# 05 — Hospital-disjoint model development, calibration and evaluation

Public source-only reproducibility notebook. Outputs and execution history were removed. No credentials, patient-level data, row-level predictions, or row-level SHAP values are included. Execution requires credentialed access to the eICU Collaborative Research Database and an authorized Google Cloud project.


In [ ]:
import os
from pathlib import Path

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT") or input("Enter your Google Cloud project ID: ").strip()
WORK_DATASET_NAME = os.environ.get("AKI_DATASET_ID", "aki_jcmc_v2")
SOURCE_DATASET = os.environ.get("EICU_SOURCE_DATASET", "physionet-data.eicu_crd")
BQ_LOCATION = os.environ.get("BIGQUERY_LOCATION", "US")
OUTPUT_ROOT = os.environ.get("AKI_OUTPUT_ROOT", "/content/AKI_JCMC_V2_PUBLIC_RUN")
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
TARGET_DATASET = f"{PROJECT_ID}.{WORK_DATASET_NAME}"

print("Target dataset:", TARGET_DATASET)
print("Output root:", OUTPUT_ROOT)


In [ ]:
from google.colab import auth, drive
import google.auth
from google.cloud import bigquery

import os
import sys
import platform
import importlib.metadata as metadata
import pandas as pd
import numpy as np
from IPython.display import display

# ------------------------------------------------------------
# 1. Google hesabı ve Drive
# ------------------------------------------------------------

auth.authenticate_user()

os.makedirs(OUTPUT_ROOT, exist_ok=True)
PROJECT_ID = globals().get("PROJECT_ID") or os.environ.get("GOOGLE_CLOUD_PROJECT") or input("Enter your Google Cloud project ID: ").strip()
SOURCE_DATASET = "physionet-data.eicu_crd"
TARGET_DATASET = f"{PROJECT_ID}.{WORK_DATASET_NAME}"
BQ_LOCATION = "US"

credentials, authenticated_project = google.auth.default(
    scopes=[
        "https://www.googleapis.com/auth/cloud-platform"
    ]
)

client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials
)

# ------------------------------------------------------------
# 2. Model çıktı klasörü
# ------------------------------------------------------------

MODEL_OUTPUT_DIR = (
    f"{OUTPUT_ROOT}/"
    "05_MODELING_OUTPUTS"
)

os.makedirs(
    MODEL_OUTPUT_DIR,
    exist_ok=True
)

# ------------------------------------------------------------
# 3. Paket sürümleri
# ------------------------------------------------------------

def installed_version(package_name):
    try:
        return metadata.version(package_name)
    except metadata.PackageNotFoundError:
        return "NOT INSTALLED"

package_names = [
    "pandas",
    "numpy",
    "scikit-learn",
    "xgboost",
    "imbalanced-learn",
    "shap",
    "google-cloud-bigquery",
    "google-cloud-bigquery-storage",
]

package_versions = pd.DataFrame(
    {
        "package": package_names,
        "version": [
            installed_version(name)
            for name in package_names
        ],
    }
)

try:
    from sklearn.model_selection import StratifiedGroupKFold
    stratified_group_available = True
    stratified_group_error = ""
except Exception as exc:
    stratified_group_available = False
    stratified_group_error = str(exc)

# ------------------------------------------------------------
# 4. BigQuery aggregate bağlantı ve görünüm kontrolü
# ------------------------------------------------------------
# "rows" BigQuery'de ayrılmış kelime olduğundan
# row_count adı kullanılmaktadır.

SQL_07A_CHECK = f"""
SELECT
  COUNT(*) AS row_count,

  COUNT(DISTINCT id_row)
    AS distinct_row_count,

  COUNT(DISTINCT group_hospital)
    AS hospital_count,

  COUNT(DISTINCT outer_fold)
    AS outer_fold_count,

  COUNTIF(label_stage23 = 1)
    AS event_count,

  COUNTIF(label_stage23 = 0)
    AS nonevent_count

FROM `{TARGET_DATASET}.feature_matrix_core_outerfold_v1`;
"""

connection_check_07A = client.query(
    SQL_07A_CHECK,
    location=BQ_LOCATION
).to_dataframe()

expected = {
    "row_count": 58491,
    "distinct_row_count": 58491,
    "hospital_count": 198,
    "outer_fold_count": 5,
    "event_count": 3032,
    "nonevent_count": 55459,
}

check_row = connection_check_07A.iloc[0]

for field, expected_value in expected.items():
    actual_value = int(check_row[field])

    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: bulunan={actual_value}, "
            f"beklenen={expected_value}"
        )

# ------------------------------------------------------------
# 5. Sonuç
# ------------------------------------------------------------

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Project:", PROJECT_ID)
print("Dataset:", TARGET_DATASET)
print("Location:", BQ_LOCATION)

print(
    "StratifiedGroupKFold available:",
    stratified_group_available
)

if not stratified_group_available:
    print(
        "StratifiedGroupKFold import error:",
        stratified_group_error
    )

print("\nPACKAGE VERSIONS")
display(package_versions)

print("\nCORE MODEL VIEW CHECK")
display(connection_check_07A)

print(
    "\n07A PASS: Modelling environment and "
    "locked outer-fold view are ready."
)

print(
    "Patient-level data have not been "
    "saved to Google Drive."
)

In [ ]:
import os
import pandas as pd
import numpy as np
from IPython.display import display

# ============================================================
# 07B — Secure in-memory loading of the locked core matrix
# ============================================================

CORE_VIEW = (
    f"{TARGET_DATASET}."
    "feature_matrix_core_outerfold_v1"
)

CONTROL_COLUMNS = [
    "id_row",
    "group_hospital",
    "label_stage23",
    "outer_fold",
]

# ------------------------------------------------------------
# 1. BigQuery şemasını oku
# ------------------------------------------------------------

SQL_07B_SCHEMA = f"""
SELECT
  column_name,
  data_type,
  ordinal_position
FROM `{TARGET_DATASET}.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'feature_matrix_core_outerfold_v1'
ORDER BY ordinal_position;
"""

core_schema_07B = client.query(
    SQL_07B_SCHEMA,
    location=BQ_LOCATION
).to_dataframe()

all_columns_07B = (
    core_schema_07B["column_name"]
    .astype(str)
    .tolist()
)

predictor_columns_07B = [
    column
    for column in all_columns_07B
    if column.startswith("x_")
]

categorical_columns_07B = (
    core_schema_07B.loc[
        core_schema_07B["column_name"].isin(
            predictor_columns_07B
        )
        & core_schema_07B["data_type"].isin(
            ["STRING", "BOOL"]
        ),
        "column_name"
    ]
    .astype(str)
    .tolist()
)

numeric_columns_07B = [
    column
    for column in predictor_columns_07B
    if column not in categorical_columns_07B
]

if len(predictor_columns_07B) != 159:
    raise RuntimeError(
        f"159 core predictor bekleniyordu; "
        f"{len(predictor_columns_07B)} bulundu."
    )

missing_control_columns = [
    column
    for column in CONTROL_COLUMNS
    if column not in all_columns_07B
]

if missing_control_columns:
    raise RuntimeError(
        "Eksik kontrol sütunları: "
        + ", ".join(missing_control_columns)
    )

for forbidden_column in [
    "patientUnitStayID",
    "hospitalID",
]:
    if forbidden_column in all_columns_07B:
        raise RuntimeError(
            f"Yasaklı doğrudan kimlik sütunu bulundu: "
            f"{forbidden_column}"
        )

# ------------------------------------------------------------
# 2. Hasta düzeyindeki matrisi yalnızca RAM'e yükle
# ------------------------------------------------------------

SQL_07B_LOAD = f"""
SELECT *
FROM `{CORE_VIEW}`;
"""

print(
    "Loading locked core matrix into Colab RAM..."
)

query_job_07B = client.query(
    SQL_07B_LOAD,
    location=BQ_LOCATION
)

try:
    core_df_07B = query_job_07B.to_dataframe(
        create_bqstorage_client=True
    )
    load_method_07B = "BigQuery Storage API"

except Exception as fast_path_error:
    print(
        "Fast path unavailable; using standard "
        "BigQuery download."
    )
    print(
        "Fast-path message:",
        type(fast_path_error).__name__
    )

    core_df_07B = query_job_07B.to_dataframe(
        create_bqstorage_client=False
    )
    load_method_07B = "Standard BigQuery API"

# ------------------------------------------------------------
# 3. Kesin bütünlük kontrolleri
# ------------------------------------------------------------

expected_shape = (
    58491,
    163
)

if core_df_07B.shape != expected_shape:
    raise RuntimeError(
        f"Beklenen şekil {expected_shape}; "
        f"bulunan {core_df_07B.shape}."
    )

if core_df_07B["id_row"].duplicated().any():
    raise RuntimeError(
        "Yinelenen id_row bulundu."
    )

if core_df_07B["id_row"].isna().any():
    raise RuntimeError(
        "Eksik id_row bulundu."
    )

if core_df_07B["group_hospital"].isna().any():
    raise RuntimeError(
        "Eksik group_hospital bulundu."
    )

if core_df_07B["label_stage23"].isna().any():
    raise RuntimeError(
        "Eksik label_stage23 bulundu."
    )

if core_df_07B["outer_fold"].isna().any():
    raise RuntimeError(
        "Eksik outer_fold bulundu."
    )

if set(
    core_df_07B["outer_fold"]
    .astype(int)
    .unique()
) != {1, 2, 3, 4, 5}:
    raise RuntimeError(
        "Outer fold değerleri 1–5 değil."
    )

hospital_fold_counts_07B = (
    core_df_07B
    .groupby("group_hospital")["outer_fold"]
    .nunique()
)

if (
    hospital_fold_counts_07B > 1
).any():
    raise RuntimeError(
        "Bir hastane birden fazla dış katta bulundu."
    )

if int(
    core_df_07B["label_stage23"].sum()
) != 3032:
    raise RuntimeError(
        "Olay sayısı 3.032 değil."
    )

# ------------------------------------------------------------
# 4. Yalnızca toplulaştırılmış özetleri oluştur
# ------------------------------------------------------------

memory_mb_07B = (
    core_df_07B
    .memory_usage(deep=True)
    .sum()
    / (1024 ** 2)
)

summary_07B = pd.DataFrame(
    {
        "metric": [
            "rows_loaded_to_ram",
            "total_columns",
            "control_columns",
            "predictor_columns",
            "numeric_predictors",
            "categorical_predictors",
            "hospitals",
            "outer_folds",
            "events",
            "nonevents",
            "event_rate",
            "duplicate_row_keys",
            "hospitals_crossing_folds",
            "ram_memory_mb",
        ],
        "value": [
            len(core_df_07B),
            core_df_07B.shape[1],
            len(CONTROL_COLUMNS),
            len(predictor_columns_07B),
            len(numeric_columns_07B),
            len(categorical_columns_07B),
            core_df_07B[
                "group_hospital"
            ].nunique(),
            core_df_07B[
                "outer_fold"
            ].nunique(),
            int(
                core_df_07B[
                    "label_stage23"
                ].sum()
            ),
            int(
                (
                    core_df_07B[
                        "label_stage23"
                    ] == 0
                ).sum()
            ),
            float(
                core_df_07B[
                    "label_stage23"
                ].mean()
            ),
            int(
                core_df_07B[
                    "id_row"
                ].duplicated().sum()
            ),
            int(
                (
                    hospital_fold_counts_07B
                    > 1
                ).sum()
            ),
            float(memory_mb_07B),
        ],
    }
)

dtype_summary_07B = (
    core_schema_07B
    .assign(
        column_role=lambda frame: np.where(
            frame["column_name"].isin(
                CONTROL_COLUMNS
            ),
            "control",
            np.where(
                frame["column_name"].str.startswith(
                    "x_"
                ),
                "predictor",
                "other",
            )
        )
    )
    .groupby(
        ["column_role", "data_type"],
        as_index=False
    )
    .size()
    .rename(columns={"size": "columns"})
)

categorical_summary_07B = pd.DataFrame(
    {
        "categorical_predictor": (
            categorical_columns_07B
        ),
        "unique_nonnull_values": [
            int(
                core_df_07B[column]
                .nunique(dropna=True)
            )
            for column in categorical_columns_07B
        ],
        "missing_count": [
            int(
                core_df_07B[column]
                .isna()
                .sum()
            )
            for column in categorical_columns_07B
        ],
        "missing_rate": [
            float(
                core_df_07B[column]
                .isna()
                .mean()
            )
            for column in categorical_columns_07B
        ],
    }
)

missingness_07B = pd.DataFrame(
    {
        "feature": predictor_columns_07B,
        "missing_count": [
            int(
                core_df_07B[column]
                .isna()
                .sum()
            )
            for column in predictor_columns_07B
        ],
        "missing_rate": [
            float(
                core_df_07B[column]
                .isna()
                .mean()
            )
            for column in predictor_columns_07B
        ],
    }
).sort_values(
    ["missing_rate", "feature"],
    ascending=[False, True]
).reset_index(drop=True)

fold_summary_07B = (
    core_df_07B
    .groupby(
        "outer_fold",
        as_index=False
    )
    .agg(
        patients=("id_row", "size"),
        hospitals=(
            "group_hospital",
            "nunique"
        ),
        events=(
            "label_stage23",
            "sum"
        ),
    )
)

fold_summary_07B["nonevents"] = (
    fold_summary_07B["patients"]
    - fold_summary_07B["events"]
)

fold_summary_07B["event_rate"] = (
    fold_summary_07B["events"]
    / fold_summary_07B["patients"]
)

# ------------------------------------------------------------
# 5. Ekranda yalnızca aggregate sonuçları göster
# ------------------------------------------------------------

print("\n07B CORE RAM LOAD SUMMARY")
display(summary_07B)

print("\n07B DATA TYPE SUMMARY")
display(dtype_summary_07B)

print("\n07B CATEGORICAL PREDICTORS")
display(categorical_summary_07B)

print("\n07B TOP 20 MISSING PREDICTORS")
display(missingness_07B.head(20))

print("\n07B OUTER FOLD SUMMARY")
display(fold_summary_07B)

print("\nLoad method:", load_method_07B)

# ------------------------------------------------------------
# 6. Yalnızca aggregate audit dosyalarını kaydet
# ------------------------------------------------------------

summary_07B.to_csv(
    os.path.join(
        MODEL_OUTPUT_DIR,
        "07B_core_ram_load_summary.csv"
    ),
    index=False
)

dtype_summary_07B.to_csv(
    os.path.join(
        MODEL_OUTPUT_DIR,
        "07B_data_type_summary.csv"
    ),
    index=False
)

categorical_summary_07B.to_csv(
    os.path.join(
        MODEL_OUTPUT_DIR,
        "07B_categorical_predictor_summary.csv"
    ),
    index=False
)

missingness_07B.to_csv(
    os.path.join(
        MODEL_OUTPUT_DIR,
        "07B_predictor_missingness_aggregate.csv"
    ),
    index=False
)

fold_summary_07B.to_csv(
    os.path.join(
        MODEL_OUTPUT_DIR,
        "07B_outer_fold_summary.csv"
    ),
    index=False
)

print(
    "\n07B PASS: Core matrix is available only "
    "in Colab RAM."
)

print(
    "No patient-level modelling matrix was "
    "written to Google Drive."
)

In [ ]:
import os
import numpy as np
import pandas as pd

from scipy import sparse

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)

from IPython.display import display

# ============================================================
# 07C — Leakage-safe preprocessing dry run
# Outer fold 1 is used only as a controlled audit.
# The preprocessor is fitted exclusively on its training set.
# ============================================================

AUDIT_OUTER_FOLD = 1

# ------------------------------------------------------------
# 1. Gerekli RAM değişkenlerini doğrula
# ------------------------------------------------------------

required_objects = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Eksik RAM değişkenleri var: "
        + ", ".join(missing_objects)
        + ". Önce 07B hücresini çalıştır."
    )

if len(predictor_columns_07B) != 159:
    raise RuntimeError(
        f"159 predictor bekleniyordu; "
        f"{len(predictor_columns_07B)} bulundu."
    )

if len(numeric_columns_07B) != 156:
    raise RuntimeError(
        f"156 sayısal predictor bekleniyordu; "
        f"{len(numeric_columns_07B)} bulundu."
    )

if len(categorical_columns_07B) != 3:
    raise RuntimeError(
        f"3 kategorik predictor bekleniyordu; "
        f"{len(categorical_columns_07B)} bulundu."
    )

# ------------------------------------------------------------
# 2. Predictor matrisini RAM içinde modellemeye hazırla
# ------------------------------------------------------------

X_core_07C = core_df_07B[
    predictor_columns_07B
].copy()

# Sayısal predictorları açık biçimde float64'e çevir.
for column in numeric_columns_07B:
    X_core_07C[column] = pd.to_numeric(
        X_core_07C[column],
        errors="coerce"
    ).astype("float64")

# Kategorik predictorları object yapısında tut.
# Eksik değerler np.nan olarak bırakılır ve pipeline içinde işlenir.
for column in categorical_columns_07B:
    categorical_series = (
        X_core_07C[column]
        .astype("object")
    )

    X_core_07C[column] = categorical_series.where(
        pd.notna(categorical_series),
        np.nan
    )

y_core_07C = (
    core_df_07B["label_stage23"]
    .astype("int8")
    .to_numpy()
)

groups_core_07C = (
    core_df_07B["group_hospital"]
    .astype(str)
    .to_numpy()
)

outer_fold_core_07C = (
    core_df_07B["outer_fold"]
    .astype(int)
    .to_numpy()
)

# ------------------------------------------------------------
# 3. Dış kat 1 eğitim ve test ayrımı
# ------------------------------------------------------------

train_mask_07C = (
    outer_fold_core_07C != AUDIT_OUTER_FOLD
)

test_mask_07C = (
    outer_fold_core_07C == AUDIT_OUTER_FOLD
)

X_train_07C = X_core_07C.loc[
    train_mask_07C
].copy()

X_test_07C = X_core_07C.loc[
    test_mask_07C
].copy()

y_train_07C = y_core_07C[
    train_mask_07C
]

y_test_07C = y_core_07C[
    test_mask_07C
]

groups_train_07C = groups_core_07C[
    train_mask_07C
]

groups_test_07C = groups_core_07C[
    test_mask_07C
]

training_hospitals_07C = set(
    groups_train_07C
)

test_hospitals_07C = set(
    groups_test_07C
)

hospital_overlap_07C = (
    training_hospitals_07C
    & test_hospitals_07C
)

if hospital_overlap_07C:
    raise RuntimeError(
        "Eğitim ve dış test kümeleri arasında "
        "hastane çakışması bulundu."
    )

expected_split = {
    "training_rows": 46803,
    "test_rows": 11688,
    "training_hospitals": 158,
    "test_hospitals": 40,
    "training_events": 2426,
    "test_events": 606,
}

actual_split = {
    "training_rows": len(X_train_07C),
    "test_rows": len(X_test_07C),
    "training_hospitals": len(
        training_hospitals_07C
    ),
    "test_hospitals": len(
        test_hospitals_07C
    ),
    "training_events": int(
        y_train_07C.sum()
    ),
    "test_events": int(
        y_test_07C.sum()
    ),
}

for metric, expected_value in expected_split.items():
    actual_value = actual_split[metric]

    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: bulunan={actual_value}, "
            f"beklenen={expected_value}"
        )

# ------------------------------------------------------------
# 4. Preprocessing üretici fonksiyonu
# ------------------------------------------------------------

def make_core_linear_preprocessor():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
            (
                "scaler",
                StandardScaler(
                    with_mean=False,
                ),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns_07B,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns_07B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )

# ------------------------------------------------------------
# 5. Yalnızca eğitim verisinde fit et
# ------------------------------------------------------------

preprocessor_linear_07C = (
    make_core_linear_preprocessor()
)

print(
    "Fitting preprocessing only on "
    "outer-fold-1 training hospitals..."
)

X_train_processed_07C = (
    preprocessor_linear_07C.fit_transform(
        X_train_07C
    )
)

X_test_processed_07C = (
    preprocessor_linear_07C.transform(
        X_test_07C
    )
)

# ------------------------------------------------------------
# 6. Dönüştürülmüş feature adları
# ------------------------------------------------------------

processed_feature_names_07C = (
    preprocessor_linear_07C
    .get_feature_names_out()
)

if (
    X_train_processed_07C.shape[1]
    != len(processed_feature_names_07C)
):
    raise RuntimeError(
        "Dönüştürülmüş sütun sayısı ile "
        "feature-name sayısı uyuşmuyor."
    )

if (
    X_test_processed_07C.shape[1]
    != X_train_processed_07C.shape[1]
):
    raise RuntimeError(
        "Eğitim ve test dönüşüm sütun "
        "sayıları farklı."
    )

if len(set(processed_feature_names_07C)) != len(
    processed_feature_names_07C
):
    raise RuntimeError(
        "Dönüştürülmüş feature adlarında "
        "yinelenme bulundu."
    )

# ------------------------------------------------------------
# 7. Eksiklik göstergelerini denetle
# ------------------------------------------------------------

numeric_imputer_07C = (
    preprocessor_linear_07C
    .named_transformers_["numeric"]
    .named_steps["imputer"]
)

indicator_indices_07C = (
    numeric_imputer_07C
    .indicator_
    .features_
)

indicator_feature_names_07C = [
    numeric_columns_07B[index]
    for index in indicator_indices_07C
]

indicator_summary_07C = pd.DataFrame(
    {
        "source_numeric_feature": (
            indicator_feature_names_07C
        )
    }
)

# ------------------------------------------------------------
# 8. Kategorik öğrenme ve bilinmeyen test kategorileri
# ------------------------------------------------------------

onehot_encoder_07C = (
    preprocessor_linear_07C
    .named_transformers_["categorical"]
    .named_steps["onehot"]
)

categorical_audit_rows_07C = []

for index, column in enumerate(
    categorical_columns_07B
):
    learned_categories = {
        str(value)
        for value in onehot_encoder_07C.categories_[
            index
        ]
    }

    test_values = (
        X_test_07C[column]
        .where(
            pd.notna(X_test_07C[column]),
            "__MISSING__",
        )
        .astype(str)
    )

    unseen_mask = ~test_values.isin(
        learned_categories
    )

    categorical_audit_rows_07C.append(
        {
            "feature": column,
            "training_categories_learned": len(
                learned_categories
            ),
            "test_unique_categories": int(
                test_values.nunique()
            ),
            "unseen_test_categories": int(
                test_values[
                    unseen_mask
                ].nunique()
            ),
            "test_rows_with_unseen_category": int(
                unseen_mask.sum()
            ),
        }
    )

categorical_audit_07C = pd.DataFrame(
    categorical_audit_rows_07C
)

# ------------------------------------------------------------
# 9. Sayısal bütünlük ve sparse matris denetimi
# ------------------------------------------------------------

def matrix_nonfinite_count(matrix):
    if sparse.issparse(matrix):
        return int(
            np.count_nonzero(
                ~np.isfinite(matrix.data)
            )
        )

    return int(
        np.count_nonzero(
            ~np.isfinite(matrix)
        )
    )

def matrix_density(matrix):
    if sparse.issparse(matrix):
        return float(
            matrix.nnz
            / (
                matrix.shape[0]
                * matrix.shape[1]
            )
        )

    return float(
        np.count_nonzero(matrix)
        / matrix.size
    )

train_nonfinite_07C = matrix_nonfinite_count(
    X_train_processed_07C
)

test_nonfinite_07C = matrix_nonfinite_count(
    X_test_processed_07C
)

if train_nonfinite_07C != 0:
    raise RuntimeError(
        "İşlenmiş eğitim matrisinde "
        "NaN veya sonsuz değer bulundu."
    )

if test_nonfinite_07C != 0:
    raise RuntimeError(
        "İşlenmiş test matrisinde "
        "NaN veya sonsuz değer bulundu."
    )

# ------------------------------------------------------------
# 10. Toplulaştırılmış özet
# ------------------------------------------------------------

summary_07C = pd.DataFrame(
    {
        "metric": [
            "audit_outer_fold",
            "training_rows",
            "test_rows",
            "training_hospitals",
            "test_hospitals",
            "hospital_overlap",
            "training_events",
            "test_events",
            "input_predictors",
            "numeric_predictors",
            "categorical_predictors",
            "numeric_missing_indicators",
            "processed_feature_columns",
            "training_matrix_sparse",
            "test_matrix_sparse",
            "training_nonfinite_values",
            "test_nonfinite_values",
            "training_matrix_density",
            "test_matrix_density",
        ],
        "value": [
            AUDIT_OUTER_FOLD,
            len(X_train_07C),
            len(X_test_07C),
            len(training_hospitals_07C),
            len(test_hospitals_07C),
            len(hospital_overlap_07C),
            int(y_train_07C.sum()),
            int(y_test_07C.sum()),
            len(predictor_columns_07B),
            len(numeric_columns_07B),
            len(categorical_columns_07B),
            len(indicator_feature_names_07C),
            X_train_processed_07C.shape[1],
            sparse.issparse(
                X_train_processed_07C
            ),
            sparse.issparse(
                X_test_processed_07C
            ),
            train_nonfinite_07C,
            test_nonfinite_07C,
            matrix_density(
                X_train_processed_07C
            ),
            matrix_density(
                X_test_processed_07C
            ),
        ],
    }
)

# ------------------------------------------------------------
# 11. Ekranda yalnızca aggregate sonuçları göster
# ------------------------------------------------------------

print("\n07C PREPROCESSING SUMMARY")
display(summary_07C)

print("\n07C CATEGORICAL ENCODING AUDIT")
display(categorical_audit_07C)

print(
    "\nNUMERIC FEATURES WITH "
    "MISSINGNESS INDICATORS:",
    len(indicator_feature_names_07C)
)

display(
    indicator_summary_07C.head(30)
)

# ------------------------------------------------------------
# 12. Yalnızca aggregate ve feature-name dosyalarını kaydet
# ------------------------------------------------------------

summary_07C.to_csv(
    os.path.join(
        MODEL_OUTPUT_DIR,
        "07C_preprocessing_summary.csv"
    ),
    index=False
)

categorical_audit_07C.to_csv(
    os.path.join(
        MODEL_OUTPUT_DIR,
        "07C_categorical_encoding_audit.csv"
    ),
    index=False
)

indicator_summary_07C.to_csv(
    os.path.join(
        MODEL_OUTPUT_DIR,
        "07C_numeric_missing_indicator_features.csv"
    ),
    index=False
)

pd.DataFrame(
    {
        "processed_feature_name": (
            processed_feature_names_07C
        )
    }
).to_csv(
    os.path.join(
        MODEL_OUTPUT_DIR,
        "07C_processed_feature_names_fold1.csv"
    ),
    index=False
)

print(
    "\n07C PASS: Preprocessing was fitted "
    "only on outer-fold-1 training hospitals."
)

print(
    "The locked test hospitals were transformed "
    "without fitting or refitting."
)

print(
    "No patient-level transformed matrix was "
    "written to Google Drive."
)

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd

from scipy.special import expit

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from sklearn.exceptions import ConvergenceWarning
from IPython.display import display

# ============================================================
# 08A-R — Runtime recovery
# Rebuild only the already-selected final LR06 model.
# The hyperparameter search is NOT repeated.
# ============================================================

required_objects_08AR = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_08AR = [
    name
    for name in required_objects_08AR
    if name not in globals()
]

if missing_objects_08AR:
    raise RuntimeError(
        "Eksik nesneler var: "
        + ", ".join(missing_objects_08AR)
        + ". Önce 07A ve 07B hücrelerini yeniden çalıştır."
    )

# ------------------------------------------------------------
# 1. 08A sırasında Drive'a kaydedilen aggregate sonuçları yükle
# ------------------------------------------------------------

candidate_path_08AR = os.path.join(
    MODEL_OUTPUT_DIR,
    "08A_logistic_candidate_results_outer1.csv",
)

selected_path_08AR = os.path.join(
    MODEL_OUTPUT_DIR,
    "08A_logistic_selected_model_outer1.csv",
)

saved_test_path_08AR = os.path.join(
    MODEL_OUTPUT_DIR,
    "08A_logistic_outer1_test_results.csv",
)

for required_path in [
    candidate_path_08AR,
    selected_path_08AR,
    saved_test_path_08AR,
]:
    if not os.path.exists(required_path):
        raise FileNotFoundError(
            "Gerekli 08A sonuç dosyası bulunamadı: "
            + required_path
        )

candidate_results_08A = pd.read_csv(
    candidate_path_08AR
)

selected_model_08A = pd.read_csv(
    selected_path_08AR
)

saved_outer_test_results_08AR = pd.read_csv(
    saved_test_path_08AR
)

if len(selected_model_08A) != 1:
    raise RuntimeError(
        "Kaydedilmiş seçili model tablosunda "
        "tam olarak bir satır bulunmalı."
    )

selected_candidate_08AR = str(
    selected_model_08A.loc[
        0,
        "selected_candidate",
    ]
)

selected_C_08AR = float(
    selected_model_08A.loc[
        0,
        "selected_C",
    ]
)

selected_l1_ratio_08AR = float(
    selected_model_08A.loc[
        0,
        "selected_l1_ratio",
    ]
)

platt_intercept_08AR = float(
    selected_model_08A.loc[
        0,
        "platt_intercept",
    ]
)

platt_slope_08AR = float(
    selected_model_08A.loc[
        0,
        "platt_slope",
    ]
)

if selected_candidate_08AR != "LR06":
    raise RuntimeError(
        "Kaydedilmiş seçili aday LR06 değil: "
        + selected_candidate_08AR
    )

if not np.isclose(
    selected_C_08AR,
    0.30,
):
    raise RuntimeError(
        "Kaydedilmiş C değeri 0.30 değil."
    )

if not np.isclose(
    selected_l1_ratio_08AR,
    0.50,
):
    raise RuntimeError(
        "Kaydedilmiş l1_ratio değeri 0.50 değil."
    )

print("Recovered selected candidate:", selected_candidate_08AR)
print("Recovered C:", selected_C_08AR)
print("Recovered l1_ratio:", selected_l1_ratio_08AR)
print("Recovered Platt intercept:", platt_intercept_08AR)
print("Recovered Platt slope:", platt_slope_08AR)

# ------------------------------------------------------------
# 2. Dış kat 1 eğitim ve test verisini yeniden hazırla
# ------------------------------------------------------------

OUTER_FOLD_08A = 1
MODEL_RANDOM_SEED_08A = 20260721

X_all_08AR = core_df_07B[
    predictor_columns_07B
].copy()

for column in numeric_columns_07B:
    X_all_08AR[column] = pd.to_numeric(
        X_all_08AR[column],
        errors="coerce",
    ).astype("float64")

for column in categorical_columns_07B:
    category_series = (
        X_all_08AR[column]
        .astype("object")
    )

    X_all_08AR[column] = category_series.where(
        pd.notna(category_series),
        np.nan,
    )

y_all_08AR = (
    core_df_07B["label_stage23"]
    .astype(int)
    .to_numpy(dtype=np.int8)
)

groups_all_08AR = (
    core_df_07B["group_hospital"]
    .astype(str)
    .to_numpy()
)

outer_all_08AR = (
    core_df_07B["outer_fold"]
    .astype(int)
    .to_numpy()
)

outer_train_mask_08A = (
    outer_all_08AR != OUTER_FOLD_08A
)

outer_test_mask_08A = (
    outer_all_08AR == OUTER_FOLD_08A
)

X_outer_train_08AR = (
    X_all_08AR.loc[
        outer_train_mask_08A
    ]
    .reset_index(drop=True)
)

X_outer_test_08AR = (
    X_all_08AR.loc[
        outer_test_mask_08A
    ]
    .reset_index(drop=True)
)

y_outer_train_08AR = y_all_08AR[
    outer_train_mask_08A
]

y_outer_test_08A = y_all_08AR[
    outer_test_mask_08A
]

groups_outer_train_08AR = groups_all_08AR[
    outer_train_mask_08A
]

groups_outer_test_08AR = groups_all_08AR[
    outer_test_mask_08A
]

if (
    set(groups_outer_train_08AR)
    & set(groups_outer_test_08AR)
):
    raise RuntimeError(
        "Dış eğitim ve test hastaneleri çakışıyor."
    )

if len(X_outer_train_08AR) != 46803:
    raise RuntimeError(
        "Dış eğitim satır sayısı 46.803 değil."
    )

if len(X_outer_test_08AR) != 11688:
    raise RuntimeError(
        "Dış test satır sayısı 11.688 değil."
    )

if int(y_outer_train_08AR.sum()) != 2426:
    raise RuntimeError(
        "Dış eğitim olay sayısı 2.426 değil."
    )

if int(y_outer_test_08A.sum()) != 606:
    raise RuntimeError(
        "Dış test olay sayısı 606 değil."
    )

# ------------------------------------------------------------
# 3. Orijinal 08A ile aynı preprocessing ve model
# ------------------------------------------------------------

def make_preprocessor_08AR():

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
            (
                "scaler",
                StandardScaler(
                    with_mean=False,
                ),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns_07B,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns_07B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


final_pipeline_08A = Pipeline(
    steps=[
        (
            "preprocessor",
            make_preprocessor_08AR(),
        ),
        (
            "model",
            LogisticRegression(
                penalty="elasticnet",
                solver="saga",
                C=selected_C_08AR,
                l1_ratio=selected_l1_ratio_08AR,
                class_weight=None,
                max_iter=5000,
                tol=1e-4,
                random_state=MODEL_RANDOM_SEED_08A,
            ),
        ),
    ]
)

# ------------------------------------------------------------
# 4. Yalnızca seçilmiş nihai modeli yeniden eğit
# ------------------------------------------------------------

print(
    "\nRebuilding only the selected LR06 final model..."
)

started_08AR = time.time()

with warnings.catch_warnings(
    record=True
) as final_warning_records_08AR:

    warnings.simplefilter(
        "always",
        ConvergenceWarning,
    )

    final_pipeline_08A.fit(
        X_outer_train_08AR,
        y_outer_train_08AR,
    )

elapsed_08AR = time.time() - started_08AR

final_convergence_warnings_08AR = sum(
    issubclass(
        warning.category,
        ConvergenceWarning,
    )
    for warning in final_warning_records_08AR
)

if final_convergence_warnings_08AR != 0:
    raise RuntimeError(
        "Yeniden oluşturulan nihai modelde "
        "yakınsama uyarısı oluştu."
    )

# ------------------------------------------------------------
# 5. Ham ve kaydedilmiş Platt katsayılı tahminler
# ------------------------------------------------------------

outer_test_raw_probabilities_08A = (
    final_pipeline_08A.predict_proba(
        X_outer_test_08AR
    )[:, 1]
)

raw_clipped_08AR = np.clip(
    outer_test_raw_probabilities_08A,
    1e-6,
    1 - 1e-6,
)

raw_logit_08AR = np.log(
    raw_clipped_08AR
    / (1 - raw_clipped_08AR)
)

outer_test_calibrated_probabilities_08A = expit(
    platt_intercept_08AR
    + platt_slope_08AR
    * raw_logit_08AR
)

for probability_array, name in [
    (
        outer_test_raw_probabilities_08A,
        "raw",
    ),
    (
        outer_test_calibrated_probabilities_08A,
        "platt",
    ),
]:
    if np.isnan(probability_array).any():
        raise RuntimeError(
            f"{name} tahminlerinde eksik değer var."
        )

    if not np.all(
        (probability_array >= 0)
        & (probability_array <= 1)
    ):
        raise RuntimeError(
            f"{name} tahminlerinde 0–1 dışında değer var."
        )

# ------------------------------------------------------------
# 6. Test metriklerini yeniden hesapla
# ------------------------------------------------------------

def probability_metrics_08AR(
    y_true,
    probabilities,
):

    probabilities = np.clip(
        np.asarray(
            probabilities,
            dtype=float,
        ),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1],
            )
        ),
        "mean_predicted_risk": float(
            probabilities.mean()
        ),
        "observed_event_rate": float(
            np.mean(y_true)
        ),
    }


def probability_logit_08AR(
    probabilities,
):

    probabilities = np.clip(
        np.asarray(
            probabilities,
            dtype=float,
        ),
        1e-6,
        1 - 1e-6,
    )

    return np.log(
        probabilities
        / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_08AR(
    y_true,
    probabilities,
):

    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )

    calibration_model.fit(
        probability_logit_08AR(
            probabilities
        ),
        y_true,
    )

    return (
        float(
            calibration_model.intercept_[0]
        ),
        float(
            calibration_model.coef_[0][0]
        ),
    )


raw_metrics_08AR = probability_metrics_08AR(
    y_outer_test_08A,
    outer_test_raw_probabilities_08A,
)

platt_metrics_08AR = probability_metrics_08AR(
    y_outer_test_08A,
    outer_test_calibrated_probabilities_08A,
)

raw_intercept_08AR, raw_slope_08AR = (
    calibration_intercept_slope_08AR(
        y_outer_test_08A,
        outer_test_raw_probabilities_08A,
    )
)

platt_intercept_test_08AR, platt_slope_test_08AR = (
    calibration_intercept_slope_08AR(
        y_outer_test_08A,
        outer_test_calibrated_probabilities_08A,
    )
)

outer_test_results_08A = pd.DataFrame(
    [
        {
            "outer_fold": 1,
            "model": "elastic_net_logistic",
            "probability_type": "raw",
            **raw_metrics_08AR,
            "calibration_intercept": (
                raw_intercept_08AR
            ),
            "calibration_slope": (
                raw_slope_08AR
            ),
        },
        {
            "outer_fold": 1,
            "model": "elastic_net_logistic",
            "probability_type": (
                "platt_calibrated"
            ),
            **platt_metrics_08AR,
            "calibration_intercept": (
                platt_intercept_test_08AR
            ),
            "calibration_slope": (
                platt_slope_test_08AR
            ),
        },
    ]
)

# ------------------------------------------------------------
# 7. Önceki kayıtla yeniden üretilebilirlik kontrolü
# ------------------------------------------------------------

comparison_rows_08AR = []

for probability_type in [
    "raw",
    "platt_calibrated",
]:

    recovered_row = (
        outer_test_results_08A.loc[
            outer_test_results_08A[
                "probability_type"
            ] == probability_type
        ]
        .iloc[0]
    )

    saved_row = (
        saved_outer_test_results_08AR.loc[
            saved_outer_test_results_08AR[
                "probability_type"
            ] == probability_type
        ]
        .iloc[0]
    )

    for metric in [
        "auroc",
        "auprc",
        "brier",
        "log_loss",
        "mean_predicted_risk",
    ]:
        comparison_rows_08AR.append(
            {
                "probability_type": probability_type,
                "metric": metric,
                "saved_value": float(
                    saved_row[metric]
                ),
                "recovered_value": float(
                    recovered_row[metric]
                ),
                "absolute_difference": abs(
                    float(saved_row[metric])
                    - float(recovered_row[metric])
                ),
            }
        )

recovery_comparison_08AR = pd.DataFrame(
    comparison_rows_08AR
)

maximum_difference_08AR = float(
    recovery_comparison_08AR[
        "absolute_difference"
    ].max()
)

# Küçük sayısal farklara izin verilir.
if maximum_difference_08AR > 0.001:
    raise RuntimeError(
        "Yeniden üretilen sonuç ile kayıtlı "
        "08A sonucu arasında beklenenden büyük fark var: "
        f"{maximum_difference_08AR}"
    )

print("\n08A-R RECOVERED TEST RESULTS")
display(outer_test_results_08A)

print("\n08A-R REPRODUCIBILITY CHECK")
display(recovery_comparison_08AR)

print("\nRebuild elapsed seconds:", elapsed_08AR)
print(
    "Maximum metric difference:",
    maximum_difference_08AR,
)

print(
    "\n08A-R PASS: Required outer-fold-1 "
    "prediction variables were restored."
)

print(
    "Now rerun the original 08B checkpoint cell."
)

In [ ]:
import os
import json
import hashlib
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from IPython.display import display

# ============================================================
# 07D — Locked nested inner hospital folds
# ============================================================

N_INNER_FOLDS = 5
INNER_CV_BASE_SEED = 20260721

# ------------------------------------------------------------
# 1. Gerekli RAM nesnelerini doğrula
# ------------------------------------------------------------

required_objects_07D = [
    "core_df_07B",
    "MODEL_OUTPUT_DIR",
]

missing_objects_07D = [
    object_name
    for object_name in required_objects_07D
    if object_name not in globals()
]

if missing_objects_07D:
    raise RuntimeError(
        "Eksik nesneler var: "
        + ", ".join(missing_objects_07D)
        + ". Önce 07B hücresini çalıştır."
    )

required_columns_07D = {
    "id_row",
    "group_hospital",
    "label_stage23",
    "outer_fold",
}

missing_columns_07D = (
    required_columns_07D
    - set(core_df_07B.columns)
)

if missing_columns_07D:
    raise RuntimeError(
        "Core RAM matrisinde eksik sütunlar var: "
        + ", ".join(sorted(missing_columns_07D))
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"58.491 satır bekleniyordu; "
        f"{len(core_df_07B)} bulundu."
    )

if (
    core_df_07B["group_hospital"]
    .nunique()
    != 198
):
    raise RuntimeError(
        "198 hastane bulunmuyor."
    )

if set(
    core_df_07B["outer_fold"]
    .astype(int)
    .unique()
) != {1, 2, 3, 4, 5}:
    raise RuntimeError(
        "Dış kat numaraları 1–5 değil."
    )

# ------------------------------------------------------------
# 2. İç katları üret
# ------------------------------------------------------------

inner_mapping_rows_07D = []
inner_summary_rows_07D = []

for outer_fold in range(1, 6):

    inner_seed = (
        INNER_CV_BASE_SEED
        + outer_fold
    )

    outer_training_df = (
        core_df_07B.loc[
            core_df_07B["outer_fold"].astype(int)
            != outer_fold,
            [
                "group_hospital",
                "label_stage23",
            ],
        ]
        .copy()
        .reset_index(drop=True)
    )

    outer_test_hospitals = set(
        core_df_07B.loc[
            core_df_07B["outer_fold"].astype(int)
            == outer_fold,
            "group_hospital",
        ].astype(str)
    )

    outer_training_df[
        "group_hospital"
    ] = (
        outer_training_df[
            "group_hospital"
        ].astype(str)
    )

    outer_training_df[
        "label_stage23"
    ] = (
        outer_training_df[
            "label_stage23"
        ].astype(int)
    )

    y_outer_train = (
        outer_training_df[
            "label_stage23"
        ].to_numpy(dtype=np.int8)
    )

    groups_outer_train = (
        outer_training_df[
            "group_hospital"
        ].to_numpy(dtype=str)
    )

    X_dummy = np.zeros(
        (
            len(outer_training_df),
            1,
        ),
        dtype=np.uint8,
    )

    inner_splitter = StratifiedGroupKFold(
        n_splits=N_INNER_FOLDS,
        shuffle=True,
        random_state=inner_seed,
    )

    hospital_to_inner_fold = {}

    for (
        inner_fold,
        (
            inner_train_index,
            inner_validation_index,
        ),
    ) in enumerate(
        inner_splitter.split(
            X_dummy,
            y_outer_train,
            groups_outer_train,
        ),
        start=1,
    ):

        inner_train_groups = set(
            groups_outer_train[
                inner_train_index
            ]
        )

        inner_validation_groups = set(
            groups_outer_train[
                inner_validation_index
            ]
        )

        group_overlap = (
            inner_train_groups
            & inner_validation_groups
        )

        if group_overlap:
            raise RuntimeError(
                f"Outer fold {outer_fold}, "
                f"inner fold {inner_fold}: "
                "hastane çakışması bulundu."
            )

        for hospital in inner_validation_groups:

            if hospital in hospital_to_inner_fold:
                raise RuntimeError(
                    f"Outer fold {outer_fold}: "
                    f"{hospital} hastanesi birden "
                    "fazla iç doğrulama katında."
                )

            hospital_to_inner_fold[
                hospital
            ] = inner_fold

        y_inner_train = y_outer_train[
            inner_train_index
        ]

        y_inner_validation = y_outer_train[
            inner_validation_index
        ]

        inner_summary_rows_07D.append(
            {
                "outer_fold": outer_fold,
                "inner_fold": inner_fold,
                "inner_cv_seed": inner_seed,

                "training_hospitals": len(
                    inner_train_groups
                ),

                "validation_hospitals": len(
                    inner_validation_groups
                ),

                "training_patients": len(
                    inner_train_index
                ),

                "validation_patients": len(
                    inner_validation_index
                ),

                "training_events": int(
                    y_inner_train.sum()
                ),

                "validation_events": int(
                    y_inner_validation.sum()
                ),

                "training_nonevents": int(
                    len(y_inner_train)
                    - y_inner_train.sum()
                ),

                "validation_nonevents": int(
                    len(y_inner_validation)
                    - y_inner_validation.sum()
                ),

                "training_event_rate": float(
                    y_inner_train.mean()
                ),

                "validation_event_rate": float(
                    y_inner_validation.mean()
                ),

                "hospital_overlap": len(
                    group_overlap
                ),
            }
        )

    # --------------------------------------------------------
    # 3. Her dış eğitim hastanesinin tam bir iç kata atanması
    # --------------------------------------------------------

    expected_training_hospitals = set(
        groups_outer_train
    )

    assigned_training_hospitals = set(
        hospital_to_inner_fold.keys()
    )

    missing_hospitals = (
        expected_training_hospitals
        - assigned_training_hospitals
    )

    extra_hospitals = (
        assigned_training_hospitals
        - expected_training_hospitals
    )

    if missing_hospitals:
        raise RuntimeError(
            f"Outer fold {outer_fold}: "
            f"{len(missing_hospitals)} eğitim "
            "hastanesine iç kat atanmadı."
        )

    if extra_hospitals:
        raise RuntimeError(
            f"Outer fold {outer_fold}: "
            "beklenmeyen hastane ataması bulundu."
        )

    if (
        assigned_training_hospitals
        & outer_test_hospitals
    ):
        raise RuntimeError(
            f"Outer fold {outer_fold}: "
            "dış test hastanesi iç CV "
            "haritasına girdi."
        )

    # Hastane düzeyindeki hasta ve olay sayıları
    outer_hospital_profile = (
        outer_training_df
        .groupby(
            "group_hospital",
            as_index=False,
        )
        .agg(
            patients=(
                "label_stage23",
                "size",
            ),
            events=(
                "label_stage23",
                "sum",
            ),
        )
    )

    outer_hospital_profile[
        "nonevents"
    ] = (
        outer_hospital_profile[
            "patients"
        ]
        - outer_hospital_profile[
            "events"
        ]
    )

    outer_hospital_profile[
        "event_rate"
    ] = (
        outer_hospital_profile[
            "events"
        ]
        / outer_hospital_profile[
            "patients"
        ]
    )

    outer_hospital_profile[
        "inner_fold"
    ] = (
        outer_hospital_profile[
            "group_hospital"
        ]
        .map(hospital_to_inner_fold)
        .astype(int)
    )

    outer_hospital_profile[
        "outer_fold"
    ] = outer_fold

    outer_hospital_profile[
        "inner_cv_seed"
    ] = inner_seed

    inner_mapping_rows_07D.extend(
        outer_hospital_profile[
            [
                "outer_fold",
                "inner_fold",
                "group_hospital",
                "patients",
                "events",
                "nonevents",
                "event_rate",
                "inner_cv_seed",
            ]
        ].to_dict(
            orient="records"
        )
    )

# ------------------------------------------------------------
# 4. Kilitli tabloları oluştur
# ------------------------------------------------------------

inner_hospital_mapping_07D = (
    pd.DataFrame(
        inner_mapping_rows_07D
    )
    .sort_values(
        [
            "outer_fold",
            "inner_fold",
            "group_hospital",
        ]
    )
    .reset_index(drop=True)
)

inner_fold_summary_07D = (
    pd.DataFrame(
        inner_summary_rows_07D
    )
    .sort_values(
        [
            "outer_fold",
            "inner_fold",
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5. Kesin bütünlük kontrolleri
# ------------------------------------------------------------

if len(
    inner_hospital_mapping_07D
) != 792:
    raise RuntimeError(
        "Toplam 792 outer-fold/hospital "
        "ataması bekleniyordu; "
        f"{len(inner_hospital_mapping_07D)} bulundu."
    )

if (
    inner_hospital_mapping_07D
    .duplicated(
        subset=[
            "outer_fold",
            "group_hospital",
        ]
    )
    .any()
):
    raise RuntimeError(
        "Bir dış eğitim setinde aynı hastane "
        "birden fazla iç kata atanmış."
    )

if len(inner_fold_summary_07D) != 25:
    raise RuntimeError(
        "25 iç kat özeti bekleniyordu."
    )

if (
    inner_fold_summary_07D[
        "hospital_overlap"
    ] != 0
).any():
    raise RuntimeError(
        "İç eğitim ve doğrulama arasında "
        "hastane çakışması bulundu."
    )

if (
    inner_fold_summary_07D[
        "validation_events"
    ] <= 0
).any():
    raise RuntimeError(
        "Olay içermeyen iç doğrulama katı var."
    )

if (
    inner_fold_summary_07D[
        "validation_nonevents"
    ] <= 0
).any():
    raise RuntimeError(
        "Olay olmayan hasta içermeyen "
        "iç doğrulama katı var."
    )

if (
    inner_fold_summary_07D[
        "training_events"
    ] <= 0
).any():
    raise RuntimeError(
        "Olay içermeyen iç eğitim katı var."
    )

# Her dış eğitim setindeki iç doğrulama katları,
# dış eğitim setinin tamamını bir kez kapsamalıdır.
for outer_fold in range(1, 6):

    outer_summary_part = (
        inner_fold_summary_07D.loc[
            inner_fold_summary_07D[
                "outer_fold"
            ] == outer_fold
        ]
    )

    expected_outer_training_rows = int(
        (
            core_df_07B[
                "outer_fold"
            ].astype(int)
            != outer_fold
        ).sum()
    )

    expected_outer_training_events = int(
        core_df_07B.loc[
            core_df_07B[
                "outer_fold"
            ].astype(int)
            != outer_fold,
            "label_stage23",
        ].sum()
    )

    expected_outer_training_hospitals = int(
        core_df_07B.loc[
            core_df_07B[
                "outer_fold"
            ].astype(int)
            != outer_fold,
            "group_hospital",
        ].nunique()
    )

    if int(
        outer_summary_part[
            "validation_patients"
        ].sum()
    ) != expected_outer_training_rows:
        raise RuntimeError(
            f"Outer fold {outer_fold}: "
            "iç doğrulama hasta toplamı hatalı."
        )

    if int(
        outer_summary_part[
            "validation_events"
        ].sum()
    ) != expected_outer_training_events:
        raise RuntimeError(
            f"Outer fold {outer_fold}: "
            "iç doğrulama olay toplamı hatalı."
        )

    if int(
        outer_summary_part[
            "validation_hospitals"
        ].sum()
    ) != expected_outer_training_hospitals:
        raise RuntimeError(
            f"Outer fold {outer_fold}: "
            "iç doğrulama hastane toplamı hatalı."
        )

# ------------------------------------------------------------
# 6. Dış kat düzeyinde iç-CV denge özeti
# ------------------------------------------------------------

outer_inner_balance_07D = (
    inner_fold_summary_07D
    .groupby(
        "outer_fold",
        as_index=False,
    )
    .agg(
        inner_folds=(
            "inner_fold",
            "nunique",
        ),

        total_validation_hospitals=(
            "validation_hospitals",
            "sum",
        ),

        minimum_validation_hospitals=(
            "validation_hospitals",
            "min",
        ),

        maximum_validation_hospitals=(
            "validation_hospitals",
            "max",
        ),

        minimum_validation_patients=(
            "validation_patients",
            "min",
        ),

        maximum_validation_patients=(
            "validation_patients",
            "max",
        ),

        minimum_validation_events=(
            "validation_events",
            "min",
        ),

        maximum_validation_events=(
            "validation_events",
            "max",
        ),

        minimum_validation_event_rate=(
            "validation_event_rate",
            "min",
        ),

        maximum_validation_event_rate=(
            "validation_event_rate",
            "max",
        ),

        maximum_hospital_overlap=(
            "hospital_overlap",
            "max",
        ),
    )
)

# ------------------------------------------------------------
# 7. Dosyaları kilitle ve SHA-256 üret
# ------------------------------------------------------------

mapping_path_07D = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv"
)

summary_path_07D = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_inner_fold_summary.csv"
)

balance_path_07D = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_outer_inner_balance_summary.csv"
)

config_path_07D = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_inner_cv_configuration.json"
)

sha_path_07D = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1_SHA256.txt"
)

inner_hospital_mapping_07D.to_csv(
    mapping_path_07D,
    index=False,
)

inner_fold_summary_07D.to_csv(
    summary_path_07D,
    index=False,
)

outer_inner_balance_07D.to_csv(
    balance_path_07D,
    index=False,
)

configuration_07D = {
    "method": "StratifiedGroupKFold",
    "n_inner_folds": N_INNER_FOLDS,
    "shuffle": True,
    "base_random_seed": INNER_CV_BASE_SEED,
    "outer_specific_seed_rule": (
        "base_random_seed + outer_fold"
    ),
    "group_variable": "group_hospital",
    "target_variable": "label_stage23",
    "outer_fold_variable": "outer_fold",
}

with open(
    config_path_07D,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        configuration_07D,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(
    mapping_path_07D,
    "rb",
) as file_handle:
    inner_mapping_sha256_07D = (
        hashlib.sha256(
            file_handle.read()
        ).hexdigest()
    )

with open(
    sha_path_07D,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(
        inner_mapping_sha256_07D
        + "\n"
    )

# ------------------------------------------------------------
# 8. Aggregate sonuçları göster
# ------------------------------------------------------------

print("\n07D INNER FOLD SUMMARY")
display(inner_fold_summary_07D)

print("\n07D OUTER–INNER BALANCE SUMMARY")
display(outer_inner_balance_07D)

print("\nLocked inner-fold mapping SHA-256:")
print(inner_mapping_sha256_07D)

print("\nSaved:")
print(mapping_path_07D)
print(summary_path_07D)
print(balance_path_07D)
print(config_path_07D)
print(sha_path_07D)

print(
    "\n07D PASS: Five nested hospital-level "
    "inner folds were locked inside every "
    "outer training set."
)

print(
    "No outer-test hospital entered inner "
    "cross-validation."
)

print(
    "No patient-level fold assignment file "
    "was written to Google Drive."
)

In [ ]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from sklearn.exceptions import ConvergenceWarning
from IPython.display import display

# ============================================================
# 08A — Nested elastic-net logistic regression dry run
# Controlled audit on outer fold 1
# ============================================================

OUTER_FOLD_08A = 1
MODEL_RANDOM_SEED_08A = 20260721

required_objects_08A = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "inner_hospital_mapping_07D",
    "MODEL_OUTPUT_DIR",
]

missing_objects_08A = [
    name
    for name in required_objects_08A
    if name not in globals()
]

if missing_objects_08A:
    raise RuntimeError(
        "Eksik RAM nesneleri var: "
        + ", ".join(missing_objects_08A)
    )

# ------------------------------------------------------------
# 1. Model matrisi
# ------------------------------------------------------------

X_all_08A = core_df_07B[
    predictor_columns_07B
].copy()

for column in numeric_columns_07B:
    X_all_08A[column] = pd.to_numeric(
        X_all_08A[column],
        errors="coerce",
    ).astype("float64")

for column in categorical_columns_07B:
    series = X_all_08A[column].astype("object")

    X_all_08A[column] = series.where(
        pd.notna(series),
        np.nan,
    )

y_all_08A = (
    core_df_07B["label_stage23"]
    .astype(int)
    .to_numpy(dtype=np.int8)
)

groups_all_08A = (
    core_df_07B["group_hospital"]
    .astype(str)
    .to_numpy()
)

outer_all_08A = (
    core_df_07B["outer_fold"]
    .astype(int)
    .to_numpy()
)

outer_train_mask_08A = (
    outer_all_08A != OUTER_FOLD_08A
)

outer_test_mask_08A = (
    outer_all_08A == OUTER_FOLD_08A
)

X_outer_train_08A = (
    X_all_08A.loc[
        outer_train_mask_08A
    ]
    .reset_index(drop=True)
)

X_outer_test_08A = (
    X_all_08A.loc[
        outer_test_mask_08A
    ]
    .reset_index(drop=True)
)

y_outer_train_08A = y_all_08A[
    outer_train_mask_08A
]

y_outer_test_08A = y_all_08A[
    outer_test_mask_08A
]

groups_outer_train_08A = groups_all_08A[
    outer_train_mask_08A
]

groups_outer_test_08A = groups_all_08A[
    outer_test_mask_08A
]

if (
    set(groups_outer_train_08A)
    & set(groups_outer_test_08A)
):
    raise RuntimeError(
        "Dış eğitim ve test hastaneleri çakışıyor."
    )

# ------------------------------------------------------------
# 2. Kilitli iç kat haritasını uygula
# ------------------------------------------------------------

mapping_part_08A = (
    inner_hospital_mapping_07D.loc[
        inner_hospital_mapping_07D[
            "outer_fold"
        ].astype(int) == OUTER_FOLD_08A,
        [
            "group_hospital",
            "inner_fold",
        ],
    ]
    .copy()
)

mapping_part_08A[
    "group_hospital"
] = mapping_part_08A[
    "group_hospital"
].astype(str)

hospital_to_inner_fold_08A = dict(
    zip(
        mapping_part_08A["group_hospital"],
        mapping_part_08A["inner_fold"].astype(int),
    )
)

inner_fold_vector_08A = np.array(
    [
        hospital_to_inner_fold_08A.get(
            hospital,
            -1,
        )
        for hospital in groups_outer_train_08A
    ],
    dtype=int,
)

if (
    inner_fold_vector_08A == -1
).any():
    raise RuntimeError(
        "Bazı dış eğitim hastanelerine "
        "kilitli iç kat atanmadı."
    )

if set(
    np.unique(inner_fold_vector_08A)
) != {1, 2, 3, 4, 5}:
    raise RuntimeError(
        "İç kat değerleri 1–5 değil."
    )

# ------------------------------------------------------------
# 3. Preprocessing ve model üreticileri
# ------------------------------------------------------------

def make_preprocessor_08A():

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
            (
                "scaler",
                StandardScaler(
                    with_mean=False,
                ),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns_07B,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns_07B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_logistic_pipeline_08A(
    C_value,
    l1_ratio_value,
):

    model = LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=float(C_value),
        l1_ratio=float(l1_ratio_value),
        class_weight=None,
        max_iter=5000,
        tol=1e-4,
        random_state=MODEL_RANDOM_SEED_08A,
    )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                make_preprocessor_08A(),
            ),
            (
                "model",
                model,
            ),
        ]
    )

# ------------------------------------------------------------
# 4. Önceden belirlenmiş sınırlı hiperparametre ızgarası
# ------------------------------------------------------------

candidate_grid_08A = [
    {
        "candidate_id": "LR01",
        "C": 0.03,
        "l1_ratio": 0.00,
    },
    {
        "candidate_id": "LR02",
        "C": 0.10,
        "l1_ratio": 0.00,
    },
    {
        "candidate_id": "LR03",
        "C": 0.30,
        "l1_ratio": 0.00,
    },
    {
        "candidate_id": "LR04",
        "C": 0.10,
        "l1_ratio": 0.25,
    },
    {
        "candidate_id": "LR05",
        "C": 0.30,
        "l1_ratio": 0.25,
    },
    {
        "candidate_id": "LR06",
        "C": 0.30,
        "l1_ratio": 0.50,
    },
]

# ------------------------------------------------------------
# 5. Metrik yardımcıları
# ------------------------------------------------------------

def probability_metrics_08A(
    y_true,
    probabilities,
):

    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1],
            )
        ),
        "mean_predicted_risk": float(
            probabilities.mean()
        ),
        "observed_event_rate": float(
            np.mean(y_true)
        ),
    }


def probability_logit_08A(probabilities):

    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )

    return np.log(
        probabilities
        / (1 - probabilities)
    ).reshape(-1, 1)


def fit_platt_calibrator_08A(
    y_true,
    raw_probabilities,
):

    calibrator = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )

    calibrator.fit(
        probability_logit_08A(
            raw_probabilities
        ),
        y_true,
    )

    return calibrator


def calibration_intercept_slope_08A(
    y_true,
    probabilities,
):

    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )

    calibration_model.fit(
        probability_logit_08A(
            probabilities
        ),
        y_true,
    )

    return (
        float(
            calibration_model.intercept_[0]
        ),
        float(
            calibration_model.coef_[0][0]
        ),
    )

# ------------------------------------------------------------
# 6. İç out-of-fold hiperparametre değerlendirmesi
# ------------------------------------------------------------

candidate_results_08A = []
candidate_oof_predictions_08A = {}

for candidate in candidate_grid_08A:

    candidate_id = candidate[
        "candidate_id"
    ]

    print(
        "Evaluating",
        candidate_id,
        "| C =",
        candidate["C"],
        "| l1_ratio =",
        candidate["l1_ratio"],
    )

    started = time.time()

    oof_predictions = np.full(
        len(y_outer_train_08A),
        np.nan,
        dtype=float,
    )

    convergence_warning_count = 0

    for inner_fold in range(1, 6):

        inner_train_mask = (
            inner_fold_vector_08A
            != inner_fold
        )

        inner_validation_mask = (
            inner_fold_vector_08A
            == inner_fold
        )

        training_hospitals = set(
            groups_outer_train_08A[
                inner_train_mask
            ]
        )

        validation_hospitals = set(
            groups_outer_train_08A[
                inner_validation_mask
            ]
        )

        if (
            training_hospitals
            & validation_hospitals
        ):
            raise RuntimeError(
                f"{candidate_id}, inner fold "
                f"{inner_fold}: hastane çakışması."
            )

        pipeline = make_logistic_pipeline_08A(
            candidate["C"],
            candidate["l1_ratio"],
        )

        with warnings.catch_warnings(
            record=True
        ) as warning_records:

            warnings.simplefilter(
                "always",
                ConvergenceWarning,
            )

            pipeline.fit(
                X_outer_train_08A.loc[
                    inner_train_mask
                ],
                y_outer_train_08A[
                    inner_train_mask
                ],
            )

        convergence_warning_count += sum(
            issubclass(
                warning.category,
                ConvergenceWarning,
            )
            for warning in warning_records
        )

        oof_predictions[
            inner_validation_mask
        ] = pipeline.predict_proba(
            X_outer_train_08A.loc[
                inner_validation_mask
            ]
        )[:, 1]

    if np.isnan(
        oof_predictions
    ).any():
        raise RuntimeError(
            f"{candidate_id}: bazı OOF "
            "tahminleri eksik."
        )

    metrics = probability_metrics_08A(
        y_outer_train_08A,
        oof_predictions,
    )

    elapsed_seconds = (
        time.time() - started
    )

    candidate_results_08A.append(
        {
            "candidate_id": candidate_id,
            "C": candidate["C"],
            "l1_ratio": candidate[
                "l1_ratio"
            ],
            **metrics,
            "convergence_warnings": (
                convergence_warning_count
            ),
            "elapsed_seconds": float(
                elapsed_seconds
            ),
        }
    )

    candidate_oof_predictions_08A[
        candidate_id
    ] = oof_predictions

candidate_results_08A = pd.DataFrame(
    candidate_results_08A
)

candidate_results_08A = (
    candidate_results_08A
    .sort_values(
        [
            "auprc",
            "auroc",
            "brier",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

candidate_results_08A[
    "selection_rank"
] = (
    np.arange(
        1,
        len(candidate_results_08A) + 1
    )
)

best_row_08A = (
    candidate_results_08A.iloc[0]
)

best_candidate_id_08A = str(
    best_row_08A["candidate_id"]
)

best_C_08A = float(
    best_row_08A["C"]
)

best_l1_ratio_08A = float(
    best_row_08A["l1_ratio"]
)

best_oof_predictions_08A = (
    candidate_oof_predictions_08A[
        best_candidate_id_08A
    ]
)

# ------------------------------------------------------------
# 7. Nested kalibrasyon
# ------------------------------------------------------------

platt_calibrator_08A = (
    fit_platt_calibrator_08A(
        y_outer_train_08A,
        best_oof_predictions_08A,
    )
)

final_pipeline_08A = (
    make_logistic_pipeline_08A(
        best_C_08A,
        best_l1_ratio_08A,
    )
)

print(
    "\nFitting selected model on the complete "
    "outer-fold-1 training set..."
)

with warnings.catch_warnings(
    record=True
) as final_warning_records:

    warnings.simplefilter(
        "always",
        ConvergenceWarning,
    )

    final_pipeline_08A.fit(
        X_outer_train_08A,
        y_outer_train_08A,
    )

final_convergence_warnings_08A = sum(
    issubclass(
        warning.category,
        ConvergenceWarning,
    )
    for warning in final_warning_records
)

outer_test_raw_probabilities_08A = (
    final_pipeline_08A.predict_proba(
        X_outer_test_08A
    )[:, 1]
)

outer_test_calibrated_probabilities_08A = (
    platt_calibrator_08A.predict_proba(
        probability_logit_08A(
            outer_test_raw_probabilities_08A
        )
    )[:, 1]
)

# ------------------------------------------------------------
# 8. Dış test sonuçları
# ------------------------------------------------------------

raw_test_metrics_08A = (
    probability_metrics_08A(
        y_outer_test_08A,
        outer_test_raw_probabilities_08A,
    )
)

calibrated_test_metrics_08A = (
    probability_metrics_08A(
        y_outer_test_08A,
        outer_test_calibrated_probabilities_08A,
    )
)

raw_calibration_intercept_08A, \
raw_calibration_slope_08A = (
    calibration_intercept_slope_08A(
        y_outer_test_08A,
        outer_test_raw_probabilities_08A,
    )
)

cal_calibration_intercept_08A, \
cal_calibration_slope_08A = (
    calibration_intercept_slope_08A(
        y_outer_test_08A,
        outer_test_calibrated_probabilities_08A,
    )
)

outer_test_results_08A = pd.DataFrame(
    [
        {
            "outer_fold": OUTER_FOLD_08A,
            "model": "elastic_net_logistic",
            "probability_type": "raw",
            **raw_test_metrics_08A,
            "calibration_intercept": (
                raw_calibration_intercept_08A
            ),
            "calibration_slope": (
                raw_calibration_slope_08A
            ),
        },
        {
            "outer_fold": OUTER_FOLD_08A,
            "model": "elastic_net_logistic",
            "probability_type": "platt_calibrated",
            **calibrated_test_metrics_08A,
            "calibration_intercept": (
                cal_calibration_intercept_08A
            ),
            "calibration_slope": (
                cal_calibration_slope_08A
            ),
        },
    ]
)

selected_model_08A = pd.DataFrame(
    [
        {
            "outer_fold": OUTER_FOLD_08A,
            "selected_candidate": (
                best_candidate_id_08A
            ),
            "selected_C": best_C_08A,
            "selected_l1_ratio": (
                best_l1_ratio_08A
            ),
            "selection_metric_primary": (
                "pooled_inner_oof_auprc"
            ),
            "inner_oof_auprc": float(
                best_row_08A["auprc"]
            ),
            "inner_oof_auroc": float(
                best_row_08A["auroc"]
            ),
            "inner_oof_brier": float(
                best_row_08A["brier"]
            ),
            "final_convergence_warnings": (
                final_convergence_warnings_08A
            ),
            "platt_intercept": float(
                platt_calibrator_08A.intercept_[0]
            ),
            "platt_slope": float(
                platt_calibrator_08A.coef_[0][0]
            ),
        }
    ]
)

# ------------------------------------------------------------
# 9. Göster ve toplulaştırılmış sonuçları kaydet
# ------------------------------------------------------------

print("\n08A INNER OOF CANDIDATE RESULTS")
display(candidate_results_08A)

print("\n08A SELECTED MODEL")
display(selected_model_08A)

print("\n08A OUTER-FOLD-1 TEST RESULTS")
display(outer_test_results_08A)

candidate_results_08A.to_csv(
    os.path.join(
        MODEL_OUTPUT_DIR,
        "08A_logistic_candidate_results_outer1.csv",
    ),
    index=False,
)

selected_model_08A.to_csv(
    os.path.join(
        MODEL_OUTPUT_DIR,
        "08A_logistic_selected_model_outer1.csv",
    ),
    index=False,
)

outer_test_results_08A.to_csv(
    os.path.join(
        MODEL_OUTPUT_DIR,
        "08A_logistic_outer1_test_results.csv",
    ),
    index=False,
)

configuration_08A = {
    "outer_fold": OUTER_FOLD_08A,
    "model": "elastic_net_logistic_regression",
    "primary_selection_metric": (
        "pooled inner out-of-fold AUPRC"
    ),
    "secondary_selection_metrics": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "calibration": (
        "Platt calibration learned from "
        "best-candidate inner OOF predictions"
    ),
    "class_weight": None,
    "random_seed": MODEL_RANDOM_SEED_08A,
    "candidate_grid": candidate_grid_08A,
}

with open(
    os.path.join(
        MODEL_OUTPUT_DIR,
        "08A_logistic_configuration_outer1.json",
    ),
    "w",
    encoding="utf-8",
) as file_handle:

    json.dump(
        configuration_08A,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

print(
    "\n08A PASS: Nested logistic-regression "
    "dry run completed on outer fold 1."
)

print(
    "Hyperparameters and calibration were learned "
    "without using outer-test outcomes."
)

print(
    "No patient-level predictions were written "
    "to Google Drive."
)

In [ ]:
import os
import json
import hashlib
import numpy as np
import pandas as pd

from google.cloud import bigquery
from IPython.display import display

# ============================================================
# 08B — Secure checkpoint of outer-fold-1 predictions
# ============================================================

required_objects_08B = [
    "core_df_07B",
    "outer_test_mask_08A",
    "y_outer_test_08A",
    "outer_test_raw_probabilities_08A",
    "outer_test_calibrated_probabilities_08A",
    "candidate_results_08A",
    "selected_model_08A",
    "outer_test_results_08A",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
    "MODEL_OUTPUT_DIR",
]

missing_objects_08B = [
    name
    for name in required_objects_08B
    if name not in globals()
]

if missing_objects_08B:
    raise RuntimeError(
        "Eksik RAM nesneleri var: "
        + ", ".join(missing_objects_08B)
        + ". Önce 08A hücresini çalıştır."
    )

# ------------------------------------------------------------
# 1. Dış kat 1 tahmin tablosunu RAM'de oluştur
# ------------------------------------------------------------

id_values_08B = (
    core_df_07B.loc[
        outer_test_mask_08A,
        "id_row",
    ]
    .astype(str)
    .to_numpy()
)

prediction_outer1_08B = pd.DataFrame(
    {
        "id_row": id_values_08B,
        "outer_fold": np.full(
            len(id_values_08B),
            1,
            dtype=np.int64,
        ),
        "label_stage23": np.asarray(
            y_outer_test_08A,
            dtype=np.int64,
        ),
        "prediction_raw": np.asarray(
            outer_test_raw_probabilities_08A,
            dtype=np.float64,
        ),
        "prediction_platt": np.asarray(
            outer_test_calibrated_probabilities_08A,
            dtype=np.float64,
        ),
        "model_name": (
            "elastic_net_logistic"
        ),
        "model_version": (
            "core_v1_nested_cv"
        ),
    }
)

# ------------------------------------------------------------
# 2. Kesin bütünlük kontrolleri
# ------------------------------------------------------------

if len(prediction_outer1_08B) != 11688:
    raise RuntimeError(
        f"11.688 tahmin bekleniyordu; "
        f"{len(prediction_outer1_08B)} bulundu."
    )

if prediction_outer1_08B[
    "id_row"
].duplicated().any():
    raise RuntimeError(
        "Tahmin tablosunda yinelenen id_row var."
    )

if int(
    prediction_outer1_08B[
        "label_stage23"
    ].sum()
) != 606:
    raise RuntimeError(
        "Dış kat 1 olay sayısı 606 değil."
    )

probability_columns_08B = [
    "prediction_raw",
    "prediction_platt",
]

for column in probability_columns_08B:

    if prediction_outer1_08B[
        column
    ].isna().any():
        raise RuntimeError(
            f"{column} içinde eksik tahmin var."
        )

    if not prediction_outer1_08B[
        column
    ].between(0, 1).all():
        raise RuntimeError(
            f"{column} içinde 0–1 dışında değer var."
        )

# ------------------------------------------------------------
# 3. BigQuery güvenli tahmin tablosuna yükle
# ------------------------------------------------------------

prediction_table_id_08B = (
    f"{TARGET_DATASET}."
    "model_lr_outer_predictions_v1"
)

prediction_job_config_08B = (
    bigquery.LoadJobConfig(
        schema=[
            bigquery.SchemaField(
                "id_row",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "outer_fold",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "label_stage23",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_raw",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_platt",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_name",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_version",
                "STRING",
                mode="REQUIRED",
            ),
        ],
        write_disposition=(
            bigquery.WriteDisposition.WRITE_TRUNCATE
        ),
    )
)

print(
    "Uploading secure outer-fold-1 "
    "prediction checkpoint:"
)

print(prediction_table_id_08B)

load_job_08B = (
    client.load_table_from_dataframe(
        prediction_outer1_08B,
        prediction_table_id_08B,
        job_config=prediction_job_config_08B,
        location=BQ_LOCATION,
    )
)

load_job_08B.result()

# ------------------------------------------------------------
# 4. BigQuery yüklemesini aggregate olarak doğrula
# ------------------------------------------------------------

SQL_08B_VERIFY = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  MIN(prediction_raw) AS minimum_raw_probability,
  MAX(prediction_raw) AS maximum_raw_probability,
  MIN(prediction_platt) AS minimum_platt_probability,
  MAX(prediction_platt) AS maximum_platt_probability
FROM `{prediction_table_id_08B}`;
"""

verification_08B = client.query(
    SQL_08B_VERIFY,
    location=BQ_LOCATION,
).to_dataframe()

verification_row_08B = (
    verification_08B.iloc[0]
)

expected_verification_08B = {
    "prediction_rows": 11688,
    "distinct_rows": 11688,
    "outer_folds": 1,
    "events": 606,
    "nonevents": 11082,
}

for field, expected_value in (
    expected_verification_08B.items()
):
    actual_value = int(
        verification_row_08B[field]
    )

    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: bulunan={actual_value}, "
            f"beklenen={expected_value}"
        )

# ------------------------------------------------------------
# 5. Model protokolünü kilitle
# ------------------------------------------------------------

locked_protocol_08B = {
    "model_family": (
        "elastic_net_logistic_regression"
    ),
    "predictor_set": (
        "159-variable locked core predictor set"
    ),
    "outer_validation": (
        "five locked hospital-disjoint "
        "outer folds"
    ),
    "inner_validation": (
        "five locked hospital-disjoint "
        "inner folds within each outer "
        "training set"
    ),
    "selection_metric_primary": (
        "pooled inner out-of-fold AUPRC"
    ),
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "candidate_grid": [
        {
            "candidate_id": "LR01",
            "C": 0.03,
            "l1_ratio": 0.00,
        },
        {
            "candidate_id": "LR02",
            "C": 0.10,
            "l1_ratio": 0.00,
        },
        {
            "candidate_id": "LR03",
            "C": 0.30,
            "l1_ratio": 0.00,
        },
        {
            "candidate_id": "LR04",
            "C": 0.10,
            "l1_ratio": 0.25,
        },
        {
            "candidate_id": "LR05",
            "C": 0.30,
            "l1_ratio": 0.25,
        },
        {
            "candidate_id": "LR06",
            "C": 0.30,
            "l1_ratio": 0.50,
        },
    ],
    "solver": "saga",
    "maximum_iterations": 5000,
    "tolerance": 0.0001,
    "class_weight": None,
    "numeric_imputation": (
        "median learned only from each "
        "training partition"
    ),
    "missingness_indicators": True,
    "numeric_scaling": (
        "StandardScaler with_mean=False"
    ),
    "categorical_imputation": "__MISSING__",
    "categorical_encoding": (
        "OneHotEncoder handle_unknown=ignore"
    ),
    "calibration": (
        "Platt scaling learned from the "
        "selected candidate's pooled "
        "inner out-of-fold predictions"
    ),
    "outer_fold_1_selected_candidate": (
        str(
            selected_model_08A.loc[
                0,
                "selected_candidate",
            ]
        )
    ),
    "protocol_change_after_outer_fold_1": False,
}

protocol_path_08B = os.path.join(
    MODEL_OUTPUT_DIR,
    "08B_locked_logistic_model_protocol_v1.json",
)

with open(
    protocol_path_08B,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        locked_protocol_08B,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(
    protocol_path_08B,
    "rb",
) as file_handle:
    protocol_sha256_08B = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

protocol_sha_path_08B = os.path.join(
    MODEL_OUTPUT_DIR,
    "08B_locked_logistic_model_protocol_v1_SHA256.txt",
)

with open(
    protocol_sha_path_08B,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(
        protocol_sha256_08B + "\n"
    )

# ------------------------------------------------------------
# 6. Yalnızca aggregate sonuçları göster
# ------------------------------------------------------------

checkpoint_summary_08B = pd.DataFrame(
    {
        "metric": [
            "prediction_rows_in_bigquery",
            "distinct_prediction_rows",
            "outer_folds_checkpointed",
            "events",
            "nonevents",
            "selected_candidate",
            "selected_C",
            "selected_l1_ratio",
            "raw_test_auroc",
            "raw_test_auprc",
            "platt_test_auroc",
            "platt_test_auprc",
        ],
        "value": [
            int(
                verification_row_08B[
                    "prediction_rows"
                ]
            ),
            int(
                verification_row_08B[
                    "distinct_rows"
                ]
            ),
            int(
                verification_row_08B[
                    "outer_folds"
                ]
            ),
            int(
                verification_row_08B[
                    "events"
                ]
            ),
            int(
                verification_row_08B[
                    "nonevents"
                ]
            ),
            selected_model_08A.loc[
                0,
                "selected_candidate",
            ],
            float(
                selected_model_08A.loc[
                    0,
                    "selected_C",
                ]
            ),
            float(
                selected_model_08A.loc[
                    0,
                    "selected_l1_ratio",
                ]
            ),
            float(
                outer_test_results_08A.loc[
                    outer_test_results_08A[
                        "probability_type"
                    ] == "raw",
                    "auroc",
                ].iloc[0]
            ),
            float(
                outer_test_results_08A.loc[
                    outer_test_results_08A[
                        "probability_type"
                    ] == "raw",
                    "auprc",
                ].iloc[0]
            ),
            float(
                outer_test_results_08A.loc[
                    outer_test_results_08A[
                        "probability_type"
                    ] == "platt_calibrated",
                    "auroc",
                ].iloc[0]
            ),
            float(
                outer_test_results_08A.loc[
                    outer_test_results_08A[
                        "probability_type"
                    ] == "platt_calibrated",
                    "auprc",
                ].iloc[0]
            ),
        ],
    }
)

print("\n08B SECURE CHECKPOINT SUMMARY")
display(checkpoint_summary_08B)

print("\n08B BIGQUERY VERIFICATION")
display(verification_08B)

print("\nLocked protocol SHA-256:")
print(protocol_sha256_08B)

print("\nSaved protocol:")
print(protocol_path_08B)
print(protocol_sha_path_08B)

print(
    "\n08B PASS: Outer-fold-1 predictions "
    "were securely checkpointed in BigQuery."
)

print(
    "The logistic model protocol is now locked "
    "and will not be modified for outer folds 2–5."
)

print(
    "No patient-level prediction file was "
    "written to Google Drive."
)

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from google.cloud import bigquery

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from sklearn.exceptions import ConvergenceWarning
from IPython.display import display

# ============================================================
# 08C1 — Resume-safe nested inner search for outer fold 2
#
# - Fixed six-candidate grid
# - Five locked hospital-disjoint inner folds
# - Preprocessing fitted once per inner training partition
# - Candidate OOF predictions checkpointed in BigQuery
# - Patient-level predictions are NOT written to Drive
# ============================================================

TARGET_OUTER_FOLD_08C1 = 2
MODEL_RANDOM_SEED_08C1 = 20260721

# ------------------------------------------------------------
# 1. Gerekli çalışma nesnelerini doğrula
# ------------------------------------------------------------

required_objects_08C1 = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_08C1 = [
    name
    for name in required_objects_08C1
    if name not in globals()
]

if missing_objects_08C1:
    raise RuntimeError(
        "Eksik RAM nesneleri var: "
        + ", ".join(missing_objects_08C1)
        + ". Önce yalnızca 07A ve 07B hücrelerini çalıştır."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"58.491 satır bekleniyordu; "
        f"{len(core_df_07B)} bulundu."
    )

if len(predictor_columns_07B) != 159:
    raise RuntimeError(
        "Core predictor sayısı 159 değil."
    )

if len(numeric_columns_07B) != 156:
    raise RuntimeError(
        "Sayısal predictor sayısı 156 değil."
    )

if len(categorical_columns_07B) != 3:
    raise RuntimeError(
        "Kategorik predictor sayısı 3 değil."
    )

# ------------------------------------------------------------
# 2. Kilitli iç hastane katlarını Drive'dan yükle
# ------------------------------------------------------------

inner_mapping_path_08C1 = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_08C1):
    raise FileNotFoundError(
        "Kilitli iç kat dosyası bulunamadı: "
        + inner_mapping_path_08C1
    )

inner_mapping_all_08C1 = pd.read_csv(
    inner_mapping_path_08C1,
    dtype={
        "group_hospital": str,
    },
)

inner_mapping_part_08C1 = (
    inner_mapping_all_08C1.loc[
        inner_mapping_all_08C1[
            "outer_fold"
        ].astype(int)
        == TARGET_OUTER_FOLD_08C1,
        [
            "group_hospital",
            "inner_fold",
        ],
    ]
    .copy()
)

inner_mapping_part_08C1[
    "group_hospital"
] = (
    inner_mapping_part_08C1[
        "group_hospital"
    ].astype(str)
)

inner_mapping_part_08C1[
    "inner_fold"
] = (
    inner_mapping_part_08C1[
        "inner_fold"
    ].astype(int)
)

if len(inner_mapping_part_08C1) != 158:
    raise RuntimeError(
        "Dış kat 2 eğitim kümesi için "
        "158 hastane ataması bekleniyordu."
    )

if inner_mapping_part_08C1[
    "group_hospital"
].duplicated().any():
    raise RuntimeError(
        "İç kat haritasında yinelenen hastane var."
    )

hospital_to_inner_fold_08C1 = dict(
    zip(
        inner_mapping_part_08C1[
            "group_hospital"
        ],
        inner_mapping_part_08C1[
            "inner_fold"
        ],
    )
)

# ------------------------------------------------------------
# 3. Dış kat 2 eğitim matrisi
# ------------------------------------------------------------

X_all_08C1 = core_df_07B[
    predictor_columns_07B
].copy()

for column in numeric_columns_07B:
    X_all_08C1[column] = pd.to_numeric(
        X_all_08C1[column],
        errors="coerce",
    ).astype("float64")

for column in categorical_columns_07B:
    categorical_series = (
        X_all_08C1[column]
        .astype("object")
    )

    X_all_08C1[column] = categorical_series.where(
        pd.notna(categorical_series),
        np.nan,
    )

outer_fold_vector_all_08C1 = (
    core_df_07B["outer_fold"]
    .astype(int)
    .to_numpy()
)

outer_training_mask_08C1 = (
    outer_fold_vector_all_08C1
    != TARGET_OUTER_FOLD_08C1
)

outer_test_mask_08C1 = (
    outer_fold_vector_all_08C1
    == TARGET_OUTER_FOLD_08C1
)

X_outer_training_08C1 = (
    X_all_08C1.loc[
        outer_training_mask_08C1
    ]
    .reset_index(drop=True)
)

outer_training_meta_08C1 = (
    core_df_07B.loc[
        outer_training_mask_08C1,
        [
            "id_row",
            "group_hospital",
            "label_stage23",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_08C1 = (
    core_df_07B.loc[
        outer_test_mask_08C1,
        [
            "id_row",
            "group_hospital",
            "label_stage23",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_training_meta_08C1[
    "id_row"
] = outer_training_meta_08C1[
    "id_row"
].astype(str)

outer_training_meta_08C1[
    "group_hospital"
] = outer_training_meta_08C1[
    "group_hospital"
].astype(str)

outer_training_meta_08C1[
    "label_stage23"
] = outer_training_meta_08C1[
    "label_stage23"
].astype(int)

outer_test_meta_08C1[
    "group_hospital"
] = outer_test_meta_08C1[
    "group_hospital"
].astype(str)

y_outer_training_08C1 = (
    outer_training_meta_08C1[
        "label_stage23"
    ]
    .to_numpy(dtype=np.int8)
)

groups_outer_training_08C1 = (
    outer_training_meta_08C1[
        "group_hospital"
    ]
    .to_numpy(dtype=str)
)

inner_fold_vector_08C1 = np.array(
    [
        hospital_to_inner_fold_08C1.get(
            hospital,
            -1,
        )
        for hospital in groups_outer_training_08C1
    ],
    dtype=int,
)

if (
    inner_fold_vector_08C1 == -1
).any():
    raise RuntimeError(
        "Bazı dış eğitim hastanelerine "
        "iç kat atanmadı."
    )

if set(
    np.unique(inner_fold_vector_08C1)
) != {1, 2, 3, 4, 5}:
    raise RuntimeError(
        "İç kat değerleri 1–5 değil."
    )

if len(X_outer_training_08C1) != 46800:
    raise RuntimeError(
        f"46.800 dış eğitim satırı bekleniyordu; "
        f"{len(X_outer_training_08C1)} bulundu."
    )

if int(y_outer_training_08C1.sum()) != 2426:
    raise RuntimeError(
        "Dış eğitim olay sayısı 2.426 değil."
    )

if len(outer_test_meta_08C1) != 11691:
    raise RuntimeError(
        "Dış test satır sayısı 11.691 değil."
    )

if int(
    outer_test_meta_08C1[
        "label_stage23"
    ].sum()
) != 606:
    raise RuntimeError(
        "Dış test olay sayısı 606 değil."
    )

if (
    set(
        outer_training_meta_08C1[
            "group_hospital"
        ]
    )
    & set(
        outer_test_meta_08C1[
            "group_hospital"
        ]
    )
):
    raise RuntimeError(
        "Dış eğitim ve test hastaneleri çakışıyor."
    )

# ------------------------------------------------------------
# 4. Kilitli aday ızgarası
# ------------------------------------------------------------

candidate_grid_08C1 = [
    {
        "candidate_id": "LR01",
        "C": 0.03,
        "l1_ratio": 0.00,
    },
    {
        "candidate_id": "LR02",
        "C": 0.10,
        "l1_ratio": 0.00,
    },
    {
        "candidate_id": "LR03",
        "C": 0.30,
        "l1_ratio": 0.00,
    },
    {
        "candidate_id": "LR04",
        "C": 0.10,
        "l1_ratio": 0.25,
    },
    {
        "candidate_id": "LR05",
        "C": 0.30,
        "l1_ratio": 0.25,
    },
    {
        "candidate_id": "LR06",
        "C": 0.30,
        "l1_ratio": 0.50,
    },
]

candidate_ids_08C1 = [
    candidate["candidate_id"]
    for candidate in candidate_grid_08C1
]

# ------------------------------------------------------------
# 5. Preprocessing ve model üreticileri
# ------------------------------------------------------------

def make_preprocessor_08C1():

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
            (
                "scaler",
                StandardScaler(
                    with_mean=False,
                ),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns_07B,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns_07B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_model_08C1(
    C_value,
    l1_ratio_value,
):

    return LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=float(C_value),
        l1_ratio=float(l1_ratio_value),
        class_weight=None,
        max_iter=5000,
        tol=1e-4,
        random_state=MODEL_RANDOM_SEED_08C1,
    )

# ------------------------------------------------------------
# 6. Güvenli BigQuery çalışma tablosu
# ------------------------------------------------------------

oof_work_table_08C1 = (
    f"{TARGET_DATASET}."
    "model_lr_inner_oof_work_v1"
)

oof_stage_table_08C1 = (
    f"{TARGET_DATASET}."
    "model_lr_inner_oof_stage_v1"
)

create_work_table_sql_08C1 = f"""
CREATE TABLE IF NOT EXISTS `{oof_work_table_08C1}` (
  id_row STRING NOT NULL,
  outer_fold INT64 NOT NULL,
  inner_fold INT64 NOT NULL,
  candidate_id STRING NOT NULL,
  label_stage23 INT64 NOT NULL,
  prediction_raw FLOAT64 NOT NULL
);
"""

client.query(
    create_work_table_sql_08C1,
    location=BQ_LOCATION,
).result()

oof_load_config_08C1 = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField(
            "id_row",
            "STRING",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "outer_fold",
            "INTEGER",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "inner_fold",
            "INTEGER",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "candidate_id",
            "STRING",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "label_stage23",
            "INTEGER",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "prediction_raw",
            "FLOAT",
            mode="REQUIRED",
        ),
    ],
    write_disposition=(
        bigquery.WriteDisposition.WRITE_TRUNCATE
    ),
)

# ------------------------------------------------------------
# 7. Aggregate fit-audit checkpoint dosyası
# ------------------------------------------------------------

fit_audit_path_08C1 = os.path.join(
    MODEL_OUTPUT_DIR,
    "08C1_logistic_inner_fit_audit_outer2.csv",
)

if os.path.exists(fit_audit_path_08C1):
    fit_audit_08C1 = pd.read_csv(
        fit_audit_path_08C1
    )
else:
    fit_audit_08C1 = pd.DataFrame(
        columns=[
            "outer_fold",
            "inner_fold",
            "candidate_id",
            "C",
            "l1_ratio",
            "training_rows",
            "validation_rows",
            "training_events",
            "validation_events",
            "processed_columns",
            "convergence_warnings",
            "elapsed_seconds",
        ]
    )

# ------------------------------------------------------------
# 8. İç katları sırayla işle
# ------------------------------------------------------------

for inner_fold in range(1, 6):

    inner_training_mask = (
        inner_fold_vector_08C1
        != inner_fold
    )

    inner_validation_mask = (
        inner_fold_vector_08C1
        == inner_fold
    )

    expected_validation_rows = int(
        inner_validation_mask.sum()
    )

    expected_training_rows = int(
        inner_training_mask.sum()
    )

    existing_status_sql = f"""
    SELECT
      COUNT(*) AS saved_rows,
      COUNT(DISTINCT id_row) AS distinct_ids,
      COUNT(DISTINCT candidate_id) AS candidates,
      COUNT(DISTINCT inner_fold) AS inner_folds
    FROM `{oof_work_table_08C1}`
    WHERE
      outer_fold = {TARGET_OUTER_FOLD_08C1}
      AND inner_fold = {inner_fold};
    """

    existing_status = client.query(
        existing_status_sql,
        location=BQ_LOCATION,
    ).to_dataframe().iloc[0]

    expected_saved_rows = (
        expected_validation_rows
        * len(candidate_grid_08C1)
    )

    checkpoint_complete = (
        int(existing_status["saved_rows"])
        == expected_saved_rows
        and int(existing_status["distinct_ids"])
        == expected_validation_rows
        and int(existing_status["candidates"])
        == len(candidate_grid_08C1)
        and int(existing_status["inner_folds"])
        == 1
    )

    if checkpoint_complete:
        print(
            f"Outer 2 / inner {inner_fold}: "
            "secure checkpoint already complete; skipping."
        )
        continue

    training_hospitals = set(
        groups_outer_training_08C1[
            inner_training_mask
        ]
    )

    validation_hospitals = set(
        groups_outer_training_08C1[
            inner_validation_mask
        ]
    )

    if training_hospitals & validation_hospitals:
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "hastane çakışması bulundu."
        )

    print(
        f"\nOuter 2 / inner {inner_fold}"
    )

    print(
        "Training rows:",
        expected_training_rows,
        "| Validation rows:",
        expected_validation_rows,
    )

    # --------------------------------------------------------
    # Preprocessing yalnızca bu iç eğitim kümesinde öğrenilir
    # --------------------------------------------------------

    preprocessor = make_preprocessor_08C1()

    preprocessing_started = time.time()

    X_inner_training_processed = (
        preprocessor.fit_transform(
            X_outer_training_08C1.loc[
                inner_training_mask
            ]
        )
    )

    X_inner_validation_processed = (
        preprocessor.transform(
            X_outer_training_08C1.loc[
                inner_validation_mask
            ]
        )
    )

    preprocessing_seconds = (
        time.time()
        - preprocessing_started
    )

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "İç eğitim ve doğrulama dönüşüm "
            "sütun sayıları farklı."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_seconds, 2),
    )

    y_inner_training = (
        y_outer_training_08C1[
            inner_training_mask
        ]
    )

    y_inner_validation = (
        y_outer_training_08C1[
            inner_validation_mask
        ]
    )

    validation_ids = (
        outer_training_meta_08C1.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_fold_audit_rows = []

    # --------------------------------------------------------
    # Aynı dönüştürülmüş veri üzerinde altı kilitli aday
    # --------------------------------------------------------

    for candidate in candidate_grid_08C1:

        candidate_id = candidate[
            "candidate_id"
        ]

        print(
            "  Fitting",
            candidate_id,
            "| C =",
            candidate["C"],
            "| l1_ratio =",
            candidate["l1_ratio"],
        )

        model = make_model_08C1(
            candidate["C"],
            candidate["l1_ratio"],
        )

        fitting_started = time.time()

        with warnings.catch_warnings(
            record=True
        ) as warning_records:

            warnings.simplefilter(
                "always",
                ConvergenceWarning,
            )

            model.fit(
                X_inner_training_processed,
                y_inner_training,
            )

        fitting_seconds = (
            time.time()
            - fitting_started
        )

        convergence_warning_count = sum(
            issubclass(
                warning.category,
                ConvergenceWarning,
            )
            for warning in warning_records
        )

        validation_probabilities = (
            model.predict_proba(
                X_inner_validation_processed
            )[:, 1]
        )

        if np.isnan(
            validation_probabilities
        ).any():
            raise RuntimeError(
                f"{candidate_id}, inner "
                f"{inner_fold}: eksik tahmin."
            )

        if not np.all(
            (
                validation_probabilities
                >= 0
            )
            & (
                validation_probabilities
                <= 1
            )
        ):
            raise RuntimeError(
                f"{candidate_id}: 0–1 dışında "
                "olasılık değeri var."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        expected_validation_rows,
                        TARGET_OUTER_FOLD_08C1,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        expected_validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": (
                        y_inner_validation
                        .astype(np.int64)
                    ),
                    "prediction_raw": (
                        validation_probabilities
                        .astype(np.float64)
                    ),
                }
            )
        )

        current_fold_audit_rows.append(
            {
                "outer_fold": (
                    TARGET_OUTER_FOLD_08C1
                ),
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "C": candidate["C"],
                "l1_ratio": candidate[
                    "l1_ratio"
                ],
                "training_rows": int(
                    len(y_inner_training)
                ),
                "validation_rows": int(
                    len(y_inner_validation)
                ),
                "training_events": int(
                    y_inner_training.sum()
                ),
                "validation_events": int(
                    y_inner_validation.sum()
                ),
                "processed_columns": int(
                    X_inner_training_processed
                    .shape[1]
                ),
                "convergence_warnings": int(
                    convergence_warning_count
                ),
                "elapsed_seconds": float(
                    fitting_seconds
                ),
            }
        )

        del model
        gc.collect()

    inner_oof_checkpoint = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    if len(inner_oof_checkpoint) != (
        expected_validation_rows
        * len(candidate_grid_08C1)
    ):
        raise RuntimeError(
            "İç kat OOF checkpoint satır "
            "sayısı hatalı."
        )

    if inner_oof_checkpoint.duplicated(
        subset=[
            "id_row",
            "candidate_id",
        ]
    ).any():
        raise RuntimeError(
            "İç kat OOF checkpoint içinde "
            "yinelenen tahmin var."
        )

    # --------------------------------------------------------
    # Stage tablosuna yükle ve hedef iç katı değiştir
    # --------------------------------------------------------

    client.load_table_from_dataframe(
        inner_oof_checkpoint,
        oof_stage_table_08C1,
        job_config=oof_load_config_08C1,
        location=BQ_LOCATION,
    ).result()

    replace_inner_fold_sql = f"""
    DELETE FROM `{oof_work_table_08C1}`
    WHERE
      outer_fold = {TARGET_OUTER_FOLD_08C1}
      AND inner_fold = {inner_fold};

    INSERT INTO `{oof_work_table_08C1}` (
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    )
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{oof_stage_table_08C1}`;
    """

    client.query(
        replace_inner_fold_sql,
        location=BQ_LOCATION,
    ).result()

    # --------------------------------------------------------
    # Aggregate fit audit dosyasını güncelle
    # --------------------------------------------------------

    fit_audit_08C1 = fit_audit_08C1.loc[
        ~(
            (
                fit_audit_08C1[
                    "outer_fold"
                ].astype(int)
                == TARGET_OUTER_FOLD_08C1
            )
            & (
                fit_audit_08C1[
                    "inner_fold"
                ].astype(int)
                == inner_fold
            )
        )
    ].copy()

    fit_audit_08C1 = pd.concat(
        [
            fit_audit_08C1,
            pd.DataFrame(
                current_fold_audit_rows
            ),
        ],
        ignore_index=True,
    )

    fit_audit_08C1 = (
        fit_audit_08C1
        .sort_values(
            [
                "outer_fold",
                "inner_fold",
                "candidate_id",
            ]
        )
        .reset_index(drop=True)
    )

    fit_audit_08C1.to_csv(
        fit_audit_path_08C1,
        index=False,
    )

    verify_inner_sql = f"""
    SELECT
      COUNT(*) AS saved_rows,
      COUNT(DISTINCT id_row) AS distinct_ids,
      COUNT(DISTINCT candidate_id) AS candidates
    FROM `{oof_work_table_08C1}`
    WHERE
      outer_fold = {TARGET_OUTER_FOLD_08C1}
      AND inner_fold = {inner_fold};
    """

    verify_inner = client.query(
        verify_inner_sql,
        location=BQ_LOCATION,
    ).to_dataframe().iloc[0]

    if int(
        verify_inner["saved_rows"]
    ) != expected_saved_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "BigQuery checkpoint satır sayısı hatalı."
        )

    if int(
        verify_inner["distinct_ids"]
    ) != expected_validation_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "BigQuery distinct id sayısı hatalı."
        )

    if int(
        verify_inner["candidates"]
    ) != 6:
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "altı aday kaydedilmedi."
        )

    print(
        f"Outer 2 / inner {inner_fold}: "
        "secure checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        inner_oof_checkpoint,
        prediction_frames,
    )

    gc.collect()

# ------------------------------------------------------------
# 9. Beş iç katın tamamını doğrula
# ------------------------------------------------------------

overall_checkpoint_sql_08C1 = f"""
SELECT
  COUNT(*) AS saved_rows,
  COUNT(DISTINCT id_row) AS distinct_ids,
  COUNT(DISTINCT candidate_id) AS candidates,
  COUNT(DISTINCT inner_fold) AS inner_folds,
  COUNT(DISTINCT CONCAT(
    candidate_id,
    '|',
    id_row
  )) AS distinct_candidate_rows
FROM `{oof_work_table_08C1}`
WHERE outer_fold = {TARGET_OUTER_FOLD_08C1};
"""

overall_checkpoint_08C1 = client.query(
    overall_checkpoint_sql_08C1,
    location=BQ_LOCATION,
).to_dataframe()

overall_row_08C1 = (
    overall_checkpoint_08C1.iloc[0]
)

expected_total_oof_rows_08C1 = (
    len(y_outer_training_08C1)
    * len(candidate_grid_08C1)
)

if int(
    overall_row_08C1["saved_rows"]
) != expected_total_oof_rows_08C1:
    raise RuntimeError(
        "Dış kat 2 toplam OOF satır sayısı hatalı."
    )

if int(
    overall_row_08C1["distinct_ids"]
) != len(y_outer_training_08C1):
    raise RuntimeError(
        "Dış kat 2 distinct OOF hasta sayısı hatalı."
    )

if int(
    overall_row_08C1["candidates"]
) != 6:
    raise RuntimeError(
        "Dış kat 2 için altı aday tamamlanmadı."
    )

if int(
    overall_row_08C1["inner_folds"]
) != 5:
    raise RuntimeError(
        "Dış kat 2 için beş iç kat tamamlanmadı."
    )

if int(
    overall_row_08C1[
        "distinct_candidate_rows"
    ]
) != expected_total_oof_rows_08C1:
    raise RuntimeError(
        "Aday–hasta OOF tahminlerinde "
        "yinelenme bulundu."
    )

# ------------------------------------------------------------
# 10. Güvenli BigQuery OOF tahminlerini RAM'e al
# ------------------------------------------------------------

load_oof_sql_08C1 = f"""
SELECT
  id_row,
  inner_fold,
  candidate_id,
  label_stage23,
  prediction_raw
FROM `{oof_work_table_08C1}`
WHERE outer_fold = {TARGET_OUTER_FOLD_08C1}
ORDER BY candidate_id, id_row;
"""

print(
    "\nLoading pooled outer-fold-2 "
    "inner OOF predictions from BigQuery..."
)

pooled_oof_long_08C1 = client.query(
    load_oof_sql_08C1,
    location=BQ_LOCATION,
).to_dataframe()

if len(
    pooled_oof_long_08C1
) != expected_total_oof_rows_08C1:
    raise RuntimeError(
        "RAM'e alınan pooled OOF "
        "satır sayısı hatalı."
    )

if pooled_oof_long_08C1.duplicated(
    subset=[
        "candidate_id",
        "id_row",
    ]
).any():
    raise RuntimeError(
        "Pooled OOF içinde yinelenen "
        "aday–hasta tahmini var."
    )

# ------------------------------------------------------------
# 11. Aday metrikleri
# ------------------------------------------------------------

def probability_metrics_08C1(
    y_true,
    probabilities,
):

    probabilities = np.clip(
        np.asarray(
            probabilities,
            dtype=float,
        ),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1],
            )
        ),
        "mean_predicted_risk": float(
            probabilities.mean()
        ),
        "observed_event_rate": float(
            np.mean(y_true)
        ),
    }

candidate_result_rows_08C1 = []

fit_audit_outer2_08C1 = (
    fit_audit_08C1.loc[
        fit_audit_08C1[
            "outer_fold"
        ].astype(int)
        == TARGET_OUTER_FOLD_08C1
    ]
    .copy()
)

for candidate in candidate_grid_08C1:

    candidate_id = candidate[
        "candidate_id"
    ]

    candidate_oof = (
        pooled_oof_long_08C1.loc[
            pooled_oof_long_08C1[
                "candidate_id"
            ] == candidate_id
        ]
        .copy()
    )

    if len(candidate_oof) != len(
        y_outer_training_08C1
    ):
        raise RuntimeError(
            f"{candidate_id}: pooled OOF "
            "satır sayısı hatalı."
        )

    if candidate_oof[
        "id_row"
    ].duplicated().any():
        raise RuntimeError(
            f"{candidate_id}: yinelenen id_row."
        )

    metrics = probability_metrics_08C1(
        candidate_oof[
            "label_stage23"
        ].astype(int),
        candidate_oof[
            "prediction_raw"
        ].astype(float),
    )

    fit_part = (
        fit_audit_outer2_08C1.loc[
            fit_audit_outer2_08C1[
                "candidate_id"
            ] == candidate_id
        ]
    )

    convergence_warnings = (
        int(
            fit_part[
                "convergence_warnings"
            ].sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    elapsed_seconds = (
        float(
            fit_part[
                "elapsed_seconds"
            ].sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_08C1.append(
        {
            "candidate_id": candidate_id,
            "C": candidate["C"],
            "l1_ratio": candidate[
                "l1_ratio"
            ],
            **metrics,
            "convergence_warnings": (
                convergence_warnings
            ),
            "elapsed_seconds": (
                elapsed_seconds
            ),
        }
    )

candidate_results_08C1 = pd.DataFrame(
    candidate_result_rows_08C1
)

candidate_results_08C1 = (
    candidate_results_08C1
    .sort_values(
        [
            "auprc",
            "auroc",
            "brier",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

candidate_results_08C1[
    "selection_rank"
] = np.arange(
    1,
    len(candidate_results_08C1) + 1,
)

best_row_08C1 = (
    candidate_results_08C1.iloc[0]
)

selected_model_08C1 = pd.DataFrame(
    [
        {
            "outer_fold": (
                TARGET_OUTER_FOLD_08C1
            ),
            "selected_candidate": str(
                best_row_08C1[
                    "candidate_id"
                ]
            ),
            "selected_C": float(
                best_row_08C1["C"]
            ),
            "selected_l1_ratio": float(
                best_row_08C1[
                    "l1_ratio"
                ]
            ),
            "selection_metric_primary": (
                "pooled_inner_oof_auprc"
            ),
            "inner_oof_auprc": float(
                best_row_08C1["auprc"]
            ),
            "inner_oof_auroc": float(
                best_row_08C1["auroc"]
            ),
            "inner_oof_brier": float(
                best_row_08C1["brier"]
            ),
            "protocol_sha256": (
                "400c3b4b510c836794543bc685c62fae"
                "f49df0c1caa197d78dffda8c2207952d"
            ),
        }
    ]
)

# ------------------------------------------------------------
# 12. Aggregate sonuçları ve seçim kilidini kaydet
# ------------------------------------------------------------

candidate_results_path_08C1 = os.path.join(
    MODEL_OUTPUT_DIR,
    "08C1_logistic_candidate_results_outer2.csv",
)

selected_model_path_08C1 = os.path.join(
    MODEL_OUTPUT_DIR,
    "08C1_logistic_selected_model_outer2.csv",
)

selection_json_path_08C1 = os.path.join(
    MODEL_OUTPUT_DIR,
    "08C1_logistic_selection_outer2.json",
)

selection_sha_path_08C1 = os.path.join(
    MODEL_OUTPUT_DIR,
    "08C1_logistic_selection_outer2_SHA256.txt",
)

candidate_results_08C1.to_csv(
    candidate_results_path_08C1,
    index=False,
)

selected_model_08C1.to_csv(
    selected_model_path_08C1,
    index=False,
)

selection_configuration_08C1 = {
    "outer_fold": TARGET_OUTER_FOLD_08C1,
    "protocol_sha256": (
        "400c3b4b510c836794543bc685c62fae"
        "f49df0c1caa197d78dffda8c2207952d"
    ),
    "selection_metric_primary": (
        "pooled inner out-of-fold AUPRC"
    ),
    "tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "selected_candidate": str(
        best_row_08C1["candidate_id"]
    ),
    "selected_C": float(
        best_row_08C1["C"]
    ),
    "selected_l1_ratio": float(
        best_row_08C1["l1_ratio"]
    ),
    "candidate_grid": candidate_grid_08C1,
    "oof_checkpoint_table": (
        oof_work_table_08C1
    ),
}

with open(
    selection_json_path_08C1,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        selection_configuration_08C1,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(
    selection_json_path_08C1,
    "rb",
) as file_handle:
    selection_sha256_08C1 = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    selection_sha_path_08C1,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(
        selection_sha256_08C1 + "\n"
    )

# ------------------------------------------------------------
# 13. Sonuç
# ------------------------------------------------------------

print("\n08C1 OUTER-FOLD-2 INNER OOF RESULTS")
display(candidate_results_08C1)

print("\n08C1 OUTER-FOLD-2 SELECTED MODEL")
display(selected_model_08C1)

print("\n08C1 BIGQUERY OOF CHECKPOINT")
display(overall_checkpoint_08C1)

print("\nSelection SHA-256:")
print(selection_sha256_08C1)

print("\nSaved:")
print(candidate_results_path_08C1)
print(selected_model_path_08C1)
print(selection_json_path_08C1)
print(selection_sha_path_08C1)

print(
    "\n08C1 PASS: Outer-fold-2 inner "
    "hyperparameter search completed."
)

print(
    "All six candidates were evaluated "
    "using pooled hospital-disjoint OOF predictions."
)

print(
    "Inner OOF patient-level predictions were "
    "checkpointed only in secure BigQuery."
)

print(
    "No patient-level prediction file was "
    "written to Google Drive."
)

In [ ]:
import os
import pandas as pd

from google.cloud import bigquery
from google.api_core.exceptions import NotFound
from IPython.display import display

# ============================================================
# 08C1-R1 — Recover completed outer-2 / inner-1 checkpoint
#
# No DELETE, INSERT, UPDATE or MERGE is used.
# The completed stage table is copied/loaded into a dedicated
# permanent checkpoint table.
# ============================================================

required_objects_08C1R1 = [
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
    "MODEL_OUTPUT_DIR",
]

missing_objects_08C1R1 = [
    name
    for name in required_objects_08C1R1
    if name not in globals()
]

if missing_objects_08C1R1:
    raise RuntimeError(
        "Eksik çalışma nesneleri var: "
        + ", ".join(missing_objects_08C1R1)
        + ". Önce 07A hücresini çalıştır."
    )

STAGE_TABLE_08C1R1 = (
    f"{TARGET_DATASET}."
    "model_lr_inner_oof_stage_v1"
)

PERMANENT_TABLE_08C1R1 = (
    f"{TARGET_DATASET}."
    "model_lr_inner_oof_outer2_inner1_v1"
)

EXPECTED_VALIDATION_ROWS_08C1R1 = 5805
EXPECTED_CANDIDATES_08C1R1 = 6
EXPECTED_TOTAL_ROWS_08C1R1 = (
    EXPECTED_VALIDATION_ROWS_08C1R1
    * EXPECTED_CANDIDATES_08C1R1
)

# ------------------------------------------------------------
# 1. Geçici stage tablosunu doğrula
# ------------------------------------------------------------

try:
    client.get_table(STAGE_TABLE_08C1R1)
except NotFound:
    raise RuntimeError(
        "Geçici stage tablosu bulunamadı. "
        "İç kat 1 tahminleri yeniden üretilmek zorunda kalabilir."
    )

SQL_STAGE_CHECK_08C1R1 = f"""
SELECT
  COUNT(*) AS row_count,
  COUNT(DISTINCT id_row) AS distinct_id_count,
  COUNT(DISTINCT candidate_id) AS candidate_count,
  COUNT(DISTINCT outer_fold) AS outer_fold_count,
  COUNT(DISTINCT inner_fold) AS inner_fold_count,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  MIN(inner_fold) AS minimum_inner_fold,
  MAX(inner_fold) AS maximum_inner_fold,
  COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
  COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
  COUNTIF(prediction_raw IS NULL) AS missing_predictions,
  MIN(prediction_raw) AS minimum_probability,
  MAX(prediction_raw) AS maximum_probability
FROM `{STAGE_TABLE_08C1R1}`;
"""

stage_check_08C1R1 = client.query(
    SQL_STAGE_CHECK_08C1R1,
    location=BQ_LOCATION,
).to_dataframe()

stage_row_08C1R1 = stage_check_08C1R1.iloc[0]

expected_stage_values_08C1R1 = {
    "row_count": EXPECTED_TOTAL_ROWS_08C1R1,
    "distinct_id_count": EXPECTED_VALIDATION_ROWS_08C1R1,
    "candidate_count": EXPECTED_CANDIDATES_08C1R1,
    "outer_fold_count": 1,
    "inner_fold_count": 1,
    "minimum_outer_fold": 2,
    "maximum_outer_fold": 2,
    "minimum_inner_fold": 1,
    "maximum_inner_fold": 1,
    "missing_predictions": 0,
}

for field, expected_value in expected_stage_values_08C1R1.items():
    actual_value = int(stage_row_08C1R1[field])

    if actual_value != expected_value:
        raise RuntimeError(
            f"Stage tablosunda {field}={actual_value}; "
            f"beklenen={expected_value}"
        )

minimum_probability_08C1R1 = float(
    stage_row_08C1R1["minimum_probability"]
)

maximum_probability_08C1R1 = float(
    stage_row_08C1R1["maximum_probability"]
)

if not (
    0.0 <= minimum_probability_08C1R1 <= 1.0
    and 0.0 <= maximum_probability_08C1R1 <= 1.0
):
    raise RuntimeError(
        "Stage tablosunda 0–1 dışında tahmin bulundu."
    )

print("08C1-R1 STAGE TABLE CHECK")
display(stage_check_08C1R1)

# ------------------------------------------------------------
# 2. Stage tablosunu kalıcı iç-kat tablosuna kopyala
# ------------------------------------------------------------

copy_method_08C1R1 = None

try:
    copy_config_08C1R1 = bigquery.CopyJobConfig(
        write_disposition=(
            bigquery.WriteDisposition.WRITE_TRUNCATE
        )
    )

    copy_job_08C1R1 = client.copy_table(
        STAGE_TABLE_08C1R1,
        PERMANENT_TABLE_08C1R1,
        job_config=copy_config_08C1R1,
        location=BQ_LOCATION,
    )

    copy_job_08C1R1.result()

    copy_method_08C1R1 = "BigQuery table copy"

except Exception as copy_error_08C1R1:
    print(
        "Direct table copy was unavailable; "
        "using query-download plus load-job fallback."
    )
    print(
        "Copy message:",
        type(copy_error_08C1R1).__name__
    )

    SQL_STAGE_LOAD_08C1R1 = f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{STAGE_TABLE_08C1R1}`;
    """

    recovered_stage_df_08C1R1 = client.query(
        SQL_STAGE_LOAD_08C1R1,
        location=BQ_LOCATION,
    ).to_dataframe()

    load_config_08C1R1 = bigquery.LoadJobConfig(
        schema=[
            bigquery.SchemaField(
                "id_row",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "outer_fold",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "inner_fold",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "candidate_id",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "label_stage23",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_raw",
                "FLOAT",
                mode="REQUIRED",
            ),
        ],
        write_disposition=(
            bigquery.WriteDisposition.WRITE_TRUNCATE
        ),
    )

    client.load_table_from_dataframe(
        recovered_stage_df_08C1R1,
        PERMANENT_TABLE_08C1R1,
        job_config=load_config_08C1R1,
        location=BQ_LOCATION,
    ).result()

    copy_method_08C1R1 = (
        "BigQuery query-download and load job"
    )

# ------------------------------------------------------------
# 3. Kalıcı tabloyu doğrula
# ------------------------------------------------------------

SQL_PERMANENT_CHECK_08C1R1 = f"""
SELECT
  COUNT(*) AS row_count,
  COUNT(DISTINCT id_row) AS distinct_id_count,
  COUNT(DISTINCT candidate_id) AS candidate_count,
  COUNT(DISTINCT outer_fold) AS outer_fold_count,
  COUNT(DISTINCT inner_fold) AS inner_fold_count,
  COUNT(
    DISTINCT CONCAT(
      candidate_id,
      '|',
      id_row
    )
  ) AS distinct_candidate_patient_rows,
  COUNTIF(prediction_raw IS NULL) AS missing_predictions,
  MIN(prediction_raw) AS minimum_probability,
  MAX(prediction_raw) AS maximum_probability
FROM `{PERMANENT_TABLE_08C1R1}`;
"""

permanent_check_08C1R1 = client.query(
    SQL_PERMANENT_CHECK_08C1R1,
    location=BQ_LOCATION,
).to_dataframe()

permanent_row_08C1R1 = (
    permanent_check_08C1R1.iloc[0]
)

expected_permanent_values_08C1R1 = {
    "row_count": EXPECTED_TOTAL_ROWS_08C1R1,
    "distinct_id_count": EXPECTED_VALIDATION_ROWS_08C1R1,
    "candidate_count": EXPECTED_CANDIDATES_08C1R1,
    "outer_fold_count": 1,
    "inner_fold_count": 1,
    "distinct_candidate_patient_rows": (
        EXPECTED_TOTAL_ROWS_08C1R1
    ),
    "missing_predictions": 0,
}

for field, expected_value in (
    expected_permanent_values_08C1R1.items()
):
    actual_value = int(
        permanent_row_08C1R1[field]
    )

    if actual_value != expected_value:
        raise RuntimeError(
            f"Kalıcı tabloda {field}={actual_value}; "
            f"beklenen={expected_value}"
        )

# ------------------------------------------------------------
# 4. RAM'deki fit audit bilgisini mümkünse koru
# ------------------------------------------------------------

fit_audit_path_08C1R1 = os.path.join(
    MODEL_OUTPUT_DIR,
    "08C1_logistic_inner_fit_audit_outer2.csv",
)

fit_audit_saved_08C1R1 = False
fit_audit_rows_saved_08C1R1 = 0

if (
    "current_fold_audit_rows" in globals()
    and isinstance(
        current_fold_audit_rows,
        list,
    )
    and len(current_fold_audit_rows) == 6
):
    recovered_fit_audit_08C1R1 = pd.DataFrame(
        current_fold_audit_rows
    )

    required_audit_columns_08C1R1 = {
        "outer_fold",
        "inner_fold",
        "candidate_id",
    }

    if required_audit_columns_08C1R1.issubset(
        recovered_fit_audit_08C1R1.columns
    ):
        if os.path.exists(
            fit_audit_path_08C1R1
        ):
            existing_fit_audit_08C1R1 = pd.read_csv(
                fit_audit_path_08C1R1
            )
        else:
            existing_fit_audit_08C1R1 = pd.DataFrame()

        combined_fit_audit_08C1R1 = pd.concat(
            [
                existing_fit_audit_08C1R1,
                recovered_fit_audit_08C1R1,
            ],
            ignore_index=True,
        )

        combined_fit_audit_08C1R1 = (
            combined_fit_audit_08C1R1
            .drop_duplicates(
                subset=[
                    "outer_fold",
                    "inner_fold",
                    "candidate_id",
                ],
                keep="last",
            )
            .sort_values(
                [
                    "outer_fold",
                    "inner_fold",
                    "candidate_id",
                ]
            )
            .reset_index(drop=True)
        )

        combined_fit_audit_08C1R1.to_csv(
            fit_audit_path_08C1R1,
            index=False,
        )

        fit_audit_saved_08C1R1 = True
        fit_audit_rows_saved_08C1R1 = len(
            recovered_fit_audit_08C1R1
        )

# ------------------------------------------------------------
# 5. Sonuç
# ------------------------------------------------------------

recovery_summary_08C1R1 = pd.DataFrame(
    {
        "metric": [
            "source_stage_rows",
            "permanent_checkpoint_rows",
            "distinct_validation_patients",
            "candidate_models",
            "outer_fold",
            "inner_fold",
            "copy_method",
            "fit_audit_recovered",
            "fit_audit_rows_recovered",
        ],
        "value": [
            int(stage_row_08C1R1["row_count"]),
            int(
                permanent_row_08C1R1[
                    "row_count"
                ]
            ),
            int(
                permanent_row_08C1R1[
                    "distinct_id_count"
                ]
            ),
            int(
                permanent_row_08C1R1[
                    "candidate_count"
                ]
            ),
            2,
            1,
            copy_method_08C1R1,
            fit_audit_saved_08C1R1,
            fit_audit_rows_saved_08C1R1,
        ],
    }
)

print("\n08C1-R1 RECOVERY SUMMARY")
display(recovery_summary_08C1R1)

print("\n08C1-R1 PERMANENT CHECKPOINT VERIFICATION")
display(permanent_check_08C1R1)

print("\nPermanent checkpoint table:")
print(PERMANENT_TABLE_08C1R1)

print(
    "\n08C1-R1 PASS: Outer-fold-2 / "
    "inner-fold-1 predictions were recovered "
    "without rerunning the six models."
)

print(
    "No billing account and no DML statement "
    "were required."
)

In [ ]:
import os
import gc
import time
import warnings

import numpy as np
import pandas as pd

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning

from IPython.display import display

# ============================================================
# 08C2-A
# Resume-safe outer-fold-2 inner-fold checkpoints
#
# Her iç kat ayrı bir BigQuery tablosuna yazılır.
# DELETE / INSERT / UPDATE / MERGE kullanılmaz.
# Hasta düzeyindeki tahminler Drive'a yazılmaz.
# ============================================================

TARGET_OUTER_FOLD_08C2A = 2
MODEL_RANDOM_SEED_08C2A = 20260721

# ------------------------------------------------------------
# 1. Gerekli nesneleri doğrula
# ------------------------------------------------------------

required_objects_08C2A = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_08C2A = [
    name
    for name in required_objects_08C2A
    if name not in globals()
]

if missing_objects_08C2A:
    raise RuntimeError(
        "Eksik RAM nesneleri var: "
        + ", ".join(missing_objects_08C2A)
        + ". Önce 07A ve ardından 07B hücresini çalıştır."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"58.491 satır bekleniyordu; "
        f"{len(core_df_07B)} bulundu."
    )

if len(predictor_columns_07B) != 159:
    raise RuntimeError(
        "Core predictor sayısı 159 değil."
    )

if len(numeric_columns_07B) != 156:
    raise RuntimeError(
        "Sayısal predictor sayısı 156 değil."
    )

if len(categorical_columns_07B) != 3:
    raise RuntimeError(
        "Kategorik predictor sayısı 3 değil."
    )

# ------------------------------------------------------------
# 2. Kilitli iç kat haritasını yükle
# ------------------------------------------------------------

inner_mapping_path_08C2A = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_08C2A):
    raise FileNotFoundError(
        "Kilitli iç kat dosyası bulunamadı: "
        + inner_mapping_path_08C2A
    )

inner_mapping_all_08C2A = pd.read_csv(
    inner_mapping_path_08C2A,
    dtype={"group_hospital": str},
)

inner_mapping_part_08C2A = (
    inner_mapping_all_08C2A.loc[
        inner_mapping_all_08C2A[
            "outer_fold"
        ].astype(int) == TARGET_OUTER_FOLD_08C2A,
        [
            "group_hospital",
            "inner_fold",
        ],
    ]
    .copy()
)

inner_mapping_part_08C2A[
    "group_hospital"
] = inner_mapping_part_08C2A[
    "group_hospital"
].astype(str)

inner_mapping_part_08C2A[
    "inner_fold"
] = inner_mapping_part_08C2A[
    "inner_fold"
].astype(int)

if len(inner_mapping_part_08C2A) != 158:
    raise RuntimeError(
        "Dış kat 2 eğitim kümesi için "
        "158 hastane ataması bekleniyordu."
    )

if inner_mapping_part_08C2A[
    "group_hospital"
].duplicated().any():
    raise RuntimeError(
        "Kilitli iç kat haritasında "
        "yinelenen hastane bulundu."
    )

hospital_to_inner_fold_08C2A = dict(
    zip(
        inner_mapping_part_08C2A[
            "group_hospital"
        ],
        inner_mapping_part_08C2A[
            "inner_fold"
        ],
    )
)

# ------------------------------------------------------------
# 3. Dış kat 2 eğitim matrisi
# ------------------------------------------------------------

X_all_08C2A = core_df_07B[
    predictor_columns_07B
].copy()

for column in numeric_columns_07B:
    X_all_08C2A[column] = pd.to_numeric(
        X_all_08C2A[column],
        errors="coerce",
    ).astype("float64")

for column in categorical_columns_07B:
    category_series = (
        X_all_08C2A[column]
        .astype("object")
    )

    X_all_08C2A[column] = category_series.where(
        pd.notna(category_series),
        np.nan,
    )

outer_fold_all_08C2A = (
    core_df_07B["outer_fold"]
    .astype(int)
    .to_numpy()
)

outer_training_mask_08C2A = (
    outer_fold_all_08C2A
    != TARGET_OUTER_FOLD_08C2A
)

outer_test_mask_08C2A = (
    outer_fold_all_08C2A
    == TARGET_OUTER_FOLD_08C2A
)

X_outer_training_08C2A = (
    X_all_08C2A.loc[
        outer_training_mask_08C2A
    ]
    .reset_index(drop=True)
)

outer_training_meta_08C2A = (
    core_df_07B.loc[
        outer_training_mask_08C2A,
        [
            "id_row",
            "group_hospital",
            "label_stage23",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_08C2A = (
    core_df_07B.loc[
        outer_test_mask_08C2A,
        [
            "id_row",
            "group_hospital",
            "label_stage23",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_training_meta_08C2A[
    "id_row"
] = outer_training_meta_08C2A[
    "id_row"
].astype(str)

outer_training_meta_08C2A[
    "group_hospital"
] = outer_training_meta_08C2A[
    "group_hospital"
].astype(str)

outer_training_meta_08C2A[
    "label_stage23"
] = outer_training_meta_08C2A[
    "label_stage23"
].astype(int)

outer_test_meta_08C2A[
    "group_hospital"
] = outer_test_meta_08C2A[
    "group_hospital"
].astype(str)

y_outer_training_08C2A = (
    outer_training_meta_08C2A[
        "label_stage23"
    ]
    .to_numpy(dtype=np.int8)
)

groups_outer_training_08C2A = (
    outer_training_meta_08C2A[
        "group_hospital"
    ]
    .to_numpy(dtype=str)
)

inner_fold_vector_08C2A = np.array(
    [
        hospital_to_inner_fold_08C2A.get(
            hospital,
            -1,
        )
        for hospital in groups_outer_training_08C2A
    ],
    dtype=int,
)

if (
    inner_fold_vector_08C2A == -1
).any():
    raise RuntimeError(
        "Bazı dış eğitim hastanelerine "
        "iç kat atanmadı."
    )

if set(
    np.unique(inner_fold_vector_08C2A)
) != {1, 2, 3, 4, 5}:
    raise RuntimeError(
        "İç kat değerleri 1–5 değil."
    )

if len(X_outer_training_08C2A) != 46800:
    raise RuntimeError(
        f"46.800 eğitim satırı bekleniyordu; "
        f"{len(X_outer_training_08C2A)} bulundu."
    )

if int(y_outer_training_08C2A.sum()) != 2426:
    raise RuntimeError(
        "Dış eğitim olay sayısı 2.426 değil."
    )

if len(outer_test_meta_08C2A) != 11691:
    raise RuntimeError(
        "Dış test satır sayısı 11.691 değil."
    )

if int(
    outer_test_meta_08C2A[
        "label_stage23"
    ].sum()
) != 606:
    raise RuntimeError(
        "Dış test olay sayısı 606 değil."
    )

if (
    set(
        outer_training_meta_08C2A[
            "group_hospital"
        ]
    )
    & set(
        outer_test_meta_08C2A[
            "group_hospital"
        ]
    )
):
    raise RuntimeError(
        "Dış eğitim ve test hastaneleri çakışıyor."
    )

# ------------------------------------------------------------
# 4. Kilitli aday ızgarası
# ------------------------------------------------------------

candidate_grid_08C2A = [
    {
        "candidate_id": "LR01",
        "C": 0.03,
        "l1_ratio": 0.00,
    },
    {
        "candidate_id": "LR02",
        "C": 0.10,
        "l1_ratio": 0.00,
    },
    {
        "candidate_id": "LR03",
        "C": 0.30,
        "l1_ratio": 0.00,
    },
    {
        "candidate_id": "LR04",
        "C": 0.10,
        "l1_ratio": 0.25,
    },
    {
        "candidate_id": "LR05",
        "C": 0.30,
        "l1_ratio": 0.25,
    },
    {
        "candidate_id": "LR06",
        "C": 0.30,
        "l1_ratio": 0.50,
    },
]

# ------------------------------------------------------------
# 5. Preprocessing ve model üreticileri
# ------------------------------------------------------------

def make_preprocessor_08C2A():

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
            (
                "scaler",
                StandardScaler(
                    with_mean=False,
                ),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns_07B,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns_07B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_model_08C2A(
    C_value,
    l1_ratio_value,
):

    return LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=float(C_value),
        l1_ratio=float(l1_ratio_value),
        class_weight=None,
        max_iter=5000,
        tol=1e-4,
        random_state=MODEL_RANDOM_SEED_08C2A,
    )

# ------------------------------------------------------------
# 6. Kalıcı checkpoint tablo adı
# ------------------------------------------------------------

def checkpoint_table_id_08C2A(
    inner_fold,
):

    return (
        f"{TARGET_DATASET}."
        f"model_lr_inner_oof_outer2_"
        f"inner{inner_fold}_v1"
    )

# ------------------------------------------------------------
# 7. Checkpoint doğrulama fonksiyonu
# ------------------------------------------------------------

def verify_checkpoint_08C2A(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):

    table_id = checkpoint_table_id_08C2A(
        inner_fold
    )

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row)
        AS distinct_id_count,
      COUNT(DISTINCT candidate_id)
        AS candidate_count,
      COUNT(DISTINCT outer_fold)
        AS outer_fold_count,
      COUNT(DISTINCT inner_fold)
        AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(
          candidate_id,
          '|',
          id_row
        )
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1)
        AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0)
        AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL)
        AS missing_predictions,
      COUNTIF(
        prediction_raw < 0
        OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(
        sql,
        location=BQ_LOCATION,
    ).to_dataframe()

    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows
        * len(candidate_grid_08C2A)
    )

    expected_positive_rows = (
        expected_validation_events
        * len(candidate_grid_08C2A)
    )

    expected_negative_rows = (
        (
            expected_validation_rows
            - expected_validation_events
        )
        * len(candidate_grid_08C2A)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": (
            expected_validation_rows
        ),
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": (
            expected_total_rows
        ),
        "positive_prediction_rows": (
            expected_positive_rows
        ),
        "negative_prediction_rows": (
            expected_negative_rows
        ),
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": 2,
        "maximum_outer_fold": 2,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failure_items = []

    for field, expected_value in (
        expected_values.items()
    ):
        actual_value = int(row[field])

        if actual_value != expected_value:
            complete = False

            failure_items.append(
                f"{field}={actual_value}, "
                f"expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failure_items),
        "check": check,
        "row": row,
    }

# ------------------------------------------------------------
# 8. Load-job şeması
# ------------------------------------------------------------

checkpoint_load_config_08C2A = (
    bigquery.LoadJobConfig(
        schema=[
            bigquery.SchemaField(
                "id_row",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "outer_fold",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "inner_fold",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "candidate_id",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "label_stage23",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_raw",
                "FLOAT",
                mode="REQUIRED",
            ),
        ],
        write_disposition=(
            bigquery.WriteDisposition.WRITE_TRUNCATE
        ),
    )
)

# ------------------------------------------------------------
# 9. Aggregate fit audit dosyası
# ------------------------------------------------------------

fit_audit_path_08C2A = os.path.join(
    MODEL_OUTPUT_DIR,
    "08C1_logistic_inner_fit_audit_outer2.csv",
)

fit_audit_columns_08C2A = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "C",
    "l1_ratio",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "convergence_warnings",
    "elapsed_seconds",
]

if os.path.exists(fit_audit_path_08C2A):
    fit_audit_08C2A = pd.read_csv(
        fit_audit_path_08C2A
    )
else:
    fit_audit_08C2A = pd.DataFrame(
        columns=fit_audit_columns_08C2A
    )

for column in fit_audit_columns_08C2A:
    if column not in fit_audit_08C2A.columns:
        fit_audit_08C2A[column] = np.nan

fit_audit_08C2A = fit_audit_08C2A[
    fit_audit_columns_08C2A
].copy()

# ------------------------------------------------------------
# 10. Beş iç katı sırayla işle
# ------------------------------------------------------------

checkpoint_status_rows_08C2A = []

for inner_fold in range(1, 6):

    inner_training_mask = (
        inner_fold_vector_08C2A
        != inner_fold
    )

    inner_validation_mask = (
        inner_fold_vector_08C2A
        == inner_fold
    )

    training_rows = int(
        inner_training_mask.sum()
    )

    validation_rows = int(
        inner_validation_mask.sum()
    )

    training_events = int(
        y_outer_training_08C2A[
            inner_training_mask
        ].sum()
    )

    validation_events = int(
        y_outer_training_08C2A[
            inner_validation_mask
        ].sum()
    )

    existing_check = verify_checkpoint_08C2A(
        inner_fold=inner_fold,
        expected_validation_rows=(
            validation_rows
        ),
        expected_validation_events=(
            validation_events
        ),
    )

    if existing_check["complete"]:

        print(
            f"Outer 2 / inner {inner_fold}: "
            "permanent checkpoint already complete; "
            "skipping model fitting."
        )

        row = existing_check["row"]

        checkpoint_status_rows_08C2A.append(
            {
                "outer_fold": 2,
                "inner_fold": inner_fold,
                "checkpoint_status": "existing_complete",
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "validation_events": validation_events,
                "checkpoint_rows": int(
                    row["row_count"]
                ),
                "distinct_validation_patients": int(
                    row["distinct_id_count"]
                ),
                "candidate_models": int(
                    row["candidate_count"]
                ),
                "minimum_probability": float(
                    row["minimum_probability"]
                ),
                "maximum_probability": float(
                    row["maximum_probability"]
                ),
                "table_id": existing_check[
                    "table_id"
                ],
            }
        )

        continue

    print(
        f"\nOuter 2 / inner {inner_fold}"
    )

    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    training_hospitals = set(
        groups_outer_training_08C2A[
            inner_training_mask
        ]
    )

    validation_hospitals = set(
        groups_outer_training_08C2A[
            inner_validation_mask
        ]
    )

    if (
        training_hospitals
        & validation_hospitals
    ):
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "hastane çakışması bulundu."
        )

    preprocessor = make_preprocessor_08C2A()

    preprocessing_started = time.time()

    X_inner_training_processed = (
        preprocessor.fit_transform(
            X_outer_training_08C2A.loc[
                inner_training_mask
            ]
        )
    )

    X_inner_validation_processed = (
        preprocessor.transform(
            X_outer_training_08C2A.loc[
                inner_validation_mask
            ]
        )
    )

    preprocessing_seconds = (
        time.time()
        - preprocessing_started
    )

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "İç eğitim ve doğrulama işlenmiş "
            "sütun sayıları farklı."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_seconds, 2),
    )

    y_inner_training = (
        y_outer_training_08C2A[
            inner_training_mask
        ]
    )

    y_inner_validation = (
        y_outer_training_08C2A[
            inner_validation_mask
        ]
    )

    validation_ids = (
        outer_training_meta_08C2A.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_08C2A:

        candidate_id = candidate[
            "candidate_id"
        ]

        print(
            "  Fitting",
            candidate_id,
            "| C =",
            candidate["C"],
            "| l1_ratio =",
            candidate["l1_ratio"],
        )

        model = make_model_08C2A(
            candidate["C"],
            candidate["l1_ratio"],
        )

        fitting_started = time.time()

        with warnings.catch_warnings(
            record=True
        ) as warning_records:

            warnings.simplefilter(
                "always",
                ConvergenceWarning,
            )

            model.fit(
                X_inner_training_processed,
                y_inner_training,
            )

        fitting_seconds = (
            time.time()
            - fitting_started
        )

        convergence_warning_count = sum(
            issubclass(
                warning.category,
                ConvergenceWarning,
            )
            for warning in warning_records
        )

        validation_probabilities = (
            model.predict_proba(
                X_inner_validation_processed
            )[:, 1]
        )

        if np.isnan(
            validation_probabilities
        ).any():
            raise RuntimeError(
                f"{candidate_id}, inner "
                f"{inner_fold}: eksik tahmin."
            )

        if not np.all(
            (
                validation_probabilities >= 0
            )
            & (
                validation_probabilities <= 1
            )
        ):
            raise RuntimeError(
                f"{candidate_id}: 0–1 dışında "
                "olasılık bulundu."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        2,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": (
                        candidate_id
                    ),
                    "label_stage23": (
                        y_inner_validation
                        .astype(np.int64)
                    ),
                    "prediction_raw": (
                        validation_probabilities
                        .astype(np.float64)
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": 2,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "C": candidate["C"],
                "l1_ratio": candidate[
                    "l1_ratio"
                ],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": (
                    validation_events
                ),
                "processed_columns": int(
                    X_inner_training_processed
                    .shape[1]
                ),
                "convergence_warnings": int(
                    convergence_warning_count
                ),
                "elapsed_seconds": float(
                    fitting_seconds
                ),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows
        * len(candidate_grid_08C2A)
    )

    if len(checkpoint_df) != (
        expected_checkpoint_rows
    ):
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "checkpoint satır sayısı hatalı."
        )

    if checkpoint_df.duplicated(
        subset=[
            "id_row",
            "candidate_id",
        ]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "yinelenen aday–hasta tahmini var."
        )

    target_table_id = (
        checkpoint_table_id_08C2A(
            inner_fold
        )
    )

    print(
        "Uploading permanent checkpoint:",
        target_table_id,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_table_id,
        job_config=(
            checkpoint_load_config_08C2A
        ),
        location=BQ_LOCATION,
    ).result()

    # --------------------------------------------------------
    # Aggregate fit audit kaydı
    # --------------------------------------------------------

    if len(fit_audit_08C2A) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_08C2A[
                        "outer_fold"
                    ],
                    errors="coerce",
                ) == 2
            )
            & (
                pd.to_numeric(
                    fit_audit_08C2A[
                        "inner_fold"
                    ],
                    errors="coerce",
                ) == inner_fold
            )
        )

        fit_audit_08C2A = (
            fit_audit_08C2A.loc[
                keep_mask
            ].copy()
        )

    fit_audit_08C2A = pd.concat(
        [
            fit_audit_08C2A,
            pd.DataFrame(
                current_audit_rows
            ),
        ],
        ignore_index=True,
    )

    fit_audit_08C2A = (
        fit_audit_08C2A[
            fit_audit_columns_08C2A
        ]
        .sort_values(
            [
                "outer_fold",
                "inner_fold",
                "candidate_id",
            ]
        )
        .reset_index(drop=True)
    )

    fit_audit_08C2A.to_csv(
        fit_audit_path_08C2A,
        index=False,
    )

    # --------------------------------------------------------
    # Kalıcı checkpoint doğrulaması
    # --------------------------------------------------------

    completed_check = verify_checkpoint_08C2A(
        inner_fold=inner_fold,
        expected_validation_rows=(
            validation_rows
        ),
        expected_validation_events=(
            validation_events
        ),
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} "
            "kalıcı checkpoint doğrulanamadı: "
            + completed_check["reason"]
        )

    row = completed_check["row"]

    checkpoint_status_rows_08C2A.append(
        {
            "outer_fold": 2,
            "inner_fold": inner_fold,
            "checkpoint_status": (
                "newly_completed"
            ),
            "training_rows": training_rows,
            "validation_rows": validation_rows,
            "validation_events": (
                validation_events
            ),
            "checkpoint_rows": int(
                row["row_count"]
            ),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(
                row["candidate_count"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": target_table_id,
        }
    )

    print(
        f"Outer 2 / inner {inner_fold}: "
        "permanent checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )

    gc.collect()

# ------------------------------------------------------------
# 11. Beş ayrı checkpoint tablosunun son denetimi
# ------------------------------------------------------------

final_status_rows_08C2A = []

for inner_fold in range(1, 6):

    validation_mask = (
        inner_fold_vector_08C2A
        == inner_fold
    )

    validation_rows = int(
        validation_mask.sum()
    )

    validation_events = int(
        y_outer_training_08C2A[
            validation_mask
        ].sum()
    )

    final_check = verify_checkpoint_08C2A(
        inner_fold=inner_fold,
        expected_validation_rows=(
            validation_rows
        ),
        expected_validation_events=(
            validation_events
        ),
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "son checkpoint kontrolü başarısız. "
            + final_check["reason"]
        )

    row = final_check["row"]

    final_status_rows_08C2A.append(
        {
            "outer_fold": 2,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(
                row["row_count"]
            ),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(
                row["candidate_count"]
            ),
            "positive_prediction_rows": int(
                row[
                    "positive_prediction_rows"
                ]
            ),
            "negative_prediction_rows": int(
                row[
                    "negative_prediction_rows"
                ]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check[
                "table_id"
            ],
        }
    )

checkpoint_summary_08C2A = (
    pd.DataFrame(
        final_status_rows_08C2A
    )
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_08C2A[
        "distinct_validation_patients"
    ].sum()
) != 46800:
    raise RuntimeError(
        "Beş checkpoint tablosundaki "
        "doğrulama hasta toplamı 46.800 değil."
    )

if int(
    checkpoint_summary_08C2A[
        "checkpoint_rows"
    ].sum()
) != 280800:
    raise RuntimeError(
        "Beş checkpoint tablosundaki "
        "tahmin satırı toplamı 280.800 değil."
    )

if not (
    checkpoint_summary_08C2A[
        "candidate_models"
    ] == 6
).all():
    raise RuntimeError(
        "Her iç katta altı aday model yok."
    )

if int(
    checkpoint_summary_08C2A[
        "missing_predictions"
    ].sum()
) != 0:
    raise RuntimeError(
        "Checkpoint tablolarında eksik tahmin var."
    )

if int(
    checkpoint_summary_08C2A[
        "invalid_probabilities"
    ].sum()
) != 0:
    raise RuntimeError(
        "Checkpoint tablolarında geçersiz "
        "olasılık değeri var."
    )

# ------------------------------------------------------------
# 12. Aggregate checkpoint özetini kaydet
# ------------------------------------------------------------

checkpoint_summary_path_08C2A = os.path.join(
    MODEL_OUTPUT_DIR,
    "08C2A_outer2_inner_checkpoint_summary.csv",
)

checkpoint_summary_08C2A.to_csv(
    checkpoint_summary_path_08C2A,
    index=False,
)

print(
    "\n08C2-A OUTER-FOLD-2 INNER "
    "CHECKPOINT SUMMARY"
)

display(checkpoint_summary_08C2A)

print("\nAggregate fit-audit file:")
print(fit_audit_path_08C2A)

print("\nCheckpoint summary file:")
print(checkpoint_summary_path_08C2A)

print(
    "\n08C2-A PASS: All five outer-fold-2 "
    "inner OOF checkpoints are complete."
)

print(
    "Each inner fold is stored in a separate "
    "permanent BigQuery table."
)

print(
    "No DML statement and no billing account "
    "were required."
)

print(
    "No patient-level prediction file was "
    "written to Google Drive."
)

In [ ]:
import os
import json
import hashlib

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

from IPython.display import display

# ============================================================
# 08C2-B
# Pool five permanent inner-fold checkpoint tables,
# select the outer-fold-2 hyperparameters and lock
# the Platt calibration coefficients.
#
# No model refitting occurs in this cell.
# No patient-level data are written to Google Drive.
# ============================================================

TARGET_OUTER_FOLD_08C2B = 2

# ------------------------------------------------------------
# 1. Gerekli nesneleri doğrula
# ------------------------------------------------------------

required_objects_08C2B = [
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
    "MODEL_OUTPUT_DIR",
]

missing_objects_08C2B = [
    name
    for name in required_objects_08C2B
    if name not in globals()
]

if missing_objects_08C2B:
    raise RuntimeError(
        "Eksik çalışma nesneleri var: "
        + ", ".join(missing_objects_08C2B)
        + ". Önce 07A hücresini çalıştır."
    )

# ------------------------------------------------------------
# 2. Kilitli protokol SHA değerini doğrula
# ------------------------------------------------------------

expected_protocol_sha256_08C2B = (
    "400c3b4b510c836794543bc685c62fae"
    "f49df0c1caa197d78dffda8c2207952d"
)

protocol_sha_path_08C2B = os.path.join(
    MODEL_OUTPUT_DIR,
    "08B_locked_logistic_model_protocol_v1_SHA256.txt",
)

if not os.path.exists(protocol_sha_path_08C2B):
    raise FileNotFoundError(
        "Kilitli model protokolü SHA dosyası bulunamadı: "
        + protocol_sha_path_08C2B
    )

with open(
    protocol_sha_path_08C2B,
    "r",
    encoding="utf-8",
) as file_handle:
    observed_protocol_sha256_08C2B = (
        file_handle.read().strip()
    )

if (
    observed_protocol_sha256_08C2B
    != expected_protocol_sha256_08C2B
):
    raise RuntimeError(
        "Model protokolü SHA-256 değeri değişmiş. "
        f"Bulunan: {observed_protocol_sha256_08C2B}"
    )

# ------------------------------------------------------------
# 3. Beş kalıcı checkpoint tablosunu tanımla
# ------------------------------------------------------------

checkpoint_tables_08C2B = [
    (
        f"{TARGET_DATASET}."
        f"model_lr_inner_oof_outer2_inner"
        f"{inner_fold}_v1"
    )
    for inner_fold in range(1, 6)
]

union_parts_08C2B = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_08C2B
]

SQL_LOAD_POOLED_OOF_08C2B = (
    "\nUNION ALL\n".join(
        union_parts_08C2B
    )
)

print(
    "Loading five permanent outer-fold-2 "
    "inner OOF checkpoint tables..."
)

query_job_08C2B = client.query(
    SQL_LOAD_POOLED_OOF_08C2B,
    location=BQ_LOCATION,
)

try:
    pooled_oof_outer2_08C2B = (
        query_job_08C2B.to_dataframe(
            create_bqstorage_client=True
        )
    )
    load_method_08C2B = (
        "BigQuery Storage API"
    )

except Exception as fast_path_error_08C2B:
    print(
        "Storage API fast path unavailable; "
        "using standard BigQuery download."
    )

    print(
        "Fast-path message:",
        type(fast_path_error_08C2B).__name__,
    )

    pooled_oof_outer2_08C2B = (
        query_job_08C2B.to_dataframe(
            create_bqstorage_client=False
        )
    )

    load_method_08C2B = (
        "Standard BigQuery API"
    )

# ------------------------------------------------------------
# 4. Veri tiplerini düzenle
# ------------------------------------------------------------

pooled_oof_outer2_08C2B[
    "id_row"
] = pooled_oof_outer2_08C2B[
    "id_row"
].astype(str)

pooled_oof_outer2_08C2B[
    "candidate_id"
] = pooled_oof_outer2_08C2B[
    "candidate_id"
].astype(str)

for column in [
    "outer_fold",
    "inner_fold",
    "label_stage23",
]:
    pooled_oof_outer2_08C2B[
        column
    ] = pd.to_numeric(
        pooled_oof_outer2_08C2B[column],
        errors="raise",
    ).astype(int)

pooled_oof_outer2_08C2B[
    "prediction_raw"
] = pd.to_numeric(
    pooled_oof_outer2_08C2B[
        "prediction_raw"
    ],
    errors="raise",
).astype(float)

# ------------------------------------------------------------
# 5. Kesin bütünlük kontrolleri
# ------------------------------------------------------------

EXPECTED_PATIENTS_08C2B = 46800
EXPECTED_EVENTS_08C2B = 2426
EXPECTED_CANDIDATES_08C2B = 6
EXPECTED_TOTAL_ROWS_08C2B = (
    EXPECTED_PATIENTS_08C2B
    * EXPECTED_CANDIDATES_08C2B
)

if len(
    pooled_oof_outer2_08C2B
) != EXPECTED_TOTAL_ROWS_08C2B:
    raise RuntimeError(
        f"280.800 pooled OOF satırı bekleniyordu; "
        f"{len(pooled_oof_outer2_08C2B)} bulundu."
    )

if set(
    pooled_oof_outer2_08C2B[
        "outer_fold"
    ].unique()
) != {2}:
    raise RuntimeError(
        "Pooled tabloda dış kat 2 dışında "
        "kayıt bulundu."
    )

if set(
    pooled_oof_outer2_08C2B[
        "inner_fold"
    ].unique()
) != {1, 2, 3, 4, 5}:
    raise RuntimeError(
        "Pooled tabloda iç kat değerleri 1–5 değil."
    )

expected_candidate_ids_08C2B = {
    "LR01",
    "LR02",
    "LR03",
    "LR04",
    "LR05",
    "LR06",
}

observed_candidate_ids_08C2B = set(
    pooled_oof_outer2_08C2B[
        "candidate_id"
    ].unique()
)

if (
    observed_candidate_ids_08C2B
    != expected_candidate_ids_08C2B
):
    raise RuntimeError(
        "Altı kilitli adayın listesi uyuşmuyor."
    )

if pooled_oof_outer2_08C2B[
    "prediction_raw"
].isna().any():
    raise RuntimeError(
        "Pooled OOF tahminlerinde eksik değer var."
    )

if not pooled_oof_outer2_08C2B[
    "prediction_raw"
].between(0, 1).all():
    raise RuntimeError(
        "Pooled OOF tahminlerinde 0–1 "
        "dışında değer var."
    )

if pooled_oof_outer2_08C2B.duplicated(
    subset=[
        "candidate_id",
        "id_row",
    ]
).any():
    raise RuntimeError(
        "Aynı aday–hasta kombinasyonu "
        "birden fazla kez bulundu."
    )

candidate_patient_counts_08C2B = (
    pooled_oof_outer2_08C2B
    .groupby("candidate_id")[
        "id_row"
    ]
    .nunique()
)

if not (
    candidate_patient_counts_08C2B
    == EXPECTED_PATIENTS_08C2B
).all():
    raise RuntimeError(
        "Her aday için 46.800 farklı hasta yok."
    )

candidate_event_counts_08C2B = (
    pooled_oof_outer2_08C2B
    .groupby("candidate_id")[
        "label_stage23"
    ]
    .sum()
)

if not (
    candidate_event_counts_08C2B
    == EXPECTED_EVENTS_08C2B
).all():
    raise RuntimeError(
        "Her aday için olay sayısı 2.426 değil."
    )

patient_label_consistency_08C2B = (
    pooled_oof_outer2_08C2B
    .groupby("id_row")[
        "label_stage23"
    ]
    .nunique()
)

if (
    patient_label_consistency_08C2B
    > 1
).any():
    raise RuntimeError(
        "Aynı hastanın adaylar arasında "
        "farklı outcome etiketi var."
    )

patient_inner_fold_consistency_08C2B = (
    pooled_oof_outer2_08C2B
    .groupby("id_row")[
        "inner_fold"
    ]
    .nunique()
)

if (
    patient_inner_fold_consistency_08C2B
    > 1
).any():
    raise RuntimeError(
        "Aynı hasta birden fazla iç "
        "doğrulama katında bulundu."
    )

# ------------------------------------------------------------
# 6. Kilitli aday ızgarası
# ------------------------------------------------------------

candidate_grid_08C2B = [
    {
        "candidate_id": "LR01",
        "C": 0.03,
        "l1_ratio": 0.00,
    },
    {
        "candidate_id": "LR02",
        "C": 0.10,
        "l1_ratio": 0.00,
    },
    {
        "candidate_id": "LR03",
        "C": 0.30,
        "l1_ratio": 0.00,
    },
    {
        "candidate_id": "LR04",
        "C": 0.10,
        "l1_ratio": 0.25,
    },
    {
        "candidate_id": "LR05",
        "C": 0.30,
        "l1_ratio": 0.25,
    },
    {
        "candidate_id": "LR06",
        "C": 0.30,
        "l1_ratio": 0.50,
    },
]

# ------------------------------------------------------------
# 7. Metrik yardımcıları
# ------------------------------------------------------------

def probability_metrics_08C2B(
    y_true,
    probabilities,
):

    probabilities = np.clip(
        np.asarray(
            probabilities,
            dtype=float,
        ),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1],
            )
        ),
        "mean_predicted_risk": float(
            probabilities.mean()
        ),
        "observed_event_rate": float(
            np.mean(y_true)
        ),
    }


def probability_logit_08C2B(
    probabilities,
):

    probabilities = np.clip(
        np.asarray(
            probabilities,
            dtype=float,
        ),
        1e-6,
        1 - 1e-6,
    )

    return np.log(
        probabilities
        / (1 - probabilities)
    ).reshape(-1, 1)

# ------------------------------------------------------------
# 8. Aggregate fit-audit bilgisini yükle
# ------------------------------------------------------------

fit_audit_path_08C2B = os.path.join(
    MODEL_OUTPUT_DIR,
    "08C1_logistic_inner_fit_audit_outer2.csv",
)

if os.path.exists(fit_audit_path_08C2B):
    fit_audit_08C2B = pd.read_csv(
        fit_audit_path_08C2B
    )
else:
    fit_audit_08C2B = pd.DataFrame()

# ------------------------------------------------------------
# 9. Altı adayın pooled OOF performansı
# ------------------------------------------------------------

candidate_result_rows_08C2B = []

for candidate in candidate_grid_08C2B:

    candidate_id = candidate[
        "candidate_id"
    ]

    candidate_oof = (
        pooled_oof_outer2_08C2B.loc[
            pooled_oof_outer2_08C2B[
                "candidate_id"
            ] == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    if len(candidate_oof) != (
        EXPECTED_PATIENTS_08C2B
    ):
        raise RuntimeError(
            f"{candidate_id}: 46.800 OOF "
            "tahmini bulunmuyor."
        )

    metrics = probability_metrics_08C2B(
        candidate_oof[
            "label_stage23"
        ].to_numpy(dtype=int),
        candidate_oof[
            "prediction_raw"
        ].to_numpy(dtype=float),
    )

    if len(fit_audit_08C2B) > 0:

        fit_part = fit_audit_08C2B.loc[
            (
                pd.to_numeric(
                    fit_audit_08C2B[
                        "outer_fold"
                    ],
                    errors="coerce",
                ) == 2
            )
            & (
                fit_audit_08C2B[
                    "candidate_id"
                ].astype(str)
                == candidate_id
            )
        ]

        convergence_warnings = int(
            pd.to_numeric(
                fit_part[
                    "convergence_warnings"
                ],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )

        elapsed_seconds = float(
            pd.to_numeric(
                fit_part[
                    "elapsed_seconds"
                ],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )

    else:
        convergence_warnings = np.nan
        elapsed_seconds = np.nan

    candidate_result_rows_08C2B.append(
        {
            "candidate_id": candidate_id,
            "C": candidate["C"],
            "l1_ratio": candidate[
                "l1_ratio"
            ],
            **metrics,
            "convergence_warnings": (
                convergence_warnings
            ),
            "elapsed_seconds": (
                elapsed_seconds
            ),
        }
    )

candidate_results_08C2B = pd.DataFrame(
    candidate_result_rows_08C2B
)

candidate_results_08C2B = (
    candidate_results_08C2B
    .sort_values(
        [
            "auprc",
            "auroc",
            "brier",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

candidate_results_08C2B[
    "selection_rank"
] = np.arange(
    1,
    len(candidate_results_08C2B) + 1,
)

best_row_08C2B = (
    candidate_results_08C2B.iloc[0]
)

selected_candidate_id_08C2B = str(
    best_row_08C2B["candidate_id"]
)

selected_C_08C2B = float(
    best_row_08C2B["C"]
)

selected_l1_ratio_08C2B = float(
    best_row_08C2B["l1_ratio"]
)

# ------------------------------------------------------------
# 10. Seçilen adayın Platt kalibrasyonunu öğren
# ------------------------------------------------------------

selected_oof_08C2B = (
    pooled_oof_outer2_08C2B.loc[
        pooled_oof_outer2_08C2B[
            "candidate_id"
        ] == selected_candidate_id_08C2B
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_08C2B = (
    selected_oof_08C2B[
        "label_stage23"
    ].to_numpy(dtype=int)
)

selected_oof_probability_08C2B = (
    selected_oof_08C2B[
        "prediction_raw"
    ].to_numpy(dtype=float)
)

platt_calibrator_08C2B = (
    LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
)

platt_calibrator_08C2B.fit(
    probability_logit_08C2B(
        selected_oof_probability_08C2B
    ),
    selected_oof_y_08C2B,
)

platt_intercept_08C2B = float(
    platt_calibrator_08C2B.intercept_[0]
)

platt_slope_08C2B = float(
    platt_calibrator_08C2B.coef_[0][0]
)

if not np.isfinite(
    platt_intercept_08C2B
):
    raise RuntimeError(
        "Platt kesişimi sonlu değil."
    )

if not np.isfinite(
    platt_slope_08C2B
):
    raise RuntimeError(
        "Platt eğimi sonlu değil."
    )

if platt_slope_08C2B <= 0:
    raise RuntimeError(
        "Platt eğimi pozitif değil."
    )

# ------------------------------------------------------------
# 11. Seçim ve kalibrasyon özetini oluştur
# ------------------------------------------------------------

selected_model_08C2B = pd.DataFrame(
    [
        {
            "outer_fold": 2,
            "selected_candidate": (
                selected_candidate_id_08C2B
            ),
            "selected_C": (
                selected_C_08C2B
            ),
            "selected_l1_ratio": (
                selected_l1_ratio_08C2B
            ),
            "selection_metric_primary": (
                "pooled_inner_oof_auprc"
            ),
            "inner_oof_auprc": float(
                best_row_08C2B["auprc"]
            ),
            "inner_oof_auroc": float(
                best_row_08C2B["auroc"]
            ),
            "inner_oof_brier": float(
                best_row_08C2B["brier"]
            ),
            "inner_oof_log_loss": float(
                best_row_08C2B[
                    "log_loss"
                ]
            ),
            "inner_oof_mean_predicted_risk": float(
                best_row_08C2B[
                    "mean_predicted_risk"
                ]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_08C2B[
                    "observed_event_rate"
                ]
            ),
            "platt_intercept": (
                platt_intercept_08C2B
            ),
            "platt_slope": (
                platt_slope_08C2B
            ),
            "protocol_sha256": (
                expected_protocol_sha256_08C2B
            ),
        }
    ]
)

# ------------------------------------------------------------
# 12. Yalnızca aggregate dosyaları kaydet
# ------------------------------------------------------------

candidate_results_path_08C2B = os.path.join(
    MODEL_OUTPUT_DIR,
    "08C2B_logistic_candidate_results_outer2.csv",
)

selected_model_path_08C2B = os.path.join(
    MODEL_OUTPUT_DIR,
    "08C2B_logistic_selected_model_outer2.csv",
)

selection_json_path_08C2B = os.path.join(
    MODEL_OUTPUT_DIR,
    "08C2B_logistic_selection_calibration_outer2.json",
)

selection_sha_path_08C2B = os.path.join(
    MODEL_OUTPUT_DIR,
    "08C2B_logistic_selection_calibration_outer2_SHA256.txt",
)

candidate_results_08C2B.to_csv(
    candidate_results_path_08C2B,
    index=False,
)

selected_model_08C2B.to_csv(
    selected_model_path_08C2B,
    index=False,
)

selection_configuration_08C2B = {
    "outer_fold": 2,
    "protocol_sha256": (
        expected_protocol_sha256_08C2B
    ),
    "selection_metric_primary": (
        "pooled inner out-of-fold AUPRC"
    ),
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "selected_candidate": (
        selected_candidate_id_08C2B
    ),
    "selected_C": (
        selected_C_08C2B
    ),
    "selected_l1_ratio": (
        selected_l1_ratio_08C2B
    ),
    "inner_oof_auprc": float(
        best_row_08C2B["auprc"]
    ),
    "inner_oof_auroc": float(
        best_row_08C2B["auroc"]
    ),
    "inner_oof_brier": float(
        best_row_08C2B["brier"]
    ),
    "platt_intercept": (
        platt_intercept_08C2B
    ),
    "platt_slope": (
        platt_slope_08C2B
    ),
    "inner_checkpoint_tables": (
        checkpoint_tables_08C2B
    ),
    "patient_level_oof_written_to_drive": False,
}

with open(
    selection_json_path_08C2B,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        selection_configuration_08C2B,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(
    selection_json_path_08C2B,
    "rb",
) as file_handle:
    selection_sha256_08C2B = (
        hashlib.sha256(
            file_handle.read()
        ).hexdigest()
    )

with open(
    selection_sha_path_08C2B,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(
        selection_sha256_08C2B
        + "\n"
    )

# ------------------------------------------------------------
# 13. Sonuçları göster
# ------------------------------------------------------------

pooled_integrity_summary_08C2B = (
    pd.DataFrame(
        {
            "metric": [
                "pooled_prediction_rows",
                "distinct_patients",
                "candidate_models",
                "inner_folds",
                "events_per_candidate",
                "nonevents_per_candidate",
                "duplicate_candidate_patient_rows",
                "missing_predictions",
                "invalid_probabilities",
                "load_method",
            ],
            "value": [
                len(
                    pooled_oof_outer2_08C2B
                ),
                pooled_oof_outer2_08C2B[
                    "id_row"
                ].nunique(),
                pooled_oof_outer2_08C2B[
                    "candidate_id"
                ].nunique(),
                pooled_oof_outer2_08C2B[
                    "inner_fold"
                ].nunique(),
                EXPECTED_EVENTS_08C2B,
                (
                    EXPECTED_PATIENTS_08C2B
                    - EXPECTED_EVENTS_08C2B
                ),
                int(
                    pooled_oof_outer2_08C2B
                    .duplicated(
                        subset=[
                            "candidate_id",
                            "id_row",
                        ]
                    )
                    .sum()
                ),
                int(
                    pooled_oof_outer2_08C2B[
                        "prediction_raw"
                    ]
                    .isna()
                    .sum()
                ),
                int(
                    (
                        ~pooled_oof_outer2_08C2B[
                            "prediction_raw"
                        ].between(0, 1)
                    )
                    .sum()
                ),
                load_method_08C2B,
            ],
        }
    )
)

print(
    "\n08C2-B POOLED OOF INTEGRITY"
)
display(pooled_integrity_summary_08C2B)

print(
    "\n08C2-B OUTER-FOLD-2 "
    "CANDIDATE RESULTS"
)
display(candidate_results_08C2B)

print(
    "\n08C2-B OUTER-FOLD-2 "
    "SELECTED MODEL AND CALIBRATION"
)
display(selected_model_08C2B)

print("\nSelection/calibration SHA-256:")
print(selection_sha256_08C2B)

print("\nSaved:")
print(candidate_results_path_08C2B)
print(selected_model_path_08C2B)
print(selection_json_path_08C2B)
print(selection_sha_path_08C2B)

print(
    "\n08C2-B PASS: Outer-fold-2 "
    "hyperparameters and Platt calibration "
    "coefficients were locked."
)

print(
    "No outer-fold-2 test outcome was used."
)

print(
    "No patient-level OOF prediction file "
    "was written to Google Drive."
)

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from sklearn.exceptions import ConvergenceWarning

from IPython.display import display

# ============================================================
# 08C2-C
# Final outer-fold-2 fit, locked test evaluation and
# secure BigQuery prediction checkpoint.
#
# Hyperparameters and Platt coefficients are loaded from
# the previously locked inner-OOF selection.
#
# No outer-test information is used during model fitting
# or calibration fitting.
# ============================================================

TARGET_OUTER_FOLD_08C2C = 2
MODEL_RANDOM_SEED_08C2C = 20260721

EXPECTED_PROTOCOL_SHA_08C2C = (
    "400c3b4b510c836794543bc685c62fae"
    "f49df0c1caa197d78dffda8c2207952d"
)

EXPECTED_SELECTION_SHA_08C2C = (
    "7bf7bbc97fcfcd69397579b21a1d9b9b"
    "3fc3250367b1b110658c3dd20b251479"
)

# ------------------------------------------------------------
# 1. Gerekli RAM nesnelerini doğrula
# ------------------------------------------------------------

required_objects_08C2C = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_08C2C = [
    name
    for name in required_objects_08C2C
    if name not in globals()
]

if missing_objects_08C2C:
    raise RuntimeError(
        "Eksik RAM nesneleri var: "
        + ", ".join(missing_objects_08C2C)
        + ". Önce 07A ve 07B hücrelerini çalıştır."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"58.491 satır bekleniyordu; "
        f"{len(core_df_07B)} bulundu."
    )

if len(predictor_columns_07B) != 159:
    raise RuntimeError(
        "Core predictor sayısı 159 değil."
    )

if len(numeric_columns_07B) != 156:
    raise RuntimeError(
        "Sayısal predictor sayısı 156 değil."
    )

if len(categorical_columns_07B) != 3:
    raise RuntimeError(
        "Kategorik predictor sayısı 3 değil."
    )

# ------------------------------------------------------------
# 2. Kilitli seçim ve kalibrasyon sonuçlarını yükle
# ------------------------------------------------------------

selected_model_path_08C2C = os.path.join(
    MODEL_OUTPUT_DIR,
    "08C2B_logistic_selected_model_outer2.csv",
)

selection_sha_path_08C2C = os.path.join(
    MODEL_OUTPUT_DIR,
    "08C2B_logistic_selection_calibration_outer2_SHA256.txt",
)

protocol_sha_path_08C2C = os.path.join(
    MODEL_OUTPUT_DIR,
    "08B_locked_logistic_model_protocol_v1_SHA256.txt",
)

for required_path in [
    selected_model_path_08C2C,
    selection_sha_path_08C2C,
    protocol_sha_path_08C2C,
]:
    if not os.path.exists(required_path):
        raise FileNotFoundError(
            "Gerekli kilit dosyası bulunamadı: "
            + required_path
        )

with open(
    selection_sha_path_08C2C,
    "r",
    encoding="utf-8",
) as file_handle:
    observed_selection_sha_08C2C = (
        file_handle.read().strip()
    )

with open(
    protocol_sha_path_08C2C,
    "r",
    encoding="utf-8",
) as file_handle:
    observed_protocol_sha_08C2C = (
        file_handle.read().strip()
    )

if (
    observed_selection_sha_08C2C
    != EXPECTED_SELECTION_SHA_08C2C
):
    raise RuntimeError(
        "Dış kat 2 seçim SHA-256 değeri değişmiş: "
        + observed_selection_sha_08C2C
    )

if (
    observed_protocol_sha_08C2C
    != EXPECTED_PROTOCOL_SHA_08C2C
):
    raise RuntimeError(
        "Model protokolü SHA-256 değeri değişmiş: "
        + observed_protocol_sha_08C2C
    )

selected_model_locked_08C2C = pd.read_csv(
    selected_model_path_08C2C
)

if len(selected_model_locked_08C2C) != 1:
    raise RuntimeError(
        "Seçili model dosyasında tam olarak "
        "bir satır bulunmalı."
    )

selected_row_08C2C = (
    selected_model_locked_08C2C.iloc[0]
)

selected_candidate_08C2C = str(
    selected_row_08C2C[
        "selected_candidate"
    ]
)

selected_C_08C2C = float(
    selected_row_08C2C[
        "selected_C"
    ]
)

selected_l1_ratio_08C2C = float(
    selected_row_08C2C[
        "selected_l1_ratio"
    ]
)

platt_intercept_locked_08C2C = float(
    selected_row_08C2C[
        "platt_intercept"
    ]
)

platt_slope_locked_08C2C = float(
    selected_row_08C2C[
        "platt_slope"
    ]
)

if selected_candidate_08C2C != "LR06":
    raise RuntimeError(
        "Dış kat 2 seçili aday LR06 değil."
    )

if not np.isclose(
    selected_C_08C2C,
    0.30,
):
    raise RuntimeError(
        "Dış kat 2 seçili C değeri 0.30 değil."
    )

if not np.isclose(
    selected_l1_ratio_08C2C,
    0.50,
):
    raise RuntimeError(
        "Dış kat 2 seçili l1_ratio 0.50 değil."
    )

if not np.isfinite(
    platt_intercept_locked_08C2C
):
    raise RuntimeError(
        "Kilitli Platt kesişimi sonlu değil."
    )

if (
    not np.isfinite(
        platt_slope_locked_08C2C
    )
    or platt_slope_locked_08C2C <= 0
):
    raise RuntimeError(
        "Kilitli Platt eğimi geçersiz."
    )

print(
    "Locked candidate:",
    selected_candidate_08C2C,
)

print(
    "Locked C:",
    selected_C_08C2C,
)

print(
    "Locked l1_ratio:",
    selected_l1_ratio_08C2C,
)

print(
    "Locked Platt intercept:",
    platt_intercept_locked_08C2C,
)

print(
    "Locked Platt slope:",
    platt_slope_locked_08C2C,
)

# ------------------------------------------------------------
# 3. Dış kat 2 eğitim ve test matrislerini hazırla
# ------------------------------------------------------------

X_all_08C2C = core_df_07B[
    predictor_columns_07B
].copy()

for column in numeric_columns_07B:
    X_all_08C2C[column] = pd.to_numeric(
        X_all_08C2C[column],
        errors="coerce",
    ).astype("float64")

for column in categorical_columns_07B:
    category_series = (
        X_all_08C2C[column]
        .astype("object")
    )

    X_all_08C2C[column] = (
        category_series.where(
            pd.notna(category_series),
            np.nan,
        )
    )

outer_fold_all_08C2C = (
    core_df_07B["outer_fold"]
    .astype(int)
    .to_numpy()
)

outer_training_mask_08C2C = (
    outer_fold_all_08C2C
    != TARGET_OUTER_FOLD_08C2C
)

outer_test_mask_08C2C = (
    outer_fold_all_08C2C
    == TARGET_OUTER_FOLD_08C2C
)

X_outer_training_08C2C = (
    X_all_08C2C.loc[
        outer_training_mask_08C2C
    ]
    .reset_index(drop=True)
)

X_outer_test_08C2C = (
    X_all_08C2C.loc[
        outer_test_mask_08C2C
    ]
    .reset_index(drop=True)
)

training_meta_08C2C = (
    core_df_07B.loc[
        outer_training_mask_08C2C,
        [
            "id_row",
            "group_hospital",
            "label_stage23",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

test_meta_08C2C = (
    core_df_07B.loc[
        outer_test_mask_08C2C,
        [
            "id_row",
            "group_hospital",
            "label_stage23",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

training_meta_08C2C[
    "group_hospital"
] = training_meta_08C2C[
    "group_hospital"
].astype(str)

test_meta_08C2C[
    "group_hospital"
] = test_meta_08C2C[
    "group_hospital"
].astype(str)

test_meta_08C2C[
    "id_row"
] = test_meta_08C2C[
    "id_row"
].astype(str)

y_outer_training_08C2C = (
    training_meta_08C2C[
        "label_stage23"
    ]
    .astype(int)
    .to_numpy(dtype=np.int8)
)

y_outer_test_08C2C = (
    test_meta_08C2C[
        "label_stage23"
    ]
    .astype(int)
    .to_numpy(dtype=np.int8)
)

training_hospitals_08C2C = set(
    training_meta_08C2C[
        "group_hospital"
    ]
)

test_hospitals_08C2C = set(
    test_meta_08C2C[
        "group_hospital"
    ]
)

hospital_overlap_08C2C = (
    training_hospitals_08C2C
    & test_hospitals_08C2C
)

if hospital_overlap_08C2C:
    raise RuntimeError(
        "Dış eğitim ve test kümeleri arasında "
        "hastane çakışması bulundu."
    )

expected_split_08C2C = {
    "training_rows": 46800,
    "test_rows": 11691,
    "training_hospitals": 158,
    "test_hospitals": 40,
    "training_events": 2426,
    "test_events": 606,
}

actual_split_08C2C = {
    "training_rows": len(
        X_outer_training_08C2C
    ),
    "test_rows": len(
        X_outer_test_08C2C
    ),
    "training_hospitals": len(
        training_hospitals_08C2C
    ),
    "test_hospitals": len(
        test_hospitals_08C2C
    ),
    "training_events": int(
        y_outer_training_08C2C.sum()
    ),
    "test_events": int(
        y_outer_test_08C2C.sum()
    ),
}

for metric, expected_value in (
    expected_split_08C2C.items()
):
    actual_value = actual_split_08C2C[
        metric
    ]

    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: bulunan={actual_value}, "
            f"beklenen={expected_value}"
        )

# ------------------------------------------------------------
# 4. Kilitli preprocessing ve model pipeline'ı
# ------------------------------------------------------------

def make_preprocessor_08C2C():

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
            (
                "scaler",
                StandardScaler(
                    with_mean=False,
                ),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns_07B,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns_07B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


final_pipeline_outer2_08C2C = Pipeline(
    steps=[
        (
            "preprocessor",
            make_preprocessor_08C2C(),
        ),
        (
            "model",
            LogisticRegression(
                penalty="elasticnet",
                solver="saga",
                C=selected_C_08C2C,
                l1_ratio=(
                    selected_l1_ratio_08C2C
                ),
                class_weight=None,
                max_iter=5000,
                tol=1e-4,
                random_state=(
                    MODEL_RANDOM_SEED_08C2C
                ),
            ),
        ),
    ]
)

# ------------------------------------------------------------
# 5. Nihai modeli yalnızca dış eğitim kümesinde fit et
# ------------------------------------------------------------

print(
    "\nFitting selected outer-fold-2 model "
    "on all 46,800 training patients..."
)

fit_started_08C2C = time.time()

with warnings.catch_warnings(
    record=True
) as fit_warning_records_08C2C:

    warnings.simplefilter(
        "always",
        ConvergenceWarning,
    )

    final_pipeline_outer2_08C2C.fit(
        X_outer_training_08C2C,
        y_outer_training_08C2C,
    )

fit_elapsed_seconds_08C2C = (
    time.time()
    - fit_started_08C2C
)

convergence_warnings_08C2C = sum(
    issubclass(
        warning.category,
        ConvergenceWarning,
    )
    for warning in fit_warning_records_08C2C
)

if convergence_warnings_08C2C != 0:
    raise RuntimeError(
        "Nihai dış kat 2 modelinde "
        "yakınsama uyarısı oluştu."
    )

# ------------------------------------------------------------
# 6. Dış test olasılıklarını üret
# ------------------------------------------------------------

outer2_raw_probabilities_08C2C = (
    final_pipeline_outer2_08C2C
    .predict_proba(
        X_outer_test_08C2C
    )[:, 1]
)

raw_clipped_08C2C = np.clip(
    outer2_raw_probabilities_08C2C,
    1e-6,
    1 - 1e-6,
)

raw_logit_08C2C = np.log(
    raw_clipped_08C2C
    / (1 - raw_clipped_08C2C)
)

outer2_platt_probabilities_08C2C = (
    expit(
        platt_intercept_locked_08C2C
        + platt_slope_locked_08C2C
        * raw_logit_08C2C
    )
)

for probabilities, name in [
    (
        outer2_raw_probabilities_08C2C,
        "raw",
    ),
    (
        outer2_platt_probabilities_08C2C,
        "platt",
    ),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(
            f"{name} tahminlerinde eksik değer var."
        )

    if not np.all(
        (
            probabilities >= 0
        )
        & (
            probabilities <= 1
        )
    ):
        raise RuntimeError(
            f"{name} tahminlerinde 0–1 "
            "dışında değer bulundu."
        )

# ------------------------------------------------------------
# 7. Performans ve kalibrasyon yardımcıları
# ------------------------------------------------------------

def probability_metrics_08C2C(
    y_true,
    probabilities,
):

    probabilities = np.clip(
        np.asarray(
            probabilities,
            dtype=float,
        ),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1],
            )
        ),
        "mean_predicted_risk": float(
            probabilities.mean()
        ),
        "observed_event_rate": float(
            np.mean(y_true)
        ),
    }


def probability_logit_08C2C(
    probabilities,
):

    probabilities = np.clip(
        np.asarray(
            probabilities,
            dtype=float,
        ),
        1e-6,
        1 - 1e-6,
    )

    return np.log(
        probabilities
        / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_08C2C(
    y_true,
    probabilities,
):

    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )

    calibration_model.fit(
        probability_logit_08C2C(
            probabilities
        ),
        y_true,
    )

    return (
        float(
            calibration_model.intercept_[0]
        ),
        float(
            calibration_model.coef_[0][0]
        ),
    )

# ------------------------------------------------------------
# 8. Dış test metriklerini hesapla
# ------------------------------------------------------------

raw_metrics_08C2C = (
    probability_metrics_08C2C(
        y_outer_test_08C2C,
        outer2_raw_probabilities_08C2C,
    )
)

platt_metrics_08C2C = (
    probability_metrics_08C2C(
        y_outer_test_08C2C,
        outer2_platt_probabilities_08C2C,
    )
)

raw_calibration_intercept_08C2C, \
raw_calibration_slope_08C2C = (
    calibration_intercept_slope_08C2C(
        y_outer_test_08C2C,
        outer2_raw_probabilities_08C2C,
    )
)

platt_calibration_intercept_08C2C, \
platt_calibration_slope_08C2C = (
    calibration_intercept_slope_08C2C(
        y_outer_test_08C2C,
        outer2_platt_probabilities_08C2C,
    )
)

outer2_test_results_08C2C = pd.DataFrame(
    [
        {
            "outer_fold": 2,
            "model": (
                "elastic_net_logistic"
            ),
            "probability_type": "raw",
            **raw_metrics_08C2C,
            "calibration_intercept": (
                raw_calibration_intercept_08C2C
            ),
            "calibration_slope": (
                raw_calibration_slope_08C2C
            ),
        },
        {
            "outer_fold": 2,
            "model": (
                "elastic_net_logistic"
            ),
            "probability_type": (
                "platt_calibrated"
            ),
            **platt_metrics_08C2C,
            "calibration_intercept": (
                platt_calibration_intercept_08C2C
            ),
            "calibration_slope": (
                platt_calibration_slope_08C2C
            ),
        },
    ]
)

# ------------------------------------------------------------
# 9. İşlenmiş feature ve katsayı denetimi
# ------------------------------------------------------------

fitted_preprocessor_08C2C = (
    final_pipeline_outer2_08C2C
    .named_steps["preprocessor"]
)

fitted_model_08C2C = (
    final_pipeline_outer2_08C2C
    .named_steps["model"]
)

processed_feature_names_08C2C = (
    fitted_preprocessor_08C2C
    .get_feature_names_out()
)

model_coefficients_08C2C = (
    fitted_model_08C2C
    .coef_
    .reshape(-1)
)

if len(
    processed_feature_names_08C2C
) != len(model_coefficients_08C2C):
    raise RuntimeError(
        "İşlenmiş feature sayısı ile "
        "model katsayı sayısı uyuşmuyor."
    )

if len(
    set(processed_feature_names_08C2C)
) != len(processed_feature_names_08C2C):
    raise RuntimeError(
        "İşlenmiş feature adlarında "
        "yinelenme bulundu."
    )

coefficient_table_08C2C = pd.DataFrame(
    {
        "processed_feature": (
            processed_feature_names_08C2C
        ),
        "coefficient": (
            model_coefficients_08C2C
        ),
    }
)

coefficient_table_08C2C[
    "absolute_coefficient"
] = coefficient_table_08C2C[
    "coefficient"
].abs()

coefficient_table_08C2C[
    "is_nonzero"
] = ~np.isclose(
    coefficient_table_08C2C[
        "coefficient"
    ],
    0.0,
    atol=1e-12,
)

coefficient_table_08C2C[
    "absolute_rank"
] = (
    coefficient_table_08C2C[
        "absolute_coefficient"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)

coefficient_table_08C2C = (
    coefficient_table_08C2C
    .sort_values(
        [
            "absolute_coefficient",
            "processed_feature",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

nonzero_coefficients_08C2C = int(
    coefficient_table_08C2C[
        "is_nonzero"
    ].sum()
)

# ------------------------------------------------------------
# 10. Nihai model özeti
# ------------------------------------------------------------

final_model_summary_08C2C = pd.DataFrame(
    [
        {
            "outer_fold": 2,
            "selected_candidate": (
                selected_candidate_08C2C
            ),
            "selected_C": (
                selected_C_08C2C
            ),
            "selected_l1_ratio": (
                selected_l1_ratio_08C2C
            ),
            "training_patients": len(
                X_outer_training_08C2C
            ),
            "training_hospitals": len(
                training_hospitals_08C2C
            ),
            "training_events": int(
                y_outer_training_08C2C.sum()
            ),
            "test_patients": len(
                X_outer_test_08C2C
            ),
            "test_hospitals": len(
                test_hospitals_08C2C
            ),
            "test_events": int(
                y_outer_test_08C2C.sum()
            ),
            "hospital_overlap": len(
                hospital_overlap_08C2C
            ),
            "processed_feature_columns": len(
                processed_feature_names_08C2C
            ),
            "nonzero_coefficients": (
                nonzero_coefficients_08C2C
            ),
            "model_intercept": float(
                fitted_model_08C2C
                .intercept_[0]
            ),
            "convergence_warnings": (
                convergence_warnings_08C2C
            ),
            "fit_elapsed_seconds": float(
                fit_elapsed_seconds_08C2C
            ),
            "locked_platt_intercept": (
                platt_intercept_locked_08C2C
            ),
            "locked_platt_slope": (
                platt_slope_locked_08C2C
            ),
            "protocol_sha256": (
                EXPECTED_PROTOCOL_SHA_08C2C
            ),
            "selection_sha256": (
                EXPECTED_SELECTION_SHA_08C2C
            ),
        }
    ]
)

# ------------------------------------------------------------
# 11. Hasta düzeyi dış test tahmin tablosu
#     Yalnızca güvenli BigQuery'ye yazılacaktır.
# ------------------------------------------------------------

outer2_prediction_table_08C2C = pd.DataFrame(
    {
        "id_row": (
            test_meta_08C2C[
                "id_row"
            ].astype(str)
        ),
        "outer_fold": np.full(
            len(test_meta_08C2C),
            2,
            dtype=np.int64,
        ),
        "label_stage23": (
            y_outer_test_08C2C
            .astype(np.int64)
        ),
        "prediction_raw": (
            outer2_raw_probabilities_08C2C
            .astype(np.float64)
        ),
        "prediction_platt": (
            outer2_platt_probabilities_08C2C
            .astype(np.float64)
        ),
        "model_name": (
            "elastic_net_logistic"
        ),
        "model_version": (
            "core_v1_nested_cv"
        ),
    }
)

if len(
    outer2_prediction_table_08C2C
) != 11691:
    raise RuntimeError(
        "Dış kat 2 tahmin satır sayısı "
        "11.691 değil."
    )

if outer2_prediction_table_08C2C[
    "id_row"
].duplicated().any():
    raise RuntimeError(
        "Dış kat 2 tahmin tablosunda "
        "yinelenen id_row bulundu."
    )

if int(
    outer2_prediction_table_08C2C[
        "label_stage23"
    ].sum()
) != 606:
    raise RuntimeError(
        "Dış kat 2 tahmin tablosunda "
        "olay sayısı 606 değil."
    )

# ------------------------------------------------------------
# 12. BigQuery kalıcı dış test checkpoint
# ------------------------------------------------------------

prediction_table_id_08C2C = (
    f"{TARGET_DATASET}."
    "model_lr_outer_predictions_outer2_v1"
)

prediction_load_config_08C2C = (
    bigquery.LoadJobConfig(
        schema=[
            bigquery.SchemaField(
                "id_row",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "outer_fold",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "label_stage23",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_raw",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_platt",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_name",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_version",
                "STRING",
                mode="REQUIRED",
            ),
        ],
        write_disposition=(
            bigquery.WriteDisposition.WRITE_TRUNCATE
        ),
    )
)

print(
    "\nUploading secure outer-fold-2 "
    "test prediction checkpoint:"
)

print(prediction_table_id_08C2C)

client.load_table_from_dataframe(
    outer2_prediction_table_08C2C,
    prediction_table_id_08C2C,
    job_config=prediction_load_config_08C2C,
    location=BQ_LOCATION,
).result()

# ------------------------------------------------------------
# 13. BigQuery checkpoint doğrulaması
# ------------------------------------------------------------

SQL_VERIFY_08C2C = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL)
    AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL)
    AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0
    OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0
    OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw)
    AS minimum_raw_probability,
  MAX(prediction_raw)
    AS maximum_raw_probability,
  MIN(prediction_platt)
    AS minimum_platt_probability,
  MAX(prediction_platt)
    AS maximum_platt_probability
FROM `{prediction_table_id_08C2C}`;
"""

prediction_verification_08C2C = (
    client.query(
        SQL_VERIFY_08C2C,
        location=BQ_LOCATION,
    )
    .to_dataframe()
)

verification_row_08C2C = (
    prediction_verification_08C2C.iloc[0]
)

expected_verification_08C2C = {
    "prediction_rows": 11691,
    "distinct_rows": 11691,
    "outer_folds": 1,
    "minimum_outer_fold": 2,
    "maximum_outer_fold": 2,
    "events": 606,
    "nonevents": 11085,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in (
    expected_verification_08C2C.items()
):
    actual_value = int(
        verification_row_08C2C[field]
    )

    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: bulunan={actual_value}, "
            f"beklenen={expected_value}"
        )

# ------------------------------------------------------------
# 14. Yalnızca aggregate ve feature-level dosyaları Drive'a yaz
# ------------------------------------------------------------

test_results_path_08C2C = os.path.join(
    MODEL_OUTPUT_DIR,
    "08C2C_logistic_outer2_test_results.csv",
)

model_summary_path_08C2C = os.path.join(
    MODEL_OUTPUT_DIR,
    "08C2C_logistic_final_model_outer2.csv",
)

coefficient_path_08C2C = os.path.join(
    MODEL_OUTPUT_DIR,
    "08C2C_logistic_coefficients_outer2.csv",
)

configuration_path_08C2C = os.path.join(
    MODEL_OUTPUT_DIR,
    "08C2C_logistic_final_evaluation_outer2.json",
)

configuration_sha_path_08C2C = os.path.join(
    MODEL_OUTPUT_DIR,
    "08C2C_logistic_final_evaluation_outer2_SHA256.txt",
)

outer2_test_results_08C2C.to_csv(
    test_results_path_08C2C,
    index=False,
)

final_model_summary_08C2C.to_csv(
    model_summary_path_08C2C,
    index=False,
)

coefficient_table_08C2C.to_csv(
    coefficient_path_08C2C,
    index=False,
)

configuration_08C2C = {
    "outer_fold": 2,
    "model_family": (
        "elastic_net_logistic_regression"
    ),
    "selected_candidate": (
        selected_candidate_08C2C
    ),
    "selected_C": selected_C_08C2C,
    "selected_l1_ratio": (
        selected_l1_ratio_08C2C
    ),
    "training_patients": 46800,
    "training_hospitals": 158,
    "test_patients": 11691,
    "test_hospitals": 40,
    "hospital_overlap": 0,
    "locked_platt_intercept": (
        platt_intercept_locked_08C2C
    ),
    "locked_platt_slope": (
        platt_slope_locked_08C2C
    ),
    "processed_feature_columns": int(
        len(processed_feature_names_08C2C)
    ),
    "nonzero_coefficients": int(
        nonzero_coefficients_08C2C
    ),
    "protocol_sha256": (
        EXPECTED_PROTOCOL_SHA_08C2C
    ),
    "selection_sha256": (
        EXPECTED_SELECTION_SHA_08C2C
    ),
    "secure_prediction_table": (
        prediction_table_id_08C2C
    ),
    "patient_level_prediction_written_to_drive": (
        False
    ),
}

with open(
    configuration_path_08C2C,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        configuration_08C2C,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(
    configuration_path_08C2C,
    "rb",
) as file_handle:
    evaluation_sha256_08C2C = (
        hashlib.sha256(
            file_handle.read()
        ).hexdigest()
    )

with open(
    configuration_sha_path_08C2C,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(
        evaluation_sha256_08C2C
        + "\n"
    )

# ------------------------------------------------------------
# 15. Yalnızca aggregate sonuçları göster
# ------------------------------------------------------------

print(
    "\n08C2-C OUTER-FOLD-2 FINAL MODEL SUMMARY"
)
display(final_model_summary_08C2C)

print(
    "\n08C2-C OUTER-FOLD-2 TEST RESULTS"
)
display(outer2_test_results_08C2C)

print(
    "\n08C2-C BIGQUERY PREDICTION VERIFICATION"
)
display(prediction_verification_08C2C)

print(
    "\n08C2-C TOP 20 ABSOLUTE COEFFICIENTS"
)
display(
    coefficient_table_08C2C.head(20)
)

print("\nEvaluation SHA-256:")
print(evaluation_sha256_08C2C)

print("\nSaved:")
print(test_results_path_08C2C)
print(model_summary_path_08C2C)
print(coefficient_path_08C2C)
print(configuration_path_08C2C)
print(configuration_sha_path_08C2C)

print(
    "\n08C2-C PASS: Outer-fold-2 final model "
    "was evaluated on the locked test hospitals."
)

print(
    "Hyperparameters and calibration coefficients "
    "were learned only from the outer training set."
)

print(
    "Patient-level outer-test predictions were "
    "checkpointed only in secure BigQuery."
)

print(
    "No patient-level prediction file was "
    "written to Google Drive."
)

# Büyük ara nesnelerin temizlenmesi
gc.collect()

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from sklearn.exceptions import ConvergenceWarning

from IPython.display import display

# ============================================================
# 09 — OUTER FOLD 3 COMPLETE NESTED MODELLING
#
# Resume-safe:
# - Each inner fold has a separate BigQuery table.
# - Completed inner folds are skipped.
#
# Free-tier compatible:
# - No DELETE / INSERT / UPDATE / MERGE.
#
# Privacy:
# - Patient-level predictions go only to BigQuery.
# - No patient-level prediction file is written to Drive.
# ============================================================

OUTER_FOLD_09 = 3
MODEL_RANDOM_SEED_09 = 20260721

EXPECTED_PROTOCOL_SHA_09 = (
    "400c3b4b510c836794543bc685c62fae"
    "f49df0c1caa197d78dffda8c2207952d"
)

EXPECTED_SPLIT_09 = {
    "training_rows": 46755,
    "test_rows": 11736,
    "training_hospitals": 158,
    "test_hospitals": 40,
    "training_events": 2424,
    "test_events": 608,
}

# ------------------------------------------------------------
# 1. Gerekli çalışma nesnelerini doğrula
# ------------------------------------------------------------

required_objects_09 = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_09 = [
    name
    for name in required_objects_09
    if name not in globals()
]

if missing_objects_09:
    raise RuntimeError(
        "Eksik RAM nesneleri var: "
        + ", ".join(missing_objects_09)
        + ". Önce 07A ve 07B hücrelerini çalıştır."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"58.491 satır bekleniyordu; "
        f"{len(core_df_07B)} bulundu."
    )

if len(predictor_columns_07B) != 159:
    raise RuntimeError(
        "Core predictor sayısı 159 değil."
    )

if len(numeric_columns_07B) != 156:
    raise RuntimeError(
        "Sayısal predictor sayısı 156 değil."
    )

if len(categorical_columns_07B) != 3:
    raise RuntimeError(
        "Kategorik predictor sayısı 3 değil."
    )

# ------------------------------------------------------------
# 2. Kilitli protokol SHA kontrolü
# ------------------------------------------------------------

protocol_sha_path_09 = os.path.join(
    MODEL_OUTPUT_DIR,
    "08B_locked_logistic_model_protocol_v1_SHA256.txt",
)

if not os.path.exists(protocol_sha_path_09):
    raise FileNotFoundError(
        "Model protokolü SHA dosyası bulunamadı: "
        + protocol_sha_path_09
    )

with open(
    protocol_sha_path_09,
    "r",
    encoding="utf-8",
) as file_handle:
    observed_protocol_sha_09 = (
        file_handle.read().strip()
    )

if observed_protocol_sha_09 != EXPECTED_PROTOCOL_SHA_09:
    raise RuntimeError(
        "Kilitli model protokolü SHA değeri değişmiş: "
        + observed_protocol_sha_09
    )

# ------------------------------------------------------------
# 3. Kilitli iç hastane katlarını yükle
# ------------------------------------------------------------

inner_mapping_path_09 = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_09):
    raise FileNotFoundError(
        "Kilitli iç kat haritası bulunamadı: "
        + inner_mapping_path_09
    )

inner_mapping_all_09 = pd.read_csv(
    inner_mapping_path_09,
    dtype={
        "group_hospital": str,
    },
)

inner_mapping_part_09 = (
    inner_mapping_all_09.loc[
        inner_mapping_all_09[
            "outer_fold"
        ].astype(int) == OUTER_FOLD_09,
        [
            "group_hospital",
            "inner_fold",
        ],
    ]
    .copy()
)

inner_mapping_part_09[
    "group_hospital"
] = inner_mapping_part_09[
    "group_hospital"
].astype(str)

inner_mapping_part_09[
    "inner_fold"
] = inner_mapping_part_09[
    "inner_fold"
].astype(int)

if len(inner_mapping_part_09) != 158:
    raise RuntimeError(
        "Dış kat 3 eğitim kümesi için "
        "158 hastane ataması bekleniyordu."
    )

if inner_mapping_part_09[
    "group_hospital"
].duplicated().any():
    raise RuntimeError(
        "İç kat haritasında yinelenen hastane var."
    )

hospital_to_inner_fold_09 = dict(
    zip(
        inner_mapping_part_09[
            "group_hospital"
        ],
        inner_mapping_part_09[
            "inner_fold"
        ],
    )
)

# ------------------------------------------------------------
# 4. Model matrisini hazırla
# ------------------------------------------------------------

X_all_09 = core_df_07B[
    predictor_columns_07B
].copy()

for column in numeric_columns_07B:
    X_all_09[column] = pd.to_numeric(
        X_all_09[column],
        errors="coerce",
    ).astype("float64")

for column in categorical_columns_07B:
    category_series = (
        X_all_09[column]
        .astype("object")
    )

    X_all_09[column] = category_series.where(
        pd.notna(category_series),
        np.nan,
    )

outer_fold_vector_09 = (
    core_df_07B["outer_fold"]
    .astype(int)
    .to_numpy()
)

outer_training_mask_09 = (
    outer_fold_vector_09 != OUTER_FOLD_09
)

outer_test_mask_09 = (
    outer_fold_vector_09 == OUTER_FOLD_09
)

X_outer_training_09 = (
    X_all_09.loc[
        outer_training_mask_09
    ]
    .reset_index(drop=True)
)

X_outer_test_09 = (
    X_all_09.loc[
        outer_test_mask_09
    ]
    .reset_index(drop=True)
)

outer_training_meta_09 = (
    core_df_07B.loc[
        outer_training_mask_09,
        [
            "id_row",
            "group_hospital",
            "label_stage23",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_09 = (
    core_df_07B.loc[
        outer_test_mask_09,
        [
            "id_row",
            "group_hospital",
            "label_stage23",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [
    outer_training_meta_09,
    outer_test_meta_09,
]:
    dataframe[
        "id_row"
    ] = dataframe[
        "id_row"
    ].astype(str)

    dataframe[
        "group_hospital"
    ] = dataframe[
        "group_hospital"
    ].astype(str)

    dataframe[
        "label_stage23"
    ] = dataframe[
        "label_stage23"
    ].astype(int)

y_outer_training_09 = (
    outer_training_meta_09[
        "label_stage23"
    ]
    .to_numpy(dtype=np.int8)
)

y_outer_test_09 = (
    outer_test_meta_09[
        "label_stage23"
    ]
    .to_numpy(dtype=np.int8)
)

groups_outer_training_09 = (
    outer_training_meta_09[
        "group_hospital"
    ]
    .to_numpy(dtype=str)
)

training_hospitals_09 = set(
    outer_training_meta_09[
        "group_hospital"
    ]
)

test_hospitals_09 = set(
    outer_test_meta_09[
        "group_hospital"
    ]
)

hospital_overlap_09 = (
    training_hospitals_09
    & test_hospitals_09
)

if hospital_overlap_09:
    raise RuntimeError(
        "Dış eğitim ve test hastaneleri çakışıyor."
    )

actual_split_09 = {
    "training_rows": len(
        X_outer_training_09
    ),
    "test_rows": len(
        X_outer_test_09
    ),
    "training_hospitals": len(
        training_hospitals_09
    ),
    "test_hospitals": len(
        test_hospitals_09
    ),
    "training_events": int(
        y_outer_training_09.sum()
    ),
    "test_events": int(
        y_outer_test_09.sum()
    ),
}

for metric, expected_value in (
    EXPECTED_SPLIT_09.items()
):
    actual_value = actual_split_09[
        metric
    ]

    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: bulunan={actual_value}, "
            f"beklenen={expected_value}"
        )

inner_fold_vector_09 = np.array(
    [
        hospital_to_inner_fold_09.get(
            hospital,
            -1,
        )
        for hospital in groups_outer_training_09
    ],
    dtype=int,
)

if (
    inner_fold_vector_09 == -1
).any():
    raise RuntimeError(
        "Bazı dış eğitim hastanelerine "
        "iç kat atanmadı."
    )

if set(
    np.unique(inner_fold_vector_09)
) != {1, 2, 3, 4, 5}:
    raise RuntimeError(
        "İç kat değerleri 1–5 değil."
    )

# ------------------------------------------------------------
# 5. Kilitli aday ızgarası
# ------------------------------------------------------------

candidate_grid_09 = [
    {
        "candidate_id": "LR01",
        "C": 0.03,
        "l1_ratio": 0.00,
    },
    {
        "candidate_id": "LR02",
        "C": 0.10,
        "l1_ratio": 0.00,
    },
    {
        "candidate_id": "LR03",
        "C": 0.30,
        "l1_ratio": 0.00,
    },
    {
        "candidate_id": "LR04",
        "C": 0.10,
        "l1_ratio": 0.25,
    },
    {
        "candidate_id": "LR05",
        "C": 0.30,
        "l1_ratio": 0.25,
    },
    {
        "candidate_id": "LR06",
        "C": 0.30,
        "l1_ratio": 0.50,
    },
]

# ------------------------------------------------------------
# 6. Preprocessing ve model üreticileri
# ------------------------------------------------------------

def make_preprocessor_09():

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
            (
                "scaler",
                StandardScaler(
                    with_mean=False,
                ),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns_07B,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns_07B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_model_09(
    C_value,
    l1_ratio_value,
):

    return LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=float(C_value),
        l1_ratio=float(l1_ratio_value),
        class_weight=None,
        max_iter=5000,
        tol=1e-4,
        random_state=MODEL_RANDOM_SEED_09,
    )


def checkpoint_table_id_09(
    inner_fold,
):

    return (
        f"{TARGET_DATASET}."
        f"model_lr_inner_oof_outer3_"
        f"inner{inner_fold}_v1"
    )

# ------------------------------------------------------------
# 7. Checkpoint doğrulama fonksiyonu
# ------------------------------------------------------------

def verify_checkpoint_09(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):

    table_id = checkpoint_table_id_09(
        inner_fold
    )

    try:
        client.get_table(table_id)

    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row)
        AS distinct_id_count,
      COUNT(DISTINCT candidate_id)
        AS candidate_count,
      COUNT(DISTINCT outer_fold)
        AS outer_fold_count,
      COUNT(DISTINCT inner_fold)
        AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(
          candidate_id,
          '|',
          id_row
        )
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1)
        AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0)
        AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL)
        AS missing_predictions,
      COUNTIF(
        prediction_raw < 0
        OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(
        sql,
        location=BQ_LOCATION,
    ).to_dataframe()

    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows
        * len(candidate_grid_09)
    )

    expected_positive_rows = (
        expected_validation_events
        * len(candidate_grid_09)
    )

    expected_negative_rows = (
        (
            expected_validation_rows
            - expected_validation_events
        )
        * len(candidate_grid_09)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": (
            expected_validation_rows
        ),
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": (
            expected_total_rows
        ),
        "positive_prediction_rows": (
            expected_positive_rows
        ),
        "negative_prediction_rows": (
            expected_negative_rows
        ),
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": 3,
        "maximum_outer_fold": 3,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failure_items = []

    for field, expected_value in (
        expected_values.items()
    ):
        actual_value = int(row[field])

        if actual_value != expected_value:
            complete = False

            failure_items.append(
                f"{field}={actual_value}, "
                f"expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failure_items),
        "check": check,
        "row": row,
    }

# ------------------------------------------------------------
# 8. BigQuery load-job şeması
# ------------------------------------------------------------

checkpoint_load_config_09 = (
    bigquery.LoadJobConfig(
        schema=[
            bigquery.SchemaField(
                "id_row",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "outer_fold",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "inner_fold",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "candidate_id",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "label_stage23",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_raw",
                "FLOAT",
                mode="REQUIRED",
            ),
        ],
        write_disposition=(
            bigquery.WriteDisposition.WRITE_TRUNCATE
        ),
    )
)

# ------------------------------------------------------------
# 9. Aggregate fit-audit dosyası
# ------------------------------------------------------------

fit_audit_columns_09 = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "C",
    "l1_ratio",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "convergence_warnings",
    "elapsed_seconds",
]

fit_audit_path_09 = os.path.join(
    MODEL_OUTPUT_DIR,
    "09A_logistic_inner_fit_audit_outer3.csv",
)

if os.path.exists(fit_audit_path_09):
    fit_audit_09 = pd.read_csv(
        fit_audit_path_09
    )
else:
    fit_audit_09 = pd.DataFrame(
        columns=fit_audit_columns_09
    )

for column in fit_audit_columns_09:
    if column not in fit_audit_09.columns:
        fit_audit_09[column] = np.nan

fit_audit_09 = fit_audit_09[
    fit_audit_columns_09
].copy()

# ------------------------------------------------------------
# 10. Beş iç katın aday modellerini çalıştır
# ------------------------------------------------------------

for inner_fold in range(1, 6):

    inner_training_mask = (
        inner_fold_vector_09 != inner_fold
    )

    inner_validation_mask = (
        inner_fold_vector_09 == inner_fold
    )

    training_rows = int(
        inner_training_mask.sum()
    )

    validation_rows = int(
        inner_validation_mask.sum()
    )

    training_events = int(
        y_outer_training_09[
            inner_training_mask
        ].sum()
    )

    validation_events = int(
        y_outer_training_09[
            inner_validation_mask
        ].sum()
    )

    existing_check = verify_checkpoint_09(
        inner_fold=inner_fold,
        expected_validation_rows=(
            validation_rows
        ),
        expected_validation_events=(
            validation_events
        ),
    )

    if existing_check["complete"]:

        print(
            f"Outer 3 / inner {inner_fold}: "
            "permanent checkpoint already complete; "
            "skipping model fitting."
        )

        continue

    training_hospital_set = set(
        groups_outer_training_09[
            inner_training_mask
        ]
    )

    validation_hospital_set = set(
        groups_outer_training_09[
            inner_validation_mask
        ]
    )

    if (
        training_hospital_set
        & validation_hospital_set
    ):
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "hastane çakışması bulundu."
        )

    print(
        f"\nOuter 3 / inner {inner_fold}"
    )

    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_09()

    preprocessing_started = time.time()

    X_inner_training_processed = (
        preprocessor.fit_transform(
            X_outer_training_09.loc[
                inner_training_mask
            ]
        )
    )

    X_inner_validation_processed = (
        preprocessor.transform(
            X_outer_training_09.loc[
                inner_validation_mask
            ]
        )
    )

    preprocessing_elapsed = (
        time.time()
        - preprocessing_started
    )

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "İşlenmiş eğitim ve doğrulama "
            "sütun sayıları farklı."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_elapsed, 2),
    )

    y_inner_training = (
        y_outer_training_09[
            inner_training_mask
        ]
    )

    y_inner_validation = (
        y_outer_training_09[
            inner_validation_mask
        ]
    )

    validation_ids = (
        outer_training_meta_09.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_09:

        candidate_id = candidate[
            "candidate_id"
        ]

        print(
            "  Fitting",
            candidate_id,
            "| C =",
            candidate["C"],
            "| l1_ratio =",
            candidate["l1_ratio"],
        )

        model = make_model_09(
            candidate["C"],
            candidate["l1_ratio"],
        )

        fitting_started = time.time()

        with warnings.catch_warnings(
            record=True
        ) as warning_records:

            warnings.simplefilter(
                "always",
                ConvergenceWarning,
            )

            model.fit(
                X_inner_training_processed,
                y_inner_training,
            )

        fitting_elapsed = (
            time.time()
            - fitting_started
        )

        convergence_warning_count = sum(
            issubclass(
                warning.category,
                ConvergenceWarning,
            )
            for warning in warning_records
        )

        validation_probabilities = (
            model.predict_proba(
                X_inner_validation_processed
            )[:, 1]
        )

        if np.isnan(
            validation_probabilities
        ).any():
            raise RuntimeError(
                f"{candidate_id}, inner "
                f"{inner_fold}: eksik tahmin."
            )

        if not np.all(
            (
                validation_probabilities >= 0
            )
            & (
                validation_probabilities <= 1
            )
        ):
            raise RuntimeError(
                f"{candidate_id}: geçersiz olasılık."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        3,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": (
                        candidate_id
                    ),
                    "label_stage23": (
                        y_inner_validation
                        .astype(np.int64)
                    ),
                    "prediction_raw": (
                        validation_probabilities
                        .astype(np.float64)
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": 3,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "C": candidate["C"],
                "l1_ratio": candidate[
                    "l1_ratio"
                ],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": (
                    validation_events
                ),
                "processed_columns": int(
                    X_inner_training_processed
                    .shape[1]
                ),
                "convergence_warnings": int(
                    convergence_warning_count
                ),
                "elapsed_seconds": float(
                    fitting_elapsed
                ),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows
        * len(candidate_grid_09)
    )

    if len(checkpoint_df) != (
        expected_checkpoint_rows
    ):
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "checkpoint satır sayısı hatalı."
        )

    if checkpoint_df.duplicated(
        subset=[
            "id_row",
            "candidate_id",
        ]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "yinelenen aday–hasta tahmini var."
        )

    target_checkpoint_table = (
        checkpoint_table_id_09(
            inner_fold
        )
    )

    print(
        "Uploading permanent checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_09,
        location=BQ_LOCATION,
    ).result()

    if len(fit_audit_09) > 0:

        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_09[
                        "outer_fold"
                    ],
                    errors="coerce",
                ) == 3
            )
            & (
                pd.to_numeric(
                    fit_audit_09[
                        "inner_fold"
                    ],
                    errors="coerce",
                ) == inner_fold
            )
        )

        fit_audit_09 = (
            fit_audit_09.loc[
                keep_mask
            ].copy()
        )

    fit_audit_09 = pd.concat(
        [
            fit_audit_09,
            pd.DataFrame(
                current_audit_rows
            ),
        ],
        ignore_index=True,
    )

    fit_audit_09 = (
        fit_audit_09[
            fit_audit_columns_09
        ]
        .sort_values(
            [
                "outer_fold",
                "inner_fold",
                "candidate_id",
            ]
        )
        .reset_index(drop=True)
    )

    fit_audit_09.to_csv(
        fit_audit_path_09,
        index=False,
    )

    completed_check = verify_checkpoint_09(
        inner_fold=inner_fold,
        expected_validation_rows=(
            validation_rows
        ),
        expected_validation_events=(
            validation_events
        ),
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint "
            "doğrulanamadı: "
            + completed_check["reason"]
        )

    print(
        f"Outer 3 / inner {inner_fold}: "
        "permanent checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )

    gc.collect()

# ------------------------------------------------------------
# 11. Beş checkpoint tablosunun son özeti
# ------------------------------------------------------------

checkpoint_summary_rows_09 = []

for inner_fold in range(1, 6):

    validation_mask = (
        inner_fold_vector_09 == inner_fold
    )

    validation_rows = int(
        validation_mask.sum()
    )

    validation_events = int(
        y_outer_training_09[
            validation_mask
        ].sum()
    )

    final_check = verify_checkpoint_09(
        inner_fold=inner_fold,
        expected_validation_rows=(
            validation_rows
        ),
        expected_validation_events=(
            validation_events
        ),
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "son checkpoint denetimi başarısız. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_09.append(
        {
            "outer_fold": 3,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(
                row["row_count"]
            ),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(
                row["candidate_count"]
            ),
            "positive_prediction_rows": int(
                row[
                    "positive_prediction_rows"
                ]
            ),
            "negative_prediction_rows": int(
                row[
                    "negative_prediction_rows"
                ]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check[
                "table_id"
            ],
        }
    )

checkpoint_summary_09 = (
    pd.DataFrame(
        checkpoint_summary_rows_09
    )
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_09[
        "distinct_validation_patients"
    ].sum()
) != EXPECTED_SPLIT_09[
    "training_rows"
]:
    raise RuntimeError(
        "Toplam doğrulama hasta sayısı "
        "46.755 değil."
    )

expected_total_oof_rows_09 = (
    EXPECTED_SPLIT_09[
        "training_rows"
    ]
    * len(candidate_grid_09)
)

if int(
    checkpoint_summary_09[
        "checkpoint_rows"
    ].sum()
) != expected_total_oof_rows_09:
    raise RuntimeError(
        "Toplam OOF tahmin satırı hatalı."
    )

checkpoint_summary_path_09 = os.path.join(
    MODEL_OUTPUT_DIR,
    "09A_outer3_inner_checkpoint_summary.csv",
)

checkpoint_summary_09.to_csv(
    checkpoint_summary_path_09,
    index=False,
)

# ------------------------------------------------------------
# 12. Beş OOF tablosunu UNION ALL ile RAM'e yükle
# ------------------------------------------------------------

checkpoint_tables_09 = [
    checkpoint_table_id_09(
        inner_fold
    )
    for inner_fold in range(1, 6)
]

union_parts_09 = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_09
]

SQL_LOAD_POOLED_OOF_09 = (
    "\nUNION ALL\n".join(
        union_parts_09
    )
)

print(
    "\nLoading pooled outer-fold-3 "
    "inner OOF predictions..."
)

query_job_09 = client.query(
    SQL_LOAD_POOLED_OOF_09,
    location=BQ_LOCATION,
)

try:
    pooled_oof_09 = (
        query_job_09.to_dataframe(
            create_bqstorage_client=True
        )
    )

    pooled_load_method_09 = (
        "BigQuery Storage API"
    )

except Exception as fast_path_error_09:

    print(
        "Storage API unavailable; using "
        "standard BigQuery download."
    )

    print(
        "Message:",
        type(fast_path_error_09).__name__,
    )

    pooled_oof_09 = (
        query_job_09.to_dataframe(
            create_bqstorage_client=False
        )
    )

    pooled_load_method_09 = (
        "Standard BigQuery API"
    )

pooled_oof_09[
    "id_row"
] = pooled_oof_09[
    "id_row"
].astype(str)

pooled_oof_09[
    "candidate_id"
] = pooled_oof_09[
    "candidate_id"
].astype(str)

for column in [
    "outer_fold",
    "inner_fold",
    "label_stage23",
]:
    pooled_oof_09[column] = pd.to_numeric(
        pooled_oof_09[column],
        errors="raise",
    ).astype(int)

pooled_oof_09[
    "prediction_raw"
] = pd.to_numeric(
    pooled_oof_09[
        "prediction_raw"
    ],
    errors="raise",
).astype(float)

# ------------------------------------------------------------
# 13. Pooled OOF bütünlük denetimleri
# ------------------------------------------------------------

if len(pooled_oof_09) != (
    expected_total_oof_rows_09
):
    raise RuntimeError(
        "Pooled OOF satır sayısı hatalı."
    )

if pooled_oof_09.duplicated(
    subset=[
        "candidate_id",
        "id_row",
    ]
).any():
    raise RuntimeError(
        "Pooled OOF içinde yinelenen "
        "aday–hasta tahmini bulundu."
    )

if pooled_oof_09[
    "prediction_raw"
].isna().any():
    raise RuntimeError(
        "Pooled OOF içinde eksik tahmin var."
    )

if not pooled_oof_09[
    "prediction_raw"
].between(0, 1).all():
    raise RuntimeError(
        "Pooled OOF içinde geçersiz "
        "olasılık değeri var."
    )

if set(
    pooled_oof_09[
        "candidate_id"
    ].unique()
) != {
    "LR01",
    "LR02",
    "LR03",
    "LR04",
    "LR05",
    "LR06",
}:
    raise RuntimeError(
        "Altı kilitli aday bulunmuyor."
    )

candidate_patient_counts_09 = (
    pooled_oof_09
    .groupby("candidate_id")[
        "id_row"
    ]
    .nunique()
)

if not (
    candidate_patient_counts_09
    == EXPECTED_SPLIT_09[
        "training_rows"
    ]
).all():
    raise RuntimeError(
        "Her aday için 46.755 farklı "
        "OOF hastası yok."
    )

candidate_event_counts_09 = (
    pooled_oof_09
    .groupby("candidate_id")[
        "label_stage23"
    ]
    .sum()
)

if not (
    candidate_event_counts_09
    == EXPECTED_SPLIT_09[
        "training_events"
    ]
).all():
    raise RuntimeError(
        "Her aday için 2.424 olay yok."
    )

# ------------------------------------------------------------
# 14. Metrik yardımcıları
# ------------------------------------------------------------

def probability_metrics_09(
    y_true,
    probabilities,
):

    probabilities = np.clip(
        np.asarray(
            probabilities,
            dtype=float,
        ),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1],
            )
        ),
        "mean_predicted_risk": float(
            probabilities.mean()
        ),
        "observed_event_rate": float(
            np.mean(y_true)
        ),
    }


def probability_logit_09(
    probabilities,
):

    probabilities = np.clip(
        np.asarray(
            probabilities,
            dtype=float,
        ),
        1e-6,
        1 - 1e-6,
    )

    return np.log(
        probabilities
        / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_09(
    y_true,
    probabilities,
):

    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )

    calibration_model.fit(
        probability_logit_09(
            probabilities
        ),
        y_true,
    )

    return (
        float(
            calibration_model.intercept_[0]
        ),
        float(
            calibration_model.coef_[0][0]
        ),
    )

# ------------------------------------------------------------
# 15. Altı adayın pooled OOF performansı
# ------------------------------------------------------------

candidate_result_rows_09 = []

for candidate in candidate_grid_09:

    candidate_id = candidate[
        "candidate_id"
    ]

    candidate_oof = (
        pooled_oof_09.loc[
            pooled_oof_09[
                "candidate_id"
            ] == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_09(
        candidate_oof[
            "label_stage23"
        ].to_numpy(dtype=int),
        candidate_oof[
            "prediction_raw"
        ].to_numpy(dtype=float),
    )

    fit_part = fit_audit_09.loc[
        fit_audit_09[
            "candidate_id"
        ].astype(str) == candidate_id
    ]

    convergence_warnings = (
        int(
            pd.to_numeric(
                fit_part[
                    "convergence_warnings"
                ],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    elapsed_seconds = (
        float(
            pd.to_numeric(
                fit_part[
                    "elapsed_seconds"
                ],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_09.append(
        {
            "candidate_id": candidate_id,
            "C": candidate["C"],
            "l1_ratio": candidate[
                "l1_ratio"
            ],
            **metrics,
            "convergence_warnings": (
                convergence_warnings
            ),
            "elapsed_seconds": (
                elapsed_seconds
            ),
        }
    )

candidate_results_09 = pd.DataFrame(
    candidate_result_rows_09
)

candidate_results_09 = (
    candidate_results_09
    .sort_values(
        [
            "auprc",
            "auroc",
            "brier",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

candidate_results_09[
    "selection_rank"
] = np.arange(
    1,
    len(candidate_results_09) + 1,
)

best_row_09 = candidate_results_09.iloc[0]

selected_candidate_09 = str(
    best_row_09["candidate_id"]
)

selected_C_09 = float(
    best_row_09["C"]
)

selected_l1_ratio_09 = float(
    best_row_09["l1_ratio"]
)

# ------------------------------------------------------------
# 16. Seçilen adayın Platt katsayıları
# ------------------------------------------------------------

selected_oof_09 = (
    pooled_oof_09.loc[
        pooled_oof_09[
            "candidate_id"
        ] == selected_candidate_09
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_09 = (
    selected_oof_09[
        "label_stage23"
    ].to_numpy(dtype=int)
)

selected_oof_probability_09 = (
    selected_oof_09[
        "prediction_raw"
    ].to_numpy(dtype=float)
)

platt_calibrator_09 = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_09.fit(
    probability_logit_09(
        selected_oof_probability_09
    ),
    selected_oof_y_09,
)

platt_intercept_09 = float(
    platt_calibrator_09.intercept_[0]
)

platt_slope_09 = float(
    platt_calibrator_09.coef_[0][0]
)

if (
    not np.isfinite(platt_intercept_09)
    or not np.isfinite(platt_slope_09)
    or platt_slope_09 <= 0
):
    raise RuntimeError(
        "Platt kalibrasyon katsayıları geçersiz."
    )

selected_model_09 = pd.DataFrame(
    [
        {
            "outer_fold": 3,
            "selected_candidate": (
                selected_candidate_09
            ),
            "selected_C": (
                selected_C_09
            ),
            "selected_l1_ratio": (
                selected_l1_ratio_09
            ),
            "selection_metric_primary": (
                "pooled_inner_oof_auprc"
            ),
            "inner_oof_auprc": float(
                best_row_09["auprc"]
            ),
            "inner_oof_auroc": float(
                best_row_09["auroc"]
            ),
            "inner_oof_brier": float(
                best_row_09["brier"]
            ),
            "inner_oof_log_loss": float(
                best_row_09["log_loss"]
            ),
            "inner_oof_mean_predicted_risk": float(
                best_row_09[
                    "mean_predicted_risk"
                ]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_09[
                    "observed_event_rate"
                ]
            ),
            "platt_intercept": (
                platt_intercept_09
            ),
            "platt_slope": (
                platt_slope_09
            ),
            "protocol_sha256": (
                EXPECTED_PROTOCOL_SHA_09
            ),
        }
    ]
)

# ------------------------------------------------------------
# 17. Seçim ve kalibrasyon dosyalarını kilitle
# ------------------------------------------------------------

candidate_results_path_09 = os.path.join(
    MODEL_OUTPUT_DIR,
    "09B_logistic_candidate_results_outer3.csv",
)

selected_model_path_09 = os.path.join(
    MODEL_OUTPUT_DIR,
    "09B_logistic_selected_model_outer3.csv",
)

selection_json_path_09 = os.path.join(
    MODEL_OUTPUT_DIR,
    "09B_logistic_selection_calibration_outer3.json",
)

selection_sha_path_09 = os.path.join(
    MODEL_OUTPUT_DIR,
    "09B_logistic_selection_calibration_outer3_SHA256.txt",
)

candidate_results_09.to_csv(
    candidate_results_path_09,
    index=False,
)

selected_model_09.to_csv(
    selected_model_path_09,
    index=False,
)

selection_configuration_09 = {
    "outer_fold": 3,
    "protocol_sha256": (
        EXPECTED_PROTOCOL_SHA_09
    ),
    "selection_metric_primary": (
        "pooled inner out-of-fold AUPRC"
    ),
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "selected_candidate": (
        selected_candidate_09
    ),
    "selected_C": selected_C_09,
    "selected_l1_ratio": (
        selected_l1_ratio_09
    ),
    "inner_oof_auprc": float(
        best_row_09["auprc"]
    ),
    "inner_oof_auroc": float(
        best_row_09["auroc"]
    ),
    "inner_oof_brier": float(
        best_row_09["brier"]
    ),
    "platt_intercept": (
        platt_intercept_09
    ),
    "platt_slope": (
        platt_slope_09
    ),
    "inner_checkpoint_tables": (
        checkpoint_tables_09
    ),
    "patient_level_oof_written_to_drive": False,
}

with open(
    selection_json_path_09,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        selection_configuration_09,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(
    selection_json_path_09,
    "rb",
) as file_handle:
    selection_sha_09 = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    selection_sha_path_09,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(
        selection_sha_09 + "\n"
    )

# ------------------------------------------------------------
# 18. Seçilen nihai dış kat 3 modelini fit et
# ------------------------------------------------------------

final_pipeline_09 = Pipeline(
    steps=[
        (
            "preprocessor",
            make_preprocessor_09(),
        ),
        (
            "model",
            make_model_09(
                selected_C_09,
                selected_l1_ratio_09,
            ),
        ),
    ]
)

print(
    "\nFitting selected outer-fold-3 model "
    "on all 46,755 training patients..."
)

final_fit_started_09 = time.time()

with warnings.catch_warnings(
    record=True
) as final_warning_records_09:

    warnings.simplefilter(
        "always",
        ConvergenceWarning,
    )

    final_pipeline_09.fit(
        X_outer_training_09,
        y_outer_training_09,
    )

final_fit_elapsed_09 = (
    time.time()
    - final_fit_started_09
)

final_convergence_warnings_09 = sum(
    issubclass(
        warning.category,
        ConvergenceWarning,
    )
    for warning in final_warning_records_09
)

if final_convergence_warnings_09 != 0:
    raise RuntimeError(
        "Nihai dış kat 3 modelinde "
        "yakınsama uyarısı oluştu."
    )

# ------------------------------------------------------------
# 19. Dış kat 3 ham ve kalibre tahminleri
# ------------------------------------------------------------

outer3_raw_probabilities_09 = (
    final_pipeline_09.predict_proba(
        X_outer_test_09
    )[:, 1]
)

raw_clipped_09 = np.clip(
    outer3_raw_probabilities_09,
    1e-6,
    1 - 1e-6,
)

raw_logit_09 = np.log(
    raw_clipped_09
    / (1 - raw_clipped_09)
)

outer3_platt_probabilities_09 = expit(
    platt_intercept_09
    + platt_slope_09
    * raw_logit_09
)

for probabilities, name in [
    (
        outer3_raw_probabilities_09,
        "raw",
    ),
    (
        outer3_platt_probabilities_09,
        "platt",
    ),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(
            f"{name} tahminlerinde eksik değer var."
        )

    if not np.all(
        (
            probabilities >= 0
        )
        & (
            probabilities <= 1
        )
    ):
        raise RuntimeError(
            f"{name} tahminlerinde geçersiz "
            "olasılık değeri var."
        )

raw_metrics_09 = probability_metrics_09(
    y_outer_test_09,
    outer3_raw_probabilities_09,
)

platt_metrics_09 = probability_metrics_09(
    y_outer_test_09,
    outer3_platt_probabilities_09,
)

raw_calibration_intercept_09, \
raw_calibration_slope_09 = (
    calibration_intercept_slope_09(
        y_outer_test_09,
        outer3_raw_probabilities_09,
    )
)

platt_calibration_intercept_09, \
platt_calibration_slope_09 = (
    calibration_intercept_slope_09(
        y_outer_test_09,
        outer3_platt_probabilities_09,
    )
)

outer3_test_results_09 = pd.DataFrame(
    [
        {
            "outer_fold": 3,
            "model": (
                "elastic_net_logistic"
            ),
            "probability_type": "raw",
            **raw_metrics_09,
            "calibration_intercept": (
                raw_calibration_intercept_09
            ),
            "calibration_slope": (
                raw_calibration_slope_09
            ),
        },
        {
            "outer_fold": 3,
            "model": (
                "elastic_net_logistic"
            ),
            "probability_type": (
                "platt_calibrated"
            ),
            **platt_metrics_09,
            "calibration_intercept": (
                platt_calibration_intercept_09
            ),
            "calibration_slope": (
                platt_calibration_slope_09
            ),
        },
    ]
)

# ------------------------------------------------------------
# 20. Feature katsayı tablosu
# ------------------------------------------------------------

fitted_preprocessor_09 = (
    final_pipeline_09
    .named_steps["preprocessor"]
)

fitted_model_09 = (
    final_pipeline_09
    .named_steps["model"]
)

processed_feature_names_09 = (
    fitted_preprocessor_09
    .get_feature_names_out()
)

model_coefficients_09 = (
    fitted_model_09
    .coef_
    .reshape(-1)
)

if len(
    processed_feature_names_09
) != len(model_coefficients_09):
    raise RuntimeError(
        "Feature ve katsayı sayıları uyuşmuyor."
    )

coefficient_table_09 = pd.DataFrame(
    {
        "processed_feature": (
            processed_feature_names_09
        ),
        "coefficient": (
            model_coefficients_09
        ),
    }
)

coefficient_table_09[
    "absolute_coefficient"
] = coefficient_table_09[
    "coefficient"
].abs()

coefficient_table_09[
    "is_nonzero"
] = ~np.isclose(
    coefficient_table_09[
        "coefficient"
    ],
    0.0,
    atol=1e-12,
)

coefficient_table_09[
    "absolute_rank"
] = (
    coefficient_table_09[
        "absolute_coefficient"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)

coefficient_table_09 = (
    coefficient_table_09
    .sort_values(
        [
            "absolute_coefficient",
            "processed_feature",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

nonzero_coefficients_09 = int(
    coefficient_table_09[
        "is_nonzero"
    ].sum()
)

final_model_summary_09 = pd.DataFrame(
    [
        {
            "outer_fold": 3,
            "selected_candidate": (
                selected_candidate_09
            ),
            "selected_C": selected_C_09,
            "selected_l1_ratio": (
                selected_l1_ratio_09
            ),
            "training_patients": len(
                X_outer_training_09
            ),
            "training_hospitals": len(
                training_hospitals_09
            ),
            "training_events": int(
                y_outer_training_09.sum()
            ),
            "test_patients": len(
                X_outer_test_09
            ),
            "test_hospitals": len(
                test_hospitals_09
            ),
            "test_events": int(
                y_outer_test_09.sum()
            ),
            "hospital_overlap": len(
                hospital_overlap_09
            ),
            "processed_feature_columns": len(
                processed_feature_names_09
            ),
            "nonzero_coefficients": (
                nonzero_coefficients_09
            ),
            "model_intercept": float(
                fitted_model_09.intercept_[0]
            ),
            "convergence_warnings": (
                final_convergence_warnings_09
            ),
            "fit_elapsed_seconds": float(
                final_fit_elapsed_09
            ),
            "locked_platt_intercept": (
                platt_intercept_09
            ),
            "locked_platt_slope": (
                platt_slope_09
            ),
            "protocol_sha256": (
                EXPECTED_PROTOCOL_SHA_09
            ),
            "selection_sha256": (
                selection_sha_09
            ),
        }
    ]
)

# ------------------------------------------------------------
# 21. Dış test tahminlerini BigQuery'ye yaz
# ------------------------------------------------------------

outer3_prediction_df_09 = pd.DataFrame(
    {
        "id_row": (
            outer_test_meta_09[
                "id_row"
            ].astype(str)
        ),
        "outer_fold": np.full(
            len(outer_test_meta_09),
            3,
            dtype=np.int64,
        ),
        "label_stage23": (
            y_outer_test_09
            .astype(np.int64)
        ),
        "prediction_raw": (
            outer3_raw_probabilities_09
            .astype(np.float64)
        ),
        "prediction_platt": (
            outer3_platt_probabilities_09
            .astype(np.float64)
        ),
        "model_name": (
            "elastic_net_logistic"
        ),
        "model_version": (
            "core_v1_nested_cv"
        ),
    }
)

if len(outer3_prediction_df_09) != 11736:
    raise RuntimeError(
        "Dış kat 3 tahmin satır sayısı "
        "11.736 değil."
    )

if outer3_prediction_df_09[
    "id_row"
].duplicated().any():
    raise RuntimeError(
        "Dış kat 3 tahminlerinde "
        "yinelenen id_row var."
    )

if int(
    outer3_prediction_df_09[
        "label_stage23"
    ].sum()
) != 608:
    raise RuntimeError(
        "Dış kat 3 olay sayısı 608 değil."
    )

prediction_table_id_09 = (
    f"{TARGET_DATASET}."
    "model_lr_outer_predictions_outer3_v1"
)

prediction_load_config_09 = (
    bigquery.LoadJobConfig(
        schema=[
            bigquery.SchemaField(
                "id_row",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "outer_fold",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "label_stage23",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_raw",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_platt",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_name",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_version",
                "STRING",
                mode="REQUIRED",
            ),
        ],
        write_disposition=(
            bigquery.WriteDisposition.WRITE_TRUNCATE
        ),
    )
)

print(
    "\nUploading secure outer-fold-3 "
    "prediction checkpoint:"
)

print(prediction_table_id_09)

client.load_table_from_dataframe(
    outer3_prediction_df_09,
    prediction_table_id_09,
    job_config=prediction_load_config_09,
    location=BQ_LOCATION,
).result()

# ------------------------------------------------------------
# 22. BigQuery dış test checkpoint doğrulaması
# ------------------------------------------------------------

SQL_VERIFY_PREDICTIONS_09 = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL)
    AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL)
    AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0
    OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0
    OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw)
    AS minimum_raw_probability,
  MAX(prediction_raw)
    AS maximum_raw_probability,
  MIN(prediction_platt)
    AS minimum_platt_probability,
  MAX(prediction_platt)
    AS maximum_platt_probability
FROM `{prediction_table_id_09}`;
"""

prediction_verification_09 = (
    client.query(
        SQL_VERIFY_PREDICTIONS_09,
        location=BQ_LOCATION,
    )
    .to_dataframe()
)

verification_row_09 = (
    prediction_verification_09.iloc[0]
)

expected_prediction_values_09 = {
    "prediction_rows": 11736,
    "distinct_rows": 11736,
    "outer_folds": 1,
    "minimum_outer_fold": 3,
    "maximum_outer_fold": 3,
    "events": 608,
    "nonevents": 11128,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in (
    expected_prediction_values_09.items()
):
    actual_value = int(
        verification_row_09[field]
    )

    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: bulunan={actual_value}, "
            f"beklenen={expected_value}"
        )

# ------------------------------------------------------------
# 23. Aggregate ve feature-level çıktıları kaydet
# ------------------------------------------------------------

test_results_path_09 = os.path.join(
    MODEL_OUTPUT_DIR,
    "09C_logistic_outer3_test_results.csv",
)

model_summary_path_09 = os.path.join(
    MODEL_OUTPUT_DIR,
    "09C_logistic_final_model_outer3.csv",
)

coefficient_path_09 = os.path.join(
    MODEL_OUTPUT_DIR,
    "09C_logistic_coefficients_outer3.csv",
)

evaluation_json_path_09 = os.path.join(
    MODEL_OUTPUT_DIR,
    "09C_logistic_final_evaluation_outer3.json",
)

evaluation_sha_path_09 = os.path.join(
    MODEL_OUTPUT_DIR,
    "09C_logistic_final_evaluation_outer3_SHA256.txt",
)

outer3_test_results_09.to_csv(
    test_results_path_09,
    index=False,
)

final_model_summary_09.to_csv(
    model_summary_path_09,
    index=False,
)

coefficient_table_09.to_csv(
    coefficient_path_09,
    index=False,
)

evaluation_configuration_09 = {
    "outer_fold": 3,
    "model_family": (
        "elastic_net_logistic_regression"
    ),
    "selected_candidate": (
        selected_candidate_09
    ),
    "selected_C": selected_C_09,
    "selected_l1_ratio": (
        selected_l1_ratio_09
    ),
    "training_patients": 46755,
    "training_hospitals": 158,
    "test_patients": 11736,
    "test_hospitals": 40,
    "hospital_overlap": 0,
    "locked_platt_intercept": (
        platt_intercept_09
    ),
    "locked_platt_slope": (
        platt_slope_09
    ),
    "processed_feature_columns": int(
        len(processed_feature_names_09)
    ),
    "nonzero_coefficients": int(
        nonzero_coefficients_09
    ),
    "protocol_sha256": (
        EXPECTED_PROTOCOL_SHA_09
    ),
    "selection_sha256": (
        selection_sha_09
    ),
    "secure_prediction_table": (
        prediction_table_id_09
    ),
    "patient_level_prediction_written_to_drive": (
        False
    ),
}

with open(
    evaluation_json_path_09,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        evaluation_configuration_09,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(
    evaluation_json_path_09,
    "rb",
) as file_handle:
    evaluation_sha_09 = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    evaluation_sha_path_09,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(
        evaluation_sha_09 + "\n"
    )

# ------------------------------------------------------------
# 24. Sonuçları göster
# ------------------------------------------------------------

pooled_integrity_09 = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_09),
            pooled_oof_09[
                "id_row"
            ].nunique(),
            pooled_oof_09[
                "candidate_id"
            ].nunique(),
            pooled_oof_09[
                "inner_fold"
            ].nunique(),
            EXPECTED_SPLIT_09[
                "training_events"
            ],
            (
                EXPECTED_SPLIT_09[
                    "training_rows"
                ]
                - EXPECTED_SPLIT_09[
                    "training_events"
                ]
            ),
            int(
                pooled_oof_09.duplicated(
                    subset=[
                        "candidate_id",
                        "id_row",
                    ]
                ).sum()
            ),
            int(
                pooled_oof_09[
                    "prediction_raw"
                ].isna().sum()
            ),
            int(
                (
                    ~pooled_oof_09[
                        "prediction_raw"
                    ].between(0, 1)
                ).sum()
            ),
            pooled_load_method_09,
        ],
    }
)

print(
    "\n09 OUTER-FOLD-3 INNER CHECKPOINT SUMMARY"
)
display(checkpoint_summary_09)

print(
    "\n09 OUTER-FOLD-3 POOLED OOF INTEGRITY"
)
display(pooled_integrity_09)

print(
    "\n09 OUTER-FOLD-3 CANDIDATE RESULTS"
)
display(candidate_results_09)

print(
    "\n09 OUTER-FOLD-3 SELECTED MODEL"
)
display(selected_model_09)

print(
    "\n09 OUTER-FOLD-3 FINAL MODEL SUMMARY"
)
display(final_model_summary_09)

print(
    "\n09 OUTER-FOLD-3 TEST RESULTS"
)
display(outer3_test_results_09)

print(
    "\n09 OUTER-FOLD-3 BIGQUERY VERIFICATION"
)
display(prediction_verification_09)

print(
    "\n09 OUTER-FOLD-3 TOP 20 ABSOLUTE COEFFICIENTS"
)
display(
    coefficient_table_09.head(20)
)

print("\nSelection SHA-256:")
print(selection_sha_09)

print("\nEvaluation SHA-256:")
print(evaluation_sha_09)

print(
    "\n09 PASS: Outer-fold-3 nested modelling "
    "and locked test evaluation are complete."
)

print(
    "All inner OOF and outer-test patient-level "
    "predictions were stored only in BigQuery."
)

print(
    "No patient-level prediction file was "
    "written to Google Drive."
)

_ = gc.collect()

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from sklearn.exceptions import ConvergenceWarning

from IPython.display import display

# ============================================================
# 10 — OUTER FOLD 4 COMPLETE NESTED MODELLING
#
# Resume-safe:
# - Each inner fold is stored in a separate BigQuery table.
# - Completed inner folds are automatically skipped.
#
# BigQuery free-tier compatible:
# - No DELETE / INSERT / UPDATE / MERGE is used.
#
# Privacy:
# - Patient-level predictions are stored only in BigQuery.
# - No patient-level prediction file is written to Drive.
# ============================================================

print("STARTING OUTER FOLD 4 — CODE VERSION 10")

OUTER_FOLD_10 = 4
MODEL_RANDOM_SEED_10 = 20260721

EXPECTED_PROTOCOL_SHA_10 = (
    "400c3b4b510c836794543bc685c62fae"
    "f49df0c1caa197d78dffda8c2207952d"
)

EXPECTED_SPLIT_10 = {
    "training_rows": 46803,
    "test_rows": 11688,
    "training_hospitals": 159,
    "test_hospitals": 39,
    "training_events": 2426,
    "test_events": 606,
}

EXPECTED_INNER_10 = {
    1: {"validation_rows": 10116, "validation_events": 554},
    2: {"validation_rows": 8161, "validation_events": 381},
    3: {"validation_rows": 6957, "validation_events": 358},
    4: {"validation_rows": 12682, "validation_events": 663},
    5: {"validation_rows": 8887, "validation_events": 470},
}

# ------------------------------------------------------------
# 1. Required runtime objects
# ------------------------------------------------------------

required_objects_10 = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_10 = [
    name for name in required_objects_10 if name not in globals()
]

if missing_objects_10:
    raise RuntimeError(
        "Eksik RAM nesneleri var: "
        + ", ".join(missing_objects_10)
        + ". Önce 07A ve 07B hücrelerini çalıştır."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"58.491 satır bekleniyordu; {len(core_df_07B)} bulundu."
    )

if len(predictor_columns_07B) != 159:
    raise RuntimeError("Core predictor sayısı 159 değil.")

if len(numeric_columns_07B) != 156:
    raise RuntimeError("Sayısal predictor sayısı 156 değil.")

if len(categorical_columns_07B) != 3:
    raise RuntimeError("Kategorik predictor sayısı 3 değil.")

# ------------------------------------------------------------
# 2. Locked protocol SHA check
# ------------------------------------------------------------

protocol_sha_path_10 = os.path.join(
    MODEL_OUTPUT_DIR,
    "08B_locked_logistic_model_protocol_v1_SHA256.txt",
)

if not os.path.exists(protocol_sha_path_10):
    raise FileNotFoundError(
        "Model protokolü SHA dosyası bulunamadı: "
        + protocol_sha_path_10
    )

with open(protocol_sha_path_10, "r", encoding="utf-8") as file_handle:
    observed_protocol_sha_10 = file_handle.read().strip()

if observed_protocol_sha_10 != EXPECTED_PROTOCOL_SHA_10:
    raise RuntimeError(
        "Kilitli model protokolü SHA değeri değişmiş: "
        + observed_protocol_sha_10
    )

# ------------------------------------------------------------
# 3. Locked inner-hospital mapping
# ------------------------------------------------------------

inner_mapping_path_10 = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_10):
    raise FileNotFoundError(
        "Kilitli iç kat haritası bulunamadı: "
        + inner_mapping_path_10
    )

inner_mapping_all_10 = pd.read_csv(
    inner_mapping_path_10,
    dtype={"group_hospital": str},
)

inner_mapping_part_10 = (
    inner_mapping_all_10.loc[
        inner_mapping_all_10["outer_fold"].astype(int) == OUTER_FOLD_10,
        ["group_hospital", "inner_fold"],
    ]
    .copy()
)

inner_mapping_part_10["group_hospital"] = (
    inner_mapping_part_10["group_hospital"].astype(str)
)
inner_mapping_part_10["inner_fold"] = (
    inner_mapping_part_10["inner_fold"].astype(int)
)

if len(inner_mapping_part_10) != 159:
    raise RuntimeError(
        "Dış kat 4 eğitim kümesi için 159 hastane ataması bekleniyordu."
    )

if inner_mapping_part_10["group_hospital"].duplicated().any():
    raise RuntimeError("İç kat haritasında yinelenen hastane var.")

hospital_to_inner_fold_10 = dict(
    zip(
        inner_mapping_part_10["group_hospital"],
        inner_mapping_part_10["inner_fold"],
    )
)

# ------------------------------------------------------------
# 4. Prepare model matrices
# ------------------------------------------------------------

X_all_10 = core_df_07B[predictor_columns_07B].copy()

for column in numeric_columns_07B:
    X_all_10[column] = pd.to_numeric(
        X_all_10[column], errors="coerce"
    ).astype("float64")

for column in categorical_columns_07B:
    category_series = X_all_10[column].astype("object")
    X_all_10[column] = category_series.where(
        pd.notna(category_series), np.nan
    )

outer_fold_vector_10 = (
    core_df_07B["outer_fold"].astype(int).to_numpy()
)

outer_training_mask_10 = outer_fold_vector_10 != OUTER_FOLD_10
outer_test_mask_10 = outer_fold_vector_10 == OUTER_FOLD_10

X_outer_training_10 = (
    X_all_10.loc[outer_training_mask_10].reset_index(drop=True)
)
X_outer_test_10 = (
    X_all_10.loc[outer_test_mask_10].reset_index(drop=True)
)

outer_training_meta_10 = (
    core_df_07B.loc[
        outer_training_mask_10,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_10 = (
    core_df_07B.loc[
        outer_test_mask_10,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [outer_training_meta_10, outer_test_meta_10]:
    dataframe["id_row"] = dataframe["id_row"].astype(str)
    dataframe["group_hospital"] = dataframe["group_hospital"].astype(str)
    dataframe["label_stage23"] = dataframe["label_stage23"].astype(int)

y_outer_training_10 = (
    outer_training_meta_10["label_stage23"].to_numpy(dtype=np.int8)
)
y_outer_test_10 = (
    outer_test_meta_10["label_stage23"].to_numpy(dtype=np.int8)
)

groups_outer_training_10 = (
    outer_training_meta_10["group_hospital"].to_numpy(dtype=str)
)

training_hospitals_10 = set(
    outer_training_meta_10["group_hospital"]
)
test_hospitals_10 = set(
    outer_test_meta_10["group_hospital"]
)
hospital_overlap_10 = training_hospitals_10 & test_hospitals_10

if hospital_overlap_10:
    raise RuntimeError("Dış eğitim ve test hastaneleri çakışıyor.")

actual_split_10 = {
    "training_rows": len(X_outer_training_10),
    "test_rows": len(X_outer_test_10),
    "training_hospitals": len(training_hospitals_10),
    "test_hospitals": len(test_hospitals_10),
    "training_events": int(y_outer_training_10.sum()),
    "test_events": int(y_outer_test_10.sum()),
}

for metric, expected_value in EXPECTED_SPLIT_10.items():
    actual_value = actual_split_10[metric]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: bulunan={actual_value}, beklenen={expected_value}"
        )

inner_fold_vector_10 = np.array(
    [
        hospital_to_inner_fold_10.get(hospital, -1)
        for hospital in groups_outer_training_10
    ],
    dtype=int,
)

if (inner_fold_vector_10 == -1).any():
    raise RuntimeError(
        "Bazı dış eğitim hastanelerine iç kat atanmadı."
    )

if set(np.unique(inner_fold_vector_10)) != {1, 2, 3, 4, 5}:
    raise RuntimeError("İç kat değerleri 1–5 değil.")

for inner_fold, expected in EXPECTED_INNER_10.items():
    validation_mask = inner_fold_vector_10 == inner_fold
    observed_rows = int(validation_mask.sum())
    observed_events = int(y_outer_training_10[validation_mask].sum())

    if observed_rows != expected["validation_rows"]:
        raise RuntimeError(
            f"Inner {inner_fold} validation_rows: "
            f"bulunan={observed_rows}, "
            f"beklenen={expected['validation_rows']}"
        )

    if observed_events != expected["validation_events"]:
        raise RuntimeError(
            f"Inner {inner_fold} validation_events: "
            f"bulunan={observed_events}, "
            f"beklenen={expected['validation_events']}"
        )

# ------------------------------------------------------------
# 5. Locked candidate grid
# ------------------------------------------------------------

candidate_grid_10 = [
    {"candidate_id": "LR01", "C": 0.03, "l1_ratio": 0.00},
    {"candidate_id": "LR02", "C": 0.10, "l1_ratio": 0.00},
    {"candidate_id": "LR03", "C": 0.30, "l1_ratio": 0.00},
    {"candidate_id": "LR04", "C": 0.10, "l1_ratio": 0.25},
    {"candidate_id": "LR05", "C": 0.30, "l1_ratio": 0.25},
    {"candidate_id": "LR06", "C": 0.30, "l1_ratio": 0.50},
]

# ------------------------------------------------------------
# 6. Preprocessing and model factories
# ------------------------------------------------------------

def make_preprocessor_10():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
            (
                "scaler",
                StandardScaler(with_mean=False),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns_07B,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns_07B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_model_10(C_value, l1_ratio_value):
    return LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=float(C_value),
        l1_ratio=float(l1_ratio_value),
        class_weight=None,
        max_iter=5000,
        tol=1e-4,
        random_state=MODEL_RANDOM_SEED_10,
    )


def checkpoint_table_id_10(inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_lr_inner_oof_outer4_inner{inner_fold}_v1"
    )

# ------------------------------------------------------------
# 7. BigQuery checkpoint verification
# ------------------------------------------------------------

def verify_checkpoint_10(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):
    table_id = checkpoint_table_id_10(inner_fold)

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS distinct_id_count,
      COUNT(DISTINCT candidate_id) AS candidate_count,
      COUNT(DISTINCT outer_fold) AS outer_fold_count,
      COUNT(DISTINCT inner_fold) AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(candidate_id, '|', id_row)
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(
        prediction_raw < 0 OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(sql, location=BQ_LOCATION).to_dataframe()
    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows * len(candidate_grid_10)
    )
    expected_positive_rows = (
        expected_validation_events * len(candidate_grid_10)
    )
    expected_negative_rows = (
        (expected_validation_rows - expected_validation_events)
        * len(candidate_grid_10)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": expected_validation_rows,
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": expected_total_rows,
        "positive_prediction_rows": expected_positive_rows,
        "negative_prediction_rows": expected_negative_rows,
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": 4,
        "maximum_outer_fold": 4,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failure_items = []

    for field, expected_value in expected_values.items():
        actual_value = int(row[field])
        if actual_value != expected_value:
            complete = False
            failure_items.append(
                f"{field}={actual_value}, expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failure_items),
        "check": check,
        "row": row,
    }

# ------------------------------------------------------------
# 8. BigQuery load schema
# ------------------------------------------------------------

checkpoint_load_config_10 = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("inner_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("candidate_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

# ------------------------------------------------------------
# 9. Aggregate fit-audit file
# ------------------------------------------------------------

fit_audit_columns_10 = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "C",
    "l1_ratio",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "convergence_warnings",
    "elapsed_seconds",
]

fit_audit_path_10 = os.path.join(
    MODEL_OUTPUT_DIR,
    "10A_logistic_inner_fit_audit_outer4.csv",
)

if os.path.exists(fit_audit_path_10):
    fit_audit_10 = pd.read_csv(fit_audit_path_10)
else:
    fit_audit_10 = pd.DataFrame(columns=fit_audit_columns_10)

for column in fit_audit_columns_10:
    if column not in fit_audit_10.columns:
        fit_audit_10[column] = np.nan

fit_audit_10 = fit_audit_10[fit_audit_columns_10].copy()

# ------------------------------------------------------------
# 10. Train six candidates in five locked inner folds
# ------------------------------------------------------------

for inner_fold in range(1, 6):
    inner_training_mask = inner_fold_vector_10 != inner_fold
    inner_validation_mask = inner_fold_vector_10 == inner_fold

    training_rows = int(inner_training_mask.sum())
    validation_rows = int(inner_validation_mask.sum())
    training_events = int(
        y_outer_training_10[inner_training_mask].sum()
    )
    validation_events = int(
        y_outer_training_10[inner_validation_mask].sum()
    )

    expected_inner = EXPECTED_INNER_10[inner_fold]
    expected_training_rows = (
        EXPECTED_SPLIT_10["training_rows"]
        - expected_inner["validation_rows"]
    )
    expected_training_events = (
        EXPECTED_SPLIT_10["training_events"]
        - expected_inner["validation_events"]
    )

    if training_rows != expected_training_rows:
        raise RuntimeError(
            f"Inner {inner_fold} training_rows: "
            f"bulunan={training_rows}, "
            f"beklenen={expected_training_rows}"
        )

    if training_events != expected_training_events:
        raise RuntimeError(
            f"Inner {inner_fold} training_events: "
            f"bulunan={training_events}, "
            f"beklenen={expected_training_events}"
        )

    existing_check = verify_checkpoint_10(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if existing_check["complete"]:
        print(
            f"Outer 4 / inner {inner_fold}: "
            "permanent checkpoint already complete; "
            "skipping model fitting."
        )
        continue

    training_hospital_set = set(
        groups_outer_training_10[inner_training_mask]
    )
    validation_hospital_set = set(
        groups_outer_training_10[inner_validation_mask]
    )

    if training_hospital_set & validation_hospital_set:
        raise RuntimeError(
            f"Inner fold {inner_fold}: hastane çakışması bulundu."
        )

    print(f"\nOuter 4 / inner {inner_fold}")
    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_10()
    preprocessing_started = time.time()

    X_inner_training_processed = preprocessor.fit_transform(
        X_outer_training_10.loc[inner_training_mask]
    )
    X_inner_validation_processed = preprocessor.transform(
        X_outer_training_10.loc[inner_validation_mask]
    )

    preprocessing_elapsed = time.time() - preprocessing_started

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "İşlenmiş eğitim ve doğrulama sütun sayıları farklı."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_elapsed, 2),
    )

    y_inner_training = y_outer_training_10[inner_training_mask]
    y_inner_validation = y_outer_training_10[inner_validation_mask]

    validation_ids = (
        outer_training_meta_10.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_10:
        candidate_id = candidate["candidate_id"]

        print(
            "  Fitting",
            candidate_id,
            "| C =",
            candidate["C"],
            "| l1_ratio =",
            candidate["l1_ratio"],
        )

        model = make_model_10(
            candidate["C"],
            candidate["l1_ratio"],
        )

        fitting_started = time.time()

        with warnings.catch_warnings(record=True) as warning_records:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(
                X_inner_training_processed,
                y_inner_training,
            )

        fitting_elapsed = time.time() - fitting_started

        convergence_warning_count = sum(
            issubclass(warning.category, ConvergenceWarning)
            for warning in warning_records
        )

        validation_probabilities = model.predict_proba(
            X_inner_validation_processed
        )[:, 1]

        if np.isnan(validation_probabilities).any():
            raise RuntimeError(
                f"{candidate_id}, inner {inner_fold}: eksik tahmin."
            )

        if not np.all(
            (validation_probabilities >= 0)
            & (validation_probabilities <= 1)
        ):
            raise RuntimeError(
                f"{candidate_id}: geçersiz olasılık."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        4,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": y_inner_validation.astype(np.int64),
                    "prediction_raw": validation_probabilities.astype(
                        np.float64
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": 4,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "C": candidate["C"],
                "l1_ratio": candidate["l1_ratio"],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": validation_events,
                "processed_columns": int(
                    X_inner_training_processed.shape[1]
                ),
                "convergence_warnings": int(
                    convergence_warning_count
                ),
                "elapsed_seconds": float(fitting_elapsed),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows * len(candidate_grid_10)
    )

    if len(checkpoint_df) != expected_checkpoint_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: checkpoint satır sayısı hatalı."
        )

    if checkpoint_df.duplicated(
        subset=["id_row", "candidate_id"]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "yinelenen aday–hasta tahmini var."
        )

    target_checkpoint_table = checkpoint_table_id_10(inner_fold)

    print(
        "Uploading permanent checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_10,
        location=BQ_LOCATION,
    ).result()

    new_audit_df = pd.DataFrame(current_audit_rows)

    if len(fit_audit_10) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_10["outer_fold"],
                    errors="coerce",
                )
                == 4
            )
            & (
                pd.to_numeric(
                    fit_audit_10["inner_fold"],
                    errors="coerce",
                )
                == inner_fold
            )
        )
        fit_audit_10 = fit_audit_10.loc[keep_mask].copy()

    if fit_audit_10.empty:
        fit_audit_10 = new_audit_df.copy()
    else:
        fit_audit_10 = pd.concat(
            [fit_audit_10, new_audit_df],
            ignore_index=True,
        )

    fit_audit_10 = (
        fit_audit_10[fit_audit_columns_10]
        .sort_values(
            ["outer_fold", "inner_fold", "candidate_id"]
        )
        .reset_index(drop=True)
    )

    fit_audit_10.to_csv(
        fit_audit_path_10,
        index=False,
    )

    completed_check = verify_checkpoint_10(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint doğrulanamadı: "
            + completed_check["reason"]
        )

    print(
        f"Outer 4 / inner {inner_fold}: "
        "permanent checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 11. Final checkpoint summary
# ------------------------------------------------------------

checkpoint_summary_rows_10 = []

for inner_fold in range(1, 6):
    validation_mask = inner_fold_vector_10 == inner_fold
    validation_rows = int(validation_mask.sum())
    validation_events = int(
        y_outer_training_10[validation_mask].sum()
    )

    final_check = verify_checkpoint_10(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "son checkpoint denetimi başarısız. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_10.append(
        {
            "outer_fold": 4,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(row["row_count"]),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(row["candidate_count"]),
            "positive_prediction_rows": int(
                row["positive_prediction_rows"]
            ),
            "negative_prediction_rows": int(
                row["negative_prediction_rows"]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check["table_id"],
        }
    )

checkpoint_summary_10 = (
    pd.DataFrame(checkpoint_summary_rows_10)
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_10["distinct_validation_patients"].sum()
) != EXPECTED_SPLIT_10["training_rows"]:
    raise RuntimeError(
        "Toplam doğrulama hasta sayısı 46.803 değil."
    )

expected_total_oof_rows_10 = (
    EXPECTED_SPLIT_10["training_rows"]
    * len(candidate_grid_10)
)

if int(checkpoint_summary_10["checkpoint_rows"].sum()) != (
    expected_total_oof_rows_10
):
    raise RuntimeError("Toplam OOF tahmin satırı hatalı.")

checkpoint_summary_path_10 = os.path.join(
    MODEL_OUTPUT_DIR,
    "10A_outer4_inner_checkpoint_summary.csv",
)
checkpoint_summary_10.to_csv(
    checkpoint_summary_path_10,
    index=False,
)

# ------------------------------------------------------------
# 12. Pool all inner OOF predictions
# ------------------------------------------------------------

checkpoint_tables_10 = [
    checkpoint_table_id_10(inner_fold)
    for inner_fold in range(1, 6)
]

union_parts_10 = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_10
]

SQL_LOAD_POOLED_OOF_10 = "\nUNION ALL\n".join(union_parts_10)

print("\nLoading pooled outer-fold-4 inner OOF predictions...")

query_job_10 = client.query(
    SQL_LOAD_POOLED_OOF_10,
    location=BQ_LOCATION,
)

try:
    pooled_oof_10 = query_job_10.to_dataframe(
        create_bqstorage_client=True
    )
    pooled_load_method_10 = "BigQuery Storage API"
except Exception as fast_path_error_10:
    print(
        "Storage API unavailable; using standard BigQuery download."
    )
    print("Message:", type(fast_path_error_10).__name__)
    pooled_oof_10 = query_job_10.to_dataframe(
        create_bqstorage_client=False
    )
    pooled_load_method_10 = "Standard BigQuery API"

pooled_oof_10["id_row"] = pooled_oof_10["id_row"].astype(str)
pooled_oof_10["candidate_id"] = pooled_oof_10["candidate_id"].astype(str)

for column in ["outer_fold", "inner_fold", "label_stage23"]:
    pooled_oof_10[column] = pd.to_numeric(
        pooled_oof_10[column], errors="raise"
    ).astype(int)

pooled_oof_10["prediction_raw"] = pd.to_numeric(
    pooled_oof_10["prediction_raw"], errors="raise"
).astype(float)

# ------------------------------------------------------------
# 13. Pooled OOF integrity checks
# ------------------------------------------------------------

if len(pooled_oof_10) != expected_total_oof_rows_10:
    raise RuntimeError("Pooled OOF satır sayısı hatalı.")

if set(pooled_oof_10["outer_fold"].unique()) != {4}:
    raise RuntimeError("Pooled OOF içinde dış kat 4 dışında kayıt var.")

if set(pooled_oof_10["inner_fold"].unique()) != {1, 2, 3, 4, 5}:
    raise RuntimeError("Pooled OOF iç katları 1–5 değil.")

if pooled_oof_10.duplicated(
    subset=["candidate_id", "id_row"]
).any():
    raise RuntimeError(
        "Pooled OOF içinde yinelenen aday–hasta tahmini bulundu."
    )

if pooled_oof_10["prediction_raw"].isna().any():
    raise RuntimeError("Pooled OOF içinde eksik tahmin var.")

if not pooled_oof_10["prediction_raw"].between(0, 1).all():
    raise RuntimeError(
        "Pooled OOF içinde geçersiz olasılık değeri var."
    )

expected_candidate_ids_10 = {
    "LR01",
    "LR02",
    "LR03",
    "LR04",
    "LR05",
    "LR06",
}

if set(pooled_oof_10["candidate_id"].unique()) != (
    expected_candidate_ids_10
):
    raise RuntimeError("Altı kilitli aday bulunmuyor.")

candidate_patient_counts_10 = (
    pooled_oof_10.groupby("candidate_id")["id_row"].nunique()
)

if not (
    candidate_patient_counts_10
    == EXPECTED_SPLIT_10["training_rows"]
).all():
    raise RuntimeError(
        "Her aday için 46.803 farklı OOF hastası yok."
    )

candidate_event_counts_10 = (
    pooled_oof_10.groupby("candidate_id")["label_stage23"].sum()
)

if not (
    candidate_event_counts_10
    == EXPECTED_SPLIT_10["training_events"]
).all():
    raise RuntimeError("Her aday için 2.426 olay yok.")

patient_label_consistency_10 = (
    pooled_oof_10.groupby("id_row")["label_stage23"].nunique()
)
if (patient_label_consistency_10 > 1).any():
    raise RuntimeError(
        "Aynı hastanın adaylar arasında outcome etiketi farklı."
    )

patient_inner_fold_consistency_10 = (
    pooled_oof_10.groupby("id_row")["inner_fold"].nunique()
)
if (patient_inner_fold_consistency_10 > 1).any():
    raise RuntimeError(
        "Aynı hasta birden fazla iç doğrulama katında bulundu."
    )

# ------------------------------------------------------------
# 14. Metric helpers
# ------------------------------------------------------------

def probability_metrics_10(y_true, probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(roc_auc_score(y_true, probabilities)),
        "auprc": float(average_precision_score(y_true, probabilities)),
        "brier": float(brier_score_loss(y_true, probabilities)),
        "log_loss": float(
            log_loss(y_true, probabilities, labels=[0, 1])
        ),
        "mean_predicted_risk": float(probabilities.mean()),
        "observed_event_rate": float(np.mean(y_true)),
    }


def probability_logit_10(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        probabilities / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_10(y_true, probabilities):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        probability_logit_10(probabilities),
        y_true,
    )
    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 15. Candidate pooled-OOF performance
# ------------------------------------------------------------

candidate_result_rows_10 = []

for candidate in candidate_grid_10:
    candidate_id = candidate["candidate_id"]

    candidate_oof = (
        pooled_oof_10.loc[
            pooled_oof_10["candidate_id"] == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_10(
        candidate_oof["label_stage23"].to_numpy(dtype=int),
        candidate_oof["prediction_raw"].to_numpy(dtype=float),
    )

    fit_part = fit_audit_10.loc[
        (
            pd.to_numeric(
                fit_audit_10["outer_fold"],
                errors="coerce",
            )
            == 4
        )
        & (
            fit_audit_10["candidate_id"].astype(str)
            == candidate_id
        )
    ]

    convergence_warnings = (
        int(
            pd.to_numeric(
                fit_part["convergence_warnings"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    elapsed_seconds = (
        float(
            pd.to_numeric(
                fit_part["elapsed_seconds"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_10.append(
        {
            "candidate_id": candidate_id,
            "C": candidate["C"],
            "l1_ratio": candidate["l1_ratio"],
            **metrics,
            "convergence_warnings": convergence_warnings,
            "elapsed_seconds": elapsed_seconds,
        }
    )

candidate_results_10 = pd.DataFrame(candidate_result_rows_10)

candidate_results_10 = (
    candidate_results_10.sort_values(
        ["auprc", "auroc", "brier"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

candidate_results_10["selection_rank"] = np.arange(
    1,
    len(candidate_results_10) + 1,
)

best_row_10 = candidate_results_10.iloc[0]
selected_candidate_10 = str(best_row_10["candidate_id"])
selected_C_10 = float(best_row_10["C"])
selected_l1_ratio_10 = float(best_row_10["l1_ratio"])

# ------------------------------------------------------------
# 16. Platt calibration on selected pooled inner OOF
# ------------------------------------------------------------

selected_oof_10 = (
    pooled_oof_10.loc[
        pooled_oof_10["candidate_id"] == selected_candidate_10
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_10 = selected_oof_10[
    "label_stage23"
].to_numpy(dtype=int)
selected_oof_probability_10 = selected_oof_10[
    "prediction_raw"
].to_numpy(dtype=float)

platt_calibrator_10 = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_10.fit(
    probability_logit_10(selected_oof_probability_10),
    selected_oof_y_10,
)

platt_intercept_10 = float(platt_calibrator_10.intercept_[0])
platt_slope_10 = float(platt_calibrator_10.coef_[0][0])

if (
    not np.isfinite(platt_intercept_10)
    or not np.isfinite(platt_slope_10)
    or platt_slope_10 <= 0
):
    raise RuntimeError("Platt kalibrasyon katsayıları geçersiz.")

selected_model_10 = pd.DataFrame(
    [
        {
            "outer_fold": 4,
            "selected_candidate": selected_candidate_10,
            "selected_C": selected_C_10,
            "selected_l1_ratio": selected_l1_ratio_10,
            "selection_metric_primary": "pooled_inner_oof_auprc",
            "inner_oof_auprc": float(best_row_10["auprc"]),
            "inner_oof_auroc": float(best_row_10["auroc"]),
            "inner_oof_brier": float(best_row_10["brier"]),
            "inner_oof_log_loss": float(best_row_10["log_loss"]),
            "inner_oof_mean_predicted_risk": float(
                best_row_10["mean_predicted_risk"]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_10["observed_event_rate"]
            ),
            "platt_intercept": platt_intercept_10,
            "platt_slope": platt_slope_10,
            "protocol_sha256": EXPECTED_PROTOCOL_SHA_10,
        }
    ]
)

# ------------------------------------------------------------
# 17. Lock selection/calibration aggregate files
# ------------------------------------------------------------

candidate_results_path_10 = os.path.join(
    MODEL_OUTPUT_DIR,
    "10B_logistic_candidate_results_outer4.csv",
)
selected_model_path_10 = os.path.join(
    MODEL_OUTPUT_DIR,
    "10B_logistic_selected_model_outer4.csv",
)
selection_json_path_10 = os.path.join(
    MODEL_OUTPUT_DIR,
    "10B_logistic_selection_calibration_outer4.json",
)
selection_sha_path_10 = os.path.join(
    MODEL_OUTPUT_DIR,
    "10B_logistic_selection_calibration_outer4_SHA256.txt",
)

candidate_results_10.to_csv(
    candidate_results_path_10,
    index=False,
)
selected_model_10.to_csv(
    selected_model_path_10,
    index=False,
)

selection_configuration_10 = {
    "outer_fold": 4,
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_10,
    "selection_metric_primary": "pooled inner out-of-fold AUPRC",
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "selected_candidate": selected_candidate_10,
    "selected_C": selected_C_10,
    "selected_l1_ratio": selected_l1_ratio_10,
    "inner_oof_auprc": float(best_row_10["auprc"]),
    "inner_oof_auroc": float(best_row_10["auroc"]),
    "inner_oof_brier": float(best_row_10["brier"]),
    "platt_intercept": platt_intercept_10,
    "platt_slope": platt_slope_10,
    "inner_checkpoint_tables": checkpoint_tables_10,
    "patient_level_oof_written_to_drive": False,
}

with open(
    selection_json_path_10,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        selection_configuration_10,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(selection_json_path_10, "rb") as file_handle:
    selection_sha_10 = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    selection_sha_path_10,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(selection_sha_10 + "\n")

# ------------------------------------------------------------
# 18. Fit final selected outer-fold-4 model
# ------------------------------------------------------------

final_pipeline_10 = Pipeline(
    steps=[
        ("preprocessor", make_preprocessor_10()),
        (
            "model",
            make_model_10(
                selected_C_10,
                selected_l1_ratio_10,
            ),
        ),
    ]
)

print(
    "\nFitting selected outer-fold-4 model "
    "on all 46,803 training patients..."
)

final_fit_started_10 = time.time()

with warnings.catch_warnings(record=True) as final_warning_records_10:
    warnings.simplefilter("always", ConvergenceWarning)
    final_pipeline_10.fit(
        X_outer_training_10,
        y_outer_training_10,
    )

final_fit_elapsed_10 = time.time() - final_fit_started_10

final_convergence_warnings_10 = sum(
    issubclass(warning.category, ConvergenceWarning)
    for warning in final_warning_records_10
)

if final_convergence_warnings_10 != 0:
    raise RuntimeError(
        "Nihai dış kat 4 modelinde yakınsama uyarısı oluştu."
    )

# ------------------------------------------------------------
# 19. Outer-fold-4 test predictions and metrics
# ------------------------------------------------------------

outer4_raw_probabilities_10 = final_pipeline_10.predict_proba(
    X_outer_test_10
)[:, 1]

raw_clipped_10 = np.clip(
    outer4_raw_probabilities_10,
    1e-6,
    1 - 1e-6,
)
raw_logit_10 = np.log(
    raw_clipped_10 / (1 - raw_clipped_10)
)

outer4_platt_probabilities_10 = expit(
    platt_intercept_10 + platt_slope_10 * raw_logit_10
)

for probabilities, name in [
    (outer4_raw_probabilities_10, "raw"),
    (outer4_platt_probabilities_10, "platt"),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(f"{name} tahminlerinde eksik değer var.")

    if not np.all(
        (probabilities >= 0) & (probabilities <= 1)
    ):
        raise RuntimeError(
            f"{name} tahminlerinde geçersiz olasılık değeri var."
        )

raw_metrics_10 = probability_metrics_10(
    y_outer_test_10,
    outer4_raw_probabilities_10,
)
platt_metrics_10 = probability_metrics_10(
    y_outer_test_10,
    outer4_platt_probabilities_10,
)

raw_calibration_intercept_10, raw_calibration_slope_10 = (
    calibration_intercept_slope_10(
        y_outer_test_10,
        outer4_raw_probabilities_10,
    )
)

platt_calibration_intercept_10, platt_calibration_slope_10 = (
    calibration_intercept_slope_10(
        y_outer_test_10,
        outer4_platt_probabilities_10,
    )
)

outer4_test_results_10 = pd.DataFrame(
    [
        {
            "outer_fold": 4,
            "model": "elastic_net_logistic",
            "probability_type": "raw",
            **raw_metrics_10,
            "calibration_intercept": raw_calibration_intercept_10,
            "calibration_slope": raw_calibration_slope_10,
        },
        {
            "outer_fold": 4,
            "model": "elastic_net_logistic",
            "probability_type": "platt_calibrated",
            **platt_metrics_10,
            "calibration_intercept": platt_calibration_intercept_10,
            "calibration_slope": platt_calibration_slope_10,
        },
    ]
)

# ------------------------------------------------------------
# 20. Feature coefficient audit
# ------------------------------------------------------------

fitted_preprocessor_10 = final_pipeline_10.named_steps[
    "preprocessor"
]
fitted_model_10 = final_pipeline_10.named_steps["model"]

processed_feature_names_10 = (
    fitted_preprocessor_10.get_feature_names_out()
)
model_coefficients_10 = fitted_model_10.coef_.reshape(-1)

if len(processed_feature_names_10) != len(model_coefficients_10):
    raise RuntimeError("Feature ve katsayı sayıları uyuşmuyor.")

if len(set(processed_feature_names_10)) != len(
    processed_feature_names_10
):
    raise RuntimeError("İşlenmiş feature adlarında yinelenme var.")

coefficient_table_10 = pd.DataFrame(
    {
        "processed_feature": processed_feature_names_10,
        "coefficient": model_coefficients_10,
    }
)
coefficient_table_10["absolute_coefficient"] = (
    coefficient_table_10["coefficient"].abs()
)
coefficient_table_10["is_nonzero"] = ~np.isclose(
    coefficient_table_10["coefficient"],
    0.0,
    atol=1e-12,
)
coefficient_table_10["absolute_rank"] = (
    coefficient_table_10["absolute_coefficient"]
    .rank(method="first", ascending=False)
    .astype(int)
)
coefficient_table_10 = (
    coefficient_table_10.sort_values(
        ["absolute_coefficient", "processed_feature"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

nonzero_coefficients_10 = int(
    coefficient_table_10["is_nonzero"].sum()
)

final_model_summary_10 = pd.DataFrame(
    [
        {
            "outer_fold": 4,
            "selected_candidate": selected_candidate_10,
            "selected_C": selected_C_10,
            "selected_l1_ratio": selected_l1_ratio_10,
            "training_patients": len(X_outer_training_10),
            "training_hospitals": len(training_hospitals_10),
            "training_events": int(y_outer_training_10.sum()),
            "test_patients": len(X_outer_test_10),
            "test_hospitals": len(test_hospitals_10),
            "test_events": int(y_outer_test_10.sum()),
            "hospital_overlap": len(hospital_overlap_10),
            "processed_feature_columns": len(
                processed_feature_names_10
            ),
            "nonzero_coefficients": nonzero_coefficients_10,
            "model_intercept": float(
                fitted_model_10.intercept_[0]
            ),
            "convergence_warnings": final_convergence_warnings_10,
            "fit_elapsed_seconds": float(final_fit_elapsed_10),
            "locked_platt_intercept": platt_intercept_10,
            "locked_platt_slope": platt_slope_10,
            "protocol_sha256": EXPECTED_PROTOCOL_SHA_10,
            "selection_sha256": selection_sha_10,
        }
    ]
)

# ------------------------------------------------------------
# 21. Secure BigQuery outer-test checkpoint
# ------------------------------------------------------------

outer4_prediction_df_10 = pd.DataFrame(
    {
        "id_row": outer_test_meta_10["id_row"].astype(str),
        "outer_fold": np.full(
            len(outer_test_meta_10),
            4,
            dtype=np.int64,
        ),
        "label_stage23": y_outer_test_10.astype(np.int64),
        "prediction_raw": outer4_raw_probabilities_10.astype(
            np.float64
        ),
        "prediction_platt": outer4_platt_probabilities_10.astype(
            np.float64
        ),
        "model_name": "elastic_net_logistic",
        "model_version": "core_v1_nested_cv",
    }
)

if len(outer4_prediction_df_10) != 11688:
    raise RuntimeError(
        "Dış kat 4 tahmin satır sayısı 11.688 değil."
    )

if outer4_prediction_df_10["id_row"].duplicated().any():
    raise RuntimeError(
        "Dış kat 4 tahminlerinde yinelenen id_row var."
    )

if int(
    outer4_prediction_df_10["label_stage23"].sum()
) != 606:
    raise RuntimeError("Dış kat 4 olay sayısı 606 değil.")

prediction_table_id_10 = (
    f"{TARGET_DATASET}."
    "model_lr_outer_predictions_outer4_v1"
)

prediction_load_config_10 = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("prediction_platt", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("model_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("model_version", "STRING", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

print("\nUploading secure outer-fold-4 prediction checkpoint:")
print(prediction_table_id_10)

client.load_table_from_dataframe(
    outer4_prediction_df_10,
    prediction_table_id_10,
    job_config=prediction_load_config_10,
    location=BQ_LOCATION,
).result()

# ------------------------------------------------------------
# 22. BigQuery outer-test checkpoint verification
# ------------------------------------------------------------

SQL_VERIFY_PREDICTIONS_10 = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL) AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL) AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0 OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0 OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw) AS minimum_raw_probability,
  MAX(prediction_raw) AS maximum_raw_probability,
  MIN(prediction_platt) AS minimum_platt_probability,
  MAX(prediction_platt) AS maximum_platt_probability
FROM `{prediction_table_id_10}`;
"""

prediction_verification_10 = client.query(
    SQL_VERIFY_PREDICTIONS_10,
    location=BQ_LOCATION,
).to_dataframe()

verification_row_10 = prediction_verification_10.iloc[0]

expected_prediction_values_10 = {
    "prediction_rows": 11688,
    "distinct_rows": 11688,
    "outer_folds": 1,
    "minimum_outer_fold": 4,
    "maximum_outer_fold": 4,
    "events": 606,
    "nonevents": 11082,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in expected_prediction_values_10.items():
    actual_value = int(verification_row_10[field])
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: bulunan={actual_value}, beklenen={expected_value}"
        )

# ------------------------------------------------------------
# 23. Save only aggregate and feature-level outputs to Drive
# ------------------------------------------------------------

test_results_path_10 = os.path.join(
    MODEL_OUTPUT_DIR,
    "10C_logistic_outer4_test_results.csv",
)
model_summary_path_10 = os.path.join(
    MODEL_OUTPUT_DIR,
    "10C_logistic_final_model_outer4.csv",
)
coefficient_path_10 = os.path.join(
    MODEL_OUTPUT_DIR,
    "10C_logistic_coefficients_outer4.csv",
)
evaluation_json_path_10 = os.path.join(
    MODEL_OUTPUT_DIR,
    "10C_logistic_final_evaluation_outer4.json",
)
evaluation_sha_path_10 = os.path.join(
    MODEL_OUTPUT_DIR,
    "10C_logistic_final_evaluation_outer4_SHA256.txt",
)

outer4_test_results_10.to_csv(
    test_results_path_10,
    index=False,
)
final_model_summary_10.to_csv(
    model_summary_path_10,
    index=False,
)
coefficient_table_10.to_csv(
    coefficient_path_10,
    index=False,
)

evaluation_configuration_10 = {
    "outer_fold": 4,
    "model_family": "elastic_net_logistic_regression",
    "selected_candidate": selected_candidate_10,
    "selected_C": selected_C_10,
    "selected_l1_ratio": selected_l1_ratio_10,
    "training_patients": 46803,
    "training_hospitals": 159,
    "test_patients": 11688,
    "test_hospitals": 39,
    "hospital_overlap": 0,
    "locked_platt_intercept": platt_intercept_10,
    "locked_platt_slope": platt_slope_10,
    "processed_feature_columns": int(
        len(processed_feature_names_10)
    ),
    "nonzero_coefficients": int(nonzero_coefficients_10),
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_10,
    "selection_sha256": selection_sha_10,
    "secure_prediction_table": prediction_table_id_10,
    "patient_level_prediction_written_to_drive": False,
}

with open(
    evaluation_json_path_10,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        evaluation_configuration_10,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(evaluation_json_path_10, "rb") as file_handle:
    evaluation_sha_10 = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    evaluation_sha_path_10,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(evaluation_sha_10 + "\n")

# ------------------------------------------------------------
# 24. Final outputs
# ------------------------------------------------------------

pooled_integrity_10 = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_10),
            pooled_oof_10["id_row"].nunique(),
            pooled_oof_10["candidate_id"].nunique(),
            pooled_oof_10["inner_fold"].nunique(),
            EXPECTED_SPLIT_10["training_events"],
            (
                EXPECTED_SPLIT_10["training_rows"]
                - EXPECTED_SPLIT_10["training_events"]
            ),
            int(
                pooled_oof_10.duplicated(
                    subset=["candidate_id", "id_row"]
                ).sum()
            ),
            int(pooled_oof_10["prediction_raw"].isna().sum()),
            int(
                (~pooled_oof_10["prediction_raw"].between(0, 1)).sum()
            ),
            pooled_load_method_10,
        ],
    }
)

print("\n10 OUTER-FOLD-4 INNER CHECKPOINT SUMMARY")
display(checkpoint_summary_10)

print("\n10 OUTER-FOLD-4 POOLED OOF INTEGRITY")
display(pooled_integrity_10)

print("\n10 OUTER-FOLD-4 CANDIDATE RESULTS")
display(candidate_results_10)

print("\n10 OUTER-FOLD-4 SELECTED MODEL")
display(selected_model_10)

print("\n10 OUTER-FOLD-4 FINAL MODEL SUMMARY")
display(final_model_summary_10)

print("\n10 OUTER-FOLD-4 TEST RESULTS")
display(outer4_test_results_10)

print("\n10 OUTER-FOLD-4 BIGQUERY VERIFICATION")
display(prediction_verification_10)

print("\n10 OUTER-FOLD-4 TOP 20 ABSOLUTE COEFFICIENTS")
display(coefficient_table_10.head(20))

print("\nSelection SHA-256:")
print(selection_sha_10)

print("\nEvaluation SHA-256:")
print(evaluation_sha_10)

print("\nSaved:")
print(fit_audit_path_10)
print(checkpoint_summary_path_10)
print(candidate_results_path_10)
print(selected_model_path_10)
print(selection_json_path_10)
print(selection_sha_path_10)
print(test_results_path_10)
print(model_summary_path_10)
print(coefficient_path_10)
print(evaluation_json_path_10)
print(evaluation_sha_path_10)

print(
    "\n10 PASS: Outer-fold-4 nested modelling "
    "and locked test evaluation are complete."
)
print(
    "All inner OOF and outer-test patient-level "
    "predictions were stored only in BigQuery."
)
print(
    "No patient-level prediction file was "
    "written to Google Drive."
)

_ = gc.collect()

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from sklearn.exceptions import ConvergenceWarning

from IPython.display import display

# ============================================================
# 11 — OUTER FOLD 5 COMPLETE NESTED MODELLING
#
# Resume-safe:
# - Each inner fold is stored in a separate BigQuery table.
# - Completed inner folds are automatically skipped.
#
# BigQuery free-tier compatible:
# - No DELETE / INSERT / UPDATE / MERGE is used.
#
# Privacy:
# - Patient-level predictions are stored only in BigQuery.
# - No patient-level prediction file is written to Drive.
# ============================================================

print("STARTING OUTER FOLD 5 — CODE VERSION 11")

OUTER_FOLD_11 = 5
MODEL_RANDOM_SEED_11 = 20260721

EXPECTED_PROTOCOL_SHA_11 = (
    "400c3b4b510c836794543bc685c62fae"
    "f49df0c1caa197d78dffda8c2207952d"
)

EXPECTED_SPLIT_11 = {
    "training_rows": 46803,
    "test_rows": 11688,
    "training_hospitals": 159,
    "test_hospitals": 39,
    "training_events": 2426,
    "test_events": 606,
}

EXPECTED_INNER_11 = {
    1: {"validation_rows": 9010, "validation_events": 420},
    2: {"validation_rows": 6838, "validation_events": 360},
    3: {"validation_rows": 9532, "validation_events": 515},
    4: {"validation_rows": 12455, "validation_events": 651},
    5: {"validation_rows": 8968, "validation_events": 480},
}

# ------------------------------------------------------------
# 1. Required runtime objects
# ------------------------------------------------------------

required_objects_11 = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_11 = [
    name for name in required_objects_11 if name not in globals()
]

if missing_objects_11:
    raise RuntimeError(
        "Eksik RAM nesneleri var: "
        + ", ".join(missing_objects_11)
        + ". Önce 07A ve 07B hücrelerini çalıştır."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"58.491 satır bekleniyordu; {len(core_df_07B)} bulundu."
    )

if len(predictor_columns_07B) != 159:
    raise RuntimeError("Core predictor sayısı 159 değil.")

if len(numeric_columns_07B) != 156:
    raise RuntimeError("Sayısal predictor sayısı 156 değil.")

if len(categorical_columns_07B) != 3:
    raise RuntimeError("Kategorik predictor sayısı 3 değil.")

# ------------------------------------------------------------
# 2. Locked protocol SHA check
# ------------------------------------------------------------

protocol_sha_path_11 = os.path.join(
    MODEL_OUTPUT_DIR,
    "08B_locked_logistic_model_protocol_v1_SHA256.txt",
)

if not os.path.exists(protocol_sha_path_11):
    raise FileNotFoundError(
        "Model protokolü SHA dosyası bulunamadı: "
        + protocol_sha_path_11
    )

with open(protocol_sha_path_11, "r", encoding="utf-8") as file_handle:
    observed_protocol_sha_11 = file_handle.read().strip()

if observed_protocol_sha_11 != EXPECTED_PROTOCOL_SHA_11:
    raise RuntimeError(
        "Kilitli model protokolü SHA değeri değişmiş: "
        + observed_protocol_sha_11
    )

# ------------------------------------------------------------
# 3. Locked inner-hospital mapping
# ------------------------------------------------------------

inner_mapping_path_11 = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_11):
    raise FileNotFoundError(
        "Kilitli iç kat haritası bulunamadı: "
        + inner_mapping_path_11
    )

inner_mapping_all_11 = pd.read_csv(
    inner_mapping_path_11,
    dtype={"group_hospital": str},
)

inner_mapping_part_11 = (
    inner_mapping_all_11.loc[
        inner_mapping_all_11["outer_fold"].astype(int) == OUTER_FOLD_11,
        ["group_hospital", "inner_fold"],
    ]
    .copy()
)

inner_mapping_part_11["group_hospital"] = (
    inner_mapping_part_11["group_hospital"].astype(str)
)
inner_mapping_part_11["inner_fold"] = (
    inner_mapping_part_11["inner_fold"].astype(int)
)

if len(inner_mapping_part_11) != 159:
    raise RuntimeError(
        "Dış kat 5 eğitim kümesi için 159 hastane ataması bekleniyordu."
    )

if inner_mapping_part_11["group_hospital"].duplicated().any():
    raise RuntimeError("İç kat haritasında yinelenen hastane var.")

hospital_to_inner_fold_11 = dict(
    zip(
        inner_mapping_part_11["group_hospital"],
        inner_mapping_part_11["inner_fold"],
    )
)

# ------------------------------------------------------------
# 4. Prepare model matrices
# ------------------------------------------------------------

X_all_11 = core_df_07B[predictor_columns_07B].copy()

for column in numeric_columns_07B:
    X_all_11[column] = pd.to_numeric(
        X_all_11[column], errors="coerce"
    ).astype("float64")

for column in categorical_columns_07B:
    category_series = X_all_11[column].astype("object")
    X_all_11[column] = category_series.where(
        pd.notna(category_series), np.nan
    )

outer_fold_vector_11 = (
    core_df_07B["outer_fold"].astype(int).to_numpy()
)

outer_training_mask_11 = outer_fold_vector_11 != OUTER_FOLD_11
outer_test_mask_11 = outer_fold_vector_11 == OUTER_FOLD_11

X_outer_training_11 = (
    X_all_11.loc[outer_training_mask_11].reset_index(drop=True)
)
X_outer_test_11 = (
    X_all_11.loc[outer_test_mask_11].reset_index(drop=True)
)

outer_training_meta_11 = (
    core_df_07B.loc[
        outer_training_mask_11,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_11 = (
    core_df_07B.loc[
        outer_test_mask_11,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [outer_training_meta_11, outer_test_meta_11]:
    dataframe["id_row"] = dataframe["id_row"].astype(str)
    dataframe["group_hospital"] = dataframe["group_hospital"].astype(str)
    dataframe["label_stage23"] = dataframe["label_stage23"].astype(int)

y_outer_training_11 = (
    outer_training_meta_11["label_stage23"].to_numpy(dtype=np.int8)
)
y_outer_test_11 = (
    outer_test_meta_11["label_stage23"].to_numpy(dtype=np.int8)
)

groups_outer_training_11 = (
    outer_training_meta_11["group_hospital"].to_numpy(dtype=str)
)

training_hospitals_11 = set(
    outer_training_meta_11["group_hospital"]
)
test_hospitals_11 = set(
    outer_test_meta_11["group_hospital"]
)
hospital_overlap_11 = training_hospitals_11 & test_hospitals_11

if hospital_overlap_11:
    raise RuntimeError("Dış eğitim ve test hastaneleri çakışıyor.")

actual_split_11 = {
    "training_rows": len(X_outer_training_11),
    "test_rows": len(X_outer_test_11),
    "training_hospitals": len(training_hospitals_11),
    "test_hospitals": len(test_hospitals_11),
    "training_events": int(y_outer_training_11.sum()),
    "test_events": int(y_outer_test_11.sum()),
}

for metric, expected_value in EXPECTED_SPLIT_11.items():
    actual_value = actual_split_11[metric]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: bulunan={actual_value}, beklenen={expected_value}"
        )

inner_fold_vector_11 = np.array(
    [
        hospital_to_inner_fold_11.get(hospital, -1)
        for hospital in groups_outer_training_11
    ],
    dtype=int,
)

if (inner_fold_vector_11 == -1).any():
    raise RuntimeError(
        "Bazı dış eğitim hastanelerine iç kat atanmadı."
    )

if set(np.unique(inner_fold_vector_11)) != {1, 2, 3, 4, 5}:
    raise RuntimeError("İç kat değerleri 1–5 değil.")

for inner_fold, expected in EXPECTED_INNER_11.items():
    validation_mask = inner_fold_vector_11 == inner_fold
    observed_rows = int(validation_mask.sum())
    observed_events = int(y_outer_training_11[validation_mask].sum())

    if observed_rows != expected["validation_rows"]:
        raise RuntimeError(
            f"Inner {inner_fold} validation_rows: "
            f"bulunan={observed_rows}, "
            f"beklenen={expected['validation_rows']}"
        )

    if observed_events != expected["validation_events"]:
        raise RuntimeError(
            f"Inner {inner_fold} validation_events: "
            f"bulunan={observed_events}, "
            f"beklenen={expected['validation_events']}"
        )

# ------------------------------------------------------------
# 5. Locked candidate grid
# ------------------------------------------------------------

candidate_grid_11 = [
    {"candidate_id": "LR01", "C": 0.03, "l1_ratio": 0.00},
    {"candidate_id": "LR02", "C": 0.10, "l1_ratio": 0.00},
    {"candidate_id": "LR03", "C": 0.30, "l1_ratio": 0.00},
    {"candidate_id": "LR04", "C": 0.10, "l1_ratio": 0.25},
    {"candidate_id": "LR05", "C": 0.30, "l1_ratio": 0.25},
    {"candidate_id": "LR06", "C": 0.30, "l1_ratio": 0.50},
]

# ------------------------------------------------------------
# 6. Preprocessing and model factories
# ------------------------------------------------------------

def make_preprocessor_11():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
            (
                "scaler",
                StandardScaler(with_mean=False),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns_07B,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns_07B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_model_11(C_value, l1_ratio_value):
    return LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=float(C_value),
        l1_ratio=float(l1_ratio_value),
        class_weight=None,
        max_iter=5000,
        tol=1e-4,
        random_state=MODEL_RANDOM_SEED_11,
    )


def checkpoint_table_id_11(inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_lr_inner_oof_outer5_inner{inner_fold}_v1"
    )

# ------------------------------------------------------------
# 7. BigQuery checkpoint verification
# ------------------------------------------------------------

def verify_checkpoint_11(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):
    table_id = checkpoint_table_id_11(inner_fold)

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS distinct_id_count,
      COUNT(DISTINCT candidate_id) AS candidate_count,
      COUNT(DISTINCT outer_fold) AS outer_fold_count,
      COUNT(DISTINCT inner_fold) AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(candidate_id, '|', id_row)
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(
        prediction_raw < 0 OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(sql, location=BQ_LOCATION).to_dataframe()
    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows * len(candidate_grid_11)
    )
    expected_positive_rows = (
        expected_validation_events * len(candidate_grid_11)
    )
    expected_negative_rows = (
        (expected_validation_rows - expected_validation_events)
        * len(candidate_grid_11)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": expected_validation_rows,
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": expected_total_rows,
        "positive_prediction_rows": expected_positive_rows,
        "negative_prediction_rows": expected_negative_rows,
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": 5,
        "maximum_outer_fold": 5,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failure_items = []

    for field, expected_value in expected_values.items():
        actual_value = int(row[field])
        if actual_value != expected_value:
            complete = False
            failure_items.append(
                f"{field}={actual_value}, expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failure_items),
        "check": check,
        "row": row,
    }

# ------------------------------------------------------------
# 8. BigQuery load schema
# ------------------------------------------------------------

checkpoint_load_config_11 = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("inner_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("candidate_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

# ------------------------------------------------------------
# 9. Aggregate fit-audit file
# ------------------------------------------------------------

fit_audit_columns_11 = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "C",
    "l1_ratio",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "convergence_warnings",
    "elapsed_seconds",
]

fit_audit_path_11 = os.path.join(
    MODEL_OUTPUT_DIR,
    "11A_logistic_inner_fit_audit_outer5.csv",
)

if os.path.exists(fit_audit_path_11):
    fit_audit_11 = pd.read_csv(fit_audit_path_11)
else:
    fit_audit_11 = pd.DataFrame(columns=fit_audit_columns_11)

for column in fit_audit_columns_11:
    if column not in fit_audit_11.columns:
        fit_audit_11[column] = np.nan

fit_audit_11 = fit_audit_11[fit_audit_columns_11].copy()

# ------------------------------------------------------------
# 10. Train six candidates in five locked inner folds
# ------------------------------------------------------------

for inner_fold in range(1, 6):
    inner_training_mask = inner_fold_vector_11 != inner_fold
    inner_validation_mask = inner_fold_vector_11 == inner_fold

    training_rows = int(inner_training_mask.sum())
    validation_rows = int(inner_validation_mask.sum())
    training_events = int(
        y_outer_training_11[inner_training_mask].sum()
    )
    validation_events = int(
        y_outer_training_11[inner_validation_mask].sum()
    )

    expected_inner = EXPECTED_INNER_11[inner_fold]
    expected_training_rows = (
        EXPECTED_SPLIT_11["training_rows"]
        - expected_inner["validation_rows"]
    )
    expected_training_events = (
        EXPECTED_SPLIT_11["training_events"]
        - expected_inner["validation_events"]
    )

    if training_rows != expected_training_rows:
        raise RuntimeError(
            f"Inner {inner_fold} training_rows: "
            f"bulunan={training_rows}, "
            f"beklenen={expected_training_rows}"
        )

    if training_events != expected_training_events:
        raise RuntimeError(
            f"Inner {inner_fold} training_events: "
            f"bulunan={training_events}, "
            f"beklenen={expected_training_events}"
        )

    existing_check = verify_checkpoint_11(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if existing_check["complete"]:
        print(
            f"Outer 5 / inner {inner_fold}: "
            "permanent checkpoint already complete; "
            "skipping model fitting."
        )
        continue

    training_hospital_set = set(
        groups_outer_training_11[inner_training_mask]
    )
    validation_hospital_set = set(
        groups_outer_training_11[inner_validation_mask]
    )

    if training_hospital_set & validation_hospital_set:
        raise RuntimeError(
            f"Inner fold {inner_fold}: hastane çakışması bulundu."
        )

    print(f"\nOuter 5 / inner {inner_fold}")
    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_11()
    preprocessing_started = time.time()

    X_inner_training_processed = preprocessor.fit_transform(
        X_outer_training_11.loc[inner_training_mask]
    )
    X_inner_validation_processed = preprocessor.transform(
        X_outer_training_11.loc[inner_validation_mask]
    )

    preprocessing_elapsed = time.time() - preprocessing_started

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "İşlenmiş eğitim ve doğrulama sütun sayıları farklı."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_elapsed, 2),
    )

    y_inner_training = y_outer_training_11[inner_training_mask]
    y_inner_validation = y_outer_training_11[inner_validation_mask]

    validation_ids = (
        outer_training_meta_11.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_11:
        candidate_id = candidate["candidate_id"]

        print(
            "  Fitting",
            candidate_id,
            "| C =",
            candidate["C"],
            "| l1_ratio =",
            candidate["l1_ratio"],
        )

        model = make_model_11(
            candidate["C"],
            candidate["l1_ratio"],
        )

        fitting_started = time.time()

        with warnings.catch_warnings(record=True) as warning_records:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(
                X_inner_training_processed,
                y_inner_training,
            )

        fitting_elapsed = time.time() - fitting_started

        convergence_warning_count = sum(
            issubclass(warning.category, ConvergenceWarning)
            for warning in warning_records
        )

        validation_probabilities = model.predict_proba(
            X_inner_validation_processed
        )[:, 1]

        if np.isnan(validation_probabilities).any():
            raise RuntimeError(
                f"{candidate_id}, inner {inner_fold}: eksik tahmin."
            )

        if not np.all(
            (validation_probabilities >= 0)
            & (validation_probabilities <= 1)
        ):
            raise RuntimeError(
                f"{candidate_id}: geçersiz olasılık."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        5,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": y_inner_validation.astype(np.int64),
                    "prediction_raw": validation_probabilities.astype(
                        np.float64
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": 5,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "C": candidate["C"],
                "l1_ratio": candidate["l1_ratio"],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": validation_events,
                "processed_columns": int(
                    X_inner_training_processed.shape[1]
                ),
                "convergence_warnings": int(
                    convergence_warning_count
                ),
                "elapsed_seconds": float(fitting_elapsed),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows * len(candidate_grid_11)
    )

    if len(checkpoint_df) != expected_checkpoint_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: checkpoint satır sayısı hatalı."
        )

    if checkpoint_df.duplicated(
        subset=["id_row", "candidate_id"]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "yinelenen aday–hasta tahmini var."
        )

    target_checkpoint_table = checkpoint_table_id_11(inner_fold)

    print(
        "Uploading permanent checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_11,
        location=BQ_LOCATION,
    ).result()

    new_audit_df = pd.DataFrame(current_audit_rows)

    if len(fit_audit_11) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_11["outer_fold"],
                    errors="coerce",
                )
                == 4
            )
            & (
                pd.to_numeric(
                    fit_audit_11["inner_fold"],
                    errors="coerce",
                )
                == inner_fold
            )
        )
        fit_audit_11 = fit_audit_11.loc[keep_mask].copy()

    if fit_audit_11.empty:
        fit_audit_11 = new_audit_df.copy()
    else:
        fit_audit_11 = pd.concat(
            [fit_audit_11, new_audit_df],
            ignore_index=True,
        )

    fit_audit_11 = (
        fit_audit_11[fit_audit_columns_11]
        .sort_values(
            ["outer_fold", "inner_fold", "candidate_id"]
        )
        .reset_index(drop=True)
    )

    fit_audit_11.to_csv(
        fit_audit_path_11,
        index=False,
    )

    completed_check = verify_checkpoint_11(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint doğrulanamadı: "
            + completed_check["reason"]
        )

    print(
        f"Outer 5 / inner {inner_fold}: "
        "permanent checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 11. Final checkpoint summary
# ------------------------------------------------------------

checkpoint_summary_rows_11 = []

for inner_fold in range(1, 6):
    validation_mask = inner_fold_vector_11 == inner_fold
    validation_rows = int(validation_mask.sum())
    validation_events = int(
        y_outer_training_11[validation_mask].sum()
    )

    final_check = verify_checkpoint_11(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "son checkpoint denetimi başarısız. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_11.append(
        {
            "outer_fold": 5,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(row["row_count"]),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(row["candidate_count"]),
            "positive_prediction_rows": int(
                row["positive_prediction_rows"]
            ),
            "negative_prediction_rows": int(
                row["negative_prediction_rows"]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check["table_id"],
        }
    )

checkpoint_summary_11 = (
    pd.DataFrame(checkpoint_summary_rows_11)
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_11["distinct_validation_patients"].sum()
) != EXPECTED_SPLIT_11["training_rows"]:
    raise RuntimeError(
        "Toplam doğrulama hasta sayısı 46.803 değil."
    )

expected_total_oof_rows_11 = (
    EXPECTED_SPLIT_11["training_rows"]
    * len(candidate_grid_11)
)

if int(checkpoint_summary_11["checkpoint_rows"].sum()) != (
    expected_total_oof_rows_11
):
    raise RuntimeError("Toplam OOF tahmin satırı hatalı.")

checkpoint_summary_path_11 = os.path.join(
    MODEL_OUTPUT_DIR,
    "11A_outer5_inner_checkpoint_summary.csv",
)
checkpoint_summary_11.to_csv(
    checkpoint_summary_path_11,
    index=False,
)

# ------------------------------------------------------------
# 12. Pool all inner OOF predictions
# ------------------------------------------------------------

checkpoint_tables_11 = [
    checkpoint_table_id_11(inner_fold)
    for inner_fold in range(1, 6)
]

union_parts_11 = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_11
]

SQL_LOAD_POOLED_OOF_11 = "\nUNION ALL\n".join(union_parts_11)

print("\nLoading pooled outer-fold-5 inner OOF predictions...")

query_job_11 = client.query(
    SQL_LOAD_POOLED_OOF_11,
    location=BQ_LOCATION,
)

try:
    pooled_oof_11 = query_job_11.to_dataframe(
        create_bqstorage_client=True
    )
    pooled_load_method_11 = "BigQuery Storage API"
except Exception as fast_path_error_11:
    print(
        "Storage API unavailable; using standard BigQuery download."
    )
    print("Message:", type(fast_path_error_11).__name__)
    pooled_oof_11 = query_job_11.to_dataframe(
        create_bqstorage_client=False
    )
    pooled_load_method_11 = "Standard BigQuery API"

pooled_oof_11["id_row"] = pooled_oof_11["id_row"].astype(str)
pooled_oof_11["candidate_id"] = pooled_oof_11["candidate_id"].astype(str)

for column in ["outer_fold", "inner_fold", "label_stage23"]:
    pooled_oof_11[column] = pd.to_numeric(
        pooled_oof_11[column], errors="raise"
    ).astype(int)

pooled_oof_11["prediction_raw"] = pd.to_numeric(
    pooled_oof_11["prediction_raw"], errors="raise"
).astype(float)

# ------------------------------------------------------------
# 13. Pooled OOF integrity checks
# ------------------------------------------------------------

if len(pooled_oof_11) != expected_total_oof_rows_11:
    raise RuntimeError("Pooled OOF satır sayısı hatalı.")

if set(pooled_oof_11["outer_fold"].unique()) != {5}:
    raise RuntimeError("Pooled OOF içinde dış kat 5 dışında kayıt var.")

if set(pooled_oof_11["inner_fold"].unique()) != {1, 2, 3, 4, 5}:
    raise RuntimeError("Pooled OOF iç katları 1–5 değil.")

if pooled_oof_11.duplicated(
    subset=["candidate_id", "id_row"]
).any():
    raise RuntimeError(
        "Pooled OOF içinde yinelenen aday–hasta tahmini bulundu."
    )

if pooled_oof_11["prediction_raw"].isna().any():
    raise RuntimeError("Pooled OOF içinde eksik tahmin var.")

if not pooled_oof_11["prediction_raw"].between(0, 1).all():
    raise RuntimeError(
        "Pooled OOF içinde geçersiz olasılık değeri var."
    )

expected_candidate_ids_11 = {
    "LR01",
    "LR02",
    "LR03",
    "LR04",
    "LR05",
    "LR06",
}

if set(pooled_oof_11["candidate_id"].unique()) != (
    expected_candidate_ids_11
):
    raise RuntimeError("Altı kilitli aday bulunmuyor.")

candidate_patient_counts_11 = (
    pooled_oof_11.groupby("candidate_id")["id_row"].nunique()
)

if not (
    candidate_patient_counts_11
    == EXPECTED_SPLIT_11["training_rows"]
).all():
    raise RuntimeError(
        "Her aday için 46.803 farklı OOF hastası yok."
    )

candidate_event_counts_11 = (
    pooled_oof_11.groupby("candidate_id")["label_stage23"].sum()
)

if not (
    candidate_event_counts_11
    == EXPECTED_SPLIT_11["training_events"]
).all():
    raise RuntimeError("Her aday için 2.426 olay yok.")

patient_label_consistency_11 = (
    pooled_oof_11.groupby("id_row")["label_stage23"].nunique()
)
if (patient_label_consistency_11 > 1).any():
    raise RuntimeError(
        "Aynı hastanın adaylar arasında outcome etiketi farklı."
    )

patient_inner_fold_consistency_11 = (
    pooled_oof_11.groupby("id_row")["inner_fold"].nunique()
)
if (patient_inner_fold_consistency_11 > 1).any():
    raise RuntimeError(
        "Aynı hasta birden fazla iç doğrulama katında bulundu."
    )

# ------------------------------------------------------------
# 14. Metric helpers
# ------------------------------------------------------------

def probability_metrics_11(y_true, probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(roc_auc_score(y_true, probabilities)),
        "auprc": float(average_precision_score(y_true, probabilities)),
        "brier": float(brier_score_loss(y_true, probabilities)),
        "log_loss": float(
            log_loss(y_true, probabilities, labels=[0, 1])
        ),
        "mean_predicted_risk": float(probabilities.mean()),
        "observed_event_rate": float(np.mean(y_true)),
    }


def probability_logit_11(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        probabilities / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_11(y_true, probabilities):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        probability_logit_11(probabilities),
        y_true,
    )
    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 15. Candidate pooled-OOF performance
# ------------------------------------------------------------

candidate_result_rows_11 = []

for candidate in candidate_grid_11:
    candidate_id = candidate["candidate_id"]

    candidate_oof = (
        pooled_oof_11.loc[
            pooled_oof_11["candidate_id"] == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_11(
        candidate_oof["label_stage23"].to_numpy(dtype=int),
        candidate_oof["prediction_raw"].to_numpy(dtype=float),
    )

    fit_part = fit_audit_11.loc[
        (
            pd.to_numeric(
                fit_audit_11["outer_fold"],
                errors="coerce",
            )
            == 4
        )
        & (
            fit_audit_11["candidate_id"].astype(str)
            == candidate_id
        )
    ]

    convergence_warnings = (
        int(
            pd.to_numeric(
                fit_part["convergence_warnings"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    elapsed_seconds = (
        float(
            pd.to_numeric(
                fit_part["elapsed_seconds"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_11.append(
        {
            "candidate_id": candidate_id,
            "C": candidate["C"],
            "l1_ratio": candidate["l1_ratio"],
            **metrics,
            "convergence_warnings": convergence_warnings,
            "elapsed_seconds": elapsed_seconds,
        }
    )

candidate_results_11 = pd.DataFrame(candidate_result_rows_11)

candidate_results_11 = (
    candidate_results_11.sort_values(
        ["auprc", "auroc", "brier"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

candidate_results_11["selection_rank"] = np.arange(
    1,
    len(candidate_results_11) + 1,
)

best_row_11 = candidate_results_11.iloc[0]
selected_candidate_11 = str(best_row_11["candidate_id"])
selected_C_11 = float(best_row_11["C"])
selected_l1_ratio_11 = float(best_row_11["l1_ratio"])

# ------------------------------------------------------------
# 16. Platt calibration on selected pooled inner OOF
# ------------------------------------------------------------

selected_oof_11 = (
    pooled_oof_11.loc[
        pooled_oof_11["candidate_id"] == selected_candidate_11
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_11 = selected_oof_11[
    "label_stage23"
].to_numpy(dtype=int)
selected_oof_probability_11 = selected_oof_11[
    "prediction_raw"
].to_numpy(dtype=float)

platt_calibrator_11 = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_11.fit(
    probability_logit_11(selected_oof_probability_11),
    selected_oof_y_11,
)

platt_intercept_11 = float(platt_calibrator_11.intercept_[0])
platt_slope_11 = float(platt_calibrator_11.coef_[0][0])

if (
    not np.isfinite(platt_intercept_11)
    or not np.isfinite(platt_slope_11)
    or platt_slope_11 <= 0
):
    raise RuntimeError("Platt kalibrasyon katsayıları geçersiz.")

selected_model_11 = pd.DataFrame(
    [
        {
            "outer_fold": 5,
            "selected_candidate": selected_candidate_11,
            "selected_C": selected_C_11,
            "selected_l1_ratio": selected_l1_ratio_11,
            "selection_metric_primary": "pooled_inner_oof_auprc",
            "inner_oof_auprc": float(best_row_11["auprc"]),
            "inner_oof_auroc": float(best_row_11["auroc"]),
            "inner_oof_brier": float(best_row_11["brier"]),
            "inner_oof_log_loss": float(best_row_11["log_loss"]),
            "inner_oof_mean_predicted_risk": float(
                best_row_11["mean_predicted_risk"]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_11["observed_event_rate"]
            ),
            "platt_intercept": platt_intercept_11,
            "platt_slope": platt_slope_11,
            "protocol_sha256": EXPECTED_PROTOCOL_SHA_11,
        }
    ]
)

# ------------------------------------------------------------
# 17. Lock selection/calibration aggregate files
# ------------------------------------------------------------

candidate_results_path_11 = os.path.join(
    MODEL_OUTPUT_DIR,
    "11B_logistic_candidate_results_outer5.csv",
)
selected_model_path_11 = os.path.join(
    MODEL_OUTPUT_DIR,
    "11B_logistic_selected_model_outer5.csv",
)
selection_json_path_11 = os.path.join(
    MODEL_OUTPUT_DIR,
    "11B_logistic_selection_calibration_outer5.json",
)
selection_sha_path_11 = os.path.join(
    MODEL_OUTPUT_DIR,
    "11B_logistic_selection_calibration_outer5_SHA256.txt",
)

candidate_results_11.to_csv(
    candidate_results_path_11,
    index=False,
)
selected_model_11.to_csv(
    selected_model_path_11,
    index=False,
)

selection_configuration_11 = {
    "outer_fold": 5,
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_11,
    "selection_metric_primary": "pooled inner out-of-fold AUPRC",
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "selected_candidate": selected_candidate_11,
    "selected_C": selected_C_11,
    "selected_l1_ratio": selected_l1_ratio_11,
    "inner_oof_auprc": float(best_row_11["auprc"]),
    "inner_oof_auroc": float(best_row_11["auroc"]),
    "inner_oof_brier": float(best_row_11["brier"]),
    "platt_intercept": platt_intercept_11,
    "platt_slope": platt_slope_11,
    "inner_checkpoint_tables": checkpoint_tables_11,
    "patient_level_oof_written_to_drive": False,
}

with open(
    selection_json_path_11,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        selection_configuration_11,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(selection_json_path_11, "rb") as file_handle:
    selection_sha_11 = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    selection_sha_path_11,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(selection_sha_11 + "\n")

# ------------------------------------------------------------
# 18. Fit final selected outer-fold-5 model
# ------------------------------------------------------------

final_pipeline_11 = Pipeline(
    steps=[
        ("preprocessor", make_preprocessor_11()),
        (
            "model",
            make_model_11(
                selected_C_11,
                selected_l1_ratio_11,
            ),
        ),
    ]
)

print(
    "\nFitting selected outer-fold-5 model "
    "on all 46,803 training patients..."
)

final_fit_started_11 = time.time()

with warnings.catch_warnings(record=True) as final_warning_records_11:
    warnings.simplefilter("always", ConvergenceWarning)
    final_pipeline_11.fit(
        X_outer_training_11,
        y_outer_training_11,
    )

final_fit_elapsed_11 = time.time() - final_fit_started_11

final_convergence_warnings_11 = sum(
    issubclass(warning.category, ConvergenceWarning)
    for warning in final_warning_records_11
)

if final_convergence_warnings_11 != 0:
    raise RuntimeError(
        "Nihai dış kat 5 modelinde yakınsama uyarısı oluştu."
    )

# ------------------------------------------------------------
# 19. Outer-fold-4 test predictions and metrics
# ------------------------------------------------------------

outer5_raw_probabilities_11 = final_pipeline_11.predict_proba(
    X_outer_test_11
)[:, 1]

raw_clipped_11 = np.clip(
    outer5_raw_probabilities_11,
    1e-6,
    1 - 1e-6,
)
raw_logit_11 = np.log(
    raw_clipped_11 / (1 - raw_clipped_11)
)

outer5_platt_probabilities_11 = expit(
    platt_intercept_11 + platt_slope_11 * raw_logit_11
)

for probabilities, name in [
    (outer5_raw_probabilities_11, "raw"),
    (outer5_platt_probabilities_11, "platt"),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(f"{name} tahminlerinde eksik değer var.")

    if not np.all(
        (probabilities >= 0) & (probabilities <= 1)
    ):
        raise RuntimeError(
            f"{name} tahminlerinde geçersiz olasılık değeri var."
        )

raw_metrics_11 = probability_metrics_11(
    y_outer_test_11,
    outer5_raw_probabilities_11,
)
platt_metrics_11 = probability_metrics_11(
    y_outer_test_11,
    outer5_platt_probabilities_11,
)

raw_calibration_intercept_11, raw_calibration_slope_11 = (
    calibration_intercept_slope_11(
        y_outer_test_11,
        outer5_raw_probabilities_11,
    )
)

platt_calibration_intercept_11, platt_calibration_slope_11 = (
    calibration_intercept_slope_11(
        y_outer_test_11,
        outer5_platt_probabilities_11,
    )
)

outer5_test_results_11 = pd.DataFrame(
    [
        {
            "outer_fold": 5,
            "model": "elastic_net_logistic",
            "probability_type": "raw",
            **raw_metrics_11,
            "calibration_intercept": raw_calibration_intercept_11,
            "calibration_slope": raw_calibration_slope_11,
        },
        {
            "outer_fold": 5,
            "model": "elastic_net_logistic",
            "probability_type": "platt_calibrated",
            **platt_metrics_11,
            "calibration_intercept": platt_calibration_intercept_11,
            "calibration_slope": platt_calibration_slope_11,
        },
    ]
)

# ------------------------------------------------------------
# 20. Feature coefficient audit
# ------------------------------------------------------------

fitted_preprocessor_11 = final_pipeline_11.named_steps[
    "preprocessor"
]
fitted_model_11 = final_pipeline_11.named_steps["model"]

processed_feature_names_11 = (
    fitted_preprocessor_11.get_feature_names_out()
)
model_coefficients_11 = fitted_model_11.coef_.reshape(-1)

if len(processed_feature_names_11) != len(model_coefficients_11):
    raise RuntimeError("Feature ve katsayı sayıları uyuşmuyor.")

if len(set(processed_feature_names_11)) != len(
    processed_feature_names_11
):
    raise RuntimeError("İşlenmiş feature adlarında yinelenme var.")

coefficient_table_11 = pd.DataFrame(
    {
        "processed_feature": processed_feature_names_11,
        "coefficient": model_coefficients_11,
    }
)
coefficient_table_11["absolute_coefficient"] = (
    coefficient_table_11["coefficient"].abs()
)
coefficient_table_11["is_nonzero"] = ~np.isclose(
    coefficient_table_11["coefficient"],
    0.0,
    atol=1e-12,
)
coefficient_table_11["absolute_rank"] = (
    coefficient_table_11["absolute_coefficient"]
    .rank(method="first", ascending=False)
    .astype(int)
)
coefficient_table_11 = (
    coefficient_table_11.sort_values(
        ["absolute_coefficient", "processed_feature"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

nonzero_coefficients_11 = int(
    coefficient_table_11["is_nonzero"].sum()
)

final_model_summary_11 = pd.DataFrame(
    [
        {
            "outer_fold": 5,
            "selected_candidate": selected_candidate_11,
            "selected_C": selected_C_11,
            "selected_l1_ratio": selected_l1_ratio_11,
            "training_patients": len(X_outer_training_11),
            "training_hospitals": len(training_hospitals_11),
            "training_events": int(y_outer_training_11.sum()),
            "test_patients": len(X_outer_test_11),
            "test_hospitals": len(test_hospitals_11),
            "test_events": int(y_outer_test_11.sum()),
            "hospital_overlap": len(hospital_overlap_11),
            "processed_feature_columns": len(
                processed_feature_names_11
            ),
            "nonzero_coefficients": nonzero_coefficients_11,
            "model_intercept": float(
                fitted_model_11.intercept_[0]
            ),
            "convergence_warnings": final_convergence_warnings_11,
            "fit_elapsed_seconds": float(final_fit_elapsed_11),
            "locked_platt_intercept": platt_intercept_11,
            "locked_platt_slope": platt_slope_11,
            "protocol_sha256": EXPECTED_PROTOCOL_SHA_11,
            "selection_sha256": selection_sha_11,
        }
    ]
)

# ------------------------------------------------------------
# 21. Secure BigQuery outer-test checkpoint
# ------------------------------------------------------------

outer5_prediction_df_11 = pd.DataFrame(
    {
        "id_row": outer_test_meta_11["id_row"].astype(str),
        "outer_fold": np.full(
            len(outer_test_meta_11),
            5,
            dtype=np.int64,
        ),
        "label_stage23": y_outer_test_11.astype(np.int64),
        "prediction_raw": outer5_raw_probabilities_11.astype(
            np.float64
        ),
        "prediction_platt": outer5_platt_probabilities_11.astype(
            np.float64
        ),
        "model_name": "elastic_net_logistic",
        "model_version": "core_v1_nested_cv",
    }
)

if len(outer5_prediction_df_11) != 11688:
    raise RuntimeError(
        "Dış kat 5 tahmin satır sayısı 11.688 değil."
    )

if outer5_prediction_df_11["id_row"].duplicated().any():
    raise RuntimeError(
        "Dış kat 5 tahminlerinde yinelenen id_row var."
    )

if int(
    outer5_prediction_df_11["label_stage23"].sum()
) != 606:
    raise RuntimeError("Dış kat 5 olay sayısı 606 değil.")

prediction_table_id_11 = (
    f"{TARGET_DATASET}."
    "model_lr_outer_predictions_outer5_v1"
)

prediction_load_config_11 = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("prediction_platt", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("model_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("model_version", "STRING", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

print("\nUploading secure outer-fold-5 prediction checkpoint:")
print(prediction_table_id_11)

client.load_table_from_dataframe(
    outer5_prediction_df_11,
    prediction_table_id_11,
    job_config=prediction_load_config_11,
    location=BQ_LOCATION,
).result()

# ------------------------------------------------------------
# 22. BigQuery outer-test checkpoint verification
# ------------------------------------------------------------

SQL_VERIFY_PREDICTIONS_11 = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL) AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL) AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0 OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0 OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw) AS minimum_raw_probability,
  MAX(prediction_raw) AS maximum_raw_probability,
  MIN(prediction_platt) AS minimum_platt_probability,
  MAX(prediction_platt) AS maximum_platt_probability
FROM `{prediction_table_id_11}`;
"""

prediction_verification_11 = client.query(
    SQL_VERIFY_PREDICTIONS_11,
    location=BQ_LOCATION,
).to_dataframe()

verification_row_11 = prediction_verification_11.iloc[0]

expected_prediction_values_11 = {
    "prediction_rows": 11688,
    "distinct_rows": 11688,
    "outer_folds": 1,
    "minimum_outer_fold": 5,
    "maximum_outer_fold": 5,
    "events": 606,
    "nonevents": 11082,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in expected_prediction_values_11.items():
    actual_value = int(verification_row_11[field])
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: bulunan={actual_value}, beklenen={expected_value}"
        )

# ------------------------------------------------------------
# 23. Save only aggregate and feature-level outputs to Drive
# ------------------------------------------------------------

test_results_path_11 = os.path.join(
    MODEL_OUTPUT_DIR,
    "11C_logistic_outer5_test_results.csv",
)
model_summary_path_11 = os.path.join(
    MODEL_OUTPUT_DIR,
    "11C_logistic_final_model_outer5.csv",
)
coefficient_path_11 = os.path.join(
    MODEL_OUTPUT_DIR,
    "11C_logistic_coefficients_outer5.csv",
)
evaluation_json_path_11 = os.path.join(
    MODEL_OUTPUT_DIR,
    "11C_logistic_final_evaluation_outer5.json",
)
evaluation_sha_path_11 = os.path.join(
    MODEL_OUTPUT_DIR,
    "11C_logistic_final_evaluation_outer5_SHA256.txt",
)

outer5_test_results_11.to_csv(
    test_results_path_11,
    index=False,
)
final_model_summary_11.to_csv(
    model_summary_path_11,
    index=False,
)
coefficient_table_11.to_csv(
    coefficient_path_11,
    index=False,
)

evaluation_configuration_11 = {
    "outer_fold": 5,
    "model_family": "elastic_net_logistic_regression",
    "selected_candidate": selected_candidate_11,
    "selected_C": selected_C_11,
    "selected_l1_ratio": selected_l1_ratio_11,
    "training_patients": 46803,
    "training_hospitals": 159,
    "test_patients": 11688,
    "test_hospitals": 39,
    "hospital_overlap": 0,
    "locked_platt_intercept": platt_intercept_11,
    "locked_platt_slope": platt_slope_11,
    "processed_feature_columns": int(
        len(processed_feature_names_11)
    ),
    "nonzero_coefficients": int(nonzero_coefficients_11),
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_11,
    "selection_sha256": selection_sha_11,
    "secure_prediction_table": prediction_table_id_11,
    "patient_level_prediction_written_to_drive": False,
}

with open(
    evaluation_json_path_11,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        evaluation_configuration_11,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(evaluation_json_path_11, "rb") as file_handle:
    evaluation_sha_11 = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    evaluation_sha_path_11,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(evaluation_sha_11 + "\n")

# ------------------------------------------------------------
# 24. Final outputs
# ------------------------------------------------------------

pooled_integrity_11 = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_11),
            pooled_oof_11["id_row"].nunique(),
            pooled_oof_11["candidate_id"].nunique(),
            pooled_oof_11["inner_fold"].nunique(),
            EXPECTED_SPLIT_11["training_events"],
            (
                EXPECTED_SPLIT_11["training_rows"]
                - EXPECTED_SPLIT_11["training_events"]
            ),
            int(
                pooled_oof_11.duplicated(
                    subset=["candidate_id", "id_row"]
                ).sum()
            ),
            int(pooled_oof_11["prediction_raw"].isna().sum()),
            int(
                (~pooled_oof_11["prediction_raw"].between(0, 1)).sum()
            ),
            pooled_load_method_11,
        ],
    }
)

print("\n11 OUTER-FOLD-5 INNER CHECKPOINT SUMMARY")
display(checkpoint_summary_11)

print("\n11 OUTER-FOLD-5 POOLED OOF INTEGRITY")
display(pooled_integrity_11)

print("\n11 OUTER-FOLD-5 CANDIDATE RESULTS")
display(candidate_results_11)

print("\n11 OUTER-FOLD-5 SELECTED MODEL")
display(selected_model_11)

print("\n11 OUTER-FOLD-5 FINAL MODEL SUMMARY")
display(final_model_summary_11)

print("\n11 OUTER-FOLD-5 TEST RESULTS")
display(outer5_test_results_11)

print("\n11 OUTER-FOLD-5 BIGQUERY VERIFICATION")
display(prediction_verification_11)

print("\n11 OUTER-FOLD-5 TOP 20 ABSOLUTE COEFFICIENTS")
display(coefficient_table_11.head(20))

print("\nSelection SHA-256:")
print(selection_sha_11)

print("\nEvaluation SHA-256:")
print(evaluation_sha_11)

print("\nSaved:")
print(fit_audit_path_11)
print(checkpoint_summary_path_11)
print(candidate_results_path_11)
print(selected_model_path_11)
print(selection_json_path_11)
print(selection_sha_path_11)
print(test_results_path_11)
print(model_summary_path_11)
print(coefficient_path_11)
print(evaluation_json_path_11)
print(evaluation_sha_path_11)

print(
    "\n11 PASS: Outer-fold-5 nested modelling "
    "and locked test evaluation are complete."
)
print(
    "All inner OOF and outer-test patient-level "
    "predictions were stored only in BigQuery."
)
print(
    "No patient-level prediction file was "
    "written to Google Drive."
)

_ = gc.collect()

In [ ]:
import os
import json
import hashlib
import warnings

import numpy as np
import pandas as pd

from google.cloud import bigquery

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

from IPython.display import display

print("STARTING FIVE-FOLD POOLED LOGISTIC EVALUATION — CODE VERSION 12")

# ============================================================
# 12 — FINAL FIVE-FOLD POOLED LOGISTIC EVALUATION
#
# Primary analysis:
# - Pool exactly one locked outer-test prediction per patient.
# - Verify all 58,491 patients and 3,032 events.
# - Calculate pooled discrimination and calibration metrics.
# - Calculate hospital-cluster bootstrap confidence intervals.
# - Store patient-level pooled predictions only in BigQuery.
# - Store only aggregate outputs on Google Drive.
#
# No model fitting or hyperparameter modification occurs here.
# No DELETE / INSERT / UPDATE / MERGE statement is used.
# ============================================================

MODEL_NAME_12 = "elastic_net_logistic"
MODEL_VERSION_12 = "core_v1_nested_cv"
BOOTSTRAP_REPLICATES_12 = 2000
BOOTSTRAP_SEED_12 = 20260723

EXPECTED_PROTOCOL_SHA_12 = (
    "400c3b4b510c836794543bc685c62fae"
    "f49df0c1caa197d78dffda8c2207952d"
)

EXPECTED_TOTAL_ROWS_12 = 58491
EXPECTED_TOTAL_EVENTS_12 = 3032
EXPECTED_TOTAL_NONEVENTS_12 = 55459
EXPECTED_TOTAL_HOSPITALS_12 = 198

EXPECTED_FOLD_STRUCTURE_12 = {
    1: {
        "rows": 11688,
        "events": 606,
        "nonevents": 11082,
        "hospitals": 40,
    },
    2: {
        "rows": 11691,
        "events": 606,
        "nonevents": 11085,
        "hospitals": 40,
    },
    3: {
        "rows": 11736,
        "events": 608,
        "nonevents": 11128,
        "hospitals": 40,
    },
    4: {
        "rows": 11688,
        "events": 606,
        "nonevents": 11082,
        "hospitals": 39,
    },
    5: {
        "rows": 11688,
        "events": 606,
        "nonevents": 11082,
        "hospitals": 39,
    },
}

# ------------------------------------------------------------
# 1. Required objects and locked protocol
# ------------------------------------------------------------

required_objects_12 = [
    "core_df_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_12 = [
    name
    for name in required_objects_12
    if name not in globals()
]

if missing_objects_12:
    raise RuntimeError(
        "Eksik çalışma nesneleri var: "
        + ", ".join(missing_objects_12)
        + ". Önce 07A ve 07B hücrelerini çalıştır."
    )

if len(core_df_07B) != EXPECTED_TOTAL_ROWS_12:
    raise RuntimeError(
        f"core_df_07B satır sayısı {len(core_df_07B)}; "
        f"beklenen {EXPECTED_TOTAL_ROWS_12}."
    )

protocol_sha_path_12 = os.path.join(
    MODEL_OUTPUT_DIR,
    "08B_locked_logistic_model_protocol_v1_SHA256.txt",
)

if not os.path.exists(protocol_sha_path_12):
    raise FileNotFoundError(
        "Kilitli lojistik model protokolü SHA dosyası bulunamadı: "
        + protocol_sha_path_12
    )

with open(
    protocol_sha_path_12,
    "r",
    encoding="utf-8",
) as file_handle:
    observed_protocol_sha_12 = file_handle.read().strip()

if observed_protocol_sha_12 != EXPECTED_PROTOCOL_SHA_12:
    raise RuntimeError(
        "Kilitli protokol SHA-256 değeri değişmiş: "
        + observed_protocol_sha_12
    )

# ------------------------------------------------------------
# 2. Source prediction tables
# ------------------------------------------------------------

source_prediction_tables_12 = {
    1: (
        f"{TARGET_DATASET}."
        "model_lr_outer_predictions_v1"
    ),
    2: (
        f"{TARGET_DATASET}."
        "model_lr_outer_predictions_outer2_v1"
    ),
    3: (
        f"{TARGET_DATASET}."
        "model_lr_outer_predictions_outer3_v1"
    ),
    4: (
        f"{TARGET_DATASET}."
        "model_lr_outer_predictions_outer4_v1"
    ),
    5: (
        f"{TARGET_DATASET}."
        "model_lr_outer_predictions_outer5_v1"
    ),
}

for fold_number, table_id in source_prediction_tables_12.items():
    try:
        client.get_table(table_id)
    except Exception as table_error:
        raise RuntimeError(
            f"Dış kat {fold_number} tahmin tablosu okunamadı: "
            f"{table_id}. Hata: {type(table_error).__name__}: "
            f"{table_error}"
        )

# ------------------------------------------------------------
# 3. Robust schema normalization
# ------------------------------------------------------------

def resolve_column_12(
    columns,
    exact_candidates,
    contains_all=None,
    contains_any=None,
    required=True,
):
    """Resolve one column without relying on a single historical schema."""

    original_columns = list(columns)
    lower_to_original = {
        str(column).lower(): column
        for column in original_columns
    }

    for candidate in exact_candidates:
        if candidate.lower() in lower_to_original:
            return lower_to_original[candidate.lower()]

    matches = []

    for original in original_columns:
        lowered = str(original).lower()

        if contains_all and not all(
            token.lower() in lowered
            for token in contains_all
        ):
            continue

        if contains_any and not any(
            token.lower() in lowered
            for token in contains_any
        ):
            continue

        if contains_all or contains_any:
            matches.append(original)

    if len(matches) == 1:
        return matches[0]

    if required:
        raise RuntimeError(
            "Gerekli sütun çözümlenemedi. "
            f"Adaylar={exact_candidates}; bulunan sütunlar="
            f"{original_columns}; sezgisel eşleşmeler={matches}"
        )

    return None


def normalize_prediction_table_12(
    raw_dataframe,
    expected_fold,
    source_table,
):
    dataframe = raw_dataframe.copy()

    id_column = resolve_column_12(
        dataframe.columns,
        exact_candidates=[
            "id_row",
            "patient_id",
            "patientunitstayid",
        ],
        contains_any=["id_row"],
    )

    label_column = resolve_column_12(
        dataframe.columns,
        exact_candidates=[
            "label_stage23",
            "label",
            "outcome",
            "y_true",
        ],
        contains_any=["label_stage23"],
    )

    outer_fold_column = resolve_column_12(
        dataframe.columns,
        exact_candidates=[
            "outer_fold",
            "fold",
        ],
        contains_all=["outer", "fold"],
        required=False,
    )

    raw_probability_column = resolve_column_12(
        dataframe.columns,
        exact_candidates=[
            "prediction_raw",
            "raw_probability",
            "probability_raw",
            "prediction_raw_probability",
            "raw_predicted_probability",
        ],
        contains_all=["raw"],
        contains_any=["prediction", "probability", "prob"],
    )

    platt_probability_column = resolve_column_12(
        dataframe.columns,
        exact_candidates=[
            "prediction_platt",
            "prediction_platt_calibrated",
            "platt_probability",
            "probability_platt",
            "calibrated_probability",
            "prediction_calibrated",
            "platt_predicted_probability",
        ],
        contains_any=["platt", "calibrated"],
    )

    model_name_column = resolve_column_12(
        dataframe.columns,
        exact_candidates=[
            "model_name",
            "model",
        ],
        contains_all=["model", "name"],
        required=False,
    )

    model_version_column = resolve_column_12(
        dataframe.columns,
        exact_candidates=[
            "model_version",
            "version",
        ],
        contains_all=["model", "version"],
        required=False,
    )

    normalized = pd.DataFrame(
        {
            "id_row": dataframe[id_column].astype(str),
            "label_stage23": pd.to_numeric(
                dataframe[label_column],
                errors="raise",
            ).astype(np.int64),
            "prediction_raw": pd.to_numeric(
                dataframe[raw_probability_column],
                errors="raise",
            ).astype(np.float64),
            "prediction_platt": pd.to_numeric(
                dataframe[platt_probability_column],
                errors="raise",
            ).astype(np.float64),
        }
    )

    if outer_fold_column is None:
        normalized["outer_fold"] = np.full(
            len(normalized),
            expected_fold,
            dtype=np.int64,
        )
    else:
        normalized["outer_fold"] = pd.to_numeric(
            dataframe[outer_fold_column],
            errors="raise",
        ).astype(np.int64)

    if model_name_column is None:
        normalized["model_name"] = MODEL_NAME_12
    else:
        normalized["model_name"] = (
            dataframe[model_name_column]
            .astype(str)
        )

    if model_version_column is None:
        normalized["model_version"] = MODEL_VERSION_12
    else:
        normalized["model_version"] = (
            dataframe[model_version_column]
            .astype(str)
        )

    normalized["source_table"] = source_table

    normalized = normalized[
        [
            "id_row",
            "outer_fold",
            "label_stage23",
            "prediction_raw",
            "prediction_platt",
            "model_name",
            "model_version",
            "source_table",
        ]
    ].copy()

    if set(normalized["outer_fold"].unique()) != {expected_fold}:
        raise RuntimeError(
            f"{source_table}: beklenen dış kat {expected_fold}; "
            f"bulunan={sorted(normalized['outer_fold'].unique())}"
        )

    return normalized

# ------------------------------------------------------------
# 4. Load and normalize all five source tables
# ------------------------------------------------------------

normalized_prediction_frames_12 = []
source_schema_rows_12 = []

for fold_number in range(1, 6):
    table_id = source_prediction_tables_12[fold_number]

    print(
        f"Loading outer-fold-{fold_number} predictions: "
        f"{table_id}"
    )

    query_job = client.query(
        f"SELECT * FROM `{table_id}`",
        location=BQ_LOCATION,
    )

    try:
        raw_fold_df = query_job.to_dataframe(
            create_bqstorage_client=True
        )
        load_method = "BigQuery Storage API"
    except Exception:
        raw_fold_df = query_job.to_dataframe(
            create_bqstorage_client=False
        )
        load_method = "Standard BigQuery API"

    source_schema_rows_12.append(
        {
            "outer_fold": fold_number,
            "source_table": table_id,
            "source_rows": len(raw_fold_df),
            "source_columns": " | ".join(
                map(str, raw_fold_df.columns)
            ),
            "load_method": load_method,
        }
    )

    normalized_fold_df = normalize_prediction_table_12(
        raw_dataframe=raw_fold_df,
        expected_fold=fold_number,
        source_table=table_id,
    )

    normalized_prediction_frames_12.append(
        normalized_fold_df
    )

    del raw_fold_df, normalized_fold_df

source_schema_summary_12 = pd.DataFrame(
    source_schema_rows_12
)

pooled_predictions_12 = pd.concat(
    normalized_prediction_frames_12,
    ignore_index=True,
)

# ------------------------------------------------------------
# 5. Strict patient-level integrity checks
# ------------------------------------------------------------

if len(pooled_predictions_12) != EXPECTED_TOTAL_ROWS_12:
    raise RuntimeError(
        f"Toplam tahmin satırı {len(pooled_predictions_12)}; "
        f"beklenen {EXPECTED_TOTAL_ROWS_12}."
    )

if pooled_predictions_12["id_row"].duplicated().any():
    duplicated_count = int(
        pooled_predictions_12["id_row"]
        .duplicated()
        .sum()
    )

    raise RuntimeError(
        f"Pooled dış test tahminlerinde {duplicated_count} "
        "yinelenen id_row bulundu."
    )

if set(
    pooled_predictions_12["outer_fold"].unique()
) != {1, 2, 3, 4, 5}:
    raise RuntimeError(
        "Pooled tahminlerde dış kat değerleri 1–5 değil."
    )

if pooled_predictions_12[
    [
        "prediction_raw",
        "prediction_platt",
    ]
].isna().any().any():
    raise RuntimeError(
        "Pooled tahminlerde eksik olasılık değeri var."
    )

for probability_column in [
    "prediction_raw",
    "prediction_platt",
]:
    if not pooled_predictions_12[
        probability_column
    ].between(0, 1).all():
        raise RuntimeError(
            f"{probability_column} içinde 0–1 dışında değer var."
        )

if int(
    pooled_predictions_12["label_stage23"].sum()
) != EXPECTED_TOTAL_EVENTS_12:
    raise RuntimeError(
        "Pooled tahminlerde toplam olay sayısı 3.032 değil."
    )

# ------------------------------------------------------------
# 6. Audit against the locked core matrix
# ------------------------------------------------------------

required_core_columns_12 = {
    "id_row",
    "outer_fold",
    "label_stage23",
    "group_hospital",
}

missing_core_columns_12 = (
    required_core_columns_12
    - set(core_df_07B.columns)
)

if missing_core_columns_12:
    raise RuntimeError(
        "core_df_07B içinde gerekli denetim sütunları eksik: "
        + ", ".join(sorted(missing_core_columns_12))
    )

core_audit_12 = core_df_07B[
    [
        "id_row",
        "outer_fold",
        "label_stage23",
        "group_hospital",
    ]
].copy()

core_audit_12["id_row"] = (
    core_audit_12["id_row"].astype(str)
)

core_audit_12["outer_fold"] = pd.to_numeric(
    core_audit_12["outer_fold"],
    errors="raise",
).astype(np.int64)

core_audit_12["label_stage23"] = pd.to_numeric(
    core_audit_12["label_stage23"],
    errors="raise",
).astype(np.int64)

core_audit_12["group_hospital"] = (
    core_audit_12["group_hospital"].astype(str)
)

if core_audit_12["id_row"].duplicated().any():
    raise RuntimeError(
        "core_df_07B içinde yinelenen id_row bulundu."
    )

pooled_audited_12 = pooled_predictions_12.merge(
    core_audit_12,
    on="id_row",
    how="inner",
    suffixes=("_prediction", "_core"),
    validate="one_to_one",
)

if len(pooled_audited_12) != EXPECTED_TOTAL_ROWS_12:
    raise RuntimeError(
        "Prediction–core birleşiminde 58.491 hasta bulunamadı."
    )

fold_mismatch_count_12 = int(
    (
        pooled_audited_12[
            "outer_fold_prediction"
        ]
        != pooled_audited_12[
            "outer_fold_core"
        ]
    ).sum()
)

label_mismatch_count_12 = int(
    (
        pooled_audited_12[
            "label_stage23_prediction"
        ]
        != pooled_audited_12[
            "label_stage23_core"
        ]
    ).sum()
)

if fold_mismatch_count_12 != 0:
    raise RuntimeError(
        f"{fold_mismatch_count_12} hastada dış kat uyuşmazlığı var."
    )

if label_mismatch_count_12 != 0:
    raise RuntimeError(
        f"{label_mismatch_count_12} hastada outcome etiketi uyuşmazlığı var."
    )

pooled_audited_12 = pooled_audited_12.rename(
    columns={
        "outer_fold_prediction": "outer_fold",
        "label_stage23_prediction": "label_stage23",
    }
)

pooled_audited_12 = pooled_audited_12.drop(
    columns=[
        "outer_fold_core",
        "label_stage23_core",
    ]
)

if pooled_audited_12[
    "group_hospital"
].nunique() != EXPECTED_TOTAL_HOSPITALS_12:
    raise RuntimeError(
        "Pooled veri 198 farklı hastane içermiyor."
    )

hospital_fold_counts_12 = (
    pooled_audited_12
    .groupby("group_hospital")[
        "outer_fold"
    ]
    .nunique()
)

if (
    hospital_fold_counts_12 > 1
).any():
    raise RuntimeError(
        "En az bir hastane birden fazla dış test katında bulundu."
    )

# ------------------------------------------------------------
# 7. Fold-level structure verification
# ------------------------------------------------------------

fold_integrity_rows_12 = []

for fold_number in range(1, 6):
    fold_part = pooled_audited_12.loc[
        pooled_audited_12["outer_fold"]
        == fold_number
    ]

    expected = EXPECTED_FOLD_STRUCTURE_12[
        fold_number
    ]

    actual = {
        "rows": len(fold_part),
        "events": int(
            fold_part["label_stage23"].sum()
        ),
        "nonevents": int(
            len(fold_part)
            - fold_part["label_stage23"].sum()
        ),
        "hospitals": int(
            fold_part["group_hospital"].nunique()
        ),
    }

    for metric_name, expected_value in expected.items():
        if actual[metric_name] != expected_value:
            raise RuntimeError(
                f"Dış kat {fold_number}, {metric_name}: "
                f"bulunan={actual[metric_name]}, "
                f"beklenen={expected_value}"
            )

    fold_integrity_rows_12.append(
        {
            "outer_fold": fold_number,
            "prediction_rows": actual["rows"],
            "distinct_patients": fold_part[
                "id_row"
            ].nunique(),
            "hospitals": actual["hospitals"],
            "events": actual["events"],
            "nonevents": actual["nonevents"],
            "minimum_raw_probability": float(
                fold_part[
                    "prediction_raw"
                ].min()
            ),
            "maximum_raw_probability": float(
                fold_part[
                    "prediction_raw"
                ].max()
            ),
            "minimum_platt_probability": float(
                fold_part[
                    "prediction_platt"
                ].min()
            ),
            "maximum_platt_probability": float(
                fold_part[
                    "prediction_platt"
                ].max()
            ),
        }
    )

fold_integrity_summary_12 = pd.DataFrame(
    fold_integrity_rows_12
)

# ------------------------------------------------------------
# 8. Create one secure all-five-fold BigQuery table
# ------------------------------------------------------------

pooled_secure_table_id_12 = (
    f"{TARGET_DATASET}."
    "model_lr_outer_predictions_all5_v1"
)

pooled_secure_df_12 = pooled_audited_12[
    [
        "id_row",
        "outer_fold",
        "label_stage23",
        "prediction_raw",
        "prediction_platt",
        "model_name",
        "model_version",
        "group_hospital",
    ]
].copy()

pooled_secure_df_12["id_row"] = (
    pooled_secure_df_12["id_row"].astype(str)
)

pooled_secure_df_12["group_hospital"] = (
    pooled_secure_df_12[
        "group_hospital"
    ].astype(str)
)

pooled_load_config_12 = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField(
            "id_row",
            "STRING",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "outer_fold",
            "INTEGER",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "label_stage23",
            "INTEGER",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "prediction_raw",
            "FLOAT",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "prediction_platt",
            "FLOAT",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "model_name",
            "STRING",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "model_version",
            "STRING",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "group_hospital",
            "STRING",
            mode="REQUIRED",
        ),
    ],
    write_disposition=(
        bigquery.WriteDisposition.WRITE_TRUNCATE
    ),
)

print(
    "\nUploading secure pooled five-fold prediction table:"
)
print(pooled_secure_table_id_12)

client.load_table_from_dataframe(
    pooled_secure_df_12,
    pooled_secure_table_id_12,
    job_config=pooled_load_config_12,
    location=BQ_LOCATION,
).result()

SQL_VERIFY_POOLED_TABLE_12 = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_patients,
  COUNT(DISTINCT group_hospital) AS hospitals,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL)
    AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL)
    AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0
    OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0
    OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw) AS minimum_raw_probability,
  MAX(prediction_raw) AS maximum_raw_probability,
  MIN(prediction_platt) AS minimum_platt_probability,
  MAX(prediction_platt) AS maximum_platt_probability
FROM `{pooled_secure_table_id_12}`;
"""

pooled_bigquery_verification_12 = (
    client.query(
        SQL_VERIFY_POOLED_TABLE_12,
        location=BQ_LOCATION,
    )
    .to_dataframe()
)

verification_row_12 = (
    pooled_bigquery_verification_12.iloc[0]
)

expected_verification_12 = {
    "prediction_rows": EXPECTED_TOTAL_ROWS_12,
    "distinct_patients": EXPECTED_TOTAL_ROWS_12,
    "hospitals": EXPECTED_TOTAL_HOSPITALS_12,
    "outer_folds": 5,
    "events": EXPECTED_TOTAL_EVENTS_12,
    "nonevents": EXPECTED_TOTAL_NONEVENTS_12,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in expected_verification_12.items():
    actual_value = int(
        verification_row_12[field]
    )

    if actual_value != expected_value:
        raise RuntimeError(
            f"Pooled BigQuery doğrulaması, {field}: "
            f"bulunan={actual_value}, beklenen={expected_value}"
        )

# ------------------------------------------------------------
# 9. Metric helpers
# ------------------------------------------------------------

def probability_logit_12(probabilities):
    clipped = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )

    return np.log(
        clipped / (1 - clipped)
    ).reshape(-1, 1)


def calibration_intercept_slope_12(
    y_true,
    probabilities,
):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )

    calibration_model.fit(
        probability_logit_12(probabilities),
        y_true,
    )

    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )


def probability_metrics_12(
    y_true,
    probabilities,
):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    intercept, slope = (
        calibration_intercept_slope_12(
            y_true,
            probabilities,
        )
    )

    return {
        "auroc": float(
            roc_auc_score(
                y_true,
                probabilities,
            )
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1],
            )
        ),
        "mean_predicted_risk": float(
            probabilities.mean()
        ),
        "observed_event_rate": float(
            np.mean(y_true)
        ),
        "calibration_intercept": intercept,
        "calibration_slope": slope,
    }

# ------------------------------------------------------------
# 10. Per-fold and pooled point estimates
# ------------------------------------------------------------

y_all_12 = pooled_audited_12[
    "label_stage23"
].to_numpy(dtype=np.int8)

raw_all_12 = pooled_audited_12[
    "prediction_raw"
].to_numpy(dtype=float)

platt_all_12 = pooled_audited_12[
    "prediction_platt"
].to_numpy(dtype=float)

fold_metric_rows_12 = []

for fold_number in range(1, 6):
    fold_mask = (
        pooled_audited_12["outer_fold"]
        .to_numpy(dtype=int)
        == fold_number
    )

    fold_y = y_all_12[fold_mask]

    for probability_type, probability_vector in [
        ("raw", raw_all_12),
        ("platt_calibrated", platt_all_12),
    ]:
        metrics = probability_metrics_12(
            fold_y,
            probability_vector[fold_mask],
        )

        fold_metric_rows_12.append(
            {
                "outer_fold": fold_number,
                "probability_type": probability_type,
                "patients": int(fold_mask.sum()),
                "events": int(fold_y.sum()),
                **metrics,
            }
        )

fold_metrics_12 = pd.DataFrame(
    fold_metric_rows_12
)

macro_summary_rows_12 = []

for probability_type in [
    "raw",
    "platt_calibrated",
]:
    subset = fold_metrics_12.loc[
        fold_metrics_12[
            "probability_type"
        ] == probability_type
    ]

    for metric_name in [
        "auroc",
        "auprc",
        "brier",
        "log_loss",
        "mean_predicted_risk",
        "calibration_intercept",
        "calibration_slope",
    ]:
        values = subset[
            metric_name
        ].to_numpy(dtype=float)

        macro_summary_rows_12.append(
            {
                "probability_type": probability_type,
                "metric": metric_name,
                "fold_mean": float(np.mean(values)),
                "fold_sd": float(
                    np.std(values, ddof=1)
                ),
                "fold_minimum": float(
                    np.min(values)
                ),
                "fold_maximum": float(
                    np.max(values)
                ),
            }
        )

macro_fold_summary_12 = pd.DataFrame(
    macro_summary_rows_12
)

pooled_point_rows_12 = []

for probability_type, probability_vector in [
    ("raw", raw_all_12),
    ("platt_calibrated", platt_all_12),
]:
    metrics = probability_metrics_12(
        y_all_12,
        probability_vector,
    )

    pooled_point_rows_12.append(
        {
            "model": MODEL_NAME_12,
            "probability_type": probability_type,
            "patients": EXPECTED_TOTAL_ROWS_12,
            "hospitals": EXPECTED_TOTAL_HOSPITALS_12,
            "events": EXPECTED_TOTAL_EVENTS_12,
            "nonevents": EXPECTED_TOTAL_NONEVENTS_12,
            **metrics,
        }
    )

pooled_point_metrics_12 = pd.DataFrame(
    pooled_point_rows_12
)

# ------------------------------------------------------------
# 11. Hospital-cluster bootstrap confidence intervals
# ------------------------------------------------------------

print(
    f"\nRunning {BOOTSTRAP_REPLICATES_12:,} "
    "hospital-cluster bootstrap replicates..."
)

hospital_categorical_12 = pd.Categorical(
    pooled_audited_12["group_hospital"]
)

hospital_codes_12 = (
    hospital_categorical_12.codes.astype(int)
)

hospital_labels_12 = list(
    hospital_categorical_12.categories
)

n_hospitals_12 = len(hospital_labels_12)

if n_hospitals_12 != EXPECTED_TOTAL_HOSPITALS_12:
    raise RuntimeError(
        "Bootstrap öncesi hastane sayısı 198 değil."
    )

rng_12 = np.random.default_rng(
    BOOTSTRAP_SEED_12
)

bootstrap_rows_12 = []

for bootstrap_index in range(
    BOOTSTRAP_REPLICATES_12
):
    sampled_hospital_codes = rng_12.integers(
        low=0,
        high=n_hospitals_12,
        size=n_hospitals_12,
    )

    hospital_multiplicity = np.bincount(
        sampled_hospital_codes,
        minlength=n_hospitals_12,
    )

    patient_weights = hospital_multiplicity[
        hospital_codes_12
    ].astype(float)

    positive_weight = float(
        patient_weights[y_all_12 == 1].sum()
    )

    negative_weight = float(
        patient_weights[y_all_12 == 0].sum()
    )

    if positive_weight <= 0 or negative_weight <= 0:
        continue

    bootstrap_rows_12.append(
        {
            "bootstrap_replicate": (
                bootstrap_index + 1
            ),
            "sampled_hospital_draws": (
                n_hospitals_12
            ),
            "unique_sampled_hospitals": int(
                np.count_nonzero(
                    hospital_multiplicity
                )
            ),
            "weighted_patients": float(
                patient_weights.sum()
            ),
            "weighted_events": positive_weight,
            "auroc": float(
                roc_auc_score(
                    y_all_12,
                    raw_all_12,
                    sample_weight=patient_weights,
                )
            ),
            "auprc": float(
                average_precision_score(
                    y_all_12,
                    raw_all_12,
                    sample_weight=patient_weights,
                )
            ),
            "brier_raw": float(
                brier_score_loss(
                    y_all_12,
                    raw_all_12,
                    sample_weight=patient_weights,
                )
            ),
            "brier_platt": float(
                brier_score_loss(
                    y_all_12,
                    platt_all_12,
                    sample_weight=patient_weights,
                )
            ),
            "log_loss_raw": float(
                log_loss(
                    y_all_12,
                    raw_all_12,
                    labels=[0, 1],
                    sample_weight=patient_weights,
                )
            ),
            "log_loss_platt": float(
                log_loss(
                    y_all_12,
                    platt_all_12,
                    labels=[0, 1],
                    sample_weight=patient_weights,
                )
            ),
        }
    )

    if (
        (bootstrap_index + 1) % 100 == 0
        or bootstrap_index == 0
    ):
        print(
            "  Completed bootstrap replicate",
            bootstrap_index + 1,
            "/",
            BOOTSTRAP_REPLICATES_12,
        )

bootstrap_replicates_12 = pd.DataFrame(
    bootstrap_rows_12
)

if len(bootstrap_replicates_12) < (
    BOOTSTRAP_REPLICATES_12 * 0.99
):
    raise RuntimeError(
        "Hastane bootstrap tekrarlarının %99'undan azı tamamlandı."
    )

bootstrap_ci_rows_12 = []

bootstrap_metric_mapping_12 = {
    "auroc": (
        "raw_and_platt",
        float(
            pooled_point_metrics_12.loc[
                pooled_point_metrics_12[
                    "probability_type"
                ] == "raw",
                "auroc",
            ].iloc[0]
        ),
    ),
    "auprc": (
        "raw_and_platt",
        float(
            pooled_point_metrics_12.loc[
                pooled_point_metrics_12[
                    "probability_type"
                ] == "raw",
                "auprc",
            ].iloc[0]
        ),
    ),
    "brier_raw": (
        "raw",
        float(
            pooled_point_metrics_12.loc[
                pooled_point_metrics_12[
                    "probability_type"
                ] == "raw",
                "brier",
            ].iloc[0]
        ),
    ),
    "brier_platt": (
        "platt_calibrated",
        float(
            pooled_point_metrics_12.loc[
                pooled_point_metrics_12[
                    "probability_type"
                ] == "platt_calibrated",
                "brier",
            ].iloc[0]
        ),
    ),
    "log_loss_raw": (
        "raw",
        float(
            pooled_point_metrics_12.loc[
                pooled_point_metrics_12[
                    "probability_type"
                ] == "raw",
                "log_loss",
            ].iloc[0]
        ),
    ),
    "log_loss_platt": (
        "platt_calibrated",
        float(
            pooled_point_metrics_12.loc[
                pooled_point_metrics_12[
                    "probability_type"
                ] == "platt_calibrated",
                "log_loss",
            ].iloc[0]
        ),
    ),
}

for metric_column, (
    probability_type,
    point_estimate,
) in bootstrap_metric_mapping_12.items():
    values = bootstrap_replicates_12[
        metric_column
    ].dropna().to_numpy(dtype=float)

    bootstrap_ci_rows_12.append(
        {
            "metric": metric_column,
            "probability_type": probability_type,
            "point_estimate": point_estimate,
            "bootstrap_replicates": len(values),
            "bootstrap_mean": float(
                np.mean(values)
            ),
            "bootstrap_standard_error": float(
                np.std(values, ddof=1)
            ),
            "ci_95_lower": float(
                np.quantile(values, 0.025)
            ),
            "ci_95_upper": float(
                np.quantile(values, 0.975)
            ),
            "bootstrap_unit": "hospital",
            "bootstrap_seed": BOOTSTRAP_SEED_12,
        }
    )

bootstrap_confidence_intervals_12 = pd.DataFrame(
    bootstrap_ci_rows_12
)

# ------------------------------------------------------------
# 12. Quantile calibration tables
# ------------------------------------------------------------

def calibration_quantiles_12(
    dataframe,
    probability_column,
    probability_type,
    requested_bins=10,
):
    working = dataframe[
        [
            "label_stage23",
            probability_column,
        ]
    ].copy()

    working["risk_group"] = pd.qcut(
        working[probability_column],
        q=requested_bins,
        labels=False,
        duplicates="drop",
    )

    summary = (
        working
        .groupby(
            "risk_group",
            observed=True,
        )
        .agg(
            patients=(
                "label_stage23",
                "size",
            ),
            events=(
                "label_stage23",
                "sum",
            ),
            mean_predicted_risk=(
                probability_column,
                "mean",
            ),
            minimum_predicted_risk=(
                probability_column,
                "min",
            ),
            maximum_predicted_risk=(
                probability_column,
                "max",
            ),
        )
        .reset_index()
    )

    summary["observed_event_rate"] = (
        summary["events"]
        / summary["patients"]
    )

    summary["probability_type"] = probability_type
    summary["requested_bins"] = requested_bins
    summary["actual_bins"] = len(summary)

    return summary[
        [
            "probability_type",
            "requested_bins",
            "actual_bins",
            "risk_group",
            "patients",
            "events",
            "mean_predicted_risk",
            "observed_event_rate",
            "minimum_predicted_risk",
            "maximum_predicted_risk",
        ]
    ]


calibration_quantile_summary_12 = pd.concat(
    [
        calibration_quantiles_12(
            pooled_audited_12,
            "prediction_raw",
            "raw",
            requested_bins=10,
        ),
        calibration_quantiles_12(
            pooled_audited_12,
            "prediction_platt",
            "platt_calibrated",
            requested_bins=10,
        ),
    ],
    ignore_index=True,
)

# ------------------------------------------------------------
# 13. Selected-model consistency audit
# ------------------------------------------------------------

selected_model_paths_12 = {
    1: os.path.join(
        MODEL_OUTPUT_DIR,
        "08A_logistic_selected_model_outer1.csv",
    ),
    2: os.path.join(
        MODEL_OUTPUT_DIR,
        "08C2B_logistic_selected_model_outer2.csv",
    ),
    3: os.path.join(
        MODEL_OUTPUT_DIR,
        "09B_logistic_selected_model_outer3.csv",
    ),
    4: os.path.join(
        MODEL_OUTPUT_DIR,
        "10B_logistic_selected_model_outer4.csv",
    ),
    5: os.path.join(
        MODEL_OUTPUT_DIR,
        "11B_logistic_selected_model_outer5.csv",
    ),
}

selected_model_rows_12 = []

for fold_number, file_path in selected_model_paths_12.items():
    if not os.path.exists(file_path):
        selected_model_rows_12.append(
            {
                "outer_fold": fold_number,
                "selected_candidate": np.nan,
                "selected_C": np.nan,
                "selected_l1_ratio": np.nan,
                "selection_file_found": False,
                "selection_file": file_path,
            }
        )
        continue

    selected_df = pd.read_csv(file_path)

    if len(selected_df) != 1:
        raise RuntimeError(
            f"Dış kat {fold_number} seçim dosyasında "
            "tam olarak bir satır yok."
        )

    row = selected_df.iloc[0]

    candidate_value = (
        row.get("selected_candidate")
        if "selected_candidate" in selected_df.columns
        else row.get("candidate_id")
    )

    C_value = (
        row.get("selected_C")
        if "selected_C" in selected_df.columns
        else row.get("C")
    )

    l1_value = (
        row.get("selected_l1_ratio")
        if "selected_l1_ratio" in selected_df.columns
        else row.get("l1_ratio")
    )

    selected_model_rows_12.append(
        {
            "outer_fold": fold_number,
            "selected_candidate": str(
                candidate_value
            ),
            "selected_C": float(C_value),
            "selected_l1_ratio": float(l1_value),
            "selection_file_found": True,
            "selection_file": file_path,
        }
    )

selected_model_consistency_12 = pd.DataFrame(
    selected_model_rows_12
)

found_selection_rows_12 = selected_model_consistency_12.loc[
    selected_model_consistency_12[
        "selection_file_found"
    ]
]

if len(found_selection_rows_12) == 5:
    if not (
        found_selection_rows_12[
            "selected_candidate"
        ] == "LR06"
    ).all():
        raise RuntimeError(
            "Beş dış katın tamamında LR06 seçilmemiş."
        )

    if not np.allclose(
        found_selection_rows_12[
            "selected_C"
        ].to_numpy(dtype=float),
        0.30,
    ):
        raise RuntimeError(
            "Beş dış katın tamamında C=0.30 seçilmemiş."
        )

    if not np.allclose(
        found_selection_rows_12[
            "selected_l1_ratio"
        ].to_numpy(dtype=float),
        0.50,
    ):
        raise RuntimeError(
            "Beş dış katın tamamında l1_ratio=0.50 seçilmemiş."
        )

# ------------------------------------------------------------
# 14. Save aggregate outputs only
# ------------------------------------------------------------

source_schema_path_12 = os.path.join(
    MODEL_OUTPUT_DIR,
    "12A_source_prediction_schema_audit.csv",
)

fold_integrity_path_12 = os.path.join(
    MODEL_OUTPUT_DIR,
    "12A_five_fold_prediction_integrity.csv",
)

fold_metrics_path_12 = os.path.join(
    MODEL_OUTPUT_DIR,
    "12B_logistic_outer_fold_metrics.csv",
)

macro_summary_path_12 = os.path.join(
    MODEL_OUTPUT_DIR,
    "12B_logistic_macro_fold_summary.csv",
)

pooled_metrics_path_12 = os.path.join(
    MODEL_OUTPUT_DIR,
    "12B_logistic_pooled_point_metrics.csv",
)

bootstrap_replicates_path_12 = os.path.join(
    MODEL_OUTPUT_DIR,
    "12C_logistic_hospital_bootstrap_replicates.csv",
)

bootstrap_ci_path_12 = os.path.join(
    MODEL_OUTPUT_DIR,
    "12C_logistic_hospital_bootstrap_confidence_intervals.csv",
)

calibration_quantiles_path_12 = os.path.join(
    MODEL_OUTPUT_DIR,
    "12D_logistic_pooled_calibration_deciles.csv",
)

selected_consistency_path_12 = os.path.join(
    MODEL_OUTPUT_DIR,
    "12D_logistic_selected_model_consistency.csv",
)

final_manifest_path_12 = os.path.join(
    MODEL_OUTPUT_DIR,
    "12E_logistic_final_pooled_evaluation_manifest.json",
)

final_manifest_sha_path_12 = os.path.join(
    MODEL_OUTPUT_DIR,
    "12E_logistic_final_pooled_evaluation_manifest_SHA256.txt",
)

source_schema_summary_12.to_csv(
    source_schema_path_12,
    index=False,
)

fold_integrity_summary_12.to_csv(
    fold_integrity_path_12,
    index=False,
)

fold_metrics_12.to_csv(
    fold_metrics_path_12,
    index=False,
)

macro_fold_summary_12.to_csv(
    macro_summary_path_12,
    index=False,
)

pooled_point_metrics_12.to_csv(
    pooled_metrics_path_12,
    index=False,
)

bootstrap_replicates_12.to_csv(
    bootstrap_replicates_path_12,
    index=False,
)

bootstrap_confidence_intervals_12.to_csv(
    bootstrap_ci_path_12,
    index=False,
)

calibration_quantile_summary_12.to_csv(
    calibration_quantiles_path_12,
    index=False,
)

selected_model_consistency_12.to_csv(
    selected_consistency_path_12,
    index=False,
)

final_manifest_12 = {
    "analysis_stage": (
        "final_five_fold_pooled_logistic_evaluation"
    ),
    "model_family": (
        "elastic_net_logistic_regression"
    ),
    "model_name": MODEL_NAME_12,
    "model_version": MODEL_VERSION_12,
    "patients": EXPECTED_TOTAL_ROWS_12,
    "hospitals": EXPECTED_TOTAL_HOSPITALS_12,
    "events": EXPECTED_TOTAL_EVENTS_12,
    "nonevents": EXPECTED_TOTAL_NONEVENTS_12,
    "outer_folds": 5,
    "source_prediction_tables": (
        source_prediction_tables_12
    ),
    "secure_pooled_prediction_table": (
        pooled_secure_table_id_12
    ),
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_12,
    "primary_performance_estimator": (
        "pooled locked outer-test predictions"
    ),
    "confidence_interval_method": (
        "percentile hospital-cluster bootstrap"
    ),
    "bootstrap_replicates_requested": (
        BOOTSTRAP_REPLICATES_12
    ),
    "bootstrap_replicates_completed": int(
        len(bootstrap_replicates_12)
    ),
    "bootstrap_seed": BOOTSTRAP_SEED_12,
    "patient_level_prediction_written_to_drive": False,
    "aggregate_output_files": {
        "source_schema_audit": source_schema_path_12,
        "fold_integrity": fold_integrity_path_12,
        "fold_metrics": fold_metrics_path_12,
        "macro_fold_summary": macro_summary_path_12,
        "pooled_point_metrics": pooled_metrics_path_12,
        "bootstrap_replicates": bootstrap_replicates_path_12,
        "bootstrap_confidence_intervals": bootstrap_ci_path_12,
        "calibration_deciles": calibration_quantiles_path_12,
        "selected_model_consistency": selected_consistency_path_12,
    },
}

with open(
    final_manifest_path_12,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        final_manifest_12,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(
    final_manifest_path_12,
    "rb",
) as file_handle:
    final_manifest_sha_12 = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    final_manifest_sha_path_12,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(
        final_manifest_sha_12 + "\n"
    )

# ------------------------------------------------------------
# 15. Display final aggregate results
# ------------------------------------------------------------

print("\n12 SOURCE TABLE SCHEMA AUDIT")
display(source_schema_summary_12)

print("\n12 FIVE-FOLD PREDICTION INTEGRITY")
display(fold_integrity_summary_12)

print("\n12 SELECTED MODEL CONSISTENCY")
display(selected_model_consistency_12)

print("\n12 OUTER-FOLD METRICS")
display(fold_metrics_12)

print("\n12 MACRO FOLD SUMMARY")
display(macro_fold_summary_12)

print("\n12 PRIMARY POOLED POINT METRICS")
display(pooled_point_metrics_12)

print("\n12 HOSPITAL-CLUSTER BOOTSTRAP CONFIDENCE INTERVALS")
display(bootstrap_confidence_intervals_12)

print("\n12 POOLED CALIBRATION DECILES")
display(calibration_quantile_summary_12)

print("\n12 BIGQUERY POOLED PREDICTION VERIFICATION")
display(pooled_bigquery_verification_12)

print("\nFinal pooled evaluation manifest SHA-256:")
print(final_manifest_sha_12)

print("\nSaved aggregate outputs:")
for output_path in [
    source_schema_path_12,
    fold_integrity_path_12,
    fold_metrics_path_12,
    macro_summary_path_12,
    pooled_metrics_path_12,
    bootstrap_replicates_path_12,
    bootstrap_ci_path_12,
    calibration_quantiles_path_12,
    selected_consistency_path_12,
    final_manifest_path_12,
    final_manifest_sha_path_12,
]:
    print(output_path)

print(
    "\n12 PASS: All five locked outer-test folds were "
    "pooled and evaluated."
)

print(
    "Primary metrics were calculated from exactly one "
    "outer-test prediction per patient."
)

print(
    "Confidence intervals were estimated by resampling "
    "hospitals as clusters."
)

print(
    "Patient-level pooled predictions were stored only "
    "in secure BigQuery."
)

print(
    "No patient-level prediction file was written to "
    "Google Drive."
)

In [ ]:
import os
import json
import hashlib

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from IPython.display import display

print("STARTING CORRECTED RAW/PLATT HOSPITAL BOOTSTRAP — CODE VERSION 12R")

# ============================================================
# 12R — CORRECTED HOSPITAL-CLUSTER BOOTSTRAP
#
# Purpose:
# - Recalculate discrimination confidence intervals separately
#   for raw and fold-specific Platt-calibrated probabilities.
# - Preserve the valid Brier and log-loss bootstrap analyses.
# - Use the already secured pooled five-fold BigQuery table.
# - Write no patient-level data to Google Drive.
#
# Why this correction is needed:
# - Platt scaling is monotonic within each outer fold.
# - Because each fold has different Platt coefficients, pooled
#   cross-fold ranking can change slightly.
# - Therefore pooled AUROC/AUPRC for raw and Platt probabilities
#   should have separate point estimates and confidence intervals.
# ============================================================

BOOTSTRAP_REPLICATES_12R = 2000
BOOTSTRAP_SEED_12R = 20260723
EXPECTED_ROWS_12R = 58491
EXPECTED_EVENTS_12R = 3032
EXPECTED_NONEVENTS_12R = 55459
EXPECTED_HOSPITALS_12R = 198
EXPECTED_FOLDS_12R = 5

required_objects_12R = [
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
    "MODEL_OUTPUT_DIR",
]

missing_objects_12R = [
    name
    for name in required_objects_12R
    if name not in globals()
]

if missing_objects_12R:
    raise RuntimeError(
        "Eksik çalışma nesneleri var: "
        + ", ".join(missing_objects_12R)
        + ". Önce 07A hücresini çalıştır."
    )

POOLED_TABLE_12R = (
    f"{TARGET_DATASET}."
    "model_lr_outer_predictions_all5_v1"
)

SQL_LOAD_12R = f"""
SELECT
  id_row,
  outer_fold,
  label_stage23,
  prediction_raw,
  prediction_platt,
  group_hospital
FROM `{POOLED_TABLE_12R}`
ORDER BY outer_fold, id_row;
"""

print("Loading secure pooled five-fold prediction table:")
print(POOLED_TABLE_12R)

query_job_12R = client.query(
    SQL_LOAD_12R,
    location=BQ_LOCATION,
)

try:
    pooled_12R = query_job_12R.to_dataframe(
        create_bqstorage_client=True
    )
    load_method_12R = "BigQuery Storage API"
except Exception as fast_path_error_12R:
    print(
        "Storage API unavailable; using standard "
        "BigQuery download."
    )
    print(
        "Fast-path message:",
        type(fast_path_error_12R).__name__,
    )
    pooled_12R = query_job_12R.to_dataframe(
        create_bqstorage_client=False
    )
    load_method_12R = "Standard BigQuery API"

pooled_12R["id_row"] = pooled_12R["id_row"].astype(str)
pooled_12R["group_hospital"] = pooled_12R[
    "group_hospital"
].astype(str)

for column in [
    "outer_fold",
    "label_stage23",
]:
    pooled_12R[column] = pd.to_numeric(
        pooled_12R[column],
        errors="raise",
    ).astype(int)

for column in [
    "prediction_raw",
    "prediction_platt",
]:
    pooled_12R[column] = pd.to_numeric(
        pooled_12R[column],
        errors="raise",
    ).astype(float)

# ------------------------------------------------------------
# 1. Integrity checks
# ------------------------------------------------------------

integrity_values_12R = {
    "prediction_rows": len(pooled_12R),
    "distinct_patients": pooled_12R["id_row"].nunique(),
    "hospitals": pooled_12R["group_hospital"].nunique(),
    "outer_folds": pooled_12R["outer_fold"].nunique(),
    "events": int(pooled_12R["label_stage23"].sum()),
    "nonevents": int(
        len(pooled_12R)
        - pooled_12R["label_stage23"].sum()
    ),
    "duplicate_patients": int(
        pooled_12R["id_row"].duplicated().sum()
    ),
    "missing_raw": int(
        pooled_12R["prediction_raw"].isna().sum()
    ),
    "missing_platt": int(
        pooled_12R["prediction_platt"].isna().sum()
    ),
    "invalid_raw": int(
        (~pooled_12R["prediction_raw"].between(0, 1)).sum()
    ),
    "invalid_platt": int(
        (~pooled_12R["prediction_platt"].between(0, 1)).sum()
    ),
}

expected_integrity_12R = {
    "prediction_rows": EXPECTED_ROWS_12R,
    "distinct_patients": EXPECTED_ROWS_12R,
    "hospitals": EXPECTED_HOSPITALS_12R,
    "outer_folds": EXPECTED_FOLDS_12R,
    "events": EXPECTED_EVENTS_12R,
    "nonevents": EXPECTED_NONEVENTS_12R,
    "duplicate_patients": 0,
    "missing_raw": 0,
    "missing_platt": 0,
    "invalid_raw": 0,
    "invalid_platt": 0,
}

for field, expected_value in expected_integrity_12R.items():
    actual_value = integrity_values_12R[field]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: bulunan={actual_value}, "
            f"beklenen={expected_value}"
        )

integrity_summary_12R = pd.DataFrame(
    {
        "metric": list(integrity_values_12R.keys())
        + ["load_method"],
        "value": list(integrity_values_12R.values())
        + [load_method_12R],
    }
)

# ------------------------------------------------------------
# 2. Point estimates
# ------------------------------------------------------------

y_12R = pooled_12R["label_stage23"].to_numpy(
    dtype=np.int8
)
raw_12R = pooled_12R["prediction_raw"].to_numpy(
    dtype=float
)
platt_12R = pooled_12R["prediction_platt"].to_numpy(
    dtype=float
)


def metric_point_estimates_12R(
    y_true,
    probabilities,
):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(
            roc_auc_score(y_true, probabilities)
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1],
            )
        ),
        "mean_predicted_risk": float(
            probabilities.mean()
        ),
        "observed_event_rate": float(
            np.mean(y_true)
        ),
    }

raw_point_12R = metric_point_estimates_12R(
    y_12R,
    raw_12R,
)
platt_point_12R = metric_point_estimates_12R(
    y_12R,
    platt_12R,
)

point_metrics_12R = pd.DataFrame(
    [
        {
            "probability_type": "raw",
            "patients": EXPECTED_ROWS_12R,
            "hospitals": EXPECTED_HOSPITALS_12R,
            "events": EXPECTED_EVENTS_12R,
            **raw_point_12R,
        },
        {
            "probability_type": "platt_calibrated",
            "patients": EXPECTED_ROWS_12R,
            "hospitals": EXPECTED_HOSPITALS_12R,
            "events": EXPECTED_EVENTS_12R,
            **platt_point_12R,
        },
    ]
)

# ------------------------------------------------------------
# 3. Hospital-cluster bootstrap
# ------------------------------------------------------------

hospital_categorical_12R = pd.Categorical(
    pooled_12R["group_hospital"]
)
hospital_codes_12R = hospital_categorical_12R.codes.astype(int)
hospital_labels_12R = list(
    hospital_categorical_12R.categories
)
n_hospitals_12R = len(hospital_labels_12R)

if n_hospitals_12R != EXPECTED_HOSPITALS_12R:
    raise RuntimeError(
        "Bootstrap öncesi hastane sayısı 198 değil."
    )

rng_12R = np.random.default_rng(
    BOOTSTRAP_SEED_12R
)

bootstrap_rows_12R = []

print(
    f"\nRunning {BOOTSTRAP_REPLICATES_12R:,} corrected "
    "hospital-cluster bootstrap replicates..."
)

for bootstrap_index in range(
    BOOTSTRAP_REPLICATES_12R
):
    sampled_hospital_codes = rng_12R.integers(
        low=0,
        high=n_hospitals_12R,
        size=n_hospitals_12R,
    )

    hospital_multiplicity = np.bincount(
        sampled_hospital_codes,
        minlength=n_hospitals_12R,
    )

    patient_weights = hospital_multiplicity[
        hospital_codes_12R
    ].astype(float)

    positive_weight = float(
        patient_weights[y_12R == 1].sum()
    )
    negative_weight = float(
        patient_weights[y_12R == 0].sum()
    )

    if positive_weight <= 0 or negative_weight <= 0:
        continue

    bootstrap_rows_12R.append(
        {
            "bootstrap_replicate": bootstrap_index + 1,
            "sampled_hospital_draws": n_hospitals_12R,
            "unique_sampled_hospitals": int(
                np.count_nonzero(
                    hospital_multiplicity
                )
            ),
            "weighted_patients": float(
                patient_weights.sum()
            ),
            "weighted_events": positive_weight,
            "auroc_raw": float(
                roc_auc_score(
                    y_12R,
                    raw_12R,
                    sample_weight=patient_weights,
                )
            ),
            "auroc_platt": float(
                roc_auc_score(
                    y_12R,
                    platt_12R,
                    sample_weight=patient_weights,
                )
            ),
            "auprc_raw": float(
                average_precision_score(
                    y_12R,
                    raw_12R,
                    sample_weight=patient_weights,
                )
            ),
            "auprc_platt": float(
                average_precision_score(
                    y_12R,
                    platt_12R,
                    sample_weight=patient_weights,
                )
            ),
            "brier_raw": float(
                brier_score_loss(
                    y_12R,
                    raw_12R,
                    sample_weight=patient_weights,
                )
            ),
            "brier_platt": float(
                brier_score_loss(
                    y_12R,
                    platt_12R,
                    sample_weight=patient_weights,
                )
            ),
            "log_loss_raw": float(
                log_loss(
                    y_12R,
                    raw_12R,
                    labels=[0, 1],
                    sample_weight=patient_weights,
                )
            ),
            "log_loss_platt": float(
                log_loss(
                    y_12R,
                    platt_12R,
                    labels=[0, 1],
                    sample_weight=patient_weights,
                )
            ),
        }
    )

    if (
        (bootstrap_index + 1) % 100 == 0
        or bootstrap_index == 0
    ):
        print(
            "  Completed bootstrap replicate",
            bootstrap_index + 1,
            "/",
            BOOTSTRAP_REPLICATES_12R,
        )

bootstrap_replicates_12R = pd.DataFrame(
    bootstrap_rows_12R
)

if len(bootstrap_replicates_12R) < (
    BOOTSTRAP_REPLICATES_12R * 0.99
):
    raise RuntimeError(
        "Bootstrap tekrarlarının %99'undan azı tamamlandı."
    )

metric_definitions_12R = [
    (
        "auroc",
        "raw",
        "auroc_raw",
        raw_point_12R["auroc"],
    ),
    (
        "auroc",
        "platt_calibrated",
        "auroc_platt",
        platt_point_12R["auroc"],
    ),
    (
        "auprc",
        "raw",
        "auprc_raw",
        raw_point_12R["auprc"],
    ),
    (
        "auprc",
        "platt_calibrated",
        "auprc_platt",
        platt_point_12R["auprc"],
    ),
    (
        "brier",
        "raw",
        "brier_raw",
        raw_point_12R["brier"],
    ),
    (
        "brier",
        "platt_calibrated",
        "brier_platt",
        platt_point_12R["brier"],
    ),
    (
        "log_loss",
        "raw",
        "log_loss_raw",
        raw_point_12R["log_loss"],
    ),
    (
        "log_loss",
        "platt_calibrated",
        "log_loss_platt",
        platt_point_12R["log_loss"],
    ),
]

bootstrap_ci_rows_12R = []

for (
    metric_name,
    probability_type,
    bootstrap_column,
    point_estimate,
) in metric_definitions_12R:
    values = bootstrap_replicates_12R[
        bootstrap_column
    ].dropna().to_numpy(dtype=float)

    bootstrap_ci_rows_12R.append(
        {
            "metric": metric_name,
            "probability_type": probability_type,
            "point_estimate": float(point_estimate),
            "bootstrap_replicates": len(values),
            "bootstrap_mean": float(np.mean(values)),
            "bootstrap_standard_error": float(
                np.std(values, ddof=1)
            ),
            "ci_95_lower": float(
                np.quantile(values, 0.025)
            ),
            "ci_95_upper": float(
                np.quantile(values, 0.975)
            ),
            "bootstrap_unit": "hospital",
            "bootstrap_seed": BOOTSTRAP_SEED_12R,
        }
    )

corrected_bootstrap_ci_12R = pd.DataFrame(
    bootstrap_ci_rows_12R
)

# ------------------------------------------------------------
# 4. Save aggregate outputs only
# ------------------------------------------------------------

point_path_12R = os.path.join(
    MODEL_OUTPUT_DIR,
    "12R_logistic_raw_platt_point_metrics.csv",
)

replicates_path_12R = os.path.join(
    MODEL_OUTPUT_DIR,
    "12R_logistic_hospital_bootstrap_replicates.csv",
)

ci_path_12R = os.path.join(
    MODEL_OUTPUT_DIR,
    "12R_logistic_hospital_bootstrap_confidence_intervals.csv",
)

manifest_path_12R = os.path.join(
    MODEL_OUTPUT_DIR,
    "12R_corrected_bootstrap_manifest.json",
)

manifest_sha_path_12R = os.path.join(
    MODEL_OUTPUT_DIR,
    "12R_corrected_bootstrap_manifest_SHA256.txt",
)

point_metrics_12R.to_csv(
    point_path_12R,
    index=False,
)

bootstrap_replicates_12R.to_csv(
    replicates_path_12R,
    index=False,
)

corrected_bootstrap_ci_12R.to_csv(
    ci_path_12R,
    index=False,
)

manifest_12R = {
    "analysis": (
        "corrected separate raw and Platt "
        "hospital-cluster bootstrap"
    ),
    "source_table": POOLED_TABLE_12R,
    "patients": EXPECTED_ROWS_12R,
    "hospitals": EXPECTED_HOSPITALS_12R,
    "events": EXPECTED_EVENTS_12R,
    "bootstrap_replicates_requested": (
        BOOTSTRAP_REPLICATES_12R
    ),
    "bootstrap_replicates_completed": int(
        len(bootstrap_replicates_12R)
    ),
    "bootstrap_seed": BOOTSTRAP_SEED_12R,
    "bootstrap_unit": "hospital",
    "discrimination_confidence_intervals": (
        "calculated separately for raw and "
        "fold-specific Platt probabilities"
    ),
    "patient_level_data_written_to_drive": False,
    "aggregate_outputs": [
        point_path_12R,
        replicates_path_12R,
        ci_path_12R,
    ],
}

with open(
    manifest_path_12R,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        manifest_12R,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(
    manifest_path_12R,
    "rb",
) as file_handle:
    manifest_sha_12R = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    manifest_sha_path_12R,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(manifest_sha_12R + "\n")

# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------

print("\n12R POOLED TABLE INTEGRITY")
display(integrity_summary_12R)

print("\n12R RAW AND PLATT POINT METRICS")
display(point_metrics_12R)

print("\n12R CORRECTED HOSPITAL-CLUSTER BOOTSTRAP CIs")
display(corrected_bootstrap_ci_12R)

print("\n12R Manifest SHA-256:")
print(manifest_sha_12R)

print("\nSaved aggregate outputs:")
print(point_path_12R)
print(replicates_path_12R)
print(ci_path_12R)
print(manifest_path_12R)
print(manifest_sha_path_12R)

print(
    "\n12R PASS: Raw and fold-specific Platt "
    "discrimination confidence intervals were "
    "estimated separately."
)

print(
    "No patient-level prediction file was "
    "written to Google Drive."
)

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

from xgboost import XGBClassifier

from IPython.display import display

print("STARTING XGBOOST OUTER FOLD 1 — CODE VERSION 13")

# ============================================================
# 13 — XGBOOST OUTER FOLD 1 COMPLETE NESTED MODELLING
#
# Design:
# - Same locked hospital-disjoint outer folds.
# - Same locked hospital-disjoint inner folds.
# - Core feature set only.
# - No SMOTE.
# - No class weighting / scale_pos_weight = 1.
# - No early stopping.
# - Fixed candidate grid locked before outer-test evaluation.
# - Selection: pooled inner-OOF AUPRC descending,
#              AUROC descending, Brier ascending.
# - Platt calibration learned only from selected candidate's
#   pooled inner-OOF predictions.
# - Patient-level predictions stored only in BigQuery.
# ============================================================

OUTER_FOLD_13 = 1
MODEL_RANDOM_SEED_13 = 20260721

EXPECTED_SPLIT_13 = {
    "training_rows": 46803,
    "test_rows": 11688,
    "training_hospitals": 158,
    "test_hospitals": 40,
    "training_events": 2426,
    "test_events": 606,
}

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_objects_13 = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_13 = [
    name for name in required_objects_13
    if name not in globals()
]

if missing_objects_13:
    raise RuntimeError(
        "Missing runtime objects: "
        + ", ".join(missing_objects_13)
        + ". Run 07A and 07B first."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"Expected 58,491 cohort rows; found {len(core_df_07B)}."
    )

if len(predictor_columns_07B) != 159:
    raise RuntimeError("Expected 159 core predictors.")

if len(numeric_columns_07B) != 156:
    raise RuntimeError("Expected 156 numeric predictors.")

if len(categorical_columns_07B) != 3:
    raise RuntimeError("Expected 3 categorical predictors.")

# ------------------------------------------------------------
# 2. Lock the XGBoost protocol BEFORE test evaluation
# ------------------------------------------------------------

candidate_grid_13 = [
    {
        "candidate_id": "XGB01",
        "n_estimators": 250,
        "max_depth": 2,
        "learning_rate": 0.03,
        "min_child_weight": 5.0,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
    },
    {
        "candidate_id": "XGB02",
        "n_estimators": 350,
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 5.0,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
    },
    {
        "candidate_id": "XGB03",
        "n_estimators": 450,
        "max_depth": 3,
        "learning_rate": 0.02,
        "min_child_weight": 10.0,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "gamma": 0.0,
        "reg_alpha": 0.10,
        "reg_lambda": 10.0,
    },
    {
        "candidate_id": "XGB04",
        "n_estimators": 350,
        "max_depth": 4,
        "learning_rate": 0.03,
        "min_child_weight": 10.0,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "gamma": 0.10,
        "reg_alpha": 0.10,
        "reg_lambda": 10.0,
    },
    {
        "candidate_id": "XGB05",
        "n_estimators": 450,
        "max_depth": 2,
        "learning_rate": 0.02,
        "min_child_weight": 10.0,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "gamma": 0.0,
        "reg_alpha": 0.50,
        "reg_lambda": 10.0,
    },
    {
        "candidate_id": "XGB06",
        "n_estimators": 450,
        "max_depth": 4,
        "learning_rate": 0.02,
        "min_child_weight": 15.0,
        "subsample": 0.90,
        "colsample_bytree": 0.80,
        "gamma": 0.20,
        "reg_alpha": 0.50,
        "reg_lambda": 15.0,
    },
]

xgb_protocol_13 = {
    "protocol_name": "xgboost_core_nested_hospital_cv_v1",
    "model_family": "XGBoost",
    "feature_set": "core_159",
    "outer_cv": "locked 5-fold hospital-disjoint outer folds",
    "inner_cv": "locked 5-fold hospital-disjoint inner folds",
    "primary_selection_metric": "pooled inner OOF AUPRC descending",
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "class_weighting": False,
    "scale_pos_weight": 1.0,
    "smote": False,
    "early_stopping": False,
    "tree_method": "hist",
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": MODEL_RANDOM_SEED_13,
    "candidate_grid": candidate_grid_13,
    "calibration": (
        "Platt calibration fit only on selected candidate "
        "pooled inner-OOF logits"
    ),
    "outer_test_use": (
        "diagnostic evaluation only; never used for tuning"
    ),
}

protocol_path_13 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13A_locked_xgboost_model_protocol_v1.json",
)

protocol_sha_path_13 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13A_locked_xgboost_model_protocol_v1_SHA256.txt",
)

protocol_text_13 = json.dumps(
    xgb_protocol_13,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
)

protocol_sha_13 = hashlib.sha256(
    protocol_text_13.encode("utf-8")
).hexdigest()

if os.path.exists(protocol_path_13):
    with open(protocol_path_13, "r", encoding="utf-8") as fh:
        existing_protocol_text_13 = fh.read()
    existing_protocol_sha_13 = hashlib.sha256(
        existing_protocol_text_13.encode("utf-8")
    ).hexdigest()

    if existing_protocol_sha_13 != protocol_sha_13:
        raise RuntimeError(
            "An existing XGBoost protocol file differs from "
            "the currently locked protocol. Stop and audit."
        )
else:
    with open(protocol_path_13, "w", encoding="utf-8") as fh:
        fh.write(protocol_text_13)

with open(protocol_sha_path_13, "w", encoding="utf-8") as fh:
    fh.write(protocol_sha_13 + "\n")

print("Locked XGBoost protocol SHA-256:")
print(protocol_sha_13)

# ------------------------------------------------------------
# 3. Load locked inner hospital map
# ------------------------------------------------------------

inner_mapping_path_13 = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_13):
    raise FileNotFoundError(
        "Locked inner-fold map not found: "
        + inner_mapping_path_13
    )

inner_mapping_all_13 = pd.read_csv(
    inner_mapping_path_13,
    dtype={"group_hospital": str},
)

inner_mapping_part_13 = (
    inner_mapping_all_13.loc[
        inner_mapping_all_13["outer_fold"].astype(int)
        == OUTER_FOLD_13,
        ["group_hospital", "inner_fold"],
    ]
    .copy()
)

inner_mapping_part_13["group_hospital"] = (
    inner_mapping_part_13["group_hospital"].astype(str)
)
inner_mapping_part_13["inner_fold"] = (
    inner_mapping_part_13["inner_fold"].astype(int)
)

if len(inner_mapping_part_13) != 158:
    raise RuntimeError(
        "Expected 158 outer-fold-1 training hospitals "
        "in the locked inner map."
    )

if inner_mapping_part_13["group_hospital"].duplicated().any():
    raise RuntimeError("Duplicate hospital in locked inner map.")

hospital_to_inner_fold_13 = dict(
    zip(
        inner_mapping_part_13["group_hospital"],
        inner_mapping_part_13["inner_fold"],
    )
)

# ------------------------------------------------------------
# 4. Prepare outer fold 1 matrices
# ------------------------------------------------------------

X_all_13 = core_df_07B[predictor_columns_07B].copy()

for column in numeric_columns_07B:
    X_all_13[column] = pd.to_numeric(
        X_all_13[column],
        errors="coerce",
    ).astype("float64")

for column in categorical_columns_07B:
    category_series = X_all_13[column].astype("object")
    X_all_13[column] = category_series.where(
        pd.notna(category_series),
        np.nan,
    )

outer_fold_vector_13 = (
    core_df_07B["outer_fold"].astype(int).to_numpy()
)

outer_training_mask_13 = (
    outer_fold_vector_13 != OUTER_FOLD_13
)
outer_test_mask_13 = (
    outer_fold_vector_13 == OUTER_FOLD_13
)

X_outer_training_13 = (
    X_all_13.loc[outer_training_mask_13]
    .reset_index(drop=True)
)

X_outer_test_13 = (
    X_all_13.loc[outer_test_mask_13]
    .reset_index(drop=True)
)

outer_training_meta_13 = (
    core_df_07B.loc[
        outer_training_mask_13,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_13 = (
    core_df_07B.loc[
        outer_test_mask_13,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [
    outer_training_meta_13,
    outer_test_meta_13,
]:
    dataframe["id_row"] = dataframe["id_row"].astype(str)
    dataframe["group_hospital"] = (
        dataframe["group_hospital"].astype(str)
    )
    dataframe["label_stage23"] = (
        dataframe["label_stage23"].astype(int)
    )

y_outer_training_13 = (
    outer_training_meta_13["label_stage23"]
    .to_numpy(dtype=np.int8)
)
y_outer_test_13 = (
    outer_test_meta_13["label_stage23"]
    .to_numpy(dtype=np.int8)
)

groups_outer_training_13 = (
    outer_training_meta_13["group_hospital"]
    .to_numpy(dtype=str)
)

training_hospitals_13 = set(
    outer_training_meta_13["group_hospital"]
)
test_hospitals_13 = set(
    outer_test_meta_13["group_hospital"]
)
hospital_overlap_13 = (
    training_hospitals_13 & test_hospitals_13
)

if hospital_overlap_13:
    raise RuntimeError(
        "Outer training/test hospital overlap detected."
    )

actual_split_13 = {
    "training_rows": len(X_outer_training_13),
    "test_rows": len(X_outer_test_13),
    "training_hospitals": len(training_hospitals_13),
    "test_hospitals": len(test_hospitals_13),
    "training_events": int(y_outer_training_13.sum()),
    "test_events": int(y_outer_test_13.sum()),
}

for metric, expected_value in EXPECTED_SPLIT_13.items():
    actual_value = actual_split_13[metric]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: found={actual_value}, "
            f"expected={expected_value}"
        )

inner_fold_vector_13 = np.array(
    [
        hospital_to_inner_fold_13.get(hospital, -1)
        for hospital in groups_outer_training_13
    ],
    dtype=int,
)

if (inner_fold_vector_13 == -1).any():
    raise RuntimeError(
        "Some outer-training hospitals have no inner-fold assignment."
    )

if set(np.unique(inner_fold_vector_13)) != {1, 2, 3, 4, 5}:
    raise RuntimeError("Inner-fold values are not exactly 1–5.")

# ------------------------------------------------------------
# 5. Preprocessor and model constructors
# ------------------------------------------------------------

def make_preprocessor_13():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns_07B,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns_07B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_xgb_model_13(candidate):
    return XGBClassifier(
        n_estimators=int(candidate["n_estimators"]),
        max_depth=int(candidate["max_depth"]),
        learning_rate=float(candidate["learning_rate"]),
        min_child_weight=float(candidate["min_child_weight"]),
        subsample=float(candidate["subsample"]),
        colsample_bytree=float(candidate["colsample_bytree"]),
        gamma=float(candidate["gamma"]),
        reg_alpha=float(candidate["reg_alpha"]),
        reg_lambda=float(candidate["reg_lambda"]),
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        max_bin=256,
        scale_pos_weight=1.0,
        importance_type="gain",
        random_state=MODEL_RANDOM_SEED_13,
        n_jobs=-1,
        verbosity=0,
    )


def checkpoint_table_id_13(inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_xgb_inner_oof_outer1_inner{inner_fold}_v1"
    )

# ------------------------------------------------------------
# 6. BigQuery checkpoint verification
# ------------------------------------------------------------

def verify_checkpoint_13(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):
    table_id = checkpoint_table_id_13(inner_fold)

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS distinct_id_count,
      COUNT(DISTINCT candidate_id) AS candidate_count,
      COUNT(DISTINCT outer_fold) AS outer_fold_count,
      COUNT(DISTINCT inner_fold) AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(candidate_id, '|', id_row)
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(
        prediction_raw < 0 OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(
        sql,
        location=BQ_LOCATION,
    ).to_dataframe()

    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows * len(candidate_grid_13)
    )
    expected_positive_rows = (
        expected_validation_events * len(candidate_grid_13)
    )
    expected_negative_rows = (
        (expected_validation_rows - expected_validation_events)
        * len(candidate_grid_13)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": expected_validation_rows,
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": expected_total_rows,
        "positive_prediction_rows": expected_positive_rows,
        "negative_prediction_rows": expected_negative_rows,
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": 1,
        "maximum_outer_fold": 1,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failures = []

    for field, expected_value in expected_values.items():
        actual_value = int(row[field])
        if actual_value != expected_value:
            complete = False
            failures.append(
                f"{field}={actual_value}, expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failures),
        "check": check,
        "row": row,
    }

checkpoint_load_config_13 = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField(
            "id_row", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "outer_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "inner_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "candidate_id", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "label_stage23", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "prediction_raw", "FLOAT", mode="REQUIRED"
        ),
    ],
    write_disposition=(
        bigquery.WriteDisposition.WRITE_TRUNCATE
    ),
)

# ------------------------------------------------------------
# 7. Aggregate fit audit
# ------------------------------------------------------------

fit_audit_columns_13 = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "n_estimators",
    "max_depth",
    "learning_rate",
    "min_child_weight",
    "subsample",
    "colsample_bytree",
    "gamma",
    "reg_alpha",
    "reg_lambda",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "fit_seconds",
]

fit_audit_path_13 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13B_xgboost_inner_fit_audit_outer1.csv",
)

if os.path.exists(fit_audit_path_13):
    fit_audit_13 = pd.read_csv(fit_audit_path_13)
else:
    fit_audit_13 = pd.DataFrame(
        columns=fit_audit_columns_13
    )

for column in fit_audit_columns_13:
    if column not in fit_audit_13.columns:
        fit_audit_13[column] = np.nan

fit_audit_13 = fit_audit_13[
    fit_audit_columns_13
].copy()

# ------------------------------------------------------------
# 8. Run five inner folds
# ------------------------------------------------------------

for inner_fold in range(1, 6):
    inner_training_mask = (
        inner_fold_vector_13 != inner_fold
    )
    inner_validation_mask = (
        inner_fold_vector_13 == inner_fold
    )

    training_rows = int(inner_training_mask.sum())
    validation_rows = int(inner_validation_mask.sum())
    training_events = int(
        y_outer_training_13[inner_training_mask].sum()
    )
    validation_events = int(
        y_outer_training_13[inner_validation_mask].sum()
    )

    existing_check = verify_checkpoint_13(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if existing_check["complete"]:
        print(
            f"Outer 1 / inner {inner_fold}: "
            "permanent XGBoost checkpoint already complete; "
            "skipping model fitting."
        )
        continue

    training_hospital_set = set(
        groups_outer_training_13[inner_training_mask]
    )
    validation_hospital_set = set(
        groups_outer_training_13[inner_validation_mask]
    )

    if training_hospital_set & validation_hospital_set:
        raise RuntimeError(
            f"Inner fold {inner_fold}: hospital overlap detected."
        )

    print(f"\nOuter 1 / inner {inner_fold}")
    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_13()

    preprocessing_started = time.time()

    X_inner_training_processed = (
        preprocessor.fit_transform(
            X_outer_training_13.loc[
                inner_training_mask
            ]
        )
    )

    X_inner_validation_processed = (
        preprocessor.transform(
            X_outer_training_13.loc[
                inner_validation_mask
            ]
        )
    )

    preprocessing_seconds = (
        time.time() - preprocessing_started
    )

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "Processed training/validation column counts differ."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_seconds, 2),
    )

    y_inner_training = (
        y_outer_training_13[inner_training_mask]
    )
    y_inner_validation = (
        y_outer_training_13[inner_validation_mask]
    )

    validation_ids = (
        outer_training_meta_13.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_13:
        candidate_id = candidate["candidate_id"]

        print(
            "  Fitting",
            candidate_id,
            "| trees =",
            candidate["n_estimators"],
            "| depth =",
            candidate["max_depth"],
            "| lr =",
            candidate["learning_rate"],
        )

        model = make_xgb_model_13(candidate)

        fit_started = time.time()

        model.fit(
            X_inner_training_processed,
            y_inner_training,
        )

        fit_seconds = time.time() - fit_started

        validation_probabilities = (
            model.predict_proba(
                X_inner_validation_processed
            )[:, 1]
        )

        if np.isnan(validation_probabilities).any():
            raise RuntimeError(
                f"{candidate_id}, inner {inner_fold}: "
                "missing predictions."
            )

        if not np.all(
            (validation_probabilities >= 0)
            & (validation_probabilities <= 1)
        ):
            raise RuntimeError(
                f"{candidate_id}: invalid probabilities."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        1,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": (
                        y_inner_validation.astype(np.int64)
                    ),
                    "prediction_raw": (
                        validation_probabilities.astype(
                            np.float64
                        )
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": 1,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "n_estimators": candidate[
                    "n_estimators"
                ],
                "max_depth": candidate[
                    "max_depth"
                ],
                "learning_rate": candidate[
                    "learning_rate"
                ],
                "min_child_weight": candidate[
                    "min_child_weight"
                ],
                "subsample": candidate["subsample"],
                "colsample_bytree": candidate[
                    "colsample_bytree"
                ],
                "gamma": candidate["gamma"],
                "reg_alpha": candidate["reg_alpha"],
                "reg_lambda": candidate[
                    "reg_lambda"
                ],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": (
                    validation_events
                ),
                "processed_columns": int(
                    X_inner_training_processed.shape[1]
                ),
                "fit_seconds": float(fit_seconds),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows * len(candidate_grid_13)
    )

    if len(checkpoint_df) != expected_checkpoint_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: invalid checkpoint row count."
        )

    if checkpoint_df.duplicated(
        subset=["id_row", "candidate_id"]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: duplicate candidate-patient rows."
        )

    target_checkpoint_table = (
        checkpoint_table_id_13(inner_fold)
    )

    print(
        "Uploading permanent XGBoost checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_13,
        location=BQ_LOCATION,
    ).result()

    if len(fit_audit_13) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_13["outer_fold"],
                    errors="coerce",
                ) == 1
            )
            & (
                pd.to_numeric(
                    fit_audit_13["inner_fold"],
                    errors="coerce",
                ) == inner_fold
            )
        )

        fit_audit_13 = (
            fit_audit_13.loc[keep_mask].copy()
        )

    fit_audit_13 = pd.concat(
        [
            fit_audit_13,
            pd.DataFrame(current_audit_rows),
        ],
        ignore_index=True,
    )

    fit_audit_13 = (
        fit_audit_13[
            fit_audit_columns_13
        ]
        .sort_values(
            [
                "outer_fold",
                "inner_fold",
                "candidate_id",
            ]
        )
        .reset_index(drop=True)
    )

    fit_audit_13.to_csv(
        fit_audit_path_13,
        index=False,
    )

    completed_check = verify_checkpoint_13(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint verification failed: "
            + completed_check["reason"]
        )

    print(
        f"Outer 1 / inner {inner_fold}: "
        "permanent XGBoost checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 9. Final inner checkpoint summary
# ------------------------------------------------------------

checkpoint_summary_rows_13 = []

for inner_fold in range(1, 6):
    validation_mask = (
        inner_fold_vector_13 == inner_fold
    )
    validation_rows = int(validation_mask.sum())
    validation_events = int(
        y_outer_training_13[
            validation_mask
        ].sum()
    )

    final_check = verify_checkpoint_13(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: final checkpoint audit failed. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_13.append(
        {
            "outer_fold": 1,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(
                row["row_count"]
            ),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(
                row["candidate_count"]
            ),
            "positive_prediction_rows": int(
                row["positive_prediction_rows"]
            ),
            "negative_prediction_rows": int(
                row["negative_prediction_rows"]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check["table_id"],
        }
    )

checkpoint_summary_13 = (
    pd.DataFrame(checkpoint_summary_rows_13)
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_13[
        "distinct_validation_patients"
    ].sum()
) != EXPECTED_SPLIT_13["training_rows"]:
    raise RuntimeError(
        "Total inner validation patients != 46,803."
    )

expected_total_oof_rows_13 = (
    EXPECTED_SPLIT_13["training_rows"]
    * len(candidate_grid_13)
)

if int(
    checkpoint_summary_13[
        "checkpoint_rows"
    ].sum()
) != expected_total_oof_rows_13:
    raise RuntimeError(
        "Total XGBoost OOF prediction rows are incorrect."
    )

checkpoint_summary_path_13 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13B_xgboost_outer1_inner_checkpoint_summary.csv",
)

checkpoint_summary_13.to_csv(
    checkpoint_summary_path_13,
    index=False,
)

# ------------------------------------------------------------
# 10. Pool five inner OOF tables
# ------------------------------------------------------------

checkpoint_tables_13 = [
    checkpoint_table_id_13(inner_fold)
    for inner_fold in range(1, 6)
]

union_parts_13 = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_13
]

SQL_LOAD_POOLED_OOF_13 = (
    "\nUNION ALL\n".join(union_parts_13)
)

print(
    "\nLoading pooled outer-fold-1 XGBoost inner OOF predictions..."
)

query_job_13 = client.query(
    SQL_LOAD_POOLED_OOF_13,
    location=BQ_LOCATION,
)

try:
    pooled_oof_13 = query_job_13.to_dataframe(
        create_bqstorage_client=True
    )
    pooled_load_method_13 = (
        "BigQuery Storage API"
    )
except Exception as fast_path_error_13:
    print(
        "Storage API unavailable; using standard BigQuery download."
    )
    print(
        "Message:",
        type(fast_path_error_13).__name__,
    )
    pooled_oof_13 = query_job_13.to_dataframe(
        create_bqstorage_client=False
    )
    pooled_load_method_13 = (
        "Standard BigQuery API"
    )

pooled_oof_13["id_row"] = (
    pooled_oof_13["id_row"].astype(str)
)
pooled_oof_13["candidate_id"] = (
    pooled_oof_13["candidate_id"].astype(str)
)

for column in [
    "outer_fold",
    "inner_fold",
    "label_stage23",
]:
    pooled_oof_13[column] = pd.to_numeric(
        pooled_oof_13[column],
        errors="raise",
    ).astype(int)

pooled_oof_13["prediction_raw"] = pd.to_numeric(
    pooled_oof_13["prediction_raw"],
    errors="raise",
).astype(float)

if len(pooled_oof_13) != expected_total_oof_rows_13:
    raise RuntimeError(
        "Pooled XGBoost OOF row count is incorrect."
    )

if pooled_oof_13.duplicated(
    subset=["candidate_id", "id_row"]
).any():
    raise RuntimeError(
        "Duplicate candidate-patient row in pooled XGBoost OOF."
    )

if pooled_oof_13["prediction_raw"].isna().any():
    raise RuntimeError("Missing XGBoost OOF prediction.")

if not pooled_oof_13[
    "prediction_raw"
].between(0, 1).all():
    raise RuntimeError(
        "Invalid XGBoost OOF probability."
    )

if set(
    pooled_oof_13["candidate_id"].unique()
) != {
    "XGB01",
    "XGB02",
    "XGB03",
    "XGB04",
    "XGB05",
    "XGB06",
}:
    raise RuntimeError(
        "The six locked XGBoost candidates are not all present."
    )

candidate_patient_counts_13 = (
    pooled_oof_13
    .groupby("candidate_id")["id_row"]
    .nunique()
)

if not (
    candidate_patient_counts_13
    == EXPECTED_SPLIT_13["training_rows"]
).all():
    raise RuntimeError(
        "Each candidate must have 46,803 OOF patients."
    )

candidate_event_counts_13 = (
    pooled_oof_13
    .groupby("candidate_id")["label_stage23"]
    .sum()
)

if not (
    candidate_event_counts_13
    == EXPECTED_SPLIT_13["training_events"]
).all():
    raise RuntimeError(
        "Each candidate must have 2,426 OOF events."
    )

# ------------------------------------------------------------
# 11. Metric helpers
# ------------------------------------------------------------

def probability_metrics_13(
    y_true,
    probabilities,
):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(
            roc_auc_score(y_true, probabilities)
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1],
            )
        ),
        "mean_predicted_risk": float(
            probabilities.mean()
        ),
        "observed_event_rate": float(
            np.mean(y_true)
        ),
    }


def probability_logit_13(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        probabilities / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_13(
    y_true,
    probabilities,
):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        probability_logit_13(probabilities),
        y_true,
    )

    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 12. Candidate pooled inner OOF metrics
# ------------------------------------------------------------

candidate_result_rows_13 = []

for candidate in candidate_grid_13:
    candidate_id = candidate["candidate_id"]

    candidate_oof = (
        pooled_oof_13.loc[
            pooled_oof_13["candidate_id"]
            == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_13(
        candidate_oof[
            "label_stage23"
        ].to_numpy(dtype=int),
        candidate_oof[
            "prediction_raw"
        ].to_numpy(dtype=float),
    )

    fit_part = fit_audit_13.loc[
        fit_audit_13[
            "candidate_id"
        ].astype(str) == candidate_id
    ]

    fit_seconds_total = (
        float(
            pd.to_numeric(
                fit_part["fit_seconds"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_13.append(
        {
            "candidate_id": candidate_id,
            **{
                key: candidate[key]
                for key in candidate
                if key != "candidate_id"
            },
            **metrics,
            "fit_seconds_total": (
                fit_seconds_total
            ),
        }
    )

candidate_results_13 = pd.DataFrame(
    candidate_result_rows_13
)

candidate_results_13 = (
    candidate_results_13
    .sort_values(
        ["auprc", "auroc", "brier"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

candidate_results_13["selection_rank"] = (
    np.arange(
        1,
        len(candidate_results_13) + 1,
    )
)

best_row_13 = candidate_results_13.iloc[0]
selected_candidate_id_13 = str(
    best_row_13["candidate_id"]
)

selected_candidate_13 = next(
    candidate
    for candidate in candidate_grid_13
    if candidate["candidate_id"]
    == selected_candidate_id_13
)

# ------------------------------------------------------------
# 13. Platt calibration from selected inner OOF
# ------------------------------------------------------------

selected_oof_13 = (
    pooled_oof_13.loc[
        pooled_oof_13["candidate_id"]
        == selected_candidate_id_13
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_13 = (
    selected_oof_13[
        "label_stage23"
    ].to_numpy(dtype=int)
)
selected_oof_probability_13 = (
    selected_oof_13[
        "prediction_raw"
    ].to_numpy(dtype=float)
)

platt_calibrator_13 = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_13.fit(
    probability_logit_13(
        selected_oof_probability_13
    ),
    selected_oof_y_13,
)

platt_intercept_13 = float(
    platt_calibrator_13.intercept_[0]
)
platt_slope_13 = float(
    platt_calibrator_13.coef_[0][0]
)

if (
    not np.isfinite(platt_intercept_13)
    or not np.isfinite(platt_slope_13)
    or platt_slope_13 <= 0
):
    raise RuntimeError(
        "Invalid Platt calibration coefficients."
    )

selected_model_13 = pd.DataFrame(
    [
        {
            "outer_fold": 1,
            "selected_candidate": (
                selected_candidate_id_13
            ),
            "selection_metric_primary": (
                "pooled_inner_oof_auprc"
            ),
            "inner_oof_auprc": float(
                best_row_13["auprc"]
            ),
            "inner_oof_auroc": float(
                best_row_13["auroc"]
            ),
            "inner_oof_brier": float(
                best_row_13["brier"]
            ),
            "inner_oof_log_loss": float(
                best_row_13["log_loss"]
            ),
            "inner_oof_mean_predicted_risk": float(
                best_row_13[
                    "mean_predicted_risk"
                ]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_13[
                    "observed_event_rate"
                ]
            ),
            "platt_intercept": (
                platt_intercept_13
            ),
            "platt_slope": platt_slope_13,
            "protocol_sha256": (
                protocol_sha_13
            ),
            **{
                key: selected_candidate_13[key]
                for key in selected_candidate_13
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 14. Save locked selection
# ------------------------------------------------------------

candidate_results_path_13 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13C_xgboost_candidate_results_outer1.csv",
)

selected_model_path_13 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13C_xgboost_selected_model_outer1.csv",
)

selection_json_path_13 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13C_xgboost_selection_calibration_outer1.json",
)

selection_sha_path_13 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13C_xgboost_selection_calibration_outer1_SHA256.txt",
)

candidate_results_13.to_csv(
    candidate_results_path_13,
    index=False,
)
selected_model_13.to_csv(
    selected_model_path_13,
    index=False,
)

selection_configuration_13 = {
    "outer_fold": 1,
    "protocol_sha256": protocol_sha_13,
    "selection_metric_primary": (
        "pooled inner out-of-fold AUPRC"
    ),
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "selected_candidate": (
        selected_candidate_id_13
    ),
    "selected_hyperparameters": {
        key: selected_candidate_13[key]
        for key in selected_candidate_13
        if key != "candidate_id"
    },
    "inner_oof_auprc": float(
        best_row_13["auprc"]
    ),
    "inner_oof_auroc": float(
        best_row_13["auroc"]
    ),
    "inner_oof_brier": float(
        best_row_13["brier"]
    ),
    "platt_intercept": platt_intercept_13,
    "platt_slope": platt_slope_13,
    "inner_checkpoint_tables": (
        checkpoint_tables_13
    ),
    "patient_level_oof_written_to_drive": False,
}

with open(
    selection_json_path_13,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        selection_configuration_13,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    selection_json_path_13,
    "rb",
) as fh:
    selection_sha_13 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    selection_sha_path_13,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(selection_sha_13 + "\n")

# ------------------------------------------------------------
# 15. Fit selected model on all outer training patients
# ------------------------------------------------------------

final_preprocessor_13 = (
    make_preprocessor_13()
)

print(
    "\nFitting selected outer-fold-1 XGBoost model "
    "on all 46,803 training patients..."
)

preprocess_started_13 = time.time()

X_outer_training_processed_13 = (
    final_preprocessor_13.fit_transform(
        X_outer_training_13
    )
)
X_outer_test_processed_13 = (
    final_preprocessor_13.transform(
        X_outer_test_13
    )
)

final_preprocessing_seconds_13 = (
    time.time() - preprocess_started_13
)

final_model_13 = make_xgb_model_13(
    selected_candidate_13
)

final_fit_started_13 = time.time()

final_model_13.fit(
    X_outer_training_processed_13,
    y_outer_training_13,
)

final_fit_seconds_13 = (
    time.time() - final_fit_started_13
)

outer1_raw_probabilities_13 = (
    final_model_13.predict_proba(
        X_outer_test_processed_13
    )[:, 1]
)

raw_clipped_13 = np.clip(
    outer1_raw_probabilities_13,
    1e-6,
    1 - 1e-6,
)
raw_logit_13 = np.log(
    raw_clipped_13
    / (1 - raw_clipped_13)
)

outer1_platt_probabilities_13 = expit(
    platt_intercept_13
    + platt_slope_13 * raw_logit_13
)

for probabilities, name in [
    (outer1_raw_probabilities_13, "raw"),
    (
        outer1_platt_probabilities_13,
        "platt",
    ),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(
            f"{name} test predictions contain missing values."
        )

    if not np.all(
        (probabilities >= 0)
        & (probabilities <= 1)
    ):
        raise RuntimeError(
            f"{name} test predictions contain invalid probabilities."
        )

raw_metrics_13 = probability_metrics_13(
    y_outer_test_13,
    outer1_raw_probabilities_13,
)
platt_metrics_13 = probability_metrics_13(
    y_outer_test_13,
    outer1_platt_probabilities_13,
)

raw_calibration_intercept_13, \
raw_calibration_slope_13 = (
    calibration_intercept_slope_13(
        y_outer_test_13,
        outer1_raw_probabilities_13,
    )
)

platt_calibration_intercept_13, \
platt_calibration_slope_13 = (
    calibration_intercept_slope_13(
        y_outer_test_13,
        outer1_platt_probabilities_13,
    )
)

outer1_test_results_13 = pd.DataFrame(
    [
        {
            "outer_fold": 1,
            "model": "xgboost",
            "probability_type": "raw",
            **raw_metrics_13,
            "calibration_intercept": (
                raw_calibration_intercept_13
            ),
            "calibration_slope": (
                raw_calibration_slope_13
            ),
        },
        {
            "outer_fold": 1,
            "model": "xgboost",
            "probability_type": (
                "platt_calibrated"
            ),
            **platt_metrics_13,
            "calibration_intercept": (
                platt_calibration_intercept_13
            ),
            "calibration_slope": (
                platt_calibration_slope_13
            ),
        },
    ]
)

# ------------------------------------------------------------
# 16. Feature importance
# ------------------------------------------------------------

processed_feature_names_13 = (
    final_preprocessor_13
    .get_feature_names_out()
)

feature_importances_13 = (
    final_model_13.feature_importances_
)

if len(processed_feature_names_13) != len(
    feature_importances_13
):
    raise RuntimeError(
        "Processed feature names and XGBoost "
        "feature importances differ in length."
    )

feature_importance_table_13 = pd.DataFrame(
    {
        "processed_feature": (
            processed_feature_names_13
        ),
        "gain_importance": (
            feature_importances_13
        ),
    }
)

feature_importance_table_13[
    "importance_rank"
] = (
    feature_importance_table_13[
        "gain_importance"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)

feature_importance_table_13 = (
    feature_importance_table_13
    .sort_values(
        [
            "gain_importance",
            "processed_feature",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

nonzero_importance_features_13 = int(
    (
        feature_importance_table_13[
            "gain_importance"
        ] > 0
    ).sum()
)

final_model_summary_13 = pd.DataFrame(
    [
        {
            "outer_fold": 1,
            "selected_candidate": (
                selected_candidate_id_13
            ),
            "training_patients": len(
                X_outer_training_13
            ),
            "training_hospitals": len(
                training_hospitals_13
            ),
            "training_events": int(
                y_outer_training_13.sum()
            ),
            "test_patients": len(
                X_outer_test_13
            ),
            "test_hospitals": len(
                test_hospitals_13
            ),
            "test_events": int(
                y_outer_test_13.sum()
            ),
            "hospital_overlap": len(
                hospital_overlap_13
            ),
            "processed_feature_columns": len(
                processed_feature_names_13
            ),
            "nonzero_importance_features": (
                nonzero_importance_features_13
            ),
            "preprocessing_seconds": float(
                final_preprocessing_seconds_13
            ),
            "fit_seconds": float(
                final_fit_seconds_13
            ),
            "locked_platt_intercept": (
                platt_intercept_13
            ),
            "locked_platt_slope": (
                platt_slope_13
            ),
            "protocol_sha256": protocol_sha_13,
            "selection_sha256": (
                selection_sha_13
            ),
            **{
                key: selected_candidate_13[key]
                for key in selected_candidate_13
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 17. Secure outer-test prediction checkpoint
# ------------------------------------------------------------

outer1_prediction_df_13 = pd.DataFrame(
    {
        "id_row": (
            outer_test_meta_13[
                "id_row"
            ].astype(str)
        ),
        "outer_fold": np.full(
            len(outer_test_meta_13),
            1,
            dtype=np.int64,
        ),
        "label_stage23": (
            y_outer_test_13.astype(np.int64)
        ),
        "prediction_raw": (
            outer1_raw_probabilities_13.astype(
                np.float64
            )
        ),
        "prediction_platt": (
            outer1_platt_probabilities_13.astype(
                np.float64
            )
        ),
        "model_name": "xgboost",
        "model_version": (
            "core_v1_nested_cv"
        ),
    }
)

if len(outer1_prediction_df_13) != 11688:
    raise RuntimeError(
        "Outer-fold-1 test prediction row count != 11,688."
    )

if outer1_prediction_df_13[
    "id_row"
].duplicated().any():
    raise RuntimeError(
        "Duplicate id_row in outer-fold-1 XGBoost predictions."
    )

if int(
    outer1_prediction_df_13[
        "label_stage23"
    ].sum()
) != 606:
    raise RuntimeError(
        "Outer-fold-1 XGBoost event count != 606."
    )

prediction_table_id_13 = (
    f"{TARGET_DATASET}."
    "model_xgb_outer_predictions_outer1_v1"
)

prediction_load_config_13 = (
    bigquery.LoadJobConfig(
        schema=[
            bigquery.SchemaField(
                "id_row",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "outer_fold",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "label_stage23",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_raw",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_platt",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_name",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_version",
                "STRING",
                mode="REQUIRED",
            ),
        ],
        write_disposition=(
            bigquery.WriteDisposition.WRITE_TRUNCATE
        ),
    )
)

print(
    "\nUploading secure outer-fold-1 "
    "XGBoost prediction checkpoint:"
)
print(prediction_table_id_13)

client.load_table_from_dataframe(
    outer1_prediction_df_13,
    prediction_table_id_13,
    job_config=prediction_load_config_13,
    location=BQ_LOCATION,
).result()

SQL_VERIFY_PREDICTIONS_13 = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL)
    AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL)
    AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0
    OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0
    OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw)
    AS minimum_raw_probability,
  MAX(prediction_raw)
    AS maximum_raw_probability,
  MIN(prediction_platt)
    AS minimum_platt_probability,
  MAX(prediction_platt)
    AS maximum_platt_probability
FROM `{prediction_table_id_13}`;
"""

prediction_verification_13 = (
    client.query(
        SQL_VERIFY_PREDICTIONS_13,
        location=BQ_LOCATION,
    )
    .to_dataframe()
)

verification_row_13 = (
    prediction_verification_13.iloc[0]
)

expected_prediction_values_13 = {
    "prediction_rows": 11688,
    "distinct_rows": 11688,
    "outer_folds": 1,
    "minimum_outer_fold": 1,
    "maximum_outer_fold": 1,
    "events": 606,
    "nonevents": 11082,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in (
    expected_prediction_values_13.items()
):
    actual_value = int(
        verification_row_13[field]
    )
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: found={actual_value}, "
            f"expected={expected_value}"
        )

# ------------------------------------------------------------
# 18. Save aggregate outputs
# ------------------------------------------------------------

test_results_path_13 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13D_xgboost_outer1_test_results.csv",
)

model_summary_path_13 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13D_xgboost_final_model_outer1.csv",
)

importance_path_13 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13D_xgboost_gain_importance_outer1.csv",
)

evaluation_json_path_13 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13D_xgboost_final_evaluation_outer1.json",
)

evaluation_sha_path_13 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13D_xgboost_final_evaluation_outer1_SHA256.txt",
)

outer1_test_results_13.to_csv(
    test_results_path_13,
    index=False,
)
final_model_summary_13.to_csv(
    model_summary_path_13,
    index=False,
)
feature_importance_table_13.to_csv(
    importance_path_13,
    index=False,
)

evaluation_configuration_13 = {
    "outer_fold": 1,
    "model_family": "xgboost",
    "protocol_sha256": protocol_sha_13,
    "selection_sha256": selection_sha_13,
    "selected_candidate": (
        selected_candidate_id_13
    ),
    "selected_hyperparameters": {
        key: selected_candidate_13[key]
        for key in selected_candidate_13
        if key != "candidate_id"
    },
    "training_patients": 46803,
    "training_hospitals": 158,
    "test_patients": 11688,
    "test_hospitals": 40,
    "hospital_overlap": 0,
    "locked_platt_intercept": (
        platt_intercept_13
    ),
    "locked_platt_slope": platt_slope_13,
    "processed_feature_columns": int(
        len(processed_feature_names_13)
    ),
    "secure_prediction_table": (
        prediction_table_id_13
    ),
    "patient_level_prediction_written_to_drive": False,
}

with open(
    evaluation_json_path_13,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        evaluation_configuration_13,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    evaluation_json_path_13,
    "rb",
) as fh:
    evaluation_sha_13 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    evaluation_sha_path_13,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(evaluation_sha_13 + "\n")

# ------------------------------------------------------------
# 19. Display results
# ------------------------------------------------------------

pooled_integrity_13 = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_13),
            pooled_oof_13[
                "id_row"
            ].nunique(),
            pooled_oof_13[
                "candidate_id"
            ].nunique(),
            pooled_oof_13[
                "inner_fold"
            ].nunique(),
            EXPECTED_SPLIT_13[
                "training_events"
            ],
            (
                EXPECTED_SPLIT_13[
                    "training_rows"
                ]
                - EXPECTED_SPLIT_13[
                    "training_events"
                ]
            ),
            int(
                pooled_oof_13.duplicated(
                    subset=[
                        "candidate_id",
                        "id_row",
                    ]
                ).sum()
            ),
            int(
                pooled_oof_13[
                    "prediction_raw"
                ].isna().sum()
            ),
            int(
                (
                    ~pooled_oof_13[
                        "prediction_raw"
                    ].between(0, 1)
                ).sum()
            ),
            pooled_load_method_13,
        ],
    }
)

print(
    "\n13 XGBOOST OUTER-FOLD-1 INNER CHECKPOINT SUMMARY"
)
display(checkpoint_summary_13)

print(
    "\n13 XGBOOST OUTER-FOLD-1 POOLED OOF INTEGRITY"
)
display(pooled_integrity_13)

print(
    "\n13 XGBOOST OUTER-FOLD-1 CANDIDATE RESULTS"
)
display(candidate_results_13)

print(
    "\n13 XGBOOST OUTER-FOLD-1 SELECTED MODEL"
)
display(selected_model_13)

print(
    "\n13 XGBOOST OUTER-FOLD-1 FINAL MODEL SUMMARY"
)
display(final_model_summary_13)

print(
    "\n13 XGBOOST OUTER-FOLD-1 TEST RESULTS"
)
display(outer1_test_results_13)

print(
    "\n13 XGBOOST OUTER-FOLD-1 BIGQUERY VERIFICATION"
)
display(prediction_verification_13)

print(
    "\n13 XGBOOST OUTER-FOLD-1 TOP 20 GAIN IMPORTANCE FEATURES"
)
display(feature_importance_table_13.head(20))

print("\nXGBoost protocol SHA-256:")
print(protocol_sha_13)

print("\nSelection SHA-256:")
print(selection_sha_13)

print("\nEvaluation SHA-256:")
print(evaluation_sha_13)

print("\nSaved aggregate outputs:")
print(protocol_path_13)
print(protocol_sha_path_13)
print(fit_audit_path_13)
print(checkpoint_summary_path_13)
print(candidate_results_path_13)
print(selected_model_path_13)
print(selection_json_path_13)
print(selection_sha_path_13)
print(test_results_path_13)
print(model_summary_path_13)
print(importance_path_13)
print(evaluation_json_path_13)
print(evaluation_sha_path_13)

print(
    "\n13 PASS: XGBoost outer-fold-1 nested modelling "
    "and locked test evaluation are complete."
)

print(
    "No class weighting, SMOTE, or early stopping was used."
)

print(
    "All patient-level OOF and outer-test predictions "
    "were stored only in BigQuery."
)

print(
    "No patient-level prediction file was written to Google Drive."
)

_ = gc.collect()

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

from xgboost import XGBClassifier

from IPython.display import display

print("STARTING XGBOOST OUTER FOLD 2 — CODE VERSION 14")

# ============================================================
# 14 — XGBOOST OUTER FOLD 2 COMPLETE NESTED MODELLING
#
# Design:
# - Same locked hospital-disjoint outer folds.
# - Same locked hospital-disjoint inner folds.
# - Core feature set only.
# - No SMOTE.
# - No class weighting / scale_pos_weight = 1.
# - No early stopping.
# - Fixed candidate grid locked before outer-test evaluation.
# - Selection: pooled inner-OOF AUPRC descending,
#              AUROC descending, Brier ascending.
# - Platt calibration learned only from selected candidate's
#   pooled inner-OOF predictions.
# - Patient-level predictions stored only in BigQuery.
# ============================================================

OUTER_FOLD_14 = 2
MODEL_RANDOM_SEED_14 = 20260721

EXPECTED_SPLIT_14 = {
    "training_rows": 46800,
    "test_rows": 11691,
    "training_hospitals": 158,
    "test_hospitals": 40,
    "training_events": 2426,
    "test_events": 606,
}

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_objects_14 = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_14 = [
    name for name in required_objects_14
    if name not in globals()
]

if missing_objects_14:
    raise RuntimeError(
        "Missing runtime objects: "
        + ", ".join(missing_objects_14)
        + ". Run 07A and 07B first."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"Expected 58,491 cohort rows; found {len(core_df_07B)}."
    )

if len(predictor_columns_07B) != 159:
    raise RuntimeError("Expected 159 core predictors.")

if len(numeric_columns_07B) != 156:
    raise RuntimeError("Expected 156 numeric predictors.")

if len(categorical_columns_07B) != 3:
    raise RuntimeError("Expected 3 categorical predictors.")

# ------------------------------------------------------------
# 2. Lock the XGBoost protocol BEFORE test evaluation
# ------------------------------------------------------------

candidate_grid_14 = [
    {
        "candidate_id": "XGB01",
        "n_estimators": 250,
        "max_depth": 2,
        "learning_rate": 0.03,
        "min_child_weight": 5.0,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
    },
    {
        "candidate_id": "XGB02",
        "n_estimators": 350,
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 5.0,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
    },
    {
        "candidate_id": "XGB03",
        "n_estimators": 450,
        "max_depth": 3,
        "learning_rate": 0.02,
        "min_child_weight": 10.0,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "gamma": 0.0,
        "reg_alpha": 0.10,
        "reg_lambda": 10.0,
    },
    {
        "candidate_id": "XGB04",
        "n_estimators": 350,
        "max_depth": 4,
        "learning_rate": 0.03,
        "min_child_weight": 10.0,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "gamma": 0.10,
        "reg_alpha": 0.10,
        "reg_lambda": 10.0,
    },
    {
        "candidate_id": "XGB05",
        "n_estimators": 450,
        "max_depth": 2,
        "learning_rate": 0.02,
        "min_child_weight": 10.0,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "gamma": 0.0,
        "reg_alpha": 0.50,
        "reg_lambda": 10.0,
    },
    {
        "candidate_id": "XGB06",
        "n_estimators": 450,
        "max_depth": 4,
        "learning_rate": 0.02,
        "min_child_weight": 15.0,
        "subsample": 0.90,
        "colsample_bytree": 0.80,
        "gamma": 0.20,
        "reg_alpha": 0.50,
        "reg_lambda": 15.0,
    },
]

xgb_protocol_14 = {
    "protocol_name": "xgboost_core_nested_hospital_cv_v1",
    "model_family": "XGBoost",
    "feature_set": "core_159",
    "outer_cv": "locked 5-fold hospital-disjoint outer folds",
    "inner_cv": "locked 5-fold hospital-disjoint inner folds",
    "primary_selection_metric": "pooled inner OOF AUPRC descending",
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "class_weighting": False,
    "scale_pos_weight": 1.0,
    "smote": False,
    "early_stopping": False,
    "tree_method": "hist",
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": MODEL_RANDOM_SEED_14,
    "candidate_grid": candidate_grid_14,
    "calibration": (
        "Platt calibration fit only on selected candidate "
        "pooled inner-OOF logits"
    ),
    "outer_test_use": (
        "diagnostic evaluation only; never used for tuning"
    ),
}

protocol_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13A_locked_xgboost_model_protocol_v1.json",
)

protocol_sha_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13A_locked_xgboost_model_protocol_v1_SHA256.txt",
)

protocol_text_14 = json.dumps(
    xgb_protocol_14,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
)

protocol_sha_14 = hashlib.sha256(
    protocol_text_14.encode("utf-8")
).hexdigest()

if os.path.exists(protocol_path_14):
    with open(protocol_path_14, "r", encoding="utf-8") as fh:
        existing_protocol_text_14 = fh.read()
    existing_protocol_sha_14 = hashlib.sha256(
        existing_protocol_text_14.encode("utf-8")
    ).hexdigest()

    if existing_protocol_sha_14 != protocol_sha_14:
        raise RuntimeError(
            "An existing XGBoost protocol file differs from "
            "the currently locked protocol. Stop and audit."
        )
else:
    with open(protocol_path_14, "w", encoding="utf-8") as fh:
        fh.write(protocol_text_14)

with open(protocol_sha_path_14, "w", encoding="utf-8") as fh:
    fh.write(protocol_sha_14 + "\n")

print("Locked XGBoost protocol SHA-256:")
print(protocol_sha_14)

# ------------------------------------------------------------
# 3. Load locked inner hospital map
# ------------------------------------------------------------

inner_mapping_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_14):
    raise FileNotFoundError(
        "Locked inner-fold map not found: "
        + inner_mapping_path_14
    )

inner_mapping_all_14 = pd.read_csv(
    inner_mapping_path_14,
    dtype={"group_hospital": str},
)

inner_mapping_part_14 = (
    inner_mapping_all_14.loc[
        inner_mapping_all_14["outer_fold"].astype(int)
        == OUTER_FOLD_14,
        ["group_hospital", "inner_fold"],
    ]
    .copy()
)

inner_mapping_part_14["group_hospital"] = (
    inner_mapping_part_14["group_hospital"].astype(str)
)
inner_mapping_part_14["inner_fold"] = (
    inner_mapping_part_14["inner_fold"].astype(int)
)

if len(inner_mapping_part_14) != 158:
    raise RuntimeError(
        "Expected 158 outer-fold-2 training hospitals "
        "in the locked inner map."
    )

if inner_mapping_part_14["group_hospital"].duplicated().any():
    raise RuntimeError("Duplicate hospital in locked inner map.")

hospital_to_inner_fold_14 = dict(
    zip(
        inner_mapping_part_14["group_hospital"],
        inner_mapping_part_14["inner_fold"],
    )
)

# ------------------------------------------------------------
# 4. Prepare outer fold 1 matrices
# ------------------------------------------------------------

X_all_14 = core_df_07B[predictor_columns_07B].copy()

for column in numeric_columns_07B:
    X_all_14[column] = pd.to_numeric(
        X_all_14[column],
        errors="coerce",
    ).astype("float64")

for column in categorical_columns_07B:
    category_series = X_all_14[column].astype("object")
    X_all_14[column] = category_series.where(
        pd.notna(category_series),
        np.nan,
    )

outer_fold_vector_14 = (
    core_df_07B["outer_fold"].astype(int).to_numpy()
)

outer_training_mask_14 = (
    outer_fold_vector_14 != OUTER_FOLD_14
)
outer_test_mask_14 = (
    outer_fold_vector_14 == OUTER_FOLD_14
)

X_outer_training_14 = (
    X_all_14.loc[outer_training_mask_14]
    .reset_index(drop=True)
)

X_outer_test_14 = (
    X_all_14.loc[outer_test_mask_14]
    .reset_index(drop=True)
)

outer_training_meta_14 = (
    core_df_07B.loc[
        outer_training_mask_14,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_14 = (
    core_df_07B.loc[
        outer_test_mask_14,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [
    outer_training_meta_14,
    outer_test_meta_14,
]:
    dataframe["id_row"] = dataframe["id_row"].astype(str)
    dataframe["group_hospital"] = (
        dataframe["group_hospital"].astype(str)
    )
    dataframe["label_stage23"] = (
        dataframe["label_stage23"].astype(int)
    )

y_outer_training_14 = (
    outer_training_meta_14["label_stage23"]
    .to_numpy(dtype=np.int8)
)
y_outer_test_14 = (
    outer_test_meta_14["label_stage23"]
    .to_numpy(dtype=np.int8)
)

groups_outer_training_14 = (
    outer_training_meta_14["group_hospital"]
    .to_numpy(dtype=str)
)

training_hospitals_14 = set(
    outer_training_meta_14["group_hospital"]
)
test_hospitals_14 = set(
    outer_test_meta_14["group_hospital"]
)
hospital_overlap_14 = (
    training_hospitals_14 & test_hospitals_14
)

if hospital_overlap_14:
    raise RuntimeError(
        "Outer training/test hospital overlap detected."
    )

actual_split_14 = {
    "training_rows": len(X_outer_training_14),
    "test_rows": len(X_outer_test_14),
    "training_hospitals": len(training_hospitals_14),
    "test_hospitals": len(test_hospitals_14),
    "training_events": int(y_outer_training_14.sum()),
    "test_events": int(y_outer_test_14.sum()),
}

for metric, expected_value in EXPECTED_SPLIT_14.items():
    actual_value = actual_split_14[metric]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: found={actual_value}, "
            f"expected={expected_value}"
        )

inner_fold_vector_14 = np.array(
    [
        hospital_to_inner_fold_14.get(hospital, -1)
        for hospital in groups_outer_training_14
    ],
    dtype=int,
)

if (inner_fold_vector_14 == -1).any():
    raise RuntimeError(
        "Some outer-training hospitals have no inner-fold assignment."
    )

if set(np.unique(inner_fold_vector_14)) != {1, 2, 3, 4, 5}:
    raise RuntimeError("Inner-fold values are not exactly 1–5.")

# ------------------------------------------------------------
# 5. Preprocessor and model constructors
# ------------------------------------------------------------

def make_preprocessor_14():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns_07B,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns_07B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_xgb_model_14(candidate):
    return XGBClassifier(
        n_estimators=int(candidate["n_estimators"]),
        max_depth=int(candidate["max_depth"]),
        learning_rate=float(candidate["learning_rate"]),
        min_child_weight=float(candidate["min_child_weight"]),
        subsample=float(candidate["subsample"]),
        colsample_bytree=float(candidate["colsample_bytree"]),
        gamma=float(candidate["gamma"]),
        reg_alpha=float(candidate["reg_alpha"]),
        reg_lambda=float(candidate["reg_lambda"]),
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        max_bin=256,
        scale_pos_weight=1.0,
        importance_type="gain",
        random_state=MODEL_RANDOM_SEED_14,
        n_jobs=-1,
        verbosity=0,
    )


def checkpoint_table_id_14(inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_xgb_inner_oof_outer2_inner{inner_fold}_v1"
    )

# ------------------------------------------------------------
# 6. BigQuery checkpoint verification
# ------------------------------------------------------------

def verify_checkpoint_14(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):
    table_id = checkpoint_table_id_14(inner_fold)

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS distinct_id_count,
      COUNT(DISTINCT candidate_id) AS candidate_count,
      COUNT(DISTINCT outer_fold) AS outer_fold_count,
      COUNT(DISTINCT inner_fold) AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(candidate_id, '|', id_row)
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(
        prediction_raw < 0 OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(
        sql,
        location=BQ_LOCATION,
    ).to_dataframe()

    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows * len(candidate_grid_14)
    )
    expected_positive_rows = (
        expected_validation_events * len(candidate_grid_14)
    )
    expected_negative_rows = (
        (expected_validation_rows - expected_validation_events)
        * len(candidate_grid_14)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": expected_validation_rows,
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": expected_total_rows,
        "positive_prediction_rows": expected_positive_rows,
        "negative_prediction_rows": expected_negative_rows,
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": 2,
        "maximum_outer_fold": 2,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failures = []

    for field, expected_value in expected_values.items():
        actual_value = int(row[field])
        if actual_value != expected_value:
            complete = False
            failures.append(
                f"{field}={actual_value}, expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failures),
        "check": check,
        "row": row,
    }

checkpoint_load_config_14 = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField(
            "id_row", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "outer_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "inner_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "candidate_id", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "label_stage23", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "prediction_raw", "FLOAT", mode="REQUIRED"
        ),
    ],
    write_disposition=(
        bigquery.WriteDisposition.WRITE_TRUNCATE
    ),
)

# ------------------------------------------------------------
# 7. Aggregate fit audit
# ------------------------------------------------------------

fit_audit_columns_14 = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "n_estimators",
    "max_depth",
    "learning_rate",
    "min_child_weight",
    "subsample",
    "colsample_bytree",
    "gamma",
    "reg_alpha",
    "reg_lambda",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "fit_seconds",
]

fit_audit_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14A_xgboost_inner_fit_audit_outer2.csv",
)

if os.path.exists(fit_audit_path_14):
    fit_audit_14 = pd.read_csv(fit_audit_path_14)
else:
    fit_audit_14 = pd.DataFrame(
        columns=fit_audit_columns_14
    )

for column in fit_audit_columns_14:
    if column not in fit_audit_14.columns:
        fit_audit_14[column] = np.nan

fit_audit_14 = fit_audit_14[
    fit_audit_columns_14
].copy()

# ------------------------------------------------------------
# 8. Run five inner folds
# ------------------------------------------------------------

for inner_fold in range(1, 6):
    inner_training_mask = (
        inner_fold_vector_14 != inner_fold
    )
    inner_validation_mask = (
        inner_fold_vector_14 == inner_fold
    )

    training_rows = int(inner_training_mask.sum())
    validation_rows = int(inner_validation_mask.sum())
    training_events = int(
        y_outer_training_14[inner_training_mask].sum()
    )
    validation_events = int(
        y_outer_training_14[inner_validation_mask].sum()
    )

    existing_check = verify_checkpoint_14(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if existing_check["complete"]:
        print(
            f"Outer 2 / inner {inner_fold}: "
            "permanent XGBoost checkpoint already complete; "
            "skipping model fitting."
        )
        continue

    training_hospital_set = set(
        groups_outer_training_14[inner_training_mask]
    )
    validation_hospital_set = set(
        groups_outer_training_14[inner_validation_mask]
    )

    if training_hospital_set & validation_hospital_set:
        raise RuntimeError(
            f"Inner fold {inner_fold}: hospital overlap detected."
        )

    print(f"\nOuter 2 / inner {inner_fold}")
    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_14()

    preprocessing_started = time.time()

    X_inner_training_processed = (
        preprocessor.fit_transform(
            X_outer_training_14.loc[
                inner_training_mask
            ]
        )
    )

    X_inner_validation_processed = (
        preprocessor.transform(
            X_outer_training_14.loc[
                inner_validation_mask
            ]
        )
    )

    preprocessing_seconds = (
        time.time() - preprocessing_started
    )

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "Processed training/validation column counts differ."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_seconds, 2),
    )

    y_inner_training = (
        y_outer_training_14[inner_training_mask]
    )
    y_inner_validation = (
        y_outer_training_14[inner_validation_mask]
    )

    validation_ids = (
        outer_training_meta_14.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_14:
        candidate_id = candidate["candidate_id"]

        print(
            "  Fitting",
            candidate_id,
            "| trees =",
            candidate["n_estimators"],
            "| depth =",
            candidate["max_depth"],
            "| lr =",
            candidate["learning_rate"],
        )

        model = make_xgb_model_14(candidate)

        fit_started = time.time()

        model.fit(
            X_inner_training_processed,
            y_inner_training,
        )

        fit_seconds = time.time() - fit_started

        validation_probabilities = (
            model.predict_proba(
                X_inner_validation_processed
            )[:, 1]
        )

        if np.isnan(validation_probabilities).any():
            raise RuntimeError(
                f"{candidate_id}, inner {inner_fold}: "
                "missing predictions."
            )

        if not np.all(
            (validation_probabilities >= 0)
            & (validation_probabilities <= 1)
        ):
            raise RuntimeError(
                f"{candidate_id}: invalid probabilities."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        2,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": (
                        y_inner_validation.astype(np.int64)
                    ),
                    "prediction_raw": (
                        validation_probabilities.astype(
                            np.float64
                        )
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": 2,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "n_estimators": candidate[
                    "n_estimators"
                ],
                "max_depth": candidate[
                    "max_depth"
                ],
                "learning_rate": candidate[
                    "learning_rate"
                ],
                "min_child_weight": candidate[
                    "min_child_weight"
                ],
                "subsample": candidate["subsample"],
                "colsample_bytree": candidate[
                    "colsample_bytree"
                ],
                "gamma": candidate["gamma"],
                "reg_alpha": candidate["reg_alpha"],
                "reg_lambda": candidate[
                    "reg_lambda"
                ],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": (
                    validation_events
                ),
                "processed_columns": int(
                    X_inner_training_processed.shape[1]
                ),
                "fit_seconds": float(fit_seconds),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows * len(candidate_grid_14)
    )

    if len(checkpoint_df) != expected_checkpoint_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: invalid checkpoint row count."
        )

    if checkpoint_df.duplicated(
        subset=["id_row", "candidate_id"]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: duplicate candidate-patient rows."
        )

    target_checkpoint_table = (
        checkpoint_table_id_14(inner_fold)
    )

    print(
        "Uploading permanent XGBoost checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_14,
        location=BQ_LOCATION,
    ).result()

    if len(fit_audit_14) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_14["outer_fold"],
                    errors="coerce",
                ) == 1
            )
            & (
                pd.to_numeric(
                    fit_audit_14["inner_fold"],
                    errors="coerce",
                ) == inner_fold
            )
        )

        fit_audit_14 = (
            fit_audit_14.loc[keep_mask].copy()
        )

    fit_audit_14 = pd.concat(
        [
            fit_audit_14,
            pd.DataFrame(current_audit_rows),
        ],
        ignore_index=True,
    )

    fit_audit_14 = (
        fit_audit_14[
            fit_audit_columns_14
        ]
        .sort_values(
            [
                "outer_fold",
                "inner_fold",
                "candidate_id",
            ]
        )
        .reset_index(drop=True)
    )

    fit_audit_14.to_csv(
        fit_audit_path_14,
        index=False,
    )

    completed_check = verify_checkpoint_14(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint verification failed: "
            + completed_check["reason"]
        )

    print(
        f"Outer 2 / inner {inner_fold}: "
        "permanent XGBoost checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 9. Final inner checkpoint summary
# ------------------------------------------------------------

checkpoint_summary_rows_14 = []

for inner_fold in range(1, 6):
    validation_mask = (
        inner_fold_vector_14 == inner_fold
    )
    validation_rows = int(validation_mask.sum())
    validation_events = int(
        y_outer_training_14[
            validation_mask
        ].sum()
    )

    final_check = verify_checkpoint_14(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: final checkpoint audit failed. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_14.append(
        {
            "outer_fold": 2,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(
                row["row_count"]
            ),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(
                row["candidate_count"]
            ),
            "positive_prediction_rows": int(
                row["positive_prediction_rows"]
            ),
            "negative_prediction_rows": int(
                row["negative_prediction_rows"]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check["table_id"],
        }
    )

checkpoint_summary_14 = (
    pd.DataFrame(checkpoint_summary_rows_14)
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_14[
        "distinct_validation_patients"
    ].sum()
) != EXPECTED_SPLIT_14["training_rows"]:
    raise RuntimeError(
        "Total inner validation patients != 46,803."
    )

expected_total_oof_rows_14 = (
    EXPECTED_SPLIT_14["training_rows"]
    * len(candidate_grid_14)
)

if int(
    checkpoint_summary_14[
        "checkpoint_rows"
    ].sum()
) != expected_total_oof_rows_14:
    raise RuntimeError(
        "Total XGBoost OOF prediction rows are incorrect."
    )

checkpoint_summary_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14A_xgboost_outer2_inner_checkpoint_summary.csv",
)

checkpoint_summary_14.to_csv(
    checkpoint_summary_path_14,
    index=False,
)

# ------------------------------------------------------------
# 10. Pool five inner OOF tables
# ------------------------------------------------------------

checkpoint_tables_14 = [
    checkpoint_table_id_14(inner_fold)
    for inner_fold in range(1, 6)
]

union_parts_14 = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_14
]

SQL_LOAD_POOLED_OOF_14 = (
    "\nUNION ALL\n".join(union_parts_14)
)

print(
    "\nLoading pooled outer-fold-2 XGBoost inner OOF predictions..."
)

query_job_14 = client.query(
    SQL_LOAD_POOLED_OOF_14,
    location=BQ_LOCATION,
)

try:
    pooled_oof_14 = query_job_14.to_dataframe(
        create_bqstorage_client=True
    )
    pooled_load_method_14 = (
        "BigQuery Storage API"
    )
except Exception as fast_path_error_14:
    print(
        "Storage API unavailable; using standard BigQuery download."
    )
    print(
        "Message:",
        type(fast_path_error_14).__name__,
    )
    pooled_oof_14 = query_job_14.to_dataframe(
        create_bqstorage_client=False
    )
    pooled_load_method_14 = (
        "Standard BigQuery API"
    )

pooled_oof_14["id_row"] = (
    pooled_oof_14["id_row"].astype(str)
)
pooled_oof_14["candidate_id"] = (
    pooled_oof_14["candidate_id"].astype(str)
)

for column in [
    "outer_fold",
    "inner_fold",
    "label_stage23",
]:
    pooled_oof_14[column] = pd.to_numeric(
        pooled_oof_14[column],
        errors="raise",
    ).astype(int)

pooled_oof_14["prediction_raw"] = pd.to_numeric(
    pooled_oof_14["prediction_raw"],
    errors="raise",
).astype(float)

if len(pooled_oof_14) != expected_total_oof_rows_14:
    raise RuntimeError(
        "Pooled XGBoost OOF row count is incorrect."
    )

if pooled_oof_14.duplicated(
    subset=["candidate_id", "id_row"]
).any():
    raise RuntimeError(
        "Duplicate candidate-patient row in pooled XGBoost OOF."
    )

if pooled_oof_14["prediction_raw"].isna().any():
    raise RuntimeError("Missing XGBoost OOF prediction.")

if not pooled_oof_14[
    "prediction_raw"
].between(0, 1).all():
    raise RuntimeError(
        "Invalid XGBoost OOF probability."
    )

if set(
    pooled_oof_14["candidate_id"].unique()
) != {
    "XGB01",
    "XGB02",
    "XGB03",
    "XGB04",
    "XGB05",
    "XGB06",
}:
    raise RuntimeError(
        "The six locked XGBoost candidates are not all present."
    )

candidate_patient_counts_14 = (
    pooled_oof_14
    .groupby("candidate_id")["id_row"]
    .nunique()
)

if not (
    candidate_patient_counts_14
    == EXPECTED_SPLIT_14["training_rows"]
).all():
    raise RuntimeError(
        "Each candidate must have 46,803 OOF patients."
    )

candidate_event_counts_14 = (
    pooled_oof_14
    .groupby("candidate_id")["label_stage23"]
    .sum()
)

if not (
    candidate_event_counts_14
    == EXPECTED_SPLIT_14["training_events"]
).all():
    raise RuntimeError(
        "Each candidate must have 2,426 OOF events."
    )

# ------------------------------------------------------------
# 11. Metric helpers
# ------------------------------------------------------------

def probability_metrics_14(
    y_true,
    probabilities,
):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(
            roc_auc_score(y_true, probabilities)
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1],
            )
        ),
        "mean_predicted_risk": float(
            probabilities.mean()
        ),
        "observed_event_rate": float(
            np.mean(y_true)
        ),
    }


def probability_logit_14(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        probabilities / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_14(
    y_true,
    probabilities,
):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        probability_logit_14(probabilities),
        y_true,
    )

    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 12. Candidate pooled inner OOF metrics
# ------------------------------------------------------------

candidate_result_rows_14 = []

for candidate in candidate_grid_14:
    candidate_id = candidate["candidate_id"]

    candidate_oof = (
        pooled_oof_14.loc[
            pooled_oof_14["candidate_id"]
            == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_14(
        candidate_oof[
            "label_stage23"
        ].to_numpy(dtype=int),
        candidate_oof[
            "prediction_raw"
        ].to_numpy(dtype=float),
    )

    fit_part = fit_audit_14.loc[
        fit_audit_14[
            "candidate_id"
        ].astype(str) == candidate_id
    ]

    fit_seconds_total = (
        float(
            pd.to_numeric(
                fit_part["fit_seconds"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_14.append(
        {
            "candidate_id": candidate_id,
            **{
                key: candidate[key]
                for key in candidate
                if key != "candidate_id"
            },
            **metrics,
            "fit_seconds_total": (
                fit_seconds_total
            ),
        }
    )

candidate_results_14 = pd.DataFrame(
    candidate_result_rows_14
)

candidate_results_14 = (
    candidate_results_14
    .sort_values(
        ["auprc", "auroc", "brier"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

candidate_results_14["selection_rank"] = (
    np.arange(
        1,
        len(candidate_results_14) + 1,
    )
)

best_row_14 = candidate_results_14.iloc[0]
selected_candidate_id_14 = str(
    best_row_14["candidate_id"]
)

selected_candidate_14 = next(
    candidate
    for candidate in candidate_grid_14
    if candidate["candidate_id"]
    == selected_candidate_id_14
)

# ------------------------------------------------------------
# 13. Platt calibration from selected inner OOF
# ------------------------------------------------------------

selected_oof_14 = (
    pooled_oof_14.loc[
        pooled_oof_14["candidate_id"]
        == selected_candidate_id_14
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_14 = (
    selected_oof_14[
        "label_stage23"
    ].to_numpy(dtype=int)
)
selected_oof_probability_14 = (
    selected_oof_14[
        "prediction_raw"
    ].to_numpy(dtype=float)
)

platt_calibrator_14 = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_14.fit(
    probability_logit_14(
        selected_oof_probability_14
    ),
    selected_oof_y_14,
)

platt_intercept_14 = float(
    platt_calibrator_14.intercept_[0]
)
platt_slope_14 = float(
    platt_calibrator_14.coef_[0][0]
)

if (
    not np.isfinite(platt_intercept_14)
    or not np.isfinite(platt_slope_14)
    or platt_slope_14 <= 0
):
    raise RuntimeError(
        "Invalid Platt calibration coefficients."
    )

selected_model_14 = pd.DataFrame(
    [
        {
            "outer_fold": 2,
            "selected_candidate": (
                selected_candidate_id_14
            ),
            "selection_metric_primary": (
                "pooled_inner_oof_auprc"
            ),
            "inner_oof_auprc": float(
                best_row_14["auprc"]
            ),
            "inner_oof_auroc": float(
                best_row_14["auroc"]
            ),
            "inner_oof_brier": float(
                best_row_14["brier"]
            ),
            "inner_oof_log_loss": float(
                best_row_14["log_loss"]
            ),
            "inner_oof_mean_predicted_risk": float(
                best_row_14[
                    "mean_predicted_risk"
                ]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_14[
                    "observed_event_rate"
                ]
            ),
            "platt_intercept": (
                platt_intercept_14
            ),
            "platt_slope": platt_slope_14,
            "protocol_sha256": (
                protocol_sha_14
            ),
            **{
                key: selected_candidate_14[key]
                for key in selected_candidate_14
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 14. Save locked selection
# ------------------------------------------------------------

candidate_results_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14B_xgboost_candidate_results_outer2.csv",
)

selected_model_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14B_xgboost_selected_model_outer2.csv",
)

selection_json_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14B_xgboost_selection_calibration_outer2.json",
)

selection_sha_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14B_xgboost_selection_calibration_outer2_SHA256.txt",
)

candidate_results_14.to_csv(
    candidate_results_path_14,
    index=False,
)
selected_model_14.to_csv(
    selected_model_path_14,
    index=False,
)

selection_configuration_14 = {
    "outer_fold": 2,
    "protocol_sha256": protocol_sha_14,
    "selection_metric_primary": (
        "pooled inner out-of-fold AUPRC"
    ),
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "selected_candidate": (
        selected_candidate_id_14
    ),
    "selected_hyperparameters": {
        key: selected_candidate_14[key]
        for key in selected_candidate_14
        if key != "candidate_id"
    },
    "inner_oof_auprc": float(
        best_row_14["auprc"]
    ),
    "inner_oof_auroc": float(
        best_row_14["auroc"]
    ),
    "inner_oof_brier": float(
        best_row_14["brier"]
    ),
    "platt_intercept": platt_intercept_14,
    "platt_slope": platt_slope_14,
    "inner_checkpoint_tables": (
        checkpoint_tables_14
    ),
    "patient_level_oof_written_to_drive": False,
}

with open(
    selection_json_path_14,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        selection_configuration_14,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    selection_json_path_14,
    "rb",
) as fh:
    selection_sha_14 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    selection_sha_path_14,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(selection_sha_14 + "\n")

# ------------------------------------------------------------
# 15. Fit selected model on all outer training patients
# ------------------------------------------------------------

final_preprocessor_14 = (
    make_preprocessor_14()
)

print(
    "\nFitting selected outer-fold-2 XGBoost model "
    "on all 46,803 training patients..."
)

preprocess_started_14 = time.time()

X_outer_training_processed_14 = (
    final_preprocessor_14.fit_transform(
        X_outer_training_14
    )
)
X_outer_test_processed_14 = (
    final_preprocessor_14.transform(
        X_outer_test_14
    )
)

final_preprocessing_seconds_14 = (
    time.time() - preprocess_started_14
)

final_model_14 = make_xgb_model_14(
    selected_candidate_14
)

final_fit_started_14 = time.time()

final_model_14.fit(
    X_outer_training_processed_14,
    y_outer_training_14,
)

final_fit_seconds_14 = (
    time.time() - final_fit_started_14
)

outer2_raw_probabilities_14 = (
    final_model_14.predict_proba(
        X_outer_test_processed_14
    )[:, 1]
)

raw_clipped_14 = np.clip(
    outer2_raw_probabilities_14,
    1e-6,
    1 - 1e-6,
)
raw_logit_14 = np.log(
    raw_clipped_14
    / (1 - raw_clipped_14)
)

outer2_platt_probabilities_14 = expit(
    platt_intercept_14
    + platt_slope_14 * raw_logit_14
)

for probabilities, name in [
    (outer2_raw_probabilities_14, "raw"),
    (
        outer2_platt_probabilities_14,
        "platt",
    ),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(
            f"{name} test predictions contain missing values."
        )

    if not np.all(
        (probabilities >= 0)
        & (probabilities <= 1)
    ):
        raise RuntimeError(
            f"{name} test predictions contain invalid probabilities."
        )

raw_metrics_14 = probability_metrics_14(
    y_outer_test_14,
    outer2_raw_probabilities_14,
)
platt_metrics_14 = probability_metrics_14(
    y_outer_test_14,
    outer2_platt_probabilities_14,
)

raw_calibration_intercept_14, \
raw_calibration_slope_14 = (
    calibration_intercept_slope_14(
        y_outer_test_14,
        outer2_raw_probabilities_14,
    )
)

platt_calibration_intercept_14, \
platt_calibration_slope_14 = (
    calibration_intercept_slope_14(
        y_outer_test_14,
        outer2_platt_probabilities_14,
    )
)

outer2_test_results_14 = pd.DataFrame(
    [
        {
            "outer_fold": 2,
            "model": "xgboost",
            "probability_type": "raw",
            **raw_metrics_14,
            "calibration_intercept": (
                raw_calibration_intercept_14
            ),
            "calibration_slope": (
                raw_calibration_slope_14
            ),
        },
        {
            "outer_fold": 2,
            "model": "xgboost",
            "probability_type": (
                "platt_calibrated"
            ),
            **platt_metrics_14,
            "calibration_intercept": (
                platt_calibration_intercept_14
            ),
            "calibration_slope": (
                platt_calibration_slope_14
            ),
        },
    ]
)

# ------------------------------------------------------------
# 16. Feature importance
# ------------------------------------------------------------

processed_feature_names_14 = (
    final_preprocessor_14
    .get_feature_names_out()
)

feature_importances_14 = (
    final_model_14.feature_importances_
)

if len(processed_feature_names_14) != len(
    feature_importances_14
):
    raise RuntimeError(
        "Processed feature names and XGBoost "
        "feature importances differ in length."
    )

feature_importance_table_14 = pd.DataFrame(
    {
        "processed_feature": (
            processed_feature_names_14
        ),
        "gain_importance": (
            feature_importances_14
        ),
    }
)

feature_importance_table_14[
    "importance_rank"
] = (
    feature_importance_table_14[
        "gain_importance"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)

feature_importance_table_14 = (
    feature_importance_table_14
    .sort_values(
        [
            "gain_importance",
            "processed_feature",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

nonzero_importance_features_14 = int(
    (
        feature_importance_table_14[
            "gain_importance"
        ] > 0
    ).sum()
)

final_model_summary_14 = pd.DataFrame(
    [
        {
            "outer_fold": 2,
            "selected_candidate": (
                selected_candidate_id_14
            ),
            "training_patients": len(
                X_outer_training_14
            ),
            "training_hospitals": len(
                training_hospitals_14
            ),
            "training_events": int(
                y_outer_training_14.sum()
            ),
            "test_patients": len(
                X_outer_test_14
            ),
            "test_hospitals": len(
                test_hospitals_14
            ),
            "test_events": int(
                y_outer_test_14.sum()
            ),
            "hospital_overlap": len(
                hospital_overlap_14
            ),
            "processed_feature_columns": len(
                processed_feature_names_14
            ),
            "nonzero_importance_features": (
                nonzero_importance_features_14
            ),
            "preprocessing_seconds": float(
                final_preprocessing_seconds_14
            ),
            "fit_seconds": float(
                final_fit_seconds_14
            ),
            "locked_platt_intercept": (
                platt_intercept_14
            ),
            "locked_platt_slope": (
                platt_slope_14
            ),
            "protocol_sha256": protocol_sha_14,
            "selection_sha256": (
                selection_sha_14
            ),
            **{
                key: selected_candidate_14[key]
                for key in selected_candidate_14
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 17. Secure outer-test prediction checkpoint
# ------------------------------------------------------------

outer2_prediction_df_14 = pd.DataFrame(
    {
        "id_row": (
            outer_test_meta_14[
                "id_row"
            ].astype(str)
        ),
        "outer_fold": np.full(
            len(outer_test_meta_14),
            2,
            dtype=np.int64,
        ),
        "label_stage23": (
            y_outer_test_14.astype(np.int64)
        ),
        "prediction_raw": (
            outer2_raw_probabilities_14.astype(
                np.float64
            )
        ),
        "prediction_platt": (
            outer2_platt_probabilities_14.astype(
                np.float64
            )
        ),
        "model_name": "xgboost",
        "model_version": (
            "core_v1_nested_cv"
        ),
    }
)

if len(outer2_prediction_df_14) != 11691:
    raise RuntimeError(
        "Outer-fold-2 test prediction row count != 11,691."
    )

if outer2_prediction_df_14[
    "id_row"
].duplicated().any():
    raise RuntimeError(
        "Duplicate id_row in outer-fold-2 XGBoost predictions."
    )

if int(
    outer2_prediction_df_14[
        "label_stage23"
    ].sum()
) != 606:
    raise RuntimeError(
        "Outer-fold-2 XGBoost event count != 606."
    )

prediction_table_id_14 = (
    f"{TARGET_DATASET}."
    "model_xgb_outer_predictions_outer2_v1"
)

prediction_load_config_14 = (
    bigquery.LoadJobConfig(
        schema=[
            bigquery.SchemaField(
                "id_row",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "outer_fold",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "label_stage23",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_raw",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_platt",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_name",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_version",
                "STRING",
                mode="REQUIRED",
            ),
        ],
        write_disposition=(
            bigquery.WriteDisposition.WRITE_TRUNCATE
        ),
    )
)

print(
    "\nUploading secure outer-fold-2 "
    "XGBoost prediction checkpoint:"
)
print(prediction_table_id_14)

client.load_table_from_dataframe(
    outer2_prediction_df_14,
    prediction_table_id_14,
    job_config=prediction_load_config_14,
    location=BQ_LOCATION,
).result()

SQL_VERIFY_PREDICTIONS_14 = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL)
    AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL)
    AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0
    OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0
    OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw)
    AS minimum_raw_probability,
  MAX(prediction_raw)
    AS maximum_raw_probability,
  MIN(prediction_platt)
    AS minimum_platt_probability,
  MAX(prediction_platt)
    AS maximum_platt_probability
FROM `{prediction_table_id_14}`;
"""

prediction_verification_14 = (
    client.query(
        SQL_VERIFY_PREDICTIONS_14,
        location=BQ_LOCATION,
    )
    .to_dataframe()
)

verification_row_14 = (
    prediction_verification_14.iloc[0]
)

expected_prediction_values_14 = {
    "prediction_rows": 11691,
    "distinct_rows": 11691,
    "outer_folds": 1,
    "minimum_outer_fold": 2,
    "maximum_outer_fold": 2,
    "events": 606,
    "nonevents": 11082,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in (
    expected_prediction_values_14.items()
):
    actual_value = int(
        verification_row_14[field]
    )
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: found={actual_value}, "
            f"expected={expected_value}"
        )

# ------------------------------------------------------------
# 18. Save aggregate outputs
# ------------------------------------------------------------

test_results_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14C_xgboost_outer2_test_results.csv",
)

model_summary_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14C_xgboost_final_model_outer2.csv",
)

importance_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14C_xgboost_gain_importance_outer2.csv",
)

evaluation_json_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14C_xgboost_final_evaluation_outer2.json",
)

evaluation_sha_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14C_xgboost_final_evaluation_outer2_SHA256.txt",
)

outer2_test_results_14.to_csv(
    test_results_path_14,
    index=False,
)
final_model_summary_14.to_csv(
    model_summary_path_14,
    index=False,
)
feature_importance_table_14.to_csv(
    importance_path_14,
    index=False,
)

evaluation_configuration_14 = {
    "outer_fold": 2,
    "model_family": "xgboost",
    "protocol_sha256": protocol_sha_14,
    "selection_sha256": selection_sha_14,
    "selected_candidate": (
        selected_candidate_id_14
    ),
    "selected_hyperparameters": {
        key: selected_candidate_14[key]
        for key in selected_candidate_14
        if key != "candidate_id"
    },
    "training_patients": 46800,
    "training_hospitals": 158,
    "test_patients": 11691,
    "test_hospitals": 40,
    "hospital_overlap": 0,
    "locked_platt_intercept": (
        platt_intercept_14
    ),
    "locked_platt_slope": platt_slope_14,
    "processed_feature_columns": int(
        len(processed_feature_names_14)
    ),
    "secure_prediction_table": (
        prediction_table_id_14
    ),
    "patient_level_prediction_written_to_drive": False,
}

with open(
    evaluation_json_path_14,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        evaluation_configuration_14,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    evaluation_json_path_14,
    "rb",
) as fh:
    evaluation_sha_14 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    evaluation_sha_path_14,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(evaluation_sha_14 + "\n")

# ------------------------------------------------------------
# 19. Display results
# ------------------------------------------------------------

pooled_integrity_14 = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_14),
            pooled_oof_14[
                "id_row"
            ].nunique(),
            pooled_oof_14[
                "candidate_id"
            ].nunique(),
            pooled_oof_14[
                "inner_fold"
            ].nunique(),
            EXPECTED_SPLIT_14[
                "training_events"
            ],
            (
                EXPECTED_SPLIT_14[
                    "training_rows"
                ]
                - EXPECTED_SPLIT_14[
                    "training_events"
                ]
            ),
            int(
                pooled_oof_14.duplicated(
                    subset=[
                        "candidate_id",
                        "id_row",
                    ]
                ).sum()
            ),
            int(
                pooled_oof_14[
                    "prediction_raw"
                ].isna().sum()
            ),
            int(
                (
                    ~pooled_oof_14[
                        "prediction_raw"
                    ].between(0, 1)
                ).sum()
            ),
            pooled_load_method_14,
        ],
    }
)

print(
    "\n14 XGBOOST OUTER-FOLD-2 INNER CHECKPOINT SUMMARY"
)
display(checkpoint_summary_14)

print(
    "\n14 XGBOOST OUTER-FOLD-2 POOLED OOF INTEGRITY"
)
display(pooled_integrity_14)

print(
    "\n14 XGBOOST OUTER-FOLD-2 CANDIDATE RESULTS"
)
display(candidate_results_14)

print(
    "\n14 XGBOOST OUTER-FOLD-2 SELECTED MODEL"
)
display(selected_model_14)

print(
    "\n14 XGBOOST OUTER-FOLD-2 FINAL MODEL SUMMARY"
)
display(final_model_summary_14)

print(
    "\n14 XGBOOST OUTER-FOLD-2 TEST RESULTS"
)
display(outer2_test_results_14)

print(
    "\n14 XGBOOST OUTER-FOLD-2 BIGQUERY VERIFICATION"
)
display(prediction_verification_14)

print(
    "\n14 XGBOOST OUTER-FOLD-2 TOP 20 GAIN IMPORTANCE FEATURES"
)
display(feature_importance_table_14.head(20))

print("\nXGBoost protocol SHA-256:")
print(protocol_sha_14)

print("\nSelection SHA-256:")
print(selection_sha_14)

print("\nEvaluation SHA-256:")
print(evaluation_sha_14)

print("\nSaved aggregate outputs:")
print(protocol_path_14)
print(protocol_sha_path_14)
print(fit_audit_path_14)
print(checkpoint_summary_path_14)
print(candidate_results_path_14)
print(selected_model_path_14)
print(selection_json_path_14)
print(selection_sha_path_14)
print(test_results_path_14)
print(model_summary_path_14)
print(importance_path_14)
print(evaluation_json_path_14)
print(evaluation_sha_path_14)

print(
    "\n14 PASS: XGBoost outer-fold-2 nested modelling "
    "and locked test evaluation are complete."
)

print(
    "No class weighting, SMOTE, or early stopping was used."
)

print(
    "All patient-level OOF and outer-test predictions "
    "were stored only in BigQuery."
)

print(
    "No patient-level prediction file was written to Google Drive."
)

_ = gc.collect()

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

from xgboost import XGBClassifier

from IPython.display import display

print("STARTING XGBOOST OUTER FOLD 2 — CORRECTED CODE VERSION 14R")

# ============================================================
# 14 — XGBOOST OUTER FOLD 2 COMPLETE NESTED MODELLING
#
# Design:
# - Same locked hospital-disjoint outer folds.
# - Same locked hospital-disjoint inner folds.
# - Core feature set only.
# - No SMOTE.
# - No class weighting / scale_pos_weight = 1.
# - No early stopping.
# - Fixed candidate grid locked before outer-test evaluation.
# - Selection: pooled inner-OOF AUPRC descending,
#              AUROC descending, Brier ascending.
# - Platt calibration learned only from selected candidate's
#   pooled inner-OOF predictions.
# - Patient-level predictions stored only in BigQuery.
# ============================================================

OUTER_FOLD_14 = 2
MODEL_RANDOM_SEED_14 = 20260721

EXPECTED_SPLIT_14 = {
    "training_rows": 46800,
    "test_rows": 11691,
    "training_hospitals": 158,
    "test_hospitals": 40,
    "training_events": 2426,
    "test_events": 606,
}

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_objects_14 = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_14 = [
    name for name in required_objects_14
    if name not in globals()
]

if missing_objects_14:
    raise RuntimeError(
        "Missing runtime objects: "
        + ", ".join(missing_objects_14)
        + ". Run 07A and 07B first."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"Expected 58,491 cohort rows; found {len(core_df_07B)}."
    )

if len(predictor_columns_07B) != 159:
    raise RuntimeError("Expected 159 core predictors.")

if len(numeric_columns_07B) != 156:
    raise RuntimeError("Expected 156 numeric predictors.")

if len(categorical_columns_07B) != 3:
    raise RuntimeError("Expected 3 categorical predictors.")

# ------------------------------------------------------------
# 2. Lock the XGBoost protocol BEFORE test evaluation
# ------------------------------------------------------------

candidate_grid_14 = [
    {
        "candidate_id": "XGB01",
        "n_estimators": 250,
        "max_depth": 2,
        "learning_rate": 0.03,
        "min_child_weight": 5.0,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
    },
    {
        "candidate_id": "XGB02",
        "n_estimators": 350,
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 5.0,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
    },
    {
        "candidate_id": "XGB03",
        "n_estimators": 450,
        "max_depth": 3,
        "learning_rate": 0.02,
        "min_child_weight": 10.0,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "gamma": 0.0,
        "reg_alpha": 0.10,
        "reg_lambda": 10.0,
    },
    {
        "candidate_id": "XGB04",
        "n_estimators": 350,
        "max_depth": 4,
        "learning_rate": 0.03,
        "min_child_weight": 10.0,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "gamma": 0.10,
        "reg_alpha": 0.10,
        "reg_lambda": 10.0,
    },
    {
        "candidate_id": "XGB05",
        "n_estimators": 450,
        "max_depth": 2,
        "learning_rate": 0.02,
        "min_child_weight": 10.0,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "gamma": 0.0,
        "reg_alpha": 0.50,
        "reg_lambda": 10.0,
    },
    {
        "candidate_id": "XGB06",
        "n_estimators": 450,
        "max_depth": 4,
        "learning_rate": 0.02,
        "min_child_weight": 15.0,
        "subsample": 0.90,
        "colsample_bytree": 0.80,
        "gamma": 0.20,
        "reg_alpha": 0.50,
        "reg_lambda": 15.0,
    },
]

xgb_protocol_14 = {
    "protocol_name": "xgboost_core_nested_hospital_cv_v1",
    "model_family": "XGBoost",
    "feature_set": "core_159",
    "outer_cv": "locked 5-fold hospital-disjoint outer folds",
    "inner_cv": "locked 5-fold hospital-disjoint inner folds",
    "primary_selection_metric": "pooled inner OOF AUPRC descending",
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "class_weighting": False,
    "scale_pos_weight": 1.0,
    "smote": False,
    "early_stopping": False,
    "tree_method": "hist",
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": MODEL_RANDOM_SEED_14,
    "candidate_grid": candidate_grid_14,
    "calibration": (
        "Platt calibration fit only on selected candidate "
        "pooled inner-OOF logits"
    ),
    "outer_test_use": (
        "diagnostic evaluation only; never used for tuning"
    ),
}

protocol_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13A_locked_xgboost_model_protocol_v1.json",
)

protocol_sha_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13A_locked_xgboost_model_protocol_v1_SHA256.txt",
)

protocol_text_14 = json.dumps(
    xgb_protocol_14,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
)

protocol_sha_14 = hashlib.sha256(
    protocol_text_14.encode("utf-8")
).hexdigest()

if os.path.exists(protocol_path_14):
    with open(protocol_path_14, "r", encoding="utf-8") as fh:
        existing_protocol_text_14 = fh.read()
    existing_protocol_sha_14 = hashlib.sha256(
        existing_protocol_text_14.encode("utf-8")
    ).hexdigest()

    if existing_protocol_sha_14 != protocol_sha_14:
        raise RuntimeError(
            "An existing XGBoost protocol file differs from "
            "the currently locked protocol. Stop and audit."
        )
else:
    with open(protocol_path_14, "w", encoding="utf-8") as fh:
        fh.write(protocol_text_14)

with open(protocol_sha_path_14, "w", encoding="utf-8") as fh:
    fh.write(protocol_sha_14 + "\n")

print("Locked XGBoost protocol SHA-256:")
print(protocol_sha_14)

# ------------------------------------------------------------
# 3. Load locked inner hospital map
# ------------------------------------------------------------

inner_mapping_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_14):
    raise FileNotFoundError(
        "Locked inner-fold map not found: "
        + inner_mapping_path_14
    )

inner_mapping_all_14 = pd.read_csv(
    inner_mapping_path_14,
    dtype={"group_hospital": str},
)

inner_mapping_part_14 = (
    inner_mapping_all_14.loc[
        inner_mapping_all_14["outer_fold"].astype(int)
        == OUTER_FOLD_14,
        ["group_hospital", "inner_fold"],
    ]
    .copy()
)

inner_mapping_part_14["group_hospital"] = (
    inner_mapping_part_14["group_hospital"].astype(str)
)
inner_mapping_part_14["inner_fold"] = (
    inner_mapping_part_14["inner_fold"].astype(int)
)

if len(inner_mapping_part_14) != 158:
    raise RuntimeError(
        "Expected 158 outer-fold-2 training hospitals "
        "in the locked inner map."
    )

if inner_mapping_part_14["group_hospital"].duplicated().any():
    raise RuntimeError("Duplicate hospital in locked inner map.")

hospital_to_inner_fold_14 = dict(
    zip(
        inner_mapping_part_14["group_hospital"],
        inner_mapping_part_14["inner_fold"],
    )
)

# ------------------------------------------------------------
# 4. Prepare outer fold 1 matrices
# ------------------------------------------------------------

X_all_14 = core_df_07B[predictor_columns_07B].copy()

for column in numeric_columns_07B:
    X_all_14[column] = pd.to_numeric(
        X_all_14[column],
        errors="coerce",
    ).astype("float64")

for column in categorical_columns_07B:
    category_series = X_all_14[column].astype("object")
    X_all_14[column] = category_series.where(
        pd.notna(category_series),
        np.nan,
    )

outer_fold_vector_14 = (
    core_df_07B["outer_fold"].astype(int).to_numpy()
)

outer_training_mask_14 = (
    outer_fold_vector_14 != OUTER_FOLD_14
)
outer_test_mask_14 = (
    outer_fold_vector_14 == OUTER_FOLD_14
)

X_outer_training_14 = (
    X_all_14.loc[outer_training_mask_14]
    .reset_index(drop=True)
)

X_outer_test_14 = (
    X_all_14.loc[outer_test_mask_14]
    .reset_index(drop=True)
)

outer_training_meta_14 = (
    core_df_07B.loc[
        outer_training_mask_14,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_14 = (
    core_df_07B.loc[
        outer_test_mask_14,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [
    outer_training_meta_14,
    outer_test_meta_14,
]:
    dataframe["id_row"] = dataframe["id_row"].astype(str)
    dataframe["group_hospital"] = (
        dataframe["group_hospital"].astype(str)
    )
    dataframe["label_stage23"] = (
        dataframe["label_stage23"].astype(int)
    )

y_outer_training_14 = (
    outer_training_meta_14["label_stage23"]
    .to_numpy(dtype=np.int8)
)
y_outer_test_14 = (
    outer_test_meta_14["label_stage23"]
    .to_numpy(dtype=np.int8)
)

groups_outer_training_14 = (
    outer_training_meta_14["group_hospital"]
    .to_numpy(dtype=str)
)

training_hospitals_14 = set(
    outer_training_meta_14["group_hospital"]
)
test_hospitals_14 = set(
    outer_test_meta_14["group_hospital"]
)
hospital_overlap_14 = (
    training_hospitals_14 & test_hospitals_14
)

if hospital_overlap_14:
    raise RuntimeError(
        "Outer training/test hospital overlap detected."
    )

actual_split_14 = {
    "training_rows": len(X_outer_training_14),
    "test_rows": len(X_outer_test_14),
    "training_hospitals": len(training_hospitals_14),
    "test_hospitals": len(test_hospitals_14),
    "training_events": int(y_outer_training_14.sum()),
    "test_events": int(y_outer_test_14.sum()),
}

for metric, expected_value in EXPECTED_SPLIT_14.items():
    actual_value = actual_split_14[metric]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: found={actual_value}, "
            f"expected={expected_value}"
        )

inner_fold_vector_14 = np.array(
    [
        hospital_to_inner_fold_14.get(hospital, -1)
        for hospital in groups_outer_training_14
    ],
    dtype=int,
)

if (inner_fold_vector_14 == -1).any():
    raise RuntimeError(
        "Some outer-training hospitals have no inner-fold assignment."
    )

if set(np.unique(inner_fold_vector_14)) != {1, 2, 3, 4, 5}:
    raise RuntimeError("Inner-fold values are not exactly 1–5.")

# ------------------------------------------------------------
# 5. Preprocessor and model constructors
# ------------------------------------------------------------

def make_preprocessor_14():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns_07B,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns_07B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_xgb_model_14(candidate):
    return XGBClassifier(
        n_estimators=int(candidate["n_estimators"]),
        max_depth=int(candidate["max_depth"]),
        learning_rate=float(candidate["learning_rate"]),
        min_child_weight=float(candidate["min_child_weight"]),
        subsample=float(candidate["subsample"]),
        colsample_bytree=float(candidate["colsample_bytree"]),
        gamma=float(candidate["gamma"]),
        reg_alpha=float(candidate["reg_alpha"]),
        reg_lambda=float(candidate["reg_lambda"]),
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        max_bin=256,
        scale_pos_weight=1.0,
        importance_type="gain",
        random_state=MODEL_RANDOM_SEED_14,
        n_jobs=-1,
        verbosity=0,
    )


def checkpoint_table_id_14(inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_xgb_inner_oof_outer2_inner{inner_fold}_v1"
    )

# ------------------------------------------------------------
# 6. BigQuery checkpoint verification
# ------------------------------------------------------------

def verify_checkpoint_14(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):
    table_id = checkpoint_table_id_14(inner_fold)

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS distinct_id_count,
      COUNT(DISTINCT candidate_id) AS candidate_count,
      COUNT(DISTINCT outer_fold) AS outer_fold_count,
      COUNT(DISTINCT inner_fold) AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(candidate_id, '|', id_row)
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(
        prediction_raw < 0 OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(
        sql,
        location=BQ_LOCATION,
    ).to_dataframe()

    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows * len(candidate_grid_14)
    )
    expected_positive_rows = (
        expected_validation_events * len(candidate_grid_14)
    )
    expected_negative_rows = (
        (expected_validation_rows - expected_validation_events)
        * len(candidate_grid_14)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": expected_validation_rows,
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": expected_total_rows,
        "positive_prediction_rows": expected_positive_rows,
        "negative_prediction_rows": expected_negative_rows,
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": 2,
        "maximum_outer_fold": 2,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failures = []

    for field, expected_value in expected_values.items():
        actual_value = int(row[field])
        if actual_value != expected_value:
            complete = False
            failures.append(
                f"{field}={actual_value}, expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failures),
        "check": check,
        "row": row,
    }

checkpoint_load_config_14 = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField(
            "id_row", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "outer_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "inner_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "candidate_id", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "label_stage23", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "prediction_raw", "FLOAT", mode="REQUIRED"
        ),
    ],
    write_disposition=(
        bigquery.WriteDisposition.WRITE_TRUNCATE
    ),
)

# ------------------------------------------------------------
# 7. Aggregate fit audit
# ------------------------------------------------------------

fit_audit_columns_14 = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "n_estimators",
    "max_depth",
    "learning_rate",
    "min_child_weight",
    "subsample",
    "colsample_bytree",
    "gamma",
    "reg_alpha",
    "reg_lambda",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "fit_seconds",
]

fit_audit_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14A_xgboost_inner_fit_audit_outer2.csv",
)

if os.path.exists(fit_audit_path_14):
    fit_audit_14 = pd.read_csv(fit_audit_path_14)
else:
    fit_audit_14 = pd.DataFrame(
        columns=fit_audit_columns_14
    )

for column in fit_audit_columns_14:
    if column not in fit_audit_14.columns:
        fit_audit_14[column] = np.nan

fit_audit_14 = fit_audit_14[
    fit_audit_columns_14
].copy()

# ------------------------------------------------------------
# 8. Run five inner folds
# ------------------------------------------------------------

for inner_fold in range(1, 6):
    inner_training_mask = (
        inner_fold_vector_14 != inner_fold
    )
    inner_validation_mask = (
        inner_fold_vector_14 == inner_fold
    )

    training_rows = int(inner_training_mask.sum())
    validation_rows = int(inner_validation_mask.sum())
    training_events = int(
        y_outer_training_14[inner_training_mask].sum()
    )
    validation_events = int(
        y_outer_training_14[inner_validation_mask].sum()
    )

    existing_check = verify_checkpoint_14(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if existing_check["complete"]:
        print(
            f"Outer 2 / inner {inner_fold}: "
            "permanent XGBoost checkpoint already complete; "
            "skipping model fitting."
        )
        continue

    training_hospital_set = set(
        groups_outer_training_14[inner_training_mask]
    )
    validation_hospital_set = set(
        groups_outer_training_14[inner_validation_mask]
    )

    if training_hospital_set & validation_hospital_set:
        raise RuntimeError(
            f"Inner fold {inner_fold}: hospital overlap detected."
        )

    print(f"\nOuter 2 / inner {inner_fold}")
    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_14()

    preprocessing_started = time.time()

    X_inner_training_processed = (
        preprocessor.fit_transform(
            X_outer_training_14.loc[
                inner_training_mask
            ]
        )
    )

    X_inner_validation_processed = (
        preprocessor.transform(
            X_outer_training_14.loc[
                inner_validation_mask
            ]
        )
    )

    preprocessing_seconds = (
        time.time() - preprocessing_started
    )

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "Processed training/validation column counts differ."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_seconds, 2),
    )

    y_inner_training = (
        y_outer_training_14[inner_training_mask]
    )
    y_inner_validation = (
        y_outer_training_14[inner_validation_mask]
    )

    validation_ids = (
        outer_training_meta_14.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_14:
        candidate_id = candidate["candidate_id"]

        print(
            "  Fitting",
            candidate_id,
            "| trees =",
            candidate["n_estimators"],
            "| depth =",
            candidate["max_depth"],
            "| lr =",
            candidate["learning_rate"],
        )

        model = make_xgb_model_14(candidate)

        fit_started = time.time()

        model.fit(
            X_inner_training_processed,
            y_inner_training,
        )

        fit_seconds = time.time() - fit_started

        validation_probabilities = (
            model.predict_proba(
                X_inner_validation_processed
            )[:, 1]
        )

        if np.isnan(validation_probabilities).any():
            raise RuntimeError(
                f"{candidate_id}, inner {inner_fold}: "
                "missing predictions."
            )

        if not np.all(
            (validation_probabilities >= 0)
            & (validation_probabilities <= 1)
        ):
            raise RuntimeError(
                f"{candidate_id}: invalid probabilities."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        2,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": (
                        y_inner_validation.astype(np.int64)
                    ),
                    "prediction_raw": (
                        validation_probabilities.astype(
                            np.float64
                        )
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": 2,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "n_estimators": candidate[
                    "n_estimators"
                ],
                "max_depth": candidate[
                    "max_depth"
                ],
                "learning_rate": candidate[
                    "learning_rate"
                ],
                "min_child_weight": candidate[
                    "min_child_weight"
                ],
                "subsample": candidate["subsample"],
                "colsample_bytree": candidate[
                    "colsample_bytree"
                ],
                "gamma": candidate["gamma"],
                "reg_alpha": candidate["reg_alpha"],
                "reg_lambda": candidate[
                    "reg_lambda"
                ],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": (
                    validation_events
                ),
                "processed_columns": int(
                    X_inner_training_processed.shape[1]
                ),
                "fit_seconds": float(fit_seconds),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows * len(candidate_grid_14)
    )

    if len(checkpoint_df) != expected_checkpoint_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: invalid checkpoint row count."
        )

    if checkpoint_df.duplicated(
        subset=["id_row", "candidate_id"]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: duplicate candidate-patient rows."
        )

    target_checkpoint_table = (
        checkpoint_table_id_14(inner_fold)
    )

    print(
        "Uploading permanent XGBoost checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_14,
        location=BQ_LOCATION,
    ).result()

    if len(fit_audit_14) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_14["outer_fold"],
                    errors="coerce",
                ) == 1
            )
            & (
                pd.to_numeric(
                    fit_audit_14["inner_fold"],
                    errors="coerce",
                ) == inner_fold
            )
        )

        fit_audit_14 = (
            fit_audit_14.loc[keep_mask].copy()
        )

    fit_audit_14 = pd.concat(
        [
            fit_audit_14,
            pd.DataFrame(current_audit_rows),
        ],
        ignore_index=True,
    )

    fit_audit_14 = (
        fit_audit_14[
            fit_audit_columns_14
        ]
        .sort_values(
            [
                "outer_fold",
                "inner_fold",
                "candidate_id",
            ]
        )
        .reset_index(drop=True)
    )

    fit_audit_14.to_csv(
        fit_audit_path_14,
        index=False,
    )

    completed_check = verify_checkpoint_14(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint verification failed: "
            + completed_check["reason"]
        )

    print(
        f"Outer 2 / inner {inner_fold}: "
        "permanent XGBoost checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 9. Final inner checkpoint summary
# ------------------------------------------------------------

checkpoint_summary_rows_14 = []

for inner_fold in range(1, 6):
    validation_mask = (
        inner_fold_vector_14 == inner_fold
    )
    validation_rows = int(validation_mask.sum())
    validation_events = int(
        y_outer_training_14[
            validation_mask
        ].sum()
    )

    final_check = verify_checkpoint_14(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: final checkpoint audit failed. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_14.append(
        {
            "outer_fold": 2,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(
                row["row_count"]
            ),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(
                row["candidate_count"]
            ),
            "positive_prediction_rows": int(
                row["positive_prediction_rows"]
            ),
            "negative_prediction_rows": int(
                row["negative_prediction_rows"]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check["table_id"],
        }
    )

checkpoint_summary_14 = (
    pd.DataFrame(checkpoint_summary_rows_14)
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_14[
        "distinct_validation_patients"
    ].sum()
) != EXPECTED_SPLIT_14["training_rows"]:
    raise RuntimeError(
        "Total inner validation patients != 46,800."
    )

expected_total_oof_rows_14 = (
    EXPECTED_SPLIT_14["training_rows"]
    * len(candidate_grid_14)
)

if int(
    checkpoint_summary_14[
        "checkpoint_rows"
    ].sum()
) != expected_total_oof_rows_14:
    raise RuntimeError(
        "Total XGBoost OOF prediction rows are incorrect."
    )

checkpoint_summary_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14A_xgboost_outer2_inner_checkpoint_summary.csv",
)

checkpoint_summary_14.to_csv(
    checkpoint_summary_path_14,
    index=False,
)

# ------------------------------------------------------------
# 10. Pool five inner OOF tables
# ------------------------------------------------------------

checkpoint_tables_14 = [
    checkpoint_table_id_14(inner_fold)
    for inner_fold in range(1, 6)
]

union_parts_14 = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_14
]

SQL_LOAD_POOLED_OOF_14 = (
    "\nUNION ALL\n".join(union_parts_14)
)

print(
    "\nLoading pooled outer-fold-2 XGBoost inner OOF predictions..."
)

query_job_14 = client.query(
    SQL_LOAD_POOLED_OOF_14,
    location=BQ_LOCATION,
)

try:
    pooled_oof_14 = query_job_14.to_dataframe(
        create_bqstorage_client=True
    )
    pooled_load_method_14 = (
        "BigQuery Storage API"
    )
except Exception as fast_path_error_14:
    print(
        "Storage API unavailable; using standard BigQuery download."
    )
    print(
        "Message:",
        type(fast_path_error_14).__name__,
    )
    pooled_oof_14 = query_job_14.to_dataframe(
        create_bqstorage_client=False
    )
    pooled_load_method_14 = (
        "Standard BigQuery API"
    )

pooled_oof_14["id_row"] = (
    pooled_oof_14["id_row"].astype(str)
)
pooled_oof_14["candidate_id"] = (
    pooled_oof_14["candidate_id"].astype(str)
)

for column in [
    "outer_fold",
    "inner_fold",
    "label_stage23",
]:
    pooled_oof_14[column] = pd.to_numeric(
        pooled_oof_14[column],
        errors="raise",
    ).astype(int)

pooled_oof_14["prediction_raw"] = pd.to_numeric(
    pooled_oof_14["prediction_raw"],
    errors="raise",
).astype(float)

if len(pooled_oof_14) != expected_total_oof_rows_14:
    raise RuntimeError(
        "Pooled XGBoost OOF row count is incorrect."
    )

if pooled_oof_14.duplicated(
    subset=["candidate_id", "id_row"]
).any():
    raise RuntimeError(
        "Duplicate candidate-patient row in pooled XGBoost OOF."
    )

if pooled_oof_14["prediction_raw"].isna().any():
    raise RuntimeError("Missing XGBoost OOF prediction.")

if not pooled_oof_14[
    "prediction_raw"
].between(0, 1).all():
    raise RuntimeError(
        "Invalid XGBoost OOF probability."
    )

if set(
    pooled_oof_14["candidate_id"].unique()
) != {
    "XGB01",
    "XGB02",
    "XGB03",
    "XGB04",
    "XGB05",
    "XGB06",
}:
    raise RuntimeError(
        "The six locked XGBoost candidates are not all present."
    )

candidate_patient_counts_14 = (
    pooled_oof_14
    .groupby("candidate_id")["id_row"]
    .nunique()
)

if not (
    candidate_patient_counts_14
    == EXPECTED_SPLIT_14["training_rows"]
).all():
    raise RuntimeError(
        "Each candidate must have 46,800 OOF patients."
    )

candidate_event_counts_14 = (
    pooled_oof_14
    .groupby("candidate_id")["label_stage23"]
    .sum()
)

if not (
    candidate_event_counts_14
    == EXPECTED_SPLIT_14["training_events"]
).all():
    raise RuntimeError(
        "Each candidate must have 2,426 OOF events."
    )

# ------------------------------------------------------------
# 11. Metric helpers
# ------------------------------------------------------------

def probability_metrics_14(
    y_true,
    probabilities,
):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(
            roc_auc_score(y_true, probabilities)
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1],
            )
        ),
        "mean_predicted_risk": float(
            probabilities.mean()
        ),
        "observed_event_rate": float(
            np.mean(y_true)
        ),
    }


def probability_logit_14(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        probabilities / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_14(
    y_true,
    probabilities,
):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        probability_logit_14(probabilities),
        y_true,
    )

    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 12. Candidate pooled inner OOF metrics
# ------------------------------------------------------------

candidate_result_rows_14 = []

for candidate in candidate_grid_14:
    candidate_id = candidate["candidate_id"]

    candidate_oof = (
        pooled_oof_14.loc[
            pooled_oof_14["candidate_id"]
            == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_14(
        candidate_oof[
            "label_stage23"
        ].to_numpy(dtype=int),
        candidate_oof[
            "prediction_raw"
        ].to_numpy(dtype=float),
    )

    fit_part = fit_audit_14.loc[
        fit_audit_14[
            "candidate_id"
        ].astype(str) == candidate_id
    ]

    fit_seconds_total = (
        float(
            pd.to_numeric(
                fit_part["fit_seconds"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_14.append(
        {
            "candidate_id": candidate_id,
            **{
                key: candidate[key]
                for key in candidate
                if key != "candidate_id"
            },
            **metrics,
            "fit_seconds_total": (
                fit_seconds_total
            ),
        }
    )

candidate_results_14 = pd.DataFrame(
    candidate_result_rows_14
)

candidate_results_14 = (
    candidate_results_14
    .sort_values(
        ["auprc", "auroc", "brier"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

candidate_results_14["selection_rank"] = (
    np.arange(
        1,
        len(candidate_results_14) + 1,
    )
)

best_row_14 = candidate_results_14.iloc[0]
selected_candidate_id_14 = str(
    best_row_14["candidate_id"]
)

selected_candidate_14 = next(
    candidate
    for candidate in candidate_grid_14
    if candidate["candidate_id"]
    == selected_candidate_id_14
)

# ------------------------------------------------------------
# 13. Platt calibration from selected inner OOF
# ------------------------------------------------------------

selected_oof_14 = (
    pooled_oof_14.loc[
        pooled_oof_14["candidate_id"]
        == selected_candidate_id_14
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_14 = (
    selected_oof_14[
        "label_stage23"
    ].to_numpy(dtype=int)
)
selected_oof_probability_14 = (
    selected_oof_14[
        "prediction_raw"
    ].to_numpy(dtype=float)
)

platt_calibrator_14 = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_14.fit(
    probability_logit_14(
        selected_oof_probability_14
    ),
    selected_oof_y_14,
)

platt_intercept_14 = float(
    platt_calibrator_14.intercept_[0]
)
platt_slope_14 = float(
    platt_calibrator_14.coef_[0][0]
)

if (
    not np.isfinite(platt_intercept_14)
    or not np.isfinite(platt_slope_14)
    or platt_slope_14 <= 0
):
    raise RuntimeError(
        "Invalid Platt calibration coefficients."
    )

selected_model_14 = pd.DataFrame(
    [
        {
            "outer_fold": 2,
            "selected_candidate": (
                selected_candidate_id_14
            ),
            "selection_metric_primary": (
                "pooled_inner_oof_auprc"
            ),
            "inner_oof_auprc": float(
                best_row_14["auprc"]
            ),
            "inner_oof_auroc": float(
                best_row_14["auroc"]
            ),
            "inner_oof_brier": float(
                best_row_14["brier"]
            ),
            "inner_oof_log_loss": float(
                best_row_14["log_loss"]
            ),
            "inner_oof_mean_predicted_risk": float(
                best_row_14[
                    "mean_predicted_risk"
                ]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_14[
                    "observed_event_rate"
                ]
            ),
            "platt_intercept": (
                platt_intercept_14
            ),
            "platt_slope": platt_slope_14,
            "protocol_sha256": (
                protocol_sha_14
            ),
            **{
                key: selected_candidate_14[key]
                for key in selected_candidate_14
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 14. Save locked selection
# ------------------------------------------------------------

candidate_results_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14B_xgboost_candidate_results_outer2.csv",
)

selected_model_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14B_xgboost_selected_model_outer2.csv",
)

selection_json_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14B_xgboost_selection_calibration_outer2.json",
)

selection_sha_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14B_xgboost_selection_calibration_outer2_SHA256.txt",
)

candidate_results_14.to_csv(
    candidate_results_path_14,
    index=False,
)
selected_model_14.to_csv(
    selected_model_path_14,
    index=False,
)

selection_configuration_14 = {
    "outer_fold": 2,
    "protocol_sha256": protocol_sha_14,
    "selection_metric_primary": (
        "pooled inner out-of-fold AUPRC"
    ),
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "selected_candidate": (
        selected_candidate_id_14
    ),
    "selected_hyperparameters": {
        key: selected_candidate_14[key]
        for key in selected_candidate_14
        if key != "candidate_id"
    },
    "inner_oof_auprc": float(
        best_row_14["auprc"]
    ),
    "inner_oof_auroc": float(
        best_row_14["auroc"]
    ),
    "inner_oof_brier": float(
        best_row_14["brier"]
    ),
    "platt_intercept": platt_intercept_14,
    "platt_slope": platt_slope_14,
    "inner_checkpoint_tables": (
        checkpoint_tables_14
    ),
    "patient_level_oof_written_to_drive": False,
}

with open(
    selection_json_path_14,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        selection_configuration_14,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    selection_json_path_14,
    "rb",
) as fh:
    selection_sha_14 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    selection_sha_path_14,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(selection_sha_14 + "\n")

# ------------------------------------------------------------
# 15. Fit selected model on all outer training patients
# ------------------------------------------------------------

final_preprocessor_14 = (
    make_preprocessor_14()
)

print(
    "\nFitting selected outer-fold-2 XGBoost model "
    "on all 46,800 training patients..."
)

preprocess_started_14 = time.time()

X_outer_training_processed_14 = (
    final_preprocessor_14.fit_transform(
        X_outer_training_14
    )
)
X_outer_test_processed_14 = (
    final_preprocessor_14.transform(
        X_outer_test_14
    )
)

final_preprocessing_seconds_14 = (
    time.time() - preprocess_started_14
)

final_model_14 = make_xgb_model_14(
    selected_candidate_14
)

final_fit_started_14 = time.time()

final_model_14.fit(
    X_outer_training_processed_14,
    y_outer_training_14,
)

final_fit_seconds_14 = (
    time.time() - final_fit_started_14
)

outer2_raw_probabilities_14 = (
    final_model_14.predict_proba(
        X_outer_test_processed_14
    )[:, 1]
)

raw_clipped_14 = np.clip(
    outer2_raw_probabilities_14,
    1e-6,
    1 - 1e-6,
)
raw_logit_14 = np.log(
    raw_clipped_14
    / (1 - raw_clipped_14)
)

outer2_platt_probabilities_14 = expit(
    platt_intercept_14
    + platt_slope_14 * raw_logit_14
)

for probabilities, name in [
    (outer2_raw_probabilities_14, "raw"),
    (
        outer2_platt_probabilities_14,
        "platt",
    ),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(
            f"{name} test predictions contain missing values."
        )

    if not np.all(
        (probabilities >= 0)
        & (probabilities <= 1)
    ):
        raise RuntimeError(
            f"{name} test predictions contain invalid probabilities."
        )

raw_metrics_14 = probability_metrics_14(
    y_outer_test_14,
    outer2_raw_probabilities_14,
)
platt_metrics_14 = probability_metrics_14(
    y_outer_test_14,
    outer2_platt_probabilities_14,
)

raw_calibration_intercept_14, \
raw_calibration_slope_14 = (
    calibration_intercept_slope_14(
        y_outer_test_14,
        outer2_raw_probabilities_14,
    )
)

platt_calibration_intercept_14, \
platt_calibration_slope_14 = (
    calibration_intercept_slope_14(
        y_outer_test_14,
        outer2_platt_probabilities_14,
    )
)

outer2_test_results_14 = pd.DataFrame(
    [
        {
            "outer_fold": 2,
            "model": "xgboost",
            "probability_type": "raw",
            **raw_metrics_14,
            "calibration_intercept": (
                raw_calibration_intercept_14
            ),
            "calibration_slope": (
                raw_calibration_slope_14
            ),
        },
        {
            "outer_fold": 2,
            "model": "xgboost",
            "probability_type": (
                "platt_calibrated"
            ),
            **platt_metrics_14,
            "calibration_intercept": (
                platt_calibration_intercept_14
            ),
            "calibration_slope": (
                platt_calibration_slope_14
            ),
        },
    ]
)

# ------------------------------------------------------------
# 16. Feature importance
# ------------------------------------------------------------

processed_feature_names_14 = (
    final_preprocessor_14
    .get_feature_names_out()
)

feature_importances_14 = (
    final_model_14.feature_importances_
)

if len(processed_feature_names_14) != len(
    feature_importances_14
):
    raise RuntimeError(
        "Processed feature names and XGBoost "
        "feature importances differ in length."
    )

feature_importance_table_14 = pd.DataFrame(
    {
        "processed_feature": (
            processed_feature_names_14
        ),
        "gain_importance": (
            feature_importances_14
        ),
    }
)

feature_importance_table_14[
    "importance_rank"
] = (
    feature_importance_table_14[
        "gain_importance"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)

feature_importance_table_14 = (
    feature_importance_table_14
    .sort_values(
        [
            "gain_importance",
            "processed_feature",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

nonzero_importance_features_14 = int(
    (
        feature_importance_table_14[
            "gain_importance"
        ] > 0
    ).sum()
)

final_model_summary_14 = pd.DataFrame(
    [
        {
            "outer_fold": 2,
            "selected_candidate": (
                selected_candidate_id_14
            ),
            "training_patients": len(
                X_outer_training_14
            ),
            "training_hospitals": len(
                training_hospitals_14
            ),
            "training_events": int(
                y_outer_training_14.sum()
            ),
            "test_patients": len(
                X_outer_test_14
            ),
            "test_hospitals": len(
                test_hospitals_14
            ),
            "test_events": int(
                y_outer_test_14.sum()
            ),
            "hospital_overlap": len(
                hospital_overlap_14
            ),
            "processed_feature_columns": len(
                processed_feature_names_14
            ),
            "nonzero_importance_features": (
                nonzero_importance_features_14
            ),
            "preprocessing_seconds": float(
                final_preprocessing_seconds_14
            ),
            "fit_seconds": float(
                final_fit_seconds_14
            ),
            "locked_platt_intercept": (
                platt_intercept_14
            ),
            "locked_platt_slope": (
                platt_slope_14
            ),
            "protocol_sha256": protocol_sha_14,
            "selection_sha256": (
                selection_sha_14
            ),
            **{
                key: selected_candidate_14[key]
                for key in selected_candidate_14
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 17. Secure outer-test prediction checkpoint
# ------------------------------------------------------------

outer2_prediction_df_14 = pd.DataFrame(
    {
        "id_row": (
            outer_test_meta_14[
                "id_row"
            ].astype(str)
        ),
        "outer_fold": np.full(
            len(outer_test_meta_14),
            2,
            dtype=np.int64,
        ),
        "label_stage23": (
            y_outer_test_14.astype(np.int64)
        ),
        "prediction_raw": (
            outer2_raw_probabilities_14.astype(
                np.float64
            )
        ),
        "prediction_platt": (
            outer2_platt_probabilities_14.astype(
                np.float64
            )
        ),
        "model_name": "xgboost",
        "model_version": (
            "core_v1_nested_cv"
        ),
    }
)

if len(outer2_prediction_df_14) != 11691:
    raise RuntimeError(
        "Outer-fold-2 test prediction row count != 11,691."
    )

if outer2_prediction_df_14[
    "id_row"
].duplicated().any():
    raise RuntimeError(
        "Duplicate id_row in outer-fold-2 XGBoost predictions."
    )

if int(
    outer2_prediction_df_14[
        "label_stage23"
    ].sum()
) != 606:
    raise RuntimeError(
        "Outer-fold-2 XGBoost event count != 606."
    )

prediction_table_id_14 = (
    f"{TARGET_DATASET}."
    "model_xgb_outer_predictions_outer2_v1"
)

prediction_load_config_14 = (
    bigquery.LoadJobConfig(
        schema=[
            bigquery.SchemaField(
                "id_row",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "outer_fold",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "label_stage23",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_raw",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_platt",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_name",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_version",
                "STRING",
                mode="REQUIRED",
            ),
        ],
        write_disposition=(
            bigquery.WriteDisposition.WRITE_TRUNCATE
        ),
    )
)

print(
    "\nUploading secure outer-fold-2 "
    "XGBoost prediction checkpoint:"
)
print(prediction_table_id_14)

client.load_table_from_dataframe(
    outer2_prediction_df_14,
    prediction_table_id_14,
    job_config=prediction_load_config_14,
    location=BQ_LOCATION,
).result()

SQL_VERIFY_PREDICTIONS_14 = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL)
    AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL)
    AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0
    OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0
    OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw)
    AS minimum_raw_probability,
  MAX(prediction_raw)
    AS maximum_raw_probability,
  MIN(prediction_platt)
    AS minimum_platt_probability,
  MAX(prediction_platt)
    AS maximum_platt_probability
FROM `{prediction_table_id_14}`;
"""

prediction_verification_14 = (
    client.query(
        SQL_VERIFY_PREDICTIONS_14,
        location=BQ_LOCATION,
    )
    .to_dataframe()
)

verification_row_14 = (
    prediction_verification_14.iloc[0]
)

expected_prediction_values_14 = {
    "prediction_rows": 11691,
    "distinct_rows": 11691,
    "outer_folds": 1,
    "minimum_outer_fold": 2,
    "maximum_outer_fold": 2,
    "events": 606,
    "nonevents": 11085,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in (
    expected_prediction_values_14.items()
):
    actual_value = int(
        verification_row_14[field]
    )
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: found={actual_value}, "
            f"expected={expected_value}"
        )

# ------------------------------------------------------------
# 18. Save aggregate outputs
# ------------------------------------------------------------

test_results_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14C_xgboost_outer2_test_results.csv",
)

model_summary_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14C_xgboost_final_model_outer2.csv",
)

importance_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14C_xgboost_gain_importance_outer2.csv",
)

evaluation_json_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14C_xgboost_final_evaluation_outer2.json",
)

evaluation_sha_path_14 = os.path.join(
    MODEL_OUTPUT_DIR,
    "14C_xgboost_final_evaluation_outer2_SHA256.txt",
)

outer2_test_results_14.to_csv(
    test_results_path_14,
    index=False,
)
final_model_summary_14.to_csv(
    model_summary_path_14,
    index=False,
)
feature_importance_table_14.to_csv(
    importance_path_14,
    index=False,
)

evaluation_configuration_14 = {
    "outer_fold": 2,
    "model_family": "xgboost",
    "protocol_sha256": protocol_sha_14,
    "selection_sha256": selection_sha_14,
    "selected_candidate": (
        selected_candidate_id_14
    ),
    "selected_hyperparameters": {
        key: selected_candidate_14[key]
        for key in selected_candidate_14
        if key != "candidate_id"
    },
    "training_patients": 46800,
    "training_hospitals": 158,
    "test_patients": 11691,
    "test_hospitals": 40,
    "hospital_overlap": 0,
    "locked_platt_intercept": (
        platt_intercept_14
    ),
    "locked_platt_slope": platt_slope_14,
    "processed_feature_columns": int(
        len(processed_feature_names_14)
    ),
    "secure_prediction_table": (
        prediction_table_id_14
    ),
    "patient_level_prediction_written_to_drive": False,
}

with open(
    evaluation_json_path_14,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        evaluation_configuration_14,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    evaluation_json_path_14,
    "rb",
) as fh:
    evaluation_sha_14 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    evaluation_sha_path_14,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(evaluation_sha_14 + "\n")

# ------------------------------------------------------------
# 19. Display results
# ------------------------------------------------------------

pooled_integrity_14 = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_14),
            pooled_oof_14[
                "id_row"
            ].nunique(),
            pooled_oof_14[
                "candidate_id"
            ].nunique(),
            pooled_oof_14[
                "inner_fold"
            ].nunique(),
            EXPECTED_SPLIT_14[
                "training_events"
            ],
            (
                EXPECTED_SPLIT_14[
                    "training_rows"
                ]
                - EXPECTED_SPLIT_14[
                    "training_events"
                ]
            ),
            int(
                pooled_oof_14.duplicated(
                    subset=[
                        "candidate_id",
                        "id_row",
                    ]
                ).sum()
            ),
            int(
                pooled_oof_14[
                    "prediction_raw"
                ].isna().sum()
            ),
            int(
                (
                    ~pooled_oof_14[
                        "prediction_raw"
                    ].between(0, 1)
                ).sum()
            ),
            pooled_load_method_14,
        ],
    }
)

print(
    "\n14 XGBOOST OUTER-FOLD-2 INNER CHECKPOINT SUMMARY"
)
display(checkpoint_summary_14)

print(
    "\n14 XGBOOST OUTER-FOLD-2 POOLED OOF INTEGRITY"
)
display(pooled_integrity_14)

print(
    "\n14 XGBOOST OUTER-FOLD-2 CANDIDATE RESULTS"
)
display(candidate_results_14)

print(
    "\n14 XGBOOST OUTER-FOLD-2 SELECTED MODEL"
)
display(selected_model_14)

print(
    "\n14 XGBOOST OUTER-FOLD-2 FINAL MODEL SUMMARY"
)
display(final_model_summary_14)

print(
    "\n14 XGBOOST OUTER-FOLD-2 TEST RESULTS"
)
display(outer2_test_results_14)

print(
    "\n14 XGBOOST OUTER-FOLD-2 BIGQUERY VERIFICATION"
)
display(prediction_verification_14)

print(
    "\n14 XGBOOST OUTER-FOLD-2 TOP 20 GAIN IMPORTANCE FEATURES"
)
display(feature_importance_table_14.head(20))

print("\nXGBoost protocol SHA-256:")
print(protocol_sha_14)

print("\nSelection SHA-256:")
print(selection_sha_14)

print("\nEvaluation SHA-256:")
print(evaluation_sha_14)

print("\nSaved aggregate outputs:")
print(protocol_path_14)
print(protocol_sha_path_14)
print(fit_audit_path_14)
print(checkpoint_summary_path_14)
print(candidate_results_path_14)
print(selected_model_path_14)
print(selection_json_path_14)
print(selection_sha_path_14)
print(test_results_path_14)
print(model_summary_path_14)
print(importance_path_14)
print(evaluation_json_path_14)
print(evaluation_sha_path_14)

print(
    "\n14 PASS: XGBoost outer-fold-2 nested modelling "
    "and locked test evaluation are complete."
)

print(
    "No class weighting, SMOTE, or early stopping was used."
)

print(
    "All patient-level OOF and outer-test predictions "
    "were stored only in BigQuery."
)

print(
    "No patient-level prediction file was written to Google Drive."
)

_ = gc.collect()

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

from xgboost import XGBClassifier

from IPython.display import display

print("STARTING XGBOOST OUTER FOLD 3 — CODE VERSION 15")

# ============================================================
# 14 — XGBOOST OUTER FOLD 2 COMPLETE NESTED MODELLING
#
# Design:
# - Same locked hospital-disjoint outer folds.
# - Same locked hospital-disjoint inner folds.
# - Core feature set only.
# - No SMOTE.
# - No class weighting / scale_pos_weight = 1.
# - No early stopping.
# - Fixed candidate grid locked before outer-test evaluation.
# - Selection: pooled inner-OOF AUPRC descending,
#              AUROC descending, Brier ascending.
# - Platt calibration learned only from selected candidate's
#   pooled inner-OOF predictions.
# - Patient-level predictions stored only in BigQuery.
# ============================================================

OUTER_FOLD_15 = 3
MODEL_RANDOM_SEED_15 = 20260721

EXPECTED_SPLIT_15 = {
    "training_rows": 46755,
    "test_rows": 11736,
    "training_hospitals": 158,
    "test_hospitals": 40,
    "training_events": 2424,
    "test_events": 608,
}

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_objects_15 = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_15 = [
    name for name in required_objects_15
    if name not in globals()
]

if missing_objects_15:
    raise RuntimeError(
        "Missing runtime objects: "
        + ", ".join(missing_objects_15)
        + ". Run 07A and 07B first."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"Expected 58,491 cohort rows; found {len(core_df_07B)}."
    )

if len(predictor_columns_07B) != 159:
    raise RuntimeError("Expected 159 core predictors.")

if len(numeric_columns_07B) != 156:
    raise RuntimeError("Expected 156 numeric predictors.")

if len(categorical_columns_07B) != 3:
    raise RuntimeError("Expected 3 categorical predictors.")

# ------------------------------------------------------------
# 2. Lock the XGBoost protocol BEFORE test evaluation
# ------------------------------------------------------------

candidate_grid_15 = [
    {
        "candidate_id": "XGB01",
        "n_estimators": 250,
        "max_depth": 2,
        "learning_rate": 0.03,
        "min_child_weight": 5.0,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
    },
    {
        "candidate_id": "XGB02",
        "n_estimators": 350,
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 5.0,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
    },
    {
        "candidate_id": "XGB03",
        "n_estimators": 450,
        "max_depth": 3,
        "learning_rate": 0.02,
        "min_child_weight": 10.0,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "gamma": 0.0,
        "reg_alpha": 0.10,
        "reg_lambda": 10.0,
    },
    {
        "candidate_id": "XGB04",
        "n_estimators": 350,
        "max_depth": 4,
        "learning_rate": 0.03,
        "min_child_weight": 10.0,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "gamma": 0.10,
        "reg_alpha": 0.10,
        "reg_lambda": 10.0,
    },
    {
        "candidate_id": "XGB05",
        "n_estimators": 450,
        "max_depth": 2,
        "learning_rate": 0.02,
        "min_child_weight": 10.0,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "gamma": 0.0,
        "reg_alpha": 0.50,
        "reg_lambda": 10.0,
    },
    {
        "candidate_id": "XGB06",
        "n_estimators": 450,
        "max_depth": 4,
        "learning_rate": 0.02,
        "min_child_weight": 15.0,
        "subsample": 0.90,
        "colsample_bytree": 0.80,
        "gamma": 0.20,
        "reg_alpha": 0.50,
        "reg_lambda": 15.0,
    },
]

xgb_protocol_15 = {
    "protocol_name": "xgboost_core_nested_hospital_cv_v1",
    "model_family": "XGBoost",
    "feature_set": "core_159",
    "outer_cv": "locked 5-fold hospital-disjoint outer folds",
    "inner_cv": "locked 5-fold hospital-disjoint inner folds",
    "primary_selection_metric": "pooled inner OOF AUPRC descending",
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "class_weighting": False,
    "scale_pos_weight": 1.0,
    "smote": False,
    "early_stopping": False,
    "tree_method": "hist",
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": MODEL_RANDOM_SEED_15,
    "candidate_grid": candidate_grid_15,
    "calibration": (
        "Platt calibration fit only on selected candidate "
        "pooled inner-OOF logits"
    ),
    "outer_test_use": (
        "diagnostic evaluation only; never used for tuning"
    ),
}

protocol_path_15 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13A_locked_xgboost_model_protocol_v1.json",
)

protocol_sha_path_15 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13A_locked_xgboost_model_protocol_v1_SHA256.txt",
)

protocol_text_15 = json.dumps(
    xgb_protocol_15,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
)

protocol_sha_15 = hashlib.sha256(
    protocol_text_15.encode("utf-8")
).hexdigest()

if os.path.exists(protocol_path_15):
    with open(protocol_path_15, "r", encoding="utf-8") as fh:
        existing_protocol_text_15 = fh.read()
    existing_protocol_sha_15 = hashlib.sha256(
        existing_protocol_text_15.encode("utf-8")
    ).hexdigest()

    if existing_protocol_sha_15 != protocol_sha_15:
        raise RuntimeError(
            "An existing XGBoost protocol file differs from "
            "the currently locked protocol. Stop and audit."
        )
else:
    with open(protocol_path_15, "w", encoding="utf-8") as fh:
        fh.write(protocol_text_15)

with open(protocol_sha_path_15, "w", encoding="utf-8") as fh:
    fh.write(protocol_sha_15 + "\n")

print("Locked XGBoost protocol SHA-256:")
print(protocol_sha_15)

# ------------------------------------------------------------
# 3. Load locked inner hospital map
# ------------------------------------------------------------

inner_mapping_path_15 = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_15):
    raise FileNotFoundError(
        "Locked inner-fold map not found: "
        + inner_mapping_path_15
    )

inner_mapping_all_15 = pd.read_csv(
    inner_mapping_path_15,
    dtype={"group_hospital": str},
)

inner_mapping_part_15 = (
    inner_mapping_all_15.loc[
        inner_mapping_all_15["outer_fold"].astype(int)
        == OUTER_FOLD_15,
        ["group_hospital", "inner_fold"],
    ]
    .copy()
)

inner_mapping_part_15["group_hospital"] = (
    inner_mapping_part_15["group_hospital"].astype(str)
)
inner_mapping_part_15["inner_fold"] = (
    inner_mapping_part_15["inner_fold"].astype(int)
)

if len(inner_mapping_part_15) != 158:
    raise RuntimeError(
        "Expected 158 outer-fold-3 training hospitals "
        "in the locked inner map."
    )

if inner_mapping_part_15["group_hospital"].duplicated().any():
    raise RuntimeError("Duplicate hospital in locked inner map.")

hospital_to_inner_fold_15 = dict(
    zip(
        inner_mapping_part_15["group_hospital"],
        inner_mapping_part_15["inner_fold"],
    )
)

# ------------------------------------------------------------
# 4. Prepare outer fold 1 matrices
# ------------------------------------------------------------

X_all_15 = core_df_07B[predictor_columns_07B].copy()

for column in numeric_columns_07B:
    X_all_15[column] = pd.to_numeric(
        X_all_15[column],
        errors="coerce",
    ).astype("float64")

for column in categorical_columns_07B:
    category_series = X_all_15[column].astype("object")
    X_all_15[column] = category_series.where(
        pd.notna(category_series),
        np.nan,
    )

outer_fold_vector_15 = (
    core_df_07B["outer_fold"].astype(int).to_numpy()
)

outer_training_mask_15 = (
    outer_fold_vector_15 != OUTER_FOLD_15
)
outer_test_mask_15 = (
    outer_fold_vector_15 == OUTER_FOLD_15
)

X_outer_training_15 = (
    X_all_15.loc[outer_training_mask_15]
    .reset_index(drop=True)
)

X_outer_test_15 = (
    X_all_15.loc[outer_test_mask_15]
    .reset_index(drop=True)
)

outer_training_meta_15 = (
    core_df_07B.loc[
        outer_training_mask_15,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_15 = (
    core_df_07B.loc[
        outer_test_mask_15,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [
    outer_training_meta_15,
    outer_test_meta_15,
]:
    dataframe["id_row"] = dataframe["id_row"].astype(str)
    dataframe["group_hospital"] = (
        dataframe["group_hospital"].astype(str)
    )
    dataframe["label_stage23"] = (
        dataframe["label_stage23"].astype(int)
    )

y_outer_training_15 = (
    outer_training_meta_15["label_stage23"]
    .to_numpy(dtype=np.int8)
)
y_outer_test_15 = (
    outer_test_meta_15["label_stage23"]
    .to_numpy(dtype=np.int8)
)

groups_outer_training_15 = (
    outer_training_meta_15["group_hospital"]
    .to_numpy(dtype=str)
)

training_hospitals_15 = set(
    outer_training_meta_15["group_hospital"]
)
test_hospitals_15 = set(
    outer_test_meta_15["group_hospital"]
)
hospital_overlap_15 = (
    training_hospitals_15 & test_hospitals_15
)

if hospital_overlap_15:
    raise RuntimeError(
        "Outer training/test hospital overlap detected."
    )

actual_split_15 = {
    "training_rows": len(X_outer_training_15),
    "test_rows": len(X_outer_test_15),
    "training_hospitals": len(training_hospitals_15),
    "test_hospitals": len(test_hospitals_15),
    "training_events": int(y_outer_training_15.sum()),
    "test_events": int(y_outer_test_15.sum()),
}

for metric, expected_value in EXPECTED_SPLIT_15.items():
    actual_value = actual_split_15[metric]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: found={actual_value}, "
            f"expected={expected_value}"
        )

inner_fold_vector_15 = np.array(
    [
        hospital_to_inner_fold_15.get(hospital, -1)
        for hospital in groups_outer_training_15
    ],
    dtype=int,
)

if (inner_fold_vector_15 == -1).any():
    raise RuntimeError(
        "Some outer-training hospitals have no inner-fold assignment."
    )

if set(np.unique(inner_fold_vector_15)) != {1, 2, 3, 4, 5}:
    raise RuntimeError("Inner-fold values are not exactly 1–5.")

# ------------------------------------------------------------
# 5. Preprocessor and model constructors
# ------------------------------------------------------------

def make_preprocessor_15():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns_07B,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns_07B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_xgb_model_15(candidate):
    return XGBClassifier(
        n_estimators=int(candidate["n_estimators"]),
        max_depth=int(candidate["max_depth"]),
        learning_rate=float(candidate["learning_rate"]),
        min_child_weight=float(candidate["min_child_weight"]),
        subsample=float(candidate["subsample"]),
        colsample_bytree=float(candidate["colsample_bytree"]),
        gamma=float(candidate["gamma"]),
        reg_alpha=float(candidate["reg_alpha"]),
        reg_lambda=float(candidate["reg_lambda"]),
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        max_bin=256,
        scale_pos_weight=1.0,
        importance_type="gain",
        random_state=MODEL_RANDOM_SEED_15,
        n_jobs=-1,
        verbosity=0,
    )


def checkpoint_table_id_15(inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_xgb_inner_oof_outer3_inner{inner_fold}_v1"
    )

# ------------------------------------------------------------
# 6. BigQuery checkpoint verification
# ------------------------------------------------------------

def verify_checkpoint_15(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):
    table_id = checkpoint_table_id_15(inner_fold)

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS distinct_id_count,
      COUNT(DISTINCT candidate_id) AS candidate_count,
      COUNT(DISTINCT outer_fold) AS outer_fold_count,
      COUNT(DISTINCT inner_fold) AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(candidate_id, '|', id_row)
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(
        prediction_raw < 0 OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(
        sql,
        location=BQ_LOCATION,
    ).to_dataframe()

    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows * len(candidate_grid_15)
    )
    expected_positive_rows = (
        expected_validation_events * len(candidate_grid_15)
    )
    expected_negative_rows = (
        (expected_validation_rows - expected_validation_events)
        * len(candidate_grid_15)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": expected_validation_rows,
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": expected_total_rows,
        "positive_prediction_rows": expected_positive_rows,
        "negative_prediction_rows": expected_negative_rows,
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": 2,
        "maximum_outer_fold": 2,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failures = []

    for field, expected_value in expected_values.items():
        actual_value = int(row[field])
        if actual_value != expected_value:
            complete = False
            failures.append(
                f"{field}={actual_value}, expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failures),
        "check": check,
        "row": row,
    }

checkpoint_load_config_15 = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField(
            "id_row", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "outer_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "inner_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "candidate_id", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "label_stage23", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "prediction_raw", "FLOAT", mode="REQUIRED"
        ),
    ],
    write_disposition=(
        bigquery.WriteDisposition.WRITE_TRUNCATE
    ),
)

# ------------------------------------------------------------
# 7. Aggregate fit audit
# ------------------------------------------------------------

fit_audit_columns_15 = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "n_estimators",
    "max_depth",
    "learning_rate",
    "min_child_weight",
    "subsample",
    "colsample_bytree",
    "gamma",
    "reg_alpha",
    "reg_lambda",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "fit_seconds",
]

fit_audit_path_15 = os.path.join(
    MODEL_OUTPUT_DIR,
    "15A_xgboost_inner_fit_audit_outer3.csv",
)

if os.path.exists(fit_audit_path_15):
    fit_audit_15 = pd.read_csv(fit_audit_path_15)
else:
    fit_audit_15 = pd.DataFrame(
        columns=fit_audit_columns_15
    )

for column in fit_audit_columns_15:
    if column not in fit_audit_15.columns:
        fit_audit_15[column] = np.nan

fit_audit_15 = fit_audit_15[
    fit_audit_columns_15
].copy()

# ------------------------------------------------------------
# 8. Run five inner folds
# ------------------------------------------------------------

for inner_fold in range(1, 6):
    inner_training_mask = (
        inner_fold_vector_15 != inner_fold
    )
    inner_validation_mask = (
        inner_fold_vector_15 == inner_fold
    )

    training_rows = int(inner_training_mask.sum())
    validation_rows = int(inner_validation_mask.sum())
    training_events = int(
        y_outer_training_15[inner_training_mask].sum()
    )
    validation_events = int(
        y_outer_training_15[inner_validation_mask].sum()
    )

    existing_check = verify_checkpoint_15(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if existing_check["complete"]:
        print(
            f"Outer 3 / inner {inner_fold}: "
            "permanent XGBoost checkpoint already complete; "
            "skipping model fitting."
        )
        continue

    training_hospital_set = set(
        groups_outer_training_15[inner_training_mask]
    )
    validation_hospital_set = set(
        groups_outer_training_15[inner_validation_mask]
    )

    if training_hospital_set & validation_hospital_set:
        raise RuntimeError(
            f"Inner fold {inner_fold}: hospital overlap detected."
        )

    print(f"\nOuter 3 / inner {inner_fold}")
    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_15()

    preprocessing_started = time.time()

    X_inner_training_processed = (
        preprocessor.fit_transform(
            X_outer_training_15.loc[
                inner_training_mask
            ]
        )
    )

    X_inner_validation_processed = (
        preprocessor.transform(
            X_outer_training_15.loc[
                inner_validation_mask
            ]
        )
    )

    preprocessing_seconds = (
        time.time() - preprocessing_started
    )

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "Processed training/validation column counts differ."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_seconds, 2),
    )

    y_inner_training = (
        y_outer_training_15[inner_training_mask]
    )
    y_inner_validation = (
        y_outer_training_15[inner_validation_mask]
    )

    validation_ids = (
        outer_training_meta_15.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_15:
        candidate_id = candidate["candidate_id"]

        print(
            "  Fitting",
            candidate_id,
            "| trees =",
            candidate["n_estimators"],
            "| depth =",
            candidate["max_depth"],
            "| lr =",
            candidate["learning_rate"],
        )

        model = make_xgb_model_15(candidate)

        fit_started = time.time()

        model.fit(
            X_inner_training_processed,
            y_inner_training,
        )

        fit_seconds = time.time() - fit_started

        validation_probabilities = (
            model.predict_proba(
                X_inner_validation_processed
            )[:, 1]
        )

        if np.isnan(validation_probabilities).any():
            raise RuntimeError(
                f"{candidate_id}, inner {inner_fold}: "
                "missing predictions."
            )

        if not np.all(
            (validation_probabilities >= 0)
            & (validation_probabilities <= 1)
        ):
            raise RuntimeError(
                f"{candidate_id}: invalid probabilities."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        2,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": (
                        y_inner_validation.astype(np.int64)
                    ),
                    "prediction_raw": (
                        validation_probabilities.astype(
                            np.float64
                        )
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": 2,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "n_estimators": candidate[
                    "n_estimators"
                ],
                "max_depth": candidate[
                    "max_depth"
                ],
                "learning_rate": candidate[
                    "learning_rate"
                ],
                "min_child_weight": candidate[
                    "min_child_weight"
                ],
                "subsample": candidate["subsample"],
                "colsample_bytree": candidate[
                    "colsample_bytree"
                ],
                "gamma": candidate["gamma"],
                "reg_alpha": candidate["reg_alpha"],
                "reg_lambda": candidate[
                    "reg_lambda"
                ],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": (
                    validation_events
                ),
                "processed_columns": int(
                    X_inner_training_processed.shape[1]
                ),
                "fit_seconds": float(fit_seconds),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows * len(candidate_grid_15)
    )

    if len(checkpoint_df) != expected_checkpoint_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: invalid checkpoint row count."
        )

    if checkpoint_df.duplicated(
        subset=["id_row", "candidate_id"]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: duplicate candidate-patient rows."
        )

    target_checkpoint_table = (
        checkpoint_table_id_15(inner_fold)
    )

    print(
        "Uploading permanent XGBoost checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_15,
        location=BQ_LOCATION,
    ).result()

    if len(fit_audit_15) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_15["outer_fold"],
                    errors="coerce",
                ) == 1
            )
            & (
                pd.to_numeric(
                    fit_audit_15["inner_fold"],
                    errors="coerce",
                ) == inner_fold
            )
        )

        fit_audit_15 = (
            fit_audit_15.loc[keep_mask].copy()
        )

    fit_audit_15 = pd.concat(
        [
            fit_audit_15,
            pd.DataFrame(current_audit_rows),
        ],
        ignore_index=True,
    )

    fit_audit_15 = (
        fit_audit_15[
            fit_audit_columns_15
        ]
        .sort_values(
            [
                "outer_fold",
                "inner_fold",
                "candidate_id",
            ]
        )
        .reset_index(drop=True)
    )

    fit_audit_15.to_csv(
        fit_audit_path_15,
        index=False,
    )

    completed_check = verify_checkpoint_15(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint verification failed: "
            + completed_check["reason"]
        )

    print(
        f"Outer 3 / inner {inner_fold}: "
        "permanent XGBoost checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 9. Final inner checkpoint summary
# ------------------------------------------------------------

checkpoint_summary_rows_15 = []

for inner_fold in range(1, 6):
    validation_mask = (
        inner_fold_vector_15 == inner_fold
    )
    validation_rows = int(validation_mask.sum())
    validation_events = int(
        y_outer_training_15[
            validation_mask
        ].sum()
    )

    final_check = verify_checkpoint_15(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: final checkpoint audit failed. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_15.append(
        {
            "outer_fold": 2,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(
                row["row_count"]
            ),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(
                row["candidate_count"]
            ),
            "positive_prediction_rows": int(
                row["positive_prediction_rows"]
            ),
            "negative_prediction_rows": int(
                row["negative_prediction_rows"]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check["table_id"],
        }
    )

checkpoint_summary_15 = (
    pd.DataFrame(checkpoint_summary_rows_15)
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_15[
        "distinct_validation_patients"
    ].sum()
) != EXPECTED_SPLIT_15["training_rows"]:
    raise RuntimeError(
        "Total inner validation patients != 46,755."
    )

expected_total_oof_rows_15 = (
    EXPECTED_SPLIT_15["training_rows"]
    * len(candidate_grid_15)
)

if int(
    checkpoint_summary_15[
        "checkpoint_rows"
    ].sum()
) != expected_total_oof_rows_15:
    raise RuntimeError(
        "Total XGBoost OOF prediction rows are incorrect."
    )

checkpoint_summary_path_15 = os.path.join(
    MODEL_OUTPUT_DIR,
    "15A_xgboost_outer3_inner_checkpoint_summary.csv",
)

checkpoint_summary_15.to_csv(
    checkpoint_summary_path_15,
    index=False,
)

# ------------------------------------------------------------
# 10. Pool five inner OOF tables
# ------------------------------------------------------------

checkpoint_tables_15 = [
    checkpoint_table_id_15(inner_fold)
    for inner_fold in range(1, 6)
]

union_parts_15 = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_15
]

SQL_LOAD_POOLED_OOF_15 = (
    "\nUNION ALL\n".join(union_parts_15)
)

print(
    "\nLoading pooled outer-fold-3 XGBoost inner OOF predictions..."
)

query_job_15 = client.query(
    SQL_LOAD_POOLED_OOF_15,
    location=BQ_LOCATION,
)

try:
    pooled_oof_15 = query_job_15.to_dataframe(
        create_bqstorage_client=True
    )
    pooled_load_method_15 = (
        "BigQuery Storage API"
    )
except Exception as fast_path_error_15:
    print(
        "Storage API unavailable; using standard BigQuery download."
    )
    print(
        "Message:",
        type(fast_path_error_15).__name__,
    )
    pooled_oof_15 = query_job_15.to_dataframe(
        create_bqstorage_client=False
    )
    pooled_load_method_15 = (
        "Standard BigQuery API"
    )

pooled_oof_15["id_row"] = (
    pooled_oof_15["id_row"].astype(str)
)
pooled_oof_15["candidate_id"] = (
    pooled_oof_15["candidate_id"].astype(str)
)

for column in [
    "outer_fold",
    "inner_fold",
    "label_stage23",
]:
    pooled_oof_15[column] = pd.to_numeric(
        pooled_oof_15[column],
        errors="raise",
    ).astype(int)

pooled_oof_15["prediction_raw"] = pd.to_numeric(
    pooled_oof_15["prediction_raw"],
    errors="raise",
).astype(float)

if len(pooled_oof_15) != expected_total_oof_rows_15:
    raise RuntimeError(
        "Pooled XGBoost OOF row count is incorrect."
    )

if pooled_oof_15.duplicated(
    subset=["candidate_id", "id_row"]
).any():
    raise RuntimeError(
        "Duplicate candidate-patient row in pooled XGBoost OOF."
    )

if pooled_oof_15["prediction_raw"].isna().any():
    raise RuntimeError("Missing XGBoost OOF prediction.")

if not pooled_oof_15[
    "prediction_raw"
].between(0, 1).all():
    raise RuntimeError(
        "Invalid XGBoost OOF probability."
    )

if set(
    pooled_oof_15["candidate_id"].unique()
) != {
    "XGB01",
    "XGB02",
    "XGB03",
    "XGB04",
    "XGB05",
    "XGB06",
}:
    raise RuntimeError(
        "The six locked XGBoost candidates are not all present."
    )

candidate_patient_counts_15 = (
    pooled_oof_15
    .groupby("candidate_id")["id_row"]
    .nunique()
)

if not (
    candidate_patient_counts_15
    == EXPECTED_SPLIT_15["training_rows"]
).all():
    raise RuntimeError(
        "Each candidate must have 46,755 OOF patients."
    )

candidate_event_counts_15 = (
    pooled_oof_15
    .groupby("candidate_id")["label_stage23"]
    .sum()
)

if not (
    candidate_event_counts_15
    == EXPECTED_SPLIT_15["training_events"]
).all():
    raise RuntimeError(
        "Each candidate must have 2,426 OOF events."
    )

# ------------------------------------------------------------
# 11. Metric helpers
# ------------------------------------------------------------

def probability_metrics_15(
    y_true,
    probabilities,
):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(
            roc_auc_score(y_true, probabilities)
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1],
            )
        ),
        "mean_predicted_risk": float(
            probabilities.mean()
        ),
        "observed_event_rate": float(
            np.mean(y_true)
        ),
    }


def probability_logit_15(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        probabilities / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_15(
    y_true,
    probabilities,
):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        probability_logit_15(probabilities),
        y_true,
    )

    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 12. Candidate pooled inner OOF metrics
# ------------------------------------------------------------

candidate_result_rows_15 = []

for candidate in candidate_grid_15:
    candidate_id = candidate["candidate_id"]

    candidate_oof = (
        pooled_oof_15.loc[
            pooled_oof_15["candidate_id"]
            == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_15(
        candidate_oof[
            "label_stage23"
        ].to_numpy(dtype=int),
        candidate_oof[
            "prediction_raw"
        ].to_numpy(dtype=float),
    )

    fit_part = fit_audit_15.loc[
        fit_audit_15[
            "candidate_id"
        ].astype(str) == candidate_id
    ]

    fit_seconds_total = (
        float(
            pd.to_numeric(
                fit_part["fit_seconds"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_15.append(
        {
            "candidate_id": candidate_id,
            **{
                key: candidate[key]
                for key in candidate
                if key != "candidate_id"
            },
            **metrics,
            "fit_seconds_total": (
                fit_seconds_total
            ),
        }
    )

candidate_results_15 = pd.DataFrame(
    candidate_result_rows_15
)

candidate_results_15 = (
    candidate_results_15
    .sort_values(
        ["auprc", "auroc", "brier"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

candidate_results_15["selection_rank"] = (
    np.arange(
        1,
        len(candidate_results_15) + 1,
    )
)

best_row_15 = candidate_results_15.iloc[0]
selected_candidate_id_15 = str(
    best_row_15["candidate_id"]
)

selected_candidate_15 = next(
    candidate
    for candidate in candidate_grid_15
    if candidate["candidate_id"]
    == selected_candidate_id_15
)

# ------------------------------------------------------------
# 13. Platt calibration from selected inner OOF
# ------------------------------------------------------------

selected_oof_15 = (
    pooled_oof_15.loc[
        pooled_oof_15["candidate_id"]
        == selected_candidate_id_15
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_15 = (
    selected_oof_15[
        "label_stage23"
    ].to_numpy(dtype=int)
)
selected_oof_probability_15 = (
    selected_oof_15[
        "prediction_raw"
    ].to_numpy(dtype=float)
)

platt_calibrator_15 = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_15.fit(
    probability_logit_15(
        selected_oof_probability_15
    ),
    selected_oof_y_15,
)

platt_intercept_15 = float(
    platt_calibrator_15.intercept_[0]
)
platt_slope_15 = float(
    platt_calibrator_15.coef_[0][0]
)

if (
    not np.isfinite(platt_intercept_15)
    or not np.isfinite(platt_slope_15)
    or platt_slope_15 <= 0
):
    raise RuntimeError(
        "Invalid Platt calibration coefficients."
    )

selected_model_15 = pd.DataFrame(
    [
        {
            "outer_fold": 2,
            "selected_candidate": (
                selected_candidate_id_15
            ),
            "selection_metric_primary": (
                "pooled_inner_oof_auprc"
            ),
            "inner_oof_auprc": float(
                best_row_15["auprc"]
            ),
            "inner_oof_auroc": float(
                best_row_15["auroc"]
            ),
            "inner_oof_brier": float(
                best_row_15["brier"]
            ),
            "inner_oof_log_loss": float(
                best_row_15["log_loss"]
            ),
            "inner_oof_mean_predicted_risk": float(
                best_row_15[
                    "mean_predicted_risk"
                ]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_15[
                    "observed_event_rate"
                ]
            ),
            "platt_intercept": (
                platt_intercept_15
            ),
            "platt_slope": platt_slope_15,
            "protocol_sha256": (
                protocol_sha_15
            ),
            **{
                key: selected_candidate_15[key]
                for key in selected_candidate_15
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 14. Save locked selection
# ------------------------------------------------------------

candidate_results_path_15 = os.path.join(
    MODEL_OUTPUT_DIR,
    "15B_xgboost_candidate_results_outer3.csv",
)

selected_model_path_15 = os.path.join(
    MODEL_OUTPUT_DIR,
    "15B_xgboost_selected_model_outer3.csv",
)

selection_json_path_15 = os.path.join(
    MODEL_OUTPUT_DIR,
    "15B_xgboost_selection_calibration_outer3.json",
)

selection_sha_path_15 = os.path.join(
    MODEL_OUTPUT_DIR,
    "15B_xgboost_selection_calibration_outer3_SHA256.txt",
)

candidate_results_15.to_csv(
    candidate_results_path_15,
    index=False,
)
selected_model_15.to_csv(
    selected_model_path_15,
    index=False,
)

selection_configuration_15 = {
    "outer_fold": 2,
    "protocol_sha256": protocol_sha_15,
    "selection_metric_primary": (
        "pooled inner out-of-fold AUPRC"
    ),
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "selected_candidate": (
        selected_candidate_id_15
    ),
    "selected_hyperparameters": {
        key: selected_candidate_15[key]
        for key in selected_candidate_15
        if key != "candidate_id"
    },
    "inner_oof_auprc": float(
        best_row_15["auprc"]
    ),
    "inner_oof_auroc": float(
        best_row_15["auroc"]
    ),
    "inner_oof_brier": float(
        best_row_15["brier"]
    ),
    "platt_intercept": platt_intercept_15,
    "platt_slope": platt_slope_15,
    "inner_checkpoint_tables": (
        checkpoint_tables_15
    ),
    "patient_level_oof_written_to_drive": False,
}

with open(
    selection_json_path_15,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        selection_configuration_15,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    selection_json_path_15,
    "rb",
) as fh:
    selection_sha_15 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    selection_sha_path_15,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(selection_sha_15 + "\n")

# ------------------------------------------------------------
# 15. Fit selected model on all outer training patients
# ------------------------------------------------------------

final_preprocessor_15 = (
    make_preprocessor_15()
)

print(
    "\nFitting selected outer-fold-3 XGBoost model "
    "on all 46,755 training patients..."
)

preprocess_started_15 = time.time()

X_outer_training_processed_15 = (
    final_preprocessor_15.fit_transform(
        X_outer_training_15
    )
)
X_outer_test_processed_15 = (
    final_preprocessor_15.transform(
        X_outer_test_15
    )
)

final_preprocessing_seconds_15 = (
    time.time() - preprocess_started_15
)

final_model_15 = make_xgb_model_15(
    selected_candidate_15
)

final_fit_started_15 = time.time()

final_model_15.fit(
    X_outer_training_processed_15,
    y_outer_training_15,
)

final_fit_seconds_15 = (
    time.time() - final_fit_started_15
)

outer3_raw_probabilities_15 = (
    final_model_15.predict_proba(
        X_outer_test_processed_15
    )[:, 1]
)

raw_clipped_15 = np.clip(
    outer3_raw_probabilities_15,
    1e-6,
    1 - 1e-6,
)
raw_logit_15 = np.log(
    raw_clipped_15
    / (1 - raw_clipped_15)
)

outer3_platt_probabilities_15 = expit(
    platt_intercept_15
    + platt_slope_15 * raw_logit_15
)

for probabilities, name in [
    (outer3_raw_probabilities_15, "raw"),
    (
        outer3_platt_probabilities_15,
        "platt",
    ),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(
            f"{name} test predictions contain missing values."
        )

    if not np.all(
        (probabilities >= 0)
        & (probabilities <= 1)
    ):
        raise RuntimeError(
            f"{name} test predictions contain invalid probabilities."
        )

raw_metrics_15 = probability_metrics_15(
    y_outer_test_15,
    outer3_raw_probabilities_15,
)
platt_metrics_15 = probability_metrics_15(
    y_outer_test_15,
    outer3_platt_probabilities_15,
)

raw_calibration_intercept_15, \
raw_calibration_slope_15 = (
    calibration_intercept_slope_15(
        y_outer_test_15,
        outer3_raw_probabilities_15,
    )
)

platt_calibration_intercept_15, \
platt_calibration_slope_15 = (
    calibration_intercept_slope_15(
        y_outer_test_15,
        outer3_platt_probabilities_15,
    )
)

outer3_test_results_15 = pd.DataFrame(
    [
        {
            "outer_fold": 2,
            "model": "xgboost",
            "probability_type": "raw",
            **raw_metrics_15,
            "calibration_intercept": (
                raw_calibration_intercept_15
            ),
            "calibration_slope": (
                raw_calibration_slope_15
            ),
        },
        {
            "outer_fold": 2,
            "model": "xgboost",
            "probability_type": (
                "platt_calibrated"
            ),
            **platt_metrics_15,
            "calibration_intercept": (
                platt_calibration_intercept_15
            ),
            "calibration_slope": (
                platt_calibration_slope_15
            ),
        },
    ]
)

# ------------------------------------------------------------
# 16. Feature importance
# ------------------------------------------------------------

processed_feature_names_15 = (
    final_preprocessor_15
    .get_feature_names_out()
)

feature_importances_15 = (
    final_model_15.feature_importances_
)

if len(processed_feature_names_15) != len(
    feature_importances_15
):
    raise RuntimeError(
        "Processed feature names and XGBoost "
        "feature importances differ in length."
    )

feature_importance_table_15 = pd.DataFrame(
    {
        "processed_feature": (
            processed_feature_names_15
        ),
        "gain_importance": (
            feature_importances_15
        ),
    }
)

feature_importance_table_15[
    "importance_rank"
] = (
    feature_importance_table_15[
        "gain_importance"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)

feature_importance_table_15 = (
    feature_importance_table_15
    .sort_values(
        [
            "gain_importance",
            "processed_feature",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

nonzero_importance_features_15 = int(
    (
        feature_importance_table_15[
            "gain_importance"
        ] > 0
    ).sum()
)

final_model_summary_15 = pd.DataFrame(
    [
        {
            "outer_fold": 2,
            "selected_candidate": (
                selected_candidate_id_15
            ),
            "training_patients": len(
                X_outer_training_15
            ),
            "training_hospitals": len(
                training_hospitals_15
            ),
            "training_events": int(
                y_outer_training_15.sum()
            ),
            "test_patients": len(
                X_outer_test_15
            ),
            "test_hospitals": len(
                test_hospitals_15
            ),
            "test_events": int(
                y_outer_test_15.sum()
            ),
            "hospital_overlap": len(
                hospital_overlap_15
            ),
            "processed_feature_columns": len(
                processed_feature_names_15
            ),
            "nonzero_importance_features": (
                nonzero_importance_features_15
            ),
            "preprocessing_seconds": float(
                final_preprocessing_seconds_15
            ),
            "fit_seconds": float(
                final_fit_seconds_15
            ),
            "locked_platt_intercept": (
                platt_intercept_15
            ),
            "locked_platt_slope": (
                platt_slope_15
            ),
            "protocol_sha256": protocol_sha_15,
            "selection_sha256": (
                selection_sha_15
            ),
            **{
                key: selected_candidate_15[key]
                for key in selected_candidate_15
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 17. Secure outer-test prediction checkpoint
# ------------------------------------------------------------

outer3_prediction_df_15 = pd.DataFrame(
    {
        "id_row": (
            outer_test_meta_15[
                "id_row"
            ].astype(str)
        ),
        "outer_fold": np.full(
            len(outer_test_meta_15),
            2,
            dtype=np.int64,
        ),
        "label_stage23": (
            y_outer_test_15.astype(np.int64)
        ),
        "prediction_raw": (
            outer3_raw_probabilities_15.astype(
                np.float64
            )
        ),
        "prediction_platt": (
            outer3_platt_probabilities_15.astype(
                np.float64
            )
        ),
        "model_name": "xgboost",
        "model_version": (
            "core_v1_nested_cv"
        ),
    }
)

if len(outer3_prediction_df_15) != 11736:
    raise RuntimeError(
        "Outer-fold-2 test prediction row count != 11,736."
    )

if outer3_prediction_df_15[
    "id_row"
].duplicated().any():
    raise RuntimeError(
        "Duplicate id_row in outer-fold-3 XGBoost predictions."
    )

if int(
    outer3_prediction_df_15[
        "label_stage23"
    ].sum()
) != 608:
    raise RuntimeError(
        "Outer-fold-2 XGBoost event count != 608."
    )

prediction_table_id_15 = (
    f"{TARGET_DATASET}."
    "model_xgb_outer_predictions_outer3_v1"
)

prediction_load_config_15 = (
    bigquery.LoadJobConfig(
        schema=[
            bigquery.SchemaField(
                "id_row",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "outer_fold",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "label_stage23",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_raw",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_platt",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_name",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_version",
                "STRING",
                mode="REQUIRED",
            ),
        ],
        write_disposition=(
            bigquery.WriteDisposition.WRITE_TRUNCATE
        ),
    )
)

print(
    "\nUploading secure outer-fold-3 "
    "XGBoost prediction checkpoint:"
)
print(prediction_table_id_15)

client.load_table_from_dataframe(
    outer3_prediction_df_15,
    prediction_table_id_15,
    job_config=prediction_load_config_15,
    location=BQ_LOCATION,
).result()

SQL_VERIFY_PREDICTIONS_15 = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL)
    AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL)
    AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0
    OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0
    OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw)
    AS minimum_raw_probability,
  MAX(prediction_raw)
    AS maximum_raw_probability,
  MIN(prediction_platt)
    AS minimum_platt_probability,
  MAX(prediction_platt)
    AS maximum_platt_probability
FROM `{prediction_table_id_15}`;
"""

prediction_verification_15 = (
    client.query(
        SQL_VERIFY_PREDICTIONS_15,
        location=BQ_LOCATION,
    )
    .to_dataframe()
)

verification_row_15 = (
    prediction_verification_15.iloc[0]
)

expected_prediction_values_15 = {
    "prediction_rows": 11736,
    "distinct_rows": 11736,
    "outer_folds": 1,
    "minimum_outer_fold": 2,
    "maximum_outer_fold": 2,
    "events": 608,
    "nonevents": 11128,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in (
    expected_prediction_values_15.items()
):
    actual_value = int(
        verification_row_15[field]
    )
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: found={actual_value}, "
            f"expected={expected_value}"
        )

# ------------------------------------------------------------
# 18. Save aggregate outputs
# ------------------------------------------------------------

test_results_path_15 = os.path.join(
    MODEL_OUTPUT_DIR,
    "15C_xgboost_outer3_test_results.csv",
)

model_summary_path_15 = os.path.join(
    MODEL_OUTPUT_DIR,
    "15C_xgboost_final_model_outer3.csv",
)

importance_path_15 = os.path.join(
    MODEL_OUTPUT_DIR,
    "15C_xgboost_gain_importance_outer3.csv",
)

evaluation_json_path_15 = os.path.join(
    MODEL_OUTPUT_DIR,
    "15C_xgboost_final_evaluation_outer3.json",
)

evaluation_sha_path_15 = os.path.join(
    MODEL_OUTPUT_DIR,
    "15C_xgboost_final_evaluation_outer3_SHA256.txt",
)

outer3_test_results_15.to_csv(
    test_results_path_15,
    index=False,
)
final_model_summary_15.to_csv(
    model_summary_path_15,
    index=False,
)
feature_importance_table_15.to_csv(
    importance_path_15,
    index=False,
)

evaluation_configuration_15 = {
    "outer_fold": 2,
    "model_family": "xgboost",
    "protocol_sha256": protocol_sha_15,
    "selection_sha256": selection_sha_15,
    "selected_candidate": (
        selected_candidate_id_15
    ),
    "selected_hyperparameters": {
        key: selected_candidate_15[key]
        for key in selected_candidate_15
        if key != "candidate_id"
    },
    "training_patients": 46755,
    "training_hospitals": 158,
    "test_patients": 11736,
    "test_hospitals": 40,
    "hospital_overlap": 0,
    "locked_platt_intercept": (
        platt_intercept_15
    ),
    "locked_platt_slope": platt_slope_15,
    "processed_feature_columns": int(
        len(processed_feature_names_15)
    ),
    "secure_prediction_table": (
        prediction_table_id_15
    ),
    "patient_level_prediction_written_to_drive": False,
}

with open(
    evaluation_json_path_15,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        evaluation_configuration_15,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    evaluation_json_path_15,
    "rb",
) as fh:
    evaluation_sha_15 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    evaluation_sha_path_15,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(evaluation_sha_15 + "\n")

# ------------------------------------------------------------
# 19. Display results
# ------------------------------------------------------------

pooled_integrity_15 = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_15),
            pooled_oof_15[
                "id_row"
            ].nunique(),
            pooled_oof_15[
                "candidate_id"
            ].nunique(),
            pooled_oof_15[
                "inner_fold"
            ].nunique(),
            EXPECTED_SPLIT_15[
                "training_events"
            ],
            (
                EXPECTED_SPLIT_15[
                    "training_rows"
                ]
                - EXPECTED_SPLIT_15[
                    "training_events"
                ]
            ),
            int(
                pooled_oof_15.duplicated(
                    subset=[
                        "candidate_id",
                        "id_row",
                    ]
                ).sum()
            ),
            int(
                pooled_oof_15[
                    "prediction_raw"
                ].isna().sum()
            ),
            int(
                (
                    ~pooled_oof_15[
                        "prediction_raw"
                    ].between(0, 1)
                ).sum()
            ),
            pooled_load_method_15,
        ],
    }
)

print(
    "\n15 XGBOOST OUTER-FOLD-3 INNER CHECKPOINT SUMMARY"
)
display(checkpoint_summary_15)

print(
    "\n15 XGBOOST OUTER-FOLD-3 POOLED OOF INTEGRITY"
)
display(pooled_integrity_15)

print(
    "\n15 XGBOOST OUTER-FOLD-3 CANDIDATE RESULTS"
)
display(candidate_results_15)

print(
    "\n15 XGBOOST OUTER-FOLD-3 SELECTED MODEL"
)
display(selected_model_15)

print(
    "\n15 XGBOOST OUTER-FOLD-3 FINAL MODEL SUMMARY"
)
display(final_model_summary_15)

print(
    "\n15 XGBOOST OUTER-FOLD-3 TEST RESULTS"
)
display(outer3_test_results_15)

print(
    "\n15 XGBOOST OUTER-FOLD-3 BIGQUERY VERIFICATION"
)
display(prediction_verification_15)

print(
    "\n15 XGBOOST OUTER-FOLD-3 TOP 20 GAIN IMPORTANCE FEATURES"
)
display(feature_importance_table_15.head(20))

print("\nXGBoost protocol SHA-256:")
print(protocol_sha_15)

print("\nSelection SHA-256:")
print(selection_sha_15)

print("\nEvaluation SHA-256:")
print(evaluation_sha_15)

print("\nSaved aggregate outputs:")
print(protocol_path_15)
print(protocol_sha_path_15)
print(fit_audit_path_15)
print(checkpoint_summary_path_15)
print(candidate_results_path_15)
print(selected_model_path_15)
print(selection_json_path_15)
print(selection_sha_path_15)
print(test_results_path_15)
print(model_summary_path_15)
print(importance_path_15)
print(evaluation_json_path_15)
print(evaluation_sha_path_15)

print(
    "\n15 PASS: XGBoost outer-fold-3 nested modelling "
    "and locked test evaluation are complete."
)

print(
    "No class weighting, SMOTE, or early stopping was used."
)

print(
    "All patient-level OOF and outer-test predictions "
    "were stored only in BigQuery."
)

print(
    "No patient-level prediction file was written to Google Drive."
)

_ = gc.collect()

In [ ]:
print("STARTING XGBOOST OUTER FOLD 3 METADATA REPAIR — CODE VERSION 15R")

import os
import json
import hashlib
import numpy as np
import pandas as pd

from google.cloud import bigquery
from google.api_core.exceptions import NotFound
from IPython.display import display

# ============================================================
# 15R — REPAIR ONLY
#
# Purpose:
# Correct outer_fold metadata from 2 -> 3 in the already-completed
# XGBoost outer-fold-3 artifacts.
#
# IMPORTANT:
# - No model fitting.
# - No hyperparameter selection.
# - No recalculation of predictions.
# - No change to probabilities or labels.
# - BigQuery tables are overwritten via WRITE_TRUNCATE only.
# - No DELETE / INSERT / UPDATE / MERGE.
# ============================================================

EXPECTED_PROTOCOL_SHA_15R = (
    "3434db5dd0b4b5145950fc07fba3d007"
    "829738d800b88ec40fbdb09879188264"
)

# Resolve shared objects if available; otherwise use locked project values.
PROJECT_ID_15R = f"{PROJECT_ID}"
TARGET_DATASET_15R = (
    TARGET_DATASET
    if "TARGET_DATASET" in globals()
    else f"{PROJECT_ID}.aki_jcmc_v2"
)
BQ_LOCATION_15R = (
    BQ_LOCATION
    if "BQ_LOCATION" in globals()
    else "US"
)
MODEL_OUTPUT_DIR_15R = (
    MODEL_OUTPUT_DIR
    if "MODEL_OUTPUT_DIR" in globals()
    else f"{OUTPUT_ROOT}/05_MODELING_OUTPUTS"
)

if "client" in globals():
    client_15R = client
else:
    client_15R = bigquery.Client(project=PROJECT_ID_15R)

os.makedirs(MODEL_OUTPUT_DIR_15R, exist_ok=True)

# ------------------------------------------------------------
# 1. Verify the locked XGBoost protocol hash
# ------------------------------------------------------------

protocol_sha_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "13A_locked_xgboost_model_protocol_v1_SHA256.txt",
)

if not os.path.exists(protocol_sha_path_15R):
    raise FileNotFoundError(
        "Locked XGBoost protocol SHA file not found: "
        + protocol_sha_path_15R
    )

with open(protocol_sha_path_15R, "r", encoding="utf-8") as fh:
    observed_protocol_sha_15R = fh.read().strip()

if observed_protocol_sha_15R != EXPECTED_PROTOCOL_SHA_15R:
    raise RuntimeError(
        "Locked XGBoost protocol SHA mismatch. "
        f"Observed={observed_protocol_sha_15R}"
    )

print("Locked XGBoost protocol SHA-256:")
print(observed_protocol_sha_15R)

# ------------------------------------------------------------
# 2. Repair five inner OOF BigQuery tables
# ------------------------------------------------------------

inner_expected_15R = {
    1: {"patients": 7934, "events": 380},
    2: {"patients": 5835, "events": 307},
    3: {"patients": 8356, "events": 458},
    4: {"patients": 13173, "events": 691},
    5: {"patients": 11457, "events": 588},
}

inner_schema_15R = [
    bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("inner_fold", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("candidate_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
]

inner_load_config_15R = bigquery.LoadJobConfig(
    schema=inner_schema_15R,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

inner_repair_rows_15R = []

for inner_fold in range(1, 6):
    table_id = (
        f"{TARGET_DATASET_15R}."
        f"model_xgb_inner_oof_outer3_inner{inner_fold}_v1"
    )

    try:
        client_15R.get_table(table_id)
    except NotFound:
        raise FileNotFoundError(
            f"Missing outer-fold-3 inner checkpoint table: {table_id}"
        )

    query = f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    df = client_15R.query(
        query,
        location=BQ_LOCATION_15R,
    ).to_dataframe(create_bqstorage_client=True)

    expected_patients = inner_expected_15R[inner_fold]["patients"]
    expected_events = inner_expected_15R[inner_fold]["events"]
    expected_rows = expected_patients * 6

    if len(df) != expected_rows:
        raise RuntimeError(
            f"Inner {inner_fold}: rows={len(df)}, expected={expected_rows}"
        )

    if df["id_row"].astype(str).nunique() != expected_patients:
        raise RuntimeError(
            f"Inner {inner_fold}: distinct patient count mismatch."
        )

    if df["candidate_id"].astype(str).nunique() != 6:
        raise RuntimeError(
            f"Inner {inner_fold}: expected six candidates."
        )

    if df.duplicated(["id_row", "candidate_id"]).any():
        raise RuntimeError(
            f"Inner {inner_fold}: duplicate candidate-patient rows found."
        )

    if set(pd.to_numeric(df["inner_fold"]).astype(int).unique()) != {inner_fold}:
        raise RuntimeError(
            f"Inner {inner_fold}: incorrect inner_fold metadata."
        )

    positive_prediction_rows = int(
        pd.to_numeric(df["label_stage23"]).astype(int).sum()
    )
    if positive_prediction_rows != expected_events * 6:
        raise RuntimeError(
            f"Inner {inner_fold}: positive prediction row count mismatch."
        )

    if df["prediction_raw"].isna().any():
        raise RuntimeError(
            f"Inner {inner_fold}: missing probabilities."
        )

    probs = pd.to_numeric(df["prediction_raw"], errors="raise").astype(float)
    if not probs.between(0, 1).all():
        raise RuntimeError(
            f"Inner {inner_fold}: invalid probabilities."
        )

    before_values = sorted(
        pd.to_numeric(df["outer_fold"], errors="raise")
        .astype(int)
        .unique()
        .tolist()
    )

    # Metadata-only correction.
    df["id_row"] = df["id_row"].astype(str)
    df["outer_fold"] = np.int64(3)
    df["inner_fold"] = pd.to_numeric(
        df["inner_fold"], errors="raise"
    ).astype(np.int64)
    df["candidate_id"] = df["candidate_id"].astype(str)
    df["label_stage23"] = pd.to_numeric(
        df["label_stage23"], errors="raise"
    ).astype(np.int64)
    df["prediction_raw"] = probs.astype(np.float64)

    client_15R.load_table_from_dataframe(
        df,
        table_id,
        job_config=inner_load_config_15R,
        location=BQ_LOCATION_15R,
    ).result()

    verify_sql = f"""
    SELECT
      COUNT(*) AS rows,
      COUNT(DISTINCT id_row) AS patients,
      COUNT(DISTINCT candidate_id) AS candidates,
      COUNT(DISTINCT outer_fold) AS outer_folds,
      MIN(outer_fold) AS min_outer_fold,
      MAX(outer_fold) AS max_outer_fold,
      COUNT(DISTINCT inner_fold) AS inner_folds,
      MIN(inner_fold) AS min_inner_fold,
      MAX(inner_fold) AS max_inner_fold,
      COUNTIF(label_stage23=1) AS positive_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(prediction_raw < 0 OR prediction_raw > 1) AS invalid_predictions
    FROM `{table_id}`
    """
    v = client_15R.query(
        verify_sql,
        location=BQ_LOCATION_15R,
    ).to_dataframe().iloc[0]

    checks = {
        "rows": expected_rows,
        "patients": expected_patients,
        "candidates": 6,
        "outer_folds": 1,
        "min_outer_fold": 3,
        "max_outer_fold": 3,
        "inner_folds": 1,
        "min_inner_fold": inner_fold,
        "max_inner_fold": inner_fold,
        "positive_prediction_rows": expected_events * 6,
        "missing_predictions": 0,
        "invalid_predictions": 0,
    }
    for field, expected in checks.items():
        actual = int(v[field])
        if actual != expected:
            raise RuntimeError(
                f"{table_id}: {field}={actual}, expected={expected}"
            )

    inner_repair_rows_15R.append({
        "inner_fold": inner_fold,
        "rows": expected_rows,
        "patients": expected_patients,
        "events": expected_events,
        "outer_fold_before": str(before_values),
        "outer_fold_after": 3,
        "table_id": table_id,
    })

inner_repair_summary_15R = pd.DataFrame(inner_repair_rows_15R)

# ------------------------------------------------------------
# 3. Repair outer-test prediction BigQuery table
# ------------------------------------------------------------

outer_table_15R = (
    f"{TARGET_DATASET_15R}."
    "model_xgb_outer_predictions_outer3_v1"
)

try:
    client_15R.get_table(outer_table_15R)
except NotFound:
    raise FileNotFoundError(
        "Missing outer-fold-3 XGBoost prediction table: "
        + outer_table_15R
    )

outer_query_15R = f"""
SELECT
  id_row,
  outer_fold,
  label_stage23,
  prediction_raw,
  prediction_platt,
  model_name,
  model_version
FROM `{outer_table_15R}`
"""

outer_df_15R = client_15R.query(
    outer_query_15R,
    location=BQ_LOCATION_15R,
).to_dataframe(create_bqstorage_client=True)

if len(outer_df_15R) != 11736:
    raise RuntimeError(
        f"Outer prediction rows={len(outer_df_15R)}, expected=11736"
    )

if outer_df_15R["id_row"].astype(str).nunique() != 11736:
    raise RuntimeError("Outer prediction distinct patient count mismatch.")

if int(
    pd.to_numeric(outer_df_15R["label_stage23"]).astype(int).sum()
) != 608:
    raise RuntimeError("Outer prediction event count mismatch.")

if outer_df_15R["id_row"].astype(str).duplicated().any():
    raise RuntimeError("Duplicate outer-test patients found.")

for col in ["prediction_raw", "prediction_platt"]:
    p = pd.to_numeric(outer_df_15R[col], errors="raise").astype(float)
    if p.isna().any() or not p.between(0, 1).all():
        raise RuntimeError(f"Invalid values in {col}.")

outer_before_15R = sorted(
    pd.to_numeric(
        outer_df_15R["outer_fold"], errors="raise"
    ).astype(int).unique().tolist()
)

outer_df_15R["id_row"] = outer_df_15R["id_row"].astype(str)
outer_df_15R["outer_fold"] = np.int64(3)
outer_df_15R["label_stage23"] = pd.to_numeric(
    outer_df_15R["label_stage23"], errors="raise"
).astype(np.int64)
outer_df_15R["prediction_raw"] = pd.to_numeric(
    outer_df_15R["prediction_raw"], errors="raise"
).astype(np.float64)
outer_df_15R["prediction_platt"] = pd.to_numeric(
    outer_df_15R["prediction_platt"], errors="raise"
).astype(np.float64)
outer_df_15R["model_name"] = outer_df_15R["model_name"].astype(str)
outer_df_15R["model_version"] = outer_df_15R["model_version"].astype(str)

outer_schema_15R = [
    bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
    bigquery.SchemaField("prediction_platt", "FLOAT", mode="REQUIRED"),
    bigquery.SchemaField("model_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("model_version", "STRING", mode="REQUIRED"),
]

outer_load_config_15R = bigquery.LoadJobConfig(
    schema=outer_schema_15R,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

client_15R.load_table_from_dataframe(
    outer_df_15R,
    outer_table_15R,
    job_config=outer_load_config_15R,
    location=BQ_LOCATION_15R,
).result()

outer_verify_sql_15R = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23=1) AS events,
  COUNTIF(label_stage23=0) AS nonevents,
  COUNTIF(prediction_raw IS NULL) AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL) AS missing_platt_predictions,
  COUNTIF(prediction_raw < 0 OR prediction_raw > 1) AS invalid_raw_predictions,
  COUNTIF(prediction_platt < 0 OR prediction_platt > 1) AS invalid_platt_predictions,
  MIN(prediction_raw) AS minimum_raw_probability,
  MAX(prediction_raw) AS maximum_raw_probability,
  MIN(prediction_platt) AS minimum_platt_probability,
  MAX(prediction_platt) AS maximum_platt_probability
FROM `{outer_table_15R}`
"""

outer_verification_15R = client_15R.query(
    outer_verify_sql_15R,
    location=BQ_LOCATION_15R,
).to_dataframe()

vr = outer_verification_15R.iloc[0]
outer_checks_15R = {
    "prediction_rows": 11736,
    "distinct_rows": 11736,
    "outer_folds": 1,
    "minimum_outer_fold": 3,
    "maximum_outer_fold": 3,
    "events": 608,
    "nonevents": 11128,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}
for field, expected in outer_checks_15R.items():
    actual = int(vr[field])
    if actual != expected:
        raise RuntimeError(
            f"Outer prediction verification failed: "
            f"{field}={actual}, expected={expected}"
        )

# ------------------------------------------------------------
# 4. Repair aggregate Drive CSV metadata
# ------------------------------------------------------------

csv_outer_fold_files_15R = [
    "15A_xgboost_inner_fit_audit_outer3.csv",
    "15A_xgboost_outer3_inner_checkpoint_summary.csv",
    "15B_xgboost_selected_model_outer3.csv",
    "15C_xgboost_outer3_test_results.csv",
    "15C_xgboost_final_model_outer3.csv",
]

for filename in csv_outer_fold_files_15R:
    path = os.path.join(MODEL_OUTPUT_DIR_15R, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    df = pd.read_csv(path)
    if "outer_fold" not in df.columns:
        raise RuntimeError(
            f"{filename}: outer_fold column not found."
        )
    df["outer_fold"] = 3
    df.to_csv(path, index=False)

# ------------------------------------------------------------
# 5. Repair selection JSON, SHA, and selected-model CSV linkage
# ------------------------------------------------------------

selection_json_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15B_xgboost_selection_calibration_outer3.json",
)
selection_sha_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15B_xgboost_selection_calibration_outer3_SHA256.txt",
)
selected_model_csv_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15B_xgboost_selected_model_outer3.csv",
)

with open(selection_json_path_15R, "r", encoding="utf-8") as fh:
    selection_obj_15R = json.load(fh)

selection_obj_15R["outer_fold"] = 3

with open(selection_json_path_15R, "w", encoding="utf-8") as fh:
    json.dump(
        selection_obj_15R,
        fh,
        indent=2,
        ensure_ascii=False,
    )

with open(selection_json_path_15R, "rb") as fh:
    corrected_selection_sha_15R = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(selection_sha_path_15R, "w", encoding="utf-8") as fh:
    fh.write(corrected_selection_sha_15R + "\n")

selected_model_csv_15R = pd.read_csv(
    selected_model_csv_path_15R
)
selected_model_csv_15R["outer_fold"] = 3
selected_model_csv_15R.to_csv(
    selected_model_csv_path_15R,
    index=False,
)

# ------------------------------------------------------------
# 6. Repair final model CSV selection SHA linkage
# ------------------------------------------------------------

final_model_csv_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15C_xgboost_final_model_outer3.csv",
)

final_model_csv_15R = pd.read_csv(
    final_model_csv_path_15R
)
final_model_csv_15R["outer_fold"] = 3
if "selection_sha256" in final_model_csv_15R.columns:
    final_model_csv_15R["selection_sha256"] = (
        corrected_selection_sha_15R
    )
final_model_csv_15R.to_csv(
    final_model_csv_path_15R,
    index=False,
)

# ------------------------------------------------------------
# 7. Repair evaluation JSON and SHA
# ------------------------------------------------------------

evaluation_json_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15C_xgboost_final_evaluation_outer3.json",
)
evaluation_sha_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15C_xgboost_final_evaluation_outer3_SHA256.txt",
)

with open(evaluation_json_path_15R, "r", encoding="utf-8") as fh:
    evaluation_obj_15R = json.load(fh)

evaluation_obj_15R["outer_fold"] = 3
evaluation_obj_15R["selection_sha256"] = (
    corrected_selection_sha_15R
)

with open(evaluation_json_path_15R, "w", encoding="utf-8") as fh:
    json.dump(
        evaluation_obj_15R,
        fh,
        indent=2,
        ensure_ascii=False,
    )

with open(evaluation_json_path_15R, "rb") as fh:
    corrected_evaluation_sha_15R = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(evaluation_sha_path_15R, "w", encoding="utf-8") as fh:
    fh.write(corrected_evaluation_sha_15R + "\n")

# ------------------------------------------------------------
# 8. Final aggregate audit
# ------------------------------------------------------------

checkpoint_summary_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15A_xgboost_outer3_inner_checkpoint_summary.csv",
)
selected_model_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15B_xgboost_selected_model_outer3.csv",
)
test_results_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15C_xgboost_outer3_test_results.csv",
)
final_model_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15C_xgboost_final_model_outer3.csv",
)

checkpoint_summary_15R = pd.read_csv(
    checkpoint_summary_path_15R
)
selected_model_15R = pd.read_csv(
    selected_model_path_15R
)
test_results_15R = pd.read_csv(
    test_results_path_15R
)
final_model_summary_15R = pd.read_csv(
    final_model_path_15R
)

for name, df in [
    ("checkpoint summary", checkpoint_summary_15R),
    ("selected model", selected_model_15R),
    ("test results", test_results_15R),
    ("final model summary", final_model_summary_15R),
]:
    if set(pd.to_numeric(df["outer_fold"]).astype(int).unique()) != {3}:
        raise RuntimeError(
            f"{name}: outer_fold metadata was not fully repaired."
        )

# Ensure performance numbers remain unchanged.
raw_row_15R = test_results_15R.loc[
    test_results_15R["probability_type"].astype(str) == "raw"
].iloc[0]
platt_row_15R = test_results_15R.loc[
    test_results_15R["probability_type"].astype(str)
    == "platt_calibrated"
].iloc[0]

expected_metrics_15R = {
    ("raw", "auroc"): 0.884105,
    ("raw", "auprc"): 0.407844,
    ("platt", "auroc"): 0.884105,
    ("platt", "auprc"): 0.407844,
}
if not np.isclose(float(raw_row_15R["auroc"]), 0.884105, atol=5e-7):
    raise RuntimeError("Raw AUROC changed unexpectedly.")
if not np.isclose(float(raw_row_15R["auprc"]), 0.407844, atol=5e-7):
    raise RuntimeError("Raw AUPRC changed unexpectedly.")
if not np.isclose(float(platt_row_15R["auroc"]), 0.884105, atol=5e-7):
    raise RuntimeError("Platt AUROC changed unexpectedly.")
if not np.isclose(float(platt_row_15R["auprc"]), 0.407844, atol=5e-7):
    raise RuntimeError("Platt AUPRC changed unexpectedly.")

repair_manifest_15R = {
    "repair_version": "15R",
    "model_family": "xgboost",
    "outer_fold": 3,
    "repair_type": "metadata_only",
    "model_refit": False,
    "predictions_recomputed": False,
    "probabilities_modified": False,
    "labels_modified": False,
    "corrected_field": "outer_fold",
    "incorrect_value": 2,
    "corrected_value": 3,
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_15R,
    "corrected_selection_sha256": corrected_selection_sha_15R,
    "corrected_evaluation_sha256": corrected_evaluation_sha_15R,
    "secure_outer_prediction_table": outer_table_15R,
    "inner_checkpoint_tables_repaired": [
        row["table_id"]
        for row in inner_repair_rows_15R
    ],
    "patient_level_file_written_to_drive": False,
}

repair_manifest_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15R_xgboost_outer3_metadata_repair_manifest.json",
)
repair_manifest_sha_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15R_xgboost_outer3_metadata_repair_manifest_SHA256.txt",
)

with open(repair_manifest_path_15R, "w", encoding="utf-8") as fh:
    json.dump(
        repair_manifest_15R,
        fh,
        indent=2,
        ensure_ascii=False,
    )

with open(repair_manifest_path_15R, "rb") as fh:
    repair_manifest_sha_15R = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(repair_manifest_sha_path_15R, "w", encoding="utf-8") as fh:
    fh.write(repair_manifest_sha_15R + "\n")

print("\n15R INNER BIGQUERY METADATA REPAIR")
display(inner_repair_summary_15R)

print("\n15R OUTER-TEST BIGQUERY VERIFICATION")
display(outer_verification_15R)

print("\n15R CORRECTED SELECTED MODEL")
display(selected_model_15R)

print("\n15R CORRECTED OUTER-FOLD-3 TEST RESULTS")
display(test_results_15R)

print("\nCorrected Selection SHA-256:")
print(corrected_selection_sha_15R)

print("\nCorrected Evaluation SHA-256:")
print(corrected_evaluation_sha_15R)

print("\n15R Repair Manifest SHA-256:")
print(repair_manifest_sha_15R)

print(
    "\n15R PASS: XGBoost outer-fold-3 metadata was corrected "
    "from outer_fold=2 to outer_fold=3 without refitting any model "
    "or changing any prediction probability."
)
print(
    "Patient-level prediction data remained only in BigQuery; "
    "no patient-level file was written to Google Drive."
)

In [ ]:
print("STARTING XGBOOST OUTER FOLD 3 METADATA REPAIR — CODE VERSION 15R")

import os
import json
import hashlib
import numpy as np
import pandas as pd

from google.cloud import bigquery
from google.api_core.exceptions import NotFound
from IPython.display import display

# ============================================================
# 15R — REPAIR ONLY
#
# Purpose:
# Correct outer_fold metadata from 2 -> 3 in the already-completed
# XGBoost outer-fold-3 artifacts.
#
# IMPORTANT:
# - No model fitting.
# - No hyperparameter selection.
# - No recalculation of predictions.
# - No change to probabilities or labels.
# - BigQuery tables are overwritten via WRITE_TRUNCATE only.
# - No DELETE / INSERT / UPDATE / MERGE.
# ============================================================

EXPECTED_PROTOCOL_SHA_15R = (
    "3434db5dd0b4b5145950fc07fba3d007"
    "829738d800b88ec40fbdb09879188264"
)

# Resolve shared objects if available; otherwise use locked project values.
PROJECT_ID_15R = f"{PROJECT_ID}"
TARGET_DATASET_15R = (
    TARGET_DATASET
    if "TARGET_DATASET" in globals()
    else f"{PROJECT_ID}.aki_jcmc_v2"
)
BQ_LOCATION_15R = (
    BQ_LOCATION
    if "BQ_LOCATION" in globals()
    else "US"
)
MODEL_OUTPUT_DIR_15R = (
    MODEL_OUTPUT_DIR
    if "MODEL_OUTPUT_DIR" in globals()
    else f"{OUTPUT_ROOT}/05_MODELING_OUTPUTS"
)

if "client" in globals():
    client_15R = client
else:
    client_15R = bigquery.Client(project=PROJECT_ID_15R)

os.makedirs(MODEL_OUTPUT_DIR_15R, exist_ok=True)

# ------------------------------------------------------------
# 1. Verify the locked XGBoost protocol hash
# ------------------------------------------------------------

protocol_sha_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "13A_locked_xgboost_model_protocol_v1_SHA256.txt",
)

if not os.path.exists(protocol_sha_path_15R):
    raise FileNotFoundError(
        "Locked XGBoost protocol SHA file not found: "
        + protocol_sha_path_15R
    )

with open(protocol_sha_path_15R, "r", encoding="utf-8") as fh:
    observed_protocol_sha_15R = fh.read().strip()

if observed_protocol_sha_15R != EXPECTED_PROTOCOL_SHA_15R:
    raise RuntimeError(
        "Locked XGBoost protocol SHA mismatch. "
        f"Observed={observed_protocol_sha_15R}"
    )

print("Locked XGBoost protocol SHA-256:")
print(observed_protocol_sha_15R)

# ------------------------------------------------------------
# 2. Repair five inner OOF BigQuery tables
# ------------------------------------------------------------

inner_expected_15R = {
    1: {"patients": 7934, "events": 380},
    2: {"patients": 5835, "events": 307},
    3: {"patients": 8356, "events": 458},
    4: {"patients": 13173, "events": 691},
    5: {"patients": 11457, "events": 588},
}

inner_schema_15R = [
    bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("inner_fold", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("candidate_id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
]

inner_load_config_15R = bigquery.LoadJobConfig(
    schema=inner_schema_15R,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

inner_repair_rows_15R = []

for inner_fold in range(1, 6):
    table_id = (
        f"{TARGET_DATASET_15R}."
        f"model_xgb_inner_oof_outer3_inner{inner_fold}_v1"
    )

    try:
        client_15R.get_table(table_id)
    except NotFound:
        raise FileNotFoundError(
            f"Missing outer-fold-3 inner checkpoint table: {table_id}"
        )

    query = f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    df = client_15R.query(
        query,
        location=BQ_LOCATION_15R,
    ).to_dataframe(create_bqstorage_client=True)

    expected_patients = inner_expected_15R[inner_fold]["patients"]
    expected_events = inner_expected_15R[inner_fold]["events"]
    expected_rows = expected_patients * 6

    if len(df) != expected_rows:
        raise RuntimeError(
            f"Inner {inner_fold}: rows={len(df)}, expected={expected_rows}"
        )

    if df["id_row"].astype(str).nunique() != expected_patients:
        raise RuntimeError(
            f"Inner {inner_fold}: distinct patient count mismatch."
        )

    if df["candidate_id"].astype(str).nunique() != 6:
        raise RuntimeError(
            f"Inner {inner_fold}: expected six candidates."
        )

    if df.duplicated(["id_row", "candidate_id"]).any():
        raise RuntimeError(
            f"Inner {inner_fold}: duplicate candidate-patient rows found."
        )

    if set(pd.to_numeric(df["inner_fold"]).astype(int).unique()) != {inner_fold}:
        raise RuntimeError(
            f"Inner {inner_fold}: incorrect inner_fold metadata."
        )

    positive_prediction_rows = int(
        pd.to_numeric(df["label_stage23"]).astype(int).sum()
    )
    if positive_prediction_rows != expected_events * 6:
        raise RuntimeError(
            f"Inner {inner_fold}: positive prediction row count mismatch."
        )

    if df["prediction_raw"].isna().any():
        raise RuntimeError(
            f"Inner {inner_fold}: missing probabilities."
        )

    probs = pd.to_numeric(df["prediction_raw"], errors="raise").astype(float)
    if not probs.between(0, 1).all():
        raise RuntimeError(
            f"Inner {inner_fold}: invalid probabilities."
        )

    before_values = sorted(
        pd.to_numeric(df["outer_fold"], errors="raise")
        .astype(int)
        .unique()
        .tolist()
    )

    # Metadata-only correction.
    df["id_row"] = df["id_row"].astype(str)
    df["outer_fold"] = np.int64(3)
    df["inner_fold"] = pd.to_numeric(
        df["inner_fold"], errors="raise"
    ).astype(np.int64)
    df["candidate_id"] = df["candidate_id"].astype(str)
    df["label_stage23"] = pd.to_numeric(
        df["label_stage23"], errors="raise"
    ).astype(np.int64)
    df["prediction_raw"] = probs.astype(np.float64)

    client_15R.load_table_from_dataframe(
        df,
        table_id,
        job_config=inner_load_config_15R,
        location=BQ_LOCATION_15R,
    ).result()

    verify_sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS patients,
      COUNT(DISTINCT candidate_id) AS candidates,
      COUNT(DISTINCT outer_fold) AS outer_folds,
      MIN(outer_fold) AS min_outer_fold,
      MAX(outer_fold) AS max_outer_fold,
      COUNT(DISTINCT inner_fold) AS inner_folds,
      MIN(inner_fold) AS min_inner_fold,
      MAX(inner_fold) AS max_inner_fold,
      COUNTIF(label_stage23=1) AS positive_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(prediction_raw < 0 OR prediction_raw > 1) AS invalid_predictions
    FROM `{table_id}`
    """
    v = client_15R.query(
        verify_sql,
        location=BQ_LOCATION_15R,
    ).to_dataframe().iloc[0]

    checks = {
        "row_count": expected_rows,
        "patients": expected_patients,
        "candidates": 6,
        "outer_folds": 1,
        "min_outer_fold": 3,
        "max_outer_fold": 3,
        "inner_folds": 1,
        "min_inner_fold": inner_fold,
        "max_inner_fold": inner_fold,
        "positive_prediction_rows": expected_events * 6,
        "missing_predictions": 0,
        "invalid_predictions": 0,
    }
    for field, expected in checks.items():
        actual = int(v[field])
        if actual != expected:
            raise RuntimeError(
                f"{table_id}: {field}={actual}, expected={expected}"
            )

    inner_repair_rows_15R.append({
        "inner_fold": inner_fold,
        "row_count": expected_rows,
        "patients": expected_patients,
        "events": expected_events,
        "outer_fold_before": str(before_values),
        "outer_fold_after": 3,
        "table_id": table_id,
    })

inner_repair_summary_15R = pd.DataFrame(inner_repair_rows_15R)

# ------------------------------------------------------------
# 3. Repair outer-test prediction BigQuery table
# ------------------------------------------------------------

outer_table_15R = (
    f"{TARGET_DATASET_15R}."
    "model_xgb_outer_predictions_outer3_v1"
)

try:
    client_15R.get_table(outer_table_15R)
except NotFound:
    raise FileNotFoundError(
        "Missing outer-fold-3 XGBoost prediction table: "
        + outer_table_15R
    )

outer_query_15R = f"""
SELECT
  id_row,
  outer_fold,
  label_stage23,
  prediction_raw,
  prediction_platt,
  model_name,
  model_version
FROM `{outer_table_15R}`
"""

outer_df_15R = client_15R.query(
    outer_query_15R,
    location=BQ_LOCATION_15R,
).to_dataframe(create_bqstorage_client=True)

if len(outer_df_15R) != 11736:
    raise RuntimeError(
        f"Outer prediction rows={len(outer_df_15R)}, expected=11736"
    )

if outer_df_15R["id_row"].astype(str).nunique() != 11736:
    raise RuntimeError("Outer prediction distinct patient count mismatch.")

if int(
    pd.to_numeric(outer_df_15R["label_stage23"]).astype(int).sum()
) != 608:
    raise RuntimeError("Outer prediction event count mismatch.")

if outer_df_15R["id_row"].astype(str).duplicated().any():
    raise RuntimeError("Duplicate outer-test patients found.")

for col in ["prediction_raw", "prediction_platt"]:
    p = pd.to_numeric(outer_df_15R[col], errors="raise").astype(float)
    if p.isna().any() or not p.between(0, 1).all():
        raise RuntimeError(f"Invalid values in {col}.")

outer_before_15R = sorted(
    pd.to_numeric(
        outer_df_15R["outer_fold"], errors="raise"
    ).astype(int).unique().tolist()
)

outer_df_15R["id_row"] = outer_df_15R["id_row"].astype(str)
outer_df_15R["outer_fold"] = np.int64(3)
outer_df_15R["label_stage23"] = pd.to_numeric(
    outer_df_15R["label_stage23"], errors="raise"
).astype(np.int64)
outer_df_15R["prediction_raw"] = pd.to_numeric(
    outer_df_15R["prediction_raw"], errors="raise"
).astype(np.float64)
outer_df_15R["prediction_platt"] = pd.to_numeric(
    outer_df_15R["prediction_platt"], errors="raise"
).astype(np.float64)
outer_df_15R["model_name"] = outer_df_15R["model_name"].astype(str)
outer_df_15R["model_version"] = outer_df_15R["model_version"].astype(str)

outer_schema_15R = [
    bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
    bigquery.SchemaField("prediction_platt", "FLOAT", mode="REQUIRED"),
    bigquery.SchemaField("model_name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("model_version", "STRING", mode="REQUIRED"),
]

outer_load_config_15R = bigquery.LoadJobConfig(
    schema=outer_schema_15R,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

client_15R.load_table_from_dataframe(
    outer_df_15R,
    outer_table_15R,
    job_config=outer_load_config_15R,
    location=BQ_LOCATION_15R,
).result()

outer_verify_sql_15R = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23=1) AS events,
  COUNTIF(label_stage23=0) AS nonevents,
  COUNTIF(prediction_raw IS NULL) AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL) AS missing_platt_predictions,
  COUNTIF(prediction_raw < 0 OR prediction_raw > 1) AS invalid_raw_predictions,
  COUNTIF(prediction_platt < 0 OR prediction_platt > 1) AS invalid_platt_predictions,
  MIN(prediction_raw) AS minimum_raw_probability,
  MAX(prediction_raw) AS maximum_raw_probability,
  MIN(prediction_platt) AS minimum_platt_probability,
  MAX(prediction_platt) AS maximum_platt_probability
FROM `{outer_table_15R}`
"""

outer_verification_15R = client_15R.query(
    outer_verify_sql_15R,
    location=BQ_LOCATION_15R,
).to_dataframe()

vr = outer_verification_15R.iloc[0]
outer_checks_15R = {
    "prediction_rows": 11736,
    "distinct_rows": 11736,
    "outer_folds": 1,
    "minimum_outer_fold": 3,
    "maximum_outer_fold": 3,
    "events": 608,
    "nonevents": 11128,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}
for field, expected in outer_checks_15R.items():
    actual = int(vr[field])
    if actual != expected:
        raise RuntimeError(
            f"Outer prediction verification failed: "
            f"{field}={actual}, expected={expected}"
        )

# ------------------------------------------------------------
# 4. Repair aggregate Drive CSV metadata
# ------------------------------------------------------------

csv_outer_fold_files_15R = [
    "15A_xgboost_inner_fit_audit_outer3.csv",
    "15A_xgboost_outer3_inner_checkpoint_summary.csv",
    "15B_xgboost_selected_model_outer3.csv",
    "15C_xgboost_outer3_test_results.csv",
    "15C_xgboost_final_model_outer3.csv",
]

for filename in csv_outer_fold_files_15R:
    path = os.path.join(MODEL_OUTPUT_DIR_15R, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    df = pd.read_csv(path)
    if "outer_fold" not in df.columns:
        raise RuntimeError(
            f"{filename}: outer_fold column not found."
        )
    df["outer_fold"] = 3
    df.to_csv(path, index=False)

# ------------------------------------------------------------
# 5. Repair selection JSON, SHA, and selected-model CSV linkage
# ------------------------------------------------------------

selection_json_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15B_xgboost_selection_calibration_outer3.json",
)
selection_sha_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15B_xgboost_selection_calibration_outer3_SHA256.txt",
)
selected_model_csv_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15B_xgboost_selected_model_outer3.csv",
)

with open(selection_json_path_15R, "r", encoding="utf-8") as fh:
    selection_obj_15R = json.load(fh)

selection_obj_15R["outer_fold"] = 3

with open(selection_json_path_15R, "w", encoding="utf-8") as fh:
    json.dump(
        selection_obj_15R,
        fh,
        indent=2,
        ensure_ascii=False,
    )

with open(selection_json_path_15R, "rb") as fh:
    corrected_selection_sha_15R = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(selection_sha_path_15R, "w", encoding="utf-8") as fh:
    fh.write(corrected_selection_sha_15R + "\n")

selected_model_csv_15R = pd.read_csv(
    selected_model_csv_path_15R
)
selected_model_csv_15R["outer_fold"] = 3
selected_model_csv_15R.to_csv(
    selected_model_csv_path_15R,
    index=False,
)

# ------------------------------------------------------------
# 6. Repair final model CSV selection SHA linkage
# ------------------------------------------------------------

final_model_csv_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15C_xgboost_final_model_outer3.csv",
)

final_model_csv_15R = pd.read_csv(
    final_model_csv_path_15R
)
final_model_csv_15R["outer_fold"] = 3
if "selection_sha256" in final_model_csv_15R.columns:
    final_model_csv_15R["selection_sha256"] = (
        corrected_selection_sha_15R
    )
final_model_csv_15R.to_csv(
    final_model_csv_path_15R,
    index=False,
)

# ------------------------------------------------------------
# 7. Repair evaluation JSON and SHA
# ------------------------------------------------------------

evaluation_json_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15C_xgboost_final_evaluation_outer3.json",
)
evaluation_sha_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15C_xgboost_final_evaluation_outer3_SHA256.txt",
)

with open(evaluation_json_path_15R, "r", encoding="utf-8") as fh:
    evaluation_obj_15R = json.load(fh)

evaluation_obj_15R["outer_fold"] = 3
evaluation_obj_15R["selection_sha256"] = (
    corrected_selection_sha_15R
)

with open(evaluation_json_path_15R, "w", encoding="utf-8") as fh:
    json.dump(
        evaluation_obj_15R,
        fh,
        indent=2,
        ensure_ascii=False,
    )

with open(evaluation_json_path_15R, "rb") as fh:
    corrected_evaluation_sha_15R = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(evaluation_sha_path_15R, "w", encoding="utf-8") as fh:
    fh.write(corrected_evaluation_sha_15R + "\n")

# ------------------------------------------------------------
# 8. Final aggregate audit
# ------------------------------------------------------------

checkpoint_summary_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15A_xgboost_outer3_inner_checkpoint_summary.csv",
)
selected_model_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15B_xgboost_selected_model_outer3.csv",
)
test_results_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15C_xgboost_outer3_test_results.csv",
)
final_model_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15C_xgboost_final_model_outer3.csv",
)

checkpoint_summary_15R = pd.read_csv(
    checkpoint_summary_path_15R
)
selected_model_15R = pd.read_csv(
    selected_model_path_15R
)
test_results_15R = pd.read_csv(
    test_results_path_15R
)
final_model_summary_15R = pd.read_csv(
    final_model_path_15R
)

for name, df in [
    ("checkpoint summary", checkpoint_summary_15R),
    ("selected model", selected_model_15R),
    ("test results", test_results_15R),
    ("final model summary", final_model_summary_15R),
]:
    if set(pd.to_numeric(df["outer_fold"]).astype(int).unique()) != {3}:
        raise RuntimeError(
            f"{name}: outer_fold metadata was not fully repaired."
        )

# Ensure performance numbers remain unchanged.
raw_row_15R = test_results_15R.loc[
    test_results_15R["probability_type"].astype(str) == "raw"
].iloc[0]
platt_row_15R = test_results_15R.loc[
    test_results_15R["probability_type"].astype(str)
    == "platt_calibrated"
].iloc[0]

expected_metrics_15R = {
    ("raw", "auroc"): 0.884105,
    ("raw", "auprc"): 0.407844,
    ("platt", "auroc"): 0.884105,
    ("platt", "auprc"): 0.407844,
}
if not np.isclose(float(raw_row_15R["auroc"]), 0.884105, atol=5e-7):
    raise RuntimeError("Raw AUROC changed unexpectedly.")
if not np.isclose(float(raw_row_15R["auprc"]), 0.407844, atol=5e-7):
    raise RuntimeError("Raw AUPRC changed unexpectedly.")
if not np.isclose(float(platt_row_15R["auroc"]), 0.884105, atol=5e-7):
    raise RuntimeError("Platt AUROC changed unexpectedly.")
if not np.isclose(float(platt_row_15R["auprc"]), 0.407844, atol=5e-7):
    raise RuntimeError("Platt AUPRC changed unexpectedly.")

repair_manifest_15R = {
    "repair_version": "15R",
    "model_family": "xgboost",
    "outer_fold": 3,
    "repair_type": "metadata_only",
    "model_refit": False,
    "predictions_recomputed": False,
    "probabilities_modified": False,
    "labels_modified": False,
    "corrected_field": "outer_fold",
    "incorrect_value": 2,
    "corrected_value": 3,
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_15R,
    "corrected_selection_sha256": corrected_selection_sha_15R,
    "corrected_evaluation_sha256": corrected_evaluation_sha_15R,
    "secure_outer_prediction_table": outer_table_15R,
    "inner_checkpoint_tables_repaired": [
        row["table_id"]
        for row in inner_repair_rows_15R
    ],
    "patient_level_file_written_to_drive": False,
}

repair_manifest_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15R_xgboost_outer3_metadata_repair_manifest.json",
)
repair_manifest_sha_path_15R = os.path.join(
    MODEL_OUTPUT_DIR_15R,
    "15R_xgboost_outer3_metadata_repair_manifest_SHA256.txt",
)

with open(repair_manifest_path_15R, "w", encoding="utf-8") as fh:
    json.dump(
        repair_manifest_15R,
        fh,
        indent=2,
        ensure_ascii=False,
    )

with open(repair_manifest_path_15R, "rb") as fh:
    repair_manifest_sha_15R = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(repair_manifest_sha_path_15R, "w", encoding="utf-8") as fh:
    fh.write(repair_manifest_sha_15R + "\n")

print("\n15R INNER BIGQUERY METADATA REPAIR")
display(inner_repair_summary_15R)

print("\n15R OUTER-TEST BIGQUERY VERIFICATION")
display(outer_verification_15R)

print("\n15R CORRECTED SELECTED MODEL")
display(selected_model_15R)

print("\n15R CORRECTED OUTER-FOLD-3 TEST RESULTS")
display(test_results_15R)

print("\nCorrected Selection SHA-256:")
print(corrected_selection_sha_15R)

print("\nCorrected Evaluation SHA-256:")
print(corrected_evaluation_sha_15R)

print("\n15R Repair Manifest SHA-256:")
print(repair_manifest_sha_15R)

print(
    "\n15R PASS: XGBoost outer-fold-3 metadata was corrected "
    "from outer_fold=2 to outer_fold=3 without refitting any model "
    "or changing any prediction probability."
)
print(
    "Patient-level prediction data remained only in BigQuery; "
    "no patient-level file was written to Google Drive."
)

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

from xgboost import XGBClassifier

from IPython.display import display

print("STARTING XGBOOST OUTER FOLD 4 — CODE VERSION 16")

# ============================================================
# 14 — XGBOOST OUTER FOLD 2 COMPLETE NESTED MODELLING
#
# Design:
# - Same locked hospital-disjoint outer folds.
# - Same locked hospital-disjoint inner folds.
# - Core feature set only.
# - No SMOTE.
# - No class weighting / scale_pos_weight = 1.
# - No early stopping.
# - Fixed candidate grid locked before outer-test evaluation.
# - Selection: pooled inner-OOF AUPRC descending,
#              AUROC descending, Brier ascending.
# - Platt calibration learned only from selected candidate's
#   pooled inner-OOF predictions.
# - Patient-level predictions stored only in BigQuery.
# ============================================================

OUTER_FOLD_16 = 4
MODEL_RANDOM_SEED_16 = 20260721

EXPECTED_SPLIT_16 = {
    "training_rows": 46803,
    "test_rows": 11688,
    "training_hospitals": 159,
    "test_hospitals": 39,
    "training_events": 2426,
    "test_events": 606,
}

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_objects_16 = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_16 = [
    name for name in required_objects_16
    if name not in globals()
]

if missing_objects_16:
    raise RuntimeError(
        "Missing runtime objects: "
        + ", ".join(missing_objects_16)
        + ". Run 07A and 07B first."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"Expected 58,491 cohort rows; found {len(core_df_07B)}."
    )

if len(predictor_columns_07B) != 159:
    raise RuntimeError("Expected 159 core predictors.")

if len(numeric_columns_07B) != 156:
    raise RuntimeError("Expected 156 numeric predictors.")

if len(categorical_columns_07B) != 3:
    raise RuntimeError("Expected 3 categorical predictors.")

# ------------------------------------------------------------
# 2. Lock the XGBoost protocol BEFORE test evaluation
# ------------------------------------------------------------

candidate_grid_16 = [
    {
        "candidate_id": "XGB01",
        "n_estimators": 250,
        "max_depth": 2,
        "learning_rate": 0.03,
        "min_child_weight": 5.0,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
    },
    {
        "candidate_id": "XGB02",
        "n_estimators": 350,
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 5.0,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
    },
    {
        "candidate_id": "XGB03",
        "n_estimators": 450,
        "max_depth": 3,
        "learning_rate": 0.02,
        "min_child_weight": 10.0,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "gamma": 0.0,
        "reg_alpha": 0.10,
        "reg_lambda": 10.0,
    },
    {
        "candidate_id": "XGB04",
        "n_estimators": 350,
        "max_depth": 4,
        "learning_rate": 0.03,
        "min_child_weight": 10.0,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "gamma": 0.10,
        "reg_alpha": 0.10,
        "reg_lambda": 10.0,
    },
    {
        "candidate_id": "XGB05",
        "n_estimators": 450,
        "max_depth": 2,
        "learning_rate": 0.02,
        "min_child_weight": 10.0,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "gamma": 0.0,
        "reg_alpha": 0.50,
        "reg_lambda": 10.0,
    },
    {
        "candidate_id": "XGB06",
        "n_estimators": 450,
        "max_depth": 4,
        "learning_rate": 0.02,
        "min_child_weight": 15.0,
        "subsample": 0.90,
        "colsample_bytree": 0.80,
        "gamma": 0.20,
        "reg_alpha": 0.50,
        "reg_lambda": 15.0,
    },
]

xgb_protocol_16 = {
    "protocol_name": "xgboost_core_nested_hospital_cv_v1",
    "model_family": "XGBoost",
    "feature_set": "core_159",
    "outer_cv": "locked 5-fold hospital-disjoint outer folds",
    "inner_cv": "locked 5-fold hospital-disjoint inner folds",
    "primary_selection_metric": "pooled inner OOF AUPRC descending",
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "class_weighting": False,
    "scale_pos_weight": 1.0,
    "smote": False,
    "early_stopping": False,
    "tree_method": "hist",
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": MODEL_RANDOM_SEED_16,
    "candidate_grid": candidate_grid_16,
    "calibration": (
        "Platt calibration fit only on selected candidate "
        "pooled inner-OOF logits"
    ),
    "outer_test_use": (
        "diagnostic evaluation only; never used for tuning"
    ),
}

protocol_path_16 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13A_locked_xgboost_model_protocol_v1.json",
)

protocol_sha_path_16 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13A_locked_xgboost_model_protocol_v1_SHA256.txt",
)

protocol_text_16 = json.dumps(
    xgb_protocol_16,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
)

protocol_sha_16 = hashlib.sha256(
    protocol_text_16.encode("utf-8")
).hexdigest()

if os.path.exists(protocol_path_16):
    with open(protocol_path_16, "r", encoding="utf-8") as fh:
        existing_protocol_text_16 = fh.read()
    existing_protocol_sha_16 = hashlib.sha256(
        existing_protocol_text_16.encode("utf-8")
    ).hexdigest()

    if existing_protocol_sha_16 != protocol_sha_16:
        raise RuntimeError(
            "An existing XGBoost protocol file differs from "
            "the currently locked protocol. Stop and audit."
        )
else:
    with open(protocol_path_16, "w", encoding="utf-8") as fh:
        fh.write(protocol_text_16)

with open(protocol_sha_path_16, "w", encoding="utf-8") as fh:
    fh.write(protocol_sha_16 + "\n")

print("Locked XGBoost protocol SHA-256:")
print(protocol_sha_16)

# ------------------------------------------------------------
# 3. Load locked inner hospital map
# ------------------------------------------------------------

inner_mapping_path_16 = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_16):
    raise FileNotFoundError(
        "Locked inner-fold map not found: "
        + inner_mapping_path_16
    )

inner_mapping_all_16 = pd.read_csv(
    inner_mapping_path_16,
    dtype={"group_hospital": str},
)

inner_mapping_part_16 = (
    inner_mapping_all_16.loc[
        inner_mapping_all_16["outer_fold"].astype(int)
        == OUTER_FOLD_16,
        ["group_hospital", "inner_fold"],
    ]
    .copy()
)

inner_mapping_part_16["group_hospital"] = (
    inner_mapping_part_16["group_hospital"].astype(str)
)
inner_mapping_part_16["inner_fold"] = (
    inner_mapping_part_16["inner_fold"].astype(int)
)

if len(inner_mapping_part_16) != 159:
    raise RuntimeError(
        "Expected 159 outer-fold-4 training hospitals "
        "in the locked inner map."
    )

if inner_mapping_part_16["group_hospital"].duplicated().any():
    raise RuntimeError("Duplicate hospital in locked inner map.")

hospital_to_inner_fold_16 = dict(
    zip(
        inner_mapping_part_16["group_hospital"],
        inner_mapping_part_16["inner_fold"],
    )
)

# ------------------------------------------------------------
# 4. Prepare outer fold 1 matrices
# ------------------------------------------------------------

X_all_16 = core_df_07B[predictor_columns_07B].copy()

for column in numeric_columns_07B:
    X_all_16[column] = pd.to_numeric(
        X_all_16[column],
        errors="coerce",
    ).astype("float64")

for column in categorical_columns_07B:
    category_series = X_all_16[column].astype("object")
    X_all_16[column] = category_series.where(
        pd.notna(category_series),
        np.nan,
    )

outer_fold_vector_16 = (
    core_df_07B["outer_fold"].astype(int).to_numpy()
)

outer_training_mask_16 = (
    outer_fold_vector_16 != OUTER_FOLD_16
)
outer_test_mask_16 = (
    outer_fold_vector_16 == OUTER_FOLD_16
)

X_outer_training_16 = (
    X_all_16.loc[outer_training_mask_16]
    .reset_index(drop=True)
)

X_outer_test_16 = (
    X_all_16.loc[outer_test_mask_16]
    .reset_index(drop=True)
)

outer_training_meta_16 = (
    core_df_07B.loc[
        outer_training_mask_16,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_16 = (
    core_df_07B.loc[
        outer_test_mask_16,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [
    outer_training_meta_16,
    outer_test_meta_16,
]:
    dataframe["id_row"] = dataframe["id_row"].astype(str)
    dataframe["group_hospital"] = (
        dataframe["group_hospital"].astype(str)
    )
    dataframe["label_stage23"] = (
        dataframe["label_stage23"].astype(int)
    )

y_outer_training_16 = (
    outer_training_meta_16["label_stage23"]
    .to_numpy(dtype=np.int8)
)
y_outer_test_16 = (
    outer_test_meta_16["label_stage23"]
    .to_numpy(dtype=np.int8)
)

groups_outer_training_16 = (
    outer_training_meta_16["group_hospital"]
    .to_numpy(dtype=str)
)

training_hospitals_16 = set(
    outer_training_meta_16["group_hospital"]
)
test_hospitals_16 = set(
    outer_test_meta_16["group_hospital"]
)
hospital_overlap_16 = (
    training_hospitals_16 & test_hospitals_16
)

if hospital_overlap_16:
    raise RuntimeError(
        "Outer training/test hospital overlap detected."
    )

actual_split_16 = {
    "training_rows": len(X_outer_training_16),
    "test_rows": len(X_outer_test_16),
    "training_hospitals": len(training_hospitals_16),
    "test_hospitals": len(test_hospitals_16),
    "training_events": int(y_outer_training_16.sum()),
    "test_events": int(y_outer_test_16.sum()),
}

for metric, expected_value in EXPECTED_SPLIT_16.items():
    actual_value = actual_split_16[metric]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: found={actual_value}, "
            f"expected={expected_value}"
        )

inner_fold_vector_16 = np.array(
    [
        hospital_to_inner_fold_16.get(hospital, -1)
        for hospital in groups_outer_training_16
    ],
    dtype=int,
)

if (inner_fold_vector_16 == -1).any():
    raise RuntimeError(
        "Some outer-training hospitals have no inner-fold assignment."
    )

if set(np.unique(inner_fold_vector_16)) != {1, 2, 3, 4, 5}:
    raise RuntimeError("Inner-fold values are not exactly 1–5.")

# ------------------------------------------------------------
# 5. Preprocessor and model constructors
# ------------------------------------------------------------

def make_preprocessor_16():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns_07B,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns_07B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_xgb_model_16(candidate):
    return XGBClassifier(
        n_estimators=int(candidate["n_estimators"]),
        max_depth=int(candidate["max_depth"]),
        learning_rate=float(candidate["learning_rate"]),
        min_child_weight=float(candidate["min_child_weight"]),
        subsample=float(candidate["subsample"]),
        colsample_bytree=float(candidate["colsample_bytree"]),
        gamma=float(candidate["gamma"]),
        reg_alpha=float(candidate["reg_alpha"]),
        reg_lambda=float(candidate["reg_lambda"]),
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        max_bin=256,
        scale_pos_weight=1.0,
        importance_type="gain",
        random_state=MODEL_RANDOM_SEED_16,
        n_jobs=-1,
        verbosity=0,
    )


def checkpoint_table_id_16(inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_xgb_inner_oof_outer4_inner{inner_fold}_v1"
    )

# ------------------------------------------------------------
# 6. BigQuery checkpoint verification
# ------------------------------------------------------------

def verify_checkpoint_16(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):
    table_id = checkpoint_table_id_16(inner_fold)

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS distinct_id_count,
      COUNT(DISTINCT candidate_id) AS candidate_count,
      COUNT(DISTINCT outer_fold) AS outer_fold_count,
      COUNT(DISTINCT inner_fold) AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(candidate_id, '|', id_row)
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(
        prediction_raw < 0 OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(
        sql,
        location=BQ_LOCATION,
    ).to_dataframe()

    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows * len(candidate_grid_16)
    )
    expected_positive_rows = (
        expected_validation_events * len(candidate_grid_16)
    )
    expected_negative_rows = (
        (expected_validation_rows - expected_validation_events)
        * len(candidate_grid_16)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": expected_validation_rows,
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": expected_total_rows,
        "positive_prediction_rows": expected_positive_rows,
        "negative_prediction_rows": expected_negative_rows,
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": OUTER_FOLD_16,
        "maximum_outer_fold": OUTER_FOLD_16,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failures = []

    for field, expected_value in expected_values.items():
        actual_value = int(row[field])
        if actual_value != expected_value:
            complete = False
            failures.append(
                f"{field}={actual_value}, expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failures),
        "check": check,
        "row": row,
    }

checkpoint_load_config_16 = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField(
            "id_row", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "outer_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "inner_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "candidate_id", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "label_stage23", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "prediction_raw", "FLOAT", mode="REQUIRED"
        ),
    ],
    write_disposition=(
        bigquery.WriteDisposition.WRITE_TRUNCATE
    ),
)

# ------------------------------------------------------------
# 7. Aggregate fit audit
# ------------------------------------------------------------

fit_audit_columns_16 = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "n_estimators",
    "max_depth",
    "learning_rate",
    "min_child_weight",
    "subsample",
    "colsample_bytree",
    "gamma",
    "reg_alpha",
    "reg_lambda",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "fit_seconds",
]

fit_audit_path_16 = os.path.join(
    MODEL_OUTPUT_DIR,
    "16A_xgboost_inner_fit_audit_outer4.csv",
)

if os.path.exists(fit_audit_path_16):
    fit_audit_16 = pd.read_csv(fit_audit_path_16)
else:
    fit_audit_16 = pd.DataFrame(
        columns=fit_audit_columns_16
    )

for column in fit_audit_columns_16:
    if column not in fit_audit_16.columns:
        fit_audit_16[column] = np.nan

fit_audit_16 = fit_audit_16[
    fit_audit_columns_16
].copy()

# ------------------------------------------------------------
# 8. Run five inner folds
# ------------------------------------------------------------

for inner_fold in range(1, 6):
    inner_training_mask = (
        inner_fold_vector_16 != inner_fold
    )
    inner_validation_mask = (
        inner_fold_vector_16 == inner_fold
    )

    training_rows = int(inner_training_mask.sum())
    validation_rows = int(inner_validation_mask.sum())
    training_events = int(
        y_outer_training_16[inner_training_mask].sum()
    )
    validation_events = int(
        y_outer_training_16[inner_validation_mask].sum()
    )

    existing_check = verify_checkpoint_16(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if existing_check["complete"]:
        print(
            f"Outer 4 / inner {inner_fold}: "
            "permanent XGBoost checkpoint already complete; "
            "skipping model fitting."
        )
        continue

    training_hospital_set = set(
        groups_outer_training_16[inner_training_mask]
    )
    validation_hospital_set = set(
        groups_outer_training_16[inner_validation_mask]
    )

    if training_hospital_set & validation_hospital_set:
        raise RuntimeError(
            f"Inner fold {inner_fold}: hospital overlap detected."
        )

    print(f"\nOuter 4 / inner {inner_fold}")
    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_16()

    preprocessing_started = time.time()

    X_inner_training_processed = (
        preprocessor.fit_transform(
            X_outer_training_16.loc[
                inner_training_mask
            ]
        )
    )

    X_inner_validation_processed = (
        preprocessor.transform(
            X_outer_training_16.loc[
                inner_validation_mask
            ]
        )
    )

    preprocessing_seconds = (
        time.time() - preprocessing_started
    )

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "Processed training/validation column counts differ."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_seconds, 2),
    )

    y_inner_training = (
        y_outer_training_16[inner_training_mask]
    )
    y_inner_validation = (
        y_outer_training_16[inner_validation_mask]
    )

    validation_ids = (
        outer_training_meta_16.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_16:
        candidate_id = candidate["candidate_id"]

        print(
            "  Fitting",
            candidate_id,
            "| trees =",
            candidate["n_estimators"],
            "| depth =",
            candidate["max_depth"],
            "| lr =",
            candidate["learning_rate"],
        )

        model = make_xgb_model_16(candidate)

        fit_started = time.time()

        model.fit(
            X_inner_training_processed,
            y_inner_training,
        )

        fit_seconds = time.time() - fit_started

        validation_probabilities = (
            model.predict_proba(
                X_inner_validation_processed
            )[:, 1]
        )

        if np.isnan(validation_probabilities).any():
            raise RuntimeError(
                f"{candidate_id}, inner {inner_fold}: "
                "missing predictions."
            )

        if not np.all(
            (validation_probabilities >= 0)
            & (validation_probabilities <= 1)
        ):
            raise RuntimeError(
                f"{candidate_id}: invalid probabilities."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        OUTER_FOLD_16,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": (
                        y_inner_validation.astype(np.int64)
                    ),
                    "prediction_raw": (
                        validation_probabilities.astype(
                            np.float64
                        )
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": OUTER_FOLD_16,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "n_estimators": candidate[
                    "n_estimators"
                ],
                "max_depth": candidate[
                    "max_depth"
                ],
                "learning_rate": candidate[
                    "learning_rate"
                ],
                "min_child_weight": candidate[
                    "min_child_weight"
                ],
                "subsample": candidate["subsample"],
                "colsample_bytree": candidate[
                    "colsample_bytree"
                ],
                "gamma": candidate["gamma"],
                "reg_alpha": candidate["reg_alpha"],
                "reg_lambda": candidate[
                    "reg_lambda"
                ],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": (
                    validation_events
                ),
                "processed_columns": int(
                    X_inner_training_processed.shape[1]
                ),
                "fit_seconds": float(fit_seconds),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows * len(candidate_grid_16)
    )

    if len(checkpoint_df) != expected_checkpoint_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: invalid checkpoint row count."
        )

    if checkpoint_df.duplicated(
        subset=["id_row", "candidate_id"]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: duplicate candidate-patient rows."
        )

    target_checkpoint_table = (
        checkpoint_table_id_16(inner_fold)
    )

    print(
        "Uploading permanent XGBoost checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_16,
        location=BQ_LOCATION,
    ).result()

    if len(fit_audit_16) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_16["outer_fold"],
                    errors="coerce",
                ) == 1
            )
            & (
                pd.to_numeric(
                    fit_audit_16["inner_fold"],
                    errors="coerce",
                ) == inner_fold
            )
        )

        fit_audit_16 = (
            fit_audit_16.loc[keep_mask].copy()
        )

    fit_audit_16 = pd.concat(
        [
            fit_audit_16,
            pd.DataFrame(current_audit_rows),
        ],
        ignore_index=True,
    )

    fit_audit_16 = (
        fit_audit_16[
            fit_audit_columns_16
        ]
        .sort_values(
            [
                "outer_fold",
                "inner_fold",
                "candidate_id",
            ]
        )
        .reset_index(drop=True)
    )

    fit_audit_16.to_csv(
        fit_audit_path_16,
        index=False,
    )

    completed_check = verify_checkpoint_16(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint verification failed: "
            + completed_check["reason"]
        )

    print(
        f"Outer 4 / inner {inner_fold}: "
        "permanent XGBoost checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 9. Final inner checkpoint summary
# ------------------------------------------------------------

checkpoint_summary_rows_16 = []

for inner_fold in range(1, 6):
    validation_mask = (
        inner_fold_vector_16 == inner_fold
    )
    validation_rows = int(validation_mask.sum())
    validation_events = int(
        y_outer_training_16[
            validation_mask
        ].sum()
    )

    final_check = verify_checkpoint_16(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: final checkpoint audit failed. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_16.append(
        {
            "outer_fold": OUTER_FOLD_16,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(
                row["row_count"]
            ),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(
                row["candidate_count"]
            ),
            "positive_prediction_rows": int(
                row["positive_prediction_rows"]
            ),
            "negative_prediction_rows": int(
                row["negative_prediction_rows"]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check["table_id"],
        }
    )

checkpoint_summary_16 = (
    pd.DataFrame(checkpoint_summary_rows_16)
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_16[
        "distinct_validation_patients"
    ].sum()
) != EXPECTED_SPLIT_16["training_rows"]:
    raise RuntimeError(
        "Total inner validation patients != 46,800."
    )

expected_total_oof_rows_16 = (
    EXPECTED_SPLIT_16["training_rows"]
    * len(candidate_grid_16)
)

if int(
    checkpoint_summary_16[
        "checkpoint_rows"
    ].sum()
) != expected_total_oof_rows_16:
    raise RuntimeError(
        "Total XGBoost OOF prediction rows are incorrect."
    )

checkpoint_summary_path_16 = os.path.join(
    MODEL_OUTPUT_DIR,
    "16A_xgboost_outer4_inner_checkpoint_summary.csv",
)

checkpoint_summary_16.to_csv(
    checkpoint_summary_path_16,
    index=False,
)

# ------------------------------------------------------------
# 10. Pool five inner OOF tables
# ------------------------------------------------------------

checkpoint_tables_16 = [
    checkpoint_table_id_16(inner_fold)
    for inner_fold in range(1, 6)
]

union_parts_16 = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_16
]

SQL_LOAD_POOLED_OOF_16 = (
    "\nUNION ALL\n".join(union_parts_16)
)

print(
    "\nLoading pooled outer-fold-4 XGBoost inner OOF predictions..."
)

query_job_16 = client.query(
    SQL_LOAD_POOLED_OOF_16,
    location=BQ_LOCATION,
)

try:
    pooled_oof_16 = query_job_16.to_dataframe(
        create_bqstorage_client=True
    )
    pooled_load_method_16 = (
        "BigQuery Storage API"
    )
except Exception as fast_path_error_16:
    print(
        "Storage API unavailable; using standard BigQuery download."
    )
    print(
        "Message:",
        type(fast_path_error_16).__name__,
    )
    pooled_oof_16 = query_job_16.to_dataframe(
        create_bqstorage_client=False
    )
    pooled_load_method_16 = (
        "Standard BigQuery API"
    )

pooled_oof_16["id_row"] = (
    pooled_oof_16["id_row"].astype(str)
)
pooled_oof_16["candidate_id"] = (
    pooled_oof_16["candidate_id"].astype(str)
)

for column in [
    "outer_fold",
    "inner_fold",
    "label_stage23",
]:
    pooled_oof_16[column] = pd.to_numeric(
        pooled_oof_16[column],
        errors="raise",
    ).astype(int)

pooled_oof_16["prediction_raw"] = pd.to_numeric(
    pooled_oof_16["prediction_raw"],
    errors="raise",
).astype(float)

if len(pooled_oof_16) != expected_total_oof_rows_16:
    raise RuntimeError(
        "Pooled XGBoost OOF row count is incorrect."
    )

if pooled_oof_16.duplicated(
    subset=["candidate_id", "id_row"]
).any():
    raise RuntimeError(
        "Duplicate candidate-patient row in pooled XGBoost OOF."
    )

if pooled_oof_16["prediction_raw"].isna().any():
    raise RuntimeError("Missing XGBoost OOF prediction.")

if not pooled_oof_16[
    "prediction_raw"
].between(0, 1).all():
    raise RuntimeError(
        "Invalid XGBoost OOF probability."
    )

if set(
    pooled_oof_16["candidate_id"].unique()
) != {
    "XGB01",
    "XGB02",
    "XGB03",
    "XGB04",
    "XGB05",
    "XGB06",
}:
    raise RuntimeError(
        "The six locked XGBoost candidates are not all present."
    )

candidate_patient_counts_16 = (
    pooled_oof_16
    .groupby("candidate_id")["id_row"]
    .nunique()
)

if not (
    candidate_patient_counts_16
    == EXPECTED_SPLIT_16["training_rows"]
).all():
    raise RuntimeError(
        "Each candidate must have 46,800 OOF patients."
    )

candidate_event_counts_16 = (
    pooled_oof_16
    .groupby("candidate_id")["label_stage23"]
    .sum()
)

if not (
    candidate_event_counts_16
    == EXPECTED_SPLIT_16["training_events"]
).all():
    raise RuntimeError(
        "Each candidate must have 2,426 OOF events."
    )

# ------------------------------------------------------------
# 11. Metric helpers
# ------------------------------------------------------------

def probability_metrics_16(
    y_true,
    probabilities,
):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(
            roc_auc_score(y_true, probabilities)
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1],
            )
        ),
        "mean_predicted_risk": float(
            probabilities.mean()
        ),
        "observed_event_rate": float(
            np.mean(y_true)
        ),
    }


def probability_logit_16(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        probabilities / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_16(
    y_true,
    probabilities,
):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        probability_logit_16(probabilities),
        y_true,
    )

    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 12. Candidate pooled inner OOF metrics
# ------------------------------------------------------------

candidate_result_rows_16 = []

for candidate in candidate_grid_16:
    candidate_id = candidate["candidate_id"]

    candidate_oof = (
        pooled_oof_16.loc[
            pooled_oof_16["candidate_id"]
            == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_16(
        candidate_oof[
            "label_stage23"
        ].to_numpy(dtype=int),
        candidate_oof[
            "prediction_raw"
        ].to_numpy(dtype=float),
    )

    fit_part = fit_audit_16.loc[
        fit_audit_16[
            "candidate_id"
        ].astype(str) == candidate_id
    ]

    fit_seconds_total = (
        float(
            pd.to_numeric(
                fit_part["fit_seconds"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_16.append(
        {
            "candidate_id": candidate_id,
            **{
                key: candidate[key]
                for key in candidate
                if key != "candidate_id"
            },
            **metrics,
            "fit_seconds_total": (
                fit_seconds_total
            ),
        }
    )

candidate_results_16 = pd.DataFrame(
    candidate_result_rows_16
)

candidate_results_16 = (
    candidate_results_16
    .sort_values(
        ["auprc", "auroc", "brier"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

candidate_results_16["selection_rank"] = (
    np.arange(
        1,
        len(candidate_results_16) + 1,
    )
)

best_row_16 = candidate_results_16.iloc[0]
selected_candidate_id_16 = str(
    best_row_16["candidate_id"]
)

selected_candidate_16 = next(
    candidate
    for candidate in candidate_grid_16
    if candidate["candidate_id"]
    == selected_candidate_id_16
)

# ------------------------------------------------------------
# 13. Platt calibration from selected inner OOF
# ------------------------------------------------------------

selected_oof_16 = (
    pooled_oof_16.loc[
        pooled_oof_16["candidate_id"]
        == selected_candidate_id_16
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_16 = (
    selected_oof_16[
        "label_stage23"
    ].to_numpy(dtype=int)
)
selected_oof_probability_16 = (
    selected_oof_16[
        "prediction_raw"
    ].to_numpy(dtype=float)
)

platt_calibrator_16 = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_16.fit(
    probability_logit_16(
        selected_oof_probability_16
    ),
    selected_oof_y_16,
)

platt_intercept_16 = float(
    platt_calibrator_16.intercept_[0]
)
platt_slope_16 = float(
    platt_calibrator_16.coef_[0][0]
)

if (
    not np.isfinite(platt_intercept_16)
    or not np.isfinite(platt_slope_16)
    or platt_slope_16 <= 0
):
    raise RuntimeError(
        "Invalid Platt calibration coefficients."
    )

selected_model_16 = pd.DataFrame(
    [
        {
            "outer_fold": OUTER_FOLD_16,
            "selected_candidate": (
                selected_candidate_id_16
            ),
            "selection_metric_primary": (
                "pooled_inner_oof_auprc"
            ),
            "inner_oof_auprc": float(
                best_row_16["auprc"]
            ),
            "inner_oof_auroc": float(
                best_row_16["auroc"]
            ),
            "inner_oof_brier": float(
                best_row_16["brier"]
            ),
            "inner_oof_log_loss": float(
                best_row_16["log_loss"]
            ),
            "inner_oof_mean_predicted_risk": float(
                best_row_16[
                    "mean_predicted_risk"
                ]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_16[
                    "observed_event_rate"
                ]
            ),
            "platt_intercept": (
                platt_intercept_16
            ),
            "platt_slope": platt_slope_16,
            "protocol_sha256": (
                protocol_sha_16
            ),
            **{
                key: selected_candidate_16[key]
                for key in selected_candidate_16
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 14. Save locked selection
# ------------------------------------------------------------

candidate_results_path_16 = os.path.join(
    MODEL_OUTPUT_DIR,
    "16B_xgboost_candidate_results_outer4.csv",
)

selected_model_path_16 = os.path.join(
    MODEL_OUTPUT_DIR,
    "16B_xgboost_selected_model_outer4.csv",
)

selection_json_path_16 = os.path.join(
    MODEL_OUTPUT_DIR,
    "16B_xgboost_selection_calibration_outer4.json",
)

selection_sha_path_16 = os.path.join(
    MODEL_OUTPUT_DIR,
    "16B_xgboost_selection_calibration_outer4_SHA256.txt",
)

candidate_results_16.to_csv(
    candidate_results_path_16,
    index=False,
)
selected_model_16.to_csv(
    selected_model_path_16,
    index=False,
)

selection_configuration_16 = {
    "outer_fold": OUTER_FOLD_16,
    "protocol_sha256": protocol_sha_16,
    "selection_metric_primary": (
        "pooled inner out-of-fold AUPRC"
    ),
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "selected_candidate": (
        selected_candidate_id_16
    ),
    "selected_hyperparameters": {
        key: selected_candidate_16[key]
        for key in selected_candidate_16
        if key != "candidate_id"
    },
    "inner_oof_auprc": float(
        best_row_16["auprc"]
    ),
    "inner_oof_auroc": float(
        best_row_16["auroc"]
    ),
    "inner_oof_brier": float(
        best_row_16["brier"]
    ),
    "platt_intercept": platt_intercept_16,
    "platt_slope": platt_slope_16,
    "inner_checkpoint_tables": (
        checkpoint_tables_16
    ),
    "patient_level_oof_written_to_drive": False,
}

with open(
    selection_json_path_16,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        selection_configuration_16,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    selection_json_path_16,
    "rb",
) as fh:
    selection_sha_16 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    selection_sha_path_16,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(selection_sha_16 + "\n")

# ------------------------------------------------------------
# 15. Fit selected model on all outer training patients
# ------------------------------------------------------------

final_preprocessor_16 = (
    make_preprocessor_16()
)

print(
    "\nFitting selected outer-fold-4 XGBoost model "
    "on all 46,803 training patients..."
)

preprocess_started_16 = time.time()

X_outer_training_processed_16 = (
    final_preprocessor_16.fit_transform(
        X_outer_training_16
    )
)
X_outer_test_processed_16 = (
    final_preprocessor_16.transform(
        X_outer_test_16
    )
)

final_preprocessing_seconds_16 = (
    time.time() - preprocess_started_16
)

final_model_16 = make_xgb_model_16(
    selected_candidate_16
)

final_fit_started_16 = time.time()

final_model_16.fit(
    X_outer_training_processed_16,
    y_outer_training_16,
)

final_fit_seconds_16 = (
    time.time() - final_fit_started_16
)

outer4_raw_probabilities_16 = (
    final_model_16.predict_proba(
        X_outer_test_processed_16
    )[:, 1]
)

raw_clipped_16 = np.clip(
    outer4_raw_probabilities_16,
    1e-6,
    1 - 1e-6,
)
raw_logit_16 = np.log(
    raw_clipped_16
    / (1 - raw_clipped_16)
)

outer4_platt_probabilities_16 = expit(
    platt_intercept_16
    + platt_slope_16 * raw_logit_16
)

for probabilities, name in [
    (outer4_raw_probabilities_16, "raw"),
    (
        outer4_platt_probabilities_16,
        "platt",
    ),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(
            f"{name} test predictions contain missing values."
        )

    if not np.all(
        (probabilities >= 0)
        & (probabilities <= 1)
    ):
        raise RuntimeError(
            f"{name} test predictions contain invalid probabilities."
        )

raw_metrics_16 = probability_metrics_16(
    y_outer_test_16,
    outer4_raw_probabilities_16,
)
platt_metrics_16 = probability_metrics_16(
    y_outer_test_16,
    outer4_platt_probabilities_16,
)

raw_calibration_intercept_16, \
raw_calibration_slope_16 = (
    calibration_intercept_slope_16(
        y_outer_test_16,
        outer4_raw_probabilities_16,
    )
)

platt_calibration_intercept_16, \
platt_calibration_slope_16 = (
    calibration_intercept_slope_16(
        y_outer_test_16,
        outer4_platt_probabilities_16,
    )
)

outer4_test_results_16 = pd.DataFrame(
    [
        {
            "outer_fold": OUTER_FOLD_16,
            "model": "xgboost",
            "probability_type": "raw",
            **raw_metrics_16,
            "calibration_intercept": (
                raw_calibration_intercept_16
            ),
            "calibration_slope": (
                raw_calibration_slope_16
            ),
        },
        {
            "outer_fold": OUTER_FOLD_16,
            "model": "xgboost",
            "probability_type": (
                "platt_calibrated"
            ),
            **platt_metrics_16,
            "calibration_intercept": (
                platt_calibration_intercept_16
            ),
            "calibration_slope": (
                platt_calibration_slope_16
            ),
        },
    ]
)

# ------------------------------------------------------------
# 16. Feature importance
# ------------------------------------------------------------

processed_feature_names_16 = (
    final_preprocessor_16
    .get_feature_names_out()
)

feature_importances_16 = (
    final_model_16.feature_importances_
)

if len(processed_feature_names_16) != len(
    feature_importances_16
):
    raise RuntimeError(
        "Processed feature names and XGBoost "
        "feature importances differ in length."
    )

feature_importance_table_16 = pd.DataFrame(
    {
        "processed_feature": (
            processed_feature_names_16
        ),
        "gain_importance": (
            feature_importances_16
        ),
    }
)

feature_importance_table_16[
    "importance_rank"
] = (
    feature_importance_table_16[
        "gain_importance"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)

feature_importance_table_16 = (
    feature_importance_table_16
    .sort_values(
        [
            "gain_importance",
            "processed_feature",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

nonzero_importance_features_16 = int(
    (
        feature_importance_table_16[
            "gain_importance"
        ] > 0
    ).sum()
)

final_model_summary_16 = pd.DataFrame(
    [
        {
            "outer_fold": OUTER_FOLD_16,
            "selected_candidate": (
                selected_candidate_id_16
            ),
            "training_patients": len(
                X_outer_training_16
            ),
            "training_hospitals": len(
                training_hospitals_16
            ),
            "training_events": int(
                y_outer_training_16.sum()
            ),
            "test_patients": len(
                X_outer_test_16
            ),
            "test_hospitals": len(
                test_hospitals_16
            ),
            "test_events": int(
                y_outer_test_16.sum()
            ),
            "hospital_overlap": len(
                hospital_overlap_16
            ),
            "processed_feature_columns": len(
                processed_feature_names_16
            ),
            "nonzero_importance_features": (
                nonzero_importance_features_16
            ),
            "preprocessing_seconds": float(
                final_preprocessing_seconds_16
            ),
            "fit_seconds": float(
                final_fit_seconds_16
            ),
            "locked_platt_intercept": (
                platt_intercept_16
            ),
            "locked_platt_slope": (
                platt_slope_16
            ),
            "protocol_sha256": protocol_sha_16,
            "selection_sha256": (
                selection_sha_16
            ),
            **{
                key: selected_candidate_16[key]
                for key in selected_candidate_16
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 17. Secure outer-test prediction checkpoint
# ------------------------------------------------------------

outer4_prediction_df_16 = pd.DataFrame(
    {
        "id_row": (
            outer_test_meta_16[
                "id_row"
            ].astype(str)
        ),
        "outer_fold": np.full(
            len(outer_test_meta_16),
            OUTER_FOLD_16,
            dtype=np.int64,
        ),
        "label_stage23": (
            y_outer_test_16.astype(np.int64)
        ),
        "prediction_raw": (
            outer4_raw_probabilities_16.astype(
                np.float64
            )
        ),
        "prediction_platt": (
            outer4_platt_probabilities_16.astype(
                np.float64
            )
        ),
        "model_name": "xgboost",
        "model_version": (
            "core_v1_nested_cv"
        ),
    }
)

if len(outer4_prediction_df_16) != 11688:
    raise RuntimeError(
        "Outer-fold-2 test prediction row count != 11,691."
    )

if outer4_prediction_df_16[
    "id_row"
].duplicated().any():
    raise RuntimeError(
        "Duplicate id_row in outer-fold-4 XGBoost predictions."
    )

if int(
    outer4_prediction_df_16[
        "label_stage23"
    ].sum()
) != 606:
    raise RuntimeError(
        "Outer-fold-2 XGBoost event count != 606."
    )

prediction_table_id_16 = (
    f"{TARGET_DATASET}."
    "model_xgb_outer_predictions_outer4_v1"
)

prediction_load_config_16 = (
    bigquery.LoadJobConfig(
        schema=[
            bigquery.SchemaField(
                "id_row",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "outer_fold",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "label_stage23",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_raw",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_platt",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_name",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_version",
                "STRING",
                mode="REQUIRED",
            ),
        ],
        write_disposition=(
            bigquery.WriteDisposition.WRITE_TRUNCATE
        ),
    )
)

print(
    "\nUploading secure outer-fold-4 "
    "XGBoost prediction checkpoint:"
)
print(prediction_table_id_16)

client.load_table_from_dataframe(
    outer4_prediction_df_16,
    prediction_table_id_16,
    job_config=prediction_load_config_16,
    location=BQ_LOCATION,
).result()

SQL_VERIFY_PREDICTIONS_16 = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL)
    AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL)
    AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0
    OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0
    OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw)
    AS minimum_raw_probability,
  MAX(prediction_raw)
    AS maximum_raw_probability,
  MIN(prediction_platt)
    AS minimum_platt_probability,
  MAX(prediction_platt)
    AS maximum_platt_probability
FROM `{prediction_table_id_16}`;
"""

prediction_verification_16 = (
    client.query(
        SQL_VERIFY_PREDICTIONS_16,
        location=BQ_LOCATION,
    )
    .to_dataframe()
)

verification_row_16 = (
    prediction_verification_16.iloc[0]
)

expected_prediction_values_16 = {
    "prediction_rows": 11688,
    "distinct_rows": 11688,
    "outer_folds": 1,
    "minimum_outer_fold": OUTER_FOLD_16,
    "maximum_outer_fold": OUTER_FOLD_16,
    "events": 606,
    "nonevents": 11082,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in (
    expected_prediction_values_16.items()
):
    actual_value = int(
        verification_row_16[field]
    )
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: found={actual_value}, "
            f"expected={expected_value}"
        )

# ------------------------------------------------------------
# 18. Save aggregate outputs
# ------------------------------------------------------------

test_results_path_16 = os.path.join(
    MODEL_OUTPUT_DIR,
    "16C_xgboost_outer4_test_results.csv",
)

model_summary_path_16 = os.path.join(
    MODEL_OUTPUT_DIR,
    "16C_xgboost_final_model_outer4.csv",
)

importance_path_16 = os.path.join(
    MODEL_OUTPUT_DIR,
    "16C_xgboost_gain_importance_outer4.csv",
)

evaluation_json_path_16 = os.path.join(
    MODEL_OUTPUT_DIR,
    "16C_xgboost_final_evaluation_outer4.json",
)

evaluation_sha_path_16 = os.path.join(
    MODEL_OUTPUT_DIR,
    "16C_xgboost_final_evaluation_outer4_SHA256.txt",
)

outer4_test_results_16.to_csv(
    test_results_path_16,
    index=False,
)
final_model_summary_16.to_csv(
    model_summary_path_16,
    index=False,
)
feature_importance_table_16.to_csv(
    importance_path_16,
    index=False,
)

evaluation_configuration_16 = {
    "outer_fold": OUTER_FOLD_16,
    "model_family": "xgboost",
    "protocol_sha256": protocol_sha_16,
    "selection_sha256": selection_sha_16,
    "selected_candidate": (
        selected_candidate_id_16
    ),
    "selected_hyperparameters": {
        key: selected_candidate_16[key]
        for key in selected_candidate_16
        if key != "candidate_id"
    },
    "training_patients": 46803,
    "training_hospitals": 159,
    "test_patients": 11688,
    "test_hospitals": 39,
    "hospital_overlap": 0,
    "locked_platt_intercept": (
        platt_intercept_16
    ),
    "locked_platt_slope": platt_slope_16,
    "processed_feature_columns": int(
        len(processed_feature_names_16)
    ),
    "secure_prediction_table": (
        prediction_table_id_16
    ),
    "patient_level_prediction_written_to_drive": False,
}

with open(
    evaluation_json_path_16,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        evaluation_configuration_16,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    evaluation_json_path_16,
    "rb",
) as fh:
    evaluation_sha_16 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    evaluation_sha_path_16,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(evaluation_sha_16 + "\n")

# ------------------------------------------------------------
# 19. Display results
# ------------------------------------------------------------

pooled_integrity_16 = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_16),
            pooled_oof_16[
                "id_row"
            ].nunique(),
            pooled_oof_16[
                "candidate_id"
            ].nunique(),
            pooled_oof_16[
                "inner_fold"
            ].nunique(),
            EXPECTED_SPLIT_16[
                "training_events"
            ],
            (
                EXPECTED_SPLIT_16[
                    "training_rows"
                ]
                - EXPECTED_SPLIT_16[
                    "training_events"
                ]
            ),
            int(
                pooled_oof_16.duplicated(
                    subset=[
                        "candidate_id",
                        "id_row",
                    ]
                ).sum()
            ),
            int(
                pooled_oof_16[
                    "prediction_raw"
                ].isna().sum()
            ),
            int(
                (
                    ~pooled_oof_16[
                        "prediction_raw"
                    ].between(0, 1)
                ).sum()
            ),
            pooled_load_method_16,
        ],
    }
)

print(
    "\n16 XGBOOST OUTER-FOLD-4 INNER CHECKPOINT SUMMARY"
)
display(checkpoint_summary_16)

print(
    "\n16 XGBOOST OUTER-FOLD-4 POOLED OOF INTEGRITY"
)
display(pooled_integrity_16)

print(
    "\n16 XGBOOST OUTER-FOLD-4 CANDIDATE RESULTS"
)
display(candidate_results_16)

print(
    "\n16 XGBOOST OUTER-FOLD-4 SELECTED MODEL"
)
display(selected_model_16)

print(
    "\n16 XGBOOST OUTER-FOLD-4 FINAL MODEL SUMMARY"
)
display(final_model_summary_16)

print(
    "\n16 XGBOOST OUTER-FOLD-4 TEST RESULTS"
)
display(outer4_test_results_16)

print(
    "\n16 XGBOOST OUTER-FOLD-4 BIGQUERY VERIFICATION"
)
display(prediction_verification_16)

print(
    "\n16 XGBOOST OUTER-FOLD-4 TOP 20 GAIN IMPORTANCE FEATURES"
)
display(feature_importance_table_16.head(20))

print("\nXGBoost protocol SHA-256:")
print(protocol_sha_16)

print("\nSelection SHA-256:")
print(selection_sha_16)

print("\nEvaluation SHA-256:")
print(evaluation_sha_16)

print("\nSaved aggregate outputs:")
print(protocol_path_16)
print(protocol_sha_path_16)
print(fit_audit_path_16)
print(checkpoint_summary_path_16)
print(candidate_results_path_16)
print(selected_model_path_16)
print(selection_json_path_16)
print(selection_sha_path_16)
print(test_results_path_16)
print(model_summary_path_16)
print(importance_path_16)
print(evaluation_json_path_16)
print(evaluation_sha_path_16)

print(
    "\n16 PASS: XGBoost outer-fold-4 nested modelling "
    "and locked test evaluation are complete."
)

print(
    "No class weighting, SMOTE, or early stopping was used."
)

print(
    "All patient-level OOF and outer-test predictions "
    "were stored only in BigQuery."
)

print(
    "No patient-level prediction file was written to Google Drive."
)

_ = gc.collect()

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

from xgboost import XGBClassifier

from IPython.display import display

print("STARTING XGBOOST OUTER FOLD 5 — CODE VERSION 17")

# ============================================================
# 14 — XGBOOST OUTER FOLD 2 COMPLETE NESTED MODELLING
#
# Design:
# - Same locked hospital-disjoint outer folds.
# - Same locked hospital-disjoint inner folds.
# - Core feature set only.
# - No SMOTE.
# - No class weighting / scale_pos_weight = 1.
# - No early stopping.
# - Fixed candidate grid locked before outer-test evaluation.
# - Selection: pooled inner-OOF AUPRC descending,
#              AUROC descending, Brier ascending.
# - Platt calibration learned only from selected candidate's
#   pooled inner-OOF predictions.
# - Patient-level predictions stored only in BigQuery.
# ============================================================

OUTER_FOLD_17 = 5
MODEL_RANDOM_SEED_17 = 20260721

EXPECTED_SPLIT_17 = {
    "training_rows": 46803,
    "test_rows": 11688,
    "training_hospitals": 159,
    "test_hospitals": 39,
    "training_events": 2426,
    "test_events": 606,
}

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_objects_17 = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_17 = [
    name for name in required_objects_17
    if name not in globals()
]

if missing_objects_17:
    raise RuntimeError(
        "Missing runtime objects: "
        + ", ".join(missing_objects_17)
        + ". Run 07A and 07B first."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"Expected 58,491 cohort rows; found {len(core_df_07B)}."
    )

if len(predictor_columns_07B) != 159:
    raise RuntimeError("Expected 159 core predictors.")

if len(numeric_columns_07B) != 156:
    raise RuntimeError("Expected 156 numeric predictors.")

if len(categorical_columns_07B) != 3:
    raise RuntimeError("Expected 3 categorical predictors.")

# ------------------------------------------------------------
# 2. Lock the XGBoost protocol BEFORE test evaluation
# ------------------------------------------------------------

candidate_grid_17 = [
    {
        "candidate_id": "XGB01",
        "n_estimators": 250,
        "max_depth": 2,
        "learning_rate": 0.03,
        "min_child_weight": 5.0,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
    },
    {
        "candidate_id": "XGB02",
        "n_estimators": 350,
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 5.0,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
    },
    {
        "candidate_id": "XGB03",
        "n_estimators": 450,
        "max_depth": 3,
        "learning_rate": 0.02,
        "min_child_weight": 10.0,
        "subsample": 0.85,
        "colsample_bytree": 0.85,
        "gamma": 0.0,
        "reg_alpha": 0.10,
        "reg_lambda": 10.0,
    },
    {
        "candidate_id": "XGB04",
        "n_estimators": 350,
        "max_depth": 4,
        "learning_rate": 0.03,
        "min_child_weight": 10.0,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "gamma": 0.10,
        "reg_alpha": 0.10,
        "reg_lambda": 10.0,
    },
    {
        "candidate_id": "XGB05",
        "n_estimators": 450,
        "max_depth": 2,
        "learning_rate": 0.02,
        "min_child_weight": 10.0,
        "subsample": 0.90,
        "colsample_bytree": 0.90,
        "gamma": 0.0,
        "reg_alpha": 0.50,
        "reg_lambda": 10.0,
    },
    {
        "candidate_id": "XGB06",
        "n_estimators": 450,
        "max_depth": 4,
        "learning_rate": 0.02,
        "min_child_weight": 15.0,
        "subsample": 0.90,
        "colsample_bytree": 0.80,
        "gamma": 0.20,
        "reg_alpha": 0.50,
        "reg_lambda": 15.0,
    },
]

xgb_protocol_17 = {
    "protocol_name": "xgboost_core_nested_hospital_cv_v1",
    "model_family": "XGBoost",
    "feature_set": "core_159",
    "outer_cv": "locked 5-fold hospital-disjoint outer folds",
    "inner_cv": "locked 5-fold hospital-disjoint inner folds",
    "primary_selection_metric": "pooled inner OOF AUPRC descending",
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "class_weighting": False,
    "scale_pos_weight": 1.0,
    "smote": False,
    "early_stopping": False,
    "tree_method": "hist",
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": MODEL_RANDOM_SEED_17,
    "candidate_grid": candidate_grid_17,
    "calibration": (
        "Platt calibration fit only on selected candidate "
        "pooled inner-OOF logits"
    ),
    "outer_test_use": (
        "diagnostic evaluation only; never used for tuning"
    ),
}

protocol_path_17 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13A_locked_xgboost_model_protocol_v1.json",
)

protocol_sha_path_17 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13A_locked_xgboost_model_protocol_v1_SHA256.txt",
)

protocol_text_17 = json.dumps(
    xgb_protocol_17,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
)

protocol_sha_17 = hashlib.sha256(
    protocol_text_17.encode("utf-8")
).hexdigest()

if os.path.exists(protocol_path_17):
    with open(protocol_path_17, "r", encoding="utf-8") as fh:
        existing_protocol_text_17 = fh.read()
    existing_protocol_sha_17 = hashlib.sha256(
        existing_protocol_text_17.encode("utf-8")
    ).hexdigest()

    if existing_protocol_sha_17 != protocol_sha_17:
        raise RuntimeError(
            "An existing XGBoost protocol file differs from "
            "the currently locked protocol. Stop and audit."
        )
else:
    with open(protocol_path_17, "w", encoding="utf-8") as fh:
        fh.write(protocol_text_17)

with open(protocol_sha_path_17, "w", encoding="utf-8") as fh:
    fh.write(protocol_sha_17 + "\n")

print("Locked XGBoost protocol SHA-256:")
print(protocol_sha_17)

# ------------------------------------------------------------
# 3. Load locked inner hospital map
# ------------------------------------------------------------

inner_mapping_path_17 = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_17):
    raise FileNotFoundError(
        "Locked inner-fold map not found: "
        + inner_mapping_path_17
    )

inner_mapping_all_17 = pd.read_csv(
    inner_mapping_path_17,
    dtype={"group_hospital": str},
)

inner_mapping_part_17 = (
    inner_mapping_all_17.loc[
        inner_mapping_all_17["outer_fold"].astype(int)
        == OUTER_FOLD_17,
        ["group_hospital", "inner_fold"],
    ]
    .copy()
)

inner_mapping_part_17["group_hospital"] = (
    inner_mapping_part_17["group_hospital"].astype(str)
)
inner_mapping_part_17["inner_fold"] = (
    inner_mapping_part_17["inner_fold"].astype(int)
)

if len(inner_mapping_part_17) != 159:
    raise RuntimeError(
        "Expected 159 outer-fold-5 training hospitals "
        "in the locked inner map."
    )

if inner_mapping_part_17["group_hospital"].duplicated().any():
    raise RuntimeError("Duplicate hospital in locked inner map.")

hospital_to_inner_fold_17 = dict(
    zip(
        inner_mapping_part_17["group_hospital"],
        inner_mapping_part_17["inner_fold"],
    )
)

# ------------------------------------------------------------
# 4. Prepare outer fold 1 matrices
# ------------------------------------------------------------

X_all_17 = core_df_07B[predictor_columns_07B].copy()

for column in numeric_columns_07B:
    X_all_17[column] = pd.to_numeric(
        X_all_17[column],
        errors="coerce",
    ).astype("float64")

for column in categorical_columns_07B:
    category_series = X_all_17[column].astype("object")
    X_all_17[column] = category_series.where(
        pd.notna(category_series),
        np.nan,
    )

outer_fold_vector_17 = (
    core_df_07B["outer_fold"].astype(int).to_numpy()
)

outer_training_mask_17 = (
    outer_fold_vector_17 != OUTER_FOLD_17
)
outer_test_mask_17 = (
    outer_fold_vector_17 == OUTER_FOLD_17
)

X_outer_training_17 = (
    X_all_17.loc[outer_training_mask_17]
    .reset_index(drop=True)
)

X_outer_test_17 = (
    X_all_17.loc[outer_test_mask_17]
    .reset_index(drop=True)
)

outer_training_meta_17 = (
    core_df_07B.loc[
        outer_training_mask_17,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_17 = (
    core_df_07B.loc[
        outer_test_mask_17,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [
    outer_training_meta_17,
    outer_test_meta_17,
]:
    dataframe["id_row"] = dataframe["id_row"].astype(str)
    dataframe["group_hospital"] = (
        dataframe["group_hospital"].astype(str)
    )
    dataframe["label_stage23"] = (
        dataframe["label_stage23"].astype(int)
    )

y_outer_training_17 = (
    outer_training_meta_17["label_stage23"]
    .to_numpy(dtype=np.int8)
)
y_outer_test_17 = (
    outer_test_meta_17["label_stage23"]
    .to_numpy(dtype=np.int8)
)

groups_outer_training_17 = (
    outer_training_meta_17["group_hospital"]
    .to_numpy(dtype=str)
)

training_hospitals_17 = set(
    outer_training_meta_17["group_hospital"]
)
test_hospitals_17 = set(
    outer_test_meta_17["group_hospital"]
)
hospital_overlap_17 = (
    training_hospitals_17 & test_hospitals_17
)

if hospital_overlap_17:
    raise RuntimeError(
        "Outer training/test hospital overlap detected."
    )

actual_split_17 = {
    "training_rows": len(X_outer_training_17),
    "test_rows": len(X_outer_test_17),
    "training_hospitals": len(training_hospitals_17),
    "test_hospitals": len(test_hospitals_17),
    "training_events": int(y_outer_training_17.sum()),
    "test_events": int(y_outer_test_17.sum()),
}

for metric, expected_value in EXPECTED_SPLIT_17.items():
    actual_value = actual_split_17[metric]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: found={actual_value}, "
            f"expected={expected_value}"
        )

inner_fold_vector_17 = np.array(
    [
        hospital_to_inner_fold_17.get(hospital, -1)
        for hospital in groups_outer_training_17
    ],
    dtype=int,
)

if (inner_fold_vector_17 == -1).any():
    raise RuntimeError(
        "Some outer-training hospitals have no inner-fold assignment."
    )

if set(np.unique(inner_fold_vector_17)) != {1, 2, 3, 4, 5}:
    raise RuntimeError("Inner-fold values are not exactly 1–5.")

# ------------------------------------------------------------
# 5. Preprocessor and model constructors
# ------------------------------------------------------------

def make_preprocessor_17():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns_07B,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns_07B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_xgb_model_17(candidate):
    return XGBClassifier(
        n_estimators=int(candidate["n_estimators"]),
        max_depth=int(candidate["max_depth"]),
        learning_rate=float(candidate["learning_rate"]),
        min_child_weight=float(candidate["min_child_weight"]),
        subsample=float(candidate["subsample"]),
        colsample_bytree=float(candidate["colsample_bytree"]),
        gamma=float(candidate["gamma"]),
        reg_alpha=float(candidate["reg_alpha"]),
        reg_lambda=float(candidate["reg_lambda"]),
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        max_bin=256,
        scale_pos_weight=1.0,
        importance_type="gain",
        random_state=MODEL_RANDOM_SEED_17,
        n_jobs=-1,
        verbosity=0,
    )


def checkpoint_table_id_17(inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_xgb_inner_oof_outer5_inner{inner_fold}_v1"
    )

# ------------------------------------------------------------
# 6. BigQuery checkpoint verification
# ------------------------------------------------------------

def verify_checkpoint_17(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):
    table_id = checkpoint_table_id_17(inner_fold)

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS distinct_id_count,
      COUNT(DISTINCT candidate_id) AS candidate_count,
      COUNT(DISTINCT outer_fold) AS outer_fold_count,
      COUNT(DISTINCT inner_fold) AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(candidate_id, '|', id_row)
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(
        prediction_raw < 0 OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(
        sql,
        location=BQ_LOCATION,
    ).to_dataframe()

    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows * len(candidate_grid_17)
    )
    expected_positive_rows = (
        expected_validation_events * len(candidate_grid_17)
    )
    expected_negative_rows = (
        (expected_validation_rows - expected_validation_events)
        * len(candidate_grid_17)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": expected_validation_rows,
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": expected_total_rows,
        "positive_prediction_rows": expected_positive_rows,
        "negative_prediction_rows": expected_negative_rows,
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": OUTER_FOLD_17,
        "maximum_outer_fold": OUTER_FOLD_17,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failures = []

    for field, expected_value in expected_values.items():
        actual_value = int(row[field])
        if actual_value != expected_value:
            complete = False
            failures.append(
                f"{field}={actual_value}, expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failures),
        "check": check,
        "row": row,
    }

checkpoint_load_config_17 = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField(
            "id_row", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "outer_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "inner_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "candidate_id", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "label_stage23", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "prediction_raw", "FLOAT", mode="REQUIRED"
        ),
    ],
    write_disposition=(
        bigquery.WriteDisposition.WRITE_TRUNCATE
    ),
)

# ------------------------------------------------------------
# 7. Aggregate fit audit
# ------------------------------------------------------------

fit_audit_columns_17 = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "n_estimators",
    "max_depth",
    "learning_rate",
    "min_child_weight",
    "subsample",
    "colsample_bytree",
    "gamma",
    "reg_alpha",
    "reg_lambda",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "fit_seconds",
]

fit_audit_path_17 = os.path.join(
    MODEL_OUTPUT_DIR,
    "17A_xgboost_inner_fit_audit_outer5.csv",
)

if os.path.exists(fit_audit_path_17):
    fit_audit_17 = pd.read_csv(fit_audit_path_17)
else:
    fit_audit_17 = pd.DataFrame(
        columns=fit_audit_columns_17
    )

for column in fit_audit_columns_17:
    if column not in fit_audit_17.columns:
        fit_audit_17[column] = np.nan

fit_audit_17 = fit_audit_17[
    fit_audit_columns_17
].copy()

# ------------------------------------------------------------
# 8. Run five inner folds
# ------------------------------------------------------------

for inner_fold in range(1, 6):
    inner_training_mask = (
        inner_fold_vector_17 != inner_fold
    )
    inner_validation_mask = (
        inner_fold_vector_17 == inner_fold
    )

    training_rows = int(inner_training_mask.sum())
    validation_rows = int(inner_validation_mask.sum())
    training_events = int(
        y_outer_training_17[inner_training_mask].sum()
    )
    validation_events = int(
        y_outer_training_17[inner_validation_mask].sum()
    )

    existing_check = verify_checkpoint_17(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if existing_check["complete"]:
        print(
            f"Outer 5 / inner {inner_fold}: "
            "permanent XGBoost checkpoint already complete; "
            "skipping model fitting."
        )
        continue

    training_hospital_set = set(
        groups_outer_training_17[inner_training_mask]
    )
    validation_hospital_set = set(
        groups_outer_training_17[inner_validation_mask]
    )

    if training_hospital_set & validation_hospital_set:
        raise RuntimeError(
            f"Inner fold {inner_fold}: hospital overlap detected."
        )

    print(f"\nOuter 5 / inner {inner_fold}")
    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_17()

    preprocessing_started = time.time()

    X_inner_training_processed = (
        preprocessor.fit_transform(
            X_outer_training_17.loc[
                inner_training_mask
            ]
        )
    )

    X_inner_validation_processed = (
        preprocessor.transform(
            X_outer_training_17.loc[
                inner_validation_mask
            ]
        )
    )

    preprocessing_seconds = (
        time.time() - preprocessing_started
    )

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "Processed training/validation column counts differ."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_seconds, 2),
    )

    y_inner_training = (
        y_outer_training_17[inner_training_mask]
    )
    y_inner_validation = (
        y_outer_training_17[inner_validation_mask]
    )

    validation_ids = (
        outer_training_meta_17.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_17:
        candidate_id = candidate["candidate_id"]

        print(
            "  Fitting",
            candidate_id,
            "| trees =",
            candidate["n_estimators"],
            "| depth =",
            candidate["max_depth"],
            "| lr =",
            candidate["learning_rate"],
        )

        model = make_xgb_model_17(candidate)

        fit_started = time.time()

        model.fit(
            X_inner_training_processed,
            y_inner_training,
        )

        fit_seconds = time.time() - fit_started

        validation_probabilities = (
            model.predict_proba(
                X_inner_validation_processed
            )[:, 1]
        )

        if np.isnan(validation_probabilities).any():
            raise RuntimeError(
                f"{candidate_id}, inner {inner_fold}: "
                "missing predictions."
            )

        if not np.all(
            (validation_probabilities >= 0)
            & (validation_probabilities <= 1)
        ):
            raise RuntimeError(
                f"{candidate_id}: invalid probabilities."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        OUTER_FOLD_17,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": (
                        y_inner_validation.astype(np.int64)
                    ),
                    "prediction_raw": (
                        validation_probabilities.astype(
                            np.float64
                        )
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": OUTER_FOLD_17,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "n_estimators": candidate[
                    "n_estimators"
                ],
                "max_depth": candidate[
                    "max_depth"
                ],
                "learning_rate": candidate[
                    "learning_rate"
                ],
                "min_child_weight": candidate[
                    "min_child_weight"
                ],
                "subsample": candidate["subsample"],
                "colsample_bytree": candidate[
                    "colsample_bytree"
                ],
                "gamma": candidate["gamma"],
                "reg_alpha": candidate["reg_alpha"],
                "reg_lambda": candidate[
                    "reg_lambda"
                ],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": (
                    validation_events
                ),
                "processed_columns": int(
                    X_inner_training_processed.shape[1]
                ),
                "fit_seconds": float(fit_seconds),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows * len(candidate_grid_17)
    )

    if len(checkpoint_df) != expected_checkpoint_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: invalid checkpoint row count."
        )

    if checkpoint_df.duplicated(
        subset=["id_row", "candidate_id"]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: duplicate candidate-patient rows."
        )

    target_checkpoint_table = (
        checkpoint_table_id_17(inner_fold)
    )

    print(
        "Uploading permanent XGBoost checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_17,
        location=BQ_LOCATION,
    ).result()

    if len(fit_audit_17) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_17["outer_fold"],
                    errors="coerce",
                ) == 1
            )
            & (
                pd.to_numeric(
                    fit_audit_17["inner_fold"],
                    errors="coerce",
                ) == inner_fold
            )
        )

        fit_audit_17 = (
            fit_audit_17.loc[keep_mask].copy()
        )

    fit_audit_17 = pd.concat(
        [
            fit_audit_17,
            pd.DataFrame(current_audit_rows),
        ],
        ignore_index=True,
    )

    fit_audit_17 = (
        fit_audit_17[
            fit_audit_columns_17
        ]
        .sort_values(
            [
                "outer_fold",
                "inner_fold",
                "candidate_id",
            ]
        )
        .reset_index(drop=True)
    )

    fit_audit_17.to_csv(
        fit_audit_path_17,
        index=False,
    )

    completed_check = verify_checkpoint_17(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint verification failed: "
            + completed_check["reason"]
        )

    print(
        f"Outer 5 / inner {inner_fold}: "
        "permanent XGBoost checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 9. Final inner checkpoint summary
# ------------------------------------------------------------

checkpoint_summary_rows_17 = []

for inner_fold in range(1, 6):
    validation_mask = (
        inner_fold_vector_17 == inner_fold
    )
    validation_rows = int(validation_mask.sum())
    validation_events = int(
        y_outer_training_17[
            validation_mask
        ].sum()
    )

    final_check = verify_checkpoint_17(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: final checkpoint audit failed. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_17.append(
        {
            "outer_fold": OUTER_FOLD_17,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(
                row["row_count"]
            ),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(
                row["candidate_count"]
            ),
            "positive_prediction_rows": int(
                row["positive_prediction_rows"]
            ),
            "negative_prediction_rows": int(
                row["negative_prediction_rows"]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check["table_id"],
        }
    )

checkpoint_summary_17 = (
    pd.DataFrame(checkpoint_summary_rows_17)
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_17[
        "distinct_validation_patients"
    ].sum()
) != EXPECTED_SPLIT_17["training_rows"]:
    raise RuntimeError(
        "Total inner validation patients != 46,800."
    )

expected_total_oof_rows_17 = (
    EXPECTED_SPLIT_17["training_rows"]
    * len(candidate_grid_17)
)

if int(
    checkpoint_summary_17[
        "checkpoint_rows"
    ].sum()
) != expected_total_oof_rows_17:
    raise RuntimeError(
        "Total XGBoost OOF prediction rows are incorrect."
    )

checkpoint_summary_path_17 = os.path.join(
    MODEL_OUTPUT_DIR,
    "17A_xgboost_outer5_inner_checkpoint_summary.csv",
)

checkpoint_summary_17.to_csv(
    checkpoint_summary_path_17,
    index=False,
)

# ------------------------------------------------------------
# 10. Pool five inner OOF tables
# ------------------------------------------------------------

checkpoint_tables_17 = [
    checkpoint_table_id_17(inner_fold)
    for inner_fold in range(1, 6)
]

union_parts_17 = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_17
]

SQL_LOAD_POOLED_OOF_17 = (
    "\nUNION ALL\n".join(union_parts_17)
)

print(
    "\nLoading pooled outer-fold-5 XGBoost inner OOF predictions..."
)

query_job_17 = client.query(
    SQL_LOAD_POOLED_OOF_17,
    location=BQ_LOCATION,
)

try:
    pooled_oof_17 = query_job_17.to_dataframe(
        create_bqstorage_client=True
    )
    pooled_load_method_17 = (
        "BigQuery Storage API"
    )
except Exception as fast_path_error_17:
    print(
        "Storage API unavailable; using standard BigQuery download."
    )
    print(
        "Message:",
        type(fast_path_error_17).__name__,
    )
    pooled_oof_17 = query_job_17.to_dataframe(
        create_bqstorage_client=False
    )
    pooled_load_method_17 = (
        "Standard BigQuery API"
    )

pooled_oof_17["id_row"] = (
    pooled_oof_17["id_row"].astype(str)
)
pooled_oof_17["candidate_id"] = (
    pooled_oof_17["candidate_id"].astype(str)
)

for column in [
    "outer_fold",
    "inner_fold",
    "label_stage23",
]:
    pooled_oof_17[column] = pd.to_numeric(
        pooled_oof_17[column],
        errors="raise",
    ).astype(int)

pooled_oof_17["prediction_raw"] = pd.to_numeric(
    pooled_oof_17["prediction_raw"],
    errors="raise",
).astype(float)

if len(pooled_oof_17) != expected_total_oof_rows_17:
    raise RuntimeError(
        "Pooled XGBoost OOF row count is incorrect."
    )

if pooled_oof_17.duplicated(
    subset=["candidate_id", "id_row"]
).any():
    raise RuntimeError(
        "Duplicate candidate-patient row in pooled XGBoost OOF."
    )

if pooled_oof_17["prediction_raw"].isna().any():
    raise RuntimeError("Missing XGBoost OOF prediction.")

if not pooled_oof_17[
    "prediction_raw"
].between(0, 1).all():
    raise RuntimeError(
        "Invalid XGBoost OOF probability."
    )

if set(
    pooled_oof_17["candidate_id"].unique()
) != {
    "XGB01",
    "XGB02",
    "XGB03",
    "XGB04",
    "XGB05",
    "XGB06",
}:
    raise RuntimeError(
        "The six locked XGBoost candidates are not all present."
    )

candidate_patient_counts_17 = (
    pooled_oof_17
    .groupby("candidate_id")["id_row"]
    .nunique()
)

if not (
    candidate_patient_counts_17
    == EXPECTED_SPLIT_17["training_rows"]
).all():
    raise RuntimeError(
        "Each candidate must have 46,800 OOF patients."
    )

candidate_event_counts_17 = (
    pooled_oof_17
    .groupby("candidate_id")["label_stage23"]
    .sum()
)

if not (
    candidate_event_counts_17
    == EXPECTED_SPLIT_17["training_events"]
).all():
    raise RuntimeError(
        "Each candidate must have 2,426 OOF events."
    )

# ------------------------------------------------------------
# 11. Metric helpers
# ------------------------------------------------------------

def probability_metrics_17(
    y_true,
    probabilities,
):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(
            roc_auc_score(y_true, probabilities)
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1],
            )
        ),
        "mean_predicted_risk": float(
            probabilities.mean()
        ),
        "observed_event_rate": float(
            np.mean(y_true)
        ),
    }


def probability_logit_17(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        probabilities / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_17(
    y_true,
    probabilities,
):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        probability_logit_17(probabilities),
        y_true,
    )

    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 12. Candidate pooled inner OOF metrics
# ------------------------------------------------------------

candidate_result_rows_17 = []

for candidate in candidate_grid_17:
    candidate_id = candidate["candidate_id"]

    candidate_oof = (
        pooled_oof_17.loc[
            pooled_oof_17["candidate_id"]
            == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_17(
        candidate_oof[
            "label_stage23"
        ].to_numpy(dtype=int),
        candidate_oof[
            "prediction_raw"
        ].to_numpy(dtype=float),
    )

    fit_part = fit_audit_17.loc[
        fit_audit_17[
            "candidate_id"
        ].astype(str) == candidate_id
    ]

    fit_seconds_total = (
        float(
            pd.to_numeric(
                fit_part["fit_seconds"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_17.append(
        {
            "candidate_id": candidate_id,
            **{
                key: candidate[key]
                for key in candidate
                if key != "candidate_id"
            },
            **metrics,
            "fit_seconds_total": (
                fit_seconds_total
            ),
        }
    )

candidate_results_17 = pd.DataFrame(
    candidate_result_rows_17
)

candidate_results_17 = (
    candidate_results_17
    .sort_values(
        ["auprc", "auroc", "brier"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

candidate_results_17["selection_rank"] = (
    np.arange(
        1,
        len(candidate_results_17) + 1,
    )
)

best_row_17 = candidate_results_17.iloc[0]
selected_candidate_id_17 = str(
    best_row_17["candidate_id"]
)

selected_candidate_17 = next(
    candidate
    for candidate in candidate_grid_17
    if candidate["candidate_id"]
    == selected_candidate_id_17
)

# ------------------------------------------------------------
# 13. Platt calibration from selected inner OOF
# ------------------------------------------------------------

selected_oof_17 = (
    pooled_oof_17.loc[
        pooled_oof_17["candidate_id"]
        == selected_candidate_id_17
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_17 = (
    selected_oof_17[
        "label_stage23"
    ].to_numpy(dtype=int)
)
selected_oof_probability_17 = (
    selected_oof_17[
        "prediction_raw"
    ].to_numpy(dtype=float)
)

platt_calibrator_17 = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_17.fit(
    probability_logit_17(
        selected_oof_probability_17
    ),
    selected_oof_y_17,
)

platt_intercept_17 = float(
    platt_calibrator_17.intercept_[0]
)
platt_slope_17 = float(
    platt_calibrator_17.coef_[0][0]
)

if (
    not np.isfinite(platt_intercept_17)
    or not np.isfinite(platt_slope_17)
    or platt_slope_17 <= 0
):
    raise RuntimeError(
        "Invalid Platt calibration coefficients."
    )

selected_model_17 = pd.DataFrame(
    [
        {
            "outer_fold": OUTER_FOLD_17,
            "selected_candidate": (
                selected_candidate_id_17
            ),
            "selection_metric_primary": (
                "pooled_inner_oof_auprc"
            ),
            "inner_oof_auprc": float(
                best_row_17["auprc"]
            ),
            "inner_oof_auroc": float(
                best_row_17["auroc"]
            ),
            "inner_oof_brier": float(
                best_row_17["brier"]
            ),
            "inner_oof_log_loss": float(
                best_row_17["log_loss"]
            ),
            "inner_oof_mean_predicted_risk": float(
                best_row_17[
                    "mean_predicted_risk"
                ]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_17[
                    "observed_event_rate"
                ]
            ),
            "platt_intercept": (
                platt_intercept_17
            ),
            "platt_slope": platt_slope_17,
            "protocol_sha256": (
                protocol_sha_17
            ),
            **{
                key: selected_candidate_17[key]
                for key in selected_candidate_17
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 14. Save locked selection
# ------------------------------------------------------------

candidate_results_path_17 = os.path.join(
    MODEL_OUTPUT_DIR,
    "17B_xgboost_candidate_results_outer5.csv",
)

selected_model_path_17 = os.path.join(
    MODEL_OUTPUT_DIR,
    "17B_xgboost_selected_model_outer5.csv",
)

selection_json_path_17 = os.path.join(
    MODEL_OUTPUT_DIR,
    "17B_xgboost_selection_calibration_outer5.json",
)

selection_sha_path_17 = os.path.join(
    MODEL_OUTPUT_DIR,
    "17B_xgboost_selection_calibration_outer5_SHA256.txt",
)

candidate_results_17.to_csv(
    candidate_results_path_17,
    index=False,
)
selected_model_17.to_csv(
    selected_model_path_17,
    index=False,
)

selection_configuration_17 = {
    "outer_fold": OUTER_FOLD_17,
    "protocol_sha256": protocol_sha_17,
    "selection_metric_primary": (
        "pooled inner out-of-fold AUPRC"
    ),
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "selected_candidate": (
        selected_candidate_id_17
    ),
    "selected_hyperparameters": {
        key: selected_candidate_17[key]
        for key in selected_candidate_17
        if key != "candidate_id"
    },
    "inner_oof_auprc": float(
        best_row_17["auprc"]
    ),
    "inner_oof_auroc": float(
        best_row_17["auroc"]
    ),
    "inner_oof_brier": float(
        best_row_17["brier"]
    ),
    "platt_intercept": platt_intercept_17,
    "platt_slope": platt_slope_17,
    "inner_checkpoint_tables": (
        checkpoint_tables_17
    ),
    "patient_level_oof_written_to_drive": False,
}

with open(
    selection_json_path_17,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        selection_configuration_17,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    selection_json_path_17,
    "rb",
) as fh:
    selection_sha_17 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    selection_sha_path_17,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(selection_sha_17 + "\n")

# ------------------------------------------------------------
# 15. Fit selected model on all outer training patients
# ------------------------------------------------------------

final_preprocessor_17 = (
    make_preprocessor_17()
)

print(
    "\nFitting selected outer-fold-5 XGBoost model "
    "on all 46,803 training patients..."
)

preprocess_started_17 = time.time()

X_outer_training_processed_17 = (
    final_preprocessor_17.fit_transform(
        X_outer_training_17
    )
)
X_outer_test_processed_17 = (
    final_preprocessor_17.transform(
        X_outer_test_17
    )
)

final_preprocessing_seconds_17 = (
    time.time() - preprocess_started_17
)

final_model_17 = make_xgb_model_17(
    selected_candidate_17
)

final_fit_started_17 = time.time()

final_model_17.fit(
    X_outer_training_processed_17,
    y_outer_training_17,
)

final_fit_seconds_17 = (
    time.time() - final_fit_started_17
)

outer5_raw_probabilities_17 = (
    final_model_17.predict_proba(
        X_outer_test_processed_17
    )[:, 1]
)

raw_clipped_17 = np.clip(
    outer5_raw_probabilities_17,
    1e-6,
    1 - 1e-6,
)
raw_logit_17 = np.log(
    raw_clipped_17
    / (1 - raw_clipped_17)
)

outer5_platt_probabilities_17 = expit(
    platt_intercept_17
    + platt_slope_17 * raw_logit_17
)

for probabilities, name in [
    (outer5_raw_probabilities_17, "raw"),
    (
        outer5_platt_probabilities_17,
        "platt",
    ),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(
            f"{name} test predictions contain missing values."
        )

    if not np.all(
        (probabilities >= 0)
        & (probabilities <= 1)
    ):
        raise RuntimeError(
            f"{name} test predictions contain invalid probabilities."
        )

raw_metrics_17 = probability_metrics_17(
    y_outer_test_17,
    outer5_raw_probabilities_17,
)
platt_metrics_17 = probability_metrics_17(
    y_outer_test_17,
    outer5_platt_probabilities_17,
)

raw_calibration_intercept_17, \
raw_calibration_slope_17 = (
    calibration_intercept_slope_17(
        y_outer_test_17,
        outer5_raw_probabilities_17,
    )
)

platt_calibration_intercept_17, \
platt_calibration_slope_17 = (
    calibration_intercept_slope_17(
        y_outer_test_17,
        outer5_platt_probabilities_17,
    )
)

outer5_test_results_17 = pd.DataFrame(
    [
        {
            "outer_fold": OUTER_FOLD_17,
            "model": "xgboost",
            "probability_type": "raw",
            **raw_metrics_17,
            "calibration_intercept": (
                raw_calibration_intercept_17
            ),
            "calibration_slope": (
                raw_calibration_slope_17
            ),
        },
        {
            "outer_fold": OUTER_FOLD_17,
            "model": "xgboost",
            "probability_type": (
                "platt_calibrated"
            ),
            **platt_metrics_17,
            "calibration_intercept": (
                platt_calibration_intercept_17
            ),
            "calibration_slope": (
                platt_calibration_slope_17
            ),
        },
    ]
)

# ------------------------------------------------------------
# 16. Feature importance
# ------------------------------------------------------------

processed_feature_names_17 = (
    final_preprocessor_17
    .get_feature_names_out()
)

feature_importances_17 = (
    final_model_17.feature_importances_
)

if len(processed_feature_names_17) != len(
    feature_importances_17
):
    raise RuntimeError(
        "Processed feature names and XGBoost "
        "feature importances differ in length."
    )

feature_importance_table_17 = pd.DataFrame(
    {
        "processed_feature": (
            processed_feature_names_17
        ),
        "gain_importance": (
            feature_importances_17
        ),
    }
)

feature_importance_table_17[
    "importance_rank"
] = (
    feature_importance_table_17[
        "gain_importance"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)

feature_importance_table_17 = (
    feature_importance_table_17
    .sort_values(
        [
            "gain_importance",
            "processed_feature",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

nonzero_importance_features_17 = int(
    (
        feature_importance_table_17[
            "gain_importance"
        ] > 0
    ).sum()
)

final_model_summary_17 = pd.DataFrame(
    [
        {
            "outer_fold": OUTER_FOLD_17,
            "selected_candidate": (
                selected_candidate_id_17
            ),
            "training_patients": len(
                X_outer_training_17
            ),
            "training_hospitals": len(
                training_hospitals_17
            ),
            "training_events": int(
                y_outer_training_17.sum()
            ),
            "test_patients": len(
                X_outer_test_17
            ),
            "test_hospitals": len(
                test_hospitals_17
            ),
            "test_events": int(
                y_outer_test_17.sum()
            ),
            "hospital_overlap": len(
                hospital_overlap_17
            ),
            "processed_feature_columns": len(
                processed_feature_names_17
            ),
            "nonzero_importance_features": (
                nonzero_importance_features_17
            ),
            "preprocessing_seconds": float(
                final_preprocessing_seconds_17
            ),
            "fit_seconds": float(
                final_fit_seconds_17
            ),
            "locked_platt_intercept": (
                platt_intercept_17
            ),
            "locked_platt_slope": (
                platt_slope_17
            ),
            "protocol_sha256": protocol_sha_17,
            "selection_sha256": (
                selection_sha_17
            ),
            **{
                key: selected_candidate_17[key]
                for key in selected_candidate_17
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 17. Secure outer-test prediction checkpoint
# ------------------------------------------------------------

outer5_prediction_df_17 = pd.DataFrame(
    {
        "id_row": (
            outer_test_meta_17[
                "id_row"
            ].astype(str)
        ),
        "outer_fold": np.full(
            len(outer_test_meta_17),
            OUTER_FOLD_17,
            dtype=np.int64,
        ),
        "label_stage23": (
            y_outer_test_17.astype(np.int64)
        ),
        "prediction_raw": (
            outer5_raw_probabilities_17.astype(
                np.float64
            )
        ),
        "prediction_platt": (
            outer5_platt_probabilities_17.astype(
                np.float64
            )
        ),
        "model_name": "xgboost",
        "model_version": (
            "core_v1_nested_cv"
        ),
    }
)

if len(outer5_prediction_df_17) != 11688:
    raise RuntimeError(
        "Outer-fold-2 test prediction row count != 11,691."
    )

if outer5_prediction_df_17[
    "id_row"
].duplicated().any():
    raise RuntimeError(
        "Duplicate id_row in outer-fold-5 XGBoost predictions."
    )

if int(
    outer5_prediction_df_17[
        "label_stage23"
    ].sum()
) != 606:
    raise RuntimeError(
        "Outer-fold-2 XGBoost event count != 606."
    )

prediction_table_id_17 = (
    f"{TARGET_DATASET}."
    "model_xgb_outer_predictions_outer5_v1"
)

prediction_load_config_17 = (
    bigquery.LoadJobConfig(
        schema=[
            bigquery.SchemaField(
                "id_row",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "outer_fold",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "label_stage23",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_raw",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_platt",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_name",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_version",
                "STRING",
                mode="REQUIRED",
            ),
        ],
        write_disposition=(
            bigquery.WriteDisposition.WRITE_TRUNCATE
        ),
    )
)

print(
    "\nUploading secure outer-fold-5 "
    "XGBoost prediction checkpoint:"
)
print(prediction_table_id_17)

client.load_table_from_dataframe(
    outer5_prediction_df_17,
    prediction_table_id_17,
    job_config=prediction_load_config_17,
    location=BQ_LOCATION,
).result()

SQL_VERIFY_PREDICTIONS_17 = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL)
    AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL)
    AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0
    OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0
    OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw)
    AS minimum_raw_probability,
  MAX(prediction_raw)
    AS maximum_raw_probability,
  MIN(prediction_platt)
    AS minimum_platt_probability,
  MAX(prediction_platt)
    AS maximum_platt_probability
FROM `{prediction_table_id_17}`;
"""

prediction_verification_17 = (
    client.query(
        SQL_VERIFY_PREDICTIONS_17,
        location=BQ_LOCATION,
    )
    .to_dataframe()
)

verification_row_17 = (
    prediction_verification_17.iloc[0]
)

expected_prediction_values_17 = {
    "prediction_rows": 11688,
    "distinct_rows": 11688,
    "outer_folds": 1,
    "minimum_outer_fold": OUTER_FOLD_17,
    "maximum_outer_fold": OUTER_FOLD_17,
    "events": 606,
    "nonevents": 11082,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in (
    expected_prediction_values_17.items()
):
    actual_value = int(
        verification_row_17[field]
    )
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: found={actual_value}, "
            f"expected={expected_value}"
        )

# ------------------------------------------------------------
# 18. Save aggregate outputs
# ------------------------------------------------------------

test_results_path_17 = os.path.join(
    MODEL_OUTPUT_DIR,
    "17C_xgboost_outer5_test_results.csv",
)

model_summary_path_17 = os.path.join(
    MODEL_OUTPUT_DIR,
    "17C_xgboost_final_model_outer5.csv",
)

importance_path_17 = os.path.join(
    MODEL_OUTPUT_DIR,
    "17C_xgboost_gain_importance_outer5.csv",
)

evaluation_json_path_17 = os.path.join(
    MODEL_OUTPUT_DIR,
    "17C_xgboost_final_evaluation_outer5.json",
)

evaluation_sha_path_17 = os.path.join(
    MODEL_OUTPUT_DIR,
    "17C_xgboost_final_evaluation_outer5_SHA256.txt",
)

outer5_test_results_17.to_csv(
    test_results_path_17,
    index=False,
)
final_model_summary_17.to_csv(
    model_summary_path_17,
    index=False,
)
feature_importance_table_17.to_csv(
    importance_path_17,
    index=False,
)

evaluation_configuration_17 = {
    "outer_fold": OUTER_FOLD_17,
    "model_family": "xgboost",
    "protocol_sha256": protocol_sha_17,
    "selection_sha256": selection_sha_17,
    "selected_candidate": (
        selected_candidate_id_17
    ),
    "selected_hyperparameters": {
        key: selected_candidate_17[key]
        for key in selected_candidate_17
        if key != "candidate_id"
    },
    "training_patients": 46803,
    "training_hospitals": 159,
    "test_patients": 11688,
    "test_hospitals": 39,
    "hospital_overlap": 0,
    "locked_platt_intercept": (
        platt_intercept_17
    ),
    "locked_platt_slope": platt_slope_17,
    "processed_feature_columns": int(
        len(processed_feature_names_17)
    ),
    "secure_prediction_table": (
        prediction_table_id_17
    ),
    "patient_level_prediction_written_to_drive": False,
}

with open(
    evaluation_json_path_17,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        evaluation_configuration_17,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    evaluation_json_path_17,
    "rb",
) as fh:
    evaluation_sha_17 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    evaluation_sha_path_17,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(evaluation_sha_17 + "\n")

# ------------------------------------------------------------
# 19. Display results
# ------------------------------------------------------------

pooled_integrity_17 = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_17),
            pooled_oof_17[
                "id_row"
            ].nunique(),
            pooled_oof_17[
                "candidate_id"
            ].nunique(),
            pooled_oof_17[
                "inner_fold"
            ].nunique(),
            EXPECTED_SPLIT_17[
                "training_events"
            ],
            (
                EXPECTED_SPLIT_17[
                    "training_rows"
                ]
                - EXPECTED_SPLIT_17[
                    "training_events"
                ]
            ),
            int(
                pooled_oof_17.duplicated(
                    subset=[
                        "candidate_id",
                        "id_row",
                    ]
                ).sum()
            ),
            int(
                pooled_oof_17[
                    "prediction_raw"
                ].isna().sum()
            ),
            int(
                (
                    ~pooled_oof_17[
                        "prediction_raw"
                    ].between(0, 1)
                ).sum()
            ),
            pooled_load_method_17,
        ],
    }
)

print(
    "\n17 XGBOOST OUTER-FOLD-5 INNER CHECKPOINT SUMMARY"
)
display(checkpoint_summary_17)

print(
    "\n17 XGBOOST OUTER-FOLD-5 POOLED OOF INTEGRITY"
)
display(pooled_integrity_17)

print(
    "\n17 XGBOOST OUTER-FOLD-5 CANDIDATE RESULTS"
)
display(candidate_results_17)

print(
    "\n17 XGBOOST OUTER-FOLD-5 SELECTED MODEL"
)
display(selected_model_17)

print(
    "\n17 XGBOOST OUTER-FOLD-5 FINAL MODEL SUMMARY"
)
display(final_model_summary_17)

print(
    "\n17 XGBOOST OUTER-FOLD-5 TEST RESULTS"
)
display(outer5_test_results_17)

print(
    "\n17 XGBOOST OUTER-FOLD-5 BIGQUERY VERIFICATION"
)
display(prediction_verification_17)

print(
    "\n17 XGBOOST OUTER-FOLD-5 TOP 20 GAIN IMPORTANCE FEATURES"
)
display(feature_importance_table_17.head(20))

print("\nXGBoost protocol SHA-256:")
print(protocol_sha_17)

print("\nSelection SHA-256:")
print(selection_sha_17)

print("\nEvaluation SHA-256:")
print(evaluation_sha_17)

print("\nSaved aggregate outputs:")
print(protocol_path_17)
print(protocol_sha_path_17)
print(fit_audit_path_17)
print(checkpoint_summary_path_17)
print(candidate_results_path_17)
print(selected_model_path_17)
print(selection_json_path_17)
print(selection_sha_path_17)
print(test_results_path_17)
print(model_summary_path_17)
print(importance_path_17)
print(evaluation_json_path_17)
print(evaluation_sha_path_17)

print(
    "\n17 PASS: XGBoost outer-fold-5 nested modelling "
    "and locked test evaluation are complete."
)

print(
    "No class weighting, SMOTE, or early stopping was used."
)

print(
    "All patient-level OOF and outer-test predictions "
    "were stored only in BigQuery."
)

print(
    "No patient-level prediction file was written to Google Drive."
)

_ = gc.collect()

In [ ]:
import os
import json
import hashlib

import numpy as np
import pandas as pd

from google.cloud import bigquery
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from IPython.display import display

print("STARTING FINAL FIVE-FOLD XGBOOST POOLED EVALUATION AND LR COMPARISON — CODE VERSION 18")

# ============================================================
# 18 — FINAL FIVE-FOLD XGBOOST EVALUATION + PAIRED LR COMPARISON
#
# Primary goals:
# 1) Pool exactly one locked outer-test XGBoost prediction per patient.
# 2) Verify 58,491 patients, 198 hospitals, 3,032 events.
# 3) Calculate raw and fold-specific Platt pooled metrics.
# 4) Estimate 2,000 hospital-cluster bootstrap CIs.
# 5) Compare XGBoost vs elastic-net logistic regression on the SAME
#    patients using paired hospital-cluster bootstrap.
# 6) Keep patient-level data only in BigQuery / RAM.
#
# No model fitting, tuning, SMOTE, class weighting, or early stopping
# occurs in this cell.
# No DELETE / INSERT / UPDATE / MERGE statement is used.
# ============================================================

MODEL_NAME_18 = "xgboost"
MODEL_VERSION_18 = "core_v1_nested_cv"

BOOTSTRAP_REPLICATES_18 = 2000
BOOTSTRAP_SEED_18 = 20260723

EXPECTED_XGB_PROTOCOL_SHA_18 = (
    "3434db5dd0b4b5145950fc07fba3d007"
    "829738d800b88ec40fbdb09879188264"
)

EXPECTED_TOTAL_ROWS_18 = 58491
EXPECTED_TOTAL_EVENTS_18 = 3032
EXPECTED_TOTAL_NONEVENTS_18 = 55459
EXPECTED_TOTAL_HOSPITALS_18 = 198

EXPECTED_FOLD_STRUCTURE_18 = {
    1: {"rows": 11688, "events": 606, "nonevents": 11082, "hospitals": 40},
    2: {"rows": 11691, "events": 606, "nonevents": 11085, "hospitals": 40},
    3: {"rows": 11736, "events": 608, "nonevents": 11128, "hospitals": 40},
    4: {"rows": 11688, "events": 606, "nonevents": 11082, "hospitals": 39},
    5: {"rows": 11688, "events": 606, "nonevents": 11082, "hospitals": 39},
}

# ------------------------------------------------------------
# 1. Required objects and locked protocol
# ------------------------------------------------------------

required_objects_18 = [
    "core_df_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_18 = [
    name for name in required_objects_18 if name not in globals()
]

if missing_objects_18:
    raise RuntimeError(
        "Eksik çalışma nesneleri var: "
        + ", ".join(missing_objects_18)
        + ". Önce 07A ve 07B hücrelerini çalıştır."
    )

if len(core_df_07B) != EXPECTED_TOTAL_ROWS_18:
    raise RuntimeError(
        f"core_df_07B rows={len(core_df_07B)}, "
        f"expected={EXPECTED_TOTAL_ROWS_18}."
    )

xgb_protocol_sha_path_18 = os.path.join(
    MODEL_OUTPUT_DIR,
    "13A_locked_xgboost_model_protocol_v1_SHA256.txt",
)

if not os.path.exists(xgb_protocol_sha_path_18):
    raise FileNotFoundError(xgb_protocol_sha_path_18)

with open(xgb_protocol_sha_path_18, "r", encoding="utf-8") as fh:
    observed_xgb_protocol_sha_18 = fh.read().strip()

if observed_xgb_protocol_sha_18 != EXPECTED_XGB_PROTOCOL_SHA_18:
    raise RuntimeError(
        "Locked XGBoost protocol SHA mismatch: "
        + observed_xgb_protocol_sha_18
    )

print("Locked XGBoost protocol SHA-256:")
print(observed_xgb_protocol_sha_18)

# ------------------------------------------------------------
# 2. Source XGBoost prediction tables
# ------------------------------------------------------------

source_xgb_tables_18 = {
    1: f"{TARGET_DATASET}.model_xgb_outer_predictions_outer1_v1",
    2: f"{TARGET_DATASET}.model_xgb_outer_predictions_outer2_v1",
    3: f"{TARGET_DATASET}.model_xgb_outer_predictions_outer3_v1",
    4: f"{TARGET_DATASET}.model_xgb_outer_predictions_outer4_v1",
    5: f"{TARGET_DATASET}.model_xgb_outer_predictions_outer5_v1",
}

for fold_number, table_id in source_xgb_tables_18.items():
    try:
        client.get_table(table_id)
    except Exception as exc:
        raise RuntimeError(
            f"XGBoost outer-fold-{fold_number} table unavailable: "
            f"{table_id}. {type(exc).__name__}: {exc}"
        )

# ------------------------------------------------------------
# 3. Robust prediction-table normalization
# ------------------------------------------------------------

def resolve_column_18(
    columns,
    exact_candidates,
    contains_all=None,
    contains_any=None,
    required=True,
):
    original_columns = list(columns)
    lower_to_original = {
        str(column).lower(): column
        for column in original_columns
    }

    for candidate in exact_candidates:
        if candidate.lower() in lower_to_original:
            return lower_to_original[candidate.lower()]

    matches = []

    for original in original_columns:
        lowered = str(original).lower()

        if contains_all and not all(
            token.lower() in lowered
            for token in contains_all
        ):
            continue

        if contains_any and not any(
            token.lower() in lowered
            for token in contains_any
        ):
            continue

        if contains_all or contains_any:
            matches.append(original)

    if len(matches) == 1:
        return matches[0]

    if required:
        raise RuntimeError(
            "Required column could not be resolved. "
            f"Candidates={exact_candidates}; columns={original_columns}; "
            f"heuristic_matches={matches}"
        )

    return None


def normalize_prediction_table_18(
    raw_dataframe,
    expected_fold,
    source_table,
    default_model_name,
    default_model_version,
):
    df = raw_dataframe.copy()

    id_col = resolve_column_18(
        df.columns,
        ["id_row", "patient_id", "patientunitstayid"],
        contains_any=["id_row"],
    )

    label_col = resolve_column_18(
        df.columns,
        ["label_stage23", "label", "outcome", "y_true"],
        contains_any=["label_stage23"],
    )

    fold_col = resolve_column_18(
        df.columns,
        ["outer_fold", "fold"],
        contains_all=["outer", "fold"],
        required=False,
    )

    raw_col = resolve_column_18(
        df.columns,
        [
            "prediction_raw",
            "raw_probability",
            "probability_raw",
            "prediction_raw_probability",
        ],
        contains_all=["raw"],
        contains_any=["prediction", "probability", "prob"],
    )

    platt_col = resolve_column_18(
        df.columns,
        [
            "prediction_platt",
            "prediction_platt_calibrated",
            "platt_probability",
            "probability_platt",
            "calibrated_probability",
            "prediction_calibrated",
        ],
        contains_any=["platt", "calibrated"],
    )

    model_name_col = resolve_column_18(
        df.columns,
        ["model_name", "model"],
        contains_all=["model", "name"],
        required=False,
    )

    model_version_col = resolve_column_18(
        df.columns,
        ["model_version", "version"],
        contains_all=["model", "version"],
        required=False,
    )

    out = pd.DataFrame({
        "id_row": df[id_col].astype(str),
        "label_stage23": pd.to_numeric(
            df[label_col], errors="raise"
        ).astype(np.int64),
        "prediction_raw": pd.to_numeric(
            df[raw_col], errors="raise"
        ).astype(np.float64),
        "prediction_platt": pd.to_numeric(
            df[platt_col], errors="raise"
        ).astype(np.float64),
    })

    if fold_col is None:
        out["outer_fold"] = np.full(
            len(out), expected_fold, dtype=np.int64
        )
    else:
        out["outer_fold"] = pd.to_numeric(
            df[fold_col], errors="raise"
        ).astype(np.int64)

    out["model_name"] = (
        df[model_name_col].astype(str)
        if model_name_col is not None
        else default_model_name
    )

    out["model_version"] = (
        df[model_version_col].astype(str)
        if model_version_col is not None
        else default_model_version
    )

    out["source_table"] = source_table

    if set(out["outer_fold"].unique()) != {expected_fold}:
        raise RuntimeError(
            f"{source_table}: expected outer_fold={expected_fold}; "
            f"found={sorted(out['outer_fold'].unique())}"
        )

    if out["id_row"].duplicated().any():
        raise RuntimeError(
            f"{source_table}: duplicate id_row values found."
        )

    for col in ["prediction_raw", "prediction_platt"]:
        if out[col].isna().any():
            raise RuntimeError(
                f"{source_table}: missing values in {col}."
            )
        if not out[col].between(0, 1).all():
            raise RuntimeError(
                f"{source_table}: probabilities outside 0-1 in {col}."
            )

    return out[
        [
            "id_row",
            "outer_fold",
            "label_stage23",
            "prediction_raw",
            "prediction_platt",
            "model_name",
            "model_version",
            "source_table",
        ]
    ].copy()

# ------------------------------------------------------------
# 4. Load all five XGBoost folds
# ------------------------------------------------------------

xgb_frames_18 = []
source_schema_rows_18 = []

for fold_number in range(1, 6):
    table_id = source_xgb_tables_18[fold_number]

    print(
        f"Loading XGBoost outer-fold-{fold_number}: {table_id}"
    )

    query_job = client.query(
        f"SELECT * FROM `{table_id}`",
        location=BQ_LOCATION,
    )

    try:
        raw_df = query_job.to_dataframe(
            create_bqstorage_client=True
        )
        load_method = "BigQuery Storage API"
    except Exception:
        raw_df = query_job.to_dataframe(
            create_bqstorage_client=False
        )
        load_method = "Standard BigQuery API"

    source_schema_rows_18.append({
        "outer_fold": fold_number,
        "source_table": table_id,
        "source_rows": len(raw_df),
        "source_columns": " | ".join(map(str, raw_df.columns)),
        "load_method": load_method,
    })

    normalized = normalize_prediction_table_18(
        raw_dataframe=raw_df,
        expected_fold=fold_number,
        source_table=table_id,
        default_model_name=MODEL_NAME_18,
        default_model_version=MODEL_VERSION_18,
    )

    xgb_frames_18.append(normalized)

source_schema_summary_18 = pd.DataFrame(
    source_schema_rows_18
)

xgb_pooled_18 = pd.concat(
    xgb_frames_18,
    ignore_index=True,
)

# ------------------------------------------------------------
# 5. Locked-core audit and hospital attachment
# ------------------------------------------------------------

required_core_columns_18 = {
    "id_row",
    "outer_fold",
    "label_stage23",
    "group_hospital",
}

missing_core_columns_18 = (
    required_core_columns_18
    - set(core_df_07B.columns)
)

if missing_core_columns_18:
    raise RuntimeError(
        "core_df_07B missing audit columns: "
        + ", ".join(sorted(missing_core_columns_18))
    )

core_audit_18 = core_df_07B[
    [
        "id_row",
        "outer_fold",
        "label_stage23",
        "group_hospital",
    ]
].copy()

core_audit_18["id_row"] = (
    core_audit_18["id_row"].astype(str)
)
core_audit_18["outer_fold"] = pd.to_numeric(
    core_audit_18["outer_fold"],
    errors="raise",
).astype(np.int64)
core_audit_18["label_stage23"] = pd.to_numeric(
    core_audit_18["label_stage23"],
    errors="raise",
).astype(np.int64)
core_audit_18["group_hospital"] = (
    core_audit_18["group_hospital"].astype(str)
)

if core_audit_18["id_row"].duplicated().any():
    raise RuntimeError("core_df_07B contains duplicate id_row values.")

if len(xgb_pooled_18) != EXPECTED_TOTAL_ROWS_18:
    raise RuntimeError(
        f"XGBoost pooled rows={len(xgb_pooled_18)}, "
        f"expected={EXPECTED_TOTAL_ROWS_18}"
    )

if xgb_pooled_18["id_row"].duplicated().any():
    raise RuntimeError(
        "XGBoost pooled predictions contain duplicate patients."
    )

xgb_audited_18 = xgb_pooled_18.merge(
    core_audit_18,
    on="id_row",
    how="inner",
    suffixes=("_prediction", "_core"),
    validate="one_to_one",
)

if len(xgb_audited_18) != EXPECTED_TOTAL_ROWS_18:
    raise RuntimeError(
        "XGBoost-core merge did not retain all 58,491 patients."
    )

fold_mismatch_18 = int(
    (
        xgb_audited_18["outer_fold_prediction"]
        != xgb_audited_18["outer_fold_core"]
    ).sum()
)

label_mismatch_18 = int(
    (
        xgb_audited_18["label_stage23_prediction"]
        != xgb_audited_18["label_stage23_core"]
    ).sum()
)

if fold_mismatch_18 != 0:
    raise RuntimeError(
        f"XGBoost/core outer-fold mismatches={fold_mismatch_18}"
    )

if label_mismatch_18 != 0:
    raise RuntimeError(
        f"XGBoost/core outcome mismatches={label_mismatch_18}"
    )

xgb_audited_18 = xgb_audited_18.rename(
    columns={
        "outer_fold_prediction": "outer_fold",
        "label_stage23_prediction": "label_stage23",
    }
).drop(
    columns=[
        "outer_fold_core",
        "label_stage23_core",
    ]
)

if int(xgb_audited_18["label_stage23"].sum()) != EXPECTED_TOTAL_EVENTS_18:
    raise RuntimeError("XGBoost pooled event count is not 3,032.")

if xgb_audited_18["group_hospital"].nunique() != EXPECTED_TOTAL_HOSPITALS_18:
    raise RuntimeError("XGBoost pooled hospital count is not 198.")

hospital_fold_count_18 = (
    xgb_audited_18
    .groupby("group_hospital")["outer_fold"]
    .nunique()
)

if (hospital_fold_count_18 > 1).any():
    raise RuntimeError(
        "A hospital appears in more than one outer-test fold."
    )

# ------------------------------------------------------------
# 6. Fold integrity
# ------------------------------------------------------------

fold_integrity_rows_18 = []

for fold_number in range(1, 6):
    part = xgb_audited_18.loc[
        xgb_audited_18["outer_fold"] == fold_number
    ].copy()

    expected = EXPECTED_FOLD_STRUCTURE_18[fold_number]

    actual = {
        "rows": len(part),
        "events": int(part["label_stage23"].sum()),
        "nonevents": int(
            len(part) - part["label_stage23"].sum()
        ),
        "hospitals": int(
            part["group_hospital"].nunique()
        ),
    }

    for key, expected_value in expected.items():
        if actual[key] != expected_value:
            raise RuntimeError(
                f"Fold {fold_number} {key}: "
                f"found={actual[key]}, expected={expected_value}"
            )

    fold_integrity_rows_18.append({
        "outer_fold": fold_number,
        "prediction_rows": actual["rows"],
        "distinct_patients": part["id_row"].nunique(),
        "hospitals": actual["hospitals"],
        "events": actual["events"],
        "nonevents": actual["nonevents"],
        "minimum_raw_probability": float(
            part["prediction_raw"].min()
        ),
        "maximum_raw_probability": float(
            part["prediction_raw"].max()
        ),
        "minimum_platt_probability": float(
            part["prediction_platt"].min()
        ),
        "maximum_platt_probability": float(
            part["prediction_platt"].max()
        ),
    })

fold_integrity_summary_18 = pd.DataFrame(
    fold_integrity_rows_18
)

# ------------------------------------------------------------
# 7. Secure pooled XGBoost BigQuery table
# ------------------------------------------------------------

xgb_pooled_table_id_18 = (
    f"{TARGET_DATASET}."
    "model_xgb_outer_predictions_all5_v1"
)

xgb_secure_df_18 = xgb_audited_18[
    [
        "id_row",
        "outer_fold",
        "label_stage23",
        "prediction_raw",
        "prediction_platt",
        "model_name",
        "model_version",
        "group_hospital",
    ]
].copy()

xgb_secure_df_18["id_row"] = (
    xgb_secure_df_18["id_row"].astype(str)
)
xgb_secure_df_18["group_hospital"] = (
    xgb_secure_df_18["group_hospital"].astype(str)
)

pooled_load_config_18 = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("prediction_platt", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("model_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("model_version", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("group_hospital", "STRING", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

print("\nUploading secure pooled XGBoost prediction table:")
print(xgb_pooled_table_id_18)

client.load_table_from_dataframe(
    xgb_secure_df_18,
    xgb_pooled_table_id_18,
    job_config=pooled_load_config_18,
    location=BQ_LOCATION,
).result()

verify_sql_18 = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_patients,
  COUNT(DISTINCT group_hospital) AS hospitals,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL) AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL) AS missing_platt_predictions,
  COUNTIF(prediction_raw < 0 OR prediction_raw > 1)
    AS invalid_raw_predictions,
  COUNTIF(prediction_platt < 0 OR prediction_platt > 1)
    AS invalid_platt_predictions,
  MIN(prediction_raw) AS minimum_raw_probability,
  MAX(prediction_raw) AS maximum_raw_probability,
  MIN(prediction_platt) AS minimum_platt_probability,
  MAX(prediction_platt) AS maximum_platt_probability
FROM `{xgb_pooled_table_id_18}`
"""

xgb_bigquery_verification_18 = (
    client.query(
        verify_sql_18,
        location=BQ_LOCATION,
    )
    .to_dataframe()
)

vr = xgb_bigquery_verification_18.iloc[0]

expected_verification_18 = {
    "prediction_rows": EXPECTED_TOTAL_ROWS_18,
    "distinct_patients": EXPECTED_TOTAL_ROWS_18,
    "hospitals": EXPECTED_TOTAL_HOSPITALS_18,
    "outer_folds": 5,
    "events": EXPECTED_TOTAL_EVENTS_18,
    "nonevents": EXPECTED_TOTAL_NONEVENTS_18,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in expected_verification_18.items():
    actual_value = int(vr[field])
    if actual_value != expected_value:
        raise RuntimeError(
            f"XGBoost pooled BigQuery verification {field}: "
            f"found={actual_value}, expected={expected_value}"
        )

# ------------------------------------------------------------
# 8. Metric helpers
# ------------------------------------------------------------

def logit_18(probabilities):
    p = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        p / (1 - p)
    ).reshape(-1, 1)


def calibration_intercept_slope_18(
    y_true,
    probabilities,
):
    model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    model.fit(
        logit_18(probabilities),
        y_true,
    )
    return (
        float(model.intercept_[0]),
        float(model.coef_[0][0]),
    )


def probability_metrics_18(
    y_true,
    probabilities,
):
    p = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )
    intercept, slope = (
        calibration_intercept_slope_18(
            y_true,
            p,
        )
    )

    return {
        "auroc": float(
            roc_auc_score(y_true, p)
        ),
        "auprc": float(
            average_precision_score(y_true, p)
        ),
        "brier": float(
            brier_score_loss(y_true, p)
        ),
        "log_loss": float(
            log_loss(
                y_true,
                p,
                labels=[0, 1],
            )
        ),
        "mean_predicted_risk": float(p.mean()),
        "observed_event_rate": float(np.mean(y_true)),
        "calibration_intercept": intercept,
        "calibration_slope": slope,
    }


def weighted_metric_bundle_18(
    y_true,
    probabilities,
    sample_weight,
):
    p = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(
            roc_auc_score(
                y_true,
                p,
                sample_weight=sample_weight,
            )
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                p,
                sample_weight=sample_weight,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                p,
                sample_weight=sample_weight,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                p,
                labels=[0, 1],
                sample_weight=sample_weight,
            )
        ),
    }

# ------------------------------------------------------------
# 9. XGBoost fold-level and pooled point metrics
# ------------------------------------------------------------

y_xgb_18 = xgb_audited_18[
    "label_stage23"
].to_numpy(dtype=np.int8)

raw_xgb_18 = xgb_audited_18[
    "prediction_raw"
].to_numpy(dtype=float)

platt_xgb_18 = xgb_audited_18[
    "prediction_platt"
].to_numpy(dtype=float)

fold_metric_rows_18 = []

for fold_number in range(1, 6):
    mask = (
        xgb_audited_18["outer_fold"]
        .to_numpy(dtype=int)
        == fold_number
    )

    fold_y = y_xgb_18[mask]

    for probability_type, vector in [
        ("raw", raw_xgb_18),
        ("platt_calibrated", platt_xgb_18),
    ]:
        metrics = probability_metrics_18(
            fold_y,
            vector[mask],
        )

        fold_metric_rows_18.append({
            "outer_fold": fold_number,
            "model": MODEL_NAME_18,
            "probability_type": probability_type,
            "patients": int(mask.sum()),
            "events": int(fold_y.sum()),
            **metrics,
        })

xgb_fold_metrics_18 = pd.DataFrame(
    fold_metric_rows_18
)

macro_rows_18 = []

for probability_type in ["raw", "platt_calibrated"]:
    subset = xgb_fold_metrics_18.loc[
        xgb_fold_metrics_18[
            "probability_type"
        ] == probability_type
    ]

    for metric_name in [
        "auroc",
        "auprc",
        "brier",
        "log_loss",
        "mean_predicted_risk",
        "calibration_intercept",
        "calibration_slope",
    ]:
        values = subset[
            metric_name
        ].to_numpy(dtype=float)

        macro_rows_18.append({
            "probability_type": probability_type,
            "metric": metric_name,
            "fold_mean": float(np.mean(values)),
            "fold_sd": float(np.std(values, ddof=1)),
            "fold_minimum": float(np.min(values)),
            "fold_maximum": float(np.max(values)),
        })

xgb_macro_summary_18 = pd.DataFrame(
    macro_rows_18
)

pooled_point_rows_18 = []

for probability_type, vector in [
    ("raw", raw_xgb_18),
    ("platt_calibrated", platt_xgb_18),
]:
    metrics = probability_metrics_18(
        y_xgb_18,
        vector,
    )

    pooled_point_rows_18.append({
        "model": MODEL_NAME_18,
        "probability_type": probability_type,
        "patients": EXPECTED_TOTAL_ROWS_18,
        "hospitals": EXPECTED_TOTAL_HOSPITALS_18,
        "events": EXPECTED_TOTAL_EVENTS_18,
        "nonevents": EXPECTED_TOTAL_NONEVENTS_18,
        **metrics,
    })

xgb_pooled_point_metrics_18 = pd.DataFrame(
    pooled_point_rows_18
)

# ------------------------------------------------------------
# 10. Selected-model consistency audit
# ------------------------------------------------------------

selected_model_paths_18 = {
    1: os.path.join(
        MODEL_OUTPUT_DIR,
        "13C_xgboost_selected_model_outer1.csv",
    ),
    2: os.path.join(
        MODEL_OUTPUT_DIR,
        "14B_xgboost_selected_model_outer2.csv",
    ),
    3: os.path.join(
        MODEL_OUTPUT_DIR,
        "15B_xgboost_selected_model_outer3.csv",
    ),
    4: os.path.join(
        MODEL_OUTPUT_DIR,
        "16B_xgboost_selected_model_outer4.csv",
    ),
    5: os.path.join(
        MODEL_OUTPUT_DIR,
        "17B_xgboost_selected_model_outer5.csv",
    ),
}

selection_rows_18 = []

for fold_number, path in selected_model_paths_18.items():
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    df = pd.read_csv(path)

    if len(df) != 1:
        raise RuntimeError(
            f"Fold {fold_number} selected-model file must have one row."
        )

    row = df.iloc[0]

    file_outer_fold = int(
        pd.to_numeric(
            row["outer_fold"],
            errors="raise",
        )
    )

    if file_outer_fold != fold_number:
        raise RuntimeError(
            f"Fold {fold_number} selection file outer_fold="
            f"{file_outer_fold}."
        )

    selection_rows_18.append({
        "outer_fold": fold_number,
        "selected_candidate": str(
            row["selected_candidate"]
        ),
        "inner_oof_auprc": float(
            row["inner_oof_auprc"]
        ),
        "inner_oof_auroc": float(
            row["inner_oof_auroc"]
        ),
        "platt_intercept": float(
            row["platt_intercept"]
        ),
        "platt_slope": float(
            row["platt_slope"]
        ),
        "n_estimators": int(
            row["n_estimators"]
        ),
        "max_depth": int(
            row["max_depth"]
        ),
        "learning_rate": float(
            row["learning_rate"]
        ),
        "min_child_weight": float(
            row["min_child_weight"]
        ),
        "subsample": float(
            row["subsample"]
        ),
        "colsample_bytree": float(
            row["colsample_bytree"]
        ),
        "gamma": float(
            row["gamma"]
        ),
        "reg_alpha": float(
            row["reg_alpha"]
        ),
        "reg_lambda": float(
            row["reg_lambda"]
        ),
        "selection_file": path,
    })

selected_model_consistency_18 = pd.DataFrame(
    selection_rows_18
)

expected_selected_18 = {
    1: "XGB04",
    2: "XGB04",
    3: "XGB04",
    4: "XGB04",
    5: "XGB06",
}

for fold_number, expected_candidate in expected_selected_18.items():
    observed_candidate = str(
        selected_model_consistency_18.loc[
            selected_model_consistency_18["outer_fold"]
            == fold_number,
            "selected_candidate",
        ].iloc[0]
    )
    if observed_candidate != expected_candidate:
        raise RuntimeError(
            f"Fold {fold_number}: selected candidate "
            f"{observed_candidate}, expected {expected_candidate}."
        )

# ------------------------------------------------------------
# 11. Correct raw and Platt hospital-cluster bootstrap for XGBoost
# ------------------------------------------------------------

print(
    f"\nRunning {BOOTSTRAP_REPLICATES_18:,} "
    "hospital-cluster bootstrap replicates for XGBoost..."
)

hospital_categorical_18 = pd.Categorical(
    xgb_audited_18["group_hospital"]
)

hospital_codes_18 = (
    hospital_categorical_18.codes.astype(int)
)

hospital_labels_18 = list(
    hospital_categorical_18.categories
)

n_hospitals_18 = len(hospital_labels_18)

if n_hospitals_18 != EXPECTED_TOTAL_HOSPITALS_18:
    raise RuntimeError(
        "Bootstrap hospital count is not 198."
    )

rng_18 = np.random.default_rng(
    BOOTSTRAP_SEED_18
)

bootstrap_rows_18 = []

for bootstrap_index in range(
    BOOTSTRAP_REPLICATES_18
):
    sampled_codes = rng_18.integers(
        low=0,
        high=n_hospitals_18,
        size=n_hospitals_18,
    )

    multiplicity = np.bincount(
        sampled_codes,
        minlength=n_hospitals_18,
    )

    weights = multiplicity[
        hospital_codes_18
    ].astype(float)

    positive_weight = float(
        weights[y_xgb_18 == 1].sum()
    )
    negative_weight = float(
        weights[y_xgb_18 == 0].sum()
    )

    if positive_weight <= 0 or negative_weight <= 0:
        continue

    raw_metrics = weighted_metric_bundle_18(
        y_xgb_18,
        raw_xgb_18,
        weights,
    )
    platt_metrics = weighted_metric_bundle_18(
        y_xgb_18,
        platt_xgb_18,
        weights,
    )

    bootstrap_rows_18.append({
        "bootstrap_replicate": bootstrap_index + 1,
        "unique_sampled_hospitals": int(
            np.count_nonzero(multiplicity)
        ),
        "weighted_patients": float(weights.sum()),
        "weighted_events": positive_weight,
        "auroc_raw": raw_metrics["auroc"],
        "auroc_platt": platt_metrics["auroc"],
        "auprc_raw": raw_metrics["auprc"],
        "auprc_platt": platt_metrics["auprc"],
        "brier_raw": raw_metrics["brier"],
        "brier_platt": platt_metrics["brier"],
        "log_loss_raw": raw_metrics["log_loss"],
        "log_loss_platt": platt_metrics["log_loss"],
    })

    if (
        bootstrap_index == 0
        or (bootstrap_index + 1) % 100 == 0
    ):
        print(
            "  Completed XGBoost bootstrap replicate",
            bootstrap_index + 1,
            "/",
            BOOTSTRAP_REPLICATES_18,
        )

xgb_bootstrap_replicates_18 = pd.DataFrame(
    bootstrap_rows_18
)

if len(xgb_bootstrap_replicates_18) < (
    BOOTSTRAP_REPLICATES_18 * 0.99
):
    raise RuntimeError(
        "Fewer than 99% of XGBoost bootstrap replicates completed."
    )

xgb_ci_rows_18 = []

point_lookup_18 = {
    "raw": xgb_pooled_point_metrics_18.loc[
        xgb_pooled_point_metrics_18["probability_type"]
        == "raw"
    ].iloc[0],
    "platt_calibrated": xgb_pooled_point_metrics_18.loc[
        xgb_pooled_point_metrics_18["probability_type"]
        == "platt_calibrated"
    ].iloc[0],
}

ci_specs_18 = [
    ("auroc", "raw", "auroc_raw"),
    ("auroc", "platt_calibrated", "auroc_platt"),
    ("auprc", "raw", "auprc_raw"),
    ("auprc", "platt_calibrated", "auprc_platt"),
    ("brier", "raw", "brier_raw"),
    ("brier", "platt_calibrated", "brier_platt"),
    ("log_loss", "raw", "log_loss_raw"),
    ("log_loss", "platt_calibrated", "log_loss_platt"),
]

for metric, probability_type, column in ci_specs_18:
    values = xgb_bootstrap_replicates_18[
        column
    ].dropna().to_numpy(dtype=float)

    xgb_ci_rows_18.append({
        "metric": metric,
        "probability_type": probability_type,
        "point_estimate": float(
            point_lookup_18[
                probability_type
            ][metric]
        ),
        "bootstrap_replicates": len(values),
        "bootstrap_mean": float(
            np.mean(values)
        ),
        "bootstrap_standard_error": float(
            np.std(values, ddof=1)
        ),
        "ci_95_lower": float(
            np.quantile(values, 0.025)
        ),
        "ci_95_upper": float(
            np.quantile(values, 0.975)
        ),
        "bootstrap_unit": "hospital",
        "bootstrap_seed": BOOTSTRAP_SEED_18,
    })

xgb_bootstrap_ci_18 = pd.DataFrame(
    xgb_ci_rows_18
)

# ------------------------------------------------------------
# 12. XGBoost pooled calibration deciles
# ------------------------------------------------------------

def calibration_quantiles_18(
    dataframe,
    probability_column,
    probability_type,
    requested_bins=10,
):
    working = dataframe[
        ["label_stage23", probability_column]
    ].copy()

    working["risk_group"] = pd.qcut(
        working[probability_column],
        q=requested_bins,
        labels=False,
        duplicates="drop",
    )

    summary = (
        working
        .groupby(
            "risk_group",
            observed=True,
        )
        .agg(
            patients=("label_stage23", "size"),
            events=("label_stage23", "sum"),
            mean_predicted_risk=(probability_column, "mean"),
            minimum_predicted_risk=(probability_column, "min"),
            maximum_predicted_risk=(probability_column, "max"),
        )
        .reset_index()
    )

    summary["observed_event_rate"] = (
        summary["events"]
        / summary["patients"]
    )

    summary["probability_type"] = probability_type
    summary["requested_bins"] = requested_bins
    summary["actual_bins"] = len(summary)

    return summary[
        [
            "probability_type",
            "requested_bins",
            "actual_bins",
            "risk_group",
            "patients",
            "events",
            "mean_predicted_risk",
            "observed_event_rate",
            "minimum_predicted_risk",
            "maximum_predicted_risk",
        ]
    ]


xgb_calibration_deciles_18 = pd.concat(
    [
        calibration_quantiles_18(
            xgb_audited_18,
            "prediction_raw",
            "raw",
        ),
        calibration_quantiles_18(
            xgb_audited_18,
            "prediction_platt",
            "platt_calibrated",
        ),
    ],
    ignore_index=True,
)

# ------------------------------------------------------------
# 13. Load locked pooled logistic-regression predictions
# ------------------------------------------------------------

lr_pooled_table_id_18 = (
    f"{TARGET_DATASET}."
    "model_lr_outer_predictions_all5_v1"
)

try:
    client.get_table(lr_pooled_table_id_18)
except Exception as exc:
    raise RuntimeError(
        "Locked pooled logistic prediction table not available: "
        f"{lr_pooled_table_id_18}. {type(exc).__name__}: {exc}"
    )

lr_query_job_18 = client.query(
    f"SELECT * FROM `{lr_pooled_table_id_18}`",
    location=BQ_LOCATION,
)

try:
    lr_raw_df_18 = lr_query_job_18.to_dataframe(
        create_bqstorage_client=True
    )
    lr_load_method_18 = "BigQuery Storage API"
except Exception:
    lr_raw_df_18 = lr_query_job_18.to_dataframe(
        create_bqstorage_client=False
    )
    lr_load_method_18 = "Standard BigQuery API"

# The pooled logistic table contains outer folds 1..5, so it is
# normalized manually rather than through the single-fold helper.
lr_id_col_18 = resolve_column_18(
    lr_raw_df_18.columns,
    ["id_row", "patient_id", "patientunitstayid"],
    contains_any=["id_row"],
)
lr_label_col_18 = resolve_column_18(
    lr_raw_df_18.columns,
    ["label_stage23", "label", "outcome", "y_true"],
    contains_any=["label_stage23"],
)
lr_fold_col_18 = resolve_column_18(
    lr_raw_df_18.columns,
    ["outer_fold", "fold"],
    contains_all=["outer", "fold"],
)
lr_raw_col_18 = resolve_column_18(
    lr_raw_df_18.columns,
    ["prediction_raw", "raw_probability", "probability_raw"],
    contains_all=["raw"],
    contains_any=["prediction", "probability", "prob"],
)
lr_platt_col_18 = resolve_column_18(
    lr_raw_df_18.columns,
    [
        "prediction_platt",
        "prediction_platt_calibrated",
        "platt_probability",
        "calibrated_probability",
    ],
    contains_any=["platt", "calibrated"],
)
lr_hospital_col_18 = resolve_column_18(
    lr_raw_df_18.columns,
    ["group_hospital", "hospital_id", "hospital"],
    contains_any=["hospital"],
    required=False,
)

lr_pooled_18 = pd.DataFrame({
    "id_row": lr_raw_df_18[lr_id_col_18].astype(str),
    "outer_fold": pd.to_numeric(
        lr_raw_df_18[lr_fold_col_18],
        errors="raise",
    ).astype(np.int64),
    "label_stage23": pd.to_numeric(
        lr_raw_df_18[lr_label_col_18],
        errors="raise",
    ).astype(np.int64),
    "lr_prediction_raw": pd.to_numeric(
        lr_raw_df_18[lr_raw_col_18],
        errors="raise",
    ).astype(np.float64),
    "lr_prediction_platt": pd.to_numeric(
        lr_raw_df_18[lr_platt_col_18],
        errors="raise",
    ).astype(np.float64),
})

if lr_hospital_col_18 is not None:
    lr_pooled_18["group_hospital_lr"] = (
        lr_raw_df_18[lr_hospital_col_18].astype(str)
    )

if len(lr_pooled_18) != EXPECTED_TOTAL_ROWS_18:
    raise RuntimeError(
        "Pooled logistic table does not contain 58,491 rows."
    )

if lr_pooled_18["id_row"].duplicated().any():
    raise RuntimeError(
        "Pooled logistic table contains duplicate patients."
    )

if set(lr_pooled_18["outer_fold"].unique()) != {1, 2, 3, 4, 5}:
    raise RuntimeError(
        "Pooled logistic table outer_fold values are not 1-5."
    )

# ------------------------------------------------------------
# 14. Paired XGBoost-vs-logistic patient alignment
# ------------------------------------------------------------

comparison_18 = xgb_audited_18[
    [
        "id_row",
        "outer_fold",
        "label_stage23",
        "group_hospital",
        "prediction_raw",
        "prediction_platt",
    ]
].rename(
    columns={
        "prediction_raw": "xgb_prediction_raw",
        "prediction_platt": "xgb_prediction_platt",
    }
).merge(
    lr_pooled_18,
    on="id_row",
    how="inner",
    suffixes=("_xgb", "_lr"),
    validate="one_to_one",
)

if len(comparison_18) != EXPECTED_TOTAL_ROWS_18:
    raise RuntimeError(
        "XGBoost-LR paired comparison did not retain all patients."
    )

if int(
    (
        comparison_18["outer_fold_xgb"]
        != comparison_18["outer_fold_lr"]
    ).sum()
) != 0:
    raise RuntimeError(
        "XGBoost and LR outer-fold assignments do not match."
    )

if int(
    (
        comparison_18["label_stage23_xgb"]
        != comparison_18["label_stage23_lr"]
    ).sum()
) != 0:
    raise RuntimeError(
        "XGBoost and LR outcomes do not match."
    )

comparison_18 = comparison_18.rename(
    columns={
        "outer_fold_xgb": "outer_fold",
        "label_stage23_xgb": "label_stage23",
    }
).drop(
    columns=[
        "outer_fold_lr",
        "label_stage23_lr",
    ]
)

if "group_hospital_lr" in comparison_18.columns:
    if int(
        (
            comparison_18["group_hospital"].astype(str)
            != comparison_18["group_hospital_lr"].astype(str)
        ).sum()
    ) != 0:
        raise RuntimeError(
            "XGBoost and LR hospital IDs do not match."
        )
    comparison_18 = comparison_18.drop(
        columns=["group_hospital_lr"]
    )

# ------------------------------------------------------------
# 15. Point comparison: XGBoost vs logistic regression
# ------------------------------------------------------------

y_cmp_18 = comparison_18[
    "label_stage23"
].to_numpy(dtype=np.int8)

comparison_point_rows_18 = []

for probability_type, xgb_col, lr_col in [
    (
        "raw",
        "xgb_prediction_raw",
        "lr_prediction_raw",
    ),
    (
        "platt_calibrated",
        "xgb_prediction_platt",
        "lr_prediction_platt",
    ),
]:
    xgb_metrics = probability_metrics_18(
        y_cmp_18,
        comparison_18[xgb_col].to_numpy(dtype=float),
    )
    lr_metrics = probability_metrics_18(
        y_cmp_18,
        comparison_18[lr_col].to_numpy(dtype=float),
    )

    comparison_point_rows_18.append({
        "probability_type": probability_type,
        "patients": EXPECTED_TOTAL_ROWS_18,
        "hospitals": EXPECTED_TOTAL_HOSPITALS_18,
        "events": EXPECTED_TOTAL_EVENTS_18,
        "xgb_auroc": xgb_metrics["auroc"],
        "lr_auroc": lr_metrics["auroc"],
        "delta_auroc_xgb_minus_lr": (
            xgb_metrics["auroc"] - lr_metrics["auroc"]
        ),
        "xgb_auprc": xgb_metrics["auprc"],
        "lr_auprc": lr_metrics["auprc"],
        "delta_auprc_xgb_minus_lr": (
            xgb_metrics["auprc"] - lr_metrics["auprc"]
        ),
        "xgb_brier": xgb_metrics["brier"],
        "lr_brier": lr_metrics["brier"],
        "brier_improvement_lr_minus_xgb": (
            lr_metrics["brier"] - xgb_metrics["brier"]
        ),
        "xgb_log_loss": xgb_metrics["log_loss"],
        "lr_log_loss": lr_metrics["log_loss"],
        "log_loss_improvement_lr_minus_xgb": (
            lr_metrics["log_loss"] - xgb_metrics["log_loss"]
        ),
        "xgb_calibration_intercept": (
            xgb_metrics["calibration_intercept"]
        ),
        "lr_calibration_intercept": (
            lr_metrics["calibration_intercept"]
        ),
        "xgb_calibration_slope": (
            xgb_metrics["calibration_slope"]
        ),
        "lr_calibration_slope": (
            lr_metrics["calibration_slope"]
        ),
    })

model_comparison_point_metrics_18 = pd.DataFrame(
    comparison_point_rows_18
)

# Fold-by-fold raw discrimination differences.
fold_comparison_rows_18 = []

for fold_number in range(1, 6):
    part = comparison_18.loc[
        comparison_18["outer_fold"] == fold_number
    ].copy()

    y_fold = part["label_stage23"].to_numpy(dtype=np.int8)

    for probability_type, xgb_col, lr_col in [
        (
            "raw",
            "xgb_prediction_raw",
            "lr_prediction_raw",
        ),
        (
            "platt_calibrated",
            "xgb_prediction_platt",
            "lr_prediction_platt",
        ),
    ]:
        xgb_m = probability_metrics_18(
            y_fold,
            part[xgb_col].to_numpy(dtype=float),
        )
        lr_m = probability_metrics_18(
            y_fold,
            part[lr_col].to_numpy(dtype=float),
        )

        fold_comparison_rows_18.append({
            "outer_fold": fold_number,
            "probability_type": probability_type,
            "patients": len(part),
            "events": int(y_fold.sum()),
            "xgb_auroc": xgb_m["auroc"],
            "lr_auroc": lr_m["auroc"],
            "delta_auroc_xgb_minus_lr": (
                xgb_m["auroc"] - lr_m["auroc"]
            ),
            "xgb_auprc": xgb_m["auprc"],
            "lr_auprc": lr_m["auprc"],
            "delta_auprc_xgb_minus_lr": (
                xgb_m["auprc"] - lr_m["auprc"]
            ),
            "xgb_brier": xgb_m["brier"],
            "lr_brier": lr_m["brier"],
            "brier_improvement_lr_minus_xgb": (
                lr_m["brier"] - xgb_m["brier"]
            ),
            "xgb_log_loss": xgb_m["log_loss"],
            "lr_log_loss": lr_m["log_loss"],
            "log_loss_improvement_lr_minus_xgb": (
                lr_m["log_loss"] - xgb_m["log_loss"]
            ),
        })

model_comparison_fold_metrics_18 = pd.DataFrame(
    fold_comparison_rows_18
)

# ------------------------------------------------------------
# 16. Paired hospital-cluster bootstrap differences
# ------------------------------------------------------------

print(
    f"\nRunning {BOOTSTRAP_REPLICATES_18:,} paired hospital-cluster "
    "bootstrap replicates for XGBoost vs logistic regression..."
)

cmp_hospital_categorical_18 = pd.Categorical(
    comparison_18["group_hospital"]
)

cmp_hospital_codes_18 = (
    cmp_hospital_categorical_18.codes.astype(int)
)

cmp_hospital_labels_18 = list(
    cmp_hospital_categorical_18.categories
)

if len(cmp_hospital_labels_18) != EXPECTED_TOTAL_HOSPITALS_18:
    raise RuntimeError(
        "Paired comparison hospital count is not 198."
    )

paired_rng_18 = np.random.default_rng(
    BOOTSTRAP_SEED_18
)

xgb_raw_cmp_18 = comparison_18[
    "xgb_prediction_raw"
].to_numpy(dtype=float)
xgb_platt_cmp_18 = comparison_18[
    "xgb_prediction_platt"
].to_numpy(dtype=float)
lr_raw_cmp_18 = comparison_18[
    "lr_prediction_raw"
].to_numpy(dtype=float)
lr_platt_cmp_18 = comparison_18[
    "lr_prediction_platt"
].to_numpy(dtype=float)

paired_rows_18 = []

for bootstrap_index in range(
    BOOTSTRAP_REPLICATES_18
):
    sampled_codes = paired_rng_18.integers(
        low=0,
        high=EXPECTED_TOTAL_HOSPITALS_18,
        size=EXPECTED_TOTAL_HOSPITALS_18,
    )

    multiplicity = np.bincount(
        sampled_codes,
        minlength=EXPECTED_TOTAL_HOSPITALS_18,
    )

    weights = multiplicity[
        cmp_hospital_codes_18
    ].astype(float)

    positive_weight = float(
        weights[y_cmp_18 == 1].sum()
    )
    negative_weight = float(
        weights[y_cmp_18 == 0].sum()
    )

    if positive_weight <= 0 or negative_weight <= 0:
        continue

    row = {
        "bootstrap_replicate": bootstrap_index + 1,
        "unique_sampled_hospitals": int(
            np.count_nonzero(multiplicity)
        ),
        "weighted_patients": float(weights.sum()),
        "weighted_events": positive_weight,
    }

    for probability_type, xgb_vec, lr_vec in [
        ("raw", xgb_raw_cmp_18, lr_raw_cmp_18),
        (
            "platt_calibrated",
            xgb_platt_cmp_18,
            lr_platt_cmp_18,
        ),
    ]:
        xgb_m = weighted_metric_bundle_18(
            y_cmp_18,
            xgb_vec,
            weights,
        )
        lr_m = weighted_metric_bundle_18(
            y_cmp_18,
            lr_vec,
            weights,
        )

        suffix = (
            "raw"
            if probability_type == "raw"
            else "platt"
        )

        row[
            f"delta_auroc_xgb_minus_lr_{suffix}"
        ] = (
            xgb_m["auroc"] - lr_m["auroc"]
        )
        row[
            f"delta_auprc_xgb_minus_lr_{suffix}"
        ] = (
            xgb_m["auprc"] - lr_m["auprc"]
        )
        row[
            f"brier_improvement_lr_minus_xgb_{suffix}"
        ] = (
            lr_m["brier"] - xgb_m["brier"]
        )
        row[
            f"log_loss_improvement_lr_minus_xgb_{suffix}"
        ] = (
            lr_m["log_loss"] - xgb_m["log_loss"]
        )

    paired_rows_18.append(row)

    if (
        bootstrap_index == 0
        or (bootstrap_index + 1) % 100 == 0
    ):
        print(
            "  Completed paired comparison bootstrap replicate",
            bootstrap_index + 1,
            "/",
            BOOTSTRAP_REPLICATES_18,
        )

paired_bootstrap_replicates_18 = pd.DataFrame(
    paired_rows_18
)

if len(paired_bootstrap_replicates_18) < (
    BOOTSTRAP_REPLICATES_18 * 0.99
):
    raise RuntimeError(
        "Fewer than 99% of paired bootstrap replicates completed."
    )

paired_ci_rows_18 = []

paired_specs_18 = [
    (
        "delta_auroc_xgb_minus_lr",
        "raw",
        "delta_auroc_xgb_minus_lr_raw",
    ),
    (
        "delta_auroc_xgb_minus_lr",
        "platt_calibrated",
        "delta_auroc_xgb_minus_lr_platt",
    ),
    (
        "delta_auprc_xgb_minus_lr",
        "raw",
        "delta_auprc_xgb_minus_lr_raw",
    ),
    (
        "delta_auprc_xgb_minus_lr",
        "platt_calibrated",
        "delta_auprc_xgb_minus_lr_platt",
    ),
    (
        "brier_improvement_lr_minus_xgb",
        "raw",
        "brier_improvement_lr_minus_xgb_raw",
    ),
    (
        "brier_improvement_lr_minus_xgb",
        "platt_calibrated",
        "brier_improvement_lr_minus_xgb_platt",
    ),
    (
        "log_loss_improvement_lr_minus_xgb",
        "raw",
        "log_loss_improvement_lr_minus_xgb_raw",
    ),
    (
        "log_loss_improvement_lr_minus_xgb",
        "platt_calibrated",
        "log_loss_improvement_lr_minus_xgb_platt",
    ),
]

point_cmp_lookup_18 = {
    row["probability_type"]: row
    for _, row in model_comparison_point_metrics_18.iterrows()
}

for metric_name, probability_type, column in paired_specs_18:
    values = paired_bootstrap_replicates_18[
        column
    ].dropna().to_numpy(dtype=float)

    point_estimate = float(
        point_cmp_lookup_18[
            probability_type
        ][metric_name]
    )

    paired_ci_rows_18.append({
        "metric": metric_name,
        "probability_type": probability_type,
        "point_estimate": point_estimate,
        "bootstrap_replicates": len(values),
        "bootstrap_mean": float(np.mean(values)),
        "bootstrap_standard_error": float(
            np.std(values, ddof=1)
        ),
        "ci_95_lower": float(
            np.quantile(values, 0.025)
        ),
        "ci_95_upper": float(
            np.quantile(values, 0.975)
        ),
        "probability_improvement_direction": (
            "positive_favors_xgboost"
        ),
        "bootstrap_unit": "hospital",
        "paired_resampling": True,
        "bootstrap_seed": BOOTSTRAP_SEED_18,
    })

paired_model_comparison_ci_18 = pd.DataFrame(
    paired_ci_rows_18
)

# ------------------------------------------------------------
# 17. Save aggregate outputs only
# ------------------------------------------------------------

paths_18 = {
    "source_schema": os.path.join(
        MODEL_OUTPUT_DIR,
        "18A_xgboost_source_prediction_schema_audit.csv",
    ),
    "fold_integrity": os.path.join(
        MODEL_OUTPUT_DIR,
        "18A_xgboost_five_fold_prediction_integrity.csv",
    ),
    "selected_models": os.path.join(
        MODEL_OUTPUT_DIR,
        "18A_xgboost_selected_model_consistency.csv",
    ),
    "fold_metrics": os.path.join(
        MODEL_OUTPUT_DIR,
        "18B_xgboost_outer_fold_metrics.csv",
    ),
    "macro_summary": os.path.join(
        MODEL_OUTPUT_DIR,
        "18B_xgboost_macro_fold_summary.csv",
    ),
    "pooled_point": os.path.join(
        MODEL_OUTPUT_DIR,
        "18B_xgboost_pooled_point_metrics.csv",
    ),
    "xgb_bootstrap_replicates": os.path.join(
        MODEL_OUTPUT_DIR,
        "18C_xgboost_hospital_bootstrap_replicates.csv",
    ),
    "xgb_bootstrap_ci": os.path.join(
        MODEL_OUTPUT_DIR,
        "18C_xgboost_hospital_bootstrap_confidence_intervals.csv",
    ),
    "calibration_deciles": os.path.join(
        MODEL_OUTPUT_DIR,
        "18D_xgboost_pooled_calibration_deciles.csv",
    ),
    "comparison_fold": os.path.join(
        MODEL_OUTPUT_DIR,
        "18E_xgboost_vs_logistic_fold_metrics.csv",
    ),
    "comparison_point": os.path.join(
        MODEL_OUTPUT_DIR,
        "18E_xgboost_vs_logistic_pooled_point_differences.csv",
    ),
    "paired_bootstrap_replicates": os.path.join(
        MODEL_OUTPUT_DIR,
        "18F_xgboost_vs_logistic_paired_hospital_bootstrap_replicates.csv",
    ),
    "paired_bootstrap_ci": os.path.join(
        MODEL_OUTPUT_DIR,
        "18F_xgboost_vs_logistic_paired_hospital_bootstrap_confidence_intervals.csv",
    ),
}

source_schema_summary_18.to_csv(
    paths_18["source_schema"],
    index=False,
)
fold_integrity_summary_18.to_csv(
    paths_18["fold_integrity"],
    index=False,
)
selected_model_consistency_18.to_csv(
    paths_18["selected_models"],
    index=False,
)
xgb_fold_metrics_18.to_csv(
    paths_18["fold_metrics"],
    index=False,
)
xgb_macro_summary_18.to_csv(
    paths_18["macro_summary"],
    index=False,
)
xgb_pooled_point_metrics_18.to_csv(
    paths_18["pooled_point"],
    index=False,
)
xgb_bootstrap_replicates_18.to_csv(
    paths_18["xgb_bootstrap_replicates"],
    index=False,
)
xgb_bootstrap_ci_18.to_csv(
    paths_18["xgb_bootstrap_ci"],
    index=False,
)
xgb_calibration_deciles_18.to_csv(
    paths_18["calibration_deciles"],
    index=False,
)
model_comparison_fold_metrics_18.to_csv(
    paths_18["comparison_fold"],
    index=False,
)
model_comparison_point_metrics_18.to_csv(
    paths_18["comparison_point"],
    index=False,
)
paired_bootstrap_replicates_18.to_csv(
    paths_18["paired_bootstrap_replicates"],
    index=False,
)
paired_model_comparison_ci_18.to_csv(
    paths_18["paired_bootstrap_ci"],
    index=False,
)

# ------------------------------------------------------------
# 18. Final manifest
# ------------------------------------------------------------

manifest_18 = {
    "analysis_version": "18",
    "analysis_type": (
        "five_fold_pooled_xgboost_and_paired_logistic_comparison"
    ),
    "patients": EXPECTED_TOTAL_ROWS_18,
    "hospitals": EXPECTED_TOTAL_HOSPITALS_18,
    "events": EXPECTED_TOTAL_EVENTS_18,
    "nonevents": EXPECTED_TOTAL_NONEVENTS_18,
    "outer_folds": 5,
    "xgboost_protocol_sha256": EXPECTED_XGB_PROTOCOL_SHA_18,
    "xgboost_secure_pooled_prediction_table": (
        xgb_pooled_table_id_18
    ),
    "logistic_secure_pooled_prediction_table": (
        lr_pooled_table_id_18
    ),
    "bootstrap_replicates": BOOTSTRAP_REPLICATES_18,
    "bootstrap_unit": "hospital",
    "bootstrap_seed": BOOTSTRAP_SEED_18,
    "paired_model_comparison": True,
    "patient_level_file_written_to_drive": False,
    "patient_level_xgboost_predictions_written_to_bigquery": True,
    "outputs": paths_18,
}

manifest_path_18 = os.path.join(
    MODEL_OUTPUT_DIR,
    "18G_final_xgboost_and_logistic_comparison_manifest.json",
)
manifest_sha_path_18 = os.path.join(
    MODEL_OUTPUT_DIR,
    "18G_final_xgboost_and_logistic_comparison_manifest_SHA256.txt",
)

with open(
    manifest_path_18,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        manifest_18,
        fh,
        indent=2,
        ensure_ascii=False,
    )

with open(
    manifest_path_18,
    "rb",
) as fh:
    manifest_sha_18 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    manifest_sha_path_18,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(manifest_sha_18 + "\n")

# ------------------------------------------------------------
# 19. Display final audit
# ------------------------------------------------------------

print("\n18 XGBOOST FIVE-FOLD PREDICTION INTEGRITY")
display(fold_integrity_summary_18)

print("\n18 XGBOOST SELECTED MODEL CONSISTENCY")
display(selected_model_consistency_18)

print("\n18 XGBOOST OUTER-FOLD METRICS")
display(xgb_fold_metrics_18)

print("\n18 XGBOOST MACRO FOLD SUMMARY")
display(xgb_macro_summary_18)

print("\n18 XGBOOST PRIMARY POOLED POINT METRICS")
display(xgb_pooled_point_metrics_18)

print("\n18 XGBOOST HOSPITAL-CLUSTER BOOTSTRAP CIs")
display(xgb_bootstrap_ci_18)

print("\n18 XGBOOST VS LOGISTIC FOLD-BY-FOLD COMPARISON")
display(model_comparison_fold_metrics_18)

print("\n18 XGBOOST VS LOGISTIC POOLED POINT DIFFERENCES")
display(model_comparison_point_metrics_18)

print("\n18 XGBOOST VS LOGISTIC PAIRED HOSPITAL-BOOTSTRAP CIs")
display(paired_model_comparison_ci_18)

print("\n18 XGBOOST POOLED BIGQUERY VERIFICATION")
display(xgb_bigquery_verification_18)

print("\nFinal XGBoost/comparison manifest SHA-256:")
print(manifest_sha_18)

print("\nSaved aggregate outputs:")
for path in paths_18.values():
    print(path)
print(manifest_path_18)
print(manifest_sha_path_18)

print(
    "\n18 PASS: Five locked XGBoost outer-test folds were pooled, "
    "evaluated, and compared with elastic-net logistic regression "
    "using paired hospital-cluster bootstrap resampling."
)
print(
    "Patient-level prediction data remained only in BigQuery/RAM. "
    "No patient-level prediction file was written to Google Drive."
)

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

from sklearn.ensemble import RandomForestClassifier

from IPython.display import display

print("STARTING RANDOM FOREST OUTER FOLD 1 — CODE VERSION 19")

# ============================================================
# 19 — RANDOM FOREST OUTER FOLD 1 COMPLETE NESTED MODELLING
#
# Design:
# - Same locked hospital-disjoint outer folds.
# - Same locked hospital-disjoint inner folds.
# - Core feature set only.
# - No SMOTE.
# - No class weighting.
# - Fixed Random Forest candidate grid locked before outer-test evaluation.
# - Selection: pooled inner-OOF AUPRC descending,
#              AUROC descending, Brier ascending.
# - Platt calibration learned only from selected candidate's
#   pooled inner-OOF predictions.
# - Patient-level predictions stored only in BigQuery.
# ============================================================

OUTER_FOLD_19 = 1
MODEL_RANDOM_SEED_19 = 20260721

EXPECTED_SPLIT_19 = {
    "training_rows": 46803,
    "test_rows": 11688,
    "training_hospitals": 158,
    "test_hospitals": 40,
    "training_events": 2426,
    "test_events": 606,
}

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_objects_19 = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_19 = [
    name for name in required_objects_19
    if name not in globals()
]

if missing_objects_19:
    raise RuntimeError(
        "Missing runtime objects: "
        + ", ".join(missing_objects_19)
        + ". Run 07A and 07B first."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"Expected 58,491 cohort rows; found {len(core_df_07B)}."
    )

if len(predictor_columns_07B) != 159:
    raise RuntimeError("Expected 159 core predictors.")

if len(numeric_columns_07B) != 156:
    raise RuntimeError("Expected 156 numeric predictors.")

if len(categorical_columns_07B) != 3:
    raise RuntimeError("Expected 3 categorical predictors.")

# ------------------------------------------------------------
# 2. Lock the Random Forest protocol BEFORE test evaluation
# ------------------------------------------------------------

candidate_grid_19 = [
    {
        "candidate_id": "RF01",
        "n_estimators": 300,
        "max_depth": 8,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "max_samples": 0.80,
    },
    {
        "candidate_id": "RF02",
        "n_estimators": 400,
        "max_depth": 12,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
        "max_samples": 0.80,
    },
    {
        "candidate_id": "RF03",
        "n_estimators": 500,
        "max_depth": None,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
        "max_samples": 0.85,
    },
    {
        "candidate_id": "RF04",
        "n_estimators": 400,
        "max_depth": 12,
        "min_samples_leaf": 10,
        "max_features": 0.25,
        "max_samples": 0.85,
    },
    {
        "candidate_id": "RF05",
        "n_estimators": 500,
        "max_depth": None,
        "min_samples_leaf": 10,
        "max_features": 0.25,
        "max_samples": 0.90,
    },
    {
        "candidate_id": "RF06",
        "n_estimators": 600,
        "max_depth": 16,
        "min_samples_leaf": 20,
        "max_features": 0.50,
        "max_samples": 0.90,
    },
]

rf_protocol_19 = {
    "protocol_name": "random_forest_core_nested_hospital_cv_v1",
    "model_family": "RandomForestClassifier",
    "feature_set": "core_159",
    "outer_cv": "locked 5-fold hospital-disjoint outer folds",
    "inner_cv": "locked 5-fold hospital-disjoint inner folds",
    "primary_selection_metric": "pooled inner OOF AUPRC descending",
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "criterion": "gini",
    "bootstrap": True,
    "class_weighting": False,
    "class_weight": None,
    "smote": False,
    "min_samples_split": 2,
    "random_state": MODEL_RANDOM_SEED_19,
    "candidate_grid": candidate_grid_19,
    "calibration": (
        "Platt calibration fit only on selected candidate "
        "pooled inner-OOF logits"
    ),
    "outer_test_use": (
        "diagnostic evaluation only; never used for tuning"
    ),
}

protocol_path_19 = os.path.join(
    MODEL_OUTPUT_DIR,
    "19A_locked_random_forest_model_protocol_v1.json",
)

protocol_sha_path_19 = os.path.join(
    MODEL_OUTPUT_DIR,
    "19A_locked_random_forest_model_protocol_v1_SHA256.txt",
)

protocol_text_19 = json.dumps(
    rf_protocol_19,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
)

protocol_sha_19 = hashlib.sha256(
    protocol_text_19.encode("utf-8")
).hexdigest()

if os.path.exists(protocol_path_19):
    with open(protocol_path_19, "r", encoding="utf-8") as fh:
        existing_protocol_text_19 = fh.read()
    existing_protocol_sha_19 = hashlib.sha256(
        existing_protocol_text_19.encode("utf-8")
    ).hexdigest()

    if existing_protocol_sha_19 != protocol_sha_19:
        raise RuntimeError(
            "An existing Random Forest protocol file differs from "
            "the currently locked protocol. Stop and audit."
        )
else:
    with open(protocol_path_19, "w", encoding="utf-8") as fh:
        fh.write(protocol_text_19)

with open(protocol_sha_path_19, "w", encoding="utf-8") as fh:
    fh.write(protocol_sha_19 + "\n")

print("Locked Random Forest protocol SHA-256:")
print(protocol_sha_19)

# ------------------------------------------------------------
# 3. Load locked inner hospital map
# ------------------------------------------------------------

inner_mapping_path_19 = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_19):
    raise FileNotFoundError(
        "Locked inner-fold map not found: "
        + inner_mapping_path_19
    )

inner_mapping_all_19 = pd.read_csv(
    inner_mapping_path_19,
    dtype={"group_hospital": str},
)

inner_mapping_part_19 = (
    inner_mapping_all_19.loc[
        inner_mapping_all_19["outer_fold"].astype(int)
        == OUTER_FOLD_19,
        ["group_hospital", "inner_fold"],
    ]
    .copy()
)

inner_mapping_part_19["group_hospital"] = (
    inner_mapping_part_19["group_hospital"].astype(str)
)
inner_mapping_part_19["inner_fold"] = (
    inner_mapping_part_19["inner_fold"].astype(int)
)

if len(inner_mapping_part_19) != 158:
    raise RuntimeError(
        "Expected 158 outer-fold-1 training hospitals "
        "in the locked inner map."
    )

if inner_mapping_part_19["group_hospital"].duplicated().any():
    raise RuntimeError("Duplicate hospital in locked inner map.")

hospital_to_inner_fold_19 = dict(
    zip(
        inner_mapping_part_19["group_hospital"],
        inner_mapping_part_19["inner_fold"],
    )
)

# ------------------------------------------------------------
# 4. Prepare outer fold 1 matrices
# ------------------------------------------------------------

X_all_19 = core_df_07B[predictor_columns_07B].copy()

for column in numeric_columns_07B:
    X_all_19[column] = pd.to_numeric(
        X_all_19[column],
        errors="coerce",
    ).astype("float64")

for column in categorical_columns_07B:
    category_series = X_all_19[column].astype("object")
    X_all_19[column] = category_series.where(
        pd.notna(category_series),
        np.nan,
    )

outer_fold_vector_19 = (
    core_df_07B["outer_fold"].astype(int).to_numpy()
)

outer_training_mask_19 = (
    outer_fold_vector_19 != OUTER_FOLD_19
)
outer_test_mask_19 = (
    outer_fold_vector_19 == OUTER_FOLD_19
)

X_outer_training_19 = (
    X_all_19.loc[outer_training_mask_19]
    .reset_index(drop=True)
)

X_outer_test_19 = (
    X_all_19.loc[outer_test_mask_19]
    .reset_index(drop=True)
)

outer_training_meta_19 = (
    core_df_07B.loc[
        outer_training_mask_19,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_19 = (
    core_df_07B.loc[
        outer_test_mask_19,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [
    outer_training_meta_19,
    outer_test_meta_19,
]:
    dataframe["id_row"] = dataframe["id_row"].astype(str)
    dataframe["group_hospital"] = (
        dataframe["group_hospital"].astype(str)
    )
    dataframe["label_stage23"] = (
        dataframe["label_stage23"].astype(int)
    )

y_outer_training_19 = (
    outer_training_meta_19["label_stage23"]
    .to_numpy(dtype=np.int8)
)
y_outer_test_19 = (
    outer_test_meta_19["label_stage23"]
    .to_numpy(dtype=np.int8)
)

groups_outer_training_19 = (
    outer_training_meta_19["group_hospital"]
    .to_numpy(dtype=str)
)

training_hospitals_19 = set(
    outer_training_meta_19["group_hospital"]
)
test_hospitals_19 = set(
    outer_test_meta_19["group_hospital"]
)
hospital_overlap_19 = (
    training_hospitals_19 & test_hospitals_19
)

if hospital_overlap_19:
    raise RuntimeError(
        "Outer training/test hospital overlap detected."
    )

actual_split_19 = {
    "training_rows": len(X_outer_training_19),
    "test_rows": len(X_outer_test_19),
    "training_hospitals": len(training_hospitals_19),
    "test_hospitals": len(test_hospitals_19),
    "training_events": int(y_outer_training_19.sum()),
    "test_events": int(y_outer_test_19.sum()),
}

for metric, expected_value in EXPECTED_SPLIT_19.items():
    actual_value = actual_split_19[metric]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: found={actual_value}, "
            f"expected={expected_value}"
        )

inner_fold_vector_19 = np.array(
    [
        hospital_to_inner_fold_19.get(hospital, -1)
        for hospital in groups_outer_training_19
    ],
    dtype=int,
)

if (inner_fold_vector_19 == -1).any():
    raise RuntimeError(
        "Some outer-training hospitals have no inner-fold assignment."
    )

if set(np.unique(inner_fold_vector_19)) != {1, 2, 3, 4, 5}:
    raise RuntimeError("Inner-fold values are not exactly 1–5.")

# ------------------------------------------------------------
# 5. Preprocessor and model constructors
# ------------------------------------------------------------

def make_preprocessor_19():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns_07B,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns_07B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_random_forest_model_19(candidate):
    return RandomForestClassifier(
        n_estimators=int(candidate["n_estimators"]),
        criterion="gini",
        max_depth=(
            None
            if candidate["max_depth"] is None
            else int(candidate["max_depth"])
        ),
        min_samples_split=2,
        min_samples_leaf=int(candidate["min_samples_leaf"]),
        max_features=candidate["max_features"],
        bootstrap=True,
        max_samples=float(candidate["max_samples"]),
        class_weight=None,
        random_state=MODEL_RANDOM_SEED_19,
        n_jobs=-1,
        verbose=0,
    )


def checkpoint_table_id_19(inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_random_forest_inner_oof_outer1_inner{inner_fold}_v1"
    )

# ------------------------------------------------------------
# 6. BigQuery checkpoint verification
# ------------------------------------------------------------

def verify_checkpoint_19(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):
    table_id = checkpoint_table_id_19(inner_fold)

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS distinct_id_count,
      COUNT(DISTINCT candidate_id) AS candidate_count,
      COUNT(DISTINCT outer_fold) AS outer_fold_count,
      COUNT(DISTINCT inner_fold) AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(candidate_id, '|', id_row)
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(
        prediction_raw < 0 OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(
        sql,
        location=BQ_LOCATION,
    ).to_dataframe()

    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows * len(candidate_grid_19)
    )
    expected_positive_rows = (
        expected_validation_events * len(candidate_grid_19)
    )
    expected_negative_rows = (
        (expected_validation_rows - expected_validation_events)
        * len(candidate_grid_19)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": expected_validation_rows,
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": expected_total_rows,
        "positive_prediction_rows": expected_positive_rows,
        "negative_prediction_rows": expected_negative_rows,
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": 1,
        "maximum_outer_fold": 1,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failures = []

    for field, expected_value in expected_values.items():
        actual_value = int(row[field])
        if actual_value != expected_value:
            complete = False
            failures.append(
                f"{field}={actual_value}, expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failures),
        "check": check,
        "row": row,
    }

checkpoint_load_config_19 = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField(
            "id_row", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "outer_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "inner_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "candidate_id", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "label_stage23", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "prediction_raw", "FLOAT", mode="REQUIRED"
        ),
    ],
    write_disposition=(
        bigquery.WriteDisposition.WRITE_TRUNCATE
    ),
)

# ------------------------------------------------------------
# 7. Aggregate fit audit
# ------------------------------------------------------------

fit_audit_columns_19 = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "n_estimators",
    "max_depth",
    "min_samples_leaf",
    "max_features",
    "max_samples",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "fit_seconds",
]

fit_audit_path_19 = os.path.join(
    MODEL_OUTPUT_DIR,
    "19B_random_forest_inner_fit_audit_outer1.csv",
)

if os.path.exists(fit_audit_path_19):
    fit_audit_19 = pd.read_csv(fit_audit_path_19)
else:
    fit_audit_19 = pd.DataFrame(
        columns=fit_audit_columns_19
    )

for column in fit_audit_columns_19:
    if column not in fit_audit_19.columns:
        fit_audit_19[column] = np.nan

fit_audit_19 = fit_audit_19[
    fit_audit_columns_19
].copy()

# ------------------------------------------------------------
# 8. Run five inner folds
# ------------------------------------------------------------

for inner_fold in range(1, 6):
    inner_training_mask = (
        inner_fold_vector_19 != inner_fold
    )
    inner_validation_mask = (
        inner_fold_vector_19 == inner_fold
    )

    training_rows = int(inner_training_mask.sum())
    validation_rows = int(inner_validation_mask.sum())
    training_events = int(
        y_outer_training_19[inner_training_mask].sum()
    )
    validation_events = int(
        y_outer_training_19[inner_validation_mask].sum()
    )

    existing_check = verify_checkpoint_19(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if existing_check["complete"]:
        print(
            f"Outer 1 / inner {inner_fold}: "
            "permanent Random Forest checkpoint already complete; "
            "skipping model fitting."
        )
        continue

    training_hospital_set = set(
        groups_outer_training_19[inner_training_mask]
    )
    validation_hospital_set = set(
        groups_outer_training_19[inner_validation_mask]
    )

    if training_hospital_set & validation_hospital_set:
        raise RuntimeError(
            f"Inner fold {inner_fold}: hospital overlap detected."
        )

    print(f"\nOuter 1 / inner {inner_fold}")
    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_19()

    preprocessing_started = time.time()

    X_inner_training_processed = (
        preprocessor.fit_transform(
            X_outer_training_19.loc[
                inner_training_mask
            ]
        )
    )

    X_inner_validation_processed = (
        preprocessor.transform(
            X_outer_training_19.loc[
                inner_validation_mask
            ]
        )
    )

    preprocessing_seconds = (
        time.time() - preprocessing_started
    )

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "Processed training/validation column counts differ."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_seconds, 2),
    )

    y_inner_training = (
        y_outer_training_19[inner_training_mask]
    )
    y_inner_validation = (
        y_outer_training_19[inner_validation_mask]
    )

    validation_ids = (
        outer_training_meta_19.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_19:
        candidate_id = candidate["candidate_id"]

        print(
            "  Fitting",
            candidate_id,
            "| trees =",
            candidate["n_estimators"],
            "| depth =",
            candidate["max_depth"],
            "| min_leaf =",
            candidate["min_samples_leaf"],
            "| max_features =",
            candidate["max_features"],
            "| max_samples =",
            candidate["max_samples"],
        )

        model = make_random_forest_model_19(candidate)

        fit_started = time.time()

        model.fit(
            X_inner_training_processed,
            y_inner_training,
        )

        fit_seconds = time.time() - fit_started

        validation_probabilities = (
            model.predict_proba(
                X_inner_validation_processed
            )[:, 1]
        )

        if np.isnan(validation_probabilities).any():
            raise RuntimeError(
                f"{candidate_id}, inner {inner_fold}: "
                "missing predictions."
            )

        if not np.all(
            (validation_probabilities >= 0)
            & (validation_probabilities <= 1)
        ):
            raise RuntimeError(
                f"{candidate_id}: invalid probabilities."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        1,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": (
                        y_inner_validation.astype(np.int64)
                    ),
                    "prediction_raw": (
                        validation_probabilities.astype(
                            np.float64
                        )
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": 1,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "n_estimators": candidate[
                    "n_estimators"
                ],
                "max_depth": candidate[
                    "max_depth"
                ],
                "min_samples_leaf": candidate[
                    "min_samples_leaf"
                ],
                "max_features": candidate[
                    "max_features"
                ],
                "max_samples": candidate[
                    "max_samples"
                ],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": (
                    validation_events
                ),
                "processed_columns": int(
                    X_inner_training_processed.shape[1]
                ),
                "fit_seconds": float(fit_seconds),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows * len(candidate_grid_19)
    )

    if len(checkpoint_df) != expected_checkpoint_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: invalid checkpoint row count."
        )

    if checkpoint_df.duplicated(
        subset=["id_row", "candidate_id"]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: duplicate candidate-patient rows."
        )

    target_checkpoint_table = (
        checkpoint_table_id_19(inner_fold)
    )

    print(
        "Uploading permanent Random Forest checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_19,
        location=BQ_LOCATION,
    ).result()

    if len(fit_audit_19) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_19["outer_fold"],
                    errors="coerce",
                ) == 1
            )
            & (
                pd.to_numeric(
                    fit_audit_19["inner_fold"],
                    errors="coerce",
                ) == inner_fold
            )
        )

        fit_audit_19 = (
            fit_audit_19.loc[keep_mask].copy()
        )

    fit_audit_19 = pd.concat(
        [
            fit_audit_19,
            pd.DataFrame(current_audit_rows),
        ],
        ignore_index=True,
    )

    fit_audit_19 = (
        fit_audit_19[
            fit_audit_columns_19
        ]
        .sort_values(
            [
                "outer_fold",
                "inner_fold",
                "candidate_id",
            ]
        )
        .reset_index(drop=True)
    )

    fit_audit_19.to_csv(
        fit_audit_path_19,
        index=False,
    )

    completed_check = verify_checkpoint_19(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint verification failed: "
            + completed_check["reason"]
        )

    print(
        f"Outer 1 / inner {inner_fold}: "
        "permanent Random Forest checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 9. Final inner checkpoint summary
# ------------------------------------------------------------

checkpoint_summary_rows_19 = []

for inner_fold in range(1, 6):
    validation_mask = (
        inner_fold_vector_19 == inner_fold
    )
    validation_rows = int(validation_mask.sum())
    validation_events = int(
        y_outer_training_19[
            validation_mask
        ].sum()
    )

    final_check = verify_checkpoint_19(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: final checkpoint audit failed. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_19.append(
        {
            "outer_fold": 1,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(
                row["row_count"]
            ),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(
                row["candidate_count"]
            ),
            "positive_prediction_rows": int(
                row["positive_prediction_rows"]
            ),
            "negative_prediction_rows": int(
                row["negative_prediction_rows"]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check["table_id"],
        }
    )

checkpoint_summary_19 = (
    pd.DataFrame(checkpoint_summary_rows_19)
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_19[
        "distinct_validation_patients"
    ].sum()
) != EXPECTED_SPLIT_19["training_rows"]:
    raise RuntimeError(
        "Total inner validation patients != 46,803."
    )

expected_total_oof_rows_19 = (
    EXPECTED_SPLIT_19["training_rows"]
    * len(candidate_grid_19)
)

if int(
    checkpoint_summary_19[
        "checkpoint_rows"
    ].sum()
) != expected_total_oof_rows_19:
    raise RuntimeError(
        "Total Random Forest OOF prediction rows are incorrect."
    )

checkpoint_summary_path_19 = os.path.join(
    MODEL_OUTPUT_DIR,
    "19B_random_forest_outer1_inner_checkpoint_summary.csv",
)

checkpoint_summary_19.to_csv(
    checkpoint_summary_path_19,
    index=False,
)

# ------------------------------------------------------------
# 10. Pool five inner OOF tables
# ------------------------------------------------------------

checkpoint_tables_19 = [
    checkpoint_table_id_19(inner_fold)
    for inner_fold in range(1, 6)
]

union_parts_19 = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_19
]

SQL_LOAD_POOLED_OOF_19 = (
    "\nUNION ALL\n".join(union_parts_19)
)

print(
    "\nLoading pooled outer-fold-1 Random Forest inner OOF predictions..."
)

query_job_19 = client.query(
    SQL_LOAD_POOLED_OOF_19,
    location=BQ_LOCATION,
)

try:
    pooled_oof_19 = query_job_19.to_dataframe(
        create_bqstorage_client=True
    )
    pooled_load_method_19 = (
        "BigQuery Storage API"
    )
except Exception as fast_path_error_19:
    print(
        "Storage API unavailable; using standard BigQuery download."
    )
    print(
        "Message:",
        type(fast_path_error_19).__name__,
    )
    pooled_oof_19 = query_job_19.to_dataframe(
        create_bqstorage_client=False
    )
    pooled_load_method_19 = (
        "Standard BigQuery API"
    )

pooled_oof_19["id_row"] = (
    pooled_oof_19["id_row"].astype(str)
)
pooled_oof_19["candidate_id"] = (
    pooled_oof_19["candidate_id"].astype(str)
)

for column in [
    "outer_fold",
    "inner_fold",
    "label_stage23",
]:
    pooled_oof_19[column] = pd.to_numeric(
        pooled_oof_19[column],
        errors="raise",
    ).astype(int)

pooled_oof_19["prediction_raw"] = pd.to_numeric(
    pooled_oof_19["prediction_raw"],
    errors="raise",
).astype(float)

if len(pooled_oof_19) != expected_total_oof_rows_19:
    raise RuntimeError(
        "Pooled Random Forest OOF row count is incorrect."
    )

if pooled_oof_19.duplicated(
    subset=["candidate_id", "id_row"]
).any():
    raise RuntimeError(
        "Duplicate candidate-patient row in pooled Random Forest OOF."
    )

if pooled_oof_19["prediction_raw"].isna().any():
    raise RuntimeError("Missing Random Forest OOF prediction.")

if not pooled_oof_19[
    "prediction_raw"
].between(0, 1).all():
    raise RuntimeError(
        "Invalid Random Forest OOF probability."
    )

if set(
    pooled_oof_19["candidate_id"].unique()
) != {
    "RF01",
    "RF02",
    "RF03",
    "RF04",
    "RF05",
    "RF06",
}:
    raise RuntimeError(
        "The six locked Random Forest candidates are not all present."
    )

candidate_patient_counts_19 = (
    pooled_oof_19
    .groupby("candidate_id")["id_row"]
    .nunique()
)

if not (
    candidate_patient_counts_19
    == EXPECTED_SPLIT_19["training_rows"]
).all():
    raise RuntimeError(
        "Each candidate must have 46,803 OOF patients."
    )

candidate_event_counts_19 = (
    pooled_oof_19
    .groupby("candidate_id")["label_stage23"]
    .sum()
)

if not (
    candidate_event_counts_19
    == EXPECTED_SPLIT_19["training_events"]
).all():
    raise RuntimeError(
        "Each candidate must have 2,426 OOF events."
    )

# ------------------------------------------------------------
# 11. Metric helpers
# ------------------------------------------------------------

def probability_metrics_19(
    y_true,
    probabilities,
):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(
            roc_auc_score(y_true, probabilities)
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1],
            )
        ),
        "mean_predicted_risk": float(
            probabilities.mean()
        ),
        "observed_event_rate": float(
            np.mean(y_true)
        ),
    }


def probability_logit_19(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        probabilities / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_19(
    y_true,
    probabilities,
):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        probability_logit_19(probabilities),
        y_true,
    )

    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 12. Candidate pooled inner OOF metrics
# ------------------------------------------------------------

candidate_result_rows_19 = []

for candidate in candidate_grid_19:
    candidate_id = candidate["candidate_id"]

    candidate_oof = (
        pooled_oof_19.loc[
            pooled_oof_19["candidate_id"]
            == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_19(
        candidate_oof[
            "label_stage23"
        ].to_numpy(dtype=int),
        candidate_oof[
            "prediction_raw"
        ].to_numpy(dtype=float),
    )

    fit_part = fit_audit_19.loc[
        fit_audit_19[
            "candidate_id"
        ].astype(str) == candidate_id
    ]

    fit_seconds_total = (
        float(
            pd.to_numeric(
                fit_part["fit_seconds"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_19.append(
        {
            "candidate_id": candidate_id,
            **{
                key: candidate[key]
                for key in candidate
                if key != "candidate_id"
            },
            **metrics,
            "fit_seconds_total": (
                fit_seconds_total
            ),
        }
    )

candidate_results_19 = pd.DataFrame(
    candidate_result_rows_19
)

candidate_results_19 = (
    candidate_results_19
    .sort_values(
        ["auprc", "auroc", "brier"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

candidate_results_19["selection_rank"] = (
    np.arange(
        1,
        len(candidate_results_19) + 1,
    )
)

best_row_19 = candidate_results_19.iloc[0]
selected_candidate_id_19 = str(
    best_row_19["candidate_id"]
)

selected_candidate_19 = next(
    candidate
    for candidate in candidate_grid_19
    if candidate["candidate_id"]
    == selected_candidate_id_19
)

# ------------------------------------------------------------
# 13. Platt calibration from selected inner OOF
# ------------------------------------------------------------

selected_oof_19 = (
    pooled_oof_19.loc[
        pooled_oof_19["candidate_id"]
        == selected_candidate_id_19
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_19 = (
    selected_oof_19[
        "label_stage23"
    ].to_numpy(dtype=int)
)
selected_oof_probability_19 = (
    selected_oof_19[
        "prediction_raw"
    ].to_numpy(dtype=float)
)

platt_calibrator_19 = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_19.fit(
    probability_logit_19(
        selected_oof_probability_19
    ),
    selected_oof_y_19,
)

platt_intercept_19 = float(
    platt_calibrator_19.intercept_[0]
)
platt_slope_19 = float(
    platt_calibrator_19.coef_[0][0]
)

if (
    not np.isfinite(platt_intercept_19)
    or not np.isfinite(platt_slope_19)
    or platt_slope_19 <= 0
):
    raise RuntimeError(
        "Invalid Platt calibration coefficients."
    )

selected_model_19 = pd.DataFrame(
    [
        {
            "outer_fold": 1,
            "selected_candidate": (
                selected_candidate_id_19
            ),
            "selection_metric_primary": (
                "pooled_inner_oof_auprc"
            ),
            "inner_oof_auprc": float(
                best_row_19["auprc"]
            ),
            "inner_oof_auroc": float(
                best_row_19["auroc"]
            ),
            "inner_oof_brier": float(
                best_row_19["brier"]
            ),
            "inner_oof_log_loss": float(
                best_row_19["log_loss"]
            ),
            "inner_oof_mean_predicted_risk": float(
                best_row_19[
                    "mean_predicted_risk"
                ]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_19[
                    "observed_event_rate"
                ]
            ),
            "platt_intercept": (
                platt_intercept_19
            ),
            "platt_slope": platt_slope_19,
            "protocol_sha256": (
                protocol_sha_19
            ),
            **{
                key: selected_candidate_19[key]
                for key in selected_candidate_19
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 14. Save locked selection
# ------------------------------------------------------------

candidate_results_path_19 = os.path.join(
    MODEL_OUTPUT_DIR,
    "19C_random_forest_candidate_results_outer1.csv",
)

selected_model_path_19 = os.path.join(
    MODEL_OUTPUT_DIR,
    "19C_random_forest_selected_model_outer1.csv",
)

selection_json_path_19 = os.path.join(
    MODEL_OUTPUT_DIR,
    "19C_random_forest_selection_calibration_outer1.json",
)

selection_sha_path_19 = os.path.join(
    MODEL_OUTPUT_DIR,
    "19C_random_forest_selection_calibration_outer1_SHA256.txt",
)

candidate_results_19.to_csv(
    candidate_results_path_19,
    index=False,
)
selected_model_19.to_csv(
    selected_model_path_19,
    index=False,
)

selection_configuration_19 = {
    "outer_fold": 1,
    "protocol_sha256": protocol_sha_19,
    "selection_metric_primary": (
        "pooled inner out-of-fold AUPRC"
    ),
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "selected_candidate": (
        selected_candidate_id_19
    ),
    "selected_hyperparameters": {
        key: selected_candidate_19[key]
        for key in selected_candidate_19
        if key != "candidate_id"
    },
    "inner_oof_auprc": float(
        best_row_19["auprc"]
    ),
    "inner_oof_auroc": float(
        best_row_19["auroc"]
    ),
    "inner_oof_brier": float(
        best_row_19["brier"]
    ),
    "platt_intercept": platt_intercept_19,
    "platt_slope": platt_slope_19,
    "inner_checkpoint_tables": (
        checkpoint_tables_19
    ),
    "patient_level_oof_written_to_drive": False,
}

with open(
    selection_json_path_19,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        selection_configuration_19,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    selection_json_path_19,
    "rb",
) as fh:
    selection_sha_19 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    selection_sha_path_19,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(selection_sha_19 + "\n")

# ------------------------------------------------------------
# 15. Fit selected model on all outer training patients
# ------------------------------------------------------------

final_preprocessor_19 = (
    make_preprocessor_19()
)

print(
    "\nFitting selected outer-fold-1 Random Forest model "
    "on all 46,803 training patients..."
)

preprocess_started_19 = time.time()

X_outer_training_processed_19 = (
    final_preprocessor_19.fit_transform(
        X_outer_training_19
    )
)
X_outer_test_processed_19 = (
    final_preprocessor_19.transform(
        X_outer_test_19
    )
)

final_preprocessing_seconds_19 = (
    time.time() - preprocess_started_19
)

final_model_19 = make_random_forest_model_19(
    selected_candidate_19
)

final_fit_started_19 = time.time()

final_model_19.fit(
    X_outer_training_processed_19,
    y_outer_training_19,
)

final_fit_seconds_19 = (
    time.time() - final_fit_started_19
)

outer1_raw_probabilities_19 = (
    final_model_19.predict_proba(
        X_outer_test_processed_19
    )[:, 1]
)

raw_clipped_19 = np.clip(
    outer1_raw_probabilities_19,
    1e-6,
    1 - 1e-6,
)
raw_logit_19 = np.log(
    raw_clipped_19
    / (1 - raw_clipped_19)
)

outer1_platt_probabilities_19 = expit(
    platt_intercept_19
    + platt_slope_19 * raw_logit_19
)

for probabilities, name in [
    (outer1_raw_probabilities_19, "raw"),
    (
        outer1_platt_probabilities_19,
        "platt",
    ),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(
            f"{name} test predictions contain missing values."
        )

    if not np.all(
        (probabilities >= 0)
        & (probabilities <= 1)
    ):
        raise RuntimeError(
            f"{name} test predictions contain invalid probabilities."
        )

raw_metrics_19 = probability_metrics_19(
    y_outer_test_19,
    outer1_raw_probabilities_19,
)
platt_metrics_19 = probability_metrics_19(
    y_outer_test_19,
    outer1_platt_probabilities_19,
)

raw_calibration_intercept_19, \
raw_calibration_slope_19 = (
    calibration_intercept_slope_19(
        y_outer_test_19,
        outer1_raw_probabilities_19,
    )
)

platt_calibration_intercept_19, \
platt_calibration_slope_19 = (
    calibration_intercept_slope_19(
        y_outer_test_19,
        outer1_platt_probabilities_19,
    )
)

outer1_test_results_19 = pd.DataFrame(
    [
        {
            "outer_fold": 1,
            "model": "random_forest",
            "probability_type": "raw",
            **raw_metrics_19,
            "calibration_intercept": (
                raw_calibration_intercept_19
            ),
            "calibration_slope": (
                raw_calibration_slope_19
            ),
        },
        {
            "outer_fold": 1,
            "model": "random_forest",
            "probability_type": (
                "platt_calibrated"
            ),
            **platt_metrics_19,
            "calibration_intercept": (
                platt_calibration_intercept_19
            ),
            "calibration_slope": (
                platt_calibration_slope_19
            ),
        },
    ]
)

# ------------------------------------------------------------
# 16. Feature importance
# ------------------------------------------------------------

processed_feature_names_19 = (
    final_preprocessor_19
    .get_feature_names_out()
)

feature_importances_19 = (
    final_model_19.feature_importances_
)

if len(processed_feature_names_19) != len(
    feature_importances_19
):
    raise RuntimeError(
        "Processed feature names and Random Forest "
        "feature importances differ in length."
    )

feature_importance_table_19 = pd.DataFrame(
    {
        "processed_feature": (
            processed_feature_names_19
        ),
        "mdi_importance": (
            feature_importances_19
        ),
    }
)

feature_importance_table_19[
    "importance_rank"
] = (
    feature_importance_table_19[
        "mdi_importance"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)

feature_importance_table_19 = (
    feature_importance_table_19
    .sort_values(
        [
            "mdi_importance",
            "processed_feature",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

nonzero_importance_features_19 = int(
    (
        feature_importance_table_19[
            "mdi_importance"
        ] > 0
    ).sum()
)

final_model_summary_19 = pd.DataFrame(
    [
        {
            "outer_fold": 1,
            "selected_candidate": (
                selected_candidate_id_19
            ),
            "training_patients": len(
                X_outer_training_19
            ),
            "training_hospitals": len(
                training_hospitals_19
            ),
            "training_events": int(
                y_outer_training_19.sum()
            ),
            "test_patients": len(
                X_outer_test_19
            ),
            "test_hospitals": len(
                test_hospitals_19
            ),
            "test_events": int(
                y_outer_test_19.sum()
            ),
            "hospital_overlap": len(
                hospital_overlap_19
            ),
            "processed_feature_columns": len(
                processed_feature_names_19
            ),
            "nonzero_importance_features": (
                nonzero_importance_features_19
            ),
            "preprocessing_seconds": float(
                final_preprocessing_seconds_19
            ),
            "fit_seconds": float(
                final_fit_seconds_19
            ),
            "locked_platt_intercept": (
                platt_intercept_19
            ),
            "locked_platt_slope": (
                platt_slope_19
            ),
            "protocol_sha256": protocol_sha_19,
            "selection_sha256": (
                selection_sha_19
            ),
            **{
                key: selected_candidate_19[key]
                for key in selected_candidate_19
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 17. Secure outer-test prediction checkpoint
# ------------------------------------------------------------

outer1_prediction_df_19 = pd.DataFrame(
    {
        "id_row": (
            outer_test_meta_19[
                "id_row"
            ].astype(str)
        ),
        "outer_fold": np.full(
            len(outer_test_meta_19),
            1,
            dtype=np.int64,
        ),
        "label_stage23": (
            y_outer_test_19.astype(np.int64)
        ),
        "prediction_raw": (
            outer1_raw_probabilities_19.astype(
                np.float64
            )
        ),
        "prediction_platt": (
            outer1_platt_probabilities_19.astype(
                np.float64
            )
        ),
        "model_name": "random_forest",
        "model_version": (
            "core_v1_nested_cv"
        ),
    }
)

if len(outer1_prediction_df_19) != 11688:
    raise RuntimeError(
        "Outer-fold-1 test prediction row count != 11,688."
    )

if outer1_prediction_df_19[
    "id_row"
].duplicated().any():
    raise RuntimeError(
        "Duplicate id_row in outer-fold-1 Random Forest predictions."
    )

if int(
    outer1_prediction_df_19[
        "label_stage23"
    ].sum()
) != 606:
    raise RuntimeError(
        "Outer-fold-1 Random Forest event count != 606."
    )

prediction_table_id_19 = (
    f"{TARGET_DATASET}."
    "model_random_forest_outer_predictions_outer1_v1"
)

prediction_load_config_19 = (
    bigquery.LoadJobConfig(
        schema=[
            bigquery.SchemaField(
                "id_row",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "outer_fold",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "label_stage23",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_raw",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_platt",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_name",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_version",
                "STRING",
                mode="REQUIRED",
            ),
        ],
        write_disposition=(
            bigquery.WriteDisposition.WRITE_TRUNCATE
        ),
    )
)

print(
    "\nUploading secure outer-fold-1 "
    "Random Forest prediction checkpoint:"
)
print(prediction_table_id_19)

client.load_table_from_dataframe(
    outer1_prediction_df_19,
    prediction_table_id_19,
    job_config=prediction_load_config_19,
    location=BQ_LOCATION,
).result()

SQL_VERIFY_PREDICTIONS_19 = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL)
    AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL)
    AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0
    OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0
    OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw)
    AS minimum_raw_probability,
  MAX(prediction_raw)
    AS maximum_raw_probability,
  MIN(prediction_platt)
    AS minimum_platt_probability,
  MAX(prediction_platt)
    AS maximum_platt_probability
FROM `{prediction_table_id_19}`;
"""

prediction_verification_19 = (
    client.query(
        SQL_VERIFY_PREDICTIONS_19,
        location=BQ_LOCATION,
    )
    .to_dataframe()
)

verification_row_19 = (
    prediction_verification_19.iloc[0]
)

expected_prediction_values_19 = {
    "prediction_rows": 11688,
    "distinct_rows": 11688,
    "outer_folds": 1,
    "minimum_outer_fold": 1,
    "maximum_outer_fold": 1,
    "events": 606,
    "nonevents": 11082,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in (
    expected_prediction_values_19.items()
):
    actual_value = int(
        verification_row_19[field]
    )
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: found={actual_value}, "
            f"expected={expected_value}"
        )

# ------------------------------------------------------------
# 18. Save aggregate outputs
# ------------------------------------------------------------

test_results_path_19 = os.path.join(
    MODEL_OUTPUT_DIR,
    "19D_random_forest_outer1_test_results.csv",
)

model_summary_path_19 = os.path.join(
    MODEL_OUTPUT_DIR,
    "19D_random_forest_final_model_outer1.csv",
)

importance_path_19 = os.path.join(
    MODEL_OUTPUT_DIR,
    "19D_random_forest_mdi_importance_outer1.csv",
)

evaluation_json_path_19 = os.path.join(
    MODEL_OUTPUT_DIR,
    "19D_random_forest_final_evaluation_outer1.json",
)

evaluation_sha_path_19 = os.path.join(
    MODEL_OUTPUT_DIR,
    "19D_random_forest_final_evaluation_outer1_SHA256.txt",
)

outer1_test_results_19.to_csv(
    test_results_path_19,
    index=False,
)
final_model_summary_19.to_csv(
    model_summary_path_19,
    index=False,
)
feature_importance_table_19.to_csv(
    importance_path_19,
    index=False,
)

evaluation_configuration_19 = {
    "outer_fold": 1,
    "model_family": "random_forest",
    "protocol_sha256": protocol_sha_19,
    "selection_sha256": selection_sha_19,
    "selected_candidate": (
        selected_candidate_id_19
    ),
    "selected_hyperparameters": {
        key: selected_candidate_19[key]
        for key in selected_candidate_19
        if key != "candidate_id"
    },
    "training_patients": 46803,
    "training_hospitals": 158,
    "test_patients": 11688,
    "test_hospitals": 40,
    "hospital_overlap": 0,
    "locked_platt_intercept": (
        platt_intercept_19
    ),
    "locked_platt_slope": platt_slope_19,
    "processed_feature_columns": int(
        len(processed_feature_names_19)
    ),
    "secure_prediction_table": (
        prediction_table_id_19
    ),
    "patient_level_prediction_written_to_drive": False,
}

with open(
    evaluation_json_path_19,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        evaluation_configuration_19,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    evaluation_json_path_19,
    "rb",
) as fh:
    evaluation_sha_19 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    evaluation_sha_path_19,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(evaluation_sha_19 + "\n")

# ------------------------------------------------------------
# 19. Display results
# ------------------------------------------------------------

pooled_integrity_19 = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_19),
            pooled_oof_19[
                "id_row"
            ].nunique(),
            pooled_oof_19[
                "candidate_id"
            ].nunique(),
            pooled_oof_19[
                "inner_fold"
            ].nunique(),
            EXPECTED_SPLIT_19[
                "training_events"
            ],
            (
                EXPECTED_SPLIT_19[
                    "training_rows"
                ]
                - EXPECTED_SPLIT_19[
                    "training_events"
                ]
            ),
            int(
                pooled_oof_19.duplicated(
                    subset=[
                        "candidate_id",
                        "id_row",
                    ]
                ).sum()
            ),
            int(
                pooled_oof_19[
                    "prediction_raw"
                ].isna().sum()
            ),
            int(
                (
                    ~pooled_oof_19[
                        "prediction_raw"
                    ].between(0, 1)
                ).sum()
            ),
            pooled_load_method_19,
        ],
    }
)

print(
    "\n19 RANDOM FOREST OUTER-FOLD-1 INNER CHECKPOINT SUMMARY"
)
display(checkpoint_summary_19)

print(
    "\n19 RANDOM FOREST OUTER-FOLD-1 POOLED OOF INTEGRITY"
)
display(pooled_integrity_19)

print(
    "\n19 RANDOM FOREST OUTER-FOLD-1 CANDIDATE RESULTS"
)
display(candidate_results_19)

print(
    "\n19 RANDOM FOREST OUTER-FOLD-1 SELECTED MODEL"
)
display(selected_model_19)

print(
    "\n19 RANDOM FOREST OUTER-FOLD-1 FINAL MODEL SUMMARY"
)
display(final_model_summary_19)

print(
    "\n19 RANDOM FOREST OUTER-FOLD-1 TEST RESULTS"
)
display(outer1_test_results_19)

print(
    "\n19 RANDOM FOREST OUTER-FOLD-1 BIGQUERY VERIFICATION"
)
display(prediction_verification_19)

print(
    "\n19 RANDOM FOREST OUTER-FOLD-1 TOP 20 MDI IMPORTANCE FEATURES"
)
display(feature_importance_table_19.head(20))

print("\nRandom Forest protocol SHA-256:")
print(protocol_sha_19)

print("\nSelection SHA-256:")
print(selection_sha_19)

print("\nEvaluation SHA-256:")
print(evaluation_sha_19)

print("\nSaved aggregate outputs:")
print(protocol_path_19)
print(protocol_sha_path_19)
print(fit_audit_path_19)
print(checkpoint_summary_path_19)
print(candidate_results_path_19)
print(selected_model_path_19)
print(selection_json_path_19)
print(selection_sha_path_19)
print(test_results_path_19)
print(model_summary_path_19)
print(importance_path_19)
print(evaluation_json_path_19)
print(evaluation_sha_path_19)

print(
    "\n19 PASS: Random Forest outer-fold-1 nested modelling "
    "and locked test evaluation are complete."
)

print(
    "No class weighting or SMOTE was used."
)

print(
    "All patient-level OOF and outer-test predictions "
    "were stored only in BigQuery."
)

print(
    "No patient-level prediction file was written to Google Drive."
)

_ = gc.collect()

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

from sklearn.ensemble import RandomForestClassifier

from IPython.display import display

print("STARTING RANDOM FOREST OUTER FOLD 2 — CODE VERSION 20")

# ============================================================
# 20 — RANDOM FOREST OUTER FOLD 2 COMPLETE NESTED MODELLING
#
# Design:
# - Same locked hospital-disjoint outer folds.
# - Same locked hospital-disjoint inner folds.
# - Core feature set only.
# - No SMOTE.
# - No class weighting.
# - Fixed Random Forest candidate grid locked before outer-test evaluation.
# - Selection: pooled inner-OOF AUPRC descending,
#              AUROC descending, Brier ascending.
# - Platt calibration learned only from selected candidate's
#   pooled inner-OOF predictions.
# - Patient-level predictions stored only in BigQuery.
# ============================================================

OUTER_FOLD_20 = 2
MODEL_RANDOM_SEED_20 = 20260721

EXPECTED_SPLIT_20 = {
    "training_rows": 46800,
    "test_rows": 11691,
    "training_hospitals": 158,
    "test_hospitals": 40,
    "training_events": 2426,
    "test_events": 606,
}

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_objects_20 = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_20 = [
    name for name in required_objects_20
    if name not in globals()
]

if missing_objects_20:
    raise RuntimeError(
        "Missing runtime objects: "
        + ", ".join(missing_objects_20)
        + ". Run 07A and 07B first."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"Expected 58,491 cohort rows; found {len(core_df_07B)}."
    )

if len(predictor_columns_07B) != 159:
    raise RuntimeError("Expected 159 core predictors.")

if len(numeric_columns_07B) != 156:
    raise RuntimeError("Expected 156 numeric predictors.")

if len(categorical_columns_07B) != 3:
    raise RuntimeError("Expected 3 categorical predictors.")

# ------------------------------------------------------------
# 2. Lock the Random Forest protocol BEFORE test evaluation
# ------------------------------------------------------------

candidate_grid_20 = [
    {
        "candidate_id": "RF01",
        "n_estimators": 300,
        "max_depth": 8,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "max_samples": 0.80,
    },
    {
        "candidate_id": "RF02",
        "n_estimators": 400,
        "max_depth": 12,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
        "max_samples": 0.80,
    },
    {
        "candidate_id": "RF03",
        "n_estimators": 500,
        "max_depth": None,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
        "max_samples": 0.85,
    },
    {
        "candidate_id": "RF04",
        "n_estimators": 400,
        "max_depth": 12,
        "min_samples_leaf": 10,
        "max_features": 0.25,
        "max_samples": 0.85,
    },
    {
        "candidate_id": "RF05",
        "n_estimators": 500,
        "max_depth": None,
        "min_samples_leaf": 10,
        "max_features": 0.25,
        "max_samples": 0.90,
    },
    {
        "candidate_id": "RF06",
        "n_estimators": 600,
        "max_depth": 16,
        "min_samples_leaf": 20,
        "max_features": 0.50,
        "max_samples": 0.90,
    },
]

rf_protocol_20 = {
    "protocol_name": "random_forest_core_nested_hospital_cv_v1",
    "model_family": "RandomForestClassifier",
    "feature_set": "core_159",
    "outer_cv": "locked 5-fold hospital-disjoint outer folds",
    "inner_cv": "locked 5-fold hospital-disjoint inner folds",
    "primary_selection_metric": "pooled inner OOF AUPRC descending",
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "criterion": "gini",
    "bootstrap": True,
    "class_weighting": False,
    "class_weight": None,
    "smote": False,
    "min_samples_split": 2,
    "random_state": MODEL_RANDOM_SEED_20,
    "candidate_grid": candidate_grid_20,
    "calibration": (
        "Platt calibration fit only on selected candidate "
        "pooled inner-OOF logits"
    ),
    "outer_test_use": (
        "diagnostic evaluation only; never used for tuning"
    ),
}

protocol_path_20 = os.path.join(
    MODEL_OUTPUT_DIR,
    "19A_locked_random_forest_model_protocol_v1.json",
)

protocol_sha_path_20 = os.path.join(
    MODEL_OUTPUT_DIR,
    "19A_locked_random_forest_model_protocol_v1_SHA256.txt",
)

protocol_text_20 = json.dumps(
    rf_protocol_20,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
)

protocol_sha_20 = hashlib.sha256(
    protocol_text_20.encode("utf-8")
).hexdigest()

if os.path.exists(protocol_path_20):
    with open(protocol_path_20, "r", encoding="utf-8") as fh:
        existing_protocol_text_20 = fh.read()
    existing_protocol_sha_20 = hashlib.sha256(
        existing_protocol_text_20.encode("utf-8")
    ).hexdigest()

    if existing_protocol_sha_20 != protocol_sha_20:
        raise RuntimeError(
            "An existing Random Forest protocol file differs from "
            "the currently locked protocol. Stop and audit."
        )
else:
    with open(protocol_path_20, "w", encoding="utf-8") as fh:
        fh.write(protocol_text_20)

with open(protocol_sha_path_20, "w", encoding="utf-8") as fh:
    fh.write(protocol_sha_20 + "\n")

print("Locked Random Forest protocol SHA-256:")
print(protocol_sha_20)

# ------------------------------------------------------------
# 3. Load locked inner hospital map
# ------------------------------------------------------------

inner_mapping_path_20 = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_20):
    raise FileNotFoundError(
        "Locked inner-fold map not found: "
        + inner_mapping_path_20
    )

inner_mapping_all_20 = pd.read_csv(
    inner_mapping_path_20,
    dtype={"group_hospital": str},
)

inner_mapping_part_20 = (
    inner_mapping_all_20.loc[
        inner_mapping_all_20["outer_fold"].astype(int)
        == OUTER_FOLD_20,
        ["group_hospital", "inner_fold"],
    ]
    .copy()
)

inner_mapping_part_20["group_hospital"] = (
    inner_mapping_part_20["group_hospital"].astype(str)
)
inner_mapping_part_20["inner_fold"] = (
    inner_mapping_part_20["inner_fold"].astype(int)
)

if len(inner_mapping_part_20) != 158:
    raise RuntimeError(
        "Expected 158 outer-fold-2 training hospitals "
        "in the locked inner map."
    )

if inner_mapping_part_20["group_hospital"].duplicated().any():
    raise RuntimeError("Duplicate hospital in locked inner map.")

hospital_to_inner_fold_20 = dict(
    zip(
        inner_mapping_part_20["group_hospital"],
        inner_mapping_part_20["inner_fold"],
    )
)

# ------------------------------------------------------------
# 4. Prepare outer fold 2 matrices
# ------------------------------------------------------------

X_all_20 = core_df_07B[predictor_columns_07B].copy()

for column in numeric_columns_07B:
    X_all_20[column] = pd.to_numeric(
        X_all_20[column],
        errors="coerce",
    ).astype("float64")

for column in categorical_columns_07B:
    category_series = X_all_20[column].astype("object")
    X_all_20[column] = category_series.where(
        pd.notna(category_series),
        np.nan,
    )

outer_fold_vector_20 = (
    core_df_07B["outer_fold"].astype(int).to_numpy()
)

outer_training_mask_20 = (
    outer_fold_vector_20 != OUTER_FOLD_20
)
outer_test_mask_20 = (
    outer_fold_vector_20 == OUTER_FOLD_20
)

X_outer_training_20 = (
    X_all_20.loc[outer_training_mask_20]
    .reset_index(drop=True)
)

X_outer_test_20 = (
    X_all_20.loc[outer_test_mask_20]
    .reset_index(drop=True)
)

outer_training_meta_20 = (
    core_df_07B.loc[
        outer_training_mask_20,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_20 = (
    core_df_07B.loc[
        outer_test_mask_20,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [
    outer_training_meta_20,
    outer_test_meta_20,
]:
    dataframe["id_row"] = dataframe["id_row"].astype(str)
    dataframe["group_hospital"] = (
        dataframe["group_hospital"].astype(str)
    )
    dataframe["label_stage23"] = (
        dataframe["label_stage23"].astype(int)
    )

y_outer_training_20 = (
    outer_training_meta_20["label_stage23"]
    .to_numpy(dtype=np.int8)
)
y_outer_test_20 = (
    outer_test_meta_20["label_stage23"]
    .to_numpy(dtype=np.int8)
)

groups_outer_training_20 = (
    outer_training_meta_20["group_hospital"]
    .to_numpy(dtype=str)
)

training_hospitals_20 = set(
    outer_training_meta_20["group_hospital"]
)
test_hospitals_20 = set(
    outer_test_meta_20["group_hospital"]
)
hospital_overlap_20 = (
    training_hospitals_20 & test_hospitals_20
)

if hospital_overlap_20:
    raise RuntimeError(
        "Outer training/test hospital overlap detected."
    )

actual_split_20 = {
    "training_rows": len(X_outer_training_20),
    "test_rows": len(X_outer_test_20),
    "training_hospitals": len(training_hospitals_20),
    "test_hospitals": len(test_hospitals_20),
    "training_events": int(y_outer_training_20.sum()),
    "test_events": int(y_outer_test_20.sum()),
}

for metric, expected_value in EXPECTED_SPLIT_20.items():
    actual_value = actual_split_20[metric]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: found={actual_value}, "
            f"expected={expected_value}"
        )

inner_fold_vector_20 = np.array(
    [
        hospital_to_inner_fold_20.get(hospital, -1)
        for hospital in groups_outer_training_20
    ],
    dtype=int,
)

if (inner_fold_vector_20 == -1).any():
    raise RuntimeError(
        "Some outer-training hospitals have no inner-fold assignment."
    )

if set(np.unique(inner_fold_vector_20)) != {1, 2, 3, 4, 5}:
    raise RuntimeError("Inner-fold values are not exactly 1–5.")

# ------------------------------------------------------------
# 5. Preprocessor and model constructors
# ------------------------------------------------------------

def make_preprocessor_20():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns_07B,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns_07B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_random_forest_model_20(candidate):
    return RandomForestClassifier(
        n_estimators=int(candidate["n_estimators"]),
        criterion="gini",
        max_depth=(
            None
            if candidate["max_depth"] is None
            else int(candidate["max_depth"])
        ),
        min_samples_split=2,
        min_samples_leaf=int(candidate["min_samples_leaf"]),
        max_features=candidate["max_features"],
        bootstrap=True,
        max_samples=float(candidate["max_samples"]),
        class_weight=None,
        random_state=MODEL_RANDOM_SEED_20,
        n_jobs=-1,
        verbose=0,
    )


def checkpoint_table_id_20(inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_random_forest_inner_oof_outer2_inner{inner_fold}_v1"
    )

# ------------------------------------------------------------
# 6. BigQuery checkpoint verification
# ------------------------------------------------------------

def verify_checkpoint_20(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):
    table_id = checkpoint_table_id_20(inner_fold)

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS distinct_id_count,
      COUNT(DISTINCT candidate_id) AS candidate_count,
      COUNT(DISTINCT outer_fold) AS outer_fold_count,
      COUNT(DISTINCT inner_fold) AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(candidate_id, '|', id_row)
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(
        prediction_raw < 0 OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(
        sql,
        location=BQ_LOCATION,
    ).to_dataframe()

    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows * len(candidate_grid_20)
    )
    expected_positive_rows = (
        expected_validation_events * len(candidate_grid_20)
    )
    expected_negative_rows = (
        (expected_validation_rows - expected_validation_events)
        * len(candidate_grid_20)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": expected_validation_rows,
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": expected_total_rows,
        "positive_prediction_rows": expected_positive_rows,
        "negative_prediction_rows": expected_negative_rows,
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": OUTER_FOLD_20,
        "maximum_outer_fold": OUTER_FOLD_20,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failures = []

    for field, expected_value in expected_values.items():
        actual_value = int(row[field])
        if actual_value != expected_value:
            complete = False
            failures.append(
                f"{field}={actual_value}, expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failures),
        "check": check,
        "row": row,
    }

checkpoint_load_config_20 = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField(
            "id_row", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "outer_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "inner_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "candidate_id", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "label_stage23", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "prediction_raw", "FLOAT", mode="REQUIRED"
        ),
    ],
    write_disposition=(
        bigquery.WriteDisposition.WRITE_TRUNCATE
    ),
)

# ------------------------------------------------------------
# 7. Aggregate fit audit
# ------------------------------------------------------------

fit_audit_columns_20 = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "n_estimators",
    "max_depth",
    "min_samples_leaf",
    "max_features",
    "max_samples",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "fit_seconds",
]

fit_audit_path_20 = os.path.join(
    MODEL_OUTPUT_DIR,
    "20B_random_forest_inner_fit_audit_outer2.csv",
)

if os.path.exists(fit_audit_path_20):
    fit_audit_20 = pd.read_csv(fit_audit_path_20)
else:
    fit_audit_20 = pd.DataFrame(
        columns=fit_audit_columns_20
    )

for column in fit_audit_columns_20:
    if column not in fit_audit_20.columns:
        fit_audit_20[column] = np.nan

fit_audit_20 = fit_audit_20[
    fit_audit_columns_20
].copy()

# ------------------------------------------------------------
# 8. Run five inner folds
# ------------------------------------------------------------

for inner_fold in range(1, 6):
    inner_training_mask = (
        inner_fold_vector_20 != inner_fold
    )
    inner_validation_mask = (
        inner_fold_vector_20 == inner_fold
    )

    training_rows = int(inner_training_mask.sum())
    validation_rows = int(inner_validation_mask.sum())
    training_events = int(
        y_outer_training_20[inner_training_mask].sum()
    )
    validation_events = int(
        y_outer_training_20[inner_validation_mask].sum()
    )

    existing_check = verify_checkpoint_20(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if existing_check["complete"]:
        print(
            f"Outer 2 / inner {inner_fold}: "
            "permanent Random Forest checkpoint already complete; "
            "skipping model fitting."
        )
        continue

    training_hospital_set = set(
        groups_outer_training_20[inner_training_mask]
    )
    validation_hospital_set = set(
        groups_outer_training_20[inner_validation_mask]
    )

    if training_hospital_set & validation_hospital_set:
        raise RuntimeError(
            f"Inner fold {inner_fold}: hospital overlap detected."
        )

    print(f"\nOuter 2 / inner {inner_fold}")
    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_20()

    preprocessing_started = time.time()

    X_inner_training_processed = (
        preprocessor.fit_transform(
            X_outer_training_20.loc[
                inner_training_mask
            ]
        )
    )

    X_inner_validation_processed = (
        preprocessor.transform(
            X_outer_training_20.loc[
                inner_validation_mask
            ]
        )
    )

    preprocessing_seconds = (
        time.time() - preprocessing_started
    )

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "Processed training/validation column counts differ."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_seconds, 2),
    )

    y_inner_training = (
        y_outer_training_20[inner_training_mask]
    )
    y_inner_validation = (
        y_outer_training_20[inner_validation_mask]
    )

    validation_ids = (
        outer_training_meta_20.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_20:
        candidate_id = candidate["candidate_id"]

        print(
            "  Fitting",
            candidate_id,
            "| trees =",
            candidate["n_estimators"],
            "| depth =",
            candidate["max_depth"],
            "| min_leaf =",
            candidate["min_samples_leaf"],
            "| max_features =",
            candidate["max_features"],
            "| max_samples =",
            candidate["max_samples"],
        )

        model = make_random_forest_model_20(candidate)

        fit_started = time.time()

        model.fit(
            X_inner_training_processed,
            y_inner_training,
        )

        fit_seconds = time.time() - fit_started

        validation_probabilities = (
            model.predict_proba(
                X_inner_validation_processed
            )[:, 1]
        )

        if np.isnan(validation_probabilities).any():
            raise RuntimeError(
                f"{candidate_id}, inner {inner_fold}: "
                "missing predictions."
            )

        if not np.all(
            (validation_probabilities >= 0)
            & (validation_probabilities <= 1)
        ):
            raise RuntimeError(
                f"{candidate_id}: invalid probabilities."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        OUTER_FOLD_20,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": (
                        y_inner_validation.astype(np.int64)
                    ),
                    "prediction_raw": (
                        validation_probabilities.astype(
                            np.float64
                        )
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": OUTER_FOLD_20,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "n_estimators": candidate[
                    "n_estimators"
                ],
                "max_depth": candidate[
                    "max_depth"
                ],
                "min_samples_leaf": candidate[
                    "min_samples_leaf"
                ],
                "max_features": candidate[
                    "max_features"
                ],
                "max_samples": candidate[
                    "max_samples"
                ],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": (
                    validation_events
                ),
                "processed_columns": int(
                    X_inner_training_processed.shape[1]
                ),
                "fit_seconds": float(fit_seconds),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows * len(candidate_grid_20)
    )

    if len(checkpoint_df) != expected_checkpoint_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: invalid checkpoint row count."
        )

    if checkpoint_df.duplicated(
        subset=["id_row", "candidate_id"]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: duplicate candidate-patient rows."
        )

    target_checkpoint_table = (
        checkpoint_table_id_20(inner_fold)
    )

    print(
        "Uploading permanent Random Forest checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_20,
        location=BQ_LOCATION,
    ).result()

    if len(fit_audit_20) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_20["outer_fold"],
                    errors="coerce",
                ) == 1
            )
            & (
                pd.to_numeric(
                    fit_audit_20["inner_fold"],
                    errors="coerce",
                ) == inner_fold
            )
        )

        fit_audit_20 = (
            fit_audit_20.loc[keep_mask].copy()
        )

    fit_audit_20 = pd.concat(
        [
            fit_audit_20,
            pd.DataFrame(current_audit_rows),
        ],
        ignore_index=True,
    )

    fit_audit_20 = (
        fit_audit_20[
            fit_audit_columns_20
        ]
        .sort_values(
            [
                "outer_fold",
                "inner_fold",
                "candidate_id",
            ]
        )
        .reset_index(drop=True)
    )

    fit_audit_20.to_csv(
        fit_audit_path_20,
        index=False,
    )

    completed_check = verify_checkpoint_20(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint verification failed: "
            + completed_check["reason"]
        )

    print(
        f"Outer 2 / inner {inner_fold}: "
        "permanent Random Forest checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 9. Final inner checkpoint summary
# ------------------------------------------------------------

checkpoint_summary_rows_20 = []

for inner_fold in range(1, 6):
    validation_mask = (
        inner_fold_vector_20 == inner_fold
    )
    validation_rows = int(validation_mask.sum())
    validation_events = int(
        y_outer_training_20[
            validation_mask
        ].sum()
    )

    final_check = verify_checkpoint_20(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: final checkpoint audit failed. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_20.append(
        {
            "outer_fold": OUTER_FOLD_20,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(
                row["row_count"]
            ),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(
                row["candidate_count"]
            ),
            "positive_prediction_rows": int(
                row["positive_prediction_rows"]
            ),
            "negative_prediction_rows": int(
                row["negative_prediction_rows"]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check["table_id"],
        }
    )

checkpoint_summary_20 = (
    pd.DataFrame(checkpoint_summary_rows_20)
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_20[
        "distinct_validation_patients"
    ].sum()
) != EXPECTED_SPLIT_20["training_rows"]:
    raise RuntimeError(
        "Total inner validation patients != 46,803."
    )

expected_total_oof_rows_20 = (
    EXPECTED_SPLIT_20["training_rows"]
    * len(candidate_grid_20)
)

if int(
    checkpoint_summary_20[
        "checkpoint_rows"
    ].sum()
) != expected_total_oof_rows_20:
    raise RuntimeError(
        "Total Random Forest OOF prediction rows are incorrect."
    )

checkpoint_summary_path_20 = os.path.join(
    MODEL_OUTPUT_DIR,
    "20B_random_forest_outer2_inner_checkpoint_summary.csv",
)

checkpoint_summary_20.to_csv(
    checkpoint_summary_path_20,
    index=False,
)

# ------------------------------------------------------------
# 10. Pool five inner OOF tables
# ------------------------------------------------------------

checkpoint_tables_20 = [
    checkpoint_table_id_20(inner_fold)
    for inner_fold in range(1, 6)
]

union_parts_20 = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_20
]

SQL_LOAD_POOLED_OOF_20 = (
    "\nUNION ALL\n".join(union_parts_20)
)

print(
    "\nLoading pooled outer-fold-2 Random Forest inner OOF predictions..."
)

query_job_20 = client.query(
    SQL_LOAD_POOLED_OOF_20,
    location=BQ_LOCATION,
)

try:
    pooled_oof_20 = query_job_20.to_dataframe(
        create_bqstorage_client=True
    )
    pooled_load_method_20 = (
        "BigQuery Storage API"
    )
except Exception as fast_path_error_20:
    print(
        "Storage API unavailable; using standard BigQuery download."
    )
    print(
        "Message:",
        type(fast_path_error_20).__name__,
    )
    pooled_oof_20 = query_job_20.to_dataframe(
        create_bqstorage_client=False
    )
    pooled_load_method_20 = (
        "Standard BigQuery API"
    )

pooled_oof_20["id_row"] = (
    pooled_oof_20["id_row"].astype(str)
)
pooled_oof_20["candidate_id"] = (
    pooled_oof_20["candidate_id"].astype(str)
)

for column in [
    "outer_fold",
    "inner_fold",
    "label_stage23",
]:
    pooled_oof_20[column] = pd.to_numeric(
        pooled_oof_20[column],
        errors="raise",
    ).astype(int)

pooled_oof_20["prediction_raw"] = pd.to_numeric(
    pooled_oof_20["prediction_raw"],
    errors="raise",
).astype(float)

if len(pooled_oof_20) != expected_total_oof_rows_20:
    raise RuntimeError(
        "Pooled Random Forest OOF row count is incorrect."
    )

if pooled_oof_20.duplicated(
    subset=["candidate_id", "id_row"]
).any():
    raise RuntimeError(
        "Duplicate candidate-patient row in pooled Random Forest OOF."
    )

if pooled_oof_20["prediction_raw"].isna().any():
    raise RuntimeError("Missing Random Forest OOF prediction.")

if not pooled_oof_20[
    "prediction_raw"
].between(0, 1).all():
    raise RuntimeError(
        "Invalid Random Forest OOF probability."
    )

if set(
    pooled_oof_20["candidate_id"].unique()
) != {
    "RF01",
    "RF02",
    "RF03",
    "RF04",
    "RF05",
    "RF06",
}:
    raise RuntimeError(
        "The six locked Random Forest candidates are not all present."
    )

candidate_patient_counts_20 = (
    pooled_oof_20
    .groupby("candidate_id")["id_row"]
    .nunique()
)

if not (
    candidate_patient_counts_20
    == EXPECTED_SPLIT_20["training_rows"]
).all():
    raise RuntimeError(
        "Each candidate must have 46,803 OOF patients."
    )

candidate_event_counts_20 = (
    pooled_oof_20
    .groupby("candidate_id")["label_stage23"]
    .sum()
)

if not (
    candidate_event_counts_20
    == EXPECTED_SPLIT_20["training_events"]
).all():
    raise RuntimeError(
        "Each candidate must have 2,426 OOF events."
    )

# ------------------------------------------------------------
# 11. Metric helpers
# ------------------------------------------------------------

def probability_metrics_20(
    y_true,
    probabilities,
):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(
            roc_auc_score(y_true, probabilities)
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1],
            )
        ),
        "mean_predicted_risk": float(
            probabilities.mean()
        ),
        "observed_event_rate": float(
            np.mean(y_true)
        ),
    }


def probability_logit_20(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        probabilities / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_20(
    y_true,
    probabilities,
):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        probability_logit_20(probabilities),
        y_true,
    )

    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 12. Candidate pooled inner OOF metrics
# ------------------------------------------------------------

candidate_result_rows_20 = []

for candidate in candidate_grid_20:
    candidate_id = candidate["candidate_id"]

    candidate_oof = (
        pooled_oof_20.loc[
            pooled_oof_20["candidate_id"]
            == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_20(
        candidate_oof[
            "label_stage23"
        ].to_numpy(dtype=int),
        candidate_oof[
            "prediction_raw"
        ].to_numpy(dtype=float),
    )

    fit_part = fit_audit_20.loc[
        fit_audit_20[
            "candidate_id"
        ].astype(str) == candidate_id
    ]

    fit_seconds_total = (
        float(
            pd.to_numeric(
                fit_part["fit_seconds"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_20.append(
        {
            "candidate_id": candidate_id,
            **{
                key: candidate[key]
                for key in candidate
                if key != "candidate_id"
            },
            **metrics,
            "fit_seconds_total": (
                fit_seconds_total
            ),
        }
    )

candidate_results_20 = pd.DataFrame(
    candidate_result_rows_20
)

candidate_results_20 = (
    candidate_results_20
    .sort_values(
        ["auprc", "auroc", "brier"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

candidate_results_20["selection_rank"] = (
    np.arange(
        1,
        len(candidate_results_20) + 1,
    )
)

best_row_20 = candidate_results_20.iloc[0]
selected_candidate_id_20 = str(
    best_row_20["candidate_id"]
)

selected_candidate_20 = next(
    candidate
    for candidate in candidate_grid_20
    if candidate["candidate_id"]
    == selected_candidate_id_20
)

# ------------------------------------------------------------
# 13. Platt calibration from selected inner OOF
# ------------------------------------------------------------

selected_oof_20 = (
    pooled_oof_20.loc[
        pooled_oof_20["candidate_id"]
        == selected_candidate_id_20
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_20 = (
    selected_oof_20[
        "label_stage23"
    ].to_numpy(dtype=int)
)
selected_oof_probability_20 = (
    selected_oof_20[
        "prediction_raw"
    ].to_numpy(dtype=float)
)

platt_calibrator_20 = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_20.fit(
    probability_logit_20(
        selected_oof_probability_20
    ),
    selected_oof_y_20,
)

platt_intercept_20 = float(
    platt_calibrator_20.intercept_[0]
)
platt_slope_20 = float(
    platt_calibrator_20.coef_[0][0]
)

if (
    not np.isfinite(platt_intercept_20)
    or not np.isfinite(platt_slope_20)
    or platt_slope_20 <= 0
):
    raise RuntimeError(
        "Invalid Platt calibration coefficients."
    )

selected_model_20 = pd.DataFrame(
    [
        {
            "outer_fold": OUTER_FOLD_20,
            "selected_candidate": (
                selected_candidate_id_20
            ),
            "selection_metric_primary": (
                "pooled_inner_oof_auprc"
            ),
            "inner_oof_auprc": float(
                best_row_20["auprc"]
            ),
            "inner_oof_auroc": float(
                best_row_20["auroc"]
            ),
            "inner_oof_brier": float(
                best_row_20["brier"]
            ),
            "inner_oof_log_loss": float(
                best_row_20["log_loss"]
            ),
            "inner_oof_mean_predicted_risk": float(
                best_row_20[
                    "mean_predicted_risk"
                ]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_20[
                    "observed_event_rate"
                ]
            ),
            "platt_intercept": (
                platt_intercept_20
            ),
            "platt_slope": platt_slope_20,
            "protocol_sha256": (
                protocol_sha_20
            ),
            **{
                key: selected_candidate_20[key]
                for key in selected_candidate_20
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 14. Save locked selection
# ------------------------------------------------------------

candidate_results_path_20 = os.path.join(
    MODEL_OUTPUT_DIR,
    "20C_random_forest_candidate_results_outer2.csv",
)

selected_model_path_20 = os.path.join(
    MODEL_OUTPUT_DIR,
    "20C_random_forest_selected_model_outer2.csv",
)

selection_json_path_20 = os.path.join(
    MODEL_OUTPUT_DIR,
    "20C_random_forest_selection_calibration_outer2.json",
)

selection_sha_path_20 = os.path.join(
    MODEL_OUTPUT_DIR,
    "20C_random_forest_selection_calibration_outer2_SHA256.txt",
)

candidate_results_20.to_csv(
    candidate_results_path_20,
    index=False,
)
selected_model_20.to_csv(
    selected_model_path_20,
    index=False,
)

selection_configuration_20 = {
    "outer_fold": OUTER_FOLD_20,
    "protocol_sha256": protocol_sha_20,
    "selection_metric_primary": (
        "pooled inner out-of-fold AUPRC"
    ),
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "selected_candidate": (
        selected_candidate_id_20
    ),
    "selected_hyperparameters": {
        key: selected_candidate_20[key]
        for key in selected_candidate_20
        if key != "candidate_id"
    },
    "inner_oof_auprc": float(
        best_row_20["auprc"]
    ),
    "inner_oof_auroc": float(
        best_row_20["auroc"]
    ),
    "inner_oof_brier": float(
        best_row_20["brier"]
    ),
    "platt_intercept": platt_intercept_20,
    "platt_slope": platt_slope_20,
    "inner_checkpoint_tables": (
        checkpoint_tables_20
    ),
    "patient_level_oof_written_to_drive": False,
}

with open(
    selection_json_path_20,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        selection_configuration_20,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    selection_json_path_20,
    "rb",
) as fh:
    selection_sha_20 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    selection_sha_path_20,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(selection_sha_20 + "\n")

# ------------------------------------------------------------
# 15. Fit selected model on all outer training patients
# ------------------------------------------------------------

final_preprocessor_20 = (
    make_preprocessor_20()
)

print(
    "\nFitting selected outer-fold-2 Random Forest model "
    "on all 46,800 training patients..."
)

preprocess_started_20 = time.time()

X_outer_training_processed_20 = (
    final_preprocessor_20.fit_transform(
        X_outer_training_20
    )
)
X_outer_test_processed_20 = (
    final_preprocessor_20.transform(
        X_outer_test_20
    )
)

final_preprocessing_seconds_20 = (
    time.time() - preprocess_started_20
)

final_model_20 = make_random_forest_model_20(
    selected_candidate_20
)

final_fit_started_20 = time.time()

final_model_20.fit(
    X_outer_training_processed_20,
    y_outer_training_20,
)

final_fit_seconds_20 = (
    time.time() - final_fit_started_20
)

outer2_raw_probabilities_20 = (
    final_model_20.predict_proba(
        X_outer_test_processed_20
    )[:, 1]
)

raw_clipped_20 = np.clip(
    outer2_raw_probabilities_20,
    1e-6,
    1 - 1e-6,
)
raw_logit_20 = np.log(
    raw_clipped_20
    / (1 - raw_clipped_20)
)

outer2_platt_probabilities_20 = expit(
    platt_intercept_20
    + platt_slope_20 * raw_logit_20
)

for probabilities, name in [
    (outer2_raw_probabilities_20, "raw"),
    (
        outer2_platt_probabilities_20,
        "platt",
    ),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(
            f"{name} test predictions contain missing values."
        )

    if not np.all(
        (probabilities >= 0)
        & (probabilities <= 1)
    ):
        raise RuntimeError(
            f"{name} test predictions contain invalid probabilities."
        )

raw_metrics_20 = probability_metrics_20(
    y_outer_test_20,
    outer2_raw_probabilities_20,
)
platt_metrics_20 = probability_metrics_20(
    y_outer_test_20,
    outer2_platt_probabilities_20,
)

raw_calibration_intercept_20, \
raw_calibration_slope_20 = (
    calibration_intercept_slope_20(
        y_outer_test_20,
        outer2_raw_probabilities_20,
    )
)

platt_calibration_intercept_20, \
platt_calibration_slope_20 = (
    calibration_intercept_slope_20(
        y_outer_test_20,
        outer2_platt_probabilities_20,
    )
)

outer2_test_results_20 = pd.DataFrame(
    [
        {
            "outer_fold": OUTER_FOLD_20,
            "model": "random_forest",
            "probability_type": "raw",
            **raw_metrics_20,
            "calibration_intercept": (
                raw_calibration_intercept_20
            ),
            "calibration_slope": (
                raw_calibration_slope_20
            ),
        },
        {
            "outer_fold": OUTER_FOLD_20,
            "model": "random_forest",
            "probability_type": (
                "platt_calibrated"
            ),
            **platt_metrics_20,
            "calibration_intercept": (
                platt_calibration_intercept_20
            ),
            "calibration_slope": (
                platt_calibration_slope_20
            ),
        },
    ]
)

# ------------------------------------------------------------
# 16. Feature importance
# ------------------------------------------------------------

processed_feature_names_20 = (
    final_preprocessor_20
    .get_feature_names_out()
)

feature_importances_20 = (
    final_model_20.feature_importances_
)

if len(processed_feature_names_20) != len(
    feature_importances_20
):
    raise RuntimeError(
        "Processed feature names and Random Forest "
        "feature importances differ in length."
    )

feature_importance_table_20 = pd.DataFrame(
    {
        "processed_feature": (
            processed_feature_names_20
        ),
        "mdi_importance": (
            feature_importances_20
        ),
    }
)

feature_importance_table_20[
    "importance_rank"
] = (
    feature_importance_table_20[
        "mdi_importance"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)

feature_importance_table_20 = (
    feature_importance_table_20
    .sort_values(
        [
            "mdi_importance",
            "processed_feature",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

nonzero_importance_features_20 = int(
    (
        feature_importance_table_20[
            "mdi_importance"
        ] > 0
    ).sum()
)

final_model_summary_20 = pd.DataFrame(
    [
        {
            "outer_fold": OUTER_FOLD_20,
            "selected_candidate": (
                selected_candidate_id_20
            ),
            "training_patients": len(
                X_outer_training_20
            ),
            "training_hospitals": len(
                training_hospitals_20
            ),
            "training_events": int(
                y_outer_training_20.sum()
            ),
            "test_patients": len(
                X_outer_test_20
            ),
            "test_hospitals": len(
                test_hospitals_20
            ),
            "test_events": int(
                y_outer_test_20.sum()
            ),
            "hospital_overlap": len(
                hospital_overlap_20
            ),
            "processed_feature_columns": len(
                processed_feature_names_20
            ),
            "nonzero_importance_features": (
                nonzero_importance_features_20
            ),
            "preprocessing_seconds": float(
                final_preprocessing_seconds_20
            ),
            "fit_seconds": float(
                final_fit_seconds_20
            ),
            "locked_platt_intercept": (
                platt_intercept_20
            ),
            "locked_platt_slope": (
                platt_slope_20
            ),
            "protocol_sha256": protocol_sha_20,
            "selection_sha256": (
                selection_sha_20
            ),
            **{
                key: selected_candidate_20[key]
                for key in selected_candidate_20
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 17. Secure outer-test prediction checkpoint
# ------------------------------------------------------------

outer2_prediction_df_20 = pd.DataFrame(
    {
        "id_row": (
            outer_test_meta_20[
                "id_row"
            ].astype(str)
        ),
        "outer_fold": np.full(
            len(outer_test_meta_20),
            OUTER_FOLD_20,
            dtype=np.int64,
        ),
        "label_stage23": (
            y_outer_test_20.astype(np.int64)
        ),
        "prediction_raw": (
            outer2_raw_probabilities_20.astype(
                np.float64
            )
        ),
        "prediction_platt": (
            outer2_platt_probabilities_20.astype(
                np.float64
            )
        ),
        "model_name": "random_forest",
        "model_version": (
            "core_v1_nested_cv"
        ),
    }
)

if len(outer2_prediction_df_20) != 11691:
    raise RuntimeError(
        "Outer-fold-1 test prediction row count != 11,688."
    )

if outer2_prediction_df_20[
    "id_row"
].duplicated().any():
    raise RuntimeError(
        "Duplicate id_row in outer-fold-2 Random Forest predictions."
    )

if int(
    outer2_prediction_df_20[
        "label_stage23"
    ].sum()
) != 606:
    raise RuntimeError(
        "Outer-fold-1 Random Forest event count != 606."
    )

prediction_table_id_20 = (
    f"{TARGET_DATASET}."
    "model_random_forest_outer_predictions_outer2_v1"
)

prediction_load_config_20 = (
    bigquery.LoadJobConfig(
        schema=[
            bigquery.SchemaField(
                "id_row",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "outer_fold",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "label_stage23",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_raw",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_platt",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_name",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_version",
                "STRING",
                mode="REQUIRED",
            ),
        ],
        write_disposition=(
            bigquery.WriteDisposition.WRITE_TRUNCATE
        ),
    )
)

print(
    "\nUploading secure outer-fold-2 "
    "Random Forest prediction checkpoint:"
)
print(prediction_table_id_20)

client.load_table_from_dataframe(
    outer2_prediction_df_20,
    prediction_table_id_20,
    job_config=prediction_load_config_20,
    location=BQ_LOCATION,
).result()

SQL_VERIFY_PREDICTIONS_20 = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL)
    AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL)
    AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0
    OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0
    OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw)
    AS minimum_raw_probability,
  MAX(prediction_raw)
    AS maximum_raw_probability,
  MIN(prediction_platt)
    AS minimum_platt_probability,
  MAX(prediction_platt)
    AS maximum_platt_probability
FROM `{prediction_table_id_20}`;
"""

prediction_verification_20 = (
    client.query(
        SQL_VERIFY_PREDICTIONS_20,
        location=BQ_LOCATION,
    )
    .to_dataframe()
)

verification_row_20 = (
    prediction_verification_20.iloc[0]
)

expected_prediction_values_20 = {
    "prediction_rows": 11691,
    "distinct_rows": 11691,
    "outer_folds": 1,
    "minimum_outer_fold": OUTER_FOLD_20,
    "maximum_outer_fold": OUTER_FOLD_20,
    "events": 606,
    "nonevents": 11085,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in (
    expected_prediction_values_20.items()
):
    actual_value = int(
        verification_row_20[field]
    )
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: found={actual_value}, "
            f"expected={expected_value}"
        )

# ------------------------------------------------------------
# 18. Save aggregate outputs
# ------------------------------------------------------------

test_results_path_20 = os.path.join(
    MODEL_OUTPUT_DIR,
    "20D_random_forest_outer2_test_results.csv",
)

model_summary_path_20 = os.path.join(
    MODEL_OUTPUT_DIR,
    "20D_random_forest_final_model_outer2.csv",
)

importance_path_20 = os.path.join(
    MODEL_OUTPUT_DIR,
    "20D_random_forest_mdi_importance_outer2.csv",
)

evaluation_json_path_20 = os.path.join(
    MODEL_OUTPUT_DIR,
    "20D_random_forest_final_evaluation_outer2.json",
)

evaluation_sha_path_20 = os.path.join(
    MODEL_OUTPUT_DIR,
    "20D_random_forest_final_evaluation_outer2_SHA256.txt",
)

outer2_test_results_20.to_csv(
    test_results_path_20,
    index=False,
)
final_model_summary_20.to_csv(
    model_summary_path_20,
    index=False,
)
feature_importance_table_20.to_csv(
    importance_path_20,
    index=False,
)

evaluation_configuration_20 = {
    "outer_fold": OUTER_FOLD_20,
    "model_family": "random_forest",
    "protocol_sha256": protocol_sha_20,
    "selection_sha256": selection_sha_20,
    "selected_candidate": (
        selected_candidate_id_20
    ),
    "selected_hyperparameters": {
        key: selected_candidate_20[key]
        for key in selected_candidate_20
        if key != "candidate_id"
    },
    "training_patients": 46800,
    "training_hospitals": 158,
    "test_patients": 11691,
    "test_hospitals": 40,
    "hospital_overlap": 0,
    "locked_platt_intercept": (
        platt_intercept_20
    ),
    "locked_platt_slope": platt_slope_20,
    "processed_feature_columns": int(
        len(processed_feature_names_20)
    ),
    "secure_prediction_table": (
        prediction_table_id_20
    ),
    "patient_level_prediction_written_to_drive": False,
}

with open(
    evaluation_json_path_20,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        evaluation_configuration_20,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    evaluation_json_path_20,
    "rb",
) as fh:
    evaluation_sha_20 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    evaluation_sha_path_20,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(evaluation_sha_20 + "\n")

# ------------------------------------------------------------
# 19. Display results
# ------------------------------------------------------------

pooled_integrity_20 = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_20),
            pooled_oof_20[
                "id_row"
            ].nunique(),
            pooled_oof_20[
                "candidate_id"
            ].nunique(),
            pooled_oof_20[
                "inner_fold"
            ].nunique(),
            EXPECTED_SPLIT_20[
                "training_events"
            ],
            (
                EXPECTED_SPLIT_20[
                    "training_rows"
                ]
                - EXPECTED_SPLIT_20[
                    "training_events"
                ]
            ),
            int(
                pooled_oof_20.duplicated(
                    subset=[
                        "candidate_id",
                        "id_row",
                    ]
                ).sum()
            ),
            int(
                pooled_oof_20[
                    "prediction_raw"
                ].isna().sum()
            ),
            int(
                (
                    ~pooled_oof_20[
                        "prediction_raw"
                    ].between(0, 1)
                ).sum()
            ),
            pooled_load_method_20,
        ],
    }
)

print(
    "\n20 RANDOM FOREST OUTER-FOLD-2 INNER CHECKPOINT SUMMARY"
)
display(checkpoint_summary_20)

print(
    "\n20 RANDOM FOREST OUTER-FOLD-2 POOLED OOF INTEGRITY"
)
display(pooled_integrity_20)

print(
    "\n20 RANDOM FOREST OUTER-FOLD-2 CANDIDATE RESULTS"
)
display(candidate_results_20)

print(
    "\n20 RANDOM FOREST OUTER-FOLD-2 SELECTED MODEL"
)
display(selected_model_20)

print(
    "\n20 RANDOM FOREST OUTER-FOLD-2 FINAL MODEL SUMMARY"
)
display(final_model_summary_20)

print(
    "\n20 RANDOM FOREST OUTER-FOLD-2 TEST RESULTS"
)
display(outer2_test_results_20)

print(
    "\n20 RANDOM FOREST OUTER-FOLD-2 BIGQUERY VERIFICATION"
)
display(prediction_verification_20)

print(
    "\n20 RANDOM FOREST OUTER-FOLD-2 TOP 20 MDI IMPORTANCE FEATURES"
)
display(feature_importance_table_20.head(20))

print("\nRandom Forest protocol SHA-256:")
print(protocol_sha_20)

print("\nSelection SHA-256:")
print(selection_sha_20)

print("\nEvaluation SHA-256:")
print(evaluation_sha_20)

print("\nSaved aggregate outputs:")
print(protocol_path_20)
print(protocol_sha_path_20)
print(fit_audit_path_20)
print(checkpoint_summary_path_20)
print(candidate_results_path_20)
print(selected_model_path_20)
print(selection_json_path_20)
print(selection_sha_path_20)
print(test_results_path_20)
print(model_summary_path_20)
print(importance_path_20)
print(evaluation_json_path_20)
print(evaluation_sha_path_20)

print(
    "\n20 PASS: Random Forest outer-fold-2 nested modelling "
    "and locked test evaluation are complete."
)

print(
    "No class weighting or SMOTE was used."
)

print(
    "All patient-level OOF and outer-test predictions "
    "were stored only in BigQuery."
)

print(
    "No patient-level prediction file was written to Google Drive."
)

_ = gc.collect()

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

from sklearn.ensemble import RandomForestClassifier

from IPython.display import display

print("STARTING RANDOM FOREST OUTER FOLD 3 — CODE VERSION 21")

# ============================================================
# 21 — RANDOM FOREST OUTER FOLD 3 COMPLETE NESTED MODELLING
#
# Design:
# - Same locked hospital-disjoint outer folds.
# - Same locked hospital-disjoint inner folds.
# - Core feature set only.
# - No SMOTE.
# - No class weighting.
# - Fixed Random Forest candidate grid locked before outer-test evaluation.
# - Selection: pooled inner-OOF AUPRC descending,
#              AUROC descending, Brier ascending.
# - Platt calibration learned only from selected candidate's
#   pooled inner-OOF predictions.
# - Patient-level predictions stored only in BigQuery.
# ============================================================

OUTER_FOLD_21 = 3
MODEL_RANDOM_SEED_21 = 20260721

EXPECTED_SPLIT_21 = {
    "training_rows": 46755,
    "test_rows": 11736,
    "training_hospitals": 158,
    "test_hospitals": 40,
    "training_events": 2424,
    "test_events": 608,
}

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_objects_21 = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_21 = [
    name for name in required_objects_21
    if name not in globals()
]

if missing_objects_21:
    raise RuntimeError(
        "Missing runtime objects: "
        + ", ".join(missing_objects_21)
        + ". Run 07A and 07B first."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"Expected 58,491 cohort rows; found {len(core_df_07B)}."
    )

if len(predictor_columns_07B) != 159:
    raise RuntimeError("Expected 159 core predictors.")

if len(numeric_columns_07B) != 156:
    raise RuntimeError("Expected 156 numeric predictors.")

if len(categorical_columns_07B) != 3:
    raise RuntimeError("Expected 3 categorical predictors.")

# ------------------------------------------------------------
# 2. Lock the Random Forest protocol BEFORE test evaluation
# ------------------------------------------------------------

candidate_grid_21 = [
    {
        "candidate_id": "RF01",
        "n_estimators": 300,
        "max_depth": 8,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "max_samples": 0.80,
    },
    {
        "candidate_id": "RF02",
        "n_estimators": 400,
        "max_depth": 12,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
        "max_samples": 0.80,
    },
    {
        "candidate_id": "RF03",
        "n_estimators": 500,
        "max_depth": None,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
        "max_samples": 0.85,
    },
    {
        "candidate_id": "RF04",
        "n_estimators": 400,
        "max_depth": 12,
        "min_samples_leaf": 10,
        "max_features": 0.25,
        "max_samples": 0.85,
    },
    {
        "candidate_id": "RF05",
        "n_estimators": 500,
        "max_depth": None,
        "min_samples_leaf": 10,
        "max_features": 0.25,
        "max_samples": 0.90,
    },
    {
        "candidate_id": "RF06",
        "n_estimators": 600,
        "max_depth": 16,
        "min_samples_leaf": 20,
        "max_features": 0.50,
        "max_samples": 0.90,
    },
]

rf_protocol_21 = {
    "protocol_name": "random_forest_core_nested_hospital_cv_v1",
    "model_family": "RandomForestClassifier",
    "feature_set": "core_159",
    "outer_cv": "locked 5-fold hospital-disjoint outer folds",
    "inner_cv": "locked 5-fold hospital-disjoint inner folds",
    "primary_selection_metric": "pooled inner OOF AUPRC descending",
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "criterion": "gini",
    "bootstrap": True,
    "class_weighting": False,
    "class_weight": None,
    "smote": False,
    "min_samples_split": 2,
    "random_state": MODEL_RANDOM_SEED_21,
    "candidate_grid": candidate_grid_21,
    "calibration": (
        "Platt calibration fit only on selected candidate "
        "pooled inner-OOF logits"
    ),
    "outer_test_use": (
        "diagnostic evaluation only; never used for tuning"
    ),
}

protocol_path_21 = os.path.join(
    MODEL_OUTPUT_DIR,
    "19A_locked_random_forest_model_protocol_v1.json",
)

protocol_sha_path_21 = os.path.join(
    MODEL_OUTPUT_DIR,
    "19A_locked_random_forest_model_protocol_v1_SHA256.txt",
)

protocol_text_21 = json.dumps(
    rf_protocol_21,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
)

protocol_sha_21 = hashlib.sha256(
    protocol_text_21.encode("utf-8")
).hexdigest()

if os.path.exists(protocol_path_21):
    with open(protocol_path_21, "r", encoding="utf-8") as fh:
        existing_protocol_text_21 = fh.read()
    existing_protocol_sha_21 = hashlib.sha256(
        existing_protocol_text_21.encode("utf-8")
    ).hexdigest()

    if existing_protocol_sha_21 != protocol_sha_21:
        raise RuntimeError(
            "An existing Random Forest protocol file differs from "
            "the currently locked protocol. Stop and audit."
        )
else:
    with open(protocol_path_21, "w", encoding="utf-8") as fh:
        fh.write(protocol_text_21)

with open(protocol_sha_path_21, "w", encoding="utf-8") as fh:
    fh.write(protocol_sha_21 + "\n")

print("Locked Random Forest protocol SHA-256:")
print(protocol_sha_21)

# ------------------------------------------------------------
# 3. Load locked inner hospital map
# ------------------------------------------------------------

inner_mapping_path_21 = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_21):
    raise FileNotFoundError(
        "Locked inner-fold map not found: "
        + inner_mapping_path_21
    )

inner_mapping_all_21 = pd.read_csv(
    inner_mapping_path_21,
    dtype={"group_hospital": str},
)

inner_mapping_part_21 = (
    inner_mapping_all_21.loc[
        inner_mapping_all_21["outer_fold"].astype(int)
        == OUTER_FOLD_21,
        ["group_hospital", "inner_fold"],
    ]
    .copy()
)

inner_mapping_part_21["group_hospital"] = (
    inner_mapping_part_21["group_hospital"].astype(str)
)
inner_mapping_part_21["inner_fold"] = (
    inner_mapping_part_21["inner_fold"].astype(int)
)

if len(inner_mapping_part_21) != 158:
    raise RuntimeError(
        "Expected 158 outer-fold-3 training hospitals "
        "in the locked inner map."
    )

if inner_mapping_part_21["group_hospital"].duplicated().any():
    raise RuntimeError("Duplicate hospital in locked inner map.")

hospital_to_inner_fold_21 = dict(
    zip(
        inner_mapping_part_21["group_hospital"],
        inner_mapping_part_21["inner_fold"],
    )
)

# ------------------------------------------------------------
# 4. Prepare outer fold 3 matrices
# ------------------------------------------------------------

X_all_21 = core_df_07B[predictor_columns_07B].copy()

for column in numeric_columns_07B:
    X_all_21[column] = pd.to_numeric(
        X_all_21[column],
        errors="coerce",
    ).astype("float64")

for column in categorical_columns_07B:
    category_series = X_all_21[column].astype("object")
    X_all_21[column] = category_series.where(
        pd.notna(category_series),
        np.nan,
    )

outer_fold_vector_21 = (
    core_df_07B["outer_fold"].astype(int).to_numpy()
)

outer_training_mask_21 = (
    outer_fold_vector_21 != OUTER_FOLD_21
)
outer_test_mask_21 = (
    outer_fold_vector_21 == OUTER_FOLD_21
)

X_outer_training_21 = (
    X_all_21.loc[outer_training_mask_21]
    .reset_index(drop=True)
)

X_outer_test_21 = (
    X_all_21.loc[outer_test_mask_21]
    .reset_index(drop=True)
)

outer_training_meta_21 = (
    core_df_07B.loc[
        outer_training_mask_21,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_21 = (
    core_df_07B.loc[
        outer_test_mask_21,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [
    outer_training_meta_21,
    outer_test_meta_21,
]:
    dataframe["id_row"] = dataframe["id_row"].astype(str)
    dataframe["group_hospital"] = (
        dataframe["group_hospital"].astype(str)
    )
    dataframe["label_stage23"] = (
        dataframe["label_stage23"].astype(int)
    )

y_outer_training_21 = (
    outer_training_meta_21["label_stage23"]
    .to_numpy(dtype=np.int8)
)
y_outer_test_21 = (
    outer_test_meta_21["label_stage23"]
    .to_numpy(dtype=np.int8)
)

groups_outer_training_21 = (
    outer_training_meta_21["group_hospital"]
    .to_numpy(dtype=str)
)

training_hospitals_21 = set(
    outer_training_meta_21["group_hospital"]
)
test_hospitals_21 = set(
    outer_test_meta_21["group_hospital"]
)
hospital_overlap_21 = (
    training_hospitals_21 & test_hospitals_21
)

if hospital_overlap_21:
    raise RuntimeError(
        "Outer training/test hospital overlap detected."
    )

actual_split_21 = {
    "training_rows": len(X_outer_training_21),
    "test_rows": len(X_outer_test_21),
    "training_hospitals": len(training_hospitals_21),
    "test_hospitals": len(test_hospitals_21),
    "training_events": int(y_outer_training_21.sum()),
    "test_events": int(y_outer_test_21.sum()),
}

for metric, expected_value in EXPECTED_SPLIT_21.items():
    actual_value = actual_split_21[metric]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: found={actual_value}, "
            f"expected={expected_value}"
        )

inner_fold_vector_21 = np.array(
    [
        hospital_to_inner_fold_21.get(hospital, -1)
        for hospital in groups_outer_training_21
    ],
    dtype=int,
)

if (inner_fold_vector_21 == -1).any():
    raise RuntimeError(
        "Some outer-training hospitals have no inner-fold assignment."
    )

if set(np.unique(inner_fold_vector_21)) != {1, 2, 3, 4, 5}:
    raise RuntimeError("Inner-fold values are not exactly 1–5.")

# ------------------------------------------------------------
# 5. Preprocessor and model constructors
# ------------------------------------------------------------

def make_preprocessor_21():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns_07B,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns_07B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_random_forest_model_21(candidate):
    return RandomForestClassifier(
        n_estimators=int(candidate["n_estimators"]),
        criterion="gini",
        max_depth=(
            None
            if candidate["max_depth"] is None
            else int(candidate["max_depth"])
        ),
        min_samples_split=2,
        min_samples_leaf=int(candidate["min_samples_leaf"]),
        max_features=candidate["max_features"],
        bootstrap=True,
        max_samples=float(candidate["max_samples"]),
        class_weight=None,
        random_state=MODEL_RANDOM_SEED_21,
        n_jobs=-1,
        verbose=0,
    )


def checkpoint_table_id_21(inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_random_forest_inner_oof_outer3_inner{inner_fold}_v1"
    )

# ------------------------------------------------------------
# 6. BigQuery checkpoint verification
# ------------------------------------------------------------

def verify_checkpoint_21(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):
    table_id = checkpoint_table_id_21(inner_fold)

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS distinct_id_count,
      COUNT(DISTINCT candidate_id) AS candidate_count,
      COUNT(DISTINCT outer_fold) AS outer_fold_count,
      COUNT(DISTINCT inner_fold) AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(candidate_id, '|', id_row)
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(
        prediction_raw < 0 OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(
        sql,
        location=BQ_LOCATION,
    ).to_dataframe()

    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows * len(candidate_grid_21)
    )
    expected_positive_rows = (
        expected_validation_events * len(candidate_grid_21)
    )
    expected_negative_rows = (
        (expected_validation_rows - expected_validation_events)
        * len(candidate_grid_21)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": expected_validation_rows,
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": expected_total_rows,
        "positive_prediction_rows": expected_positive_rows,
        "negative_prediction_rows": expected_negative_rows,
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": OUTER_FOLD_21,
        "maximum_outer_fold": OUTER_FOLD_21,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failures = []

    for field, expected_value in expected_values.items():
        actual_value = int(row[field])
        if actual_value != expected_value:
            complete = False
            failures.append(
                f"{field}={actual_value}, expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failures),
        "check": check,
        "row": row,
    }

checkpoint_load_config_21 = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField(
            "id_row", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "outer_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "inner_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "candidate_id", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "label_stage23", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "prediction_raw", "FLOAT", mode="REQUIRED"
        ),
    ],
    write_disposition=(
        bigquery.WriteDisposition.WRITE_TRUNCATE
    ),
)

# ------------------------------------------------------------
# 7. Aggregate fit audit
# ------------------------------------------------------------

fit_audit_columns_21 = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "n_estimators",
    "max_depth",
    "min_samples_leaf",
    "max_features",
    "max_samples",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "fit_seconds",
]

fit_audit_path_21 = os.path.join(
    MODEL_OUTPUT_DIR,
    "21B_random_forest_inner_fit_audit_outer3.csv",
)

if os.path.exists(fit_audit_path_21):
    fit_audit_21 = pd.read_csv(fit_audit_path_21)
else:
    fit_audit_21 = pd.DataFrame(
        columns=fit_audit_columns_21
    )

for column in fit_audit_columns_21:
    if column not in fit_audit_21.columns:
        fit_audit_21[column] = np.nan

fit_audit_21 = fit_audit_21[
    fit_audit_columns_21
].copy()

# ------------------------------------------------------------
# 8. Run five inner folds
# ------------------------------------------------------------

for inner_fold in range(1, 6):
    inner_training_mask = (
        inner_fold_vector_21 != inner_fold
    )
    inner_validation_mask = (
        inner_fold_vector_21 == inner_fold
    )

    training_rows = int(inner_training_mask.sum())
    validation_rows = int(inner_validation_mask.sum())
    training_events = int(
        y_outer_training_21[inner_training_mask].sum()
    )
    validation_events = int(
        y_outer_training_21[inner_validation_mask].sum()
    )

    existing_check = verify_checkpoint_21(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if existing_check["complete"]:
        print(
            f"Outer 3 / inner {inner_fold}: "
            "permanent Random Forest checkpoint already complete; "
            "skipping model fitting."
        )
        continue

    training_hospital_set = set(
        groups_outer_training_21[inner_training_mask]
    )
    validation_hospital_set = set(
        groups_outer_training_21[inner_validation_mask]
    )

    if training_hospital_set & validation_hospital_set:
        raise RuntimeError(
            f"Inner fold {inner_fold}: hospital overlap detected."
        )

    print(f"\nOuter 3 / inner {inner_fold}")
    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_21()

    preprocessing_started = time.time()

    X_inner_training_processed = (
        preprocessor.fit_transform(
            X_outer_training_21.loc[
                inner_training_mask
            ]
        )
    )

    X_inner_validation_processed = (
        preprocessor.transform(
            X_outer_training_21.loc[
                inner_validation_mask
            ]
        )
    )

    preprocessing_seconds = (
        time.time() - preprocessing_started
    )

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "Processed training/validation column counts differ."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_seconds, 2),
    )

    y_inner_training = (
        y_outer_training_21[inner_training_mask]
    )
    y_inner_validation = (
        y_outer_training_21[inner_validation_mask]
    )

    validation_ids = (
        outer_training_meta_21.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_21:
        candidate_id = candidate["candidate_id"]

        print(
            "  Fitting",
            candidate_id,
            "| trees =",
            candidate["n_estimators"],
            "| depth =",
            candidate["max_depth"],
            "| min_leaf =",
            candidate["min_samples_leaf"],
            "| max_features =",
            candidate["max_features"],
            "| max_samples =",
            candidate["max_samples"],
        )

        model = make_random_forest_model_21(candidate)

        fit_started = time.time()

        model.fit(
            X_inner_training_processed,
            y_inner_training,
        )

        fit_seconds = time.time() - fit_started

        validation_probabilities = (
            model.predict_proba(
                X_inner_validation_processed
            )[:, 1]
        )

        if np.isnan(validation_probabilities).any():
            raise RuntimeError(
                f"{candidate_id}, inner {inner_fold}: "
                "missing predictions."
            )

        if not np.all(
            (validation_probabilities >= 0)
            & (validation_probabilities <= 1)
        ):
            raise RuntimeError(
                f"{candidate_id}: invalid probabilities."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        OUTER_FOLD_21,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": (
                        y_inner_validation.astype(np.int64)
                    ),
                    "prediction_raw": (
                        validation_probabilities.astype(
                            np.float64
                        )
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": OUTER_FOLD_21,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "n_estimators": candidate[
                    "n_estimators"
                ],
                "max_depth": candidate[
                    "max_depth"
                ],
                "min_samples_leaf": candidate[
                    "min_samples_leaf"
                ],
                "max_features": candidate[
                    "max_features"
                ],
                "max_samples": candidate[
                    "max_samples"
                ],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": (
                    validation_events
                ),
                "processed_columns": int(
                    X_inner_training_processed.shape[1]
                ),
                "fit_seconds": float(fit_seconds),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows * len(candidate_grid_21)
    )

    if len(checkpoint_df) != expected_checkpoint_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: invalid checkpoint row count."
        )

    if checkpoint_df.duplicated(
        subset=["id_row", "candidate_id"]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: duplicate candidate-patient rows."
        )

    target_checkpoint_table = (
        checkpoint_table_id_21(inner_fold)
    )

    print(
        "Uploading permanent Random Forest checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_21,
        location=BQ_LOCATION,
    ).result()

    if len(fit_audit_21) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_21["outer_fold"],
                    errors="coerce",
                ) == 1
            )
            & (
                pd.to_numeric(
                    fit_audit_21["inner_fold"],
                    errors="coerce",
                ) == inner_fold
            )
        )

        fit_audit_21 = (
            fit_audit_21.loc[keep_mask].copy()
        )

    fit_audit_21 = pd.concat(
        [
            fit_audit_21,
            pd.DataFrame(current_audit_rows),
        ],
        ignore_index=True,
    )

    fit_audit_21 = (
        fit_audit_21[
            fit_audit_columns_21
        ]
        .sort_values(
            [
                "outer_fold",
                "inner_fold",
                "candidate_id",
            ]
        )
        .reset_index(drop=True)
    )

    fit_audit_21.to_csv(
        fit_audit_path_21,
        index=False,
    )

    completed_check = verify_checkpoint_21(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint verification failed: "
            + completed_check["reason"]
        )

    print(
        f"Outer 3 / inner {inner_fold}: "
        "permanent Random Forest checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 9. Final inner checkpoint summary
# ------------------------------------------------------------

checkpoint_summary_rows_21 = []

for inner_fold in range(1, 6):
    validation_mask = (
        inner_fold_vector_21 == inner_fold
    )
    validation_rows = int(validation_mask.sum())
    validation_events = int(
        y_outer_training_21[
            validation_mask
        ].sum()
    )

    final_check = verify_checkpoint_21(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: final checkpoint audit failed. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_21.append(
        {
            "outer_fold": OUTER_FOLD_21,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(
                row["row_count"]
            ),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(
                row["candidate_count"]
            ),
            "positive_prediction_rows": int(
                row["positive_prediction_rows"]
            ),
            "negative_prediction_rows": int(
                row["negative_prediction_rows"]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check["table_id"],
        }
    )

checkpoint_summary_21 = (
    pd.DataFrame(checkpoint_summary_rows_21)
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_21[
        "distinct_validation_patients"
    ].sum()
) != EXPECTED_SPLIT_21["training_rows"]:
    raise RuntimeError(
        "Total inner validation patients != 46,803."
    )

expected_total_oof_rows_21 = (
    EXPECTED_SPLIT_21["training_rows"]
    * len(candidate_grid_21)
)

if int(
    checkpoint_summary_21[
        "checkpoint_rows"
    ].sum()
) != expected_total_oof_rows_21:
    raise RuntimeError(
        "Total Random Forest OOF prediction rows are incorrect."
    )

checkpoint_summary_path_21 = os.path.join(
    MODEL_OUTPUT_DIR,
    "21B_random_forest_outer3_inner_checkpoint_summary.csv",
)

checkpoint_summary_21.to_csv(
    checkpoint_summary_path_21,
    index=False,
)

# ------------------------------------------------------------
# 10. Pool five inner OOF tables
# ------------------------------------------------------------

checkpoint_tables_21 = [
    checkpoint_table_id_21(inner_fold)
    for inner_fold in range(1, 6)
]

union_parts_21 = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_21
]

SQL_LOAD_POOLED_OOF_21 = (
    "\nUNION ALL\n".join(union_parts_21)
)

print(
    "\nLoading pooled outer-fold-3 Random Forest inner OOF predictions..."
)

query_job_21 = client.query(
    SQL_LOAD_POOLED_OOF_21,
    location=BQ_LOCATION,
)

try:
    pooled_oof_21 = query_job_21.to_dataframe(
        create_bqstorage_client=True
    )
    pooled_load_method_21 = (
        "BigQuery Storage API"
    )
except Exception as fast_path_error_21:
    print(
        "Storage API unavailable; using standard BigQuery download."
    )
    print(
        "Message:",
        type(fast_path_error_21).__name__,
    )
    pooled_oof_21 = query_job_21.to_dataframe(
        create_bqstorage_client=False
    )
    pooled_load_method_21 = (
        "Standard BigQuery API"
    )

pooled_oof_21["id_row"] = (
    pooled_oof_21["id_row"].astype(str)
)
pooled_oof_21["candidate_id"] = (
    pooled_oof_21["candidate_id"].astype(str)
)

for column in [
    "outer_fold",
    "inner_fold",
    "label_stage23",
]:
    pooled_oof_21[column] = pd.to_numeric(
        pooled_oof_21[column],
        errors="raise",
    ).astype(int)

pooled_oof_21["prediction_raw"] = pd.to_numeric(
    pooled_oof_21["prediction_raw"],
    errors="raise",
).astype(float)

if len(pooled_oof_21) != expected_total_oof_rows_21:
    raise RuntimeError(
        "Pooled Random Forest OOF row count is incorrect."
    )

if pooled_oof_21.duplicated(
    subset=["candidate_id", "id_row"]
).any():
    raise RuntimeError(
        "Duplicate candidate-patient row in pooled Random Forest OOF."
    )

if pooled_oof_21["prediction_raw"].isna().any():
    raise RuntimeError("Missing Random Forest OOF prediction.")

if not pooled_oof_21[
    "prediction_raw"
].between(0, 1).all():
    raise RuntimeError(
        "Invalid Random Forest OOF probability."
    )

if set(
    pooled_oof_21["candidate_id"].unique()
) != {
    "RF01",
    "RF02",
    "RF03",
    "RF04",
    "RF05",
    "RF06",
}:
    raise RuntimeError(
        "The six locked Random Forest candidates are not all present."
    )

candidate_patient_counts_21 = (
    pooled_oof_21
    .groupby("candidate_id")["id_row"]
    .nunique()
)

if not (
    candidate_patient_counts_21
    == EXPECTED_SPLIT_21["training_rows"]
).all():
    raise RuntimeError(
        "Each candidate must have 46,803 OOF patients."
    )

candidate_event_counts_21 = (
    pooled_oof_21
    .groupby("candidate_id")["label_stage23"]
    .sum()
)

if not (
    candidate_event_counts_21
    == EXPECTED_SPLIT_21["training_events"]
).all():
    raise RuntimeError(
        "Each candidate must have 2,426 OOF events."
    )

# ------------------------------------------------------------
# 11. Metric helpers
# ------------------------------------------------------------

def probability_metrics_21(
    y_true,
    probabilities,
):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(
            roc_auc_score(y_true, probabilities)
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1],
            )
        ),
        "mean_predicted_risk": float(
            probabilities.mean()
        ),
        "observed_event_rate": float(
            np.mean(y_true)
        ),
    }


def probability_logit_21(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        probabilities / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_21(
    y_true,
    probabilities,
):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        probability_logit_21(probabilities),
        y_true,
    )

    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 12. Candidate pooled inner OOF metrics
# ------------------------------------------------------------

candidate_result_rows_21 = []

for candidate in candidate_grid_21:
    candidate_id = candidate["candidate_id"]

    candidate_oof = (
        pooled_oof_21.loc[
            pooled_oof_21["candidate_id"]
            == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_21(
        candidate_oof[
            "label_stage23"
        ].to_numpy(dtype=int),
        candidate_oof[
            "prediction_raw"
        ].to_numpy(dtype=float),
    )

    fit_part = fit_audit_21.loc[
        fit_audit_21[
            "candidate_id"
        ].astype(str) == candidate_id
    ]

    fit_seconds_total = (
        float(
            pd.to_numeric(
                fit_part["fit_seconds"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_21.append(
        {
            "candidate_id": candidate_id,
            **{
                key: candidate[key]
                for key in candidate
                if key != "candidate_id"
            },
            **metrics,
            "fit_seconds_total": (
                fit_seconds_total
            ),
        }
    )

candidate_results_21 = pd.DataFrame(
    candidate_result_rows_21
)

candidate_results_21 = (
    candidate_results_21
    .sort_values(
        ["auprc", "auroc", "brier"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

candidate_results_21["selection_rank"] = (
    np.arange(
        1,
        len(candidate_results_21) + 1,
    )
)

best_row_21 = candidate_results_21.iloc[0]
selected_candidate_id_21 = str(
    best_row_21["candidate_id"]
)

selected_candidate_21 = next(
    candidate
    for candidate in candidate_grid_21
    if candidate["candidate_id"]
    == selected_candidate_id_21
)

# ------------------------------------------------------------
# 13. Platt calibration from selected inner OOF
# ------------------------------------------------------------

selected_oof_21 = (
    pooled_oof_21.loc[
        pooled_oof_21["candidate_id"]
        == selected_candidate_id_21
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_21 = (
    selected_oof_21[
        "label_stage23"
    ].to_numpy(dtype=int)
)
selected_oof_probability_21 = (
    selected_oof_21[
        "prediction_raw"
    ].to_numpy(dtype=float)
)

platt_calibrator_21 = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_21.fit(
    probability_logit_21(
        selected_oof_probability_21
    ),
    selected_oof_y_21,
)

platt_intercept_21 = float(
    platt_calibrator_21.intercept_[0]
)
platt_slope_21 = float(
    platt_calibrator_21.coef_[0][0]
)

if (
    not np.isfinite(platt_intercept_21)
    or not np.isfinite(platt_slope_21)
    or platt_slope_21 <= 0
):
    raise RuntimeError(
        "Invalid Platt calibration coefficients."
    )

selected_model_21 = pd.DataFrame(
    [
        {
            "outer_fold": OUTER_FOLD_21,
            "selected_candidate": (
                selected_candidate_id_21
            ),
            "selection_metric_primary": (
                "pooled_inner_oof_auprc"
            ),
            "inner_oof_auprc": float(
                best_row_21["auprc"]
            ),
            "inner_oof_auroc": float(
                best_row_21["auroc"]
            ),
            "inner_oof_brier": float(
                best_row_21["brier"]
            ),
            "inner_oof_log_loss": float(
                best_row_21["log_loss"]
            ),
            "inner_oof_mean_predicted_risk": float(
                best_row_21[
                    "mean_predicted_risk"
                ]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_21[
                    "observed_event_rate"
                ]
            ),
            "platt_intercept": (
                platt_intercept_21
            ),
            "platt_slope": platt_slope_21,
            "protocol_sha256": (
                protocol_sha_21
            ),
            **{
                key: selected_candidate_21[key]
                for key in selected_candidate_21
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 14. Save locked selection
# ------------------------------------------------------------

candidate_results_path_21 = os.path.join(
    MODEL_OUTPUT_DIR,
    "21C_random_forest_candidate_results_outer3.csv",
)

selected_model_path_21 = os.path.join(
    MODEL_OUTPUT_DIR,
    "21C_random_forest_selected_model_outer3.csv",
)

selection_json_path_21 = os.path.join(
    MODEL_OUTPUT_DIR,
    "21C_random_forest_selection_calibration_outer3.json",
)

selection_sha_path_21 = os.path.join(
    MODEL_OUTPUT_DIR,
    "21C_random_forest_selection_calibration_outer3_SHA256.txt",
)

candidate_results_21.to_csv(
    candidate_results_path_21,
    index=False,
)
selected_model_21.to_csv(
    selected_model_path_21,
    index=False,
)

selection_configuration_21 = {
    "outer_fold": OUTER_FOLD_21,
    "protocol_sha256": protocol_sha_21,
    "selection_metric_primary": (
        "pooled inner out-of-fold AUPRC"
    ),
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "selected_candidate": (
        selected_candidate_id_21
    ),
    "selected_hyperparameters": {
        key: selected_candidate_21[key]
        for key in selected_candidate_21
        if key != "candidate_id"
    },
    "inner_oof_auprc": float(
        best_row_21["auprc"]
    ),
    "inner_oof_auroc": float(
        best_row_21["auroc"]
    ),
    "inner_oof_brier": float(
        best_row_21["brier"]
    ),
    "platt_intercept": platt_intercept_21,
    "platt_slope": platt_slope_21,
    "inner_checkpoint_tables": (
        checkpoint_tables_21
    ),
    "patient_level_oof_written_to_drive": False,
}

with open(
    selection_json_path_21,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        selection_configuration_21,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    selection_json_path_21,
    "rb",
) as fh:
    selection_sha_21 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    selection_sha_path_21,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(selection_sha_21 + "\n")

# ------------------------------------------------------------
# 15. Fit selected model on all outer training patients
# ------------------------------------------------------------

final_preprocessor_21 = (
    make_preprocessor_21()
)

print(
    "\nFitting selected outer-fold-3 Random Forest model "
    "on all 46,755 training patients..."
)

preprocess_started_21 = time.time()

X_outer_training_processed_21 = (
    final_preprocessor_21.fit_transform(
        X_outer_training_21
    )
)
X_outer_test_processed_21 = (
    final_preprocessor_21.transform(
        X_outer_test_21
    )
)

final_preprocessing_seconds_21 = (
    time.time() - preprocess_started_21
)

final_model_21 = make_random_forest_model_21(
    selected_candidate_21
)

final_fit_started_21 = time.time()

final_model_21.fit(
    X_outer_training_processed_21,
    y_outer_training_21,
)

final_fit_seconds_21 = (
    time.time() - final_fit_started_21
)

outer3_raw_probabilities_21 = (
    final_model_21.predict_proba(
        X_outer_test_processed_21
    )[:, 1]
)

raw_clipped_21 = np.clip(
    outer3_raw_probabilities_21,
    1e-6,
    1 - 1e-6,
)
raw_logit_21 = np.log(
    raw_clipped_21
    / (1 - raw_clipped_21)
)

outer3_platt_probabilities_21 = expit(
    platt_intercept_21
    + platt_slope_21 * raw_logit_21
)

for probabilities, name in [
    (outer3_raw_probabilities_21, "raw"),
    (
        outer3_platt_probabilities_21,
        "platt",
    ),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(
            f"{name} test predictions contain missing values."
        )

    if not np.all(
        (probabilities >= 0)
        & (probabilities <= 1)
    ):
        raise RuntimeError(
            f"{name} test predictions contain invalid probabilities."
        )

raw_metrics_21 = probability_metrics_21(
    y_outer_test_21,
    outer3_raw_probabilities_21,
)
platt_metrics_21 = probability_metrics_21(
    y_outer_test_21,
    outer3_platt_probabilities_21,
)

raw_calibration_intercept_21, \
raw_calibration_slope_21 = (
    calibration_intercept_slope_21(
        y_outer_test_21,
        outer3_raw_probabilities_21,
    )
)

platt_calibration_intercept_21, \
platt_calibration_slope_21 = (
    calibration_intercept_slope_21(
        y_outer_test_21,
        outer3_platt_probabilities_21,
    )
)

outer3_test_results_21 = pd.DataFrame(
    [
        {
            "outer_fold": OUTER_FOLD_21,
            "model": "random_forest",
            "probability_type": "raw",
            **raw_metrics_21,
            "calibration_intercept": (
                raw_calibration_intercept_21
            ),
            "calibration_slope": (
                raw_calibration_slope_21
            ),
        },
        {
            "outer_fold": OUTER_FOLD_21,
            "model": "random_forest",
            "probability_type": (
                "platt_calibrated"
            ),
            **platt_metrics_21,
            "calibration_intercept": (
                platt_calibration_intercept_21
            ),
            "calibration_slope": (
                platt_calibration_slope_21
            ),
        },
    ]
)

# ------------------------------------------------------------
# 16. Feature importance
# ------------------------------------------------------------

processed_feature_names_21 = (
    final_preprocessor_21
    .get_feature_names_out()
)

feature_importances_21 = (
    final_model_21.feature_importances_
)

if len(processed_feature_names_21) != len(
    feature_importances_21
):
    raise RuntimeError(
        "Processed feature names and Random Forest "
        "feature importances differ in length."
    )

feature_importance_table_21 = pd.DataFrame(
    {
        "processed_feature": (
            processed_feature_names_21
        ),
        "mdi_importance": (
            feature_importances_21
        ),
    }
)

feature_importance_table_21[
    "importance_rank"
] = (
    feature_importance_table_21[
        "mdi_importance"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)

feature_importance_table_21 = (
    feature_importance_table_21
    .sort_values(
        [
            "mdi_importance",
            "processed_feature",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

nonzero_importance_features_21 = int(
    (
        feature_importance_table_21[
            "mdi_importance"
        ] > 0
    ).sum()
)

final_model_summary_21 = pd.DataFrame(
    [
        {
            "outer_fold": OUTER_FOLD_21,
            "selected_candidate": (
                selected_candidate_id_21
            ),
            "training_patients": len(
                X_outer_training_21
            ),
            "training_hospitals": len(
                training_hospitals_21
            ),
            "training_events": int(
                y_outer_training_21.sum()
            ),
            "test_patients": len(
                X_outer_test_21
            ),
            "test_hospitals": len(
                test_hospitals_21
            ),
            "test_events": int(
                y_outer_test_21.sum()
            ),
            "hospital_overlap": len(
                hospital_overlap_21
            ),
            "processed_feature_columns": len(
                processed_feature_names_21
            ),
            "nonzero_importance_features": (
                nonzero_importance_features_21
            ),
            "preprocessing_seconds": float(
                final_preprocessing_seconds_21
            ),
            "fit_seconds": float(
                final_fit_seconds_21
            ),
            "locked_platt_intercept": (
                platt_intercept_21
            ),
            "locked_platt_slope": (
                platt_slope_21
            ),
            "protocol_sha256": protocol_sha_21,
            "selection_sha256": (
                selection_sha_21
            ),
            **{
                key: selected_candidate_21[key]
                for key in selected_candidate_21
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 17. Secure outer-test prediction checkpoint
# ------------------------------------------------------------

outer3_prediction_df_21 = pd.DataFrame(
    {
        "id_row": (
            outer_test_meta_21[
                "id_row"
            ].astype(str)
        ),
        "outer_fold": np.full(
            len(outer_test_meta_21),
            OUTER_FOLD_21,
            dtype=np.int64,
        ),
        "label_stage23": (
            y_outer_test_21.astype(np.int64)
        ),
        "prediction_raw": (
            outer3_raw_probabilities_21.astype(
                np.float64
            )
        ),
        "prediction_platt": (
            outer3_platt_probabilities_21.astype(
                np.float64
            )
        ),
        "model_name": "random_forest",
        "model_version": (
            "core_v1_nested_cv"
        ),
    }
)

if len(outer3_prediction_df_21) != 11736:
    raise RuntimeError(
        "Outer-fold-1 test prediction row count != 11,688."
    )

if outer3_prediction_df_21[
    "id_row"
].duplicated().any():
    raise RuntimeError(
        "Duplicate id_row in outer-fold-3 Random Forest predictions."
    )

if int(
    outer3_prediction_df_21[
        "label_stage23"
    ].sum()
) != 606:
    raise RuntimeError(
        "Outer-fold-1 Random Forest event count != 606."
    )

prediction_table_id_21 = (
    f"{TARGET_DATASET}."
    "model_random_forest_outer_predictions_outer3_v1"
)

prediction_load_config_21 = (
    bigquery.LoadJobConfig(
        schema=[
            bigquery.SchemaField(
                "id_row",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "outer_fold",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "label_stage23",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_raw",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_platt",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_name",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_version",
                "STRING",
                mode="REQUIRED",
            ),
        ],
        write_disposition=(
            bigquery.WriteDisposition.WRITE_TRUNCATE
        ),
    )
)

print(
    "\nUploading secure outer-fold-3 "
    "Random Forest prediction checkpoint:"
)
print(prediction_table_id_21)

client.load_table_from_dataframe(
    outer3_prediction_df_21,
    prediction_table_id_21,
    job_config=prediction_load_config_21,
    location=BQ_LOCATION,
).result()

SQL_VERIFY_PREDICTIONS_21 = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL)
    AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL)
    AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0
    OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0
    OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw)
    AS minimum_raw_probability,
  MAX(prediction_raw)
    AS maximum_raw_probability,
  MIN(prediction_platt)
    AS minimum_platt_probability,
  MAX(prediction_platt)
    AS maximum_platt_probability
FROM `{prediction_table_id_21}`;
"""

prediction_verification_21 = (
    client.query(
        SQL_VERIFY_PREDICTIONS_21,
        location=BQ_LOCATION,
    )
    .to_dataframe()
)

verification_row_21 = (
    prediction_verification_21.iloc[0]
)

expected_prediction_values_21 = {
    "prediction_rows": 11736,
    "distinct_rows": 11736,
    "outer_folds": 1,
    "minimum_outer_fold": OUTER_FOLD_21,
    "maximum_outer_fold": OUTER_FOLD_21,
    "events": 606,
    "nonevents": 11128,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in (
    expected_prediction_values_21.items()
):
    actual_value = int(
        verification_row_21[field]
    )
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: found={actual_value}, "
            f"expected={expected_value}"
        )

# ------------------------------------------------------------
# 18. Save aggregate outputs
# ------------------------------------------------------------

test_results_path_21 = os.path.join(
    MODEL_OUTPUT_DIR,
    "21D_random_forest_outer3_test_results.csv",
)

model_summary_path_21 = os.path.join(
    MODEL_OUTPUT_DIR,
    "21D_random_forest_final_model_outer3.csv",
)

importance_path_21 = os.path.join(
    MODEL_OUTPUT_DIR,
    "21D_random_forest_mdi_importance_outer3.csv",
)

evaluation_json_path_21 = os.path.join(
    MODEL_OUTPUT_DIR,
    "21D_random_forest_final_evaluation_outer3.json",
)

evaluation_sha_path_21 = os.path.join(
    MODEL_OUTPUT_DIR,
    "21D_random_forest_final_evaluation_outer3_SHA256.txt",
)

outer3_test_results_21.to_csv(
    test_results_path_21,
    index=False,
)
final_model_summary_21.to_csv(
    model_summary_path_21,
    index=False,
)
feature_importance_table_21.to_csv(
    importance_path_21,
    index=False,
)

evaluation_configuration_21 = {
    "outer_fold": OUTER_FOLD_21,
    "model_family": "random_forest",
    "protocol_sha256": protocol_sha_21,
    "selection_sha256": selection_sha_21,
    "selected_candidate": (
        selected_candidate_id_21
    ),
    "selected_hyperparameters": {
        key: selected_candidate_21[key]
        for key in selected_candidate_21
        if key != "candidate_id"
    },
    "training_patients": 46755,
    "training_hospitals": 158,
    "test_patients": 11736,
    "test_hospitals": 40,
    "hospital_overlap": 0,
    "locked_platt_intercept": (
        platt_intercept_21
    ),
    "locked_platt_slope": platt_slope_21,
    "processed_feature_columns": int(
        len(processed_feature_names_21)
    ),
    "secure_prediction_table": (
        prediction_table_id_21
    ),
    "patient_level_prediction_written_to_drive": False,
}

with open(
    evaluation_json_path_21,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        evaluation_configuration_21,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    evaluation_json_path_21,
    "rb",
) as fh:
    evaluation_sha_21 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    evaluation_sha_path_21,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(evaluation_sha_21 + "\n")

# ------------------------------------------------------------
# 19. Display results
# ------------------------------------------------------------

pooled_integrity_21 = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_21),
            pooled_oof_21[
                "id_row"
            ].nunique(),
            pooled_oof_21[
                "candidate_id"
            ].nunique(),
            pooled_oof_21[
                "inner_fold"
            ].nunique(),
            EXPECTED_SPLIT_21[
                "training_events"
            ],
            (
                EXPECTED_SPLIT_21[
                    "training_rows"
                ]
                - EXPECTED_SPLIT_21[
                    "training_events"
                ]
            ),
            int(
                pooled_oof_21.duplicated(
                    subset=[
                        "candidate_id",
                        "id_row",
                    ]
                ).sum()
            ),
            int(
                pooled_oof_21[
                    "prediction_raw"
                ].isna().sum()
            ),
            int(
                (
                    ~pooled_oof_21[
                        "prediction_raw"
                    ].between(0, 1)
                ).sum()
            ),
            pooled_load_method_21,
        ],
    }
)

print(
    "\n21 RANDOM FOREST OUTER-FOLD-3 INNER CHECKPOINT SUMMARY"
)
display(checkpoint_summary_21)

print(
    "\n21 RANDOM FOREST OUTER-FOLD-3 POOLED OOF INTEGRITY"
)
display(pooled_integrity_21)

print(
    "\n21 RANDOM FOREST OUTER-FOLD-3 CANDIDATE RESULTS"
)
display(candidate_results_21)

print(
    "\n21 RANDOM FOREST OUTER-FOLD-3 SELECTED MODEL"
)
display(selected_model_21)

print(
    "\n21 RANDOM FOREST OUTER-FOLD-3 FINAL MODEL SUMMARY"
)
display(final_model_summary_21)

print(
    "\n21 RANDOM FOREST OUTER-FOLD-3 TEST RESULTS"
)
display(outer3_test_results_21)

print(
    "\n21 RANDOM FOREST OUTER-FOLD-3 BIGQUERY VERIFICATION"
)
display(prediction_verification_21)

print(
    "\n21 RANDOM FOREST OUTER-FOLD-3 TOP 20 MDI IMPORTANCE FEATURES"
)
display(feature_importance_table_21.head(20))

print("\nRandom Forest protocol SHA-256:")
print(protocol_sha_21)

print("\nSelection SHA-256:")
print(selection_sha_21)

print("\nEvaluation SHA-256:")
print(evaluation_sha_21)

print("\nSaved aggregate outputs:")
print(protocol_path_21)
print(protocol_sha_path_21)
print(fit_audit_path_21)
print(checkpoint_summary_path_21)
print(candidate_results_path_21)
print(selected_model_path_21)
print(selection_json_path_21)
print(selection_sha_path_21)
print(test_results_path_21)
print(model_summary_path_21)
print(importance_path_21)
print(evaluation_json_path_21)
print(evaluation_sha_path_21)

print(
    "\n21 PASS: Random Forest outer-fold-3 nested modelling "
    "and locked test evaluation are complete."
)

print(
    "No class weighting or SMOTE was used."
)

print(
    "All patient-level OOF and outer-test predictions "
    "were stored only in BigQuery."
)

print(
    "No patient-level prediction file was written to Google Drive."
)

_ = gc.collect()

In [ ]:
# 21R — Resume Random Forest outer fold 3 after stale event-count guard fix
# IMPORTANT:
# Run this ONLY in the SAME Colab runtime immediately after the failed
# 21_random_forest_outer3_complete_nested_modeling.py execution.
# It does NOT refit the Random Forest model. It resumes from the already
# computed in-memory outer-fold-3 predictions and writes/verifies outputs.

import os
import json
import hashlib
import gc
import pandas as pd
from google.cloud import bigquery
from IPython.display import display

print("STARTING 21R RANDOM FOREST OUTER FOLD 3 — RESUME AFTER GUARD FIX")

required_globals_21r = [
    "OUTER_FOLD_21",
    "TARGET_DATASET",
    "BQ_LOCATION",
    "MODEL_OUTPUT_DIR",
    "client",
    "outer3_prediction_df_21",
    "outer3_test_results_21",
    "final_model_summary_21",
    "feature_importance_table_21",
    "checkpoint_summary_21",
    "candidate_results_21",
    "selected_model_21",
    "pooled_oof_21",
    "pooled_load_method_21",
    "EXPECTED_SPLIT_21",
    "protocol_sha_21",
    "selection_sha_21",
    "selected_candidate_id_21",
    "selected_candidate_21",
    "platt_intercept_21",
    "platt_slope_21",
    "processed_feature_names_21",
    "protocol_path_21",
    "protocol_sha_path_21",
    "fit_audit_path_21",
    "checkpoint_summary_path_21",
    "candidate_results_path_21",
    "selected_model_path_21",
    "selection_json_path_21",
    "selection_sha_path_21",
]

missing_21r = [name for name in required_globals_21r if name not in globals()]
if missing_21r:
    raise RuntimeError(
        "21R cannot resume because the Colab runtime no longer contains the "
        "required in-memory objects. Missing: " + ", ".join(missing_21r)
    )

if int(OUTER_FOLD_21) != 3:
    raise RuntimeError(f"Expected OUTER_FOLD_21=3, found {OUTER_FOLD_21!r}.")

if len(outer3_prediction_df_21) != 11736:
    raise RuntimeError(
        f"Outer-fold-3 prediction rows={len(outer3_prediction_df_21)}, expected 11736."
    )

if outer3_prediction_df_21["id_row"].duplicated().any():
    raise RuntimeError("Duplicate id_row in outer-fold-3 Random Forest predictions.")

event_count_21r = int(outer3_prediction_df_21["label_stage23"].sum())
if event_count_21r != 608:
    raise RuntimeError(
        f"Outer-fold-3 Random Forest event count={event_count_21r}, expected 608."
    )

nonevent_count_21r = int((outer3_prediction_df_21["label_stage23"] == 0).sum())
if nonevent_count_21r != 11128:
    raise RuntimeError(
        f"Outer-fold-3 Random Forest nonevent count={nonevent_count_21r}, expected 11128."
    )

if set(outer3_prediction_df_21["outer_fold"].astype(int).unique()) != {3}:
    raise RuntimeError("Outer-fold metadata is not exclusively 3.")

for col in ["prediction_raw", "prediction_platt"]:
    if outer3_prediction_df_21[col].isna().any():
        raise RuntimeError(f"Missing values found in {col}.")
    if (~outer3_prediction_df_21[col].between(0, 1)).any():
        raise RuntimeError(f"Invalid probabilities found in {col}.")

print(
    "In-memory outer-fold-3 predictions verified: "
    "11,736 patients | 608 events | 11,128 nonevents."
)
print("No model refit will be performed.")

prediction_table_id_21 = (
    f"{TARGET_DATASET}."
    "model_random_forest_outer_predictions_outer3_v1"
)

prediction_load_config_21 = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("prediction_platt", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("model_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("model_version", "STRING", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

print("\nUploading secure outer-fold-3 Random Forest prediction checkpoint:")
print(prediction_table_id_21)

client.load_table_from_dataframe(
    outer3_prediction_df_21,
    prediction_table_id_21,
    job_config=prediction_load_config_21,
    location=BQ_LOCATION,
).result()

SQL_VERIFY_PREDICTIONS_21 = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL) AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL) AS missing_platt_predictions,
  COUNTIF(prediction_raw < 0 OR prediction_raw > 1) AS invalid_raw_predictions,
  COUNTIF(prediction_platt < 0 OR prediction_platt > 1) AS invalid_platt_predictions,
  MIN(prediction_raw) AS minimum_raw_probability,
  MAX(prediction_raw) AS maximum_raw_probability,
  MIN(prediction_platt) AS minimum_platt_probability,
  MAX(prediction_platt) AS maximum_platt_probability
FROM `{prediction_table_id_21}`;
"""

prediction_verification_21 = (
    client.query(SQL_VERIFY_PREDICTIONS_21, location=BQ_LOCATION)
    .to_dataframe()
)

verification_row_21 = prediction_verification_21.iloc[0]

expected_prediction_values_21 = {
    "prediction_rows": 11736,
    "distinct_rows": 11736,
    "outer_folds": 1,
    "minimum_outer_fold": 3,
    "maximum_outer_fold": 3,
    "events": 608,
    "nonevents": 11128,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in expected_prediction_values_21.items():
    actual_value = int(verification_row_21[field])
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: found={actual_value}, expected={expected_value}"
        )

test_results_path_21 = os.path.join(
    MODEL_OUTPUT_DIR,
    "21D_random_forest_outer3_test_results.csv",
)
model_summary_path_21 = os.path.join(
    MODEL_OUTPUT_DIR,
    "21D_random_forest_final_model_outer3.csv",
)
importance_path_21 = os.path.join(
    MODEL_OUTPUT_DIR,
    "21D_random_forest_mdi_importance_outer3.csv",
)
evaluation_json_path_21 = os.path.join(
    MODEL_OUTPUT_DIR,
    "21D_random_forest_final_evaluation_outer3.json",
)
evaluation_sha_path_21 = os.path.join(
    MODEL_OUTPUT_DIR,
    "21D_random_forest_final_evaluation_outer3_SHA256.txt",
)

outer3_test_results_21.to_csv(test_results_path_21, index=False)
final_model_summary_21.to_csv(model_summary_path_21, index=False)
feature_importance_table_21.to_csv(importance_path_21, index=False)

evaluation_configuration_21 = {
    "outer_fold": 3,
    "model_family": "random_forest",
    "protocol_sha256": protocol_sha_21,
    "selection_sha256": selection_sha_21,
    "selected_candidate": selected_candidate_id_21,
    "selected_hyperparameters": {
        key: selected_candidate_21[key]
        for key in selected_candidate_21
        if key != "candidate_id"
    },
    "training_patients": 46755,
    "training_hospitals": 158,
    "test_patients": 11736,
    "test_hospitals": 40,
    "hospital_overlap": 0,
    "locked_platt_intercept": platt_intercept_21,
    "locked_platt_slope": platt_slope_21,
    "processed_feature_columns": int(len(processed_feature_names_21)),
    "secure_prediction_table": prediction_table_id_21,
    "patient_level_prediction_written_to_drive": False,
}

with open(evaluation_json_path_21, "w", encoding="utf-8") as fh:
    json.dump(
        evaluation_configuration_21,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(evaluation_json_path_21, "rb") as fh:
    evaluation_sha_21 = hashlib.sha256(fh.read()).hexdigest()

with open(evaluation_sha_path_21, "w", encoding="utf-8") as fh:
    fh.write(evaluation_sha_21 + "\n")

pooled_integrity_21 = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_21),
            pooled_oof_21["id_row"].nunique(),
            pooled_oof_21["candidate_id"].nunique(),
            pooled_oof_21["inner_fold"].nunique(),
            EXPECTED_SPLIT_21["training_events"],
            EXPECTED_SPLIT_21["training_rows"] - EXPECTED_SPLIT_21["training_events"],
            int(
                pooled_oof_21.duplicated(
                    subset=["candidate_id", "id_row"]
                ).sum()
            ),
            int(pooled_oof_21["prediction_raw"].isna().sum()),
            int((~pooled_oof_21["prediction_raw"].between(0, 1)).sum()),
            pooled_load_method_21,
        ],
    }
)

print("\n21 RANDOM FOREST OUTER-FOLD-3 INNER CHECKPOINT SUMMARY")
display(checkpoint_summary_21)

print("\n21 RANDOM FOREST OUTER-FOLD-3 POOLED OOF INTEGRITY")
display(pooled_integrity_21)

print("\n21 RANDOM FOREST OUTER-FOLD-3 CANDIDATE RESULTS")
display(candidate_results_21)

print("\n21 RANDOM FOREST OUTER-FOLD-3 SELECTED MODEL")
display(selected_model_21)

print("\n21 RANDOM FOREST OUTER-FOLD-3 FINAL MODEL SUMMARY")
display(final_model_summary_21)

print("\n21 RANDOM FOREST OUTER-FOLD-3 TEST RESULTS")
display(outer3_test_results_21)

print("\n21 RANDOM FOREST OUTER-FOLD-3 BIGQUERY VERIFICATION")
display(prediction_verification_21)

print("\n21 RANDOM FOREST OUTER-FOLD-3 TOP 20 MDI IMPORTANCE FEATURES")
display(feature_importance_table_21.head(20))

print("\nRandom Forest protocol SHA-256:")
print(protocol_sha_21)

print("\nSelection SHA-256:")
print(selection_sha_21)

print("\nEvaluation SHA-256:")
print(evaluation_sha_21)

print("\nSaved aggregate outputs:")
for p in [
    protocol_path_21,
    protocol_sha_path_21,
    fit_audit_path_21,
    checkpoint_summary_path_21,
    candidate_results_path_21,
    selected_model_path_21,
    selection_json_path_21,
    selection_sha_path_21,
    test_results_path_21,
    model_summary_path_21,
    importance_path_21,
    evaluation_json_path_21,
    evaluation_sha_path_21,
]:
    print(p)

print(
    "\n21R PASS: Random Forest outer-fold-3 post-fit resume, "
    "secure BigQuery write, verification, and aggregate output save are complete."
)
print("No Random Forest model was refit by 21R.")
print("No class weighting or SMOTE was used.")
print("No patient-level prediction file was written to Google Drive.")

_ = gc.collect()

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

from sklearn.ensemble import RandomForestClassifier

from IPython.display import display

print("STARTING RANDOM FOREST OUTER FOLD 4 — CODE VERSION 22")

# ============================================================
# 22 — RANDOM FOREST OUTER FOLD 4 COMPLETE NESTED MODELLING
#
# Design:
# - Same locked hospital-disjoint outer folds.
# - Same locked hospital-disjoint inner folds.
# - Core feature set only.
# - No SMOTE.
# - No class weighting.
# - Fixed Random Forest candidate grid locked before outer-test evaluation.
# - Selection: pooled inner-OOF AUPRC descending,
#              AUROC descending, Brier ascending.
# - Platt calibration learned only from selected candidate's
#   pooled inner-OOF predictions.
# - Patient-level predictions stored only in BigQuery.
# ============================================================

OUTER_FOLD_22 = 4
MODEL_RANDOM_SEED_22 = 20260721

EXPECTED_SPLIT_22 = {
    "training_rows": 46803,
    "test_rows": 11688,
    "training_hospitals": 159,
    "test_hospitals": 39,
    "training_events": 2426,
    "test_events": 606,
}

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_objects_22 = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_22 = [
    name for name in required_objects_22
    if name not in globals()
]

if missing_objects_22:
    raise RuntimeError(
        "Missing runtime objects: "
        + ", ".join(missing_objects_22)
        + ". Run 07A and 07B first."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"Expected 58,491 cohort rows; found {len(core_df_07B)}."
    )

if len(predictor_columns_07B) != 159:
    raise RuntimeError("Expected 159 core predictors.")

if len(numeric_columns_07B) != 156:
    raise RuntimeError("Expected 156 numeric predictors.")

if len(categorical_columns_07B) != 3:
    raise RuntimeError("Expected 3 categorical predictors.")

# ------------------------------------------------------------
# 2. Lock the Random Forest protocol BEFORE test evaluation
# ------------------------------------------------------------

candidate_grid_22 = [
    {
        "candidate_id": "RF01",
        "n_estimators": 300,
        "max_depth": 8,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "max_samples": 0.80,
    },
    {
        "candidate_id": "RF02",
        "n_estimators": 400,
        "max_depth": 12,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
        "max_samples": 0.80,
    },
    {
        "candidate_id": "RF03",
        "n_estimators": 500,
        "max_depth": None,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
        "max_samples": 0.85,
    },
    {
        "candidate_id": "RF04",
        "n_estimators": 400,
        "max_depth": 12,
        "min_samples_leaf": 10,
        "max_features": 0.25,
        "max_samples": 0.85,
    },
    {
        "candidate_id": "RF05",
        "n_estimators": 500,
        "max_depth": None,
        "min_samples_leaf": 10,
        "max_features": 0.25,
        "max_samples": 0.90,
    },
    {
        "candidate_id": "RF06",
        "n_estimators": 600,
        "max_depth": 16,
        "min_samples_leaf": 20,
        "max_features": 0.50,
        "max_samples": 0.90,
    },
]

rf_protocol_22 = {
    "protocol_name": "random_forest_core_nested_hospital_cv_v1",
    "model_family": "RandomForestClassifier",
    "feature_set": "core_159",
    "outer_cv": "locked 5-fold hospital-disjoint outer folds",
    "inner_cv": "locked 5-fold hospital-disjoint inner folds",
    "primary_selection_metric": "pooled inner OOF AUPRC descending",
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "criterion": "gini",
    "bootstrap": True,
    "class_weighting": False,
    "class_weight": None,
    "smote": False,
    "min_samples_split": 2,
    "random_state": MODEL_RANDOM_SEED_22,
    "candidate_grid": candidate_grid_22,
    "calibration": (
        "Platt calibration fit only on selected candidate "
        "pooled inner-OOF logits"
    ),
    "outer_test_use": (
        "diagnostic evaluation only; never used for tuning"
    ),
}

protocol_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "19A_locked_random_forest_model_protocol_v1.json",
)

protocol_sha_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "19A_locked_random_forest_model_protocol_v1_SHA256.txt",
)

protocol_text_22 = json.dumps(
    rf_protocol_22,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
)

protocol_sha_22 = hashlib.sha256(
    protocol_text_22.encode("utf-8")
).hexdigest()

if os.path.exists(protocol_path_22):
    with open(protocol_path_22, "r", encoding="utf-8") as fh:
        existing_protocol_text_22 = fh.read()
    existing_protocol_sha_22 = hashlib.sha256(
        existing_protocol_text_22.encode("utf-8")
    ).hexdigest()

    if existing_protocol_sha_22 != protocol_sha_22:
        raise RuntimeError(
            "An existing Random Forest protocol file differs from "
            "the currently locked protocol. Stop and audit."
        )
else:
    with open(protocol_path_22, "w", encoding="utf-8") as fh:
        fh.write(protocol_text_22)

with open(protocol_sha_path_22, "w", encoding="utf-8") as fh:
    fh.write(protocol_sha_22 + "\n")

print("Locked Random Forest protocol SHA-256:")
print(protocol_sha_22)

# ------------------------------------------------------------
# 3. Load locked inner hospital map
# ------------------------------------------------------------

inner_mapping_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_22):
    raise FileNotFoundError(
        "Locked inner-fold map not found: "
        + inner_mapping_path_22
    )

inner_mapping_all_22 = pd.read_csv(
    inner_mapping_path_22,
    dtype={"group_hospital": str},
)

inner_mapping_part_22 = (
    inner_mapping_all_22.loc[
        inner_mapping_all_22["outer_fold"].astype(int)
        == OUTER_FOLD_22,
        ["group_hospital", "inner_fold"],
    ]
    .copy()
)

inner_mapping_part_22["group_hospital"] = (
    inner_mapping_part_22["group_hospital"].astype(str)
)
inner_mapping_part_22["inner_fold"] = (
    inner_mapping_part_22["inner_fold"].astype(int)
)

if len(inner_mapping_part_22) != 158:
    raise RuntimeError(
        "Expected 158 outer-fold-4 training hospitals "
        "in the locked inner map."
    )

if inner_mapping_part_22["group_hospital"].duplicated().any():
    raise RuntimeError("Duplicate hospital in locked inner map.")

hospital_to_inner_fold_22 = dict(
    zip(
        inner_mapping_part_22["group_hospital"],
        inner_mapping_part_22["inner_fold"],
    )
)

# ------------------------------------------------------------
# 4. Prepare outer fold 4 matrices
# ------------------------------------------------------------

X_all_22 = core_df_07B[predictor_columns_07B].copy()

for column in numeric_columns_07B:
    X_all_22[column] = pd.to_numeric(
        X_all_22[column],
        errors="coerce",
    ).astype("float64")

for column in categorical_columns_07B:
    category_series = X_all_22[column].astype("object")
    X_all_22[column] = category_series.where(
        pd.notna(category_series),
        np.nan,
    )

outer_fold_vector_22 = (
    core_df_07B["outer_fold"].astype(int).to_numpy()
)

outer_training_mask_22 = (
    outer_fold_vector_22 != OUTER_FOLD_22
)
outer_test_mask_22 = (
    outer_fold_vector_22 == OUTER_FOLD_22
)

X_outer_training_22 = (
    X_all_22.loc[outer_training_mask_22]
    .reset_index(drop=True)
)

X_outer_test_22 = (
    X_all_22.loc[outer_test_mask_22]
    .reset_index(drop=True)
)

outer_training_meta_22 = (
    core_df_07B.loc[
        outer_training_mask_22,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_22 = (
    core_df_07B.loc[
        outer_test_mask_22,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [
    outer_training_meta_22,
    outer_test_meta_22,
]:
    dataframe["id_row"] = dataframe["id_row"].astype(str)
    dataframe["group_hospital"] = (
        dataframe["group_hospital"].astype(str)
    )
    dataframe["label_stage23"] = (
        dataframe["label_stage23"].astype(int)
    )

y_outer_training_22 = (
    outer_training_meta_22["label_stage23"]
    .to_numpy(dtype=np.int8)
)
y_outer_test_22 = (
    outer_test_meta_22["label_stage23"]
    .to_numpy(dtype=np.int8)
)

groups_outer_training_22 = (
    outer_training_meta_22["group_hospital"]
    .to_numpy(dtype=str)
)

training_hospitals_22 = set(
    outer_training_meta_22["group_hospital"]
)
test_hospitals_22 = set(
    outer_test_meta_22["group_hospital"]
)
hospital_overlap_22 = (
    training_hospitals_22 & test_hospitals_22
)

if hospital_overlap_22:
    raise RuntimeError(
        "Outer training/test hospital overlap detected."
    )

actual_split_22 = {
    "training_rows": len(X_outer_training_22),
    "test_rows": len(X_outer_test_22),
    "training_hospitals": len(training_hospitals_22),
    "test_hospitals": len(test_hospitals_22),
    "training_events": int(y_outer_training_22.sum()),
    "test_events": int(y_outer_test_22.sum()),
}

for metric, expected_value in EXPECTED_SPLIT_22.items():
    actual_value = actual_split_22[metric]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: found={actual_value}, "
            f"expected={expected_value}"
        )

inner_fold_vector_22 = np.array(
    [
        hospital_to_inner_fold_22.get(hospital, -1)
        for hospital in groups_outer_training_22
    ],
    dtype=int,
)

if (inner_fold_vector_22 == -1).any():
    raise RuntimeError(
        "Some outer-training hospitals have no inner-fold assignment."
    )

if set(np.unique(inner_fold_vector_22)) != {1, 2, 3, 4, 5}:
    raise RuntimeError("Inner-fold values are not exactly 1–5.")

# ------------------------------------------------------------
# 5. Preprocessor and model constructors
# ------------------------------------------------------------

def make_preprocessor_22():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns_07B,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns_07B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_random_forest_model_22(candidate):
    return RandomForestClassifier(
        n_estimators=int(candidate["n_estimators"]),
        criterion="gini",
        max_depth=(
            None
            if candidate["max_depth"] is None
            else int(candidate["max_depth"])
        ),
        min_samples_split=2,
        min_samples_leaf=int(candidate["min_samples_leaf"]),
        max_features=candidate["max_features"],
        bootstrap=True,
        max_samples=float(candidate["max_samples"]),
        class_weight=None,
        random_state=MODEL_RANDOM_SEED_22,
        n_jobs=-1,
        verbose=0,
    )


def checkpoint_table_id_22(inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_random_forest_inner_oof_outer4_inner{inner_fold}_v1"
    )

# ------------------------------------------------------------
# 6. BigQuery checkpoint verification
# ------------------------------------------------------------

def verify_checkpoint_22(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):
    table_id = checkpoint_table_id_22(inner_fold)

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS distinct_id_count,
      COUNT(DISTINCT candidate_id) AS candidate_count,
      COUNT(DISTINCT outer_fold) AS outer_fold_count,
      COUNT(DISTINCT inner_fold) AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(candidate_id, '|', id_row)
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(
        prediction_raw < 0 OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(
        sql,
        location=BQ_LOCATION,
    ).to_dataframe()

    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows * len(candidate_grid_22)
    )
    expected_positive_rows = (
        expected_validation_events * len(candidate_grid_22)
    )
    expected_negative_rows = (
        (expected_validation_rows - expected_validation_events)
        * len(candidate_grid_22)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": expected_validation_rows,
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": expected_total_rows,
        "positive_prediction_rows": expected_positive_rows,
        "negative_prediction_rows": expected_negative_rows,
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": OUTER_FOLD_22,
        "maximum_outer_fold": OUTER_FOLD_22,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failures = []

    for field, expected_value in expected_values.items():
        actual_value = int(row[field])
        if actual_value != expected_value:
            complete = False
            failures.append(
                f"{field}={actual_value}, expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failures),
        "check": check,
        "row": row,
    }

checkpoint_load_config_22 = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField(
            "id_row", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "outer_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "inner_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "candidate_id", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "label_stage23", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "prediction_raw", "FLOAT", mode="REQUIRED"
        ),
    ],
    write_disposition=(
        bigquery.WriteDisposition.WRITE_TRUNCATE
    ),
)

# ------------------------------------------------------------
# 7. Aggregate fit audit
# ------------------------------------------------------------

fit_audit_columns_22 = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "n_estimators",
    "max_depth",
    "min_samples_leaf",
    "max_features",
    "max_samples",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "fit_seconds",
]

fit_audit_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22B_random_forest_inner_fit_audit_outer4.csv",
)

if os.path.exists(fit_audit_path_22):
    fit_audit_22 = pd.read_csv(fit_audit_path_22)
else:
    fit_audit_22 = pd.DataFrame(
        columns=fit_audit_columns_22
    )

for column in fit_audit_columns_22:
    if column not in fit_audit_22.columns:
        fit_audit_22[column] = np.nan

fit_audit_22 = fit_audit_22[
    fit_audit_columns_22
].copy()

# ------------------------------------------------------------
# 8. Run five inner folds
# ------------------------------------------------------------

for inner_fold in range(1, 6):
    inner_training_mask = (
        inner_fold_vector_22 != inner_fold
    )
    inner_validation_mask = (
        inner_fold_vector_22 == inner_fold
    )

    training_rows = int(inner_training_mask.sum())
    validation_rows = int(inner_validation_mask.sum())
    training_events = int(
        y_outer_training_22[inner_training_mask].sum()
    )
    validation_events = int(
        y_outer_training_22[inner_validation_mask].sum()
    )

    existing_check = verify_checkpoint_22(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if existing_check["complete"]:
        print(
            f"Outer 4 / inner {inner_fold}: "
            "permanent Random Forest checkpoint already complete; "
            "skipping model fitting."
        )
        continue

    training_hospital_set = set(
        groups_outer_training_22[inner_training_mask]
    )
    validation_hospital_set = set(
        groups_outer_training_22[inner_validation_mask]
    )

    if training_hospital_set & validation_hospital_set:
        raise RuntimeError(
            f"Inner fold {inner_fold}: hospital overlap detected."
        )

    print(f"\nOuter 4 / inner {inner_fold}")
    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_22()

    preprocessing_started = time.time()

    X_inner_training_processed = (
        preprocessor.fit_transform(
            X_outer_training_22.loc[
                inner_training_mask
            ]
        )
    )

    X_inner_validation_processed = (
        preprocessor.transform(
            X_outer_training_22.loc[
                inner_validation_mask
            ]
        )
    )

    preprocessing_seconds = (
        time.time() - preprocessing_started
    )

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "Processed training/validation column counts differ."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_seconds, 2),
    )

    y_inner_training = (
        y_outer_training_22[inner_training_mask]
    )
    y_inner_validation = (
        y_outer_training_22[inner_validation_mask]
    )

    validation_ids = (
        outer_training_meta_22.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_22:
        candidate_id = candidate["candidate_id"]

        print(
            "  Fitting",
            candidate_id,
            "| trees =",
            candidate["n_estimators"],
            "| depth =",
            candidate["max_depth"],
            "| min_leaf =",
            candidate["min_samples_leaf"],
            "| max_features =",
            candidate["max_features"],
            "| max_samples =",
            candidate["max_samples"],
        )

        model = make_random_forest_model_22(candidate)

        fit_started = time.time()

        model.fit(
            X_inner_training_processed,
            y_inner_training,
        )

        fit_seconds = time.time() - fit_started

        validation_probabilities = (
            model.predict_proba(
                X_inner_validation_processed
            )[:, 1]
        )

        if np.isnan(validation_probabilities).any():
            raise RuntimeError(
                f"{candidate_id}, inner {inner_fold}: "
                "missing predictions."
            )

        if not np.all(
            (validation_probabilities >= 0)
            & (validation_probabilities <= 1)
        ):
            raise RuntimeError(
                f"{candidate_id}: invalid probabilities."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        OUTER_FOLD_22,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": (
                        y_inner_validation.astype(np.int64)
                    ),
                    "prediction_raw": (
                        validation_probabilities.astype(
                            np.float64
                        )
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": OUTER_FOLD_22,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "n_estimators": candidate[
                    "n_estimators"
                ],
                "max_depth": candidate[
                    "max_depth"
                ],
                "min_samples_leaf": candidate[
                    "min_samples_leaf"
                ],
                "max_features": candidate[
                    "max_features"
                ],
                "max_samples": candidate[
                    "max_samples"
                ],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": (
                    validation_events
                ),
                "processed_columns": int(
                    X_inner_training_processed.shape[1]
                ),
                "fit_seconds": float(fit_seconds),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows * len(candidate_grid_22)
    )

    if len(checkpoint_df) != expected_checkpoint_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: invalid checkpoint row count."
        )

    if checkpoint_df.duplicated(
        subset=["id_row", "candidate_id"]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: duplicate candidate-patient rows."
        )

    target_checkpoint_table = (
        checkpoint_table_id_22(inner_fold)
    )

    print(
        "Uploading permanent Random Forest checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_22,
        location=BQ_LOCATION,
    ).result()

    if len(fit_audit_22) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_22["outer_fold"],
                    errors="coerce",
                ) == 1
            )
            & (
                pd.to_numeric(
                    fit_audit_22["inner_fold"],
                    errors="coerce",
                ) == inner_fold
            )
        )

        fit_audit_22 = (
            fit_audit_22.loc[keep_mask].copy()
        )

    fit_audit_22 = pd.concat(
        [
            fit_audit_22,
            pd.DataFrame(current_audit_rows),
        ],
        ignore_index=True,
    )

    fit_audit_22 = (
        fit_audit_22[
            fit_audit_columns_22
        ]
        .sort_values(
            [
                "outer_fold",
                "inner_fold",
                "candidate_id",
            ]
        )
        .reset_index(drop=True)
    )

    fit_audit_22.to_csv(
        fit_audit_path_22,
        index=False,
    )

    completed_check = verify_checkpoint_22(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint verification failed: "
            + completed_check["reason"]
        )

    print(
        f"Outer 4 / inner {inner_fold}: "
        "permanent Random Forest checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 9. Final inner checkpoint summary
# ------------------------------------------------------------

checkpoint_summary_rows_22 = []

for inner_fold in range(1, 6):
    validation_mask = (
        inner_fold_vector_22 == inner_fold
    )
    validation_rows = int(validation_mask.sum())
    validation_events = int(
        y_outer_training_22[
            validation_mask
        ].sum()
    )

    final_check = verify_checkpoint_22(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: final checkpoint audit failed. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_22.append(
        {
            "outer_fold": OUTER_FOLD_22,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(
                row["row_count"]
            ),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(
                row["candidate_count"]
            ),
            "positive_prediction_rows": int(
                row["positive_prediction_rows"]
            ),
            "negative_prediction_rows": int(
                row["negative_prediction_rows"]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check["table_id"],
        }
    )

checkpoint_summary_22 = (
    pd.DataFrame(checkpoint_summary_rows_22)
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_22[
        "distinct_validation_patients"
    ].sum()
) != EXPECTED_SPLIT_22["training_rows"]:
    raise RuntimeError(
        "Total inner validation patients != 46,803."
    )

expected_total_oof_rows_22 = (
    EXPECTED_SPLIT_22["training_rows"]
    * len(candidate_grid_22)
)

if int(
    checkpoint_summary_22[
        "checkpoint_rows"
    ].sum()
) != expected_total_oof_rows_22:
    raise RuntimeError(
        "Total Random Forest OOF prediction rows are incorrect."
    )

checkpoint_summary_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22B_random_forest_outer4_inner_checkpoint_summary.csv",
)

checkpoint_summary_22.to_csv(
    checkpoint_summary_path_22,
    index=False,
)

# ------------------------------------------------------------
# 10. Pool five inner OOF tables
# ------------------------------------------------------------

checkpoint_tables_22 = [
    checkpoint_table_id_22(inner_fold)
    for inner_fold in range(1, 6)
]

union_parts_22 = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_22
]

SQL_LOAD_POOLED_OOF_22 = (
    "\nUNION ALL\n".join(union_parts_22)
)

print(
    "\nLoading pooled outer-fold-4 Random Forest inner OOF predictions..."
)

query_job_22 = client.query(
    SQL_LOAD_POOLED_OOF_22,
    location=BQ_LOCATION,
)

try:
    pooled_oof_22 = query_job_22.to_dataframe(
        create_bqstorage_client=True
    )
    pooled_load_method_22 = (
        "BigQuery Storage API"
    )
except Exception as fast_path_error_22:
    print(
        "Storage API unavailable; using standard BigQuery download."
    )
    print(
        "Message:",
        type(fast_path_error_22).__name__,
    )
    pooled_oof_22 = query_job_22.to_dataframe(
        create_bqstorage_client=False
    )
    pooled_load_method_22 = (
        "Standard BigQuery API"
    )

pooled_oof_22["id_row"] = (
    pooled_oof_22["id_row"].astype(str)
)
pooled_oof_22["candidate_id"] = (
    pooled_oof_22["candidate_id"].astype(str)
)

for column in [
    "outer_fold",
    "inner_fold",
    "label_stage23",
]:
    pooled_oof_22[column] = pd.to_numeric(
        pooled_oof_22[column],
        errors="raise",
    ).astype(int)

pooled_oof_22["prediction_raw"] = pd.to_numeric(
    pooled_oof_22["prediction_raw"],
    errors="raise",
).astype(float)

if len(pooled_oof_22) != expected_total_oof_rows_22:
    raise RuntimeError(
        "Pooled Random Forest OOF row count is incorrect."
    )

if pooled_oof_22.duplicated(
    subset=["candidate_id", "id_row"]
).any():
    raise RuntimeError(
        "Duplicate candidate-patient row in pooled Random Forest OOF."
    )

if pooled_oof_22["prediction_raw"].isna().any():
    raise RuntimeError("Missing Random Forest OOF prediction.")

if not pooled_oof_22[
    "prediction_raw"
].between(0, 1).all():
    raise RuntimeError(
        "Invalid Random Forest OOF probability."
    )

if set(
    pooled_oof_22["candidate_id"].unique()
) != {
    "RF01",
    "RF02",
    "RF03",
    "RF04",
    "RF05",
    "RF06",
}:
    raise RuntimeError(
        "The six locked Random Forest candidates are not all present."
    )

candidate_patient_counts_22 = (
    pooled_oof_22
    .groupby("candidate_id")["id_row"]
    .nunique()
)

if not (
    candidate_patient_counts_22
    == EXPECTED_SPLIT_22["training_rows"]
).all():
    raise RuntimeError(
        "Each candidate must have 46,803 OOF patients."
    )

candidate_event_counts_22 = (
    pooled_oof_22
    .groupby("candidate_id")["label_stage23"]
    .sum()
)

if not (
    candidate_event_counts_22
    == EXPECTED_SPLIT_22["training_events"]
).all():
    raise RuntimeError(
        "Each candidate must have 2,426 OOF events."
    )

# ------------------------------------------------------------
# 11. Metric helpers
# ------------------------------------------------------------

def probability_metrics_22(
    y_true,
    probabilities,
):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(
            roc_auc_score(y_true, probabilities)
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1],
            )
        ),
        "mean_predicted_risk": float(
            probabilities.mean()
        ),
        "observed_event_rate": float(
            np.mean(y_true)
        ),
    }


def probability_logit_22(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        probabilities / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_22(
    y_true,
    probabilities,
):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        probability_logit_22(probabilities),
        y_true,
    )

    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 12. Candidate pooled inner OOF metrics
# ------------------------------------------------------------

candidate_result_rows_22 = []

for candidate in candidate_grid_22:
    candidate_id = candidate["candidate_id"]

    candidate_oof = (
        pooled_oof_22.loc[
            pooled_oof_22["candidate_id"]
            == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_22(
        candidate_oof[
            "label_stage23"
        ].to_numpy(dtype=int),
        candidate_oof[
            "prediction_raw"
        ].to_numpy(dtype=float),
    )

    fit_part = fit_audit_22.loc[
        fit_audit_22[
            "candidate_id"
        ].astype(str) == candidate_id
    ]

    fit_seconds_total = (
        float(
            pd.to_numeric(
                fit_part["fit_seconds"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_22.append(
        {
            "candidate_id": candidate_id,
            **{
                key: candidate[key]
                for key in candidate
                if key != "candidate_id"
            },
            **metrics,
            "fit_seconds_total": (
                fit_seconds_total
            ),
        }
    )

candidate_results_22 = pd.DataFrame(
    candidate_result_rows_22
)

candidate_results_22 = (
    candidate_results_22
    .sort_values(
        ["auprc", "auroc", "brier"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

candidate_results_22["selection_rank"] = (
    np.arange(
        1,
        len(candidate_results_22) + 1,
    )
)

best_row_22 = candidate_results_22.iloc[0]
selected_candidate_id_22 = str(
    best_row_22["candidate_id"]
)

selected_candidate_22 = next(
    candidate
    for candidate in candidate_grid_22
    if candidate["candidate_id"]
    == selected_candidate_id_22
)

# ------------------------------------------------------------
# 13. Platt calibration from selected inner OOF
# ------------------------------------------------------------

selected_oof_22 = (
    pooled_oof_22.loc[
        pooled_oof_22["candidate_id"]
        == selected_candidate_id_22
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_22 = (
    selected_oof_22[
        "label_stage23"
    ].to_numpy(dtype=int)
)
selected_oof_probability_22 = (
    selected_oof_22[
        "prediction_raw"
    ].to_numpy(dtype=float)
)

platt_calibrator_22 = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_22.fit(
    probability_logit_22(
        selected_oof_probability_22
    ),
    selected_oof_y_22,
)

platt_intercept_22 = float(
    platt_calibrator_22.intercept_[0]
)
platt_slope_22 = float(
    platt_calibrator_22.coef_[0][0]
)

if (
    not np.isfinite(platt_intercept_22)
    or not np.isfinite(platt_slope_22)
    or platt_slope_22 <= 0
):
    raise RuntimeError(
        "Invalid Platt calibration coefficients."
    )

selected_model_22 = pd.DataFrame(
    [
        {
            "outer_fold": OUTER_FOLD_22,
            "selected_candidate": (
                selected_candidate_id_22
            ),
            "selection_metric_primary": (
                "pooled_inner_oof_auprc"
            ),
            "inner_oof_auprc": float(
                best_row_22["auprc"]
            ),
            "inner_oof_auroc": float(
                best_row_22["auroc"]
            ),
            "inner_oof_brier": float(
                best_row_22["brier"]
            ),
            "inner_oof_log_loss": float(
                best_row_22["log_loss"]
            ),
            "inner_oof_mean_predicted_risk": float(
                best_row_22[
                    "mean_predicted_risk"
                ]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_22[
                    "observed_event_rate"
                ]
            ),
            "platt_intercept": (
                platt_intercept_22
            ),
            "platt_slope": platt_slope_22,
            "protocol_sha256": (
                protocol_sha_22
            ),
            **{
                key: selected_candidate_22[key]
                for key in selected_candidate_22
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 14. Save locked selection
# ------------------------------------------------------------

candidate_results_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22C_random_forest_candidate_results_outer4.csv",
)

selected_model_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22C_random_forest_selected_model_outer4.csv",
)

selection_json_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22C_random_forest_selection_calibration_outer4.json",
)

selection_sha_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22C_random_forest_selection_calibration_outer4_SHA256.txt",
)

candidate_results_22.to_csv(
    candidate_results_path_22,
    index=False,
)
selected_model_22.to_csv(
    selected_model_path_22,
    index=False,
)

selection_configuration_22 = {
    "outer_fold": OUTER_FOLD_22,
    "protocol_sha256": protocol_sha_22,
    "selection_metric_primary": (
        "pooled inner out-of-fold AUPRC"
    ),
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "selected_candidate": (
        selected_candidate_id_22
    ),
    "selected_hyperparameters": {
        key: selected_candidate_22[key]
        for key in selected_candidate_22
        if key != "candidate_id"
    },
    "inner_oof_auprc": float(
        best_row_22["auprc"]
    ),
    "inner_oof_auroc": float(
        best_row_22["auroc"]
    ),
    "inner_oof_brier": float(
        best_row_22["brier"]
    ),
    "platt_intercept": platt_intercept_22,
    "platt_slope": platt_slope_22,
    "inner_checkpoint_tables": (
        checkpoint_tables_22
    ),
    "patient_level_oof_written_to_drive": False,
}

with open(
    selection_json_path_22,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        selection_configuration_22,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    selection_json_path_22,
    "rb",
) as fh:
    selection_sha_22 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    selection_sha_path_22,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(selection_sha_22 + "\n")

# ------------------------------------------------------------
# 15. Fit selected model on all outer training patients
# ------------------------------------------------------------

final_preprocessor_22 = (
    make_preprocessor_22()
)

print(
    "\nFitting selected outer-fold-4 Random Forest model "
    "on all 46,803 training patients..."
)

preprocess_started_22 = time.time()

X_outer_training_processed_22 = (
    final_preprocessor_22.fit_transform(
        X_outer_training_22
    )
)
X_outer_test_processed_22 = (
    final_preprocessor_22.transform(
        X_outer_test_22
    )
)

final_preprocessing_seconds_22 = (
    time.time() - preprocess_started_22
)

final_model_22 = make_random_forest_model_22(
    selected_candidate_22
)

final_fit_started_22 = time.time()

final_model_22.fit(
    X_outer_training_processed_22,
    y_outer_training_22,
)

final_fit_seconds_22 = (
    time.time() - final_fit_started_22
)

outer4_raw_probabilities_22 = (
    final_model_22.predict_proba(
        X_outer_test_processed_22
    )[:, 1]
)

raw_clipped_22 = np.clip(
    outer4_raw_probabilities_22,
    1e-6,
    1 - 1e-6,
)
raw_logit_22 = np.log(
    raw_clipped_22
    / (1 - raw_clipped_22)
)

outer4_platt_probabilities_22 = expit(
    platt_intercept_22
    + platt_slope_22 * raw_logit_22
)

for probabilities, name in [
    (outer4_raw_probabilities_22, "raw"),
    (
        outer4_platt_probabilities_22,
        "platt",
    ),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(
            f"{name} test predictions contain missing values."
        )

    if not np.all(
        (probabilities >= 0)
        & (probabilities <= 1)
    ):
        raise RuntimeError(
            f"{name} test predictions contain invalid probabilities."
        )

raw_metrics_22 = probability_metrics_22(
    y_outer_test_22,
    outer4_raw_probabilities_22,
)
platt_metrics_22 = probability_metrics_22(
    y_outer_test_22,
    outer4_platt_probabilities_22,
)

raw_calibration_intercept_22, \
raw_calibration_slope_22 = (
    calibration_intercept_slope_22(
        y_outer_test_22,
        outer4_raw_probabilities_22,
    )
)

platt_calibration_intercept_22, \
platt_calibration_slope_22 = (
    calibration_intercept_slope_22(
        y_outer_test_22,
        outer4_platt_probabilities_22,
    )
)

outer4_test_results_22 = pd.DataFrame(
    [
        {
            "outer_fold": OUTER_FOLD_22,
            "model": "random_forest",
            "probability_type": "raw",
            **raw_metrics_22,
            "calibration_intercept": (
                raw_calibration_intercept_22
            ),
            "calibration_slope": (
                raw_calibration_slope_22
            ),
        },
        {
            "outer_fold": OUTER_FOLD_22,
            "model": "random_forest",
            "probability_type": (
                "platt_calibrated"
            ),
            **platt_metrics_22,
            "calibration_intercept": (
                platt_calibration_intercept_22
            ),
            "calibration_slope": (
                platt_calibration_slope_22
            ),
        },
    ]
)

# ------------------------------------------------------------
# 16. Feature importance
# ------------------------------------------------------------

processed_feature_names_22 = (
    final_preprocessor_22
    .get_feature_names_out()
)

feature_importances_22 = (
    final_model_22.feature_importances_
)

if len(processed_feature_names_22) != len(
    feature_importances_22
):
    raise RuntimeError(
        "Processed feature names and Random Forest "
        "feature importances differ in length."
    )

feature_importance_table_22 = pd.DataFrame(
    {
        "processed_feature": (
            processed_feature_names_22
        ),
        "mdi_importance": (
            feature_importances_22
        ),
    }
)

feature_importance_table_22[
    "importance_rank"
] = (
    feature_importance_table_22[
        "mdi_importance"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)

feature_importance_table_22 = (
    feature_importance_table_22
    .sort_values(
        [
            "mdi_importance",
            "processed_feature",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

nonzero_importance_features_22 = int(
    (
        feature_importance_table_22[
            "mdi_importance"
        ] > 0
    ).sum()
)

final_model_summary_22 = pd.DataFrame(
    [
        {
            "outer_fold": OUTER_FOLD_22,
            "selected_candidate": (
                selected_candidate_id_22
            ),
            "training_patients": len(
                X_outer_training_22
            ),
            "training_hospitals": len(
                training_hospitals_22
            ),
            "training_events": int(
                y_outer_training_22.sum()
            ),
            "test_patients": len(
                X_outer_test_22
            ),
            "test_hospitals": len(
                test_hospitals_22
            ),
            "test_events": int(
                y_outer_test_22.sum()
            ),
            "hospital_overlap": len(
                hospital_overlap_22
            ),
            "processed_feature_columns": len(
                processed_feature_names_22
            ),
            "nonzero_importance_features": (
                nonzero_importance_features_22
            ),
            "preprocessing_seconds": float(
                final_preprocessing_seconds_22
            ),
            "fit_seconds": float(
                final_fit_seconds_22
            ),
            "locked_platt_intercept": (
                platt_intercept_22
            ),
            "locked_platt_slope": (
                platt_slope_22
            ),
            "protocol_sha256": protocol_sha_22,
            "selection_sha256": (
                selection_sha_22
            ),
            **{
                key: selected_candidate_22[key]
                for key in selected_candidate_22
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 17. Secure outer-test prediction checkpoint
# ------------------------------------------------------------

outer4_prediction_df_22 = pd.DataFrame(
    {
        "id_row": (
            outer_test_meta_22[
                "id_row"
            ].astype(str)
        ),
        "outer_fold": np.full(
            len(outer_test_meta_22),
            OUTER_FOLD_22,
            dtype=np.int64,
        ),
        "label_stage23": (
            y_outer_test_22.astype(np.int64)
        ),
        "prediction_raw": (
            outer4_raw_probabilities_22.astype(
                np.float64
            )
        ),
        "prediction_platt": (
            outer4_platt_probabilities_22.astype(
                np.float64
            )
        ),
        "model_name": "random_forest",
        "model_version": (
            "core_v1_nested_cv"
        ),
    }
)

if len(outer4_prediction_df_22) != 11688:
    raise RuntimeError(
        "Outer-fold-4 test prediction row count != 11,688."
    )

if outer4_prediction_df_22[
    "id_row"
].duplicated().any():
    raise RuntimeError(
        "Duplicate id_row in outer-fold-4 Random Forest predictions."
    )

if int(
    outer4_prediction_df_22[
        "label_stage23"
    ].sum()
) != 606:
    raise RuntimeError(
        "Outer-fold-4 Random Forest event count != 606."
    )

prediction_table_id_22 = (
    f"{TARGET_DATASET}."
    "model_random_forest_outer_predictions_outer4_v1"
)

prediction_load_config_22 = (
    bigquery.LoadJobConfig(
        schema=[
            bigquery.SchemaField(
                "id_row",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "outer_fold",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "label_stage23",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_raw",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_platt",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_name",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_version",
                "STRING",
                mode="REQUIRED",
            ),
        ],
        write_disposition=(
            bigquery.WriteDisposition.WRITE_TRUNCATE
        ),
    )
)

print(
    "\nUploading secure outer-fold-4 "
    "Random Forest prediction checkpoint:"
)
print(prediction_table_id_22)

client.load_table_from_dataframe(
    outer4_prediction_df_22,
    prediction_table_id_22,
    job_config=prediction_load_config_22,
    location=BQ_LOCATION,
).result()

SQL_VERIFY_PREDICTIONS_22 = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL)
    AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL)
    AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0
    OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0
    OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw)
    AS minimum_raw_probability,
  MAX(prediction_raw)
    AS maximum_raw_probability,
  MIN(prediction_platt)
    AS minimum_platt_probability,
  MAX(prediction_platt)
    AS maximum_platt_probability
FROM `{prediction_table_id_22}`;
"""

prediction_verification_22 = (
    client.query(
        SQL_VERIFY_PREDICTIONS_22,
        location=BQ_LOCATION,
    )
    .to_dataframe()
)

verification_row_22 = (
    prediction_verification_22.iloc[0]
)

expected_prediction_values_22 = {
    "prediction_rows": 11688,
    "distinct_rows": 11688,
    "outer_folds": 1,
    "minimum_outer_fold": OUTER_FOLD_22,
    "maximum_outer_fold": OUTER_FOLD_22,
    "events": 606,
    "nonevents": 11082,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in (
    expected_prediction_values_22.items()
):
    actual_value = int(
        verification_row_22[field]
    )
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: found={actual_value}, "
            f"expected={expected_value}"
        )

# ------------------------------------------------------------
# 18. Save aggregate outputs
# ------------------------------------------------------------

test_results_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22D_random_forest_outer4_test_results.csv",
)

model_summary_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22D_random_forest_final_model_outer4.csv",
)

importance_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22D_random_forest_mdi_importance_outer4.csv",
)

evaluation_json_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22D_random_forest_final_evaluation_outer4.json",
)

evaluation_sha_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22D_random_forest_final_evaluation_outer4_SHA256.txt",
)

outer4_test_results_22.to_csv(
    test_results_path_22,
    index=False,
)
final_model_summary_22.to_csv(
    model_summary_path_22,
    index=False,
)
feature_importance_table_22.to_csv(
    importance_path_22,
    index=False,
)

evaluation_configuration_22 = {
    "outer_fold": OUTER_FOLD_22,
    "model_family": "random_forest",
    "protocol_sha256": protocol_sha_22,
    "selection_sha256": selection_sha_22,
    "selected_candidate": (
        selected_candidate_id_22
    ),
    "selected_hyperparameters": {
        key: selected_candidate_22[key]
        for key in selected_candidate_22
        if key != "candidate_id"
    },
    "training_patients": 46803,
    "training_hospitals": 159,
    "test_patients": 11688,
    "test_hospitals": 39,
    "hospital_overlap": 0,
    "locked_platt_intercept": (
        platt_intercept_22
    ),
    "locked_platt_slope": platt_slope_22,
    "processed_feature_columns": int(
        len(processed_feature_names_22)
    ),
    "secure_prediction_table": (
        prediction_table_id_22
    ),
    "patient_level_prediction_written_to_drive": False,
}

with open(
    evaluation_json_path_22,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        evaluation_configuration_22,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    evaluation_json_path_22,
    "rb",
) as fh:
    evaluation_sha_22 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    evaluation_sha_path_22,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(evaluation_sha_22 + "\n")

# ------------------------------------------------------------
# 19. Display results
# ------------------------------------------------------------

pooled_integrity_22 = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_22),
            pooled_oof_22[
                "id_row"
            ].nunique(),
            pooled_oof_22[
                "candidate_id"
            ].nunique(),
            pooled_oof_22[
                "inner_fold"
            ].nunique(),
            EXPECTED_SPLIT_22[
                "training_events"
            ],
            (
                EXPECTED_SPLIT_22[
                    "training_rows"
                ]
                - EXPECTED_SPLIT_22[
                    "training_events"
                ]
            ),
            int(
                pooled_oof_22.duplicated(
                    subset=[
                        "candidate_id",
                        "id_row",
                    ]
                ).sum()
            ),
            int(
                pooled_oof_22[
                    "prediction_raw"
                ].isna().sum()
            ),
            int(
                (
                    ~pooled_oof_22[
                        "prediction_raw"
                    ].between(0, 1)
                ).sum()
            ),
            pooled_load_method_22,
        ],
    }
)

print(
    "\n22 RANDOM FOREST OUTER-FOLD-4 INNER CHECKPOINT SUMMARY"
)
display(checkpoint_summary_22)

print(
    "\n22 RANDOM FOREST OUTER-FOLD-4 POOLED OOF INTEGRITY"
)
display(pooled_integrity_22)

print(
    "\n22 RANDOM FOREST OUTER-FOLD-4 CANDIDATE RESULTS"
)
display(candidate_results_22)

print(
    "\n22 RANDOM FOREST OUTER-FOLD-4 SELECTED MODEL"
)
display(selected_model_22)

print(
    "\n22 RANDOM FOREST OUTER-FOLD-4 FINAL MODEL SUMMARY"
)
display(final_model_summary_22)

print(
    "\n22 RANDOM FOREST OUTER-FOLD-4 TEST RESULTS"
)
display(outer4_test_results_22)

print(
    "\n22 RANDOM FOREST OUTER-FOLD-4 BIGQUERY VERIFICATION"
)
display(prediction_verification_22)

print(
    "\n22 RANDOM FOREST OUTER-FOLD-4 TOP 20 MDI IMPORTANCE FEATURES"
)
display(feature_importance_table_22.head(20))

print("\nRandom Forest protocol SHA-256:")
print(protocol_sha_22)

print("\nSelection SHA-256:")
print(selection_sha_22)

print("\nEvaluation SHA-256:")
print(evaluation_sha_22)

print("\nSaved aggregate outputs:")
print(protocol_path_22)
print(protocol_sha_path_22)
print(fit_audit_path_22)
print(checkpoint_summary_path_22)
print(candidate_results_path_22)
print(selected_model_path_22)
print(selection_json_path_22)
print(selection_sha_path_22)
print(test_results_path_22)
print(model_summary_path_22)
print(importance_path_22)
print(evaluation_json_path_22)
print(evaluation_sha_path_22)

print(
    "\n22 PASS: Random Forest outer-fold-4 nested modelling "
    "and locked test evaluation are complete."
)

print(
    "No class weighting or SMOTE was used."
)

print(
    "All patient-level OOF and outer-test predictions "
    "were stored only in BigQuery."
)

print(
    "No patient-level prediction file was written to Google Drive."
)

_ = gc.collect()

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

from sklearn.ensemble import RandomForestClassifier

from IPython.display import display

print("STARTING RANDOM FOREST OUTER FOLD 4 — CODE VERSION 22")

# ============================================================
# 22 — RANDOM FOREST OUTER FOLD 4 COMPLETE NESTED MODELLING
#
# Design:
# - Same locked hospital-disjoint outer folds.
# - Same locked hospital-disjoint inner folds.
# - Core feature set only.
# - No SMOTE.
# - No class weighting.
# - Fixed Random Forest candidate grid locked before outer-test evaluation.
# - Selection: pooled inner-OOF AUPRC descending,
#              AUROC descending, Brier ascending.
# - Platt calibration learned only from selected candidate's
#   pooled inner-OOF predictions.
# - Patient-level predictions stored only in BigQuery.
# ============================================================

OUTER_FOLD_22 = 4
MODEL_RANDOM_SEED_22 = 20260721

EXPECTED_SPLIT_22 = {
    "training_rows": 46803,
    "test_rows": 11688,
    "training_hospitals": 159,
    "test_hospitals": 39,
    "training_events": 2426,
    "test_events": 606,
}

# ------------------------------------------------------------
# 1. Required objects
# ------------------------------------------------------------

required_objects_22 = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_22 = [
    name for name in required_objects_22
    if name not in globals()
]

if missing_objects_22:
    raise RuntimeError(
        "Missing runtime objects: "
        + ", ".join(missing_objects_22)
        + ". Run 07A and 07B first."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"Expected 58,491 cohort rows; found {len(core_df_07B)}."
    )

if len(predictor_columns_07B) != 159:
    raise RuntimeError("Expected 159 core predictors.")

if len(numeric_columns_07B) != 156:
    raise RuntimeError("Expected 156 numeric predictors.")

if len(categorical_columns_07B) != 3:
    raise RuntimeError("Expected 3 categorical predictors.")

# ------------------------------------------------------------
# 2. Lock the Random Forest protocol BEFORE test evaluation
# ------------------------------------------------------------

candidate_grid_22 = [
    {
        "candidate_id": "RF01",
        "n_estimators": 300,
        "max_depth": 8,
        "min_samples_leaf": 10,
        "max_features": "sqrt",
        "max_samples": 0.80,
    },
    {
        "candidate_id": "RF02",
        "n_estimators": 400,
        "max_depth": 12,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
        "max_samples": 0.80,
    },
    {
        "candidate_id": "RF03",
        "n_estimators": 500,
        "max_depth": None,
        "min_samples_leaf": 5,
        "max_features": "sqrt",
        "max_samples": 0.85,
    },
    {
        "candidate_id": "RF04",
        "n_estimators": 400,
        "max_depth": 12,
        "min_samples_leaf": 10,
        "max_features": 0.25,
        "max_samples": 0.85,
    },
    {
        "candidate_id": "RF05",
        "n_estimators": 500,
        "max_depth": None,
        "min_samples_leaf": 10,
        "max_features": 0.25,
        "max_samples": 0.90,
    },
    {
        "candidate_id": "RF06",
        "n_estimators": 600,
        "max_depth": 16,
        "min_samples_leaf": 20,
        "max_features": 0.50,
        "max_samples": 0.90,
    },
]

rf_protocol_22 = {
    "protocol_name": "random_forest_core_nested_hospital_cv_v1",
    "model_family": "RandomForestClassifier",
    "feature_set": "core_159",
    "outer_cv": "locked 5-fold hospital-disjoint outer folds",
    "inner_cv": "locked 5-fold hospital-disjoint inner folds",
    "primary_selection_metric": "pooled inner OOF AUPRC descending",
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "criterion": "gini",
    "bootstrap": True,
    "class_weighting": False,
    "class_weight": None,
    "smote": False,
    "min_samples_split": 2,
    "random_state": MODEL_RANDOM_SEED_22,
    "candidate_grid": candidate_grid_22,
    "calibration": (
        "Platt calibration fit only on selected candidate "
        "pooled inner-OOF logits"
    ),
    "outer_test_use": (
        "diagnostic evaluation only; never used for tuning"
    ),
}

protocol_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "19A_locked_random_forest_model_protocol_v1.json",
)

protocol_sha_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "19A_locked_random_forest_model_protocol_v1_SHA256.txt",
)

protocol_text_22 = json.dumps(
    rf_protocol_22,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
)

protocol_sha_22 = hashlib.sha256(
    protocol_text_22.encode("utf-8")
).hexdigest()

if os.path.exists(protocol_path_22):
    with open(protocol_path_22, "r", encoding="utf-8") as fh:
        existing_protocol_text_22 = fh.read()
    existing_protocol_sha_22 = hashlib.sha256(
        existing_protocol_text_22.encode("utf-8")
    ).hexdigest()

    if existing_protocol_sha_22 != protocol_sha_22:
        raise RuntimeError(
            "An existing Random Forest protocol file differs from "
            "the currently locked protocol. Stop and audit."
        )
else:
    with open(protocol_path_22, "w", encoding="utf-8") as fh:
        fh.write(protocol_text_22)

with open(protocol_sha_path_22, "w", encoding="utf-8") as fh:
    fh.write(protocol_sha_22 + "\n")

print("Locked Random Forest protocol SHA-256:")
print(protocol_sha_22)

# ------------------------------------------------------------
# 3. Load locked inner hospital map
# ------------------------------------------------------------

inner_mapping_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_22):
    raise FileNotFoundError(
        "Locked inner-fold map not found: "
        + inner_mapping_path_22
    )

inner_mapping_all_22 = pd.read_csv(
    inner_mapping_path_22,
    dtype={"group_hospital": str},
)

inner_mapping_part_22 = (
    inner_mapping_all_22.loc[
        inner_mapping_all_22["outer_fold"].astype(int)
        == OUTER_FOLD_22,
        ["group_hospital", "inner_fold"],
    ]
    .copy()
)

inner_mapping_part_22["group_hospital"] = (
    inner_mapping_part_22["group_hospital"].astype(str)
)
inner_mapping_part_22["inner_fold"] = (
    inner_mapping_part_22["inner_fold"].astype(int)
)

if len(inner_mapping_part_22) != EXPECTED_SPLIT_22["training_hospitals"]:
    raise RuntimeError(
        "Expected "
        f'{EXPECTED_SPLIT_22["training_hospitals"]} '
        "outer-fold-4 training hospitals in the locked inner map; "
        f"found {len(inner_mapping_part_22)}."
    )

if inner_mapping_part_22["group_hospital"].duplicated().any():
    raise RuntimeError("Duplicate hospital in locked inner map.")

hospital_to_inner_fold_22 = dict(
    zip(
        inner_mapping_part_22["group_hospital"],
        inner_mapping_part_22["inner_fold"],
    )
)

# ------------------------------------------------------------
# 4. Prepare outer fold 4 matrices
# ------------------------------------------------------------

X_all_22 = core_df_07B[predictor_columns_07B].copy()

for column in numeric_columns_07B:
    X_all_22[column] = pd.to_numeric(
        X_all_22[column],
        errors="coerce",
    ).astype("float64")

for column in categorical_columns_07B:
    category_series = X_all_22[column].astype("object")
    X_all_22[column] = category_series.where(
        pd.notna(category_series),
        np.nan,
    )

outer_fold_vector_22 = (
    core_df_07B["outer_fold"].astype(int).to_numpy()
)

outer_training_mask_22 = (
    outer_fold_vector_22 != OUTER_FOLD_22
)
outer_test_mask_22 = (
    outer_fold_vector_22 == OUTER_FOLD_22
)

X_outer_training_22 = (
    X_all_22.loc[outer_training_mask_22]
    .reset_index(drop=True)
)

X_outer_test_22 = (
    X_all_22.loc[outer_test_mask_22]
    .reset_index(drop=True)
)

outer_training_meta_22 = (
    core_df_07B.loc[
        outer_training_mask_22,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_22 = (
    core_df_07B.loc[
        outer_test_mask_22,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [
    outer_training_meta_22,
    outer_test_meta_22,
]:
    dataframe["id_row"] = dataframe["id_row"].astype(str)
    dataframe["group_hospital"] = (
        dataframe["group_hospital"].astype(str)
    )
    dataframe["label_stage23"] = (
        dataframe["label_stage23"].astype(int)
    )

y_outer_training_22 = (
    outer_training_meta_22["label_stage23"]
    .to_numpy(dtype=np.int8)
)
y_outer_test_22 = (
    outer_test_meta_22["label_stage23"]
    .to_numpy(dtype=np.int8)
)

groups_outer_training_22 = (
    outer_training_meta_22["group_hospital"]
    .to_numpy(dtype=str)
)

training_hospitals_22 = set(
    outer_training_meta_22["group_hospital"]
)
test_hospitals_22 = set(
    outer_test_meta_22["group_hospital"]
)
hospital_overlap_22 = (
    training_hospitals_22 & test_hospitals_22
)

if hospital_overlap_22:
    raise RuntimeError(
        "Outer training/test hospital overlap detected."
    )

actual_split_22 = {
    "training_rows": len(X_outer_training_22),
    "test_rows": len(X_outer_test_22),
    "training_hospitals": len(training_hospitals_22),
    "test_hospitals": len(test_hospitals_22),
    "training_events": int(y_outer_training_22.sum()),
    "test_events": int(y_outer_test_22.sum()),
}

for metric, expected_value in EXPECTED_SPLIT_22.items():
    actual_value = actual_split_22[metric]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: found={actual_value}, "
            f"expected={expected_value}"
        )

inner_fold_vector_22 = np.array(
    [
        hospital_to_inner_fold_22.get(hospital, -1)
        for hospital in groups_outer_training_22
    ],
    dtype=int,
)

if (inner_fold_vector_22 == -1).any():
    raise RuntimeError(
        "Some outer-training hospitals have no inner-fold assignment."
    )

if set(np.unique(inner_fold_vector_22)) != {1, 2, 3, 4, 5}:
    raise RuntimeError("Inner-fold values are not exactly 1–5.")

# ------------------------------------------------------------
# 5. Preprocessor and model constructors
# ------------------------------------------------------------

def make_preprocessor_22():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns_07B,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns_07B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_random_forest_model_22(candidate):
    return RandomForestClassifier(
        n_estimators=int(candidate["n_estimators"]),
        criterion="gini",
        max_depth=(
            None
            if candidate["max_depth"] is None
            else int(candidate["max_depth"])
        ),
        min_samples_split=2,
        min_samples_leaf=int(candidate["min_samples_leaf"]),
        max_features=candidate["max_features"],
        bootstrap=True,
        max_samples=float(candidate["max_samples"]),
        class_weight=None,
        random_state=MODEL_RANDOM_SEED_22,
        n_jobs=-1,
        verbose=0,
    )


def checkpoint_table_id_22(inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_random_forest_inner_oof_outer4_inner{inner_fold}_v1"
    )

# ------------------------------------------------------------
# 6. BigQuery checkpoint verification
# ------------------------------------------------------------

def verify_checkpoint_22(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):
    table_id = checkpoint_table_id_22(inner_fold)

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS distinct_id_count,
      COUNT(DISTINCT candidate_id) AS candidate_count,
      COUNT(DISTINCT outer_fold) AS outer_fold_count,
      COUNT(DISTINCT inner_fold) AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(candidate_id, '|', id_row)
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(
        prediction_raw < 0 OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(
        sql,
        location=BQ_LOCATION,
    ).to_dataframe()

    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows * len(candidate_grid_22)
    )
    expected_positive_rows = (
        expected_validation_events * len(candidate_grid_22)
    )
    expected_negative_rows = (
        (expected_validation_rows - expected_validation_events)
        * len(candidate_grid_22)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": expected_validation_rows,
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": expected_total_rows,
        "positive_prediction_rows": expected_positive_rows,
        "negative_prediction_rows": expected_negative_rows,
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": OUTER_FOLD_22,
        "maximum_outer_fold": OUTER_FOLD_22,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failures = []

    for field, expected_value in expected_values.items():
        actual_value = int(row[field])
        if actual_value != expected_value:
            complete = False
            failures.append(
                f"{field}={actual_value}, expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failures),
        "check": check,
        "row": row,
    }

checkpoint_load_config_22 = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField(
            "id_row", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "outer_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "inner_fold", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "candidate_id", "STRING", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "label_stage23", "INTEGER", mode="REQUIRED"
        ),
        bigquery.SchemaField(
            "prediction_raw", "FLOAT", mode="REQUIRED"
        ),
    ],
    write_disposition=(
        bigquery.WriteDisposition.WRITE_TRUNCATE
    ),
)

# ------------------------------------------------------------
# 7. Aggregate fit audit
# ------------------------------------------------------------

fit_audit_columns_22 = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "n_estimators",
    "max_depth",
    "min_samples_leaf",
    "max_features",
    "max_samples",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "fit_seconds",
]

fit_audit_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22B_random_forest_inner_fit_audit_outer4.csv",
)

if os.path.exists(fit_audit_path_22):
    fit_audit_22 = pd.read_csv(fit_audit_path_22)
else:
    fit_audit_22 = pd.DataFrame(
        columns=fit_audit_columns_22
    )

for column in fit_audit_columns_22:
    if column not in fit_audit_22.columns:
        fit_audit_22[column] = np.nan

fit_audit_22 = fit_audit_22[
    fit_audit_columns_22
].copy()

# ------------------------------------------------------------
# 8. Run five inner folds
# ------------------------------------------------------------

for inner_fold in range(1, 6):
    inner_training_mask = (
        inner_fold_vector_22 != inner_fold
    )
    inner_validation_mask = (
        inner_fold_vector_22 == inner_fold
    )

    training_rows = int(inner_training_mask.sum())
    validation_rows = int(inner_validation_mask.sum())
    training_events = int(
        y_outer_training_22[inner_training_mask].sum()
    )
    validation_events = int(
        y_outer_training_22[inner_validation_mask].sum()
    )

    existing_check = verify_checkpoint_22(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if existing_check["complete"]:
        print(
            f"Outer 4 / inner {inner_fold}: "
            "permanent Random Forest checkpoint already complete; "
            "skipping model fitting."
        )
        continue

    training_hospital_set = set(
        groups_outer_training_22[inner_training_mask]
    )
    validation_hospital_set = set(
        groups_outer_training_22[inner_validation_mask]
    )

    if training_hospital_set & validation_hospital_set:
        raise RuntimeError(
            f"Inner fold {inner_fold}: hospital overlap detected."
        )

    print(f"\nOuter 4 / inner {inner_fold}")
    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_22()

    preprocessing_started = time.time()

    X_inner_training_processed = (
        preprocessor.fit_transform(
            X_outer_training_22.loc[
                inner_training_mask
            ]
        )
    )

    X_inner_validation_processed = (
        preprocessor.transform(
            X_outer_training_22.loc[
                inner_validation_mask
            ]
        )
    )

    preprocessing_seconds = (
        time.time() - preprocessing_started
    )

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "Processed training/validation column counts differ."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_seconds, 2),
    )

    y_inner_training = (
        y_outer_training_22[inner_training_mask]
    )
    y_inner_validation = (
        y_outer_training_22[inner_validation_mask]
    )

    validation_ids = (
        outer_training_meta_22.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_22:
        candidate_id = candidate["candidate_id"]

        print(
            "  Fitting",
            candidate_id,
            "| trees =",
            candidate["n_estimators"],
            "| depth =",
            candidate["max_depth"],
            "| min_leaf =",
            candidate["min_samples_leaf"],
            "| max_features =",
            candidate["max_features"],
            "| max_samples =",
            candidate["max_samples"],
        )

        model = make_random_forest_model_22(candidate)

        fit_started = time.time()

        model.fit(
            X_inner_training_processed,
            y_inner_training,
        )

        fit_seconds = time.time() - fit_started

        validation_probabilities = (
            model.predict_proba(
                X_inner_validation_processed
            )[:, 1]
        )

        if np.isnan(validation_probabilities).any():
            raise RuntimeError(
                f"{candidate_id}, inner {inner_fold}: "
                "missing predictions."
            )

        if not np.all(
            (validation_probabilities >= 0)
            & (validation_probabilities <= 1)
        ):
            raise RuntimeError(
                f"{candidate_id}: invalid probabilities."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        OUTER_FOLD_22,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": (
                        y_inner_validation.astype(np.int64)
                    ),
                    "prediction_raw": (
                        validation_probabilities.astype(
                            np.float64
                        )
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": OUTER_FOLD_22,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "n_estimators": candidate[
                    "n_estimators"
                ],
                "max_depth": candidate[
                    "max_depth"
                ],
                "min_samples_leaf": candidate[
                    "min_samples_leaf"
                ],
                "max_features": candidate[
                    "max_features"
                ],
                "max_samples": candidate[
                    "max_samples"
                ],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": (
                    validation_events
                ),
                "processed_columns": int(
                    X_inner_training_processed.shape[1]
                ),
                "fit_seconds": float(fit_seconds),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows * len(candidate_grid_22)
    )

    if len(checkpoint_df) != expected_checkpoint_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: invalid checkpoint row count."
        )

    if checkpoint_df.duplicated(
        subset=["id_row", "candidate_id"]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: duplicate candidate-patient rows."
        )

    target_checkpoint_table = (
        checkpoint_table_id_22(inner_fold)
    )

    print(
        "Uploading permanent Random Forest checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_22,
        location=BQ_LOCATION,
    ).result()

    if len(fit_audit_22) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_22["outer_fold"],
                    errors="coerce",
                ) == 1
            )
            & (
                pd.to_numeric(
                    fit_audit_22["inner_fold"],
                    errors="coerce",
                ) == inner_fold
            )
        )

        fit_audit_22 = (
            fit_audit_22.loc[keep_mask].copy()
        )

    fit_audit_22 = pd.concat(
        [
            fit_audit_22,
            pd.DataFrame(current_audit_rows),
        ],
        ignore_index=True,
    )

    fit_audit_22 = (
        fit_audit_22[
            fit_audit_columns_22
        ]
        .sort_values(
            [
                "outer_fold",
                "inner_fold",
                "candidate_id",
            ]
        )
        .reset_index(drop=True)
    )

    fit_audit_22.to_csv(
        fit_audit_path_22,
        index=False,
    )

    completed_check = verify_checkpoint_22(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint verification failed: "
            + completed_check["reason"]
        )

    print(
        f"Outer 4 / inner {inner_fold}: "
        "permanent Random Forest checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 9. Final inner checkpoint summary
# ------------------------------------------------------------

checkpoint_summary_rows_22 = []

for inner_fold in range(1, 6):
    validation_mask = (
        inner_fold_vector_22 == inner_fold
    )
    validation_rows = int(validation_mask.sum())
    validation_events = int(
        y_outer_training_22[
            validation_mask
        ].sum()
    )

    final_check = verify_checkpoint_22(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: final checkpoint audit failed. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_22.append(
        {
            "outer_fold": OUTER_FOLD_22,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(
                row["row_count"]
            ),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(
                row["candidate_count"]
            ),
            "positive_prediction_rows": int(
                row["positive_prediction_rows"]
            ),
            "negative_prediction_rows": int(
                row["negative_prediction_rows"]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check["table_id"],
        }
    )

checkpoint_summary_22 = (
    pd.DataFrame(checkpoint_summary_rows_22)
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_22[
        "distinct_validation_patients"
    ].sum()
) != EXPECTED_SPLIT_22["training_rows"]:
    raise RuntimeError(
        "Total inner validation patients != 46,803."
    )

expected_total_oof_rows_22 = (
    EXPECTED_SPLIT_22["training_rows"]
    * len(candidate_grid_22)
)

if int(
    checkpoint_summary_22[
        "checkpoint_rows"
    ].sum()
) != expected_total_oof_rows_22:
    raise RuntimeError(
        "Total Random Forest OOF prediction rows are incorrect."
    )

checkpoint_summary_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22B_random_forest_outer4_inner_checkpoint_summary.csv",
)

checkpoint_summary_22.to_csv(
    checkpoint_summary_path_22,
    index=False,
)

# ------------------------------------------------------------
# 10. Pool five inner OOF tables
# ------------------------------------------------------------

checkpoint_tables_22 = [
    checkpoint_table_id_22(inner_fold)
    for inner_fold in range(1, 6)
]

union_parts_22 = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_22
]

SQL_LOAD_POOLED_OOF_22 = (
    "\nUNION ALL\n".join(union_parts_22)
)

print(
    "\nLoading pooled outer-fold-4 Random Forest inner OOF predictions..."
)

query_job_22 = client.query(
    SQL_LOAD_POOLED_OOF_22,
    location=BQ_LOCATION,
)

try:
    pooled_oof_22 = query_job_22.to_dataframe(
        create_bqstorage_client=True
    )
    pooled_load_method_22 = (
        "BigQuery Storage API"
    )
except Exception as fast_path_error_22:
    print(
        "Storage API unavailable; using standard BigQuery download."
    )
    print(
        "Message:",
        type(fast_path_error_22).__name__,
    )
    pooled_oof_22 = query_job_22.to_dataframe(
        create_bqstorage_client=False
    )
    pooled_load_method_22 = (
        "Standard BigQuery API"
    )

pooled_oof_22["id_row"] = (
    pooled_oof_22["id_row"].astype(str)
)
pooled_oof_22["candidate_id"] = (
    pooled_oof_22["candidate_id"].astype(str)
)

for column in [
    "outer_fold",
    "inner_fold",
    "label_stage23",
]:
    pooled_oof_22[column] = pd.to_numeric(
        pooled_oof_22[column],
        errors="raise",
    ).astype(int)

pooled_oof_22["prediction_raw"] = pd.to_numeric(
    pooled_oof_22["prediction_raw"],
    errors="raise",
).astype(float)

if len(pooled_oof_22) != expected_total_oof_rows_22:
    raise RuntimeError(
        "Pooled Random Forest OOF row count is incorrect."
    )

if pooled_oof_22.duplicated(
    subset=["candidate_id", "id_row"]
).any():
    raise RuntimeError(
        "Duplicate candidate-patient row in pooled Random Forest OOF."
    )

if pooled_oof_22["prediction_raw"].isna().any():
    raise RuntimeError("Missing Random Forest OOF prediction.")

if not pooled_oof_22[
    "prediction_raw"
].between(0, 1).all():
    raise RuntimeError(
        "Invalid Random Forest OOF probability."
    )

if set(
    pooled_oof_22["candidate_id"].unique()
) != {
    "RF01",
    "RF02",
    "RF03",
    "RF04",
    "RF05",
    "RF06",
}:
    raise RuntimeError(
        "The six locked Random Forest candidates are not all present."
    )

candidate_patient_counts_22 = (
    pooled_oof_22
    .groupby("candidate_id")["id_row"]
    .nunique()
)

if not (
    candidate_patient_counts_22
    == EXPECTED_SPLIT_22["training_rows"]
).all():
    raise RuntimeError(
        "Each candidate must have 46,803 OOF patients."
    )

candidate_event_counts_22 = (
    pooled_oof_22
    .groupby("candidate_id")["label_stage23"]
    .sum()
)

if not (
    candidate_event_counts_22
    == EXPECTED_SPLIT_22["training_events"]
).all():
    raise RuntimeError(
        "Each candidate must have 2,426 OOF events."
    )

# ------------------------------------------------------------
# 11. Metric helpers
# ------------------------------------------------------------

def probability_metrics_22(
    y_true,
    probabilities,
):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(
            roc_auc_score(y_true, probabilities)
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                probabilities,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                probabilities,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=[0, 1],
            )
        ),
        "mean_predicted_risk": float(
            probabilities.mean()
        ),
        "observed_event_rate": float(
            np.mean(y_true)
        ),
    }


def probability_logit_22(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        probabilities / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_22(
    y_true,
    probabilities,
):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        probability_logit_22(probabilities),
        y_true,
    )

    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 12. Candidate pooled inner OOF metrics
# ------------------------------------------------------------

candidate_result_rows_22 = []

for candidate in candidate_grid_22:
    candidate_id = candidate["candidate_id"]

    candidate_oof = (
        pooled_oof_22.loc[
            pooled_oof_22["candidate_id"]
            == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_22(
        candidate_oof[
            "label_stage23"
        ].to_numpy(dtype=int),
        candidate_oof[
            "prediction_raw"
        ].to_numpy(dtype=float),
    )

    fit_part = fit_audit_22.loc[
        fit_audit_22[
            "candidate_id"
        ].astype(str) == candidate_id
    ]

    fit_seconds_total = (
        float(
            pd.to_numeric(
                fit_part["fit_seconds"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_22.append(
        {
            "candidate_id": candidate_id,
            **{
                key: candidate[key]
                for key in candidate
                if key != "candidate_id"
            },
            **metrics,
            "fit_seconds_total": (
                fit_seconds_total
            ),
        }
    )

candidate_results_22 = pd.DataFrame(
    candidate_result_rows_22
)

candidate_results_22 = (
    candidate_results_22
    .sort_values(
        ["auprc", "auroc", "brier"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

candidate_results_22["selection_rank"] = (
    np.arange(
        1,
        len(candidate_results_22) + 1,
    )
)

best_row_22 = candidate_results_22.iloc[0]
selected_candidate_id_22 = str(
    best_row_22["candidate_id"]
)

selected_candidate_22 = next(
    candidate
    for candidate in candidate_grid_22
    if candidate["candidate_id"]
    == selected_candidate_id_22
)

# ------------------------------------------------------------
# 13. Platt calibration from selected inner OOF
# ------------------------------------------------------------

selected_oof_22 = (
    pooled_oof_22.loc[
        pooled_oof_22["candidate_id"]
        == selected_candidate_id_22
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_22 = (
    selected_oof_22[
        "label_stage23"
    ].to_numpy(dtype=int)
)
selected_oof_probability_22 = (
    selected_oof_22[
        "prediction_raw"
    ].to_numpy(dtype=float)
)

platt_calibrator_22 = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_22.fit(
    probability_logit_22(
        selected_oof_probability_22
    ),
    selected_oof_y_22,
)

platt_intercept_22 = float(
    platt_calibrator_22.intercept_[0]
)
platt_slope_22 = float(
    platt_calibrator_22.coef_[0][0]
)

if (
    not np.isfinite(platt_intercept_22)
    or not np.isfinite(platt_slope_22)
    or platt_slope_22 <= 0
):
    raise RuntimeError(
        "Invalid Platt calibration coefficients."
    )

selected_model_22 = pd.DataFrame(
    [
        {
            "outer_fold": OUTER_FOLD_22,
            "selected_candidate": (
                selected_candidate_id_22
            ),
            "selection_metric_primary": (
                "pooled_inner_oof_auprc"
            ),
            "inner_oof_auprc": float(
                best_row_22["auprc"]
            ),
            "inner_oof_auroc": float(
                best_row_22["auroc"]
            ),
            "inner_oof_brier": float(
                best_row_22["brier"]
            ),
            "inner_oof_log_loss": float(
                best_row_22["log_loss"]
            ),
            "inner_oof_mean_predicted_risk": float(
                best_row_22[
                    "mean_predicted_risk"
                ]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_22[
                    "observed_event_rate"
                ]
            ),
            "platt_intercept": (
                platt_intercept_22
            ),
            "platt_slope": platt_slope_22,
            "protocol_sha256": (
                protocol_sha_22
            ),
            **{
                key: selected_candidate_22[key]
                for key in selected_candidate_22
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 14. Save locked selection
# ------------------------------------------------------------

candidate_results_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22C_random_forest_candidate_results_outer4.csv",
)

selected_model_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22C_random_forest_selected_model_outer4.csv",
)

selection_json_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22C_random_forest_selection_calibration_outer4.json",
)

selection_sha_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22C_random_forest_selection_calibration_outer4_SHA256.txt",
)

candidate_results_22.to_csv(
    candidate_results_path_22,
    index=False,
)
selected_model_22.to_csv(
    selected_model_path_22,
    index=False,
)

selection_configuration_22 = {
    "outer_fold": OUTER_FOLD_22,
    "protocol_sha256": protocol_sha_22,
    "selection_metric_primary": (
        "pooled inner out-of-fold AUPRC"
    ),
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
    ],
    "selected_candidate": (
        selected_candidate_id_22
    ),
    "selected_hyperparameters": {
        key: selected_candidate_22[key]
        for key in selected_candidate_22
        if key != "candidate_id"
    },
    "inner_oof_auprc": float(
        best_row_22["auprc"]
    ),
    "inner_oof_auroc": float(
        best_row_22["auroc"]
    ),
    "inner_oof_brier": float(
        best_row_22["brier"]
    ),
    "platt_intercept": platt_intercept_22,
    "platt_slope": platt_slope_22,
    "inner_checkpoint_tables": (
        checkpoint_tables_22
    ),
    "patient_level_oof_written_to_drive": False,
}

with open(
    selection_json_path_22,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        selection_configuration_22,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    selection_json_path_22,
    "rb",
) as fh:
    selection_sha_22 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    selection_sha_path_22,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(selection_sha_22 + "\n")

# ------------------------------------------------------------
# 15. Fit selected model on all outer training patients
# ------------------------------------------------------------

final_preprocessor_22 = (
    make_preprocessor_22()
)

print(
    "\nFitting selected outer-fold-4 Random Forest model "
    "on all 46,803 training patients..."
)

preprocess_started_22 = time.time()

X_outer_training_processed_22 = (
    final_preprocessor_22.fit_transform(
        X_outer_training_22
    )
)
X_outer_test_processed_22 = (
    final_preprocessor_22.transform(
        X_outer_test_22
    )
)

final_preprocessing_seconds_22 = (
    time.time() - preprocess_started_22
)

final_model_22 = make_random_forest_model_22(
    selected_candidate_22
)

final_fit_started_22 = time.time()

final_model_22.fit(
    X_outer_training_processed_22,
    y_outer_training_22,
)

final_fit_seconds_22 = (
    time.time() - final_fit_started_22
)

outer4_raw_probabilities_22 = (
    final_model_22.predict_proba(
        X_outer_test_processed_22
    )[:, 1]
)

raw_clipped_22 = np.clip(
    outer4_raw_probabilities_22,
    1e-6,
    1 - 1e-6,
)
raw_logit_22 = np.log(
    raw_clipped_22
    / (1 - raw_clipped_22)
)

outer4_platt_probabilities_22 = expit(
    platt_intercept_22
    + platt_slope_22 * raw_logit_22
)

for probabilities, name in [
    (outer4_raw_probabilities_22, "raw"),
    (
        outer4_platt_probabilities_22,
        "platt",
    ),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(
            f"{name} test predictions contain missing values."
        )

    if not np.all(
        (probabilities >= 0)
        & (probabilities <= 1)
    ):
        raise RuntimeError(
            f"{name} test predictions contain invalid probabilities."
        )

raw_metrics_22 = probability_metrics_22(
    y_outer_test_22,
    outer4_raw_probabilities_22,
)
platt_metrics_22 = probability_metrics_22(
    y_outer_test_22,
    outer4_platt_probabilities_22,
)

raw_calibration_intercept_22, \
raw_calibration_slope_22 = (
    calibration_intercept_slope_22(
        y_outer_test_22,
        outer4_raw_probabilities_22,
    )
)

platt_calibration_intercept_22, \
platt_calibration_slope_22 = (
    calibration_intercept_slope_22(
        y_outer_test_22,
        outer4_platt_probabilities_22,
    )
)

outer4_test_results_22 = pd.DataFrame(
    [
        {
            "outer_fold": OUTER_FOLD_22,
            "model": "random_forest",
            "probability_type": "raw",
            **raw_metrics_22,
            "calibration_intercept": (
                raw_calibration_intercept_22
            ),
            "calibration_slope": (
                raw_calibration_slope_22
            ),
        },
        {
            "outer_fold": OUTER_FOLD_22,
            "model": "random_forest",
            "probability_type": (
                "platt_calibrated"
            ),
            **platt_metrics_22,
            "calibration_intercept": (
                platt_calibration_intercept_22
            ),
            "calibration_slope": (
                platt_calibration_slope_22
            ),
        },
    ]
)

# ------------------------------------------------------------
# 16. Feature importance
# ------------------------------------------------------------

processed_feature_names_22 = (
    final_preprocessor_22
    .get_feature_names_out()
)

feature_importances_22 = (
    final_model_22.feature_importances_
)

if len(processed_feature_names_22) != len(
    feature_importances_22
):
    raise RuntimeError(
        "Processed feature names and Random Forest "
        "feature importances differ in length."
    )

feature_importance_table_22 = pd.DataFrame(
    {
        "processed_feature": (
            processed_feature_names_22
        ),
        "mdi_importance": (
            feature_importances_22
        ),
    }
)

feature_importance_table_22[
    "importance_rank"
] = (
    feature_importance_table_22[
        "mdi_importance"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)

feature_importance_table_22 = (
    feature_importance_table_22
    .sort_values(
        [
            "mdi_importance",
            "processed_feature",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

nonzero_importance_features_22 = int(
    (
        feature_importance_table_22[
            "mdi_importance"
        ] > 0
    ).sum()
)

final_model_summary_22 = pd.DataFrame(
    [
        {
            "outer_fold": OUTER_FOLD_22,
            "selected_candidate": (
                selected_candidate_id_22
            ),
            "training_patients": len(
                X_outer_training_22
            ),
            "training_hospitals": len(
                training_hospitals_22
            ),
            "training_events": int(
                y_outer_training_22.sum()
            ),
            "test_patients": len(
                X_outer_test_22
            ),
            "test_hospitals": len(
                test_hospitals_22
            ),
            "test_events": int(
                y_outer_test_22.sum()
            ),
            "hospital_overlap": len(
                hospital_overlap_22
            ),
            "processed_feature_columns": len(
                processed_feature_names_22
            ),
            "nonzero_importance_features": (
                nonzero_importance_features_22
            ),
            "preprocessing_seconds": float(
                final_preprocessing_seconds_22
            ),
            "fit_seconds": float(
                final_fit_seconds_22
            ),
            "locked_platt_intercept": (
                platt_intercept_22
            ),
            "locked_platt_slope": (
                platt_slope_22
            ),
            "protocol_sha256": protocol_sha_22,
            "selection_sha256": (
                selection_sha_22
            ),
            **{
                key: selected_candidate_22[key]
                for key in selected_candidate_22
                if key != "candidate_id"
            },
        }
    ]
)

# ------------------------------------------------------------
# 17. Secure outer-test prediction checkpoint
# ------------------------------------------------------------

outer4_prediction_df_22 = pd.DataFrame(
    {
        "id_row": (
            outer_test_meta_22[
                "id_row"
            ].astype(str)
        ),
        "outer_fold": np.full(
            len(outer_test_meta_22),
            OUTER_FOLD_22,
            dtype=np.int64,
        ),
        "label_stage23": (
            y_outer_test_22.astype(np.int64)
        ),
        "prediction_raw": (
            outer4_raw_probabilities_22.astype(
                np.float64
            )
        ),
        "prediction_platt": (
            outer4_platt_probabilities_22.astype(
                np.float64
            )
        ),
        "model_name": "random_forest",
        "model_version": (
            "core_v1_nested_cv"
        ),
    }
)

if len(outer4_prediction_df_22) != 11688:
    raise RuntimeError(
        "Outer-fold-4 test prediction row count != 11,688."
    )

if outer4_prediction_df_22[
    "id_row"
].duplicated().any():
    raise RuntimeError(
        "Duplicate id_row in outer-fold-4 Random Forest predictions."
    )

if int(
    outer4_prediction_df_22[
        "label_stage23"
    ].sum()
) != 606:
    raise RuntimeError(
        "Outer-fold-4 Random Forest event count != 606."
    )

prediction_table_id_22 = (
    f"{TARGET_DATASET}."
    "model_random_forest_outer_predictions_outer4_v1"
)

prediction_load_config_22 = (
    bigquery.LoadJobConfig(
        schema=[
            bigquery.SchemaField(
                "id_row",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "outer_fold",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "label_stage23",
                "INTEGER",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_raw",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "prediction_platt",
                "FLOAT",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_name",
                "STRING",
                mode="REQUIRED",
            ),
            bigquery.SchemaField(
                "model_version",
                "STRING",
                mode="REQUIRED",
            ),
        ],
        write_disposition=(
            bigquery.WriteDisposition.WRITE_TRUNCATE
        ),
    )
)

print(
    "\nUploading secure outer-fold-4 "
    "Random Forest prediction checkpoint:"
)
print(prediction_table_id_22)

client.load_table_from_dataframe(
    outer4_prediction_df_22,
    prediction_table_id_22,
    job_config=prediction_load_config_22,
    location=BQ_LOCATION,
).result()

SQL_VERIFY_PREDICTIONS_22 = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL)
    AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL)
    AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0
    OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0
    OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw)
    AS minimum_raw_probability,
  MAX(prediction_raw)
    AS maximum_raw_probability,
  MIN(prediction_platt)
    AS minimum_platt_probability,
  MAX(prediction_platt)
    AS maximum_platt_probability
FROM `{prediction_table_id_22}`;
"""

prediction_verification_22 = (
    client.query(
        SQL_VERIFY_PREDICTIONS_22,
        location=BQ_LOCATION,
    )
    .to_dataframe()
)

verification_row_22 = (
    prediction_verification_22.iloc[0]
)

expected_prediction_values_22 = {
    "prediction_rows": 11688,
    "distinct_rows": 11688,
    "outer_folds": 1,
    "minimum_outer_fold": OUTER_FOLD_22,
    "maximum_outer_fold": OUTER_FOLD_22,
    "events": 606,
    "nonevents": 11082,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in (
    expected_prediction_values_22.items()
):
    actual_value = int(
        verification_row_22[field]
    )
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: found={actual_value}, "
            f"expected={expected_value}"
        )

# ------------------------------------------------------------
# 18. Save aggregate outputs
# ------------------------------------------------------------

test_results_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22D_random_forest_outer4_test_results.csv",
)

model_summary_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22D_random_forest_final_model_outer4.csv",
)

importance_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22D_random_forest_mdi_importance_outer4.csv",
)

evaluation_json_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22D_random_forest_final_evaluation_outer4.json",
)

evaluation_sha_path_22 = os.path.join(
    MODEL_OUTPUT_DIR,
    "22D_random_forest_final_evaluation_outer4_SHA256.txt",
)

outer4_test_results_22.to_csv(
    test_results_path_22,
    index=False,
)
final_model_summary_22.to_csv(
    model_summary_path_22,
    index=False,
)
feature_importance_table_22.to_csv(
    importance_path_22,
    index=False,
)

evaluation_configuration_22 = {
    "outer_fold": OUTER_FOLD_22,
    "model_family": "random_forest",
    "protocol_sha256": protocol_sha_22,
    "selection_sha256": selection_sha_22,
    "selected_candidate": (
        selected_candidate_id_22
    ),
    "selected_hyperparameters": {
        key: selected_candidate_22[key]
        for key in selected_candidate_22
        if key != "candidate_id"
    },
    "training_patients": 46803,
    "training_hospitals": 159,
    "test_patients": 11688,
    "test_hospitals": 39,
    "hospital_overlap": 0,
    "locked_platt_intercept": (
        platt_intercept_22
    ),
    "locked_platt_slope": platt_slope_22,
    "processed_feature_columns": int(
        len(processed_feature_names_22)
    ),
    "secure_prediction_table": (
        prediction_table_id_22
    ),
    "patient_level_prediction_written_to_drive": False,
}

with open(
    evaluation_json_path_22,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        evaluation_configuration_22,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    evaluation_json_path_22,
    "rb",
) as fh:
    evaluation_sha_22 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    evaluation_sha_path_22,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(evaluation_sha_22 + "\n")

# ------------------------------------------------------------
# 19. Display results
# ------------------------------------------------------------

pooled_integrity_22 = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_22),
            pooled_oof_22[
                "id_row"
            ].nunique(),
            pooled_oof_22[
                "candidate_id"
            ].nunique(),
            pooled_oof_22[
                "inner_fold"
            ].nunique(),
            EXPECTED_SPLIT_22[
                "training_events"
            ],
            (
                EXPECTED_SPLIT_22[
                    "training_rows"
                ]
                - EXPECTED_SPLIT_22[
                    "training_events"
                ]
            ),
            int(
                pooled_oof_22.duplicated(
                    subset=[
                        "candidate_id",
                        "id_row",
                    ]
                ).sum()
            ),
            int(
                pooled_oof_22[
                    "prediction_raw"
                ].isna().sum()
            ),
            int(
                (
                    ~pooled_oof_22[
                        "prediction_raw"
                    ].between(0, 1)
                ).sum()
            ),
            pooled_load_method_22,
        ],
    }
)

print(
    "\n22 RANDOM FOREST OUTER-FOLD-4 INNER CHECKPOINT SUMMARY"
)
display(checkpoint_summary_22)

print(
    "\n22 RANDOM FOREST OUTER-FOLD-4 POOLED OOF INTEGRITY"
)
display(pooled_integrity_22)

print(
    "\n22 RANDOM FOREST OUTER-FOLD-4 CANDIDATE RESULTS"
)
display(candidate_results_22)

print(
    "\n22 RANDOM FOREST OUTER-FOLD-4 SELECTED MODEL"
)
display(selected_model_22)

print(
    "\n22 RANDOM FOREST OUTER-FOLD-4 FINAL MODEL SUMMARY"
)
display(final_model_summary_22)

print(
    "\n22 RANDOM FOREST OUTER-FOLD-4 TEST RESULTS"
)
display(outer4_test_results_22)

print(
    "\n22 RANDOM FOREST OUTER-FOLD-4 BIGQUERY VERIFICATION"
)
display(prediction_verification_22)

print(
    "\n22 RANDOM FOREST OUTER-FOLD-4 TOP 20 MDI IMPORTANCE FEATURES"
)
display(feature_importance_table_22.head(20))

print("\nRandom Forest protocol SHA-256:")
print(protocol_sha_22)

print("\nSelection SHA-256:")
print(selection_sha_22)

print("\nEvaluation SHA-256:")
print(evaluation_sha_22)

print("\nSaved aggregate outputs:")
print(protocol_path_22)
print(protocol_sha_path_22)
print(fit_audit_path_22)
print(checkpoint_summary_path_22)
print(candidate_results_path_22)
print(selected_model_path_22)
print(selection_json_path_22)
print(selection_sha_path_22)
print(test_results_path_22)
print(model_summary_path_22)
print(importance_path_22)
print(evaluation_json_path_22)
print(evaluation_sha_path_22)

print(
    "\n22 PASS: Random Forest outer-fold-4 nested modelling "
    "and locked test evaluation are complete."
)

print(
    "No class weighting or SMOTE was used."
)

print(
    "All patient-level OOF and outer-test predictions "
    "were stored only in BigQuery."
)

print(
    "No patient-level prediction file was written to Google Drive."
)

_ = gc.collect()

In [ ]:
import os, json, hashlib
from datetime import datetime, timezone

print("STARTING PARSIMONIOUS CLINICAL BASELINE PROTOCOL LOCK — CODE VERSION 33A")

if "MODEL_OUTPUT_DIR" not in globals():
    raise RuntimeError("Önce 07A hücresini çalıştır.")

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)

protocol = {
    "analysis_version": "33A",
    "protocol_name": "locked_parsimonious_clinical_baseline_v1",
    "protocol_status": "locked_before_any_outer_test_evaluation",
    "analysis_role": "additional_post_hoc_clinical_baseline",
    "rationale": {
        "qsofa_not_used": "Standard 0–12 h qSOFA cannot be reconstructed because no compatible mentation/GCS component was found.",
        "sirs_not_used": "Standard 0–12 h SIRS was not selected because temperature coverage is approximately 12%, leaving about 9.3% complete cases.",
        "incomplete_scores_prohibited": True,
        "complete_case_score_analysis_prohibited": True,
        "apache_24h_substitution_prohibited": True
    },
    "cohort": {
        "patients": 58491,
        "hospitals": 198,
        "events": 3032,
        "landmark_hours": 12,
        "outcome_window_hours": ">12–72",
        "outcome": "KDIGO_stage_2_or_3_AKI"
    },
    "predictors": [
        "x_age_years",
        "x_sex",
        "x_reference_creatinine",
        "x_stage1_at_landmark",
        "x_lab_creatinine_last",
        "x_lab_bun_last",
        "x_vital_respiratory_rate_last",
        "x_vital_noninvasive_systolic_bp_last"
    ],
    "predictor_count": 8,
    "preprocessing": {
        "numeric_imputation": "median_training_only",
        "numeric_missing_indicators": True,
        "numeric_scaling": "StandardScaler_with_mean_false",
        "categorical_imputation": "constant___MISSING__",
        "categorical_encoding": "one_hot_unknown_ignore",
        "categorical_predictors": ["x_sex"]
    },
    "validation": {
        "outer_folds": 5,
        "outer_split_unit": "hospital",
        "inner_folds_per_outer_training_set": 5,
        "inner_split_unit": "hospital",
        "outer_test_used_once_after_selection_and_calibration_lock": True
    },
    "model": {
        "estimator": "LogisticRegression",
        "penalty": "elasticnet",
        "solver": "saga",
        "class_weight": None,
        "max_iter": 5000,
        "tol": 0.0001,
        "random_state": 20260721
    },
    "candidate_grid": [
        {"candidate_id": "CLIN01", "C": 0.03, "l1_ratio": 0.0},
        {"candidate_id": "CLIN02", "C": 0.10, "l1_ratio": 0.0},
        {"candidate_id": "CLIN03", "C": 0.30, "l1_ratio": 0.0},
        {"candidate_id": "CLIN04", "C": 0.10, "l1_ratio": 0.25},
        {"candidate_id": "CLIN05", "C": 0.30, "l1_ratio": 0.25},
        {"candidate_id": "CLIN06", "C": 0.30, "l1_ratio": 0.50}
    ],
    "selection_rule": [
        "pooled_inner_oof_AUPRC_descending",
        "pooled_inner_oof_AUROC_descending",
        "pooled_inner_oof_Brier_ascending",
        "candidate_id_ascending_immutable_tie_break"
    ],
    "calibration": {
        "method": "Platt",
        "source": "selected_candidate_pooled_inner_OOF_logits",
        "fit_scope": "outer_training_only"
    },
    "primary_probability_type": "platt_calibrated",
    "reported_metrics": [
        "AUROC", "AUPRC", "Brier_score", "log_loss",
        "calibration_intercept", "calibration_slope"
    ],
    "uncertainty": {
        "method": "hospital_cluster_bootstrap",
        "replicates": 2000,
        "seed": 20260723
    },
    "prohibited": [
        "class_weighting", "SMOTE", "oversampling", "undersampling",
        "early_stopping", "test_based_retuning",
        "patient_level_drive_export",
        "BigQuery_DML_DELETE_INSERT_UPDATE_MERGE"
    ],
    "provenance_statement": "Additional/post-hoc clinical benchmark; protocol locked before any of its outer-test results were examined.",
    "created_utc": datetime.now(timezone.utc).isoformat()
}

protocol_text = json.dumps(protocol, indent=2, sort_keys=True)
protocol_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A_locked_parsimonious_clinical_baseline_protocol_v1.json"
)
sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A_locked_parsimonious_clinical_baseline_protocol_v1_SHA256.txt"
)

with open(protocol_path, "w", encoding="utf-8") as f:
    f.write(protocol_text)

protocol_sha = hashlib.sha256(protocol_text.encode("utf-8")).hexdigest()

with open(sha_path, "w", encoding="utf-8") as f:
    f.write(protocol_sha + "\n")

with open(protocol_path, "r", encoding="utf-8") as f:
    verify = hashlib.sha256(f.read().encode("utf-8")).hexdigest()

if verify != protocol_sha:
    raise RuntimeError("Protocol SHA verification failed.")

print("\nLocked protocol SHA-256:")
print(protocol_sha)
print("\nSaved:")
print(protocol_path)
print(sha_path)
print("\n33A PASS: Protocol locked before any outer-test evaluation.")
print("No fitting, test access, recalibration, or patient-level Drive export was performed.")

In [ ]:
print("RUNTIME_OK")
print(MODEL_OUTPUT_DIR)

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from sklearn.exceptions import ConvergenceWarning

from IPython.display import display

# ============================================================
# 33B — PARSIMONIOUS CLINICAL BASELINE OUTER FOLD 1
#
# Resume-safe:
# - Each inner fold is stored in a separate BigQuery table.
# - Completed inner folds are automatically skipped.
#
# BigQuery free-tier compatible:
# - No DELETE / INSERT / UPDATE / MERGE is used.
#
# Privacy:
# - Patient-level predictions are stored only in BigQuery.
# - No patient-level prediction file is written to Drive.
# ============================================================

print("STARTING PARSIMONIOUS CLINICAL BASELINE OUTER FOLD 1 — CODE VERSION 33B")

OUTER_FOLD_33B = 1
MODEL_RANDOM_SEED_33B = 20260721

EXPECTED_PROTOCOL_SHA_33B = (
    "94b0abb218dbef4e349702ea2824ca4e"
    "31dfbe36c53efa05ba1a8bd1f38f835e"
)

EXPECTED_SPLIT_33B = {
    "training_rows": 46803,
    "test_rows": 11688,
    "training_hospitals": 158,
    "test_hospitals": 40,
    "training_events": 2426,
    "test_events": 606,
}

EXPECTED_INNER_33B = {
    1: {"validation_rows": 10650, "validation_events": 578},
    2: {"validation_rows": 7084, "validation_events": 359},
    3: {"validation_rows": 11915, "validation_events": 637},
    4: {"validation_rows": 10019, "validation_events": 488},
    5: {"validation_rows": 7135, "validation_events": 364},
}

CLINICAL_PREDICTORS_33B = [
    "x_age_years",
    "x_sex",
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
    "x_lab_bun_last",
    "x_vital_respiratory_rate_last",
    "x_vital_noninvasive_systolic_bp_last",
]

CLINICAL_NUMERIC_33B = [
    "x_age_years",
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
    "x_lab_bun_last",
    "x_vital_respiratory_rate_last",
    "x_vital_noninvasive_systolic_bp_last",
]

CLINICAL_CATEGORICAL_33B = ["x_sex"]

# ------------------------------------------------------------
# 1. Required runtime objects
# ------------------------------------------------------------

required_objects_33B = [
    "core_df_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_33B = [
    name for name in required_objects_33B if name not in globals()
]

if missing_objects_33B:
    raise RuntimeError(
        "Eksik RAM nesneleri var: "
        + ", ".join(missing_objects_33B)
        + ". Önce 07A ve 07B hücrelerini çalıştır."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"58.491 satır bekleniyordu; {len(core_df_07B)} bulundu."
    )

missing_predictors_33B = [
    column for column in CLINICAL_PREDICTORS_33B
    if column not in core_df_07B.columns
]

if missing_predictors_33B:
    raise RuntimeError(
        "Kilitli klinik predictor(lar) core_df_07B içinde yok: "
        + ", ".join(missing_predictors_33B)
    )

# ------------------------------------------------------------
# 2. Locked protocol SHA check
# ------------------------------------------------------------

protocol_sha_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A_locked_parsimonious_clinical_baseline_protocol_v1_SHA256.txt",
)

if not os.path.exists(protocol_sha_path_33B):
    raise FileNotFoundError(
        "Model protokolü SHA dosyası bulunamadı: "
        + protocol_sha_path_33B
    )

with open(protocol_sha_path_33B, "r", encoding="utf-8") as file_handle:
    observed_protocol_sha_33B = file_handle.read().strip()

if observed_protocol_sha_33B != EXPECTED_PROTOCOL_SHA_33B:
    raise RuntimeError(
        "Kilitli model protokolü SHA değeri değişmiş: "
        + observed_protocol_sha_33B
    )

# ------------------------------------------------------------
# 3. Locked inner-hospital mapping
# ------------------------------------------------------------

inner_mapping_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_33B):
    raise FileNotFoundError(
        "Kilitli iç kat haritası bulunamadı: "
        + inner_mapping_path_33B
    )

inner_mapping_all_33B = pd.read_csv(
    inner_mapping_path_33B,
    dtype={"group_hospital": str},
)

inner_mapping_part_33B = (
    inner_mapping_all_33B.loc[
        inner_mapping_all_33B["outer_fold"].astype(int) == OUTER_FOLD_33B,
        ["group_hospital", "inner_fold"],
    ]
    .copy()
)

inner_mapping_part_33B["group_hospital"] = (
    inner_mapping_part_33B["group_hospital"].astype(str)
)
inner_mapping_part_33B["inner_fold"] = (
    inner_mapping_part_33B["inner_fold"].astype(int)
)

if len(inner_mapping_part_33B) != 158:
    raise RuntimeError(
        "Dış kat 1 eğitim kümesi için 158 hastane ataması bekleniyordu."
    )

if inner_mapping_part_33B["group_hospital"].duplicated().any():
    raise RuntimeError("İç kat haritasında yinelenen hastane var.")

hospital_to_inner_fold_33B = dict(
    zip(
        inner_mapping_part_33B["group_hospital"],
        inner_mapping_part_33B["inner_fold"],
    )
)

# ------------------------------------------------------------
# 4. Prepare model matrices
# ------------------------------------------------------------

X_all_33B = core_df_07B[CLINICAL_PREDICTORS_33B].copy()

for column in CLINICAL_NUMERIC_33B:
    X_all_33B[column] = pd.to_numeric(
        X_all_33B[column], errors="coerce"
    ).astype("float64")

for column in CLINICAL_CATEGORICAL_33B:
    category_series = X_all_33B[column].astype("object")
    X_all_33B[column] = category_series.where(
        pd.notna(category_series), np.nan
    )

outer_fold_vector_33B = (
    core_df_07B["outer_fold"].astype(int).to_numpy()
)

outer_training_mask_33B = outer_fold_vector_33B != OUTER_FOLD_33B
outer_test_mask_33B = outer_fold_vector_33B == OUTER_FOLD_33B

X_outer_training_33B = (
    X_all_33B.loc[outer_training_mask_33B].reset_index(drop=True)
)
X_outer_test_33B = (
    X_all_33B.loc[outer_test_mask_33B].reset_index(drop=True)
)

outer_training_meta_33B = (
    core_df_07B.loc[
        outer_training_mask_33B,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_33B = (
    core_df_07B.loc[
        outer_test_mask_33B,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [outer_training_meta_33B, outer_test_meta_33B]:
    dataframe["id_row"] = dataframe["id_row"].astype(str)
    dataframe["group_hospital"] = dataframe["group_hospital"].astype(str)
    dataframe["label_stage23"] = dataframe["label_stage23"].astype(int)

y_outer_training_33B = (
    outer_training_meta_33B["label_stage23"].to_numpy(dtype=np.int8)
)
y_outer_test_33B = (
    outer_test_meta_33B["label_stage23"].to_numpy(dtype=np.int8)
)

groups_outer_training_33B = (
    outer_training_meta_33B["group_hospital"].to_numpy(dtype=str)
)

training_hospitals_33B = set(
    outer_training_meta_33B["group_hospital"]
)
test_hospitals_33B = set(
    outer_test_meta_33B["group_hospital"]
)
hospital_overlap_33B = training_hospitals_33B & test_hospitals_33B

if hospital_overlap_33B:
    raise RuntimeError("Dış eğitim ve test hastaneleri çakışıyor.")

actual_split_33B = {
    "training_rows": len(X_outer_training_33B),
    "test_rows": len(X_outer_test_33B),
    "training_hospitals": len(training_hospitals_33B),
    "test_hospitals": len(test_hospitals_33B),
    "training_events": int(y_outer_training_33B.sum()),
    "test_events": int(y_outer_test_33B.sum()),
}

for metric, expected_value in EXPECTED_SPLIT_33B.items():
    actual_value = actual_split_33B[metric]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: bulunan={actual_value}, beklenen={expected_value}"
        )

inner_fold_vector_33B = np.array(
    [
        hospital_to_inner_fold_33B.get(hospital, -1)
        for hospital in groups_outer_training_33B
    ],
    dtype=int,
)

if (inner_fold_vector_33B == -1).any():
    raise RuntimeError(
        "Bazı dış eğitim hastanelerine iç kat atanmadı."
    )

if set(np.unique(inner_fold_vector_33B)) != {1, 2, 3, 4, 5}:
    raise RuntimeError("İç kat değerleri 1–5 değil.")

for inner_fold, expected in EXPECTED_INNER_33B.items():
    validation_mask = inner_fold_vector_33B == inner_fold
    observed_rows = int(validation_mask.sum())
    observed_events = int(y_outer_training_33B[validation_mask].sum())

    if observed_rows != expected["validation_rows"]:
        raise RuntimeError(
            f"Inner {inner_fold} validation_rows: "
            f"bulunan={observed_rows}, "
            f"beklenen={expected['validation_rows']}"
        )

    if observed_events != expected["validation_events"]:
        raise RuntimeError(
            f"Inner {inner_fold} validation_events: "
            f"bulunan={observed_events}, "
            f"beklenen={expected['validation_events']}"
        )

# ------------------------------------------------------------
# 5. Locked candidate grid
# ------------------------------------------------------------

candidate_grid_33B = [
    {"candidate_id": "CLIN01", "C": 0.03, "l1_ratio": 0.00},
    {"candidate_id": "CLIN02", "C": 0.10, "l1_ratio": 0.00},
    {"candidate_id": "CLIN03", "C": 0.30, "l1_ratio": 0.00},
    {"candidate_id": "CLIN04", "C": 0.10, "l1_ratio": 0.25},
    {"candidate_id": "CLIN05", "C": 0.30, "l1_ratio": 0.25},
    {"candidate_id": "CLIN06", "C": 0.30, "l1_ratio": 0.50},
]

# ------------------------------------------------------------
# 6. Preprocessing and model factories
# ------------------------------------------------------------

def make_preprocessor_33B():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
            (
                "scaler",
                StandardScaler(with_mean=False),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                CLINICAL_NUMERIC_33B,
            ),
            (
                "categorical",
                categorical_pipeline,
                CLINICAL_CATEGORICAL_33B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_model_33B(C_value, l1_ratio_value):
    return LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=float(C_value),
        l1_ratio=float(l1_ratio_value),
        class_weight=None,
        max_iter=5000,
        tol=1e-4,
        random_state=MODEL_RANDOM_SEED_33B,
    )


def checkpoint_table_id_33B(inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_clinical_lr_inner_oof_outer1_inner{inner_fold}_v1"
    )

# ------------------------------------------------------------
# 7. BigQuery checkpoint verification
# ------------------------------------------------------------

def verify_checkpoint_33B(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):
    table_id = checkpoint_table_id_33B(inner_fold)

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS distinct_id_count,
      COUNT(DISTINCT candidate_id) AS candidate_count,
      COUNT(DISTINCT outer_fold) AS outer_fold_count,
      COUNT(DISTINCT inner_fold) AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(candidate_id, '|', id_row)
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(
        prediction_raw < 0 OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(sql, location=BQ_LOCATION).to_dataframe()
    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows * len(candidate_grid_33B)
    )
    expected_positive_rows = (
        expected_validation_events * len(candidate_grid_33B)
    )
    expected_negative_rows = (
        (expected_validation_rows - expected_validation_events)
        * len(candidate_grid_33B)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": expected_validation_rows,
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": expected_total_rows,
        "positive_prediction_rows": expected_positive_rows,
        "negative_prediction_rows": expected_negative_rows,
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": 1,
        "maximum_outer_fold": 1,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failure_items = []

    for field, expected_value in expected_values.items():
        actual_value = int(row[field])
        if actual_value != expected_value:
            complete = False
            failure_items.append(
                f"{field}={actual_value}, expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failure_items),
        "check": check,
        "row": row,
    }

# ------------------------------------------------------------
# 8. BigQuery load schema
# ------------------------------------------------------------

checkpoint_load_config_33B = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("inner_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("candidate_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

# ------------------------------------------------------------
# 9. Aggregate fit-audit file
# ------------------------------------------------------------

fit_audit_columns_33B = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "C",
    "l1_ratio",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "convergence_warnings",
    "elapsed_seconds",
]

fit_audit_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_inner_fit_audit_outer1.csv",
)

if os.path.exists(fit_audit_path_33B):
    fit_audit_33B = pd.read_csv(fit_audit_path_33B)
else:
    fit_audit_33B = pd.DataFrame(columns=fit_audit_columns_33B)

for column in fit_audit_columns_33B:
    if column not in fit_audit_33B.columns:
        fit_audit_33B[column] = np.nan

fit_audit_33B = fit_audit_33B[fit_audit_columns_33B].copy()

# ------------------------------------------------------------
# 10. Train six candidates in five locked inner folds
# ------------------------------------------------------------

for inner_fold in range(1, 6):
    inner_training_mask = inner_fold_vector_33B != inner_fold
    inner_validation_mask = inner_fold_vector_33B == inner_fold

    training_rows = int(inner_training_mask.sum())
    validation_rows = int(inner_validation_mask.sum())
    training_events = int(
        y_outer_training_33B[inner_training_mask].sum()
    )
    validation_events = int(
        y_outer_training_33B[inner_validation_mask].sum()
    )

    expected_inner = EXPECTED_INNER_33B[inner_fold]
    expected_training_rows = (
        EXPECTED_SPLIT_33B["training_rows"]
        - expected_inner["validation_rows"]
    )
    expected_training_events = (
        EXPECTED_SPLIT_33B["training_events"]
        - expected_inner["validation_events"]
    )

    if training_rows != expected_training_rows:
        raise RuntimeError(
            f"Inner {inner_fold} training_rows: "
            f"bulunan={training_rows}, "
            f"beklenen={expected_training_rows}"
        )

    if training_events != expected_training_events:
        raise RuntimeError(
            f"Inner {inner_fold} training_events: "
            f"bulunan={training_events}, "
            f"beklenen={expected_training_events}"
        )

    existing_check = verify_checkpoint_33B(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if existing_check["complete"]:
        print(
            f"Outer 1 / inner {inner_fold}: "
            "permanent checkpoint already complete; "
            "skipping model fitting."
        )
        continue

    training_hospital_set = set(
        groups_outer_training_33B[inner_training_mask]
    )
    validation_hospital_set = set(
        groups_outer_training_33B[inner_validation_mask]
    )

    if training_hospital_set & validation_hospital_set:
        raise RuntimeError(
            f"Inner fold {inner_fold}: hastane çakışması bulundu."
        )

    print(f"\nOuter 1 / inner {inner_fold}")
    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_33B()
    preprocessing_started = time.time()

    X_inner_training_processed = preprocessor.fit_transform(
        X_outer_training_33B.loc[inner_training_mask]
    )
    X_inner_validation_processed = preprocessor.transform(
        X_outer_training_33B.loc[inner_validation_mask]
    )

    preprocessing_elapsed = time.time() - preprocessing_started

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "İşlenmiş eğitim ve doğrulama sütun sayıları farklı."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_elapsed, 2),
    )

    y_inner_training = y_outer_training_33B[inner_training_mask]
    y_inner_validation = y_outer_training_33B[inner_validation_mask]

    validation_ids = (
        outer_training_meta_33B.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_33B:
        candidate_id = candidate["candidate_id"]

        print(
            "  Fitting",
            candidate_id,
            "| C =",
            candidate["C"],
            "| l1_ratio =",
            candidate["l1_ratio"],
        )

        model = make_model_33B(
            candidate["C"],
            candidate["l1_ratio"],
        )

        fitting_started = time.time()

        with warnings.catch_warnings(record=True) as warning_records:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(
                X_inner_training_processed,
                y_inner_training,
            )

        fitting_elapsed = time.time() - fitting_started

        convergence_warning_count = sum(
            issubclass(warning.category, ConvergenceWarning)
            for warning in warning_records
        )

        validation_probabilities = model.predict_proba(
            X_inner_validation_processed
        )[:, 1]

        if np.isnan(validation_probabilities).any():
            raise RuntimeError(
                f"{candidate_id}, inner {inner_fold}: eksik tahmin."
            )

        if not np.all(
            (validation_probabilities >= 0)
            & (validation_probabilities <= 1)
        ):
            raise RuntimeError(
                f"{candidate_id}: geçersiz olasılık."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        1,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": y_inner_validation.astype(np.int64),
                    "prediction_raw": validation_probabilities.astype(
                        np.float64
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": 1,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "C": candidate["C"],
                "l1_ratio": candidate["l1_ratio"],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": validation_events,
                "processed_columns": int(
                    X_inner_training_processed.shape[1]
                ),
                "convergence_warnings": int(
                    convergence_warning_count
                ),
                "elapsed_seconds": float(fitting_elapsed),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows * len(candidate_grid_33B)
    )

    if len(checkpoint_df) != expected_checkpoint_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: checkpoint satır sayısı hatalı."
        )

    if checkpoint_df.duplicated(
        subset=["id_row", "candidate_id"]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "yinelenen aday–hasta tahmini var."
        )

    target_checkpoint_table = checkpoint_table_id_33B(inner_fold)

    print(
        "Uploading permanent checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_33B,
        location=BQ_LOCATION,
    ).result()

    new_audit_df = pd.DataFrame(current_audit_rows)

    if len(fit_audit_33B) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_33B["outer_fold"],
                    errors="coerce",
                )
                == 1
            )
            & (
                pd.to_numeric(
                    fit_audit_33B["inner_fold"],
                    errors="coerce",
                )
                == inner_fold
            )
        )
        fit_audit_33B = fit_audit_33B.loc[keep_mask].copy()

    if fit_audit_33B.empty:
        fit_audit_33B = new_audit_df.copy()
    else:
        fit_audit_33B = pd.concat(
            [fit_audit_33B, new_audit_df],
            ignore_index=True,
        )

    fit_audit_33B = (
        fit_audit_33B[fit_audit_columns_33B]
        .sort_values(
            ["outer_fold", "inner_fold", "candidate_id"]
        )
        .reset_index(drop=True)
    )

    fit_audit_33B.to_csv(
        fit_audit_path_33B,
        index=False,
    )

    completed_check = verify_checkpoint_33B(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint doğrulanamadı: "
            + completed_check["reason"]
        )

    print(
        f"Outer 1 / inner {inner_fold}: "
        "permanent checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 11. Final checkpoint summary
# ------------------------------------------------------------

checkpoint_summary_rows_33B = []

for inner_fold in range(1, 6):
    validation_mask = inner_fold_vector_33B == inner_fold
    validation_rows = int(validation_mask.sum())
    validation_events = int(
        y_outer_training_33B[validation_mask].sum()
    )

    final_check = verify_checkpoint_33B(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "son checkpoint denetimi başarısız. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_33B.append(
        {
            "outer_fold": 1,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(row["row_count"]),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(row["candidate_count"]),
            "positive_prediction_rows": int(
                row["positive_prediction_rows"]
            ),
            "negative_prediction_rows": int(
                row["negative_prediction_rows"]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check["table_id"],
        }
    )

checkpoint_summary_33B = (
    pd.DataFrame(checkpoint_summary_rows_33B)
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_33B["distinct_validation_patients"].sum()
) != EXPECTED_SPLIT_33B["training_rows"]:
    raise RuntimeError(
        "Toplam doğrulama hasta sayısı 46.803 değil."
    )

expected_total_oof_rows_33B = (
    EXPECTED_SPLIT_33B["training_rows"]
    * len(candidate_grid_33B)
)

if int(checkpoint_summary_33B["checkpoint_rows"].sum()) != (
    expected_total_oof_rows_33B
):
    raise RuntimeError("Toplam OOF tahmin satırı hatalı.")

checkpoint_summary_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_outer1_inner_checkpoint_summary.csv",
)
checkpoint_summary_33B.to_csv(
    checkpoint_summary_path_33B,
    index=False,
)

# ------------------------------------------------------------
# 12. Pool all inner OOF predictions
# ------------------------------------------------------------

checkpoint_tables_33B = [
    checkpoint_table_id_33B(inner_fold)
    for inner_fold in range(1, 6)
]

union_parts_33B = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_33B
]

SQL_LOAD_POOLED_OOF_33B = "\nUNION ALL\n".join(union_parts_33B)

print("\nLoading pooled outer-fold-1 inner OOF predictions...")

query_job_33B = client.query(
    SQL_LOAD_POOLED_OOF_33B,
    location=BQ_LOCATION,
)

try:
    pooled_oof_33B = query_job_33B.to_dataframe(
        create_bqstorage_client=True
    )
    pooled_load_method_33B = "BigQuery Storage API"
except Exception as fast_path_error_33B:
    print(
        "Storage API unavailable; using standard BigQuery download."
    )
    print("Message:", type(fast_path_error_33B).__name__)
    pooled_oof_33B = query_job_33B.to_dataframe(
        create_bqstorage_client=False
    )
    pooled_load_method_33B = "Standard BigQuery API"

pooled_oof_33B["id_row"] = pooled_oof_33B["id_row"].astype(str)
pooled_oof_33B["candidate_id"] = pooled_oof_33B["candidate_id"].astype(str)

for column in ["outer_fold", "inner_fold", "label_stage23"]:
    pooled_oof_33B[column] = pd.to_numeric(
        pooled_oof_33B[column], errors="raise"
    ).astype(int)

pooled_oof_33B["prediction_raw"] = pd.to_numeric(
    pooled_oof_33B["prediction_raw"], errors="raise"
).astype(float)

# ------------------------------------------------------------
# 13. Pooled OOF integrity checks
# ------------------------------------------------------------

if len(pooled_oof_33B) != expected_total_oof_rows_33B:
    raise RuntimeError("Pooled OOF satır sayısı hatalı.")

if set(pooled_oof_33B["outer_fold"].unique()) != {1}:
    raise RuntimeError("Pooled OOF içinde dış kat 4 dışında kayıt var.")

if set(pooled_oof_33B["inner_fold"].unique()) != {1, 2, 3, 4, 5}:
    raise RuntimeError("Pooled OOF iç katları 1–5 değil.")

if pooled_oof_33B.duplicated(
    subset=["candidate_id", "id_row"]
).any():
    raise RuntimeError(
        "Pooled OOF içinde yinelenen aday–hasta tahmini bulundu."
    )

if pooled_oof_33B["prediction_raw"].isna().any():
    raise RuntimeError("Pooled OOF içinde eksik tahmin var.")

if not pooled_oof_33B["prediction_raw"].between(0, 1).all():
    raise RuntimeError(
        "Pooled OOF içinde geçersiz olasılık değeri var."
    )

expected_candidate_ids_33B = {
    "CLIN01",
    "CLIN02",
    "CLIN03",
    "CLIN04",
    "CLIN05",
    "CLIN06",
}

if set(pooled_oof_33B["candidate_id"].unique()) != (
    expected_candidate_ids_33B
):
    raise RuntimeError("Altı kilitli aday bulunmuyor.")

candidate_patient_counts_33B = (
    pooled_oof_33B.groupby("candidate_id")["id_row"].nunique()
)

if not (
    candidate_patient_counts_33B
    == EXPECTED_SPLIT_33B["training_rows"]
).all():
    raise RuntimeError(
        "Her aday için 46.803 farklı OOF hastası yok."
    )

candidate_event_counts_33B = (
    pooled_oof_33B.groupby("candidate_id")["label_stage23"].sum()
)

if not (
    candidate_event_counts_33B
    == EXPECTED_SPLIT_33B["training_events"]
).all():
    raise RuntimeError("Her aday için 2.426 olay yok.")

patient_label_consistency_33B = (
    pooled_oof_33B.groupby("id_row")["label_stage23"].nunique()
)
if (patient_label_consistency_33B > 1).any():
    raise RuntimeError(
        "Aynı hastanın adaylar arasında outcome etiketi farklı."
    )

patient_inner_fold_consistency_33B = (
    pooled_oof_33B.groupby("id_row")["inner_fold"].nunique()
)
if (patient_inner_fold_consistency_33B > 1).any():
    raise RuntimeError(
        "Aynı hasta birden fazla iç doğrulama katında bulundu."
    )

# ------------------------------------------------------------
# 14. Metric helpers
# ------------------------------------------------------------

def probability_metrics_33B(y_true, probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(roc_auc_score(y_true, probabilities)),
        "auprc": float(average_precision_score(y_true, probabilities)),
        "brier": float(brier_score_loss(y_true, probabilities)),
        "log_loss": float(
            log_loss(y_true, probabilities, labels=[0, 1])
        ),
        "mean_predicted_risk": float(probabilities.mean()),
        "observed_event_rate": float(np.mean(y_true)),
    }


def probability_logit_33B(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        probabilities / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_33B(y_true, probabilities):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        probability_logit_33B(probabilities),
        y_true,
    )
    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 15. Candidate pooled-OOF performance
# ------------------------------------------------------------

candidate_result_rows_33B = []

for candidate in candidate_grid_33B:
    candidate_id = candidate["candidate_id"]

    candidate_oof = (
        pooled_oof_33B.loc[
            pooled_oof_33B["candidate_id"] == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_33B(
        candidate_oof["label_stage23"].to_numpy(dtype=int),
        candidate_oof["prediction_raw"].to_numpy(dtype=float),
    )

    fit_part = fit_audit_33B.loc[
        (
            pd.to_numeric(
                fit_audit_33B["outer_fold"],
                errors="coerce",
            )
            == 1
        )
        & (
            fit_audit_33B["candidate_id"].astype(str)
            == candidate_id
        )
    ]

    convergence_warnings = (
        int(
            pd.to_numeric(
                fit_part["convergence_warnings"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    elapsed_seconds = (
        float(
            pd.to_numeric(
                fit_part["elapsed_seconds"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_33B.append(
        {
            "candidate_id": candidate_id,
            "C": candidate["C"],
            "l1_ratio": candidate["l1_ratio"],
            **metrics,
            "convergence_warnings": convergence_warnings,
            "elapsed_seconds": elapsed_seconds,
        }
    )

candidate_results_33B = pd.DataFrame(candidate_result_rows_33B)

candidate_results_33B = (
    candidate_results_33B.sort_values(
        ["auprc", "auroc", "brier", "candidate_id"],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)

candidate_results_33B["selection_rank"] = np.arange(
    1,
    len(candidate_results_33B) + 1,
)

best_row_33B = candidate_results_33B.iloc[0]
selected_candidate_33B = str(best_row_33B["candidate_id"])
selected_C_33B = float(best_row_33B["C"])
selected_l1_ratio_33B = float(best_row_33B["l1_ratio"])

# ------------------------------------------------------------
# 16. Platt calibration on selected pooled inner OOF
# ------------------------------------------------------------

selected_oof_33B = (
    pooled_oof_33B.loc[
        pooled_oof_33B["candidate_id"] == selected_candidate_33B
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_33B = selected_oof_33B[
    "label_stage23"
].to_numpy(dtype=int)
selected_oof_probability_33B = selected_oof_33B[
    "prediction_raw"
].to_numpy(dtype=float)

platt_calibrator_33B = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_33B.fit(
    probability_logit_33B(selected_oof_probability_33B),
    selected_oof_y_33B,
)

platt_intercept_33B = float(platt_calibrator_33B.intercept_[0])
platt_slope_33B = float(platt_calibrator_33B.coef_[0][0])

if (
    not np.isfinite(platt_intercept_33B)
    or not np.isfinite(platt_slope_33B)
    or platt_slope_33B <= 0
):
    raise RuntimeError("Platt kalibrasyon katsayıları geçersiz.")

selected_model_33B = pd.DataFrame(
    [
        {
            "outer_fold": 1,
            "selected_candidate": selected_candidate_33B,
            "selected_C": selected_C_33B,
            "selected_l1_ratio": selected_l1_ratio_33B,
            "selection_metric_primary": "pooled_inner_oof_auprc",
            "inner_oof_auprc": float(best_row_33B["auprc"]),
            "inner_oof_auroc": float(best_row_33B["auroc"]),
            "inner_oof_brier": float(best_row_33B["brier"]),
            "inner_oof_log_loss": float(best_row_33B["log_loss"]),
            "inner_oof_mean_predicted_risk": float(
                best_row_33B["mean_predicted_risk"]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_33B["observed_event_rate"]
            ),
            "platt_intercept": platt_intercept_33B,
            "platt_slope": platt_slope_33B,
            "protocol_sha256": EXPECTED_PROTOCOL_SHA_33B,
        }
    ]
)

# ------------------------------------------------------------
# 17. Lock selection/calibration aggregate files
# ------------------------------------------------------------

candidate_results_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_candidate_results_outer1.csv",
)
selected_model_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_selected_model_outer1.csv",
)
selection_json_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_selection_calibration_outer1.json",
)
selection_sha_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_selection_calibration_outer1_SHA256.txt",
)

candidate_results_33B.to_csv(
    candidate_results_path_33B,
    index=False,
)
selected_model_33B.to_csv(
    selected_model_path_33B,
    index=False,
)

selection_configuration_33B = {
    "outer_fold": 1,
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_33B,
    "selection_metric_primary": "pooled inner out-of-fold AUPRC",
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
        "candidate_id ascending immutable tie-break",
    ],
    "selected_candidate": selected_candidate_33B,
    "selected_C": selected_C_33B,
    "selected_l1_ratio": selected_l1_ratio_33B,
    "inner_oof_auprc": float(best_row_33B["auprc"]),
    "inner_oof_auroc": float(best_row_33B["auroc"]),
    "inner_oof_brier": float(best_row_33B["brier"]),
    "platt_intercept": platt_intercept_33B,
    "platt_slope": platt_slope_33B,
    "inner_checkpoint_tables": checkpoint_tables_33B,
    "patient_level_oof_written_to_drive": False,
    "analysis_role": "additional_post_hoc_clinical_baseline",
    "predictors": CLINICAL_PREDICTORS_33B,
}

with open(
    selection_json_path_33B,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        selection_configuration_33B,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(selection_json_path_33B, "rb") as file_handle:
    selection_sha_33B = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    selection_sha_path_33B,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(selection_sha_33B + "\n")

# ------------------------------------------------------------
# 18. Fit final selected outer-fold-1 model
# ------------------------------------------------------------

final_pipeline_33B = Pipeline(
    steps=[
        ("preprocessor", make_preprocessor_33B()),
        (
            "model",
            make_model_33B(
                selected_C_33B,
                selected_l1_ratio_33B,
            ),
        ),
    ]
)

print(
    "\nFitting selected outer-fold-1 model "
    "on all 46,803 training patients..."
)

final_fit_started_33B = time.time()

with warnings.catch_warnings(record=True) as final_warning_records_33B:
    warnings.simplefilter("always", ConvergenceWarning)
    final_pipeline_33B.fit(
        X_outer_training_33B,
        y_outer_training_33B,
    )

final_fit_elapsed_33B = time.time() - final_fit_started_33B

final_convergence_warnings_33B = sum(
    issubclass(warning.category, ConvergenceWarning)
    for warning in final_warning_records_33B
)

if final_convergence_warnings_33B != 0:
    raise RuntimeError(
        "Nihai dış kat 4 modelinde yakınsama uyarısı oluştu."
    )

# ------------------------------------------------------------
# 19. Outer-fold-1 test predictions and metrics
# ------------------------------------------------------------

outer1_raw_probabilities_33B = final_pipeline_33B.predict_proba(
    X_outer_test_33B
)[:, 1]

raw_clipped_33B = np.clip(
    outer1_raw_probabilities_33B,
    1e-6,
    1 - 1e-6,
)
raw_logit_33B = np.log(
    raw_clipped_33B / (1 - raw_clipped_33B)
)

outer1_platt_probabilities_33B = expit(
    platt_intercept_33B + platt_slope_33B * raw_logit_33B
)

for probabilities, name in [
    (outer1_raw_probabilities_33B, "raw"),
    (outer1_platt_probabilities_33B, "platt"),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(f"{name} tahminlerinde eksik değer var.")

    if not np.all(
        (probabilities >= 0) & (probabilities <= 1)
    ):
        raise RuntimeError(
            f"{name} tahminlerinde geçersiz olasılık değeri var."
        )

raw_metrics_33B = probability_metrics_33B(
    y_outer_test_33B,
    outer1_raw_probabilities_33B,
)
platt_metrics_33B = probability_metrics_33B(
    y_outer_test_33B,
    outer1_platt_probabilities_33B,
)

raw_calibration_intercept_33B, raw_calibration_slope_33B = (
    calibration_intercept_slope_33B(
        y_outer_test_33B,
        outer1_raw_probabilities_33B,
    )
)

platt_calibration_intercept_33B, platt_calibration_slope_33B = (
    calibration_intercept_slope_33B(
        y_outer_test_33B,
        outer1_platt_probabilities_33B,
    )
)

outer1_test_results_33B = pd.DataFrame(
    [
        {
            "outer_fold": 1,
            "model": "parsimonious_clinical_logistic",
            "probability_type": "raw",
            **raw_metrics_33B,
            "calibration_intercept": raw_calibration_intercept_33B,
            "calibration_slope": raw_calibration_slope_33B,
        },
        {
            "outer_fold": 1,
            "model": "parsimonious_clinical_logistic",
            "probability_type": "platt_calibrated",
            **platt_metrics_33B,
            "calibration_intercept": platt_calibration_intercept_33B,
            "calibration_slope": platt_calibration_slope_33B,
        },
    ]
)

# ------------------------------------------------------------
# 20. Feature coefficient audit
# ------------------------------------------------------------

fitted_preprocessor_33B = final_pipeline_33B.named_steps[
    "preprocessor"
]
fitted_model_33B = final_pipeline_33B.named_steps["model"]

processed_feature_names_33B = (
    fitted_preprocessor_33B.get_feature_names_out()
)
model_coefficients_33B = fitted_model_33B.coef_.reshape(-1)

if len(processed_feature_names_33B) != len(model_coefficients_33B):
    raise RuntimeError("Feature ve katsayı sayıları uyuşmuyor.")

if len(set(processed_feature_names_33B)) != len(
    processed_feature_names_33B
):
    raise RuntimeError("İşlenmiş feature adlarında yinelenme var.")

coefficient_table_33B = pd.DataFrame(
    {
        "processed_feature": processed_feature_names_33B,
        "coefficient": model_coefficients_33B,
    }
)
coefficient_table_33B["absolute_coefficient"] = (
    coefficient_table_33B["coefficient"].abs()
)
coefficient_table_33B["is_nonzero"] = ~np.isclose(
    coefficient_table_33B["coefficient"],
    0.0,
    atol=1e-12,
)
coefficient_table_33B["absolute_rank"] = (
    coefficient_table_33B["absolute_coefficient"]
    .rank(method="first", ascending=False)
    .astype(int)
)
coefficient_table_33B = (
    coefficient_table_33B.sort_values(
        ["absolute_coefficient", "processed_feature"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

nonzero_coefficients_33B = int(
    coefficient_table_33B["is_nonzero"].sum()
)

final_model_summary_33B = pd.DataFrame(
    [
        {
            "outer_fold": 1,
            "selected_candidate": selected_candidate_33B,
            "selected_C": selected_C_33B,
            "selected_l1_ratio": selected_l1_ratio_33B,
            "training_patients": len(X_outer_training_33B),
            "training_hospitals": len(training_hospitals_33B),
            "training_events": int(y_outer_training_33B.sum()),
            "test_patients": len(X_outer_test_33B),
            "test_hospitals": len(test_hospitals_33B),
            "test_events": int(y_outer_test_33B.sum()),
            "hospital_overlap": len(hospital_overlap_33B),
            "processed_feature_columns": len(
                processed_feature_names_33B
            ),
            "nonzero_coefficients": nonzero_coefficients_33B,
            "model_intercept": float(
                fitted_model_33B.intercept_[0]
            ),
            "convergence_warnings": final_convergence_warnings_33B,
            "fit_elapsed_seconds": float(final_fit_elapsed_33B),
            "locked_platt_intercept": platt_intercept_33B,
            "locked_platt_slope": platt_slope_33B,
            "protocol_sha256": EXPECTED_PROTOCOL_SHA_33B,
            "selection_sha256": selection_sha_33B,
        }
    ]
)

# ------------------------------------------------------------
# 21. Secure BigQuery outer-test checkpoint
# ------------------------------------------------------------

outer1_prediction_df_33B = pd.DataFrame(
    {
        "id_row": outer_test_meta_33B["id_row"].astype(str),
        "outer_fold": np.full(
            len(outer_test_meta_33B),
            1,
            dtype=np.int64,
        ),
        "label_stage23": y_outer_test_33B.astype(np.int64),
        "prediction_raw": outer1_raw_probabilities_33B.astype(
            np.float64
        ),
        "prediction_platt": outer1_platt_probabilities_33B.astype(
            np.float64
        ),
        "model_name": "parsimonious_clinical_logistic",
        "model_version": "clinical8_v1_nested_cv",
    }
)

if len(outer1_prediction_df_33B) != 11688:
    raise RuntimeError(
        "Dış kat 4 tahmin satır sayısı 11.688 değil."
    )

if outer1_prediction_df_33B["id_row"].duplicated().any():
    raise RuntimeError(
        "Dış kat 4 tahminlerinde yinelenen id_row var."
    )

if int(
    outer1_prediction_df_33B["label_stage23"].sum()
) != 606:
    raise RuntimeError("Dış kat 4 olay sayısı 606 değil.")

prediction_table_id_33B = (
    f"{TARGET_DATASET}."
    "model_clinical_lr_outer_predictions_outer1_v1"
)

prediction_load_config_33B = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("prediction_platt", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("model_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("model_version", "STRING", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

print("\nUploading secure outer-fold-1 prediction checkpoint:")
print(prediction_table_id_33B)

client.load_table_from_dataframe(
    outer1_prediction_df_33B,
    prediction_table_id_33B,
    job_config=prediction_load_config_33B,
    location=BQ_LOCATION,
).result()

# ------------------------------------------------------------
# 22. BigQuery outer-test checkpoint verification
# ------------------------------------------------------------

SQL_VERIFY_PREDICTIONS_33B = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL) AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL) AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0 OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0 OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw) AS minimum_raw_probability,
  MAX(prediction_raw) AS maximum_raw_probability,
  MIN(prediction_platt) AS minimum_platt_probability,
  MAX(prediction_platt) AS maximum_platt_probability
FROM `{prediction_table_id_33B}`;
"""

prediction_verification_33B = client.query(
    SQL_VERIFY_PREDICTIONS_33B,
    location=BQ_LOCATION,
).to_dataframe()

verification_row_33B = prediction_verification_33B.iloc[0]

expected_prediction_values_33B = {
    "prediction_rows": 11688,
    "distinct_rows": 11688,
    "outer_folds": 1,
    "minimum_outer_fold": 1,
    "maximum_outer_fold": 1,
    "events": 606,
    "nonevents": 11082,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in expected_prediction_values_33B.items():
    actual_value = int(verification_row_33B[field])
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: bulunan={actual_value}, beklenen={expected_value}"
        )

# ------------------------------------------------------------
# 23. Save only aggregate and feature-level outputs to Drive
# ------------------------------------------------------------

test_results_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_outer1_test_results.csv",
)
model_summary_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_final_model_outer1.csv",
)
coefficient_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_coefficients_outer1.csv",
)
evaluation_json_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_final_evaluation_outer1.json",
)
evaluation_sha_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_final_evaluation_outer1_SHA256.txt",
)

outer1_test_results_33B.to_csv(
    test_results_path_33B,
    index=False,
)
final_model_summary_33B.to_csv(
    model_summary_path_33B,
    index=False,
)
coefficient_table_33B.to_csv(
    coefficient_path_33B,
    index=False,
)

evaluation_configuration_33B = {
    "outer_fold": 1,
    "model_family": "parsimonious_clinical_elastic_net_logistic_regression",
    "selected_candidate": selected_candidate_33B,
    "selected_C": selected_C_33B,
    "selected_l1_ratio": selected_l1_ratio_33B,
    "training_patients": 46803,
    "training_hospitals": 158,
    "test_patients": 11688,
    "test_hospitals": 40,
    "hospital_overlap": 0,
    "locked_platt_intercept": platt_intercept_33B,
    "locked_platt_slope": platt_slope_33B,
    "processed_feature_columns": int(
        len(processed_feature_names_33B)
    ),
    "nonzero_coefficients": int(nonzero_coefficients_33B),
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_33B,
    "selection_sha256": selection_sha_33B,
    "secure_prediction_table": prediction_table_id_33B,
    "patient_level_prediction_written_to_drive": False,
    "analysis_role": "additional_post_hoc_clinical_baseline",
    "predictors": CLINICAL_PREDICTORS_33B,
}

with open(
    evaluation_json_path_33B,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        evaluation_configuration_33B,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(evaluation_json_path_33B, "rb") as file_handle:
    evaluation_sha_33B = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    evaluation_sha_path_33B,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(evaluation_sha_33B + "\n")

# ------------------------------------------------------------
# 24. Final outputs
# ------------------------------------------------------------

pooled_integrity_33B = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_33B),
            pooled_oof_33B["id_row"].nunique(),
            pooled_oof_33B["candidate_id"].nunique(),
            pooled_oof_33B["inner_fold"].nunique(),
            EXPECTED_SPLIT_33B["training_events"],
            (
                EXPECTED_SPLIT_33B["training_rows"]
                - EXPECTED_SPLIT_33B["training_events"]
            ),
            int(
                pooled_oof_33B.duplicated(
                    subset=["candidate_id", "id_row"]
                ).sum()
            ),
            int(pooled_oof_33B["prediction_raw"].isna().sum()),
            int(
                (~pooled_oof_33B["prediction_raw"].between(0, 1)).sum()
            ),
            pooled_load_method_33B,
        ],
    }
)

print("\n33B OUTER-FOLD-1 INNER CHECKPOINT SUMMARY")
display(checkpoint_summary_33B)

print("\n33B OUTER-FOLD-1 POOLED OOF INTEGRITY")
display(pooled_integrity_33B)

print("\n33B OUTER-FOLD-1 CANDIDATE RESULTS")
display(candidate_results_33B)

print("\n33B OUTER-FOLD-1 SELECTED MODEL")
display(selected_model_33B)

print("\n33B OUTER-FOLD-1 FINAL MODEL SUMMARY")
display(final_model_summary_33B)

print("\n33B OUTER-FOLD-1 TEST RESULTS")
display(outer1_test_results_33B)

print("\n33B OUTER-FOLD-1 BIGQUERY VERIFICATION")
display(prediction_verification_33B)

print("\n33B OUTER-FOLD-1 TOP 20 ABSOLUTE COEFFICIENTS")
display(coefficient_table_33B.head(20))

print("\nSelection SHA-256:")
print(selection_sha_33B)

print("\nEvaluation SHA-256:")
print(evaluation_sha_33B)

print("\nSaved:")
print(fit_audit_path_33B)
print(checkpoint_summary_path_33B)
print(candidate_results_path_33B)
print(selected_model_path_33B)
print(selection_json_path_33B)
print(selection_sha_path_33B)
print(test_results_path_33B)
print(model_summary_path_33B)
print(coefficient_path_33B)
print(evaluation_json_path_33B)
print(evaluation_sha_path_33B)

print(
    "\n33B PASS: Outer-fold-1 nested modelling "
    "and locked test evaluation are complete."
)
print(
    "All inner OOF and outer-test patient-level "
    "predictions were stored only in BigQuery."
)
print(
    "No patient-level prediction file was "
    "written to Google Drive."
)

_ = gc.collect()

In [ ]:
import os, json, hashlib

print("STARTING CLINICAL BASELINE NUMERICAL CONVERGENCE AMENDMENT — CODE VERSION 33A1")

if "MODEL_OUTPUT_DIR" not in globals():
    raise RuntimeError("Önce 07A hücresini çalıştır.")

ORIGINAL_PROTOCOL_SHA = "94b0abb218dbef4e349702ea2824ca4e31dfbe36c53efa05ba1a8bd1f38f835e"
EXPECTED_AMENDMENT_SHA = "fcbcf857caa9aaad7ffd6691b4b61ac6503d09a6eee80db84bc2967ddf1f1a6a"

original_sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A_locked_parsimonious_clinical_baseline_protocol_v1_SHA256.txt",
)
if not os.path.exists(original_sha_path):
    raise FileNotFoundError(original_sha_path)

with open(original_sha_path, "r", encoding="utf-8") as f:
    observed_original_sha = f.read().strip()

if observed_original_sha != ORIGINAL_PROTOCOL_SHA:
    raise RuntimeError(
        "33A original protocol SHA mismatch: " + observed_original_sha
    )

amendment = {'analysis_version': '33A1', 'amendment_name': 'parsimonious_clinical_baseline_numerical_convergence_amendment_v1', 'amends_protocol_sha256': '94b0abb218dbef4e349702ea2824ca4e31dfbe36c53efa05ba1a8bd1f38f835e', 'amendment_timing': 'before_any_outer_test_evaluation', 'reason': 'The selected final outer-fold-1 model emitted a ConvergenceWarning at max_iter=5000. All five inner-fold candidate fits completed with zero convergence warnings. The iteration ceiling is increased solely to allow convergence of the same penalized objective.', 'change': {'parameter': 'LogisticRegression.max_iter', 'old_value': 5000, 'new_value': 20000, 'tol_unchanged': 0.0001, 'solver_unchanged': 'saga', 'penalty_unchanged': 'elasticnet', 'predictors_unchanged': True, 'candidate_grid_unchanged': True, 'selection_rule_unchanged': True, 'platt_calibration_unchanged': True, 'class_weight_unchanged': None, 'random_state_unchanged': 20260721}, 'existing_inner_checkpoints': 'Reusable only because all recorded inner candidate fits converged under the original 5000-iteration ceiling.', 'scientific_guard': 'This is a numerical optimization amendment, not performance-based retuning. No outer-test prediction or metric was accessed before it.'}
amendment_text = json.dumps(amendment, indent=2, sort_keys=True)
observed_amendment_sha = hashlib.sha256(
    amendment_text.encode("utf-8")
).hexdigest()

if observed_amendment_sha != EXPECTED_AMENDMENT_SHA:
    raise RuntimeError("Internal amendment SHA mismatch.")

amendment_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A1_parsimonious_clinical_baseline_convergence_amendment_v1.json",
)
amendment_sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A1_parsimonious_clinical_baseline_convergence_amendment_v1_SHA256.txt",
)

with open(amendment_path, "w", encoding="utf-8") as f:
    f.write(amendment_text)

with open(amendment_sha_path, "w", encoding="utf-8") as f:
    f.write(observed_amendment_sha + "\n")

print("\nOriginal 33A protocol SHA-256:")
print(ORIGINAL_PROTOCOL_SHA)
print("\n33A1 amendment SHA-256:")
print(observed_amendment_sha)
print("\nOnly numerical change: max_iter 5000 -> 20000")
print("tol / solver / penalty / predictors / grid / selection / calibration: UNCHANGED")
print("\n33A1 PASS: Numerical convergence amendment locked before any outer-test evaluation.")

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from sklearn.exceptions import ConvergenceWarning

from IPython.display import display

# ============================================================
# 33B — PARSIMONIOUS CLINICAL BASELINE OUTER FOLD 1
#
# Resume-safe:
# - Each inner fold is stored in a separate BigQuery table.
# - Completed inner folds are automatically skipped.
#
# BigQuery free-tier compatible:
# - No DELETE / INSERT / UPDATE / MERGE is used.
#
# Privacy:
# - Patient-level predictions are stored only in BigQuery.
# - No patient-level prediction file is written to Drive.
# ============================================================

print("STARTING PARSIMONIOUS CLINICAL BASELINE OUTER FOLD 1 — CODE VERSION 33B-R1")

OUTER_FOLD_33B = 1
MODEL_RANDOM_SEED_33B = 20260721

EXPECTED_PROTOCOL_SHA_33B = (
    "94b0abb218dbef4e349702ea2824ca4e"
    "31dfbe36c53efa05ba1a8bd1f38f835e"
)

EXPECTED_AMENDMENT_SHA_33B = (
    "fcbcf857caa9aaad7ffd6691b4b61ac6"
    "503d09a6eee80db84bc2967ddf1f1a6a"
)

EXPECTED_SPLIT_33B = {
    "training_rows": 46803,
    "test_rows": 11688,
    "training_hospitals": 158,
    "test_hospitals": 40,
    "training_events": 2426,
    "test_events": 606,
}

EXPECTED_INNER_33B = {
    1: {"validation_rows": 10650, "validation_events": 578},
    2: {"validation_rows": 7084, "validation_events": 359},
    3: {"validation_rows": 11915, "validation_events": 637},
    4: {"validation_rows": 10019, "validation_events": 488},
    5: {"validation_rows": 7135, "validation_events": 364},
}

CLINICAL_PREDICTORS_33B = [
    "x_age_years",
    "x_sex",
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
    "x_lab_bun_last",
    "x_vital_respiratory_rate_last",
    "x_vital_noninvasive_systolic_bp_last",
]

CLINICAL_NUMERIC_33B = [
    "x_age_years",
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
    "x_lab_bun_last",
    "x_vital_respiratory_rate_last",
    "x_vital_noninvasive_systolic_bp_last",
]

CLINICAL_CATEGORICAL_33B = ["x_sex"]

# ------------------------------------------------------------
# 1. Required runtime objects
# ------------------------------------------------------------

required_objects_33B = [
    "core_df_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_33B = [
    name for name in required_objects_33B if name not in globals()
]

if missing_objects_33B:
    raise RuntimeError(
        "Eksik RAM nesneleri var: "
        + ", ".join(missing_objects_33B)
        + ". Önce 07A ve 07B hücrelerini çalıştır."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"58.491 satır bekleniyordu; {len(core_df_07B)} bulundu."
    )

missing_predictors_33B = [
    column for column in CLINICAL_PREDICTORS_33B
    if column not in core_df_07B.columns
]

if missing_predictors_33B:
    raise RuntimeError(
        "Kilitli klinik predictor(lar) core_df_07B içinde yok: "
        + ", ".join(missing_predictors_33B)
    )

# ------------------------------------------------------------
# 2. Locked protocol SHA check
# ------------------------------------------------------------

protocol_sha_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A_locked_parsimonious_clinical_baseline_protocol_v1_SHA256.txt",
)

if not os.path.exists(protocol_sha_path_33B):
    raise FileNotFoundError(
        "Model protokolü SHA dosyası bulunamadı: "
        + protocol_sha_path_33B
    )

with open(protocol_sha_path_33B, "r", encoding="utf-8") as file_handle:
    observed_protocol_sha_33B = file_handle.read().strip()

if observed_protocol_sha_33B != EXPECTED_PROTOCOL_SHA_33B:
    raise RuntimeError(
        "Kilitli model protokolü SHA değeri değişmiş: "
        + observed_protocol_sha_33B
    )

amendment_sha_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A1_parsimonious_clinical_baseline_convergence_amendment_v1_SHA256.txt",
)

if not os.path.exists(amendment_sha_path_33B):
    raise FileNotFoundError(
        "Önce 33A1 convergence amendment scriptini çalıştır: "
        + amendment_sha_path_33B
    )

with open(amendment_sha_path_33B, "r", encoding="utf-8") as file_handle:
    observed_amendment_sha_33B = file_handle.read().strip()

if observed_amendment_sha_33B != EXPECTED_AMENDMENT_SHA_33B:
    raise RuntimeError(
        "33A1 amendment SHA değeri değişmiş: "
        + observed_amendment_sha_33B
    )

print("33A protocol SHA guard: PASS")
print("33A1 convergence amendment SHA guard: PASS")

# ------------------------------------------------------------
# 3. Locked inner-hospital mapping
# ------------------------------------------------------------

inner_mapping_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_33B):
    raise FileNotFoundError(
        "Kilitli iç kat haritası bulunamadı: "
        + inner_mapping_path_33B
    )

inner_mapping_all_33B = pd.read_csv(
    inner_mapping_path_33B,
    dtype={"group_hospital": str},
)

inner_mapping_part_33B = (
    inner_mapping_all_33B.loc[
        inner_mapping_all_33B["outer_fold"].astype(int) == OUTER_FOLD_33B,
        ["group_hospital", "inner_fold"],
    ]
    .copy()
)

inner_mapping_part_33B["group_hospital"] = (
    inner_mapping_part_33B["group_hospital"].astype(str)
)
inner_mapping_part_33B["inner_fold"] = (
    inner_mapping_part_33B["inner_fold"].astype(int)
)

if len(inner_mapping_part_33B) != 158:
    raise RuntimeError(
        "Dış kat 1 eğitim kümesi için 158 hastane ataması bekleniyordu."
    )

if inner_mapping_part_33B["group_hospital"].duplicated().any():
    raise RuntimeError("İç kat haritasında yinelenen hastane var.")

hospital_to_inner_fold_33B = dict(
    zip(
        inner_mapping_part_33B["group_hospital"],
        inner_mapping_part_33B["inner_fold"],
    )
)

# ------------------------------------------------------------
# 4. Prepare model matrices
# ------------------------------------------------------------

X_all_33B = core_df_07B[CLINICAL_PREDICTORS_33B].copy()

for column in CLINICAL_NUMERIC_33B:
    X_all_33B[column] = pd.to_numeric(
        X_all_33B[column], errors="coerce"
    ).astype("float64")

for column in CLINICAL_CATEGORICAL_33B:
    category_series = X_all_33B[column].astype("object")
    X_all_33B[column] = category_series.where(
        pd.notna(category_series), np.nan
    )

outer_fold_vector_33B = (
    core_df_07B["outer_fold"].astype(int).to_numpy()
)

outer_training_mask_33B = outer_fold_vector_33B != OUTER_FOLD_33B
outer_test_mask_33B = outer_fold_vector_33B == OUTER_FOLD_33B

X_outer_training_33B = (
    X_all_33B.loc[outer_training_mask_33B].reset_index(drop=True)
)
X_outer_test_33B = (
    X_all_33B.loc[outer_test_mask_33B].reset_index(drop=True)
)

outer_training_meta_33B = (
    core_df_07B.loc[
        outer_training_mask_33B,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_33B = (
    core_df_07B.loc[
        outer_test_mask_33B,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [outer_training_meta_33B, outer_test_meta_33B]:
    dataframe["id_row"] = dataframe["id_row"].astype(str)
    dataframe["group_hospital"] = dataframe["group_hospital"].astype(str)
    dataframe["label_stage23"] = dataframe["label_stage23"].astype(int)

y_outer_training_33B = (
    outer_training_meta_33B["label_stage23"].to_numpy(dtype=np.int8)
)
y_outer_test_33B = (
    outer_test_meta_33B["label_stage23"].to_numpy(dtype=np.int8)
)

groups_outer_training_33B = (
    outer_training_meta_33B["group_hospital"].to_numpy(dtype=str)
)

training_hospitals_33B = set(
    outer_training_meta_33B["group_hospital"]
)
test_hospitals_33B = set(
    outer_test_meta_33B["group_hospital"]
)
hospital_overlap_33B = training_hospitals_33B & test_hospitals_33B

if hospital_overlap_33B:
    raise RuntimeError("Dış eğitim ve test hastaneleri çakışıyor.")

actual_split_33B = {
    "training_rows": len(X_outer_training_33B),
    "test_rows": len(X_outer_test_33B),
    "training_hospitals": len(training_hospitals_33B),
    "test_hospitals": len(test_hospitals_33B),
    "training_events": int(y_outer_training_33B.sum()),
    "test_events": int(y_outer_test_33B.sum()),
}

for metric, expected_value in EXPECTED_SPLIT_33B.items():
    actual_value = actual_split_33B[metric]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: bulunan={actual_value}, beklenen={expected_value}"
        )

inner_fold_vector_33B = np.array(
    [
        hospital_to_inner_fold_33B.get(hospital, -1)
        for hospital in groups_outer_training_33B
    ],
    dtype=int,
)

if (inner_fold_vector_33B == -1).any():
    raise RuntimeError(
        "Bazı dış eğitim hastanelerine iç kat atanmadı."
    )

if set(np.unique(inner_fold_vector_33B)) != {1, 2, 3, 4, 5}:
    raise RuntimeError("İç kat değerleri 1–5 değil.")

for inner_fold, expected in EXPECTED_INNER_33B.items():
    validation_mask = inner_fold_vector_33B == inner_fold
    observed_rows = int(validation_mask.sum())
    observed_events = int(y_outer_training_33B[validation_mask].sum())

    if observed_rows != expected["validation_rows"]:
        raise RuntimeError(
            f"Inner {inner_fold} validation_rows: "
            f"bulunan={observed_rows}, "
            f"beklenen={expected['validation_rows']}"
        )

    if observed_events != expected["validation_events"]:
        raise RuntimeError(
            f"Inner {inner_fold} validation_events: "
            f"bulunan={observed_events}, "
            f"beklenen={expected['validation_events']}"
        )

# ------------------------------------------------------------
# 5. Locked candidate grid
# ------------------------------------------------------------

candidate_grid_33B = [
    {"candidate_id": "CLIN01", "C": 0.03, "l1_ratio": 0.00},
    {"candidate_id": "CLIN02", "C": 0.10, "l1_ratio": 0.00},
    {"candidate_id": "CLIN03", "C": 0.30, "l1_ratio": 0.00},
    {"candidate_id": "CLIN04", "C": 0.10, "l1_ratio": 0.25},
    {"candidate_id": "CLIN05", "C": 0.30, "l1_ratio": 0.25},
    {"candidate_id": "CLIN06", "C": 0.30, "l1_ratio": 0.50},
]

# ------------------------------------------------------------
# 6. Preprocessing and model factories
# ------------------------------------------------------------

def make_preprocessor_33B():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
            (
                "scaler",
                StandardScaler(with_mean=False),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                CLINICAL_NUMERIC_33B,
            ),
            (
                "categorical",
                categorical_pipeline,
                CLINICAL_CATEGORICAL_33B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_model_33B(C_value, l1_ratio_value):
    return LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=float(C_value),
        l1_ratio=float(l1_ratio_value),
        class_weight=None,
        max_iter=20000,
        tol=1e-4,
        random_state=MODEL_RANDOM_SEED_33B,
    )


def checkpoint_table_id_33B(inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_clinical_lr_inner_oof_outer1_inner{inner_fold}_v1"
    )

# ------------------------------------------------------------
# 7. BigQuery checkpoint verification
# ------------------------------------------------------------

def verify_checkpoint_33B(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):
    table_id = checkpoint_table_id_33B(inner_fold)

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS distinct_id_count,
      COUNT(DISTINCT candidate_id) AS candidate_count,
      COUNT(DISTINCT outer_fold) AS outer_fold_count,
      COUNT(DISTINCT inner_fold) AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(candidate_id, '|', id_row)
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(
        prediction_raw < 0 OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(sql, location=BQ_LOCATION).to_dataframe()
    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows * len(candidate_grid_33B)
    )
    expected_positive_rows = (
        expected_validation_events * len(candidate_grid_33B)
    )
    expected_negative_rows = (
        (expected_validation_rows - expected_validation_events)
        * len(candidate_grid_33B)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": expected_validation_rows,
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": expected_total_rows,
        "positive_prediction_rows": expected_positive_rows,
        "negative_prediction_rows": expected_negative_rows,
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": 1,
        "maximum_outer_fold": 1,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failure_items = []

    for field, expected_value in expected_values.items():
        actual_value = int(row[field])
        if actual_value != expected_value:
            complete = False
            failure_items.append(
                f"{field}={actual_value}, expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failure_items),
        "check": check,
        "row": row,
    }

# ------------------------------------------------------------
# 8. BigQuery load schema
# ------------------------------------------------------------

checkpoint_load_config_33B = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("inner_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("candidate_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

# ------------------------------------------------------------
# 9. Aggregate fit-audit file
# ------------------------------------------------------------

fit_audit_columns_33B = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "C",
    "l1_ratio",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "convergence_warnings",
    "elapsed_seconds",
]

fit_audit_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_inner_fit_audit_outer1.csv",
)

if os.path.exists(fit_audit_path_33B):
    fit_audit_33B = pd.read_csv(fit_audit_path_33B)
else:
    fit_audit_33B = pd.DataFrame(columns=fit_audit_columns_33B)

for column in fit_audit_columns_33B:
    if column not in fit_audit_33B.columns:
        fit_audit_33B[column] = np.nan

fit_audit_33B = fit_audit_33B[fit_audit_columns_33B].copy()

# ------------------------------------------------------------
# 10. Train six candidates in five locked inner folds
# ------------------------------------------------------------

for inner_fold in range(1, 6):
    inner_training_mask = inner_fold_vector_33B != inner_fold
    inner_validation_mask = inner_fold_vector_33B == inner_fold

    training_rows = int(inner_training_mask.sum())
    validation_rows = int(inner_validation_mask.sum())
    training_events = int(
        y_outer_training_33B[inner_training_mask].sum()
    )
    validation_events = int(
        y_outer_training_33B[inner_validation_mask].sum()
    )

    expected_inner = EXPECTED_INNER_33B[inner_fold]
    expected_training_rows = (
        EXPECTED_SPLIT_33B["training_rows"]
        - expected_inner["validation_rows"]
    )
    expected_training_events = (
        EXPECTED_SPLIT_33B["training_events"]
        - expected_inner["validation_events"]
    )

    if training_rows != expected_training_rows:
        raise RuntimeError(
            f"Inner {inner_fold} training_rows: "
            f"bulunan={training_rows}, "
            f"beklenen={expected_training_rows}"
        )

    if training_events != expected_training_events:
        raise RuntimeError(
            f"Inner {inner_fold} training_events: "
            f"bulunan={training_events}, "
            f"beklenen={expected_training_events}"
        )

    existing_check = verify_checkpoint_33B(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if existing_check["complete"]:
        print(
            f"Outer 1 / inner {inner_fold}: "
            "permanent checkpoint already complete; "
            "skipping model fitting."
        )
        continue

    training_hospital_set = set(
        groups_outer_training_33B[inner_training_mask]
    )
    validation_hospital_set = set(
        groups_outer_training_33B[inner_validation_mask]
    )

    if training_hospital_set & validation_hospital_set:
        raise RuntimeError(
            f"Inner fold {inner_fold}: hastane çakışması bulundu."
        )

    print(f"\nOuter 1 / inner {inner_fold}")
    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_33B()
    preprocessing_started = time.time()

    X_inner_training_processed = preprocessor.fit_transform(
        X_outer_training_33B.loc[inner_training_mask]
    )
    X_inner_validation_processed = preprocessor.transform(
        X_outer_training_33B.loc[inner_validation_mask]
    )

    preprocessing_elapsed = time.time() - preprocessing_started

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "İşlenmiş eğitim ve doğrulama sütun sayıları farklı."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_elapsed, 2),
    )

    y_inner_training = y_outer_training_33B[inner_training_mask]
    y_inner_validation = y_outer_training_33B[inner_validation_mask]

    validation_ids = (
        outer_training_meta_33B.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_33B:
        candidate_id = candidate["candidate_id"]

        print(
            "  Fitting",
            candidate_id,
            "| C =",
            candidate["C"],
            "| l1_ratio =",
            candidate["l1_ratio"],
        )

        model = make_model_33B(
            candidate["C"],
            candidate["l1_ratio"],
        )

        fitting_started = time.time()

        with warnings.catch_warnings(record=True) as warning_records:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(
                X_inner_training_processed,
                y_inner_training,
            )

        fitting_elapsed = time.time() - fitting_started

        convergence_warning_count = sum(
            issubclass(warning.category, ConvergenceWarning)
            for warning in warning_records
        )

        validation_probabilities = model.predict_proba(
            X_inner_validation_processed
        )[:, 1]

        if np.isnan(validation_probabilities).any():
            raise RuntimeError(
                f"{candidate_id}, inner {inner_fold}: eksik tahmin."
            )

        if not np.all(
            (validation_probabilities >= 0)
            & (validation_probabilities <= 1)
        ):
            raise RuntimeError(
                f"{candidate_id}: geçersiz olasılık."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        1,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": y_inner_validation.astype(np.int64),
                    "prediction_raw": validation_probabilities.astype(
                        np.float64
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": 1,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "C": candidate["C"],
                "l1_ratio": candidate["l1_ratio"],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": validation_events,
                "processed_columns": int(
                    X_inner_training_processed.shape[1]
                ),
                "convergence_warnings": int(
                    convergence_warning_count
                ),
                "elapsed_seconds": float(fitting_elapsed),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows * len(candidate_grid_33B)
    )

    if len(checkpoint_df) != expected_checkpoint_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: checkpoint satır sayısı hatalı."
        )

    if checkpoint_df.duplicated(
        subset=["id_row", "candidate_id"]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "yinelenen aday–hasta tahmini var."
        )

    target_checkpoint_table = checkpoint_table_id_33B(inner_fold)

    print(
        "Uploading permanent checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_33B,
        location=BQ_LOCATION,
    ).result()

    new_audit_df = pd.DataFrame(current_audit_rows)

    if len(fit_audit_33B) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_33B["outer_fold"],
                    errors="coerce",
                )
                == 1
            )
            & (
                pd.to_numeric(
                    fit_audit_33B["inner_fold"],
                    errors="coerce",
                )
                == inner_fold
            )
        )
        fit_audit_33B = fit_audit_33B.loc[keep_mask].copy()

    if fit_audit_33B.empty:
        fit_audit_33B = new_audit_df.copy()
    else:
        fit_audit_33B = pd.concat(
            [fit_audit_33B, new_audit_df],
            ignore_index=True,
        )

    fit_audit_33B = (
        fit_audit_33B[fit_audit_columns_33B]
        .sort_values(
            ["outer_fold", "inner_fold", "candidate_id"]
        )
        .reset_index(drop=True)
    )

    fit_audit_33B.to_csv(
        fit_audit_path_33B,
        index=False,
    )

    completed_check = verify_checkpoint_33B(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint doğrulanamadı: "
            + completed_check["reason"]
        )

    print(
        f"Outer 1 / inner {inner_fold}: "
        "permanent checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 11. Final checkpoint summary
# ------------------------------------------------------------

checkpoint_summary_rows_33B = []

for inner_fold in range(1, 6):
    validation_mask = inner_fold_vector_33B == inner_fold
    validation_rows = int(validation_mask.sum())
    validation_events = int(
        y_outer_training_33B[validation_mask].sum()
    )

    final_check = verify_checkpoint_33B(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "son checkpoint denetimi başarısız. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_33B.append(
        {
            "outer_fold": 1,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(row["row_count"]),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(row["candidate_count"]),
            "positive_prediction_rows": int(
                row["positive_prediction_rows"]
            ),
            "negative_prediction_rows": int(
                row["negative_prediction_rows"]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check["table_id"],
        }
    )

checkpoint_summary_33B = (
    pd.DataFrame(checkpoint_summary_rows_33B)
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_33B["distinct_validation_patients"].sum()
) != EXPECTED_SPLIT_33B["training_rows"]:
    raise RuntimeError(
        "Toplam doğrulama hasta sayısı 46.803 değil."
    )

expected_total_oof_rows_33B = (
    EXPECTED_SPLIT_33B["training_rows"]
    * len(candidate_grid_33B)
)

if int(checkpoint_summary_33B["checkpoint_rows"].sum()) != (
    expected_total_oof_rows_33B
):
    raise RuntimeError("Toplam OOF tahmin satırı hatalı.")

checkpoint_summary_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_outer1_inner_checkpoint_summary.csv",
)
checkpoint_summary_33B.to_csv(
    checkpoint_summary_path_33B,
    index=False,
)

# ------------------------------------------------------------
# 12. Pool all inner OOF predictions
# ------------------------------------------------------------

checkpoint_tables_33B = [
    checkpoint_table_id_33B(inner_fold)
    for inner_fold in range(1, 6)
]

union_parts_33B = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_33B
]

SQL_LOAD_POOLED_OOF_33B = "\nUNION ALL\n".join(union_parts_33B)

print("\nLoading pooled outer-fold-1 inner OOF predictions...")

query_job_33B = client.query(
    SQL_LOAD_POOLED_OOF_33B,
    location=BQ_LOCATION,
)

try:
    pooled_oof_33B = query_job_33B.to_dataframe(
        create_bqstorage_client=True
    )
    pooled_load_method_33B = "BigQuery Storage API"
except Exception as fast_path_error_33B:
    print(
        "Storage API unavailable; using standard BigQuery download."
    )
    print("Message:", type(fast_path_error_33B).__name__)
    pooled_oof_33B = query_job_33B.to_dataframe(
        create_bqstorage_client=False
    )
    pooled_load_method_33B = "Standard BigQuery API"

pooled_oof_33B["id_row"] = pooled_oof_33B["id_row"].astype(str)
pooled_oof_33B["candidate_id"] = pooled_oof_33B["candidate_id"].astype(str)

for column in ["outer_fold", "inner_fold", "label_stage23"]:
    pooled_oof_33B[column] = pd.to_numeric(
        pooled_oof_33B[column], errors="raise"
    ).astype(int)

pooled_oof_33B["prediction_raw"] = pd.to_numeric(
    pooled_oof_33B["prediction_raw"], errors="raise"
).astype(float)

# ------------------------------------------------------------
# 13. Pooled OOF integrity checks
# ------------------------------------------------------------

if len(pooled_oof_33B) != expected_total_oof_rows_33B:
    raise RuntimeError("Pooled OOF satır sayısı hatalı.")

if set(pooled_oof_33B["outer_fold"].unique()) != {1}:
    raise RuntimeError("Pooled OOF içinde dış kat 4 dışında kayıt var.")

if set(pooled_oof_33B["inner_fold"].unique()) != {1, 2, 3, 4, 5}:
    raise RuntimeError("Pooled OOF iç katları 1–5 değil.")

if pooled_oof_33B.duplicated(
    subset=["candidate_id", "id_row"]
).any():
    raise RuntimeError(
        "Pooled OOF içinde yinelenen aday–hasta tahmini bulundu."
    )

if pooled_oof_33B["prediction_raw"].isna().any():
    raise RuntimeError("Pooled OOF içinde eksik tahmin var.")

if not pooled_oof_33B["prediction_raw"].between(0, 1).all():
    raise RuntimeError(
        "Pooled OOF içinde geçersiz olasılık değeri var."
    )

expected_candidate_ids_33B = {
    "CLIN01",
    "CLIN02",
    "CLIN03",
    "CLIN04",
    "CLIN05",
    "CLIN06",
}

if set(pooled_oof_33B["candidate_id"].unique()) != (
    expected_candidate_ids_33B
):
    raise RuntimeError("Altı kilitli aday bulunmuyor.")

candidate_patient_counts_33B = (
    pooled_oof_33B.groupby("candidate_id")["id_row"].nunique()
)

if not (
    candidate_patient_counts_33B
    == EXPECTED_SPLIT_33B["training_rows"]
).all():
    raise RuntimeError(
        "Her aday için 46.803 farklı OOF hastası yok."
    )

candidate_event_counts_33B = (
    pooled_oof_33B.groupby("candidate_id")["label_stage23"].sum()
)

if not (
    candidate_event_counts_33B
    == EXPECTED_SPLIT_33B["training_events"]
).all():
    raise RuntimeError("Her aday için 2.426 olay yok.")

patient_label_consistency_33B = (
    pooled_oof_33B.groupby("id_row")["label_stage23"].nunique()
)
if (patient_label_consistency_33B > 1).any():
    raise RuntimeError(
        "Aynı hastanın adaylar arasında outcome etiketi farklı."
    )

patient_inner_fold_consistency_33B = (
    pooled_oof_33B.groupby("id_row")["inner_fold"].nunique()
)
if (patient_inner_fold_consistency_33B > 1).any():
    raise RuntimeError(
        "Aynı hasta birden fazla iç doğrulama katında bulundu."
    )

# ------------------------------------------------------------
# 14. Metric helpers
# ------------------------------------------------------------

def probability_metrics_33B(y_true, probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(roc_auc_score(y_true, probabilities)),
        "auprc": float(average_precision_score(y_true, probabilities)),
        "brier": float(brier_score_loss(y_true, probabilities)),
        "log_loss": float(
            log_loss(y_true, probabilities, labels=[0, 1])
        ),
        "mean_predicted_risk": float(probabilities.mean()),
        "observed_event_rate": float(np.mean(y_true)),
    }


def probability_logit_33B(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        probabilities / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_33B(y_true, probabilities):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        probability_logit_33B(probabilities),
        y_true,
    )
    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 15. Candidate pooled-OOF performance
# ------------------------------------------------------------

candidate_result_rows_33B = []

for candidate in candidate_grid_33B:
    candidate_id = candidate["candidate_id"]

    candidate_oof = (
        pooled_oof_33B.loc[
            pooled_oof_33B["candidate_id"] == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_33B(
        candidate_oof["label_stage23"].to_numpy(dtype=int),
        candidate_oof["prediction_raw"].to_numpy(dtype=float),
    )

    fit_part = fit_audit_33B.loc[
        (
            pd.to_numeric(
                fit_audit_33B["outer_fold"],
                errors="coerce",
            )
            == 1
        )
        & (
            fit_audit_33B["candidate_id"].astype(str)
            == candidate_id
        )
    ]

    convergence_warnings = (
        int(
            pd.to_numeric(
                fit_part["convergence_warnings"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    elapsed_seconds = (
        float(
            pd.to_numeric(
                fit_part["elapsed_seconds"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_33B.append(
        {
            "candidate_id": candidate_id,
            "C": candidate["C"],
            "l1_ratio": candidate["l1_ratio"],
            **metrics,
            "convergence_warnings": convergence_warnings,
            "elapsed_seconds": elapsed_seconds,
        }
    )

candidate_results_33B = pd.DataFrame(candidate_result_rows_33B)

candidate_results_33B = (
    candidate_results_33B.sort_values(
        ["auprc", "auroc", "brier", "candidate_id"],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)

candidate_results_33B["selection_rank"] = np.arange(
    1,
    len(candidate_results_33B) + 1,
)

best_row_33B = candidate_results_33B.iloc[0]
selected_candidate_33B = str(best_row_33B["candidate_id"])
selected_C_33B = float(best_row_33B["C"])
selected_l1_ratio_33B = float(best_row_33B["l1_ratio"])

# ------------------------------------------------------------
# 16. Platt calibration on selected pooled inner OOF
# ------------------------------------------------------------

selected_oof_33B = (
    pooled_oof_33B.loc[
        pooled_oof_33B["candidate_id"] == selected_candidate_33B
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_33B = selected_oof_33B[
    "label_stage23"
].to_numpy(dtype=int)
selected_oof_probability_33B = selected_oof_33B[
    "prediction_raw"
].to_numpy(dtype=float)

platt_calibrator_33B = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_33B.fit(
    probability_logit_33B(selected_oof_probability_33B),
    selected_oof_y_33B,
)

platt_intercept_33B = float(platt_calibrator_33B.intercept_[0])
platt_slope_33B = float(platt_calibrator_33B.coef_[0][0])

if (
    not np.isfinite(platt_intercept_33B)
    or not np.isfinite(platt_slope_33B)
    or platt_slope_33B <= 0
):
    raise RuntimeError("Platt kalibrasyon katsayıları geçersiz.")

selected_model_33B = pd.DataFrame(
    [
        {
            "outer_fold": 1,
            "selected_candidate": selected_candidate_33B,
            "selected_C": selected_C_33B,
            "selected_l1_ratio": selected_l1_ratio_33B,
            "selection_metric_primary": "pooled_inner_oof_auprc",
            "inner_oof_auprc": float(best_row_33B["auprc"]),
            "inner_oof_auroc": float(best_row_33B["auroc"]),
            "inner_oof_brier": float(best_row_33B["brier"]),
            "inner_oof_log_loss": float(best_row_33B["log_loss"]),
            "inner_oof_mean_predicted_risk": float(
                best_row_33B["mean_predicted_risk"]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_33B["observed_event_rate"]
            ),
            "platt_intercept": platt_intercept_33B,
            "platt_slope": platt_slope_33B,
            "protocol_sha256": EXPECTED_PROTOCOL_SHA_33B,
            "numerical_amendment_sha256": EXPECTED_AMENDMENT_SHA_33B,
            "model_max_iter": 20000,
        }
    ]
)

# ------------------------------------------------------------
# 17. Lock selection/calibration aggregate files
# ------------------------------------------------------------

candidate_results_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_candidate_results_outer1.csv",
)
selected_model_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_selected_model_outer1.csv",
)
selection_json_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_selection_calibration_outer1.json",
)
selection_sha_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_selection_calibration_outer1_SHA256.txt",
)

candidate_results_33B.to_csv(
    candidate_results_path_33B,
    index=False,
)
selected_model_33B.to_csv(
    selected_model_path_33B,
    index=False,
)

selection_configuration_33B = {
    "outer_fold": 1,
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_33B,
    "selection_metric_primary": "pooled inner out-of-fold AUPRC",
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
        "candidate_id ascending immutable tie-break",
    ],
    "selected_candidate": selected_candidate_33B,
    "selected_C": selected_C_33B,
    "selected_l1_ratio": selected_l1_ratio_33B,
    "inner_oof_auprc": float(best_row_33B["auprc"]),
    "inner_oof_auroc": float(best_row_33B["auroc"]),
    "inner_oof_brier": float(best_row_33B["brier"]),
    "platt_intercept": platt_intercept_33B,
    "platt_slope": platt_slope_33B,
    "inner_checkpoint_tables": checkpoint_tables_33B,
    "patient_level_oof_written_to_drive": False,
    "analysis_role": "additional_post_hoc_clinical_baseline",
    "predictors": CLINICAL_PREDICTORS_33B,
}

with open(
    selection_json_path_33B,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        selection_configuration_33B,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(selection_json_path_33B, "rb") as file_handle:
    selection_sha_33B = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    selection_sha_path_33B,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(selection_sha_33B + "\n")

# ------------------------------------------------------------
# 18. Fit final selected outer-fold-1 model
# ------------------------------------------------------------

final_pipeline_33B = Pipeline(
    steps=[
        ("preprocessor", make_preprocessor_33B()),
        (
            "model",
            make_model_33B(
                selected_C_33B,
                selected_l1_ratio_33B,
            ),
        ),
    ]
)

print(
    "\nFitting selected outer-fold-1 model "
    "on all 46,803 training patients..."
)

final_fit_started_33B = time.time()

with warnings.catch_warnings(record=True) as final_warning_records_33B:
    warnings.simplefilter("always", ConvergenceWarning)
    final_pipeline_33B.fit(
        X_outer_training_33B,
        y_outer_training_33B,
    )

final_fit_elapsed_33B = time.time() - final_fit_started_33B

final_model_n_iter_33B = int(
    np.max(
        np.asarray(
            final_pipeline_33B.named_steps["model"].n_iter_
        )
    )
)

final_convergence_warnings_33B = sum(
    issubclass(warning.category, ConvergenceWarning)
    for warning in final_warning_records_33B
)

if final_convergence_warnings_33B != 0:
    raise RuntimeError(
        "Nihai outer-fold-1 klinik baseline modeli max_iter=20000 "
        f"altında da yakınsamadı. n_iter_={final_model_n_iter_33B}"
    )

# ------------------------------------------------------------
# 19. Outer-fold-1 test predictions and metrics
# ------------------------------------------------------------

outer1_raw_probabilities_33B = final_pipeline_33B.predict_proba(
    X_outer_test_33B
)[:, 1]

raw_clipped_33B = np.clip(
    outer1_raw_probabilities_33B,
    1e-6,
    1 - 1e-6,
)
raw_logit_33B = np.log(
    raw_clipped_33B / (1 - raw_clipped_33B)
)

outer1_platt_probabilities_33B = expit(
    platt_intercept_33B + platt_slope_33B * raw_logit_33B
)

for probabilities, name in [
    (outer1_raw_probabilities_33B, "raw"),
    (outer1_platt_probabilities_33B, "platt"),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(f"{name} tahminlerinde eksik değer var.")

    if not np.all(
        (probabilities >= 0) & (probabilities <= 1)
    ):
        raise RuntimeError(
            f"{name} tahminlerinde geçersiz olasılık değeri var."
        )

raw_metrics_33B = probability_metrics_33B(
    y_outer_test_33B,
    outer1_raw_probabilities_33B,
)
platt_metrics_33B = probability_metrics_33B(
    y_outer_test_33B,
    outer1_platt_probabilities_33B,
)

raw_calibration_intercept_33B, raw_calibration_slope_33B = (
    calibration_intercept_slope_33B(
        y_outer_test_33B,
        outer1_raw_probabilities_33B,
    )
)

platt_calibration_intercept_33B, platt_calibration_slope_33B = (
    calibration_intercept_slope_33B(
        y_outer_test_33B,
        outer1_platt_probabilities_33B,
    )
)

outer1_test_results_33B = pd.DataFrame(
    [
        {
            "outer_fold": 1,
            "model": "parsimonious_clinical_logistic",
            "probability_type": "raw",
            **raw_metrics_33B,
            "calibration_intercept": raw_calibration_intercept_33B,
            "calibration_slope": raw_calibration_slope_33B,
        },
        {
            "outer_fold": 1,
            "model": "parsimonious_clinical_logistic",
            "probability_type": "platt_calibrated",
            **platt_metrics_33B,
            "calibration_intercept": platt_calibration_intercept_33B,
            "calibration_slope": platt_calibration_slope_33B,
        },
    ]
)

# ------------------------------------------------------------
# 20. Feature coefficient audit
# ------------------------------------------------------------

fitted_preprocessor_33B = final_pipeline_33B.named_steps[
    "preprocessor"
]
fitted_model_33B = final_pipeline_33B.named_steps["model"]

processed_feature_names_33B = (
    fitted_preprocessor_33B.get_feature_names_out()
)
model_coefficients_33B = fitted_model_33B.coef_.reshape(-1)

if len(processed_feature_names_33B) != len(model_coefficients_33B):
    raise RuntimeError("Feature ve katsayı sayıları uyuşmuyor.")

if len(set(processed_feature_names_33B)) != len(
    processed_feature_names_33B
):
    raise RuntimeError("İşlenmiş feature adlarında yinelenme var.")

coefficient_table_33B = pd.DataFrame(
    {
        "processed_feature": processed_feature_names_33B,
        "coefficient": model_coefficients_33B,
    }
)
coefficient_table_33B["absolute_coefficient"] = (
    coefficient_table_33B["coefficient"].abs()
)
coefficient_table_33B["is_nonzero"] = ~np.isclose(
    coefficient_table_33B["coefficient"],
    0.0,
    atol=1e-12,
)
coefficient_table_33B["absolute_rank"] = (
    coefficient_table_33B["absolute_coefficient"]
    .rank(method="first", ascending=False)
    .astype(int)
)
coefficient_table_33B = (
    coefficient_table_33B.sort_values(
        ["absolute_coefficient", "processed_feature"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

nonzero_coefficients_33B = int(
    coefficient_table_33B["is_nonzero"].sum()
)

final_model_summary_33B = pd.DataFrame(
    [
        {
            "outer_fold": 1,
            "selected_candidate": selected_candidate_33B,
            "selected_C": selected_C_33B,
            "selected_l1_ratio": selected_l1_ratio_33B,
            "training_patients": len(X_outer_training_33B),
            "training_hospitals": len(training_hospitals_33B),
            "training_events": int(y_outer_training_33B.sum()),
            "test_patients": len(X_outer_test_33B),
            "test_hospitals": len(test_hospitals_33B),
            "test_events": int(y_outer_test_33B.sum()),
            "hospital_overlap": len(hospital_overlap_33B),
            "processed_feature_columns": len(
                processed_feature_names_33B
            ),
            "nonzero_coefficients": nonzero_coefficients_33B,
            "model_intercept": float(
                fitted_model_33B.intercept_[0]
            ),
            "convergence_warnings": final_convergence_warnings_33B,
            "final_model_n_iter": final_model_n_iter_33B,
            "fit_elapsed_seconds": float(final_fit_elapsed_33B),
            "locked_platt_intercept": platt_intercept_33B,
            "locked_platt_slope": platt_slope_33B,
            "protocol_sha256": EXPECTED_PROTOCOL_SHA_33B,
            "numerical_amendment_sha256": EXPECTED_AMENDMENT_SHA_33B,
            "model_max_iter": 20000,
            "selection_sha256": selection_sha_33B,
        }
    ]
)

# ------------------------------------------------------------
# 21. Secure BigQuery outer-test checkpoint
# ------------------------------------------------------------

outer1_prediction_df_33B = pd.DataFrame(
    {
        "id_row": outer_test_meta_33B["id_row"].astype(str),
        "outer_fold": np.full(
            len(outer_test_meta_33B),
            1,
            dtype=np.int64,
        ),
        "label_stage23": y_outer_test_33B.astype(np.int64),
        "prediction_raw": outer1_raw_probabilities_33B.astype(
            np.float64
        ),
        "prediction_platt": outer1_platt_probabilities_33B.astype(
            np.float64
        ),
        "model_name": "parsimonious_clinical_logistic",
        "model_version": "clinical8_v1_nested_cv",
    }
)

if len(outer1_prediction_df_33B) != 11688:
    raise RuntimeError(
        "Dış kat 4 tahmin satır sayısı 11.688 değil."
    )

if outer1_prediction_df_33B["id_row"].duplicated().any():
    raise RuntimeError(
        "Dış kat 4 tahminlerinde yinelenen id_row var."
    )

if int(
    outer1_prediction_df_33B["label_stage23"].sum()
) != 606:
    raise RuntimeError("Dış kat 4 olay sayısı 606 değil.")

prediction_table_id_33B = (
    f"{TARGET_DATASET}."
    "model_clinical_lr_outer_predictions_outer1_v1"
)

prediction_load_config_33B = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("prediction_platt", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("model_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("model_version", "STRING", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

print("\nUploading secure outer-fold-1 prediction checkpoint:")
print(prediction_table_id_33B)

client.load_table_from_dataframe(
    outer1_prediction_df_33B,
    prediction_table_id_33B,
    job_config=prediction_load_config_33B,
    location=BQ_LOCATION,
).result()

# ------------------------------------------------------------
# 22. BigQuery outer-test checkpoint verification
# ------------------------------------------------------------

SQL_VERIFY_PREDICTIONS_33B = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL) AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL) AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0 OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0 OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw) AS minimum_raw_probability,
  MAX(prediction_raw) AS maximum_raw_probability,
  MIN(prediction_platt) AS minimum_platt_probability,
  MAX(prediction_platt) AS maximum_platt_probability
FROM `{prediction_table_id_33B}`;
"""

prediction_verification_33B = client.query(
    SQL_VERIFY_PREDICTIONS_33B,
    location=BQ_LOCATION,
).to_dataframe()

verification_row_33B = prediction_verification_33B.iloc[0]

expected_prediction_values_33B = {
    "prediction_rows": 11688,
    "distinct_rows": 11688,
    "outer_folds": 1,
    "minimum_outer_fold": 1,
    "maximum_outer_fold": 1,
    "events": 606,
    "nonevents": 11082,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in expected_prediction_values_33B.items():
    actual_value = int(verification_row_33B[field])
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: bulunan={actual_value}, beklenen={expected_value}"
        )

# ------------------------------------------------------------
# 23. Save only aggregate and feature-level outputs to Drive
# ------------------------------------------------------------

test_results_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_outer1_test_results.csv",
)
model_summary_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_final_model_outer1.csv",
)
coefficient_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_coefficients_outer1.csv",
)
evaluation_json_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_final_evaluation_outer1.json",
)
evaluation_sha_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_clinical_final_evaluation_outer1_SHA256.txt",
)

outer1_test_results_33B.to_csv(
    test_results_path_33B,
    index=False,
)
final_model_summary_33B.to_csv(
    model_summary_path_33B,
    index=False,
)
coefficient_table_33B.to_csv(
    coefficient_path_33B,
    index=False,
)

evaluation_configuration_33B = {
    "outer_fold": 1,
    "model_family": "parsimonious_clinical_elastic_net_logistic_regression",
    "selected_candidate": selected_candidate_33B,
    "selected_C": selected_C_33B,
    "selected_l1_ratio": selected_l1_ratio_33B,
    "training_patients": 46803,
    "training_hospitals": 158,
    "test_patients": 11688,
    "test_hospitals": 40,
    "hospital_overlap": 0,
    "locked_platt_intercept": platt_intercept_33B,
    "locked_platt_slope": platt_slope_33B,
    "processed_feature_columns": int(
        len(processed_feature_names_33B)
    ),
    "nonzero_coefficients": int(nonzero_coefficients_33B),
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_33B,
    "selection_sha256": selection_sha_33B,
    "secure_prediction_table": prediction_table_id_33B,
    "patient_level_prediction_written_to_drive": False,
    "analysis_role": "additional_post_hoc_clinical_baseline",
    "predictors": CLINICAL_PREDICTORS_33B,
}

with open(
    evaluation_json_path_33B,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        evaluation_configuration_33B,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(evaluation_json_path_33B, "rb") as file_handle:
    evaluation_sha_33B = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    evaluation_sha_path_33B,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(evaluation_sha_33B + "\n")

# ------------------------------------------------------------
# 24. Final outputs
# ------------------------------------------------------------

pooled_integrity_33B = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_33B),
            pooled_oof_33B["id_row"].nunique(),
            pooled_oof_33B["candidate_id"].nunique(),
            pooled_oof_33B["inner_fold"].nunique(),
            EXPECTED_SPLIT_33B["training_events"],
            (
                EXPECTED_SPLIT_33B["training_rows"]
                - EXPECTED_SPLIT_33B["training_events"]
            ),
            int(
                pooled_oof_33B.duplicated(
                    subset=["candidate_id", "id_row"]
                ).sum()
            ),
            int(pooled_oof_33B["prediction_raw"].isna().sum()),
            int(
                (~pooled_oof_33B["prediction_raw"].between(0, 1)).sum()
            ),
            pooled_load_method_33B,
        ],
    }
)

print("\n33B OUTER-FOLD-1 INNER CHECKPOINT SUMMARY")
display(checkpoint_summary_33B)

print("\n33B OUTER-FOLD-1 POOLED OOF INTEGRITY")
display(pooled_integrity_33B)

print("\n33B OUTER-FOLD-1 CANDIDATE RESULTS")
display(candidate_results_33B)

print("\n33B OUTER-FOLD-1 SELECTED MODEL")
display(selected_model_33B)

print("\n33B OUTER-FOLD-1 FINAL MODEL SUMMARY")
display(final_model_summary_33B)

print("\n33B OUTER-FOLD-1 TEST RESULTS")
display(outer1_test_results_33B)

print("\n33B OUTER-FOLD-1 BIGQUERY VERIFICATION")
display(prediction_verification_33B)

print("\n33B OUTER-FOLD-1 TOP 20 ABSOLUTE COEFFICIENTS")
display(coefficient_table_33B.head(20))

print("\nSelection SHA-256:")
print(selection_sha_33B)

print("\nEvaluation SHA-256:")
print(evaluation_sha_33B)

print("\nSaved:")
print(fit_audit_path_33B)
print(checkpoint_summary_path_33B)
print(candidate_results_path_33B)
print(selected_model_path_33B)
print(selection_json_path_33B)
print(selection_sha_path_33B)
print(test_results_path_33B)
print(model_summary_path_33B)
print(coefficient_path_33B)
print(evaluation_json_path_33B)
print(evaluation_sha_path_33B)

print(
    "\n33B PASS: Outer-fold-1 nested modelling "
    "and locked test evaluation are complete."
)
print(
    "All inner OOF and outer-test patient-level "
    "predictions were stored only in BigQuery."
)
print(
    "No patient-level prediction file was "
    "written to Google Drive."
)

_ = gc.collect()

In [ ]:
import os, json, hashlib

print("STARTING CLINICAL BASELINE FULL INNER-REFIT CORRECTION — CODE VERSION 33A2")

if "MODEL_OUTPUT_DIR" not in globals():
    raise RuntimeError("Önce 07A hücresini çalıştır.")

EXPECTED_33A_SHA = "94b0abb218dbef4e349702ea2824ca4e31dfbe36c53efa05ba1a8bd1f38f835e"
EXPECTED_33A1_SHA = "fcbcf857caa9aaad7ffd6691b4b61ac6503d09a6eee80db84bc2967ddf1f1a6a"
EXPECTED_33A2_SHA = "ea7ab2747aad0c20157da0988b2fbeb16ce007b4c92bf785d9ef9d363862ecfa"

for filename, expected_sha in [
    (
        "33A_locked_parsimonious_clinical_baseline_protocol_v1_SHA256.txt",
        EXPECTED_33A_SHA,
    ),
    (
        "33A1_parsimonious_clinical_baseline_convergence_amendment_v1_SHA256.txt",
        EXPECTED_33A1_SHA,
    ),
]:
    path = os.path.join(MODEL_OUTPUT_DIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    with open(path, "r", encoding="utf-8") as f:
        observed = f.read().strip()
    if observed != expected_sha:
        raise RuntimeError(
            f"SHA mismatch for {filename}: {observed}"
        )

correction = {'analysis_version': '33A2', 'correction_name': 'parsimonious_clinical_baseline_full_inner_refit_convergence_correction_v1', 'original_protocol_sha256': '94b0abb218dbef4e349702ea2824ca4e31dfbe36c53efa05ba1a8bd1f38f835e', 'superseded_amendment_sha256': 'fcbcf857caa9aaad7ffd6691b4b61ac6503d09a6eee80db84bc2967ddf1f1a6a', 'status': 'supersedes_33A1_for_all_clinical_baseline_modeling', 'trigger': 'After 33B-R1 completed, the aggregate inner-fit audit showed that several candidate models had ConvergenceWarning counts under the original 5000-iteration inner fits: CLIN01=5, CLIN02=4, CLIN03=2, CLIN05=1, while CLIN04=0 and CLIN06=0.', 'scientific_consequence': 'The first outer-fold-1 candidate ranking and its Platt calibrator were based partly on non-converged inner models and therefore must not be treated as the corrected clinical-baseline result.', 'outer_test_exposure_statement': 'Outer-fold-1 test metrics were already produced once in 33B-R1. They are quarantined as superseded exploratory output and must not be used to choose predictors, hyperparameters, calibration, or any other modeling decision.', 'locked_correction': {'all_six_candidates_refit_in_all_five_inner_folds': True, 'new_inner_checkpoint_namespace': 'v2', 'new_outer_prediction_namespace': 'v2', 'LogisticRegression.max_iter': 20000, 'tol': 0.0001, 'solver': 'saga', 'penalty': 'elasticnet', 'candidate_grid_unchanged': True, 'predictor_set_unchanged': True, 'selection_rule_unchanged': True, 'platt_calibration_rule_unchanged': True, 'random_state_unchanged': 20260721, 'class_weight_unchanged': None, 'zero_inner_convergence_warning_required_before_selection': True, 'zero_final_convergence_warning_required_before_outer_test': True}, 'reporting_status': 'The parsimonious clinical baseline remains an additional/post-hoc exploratory benchmark. The convergence correction will be disclosed in the audit trail; no test-driven tuning is permitted.', 'prohibited': ['using_33B_R1_test_metrics_for_selection', 'predictor_changes', 'candidate_grid_changes', 'test_based_retuning', 'class_weighting', 'SMOTE', 'oversampling', 'undersampling', 'early_stopping']}
correction_text = json.dumps(correction, indent=2, sort_keys=True)

observed_sha = hashlib.sha256(
    correction_text.encode("utf-8")
).hexdigest()

if observed_sha != EXPECTED_33A2_SHA:
    raise RuntimeError("Internal 33A2 SHA mismatch.")

correction_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A2_parsimonious_clinical_baseline_full_inner_refit_correction_v1.json",
)
sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A2_parsimonious_clinical_baseline_full_inner_refit_correction_v1_SHA256.txt",
)

with open(correction_path, "w", encoding="utf-8") as f:
    f.write(correction_text)

with open(sha_path, "w", encoding="utf-8") as f:
    f.write(observed_sha + "\n")

print("\n33A original protocol SHA: PASS")
print("33A1 amendment SHA: PASS")
print("\n33A2 correction SHA-256:")
print(observed_sha)
print("\nLocked correction:")
print("- refit ALL 6 candidates in ALL 5 inner folds")
print("- max_iter=20000 for every inner and final fit")
print("- zero inner convergence warnings required before selection")
print("- new BigQuery checkpoint namespace v2")
print("- 33B-R1 outer-test output quarantined; never used for tuning")
print("\n33A2 PASS: Full inner-refit convergence correction locked.")

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from sklearn.exceptions import ConvergenceWarning

from IPython.display import display

# ============================================================
# 33B — PARSIMONIOUS CLINICAL BASELINE OUTER FOLD 1
#
# Resume-safe:
# - Each inner fold is stored in a separate BigQuery table.
# - Completed inner folds are automatically skipped.
#
# BigQuery free-tier compatible:
# - No DELETE / INSERT / UPDATE / MERGE is used.
#
# Privacy:
# - Patient-level predictions are stored only in BigQuery.
# - No patient-level prediction file is written to Drive.
# ============================================================

print("STARTING PARSIMONIOUS CLINICAL BASELINE OUTER FOLD 1 — CODE VERSION 33B-R2")

OUTER_FOLD_33B = 1
MODEL_RANDOM_SEED_33B = 20260721

EXPECTED_PROTOCOL_SHA_33B = (
    "94b0abb218dbef4e349702ea2824ca4e"
    "31dfbe36c53efa05ba1a8bd1f38f835e"
)

EXPECTED_AMENDMENT_SHA_33B = (
    "fcbcf857caa9aaad7ffd6691b4b61ac6"
    "503d09a6eee80db84bc2967ddf1f1a6a"
)

EXPECTED_CORRECTION_SHA_33B = (
    "ea7ab2747aad0c20157da0988b2fbeb1"
    "6ce007b4c92bf785d9ef9d363862ecfa"
)

EXPECTED_SPLIT_33B = {
    "training_rows": 46803,
    "test_rows": 11688,
    "training_hospitals": 158,
    "test_hospitals": 40,
    "training_events": 2426,
    "test_events": 606,
}

EXPECTED_INNER_33B = {
    1: {"validation_rows": 10650, "validation_events": 578},
    2: {"validation_rows": 7084, "validation_events": 359},
    3: {"validation_rows": 11915, "validation_events": 637},
    4: {"validation_rows": 10019, "validation_events": 488},
    5: {"validation_rows": 7135, "validation_events": 364},
}

CLINICAL_PREDICTORS_33B = [
    "x_age_years",
    "x_sex",
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
    "x_lab_bun_last",
    "x_vital_respiratory_rate_last",
    "x_vital_noninvasive_systolic_bp_last",
]

CLINICAL_NUMERIC_33B = [
    "x_age_years",
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
    "x_lab_bun_last",
    "x_vital_respiratory_rate_last",
    "x_vital_noninvasive_systolic_bp_last",
]

CLINICAL_CATEGORICAL_33B = ["x_sex"]

# ------------------------------------------------------------
# 1. Required runtime objects
# ------------------------------------------------------------

required_objects_33B = [
    "core_df_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_33B = [
    name for name in required_objects_33B if name not in globals()
]

if missing_objects_33B:
    raise RuntimeError(
        "Eksik RAM nesneleri var: "
        + ", ".join(missing_objects_33B)
        + ". Önce 07A ve 07B hücrelerini çalıştır."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"58.491 satır bekleniyordu; {len(core_df_07B)} bulundu."
    )

missing_predictors_33B = [
    column for column in CLINICAL_PREDICTORS_33B
    if column not in core_df_07B.columns
]

if missing_predictors_33B:
    raise RuntimeError(
        "Kilitli klinik predictor(lar) core_df_07B içinde yok: "
        + ", ".join(missing_predictors_33B)
    )

# ------------------------------------------------------------
# 2. Locked protocol SHA check
# ------------------------------------------------------------

protocol_sha_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A_locked_parsimonious_clinical_baseline_protocol_v1_SHA256.txt",
)

if not os.path.exists(protocol_sha_path_33B):
    raise FileNotFoundError(
        "Model protokolü SHA dosyası bulunamadı: "
        + protocol_sha_path_33B
    )

with open(protocol_sha_path_33B, "r", encoding="utf-8") as file_handle:
    observed_protocol_sha_33B = file_handle.read().strip()

if observed_protocol_sha_33B != EXPECTED_PROTOCOL_SHA_33B:
    raise RuntimeError(
        "Kilitli model protokolü SHA değeri değişmiş: "
        + observed_protocol_sha_33B
    )

amendment_sha_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A1_parsimonious_clinical_baseline_convergence_amendment_v1_SHA256.txt",
)

if not os.path.exists(amendment_sha_path_33B):
    raise FileNotFoundError(
        "Önce 33A1 convergence amendment scriptini çalıştır: "
        + amendment_sha_path_33B
    )

with open(amendment_sha_path_33B, "r", encoding="utf-8") as file_handle:
    observed_amendment_sha_33B = file_handle.read().strip()

if observed_amendment_sha_33B != EXPECTED_AMENDMENT_SHA_33B:
    raise RuntimeError(
        "33A1 amendment SHA değeri değişmiş: "
        + observed_amendment_sha_33B
    )

print("33A protocol SHA guard: PASS")
print("33A1 convergence amendment SHA guard: PASS")

correction_sha_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A2_parsimonious_clinical_baseline_full_inner_refit_correction_v1_SHA256.txt",
)

if not os.path.exists(correction_sha_path_33B):
    raise FileNotFoundError(
        "Önce 33A2 correction scriptini çalıştır: "
        + correction_sha_path_33B
    )

with open(correction_sha_path_33B, "r", encoding="utf-8") as file_handle:
    observed_correction_sha_33B = file_handle.read().strip()

if observed_correction_sha_33B != EXPECTED_CORRECTION_SHA_33B:
    raise RuntimeError(
        "33A2 correction SHA değeri değişmiş: "
        + observed_correction_sha_33B
    )

print("33A2 full inner-refit correction SHA guard: PASS")

# ------------------------------------------------------------
# 3. Locked inner-hospital mapping
# ------------------------------------------------------------

inner_mapping_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_33B):
    raise FileNotFoundError(
        "Kilitli iç kat haritası bulunamadı: "
        + inner_mapping_path_33B
    )

inner_mapping_all_33B = pd.read_csv(
    inner_mapping_path_33B,
    dtype={"group_hospital": str},
)

inner_mapping_part_33B = (
    inner_mapping_all_33B.loc[
        inner_mapping_all_33B["outer_fold"].astype(int) == OUTER_FOLD_33B,
        ["group_hospital", "inner_fold"],
    ]
    .copy()
)

inner_mapping_part_33B["group_hospital"] = (
    inner_mapping_part_33B["group_hospital"].astype(str)
)
inner_mapping_part_33B["inner_fold"] = (
    inner_mapping_part_33B["inner_fold"].astype(int)
)

if len(inner_mapping_part_33B) != 158:
    raise RuntimeError(
        "Dış kat 1 eğitim kümesi için 158 hastane ataması bekleniyordu."
    )

if inner_mapping_part_33B["group_hospital"].duplicated().any():
    raise RuntimeError("İç kat haritasında yinelenen hastane var.")

hospital_to_inner_fold_33B = dict(
    zip(
        inner_mapping_part_33B["group_hospital"],
        inner_mapping_part_33B["inner_fold"],
    )
)

# ------------------------------------------------------------
# 4. Prepare model matrices
# ------------------------------------------------------------

X_all_33B = core_df_07B[CLINICAL_PREDICTORS_33B].copy()

for column in CLINICAL_NUMERIC_33B:
    X_all_33B[column] = pd.to_numeric(
        X_all_33B[column], errors="coerce"
    ).astype("float64")

for column in CLINICAL_CATEGORICAL_33B:
    category_series = X_all_33B[column].astype("object")
    X_all_33B[column] = category_series.where(
        pd.notna(category_series), np.nan
    )

outer_fold_vector_33B = (
    core_df_07B["outer_fold"].astype(int).to_numpy()
)

outer_training_mask_33B = outer_fold_vector_33B != OUTER_FOLD_33B
outer_test_mask_33B = outer_fold_vector_33B == OUTER_FOLD_33B

X_outer_training_33B = (
    X_all_33B.loc[outer_training_mask_33B].reset_index(drop=True)
)
X_outer_test_33B = (
    X_all_33B.loc[outer_test_mask_33B].reset_index(drop=True)
)

outer_training_meta_33B = (
    core_df_07B.loc[
        outer_training_mask_33B,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_33B = (
    core_df_07B.loc[
        outer_test_mask_33B,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [outer_training_meta_33B, outer_test_meta_33B]:
    dataframe["id_row"] = dataframe["id_row"].astype(str)
    dataframe["group_hospital"] = dataframe["group_hospital"].astype(str)
    dataframe["label_stage23"] = dataframe["label_stage23"].astype(int)

y_outer_training_33B = (
    outer_training_meta_33B["label_stage23"].to_numpy(dtype=np.int8)
)
y_outer_test_33B = (
    outer_test_meta_33B["label_stage23"].to_numpy(dtype=np.int8)
)

groups_outer_training_33B = (
    outer_training_meta_33B["group_hospital"].to_numpy(dtype=str)
)

training_hospitals_33B = set(
    outer_training_meta_33B["group_hospital"]
)
test_hospitals_33B = set(
    outer_test_meta_33B["group_hospital"]
)
hospital_overlap_33B = training_hospitals_33B & test_hospitals_33B

if hospital_overlap_33B:
    raise RuntimeError("Dış eğitim ve test hastaneleri çakışıyor.")

actual_split_33B = {
    "training_rows": len(X_outer_training_33B),
    "test_rows": len(X_outer_test_33B),
    "training_hospitals": len(training_hospitals_33B),
    "test_hospitals": len(test_hospitals_33B),
    "training_events": int(y_outer_training_33B.sum()),
    "test_events": int(y_outer_test_33B.sum()),
}

for metric, expected_value in EXPECTED_SPLIT_33B.items():
    actual_value = actual_split_33B[metric]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: bulunan={actual_value}, beklenen={expected_value}"
        )

inner_fold_vector_33B = np.array(
    [
        hospital_to_inner_fold_33B.get(hospital, -1)
        for hospital in groups_outer_training_33B
    ],
    dtype=int,
)

if (inner_fold_vector_33B == -1).any():
    raise RuntimeError(
        "Bazı dış eğitim hastanelerine iç kat atanmadı."
    )

if set(np.unique(inner_fold_vector_33B)) != {1, 2, 3, 4, 5}:
    raise RuntimeError("İç kat değerleri 1–5 değil.")

for inner_fold, expected in EXPECTED_INNER_33B.items():
    validation_mask = inner_fold_vector_33B == inner_fold
    observed_rows = int(validation_mask.sum())
    observed_events = int(y_outer_training_33B[validation_mask].sum())

    if observed_rows != expected["validation_rows"]:
        raise RuntimeError(
            f"Inner {inner_fold} validation_rows: "
            f"bulunan={observed_rows}, "
            f"beklenen={expected['validation_rows']}"
        )

    if observed_events != expected["validation_events"]:
        raise RuntimeError(
            f"Inner {inner_fold} validation_events: "
            f"bulunan={observed_events}, "
            f"beklenen={expected['validation_events']}"
        )

# ------------------------------------------------------------
# 5. Locked candidate grid
# ------------------------------------------------------------

candidate_grid_33B = [
    {"candidate_id": "CLIN01", "C": 0.03, "l1_ratio": 0.00},
    {"candidate_id": "CLIN02", "C": 0.10, "l1_ratio": 0.00},
    {"candidate_id": "CLIN03", "C": 0.30, "l1_ratio": 0.00},
    {"candidate_id": "CLIN04", "C": 0.10, "l1_ratio": 0.25},
    {"candidate_id": "CLIN05", "C": 0.30, "l1_ratio": 0.25},
    {"candidate_id": "CLIN06", "C": 0.30, "l1_ratio": 0.50},
]

# ------------------------------------------------------------
# 6. Preprocessing and model factories
# ------------------------------------------------------------

def make_preprocessor_33B():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
            (
                "scaler",
                StandardScaler(with_mean=False),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                CLINICAL_NUMERIC_33B,
            ),
            (
                "categorical",
                categorical_pipeline,
                CLINICAL_CATEGORICAL_33B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_model_33B(C_value, l1_ratio_value):
    return LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=float(C_value),
        l1_ratio=float(l1_ratio_value),
        class_weight=None,
        max_iter=20000,
        tol=1e-4,
        random_state=MODEL_RANDOM_SEED_33B,
    )


def checkpoint_table_id_33B(inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_clinical_lr_inner_oof_outer1_inner{inner_fold}_v2"
    )

# ------------------------------------------------------------
# 7. BigQuery checkpoint verification
# ------------------------------------------------------------

def verify_checkpoint_33B(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):
    table_id = checkpoint_table_id_33B(inner_fold)

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS distinct_id_count,
      COUNT(DISTINCT candidate_id) AS candidate_count,
      COUNT(DISTINCT outer_fold) AS outer_fold_count,
      COUNT(DISTINCT inner_fold) AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(candidate_id, '|', id_row)
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(
        prediction_raw < 0 OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(sql, location=BQ_LOCATION).to_dataframe()
    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows * len(candidate_grid_33B)
    )
    expected_positive_rows = (
        expected_validation_events * len(candidate_grid_33B)
    )
    expected_negative_rows = (
        (expected_validation_rows - expected_validation_events)
        * len(candidate_grid_33B)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": expected_validation_rows,
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": expected_total_rows,
        "positive_prediction_rows": expected_positive_rows,
        "negative_prediction_rows": expected_negative_rows,
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": 1,
        "maximum_outer_fold": 1,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failure_items = []

    for field, expected_value in expected_values.items():
        actual_value = int(row[field])
        if actual_value != expected_value:
            complete = False
            failure_items.append(
                f"{field}={actual_value}, expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failure_items),
        "check": check,
        "row": row,
    }

# ------------------------------------------------------------
# 8. BigQuery load schema
# ------------------------------------------------------------

checkpoint_load_config_33B = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("inner_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("candidate_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

# ------------------------------------------------------------
# 9. Aggregate fit-audit file
# ------------------------------------------------------------

fit_audit_columns_33B = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "C",
    "l1_ratio",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "convergence_warnings",
    "elapsed_seconds",
]

fit_audit_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_R2_clinical_inner_fit_audit_outer1.csv",
)

if os.path.exists(fit_audit_path_33B):
    fit_audit_33B = pd.read_csv(fit_audit_path_33B)
else:
    fit_audit_33B = pd.DataFrame(columns=fit_audit_columns_33B)

for column in fit_audit_columns_33B:
    if column not in fit_audit_33B.columns:
        fit_audit_33B[column] = np.nan

fit_audit_33B = fit_audit_33B[fit_audit_columns_33B].copy()

# ------------------------------------------------------------
# 10. Train six candidates in five locked inner folds
# ------------------------------------------------------------

for inner_fold in range(1, 6):
    inner_training_mask = inner_fold_vector_33B != inner_fold
    inner_validation_mask = inner_fold_vector_33B == inner_fold

    training_rows = int(inner_training_mask.sum())
    validation_rows = int(inner_validation_mask.sum())
    training_events = int(
        y_outer_training_33B[inner_training_mask].sum()
    )
    validation_events = int(
        y_outer_training_33B[inner_validation_mask].sum()
    )

    expected_inner = EXPECTED_INNER_33B[inner_fold]
    expected_training_rows = (
        EXPECTED_SPLIT_33B["training_rows"]
        - expected_inner["validation_rows"]
    )
    expected_training_events = (
        EXPECTED_SPLIT_33B["training_events"]
        - expected_inner["validation_events"]
    )

    if training_rows != expected_training_rows:
        raise RuntimeError(
            f"Inner {inner_fold} training_rows: "
            f"bulunan={training_rows}, "
            f"beklenen={expected_training_rows}"
        )

    if training_events != expected_training_events:
        raise RuntimeError(
            f"Inner {inner_fold} training_events: "
            f"bulunan={training_events}, "
            f"beklenen={expected_training_events}"
        )

    existing_check = verify_checkpoint_33B(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if existing_check["complete"]:
        print(
            f"Outer 1 / inner {inner_fold}: "
            "permanent checkpoint already complete; "
            "skipping model fitting."
        )
        continue

    training_hospital_set = set(
        groups_outer_training_33B[inner_training_mask]
    )
    validation_hospital_set = set(
        groups_outer_training_33B[inner_validation_mask]
    )

    if training_hospital_set & validation_hospital_set:
        raise RuntimeError(
            f"Inner fold {inner_fold}: hastane çakışması bulundu."
        )

    print(f"\nOuter 1 / inner {inner_fold}")
    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_33B()
    preprocessing_started = time.time()

    X_inner_training_processed = preprocessor.fit_transform(
        X_outer_training_33B.loc[inner_training_mask]
    )
    X_inner_validation_processed = preprocessor.transform(
        X_outer_training_33B.loc[inner_validation_mask]
    )

    preprocessing_elapsed = time.time() - preprocessing_started

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "İşlenmiş eğitim ve doğrulama sütun sayıları farklı."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_elapsed, 2),
    )

    y_inner_training = y_outer_training_33B[inner_training_mask]
    y_inner_validation = y_outer_training_33B[inner_validation_mask]

    validation_ids = (
        outer_training_meta_33B.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_33B:
        candidate_id = candidate["candidate_id"]

        print(
            "  Fitting",
            candidate_id,
            "| C =",
            candidate["C"],
            "| l1_ratio =",
            candidate["l1_ratio"],
        )

        model = make_model_33B(
            candidate["C"],
            candidate["l1_ratio"],
        )

        fitting_started = time.time()

        with warnings.catch_warnings(record=True) as warning_records:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(
                X_inner_training_processed,
                y_inner_training,
            )

        fitting_elapsed = time.time() - fitting_started

        convergence_warning_count = sum(
            issubclass(warning.category, ConvergenceWarning)
            for warning in warning_records
        )

        validation_probabilities = model.predict_proba(
            X_inner_validation_processed
        )[:, 1]

        if np.isnan(validation_probabilities).any():
            raise RuntimeError(
                f"{candidate_id}, inner {inner_fold}: eksik tahmin."
            )

        if not np.all(
            (validation_probabilities >= 0)
            & (validation_probabilities <= 1)
        ):
            raise RuntimeError(
                f"{candidate_id}: geçersiz olasılık."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        1,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": y_inner_validation.astype(np.int64),
                    "prediction_raw": validation_probabilities.astype(
                        np.float64
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": 1,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "C": candidate["C"],
                "l1_ratio": candidate["l1_ratio"],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": validation_events,
                "processed_columns": int(
                    X_inner_training_processed.shape[1]
                ),
                "convergence_warnings": int(
                    convergence_warning_count
                ),
                "elapsed_seconds": float(fitting_elapsed),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows * len(candidate_grid_33B)
    )

    if len(checkpoint_df) != expected_checkpoint_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: checkpoint satır sayısı hatalı."
        )

    if checkpoint_df.duplicated(
        subset=["id_row", "candidate_id"]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "yinelenen aday–hasta tahmini var."
        )

    target_checkpoint_table = checkpoint_table_id_33B(inner_fold)

    print(
        "Uploading permanent checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_33B,
        location=BQ_LOCATION,
    ).result()

    new_audit_df = pd.DataFrame(current_audit_rows)

    if len(fit_audit_33B) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_33B["outer_fold"],
                    errors="coerce",
                )
                == 1
            )
            & (
                pd.to_numeric(
                    fit_audit_33B["inner_fold"],
                    errors="coerce",
                )
                == inner_fold
            )
        )
        fit_audit_33B = fit_audit_33B.loc[keep_mask].copy()

    if fit_audit_33B.empty:
        fit_audit_33B = new_audit_df.copy()
    else:
        fit_audit_33B = pd.concat(
            [fit_audit_33B, new_audit_df],
            ignore_index=True,
        )

    fit_audit_33B = (
        fit_audit_33B[fit_audit_columns_33B]
        .sort_values(
            ["outer_fold", "inner_fold", "candidate_id"]
        )
        .reset_index(drop=True)
    )

    fit_audit_33B.to_csv(
        fit_audit_path_33B,
        index=False,
    )

    completed_check = verify_checkpoint_33B(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint doğrulanamadı: "
            + completed_check["reason"]
        )

    print(
        f"Outer 1 / inner {inner_fold}: "
        "permanent checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 11. Final checkpoint summary
# ------------------------------------------------------------

checkpoint_summary_rows_33B = []

for inner_fold in range(1, 6):
    validation_mask = inner_fold_vector_33B == inner_fold
    validation_rows = int(validation_mask.sum())
    validation_events = int(
        y_outer_training_33B[validation_mask].sum()
    )

    final_check = verify_checkpoint_33B(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "son checkpoint denetimi başarısız. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_33B.append(
        {
            "outer_fold": 1,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(row["row_count"]),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(row["candidate_count"]),
            "positive_prediction_rows": int(
                row["positive_prediction_rows"]
            ),
            "negative_prediction_rows": int(
                row["negative_prediction_rows"]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check["table_id"],
        }
    )

checkpoint_summary_33B = (
    pd.DataFrame(checkpoint_summary_rows_33B)
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_33B["distinct_validation_patients"].sum()
) != EXPECTED_SPLIT_33B["training_rows"]:
    raise RuntimeError(
        "Toplam doğrulama hasta sayısı 46.803 değil."
    )

expected_total_oof_rows_33B = (
    EXPECTED_SPLIT_33B["training_rows"]
    * len(candidate_grid_33B)
)

if int(checkpoint_summary_33B["checkpoint_rows"].sum()) != (
    expected_total_oof_rows_33B
):
    raise RuntimeError("Toplam OOF tahmin satırı hatalı.")

checkpoint_summary_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_R2_clinical_outer1_inner_checkpoint_summary.csv",
)
checkpoint_summary_33B.to_csv(
    checkpoint_summary_path_33B,
    index=False,
)

# ------------------------------------------------------------
# 12. Pool all inner OOF predictions
# ------------------------------------------------------------

checkpoint_tables_33B = [
    checkpoint_table_id_33B(inner_fold)
    for inner_fold in range(1, 6)
]

union_parts_33B = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_33B
]

SQL_LOAD_POOLED_OOF_33B = "\nUNION ALL\n".join(union_parts_33B)

print("\nLoading pooled outer-fold-1 inner OOF predictions...")

query_job_33B = client.query(
    SQL_LOAD_POOLED_OOF_33B,
    location=BQ_LOCATION,
)

try:
    pooled_oof_33B = query_job_33B.to_dataframe(
        create_bqstorage_client=True
    )
    pooled_load_method_33B = "BigQuery Storage API"
except Exception as fast_path_error_33B:
    print(
        "Storage API unavailable; using standard BigQuery download."
    )
    print("Message:", type(fast_path_error_33B).__name__)
    pooled_oof_33B = query_job_33B.to_dataframe(
        create_bqstorage_client=False
    )
    pooled_load_method_33B = "Standard BigQuery API"

pooled_oof_33B["id_row"] = pooled_oof_33B["id_row"].astype(str)
pooled_oof_33B["candidate_id"] = pooled_oof_33B["candidate_id"].astype(str)

for column in ["outer_fold", "inner_fold", "label_stage23"]:
    pooled_oof_33B[column] = pd.to_numeric(
        pooled_oof_33B[column], errors="raise"
    ).astype(int)

pooled_oof_33B["prediction_raw"] = pd.to_numeric(
    pooled_oof_33B["prediction_raw"], errors="raise"
).astype(float)

# ------------------------------------------------------------
# 13. Pooled OOF integrity checks
# ------------------------------------------------------------

if len(pooled_oof_33B) != expected_total_oof_rows_33B:
    raise RuntimeError("Pooled OOF satır sayısı hatalı.")

if set(pooled_oof_33B["outer_fold"].unique()) != {1}:
    raise RuntimeError("Pooled OOF içinde dış kat 4 dışında kayıt var.")

if set(pooled_oof_33B["inner_fold"].unique()) != {1, 2, 3, 4, 5}:
    raise RuntimeError("Pooled OOF iç katları 1–5 değil.")

if pooled_oof_33B.duplicated(
    subset=["candidate_id", "id_row"]
).any():
    raise RuntimeError(
        "Pooled OOF içinde yinelenen aday–hasta tahmini bulundu."
    )

if pooled_oof_33B["prediction_raw"].isna().any():
    raise RuntimeError("Pooled OOF içinde eksik tahmin var.")

if not pooled_oof_33B["prediction_raw"].between(0, 1).all():
    raise RuntimeError(
        "Pooled OOF içinde geçersiz olasılık değeri var."
    )

expected_candidate_ids_33B = {
    "CLIN01",
    "CLIN02",
    "CLIN03",
    "CLIN04",
    "CLIN05",
    "CLIN06",
}

if set(pooled_oof_33B["candidate_id"].unique()) != (
    expected_candidate_ids_33B
):
    raise RuntimeError("Altı kilitli aday bulunmuyor.")

candidate_patient_counts_33B = (
    pooled_oof_33B.groupby("candidate_id")["id_row"].nunique()
)

if not (
    candidate_patient_counts_33B
    == EXPECTED_SPLIT_33B["training_rows"]
).all():
    raise RuntimeError(
        "Her aday için 46.803 farklı OOF hastası yok."
    )

candidate_event_counts_33B = (
    pooled_oof_33B.groupby("candidate_id")["label_stage23"].sum()
)

if not (
    candidate_event_counts_33B
    == EXPECTED_SPLIT_33B["training_events"]
).all():
    raise RuntimeError("Her aday için 2.426 olay yok.")

patient_label_consistency_33B = (
    pooled_oof_33B.groupby("id_row")["label_stage23"].nunique()
)
if (patient_label_consistency_33B > 1).any():
    raise RuntimeError(
        "Aynı hastanın adaylar arasında outcome etiketi farklı."
    )

patient_inner_fold_consistency_33B = (
    pooled_oof_33B.groupby("id_row")["inner_fold"].nunique()
)
if (patient_inner_fold_consistency_33B > 1).any():
    raise RuntimeError(
        "Aynı hasta birden fazla iç doğrulama katında bulundu."
    )

# ------------------------------------------------------------
# 14. Metric helpers
# ------------------------------------------------------------

def probability_metrics_33B(y_true, probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(roc_auc_score(y_true, probabilities)),
        "auprc": float(average_precision_score(y_true, probabilities)),
        "brier": float(brier_score_loss(y_true, probabilities)),
        "log_loss": float(
            log_loss(y_true, probabilities, labels=[0, 1])
        ),
        "mean_predicted_risk": float(probabilities.mean()),
        "observed_event_rate": float(np.mean(y_true)),
    }


def probability_logit_33B(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        probabilities / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_33B(y_true, probabilities):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        probability_logit_33B(probabilities),
        y_true,
    )
    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 15. Candidate pooled-OOF performance
# ------------------------------------------------------------

candidate_result_rows_33B = []

for candidate in candidate_grid_33B:
    candidate_id = candidate["candidate_id"]

    candidate_oof = (
        pooled_oof_33B.loc[
            pooled_oof_33B["candidate_id"] == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_33B(
        candidate_oof["label_stage23"].to_numpy(dtype=int),
        candidate_oof["prediction_raw"].to_numpy(dtype=float),
    )

    fit_part = fit_audit_33B.loc[
        (
            pd.to_numeric(
                fit_audit_33B["outer_fold"],
                errors="coerce",
            )
            == 1
        )
        & (
            fit_audit_33B["candidate_id"].astype(str)
            == candidate_id
        )
    ]

    convergence_warnings = (
        int(
            pd.to_numeric(
                fit_part["convergence_warnings"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    elapsed_seconds = (
        float(
            pd.to_numeric(
                fit_part["elapsed_seconds"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_33B.append(
        {
            "candidate_id": candidate_id,
            "C": candidate["C"],
            "l1_ratio": candidate["l1_ratio"],
            **metrics,
            "convergence_warnings": convergence_warnings,
            "elapsed_seconds": elapsed_seconds,
        }
    )

candidate_results_33B = pd.DataFrame(candidate_result_rows_33B)

total_inner_convergence_warnings_33B = int(
    pd.to_numeric(
        candidate_results_33B["convergence_warnings"],
        errors="coerce",
    ).fillna(0).sum()
)

if total_inner_convergence_warnings_33B != 0:
    raise RuntimeError(
        "33B-R2 stopped BEFORE candidate selection and BEFORE outer-test "
        "evaluation because at least one inner candidate fit still emitted "
        f"a ConvergenceWarning under max_iter=20000. "
        f"Total warnings={total_inner_convergence_warnings_33B}. "
        "No test result may be accessed."
    )

print(
    "All 30 corrected inner candidate fits converged with zero warnings: PASS"
)

candidate_results_33B = (
    candidate_results_33B.sort_values(
        ["auprc", "auroc", "brier", "candidate_id"],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)

candidate_results_33B["selection_rank"] = np.arange(
    1,
    len(candidate_results_33B) + 1,
)

best_row_33B = candidate_results_33B.iloc[0]
selected_candidate_33B = str(best_row_33B["candidate_id"])
selected_C_33B = float(best_row_33B["C"])
selected_l1_ratio_33B = float(best_row_33B["l1_ratio"])

# ------------------------------------------------------------
# 16. Platt calibration on selected pooled inner OOF
# ------------------------------------------------------------

selected_oof_33B = (
    pooled_oof_33B.loc[
        pooled_oof_33B["candidate_id"] == selected_candidate_33B
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_33B = selected_oof_33B[
    "label_stage23"
].to_numpy(dtype=int)
selected_oof_probability_33B = selected_oof_33B[
    "prediction_raw"
].to_numpy(dtype=float)

platt_calibrator_33B = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_33B.fit(
    probability_logit_33B(selected_oof_probability_33B),
    selected_oof_y_33B,
)

platt_intercept_33B = float(platt_calibrator_33B.intercept_[0])
platt_slope_33B = float(platt_calibrator_33B.coef_[0][0])

if (
    not np.isfinite(platt_intercept_33B)
    or not np.isfinite(platt_slope_33B)
    or platt_slope_33B <= 0
):
    raise RuntimeError("Platt kalibrasyon katsayıları geçersiz.")

selected_model_33B = pd.DataFrame(
    [
        {
            "outer_fold": 1,
            "selected_candidate": selected_candidate_33B,
            "selected_C": selected_C_33B,
            "selected_l1_ratio": selected_l1_ratio_33B,
            "selection_metric_primary": "pooled_inner_oof_auprc",
            "inner_oof_auprc": float(best_row_33B["auprc"]),
            "inner_oof_auroc": float(best_row_33B["auroc"]),
            "inner_oof_brier": float(best_row_33B["brier"]),
            "inner_oof_log_loss": float(best_row_33B["log_loss"]),
            "inner_oof_mean_predicted_risk": float(
                best_row_33B["mean_predicted_risk"]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_33B["observed_event_rate"]
            ),
            "platt_intercept": platt_intercept_33B,
            "platt_slope": platt_slope_33B,
            "protocol_sha256": EXPECTED_PROTOCOL_SHA_33B,
            "numerical_amendment_sha256": EXPECTED_AMENDMENT_SHA_33B,
            "full_inner_refit_correction_sha256": EXPECTED_CORRECTION_SHA_33B,
            "corrected_checkpoint_namespace": "v2",
            "analysis_status": "post_hoc_exploratory_corrected",
            "full_inner_refit_correction_sha256": EXPECTED_CORRECTION_SHA_33B,
            "corrected_checkpoint_namespace": "v2",
            "analysis_status": "post_hoc_exploratory_corrected",
            "model_max_iter": 20000,
        }
    ]
)

# ------------------------------------------------------------
# 17. Lock selection/calibration aggregate files
# ------------------------------------------------------------

candidate_results_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_R2_clinical_candidate_results_outer1.csv",
)
selected_model_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_R2_clinical_selected_model_outer1.csv",
)
selection_json_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_R2_clinical_selection_calibration_outer1.json",
)
selection_sha_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_R2_clinical_selection_calibration_outer1_SHA256.txt",
)

candidate_results_33B.to_csv(
    candidate_results_path_33B,
    index=False,
)
selected_model_33B.to_csv(
    selected_model_path_33B,
    index=False,
)

selection_configuration_33B = {
    "outer_fold": 1,
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_33B,
    "selection_metric_primary": "pooled inner out-of-fold AUPRC",
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
        "candidate_id ascending immutable tie-break",
    ],
    "selected_candidate": selected_candidate_33B,
    "selected_C": selected_C_33B,
    "selected_l1_ratio": selected_l1_ratio_33B,
    "inner_oof_auprc": float(best_row_33B["auprc"]),
    "inner_oof_auroc": float(best_row_33B["auroc"]),
    "inner_oof_brier": float(best_row_33B["brier"]),
    "platt_intercept": platt_intercept_33B,
    "platt_slope": platt_slope_33B,
    "inner_checkpoint_tables": checkpoint_tables_33B,
    "patient_level_oof_written_to_drive": False,
    "analysis_role": "additional_post_hoc_clinical_baseline",
    "predictors": CLINICAL_PREDICTORS_33B,
}

with open(
    selection_json_path_33B,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        selection_configuration_33B,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(selection_json_path_33B, "rb") as file_handle:
    selection_sha_33B = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    selection_sha_path_33B,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(selection_sha_33B + "\n")

# ------------------------------------------------------------
# 18. Fit final selected outer-fold-1 model
# ------------------------------------------------------------

final_pipeline_33B = Pipeline(
    steps=[
        ("preprocessor", make_preprocessor_33B()),
        (
            "model",
            make_model_33B(
                selected_C_33B,
                selected_l1_ratio_33B,
            ),
        ),
    ]
)

print(
    "\nFitting selected outer-fold-1 model "
    "on all 46,803 training patients..."
)

final_fit_started_33B = time.time()

with warnings.catch_warnings(record=True) as final_warning_records_33B:
    warnings.simplefilter("always", ConvergenceWarning)
    final_pipeline_33B.fit(
        X_outer_training_33B,
        y_outer_training_33B,
    )

final_fit_elapsed_33B = time.time() - final_fit_started_33B

final_model_n_iter_33B = int(
    np.max(
        np.asarray(
            final_pipeline_33B.named_steps["model"].n_iter_
        )
    )
)

final_convergence_warnings_33B = sum(
    issubclass(warning.category, ConvergenceWarning)
    for warning in final_warning_records_33B
)

if final_convergence_warnings_33B != 0:
    raise RuntimeError(
        "Nihai outer-fold-1 klinik baseline modeli max_iter=20000 "
        f"altında da yakınsamadı. n_iter_={final_model_n_iter_33B}"
    )

# ------------------------------------------------------------
# 19. Outer-fold-1 test predictions and metrics
# ------------------------------------------------------------

outer1_raw_probabilities_33B = final_pipeline_33B.predict_proba(
    X_outer_test_33B
)[:, 1]

raw_clipped_33B = np.clip(
    outer1_raw_probabilities_33B,
    1e-6,
    1 - 1e-6,
)
raw_logit_33B = np.log(
    raw_clipped_33B / (1 - raw_clipped_33B)
)

outer1_platt_probabilities_33B = expit(
    platt_intercept_33B + platt_slope_33B * raw_logit_33B
)

for probabilities, name in [
    (outer1_raw_probabilities_33B, "raw"),
    (outer1_platt_probabilities_33B, "platt"),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(f"{name} tahminlerinde eksik değer var.")

    if not np.all(
        (probabilities >= 0) & (probabilities <= 1)
    ):
        raise RuntimeError(
            f"{name} tahminlerinde geçersiz olasılık değeri var."
        )

raw_metrics_33B = probability_metrics_33B(
    y_outer_test_33B,
    outer1_raw_probabilities_33B,
)
platt_metrics_33B = probability_metrics_33B(
    y_outer_test_33B,
    outer1_platt_probabilities_33B,
)

raw_calibration_intercept_33B, raw_calibration_slope_33B = (
    calibration_intercept_slope_33B(
        y_outer_test_33B,
        outer1_raw_probabilities_33B,
    )
)

platt_calibration_intercept_33B, platt_calibration_slope_33B = (
    calibration_intercept_slope_33B(
        y_outer_test_33B,
        outer1_platt_probabilities_33B,
    )
)

outer1_test_results_33B = pd.DataFrame(
    [
        {
            "outer_fold": 1,
            "model": "parsimonious_clinical_logistic",
            "probability_type": "raw",
            **raw_metrics_33B,
            "calibration_intercept": raw_calibration_intercept_33B,
            "calibration_slope": raw_calibration_slope_33B,
        },
        {
            "outer_fold": 1,
            "model": "parsimonious_clinical_logistic",
            "probability_type": "platt_calibrated",
            **platt_metrics_33B,
            "calibration_intercept": platt_calibration_intercept_33B,
            "calibration_slope": platt_calibration_slope_33B,
        },
    ]
)

# ------------------------------------------------------------
# 20. Feature coefficient audit
# ------------------------------------------------------------

fitted_preprocessor_33B = final_pipeline_33B.named_steps[
    "preprocessor"
]
fitted_model_33B = final_pipeline_33B.named_steps["model"]

processed_feature_names_33B = (
    fitted_preprocessor_33B.get_feature_names_out()
)
model_coefficients_33B = fitted_model_33B.coef_.reshape(-1)

if len(processed_feature_names_33B) != len(model_coefficients_33B):
    raise RuntimeError("Feature ve katsayı sayıları uyuşmuyor.")

if len(set(processed_feature_names_33B)) != len(
    processed_feature_names_33B
):
    raise RuntimeError("İşlenmiş feature adlarında yinelenme var.")

coefficient_table_33B = pd.DataFrame(
    {
        "processed_feature": processed_feature_names_33B,
        "coefficient": model_coefficients_33B,
    }
)
coefficient_table_33B["absolute_coefficient"] = (
    coefficient_table_33B["coefficient"].abs()
)
coefficient_table_33B["is_nonzero"] = ~np.isclose(
    coefficient_table_33B["coefficient"],
    0.0,
    atol=1e-12,
)
coefficient_table_33B["absolute_rank"] = (
    coefficient_table_33B["absolute_coefficient"]
    .rank(method="first", ascending=False)
    .astype(int)
)
coefficient_table_33B = (
    coefficient_table_33B.sort_values(
        ["absolute_coefficient", "processed_feature"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

nonzero_coefficients_33B = int(
    coefficient_table_33B["is_nonzero"].sum()
)

final_model_summary_33B = pd.DataFrame(
    [
        {
            "outer_fold": 1,
            "selected_candidate": selected_candidate_33B,
            "selected_C": selected_C_33B,
            "selected_l1_ratio": selected_l1_ratio_33B,
            "training_patients": len(X_outer_training_33B),
            "training_hospitals": len(training_hospitals_33B),
            "training_events": int(y_outer_training_33B.sum()),
            "test_patients": len(X_outer_test_33B),
            "test_hospitals": len(test_hospitals_33B),
            "test_events": int(y_outer_test_33B.sum()),
            "hospital_overlap": len(hospital_overlap_33B),
            "processed_feature_columns": len(
                processed_feature_names_33B
            ),
            "nonzero_coefficients": nonzero_coefficients_33B,
            "model_intercept": float(
                fitted_model_33B.intercept_[0]
            ),
            "convergence_warnings": final_convergence_warnings_33B,
            "final_model_n_iter": final_model_n_iter_33B,
            "fit_elapsed_seconds": float(final_fit_elapsed_33B),
            "locked_platt_intercept": platt_intercept_33B,
            "locked_platt_slope": platt_slope_33B,
            "protocol_sha256": EXPECTED_PROTOCOL_SHA_33B,
            "numerical_amendment_sha256": EXPECTED_AMENDMENT_SHA_33B,
            "model_max_iter": 20000,
            "selection_sha256": selection_sha_33B,
        }
    ]
)

# ------------------------------------------------------------
# 21. Secure BigQuery outer-test checkpoint
# ------------------------------------------------------------

outer1_prediction_df_33B = pd.DataFrame(
    {
        "id_row": outer_test_meta_33B["id_row"].astype(str),
        "outer_fold": np.full(
            len(outer_test_meta_33B),
            1,
            dtype=np.int64,
        ),
        "label_stage23": y_outer_test_33B.astype(np.int64),
        "prediction_raw": outer1_raw_probabilities_33B.astype(
            np.float64
        ),
        "prediction_platt": outer1_platt_probabilities_33B.astype(
            np.float64
        ),
        "model_name": "parsimonious_clinical_logistic",
        "model_version": "clinical8_v1_nested_cv",
    }
)

if len(outer1_prediction_df_33B) != 11688:
    raise RuntimeError(
        "Dış kat 4 tahmin satır sayısı 11.688 değil."
    )

if outer1_prediction_df_33B["id_row"].duplicated().any():
    raise RuntimeError(
        "Dış kat 4 tahminlerinde yinelenen id_row var."
    )

if int(
    outer1_prediction_df_33B["label_stage23"].sum()
) != 606:
    raise RuntimeError("Dış kat 4 olay sayısı 606 değil.")

prediction_table_id_33B = (
    f"{TARGET_DATASET}."
    "model_clinical_lr_outer_predictions_outer1_v2"
)

prediction_load_config_33B = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("prediction_platt", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("model_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("model_version", "STRING", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

print("\nUploading secure outer-fold-1 prediction checkpoint:")
print(prediction_table_id_33B)

client.load_table_from_dataframe(
    outer1_prediction_df_33B,
    prediction_table_id_33B,
    job_config=prediction_load_config_33B,
    location=BQ_LOCATION,
).result()

# ------------------------------------------------------------
# 22. BigQuery outer-test checkpoint verification
# ------------------------------------------------------------

SQL_VERIFY_PREDICTIONS_33B = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL) AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL) AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0 OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0 OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw) AS minimum_raw_probability,
  MAX(prediction_raw) AS maximum_raw_probability,
  MIN(prediction_platt) AS minimum_platt_probability,
  MAX(prediction_platt) AS maximum_platt_probability
FROM `{prediction_table_id_33B}`;
"""

prediction_verification_33B = client.query(
    SQL_VERIFY_PREDICTIONS_33B,
    location=BQ_LOCATION,
).to_dataframe()

verification_row_33B = prediction_verification_33B.iloc[0]

expected_prediction_values_33B = {
    "prediction_rows": 11688,
    "distinct_rows": 11688,
    "outer_folds": 1,
    "minimum_outer_fold": 1,
    "maximum_outer_fold": 1,
    "events": 606,
    "nonevents": 11082,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in expected_prediction_values_33B.items():
    actual_value = int(verification_row_33B[field])
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: bulunan={actual_value}, beklenen={expected_value}"
        )

# ------------------------------------------------------------
# 23. Save only aggregate and feature-level outputs to Drive
# ------------------------------------------------------------

test_results_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_R2_clinical_outer1_test_results.csv",
)
model_summary_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_R2_clinical_final_model_outer1.csv",
)
coefficient_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_R2_clinical_coefficients_outer1.csv",
)
evaluation_json_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_R2_clinical_final_evaluation_outer1.json",
)
evaluation_sha_path_33B = os.path.join(
    MODEL_OUTPUT_DIR,
    "33B_R2_clinical_final_evaluation_outer1_SHA256.txt",
)

outer1_test_results_33B.to_csv(
    test_results_path_33B,
    index=False,
)
final_model_summary_33B.to_csv(
    model_summary_path_33B,
    index=False,
)
coefficient_table_33B.to_csv(
    coefficient_path_33B,
    index=False,
)

evaluation_configuration_33B = {
    "outer_fold": 1,
    "model_family": "parsimonious_clinical_elastic_net_logistic_regression",
    "selected_candidate": selected_candidate_33B,
    "selected_C": selected_C_33B,
    "selected_l1_ratio": selected_l1_ratio_33B,
    "training_patients": 46803,
    "training_hospitals": 158,
    "test_patients": 11688,
    "test_hospitals": 40,
    "hospital_overlap": 0,
    "locked_platt_intercept": platt_intercept_33B,
    "locked_platt_slope": platt_slope_33B,
    "processed_feature_columns": int(
        len(processed_feature_names_33B)
    ),
    "nonzero_coefficients": int(nonzero_coefficients_33B),
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_33B,
    "selection_sha256": selection_sha_33B,
    "secure_prediction_table": prediction_table_id_33B,
    "patient_level_prediction_written_to_drive": False,
    "analysis_role": "additional_post_hoc_clinical_baseline",
    "predictors": CLINICAL_PREDICTORS_33B,
}

with open(
    evaluation_json_path_33B,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        evaluation_configuration_33B,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(evaluation_json_path_33B, "rb") as file_handle:
    evaluation_sha_33B = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    evaluation_sha_path_33B,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(evaluation_sha_33B + "\n")

# ------------------------------------------------------------
# 24. Final outputs
# ------------------------------------------------------------

pooled_integrity_33B = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_33B),
            pooled_oof_33B["id_row"].nunique(),
            pooled_oof_33B["candidate_id"].nunique(),
            pooled_oof_33B["inner_fold"].nunique(),
            EXPECTED_SPLIT_33B["training_events"],
            (
                EXPECTED_SPLIT_33B["training_rows"]
                - EXPECTED_SPLIT_33B["training_events"]
            ),
            int(
                pooled_oof_33B.duplicated(
                    subset=["candidate_id", "id_row"]
                ).sum()
            ),
            int(pooled_oof_33B["prediction_raw"].isna().sum()),
            int(
                (~pooled_oof_33B["prediction_raw"].between(0, 1)).sum()
            ),
            pooled_load_method_33B,
        ],
    }
)

print("\n33B OUTER-FOLD-1 INNER CHECKPOINT SUMMARY")
display(checkpoint_summary_33B)

print("\n33B OUTER-FOLD-1 POOLED OOF INTEGRITY")
display(pooled_integrity_33B)

print("\n33B OUTER-FOLD-1 CANDIDATE RESULTS")
display(candidate_results_33B)

print("\n33B OUTER-FOLD-1 SELECTED MODEL")
display(selected_model_33B)

print("\n33B OUTER-FOLD-1 FINAL MODEL SUMMARY")
display(final_model_summary_33B)

print("\n33B OUTER-FOLD-1 TEST RESULTS")
display(outer1_test_results_33B)

print("\n33B OUTER-FOLD-1 BIGQUERY VERIFICATION")
display(prediction_verification_33B)

print("\n33B OUTER-FOLD-1 TOP 20 ABSOLUTE COEFFICIENTS")
display(coefficient_table_33B.head(20))

print("\nSelection SHA-256:")
print(selection_sha_33B)

print("\nEvaluation SHA-256:")
print(evaluation_sha_33B)

print("\nSaved:")
print(fit_audit_path_33B)
print(checkpoint_summary_path_33B)
print(candidate_results_path_33B)
print(selected_model_path_33B)
print(selection_json_path_33B)
print(selection_sha_path_33B)
print(test_results_path_33B)
print(model_summary_path_33B)
print(coefficient_path_33B)
print(evaluation_json_path_33B)
print(evaluation_sha_path_33B)

print(
    "\n33B-R2 PASS: Outer-fold-1 nested modelling "
    "and locked test evaluation are complete."
)
print(
    "All inner OOF and outer-test patient-level "
    "predictions were stored only in BigQuery."
)
print(
    "No patient-level prediction file was "
    "written to Google Drive."
)

_ = gc.collect()

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from sklearn.exceptions import ConvergenceWarning

from IPython.display import display

# ============================================================
# 33C — PARSIMONIOUS CLINICAL BASELINE OUTER FOLD 2
#
# Resume-safe:
# - Each inner fold is stored in a separate BigQuery table.
# - Completed inner folds are automatically skipped.
#
# BigQuery free-tier compatible:
# - No DELETE / INSERT / UPDATE / MERGE is used.
#
# Privacy:
# - Patient-level predictions are stored only in BigQuery.
# - No patient-level prediction file is written to Drive.
# ============================================================

print("STARTING PARSIMONIOUS CLINICAL BASELINE OUTER FOLD 2 — CODE VERSION 33C")

OUTER_FOLD_33C = 2
MODEL_RANDOM_SEED_33C = 20260721

EXPECTED_PROTOCOL_SHA_33C = (
    "94b0abb218dbef4e349702ea2824ca4e"
    "31dfbe36c53efa05ba1a8bd1f38f835e"
)

EXPECTED_AMENDMENT_SHA_33C = (
    "fcbcf857caa9aaad7ffd6691b4b61ac6"
    "503d09a6eee80db84bc2967ddf1f1a6a"
)

EXPECTED_CORRECTION_SHA_33C = (
    "ea7ab2747aad0c20157da0988b2fbeb1"
    "6ce007b4c92bf785d9ef9d363862ecfa"
)

EXPECTED_SPLIT_33C = {
    "training_rows": 46800,
    "test_rows": 11691,
    "training_hospitals": 158,
    "test_hospitals": 40,
    "training_events": 2426,
    "test_events": 606,
}

EXPECTED_INNER_33C = {
    1: {"validation_rows": 5805, "validation_events": 287},
    2: {"validation_rows": 8048, "validation_events": 406},
    3: {"validation_rows": 11580, "validation_events": 618},
    4: {"validation_rows": 10207, "validation_events": 495},
    5: {"validation_rows": 11160, "validation_events": 620},
}

CLINICAL_PREDICTORS_33C = [
    "x_age_years",
    "x_sex",
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
    "x_lab_bun_last",
    "x_vital_respiratory_rate_last",
    "x_vital_noninvasive_systolic_bp_last",
]

CLINICAL_NUMERIC_33C = [
    "x_age_years",
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
    "x_lab_bun_last",
    "x_vital_respiratory_rate_last",
    "x_vital_noninvasive_systolic_bp_last",
]

CLINICAL_CATEGORICAL_33C = ["x_sex"]

# ------------------------------------------------------------
# 1. Required runtime objects
# ------------------------------------------------------------

required_objects_33C = [
    "core_df_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_33C = [
    name for name in required_objects_33C if name not in globals()
]

if missing_objects_33C:
    raise RuntimeError(
        "Eksik RAM nesneleri var: "
        + ", ".join(missing_objects_33C)
        + ". Önce 07A ve 07B hücrelerini çalıştır."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"58.491 satır bekleniyordu; {len(core_df_07B)} bulundu."
    )

missing_predictors_33C = [
    column for column in CLINICAL_PREDICTORS_33C
    if column not in core_df_07B.columns
]

if missing_predictors_33C:
    raise RuntimeError(
        "Kilitli klinik predictor(lar) core_df_07B içinde yok: "
        + ", ".join(missing_predictors_33C)
    )

# ------------------------------------------------------------
# 2. Locked protocol SHA check
# ------------------------------------------------------------

protocol_sha_path_33C = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A_locked_parsimonious_clinical_baseline_protocol_v1_SHA256.txt",
)

if not os.path.exists(protocol_sha_path_33C):
    raise FileNotFoundError(
        "Model protokolü SHA dosyası bulunamadı: "
        + protocol_sha_path_33C
    )

with open(protocol_sha_path_33C, "r", encoding="utf-8") as file_handle:
    observed_protocol_sha_33C = file_handle.read().strip()

if observed_protocol_sha_33C != EXPECTED_PROTOCOL_SHA_33C:
    raise RuntimeError(
        "Kilitli model protokolü SHA değeri değişmiş: "
        + observed_protocol_sha_33C
    )

amendment_sha_path_33C = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A1_parsimonious_clinical_baseline_convergence_amendment_v1_SHA256.txt",
)

if not os.path.exists(amendment_sha_path_33C):
    raise FileNotFoundError(
        "Önce 33A1 convergence amendment scriptini çalıştır: "
        + amendment_sha_path_33C
    )

with open(amendment_sha_path_33C, "r", encoding="utf-8") as file_handle:
    observed_amendment_sha_33C = file_handle.read().strip()

if observed_amendment_sha_33C != EXPECTED_AMENDMENT_SHA_33C:
    raise RuntimeError(
        "33A1 amendment SHA değeri değişmiş: "
        + observed_amendment_sha_33C
    )

print("33A protocol SHA guard: PASS")
print("33A1 convergence amendment SHA guard: PASS")

correction_sha_path_33C = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A2_parsimonious_clinical_baseline_full_inner_refit_correction_v1_SHA256.txt",
)

if not os.path.exists(correction_sha_path_33C):
    raise FileNotFoundError(
        "Önce 33A2 correction scriptini çalıştır: "
        + correction_sha_path_33C
    )

with open(correction_sha_path_33C, "r", encoding="utf-8") as file_handle:
    observed_correction_sha_33C = file_handle.read().strip()

if observed_correction_sha_33C != EXPECTED_CORRECTION_SHA_33C:
    raise RuntimeError(
        "33A2 correction SHA değeri değişmiş: "
        + observed_correction_sha_33C
    )

print("33A2 full inner-refit correction SHA guard: PASS")

# ------------------------------------------------------------
# 3. Locked inner-hospital mapping
# ------------------------------------------------------------

inner_mapping_path_33C = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_33C):
    raise FileNotFoundError(
        "Kilitli iç kat haritası bulunamadı: "
        + inner_mapping_path_33C
    )

inner_mapping_all_33C = pd.read_csv(
    inner_mapping_path_33C,
    dtype={"group_hospital": str},
)

inner_mapping_part_33C = (
    inner_mapping_all_33C.loc[
        inner_mapping_all_33C["outer_fold"].astype(int) == OUTER_FOLD_33C,
        ["group_hospital", "inner_fold"],
    ]
    .copy()
)

inner_mapping_part_33C["group_hospital"] = (
    inner_mapping_part_33C["group_hospital"].astype(str)
)
inner_mapping_part_33C["inner_fold"] = (
    inner_mapping_part_33C["inner_fold"].astype(int)
)

if len(inner_mapping_part_33C) != 158:
    raise RuntimeError(
        "Dış kat 1 eğitim kümesi için 158 hastane ataması bekleniyordu."
    )

if inner_mapping_part_33C["group_hospital"].duplicated().any():
    raise RuntimeError("İç kat haritasında yinelenen hastane var.")

hospital_to_inner_fold_33C = dict(
    zip(
        inner_mapping_part_33C["group_hospital"],
        inner_mapping_part_33C["inner_fold"],
    )
)

# ------------------------------------------------------------
# 4. Prepare model matrices
# ------------------------------------------------------------

X_all_33C = core_df_07B[CLINICAL_PREDICTORS_33C].copy()

for column in CLINICAL_NUMERIC_33C:
    X_all_33C[column] = pd.to_numeric(
        X_all_33C[column], errors="coerce"
    ).astype("float64")

for column in CLINICAL_CATEGORICAL_33C:
    category_series = X_all_33C[column].astype("object")
    X_all_33C[column] = category_series.where(
        pd.notna(category_series), np.nan
    )

outer_fold_vector_33C = (
    core_df_07B["outer_fold"].astype(int).to_numpy()
)

outer_training_mask_33C = outer_fold_vector_33C != OUTER_FOLD_33C
outer_test_mask_33C = outer_fold_vector_33C == OUTER_FOLD_33C

X_outer_training_33C = (
    X_all_33C.loc[outer_training_mask_33C].reset_index(drop=True)
)
X_outer_test_33C = (
    X_all_33C.loc[outer_test_mask_33C].reset_index(drop=True)
)

outer_training_meta_33C = (
    core_df_07B.loc[
        outer_training_mask_33C,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_33C = (
    core_df_07B.loc[
        outer_test_mask_33C,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [outer_training_meta_33C, outer_test_meta_33C]:
    dataframe["id_row"] = dataframe["id_row"].astype(str)
    dataframe["group_hospital"] = dataframe["group_hospital"].astype(str)
    dataframe["label_stage23"] = dataframe["label_stage23"].astype(int)

y_outer_training_33C = (
    outer_training_meta_33C["label_stage23"].to_numpy(dtype=np.int8)
)
y_outer_test_33C = (
    outer_test_meta_33C["label_stage23"].to_numpy(dtype=np.int8)
)

groups_outer_training_33C = (
    outer_training_meta_33C["group_hospital"].to_numpy(dtype=str)
)

training_hospitals_33C = set(
    outer_training_meta_33C["group_hospital"]
)
test_hospitals_33C = set(
    outer_test_meta_33C["group_hospital"]
)
hospital_overlap_33C = training_hospitals_33C & test_hospitals_33C

if hospital_overlap_33C:
    raise RuntimeError("Dış eğitim ve test hastaneleri çakışıyor.")

actual_split_33C = {
    "training_rows": len(X_outer_training_33C),
    "test_rows": len(X_outer_test_33C),
    "training_hospitals": len(training_hospitals_33C),
    "test_hospitals": len(test_hospitals_33C),
    "training_events": int(y_outer_training_33C.sum()),
    "test_events": int(y_outer_test_33C.sum()),
}

for metric, expected_value in EXPECTED_SPLIT_33C.items():
    actual_value = actual_split_33C[metric]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: bulunan={actual_value}, beklenen={expected_value}"
        )

inner_fold_vector_33C = np.array(
    [
        hospital_to_inner_fold_33C.get(hospital, -1)
        for hospital in groups_outer_training_33C
    ],
    dtype=int,
)

if (inner_fold_vector_33C == -1).any():
    raise RuntimeError(
        "Bazı dış eğitim hastanelerine iç kat atanmadı."
    )

if set(np.unique(inner_fold_vector_33C)) != {1, 2, 3, 4, 5}:
    raise RuntimeError("İç kat değerleri 1–5 değil.")

for inner_fold, expected in EXPECTED_INNER_33C.items():
    validation_mask = inner_fold_vector_33C == inner_fold
    observed_rows = int(validation_mask.sum())
    observed_events = int(y_outer_training_33C[validation_mask].sum())

    if observed_rows != expected["validation_rows"]:
        raise RuntimeError(
            f"Inner {inner_fold} validation_rows: "
            f"bulunan={observed_rows}, "
            f"beklenen={expected['validation_rows']}"
        )

    if observed_events != expected["validation_events"]:
        raise RuntimeError(
            f"Inner {inner_fold} validation_events: "
            f"bulunan={observed_events}, "
            f"beklenen={expected['validation_events']}"
        )

# ------------------------------------------------------------
# 5. Locked candidate grid
# ------------------------------------------------------------

candidate_grid_33C = [
    {"candidate_id": "CLIN01", "C": 0.03, "l1_ratio": 0.00},
    {"candidate_id": "CLIN02", "C": 0.10, "l1_ratio": 0.00},
    {"candidate_id": "CLIN03", "C": 0.30, "l1_ratio": 0.00},
    {"candidate_id": "CLIN04", "C": 0.10, "l1_ratio": 0.25},
    {"candidate_id": "CLIN05", "C": 0.30, "l1_ratio": 0.25},
    {"candidate_id": "CLIN06", "C": 0.30, "l1_ratio": 0.50},
]

# ------------------------------------------------------------
# 6. Preprocessing and model factories
# ------------------------------------------------------------

def make_preprocessor_33C():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
            (
                "scaler",
                StandardScaler(with_mean=False),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                CLINICAL_NUMERIC_33C,
            ),
            (
                "categorical",
                categorical_pipeline,
                CLINICAL_CATEGORICAL_33C,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_model_33C(C_value, l1_ratio_value):
    return LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=float(C_value),
        l1_ratio=float(l1_ratio_value),
        class_weight=None,
        max_iter=20000,
        tol=1e-4,
        random_state=MODEL_RANDOM_SEED_33C,
    )


def checkpoint_table_id_33C(inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_clinical_lr_inner_oof_outer2_inner{inner_fold}_v2"
    )

# ------------------------------------------------------------
# 7. BigQuery checkpoint verification
# ------------------------------------------------------------

def verify_checkpoint_33C(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):
    table_id = checkpoint_table_id_33C(inner_fold)

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS distinct_id_count,
      COUNT(DISTINCT candidate_id) AS candidate_count,
      COUNT(DISTINCT outer_fold) AS outer_fold_count,
      COUNT(DISTINCT inner_fold) AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(candidate_id, '|', id_row)
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(
        prediction_raw < 0 OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(sql, location=BQ_LOCATION).to_dataframe()
    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows * len(candidate_grid_33C)
    )
    expected_positive_rows = (
        expected_validation_events * len(candidate_grid_33C)
    )
    expected_negative_rows = (
        (expected_validation_rows - expected_validation_events)
        * len(candidate_grid_33C)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": expected_validation_rows,
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": expected_total_rows,
        "positive_prediction_rows": expected_positive_rows,
        "negative_prediction_rows": expected_negative_rows,
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": 2,
        "maximum_outer_fold": 2,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failure_items = []

    for field, expected_value in expected_values.items():
        actual_value = int(row[field])
        if actual_value != expected_value:
            complete = False
            failure_items.append(
                f"{field}={actual_value}, expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failure_items),
        "check": check,
        "row": row,
    }

# ------------------------------------------------------------
# 8. BigQuery load schema
# ------------------------------------------------------------

checkpoint_load_config_33C = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("inner_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("candidate_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

# ------------------------------------------------------------
# 9. Aggregate fit-audit file
# ------------------------------------------------------------

fit_audit_columns_33C = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "C",
    "l1_ratio",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "convergence_warnings",
    "elapsed_seconds",
]

fit_audit_path_33C = os.path.join(
    MODEL_OUTPUT_DIR,
    "33C_clinical_inner_fit_audit_outer2.csv",
)

if os.path.exists(fit_audit_path_33C):
    fit_audit_33C = pd.read_csv(fit_audit_path_33C)
else:
    fit_audit_33C = pd.DataFrame(columns=fit_audit_columns_33C)

for column in fit_audit_columns_33C:
    if column not in fit_audit_33C.columns:
        fit_audit_33C[column] = np.nan

fit_audit_33C = fit_audit_33C[fit_audit_columns_33C].copy()

# ------------------------------------------------------------
# 10. Train six candidates in five locked inner folds
# ------------------------------------------------------------

for inner_fold in range(1, 6):
    inner_training_mask = inner_fold_vector_33C != inner_fold
    inner_validation_mask = inner_fold_vector_33C == inner_fold

    training_rows = int(inner_training_mask.sum())
    validation_rows = int(inner_validation_mask.sum())
    training_events = int(
        y_outer_training_33C[inner_training_mask].sum()
    )
    validation_events = int(
        y_outer_training_33C[inner_validation_mask].sum()
    )

    expected_inner = EXPECTED_INNER_33C[inner_fold]
    expected_training_rows = (
        EXPECTED_SPLIT_33C["training_rows"]
        - expected_inner["validation_rows"]
    )
    expected_training_events = (
        EXPECTED_SPLIT_33C["training_events"]
        - expected_inner["validation_events"]
    )

    if training_rows != expected_training_rows:
        raise RuntimeError(
            f"Inner {inner_fold} training_rows: "
            f"bulunan={training_rows}, "
            f"beklenen={expected_training_rows}"
        )

    if training_events != expected_training_events:
        raise RuntimeError(
            f"Inner {inner_fold} training_events: "
            f"bulunan={training_events}, "
            f"beklenen={expected_training_events}"
        )

    existing_check = verify_checkpoint_33C(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if existing_check["complete"]:
        print(
            f"Outer 2 / inner {inner_fold}: "
            "permanent checkpoint already complete; "
            "skipping model fitting."
        )
        continue

    training_hospital_set = set(
        groups_outer_training_33C[inner_training_mask]
    )
    validation_hospital_set = set(
        groups_outer_training_33C[inner_validation_mask]
    )

    if training_hospital_set & validation_hospital_set:
        raise RuntimeError(
            f"Inner fold {inner_fold}: hastane çakışması bulundu."
        )

    print(f"\nOuter 2 / inner {inner_fold}")
    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_33C()
    preprocessing_started = time.time()

    X_inner_training_processed = preprocessor.fit_transform(
        X_outer_training_33C.loc[inner_training_mask]
    )
    X_inner_validation_processed = preprocessor.transform(
        X_outer_training_33C.loc[inner_validation_mask]
    )

    preprocessing_elapsed = time.time() - preprocessing_started

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "İşlenmiş eğitim ve doğrulama sütun sayıları farklı."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_elapsed, 2),
    )

    y_inner_training = y_outer_training_33C[inner_training_mask]
    y_inner_validation = y_outer_training_33C[inner_validation_mask]

    validation_ids = (
        outer_training_meta_33C.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_33C:
        candidate_id = candidate["candidate_id"]

        print(
            "  Fitting",
            candidate_id,
            "| C =",
            candidate["C"],
            "| l1_ratio =",
            candidate["l1_ratio"],
        )

        model = make_model_33C(
            candidate["C"],
            candidate["l1_ratio"],
        )

        fitting_started = time.time()

        with warnings.catch_warnings(record=True) as warning_records:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(
                X_inner_training_processed,
                y_inner_training,
            )

        fitting_elapsed = time.time() - fitting_started

        convergence_warning_count = sum(
            issubclass(warning.category, ConvergenceWarning)
            for warning in warning_records
        )

        validation_probabilities = model.predict_proba(
            X_inner_validation_processed
        )[:, 1]

        if np.isnan(validation_probabilities).any():
            raise RuntimeError(
                f"{candidate_id}, inner {inner_fold}: eksik tahmin."
            )

        if not np.all(
            (validation_probabilities >= 0)
            & (validation_probabilities <= 1)
        ):
            raise RuntimeError(
                f"{candidate_id}: geçersiz olasılık."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        2,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": y_inner_validation.astype(np.int64),
                    "prediction_raw": validation_probabilities.astype(
                        np.float64
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": 2,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "C": candidate["C"],
                "l1_ratio": candidate["l1_ratio"],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": validation_events,
                "processed_columns": int(
                    X_inner_training_processed.shape[1]
                ),
                "convergence_warnings": int(
                    convergence_warning_count
                ),
                "elapsed_seconds": float(fitting_elapsed),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows * len(candidate_grid_33C)
    )

    if len(checkpoint_df) != expected_checkpoint_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: checkpoint satır sayısı hatalı."
        )

    if checkpoint_df.duplicated(
        subset=["id_row", "candidate_id"]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "yinelenen aday–hasta tahmini var."
        )

    target_checkpoint_table = checkpoint_table_id_33C(inner_fold)

    print(
        "Uploading permanent checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_33C,
        location=BQ_LOCATION,
    ).result()

    new_audit_df = pd.DataFrame(current_audit_rows)

    if len(fit_audit_33C) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_33C["outer_fold"],
                    errors="coerce",
                )
                == 2
            )
            & (
                pd.to_numeric(
                    fit_audit_33C["inner_fold"],
                    errors="coerce",
                )
                == inner_fold
            )
        )
        fit_audit_33C = fit_audit_33C.loc[keep_mask].copy()

    if fit_audit_33C.empty:
        fit_audit_33C = new_audit_df.copy()
    else:
        fit_audit_33C = pd.concat(
            [fit_audit_33C, new_audit_df],
            ignore_index=True,
        )

    fit_audit_33C = (
        fit_audit_33C[fit_audit_columns_33C]
        .sort_values(
            ["outer_fold", "inner_fold", "candidate_id"]
        )
        .reset_index(drop=True)
    )

    fit_audit_33C.to_csv(
        fit_audit_path_33C,
        index=False,
    )

    completed_check = verify_checkpoint_33C(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint doğrulanamadı: "
            + completed_check["reason"]
        )

    print(
        f"Outer 2 / inner {inner_fold}: "
        "permanent checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 11. Final checkpoint summary
# ------------------------------------------------------------

checkpoint_summary_rows_33C = []

for inner_fold in range(1, 6):
    validation_mask = inner_fold_vector_33C == inner_fold
    validation_rows = int(validation_mask.sum())
    validation_events = int(
        y_outer_training_33C[validation_mask].sum()
    )

    final_check = verify_checkpoint_33C(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "son checkpoint denetimi başarısız. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_33C.append(
        {
            "outer_fold": 2,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(row["row_count"]),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(row["candidate_count"]),
            "positive_prediction_rows": int(
                row["positive_prediction_rows"]
            ),
            "negative_prediction_rows": int(
                row["negative_prediction_rows"]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check["table_id"],
        }
    )

checkpoint_summary_33C = (
    pd.DataFrame(checkpoint_summary_rows_33C)
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_33C["distinct_validation_patients"].sum()
) != EXPECTED_SPLIT_33C["training_rows"]:
    raise RuntimeError(
        "Toplam doğrulama hasta sayısı 46.803 değil."
    )

expected_total_oof_rows_33C = (
    EXPECTED_SPLIT_33C["training_rows"]
    * len(candidate_grid_33C)
)

if int(checkpoint_summary_33C["checkpoint_rows"].sum()) != (
    expected_total_oof_rows_33C
):
    raise RuntimeError("Toplam OOF tahmin satırı hatalı.")

checkpoint_summary_path_33C = os.path.join(
    MODEL_OUTPUT_DIR,
    "33C_clinical_outer2_inner_checkpoint_summary.csv",
)
checkpoint_summary_33C.to_csv(
    checkpoint_summary_path_33C,
    index=False,
)

# ------------------------------------------------------------
# 12. Pool all inner OOF predictions
# ------------------------------------------------------------

checkpoint_tables_33C = [
    checkpoint_table_id_33C(inner_fold)
    for inner_fold in range(1, 6)
]

union_parts_33C = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_33C
]

SQL_LOAD_POOLED_OOF_33C = "\nUNION ALL\n".join(union_parts_33C)

print("\nLoading pooled outer-fold-2 inner OOF predictions...")

query_job_33C = client.query(
    SQL_LOAD_POOLED_OOF_33C,
    location=BQ_LOCATION,
)

try:
    pooled_oof_33C = query_job_33C.to_dataframe(
        create_bqstorage_client=True
    )
    pooled_load_method_33C = "BigQuery Storage API"
except Exception as fast_path_error_33C:
    print(
        "Storage API unavailable; using standard BigQuery download."
    )
    print("Message:", type(fast_path_error_33C).__name__)
    pooled_oof_33C = query_job_33C.to_dataframe(
        create_bqstorage_client=False
    )
    pooled_load_method_33C = "Standard BigQuery API"

pooled_oof_33C["id_row"] = pooled_oof_33C["id_row"].astype(str)
pooled_oof_33C["candidate_id"] = pooled_oof_33C["candidate_id"].astype(str)

for column in ["outer_fold", "inner_fold", "label_stage23"]:
    pooled_oof_33C[column] = pd.to_numeric(
        pooled_oof_33C[column], errors="raise"
    ).astype(int)

pooled_oof_33C["prediction_raw"] = pd.to_numeric(
    pooled_oof_33C["prediction_raw"], errors="raise"
).astype(float)

# ------------------------------------------------------------
# 13. Pooled OOF integrity checks
# ------------------------------------------------------------

if len(pooled_oof_33C) != expected_total_oof_rows_33C:
    raise RuntimeError("Pooled OOF satır sayısı hatalı.")

if set(pooled_oof_33C["outer_fold"].unique()) != {2}:
    raise RuntimeError("Pooled OOF içinde dış kat 4 dışında kayıt var.")

if set(pooled_oof_33C["inner_fold"].unique()) != {1, 2, 3, 4, 5}:
    raise RuntimeError("Pooled OOF iç katları 1–5 değil.")

if pooled_oof_33C.duplicated(
    subset=["candidate_id", "id_row"]
).any():
    raise RuntimeError(
        "Pooled OOF içinde yinelenen aday–hasta tahmini bulundu."
    )

if pooled_oof_33C["prediction_raw"].isna().any():
    raise RuntimeError("Pooled OOF içinde eksik tahmin var.")

if not pooled_oof_33C["prediction_raw"].between(0, 1).all():
    raise RuntimeError(
        "Pooled OOF içinde geçersiz olasılık değeri var."
    )

expected_candidate_ids_33C = {
    "CLIN01",
    "CLIN02",
    "CLIN03",
    "CLIN04",
    "CLIN05",
    "CLIN06",
}

if set(pooled_oof_33C["candidate_id"].unique()) != (
    expected_candidate_ids_33C
):
    raise RuntimeError("Altı kilitli aday bulunmuyor.")

candidate_patient_counts_33C = (
    pooled_oof_33C.groupby("candidate_id")["id_row"].nunique()
)

if not (
    candidate_patient_counts_33C
    == EXPECTED_SPLIT_33C["training_rows"]
).all():
    raise RuntimeError(
        "Her aday için 46.803 farklı OOF hastası yok."
    )

candidate_event_counts_33C = (
    pooled_oof_33C.groupby("candidate_id")["label_stage23"].sum()
)

if not (
    candidate_event_counts_33C
    == EXPECTED_SPLIT_33C["training_events"]
).all():
    raise RuntimeError("Her aday için 2.426 olay yok.")

patient_label_consistency_33C = (
    pooled_oof_33C.groupby("id_row")["label_stage23"].nunique()
)
if (patient_label_consistency_33C > 1).any():
    raise RuntimeError(
        "Aynı hastanın adaylar arasında outcome etiketi farklı."
    )

patient_inner_fold_consistency_33C = (
    pooled_oof_33C.groupby("id_row")["inner_fold"].nunique()
)
if (patient_inner_fold_consistency_33C > 1).any():
    raise RuntimeError(
        "Aynı hasta birden fazla iç doğrulama katında bulundu."
    )

# ------------------------------------------------------------
# 14. Metric helpers
# ------------------------------------------------------------

def probability_metrics_33C(y_true, probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(roc_auc_score(y_true, probabilities)),
        "auprc": float(average_precision_score(y_true, probabilities)),
        "brier": float(brier_score_loss(y_true, probabilities)),
        "log_loss": float(
            log_loss(y_true, probabilities, labels=[0, 1])
        ),
        "mean_predicted_risk": float(probabilities.mean()),
        "observed_event_rate": float(np.mean(y_true)),
    }


def probability_logit_33C(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        probabilities / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_33C(y_true, probabilities):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        probability_logit_33C(probabilities),
        y_true,
    )
    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 15. Candidate pooled-OOF performance
# ------------------------------------------------------------

candidate_result_rows_33C = []

for candidate in candidate_grid_33C:
    candidate_id = candidate["candidate_id"]

    candidate_oof = (
        pooled_oof_33C.loc[
            pooled_oof_33C["candidate_id"] == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_33C(
        candidate_oof["label_stage23"].to_numpy(dtype=int),
        candidate_oof["prediction_raw"].to_numpy(dtype=float),
    )

    fit_part = fit_audit_33C.loc[
        (
            pd.to_numeric(
                fit_audit_33C["outer_fold"],
                errors="coerce",
            )
            == 2
        )
        & (
            fit_audit_33C["candidate_id"].astype(str)
            == candidate_id
        )
    ]

    convergence_warnings = (
        int(
            pd.to_numeric(
                fit_part["convergence_warnings"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    elapsed_seconds = (
        float(
            pd.to_numeric(
                fit_part["elapsed_seconds"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_33C.append(
        {
            "candidate_id": candidate_id,
            "C": candidate["C"],
            "l1_ratio": candidate["l1_ratio"],
            **metrics,
            "convergence_warnings": convergence_warnings,
            "elapsed_seconds": elapsed_seconds,
        }
    )

candidate_results_33C = pd.DataFrame(candidate_result_rows_33C)

total_inner_convergence_warnings_33C = int(
    pd.to_numeric(
        candidate_results_33C["convergence_warnings"],
        errors="coerce",
    ).fillna(0).sum()
)

if total_inner_convergence_warnings_33C != 0:
    raise RuntimeError(
        "33B-R2 stopped BEFORE candidate selection and BEFORE outer-test "
        "evaluation because at least one inner candidate fit still emitted "
        f"a ConvergenceWarning under max_iter=20000. "
        f"Total warnings={total_inner_convergence_warnings_33C}. "
        "No test result may be accessed."
    )

print(
    "All 30 corrected inner candidate fits converged with zero warnings: PASS"
)

candidate_results_33C = (
    candidate_results_33C.sort_values(
        ["auprc", "auroc", "brier", "candidate_id"],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)

candidate_results_33C["selection_rank"] = np.arange(
    1,
    len(candidate_results_33C) + 1,
)

best_row_33C = candidate_results_33C.iloc[0]
selected_candidate_33C = str(best_row_33C["candidate_id"])
selected_C_33C = float(best_row_33C["C"])
selected_l1_ratio_33C = float(best_row_33C["l1_ratio"])

# ------------------------------------------------------------
# 16. Platt calibration on selected pooled inner OOF
# ------------------------------------------------------------

selected_oof_33C = (
    pooled_oof_33C.loc[
        pooled_oof_33C["candidate_id"] == selected_candidate_33C
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_33C = selected_oof_33C[
    "label_stage23"
].to_numpy(dtype=int)
selected_oof_probability_33C = selected_oof_33C[
    "prediction_raw"
].to_numpy(dtype=float)

platt_calibrator_33C = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_33C.fit(
    probability_logit_33C(selected_oof_probability_33C),
    selected_oof_y_33C,
)

platt_intercept_33C = float(platt_calibrator_33C.intercept_[0])
platt_slope_33C = float(platt_calibrator_33C.coef_[0][0])

if (
    not np.isfinite(platt_intercept_33C)
    or not np.isfinite(platt_slope_33C)
    or platt_slope_33C <= 0
):
    raise RuntimeError("Platt kalibrasyon katsayıları geçersiz.")

selected_model_33C = pd.DataFrame(
    [
        {
            "outer_fold": 2,
            "selected_candidate": selected_candidate_33C,
            "selected_C": selected_C_33C,
            "selected_l1_ratio": selected_l1_ratio_33C,
            "selection_metric_primary": "pooled_inner_oof_auprc",
            "inner_oof_auprc": float(best_row_33C["auprc"]),
            "inner_oof_auroc": float(best_row_33C["auroc"]),
            "inner_oof_brier": float(best_row_33C["brier"]),
            "inner_oof_log_loss": float(best_row_33C["log_loss"]),
            "inner_oof_mean_predicted_risk": float(
                best_row_33C["mean_predicted_risk"]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_33C["observed_event_rate"]
            ),
            "platt_intercept": platt_intercept_33C,
            "platt_slope": platt_slope_33C,
            "protocol_sha256": EXPECTED_PROTOCOL_SHA_33C,
            "numerical_amendment_sha256": EXPECTED_AMENDMENT_SHA_33C,
            "full_inner_refit_correction_sha256": EXPECTED_CORRECTION_SHA_33C,
            "corrected_checkpoint_namespace": "v2",
            "analysis_status": "post_hoc_exploratory_corrected",
            "full_inner_refit_correction_sha256": EXPECTED_CORRECTION_SHA_33C,
            "corrected_checkpoint_namespace": "v2",
            "analysis_status": "post_hoc_exploratory_corrected",
            "model_max_iter": 20000,
        }
    ]
)

# ------------------------------------------------------------
# 17. Lock selection/calibration aggregate files
# ------------------------------------------------------------

candidate_results_path_33C = os.path.join(
    MODEL_OUTPUT_DIR,
    "33C_clinical_candidate_results_outer2.csv",
)
selected_model_path_33C = os.path.join(
    MODEL_OUTPUT_DIR,
    "33C_clinical_selected_model_outer2.csv",
)
selection_json_path_33C = os.path.join(
    MODEL_OUTPUT_DIR,
    "33C_clinical_selection_calibration_outer2.json",
)
selection_sha_path_33C = os.path.join(
    MODEL_OUTPUT_DIR,
    "33C_clinical_selection_calibration_outer2_SHA256.txt",
)

candidate_results_33C.to_csv(
    candidate_results_path_33C,
    index=False,
)
selected_model_33C.to_csv(
    selected_model_path_33C,
    index=False,
)

selection_configuration_33C = {
    "outer_fold": 2,
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_33C,
    "selection_metric_primary": "pooled inner out-of-fold AUPRC",
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
        "candidate_id ascending immutable tie-break",
    ],
    "selected_candidate": selected_candidate_33C,
    "selected_C": selected_C_33C,
    "selected_l1_ratio": selected_l1_ratio_33C,
    "inner_oof_auprc": float(best_row_33C["auprc"]),
    "inner_oof_auroc": float(best_row_33C["auroc"]),
    "inner_oof_brier": float(best_row_33C["brier"]),
    "platt_intercept": platt_intercept_33C,
    "platt_slope": platt_slope_33C,
    "inner_checkpoint_tables": checkpoint_tables_33C,
    "patient_level_oof_written_to_drive": False,
    "analysis_role": "additional_post_hoc_clinical_baseline",
    "predictors": CLINICAL_PREDICTORS_33C,
}

with open(
    selection_json_path_33C,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        selection_configuration_33C,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(selection_json_path_33C, "rb") as file_handle:
    selection_sha_33C = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    selection_sha_path_33C,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(selection_sha_33C + "\n")

# ------------------------------------------------------------
# 18. Fit final selected outer-fold-2 model
# ------------------------------------------------------------

final_pipeline_33C = Pipeline(
    steps=[
        ("preprocessor", make_preprocessor_33C()),
        (
            "model",
            make_model_33C(
                selected_C_33C,
                selected_l1_ratio_33C,
            ),
        ),
    ]
)

print(
    "\nFitting selected outer-fold-2 model "
    "on all 46,803 training patients..."
)

final_fit_started_33C = time.time()

with warnings.catch_warnings(record=True) as final_warning_records_33C:
    warnings.simplefilter("always", ConvergenceWarning)
    final_pipeline_33C.fit(
        X_outer_training_33C,
        y_outer_training_33C,
    )

final_fit_elapsed_33C = time.time() - final_fit_started_33C

final_model_n_iter_33C = int(
    np.max(
        np.asarray(
            final_pipeline_33C.named_steps["model"].n_iter_
        )
    )
)

final_convergence_warnings_33C = sum(
    issubclass(warning.category, ConvergenceWarning)
    for warning in final_warning_records_33C
)

if final_convergence_warnings_33C != 0:
    raise RuntimeError(
        "Nihai outer-fold-2 klinik baseline modeli max_iter=20000 "
        f"altında da yakınsamadı. n_iter_={final_model_n_iter_33C}"
    )

# ------------------------------------------------------------
# 19. Outer-fold-2 test predictions and metrics
# ------------------------------------------------------------

outer2_raw_probabilities_33C = final_pipeline_33C.predict_proba(
    X_outer_test_33C
)[:, 1]

raw_clipped_33C = np.clip(
    outer2_raw_probabilities_33C,
    1e-6,
    1 - 1e-6,
)
raw_logit_33C = np.log(
    raw_clipped_33C / (1 - raw_clipped_33C)
)

outer2_platt_probabilities_33C = expit(
    platt_intercept_33C + platt_slope_33C * raw_logit_33C
)

for probabilities, name in [
    (outer2_raw_probabilities_33C, "raw"),
    (outer2_platt_probabilities_33C, "platt"),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(f"{name} tahminlerinde eksik değer var.")

    if not np.all(
        (probabilities >= 0) & (probabilities <= 1)
    ):
        raise RuntimeError(
            f"{name} tahminlerinde geçersiz olasılık değeri var."
        )

raw_metrics_33C = probability_metrics_33C(
    y_outer_test_33C,
    outer2_raw_probabilities_33C,
)
platt_metrics_33C = probability_metrics_33C(
    y_outer_test_33C,
    outer2_platt_probabilities_33C,
)

raw_calibration_intercept_33C, raw_calibration_slope_33C = (
    calibration_intercept_slope_33C(
        y_outer_test_33C,
        outer2_raw_probabilities_33C,
    )
)

platt_calibration_intercept_33C, platt_calibration_slope_33C = (
    calibration_intercept_slope_33C(
        y_outer_test_33C,
        outer2_platt_probabilities_33C,
    )
)

outer2_test_results_33C = pd.DataFrame(
    [
        {
            "outer_fold": 2,
            "model": "parsimonious_clinical_logistic",
            "probability_type": "raw",
            **raw_metrics_33C,
            "calibration_intercept": raw_calibration_intercept_33C,
            "calibration_slope": raw_calibration_slope_33C,
        },
        {
            "outer_fold": 2,
            "model": "parsimonious_clinical_logistic",
            "probability_type": "platt_calibrated",
            **platt_metrics_33C,
            "calibration_intercept": platt_calibration_intercept_33C,
            "calibration_slope": platt_calibration_slope_33C,
        },
    ]
)

# ------------------------------------------------------------
# 20. Feature coefficient audit
# ------------------------------------------------------------

fitted_preprocessor_33C = final_pipeline_33C.named_steps[
    "preprocessor"
]
fitted_model_33C = final_pipeline_33C.named_steps["model"]

processed_feature_names_33C = (
    fitted_preprocessor_33C.get_feature_names_out()
)
model_coefficients_33C = fitted_model_33C.coef_.reshape(-1)

if len(processed_feature_names_33C) != len(model_coefficients_33C):
    raise RuntimeError("Feature ve katsayı sayıları uyuşmuyor.")

if len(set(processed_feature_names_33C)) != len(
    processed_feature_names_33C
):
    raise RuntimeError("İşlenmiş feature adlarında yinelenme var.")

coefficient_table_33C = pd.DataFrame(
    {
        "processed_feature": processed_feature_names_33C,
        "coefficient": model_coefficients_33C,
    }
)
coefficient_table_33C["absolute_coefficient"] = (
    coefficient_table_33C["coefficient"].abs()
)
coefficient_table_33C["is_nonzero"] = ~np.isclose(
    coefficient_table_33C["coefficient"],
    0.0,
    atol=1e-12,
)
coefficient_table_33C["absolute_rank"] = (
    coefficient_table_33C["absolute_coefficient"]
    .rank(method="first", ascending=False)
    .astype(int)
)
coefficient_table_33C = (
    coefficient_table_33C.sort_values(
        ["absolute_coefficient", "processed_feature"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

nonzero_coefficients_33C = int(
    coefficient_table_33C["is_nonzero"].sum()
)

final_model_summary_33C = pd.DataFrame(
    [
        {
            "outer_fold": 2,
            "selected_candidate": selected_candidate_33C,
            "selected_C": selected_C_33C,
            "selected_l1_ratio": selected_l1_ratio_33C,
            "training_patients": len(X_outer_training_33C),
            "training_hospitals": len(training_hospitals_33C),
            "training_events": int(y_outer_training_33C.sum()),
            "test_patients": len(X_outer_test_33C),
            "test_hospitals": len(test_hospitals_33C),
            "test_events": int(y_outer_test_33C.sum()),
            "hospital_overlap": len(hospital_overlap_33C),
            "processed_feature_columns": len(
                processed_feature_names_33C
            ),
            "nonzero_coefficients": nonzero_coefficients_33C,
            "model_intercept": float(
                fitted_model_33C.intercept_[0]
            ),
            "convergence_warnings": final_convergence_warnings_33C,
            "final_model_n_iter": final_model_n_iter_33C,
            "fit_elapsed_seconds": float(final_fit_elapsed_33C),
            "locked_platt_intercept": platt_intercept_33C,
            "locked_platt_slope": platt_slope_33C,
            "protocol_sha256": EXPECTED_PROTOCOL_SHA_33C,
            "numerical_amendment_sha256": EXPECTED_AMENDMENT_SHA_33C,
            "model_max_iter": 20000,
            "selection_sha256": selection_sha_33C,
        }
    ]
)

# ------------------------------------------------------------
# 21. Secure BigQuery outer-test checkpoint
# ------------------------------------------------------------

outer2_prediction_df_33C = pd.DataFrame(
    {
        "id_row": outer_test_meta_33C["id_row"].astype(str),
        "outer_fold": np.full(
            len(outer_test_meta_33C),
            2,
            dtype=np.int64,
        ),
        "label_stage23": y_outer_test_33C.astype(np.int64),
        "prediction_raw": outer2_raw_probabilities_33C.astype(
            np.float64
        ),
        "prediction_platt": outer2_platt_probabilities_33C.astype(
            np.float64
        ),
        "model_name": "parsimonious_clinical_logistic",
        "model_version": "clinical8_v1_nested_cv",
    }
)

if len(outer2_prediction_df_33C) != 11688:
    raise RuntimeError(
        "Dış kat 4 tahmin satır sayısı 11.688 değil."
    )

if outer2_prediction_df_33C["id_row"].duplicated().any():
    raise RuntimeError(
        "Dış kat 4 tahminlerinde yinelenen id_row var."
    )

if int(
    outer2_prediction_df_33C["label_stage23"].sum()
) != 606:
    raise RuntimeError("Dış kat 4 olay sayısı 606 değil.")

prediction_table_id_33C = (
    f"{TARGET_DATASET}."
    "model_clinical_lr_outer_predictions_outer2_v2"
)

prediction_load_config_33C = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("prediction_platt", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("model_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("model_version", "STRING", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

print("\nUploading secure outer-fold-2 prediction checkpoint:")
print(prediction_table_id_33C)

client.load_table_from_dataframe(
    outer2_prediction_df_33C,
    prediction_table_id_33C,
    job_config=prediction_load_config_33C,
    location=BQ_LOCATION,
).result()

# ------------------------------------------------------------
# 22. BigQuery outer-test checkpoint verification
# ------------------------------------------------------------

SQL_VERIFY_PREDICTIONS_33C = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL) AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL) AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0 OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0 OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw) AS minimum_raw_probability,
  MAX(prediction_raw) AS maximum_raw_probability,
  MIN(prediction_platt) AS minimum_platt_probability,
  MAX(prediction_platt) AS maximum_platt_probability
FROM `{prediction_table_id_33C}`;
"""

prediction_verification_33C = client.query(
    SQL_VERIFY_PREDICTIONS_33C,
    location=BQ_LOCATION,
).to_dataframe()

verification_row_33C = prediction_verification_33C.iloc[0]

expected_prediction_values_33C = {
    "prediction_rows": 11688,
    "distinct_rows": 11688,
    "outer_folds": 1,
    "minimum_outer_fold": 2,
    "maximum_outer_fold": 2,
    "events": 606,
    "nonevents": 11082,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in expected_prediction_values_33C.items():
    actual_value = int(verification_row_33C[field])
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: bulunan={actual_value}, beklenen={expected_value}"
        )

# ------------------------------------------------------------
# 23. Save only aggregate and feature-level outputs to Drive
# ------------------------------------------------------------

test_results_path_33C = os.path.join(
    MODEL_OUTPUT_DIR,
    "33C_clinical_outer2_test_results.csv",
)
model_summary_path_33C = os.path.join(
    MODEL_OUTPUT_DIR,
    "33C_clinical_final_model_outer2.csv",
)
coefficient_path_33C = os.path.join(
    MODEL_OUTPUT_DIR,
    "33C_clinical_coefficients_outer2.csv",
)
evaluation_json_path_33C = os.path.join(
    MODEL_OUTPUT_DIR,
    "33C_clinical_final_evaluation_outer2.json",
)
evaluation_sha_path_33C = os.path.join(
    MODEL_OUTPUT_DIR,
    "33C_clinical_final_evaluation_outer2_SHA256.txt",
)

outer2_test_results_33C.to_csv(
    test_results_path_33C,
    index=False,
)
final_model_summary_33C.to_csv(
    model_summary_path_33C,
    index=False,
)
coefficient_table_33C.to_csv(
    coefficient_path_33C,
    index=False,
)

evaluation_configuration_33C = {
    "outer_fold": 2,
    "model_family": "parsimonious_clinical_elastic_net_logistic_regression",
    "selected_candidate": selected_candidate_33C,
    "selected_C": selected_C_33C,
    "selected_l1_ratio": selected_l1_ratio_33C,
    "training_patients": 46803,
    "training_hospitals": 158,
    "test_patients": 11688,
    "test_hospitals": 40,
    "hospital_overlap": 0,
    "locked_platt_intercept": platt_intercept_33C,
    "locked_platt_slope": platt_slope_33C,
    "processed_feature_columns": int(
        len(processed_feature_names_33C)
    ),
    "nonzero_coefficients": int(nonzero_coefficients_33C),
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_33C,
    "selection_sha256": selection_sha_33C,
    "secure_prediction_table": prediction_table_id_33C,
    "patient_level_prediction_written_to_drive": False,
    "analysis_role": "additional_post_hoc_clinical_baseline",
    "predictors": CLINICAL_PREDICTORS_33C,
}

with open(
    evaluation_json_path_33C,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        evaluation_configuration_33C,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(evaluation_json_path_33C, "rb") as file_handle:
    evaluation_sha_33C = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    evaluation_sha_path_33C,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(evaluation_sha_33C + "\n")

# ------------------------------------------------------------
# 24. Final outputs
# ------------------------------------------------------------

pooled_integrity_33C = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_33C),
            pooled_oof_33C["id_row"].nunique(),
            pooled_oof_33C["candidate_id"].nunique(),
            pooled_oof_33C["inner_fold"].nunique(),
            EXPECTED_SPLIT_33C["training_events"],
            (
                EXPECTED_SPLIT_33C["training_rows"]
                - EXPECTED_SPLIT_33C["training_events"]
            ),
            int(
                pooled_oof_33C.duplicated(
                    subset=["candidate_id", "id_row"]
                ).sum()
            ),
            int(pooled_oof_33C["prediction_raw"].isna().sum()),
            int(
                (~pooled_oof_33C["prediction_raw"].between(0, 1)).sum()
            ),
            pooled_load_method_33C,
        ],
    }
)

print("\n33C OUTER-FOLD-2 INNER CHECKPOINT SUMMARY")
display(checkpoint_summary_33C)

print("\n33C OUTER-FOLD-2 POOLED OOF INTEGRITY")
display(pooled_integrity_33C)

print("\n33C OUTER-FOLD-2 CANDIDATE RESULTS")
display(candidate_results_33C)

print("\n33C OUTER-FOLD-2 SELECTED MODEL")
display(selected_model_33C)

print("\n33C OUTER-FOLD-2 FINAL MODEL SUMMARY")
display(final_model_summary_33C)

print("\n33C OUTER-FOLD-2 TEST RESULTS")
display(outer2_test_results_33C)

print("\n33C OUTER-FOLD-2 BIGQUERY VERIFICATION")
display(prediction_verification_33C)

print("\n33C OUTER-FOLD-2 TOP 20 ABSOLUTE COEFFICIENTS")
display(coefficient_table_33C.head(20))

print("\nSelection SHA-256:")
print(selection_sha_33C)

print("\nEvaluation SHA-256:")
print(evaluation_sha_33C)

print("\nSaved:")
print(fit_audit_path_33C)
print(checkpoint_summary_path_33C)
print(candidate_results_path_33C)
print(selected_model_path_33C)
print(selection_json_path_33C)
print(selection_sha_path_33C)
print(test_results_path_33C)
print(model_summary_path_33C)
print(coefficient_path_33C)
print(evaluation_json_path_33C)
print(evaluation_sha_path_33C)

print(
    "\n33C PASS: Outer-fold-2 nested modelling "
    "and locked test evaluation are complete."
)
print(
    "All inner OOF and outer-test patient-level "
    "predictions were stored only in BigQuery."
)
print(
    "No patient-level prediction file was "
    "written to Google Drive."
)

_ = gc.collect()

In [ ]:
import os
import json
import hashlib
import gc

import numpy as np
import pandas as pd
from google.cloud import bigquery
from IPython.display import display

print("STARTING OUTER FOLD 2 POST-GUARD RESUME — CODE VERSION 33C-R1")

# ============================================================
# 33C-R1 — NO-REFIT RESUME AFTER STALE ROW-GUARD FIX
#
# The previous 33C run:
# - completed all 30 corrected inner fits;
# - confirmed zero inner ConvergenceWarnings;
# - selected/calibrated the model;
# - fitted the final outer-fold-2 model;
# - computed outer-fold-2 predictions;
# - stopped only because a stale Fold-1 row guard expected
#   11,688 instead of the correct Fold-2 count 11,691.
#
# This resume:
# - DOES NOT FIT any model;
# - DOES NOT retune or recalibrate;
# - validates the in-memory completed Fold-2 objects;
# - uploads the already-computed 11,691 predictions;
# - performs BigQuery verification;
# - saves aggregate/feature-level outputs only.
# ============================================================

EXPECTED_PROTOCOL_SHA_33CR1 = (
    "94b0abb218dbef4e349702ea2824ca4e"
    "31dfbe36c53efa05ba1a8bd1f38f835e"
)
EXPECTED_AMENDMENT_SHA_33CR1 = (
    "fcbcf857caa9aaad7ffd6691b4b61ac6"
    "503d09a6eee80db84bc2967ddf1f1a6a"
)
EXPECTED_CORRECTION_SHA_33CR1 = (
    "ea7ab2747aad0c20157da0988b2fbeb1"
    "6ce007b4c92bf785d9ef9d363862ecfa"
)

EXPECTED_ROWS_33CR1 = 11691
EXPECTED_EVENTS_33CR1 = 606
EXPECTED_NONEVENTS_33CR1 = 11085
EXPECTED_TRAINING_ROWS_33CR1 = 46800
EXPECTED_TRAINING_HOSPITALS_33CR1 = 158
EXPECTED_TEST_HOSPITALS_33CR1 = 40

required_objects_33CR1 = [
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
    "outer2_prediction_df_33C",
    "outer2_test_results_33C",
    "final_model_summary_33C",
    "coefficient_table_33C",
    "selected_candidate_33C",
    "selected_C_33C",
    "selected_l1_ratio_33C",
    "platt_intercept_33C",
    "platt_slope_33C",
    "processed_feature_names_33C",
    "nonzero_coefficients_33C",
    "selection_sha_33C",
    "CLINICAL_PREDICTORS_33C",
]

missing_objects_33CR1 = [
    name for name in required_objects_33CR1
    if name not in globals()
]

if missing_objects_33CR1:
    raise RuntimeError(
        "Bu no-refit resume aynı Colab runtime'ında çalıştırılmalıdır. "
        "Eksik nesneler: " + ", ".join(missing_objects_33CR1)
    )

# ------------------------------------------------------------
# 1. Verify locked protocol / amendments from Drive
# ------------------------------------------------------------

sha_files_33CR1 = [
    (
        "33A_locked_parsimonious_clinical_baseline_protocol_v1_SHA256.txt",
        EXPECTED_PROTOCOL_SHA_33CR1,
        "33A protocol",
    ),
    (
        "33A1_parsimonious_clinical_baseline_convergence_amendment_v1_SHA256.txt",
        EXPECTED_AMENDMENT_SHA_33CR1,
        "33A1 amendment",
    ),
    (
        "33A2_parsimonious_clinical_baseline_full_inner_refit_correction_v1_SHA256.txt",
        EXPECTED_CORRECTION_SHA_33CR1,
        "33A2 correction",
    ),
]

for filename, expected_sha, label in sha_files_33CR1:
    path = os.path.join(MODEL_OUTPUT_DIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    with open(path, "r", encoding="utf-8") as fh:
        observed_sha = fh.read().strip()
    if observed_sha != expected_sha:
        raise RuntimeError(
            f"{label} SHA mismatch: {observed_sha}"
        )

print("33A / 33A1 / 33A2 SHA guards: PASS")

# ------------------------------------------------------------
# 2. Verify that the completed in-memory prediction object is
#    exactly Fold 2 and was already computed before the stale guard
# ------------------------------------------------------------

pred_33CR1 = outer2_prediction_df_33C.copy()

if len(pred_33CR1) != EXPECTED_ROWS_33CR1:
    raise RuntimeError(
        f"Fold-2 prediction rows={len(pred_33CR1)}, "
        f"expected={EXPECTED_ROWS_33CR1}."
    )

if pred_33CR1["id_row"].duplicated().any():
    raise RuntimeError(
        "Fold-2 predictions contain duplicate id_row values."
    )

if set(pd.to_numeric(
    pred_33CR1["outer_fold"],
    errors="raise",
).astype(int).unique()) != {2}:
    raise RuntimeError(
        "Prediction object is not exclusively outer fold 2."
    )

if int(pd.to_numeric(
    pred_33CR1["label_stage23"],
    errors="raise",
).sum()) != EXPECTED_EVENTS_33CR1:
    raise RuntimeError(
        "Fold-2 event count is not 606."
    )

for column in ["prediction_raw", "prediction_platt"]:
    values = pd.to_numeric(
        pred_33CR1[column],
        errors="raise",
    ).to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise RuntimeError(
            f"Non-finite probabilities found in {column}."
        )

    if ((values < 0.0) | (values > 1.0)).any():
        raise RuntimeError(
            f"Invalid probabilities found in {column}."
        )

# Final model summary was already generated by the completed fit.
summary_row_33CR1 = final_model_summary_33C.iloc[0]

if int(summary_row_33CR1["training_patients"]) != EXPECTED_TRAINING_ROWS_33CR1:
    raise RuntimeError(
        "Final model training-patient count is not 46,800."
    )

if int(summary_row_33CR1["test_patients"]) != EXPECTED_ROWS_33CR1:
    raise RuntimeError(
        "Final model test-patient count is not 11,691."
    )

if int(summary_row_33CR1["training_hospitals"]) != EXPECTED_TRAINING_HOSPITALS_33CR1:
    raise RuntimeError(
        "Final model training-hospital count is not 158."
    )

if int(summary_row_33CR1["test_hospitals"]) != EXPECTED_TEST_HOSPITALS_33CR1:
    raise RuntimeError(
        "Final model test-hospital count is not 40."
    )

if int(summary_row_33CR1["convergence_warnings"]) != 0:
    raise RuntimeError(
        "Final Fold-2 model has a convergence warning."
    )

print("Completed Fold-2 in-memory fit/prediction integrity: PASS")
print("No model refit will be performed.")

# ------------------------------------------------------------
# 3. Upload already-computed predictions to secure BigQuery table
# ------------------------------------------------------------

prediction_table_id_33CR1 = (
    f"{TARGET_DATASET}."
    "model_clinical_lr_outer_predictions_outer2_v2"
)

prediction_load_config_33CR1 = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("prediction_platt", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("model_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("model_version", "STRING", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

print("\nUploading already-computed secure Fold-2 predictions:")
print(prediction_table_id_33CR1)

client.load_table_from_dataframe(
    pred_33CR1,
    prediction_table_id_33CR1,
    job_config=prediction_load_config_33CR1,
    location=BQ_LOCATION,
).result()

# ------------------------------------------------------------
# 4. Exact BigQuery verification
# ------------------------------------------------------------

verification_sql_33CR1 = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL) AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL) AS missing_platt_predictions,
  COUNTIF(prediction_raw < 0 OR prediction_raw > 1) AS invalid_raw_predictions,
  COUNTIF(prediction_platt < 0 OR prediction_platt > 1) AS invalid_platt_predictions,
  MIN(prediction_raw) AS minimum_raw_probability,
  MAX(prediction_raw) AS maximum_raw_probability,
  MIN(prediction_platt) AS minimum_platt_probability,
  MAX(prediction_platt) AS maximum_platt_probability
FROM `{prediction_table_id_33CR1}`
"""

prediction_verification_33CR1 = client.query(
    verification_sql_33CR1,
    location=BQ_LOCATION,
).to_dataframe()

vr = prediction_verification_33CR1.iloc[0]

expected_verification_33CR1 = {
    "prediction_rows": EXPECTED_ROWS_33CR1,
    "distinct_rows": EXPECTED_ROWS_33CR1,
    "outer_folds": 1,
    "minimum_outer_fold": 2,
    "maximum_outer_fold": 2,
    "events": EXPECTED_EVENTS_33CR1,
    "nonevents": EXPECTED_NONEVENTS_33CR1,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected in expected_verification_33CR1.items():
    actual = int(vr[field])
    if actual != expected:
        raise RuntimeError(
            f"{field}: observed={actual}, expected={expected}"
        )

# ------------------------------------------------------------
# 5. Save aggregate / feature-level outputs only
# ------------------------------------------------------------

test_results_path_33CR1 = os.path.join(
    MODEL_OUTPUT_DIR,
    "33C_clinical_outer2_test_results.csv",
)
model_summary_path_33CR1 = os.path.join(
    MODEL_OUTPUT_DIR,
    "33C_clinical_final_model_outer2.csv",
)
coefficient_path_33CR1 = os.path.join(
    MODEL_OUTPUT_DIR,
    "33C_clinical_coefficients_outer2.csv",
)
evaluation_json_path_33CR1 = os.path.join(
    MODEL_OUTPUT_DIR,
    "33C_clinical_final_evaluation_outer2.json",
)
evaluation_sha_path_33CR1 = os.path.join(
    MODEL_OUTPUT_DIR,
    "33C_clinical_final_evaluation_outer2_SHA256.txt",
)

outer2_test_results_33C.to_csv(
    test_results_path_33CR1,
    index=False,
)
final_model_summary_33C.to_csv(
    model_summary_path_33CR1,
    index=False,
)
coefficient_table_33C.to_csv(
    coefficient_path_33CR1,
    index=False,
)

evaluation_configuration_33CR1 = {
    "outer_fold": 2,
    "analysis_status": "post_hoc_exploratory_corrected",
    "analysis_role": "additional_post_hoc_clinical_baseline",
    "model_family": "parsimonious_clinical_elastic_net_logistic_regression",
    "selected_candidate": selected_candidate_33C,
    "selected_C": float(selected_C_33C),
    "selected_l1_ratio": float(selected_l1_ratio_33C),
    "training_patients": EXPECTED_TRAINING_ROWS_33CR1,
    "training_hospitals": EXPECTED_TRAINING_HOSPITALS_33CR1,
    "test_patients": EXPECTED_ROWS_33CR1,
    "test_hospitals": EXPECTED_TEST_HOSPITALS_33CR1,
    "test_events": EXPECTED_EVENTS_33CR1,
    "test_nonevents": EXPECTED_NONEVENTS_33CR1,
    "hospital_overlap": 0,
    "locked_platt_intercept": float(platt_intercept_33C),
    "locked_platt_slope": float(platt_slope_33C),
    "processed_feature_columns": int(
        len(processed_feature_names_33C)
    ),
    "nonzero_coefficients": int(nonzero_coefficients_33C),
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_33CR1,
    "numerical_amendment_sha256": EXPECTED_AMENDMENT_SHA_33CR1,
    "full_inner_refit_correction_sha256": EXPECTED_CORRECTION_SHA_33CR1,
    "selection_sha256": selection_sha_33C,
    "secure_prediction_table": prediction_table_id_33CR1,
    "model_refit_in_this_resume": False,
    "retuning_in_this_resume": False,
    "recalibration_in_this_resume": False,
    "bigquery_dml_used": False,
    "patient_level_prediction_written_to_drive": False,
    "predictors": list(CLINICAL_PREDICTORS_33C),
    "resume_reason": (
        "Previous 33C run stopped after prediction computation because "
        "a stale Fold-1 row-count guard expected 11,688 instead of the "
        "correct Fold-2 count 11,691."
    ),
}

with open(
    evaluation_json_path_33CR1,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        evaluation_configuration_33CR1,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    evaluation_json_path_33CR1,
    "rb",
) as fh:
    evaluation_sha_33CR1 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    evaluation_sha_path_33CR1,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(evaluation_sha_33CR1 + "\n")

# ------------------------------------------------------------
# 6. Display
# ------------------------------------------------------------

print("\n33C-R1 OUTER-FOLD-2 FINAL MODEL SUMMARY")
display(final_model_summary_33C)

print("\n33C-R1 OUTER-FOLD-2 TEST RESULTS")
display(outer2_test_results_33C)

print("\n33C-R1 OUTER-FOLD-2 BIGQUERY VERIFICATION")
display(prediction_verification_33CR1)

print("\n33C-R1 OUTER-FOLD-2 TOP ABSOLUTE COEFFICIENTS")
display(coefficient_table_33C.head(20))

print("\nSelection SHA-256:")
print(selection_sha_33C)

print("\nEvaluation SHA-256:")
print(evaluation_sha_33CR1)

print("\nSaved:")
print(test_results_path_33CR1)
print(model_summary_path_33CR1)
print(coefficient_path_33CR1)
print(evaluation_json_path_33CR1)
print(evaluation_sha_path_33CR1)

print(
    "\n33C-R1 PASS: Outer-fold-2 post-guard resume completed "
    "without any model refitting, retuning, or recalibration."
)
print(
    "11,691 already-computed patient predictions were verified "
    "and stored only in secure BigQuery."
)
print(
    "No patient-level prediction file was written to Google Drive."
)

_ = gc.collect()

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from sklearn.exceptions import ConvergenceWarning

from IPython.display import display

# ============================================================
# 33B — PARSIMONIOUS CLINICAL BASELINE OUTER FOLD 3
#
# Resume-safe:
# - Each inner fold is stored in a separate BigQuery table.
# - Completed inner folds are automatically skipped.
#
# BigQuery free-tier compatible:
# - No DELETE / INSERT / UPDATE / MERGE is used.
#
# Privacy:
# - Patient-level predictions are stored only in BigQuery.
# - No patient-level prediction file is written to Drive.
# ============================================================

print("STARTING PARSIMONIOUS CLINICAL BASELINE OUTER FOLD 3 — CODE VERSION 33D")

OUTER_FOLD_33D = 3
MODEL_RANDOM_SEED_33D = 20260721

EXPECTED_PROTOCOL_SHA_33D = (
    "94b0abb218dbef4e349702ea2824ca4e"
    "31dfbe36c53efa05ba1a8bd1f38f835e"
)

EXPECTED_AMENDMENT_SHA_33D = (
    "fcbcf857caa9aaad7ffd6691b4b61ac6"
    "503d09a6eee80db84bc2967ddf1f1a6a"
)

EXPECTED_CORRECTION_SHA_33D = (
    "ea7ab2747aad0c20157da0988b2fbeb1"
    "6ce007b4c92bf785d9ef9d363862ecfa"
)

EXPECTED_SPLIT_33D = {
    "training_rows": 46755,
    "test_rows": 11736,
    "training_hospitals": 158,
    "test_hospitals": 40,
    "training_events": 2424,
    "test_events": 608,
}

EXPECTED_INNER_33D = {
    1: {"validation_rows": 7934, "validation_events": 380},
    2: {"validation_rows": 5835, "validation_events": 307},
    3: {"validation_rows": 8356, "validation_events": 458},
    4: {"validation_rows": 13173, "validation_events": 691},
    5: {"validation_rows": 11457, "validation_events": 588},
}

CLINICAL_PREDICTORS_33D = [
    "x_age_years",
    "x_sex",
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
    "x_lab_bun_last",
    "x_vital_respiratory_rate_last",
    "x_vital_noninvasive_systolic_bp_last",
]

CLINICAL_NUMERIC_33D = [
    "x_age_years",
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
    "x_lab_bun_last",
    "x_vital_respiratory_rate_last",
    "x_vital_noninvasive_systolic_bp_last",
]

CLINICAL_CATEGORICAL_33D = ["x_sex"]

# ------------------------------------------------------------
# 1. Required runtime objects
# ------------------------------------------------------------

required_objects_33D = [
    "core_df_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_33D = [
    name for name in required_objects_33D if name not in globals()
]

if missing_objects_33D:
    raise RuntimeError(
        "Eksik RAM nesneleri var: "
        + ", ".join(missing_objects_33D)
        + ". Önce 07A ve 07B hücrelerini çalıştır."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"58.491 satır bekleniyordu; {len(core_df_07B)} bulundu."
    )

missing_predictors_33D = [
    column for column in CLINICAL_PREDICTORS_33D
    if column not in core_df_07B.columns
]

if missing_predictors_33D:
    raise RuntimeError(
        "Kilitli klinik predictor(lar) core_df_07B içinde yok: "
        + ", ".join(missing_predictors_33D)
    )

# ------------------------------------------------------------
# 2. Locked protocol SHA check
# ------------------------------------------------------------

protocol_sha_path_33D = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A_locked_parsimonious_clinical_baseline_protocol_v1_SHA256.txt",
)

if not os.path.exists(protocol_sha_path_33D):
    raise FileNotFoundError(
        "Model protokolü SHA dosyası bulunamadı: "
        + protocol_sha_path_33D
    )

with open(protocol_sha_path_33D, "r", encoding="utf-8") as file_handle:
    observed_protocol_sha_33D = file_handle.read().strip()

if observed_protocol_sha_33D != EXPECTED_PROTOCOL_SHA_33D:
    raise RuntimeError(
        "Kilitli model protokolü SHA değeri değişmiş: "
        + observed_protocol_sha_33D
    )

amendment_sha_path_33D = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A1_parsimonious_clinical_baseline_convergence_amendment_v1_SHA256.txt",
)

if not os.path.exists(amendment_sha_path_33D):
    raise FileNotFoundError(
        "Önce 33A1 convergence amendment scriptini çalıştır: "
        + amendment_sha_path_33D
    )

with open(amendment_sha_path_33D, "r", encoding="utf-8") as file_handle:
    observed_amendment_sha_33D = file_handle.read().strip()

if observed_amendment_sha_33D != EXPECTED_AMENDMENT_SHA_33D:
    raise RuntimeError(
        "33A1 amendment SHA değeri değişmiş: "
        + observed_amendment_sha_33D
    )

print("33A protocol SHA guard: PASS")
print("33A1 convergence amendment SHA guard: PASS")

correction_sha_path_33D = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A2_parsimonious_clinical_baseline_full_inner_refit_correction_v1_SHA256.txt",
)

if not os.path.exists(correction_sha_path_33D):
    raise FileNotFoundError(
        "Önce 33A2 correction scriptini çalıştır: "
        + correction_sha_path_33D
    )

with open(correction_sha_path_33D, "r", encoding="utf-8") as file_handle:
    observed_correction_sha_33D = file_handle.read().strip()

if observed_correction_sha_33D != EXPECTED_CORRECTION_SHA_33D:
    raise RuntimeError(
        "33A2 correction SHA değeri değişmiş: "
        + observed_correction_sha_33D
    )

print("33A2 full inner-refit correction SHA guard: PASS")

# ------------------------------------------------------------
# 3. Locked inner-hospital mapping
# ------------------------------------------------------------

inner_mapping_path_33D = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_33D):
    raise FileNotFoundError(
        "Kilitli iç kat haritası bulunamadı: "
        + inner_mapping_path_33D
    )

inner_mapping_all_33D = pd.read_csv(
    inner_mapping_path_33D,
    dtype={"group_hospital": str},
)

inner_mapping_part_33D = (
    inner_mapping_all_33D.loc[
        inner_mapping_all_33D["outer_fold"].astype(int) == OUTER_FOLD_33D,
        ["group_hospital", "inner_fold"],
    ]
    .copy()
)

inner_mapping_part_33D["group_hospital"] = (
    inner_mapping_part_33D["group_hospital"].astype(str)
)
inner_mapping_part_33D["inner_fold"] = (
    inner_mapping_part_33D["inner_fold"].astype(int)
)

if len(inner_mapping_part_33D) != 158:
    raise RuntimeError(
        "Dış kat 1 eğitim kümesi için 158 hastane ataması bekleniyordu."
    )

if inner_mapping_part_33D["group_hospital"].duplicated().any():
    raise RuntimeError("İç kat haritasında yinelenen hastane var.")

hospital_to_inner_fold_33D = dict(
    zip(
        inner_mapping_part_33D["group_hospital"],
        inner_mapping_part_33D["inner_fold"],
    )
)

# ------------------------------------------------------------
# 4. Prepare model matrices
# ------------------------------------------------------------

X_all_33D = core_df_07B[CLINICAL_PREDICTORS_33D].copy()

for column in CLINICAL_NUMERIC_33D:
    X_all_33D[column] = pd.to_numeric(
        X_all_33D[column], errors="coerce"
    ).astype("float64")

for column in CLINICAL_CATEGORICAL_33D:
    category_series = X_all_33D[column].astype("object")
    X_all_33D[column] = category_series.where(
        pd.notna(category_series), np.nan
    )

outer_fold_vector_33D = (
    core_df_07B["outer_fold"].astype(int).to_numpy()
)

outer_training_mask_33D = outer_fold_vector_33D != OUTER_FOLD_33D
outer_test_mask_33D = outer_fold_vector_33D == OUTER_FOLD_33D

X_outer_training_33D = (
    X_all_33D.loc[outer_training_mask_33D].reset_index(drop=True)
)
X_outer_test_33D = (
    X_all_33D.loc[outer_test_mask_33D].reset_index(drop=True)
)

outer_training_meta_33D = (
    core_df_07B.loc[
        outer_training_mask_33D,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_33D = (
    core_df_07B.loc[
        outer_test_mask_33D,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [outer_training_meta_33D, outer_test_meta_33D]:
    dataframe["id_row"] = dataframe["id_row"].astype(str)
    dataframe["group_hospital"] = dataframe["group_hospital"].astype(str)
    dataframe["label_stage23"] = dataframe["label_stage23"].astype(int)

y_outer_training_33D = (
    outer_training_meta_33D["label_stage23"].to_numpy(dtype=np.int8)
)
y_outer_test_33D = (
    outer_test_meta_33D["label_stage23"].to_numpy(dtype=np.int8)
)

groups_outer_training_33D = (
    outer_training_meta_33D["group_hospital"].to_numpy(dtype=str)
)

training_hospitals_33D = set(
    outer_training_meta_33D["group_hospital"]
)
test_hospitals_33D = set(
    outer_test_meta_33D["group_hospital"]
)
hospital_overlap_33D = training_hospitals_33D & test_hospitals_33D

if hospital_overlap_33D:
    raise RuntimeError("Dış eğitim ve test hastaneleri çakışıyor.")

actual_split_33D = {
    "training_rows": len(X_outer_training_33D),
    "test_rows": len(X_outer_test_33D),
    "training_hospitals": len(training_hospitals_33D),
    "test_hospitals": len(test_hospitals_33D),
    "training_events": int(y_outer_training_33D.sum()),
    "test_events": int(y_outer_test_33D.sum()),
}

for metric, expected_value in EXPECTED_SPLIT_33D.items():
    actual_value = actual_split_33D[metric]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: bulunan={actual_value}, beklenen={expected_value}"
        )

inner_fold_vector_33D = np.array(
    [
        hospital_to_inner_fold_33D.get(hospital, -1)
        for hospital in groups_outer_training_33D
    ],
    dtype=int,
)

if (inner_fold_vector_33D == -1).any():
    raise RuntimeError(
        "Bazı dış eğitim hastanelerine iç kat atanmadı."
    )

if set(np.unique(inner_fold_vector_33D)) != {1, 2, 3, 4, 5}:
    raise RuntimeError("İç kat değerleri 1–5 değil.")

for inner_fold, expected in EXPECTED_INNER_33D.items():
    validation_mask = inner_fold_vector_33D == inner_fold
    observed_rows = int(validation_mask.sum())
    observed_events = int(y_outer_training_33D[validation_mask].sum())

    if observed_rows != expected["validation_rows"]:
        raise RuntimeError(
            f"Inner {inner_fold} validation_rows: "
            f"bulunan={observed_rows}, "
            f"beklenen={expected['validation_rows']}"
        )

    if observed_events != expected["validation_events"]:
        raise RuntimeError(
            f"Inner {inner_fold} validation_events: "
            f"bulunan={observed_events}, "
            f"beklenen={expected['validation_events']}"
        )

# ------------------------------------------------------------
# 5. Locked candidate grid
# ------------------------------------------------------------

candidate_grid_33D = [
    {"candidate_id": "CLIN01", "C": 0.03, "l1_ratio": 0.00},
    {"candidate_id": "CLIN02", "C": 0.10, "l1_ratio": 0.00},
    {"candidate_id": "CLIN03", "C": 0.30, "l1_ratio": 0.00},
    {"candidate_id": "CLIN04", "C": 0.10, "l1_ratio": 0.25},
    {"candidate_id": "CLIN05", "C": 0.30, "l1_ratio": 0.25},
    {"candidate_id": "CLIN06", "C": 0.30, "l1_ratio": 0.50},
]

# ------------------------------------------------------------
# 6. Preprocessing and model factories
# ------------------------------------------------------------

def make_preprocessor_33D():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
            (
                "scaler",
                StandardScaler(with_mean=False),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                CLINICAL_NUMERIC_33D,
            ),
            (
                "categorical",
                categorical_pipeline,
                CLINICAL_CATEGORICAL_33D,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_model_33D(C_value, l1_ratio_value):
    return LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=float(C_value),
        l1_ratio=float(l1_ratio_value),
        class_weight=None,
        max_iter=20000,
        tol=1e-4,
        random_state=MODEL_RANDOM_SEED_33D,
    )


def checkpoint_table_id_33D(inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_clinical_lr_inner_oof_outer3_inner{inner_fold}_v2"
    )

# ------------------------------------------------------------
# 7. BigQuery checkpoint verification
# ------------------------------------------------------------

def verify_checkpoint_33D(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):
    table_id = checkpoint_table_id_33D(inner_fold)

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS distinct_id_count,
      COUNT(DISTINCT candidate_id) AS candidate_count,
      COUNT(DISTINCT outer_fold) AS outer_fold_count,
      COUNT(DISTINCT inner_fold) AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(candidate_id, '|', id_row)
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(
        prediction_raw < 0 OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(sql, location=BQ_LOCATION).to_dataframe()
    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows * len(candidate_grid_33D)
    )
    expected_positive_rows = (
        expected_validation_events * len(candidate_grid_33D)
    )
    expected_negative_rows = (
        (expected_validation_rows - expected_validation_events)
        * len(candidate_grid_33D)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": expected_validation_rows,
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": expected_total_rows,
        "positive_prediction_rows": expected_positive_rows,
        "negative_prediction_rows": expected_negative_rows,
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": 3,
        "maximum_outer_fold": 3,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failure_items = []

    for field, expected_value in expected_values.items():
        actual_value = int(row[field])
        if actual_value != expected_value:
            complete = False
            failure_items.append(
                f"{field}={actual_value}, expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failure_items),
        "check": check,
        "row": row,
    }

# ------------------------------------------------------------
# 8. BigQuery load schema
# ------------------------------------------------------------

checkpoint_load_config_33D = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("inner_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("candidate_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

# ------------------------------------------------------------
# 9. Aggregate fit-audit file
# ------------------------------------------------------------

fit_audit_columns_33D = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "C",
    "l1_ratio",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "convergence_warnings",
    "elapsed_seconds",
]

fit_audit_path_33D = os.path.join(
    MODEL_OUTPUT_DIR,
    "33D_clinical_inner_fit_audit_outer3.csv",
)

if os.path.exists(fit_audit_path_33D):
    fit_audit_33D = pd.read_csv(fit_audit_path_33D)
else:
    fit_audit_33D = pd.DataFrame(columns=fit_audit_columns_33D)

for column in fit_audit_columns_33D:
    if column not in fit_audit_33D.columns:
        fit_audit_33D[column] = np.nan

fit_audit_33D = fit_audit_33D[fit_audit_columns_33D].copy()

# ------------------------------------------------------------
# 10. Train six candidates in five locked inner folds
# ------------------------------------------------------------

for inner_fold in range(1, 6):
    inner_training_mask = inner_fold_vector_33D != inner_fold
    inner_validation_mask = inner_fold_vector_33D == inner_fold

    training_rows = int(inner_training_mask.sum())
    validation_rows = int(inner_validation_mask.sum())
    training_events = int(
        y_outer_training_33D[inner_training_mask].sum()
    )
    validation_events = int(
        y_outer_training_33D[inner_validation_mask].sum()
    )

    expected_inner = EXPECTED_INNER_33D[inner_fold]
    expected_training_rows = (
        EXPECTED_SPLIT_33D["training_rows"]
        - expected_inner["validation_rows"]
    )
    expected_training_events = (
        EXPECTED_SPLIT_33D["training_events"]
        - expected_inner["validation_events"]
    )

    if training_rows != expected_training_rows:
        raise RuntimeError(
            f"Inner {inner_fold} training_rows: "
            f"bulunan={training_rows}, "
            f"beklenen={expected_training_rows}"
        )

    if training_events != expected_training_events:
        raise RuntimeError(
            f"Inner {inner_fold} training_events: "
            f"bulunan={training_events}, "
            f"beklenen={expected_training_events}"
        )

    existing_check = verify_checkpoint_33D(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if existing_check["complete"]:
        print(
            f"Outer 3 / inner {inner_fold}: "
            "permanent checkpoint already complete; "
            "skipping model fitting."
        )
        continue

    training_hospital_set = set(
        groups_outer_training_33D[inner_training_mask]
    )
    validation_hospital_set = set(
        groups_outer_training_33D[inner_validation_mask]
    )

    if training_hospital_set & validation_hospital_set:
        raise RuntimeError(
            f"Inner fold {inner_fold}: hastane çakışması bulundu."
        )

    print(f"\nOuter 3 / inner {inner_fold}")
    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_33D()
    preprocessing_started = time.time()

    X_inner_training_processed = preprocessor.fit_transform(
        X_outer_training_33D.loc[inner_training_mask]
    )
    X_inner_validation_processed = preprocessor.transform(
        X_outer_training_33D.loc[inner_validation_mask]
    )

    preprocessing_elapsed = time.time() - preprocessing_started

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "İşlenmiş eğitim ve doğrulama sütun sayıları farklı."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_elapsed, 2),
    )

    y_inner_training = y_outer_training_33D[inner_training_mask]
    y_inner_validation = y_outer_training_33D[inner_validation_mask]

    validation_ids = (
        outer_training_meta_33D.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_33D:
        candidate_id = candidate["candidate_id"]

        print(
            "  Fitting",
            candidate_id,
            "| C =",
            candidate["C"],
            "| l1_ratio =",
            candidate["l1_ratio"],
        )

        model = make_model_33D(
            candidate["C"],
            candidate["l1_ratio"],
        )

        fitting_started = time.time()

        with warnings.catch_warnings(record=True) as warning_records:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(
                X_inner_training_processed,
                y_inner_training,
            )

        fitting_elapsed = time.time() - fitting_started

        convergence_warning_count = sum(
            issubclass(warning.category, ConvergenceWarning)
            for warning in warning_records
        )

        validation_probabilities = model.predict_proba(
            X_inner_validation_processed
        )[:, 1]

        if np.isnan(validation_probabilities).any():
            raise RuntimeError(
                f"{candidate_id}, inner {inner_fold}: eksik tahmin."
            )

        if not np.all(
            (validation_probabilities >= 0)
            & (validation_probabilities <= 1)
        ):
            raise RuntimeError(
                f"{candidate_id}: geçersiz olasılık."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        3,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": y_inner_validation.astype(np.int64),
                    "prediction_raw": validation_probabilities.astype(
                        np.float64
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": 3,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "C": candidate["C"],
                "l1_ratio": candidate["l1_ratio"],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": validation_events,
                "processed_columns": int(
                    X_inner_training_processed.shape[1]
                ),
                "convergence_warnings": int(
                    convergence_warning_count
                ),
                "elapsed_seconds": float(fitting_elapsed),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows * len(candidate_grid_33D)
    )

    if len(checkpoint_df) != expected_checkpoint_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: checkpoint satır sayısı hatalı."
        )

    if checkpoint_df.duplicated(
        subset=["id_row", "candidate_id"]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "yinelenen aday–hasta tahmini var."
        )

    target_checkpoint_table = checkpoint_table_id_33D(inner_fold)

    print(
        "Uploading permanent checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_33D,
        location=BQ_LOCATION,
    ).result()

    new_audit_df = pd.DataFrame(current_audit_rows)

    if len(fit_audit_33D) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_33D["outer_fold"],
                    errors="coerce",
                )
                == 3
            )
            & (
                pd.to_numeric(
                    fit_audit_33D["inner_fold"],
                    errors="coerce",
                )
                == inner_fold
            )
        )
        fit_audit_33D = fit_audit_33D.loc[keep_mask].copy()

    if fit_audit_33D.empty:
        fit_audit_33D = new_audit_df.copy()
    else:
        fit_audit_33D = pd.concat(
            [fit_audit_33D, new_audit_df],
            ignore_index=True,
        )

    fit_audit_33D = (
        fit_audit_33D[fit_audit_columns_33D]
        .sort_values(
            ["outer_fold", "inner_fold", "candidate_id"]
        )
        .reset_index(drop=True)
    )

    fit_audit_33D.to_csv(
        fit_audit_path_33D,
        index=False,
    )

    completed_check = verify_checkpoint_33D(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint doğrulanamadı: "
            + completed_check["reason"]
        )

    print(
        f"Outer 3 / inner {inner_fold}: "
        "permanent checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 11. Final checkpoint summary
# ------------------------------------------------------------

checkpoint_summary_rows_33D = []

for inner_fold in range(1, 6):
    validation_mask = inner_fold_vector_33D == inner_fold
    validation_rows = int(validation_mask.sum())
    validation_events = int(
        y_outer_training_33D[validation_mask].sum()
    )

    final_check = verify_checkpoint_33D(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "son checkpoint denetimi başarısız. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_33D.append(
        {
            "outer_fold": 3,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(row["row_count"]),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(row["candidate_count"]),
            "positive_prediction_rows": int(
                row["positive_prediction_rows"]
            ),
            "negative_prediction_rows": int(
                row["negative_prediction_rows"]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check["table_id"],
        }
    )

checkpoint_summary_33D = (
    pd.DataFrame(checkpoint_summary_rows_33D)
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_33D["distinct_validation_patients"].sum()
) != EXPECTED_SPLIT_33D["training_rows"]:
    raise RuntimeError(
        "Toplam doğrulama hasta sayısı 46.803 değil."
    )

expected_total_oof_rows_33D = (
    EXPECTED_SPLIT_33D["training_rows"]
    * len(candidate_grid_33D)
)

if int(checkpoint_summary_33D["checkpoint_rows"].sum()) != (
    expected_total_oof_rows_33D
):
    raise RuntimeError("Toplam OOF tahmin satırı hatalı.")

checkpoint_summary_path_33D = os.path.join(
    MODEL_OUTPUT_DIR,
    "33D_clinical_outer3_inner_checkpoint_summary.csv",
)
checkpoint_summary_33D.to_csv(
    checkpoint_summary_path_33D,
    index=False,
)

# ------------------------------------------------------------
# 12. Pool all inner OOF predictions
# ------------------------------------------------------------

checkpoint_tables_33D = [
    checkpoint_table_id_33D(inner_fold)
    for inner_fold in range(1, 6)
]

union_parts_33D = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_33D
]

SQL_LOAD_POOLED_OOF_33D = "\nUNION ALL\n".join(union_parts_33D)

print("\nLoading pooled outer-fold-3 inner OOF predictions...")

query_job_33D = client.query(
    SQL_LOAD_POOLED_OOF_33D,
    location=BQ_LOCATION,
)

try:
    pooled_oof_33D = query_job_33D.to_dataframe(
        create_bqstorage_client=True
    )
    pooled_load_method_33D = "BigQuery Storage API"
except Exception as fast_path_error_33D:
    print(
        "Storage API unavailable; using standard BigQuery download."
    )
    print("Message:", type(fast_path_error_33D).__name__)
    pooled_oof_33D = query_job_33D.to_dataframe(
        create_bqstorage_client=False
    )
    pooled_load_method_33D = "Standard BigQuery API"

pooled_oof_33D["id_row"] = pooled_oof_33D["id_row"].astype(str)
pooled_oof_33D["candidate_id"] = pooled_oof_33D["candidate_id"].astype(str)

for column in ["outer_fold", "inner_fold", "label_stage23"]:
    pooled_oof_33D[column] = pd.to_numeric(
        pooled_oof_33D[column], errors="raise"
    ).astype(int)

pooled_oof_33D["prediction_raw"] = pd.to_numeric(
    pooled_oof_33D["prediction_raw"], errors="raise"
).astype(float)

# ------------------------------------------------------------
# 13. Pooled OOF integrity checks
# ------------------------------------------------------------

if len(pooled_oof_33D) != expected_total_oof_rows_33D:
    raise RuntimeError("Pooled OOF satır sayısı hatalı.")

if set(pooled_oof_33D["outer_fold"].unique()) != {3}:
    raise RuntimeError("Pooled OOF içinde dış kat 4 dışında kayıt var.")

if set(pooled_oof_33D["inner_fold"].unique()) != {1, 2, 3, 4, 5}:
    raise RuntimeError("Pooled OOF iç katları 1–5 değil.")

if pooled_oof_33D.duplicated(
    subset=["candidate_id", "id_row"]
).any():
    raise RuntimeError(
        "Pooled OOF içinde yinelenen aday–hasta tahmini bulundu."
    )

if pooled_oof_33D["prediction_raw"].isna().any():
    raise RuntimeError("Pooled OOF içinde eksik tahmin var.")

if not pooled_oof_33D["prediction_raw"].between(0, 1).all():
    raise RuntimeError(
        "Pooled OOF içinde geçersiz olasılık değeri var."
    )

expected_candidate_ids_33D = {
    "CLIN01",
    "CLIN02",
    "CLIN03",
    "CLIN04",
    "CLIN05",
    "CLIN06",
}

if set(pooled_oof_33D["candidate_id"].unique()) != (
    expected_candidate_ids_33D
):
    raise RuntimeError("Altı kilitli aday bulunmuyor.")

candidate_patient_counts_33D = (
    pooled_oof_33D.groupby("candidate_id")["id_row"].nunique()
)

if not (
    candidate_patient_counts_33D
    == EXPECTED_SPLIT_33D["training_rows"]
).all():
    raise RuntimeError(
        "Her aday için 46.803 farklı OOF hastası yok."
    )

candidate_event_counts_33D = (
    pooled_oof_33D.groupby("candidate_id")["label_stage23"].sum()
)

if not (
    candidate_event_counts_33D
    == EXPECTED_SPLIT_33D["training_events"]
).all():
    raise RuntimeError("Her aday için 2.426 olay yok.")

patient_label_consistency_33D = (
    pooled_oof_33D.groupby("id_row")["label_stage23"].nunique()
)
if (patient_label_consistency_33D > 1).any():
    raise RuntimeError(
        "Aynı hastanın adaylar arasında outcome etiketi farklı."
    )

patient_inner_fold_consistency_33D = (
    pooled_oof_33D.groupby("id_row")["inner_fold"].nunique()
)
if (patient_inner_fold_consistency_33D > 1).any():
    raise RuntimeError(
        "Aynı hasta birden fazla iç doğrulama katında bulundu."
    )

# ------------------------------------------------------------
# 14. Metric helpers
# ------------------------------------------------------------

def probability_metrics_33D(y_true, probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(roc_auc_score(y_true, probabilities)),
        "auprc": float(average_precision_score(y_true, probabilities)),
        "brier": float(brier_score_loss(y_true, probabilities)),
        "log_loss": float(
            log_loss(y_true, probabilities, labels=[0, 1])
        ),
        "mean_predicted_risk": float(probabilities.mean()),
        "observed_event_rate": float(np.mean(y_true)),
    }


def probability_logit_33D(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        probabilities / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_33D(y_true, probabilities):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        probability_logit_33D(probabilities),
        y_true,
    )
    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 15. Candidate pooled-OOF performance
# ------------------------------------------------------------

candidate_result_rows_33D = []

for candidate in candidate_grid_33D:
    candidate_id = candidate["candidate_id"]

    candidate_oof = (
        pooled_oof_33D.loc[
            pooled_oof_33D["candidate_id"] == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_33D(
        candidate_oof["label_stage23"].to_numpy(dtype=int),
        candidate_oof["prediction_raw"].to_numpy(dtype=float),
    )

    fit_part = fit_audit_33D.loc[
        (
            pd.to_numeric(
                fit_audit_33D["outer_fold"],
                errors="coerce",
            )
            == 3
        )
        & (
            fit_audit_33D["candidate_id"].astype(str)
            == candidate_id
        )
    ]

    convergence_warnings = (
        int(
            pd.to_numeric(
                fit_part["convergence_warnings"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    elapsed_seconds = (
        float(
            pd.to_numeric(
                fit_part["elapsed_seconds"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_33D.append(
        {
            "candidate_id": candidate_id,
            "C": candidate["C"],
            "l1_ratio": candidate["l1_ratio"],
            **metrics,
            "convergence_warnings": convergence_warnings,
            "elapsed_seconds": elapsed_seconds,
        }
    )

candidate_results_33D = pd.DataFrame(candidate_result_rows_33D)

total_inner_convergence_warnings_33D = int(
    pd.to_numeric(
        candidate_results_33D["convergence_warnings"],
        errors="coerce",
    ).fillna(0).sum()
)

if total_inner_convergence_warnings_33D != 0:
    raise RuntimeError(
        "33B-R2 stopped BEFORE candidate selection and BEFORE outer-test "
        "evaluation because at least one inner candidate fit still emitted "
        f"a ConvergenceWarning under max_iter=20000. "
        f"Total warnings={total_inner_convergence_warnings_33D}. "
        "No test result may be accessed."
    )

print(
    "All 30 corrected inner candidate fits converged with zero warnings: PASS"
)

candidate_results_33D = (
    candidate_results_33D.sort_values(
        ["auprc", "auroc", "brier", "candidate_id"],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)

candidate_results_33D["selection_rank"] = np.arange(
    1,
    len(candidate_results_33D) + 1,
)

best_row_33D = candidate_results_33D.iloc[0]
selected_candidate_33D = str(best_row_33D["candidate_id"])
selected_C_33D = float(best_row_33D["C"])
selected_l1_ratio_33D = float(best_row_33D["l1_ratio"])

# ------------------------------------------------------------
# 16. Platt calibration on selected pooled inner OOF
# ------------------------------------------------------------

selected_oof_33D = (
    pooled_oof_33D.loc[
        pooled_oof_33D["candidate_id"] == selected_candidate_33D
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_33D = selected_oof_33D[
    "label_stage23"
].to_numpy(dtype=int)
selected_oof_probability_33D = selected_oof_33D[
    "prediction_raw"
].to_numpy(dtype=float)

platt_calibrator_33D = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_33D.fit(
    probability_logit_33D(selected_oof_probability_33D),
    selected_oof_y_33D,
)

platt_intercept_33D = float(platt_calibrator_33D.intercept_[0])
platt_slope_33D = float(platt_calibrator_33D.coef_[0][0])

if (
    not np.isfinite(platt_intercept_33D)
    or not np.isfinite(platt_slope_33D)
    or platt_slope_33D <= 0
):
    raise RuntimeError("Platt kalibrasyon katsayıları geçersiz.")

selected_model_33D = pd.DataFrame(
    [
        {
            "outer_fold": 3,
            "selected_candidate": selected_candidate_33D,
            "selected_C": selected_C_33D,
            "selected_l1_ratio": selected_l1_ratio_33D,
            "selection_metric_primary": "pooled_inner_oof_auprc",
            "inner_oof_auprc": float(best_row_33D["auprc"]),
            "inner_oof_auroc": float(best_row_33D["auroc"]),
            "inner_oof_brier": float(best_row_33D["brier"]),
            "inner_oof_log_loss": float(best_row_33D["log_loss"]),
            "inner_oof_mean_predicted_risk": float(
                best_row_33D["mean_predicted_risk"]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_33D["observed_event_rate"]
            ),
            "platt_intercept": platt_intercept_33D,
            "platt_slope": platt_slope_33D,
            "protocol_sha256": EXPECTED_PROTOCOL_SHA_33D,
            "numerical_amendment_sha256": EXPECTED_AMENDMENT_SHA_33D,
            "full_inner_refit_correction_sha256": EXPECTED_CORRECTION_SHA_33D,
            "corrected_checkpoint_namespace": "v2",
            "analysis_status": "post_hoc_exploratory_corrected",
            "full_inner_refit_correction_sha256": EXPECTED_CORRECTION_SHA_33D,
            "corrected_checkpoint_namespace": "v2",
            "analysis_status": "post_hoc_exploratory_corrected",
            "model_max_iter": 20000,
        }
    ]
)

# ------------------------------------------------------------
# 17. Lock selection/calibration aggregate files
# ------------------------------------------------------------

candidate_results_path_33D = os.path.join(
    MODEL_OUTPUT_DIR,
    "33D_clinical_candidate_results_outer3.csv",
)
selected_model_path_33D = os.path.join(
    MODEL_OUTPUT_DIR,
    "33D_clinical_selected_model_outer3.csv",
)
selection_json_path_33D = os.path.join(
    MODEL_OUTPUT_DIR,
    "33D_clinical_selection_calibration_outer3.json",
)
selection_sha_path_33D = os.path.join(
    MODEL_OUTPUT_DIR,
    "33D_clinical_selection_calibration_outer3_SHA256.txt",
)

candidate_results_33D.to_csv(
    candidate_results_path_33D,
    index=False,
)
selected_model_33D.to_csv(
    selected_model_path_33D,
    index=False,
)

selection_configuration_33D = {
    "outer_fold": 3,
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_33D,
    "selection_metric_primary": "pooled inner out-of-fold AUPRC",
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
        "candidate_id ascending immutable tie-break",
    ],
    "selected_candidate": selected_candidate_33D,
    "selected_C": selected_C_33D,
    "selected_l1_ratio": selected_l1_ratio_33D,
    "inner_oof_auprc": float(best_row_33D["auprc"]),
    "inner_oof_auroc": float(best_row_33D["auroc"]),
    "inner_oof_brier": float(best_row_33D["brier"]),
    "platt_intercept": platt_intercept_33D,
    "platt_slope": platt_slope_33D,
    "inner_checkpoint_tables": checkpoint_tables_33D,
    "patient_level_oof_written_to_drive": False,
    "analysis_role": "additional_post_hoc_clinical_baseline",
    "predictors": CLINICAL_PREDICTORS_33D,
}

with open(
    selection_json_path_33D,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        selection_configuration_33D,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(selection_json_path_33D, "rb") as file_handle:
    selection_sha_33D = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    selection_sha_path_33D,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(selection_sha_33D + "\n")

# ------------------------------------------------------------
# 18. Fit final selected outer-fold-3 model
# ------------------------------------------------------------

final_pipeline_33D = Pipeline(
    steps=[
        ("preprocessor", make_preprocessor_33D()),
        (
            "model",
            make_model_33D(
                selected_C_33D,
                selected_l1_ratio_33D,
            ),
        ),
    ]
)

print(
    "\nFitting selected outer-fold-3 model "
    "on all 46,755 training patients..."
)

final_fit_started_33D = time.time()

with warnings.catch_warnings(record=True) as final_warning_records_33D:
    warnings.simplefilter("always", ConvergenceWarning)
    final_pipeline_33D.fit(
        X_outer_training_33D,
        y_outer_training_33D,
    )

final_fit_elapsed_33D = time.time() - final_fit_started_33D

final_model_n_iter_33D = int(
    np.max(
        np.asarray(
            final_pipeline_33D.named_steps["model"].n_iter_
        )
    )
)

final_convergence_warnings_33D = sum(
    issubclass(warning.category, ConvergenceWarning)
    for warning in final_warning_records_33D
)

if final_convergence_warnings_33D != 0:
    raise RuntimeError(
        "Nihai outer-fold-3 klinik baseline modeli max_iter=20000 "
        f"altında da yakınsamadı. n_iter_={final_model_n_iter_33D}"
    )

# ------------------------------------------------------------
# 19. Outer-fold-3 test predictions and metrics
# ------------------------------------------------------------

outer3_raw_probabilities_33D = final_pipeline_33D.predict_proba(
    X_outer_test_33D
)[:, 1]

raw_clipped_33D = np.clip(
    outer3_raw_probabilities_33D,
    1e-6,
    1 - 1e-6,
)
raw_logit_33D = np.log(
    raw_clipped_33D / (1 - raw_clipped_33D)
)

outer3_platt_probabilities_33D = expit(
    platt_intercept_33D + platt_slope_33D * raw_logit_33D
)

for probabilities, name in [
    (outer3_raw_probabilities_33D, "raw"),
    (outer3_platt_probabilities_33D, "platt"),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(f"{name} tahminlerinde eksik değer var.")

    if not np.all(
        (probabilities >= 0) & (probabilities <= 1)
    ):
        raise RuntimeError(
            f"{name} tahminlerinde geçersiz olasılık değeri var."
        )

raw_metrics_33D = probability_metrics_33D(
    y_outer_test_33D,
    outer3_raw_probabilities_33D,
)
platt_metrics_33D = probability_metrics_33D(
    y_outer_test_33D,
    outer3_platt_probabilities_33D,
)

raw_calibration_intercept_33D, raw_calibration_slope_33D = (
    calibration_intercept_slope_33D(
        y_outer_test_33D,
        outer3_raw_probabilities_33D,
    )
)

platt_calibration_intercept_33D, platt_calibration_slope_33D = (
    calibration_intercept_slope_33D(
        y_outer_test_33D,
        outer3_platt_probabilities_33D,
    )
)

outer3_test_results_33D = pd.DataFrame(
    [
        {
            "outer_fold": 3,
            "model": "parsimonious_clinical_logistic",
            "probability_type": "raw",
            **raw_metrics_33D,
            "calibration_intercept": raw_calibration_intercept_33D,
            "calibration_slope": raw_calibration_slope_33D,
        },
        {
            "outer_fold": 3,
            "model": "parsimonious_clinical_logistic",
            "probability_type": "platt_calibrated",
            **platt_metrics_33D,
            "calibration_intercept": platt_calibration_intercept_33D,
            "calibration_slope": platt_calibration_slope_33D,
        },
    ]
)

# ------------------------------------------------------------
# 20. Feature coefficient audit
# ------------------------------------------------------------

fitted_preprocessor_33D = final_pipeline_33D.named_steps[
    "preprocessor"
]
fitted_model_33D = final_pipeline_33D.named_steps["model"]

processed_feature_names_33D = (
    fitted_preprocessor_33D.get_feature_names_out()
)
model_coefficients_33D = fitted_model_33D.coef_.reshape(-1)

if len(processed_feature_names_33D) != len(model_coefficients_33D):
    raise RuntimeError("Feature ve katsayı sayıları uyuşmuyor.")

if len(set(processed_feature_names_33D)) != len(
    processed_feature_names_33D
):
    raise RuntimeError("İşlenmiş feature adlarında yinelenme var.")

coefficient_table_33D = pd.DataFrame(
    {
        "processed_feature": processed_feature_names_33D,
        "coefficient": model_coefficients_33D,
    }
)
coefficient_table_33D["absolute_coefficient"] = (
    coefficient_table_33D["coefficient"].abs()
)
coefficient_table_33D["is_nonzero"] = ~np.isclose(
    coefficient_table_33D["coefficient"],
    0.0,
    atol=1e-12,
)
coefficient_table_33D["absolute_rank"] = (
    coefficient_table_33D["absolute_coefficient"]
    .rank(method="first", ascending=False)
    .astype(int)
)
coefficient_table_33D = (
    coefficient_table_33D.sort_values(
        ["absolute_coefficient", "processed_feature"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

nonzero_coefficients_33D = int(
    coefficient_table_33D["is_nonzero"].sum()
)

final_model_summary_33D = pd.DataFrame(
    [
        {
            "outer_fold": 3,
            "selected_candidate": selected_candidate_33D,
            "selected_C": selected_C_33D,
            "selected_l1_ratio": selected_l1_ratio_33D,
            "training_patients": len(X_outer_training_33D),
            "training_hospitals": len(training_hospitals_33D),
            "training_events": int(y_outer_training_33D.sum()),
            "test_patients": len(X_outer_test_33D),
            "test_hospitals": len(test_hospitals_33D),
            "test_events": int(y_outer_test_33D.sum()),
            "hospital_overlap": len(hospital_overlap_33D),
            "processed_feature_columns": len(
                processed_feature_names_33D
            ),
            "nonzero_coefficients": nonzero_coefficients_33D,
            "model_intercept": float(
                fitted_model_33D.intercept_[0]
            ),
            "convergence_warnings": final_convergence_warnings_33D,
            "final_model_n_iter": final_model_n_iter_33D,
            "fit_elapsed_seconds": float(final_fit_elapsed_33D),
            "locked_platt_intercept": platt_intercept_33D,
            "locked_platt_slope": platt_slope_33D,
            "protocol_sha256": EXPECTED_PROTOCOL_SHA_33D,
            "numerical_amendment_sha256": EXPECTED_AMENDMENT_SHA_33D,
            "model_max_iter": 20000,
            "selection_sha256": selection_sha_33D,
        }
    ]
)

# ------------------------------------------------------------
# 21. Secure BigQuery outer-test checkpoint
# ------------------------------------------------------------

outer3_prediction_df_33D = pd.DataFrame(
    {
        "id_row": outer_test_meta_33D["id_row"].astype(str),
        "outer_fold": np.full(
            len(outer_test_meta_33D),
            3,
            dtype=np.int64,
        ),
        "label_stage23": y_outer_test_33D.astype(np.int64),
        "prediction_raw": outer3_raw_probabilities_33D.astype(
            np.float64
        ),
        "prediction_platt": outer3_platt_probabilities_33D.astype(
            np.float64
        ),
        "model_name": "parsimonious_clinical_logistic",
        "model_version": "clinical8_v1_nested_cv",
    }
)

if len(outer3_prediction_df_33D) != 11736:
    raise RuntimeError(
        "Outer-fold-3 tahmin satır sayısı 11,736 değil."
    )

if outer3_prediction_df_33D["id_row"].duplicated().any():
    raise RuntimeError(
        "Outer-fold-3 tahminlerinde yinelenen id_row var."
    )

if int(
    outer3_prediction_df_33D["label_stage23"].sum()
) != 608:
    raise RuntimeError("Outer-fold-3 event count is not 608 değil.")

prediction_table_id_33D = (
    f"{TARGET_DATASET}."
    "model_clinical_lr_outer_predictions_outer3_v2"
)

prediction_load_config_33D = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("prediction_platt", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("model_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("model_version", "STRING", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

print("\nUploading secure outer-fold-3 prediction checkpoint:")
print(prediction_table_id_33D)

client.load_table_from_dataframe(
    outer3_prediction_df_33D,
    prediction_table_id_33D,
    job_config=prediction_load_config_33D,
    location=BQ_LOCATION,
).result()

# ------------------------------------------------------------
# 22. BigQuery outer-test checkpoint verification
# ------------------------------------------------------------

SQL_VERIFY_PREDICTIONS_33D = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL) AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL) AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0 OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0 OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw) AS minimum_raw_probability,
  MAX(prediction_raw) AS maximum_raw_probability,
  MIN(prediction_platt) AS minimum_platt_probability,
  MAX(prediction_platt) AS maximum_platt_probability
FROM `{prediction_table_id_33D}`;
"""

prediction_verification_33D = client.query(
    SQL_VERIFY_PREDICTIONS_33D,
    location=BQ_LOCATION,
).to_dataframe()

verification_row_33D = prediction_verification_33D.iloc[0]

expected_prediction_values_33D = {
    "prediction_rows": 11736,
    "distinct_rows": 11736,
    "outer_folds": 1,
    "minimum_outer_fold": 3,
    "maximum_outer_fold": 3,
    "events": 608,
    "nonevents": 11128,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in expected_prediction_values_33D.items():
    actual_value = int(verification_row_33D[field])
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: bulunan={actual_value}, beklenen={expected_value}"
        )

# ------------------------------------------------------------
# 23. Save only aggregate and feature-level outputs to Drive
# ------------------------------------------------------------

test_results_path_33D = os.path.join(
    MODEL_OUTPUT_DIR,
    "33D_clinical_outer3_test_results.csv",
)
model_summary_path_33D = os.path.join(
    MODEL_OUTPUT_DIR,
    "33D_clinical_final_model_outer3.csv",
)
coefficient_path_33D = os.path.join(
    MODEL_OUTPUT_DIR,
    "33D_clinical_coefficients_outer3.csv",
)
evaluation_json_path_33D = os.path.join(
    MODEL_OUTPUT_DIR,
    "33D_clinical_final_evaluation_outer3.json",
)
evaluation_sha_path_33D = os.path.join(
    MODEL_OUTPUT_DIR,
    "33D_clinical_final_evaluation_outer3_SHA256.txt",
)

outer3_test_results_33D.to_csv(
    test_results_path_33D,
    index=False,
)
final_model_summary_33D.to_csv(
    model_summary_path_33D,
    index=False,
)
coefficient_table_33D.to_csv(
    coefficient_path_33D,
    index=False,
)

evaluation_configuration_33D = {
    "outer_fold": 3,
    "model_family": "parsimonious_clinical_elastic_net_logistic_regression",
    "selected_candidate": selected_candidate_33D,
    "selected_C": selected_C_33D,
    "selected_l1_ratio": selected_l1_ratio_33D,
    "training_patients": 46755,
    "training_hospitals": 158,
    "test_patients": 11736,
    "test_hospitals": 40,
    "hospital_overlap": 0,
    "locked_platt_intercept": platt_intercept_33D,
    "locked_platt_slope": platt_slope_33D,
    "processed_feature_columns": int(
        len(processed_feature_names_33D)
    ),
    "nonzero_coefficients": int(nonzero_coefficients_33D),
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_33D,
    "selection_sha256": selection_sha_33D,
    "secure_prediction_table": prediction_table_id_33D,
    "patient_level_prediction_written_to_drive": False,
    "analysis_role": "additional_post_hoc_clinical_baseline",
    "predictors": CLINICAL_PREDICTORS_33D,
}

with open(
    evaluation_json_path_33D,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        evaluation_configuration_33D,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(evaluation_json_path_33D, "rb") as file_handle:
    evaluation_sha_33D = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    evaluation_sha_path_33D,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(evaluation_sha_33D + "\n")

# ------------------------------------------------------------
# 24. Final outputs
# ------------------------------------------------------------

pooled_integrity_33D = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_33D),
            pooled_oof_33D["id_row"].nunique(),
            pooled_oof_33D["candidate_id"].nunique(),
            pooled_oof_33D["inner_fold"].nunique(),
            EXPECTED_SPLIT_33D["training_events"],
            (
                EXPECTED_SPLIT_33D["training_rows"]
                - EXPECTED_SPLIT_33D["training_events"]
            ),
            int(
                pooled_oof_33D.duplicated(
                    subset=["candidate_id", "id_row"]
                ).sum()
            ),
            int(pooled_oof_33D["prediction_raw"].isna().sum()),
            int(
                (~pooled_oof_33D["prediction_raw"].between(0, 1)).sum()
            ),
            pooled_load_method_33D,
        ],
    }
)

print("\n33D OUTER-FOLD-3 INNER CHECKPOINT SUMMARY")
display(checkpoint_summary_33D)

print("\n33D OUTER-FOLD-3 POOLED OOF INTEGRITY")
display(pooled_integrity_33D)

print("\n33D OUTER-FOLD-3 CANDIDATE RESULTS")
display(candidate_results_33D)

print("\n33D OUTER-FOLD-3 SELECTED MODEL")
display(selected_model_33D)

print("\n33D OUTER-FOLD-3 FINAL MODEL SUMMARY")
display(final_model_summary_33D)

print("\n33D OUTER-FOLD-3 TEST RESULTS")
display(outer3_test_results_33D)

print("\n33D OUTER-FOLD-3 BIGQUERY VERIFICATION")
display(prediction_verification_33D)

print("\n33D OUTER-FOLD-3 TOP 20 ABSOLUTE COEFFICIENTS")
display(coefficient_table_33D.head(20))

print("\nSelection SHA-256:")
print(selection_sha_33D)

print("\nEvaluation SHA-256:")
print(evaluation_sha_33D)

print("\nSaved:")
print(fit_audit_path_33D)
print(checkpoint_summary_path_33D)
print(candidate_results_path_33D)
print(selected_model_path_33D)
print(selection_json_path_33D)
print(selection_sha_path_33D)
print(test_results_path_33D)
print(model_summary_path_33D)
print(coefficient_path_33D)
print(evaluation_json_path_33D)
print(evaluation_sha_path_33D)

print(
    "\n33D PASS: Outer-fold-3 nested modelling "
    "and locked test evaluation are complete."
)
print(
    "All inner OOF and outer-test patient-level "
    "predictions were stored only in BigQuery."
)
print(
    "No patient-level prediction file was "
    "written to Google Drive."
)

_ = gc.collect()

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from sklearn.exceptions import ConvergenceWarning

from IPython.display import display

# ============================================================
# 33B — PARSIMONIOUS CLINICAL BASELINE OUTER FOLD 4
#
# Resume-safe:
# - Each inner fold is stored in a separate BigQuery table.
# - Completed inner folds are automatically skipped.
#
# BigQuery free-tier compatible:
# - No DELETE / INSERT / UPDATE / MERGE is used.
#
# Privacy:
# - Patient-level predictions are stored only in BigQuery.
# - No patient-level prediction file is written to Drive.
# ============================================================

print("STARTING PARSIMONIOUS CLINICAL BASELINE OUTER FOLD 4 — CODE VERSION 33E")

OUTER_FOLD_33E = 4
MODEL_RANDOM_SEED_33E = 20260721

EXPECTED_PROTOCOL_SHA_33E = (
    "94b0abb218dbef4e349702ea2824ca4e"
    "31dfbe36c53efa05ba1a8bd1f38f835e"
)

EXPECTED_AMENDMENT_SHA_33E = (
    "fcbcf857caa9aaad7ffd6691b4b61ac6"
    "503d09a6eee80db84bc2967ddf1f1a6a"
)

EXPECTED_CORRECTION_SHA_33E = (
    "ea7ab2747aad0c20157da0988b2fbeb1"
    "6ce007b4c92bf785d9ef9d363862ecfa"
)

EXPECTED_SPLIT_33E = {
    "training_rows": 46803,
    "test_rows": 11688,
    "training_hospitals": 159,
    "test_hospitals": 39,
    "training_events": 2426,
    "test_events": 606,
}

EXPECTED_INNER_33E = {
    1: {"validation_rows": 10116, "validation_events": 554},
    2: {"validation_rows": 8161, "validation_events": 381},
    3: {"validation_rows": 6957, "validation_events": 358},
    4: {"validation_rows": 12682, "validation_events": 663},
    5: {"validation_rows": 8887, "validation_events": 470},
}

CLINICAL_PREDICTORS_33E = [
    "x_age_years",
    "x_sex",
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
    "x_lab_bun_last",
    "x_vital_respiratory_rate_last",
    "x_vital_noninvasive_systolic_bp_last",
]

CLINICAL_NUMERIC_33E = [
    "x_age_years",
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
    "x_lab_bun_last",
    "x_vital_respiratory_rate_last",
    "x_vital_noninvasive_systolic_bp_last",
]

CLINICAL_CATEGORICAL_33E = ["x_sex"]

# ------------------------------------------------------------
# 1. Required runtime objects
# ------------------------------------------------------------

required_objects_33E = [
    "core_df_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_33E = [
    name for name in required_objects_33E if name not in globals()
]

if missing_objects_33E:
    raise RuntimeError(
        "Eksik RAM nesneleri var: "
        + ", ".join(missing_objects_33E)
        + ". Önce 07A ve 07B hücrelerini çalıştır."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"58.491 satır bekleniyordu; {len(core_df_07B)} bulundu."
    )

missing_predictors_33E = [
    column for column in CLINICAL_PREDICTORS_33E
    if column not in core_df_07B.columns
]

if missing_predictors_33E:
    raise RuntimeError(
        "Kilitli klinik predictor(lar) core_df_07B içinde yok: "
        + ", ".join(missing_predictors_33E)
    )

# ------------------------------------------------------------
# 2. Locked protocol SHA check
# ------------------------------------------------------------

protocol_sha_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A_locked_parsimonious_clinical_baseline_protocol_v1_SHA256.txt",
)

if not os.path.exists(protocol_sha_path_33E):
    raise FileNotFoundError(
        "Model protokolü SHA dosyası bulunamadı: "
        + protocol_sha_path_33E
    )

with open(protocol_sha_path_33E, "r", encoding="utf-8") as file_handle:
    observed_protocol_sha_33E = file_handle.read().strip()

if observed_protocol_sha_33E != EXPECTED_PROTOCOL_SHA_33E:
    raise RuntimeError(
        "Kilitli model protokolü SHA değeri değişmiş: "
        + observed_protocol_sha_33E
    )

amendment_sha_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A1_parsimonious_clinical_baseline_convergence_amendment_v1_SHA256.txt",
)

if not os.path.exists(amendment_sha_path_33E):
    raise FileNotFoundError(
        "Önce 33A1 convergence amendment scriptini çalıştır: "
        + amendment_sha_path_33E
    )

with open(amendment_sha_path_33E, "r", encoding="utf-8") as file_handle:
    observed_amendment_sha_33E = file_handle.read().strip()

if observed_amendment_sha_33E != EXPECTED_AMENDMENT_SHA_33E:
    raise RuntimeError(
        "33A1 amendment SHA değeri değişmiş: "
        + observed_amendment_sha_33E
    )

print("33A protocol SHA guard: PASS")
print("33A1 convergence amendment SHA guard: PASS")

correction_sha_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A2_parsimonious_clinical_baseline_full_inner_refit_correction_v1_SHA256.txt",
)

if not os.path.exists(correction_sha_path_33E):
    raise FileNotFoundError(
        "Önce 33A2 correction scriptini çalıştır: "
        + correction_sha_path_33E
    )

with open(correction_sha_path_33E, "r", encoding="utf-8") as file_handle:
    observed_correction_sha_33E = file_handle.read().strip()

if observed_correction_sha_33E != EXPECTED_CORRECTION_SHA_33E:
    raise RuntimeError(
        "33A2 correction SHA değeri değişmiş: "
        + observed_correction_sha_33E
    )

print("33A2 full inner-refit correction SHA guard: PASS")

# ------------------------------------------------------------
# 3. Locked inner-hospital mapping
# ------------------------------------------------------------

inner_mapping_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_33E):
    raise FileNotFoundError(
        "Kilitli iç kat haritası bulunamadı: "
        + inner_mapping_path_33E
    )

inner_mapping_all_33E = pd.read_csv(
    inner_mapping_path_33E,
    dtype={"group_hospital": str},
)

inner_mapping_part_33E = (
    inner_mapping_all_33E.loc[
        inner_mapping_all_33E["outer_fold"].astype(int) == OUTER_FOLD_33E,
        ["group_hospital", "inner_fold"],
    ]
    .copy()
)

inner_mapping_part_33E["group_hospital"] = (
    inner_mapping_part_33E["group_hospital"].astype(str)
)
inner_mapping_part_33E["inner_fold"] = (
    inner_mapping_part_33E["inner_fold"].astype(int)
)

if len(inner_mapping_part_33E) != 158:
    raise RuntimeError(
        "Dış kat 1 eğitim kümesi için 158 hastane ataması bekleniyordu."
    )

if inner_mapping_part_33E["group_hospital"].duplicated().any():
    raise RuntimeError("İç kat haritasında yinelenen hastane var.")

hospital_to_inner_fold_33E = dict(
    zip(
        inner_mapping_part_33E["group_hospital"],
        inner_mapping_part_33E["inner_fold"],
    )
)

# ------------------------------------------------------------
# 4. Prepare model matrices
# ------------------------------------------------------------

X_all_33E = core_df_07B[CLINICAL_PREDICTORS_33E].copy()

for column in CLINICAL_NUMERIC_33E:
    X_all_33E[column] = pd.to_numeric(
        X_all_33E[column], errors="coerce"
    ).astype("float64")

for column in CLINICAL_CATEGORICAL_33E:
    category_series = X_all_33E[column].astype("object")
    X_all_33E[column] = category_series.where(
        pd.notna(category_series), np.nan
    )

outer_fold_vector_33E = (
    core_df_07B["outer_fold"].astype(int).to_numpy()
)

outer_training_mask_33E = outer_fold_vector_33E != OUTER_FOLD_33E
outer_test_mask_33E = outer_fold_vector_33E == OUTER_FOLD_33E

X_outer_training_33E = (
    X_all_33E.loc[outer_training_mask_33E].reset_index(drop=True)
)
X_outer_test_33E = (
    X_all_33E.loc[outer_test_mask_33E].reset_index(drop=True)
)

outer_training_meta_33E = (
    core_df_07B.loc[
        outer_training_mask_33E,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_33E = (
    core_df_07B.loc[
        outer_test_mask_33E,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [outer_training_meta_33E, outer_test_meta_33E]:
    dataframe["id_row"] = dataframe["id_row"].astype(str)
    dataframe["group_hospital"] = dataframe["group_hospital"].astype(str)
    dataframe["label_stage23"] = dataframe["label_stage23"].astype(int)

y_outer_training_33E = (
    outer_training_meta_33E["label_stage23"].to_numpy(dtype=np.int8)
)
y_outer_test_33E = (
    outer_test_meta_33E["label_stage23"].to_numpy(dtype=np.int8)
)

groups_outer_training_33E = (
    outer_training_meta_33E["group_hospital"].to_numpy(dtype=str)
)

training_hospitals_33E = set(
    outer_training_meta_33E["group_hospital"]
)
test_hospitals_33E = set(
    outer_test_meta_33E["group_hospital"]
)
hospital_overlap_33E = training_hospitals_33E & test_hospitals_33E

if hospital_overlap_33E:
    raise RuntimeError("Dış eğitim ve test hastaneleri çakışıyor.")

actual_split_33E = {
    "training_rows": len(X_outer_training_33E),
    "test_rows": len(X_outer_test_33E),
    "training_hospitals": len(training_hospitals_33E),
    "test_hospitals": len(test_hospitals_33E),
    "training_events": int(y_outer_training_33E.sum()),
    "test_events": int(y_outer_test_33E.sum()),
}

for metric, expected_value in EXPECTED_SPLIT_33E.items():
    actual_value = actual_split_33E[metric]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: bulunan={actual_value}, beklenen={expected_value}"
        )

inner_fold_vector_33E = np.array(
    [
        hospital_to_inner_fold_33E.get(hospital, -1)
        for hospital in groups_outer_training_33E
    ],
    dtype=int,
)

if (inner_fold_vector_33E == -1).any():
    raise RuntimeError(
        "Bazı dış eğitim hastanelerine iç kat atanmadı."
    )

if set(np.unique(inner_fold_vector_33E)) != {1, 2, 3, 4, 5}:
    raise RuntimeError("İç kat değerleri 1–5 değil.")

for inner_fold, expected in EXPECTED_INNER_33E.items():
    validation_mask = inner_fold_vector_33E == inner_fold
    observed_rows = int(validation_mask.sum())
    observed_events = int(y_outer_training_33E[validation_mask].sum())

    if observed_rows != expected["validation_rows"]:
        raise RuntimeError(
            f"Inner {inner_fold} validation_rows: "
            f"bulunan={observed_rows}, "
            f"beklenen={expected['validation_rows']}"
        )

    if observed_events != expected["validation_events"]:
        raise RuntimeError(
            f"Inner {inner_fold} validation_events: "
            f"bulunan={observed_events}, "
            f"beklenen={expected['validation_events']}"
        )

# ------------------------------------------------------------
# 5. Locked candidate grid
# ------------------------------------------------------------

candidate_grid_33E = [
    {"candidate_id": "CLIN01", "C": 0.03, "l1_ratio": 0.00},
    {"candidate_id": "CLIN02", "C": 0.10, "l1_ratio": 0.00},
    {"candidate_id": "CLIN03", "C": 0.30, "l1_ratio": 0.00},
    {"candidate_id": "CLIN04", "C": 0.10, "l1_ratio": 0.25},
    {"candidate_id": "CLIN05", "C": 0.30, "l1_ratio": 0.25},
    {"candidate_id": "CLIN06", "C": 0.30, "l1_ratio": 0.50},
]

# ------------------------------------------------------------
# 6. Preprocessing and model factories
# ------------------------------------------------------------

def make_preprocessor_33E():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
            (
                "scaler",
                StandardScaler(with_mean=False),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                CLINICAL_NUMERIC_33E,
            ),
            (
                "categorical",
                categorical_pipeline,
                CLINICAL_CATEGORICAL_33E,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_model_33E(C_value, l1_ratio_value):
    return LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=float(C_value),
        l1_ratio=float(l1_ratio_value),
        class_weight=None,
        max_iter=20000,
        tol=1e-4,
        random_state=MODEL_RANDOM_SEED_33E,
    )


def checkpoint_table_id_33E(inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_clinical_lr_inner_oof_outer4_inner{inner_fold}_v2"
    )

# ------------------------------------------------------------
# 7. BigQuery checkpoint verification
# ------------------------------------------------------------

def verify_checkpoint_33E(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):
    table_id = checkpoint_table_id_33E(inner_fold)

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS distinct_id_count,
      COUNT(DISTINCT candidate_id) AS candidate_count,
      COUNT(DISTINCT outer_fold) AS outer_fold_count,
      COUNT(DISTINCT inner_fold) AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(candidate_id, '|', id_row)
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(
        prediction_raw < 0 OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(sql, location=BQ_LOCATION).to_dataframe()
    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows * len(candidate_grid_33E)
    )
    expected_positive_rows = (
        expected_validation_events * len(candidate_grid_33E)
    )
    expected_negative_rows = (
        (expected_validation_rows - expected_validation_events)
        * len(candidate_grid_33E)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": expected_validation_rows,
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": expected_total_rows,
        "positive_prediction_rows": expected_positive_rows,
        "negative_prediction_rows": expected_negative_rows,
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": 4,
        "maximum_outer_fold": 4,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failure_items = []

    for field, expected_value in expected_values.items():
        actual_value = int(row[field])
        if actual_value != expected_value:
            complete = False
            failure_items.append(
                f"{field}={actual_value}, expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failure_items),
        "check": check,
        "row": row,
    }

# ------------------------------------------------------------
# 8. BigQuery load schema
# ------------------------------------------------------------

checkpoint_load_config_33E = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("inner_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("candidate_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

# ------------------------------------------------------------
# 9. Aggregate fit-audit file
# ------------------------------------------------------------

fit_audit_columns_33E = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "C",
    "l1_ratio",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "convergence_warnings",
    "elapsed_seconds",
]

fit_audit_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_inner_fit_audit_outer4.csv",
)

if os.path.exists(fit_audit_path_33E):
    fit_audit_33E = pd.read_csv(fit_audit_path_33E)
else:
    fit_audit_33E = pd.DataFrame(columns=fit_audit_columns_33E)

for column in fit_audit_columns_33E:
    if column not in fit_audit_33E.columns:
        fit_audit_33E[column] = np.nan

fit_audit_33E = fit_audit_33E[fit_audit_columns_33E].copy()

# ------------------------------------------------------------
# 10. Train six candidates in five locked inner folds
# ------------------------------------------------------------

for inner_fold in range(1, 6):
    inner_training_mask = inner_fold_vector_33E != inner_fold
    inner_validation_mask = inner_fold_vector_33E == inner_fold

    training_rows = int(inner_training_mask.sum())
    validation_rows = int(inner_validation_mask.sum())
    training_events = int(
        y_outer_training_33E[inner_training_mask].sum()
    )
    validation_events = int(
        y_outer_training_33E[inner_validation_mask].sum()
    )

    expected_inner = EXPECTED_INNER_33E[inner_fold]
    expected_training_rows = (
        EXPECTED_SPLIT_33E["training_rows"]
        - expected_inner["validation_rows"]
    )
    expected_training_events = (
        EXPECTED_SPLIT_33E["training_events"]
        - expected_inner["validation_events"]
    )

    if training_rows != expected_training_rows:
        raise RuntimeError(
            f"Inner {inner_fold} training_rows: "
            f"bulunan={training_rows}, "
            f"beklenen={expected_training_rows}"
        )

    if training_events != expected_training_events:
        raise RuntimeError(
            f"Inner {inner_fold} training_events: "
            f"bulunan={training_events}, "
            f"beklenen={expected_training_events}"
        )

    existing_check = verify_checkpoint_33E(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if existing_check["complete"]:
        print(
            f"Outer 4 / inner {inner_fold}: "
            "permanent checkpoint already complete; "
            "skipping model fitting."
        )
        continue

    training_hospital_set = set(
        groups_outer_training_33E[inner_training_mask]
    )
    validation_hospital_set = set(
        groups_outer_training_33E[inner_validation_mask]
    )

    if training_hospital_set & validation_hospital_set:
        raise RuntimeError(
            f"Inner fold {inner_fold}: hastane çakışması bulundu."
        )

    print(f"\nOuter 4 / inner {inner_fold}")
    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_33E()
    preprocessing_started = time.time()

    X_inner_training_processed = preprocessor.fit_transform(
        X_outer_training_33E.loc[inner_training_mask]
    )
    X_inner_validation_processed = preprocessor.transform(
        X_outer_training_33E.loc[inner_validation_mask]
    )

    preprocessing_elapsed = time.time() - preprocessing_started

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "İşlenmiş eğitim ve doğrulama sütun sayıları farklı."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_elapsed, 2),
    )

    y_inner_training = y_outer_training_33E[inner_training_mask]
    y_inner_validation = y_outer_training_33E[inner_validation_mask]

    validation_ids = (
        outer_training_meta_33E.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_33E:
        candidate_id = candidate["candidate_id"]

        print(
            "  Fitting",
            candidate_id,
            "| C =",
            candidate["C"],
            "| l1_ratio =",
            candidate["l1_ratio"],
        )

        model = make_model_33E(
            candidate["C"],
            candidate["l1_ratio"],
        )

        fitting_started = time.time()

        with warnings.catch_warnings(record=True) as warning_records:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(
                X_inner_training_processed,
                y_inner_training,
            )

        fitting_elapsed = time.time() - fitting_started

        convergence_warning_count = sum(
            issubclass(warning.category, ConvergenceWarning)
            for warning in warning_records
        )

        validation_probabilities = model.predict_proba(
            X_inner_validation_processed
        )[:, 1]

        if np.isnan(validation_probabilities).any():
            raise RuntimeError(
                f"{candidate_id}, inner {inner_fold}: eksik tahmin."
            )

        if not np.all(
            (validation_probabilities >= 0)
            & (validation_probabilities <= 1)
        ):
            raise RuntimeError(
                f"{candidate_id}: geçersiz olasılık."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        4,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": y_inner_validation.astype(np.int64),
                    "prediction_raw": validation_probabilities.astype(
                        np.float64
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": 4,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "C": candidate["C"],
                "l1_ratio": candidate["l1_ratio"],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": validation_events,
                "processed_columns": int(
                    X_inner_training_processed.shape[1]
                ),
                "convergence_warnings": int(
                    convergence_warning_count
                ),
                "elapsed_seconds": float(fitting_elapsed),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows * len(candidate_grid_33E)
    )

    if len(checkpoint_df) != expected_checkpoint_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: checkpoint satır sayısı hatalı."
        )

    if checkpoint_df.duplicated(
        subset=["id_row", "candidate_id"]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "yinelenen aday–hasta tahmini var."
        )

    target_checkpoint_table = checkpoint_table_id_33E(inner_fold)

    print(
        "Uploading permanent checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_33E,
        location=BQ_LOCATION,
    ).result()

    new_audit_df = pd.DataFrame(current_audit_rows)

    if len(fit_audit_33E) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_33E["outer_fold"],
                    errors="coerce",
                )
                == 4
            )
            & (
                pd.to_numeric(
                    fit_audit_33E["inner_fold"],
                    errors="coerce",
                )
                == inner_fold
            )
        )
        fit_audit_33E = fit_audit_33E.loc[keep_mask].copy()

    if fit_audit_33E.empty:
        fit_audit_33E = new_audit_df.copy()
    else:
        fit_audit_33E = pd.concat(
            [fit_audit_33E, new_audit_df],
            ignore_index=True,
        )

    fit_audit_33E = (
        fit_audit_33E[fit_audit_columns_33E]
        .sort_values(
            ["outer_fold", "inner_fold", "candidate_id"]
        )
        .reset_index(drop=True)
    )

    fit_audit_33E.to_csv(
        fit_audit_path_33E,
        index=False,
    )

    completed_check = verify_checkpoint_33E(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint doğrulanamadı: "
            + completed_check["reason"]
        )

    print(
        f"Outer 4 / inner {inner_fold}: "
        "permanent checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 11. Final checkpoint summary
# ------------------------------------------------------------

checkpoint_summary_rows_33E = []

for inner_fold in range(1, 6):
    validation_mask = inner_fold_vector_33E == inner_fold
    validation_rows = int(validation_mask.sum())
    validation_events = int(
        y_outer_training_33E[validation_mask].sum()
    )

    final_check = verify_checkpoint_33E(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "son checkpoint denetimi başarısız. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_33E.append(
        {
            "outer_fold": 4,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(row["row_count"]),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(row["candidate_count"]),
            "positive_prediction_rows": int(
                row["positive_prediction_rows"]
            ),
            "negative_prediction_rows": int(
                row["negative_prediction_rows"]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check["table_id"],
        }
    )

checkpoint_summary_33E = (
    pd.DataFrame(checkpoint_summary_rows_33E)
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_33E["distinct_validation_patients"].sum()
) != EXPECTED_SPLIT_33E["training_rows"]:
    raise RuntimeError(
        "Toplam doğrulama hasta sayısı 46.803 değil."
    )

expected_total_oof_rows_33E = (
    EXPECTED_SPLIT_33E["training_rows"]
    * len(candidate_grid_33E)
)

if int(checkpoint_summary_33E["checkpoint_rows"].sum()) != (
    expected_total_oof_rows_33E
):
    raise RuntimeError("Toplam OOF tahmin satırı hatalı.")

checkpoint_summary_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_outer4_inner_checkpoint_summary.csv",
)
checkpoint_summary_33E.to_csv(
    checkpoint_summary_path_33E,
    index=False,
)

# ------------------------------------------------------------
# 12. Pool all inner OOF predictions
# ------------------------------------------------------------

checkpoint_tables_33E = [
    checkpoint_table_id_33E(inner_fold)
    for inner_fold in range(1, 6)
]

union_parts_33E = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_33E
]

SQL_LOAD_POOLED_OOF_33E = "\nUNION ALL\n".join(union_parts_33E)

print("\nLoading pooled outer-fold-4 inner OOF predictions...")

query_job_33E = client.query(
    SQL_LOAD_POOLED_OOF_33E,
    location=BQ_LOCATION,
)

try:
    pooled_oof_33E = query_job_33E.to_dataframe(
        create_bqstorage_client=True
    )
    pooled_load_method_33E = "BigQuery Storage API"
except Exception as fast_path_error_33E:
    print(
        "Storage API unavailable; using standard BigQuery download."
    )
    print("Message:", type(fast_path_error_33E).__name__)
    pooled_oof_33E = query_job_33E.to_dataframe(
        create_bqstorage_client=False
    )
    pooled_load_method_33E = "Standard BigQuery API"

pooled_oof_33E["id_row"] = pooled_oof_33E["id_row"].astype(str)
pooled_oof_33E["candidate_id"] = pooled_oof_33E["candidate_id"].astype(str)

for column in ["outer_fold", "inner_fold", "label_stage23"]:
    pooled_oof_33E[column] = pd.to_numeric(
        pooled_oof_33E[column], errors="raise"
    ).astype(int)

pooled_oof_33E["prediction_raw"] = pd.to_numeric(
    pooled_oof_33E["prediction_raw"], errors="raise"
).astype(float)

# ------------------------------------------------------------
# 13. Pooled OOF integrity checks
# ------------------------------------------------------------

if len(pooled_oof_33E) != expected_total_oof_rows_33E:
    raise RuntimeError("Pooled OOF satır sayısı hatalı.")

if set(pooled_oof_33E["outer_fold"].unique()) != {4}:
    raise RuntimeError("Pooled OOF içinde dış kat 4 dışında kayıt var.")

if set(pooled_oof_33E["inner_fold"].unique()) != {1, 2, 3, 4, 5}:
    raise RuntimeError("Pooled OOF iç katları 1–5 değil.")

if pooled_oof_33E.duplicated(
    subset=["candidate_id", "id_row"]
).any():
    raise RuntimeError(
        "Pooled OOF içinde yinelenen aday–hasta tahmini bulundu."
    )

if pooled_oof_33E["prediction_raw"].isna().any():
    raise RuntimeError("Pooled OOF içinde eksik tahmin var.")

if not pooled_oof_33E["prediction_raw"].between(0, 1).all():
    raise RuntimeError(
        "Pooled OOF içinde geçersiz olasılık değeri var."
    )

expected_candidate_ids_33E = {
    "CLIN01",
    "CLIN02",
    "CLIN03",
    "CLIN04",
    "CLIN05",
    "CLIN06",
}

if set(pooled_oof_33E["candidate_id"].unique()) != (
    expected_candidate_ids_33E
):
    raise RuntimeError("Altı kilitli aday bulunmuyor.")

candidate_patient_counts_33E = (
    pooled_oof_33E.groupby("candidate_id")["id_row"].nunique()
)

if not (
    candidate_patient_counts_33E
    == EXPECTED_SPLIT_33E["training_rows"]
).all():
    raise RuntimeError(
        "Her aday için 46.803 farklı OOF hastası yok."
    )

candidate_event_counts_33E = (
    pooled_oof_33E.groupby("candidate_id")["label_stage23"].sum()
)

if not (
    candidate_event_counts_33E
    == EXPECTED_SPLIT_33E["training_events"]
).all():
    raise RuntimeError("Her aday için 2.426 olay yok.")

patient_label_consistency_33E = (
    pooled_oof_33E.groupby("id_row")["label_stage23"].nunique()
)
if (patient_label_consistency_33E > 1).any():
    raise RuntimeError(
        "Aynı hastanın adaylar arasında outcome etiketi farklı."
    )

patient_inner_fold_consistency_33E = (
    pooled_oof_33E.groupby("id_row")["inner_fold"].nunique()
)
if (patient_inner_fold_consistency_33E > 1).any():
    raise RuntimeError(
        "Aynı hasta birden fazla iç doğrulama katında bulundu."
    )

# ------------------------------------------------------------
# 14. Metric helpers
# ------------------------------------------------------------

def probability_metrics_33E(y_true, probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(roc_auc_score(y_true, probabilities)),
        "auprc": float(average_precision_score(y_true, probabilities)),
        "brier": float(brier_score_loss(y_true, probabilities)),
        "log_loss": float(
            log_loss(y_true, probabilities, labels=[0, 1])
        ),
        "mean_predicted_risk": float(probabilities.mean()),
        "observed_event_rate": float(np.mean(y_true)),
    }


def probability_logit_33E(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        probabilities / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_33E(y_true, probabilities):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        probability_logit_33E(probabilities),
        y_true,
    )
    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 15. Candidate pooled-OOF performance
# ------------------------------------------------------------

candidate_result_rows_33E = []

for candidate in candidate_grid_33E:
    candidate_id = candidate["candidate_id"]

    candidate_oof = (
        pooled_oof_33E.loc[
            pooled_oof_33E["candidate_id"] == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_33E(
        candidate_oof["label_stage23"].to_numpy(dtype=int),
        candidate_oof["prediction_raw"].to_numpy(dtype=float),
    )

    fit_part = fit_audit_33E.loc[
        (
            pd.to_numeric(
                fit_audit_33E["outer_fold"],
                errors="coerce",
            )
            == 4
        )
        & (
            fit_audit_33E["candidate_id"].astype(str)
            == candidate_id
        )
    ]

    convergence_warnings = (
        int(
            pd.to_numeric(
                fit_part["convergence_warnings"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    elapsed_seconds = (
        float(
            pd.to_numeric(
                fit_part["elapsed_seconds"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_33E.append(
        {
            "candidate_id": candidate_id,
            "C": candidate["C"],
            "l1_ratio": candidate["l1_ratio"],
            **metrics,
            "convergence_warnings": convergence_warnings,
            "elapsed_seconds": elapsed_seconds,
        }
    )

candidate_results_33E = pd.DataFrame(candidate_result_rows_33E)

total_inner_convergence_warnings_33E = int(
    pd.to_numeric(
        candidate_results_33E["convergence_warnings"],
        errors="coerce",
    ).fillna(0).sum()
)

if total_inner_convergence_warnings_33E != 0:
    raise RuntimeError(
        "33B-R2 stopped BEFORE candidate selection and BEFORE outer-test "
        "evaluation because at least one inner candidate fit still emitted "
        f"a ConvergenceWarning under max_iter=20000. "
        f"Total warnings={total_inner_convergence_warnings_33E}. "
        "No test result may be accessed."
    )

print(
    "All 30 corrected inner candidate fits converged with zero warnings: PASS"
)

candidate_results_33E = (
    candidate_results_33E.sort_values(
        ["auprc", "auroc", "brier", "candidate_id"],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)

candidate_results_33E["selection_rank"] = np.arange(
    1,
    len(candidate_results_33E) + 1,
)

best_row_33E = candidate_results_33E.iloc[0]
selected_candidate_33E = str(best_row_33E["candidate_id"])
selected_C_33E = float(best_row_33E["C"])
selected_l1_ratio_33E = float(best_row_33E["l1_ratio"])

# ------------------------------------------------------------
# 16. Platt calibration on selected pooled inner OOF
# ------------------------------------------------------------

selected_oof_33E = (
    pooled_oof_33E.loc[
        pooled_oof_33E["candidate_id"] == selected_candidate_33E
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_33E = selected_oof_33E[
    "label_stage23"
].to_numpy(dtype=int)
selected_oof_probability_33E = selected_oof_33E[
    "prediction_raw"
].to_numpy(dtype=float)

platt_calibrator_33E = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_33E.fit(
    probability_logit_33E(selected_oof_probability_33E),
    selected_oof_y_33E,
)

platt_intercept_33E = float(platt_calibrator_33E.intercept_[0])
platt_slope_33E = float(platt_calibrator_33E.coef_[0][0])

if (
    not np.isfinite(platt_intercept_33E)
    or not np.isfinite(platt_slope_33E)
    or platt_slope_33E <= 0
):
    raise RuntimeError("Platt kalibrasyon katsayıları geçersiz.")

selected_model_33E = pd.DataFrame(
    [
        {
            "outer_fold": 4,
            "selected_candidate": selected_candidate_33E,
            "selected_C": selected_C_33E,
            "selected_l1_ratio": selected_l1_ratio_33E,
            "selection_metric_primary": "pooled_inner_oof_auprc",
            "inner_oof_auprc": float(best_row_33E["auprc"]),
            "inner_oof_auroc": float(best_row_33E["auroc"]),
            "inner_oof_brier": float(best_row_33E["brier"]),
            "inner_oof_log_loss": float(best_row_33E["log_loss"]),
            "inner_oof_mean_predicted_risk": float(
                best_row_33E["mean_predicted_risk"]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_33E["observed_event_rate"]
            ),
            "platt_intercept": platt_intercept_33E,
            "platt_slope": platt_slope_33E,
            "protocol_sha256": EXPECTED_PROTOCOL_SHA_33E,
            "numerical_amendment_sha256": EXPECTED_AMENDMENT_SHA_33E,
            "full_inner_refit_correction_sha256": EXPECTED_CORRECTION_SHA_33E,
            "corrected_checkpoint_namespace": "v2",
            "analysis_status": "post_hoc_exploratory_corrected",
            "full_inner_refit_correction_sha256": EXPECTED_CORRECTION_SHA_33E,
            "corrected_checkpoint_namespace": "v2",
            "analysis_status": "post_hoc_exploratory_corrected",
            "model_max_iter": 20000,
        }
    ]
)

# ------------------------------------------------------------
# 17. Lock selection/calibration aggregate files
# ------------------------------------------------------------

candidate_results_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_candidate_results_outer4.csv",
)
selected_model_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_selected_model_outer4.csv",
)
selection_json_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_selection_calibration_outer4.json",
)
selection_sha_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_selection_calibration_outer4_SHA256.txt",
)

candidate_results_33E.to_csv(
    candidate_results_path_33E,
    index=False,
)
selected_model_33E.to_csv(
    selected_model_path_33E,
    index=False,
)

selection_configuration_33E = {
    "outer_fold": 4,
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_33E,
    "selection_metric_primary": "pooled inner out-of-fold AUPRC",
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
        "candidate_id ascending immutable tie-break",
    ],
    "selected_candidate": selected_candidate_33E,
    "selected_C": selected_C_33E,
    "selected_l1_ratio": selected_l1_ratio_33E,
    "inner_oof_auprc": float(best_row_33E["auprc"]),
    "inner_oof_auroc": float(best_row_33E["auroc"]),
    "inner_oof_brier": float(best_row_33E["brier"]),
    "platt_intercept": platt_intercept_33E,
    "platt_slope": platt_slope_33E,
    "inner_checkpoint_tables": checkpoint_tables_33E,
    "patient_level_oof_written_to_drive": False,
    "analysis_role": "additional_post_hoc_clinical_baseline",
    "predictors": CLINICAL_PREDICTORS_33E,
}

with open(
    selection_json_path_33E,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        selection_configuration_33E,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(selection_json_path_33E, "rb") as file_handle:
    selection_sha_33E = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    selection_sha_path_33E,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(selection_sha_33E + "\n")

# ------------------------------------------------------------
# 18. Fit final selected outer-fold-4 model
# ------------------------------------------------------------

final_pipeline_33E = Pipeline(
    steps=[
        ("preprocessor", make_preprocessor_33E()),
        (
            "model",
            make_model_33E(
                selected_C_33E,
                selected_l1_ratio_33E,
            ),
        ),
    ]
)

print(
    "\nFitting selected outer-fold-4 model "
    "on all 46,803 training patients..."
)

final_fit_started_33E = time.time()

with warnings.catch_warnings(record=True) as final_warning_records_33E:
    warnings.simplefilter("always", ConvergenceWarning)
    final_pipeline_33E.fit(
        X_outer_training_33E,
        y_outer_training_33E,
    )

final_fit_elapsed_33E = time.time() - final_fit_started_33E

final_model_n_iter_33E = int(
    np.max(
        np.asarray(
            final_pipeline_33E.named_steps["model"].n_iter_
        )
    )
)

final_convergence_warnings_33E = sum(
    issubclass(warning.category, ConvergenceWarning)
    for warning in final_warning_records_33E
)

if final_convergence_warnings_33E != 0:
    raise RuntimeError(
        "Nihai outer-fold-4 klinik baseline modeli max_iter=20000 "
        f"altında da yakınsamadı. n_iter_={final_model_n_iter_33E}"
    )

# ------------------------------------------------------------
# 19. Outer-fold-4 test predictions and metrics
# ------------------------------------------------------------

outer4_raw_probabilities_33E = final_pipeline_33E.predict_proba(
    X_outer_test_33E
)[:, 1]

raw_clipped_33E = np.clip(
    outer4_raw_probabilities_33E,
    1e-6,
    1 - 1e-6,
)
raw_logit_33E = np.log(
    raw_clipped_33E / (1 - raw_clipped_33E)
)

outer4_platt_probabilities_33E = expit(
    platt_intercept_33E + platt_slope_33E * raw_logit_33E
)

for probabilities, name in [
    (outer4_raw_probabilities_33E, "raw"),
    (outer4_platt_probabilities_33E, "platt"),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(f"{name} tahminlerinde eksik değer var.")

    if not np.all(
        (probabilities >= 0) & (probabilities <= 1)
    ):
        raise RuntimeError(
            f"{name} tahminlerinde geçersiz olasılık değeri var."
        )

raw_metrics_33E = probability_metrics_33E(
    y_outer_test_33E,
    outer4_raw_probabilities_33E,
)
platt_metrics_33E = probability_metrics_33E(
    y_outer_test_33E,
    outer4_platt_probabilities_33E,
)

raw_calibration_intercept_33E, raw_calibration_slope_33E = (
    calibration_intercept_slope_33E(
        y_outer_test_33E,
        outer4_raw_probabilities_33E,
    )
)

platt_calibration_intercept_33E, platt_calibration_slope_33E = (
    calibration_intercept_slope_33E(
        y_outer_test_33E,
        outer4_platt_probabilities_33E,
    )
)

outer4_test_results_33E = pd.DataFrame(
    [
        {
            "outer_fold": 4,
            "model": "parsimonious_clinical_logistic",
            "probability_type": "raw",
            **raw_metrics_33E,
            "calibration_intercept": raw_calibration_intercept_33E,
            "calibration_slope": raw_calibration_slope_33E,
        },
        {
            "outer_fold": 4,
            "model": "parsimonious_clinical_logistic",
            "probability_type": "platt_calibrated",
            **platt_metrics_33E,
            "calibration_intercept": platt_calibration_intercept_33E,
            "calibration_slope": platt_calibration_slope_33E,
        },
    ]
)

# ------------------------------------------------------------
# 20. Feature coefficient audit
# ------------------------------------------------------------

fitted_preprocessor_33E = final_pipeline_33E.named_steps[
    "preprocessor"
]
fitted_model_33E = final_pipeline_33E.named_steps["model"]

processed_feature_names_33E = (
    fitted_preprocessor_33E.get_feature_names_out()
)
model_coefficients_33E = fitted_model_33E.coef_.reshape(-1)

if len(processed_feature_names_33E) != len(model_coefficients_33E):
    raise RuntimeError("Feature ve katsayı sayıları uyuşmuyor.")

if len(set(processed_feature_names_33E)) != len(
    processed_feature_names_33E
):
    raise RuntimeError("İşlenmiş feature adlarında yinelenme var.")

coefficient_table_33E = pd.DataFrame(
    {
        "processed_feature": processed_feature_names_33E,
        "coefficient": model_coefficients_33E,
    }
)
coefficient_table_33E["absolute_coefficient"] = (
    coefficient_table_33E["coefficient"].abs()
)
coefficient_table_33E["is_nonzero"] = ~np.isclose(
    coefficient_table_33E["coefficient"],
    0.0,
    atol=1e-12,
)
coefficient_table_33E["absolute_rank"] = (
    coefficient_table_33E["absolute_coefficient"]
    .rank(method="first", ascending=False)
    .astype(int)
)
coefficient_table_33E = (
    coefficient_table_33E.sort_values(
        ["absolute_coefficient", "processed_feature"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

nonzero_coefficients_33E = int(
    coefficient_table_33E["is_nonzero"].sum()
)

final_model_summary_33E = pd.DataFrame(
    [
        {
            "outer_fold": 4,
            "selected_candidate": selected_candidate_33E,
            "selected_C": selected_C_33E,
            "selected_l1_ratio": selected_l1_ratio_33E,
            "training_patients": len(X_outer_training_33E),
            "training_hospitals": len(training_hospitals_33E),
            "training_events": int(y_outer_training_33E.sum()),
            "test_patients": len(X_outer_test_33E),
            "test_hospitals": len(test_hospitals_33E),
            "test_events": int(y_outer_test_33E.sum()),
            "hospital_overlap": len(hospital_overlap_33E),
            "processed_feature_columns": len(
                processed_feature_names_33E
            ),
            "nonzero_coefficients": nonzero_coefficients_33E,
            "model_intercept": float(
                fitted_model_33E.intercept_[0]
            ),
            "convergence_warnings": final_convergence_warnings_33E,
            "final_model_n_iter": final_model_n_iter_33E,
            "fit_elapsed_seconds": float(final_fit_elapsed_33E),
            "locked_platt_intercept": platt_intercept_33E,
            "locked_platt_slope": platt_slope_33E,
            "protocol_sha256": EXPECTED_PROTOCOL_SHA_33E,
            "numerical_amendment_sha256": EXPECTED_AMENDMENT_SHA_33E,
            "model_max_iter": 20000,
            "selection_sha256": selection_sha_33E,
        }
    ]
)

# ------------------------------------------------------------
# 21. Secure BigQuery outer-test checkpoint
# ------------------------------------------------------------

outer4_prediction_df_33E = pd.DataFrame(
    {
        "id_row": outer_test_meta_33E["id_row"].astype(str),
        "outer_fold": np.full(
            len(outer_test_meta_33E),
            4,
            dtype=np.int64,
        ),
        "label_stage23": y_outer_test_33E.astype(np.int64),
        "prediction_raw": outer4_raw_probabilities_33E.astype(
            np.float64
        ),
        "prediction_platt": outer4_platt_probabilities_33E.astype(
            np.float64
        ),
        "model_name": "parsimonious_clinical_logistic",
        "model_version": "clinical8_v1_nested_cv",
    }
)

if len(outer4_prediction_df_33E) != 11688:
    raise RuntimeError(
        "Outer-fold-4 tahmin satır sayısı 11,688 değil."
    )

if outer4_prediction_df_33E["id_row"].duplicated().any():
    raise RuntimeError(
        "Outer-fold-4 tahminlerinde yinelenen id_row var."
    )

if int(
    outer4_prediction_df_33E["label_stage23"].sum()
) != 606:
    raise RuntimeError("Outer-fold-4 event count is not 606 değil.")

prediction_table_id_33E = (
    f"{TARGET_DATASET}."
    "model_clinical_lr_outer_predictions_outer4_v2"
)

prediction_load_config_33E = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("prediction_platt", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("model_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("model_version", "STRING", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

print("\nUploading secure outer-fold-4 prediction checkpoint:")
print(prediction_table_id_33E)

client.load_table_from_dataframe(
    outer4_prediction_df_33E,
    prediction_table_id_33E,
    job_config=prediction_load_config_33E,
    location=BQ_LOCATION,
).result()

# ------------------------------------------------------------
# 22. BigQuery outer-test checkpoint verification
# ------------------------------------------------------------

SQL_VERIFY_PREDICTIONS_33E = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL) AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL) AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0 OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0 OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw) AS minimum_raw_probability,
  MAX(prediction_raw) AS maximum_raw_probability,
  MIN(prediction_platt) AS minimum_platt_probability,
  MAX(prediction_platt) AS maximum_platt_probability
FROM `{prediction_table_id_33E}`;
"""

prediction_verification_33E = client.query(
    SQL_VERIFY_PREDICTIONS_33E,
    location=BQ_LOCATION,
).to_dataframe()

verification_row_33E = prediction_verification_33E.iloc[0]

expected_prediction_values_33E = {
    "prediction_rows": 11688,
    "distinct_rows": 11688,
    "outer_folds": 1,
    "minimum_outer_fold": 4,
    "maximum_outer_fold": 4,
    "events": 606,
    "nonevents": 11082,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in expected_prediction_values_33E.items():
    actual_value = int(verification_row_33E[field])
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: bulunan={actual_value}, beklenen={expected_value}"
        )

# ------------------------------------------------------------
# 23. Save only aggregate and feature-level outputs to Drive
# ------------------------------------------------------------

test_results_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_outer4_test_results.csv",
)
model_summary_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_final_model_outer4.csv",
)
coefficient_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_coefficients_outer4.csv",
)
evaluation_json_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_final_evaluation_outer4.json",
)
evaluation_sha_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_final_evaluation_outer4_SHA256.txt",
)

outer4_test_results_33E.to_csv(
    test_results_path_33E,
    index=False,
)
final_model_summary_33E.to_csv(
    model_summary_path_33E,
    index=False,
)
coefficient_table_33E.to_csv(
    coefficient_path_33E,
    index=False,
)

evaluation_configuration_33E = {
    "outer_fold": 4,
    "model_family": "parsimonious_clinical_elastic_net_logistic_regression",
    "selected_candidate": selected_candidate_33E,
    "selected_C": selected_C_33E,
    "selected_l1_ratio": selected_l1_ratio_33E,
    "training_patients": 46803,
    "training_hospitals": 159,
    "test_patients": 11688,
    "test_hospitals": 39,
    "hospital_overlap": 0,
    "locked_platt_intercept": platt_intercept_33E,
    "locked_platt_slope": platt_slope_33E,
    "processed_feature_columns": int(
        len(processed_feature_names_33E)
    ),
    "nonzero_coefficients": int(nonzero_coefficients_33E),
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_33E,
    "selection_sha256": selection_sha_33E,
    "secure_prediction_table": prediction_table_id_33E,
    "patient_level_prediction_written_to_drive": False,
    "analysis_role": "additional_post_hoc_clinical_baseline",
    "predictors": CLINICAL_PREDICTORS_33E,
}

with open(
    evaluation_json_path_33E,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        evaluation_configuration_33E,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(evaluation_json_path_33E, "rb") as file_handle:
    evaluation_sha_33E = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    evaluation_sha_path_33E,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(evaluation_sha_33E + "\n")

# ------------------------------------------------------------
# 24. Final outputs
# ------------------------------------------------------------

pooled_integrity_33E = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_33E),
            pooled_oof_33E["id_row"].nunique(),
            pooled_oof_33E["candidate_id"].nunique(),
            pooled_oof_33E["inner_fold"].nunique(),
            EXPECTED_SPLIT_33E["training_events"],
            (
                EXPECTED_SPLIT_33E["training_rows"]
                - EXPECTED_SPLIT_33E["training_events"]
            ),
            int(
                pooled_oof_33E.duplicated(
                    subset=["candidate_id", "id_row"]
                ).sum()
            ),
            int(pooled_oof_33E["prediction_raw"].isna().sum()),
            int(
                (~pooled_oof_33E["prediction_raw"].between(0, 1)).sum()
            ),
            pooled_load_method_33E,
        ],
    }
)

print("\n33E OUTER-FOLD-4 INNER CHECKPOINT SUMMARY")
display(checkpoint_summary_33E)

print("\n33E OUTER-FOLD-4 POOLED OOF INTEGRITY")
display(pooled_integrity_33E)

print("\n33E OUTER-FOLD-4 CANDIDATE RESULTS")
display(candidate_results_33E)

print("\n33E OUTER-FOLD-4 SELECTED MODEL")
display(selected_model_33E)

print("\n33E OUTER-FOLD-4 FINAL MODEL SUMMARY")
display(final_model_summary_33E)

print("\n33E OUTER-FOLD-4 TEST RESULTS")
display(outer4_test_results_33E)

print("\n33E OUTER-FOLD-4 BIGQUERY VERIFICATION")
display(prediction_verification_33E)

print("\n33E OUTER-FOLD-4 TOP 20 ABSOLUTE COEFFICIENTS")
display(coefficient_table_33E.head(20))

print("\nSelection SHA-256:")
print(selection_sha_33E)

print("\nEvaluation SHA-256:")
print(evaluation_sha_33E)

print("\nSaved:")
print(fit_audit_path_33E)
print(checkpoint_summary_path_33E)
print(candidate_results_path_33E)
print(selected_model_path_33E)
print(selection_json_path_33E)
print(selection_sha_path_33E)
print(test_results_path_33E)
print(model_summary_path_33E)
print(coefficient_path_33E)
print(evaluation_json_path_33E)
print(evaluation_sha_path_33E)

print(
    "\n33E PASS: Outer-fold-4 nested modelling "
    "and locked test evaluation are complete."
)
print(
    "All inner OOF and outer-test patient-level "
    "predictions were stored only in BigQuery."
)
print(
    "No patient-level prediction file was "
    "written to Google Drive."
)

_ = gc.collect()

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from sklearn.exceptions import ConvergenceWarning

from IPython.display import display

# ============================================================
# 33B — PARSIMONIOUS CLINICAL BASELINE OUTER FOLD 4
#
# Resume-safe:
# - Each inner fold is stored in a separate BigQuery table.
# - Completed inner folds are automatically skipped.
#
# BigQuery free-tier compatible:
# - No DELETE / INSERT / UPDATE / MERGE is used.
#
# Privacy:
# - Patient-level predictions are stored only in BigQuery.
# - No patient-level prediction file is written to Drive.
# ============================================================

print("STARTING PARSIMONIOUS CLINICAL BASELINE OUTER FOLD 4 — CODE VERSION 33E-R1")

OUTER_FOLD_33E = 4
MODEL_RANDOM_SEED_33E = 20260721

EXPECTED_PROTOCOL_SHA_33E = (
    "94b0abb218dbef4e349702ea2824ca4e"
    "31dfbe36c53efa05ba1a8bd1f38f835e"
)

EXPECTED_AMENDMENT_SHA_33E = (
    "fcbcf857caa9aaad7ffd6691b4b61ac6"
    "503d09a6eee80db84bc2967ddf1f1a6a"
)

EXPECTED_CORRECTION_SHA_33E = (
    "ea7ab2747aad0c20157da0988b2fbeb1"
    "6ce007b4c92bf785d9ef9d363862ecfa"
)

EXPECTED_SPLIT_33E = {
    "training_rows": 46803,
    "test_rows": 11688,
    "training_hospitals": 159,
    "test_hospitals": 39,
    "training_events": 2426,
    "test_events": 606,
}

EXPECTED_INNER_33E = {
    1: {"validation_rows": 10116, "validation_events": 554},
    2: {"validation_rows": 8161, "validation_events": 381},
    3: {"validation_rows": 6957, "validation_events": 358},
    4: {"validation_rows": 12682, "validation_events": 663},
    5: {"validation_rows": 8887, "validation_events": 470},
}

CLINICAL_PREDICTORS_33E = [
    "x_age_years",
    "x_sex",
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
    "x_lab_bun_last",
    "x_vital_respiratory_rate_last",
    "x_vital_noninvasive_systolic_bp_last",
]

CLINICAL_NUMERIC_33E = [
    "x_age_years",
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
    "x_lab_bun_last",
    "x_vital_respiratory_rate_last",
    "x_vital_noninvasive_systolic_bp_last",
]

CLINICAL_CATEGORICAL_33E = ["x_sex"]

# ------------------------------------------------------------
# 1. Required runtime objects
# ------------------------------------------------------------

required_objects_33E = [
    "core_df_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_33E = [
    name for name in required_objects_33E if name not in globals()
]

if missing_objects_33E:
    raise RuntimeError(
        "Eksik RAM nesneleri var: "
        + ", ".join(missing_objects_33E)
        + ". Önce 07A ve 07B hücrelerini çalıştır."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"58.491 satır bekleniyordu; {len(core_df_07B)} bulundu."
    )

missing_predictors_33E = [
    column for column in CLINICAL_PREDICTORS_33E
    if column not in core_df_07B.columns
]

if missing_predictors_33E:
    raise RuntimeError(
        "Kilitli klinik predictor(lar) core_df_07B içinde yok: "
        + ", ".join(missing_predictors_33E)
    )

# ------------------------------------------------------------
# 2. Locked protocol SHA check
# ------------------------------------------------------------

protocol_sha_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A_locked_parsimonious_clinical_baseline_protocol_v1_SHA256.txt",
)

if not os.path.exists(protocol_sha_path_33E):
    raise FileNotFoundError(
        "Model protokolü SHA dosyası bulunamadı: "
        + protocol_sha_path_33E
    )

with open(protocol_sha_path_33E, "r", encoding="utf-8") as file_handle:
    observed_protocol_sha_33E = file_handle.read().strip()

if observed_protocol_sha_33E != EXPECTED_PROTOCOL_SHA_33E:
    raise RuntimeError(
        "Kilitli model protokolü SHA değeri değişmiş: "
        + observed_protocol_sha_33E
    )

amendment_sha_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A1_parsimonious_clinical_baseline_convergence_amendment_v1_SHA256.txt",
)

if not os.path.exists(amendment_sha_path_33E):
    raise FileNotFoundError(
        "Önce 33A1 convergence amendment scriptini çalıştır: "
        + amendment_sha_path_33E
    )

with open(amendment_sha_path_33E, "r", encoding="utf-8") as file_handle:
    observed_amendment_sha_33E = file_handle.read().strip()

if observed_amendment_sha_33E != EXPECTED_AMENDMENT_SHA_33E:
    raise RuntimeError(
        "33A1 amendment SHA değeri değişmiş: "
        + observed_amendment_sha_33E
    )

print("33A protocol SHA guard: PASS")
print("33A1 convergence amendment SHA guard: PASS")

correction_sha_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A2_parsimonious_clinical_baseline_full_inner_refit_correction_v1_SHA256.txt",
)

if not os.path.exists(correction_sha_path_33E):
    raise FileNotFoundError(
        "Önce 33A2 correction scriptini çalıştır: "
        + correction_sha_path_33E
    )

with open(correction_sha_path_33E, "r", encoding="utf-8") as file_handle:
    observed_correction_sha_33E = file_handle.read().strip()

if observed_correction_sha_33E != EXPECTED_CORRECTION_SHA_33E:
    raise RuntimeError(
        "33A2 correction SHA değeri değişmiş: "
        + observed_correction_sha_33E
    )

print("33A2 full inner-refit correction SHA guard: PASS")

# ------------------------------------------------------------
# 3. Locked inner-hospital mapping
# ------------------------------------------------------------

inner_mapping_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_33E):
    raise FileNotFoundError(
        "Kilitli iç kat haritası bulunamadı: "
        + inner_mapping_path_33E
    )

inner_mapping_all_33E = pd.read_csv(
    inner_mapping_path_33E,
    dtype={"group_hospital": str},
)

inner_mapping_part_33E = (
    inner_mapping_all_33E.loc[
        inner_mapping_all_33E["outer_fold"].astype(int) == OUTER_FOLD_33E,
        ["group_hospital", "inner_fold"],
    ]
    .copy()
)

inner_mapping_part_33E["group_hospital"] = (
    inner_mapping_part_33E["group_hospital"].astype(str)
)
inner_mapping_part_33E["inner_fold"] = (
    inner_mapping_part_33E["inner_fold"].astype(int)
)

if len(inner_mapping_part_33E) != 159:
    raise RuntimeError(
        "Outer-fold-4 training set için 159 hospital assignment bekleniyordu."
    )

if inner_mapping_part_33E["group_hospital"].duplicated().any():
    raise RuntimeError("İç kat haritasında yinelenen hastane var.")

hospital_to_inner_fold_33E = dict(
    zip(
        inner_mapping_part_33E["group_hospital"],
        inner_mapping_part_33E["inner_fold"],
    )
)

# ------------------------------------------------------------
# 4. Prepare model matrices
# ------------------------------------------------------------

X_all_33E = core_df_07B[CLINICAL_PREDICTORS_33E].copy()

for column in CLINICAL_NUMERIC_33E:
    X_all_33E[column] = pd.to_numeric(
        X_all_33E[column], errors="coerce"
    ).astype("float64")

for column in CLINICAL_CATEGORICAL_33E:
    category_series = X_all_33E[column].astype("object")
    X_all_33E[column] = category_series.where(
        pd.notna(category_series), np.nan
    )

outer_fold_vector_33E = (
    core_df_07B["outer_fold"].astype(int).to_numpy()
)

outer_training_mask_33E = outer_fold_vector_33E != OUTER_FOLD_33E
outer_test_mask_33E = outer_fold_vector_33E == OUTER_FOLD_33E

X_outer_training_33E = (
    X_all_33E.loc[outer_training_mask_33E].reset_index(drop=True)
)
X_outer_test_33E = (
    X_all_33E.loc[outer_test_mask_33E].reset_index(drop=True)
)

outer_training_meta_33E = (
    core_df_07B.loc[
        outer_training_mask_33E,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_33E = (
    core_df_07B.loc[
        outer_test_mask_33E,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [outer_training_meta_33E, outer_test_meta_33E]:
    dataframe["id_row"] = dataframe["id_row"].astype(str)
    dataframe["group_hospital"] = dataframe["group_hospital"].astype(str)
    dataframe["label_stage23"] = dataframe["label_stage23"].astype(int)

y_outer_training_33E = (
    outer_training_meta_33E["label_stage23"].to_numpy(dtype=np.int8)
)
y_outer_test_33E = (
    outer_test_meta_33E["label_stage23"].to_numpy(dtype=np.int8)
)

groups_outer_training_33E = (
    outer_training_meta_33E["group_hospital"].to_numpy(dtype=str)
)

training_hospitals_33E = set(
    outer_training_meta_33E["group_hospital"]
)
test_hospitals_33E = set(
    outer_test_meta_33E["group_hospital"]
)
hospital_overlap_33E = training_hospitals_33E & test_hospitals_33E

if hospital_overlap_33E:
    raise RuntimeError("Dış eğitim ve test hastaneleri çakışıyor.")

actual_split_33E = {
    "training_rows": len(X_outer_training_33E),
    "test_rows": len(X_outer_test_33E),
    "training_hospitals": len(training_hospitals_33E),
    "test_hospitals": len(test_hospitals_33E),
    "training_events": int(y_outer_training_33E.sum()),
    "test_events": int(y_outer_test_33E.sum()),
}

for metric, expected_value in EXPECTED_SPLIT_33E.items():
    actual_value = actual_split_33E[metric]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: bulunan={actual_value}, beklenen={expected_value}"
        )

inner_fold_vector_33E = np.array(
    [
        hospital_to_inner_fold_33E.get(hospital, -1)
        for hospital in groups_outer_training_33E
    ],
    dtype=int,
)

if (inner_fold_vector_33E == -1).any():
    raise RuntimeError(
        "Bazı dış eğitim hastanelerine iç kat atanmadı."
    )

if set(np.unique(inner_fold_vector_33E)) != {1, 2, 3, 4, 5}:
    raise RuntimeError("İç kat değerleri 1–5 değil.")

for inner_fold, expected in EXPECTED_INNER_33E.items():
    validation_mask = inner_fold_vector_33E == inner_fold
    observed_rows = int(validation_mask.sum())
    observed_events = int(y_outer_training_33E[validation_mask].sum())

    if observed_rows != expected["validation_rows"]:
        raise RuntimeError(
            f"Inner {inner_fold} validation_rows: "
            f"bulunan={observed_rows}, "
            f"beklenen={expected['validation_rows']}"
        )

    if observed_events != expected["validation_events"]:
        raise RuntimeError(
            f"Inner {inner_fold} validation_events: "
            f"bulunan={observed_events}, "
            f"beklenen={expected['validation_events']}"
        )

# ------------------------------------------------------------
# 5. Locked candidate grid
# ------------------------------------------------------------

candidate_grid_33E = [
    {"candidate_id": "CLIN01", "C": 0.03, "l1_ratio": 0.00},
    {"candidate_id": "CLIN02", "C": 0.10, "l1_ratio": 0.00},
    {"candidate_id": "CLIN03", "C": 0.30, "l1_ratio": 0.00},
    {"candidate_id": "CLIN04", "C": 0.10, "l1_ratio": 0.25},
    {"candidate_id": "CLIN05", "C": 0.30, "l1_ratio": 0.25},
    {"candidate_id": "CLIN06", "C": 0.30, "l1_ratio": 0.50},
]

# ------------------------------------------------------------
# 6. Preprocessing and model factories
# ------------------------------------------------------------

def make_preprocessor_33E():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
            (
                "scaler",
                StandardScaler(with_mean=False),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                CLINICAL_NUMERIC_33E,
            ),
            (
                "categorical",
                categorical_pipeline,
                CLINICAL_CATEGORICAL_33E,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_model_33E(C_value, l1_ratio_value):
    return LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=float(C_value),
        l1_ratio=float(l1_ratio_value),
        class_weight=None,
        max_iter=20000,
        tol=1e-4,
        random_state=MODEL_RANDOM_SEED_33E,
    )


def checkpoint_table_id_33E(inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_clinical_lr_inner_oof_outer4_inner{inner_fold}_v2"
    )

# ------------------------------------------------------------
# 7. BigQuery checkpoint verification
# ------------------------------------------------------------

def verify_checkpoint_33E(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):
    table_id = checkpoint_table_id_33E(inner_fold)

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS distinct_id_count,
      COUNT(DISTINCT candidate_id) AS candidate_count,
      COUNT(DISTINCT outer_fold) AS outer_fold_count,
      COUNT(DISTINCT inner_fold) AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(candidate_id, '|', id_row)
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(
        prediction_raw < 0 OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(sql, location=BQ_LOCATION).to_dataframe()
    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows * len(candidate_grid_33E)
    )
    expected_positive_rows = (
        expected_validation_events * len(candidate_grid_33E)
    )
    expected_negative_rows = (
        (expected_validation_rows - expected_validation_events)
        * len(candidate_grid_33E)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": expected_validation_rows,
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": expected_total_rows,
        "positive_prediction_rows": expected_positive_rows,
        "negative_prediction_rows": expected_negative_rows,
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": 4,
        "maximum_outer_fold": 4,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failure_items = []

    for field, expected_value in expected_values.items():
        actual_value = int(row[field])
        if actual_value != expected_value:
            complete = False
            failure_items.append(
                f"{field}={actual_value}, expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failure_items),
        "check": check,
        "row": row,
    }

# ------------------------------------------------------------
# 8. BigQuery load schema
# ------------------------------------------------------------

checkpoint_load_config_33E = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("inner_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("candidate_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

# ------------------------------------------------------------
# 9. Aggregate fit-audit file
# ------------------------------------------------------------

fit_audit_columns_33E = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "C",
    "l1_ratio",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "convergence_warnings",
    "elapsed_seconds",
]

fit_audit_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_inner_fit_audit_outer4.csv",
)

if os.path.exists(fit_audit_path_33E):
    fit_audit_33E = pd.read_csv(fit_audit_path_33E)
else:
    fit_audit_33E = pd.DataFrame(columns=fit_audit_columns_33E)

for column in fit_audit_columns_33E:
    if column not in fit_audit_33E.columns:
        fit_audit_33E[column] = np.nan

fit_audit_33E = fit_audit_33E[fit_audit_columns_33E].copy()

# ------------------------------------------------------------
# 10. Train six candidates in five locked inner folds
# ------------------------------------------------------------

for inner_fold in range(1, 6):
    inner_training_mask = inner_fold_vector_33E != inner_fold
    inner_validation_mask = inner_fold_vector_33E == inner_fold

    training_rows = int(inner_training_mask.sum())
    validation_rows = int(inner_validation_mask.sum())
    training_events = int(
        y_outer_training_33E[inner_training_mask].sum()
    )
    validation_events = int(
        y_outer_training_33E[inner_validation_mask].sum()
    )

    expected_inner = EXPECTED_INNER_33E[inner_fold]
    expected_training_rows = (
        EXPECTED_SPLIT_33E["training_rows"]
        - expected_inner["validation_rows"]
    )
    expected_training_events = (
        EXPECTED_SPLIT_33E["training_events"]
        - expected_inner["validation_events"]
    )

    if training_rows != expected_training_rows:
        raise RuntimeError(
            f"Inner {inner_fold} training_rows: "
            f"bulunan={training_rows}, "
            f"beklenen={expected_training_rows}"
        )

    if training_events != expected_training_events:
        raise RuntimeError(
            f"Inner {inner_fold} training_events: "
            f"bulunan={training_events}, "
            f"beklenen={expected_training_events}"
        )

    existing_check = verify_checkpoint_33E(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if existing_check["complete"]:
        print(
            f"Outer 4 / inner {inner_fold}: "
            "permanent checkpoint already complete; "
            "skipping model fitting."
        )
        continue

    training_hospital_set = set(
        groups_outer_training_33E[inner_training_mask]
    )
    validation_hospital_set = set(
        groups_outer_training_33E[inner_validation_mask]
    )

    if training_hospital_set & validation_hospital_set:
        raise RuntimeError(
            f"Inner fold {inner_fold}: hastane çakışması bulundu."
        )

    print(f"\nOuter 4 / inner {inner_fold}")
    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_33E()
    preprocessing_started = time.time()

    X_inner_training_processed = preprocessor.fit_transform(
        X_outer_training_33E.loc[inner_training_mask]
    )
    X_inner_validation_processed = preprocessor.transform(
        X_outer_training_33E.loc[inner_validation_mask]
    )

    preprocessing_elapsed = time.time() - preprocessing_started

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "İşlenmiş eğitim ve doğrulama sütun sayıları farklı."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_elapsed, 2),
    )

    y_inner_training = y_outer_training_33E[inner_training_mask]
    y_inner_validation = y_outer_training_33E[inner_validation_mask]

    validation_ids = (
        outer_training_meta_33E.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_33E:
        candidate_id = candidate["candidate_id"]

        print(
            "  Fitting",
            candidate_id,
            "| C =",
            candidate["C"],
            "| l1_ratio =",
            candidate["l1_ratio"],
        )

        model = make_model_33E(
            candidate["C"],
            candidate["l1_ratio"],
        )

        fitting_started = time.time()

        with warnings.catch_warnings(record=True) as warning_records:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(
                X_inner_training_processed,
                y_inner_training,
            )

        fitting_elapsed = time.time() - fitting_started

        convergence_warning_count = sum(
            issubclass(warning.category, ConvergenceWarning)
            for warning in warning_records
        )

        validation_probabilities = model.predict_proba(
            X_inner_validation_processed
        )[:, 1]

        if np.isnan(validation_probabilities).any():
            raise RuntimeError(
                f"{candidate_id}, inner {inner_fold}: eksik tahmin."
            )

        if not np.all(
            (validation_probabilities >= 0)
            & (validation_probabilities <= 1)
        ):
            raise RuntimeError(
                f"{candidate_id}: geçersiz olasılık."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        4,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": y_inner_validation.astype(np.int64),
                    "prediction_raw": validation_probabilities.astype(
                        np.float64
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": 4,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "C": candidate["C"],
                "l1_ratio": candidate["l1_ratio"],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": validation_events,
                "processed_columns": int(
                    X_inner_training_processed.shape[1]
                ),
                "convergence_warnings": int(
                    convergence_warning_count
                ),
                "elapsed_seconds": float(fitting_elapsed),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows * len(candidate_grid_33E)
    )

    if len(checkpoint_df) != expected_checkpoint_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: checkpoint satır sayısı hatalı."
        )

    if checkpoint_df.duplicated(
        subset=["id_row", "candidate_id"]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "yinelenen aday–hasta tahmini var."
        )

    target_checkpoint_table = checkpoint_table_id_33E(inner_fold)

    print(
        "Uploading permanent checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_33E,
        location=BQ_LOCATION,
    ).result()

    new_audit_df = pd.DataFrame(current_audit_rows)

    if len(fit_audit_33E) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_33E["outer_fold"],
                    errors="coerce",
                )
                == 4
            )
            & (
                pd.to_numeric(
                    fit_audit_33E["inner_fold"],
                    errors="coerce",
                )
                == inner_fold
            )
        )
        fit_audit_33E = fit_audit_33E.loc[keep_mask].copy()

    if fit_audit_33E.empty:
        fit_audit_33E = new_audit_df.copy()
    else:
        fit_audit_33E = pd.concat(
            [fit_audit_33E, new_audit_df],
            ignore_index=True,
        )

    fit_audit_33E = (
        fit_audit_33E[fit_audit_columns_33E]
        .sort_values(
            ["outer_fold", "inner_fold", "candidate_id"]
        )
        .reset_index(drop=True)
    )

    fit_audit_33E.to_csv(
        fit_audit_path_33E,
        index=False,
    )

    completed_check = verify_checkpoint_33E(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint doğrulanamadı: "
            + completed_check["reason"]
        )

    print(
        f"Outer 4 / inner {inner_fold}: "
        "permanent checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 11. Final checkpoint summary
# ------------------------------------------------------------

checkpoint_summary_rows_33E = []

for inner_fold in range(1, 6):
    validation_mask = inner_fold_vector_33E == inner_fold
    validation_rows = int(validation_mask.sum())
    validation_events = int(
        y_outer_training_33E[validation_mask].sum()
    )

    final_check = verify_checkpoint_33E(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "son checkpoint denetimi başarısız. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_33E.append(
        {
            "outer_fold": 4,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(row["row_count"]),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(row["candidate_count"]),
            "positive_prediction_rows": int(
                row["positive_prediction_rows"]
            ),
            "negative_prediction_rows": int(
                row["negative_prediction_rows"]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check["table_id"],
        }
    )

checkpoint_summary_33E = (
    pd.DataFrame(checkpoint_summary_rows_33E)
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_33E["distinct_validation_patients"].sum()
) != EXPECTED_SPLIT_33E["training_rows"]:
    raise RuntimeError(
        "Toplam doğrulama hasta sayısı 46.803 değil."
    )

expected_total_oof_rows_33E = (
    EXPECTED_SPLIT_33E["training_rows"]
    * len(candidate_grid_33E)
)

if int(checkpoint_summary_33E["checkpoint_rows"].sum()) != (
    expected_total_oof_rows_33E
):
    raise RuntimeError("Toplam OOF tahmin satırı hatalı.")

checkpoint_summary_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_outer4_inner_checkpoint_summary.csv",
)
checkpoint_summary_33E.to_csv(
    checkpoint_summary_path_33E,
    index=False,
)

# ------------------------------------------------------------
# 12. Pool all inner OOF predictions
# ------------------------------------------------------------

checkpoint_tables_33E = [
    checkpoint_table_id_33E(inner_fold)
    for inner_fold in range(1, 6)
]

union_parts_33E = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_33E
]

SQL_LOAD_POOLED_OOF_33E = "\nUNION ALL\n".join(union_parts_33E)

print("\nLoading pooled outer-fold-4 inner OOF predictions...")

query_job_33E = client.query(
    SQL_LOAD_POOLED_OOF_33E,
    location=BQ_LOCATION,
)

try:
    pooled_oof_33E = query_job_33E.to_dataframe(
        create_bqstorage_client=True
    )
    pooled_load_method_33E = "BigQuery Storage API"
except Exception as fast_path_error_33E:
    print(
        "Storage API unavailable; using standard BigQuery download."
    )
    print("Message:", type(fast_path_error_33E).__name__)
    pooled_oof_33E = query_job_33E.to_dataframe(
        create_bqstorage_client=False
    )
    pooled_load_method_33E = "Standard BigQuery API"

pooled_oof_33E["id_row"] = pooled_oof_33E["id_row"].astype(str)
pooled_oof_33E["candidate_id"] = pooled_oof_33E["candidate_id"].astype(str)

for column in ["outer_fold", "inner_fold", "label_stage23"]:
    pooled_oof_33E[column] = pd.to_numeric(
        pooled_oof_33E[column], errors="raise"
    ).astype(int)

pooled_oof_33E["prediction_raw"] = pd.to_numeric(
    pooled_oof_33E["prediction_raw"], errors="raise"
).astype(float)

# ------------------------------------------------------------
# 13. Pooled OOF integrity checks
# ------------------------------------------------------------

if len(pooled_oof_33E) != expected_total_oof_rows_33E:
    raise RuntimeError("Pooled OOF satır sayısı hatalı.")

if set(pooled_oof_33E["outer_fold"].unique()) != {4}:
    raise RuntimeError("Pooled OOF içinde dış kat 4 dışında kayıt var.")

if set(pooled_oof_33E["inner_fold"].unique()) != {1, 2, 3, 4, 5}:
    raise RuntimeError("Pooled OOF iç katları 1–5 değil.")

if pooled_oof_33E.duplicated(
    subset=["candidate_id", "id_row"]
).any():
    raise RuntimeError(
        "Pooled OOF içinde yinelenen aday–hasta tahmini bulundu."
    )

if pooled_oof_33E["prediction_raw"].isna().any():
    raise RuntimeError("Pooled OOF içinde eksik tahmin var.")

if not pooled_oof_33E["prediction_raw"].between(0, 1).all():
    raise RuntimeError(
        "Pooled OOF içinde geçersiz olasılık değeri var."
    )

expected_candidate_ids_33E = {
    "CLIN01",
    "CLIN02",
    "CLIN03",
    "CLIN04",
    "CLIN05",
    "CLIN06",
}

if set(pooled_oof_33E["candidate_id"].unique()) != (
    expected_candidate_ids_33E
):
    raise RuntimeError("Altı kilitli aday bulunmuyor.")

candidate_patient_counts_33E = (
    pooled_oof_33E.groupby("candidate_id")["id_row"].nunique()
)

if not (
    candidate_patient_counts_33E
    == EXPECTED_SPLIT_33E["training_rows"]
).all():
    raise RuntimeError(
        "Her aday için 46.803 farklı OOF hastası yok."
    )

candidate_event_counts_33E = (
    pooled_oof_33E.groupby("candidate_id")["label_stage23"].sum()
)

if not (
    candidate_event_counts_33E
    == EXPECTED_SPLIT_33E["training_events"]
).all():
    raise RuntimeError("Her aday için 2.426 olay yok.")

patient_label_consistency_33E = (
    pooled_oof_33E.groupby("id_row")["label_stage23"].nunique()
)
if (patient_label_consistency_33E > 1).any():
    raise RuntimeError(
        "Aynı hastanın adaylar arasında outcome etiketi farklı."
    )

patient_inner_fold_consistency_33E = (
    pooled_oof_33E.groupby("id_row")["inner_fold"].nunique()
)
if (patient_inner_fold_consistency_33E > 1).any():
    raise RuntimeError(
        "Aynı hasta birden fazla iç doğrulama katında bulundu."
    )

# ------------------------------------------------------------
# 14. Metric helpers
# ------------------------------------------------------------

def probability_metrics_33E(y_true, probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(roc_auc_score(y_true, probabilities)),
        "auprc": float(average_precision_score(y_true, probabilities)),
        "brier": float(brier_score_loss(y_true, probabilities)),
        "log_loss": float(
            log_loss(y_true, probabilities, labels=[0, 1])
        ),
        "mean_predicted_risk": float(probabilities.mean()),
        "observed_event_rate": float(np.mean(y_true)),
    }


def probability_logit_33E(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        probabilities / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_33E(y_true, probabilities):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        probability_logit_33E(probabilities),
        y_true,
    )
    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 15. Candidate pooled-OOF performance
# ------------------------------------------------------------

candidate_result_rows_33E = []

for candidate in candidate_grid_33E:
    candidate_id = candidate["candidate_id"]

    candidate_oof = (
        pooled_oof_33E.loc[
            pooled_oof_33E["candidate_id"] == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_33E(
        candidate_oof["label_stage23"].to_numpy(dtype=int),
        candidate_oof["prediction_raw"].to_numpy(dtype=float),
    )

    fit_part = fit_audit_33E.loc[
        (
            pd.to_numeric(
                fit_audit_33E["outer_fold"],
                errors="coerce",
            )
            == 4
        )
        & (
            fit_audit_33E["candidate_id"].astype(str)
            == candidate_id
        )
    ]

    convergence_warnings = (
        int(
            pd.to_numeric(
                fit_part["convergence_warnings"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    elapsed_seconds = (
        float(
            pd.to_numeric(
                fit_part["elapsed_seconds"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_33E.append(
        {
            "candidate_id": candidate_id,
            "C": candidate["C"],
            "l1_ratio": candidate["l1_ratio"],
            **metrics,
            "convergence_warnings": convergence_warnings,
            "elapsed_seconds": elapsed_seconds,
        }
    )

candidate_results_33E = pd.DataFrame(candidate_result_rows_33E)

total_inner_convergence_warnings_33E = int(
    pd.to_numeric(
        candidate_results_33E["convergence_warnings"],
        errors="coerce",
    ).fillna(0).sum()
)

if total_inner_convergence_warnings_33E != 0:
    raise RuntimeError(
        "33B-R2 stopped BEFORE candidate selection and BEFORE outer-test "
        "evaluation because at least one inner candidate fit still emitted "
        f"a ConvergenceWarning under max_iter=20000. "
        f"Total warnings={total_inner_convergence_warnings_33E}. "
        "No test result may be accessed."
    )

print(
    "All 30 corrected inner candidate fits converged with zero warnings: PASS"
)

candidate_results_33E = (
    candidate_results_33E.sort_values(
        ["auprc", "auroc", "brier", "candidate_id"],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)

candidate_results_33E["selection_rank"] = np.arange(
    1,
    len(candidate_results_33E) + 1,
)

best_row_33E = candidate_results_33E.iloc[0]
selected_candidate_33E = str(best_row_33E["candidate_id"])
selected_C_33E = float(best_row_33E["C"])
selected_l1_ratio_33E = float(best_row_33E["l1_ratio"])

# ------------------------------------------------------------
# 16. Platt calibration on selected pooled inner OOF
# ------------------------------------------------------------

selected_oof_33E = (
    pooled_oof_33E.loc[
        pooled_oof_33E["candidate_id"] == selected_candidate_33E
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_33E = selected_oof_33E[
    "label_stage23"
].to_numpy(dtype=int)
selected_oof_probability_33E = selected_oof_33E[
    "prediction_raw"
].to_numpy(dtype=float)

platt_calibrator_33E = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_33E.fit(
    probability_logit_33E(selected_oof_probability_33E),
    selected_oof_y_33E,
)

platt_intercept_33E = float(platt_calibrator_33E.intercept_[0])
platt_slope_33E = float(platt_calibrator_33E.coef_[0][0])

if (
    not np.isfinite(platt_intercept_33E)
    or not np.isfinite(platt_slope_33E)
    or platt_slope_33E <= 0
):
    raise RuntimeError("Platt kalibrasyon katsayıları geçersiz.")

selected_model_33E = pd.DataFrame(
    [
        {
            "outer_fold": 4,
            "selected_candidate": selected_candidate_33E,
            "selected_C": selected_C_33E,
            "selected_l1_ratio": selected_l1_ratio_33E,
            "selection_metric_primary": "pooled_inner_oof_auprc",
            "inner_oof_auprc": float(best_row_33E["auprc"]),
            "inner_oof_auroc": float(best_row_33E["auroc"]),
            "inner_oof_brier": float(best_row_33E["brier"]),
            "inner_oof_log_loss": float(best_row_33E["log_loss"]),
            "inner_oof_mean_predicted_risk": float(
                best_row_33E["mean_predicted_risk"]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_33E["observed_event_rate"]
            ),
            "platt_intercept": platt_intercept_33E,
            "platt_slope": platt_slope_33E,
            "protocol_sha256": EXPECTED_PROTOCOL_SHA_33E,
            "numerical_amendment_sha256": EXPECTED_AMENDMENT_SHA_33E,
            "full_inner_refit_correction_sha256": EXPECTED_CORRECTION_SHA_33E,
            "corrected_checkpoint_namespace": "v2",
            "analysis_status": "post_hoc_exploratory_corrected",
            "full_inner_refit_correction_sha256": EXPECTED_CORRECTION_SHA_33E,
            "corrected_checkpoint_namespace": "v2",
            "analysis_status": "post_hoc_exploratory_corrected",
            "model_max_iter": 20000,
        }
    ]
)

# ------------------------------------------------------------
# 17. Lock selection/calibration aggregate files
# ------------------------------------------------------------

candidate_results_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_candidate_results_outer4.csv",
)
selected_model_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_selected_model_outer4.csv",
)
selection_json_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_selection_calibration_outer4.json",
)
selection_sha_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_selection_calibration_outer4_SHA256.txt",
)

candidate_results_33E.to_csv(
    candidate_results_path_33E,
    index=False,
)
selected_model_33E.to_csv(
    selected_model_path_33E,
    index=False,
)

selection_configuration_33E = {
    "outer_fold": 4,
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_33E,
    "selection_metric_primary": "pooled inner out-of-fold AUPRC",
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
        "candidate_id ascending immutable tie-break",
    ],
    "selected_candidate": selected_candidate_33E,
    "selected_C": selected_C_33E,
    "selected_l1_ratio": selected_l1_ratio_33E,
    "inner_oof_auprc": float(best_row_33E["auprc"]),
    "inner_oof_auroc": float(best_row_33E["auroc"]),
    "inner_oof_brier": float(best_row_33E["brier"]),
    "platt_intercept": platt_intercept_33E,
    "platt_slope": platt_slope_33E,
    "inner_checkpoint_tables": checkpoint_tables_33E,
    "patient_level_oof_written_to_drive": False,
    "analysis_role": "additional_post_hoc_clinical_baseline",
    "predictors": CLINICAL_PREDICTORS_33E,
}

with open(
    selection_json_path_33E,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        selection_configuration_33E,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(selection_json_path_33E, "rb") as file_handle:
    selection_sha_33E = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    selection_sha_path_33E,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(selection_sha_33E + "\n")

# ------------------------------------------------------------
# 18. Fit final selected outer-fold-4 model
# ------------------------------------------------------------

final_pipeline_33E = Pipeline(
    steps=[
        ("preprocessor", make_preprocessor_33E()),
        (
            "model",
            make_model_33E(
                selected_C_33E,
                selected_l1_ratio_33E,
            ),
        ),
    ]
)

print(
    "\nFitting selected outer-fold-4 model "
    "on all 46,803 training patients..."
)

final_fit_started_33E = time.time()

with warnings.catch_warnings(record=True) as final_warning_records_33E:
    warnings.simplefilter("always", ConvergenceWarning)
    final_pipeline_33E.fit(
        X_outer_training_33E,
        y_outer_training_33E,
    )

final_fit_elapsed_33E = time.time() - final_fit_started_33E

final_model_n_iter_33E = int(
    np.max(
        np.asarray(
            final_pipeline_33E.named_steps["model"].n_iter_
        )
    )
)

final_convergence_warnings_33E = sum(
    issubclass(warning.category, ConvergenceWarning)
    for warning in final_warning_records_33E
)

if final_convergence_warnings_33E != 0:
    raise RuntimeError(
        "Nihai outer-fold-4 klinik baseline modeli max_iter=20000 "
        f"altında da yakınsamadı. n_iter_={final_model_n_iter_33E}"
    )

# ------------------------------------------------------------
# 19. Outer-fold-4 test predictions and metrics
# ------------------------------------------------------------

outer4_raw_probabilities_33E = final_pipeline_33E.predict_proba(
    X_outer_test_33E
)[:, 1]

raw_clipped_33E = np.clip(
    outer4_raw_probabilities_33E,
    1e-6,
    1 - 1e-6,
)
raw_logit_33E = np.log(
    raw_clipped_33E / (1 - raw_clipped_33E)
)

outer4_platt_probabilities_33E = expit(
    platt_intercept_33E + platt_slope_33E * raw_logit_33E
)

for probabilities, name in [
    (outer4_raw_probabilities_33E, "raw"),
    (outer4_platt_probabilities_33E, "platt"),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(f"{name} tahminlerinde eksik değer var.")

    if not np.all(
        (probabilities >= 0) & (probabilities <= 1)
    ):
        raise RuntimeError(
            f"{name} tahminlerinde geçersiz olasılık değeri var."
        )

raw_metrics_33E = probability_metrics_33E(
    y_outer_test_33E,
    outer4_raw_probabilities_33E,
)
platt_metrics_33E = probability_metrics_33E(
    y_outer_test_33E,
    outer4_platt_probabilities_33E,
)

raw_calibration_intercept_33E, raw_calibration_slope_33E = (
    calibration_intercept_slope_33E(
        y_outer_test_33E,
        outer4_raw_probabilities_33E,
    )
)

platt_calibration_intercept_33E, platt_calibration_slope_33E = (
    calibration_intercept_slope_33E(
        y_outer_test_33E,
        outer4_platt_probabilities_33E,
    )
)

outer4_test_results_33E = pd.DataFrame(
    [
        {
            "outer_fold": 4,
            "model": "parsimonious_clinical_logistic",
            "probability_type": "raw",
            **raw_metrics_33E,
            "calibration_intercept": raw_calibration_intercept_33E,
            "calibration_slope": raw_calibration_slope_33E,
        },
        {
            "outer_fold": 4,
            "model": "parsimonious_clinical_logistic",
            "probability_type": "platt_calibrated",
            **platt_metrics_33E,
            "calibration_intercept": platt_calibration_intercept_33E,
            "calibration_slope": platt_calibration_slope_33E,
        },
    ]
)

# ------------------------------------------------------------
# 20. Feature coefficient audit
# ------------------------------------------------------------

fitted_preprocessor_33E = final_pipeline_33E.named_steps[
    "preprocessor"
]
fitted_model_33E = final_pipeline_33E.named_steps["model"]

processed_feature_names_33E = (
    fitted_preprocessor_33E.get_feature_names_out()
)
model_coefficients_33E = fitted_model_33E.coef_.reshape(-1)

if len(processed_feature_names_33E) != len(model_coefficients_33E):
    raise RuntimeError("Feature ve katsayı sayıları uyuşmuyor.")

if len(set(processed_feature_names_33E)) != len(
    processed_feature_names_33E
):
    raise RuntimeError("İşlenmiş feature adlarında yinelenme var.")

coefficient_table_33E = pd.DataFrame(
    {
        "processed_feature": processed_feature_names_33E,
        "coefficient": model_coefficients_33E,
    }
)
coefficient_table_33E["absolute_coefficient"] = (
    coefficient_table_33E["coefficient"].abs()
)
coefficient_table_33E["is_nonzero"] = ~np.isclose(
    coefficient_table_33E["coefficient"],
    0.0,
    atol=1e-12,
)
coefficient_table_33E["absolute_rank"] = (
    coefficient_table_33E["absolute_coefficient"]
    .rank(method="first", ascending=False)
    .astype(int)
)
coefficient_table_33E = (
    coefficient_table_33E.sort_values(
        ["absolute_coefficient", "processed_feature"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

nonzero_coefficients_33E = int(
    coefficient_table_33E["is_nonzero"].sum()
)

final_model_summary_33E = pd.DataFrame(
    [
        {
            "outer_fold": 4,
            "selected_candidate": selected_candidate_33E,
            "selected_C": selected_C_33E,
            "selected_l1_ratio": selected_l1_ratio_33E,
            "training_patients": len(X_outer_training_33E),
            "training_hospitals": len(training_hospitals_33E),
            "training_events": int(y_outer_training_33E.sum()),
            "test_patients": len(X_outer_test_33E),
            "test_hospitals": len(test_hospitals_33E),
            "test_events": int(y_outer_test_33E.sum()),
            "hospital_overlap": len(hospital_overlap_33E),
            "processed_feature_columns": len(
                processed_feature_names_33E
            ),
            "nonzero_coefficients": nonzero_coefficients_33E,
            "model_intercept": float(
                fitted_model_33E.intercept_[0]
            ),
            "convergence_warnings": final_convergence_warnings_33E,
            "final_model_n_iter": final_model_n_iter_33E,
            "fit_elapsed_seconds": float(final_fit_elapsed_33E),
            "locked_platt_intercept": platt_intercept_33E,
            "locked_platt_slope": platt_slope_33E,
            "protocol_sha256": EXPECTED_PROTOCOL_SHA_33E,
            "numerical_amendment_sha256": EXPECTED_AMENDMENT_SHA_33E,
            "model_max_iter": 20000,
            "selection_sha256": selection_sha_33E,
        }
    ]
)

# ------------------------------------------------------------
# 21. Secure BigQuery outer-test checkpoint
# ------------------------------------------------------------

outer4_prediction_df_33E = pd.DataFrame(
    {
        "id_row": outer_test_meta_33E["id_row"].astype(str),
        "outer_fold": np.full(
            len(outer_test_meta_33E),
            4,
            dtype=np.int64,
        ),
        "label_stage23": y_outer_test_33E.astype(np.int64),
        "prediction_raw": outer4_raw_probabilities_33E.astype(
            np.float64
        ),
        "prediction_platt": outer4_platt_probabilities_33E.astype(
            np.float64
        ),
        "model_name": "parsimonious_clinical_logistic",
        "model_version": "clinical8_v1_nested_cv",
    }
)

if len(outer4_prediction_df_33E) != 11688:
    raise RuntimeError(
        "Outer-fold-4 tahmin satır sayısı 11,688 değil."
    )

if outer4_prediction_df_33E["id_row"].duplicated().any():
    raise RuntimeError(
        "Outer-fold-4 tahminlerinde yinelenen id_row var."
    )

if int(
    outer4_prediction_df_33E["label_stage23"].sum()
) != 606:
    raise RuntimeError("Outer-fold-4 event count is not 606 değil.")

prediction_table_id_33E = (
    f"{TARGET_DATASET}."
    "model_clinical_lr_outer_predictions_outer4_v2"
)

prediction_load_config_33E = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("prediction_platt", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("model_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("model_version", "STRING", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

print("\nUploading secure outer-fold-4 prediction checkpoint:")
print(prediction_table_id_33E)

client.load_table_from_dataframe(
    outer4_prediction_df_33E,
    prediction_table_id_33E,
    job_config=prediction_load_config_33E,
    location=BQ_LOCATION,
).result()

# ------------------------------------------------------------
# 22. BigQuery outer-test checkpoint verification
# ------------------------------------------------------------

SQL_VERIFY_PREDICTIONS_33E = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL) AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL) AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0 OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0 OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw) AS minimum_raw_probability,
  MAX(prediction_raw) AS maximum_raw_probability,
  MIN(prediction_platt) AS minimum_platt_probability,
  MAX(prediction_platt) AS maximum_platt_probability
FROM `{prediction_table_id_33E}`;
"""

prediction_verification_33E = client.query(
    SQL_VERIFY_PREDICTIONS_33E,
    location=BQ_LOCATION,
).to_dataframe()

verification_row_33E = prediction_verification_33E.iloc[0]

expected_prediction_values_33E = {
    "prediction_rows": 11688,
    "distinct_rows": 11688,
    "outer_folds": 1,
    "minimum_outer_fold": 4,
    "maximum_outer_fold": 4,
    "events": 606,
    "nonevents": 11082,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in expected_prediction_values_33E.items():
    actual_value = int(verification_row_33E[field])
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: bulunan={actual_value}, beklenen={expected_value}"
        )

# ------------------------------------------------------------
# 23. Save only aggregate and feature-level outputs to Drive
# ------------------------------------------------------------

test_results_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_outer4_test_results.csv",
)
model_summary_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_final_model_outer4.csv",
)
coefficient_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_coefficients_outer4.csv",
)
evaluation_json_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_final_evaluation_outer4.json",
)
evaluation_sha_path_33E = os.path.join(
    MODEL_OUTPUT_DIR,
    "33E_clinical_final_evaluation_outer4_SHA256.txt",
)

outer4_test_results_33E.to_csv(
    test_results_path_33E,
    index=False,
)
final_model_summary_33E.to_csv(
    model_summary_path_33E,
    index=False,
)
coefficient_table_33E.to_csv(
    coefficient_path_33E,
    index=False,
)

evaluation_configuration_33E = {
    "outer_fold": 4,
    "model_family": "parsimonious_clinical_elastic_net_logistic_regression",
    "selected_candidate": selected_candidate_33E,
    "selected_C": selected_C_33E,
    "selected_l1_ratio": selected_l1_ratio_33E,
    "training_patients": 46803,
    "training_hospitals": 159,
    "test_patients": 11688,
    "test_hospitals": 39,
    "hospital_overlap": 0,
    "locked_platt_intercept": platt_intercept_33E,
    "locked_platt_slope": platt_slope_33E,
    "processed_feature_columns": int(
        len(processed_feature_names_33E)
    ),
    "nonzero_coefficients": int(nonzero_coefficients_33E),
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_33E,
    "selection_sha256": selection_sha_33E,
    "secure_prediction_table": prediction_table_id_33E,
    "patient_level_prediction_written_to_drive": False,
    "analysis_role": "additional_post_hoc_clinical_baseline",
    "predictors": CLINICAL_PREDICTORS_33E,
}

with open(
    evaluation_json_path_33E,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        evaluation_configuration_33E,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(evaluation_json_path_33E, "rb") as file_handle:
    evaluation_sha_33E = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    evaluation_sha_path_33E,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(evaluation_sha_33E + "\n")

# ------------------------------------------------------------
# 24. Final outputs
# ------------------------------------------------------------

pooled_integrity_33E = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_33E),
            pooled_oof_33E["id_row"].nunique(),
            pooled_oof_33E["candidate_id"].nunique(),
            pooled_oof_33E["inner_fold"].nunique(),
            EXPECTED_SPLIT_33E["training_events"],
            (
                EXPECTED_SPLIT_33E["training_rows"]
                - EXPECTED_SPLIT_33E["training_events"]
            ),
            int(
                pooled_oof_33E.duplicated(
                    subset=["candidate_id", "id_row"]
                ).sum()
            ),
            int(pooled_oof_33E["prediction_raw"].isna().sum()),
            int(
                (~pooled_oof_33E["prediction_raw"].between(0, 1)).sum()
            ),
            pooled_load_method_33E,
        ],
    }
)

print("\n33E OUTER-FOLD-4 INNER CHECKPOINT SUMMARY")
display(checkpoint_summary_33E)

print("\n33E OUTER-FOLD-4 POOLED OOF INTEGRITY")
display(pooled_integrity_33E)

print("\n33E OUTER-FOLD-4 CANDIDATE RESULTS")
display(candidate_results_33E)

print("\n33E OUTER-FOLD-4 SELECTED MODEL")
display(selected_model_33E)

print("\n33E OUTER-FOLD-4 FINAL MODEL SUMMARY")
display(final_model_summary_33E)

print("\n33E OUTER-FOLD-4 TEST RESULTS")
display(outer4_test_results_33E)

print("\n33E OUTER-FOLD-4 BIGQUERY VERIFICATION")
display(prediction_verification_33E)

print("\n33E OUTER-FOLD-4 TOP 20 ABSOLUTE COEFFICIENTS")
display(coefficient_table_33E.head(20))

print("\nSelection SHA-256:")
print(selection_sha_33E)

print("\nEvaluation SHA-256:")
print(evaluation_sha_33E)

print("\nSaved:")
print(fit_audit_path_33E)
print(checkpoint_summary_path_33E)
print(candidate_results_path_33E)
print(selected_model_path_33E)
print(selection_json_path_33E)
print(selection_sha_path_33E)
print(test_results_path_33E)
print(model_summary_path_33E)
print(coefficient_path_33E)
print(evaluation_json_path_33E)
print(evaluation_sha_path_33E)

print(
    "\n33E PASS: Outer-fold-4 nested modelling "
    "and locked test evaluation are complete."
)
print(
    "All inner OOF and outer-test patient-level "
    "predictions were stored only in BigQuery."
)
print(
    "No patient-level prediction file was "
    "written to Google Drive."
)

_ = gc.collect()

In [ ]:
import os
import gc
import json
import time
import hashlib
import warnings

import numpy as np
import pandas as pd

from scipy.special import expit

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from sklearn.exceptions import ConvergenceWarning

from IPython.display import display

# ============================================================
# 33B — PARSIMONIOUS CLINICAL BASELINE OUTER FOLD 5
#
# Resume-safe:
# - Each inner fold is stored in a separate BigQuery table.
# - Completed inner folds are automatically skipped.
#
# BigQuery free-tier compatible:
# - No DELETE / INSERT / UPDATE / MERGE is used.
#
# Privacy:
# - Patient-level predictions are stored only in BigQuery.
# - No patient-level prediction file is written to Drive.
# ============================================================

print("STARTING PARSIMONIOUS CLINICAL BASELINE OUTER FOLD 5 — CODE VERSION 33F")

OUTER_FOLD_33F = 5
MODEL_RANDOM_SEED_33F = 20260721

EXPECTED_PROTOCOL_SHA_33F = (
    "94b0abb218dbef4e349702ea2824ca4e"
    "31dfbe36c53efa05ba1a8bd1f38f835e"
)

EXPECTED_AMENDMENT_SHA_33F = (
    "fcbcf857caa9aaad7ffd6691b4b61ac6"
    "503d09a6eee80db84bc2967ddf1f1a6a"
)

EXPECTED_CORRECTION_SHA_33F = (
    "ea7ab2747aad0c20157da0988b2fbeb1"
    "6ce007b4c92bf785d9ef9d363862ecfa"
)

EXPECTED_SPLIT_33F = {
    "training_rows": 46803,
    "test_rows": 11688,
    "training_hospitals": 159,
    "test_hospitals": 39,
    "training_events": 2426,
    "test_events": 606,
}

EXPECTED_INNER_33F = {
    1: {"validation_rows": 9010, "validation_events": 420},
    2: {"validation_rows": 6838, "validation_events": 360},
    3: {"validation_rows": 9532, "validation_events": 515},
    4: {"validation_rows": 12455, "validation_events": 651},
    5: {"validation_rows": 8968, "validation_events": 480},
}

CLINICAL_PREDICTORS_33F = [
    "x_age_years",
    "x_sex",
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
    "x_lab_bun_last",
    "x_vital_respiratory_rate_last",
    "x_vital_noninvasive_systolic_bp_last",
]

CLINICAL_NUMERIC_33F = [
    "x_age_years",
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
    "x_lab_bun_last",
    "x_vital_respiratory_rate_last",
    "x_vital_noninvasive_systolic_bp_last",
]

CLINICAL_CATEGORICAL_33F = ["x_sex"]

# ------------------------------------------------------------
# 1. Required runtime objects
# ------------------------------------------------------------

required_objects_33F = [
    "core_df_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_33F = [
    name for name in required_objects_33F if name not in globals()
]

if missing_objects_33F:
    raise RuntimeError(
        "Eksik RAM nesneleri var: "
        + ", ".join(missing_objects_33F)
        + ". Önce 07A ve 07B hücrelerini çalıştır."
    )

if len(core_df_07B) != 58491:
    raise RuntimeError(
        f"58.491 satır bekleniyordu; {len(core_df_07B)} bulundu."
    )

missing_predictors_33F = [
    column for column in CLINICAL_PREDICTORS_33F
    if column not in core_df_07B.columns
]

if missing_predictors_33F:
    raise RuntimeError(
        "Kilitli klinik predictor(lar) core_df_07B içinde yok: "
        + ", ".join(missing_predictors_33F)
    )

# ------------------------------------------------------------
# 2. Locked protocol SHA check
# ------------------------------------------------------------

protocol_sha_path_33F = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A_locked_parsimonious_clinical_baseline_protocol_v1_SHA256.txt",
)

if not os.path.exists(protocol_sha_path_33F):
    raise FileNotFoundError(
        "Model protokolü SHA dosyası bulunamadı: "
        + protocol_sha_path_33F
    )

with open(protocol_sha_path_33F, "r", encoding="utf-8") as file_handle:
    observed_protocol_sha_33F = file_handle.read().strip()

if observed_protocol_sha_33F != EXPECTED_PROTOCOL_SHA_33F:
    raise RuntimeError(
        "Kilitli model protokolü SHA değeri değişmiş: "
        + observed_protocol_sha_33F
    )

amendment_sha_path_33F = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A1_parsimonious_clinical_baseline_convergence_amendment_v1_SHA256.txt",
)

if not os.path.exists(amendment_sha_path_33F):
    raise FileNotFoundError(
        "Önce 33A1 convergence amendment scriptini çalıştır: "
        + amendment_sha_path_33F
    )

with open(amendment_sha_path_33F, "r", encoding="utf-8") as file_handle:
    observed_amendment_sha_33F = file_handle.read().strip()

if observed_amendment_sha_33F != EXPECTED_AMENDMENT_SHA_33F:
    raise RuntimeError(
        "33A1 amendment SHA değeri değişmiş: "
        + observed_amendment_sha_33F
    )

print("33A protocol SHA guard: PASS")
print("33A1 convergence amendment SHA guard: PASS")

correction_sha_path_33F = os.path.join(
    MODEL_OUTPUT_DIR,
    "33A2_parsimonious_clinical_baseline_full_inner_refit_correction_v1_SHA256.txt",
)

if not os.path.exists(correction_sha_path_33F):
    raise FileNotFoundError(
        "Önce 33A2 correction scriptini çalıştır: "
        + correction_sha_path_33F
    )

with open(correction_sha_path_33F, "r", encoding="utf-8") as file_handle:
    observed_correction_sha_33F = file_handle.read().strip()

if observed_correction_sha_33F != EXPECTED_CORRECTION_SHA_33F:
    raise RuntimeError(
        "33A2 correction SHA değeri değişmiş: "
        + observed_correction_sha_33F
    )

print("33A2 full inner-refit correction SHA guard: PASS")

# ------------------------------------------------------------
# 3. Locked inner-hospital mapping
# ------------------------------------------------------------

inner_mapping_path_33F = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_mapping_path_33F):
    raise FileNotFoundError(
        "Kilitli iç kat haritası bulunamadı: "
        + inner_mapping_path_33F
    )

inner_mapping_all_33F = pd.read_csv(
    inner_mapping_path_33F,
    dtype={"group_hospital": str},
)

inner_mapping_part_33F = (
    inner_mapping_all_33F.loc[
        inner_mapping_all_33F["outer_fold"].astype(int) == OUTER_FOLD_33F,
        ["group_hospital", "inner_fold"],
    ]
    .copy()
)

inner_mapping_part_33F["group_hospital"] = (
    inner_mapping_part_33F["group_hospital"].astype(str)
)
inner_mapping_part_33F["inner_fold"] = (
    inner_mapping_part_33F["inner_fold"].astype(int)
)

if len(inner_mapping_part_33F) != 159:
    raise RuntimeError(
        "Outer-fold-5 training set için 159 hospital assignment bekleniyordu."
    )

if inner_mapping_part_33F["group_hospital"].duplicated().any():
    raise RuntimeError("İç kat haritasında yinelenen hastane var.")

hospital_to_inner_fold_33F = dict(
    zip(
        inner_mapping_part_33F["group_hospital"],
        inner_mapping_part_33F["inner_fold"],
    )
)

# ------------------------------------------------------------
# 4. Prepare model matrices
# ------------------------------------------------------------

X_all_33F = core_df_07B[CLINICAL_PREDICTORS_33F].copy()

for column in CLINICAL_NUMERIC_33F:
    X_all_33F[column] = pd.to_numeric(
        X_all_33F[column], errors="coerce"
    ).astype("float64")

for column in CLINICAL_CATEGORICAL_33F:
    category_series = X_all_33F[column].astype("object")
    X_all_33F[column] = category_series.where(
        pd.notna(category_series), np.nan
    )

outer_fold_vector_33F = (
    core_df_07B["outer_fold"].astype(int).to_numpy()
)

outer_training_mask_33F = outer_fold_vector_33F != OUTER_FOLD_33F
outer_test_mask_33F = outer_fold_vector_33F == OUTER_FOLD_33F

X_outer_training_33F = (
    X_all_33F.loc[outer_training_mask_33F].reset_index(drop=True)
)
X_outer_test_33F = (
    X_all_33F.loc[outer_test_mask_33F].reset_index(drop=True)
)

outer_training_meta_33F = (
    core_df_07B.loc[
        outer_training_mask_33F,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

outer_test_meta_33F = (
    core_df_07B.loc[
        outer_test_mask_33F,
        ["id_row", "group_hospital", "label_stage23"],
    ]
    .copy()
    .reset_index(drop=True)
)

for dataframe in [outer_training_meta_33F, outer_test_meta_33F]:
    dataframe["id_row"] = dataframe["id_row"].astype(str)
    dataframe["group_hospital"] = dataframe["group_hospital"].astype(str)
    dataframe["label_stage23"] = dataframe["label_stage23"].astype(int)

y_outer_training_33F = (
    outer_training_meta_33F["label_stage23"].to_numpy(dtype=np.int8)
)
y_outer_test_33F = (
    outer_test_meta_33F["label_stage23"].to_numpy(dtype=np.int8)
)

groups_outer_training_33F = (
    outer_training_meta_33F["group_hospital"].to_numpy(dtype=str)
)

training_hospitals_33F = set(
    outer_training_meta_33F["group_hospital"]
)
test_hospitals_33F = set(
    outer_test_meta_33F["group_hospital"]
)
hospital_overlap_33F = training_hospitals_33F & test_hospitals_33F

if hospital_overlap_33F:
    raise RuntimeError("Dış eğitim ve test hastaneleri çakışıyor.")

actual_split_33F = {
    "training_rows": len(X_outer_training_33F),
    "test_rows": len(X_outer_test_33F),
    "training_hospitals": len(training_hospitals_33F),
    "test_hospitals": len(test_hospitals_33F),
    "training_events": int(y_outer_training_33F.sum()),
    "test_events": int(y_outer_test_33F.sum()),
}

for metric, expected_value in EXPECTED_SPLIT_33F.items():
    actual_value = actual_split_33F[metric]
    if actual_value != expected_value:
        raise RuntimeError(
            f"{metric}: bulunan={actual_value}, beklenen={expected_value}"
        )

inner_fold_vector_33F = np.array(
    [
        hospital_to_inner_fold_33F.get(hospital, -1)
        for hospital in groups_outer_training_33F
    ],
    dtype=int,
)

if (inner_fold_vector_33F == -1).any():
    raise RuntimeError(
        "Bazı dış eğitim hastanelerine iç kat atanmadı."
    )

if set(np.unique(inner_fold_vector_33F)) != {1, 2, 3, 4, 5}:
    raise RuntimeError("İç kat değerleri 1–5 değil.")

for inner_fold, expected in EXPECTED_INNER_33F.items():
    validation_mask = inner_fold_vector_33F == inner_fold
    observed_rows = int(validation_mask.sum())
    observed_events = int(y_outer_training_33F[validation_mask].sum())

    if observed_rows != expected["validation_rows"]:
        raise RuntimeError(
            f"Inner {inner_fold} validation_rows: "
            f"bulunan={observed_rows}, "
            f"beklenen={expected['validation_rows']}"
        )

    if observed_events != expected["validation_events"]:
        raise RuntimeError(
            f"Inner {inner_fold} validation_events: "
            f"bulunan={observed_events}, "
            f"beklenen={expected['validation_events']}"
        )

# ------------------------------------------------------------
# 5. Locked candidate grid
# ------------------------------------------------------------

candidate_grid_33F = [
    {"candidate_id": "CLIN01", "C": 0.03, "l1_ratio": 0.00},
    {"candidate_id": "CLIN02", "C": 0.10, "l1_ratio": 0.00},
    {"candidate_id": "CLIN03", "C": 0.30, "l1_ratio": 0.00},
    {"candidate_id": "CLIN04", "C": 0.10, "l1_ratio": 0.25},
    {"candidate_id": "CLIN05", "C": 0.30, "l1_ratio": 0.25},
    {"candidate_id": "CLIN06", "C": 0.30, "l1_ratio": 0.50},
]

# ------------------------------------------------------------
# 6. Preprocessing and model factories
# ------------------------------------------------------------

def make_preprocessor_33F():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
            (
                "scaler",
                StandardScaler(with_mean=False),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                CLINICAL_NUMERIC_33F,
            ),
            (
                "categorical",
                categorical_pipeline,
                CLINICAL_CATEGORICAL_33F,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_model_33F(C_value, l1_ratio_value):
    return LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=float(C_value),
        l1_ratio=float(l1_ratio_value),
        class_weight=None,
        max_iter=20000,
        tol=1e-4,
        random_state=MODEL_RANDOM_SEED_33F,
    )


def checkpoint_table_id_33F(inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_clinical_lr_inner_oof_outer5_inner{inner_fold}_v2"
    )

# ------------------------------------------------------------
# 7. BigQuery checkpoint verification
# ------------------------------------------------------------

def verify_checkpoint_33F(
    inner_fold,
    expected_validation_rows,
    expected_validation_events,
):
    table_id = checkpoint_table_id_33F(inner_fold)

    try:
        client.get_table(table_id)
    except NotFound:
        return {
            "complete": False,
            "table_id": table_id,
            "reason": "table_not_found",
        }

    sql = f"""
    SELECT
      COUNT(*) AS row_count,
      COUNT(DISTINCT id_row) AS distinct_id_count,
      COUNT(DISTINCT candidate_id) AS candidate_count,
      COUNT(DISTINCT outer_fold) AS outer_fold_count,
      COUNT(DISTINCT inner_fold) AS inner_fold_count,
      COUNT(
        DISTINCT CONCAT(candidate_id, '|', id_row)
      ) AS distinct_candidate_patient_rows,
      COUNTIF(label_stage23 = 1) AS positive_prediction_rows,
      COUNTIF(label_stage23 = 0) AS negative_prediction_rows,
      COUNTIF(prediction_raw IS NULL) AS missing_predictions,
      COUNTIF(
        prediction_raw < 0 OR prediction_raw > 1
      ) AS invalid_probabilities,
      MIN(outer_fold) AS minimum_outer_fold,
      MAX(outer_fold) AS maximum_outer_fold,
      MIN(inner_fold) AS minimum_inner_fold,
      MAX(inner_fold) AS maximum_inner_fold,
      MIN(prediction_raw) AS minimum_probability,
      MAX(prediction_raw) AS maximum_probability
    FROM `{table_id}`;
    """

    check = client.query(sql, location=BQ_LOCATION).to_dataframe()
    row = check.iloc[0]

    expected_total_rows = (
        expected_validation_rows * len(candidate_grid_33F)
    )
    expected_positive_rows = (
        expected_validation_events * len(candidate_grid_33F)
    )
    expected_negative_rows = (
        (expected_validation_rows - expected_validation_events)
        * len(candidate_grid_33F)
    )

    expected_values = {
        "row_count": expected_total_rows,
        "distinct_id_count": expected_validation_rows,
        "candidate_count": 6,
        "outer_fold_count": 1,
        "inner_fold_count": 1,
        "distinct_candidate_patient_rows": expected_total_rows,
        "positive_prediction_rows": expected_positive_rows,
        "negative_prediction_rows": expected_negative_rows,
        "missing_predictions": 0,
        "invalid_probabilities": 0,
        "minimum_outer_fold": 5,
        "maximum_outer_fold": 5,
        "minimum_inner_fold": inner_fold,
        "maximum_inner_fold": inner_fold,
    }

    complete = True
    failure_items = []

    for field, expected_value in expected_values.items():
        actual_value = int(row[field])
        if actual_value != expected_value:
            complete = False
            failure_items.append(
                f"{field}={actual_value}, expected={expected_value}"
            )

    return {
        "complete": complete,
        "table_id": table_id,
        "reason": "; ".join(failure_items),
        "check": check,
        "row": row,
    }

# ------------------------------------------------------------
# 8. BigQuery load schema
# ------------------------------------------------------------

checkpoint_load_config_33F = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("inner_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("candidate_id", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

# ------------------------------------------------------------
# 9. Aggregate fit-audit file
# ------------------------------------------------------------

fit_audit_columns_33F = [
    "outer_fold",
    "inner_fold",
    "candidate_id",
    "C",
    "l1_ratio",
    "training_rows",
    "validation_rows",
    "training_events",
    "validation_events",
    "processed_columns",
    "convergence_warnings",
    "elapsed_seconds",
]

fit_audit_path_33F = os.path.join(
    MODEL_OUTPUT_DIR,
    "33F_clinical_inner_fit_audit_outer5.csv",
)

if os.path.exists(fit_audit_path_33F):
    fit_audit_33F = pd.read_csv(fit_audit_path_33F)
else:
    fit_audit_33F = pd.DataFrame(columns=fit_audit_columns_33F)

for column in fit_audit_columns_33F:
    if column not in fit_audit_33F.columns:
        fit_audit_33F[column] = np.nan

fit_audit_33F = fit_audit_33F[fit_audit_columns_33F].copy()

# ------------------------------------------------------------
# 10. Train six candidates in five locked inner folds
# ------------------------------------------------------------

for inner_fold in range(1, 6):
    inner_training_mask = inner_fold_vector_33F != inner_fold
    inner_validation_mask = inner_fold_vector_33F == inner_fold

    training_rows = int(inner_training_mask.sum())
    validation_rows = int(inner_validation_mask.sum())
    training_events = int(
        y_outer_training_33F[inner_training_mask].sum()
    )
    validation_events = int(
        y_outer_training_33F[inner_validation_mask].sum()
    )

    expected_inner = EXPECTED_INNER_33F[inner_fold]
    expected_training_rows = (
        EXPECTED_SPLIT_33F["training_rows"]
        - expected_inner["validation_rows"]
    )
    expected_training_events = (
        EXPECTED_SPLIT_33F["training_events"]
        - expected_inner["validation_events"]
    )

    if training_rows != expected_training_rows:
        raise RuntimeError(
            f"Inner {inner_fold} training_rows: "
            f"bulunan={training_rows}, "
            f"beklenen={expected_training_rows}"
        )

    if training_events != expected_training_events:
        raise RuntimeError(
            f"Inner {inner_fold} training_events: "
            f"bulunan={training_events}, "
            f"beklenen={expected_training_events}"
        )

    existing_check = verify_checkpoint_33F(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if existing_check["complete"]:
        print(
            f"Outer 5 / inner {inner_fold}: "
            "permanent checkpoint already complete; "
            "skipping model fitting."
        )
        continue

    training_hospital_set = set(
        groups_outer_training_33F[inner_training_mask]
    )
    validation_hospital_set = set(
        groups_outer_training_33F[inner_validation_mask]
    )

    if training_hospital_set & validation_hospital_set:
        raise RuntimeError(
            f"Inner fold {inner_fold}: hastane çakışması bulundu."
        )

    print(f"\nOuter 5 / inner {inner_fold}")
    print(
        "Training rows:",
        training_rows,
        "| Validation rows:",
        validation_rows,
        "| Validation events:",
        validation_events,
    )

    preprocessor = make_preprocessor_33F()
    preprocessing_started = time.time()

    X_inner_training_processed = preprocessor.fit_transform(
        X_outer_training_33F.loc[inner_training_mask]
    )
    X_inner_validation_processed = preprocessor.transform(
        X_outer_training_33F.loc[inner_validation_mask]
    )

    preprocessing_elapsed = time.time() - preprocessing_started

    if (
        X_inner_training_processed.shape[1]
        != X_inner_validation_processed.shape[1]
    ):
        raise RuntimeError(
            "İşlenmiş eğitim ve doğrulama sütun sayıları farklı."
        )

    print(
        "Processed columns:",
        X_inner_training_processed.shape[1],
        "| Preprocessing seconds:",
        round(preprocessing_elapsed, 2),
    )

    y_inner_training = y_outer_training_33F[inner_training_mask]
    y_inner_validation = y_outer_training_33F[inner_validation_mask]

    validation_ids = (
        outer_training_meta_33F.loc[
            inner_validation_mask,
            "id_row",
        ]
        .astype(str)
        .to_numpy()
    )

    prediction_frames = []
    current_audit_rows = []

    for candidate in candidate_grid_33F:
        candidate_id = candidate["candidate_id"]

        print(
            "  Fitting",
            candidate_id,
            "| C =",
            candidate["C"],
            "| l1_ratio =",
            candidate["l1_ratio"],
        )

        model = make_model_33F(
            candidate["C"],
            candidate["l1_ratio"],
        )

        fitting_started = time.time()

        with warnings.catch_warnings(record=True) as warning_records:
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(
                X_inner_training_processed,
                y_inner_training,
            )

        fitting_elapsed = time.time() - fitting_started

        convergence_warning_count = sum(
            issubclass(warning.category, ConvergenceWarning)
            for warning in warning_records
        )

        validation_probabilities = model.predict_proba(
            X_inner_validation_processed
        )[:, 1]

        if np.isnan(validation_probabilities).any():
            raise RuntimeError(
                f"{candidate_id}, inner {inner_fold}: eksik tahmin."
            )

        if not np.all(
            (validation_probabilities >= 0)
            & (validation_probabilities <= 1)
        ):
            raise RuntimeError(
                f"{candidate_id}: geçersiz olasılık."
            )

        prediction_frames.append(
            pd.DataFrame(
                {
                    "id_row": validation_ids,
                    "outer_fold": np.full(
                        validation_rows,
                        5,
                        dtype=np.int64,
                    ),
                    "inner_fold": np.full(
                        validation_rows,
                        inner_fold,
                        dtype=np.int64,
                    ),
                    "candidate_id": candidate_id,
                    "label_stage23": y_inner_validation.astype(np.int64),
                    "prediction_raw": validation_probabilities.astype(
                        np.float64
                    ),
                }
            )
        )

        current_audit_rows.append(
            {
                "outer_fold": 5,
                "inner_fold": inner_fold,
                "candidate_id": candidate_id,
                "C": candidate["C"],
                "l1_ratio": candidate["l1_ratio"],
                "training_rows": training_rows,
                "validation_rows": validation_rows,
                "training_events": training_events,
                "validation_events": validation_events,
                "processed_columns": int(
                    X_inner_training_processed.shape[1]
                ),
                "convergence_warnings": int(
                    convergence_warning_count
                ),
                "elapsed_seconds": float(fitting_elapsed),
            }
        )

        del model
        gc.collect()

    checkpoint_df = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    expected_checkpoint_rows = (
        validation_rows * len(candidate_grid_33F)
    )

    if len(checkpoint_df) != expected_checkpoint_rows:
        raise RuntimeError(
            f"Inner fold {inner_fold}: checkpoint satır sayısı hatalı."
        )

    if checkpoint_df.duplicated(
        subset=["id_row", "candidate_id"]
    ).any():
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "yinelenen aday–hasta tahmini var."
        )

    target_checkpoint_table = checkpoint_table_id_33F(inner_fold)

    print(
        "Uploading permanent checkpoint:",
        target_checkpoint_table,
    )

    client.load_table_from_dataframe(
        checkpoint_df,
        target_checkpoint_table,
        job_config=checkpoint_load_config_33F,
        location=BQ_LOCATION,
    ).result()

    new_audit_df = pd.DataFrame(current_audit_rows)

    if len(fit_audit_33F) > 0:
        keep_mask = ~(
            (
                pd.to_numeric(
                    fit_audit_33F["outer_fold"],
                    errors="coerce",
                )
                == 5
            )
            & (
                pd.to_numeric(
                    fit_audit_33F["inner_fold"],
                    errors="coerce",
                )
                == inner_fold
            )
        )
        fit_audit_33F = fit_audit_33F.loc[keep_mask].copy()

    if fit_audit_33F.empty:
        fit_audit_33F = new_audit_df.copy()
    else:
        fit_audit_33F = pd.concat(
            [fit_audit_33F, new_audit_df],
            ignore_index=True,
        )

    fit_audit_33F = (
        fit_audit_33F[fit_audit_columns_33F]
        .sort_values(
            ["outer_fold", "inner_fold", "candidate_id"]
        )
        .reset_index(drop=True)
    )

    fit_audit_33F.to_csv(
        fit_audit_path_33F,
        index=False,
    )

    completed_check = verify_checkpoint_33F(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not completed_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold} checkpoint doğrulanamadı: "
            + completed_check["reason"]
        )

    print(
        f"Outer 5 / inner {inner_fold}: "
        "permanent checkpoint PASS."
    )

    del (
        preprocessor,
        X_inner_training_processed,
        X_inner_validation_processed,
        prediction_frames,
        checkpoint_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 11. Final checkpoint summary
# ------------------------------------------------------------

checkpoint_summary_rows_33F = []

for inner_fold in range(1, 6):
    validation_mask = inner_fold_vector_33F == inner_fold
    validation_rows = int(validation_mask.sum())
    validation_events = int(
        y_outer_training_33F[validation_mask].sum()
    )

    final_check = verify_checkpoint_33F(
        inner_fold=inner_fold,
        expected_validation_rows=validation_rows,
        expected_validation_events=validation_events,
    )

    if not final_check["complete"]:
        raise RuntimeError(
            f"Inner fold {inner_fold}: "
            "son checkpoint denetimi başarısız. "
            + final_check["reason"]
        )

    row = final_check["row"]

    checkpoint_summary_rows_33F.append(
        {
            "outer_fold": 5,
            "inner_fold": inner_fold,
            "checkpoint_rows": int(row["row_count"]),
            "distinct_validation_patients": int(
                row["distinct_id_count"]
            ),
            "candidate_models": int(row["candidate_count"]),
            "positive_prediction_rows": int(
                row["positive_prediction_rows"]
            ),
            "negative_prediction_rows": int(
                row["negative_prediction_rows"]
            ),
            "missing_predictions": int(
                row["missing_predictions"]
            ),
            "invalid_probabilities": int(
                row["invalid_probabilities"]
            ),
            "minimum_probability": float(
                row["minimum_probability"]
            ),
            "maximum_probability": float(
                row["maximum_probability"]
            ),
            "table_id": final_check["table_id"],
        }
    )

checkpoint_summary_33F = (
    pd.DataFrame(checkpoint_summary_rows_33F)
    .sort_values("inner_fold")
    .reset_index(drop=True)
)

if int(
    checkpoint_summary_33F["distinct_validation_patients"].sum()
) != EXPECTED_SPLIT_33F["training_rows"]:
    raise RuntimeError(
        "Toplam doğrulama hasta sayısı 46.803 değil."
    )

expected_total_oof_rows_33F = (
    EXPECTED_SPLIT_33F["training_rows"]
    * len(candidate_grid_33F)
)

if int(checkpoint_summary_33F["checkpoint_rows"].sum()) != (
    expected_total_oof_rows_33F
):
    raise RuntimeError("Toplam OOF tahmin satırı hatalı.")

checkpoint_summary_path_33F = os.path.join(
    MODEL_OUTPUT_DIR,
    "33F_clinical_outer5_inner_checkpoint_summary.csv",
)
checkpoint_summary_33F.to_csv(
    checkpoint_summary_path_33F,
    index=False,
)

# ------------------------------------------------------------
# 12. Pool all inner OOF predictions
# ------------------------------------------------------------

checkpoint_tables_33F = [
    checkpoint_table_id_33F(inner_fold)
    for inner_fold in range(1, 6)
]

union_parts_33F = [
    f"""
    SELECT
      id_row,
      outer_fold,
      inner_fold,
      candidate_id,
      label_stage23,
      prediction_raw
    FROM `{table_id}`
    """
    for table_id in checkpoint_tables_33F
]

SQL_LOAD_POOLED_OOF_33F = "\nUNION ALL\n".join(union_parts_33F)

print("\nLoading pooled outer-fold-5 inner OOF predictions...")

query_job_33F = client.query(
    SQL_LOAD_POOLED_OOF_33F,
    location=BQ_LOCATION,
)

try:
    pooled_oof_33F = query_job_33F.to_dataframe(
        create_bqstorage_client=True
    )
    pooled_load_method_33F = "BigQuery Storage API"
except Exception as fast_path_error_33F:
    print(
        "Storage API unavailable; using standard BigQuery download."
    )
    print("Message:", type(fast_path_error_33F).__name__)
    pooled_oof_33F = query_job_33F.to_dataframe(
        create_bqstorage_client=False
    )
    pooled_load_method_33F = "Standard BigQuery API"

pooled_oof_33F["id_row"] = pooled_oof_33F["id_row"].astype(str)
pooled_oof_33F["candidate_id"] = pooled_oof_33F["candidate_id"].astype(str)

for column in ["outer_fold", "inner_fold", "label_stage23"]:
    pooled_oof_33F[column] = pd.to_numeric(
        pooled_oof_33F[column], errors="raise"
    ).astype(int)

pooled_oof_33F["prediction_raw"] = pd.to_numeric(
    pooled_oof_33F["prediction_raw"], errors="raise"
).astype(float)

# ------------------------------------------------------------
# 13. Pooled OOF integrity checks
# ------------------------------------------------------------

if len(pooled_oof_33F) != expected_total_oof_rows_33F:
    raise RuntimeError("Pooled OOF satır sayısı hatalı.")

if set(pooled_oof_33F["outer_fold"].unique()) != {5}:
    raise RuntimeError("Pooled OOF içinde dış kat 4 dışında kayıt var.")

if set(pooled_oof_33F["inner_fold"].unique()) != {1, 2, 3, 4, 5}:
    raise RuntimeError("Pooled OOF iç katları 1–5 değil.")

if pooled_oof_33F.duplicated(
    subset=["candidate_id", "id_row"]
).any():
    raise RuntimeError(
        "Pooled OOF içinde yinelenen aday–hasta tahmini bulundu."
    )

if pooled_oof_33F["prediction_raw"].isna().any():
    raise RuntimeError("Pooled OOF içinde eksik tahmin var.")

if not pooled_oof_33F["prediction_raw"].between(0, 1).all():
    raise RuntimeError(
        "Pooled OOF içinde geçersiz olasılık değeri var."
    )

expected_candidate_ids_33F = {
    "CLIN01",
    "CLIN02",
    "CLIN03",
    "CLIN04",
    "CLIN05",
    "CLIN06",
}

if set(pooled_oof_33F["candidate_id"].unique()) != (
    expected_candidate_ids_33F
):
    raise RuntimeError("Altı kilitli aday bulunmuyor.")

candidate_patient_counts_33F = (
    pooled_oof_33F.groupby("candidate_id")["id_row"].nunique()
)

if not (
    candidate_patient_counts_33F
    == EXPECTED_SPLIT_33F["training_rows"]
).all():
    raise RuntimeError(
        "Her aday için 46.803 farklı OOF hastası yok."
    )

candidate_event_counts_33F = (
    pooled_oof_33F.groupby("candidate_id")["label_stage23"].sum()
)

if not (
    candidate_event_counts_33F
    == EXPECTED_SPLIT_33F["training_events"]
).all():
    raise RuntimeError("Her aday için 2.426 olay yok.")

patient_label_consistency_33F = (
    pooled_oof_33F.groupby("id_row")["label_stage23"].nunique()
)
if (patient_label_consistency_33F > 1).any():
    raise RuntimeError(
        "Aynı hastanın adaylar arasında outcome etiketi farklı."
    )

patient_inner_fold_consistency_33F = (
    pooled_oof_33F.groupby("id_row")["inner_fold"].nunique()
)
if (patient_inner_fold_consistency_33F > 1).any():
    raise RuntimeError(
        "Aynı hasta birden fazla iç doğrulama katında bulundu."
    )

# ------------------------------------------------------------
# 14. Metric helpers
# ------------------------------------------------------------

def probability_metrics_33F(y_true, probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(roc_auc_score(y_true, probabilities)),
        "auprc": float(average_precision_score(y_true, probabilities)),
        "brier": float(brier_score_loss(y_true, probabilities)),
        "log_loss": float(
            log_loss(y_true, probabilities, labels=[0, 1])
        ),
        "mean_predicted_risk": float(probabilities.mean()),
        "observed_event_rate": float(np.mean(y_true)),
    }


def probability_logit_33F(probabilities):
    probabilities = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        probabilities / (1 - probabilities)
    ).reshape(-1, 1)


def calibration_intercept_slope_33F(y_true, probabilities):
    calibration_model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibration_model.fit(
        probability_logit_33F(probabilities),
        y_true,
    )
    return (
        float(calibration_model.intercept_[0]),
        float(calibration_model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 15. Candidate pooled-OOF performance
# ------------------------------------------------------------

candidate_result_rows_33F = []

for candidate in candidate_grid_33F:
    candidate_id = candidate["candidate_id"]

    candidate_oof = (
        pooled_oof_33F.loc[
            pooled_oof_33F["candidate_id"] == candidate_id
        ]
        .copy()
        .sort_values("id_row")
        .reset_index(drop=True)
    )

    metrics = probability_metrics_33F(
        candidate_oof["label_stage23"].to_numpy(dtype=int),
        candidate_oof["prediction_raw"].to_numpy(dtype=float),
    )

    fit_part = fit_audit_33F.loc[
        (
            pd.to_numeric(
                fit_audit_33F["outer_fold"],
                errors="coerce",
            )
            == 5
        )
        & (
            fit_audit_33F["candidate_id"].astype(str)
            == candidate_id
        )
    ]

    convergence_warnings = (
        int(
            pd.to_numeric(
                fit_part["convergence_warnings"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    elapsed_seconds = (
        float(
            pd.to_numeric(
                fit_part["elapsed_seconds"],
                errors="coerce",
            )
            .fillna(0)
            .sum()
        )
        if len(fit_part) > 0
        else np.nan
    )

    candidate_result_rows_33F.append(
        {
            "candidate_id": candidate_id,
            "C": candidate["C"],
            "l1_ratio": candidate["l1_ratio"],
            **metrics,
            "convergence_warnings": convergence_warnings,
            "elapsed_seconds": elapsed_seconds,
        }
    )

candidate_results_33F = pd.DataFrame(candidate_result_rows_33F)

total_inner_convergence_warnings_33F = int(
    pd.to_numeric(
        candidate_results_33F["convergence_warnings"],
        errors="coerce",
    ).fillna(0).sum()
)

if total_inner_convergence_warnings_33F != 0:
    raise RuntimeError(
        "33B-R2 stopped BEFORE candidate selection and BEFORE outer-test "
        "evaluation because at least one inner candidate fit still emitted "
        f"a ConvergenceWarning under max_iter=20000. "
        f"Total warnings={total_inner_convergence_warnings_33F}. "
        "No test result may be accessed."
    )

print(
    "All 30 corrected inner candidate fits converged with zero warnings: PASS"
)

candidate_results_33F = (
    candidate_results_33F.sort_values(
        ["auprc", "auroc", "brier", "candidate_id"],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)

candidate_results_33F["selection_rank"] = np.arange(
    1,
    len(candidate_results_33F) + 1,
)

best_row_33F = candidate_results_33F.iloc[0]
selected_candidate_33F = str(best_row_33F["candidate_id"])
selected_C_33F = float(best_row_33F["C"])
selected_l1_ratio_33F = float(best_row_33F["l1_ratio"])

# ------------------------------------------------------------
# 16. Platt calibration on selected pooled inner OOF
# ------------------------------------------------------------

selected_oof_33F = (
    pooled_oof_33F.loc[
        pooled_oof_33F["candidate_id"] == selected_candidate_33F
    ]
    .copy()
    .sort_values("id_row")
    .reset_index(drop=True)
)

selected_oof_y_33F = selected_oof_33F[
    "label_stage23"
].to_numpy(dtype=int)
selected_oof_probability_33F = selected_oof_33F[
    "prediction_raw"
].to_numpy(dtype=float)

platt_calibrator_33F = LogisticRegression(
    penalty=None,
    solver="lbfgs",
    max_iter=2000,
)

platt_calibrator_33F.fit(
    probability_logit_33F(selected_oof_probability_33F),
    selected_oof_y_33F,
)

platt_intercept_33F = float(platt_calibrator_33F.intercept_[0])
platt_slope_33F = float(platt_calibrator_33F.coef_[0][0])

if (
    not np.isfinite(platt_intercept_33F)
    or not np.isfinite(platt_slope_33F)
    or platt_slope_33F <= 0
):
    raise RuntimeError("Platt kalibrasyon katsayıları geçersiz.")

selected_model_33F = pd.DataFrame(
    [
        {
            "outer_fold": 5,
            "selected_candidate": selected_candidate_33F,
            "selected_C": selected_C_33F,
            "selected_l1_ratio": selected_l1_ratio_33F,
            "selection_metric_primary": "pooled_inner_oof_auprc",
            "inner_oof_auprc": float(best_row_33F["auprc"]),
            "inner_oof_auroc": float(best_row_33F["auroc"]),
            "inner_oof_brier": float(best_row_33F["brier"]),
            "inner_oof_log_loss": float(best_row_33F["log_loss"]),
            "inner_oof_mean_predicted_risk": float(
                best_row_33F["mean_predicted_risk"]
            ),
            "inner_oof_observed_event_rate": float(
                best_row_33F["observed_event_rate"]
            ),
            "platt_intercept": platt_intercept_33F,
            "platt_slope": platt_slope_33F,
            "protocol_sha256": EXPECTED_PROTOCOL_SHA_33F,
            "numerical_amendment_sha256": EXPECTED_AMENDMENT_SHA_33F,
            "full_inner_refit_correction_sha256": EXPECTED_CORRECTION_SHA_33F,
            "corrected_checkpoint_namespace": "v2",
            "analysis_status": "post_hoc_exploratory_corrected",
            "full_inner_refit_correction_sha256": EXPECTED_CORRECTION_SHA_33F,
            "corrected_checkpoint_namespace": "v2",
            "analysis_status": "post_hoc_exploratory_corrected",
            "model_max_iter": 20000,
        }
    ]
)

# ------------------------------------------------------------
# 17. Lock selection/calibration aggregate files
# ------------------------------------------------------------

candidate_results_path_33F = os.path.join(
    MODEL_OUTPUT_DIR,
    "33F_clinical_candidate_results_outer5.csv",
)
selected_model_path_33F = os.path.join(
    MODEL_OUTPUT_DIR,
    "33F_clinical_selected_model_outer5.csv",
)
selection_json_path_33F = os.path.join(
    MODEL_OUTPUT_DIR,
    "33F_clinical_selection_calibration_outer5.json",
)
selection_sha_path_33F = os.path.join(
    MODEL_OUTPUT_DIR,
    "33F_clinical_selection_calibration_outer5_SHA256.txt",
)

candidate_results_33F.to_csv(
    candidate_results_path_33F,
    index=False,
)
selected_model_33F.to_csv(
    selected_model_path_33F,
    index=False,
)

selection_configuration_33F = {
    "outer_fold": 5,
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_33F,
    "selection_metric_primary": "pooled inner out-of-fold AUPRC",
    "selection_tie_breakers": [
        "AUROC descending",
        "Brier score ascending",
        "candidate_id ascending immutable tie-break",
    ],
    "selected_candidate": selected_candidate_33F,
    "selected_C": selected_C_33F,
    "selected_l1_ratio": selected_l1_ratio_33F,
    "inner_oof_auprc": float(best_row_33F["auprc"]),
    "inner_oof_auroc": float(best_row_33F["auroc"]),
    "inner_oof_brier": float(best_row_33F["brier"]),
    "platt_intercept": platt_intercept_33F,
    "platt_slope": platt_slope_33F,
    "inner_checkpoint_tables": checkpoint_tables_33F,
    "patient_level_oof_written_to_drive": False,
    "analysis_role": "additional_post_hoc_clinical_baseline",
    "predictors": CLINICAL_PREDICTORS_33F,
}

with open(
    selection_json_path_33F,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        selection_configuration_33F,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(selection_json_path_33F, "rb") as file_handle:
    selection_sha_33F = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    selection_sha_path_33F,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(selection_sha_33F + "\n")

# ------------------------------------------------------------
# 18. Fit final selected outer-fold-5 model
# ------------------------------------------------------------

final_pipeline_33F = Pipeline(
    steps=[
        ("preprocessor", make_preprocessor_33F()),
        (
            "model",
            make_model_33F(
                selected_C_33F,
                selected_l1_ratio_33F,
            ),
        ),
    ]
)

print(
    "\nFitting selected outer-fold-5 model "
    "on all 46,803 training patients..."
)

final_fit_started_33F = time.time()

with warnings.catch_warnings(record=True) as final_warning_records_33F:
    warnings.simplefilter("always", ConvergenceWarning)
    final_pipeline_33F.fit(
        X_outer_training_33F,
        y_outer_training_33F,
    )

final_fit_elapsed_33F = time.time() - final_fit_started_33F

final_model_n_iter_33F = int(
    np.max(
        np.asarray(
            final_pipeline_33F.named_steps["model"].n_iter_
        )
    )
)

final_convergence_warnings_33F = sum(
    issubclass(warning.category, ConvergenceWarning)
    for warning in final_warning_records_33F
)

if final_convergence_warnings_33F != 0:
    raise RuntimeError(
        "Nihai outer-fold-5 klinik baseline modeli max_iter=20000 "
        f"altında da yakınsamadı. n_iter_={final_model_n_iter_33F}"
    )

# ------------------------------------------------------------
# 19. Outer-fold-5 test predictions and metrics
# ------------------------------------------------------------

outer5_raw_probabilities_33F = final_pipeline_33F.predict_proba(
    X_outer_test_33F
)[:, 1]

raw_clipped_33F = np.clip(
    outer5_raw_probabilities_33F,
    1e-6,
    1 - 1e-6,
)
raw_logit_33F = np.log(
    raw_clipped_33F / (1 - raw_clipped_33F)
)

outer5_platt_probabilities_33F = expit(
    platt_intercept_33F + platt_slope_33F * raw_logit_33F
)

for probabilities, name in [
    (outer5_raw_probabilities_33F, "raw"),
    (outer5_platt_probabilities_33F, "platt"),
]:
    if np.isnan(probabilities).any():
        raise RuntimeError(f"{name} tahminlerinde eksik değer var.")

    if not np.all(
        (probabilities >= 0) & (probabilities <= 1)
    ):
        raise RuntimeError(
            f"{name} tahminlerinde geçersiz olasılık değeri var."
        )

raw_metrics_33F = probability_metrics_33F(
    y_outer_test_33F,
    outer5_raw_probabilities_33F,
)
platt_metrics_33F = probability_metrics_33F(
    y_outer_test_33F,
    outer5_platt_probabilities_33F,
)

raw_calibration_intercept_33F, raw_calibration_slope_33F = (
    calibration_intercept_slope_33F(
        y_outer_test_33F,
        outer5_raw_probabilities_33F,
    )
)

platt_calibration_intercept_33F, platt_calibration_slope_33F = (
    calibration_intercept_slope_33F(
        y_outer_test_33F,
        outer5_platt_probabilities_33F,
    )
)

outer5_test_results_33F = pd.DataFrame(
    [
        {
            "outer_fold": 5,
            "model": "parsimonious_clinical_logistic",
            "probability_type": "raw",
            **raw_metrics_33F,
            "calibration_intercept": raw_calibration_intercept_33F,
            "calibration_slope": raw_calibration_slope_33F,
        },
        {
            "outer_fold": 5,
            "model": "parsimonious_clinical_logistic",
            "probability_type": "platt_calibrated",
            **platt_metrics_33F,
            "calibration_intercept": platt_calibration_intercept_33F,
            "calibration_slope": platt_calibration_slope_33F,
        },
    ]
)

# ------------------------------------------------------------
# 20. Feature coefficient audit
# ------------------------------------------------------------

fitted_preprocessor_33F = final_pipeline_33F.named_steps[
    "preprocessor"
]
fitted_model_33F = final_pipeline_33F.named_steps["model"]

processed_feature_names_33F = (
    fitted_preprocessor_33F.get_feature_names_out()
)
model_coefficients_33F = fitted_model_33F.coef_.reshape(-1)

if len(processed_feature_names_33F) != len(model_coefficients_33F):
    raise RuntimeError("Feature ve katsayı sayıları uyuşmuyor.")

if len(set(processed_feature_names_33F)) != len(
    processed_feature_names_33F
):
    raise RuntimeError("İşlenmiş feature adlarında yinelenme var.")

coefficient_table_33F = pd.DataFrame(
    {
        "processed_feature": processed_feature_names_33F,
        "coefficient": model_coefficients_33F,
    }
)
coefficient_table_33F["absolute_coefficient"] = (
    coefficient_table_33F["coefficient"].abs()
)
coefficient_table_33F["is_nonzero"] = ~np.isclose(
    coefficient_table_33F["coefficient"],
    0.0,
    atol=1e-12,
)
coefficient_table_33F["absolute_rank"] = (
    coefficient_table_33F["absolute_coefficient"]
    .rank(method="first", ascending=False)
    .astype(int)
)
coefficient_table_33F = (
    coefficient_table_33F.sort_values(
        ["absolute_coefficient", "processed_feature"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

nonzero_coefficients_33F = int(
    coefficient_table_33F["is_nonzero"].sum()
)

final_model_summary_33F = pd.DataFrame(
    [
        {
            "outer_fold": 5,
            "selected_candidate": selected_candidate_33F,
            "selected_C": selected_C_33F,
            "selected_l1_ratio": selected_l1_ratio_33F,
            "training_patients": len(X_outer_training_33F),
            "training_hospitals": len(training_hospitals_33F),
            "training_events": int(y_outer_training_33F.sum()),
            "test_patients": len(X_outer_test_33F),
            "test_hospitals": len(test_hospitals_33F),
            "test_events": int(y_outer_test_33F.sum()),
            "hospital_overlap": len(hospital_overlap_33F),
            "processed_feature_columns": len(
                processed_feature_names_33F
            ),
            "nonzero_coefficients": nonzero_coefficients_33F,
            "model_intercept": float(
                fitted_model_33F.intercept_[0]
            ),
            "convergence_warnings": final_convergence_warnings_33F,
            "final_model_n_iter": final_model_n_iter_33F,
            "fit_elapsed_seconds": float(final_fit_elapsed_33F),
            "locked_platt_intercept": platt_intercept_33F,
            "locked_platt_slope": platt_slope_33F,
            "protocol_sha256": EXPECTED_PROTOCOL_SHA_33F,
            "numerical_amendment_sha256": EXPECTED_AMENDMENT_SHA_33F,
            "model_max_iter": 20000,
            "selection_sha256": selection_sha_33F,
        }
    ]
)

# ------------------------------------------------------------
# 21. Secure BigQuery outer-test checkpoint
# ------------------------------------------------------------

outer5_prediction_df_33F = pd.DataFrame(
    {
        "id_row": outer_test_meta_33F["id_row"].astype(str),
        "outer_fold": np.full(
            len(outer_test_meta_33F),
            5,
            dtype=np.int64,
        ),
        "label_stage23": y_outer_test_33F.astype(np.int64),
        "prediction_raw": outer5_raw_probabilities_33F.astype(
            np.float64
        ),
        "prediction_platt": outer5_platt_probabilities_33F.astype(
            np.float64
        ),
        "model_name": "parsimonious_clinical_logistic",
        "model_version": "clinical8_v1_nested_cv",
    }
)

if len(outer5_prediction_df_33F) != 11688:
    raise RuntimeError(
        "Outer-fold-5 tahmin satır sayısı 11,688 değil."
    )

if outer5_prediction_df_33F["id_row"].duplicated().any():
    raise RuntimeError(
        "Outer-fold-5 tahminlerinde yinelenen id_row var."
    )

if int(
    outer5_prediction_df_33F["label_stage23"].sum()
) != 606:
    raise RuntimeError("Outer-fold-5 event count is not 606 değil.")

prediction_table_id_33F = (
    f"{TARGET_DATASET}."
    "model_clinical_lr_outer_predictions_outer5_v2"
)

prediction_load_config_33F = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("prediction_platt", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("model_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("model_version", "STRING", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

print("\nUploading secure outer-fold-5 prediction checkpoint:")
print(prediction_table_id_33F)

client.load_table_from_dataframe(
    outer5_prediction_df_33F,
    prediction_table_id_33F,
    job_config=prediction_load_config_33F,
    location=BQ_LOCATION,
).result()

# ------------------------------------------------------------
# 22. BigQuery outer-test checkpoint verification
# ------------------------------------------------------------

SQL_VERIFY_PREDICTIONS_33F = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_rows,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  MIN(outer_fold) AS minimum_outer_fold,
  MAX(outer_fold) AS maximum_outer_fold,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL) AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL) AS missing_platt_predictions,
  COUNTIF(
    prediction_raw < 0 OR prediction_raw > 1
  ) AS invalid_raw_predictions,
  COUNTIF(
    prediction_platt < 0 OR prediction_platt > 1
  ) AS invalid_platt_predictions,
  MIN(prediction_raw) AS minimum_raw_probability,
  MAX(prediction_raw) AS maximum_raw_probability,
  MIN(prediction_platt) AS minimum_platt_probability,
  MAX(prediction_platt) AS maximum_platt_probability
FROM `{prediction_table_id_33F}`;
"""

prediction_verification_33F = client.query(
    SQL_VERIFY_PREDICTIONS_33F,
    location=BQ_LOCATION,
).to_dataframe()

verification_row_33F = prediction_verification_33F.iloc[0]

expected_prediction_values_33F = {
    "prediction_rows": 11688,
    "distinct_rows": 11688,
    "outer_folds": 1,
    "minimum_outer_fold": 5,
    "maximum_outer_fold": 5,
    "events": 606,
    "nonevents": 11082,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in expected_prediction_values_33F.items():
    actual_value = int(verification_row_33F[field])
    if actual_value != expected_value:
        raise RuntimeError(
            f"{field}: bulunan={actual_value}, beklenen={expected_value}"
        )

# ------------------------------------------------------------
# 23. Save only aggregate and feature-level outputs to Drive
# ------------------------------------------------------------

test_results_path_33F = os.path.join(
    MODEL_OUTPUT_DIR,
    "33F_clinical_outer5_test_results.csv",
)
model_summary_path_33F = os.path.join(
    MODEL_OUTPUT_DIR,
    "33F_clinical_final_model_outer5.csv",
)
coefficient_path_33F = os.path.join(
    MODEL_OUTPUT_DIR,
    "33F_clinical_coefficients_outer5.csv",
)
evaluation_json_path_33F = os.path.join(
    MODEL_OUTPUT_DIR,
    "33F_clinical_final_evaluation_outer5.json",
)
evaluation_sha_path_33F = os.path.join(
    MODEL_OUTPUT_DIR,
    "33F_clinical_final_evaluation_outer5_SHA256.txt",
)

outer5_test_results_33F.to_csv(
    test_results_path_33F,
    index=False,
)
final_model_summary_33F.to_csv(
    model_summary_path_33F,
    index=False,
)
coefficient_table_33F.to_csv(
    coefficient_path_33F,
    index=False,
)

evaluation_configuration_33F = {
    "outer_fold": 5,
    "model_family": "parsimonious_clinical_elastic_net_logistic_regression",
    "selected_candidate": selected_candidate_33F,
    "selected_C": selected_C_33F,
    "selected_l1_ratio": selected_l1_ratio_33F,
    "training_patients": 46803,
    "training_hospitals": 159,
    "test_patients": 11688,
    "test_hospitals": 39,
    "hospital_overlap": 0,
    "locked_platt_intercept": platt_intercept_33F,
    "locked_platt_slope": platt_slope_33F,
    "processed_feature_columns": int(
        len(processed_feature_names_33F)
    ),
    "nonzero_coefficients": int(nonzero_coefficients_33F),
    "protocol_sha256": EXPECTED_PROTOCOL_SHA_33F,
    "selection_sha256": selection_sha_33F,
    "secure_prediction_table": prediction_table_id_33F,
    "patient_level_prediction_written_to_drive": False,
    "analysis_role": "additional_post_hoc_clinical_baseline",
    "predictors": CLINICAL_PREDICTORS_33F,
}

with open(
    evaluation_json_path_33F,
    "w",
    encoding="utf-8",
) as file_handle:
    json.dump(
        evaluation_configuration_33F,
        file_handle,
        indent=2,
        ensure_ascii=False,
    )

with open(evaluation_json_path_33F, "rb") as file_handle:
    evaluation_sha_33F = hashlib.sha256(
        file_handle.read()
    ).hexdigest()

with open(
    evaluation_sha_path_33F,
    "w",
    encoding="utf-8",
) as file_handle:
    file_handle.write(evaluation_sha_33F + "\n")

# ------------------------------------------------------------
# 24. Final outputs
# ------------------------------------------------------------

pooled_integrity_33F = pd.DataFrame(
    {
        "metric": [
            "pooled_prediction_rows",
            "distinct_patients",
            "candidate_models",
            "inner_folds",
            "events_per_candidate",
            "nonevents_per_candidate",
            "duplicate_candidate_patient_rows",
            "missing_predictions",
            "invalid_probabilities",
            "load_method",
        ],
        "value": [
            len(pooled_oof_33F),
            pooled_oof_33F["id_row"].nunique(),
            pooled_oof_33F["candidate_id"].nunique(),
            pooled_oof_33F["inner_fold"].nunique(),
            EXPECTED_SPLIT_33F["training_events"],
            (
                EXPECTED_SPLIT_33F["training_rows"]
                - EXPECTED_SPLIT_33F["training_events"]
            ),
            int(
                pooled_oof_33F.duplicated(
                    subset=["candidate_id", "id_row"]
                ).sum()
            ),
            int(pooled_oof_33F["prediction_raw"].isna().sum()),
            int(
                (~pooled_oof_33F["prediction_raw"].between(0, 1)).sum()
            ),
            pooled_load_method_33F,
        ],
    }
)

print("\n33F OUTER-FOLD-5 INNER CHECKPOINT SUMMARY")
display(checkpoint_summary_33F)

print("\n33F OUTER-FOLD-5 POOLED OOF INTEGRITY")
display(pooled_integrity_33F)

print("\n33F OUTER-FOLD-5 CANDIDATE RESULTS")
display(candidate_results_33F)

print("\n33F OUTER-FOLD-5 SELECTED MODEL")
display(selected_model_33F)

print("\n33F OUTER-FOLD-5 FINAL MODEL SUMMARY")
display(final_model_summary_33F)

print("\n33F OUTER-FOLD-5 TEST RESULTS")
display(outer5_test_results_33F)

print("\n33F OUTER-FOLD-5 BIGQUERY VERIFICATION")
display(prediction_verification_33F)

print("\n33F OUTER-FOLD-5 TOP 20 ABSOLUTE COEFFICIENTS")
display(coefficient_table_33F.head(20))

print("\nSelection SHA-256:")
print(selection_sha_33F)

print("\nEvaluation SHA-256:")
print(evaluation_sha_33F)

print("\nSaved:")
print(fit_audit_path_33F)
print(checkpoint_summary_path_33F)
print(candidate_results_path_33F)
print(selected_model_path_33F)
print(selection_json_path_33F)
print(selection_sha_path_33F)
print(test_results_path_33F)
print(model_summary_path_33F)
print(coefficient_path_33F)
print(evaluation_json_path_33F)
print(evaluation_sha_path_33F)

print(
    "\n33F PASS: Outer-fold-5 nested modelling "
    "and locked test evaluation are complete."
)
print(
    "All inner OOF and outer-test patient-level "
    "predictions were stored only in BigQuery."
)
print(
    "No patient-level prediction file was "
    "written to Google Drive."
)

_ = gc.collect()

In [ ]:
import os
import json
import hashlib

import numpy as np
import pandas as pd

from google.cloud import bigquery
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from IPython.display import display

print("STARTING FINAL FIVE-FOLD PARSIMONIOUS CLINICAL BASELINE POOLED EVALUATION — CODE VERSION 34")

# ============================================================
# 30 — FINAL FIVE-FOLD PARSIMONIOUS CLINICAL BASELINE POOLED EVALUATION
#
# Primary goals:
# 1) Pool exactly one locked outer-test parsimonious clinical baseline prediction
#    per patient across the five hospital-disjoint outer folds.
# 2) Verify 58,491 patients, 198 hospitals, 3,032 events.
# 3) Calculate raw and fold-specific Platt pooled metrics.
# 4) Calculate macro fold metrics.
# 5) Estimate 2,000 hospital-cluster bootstrap CIs using the same
#    locked seed as prior model evaluations.
# 6) Audit selected-model consistency across folds.
# 7) Keep patient-level prediction data only in BigQuery / RAM.
#
# This script does NOT fit or tune a parsimonious clinical baseline model.
# It does NOT compare models or trigger retuning.
# No SMOTE or class weighting is used.
# No DELETE / INSERT / UPDATE / MERGE statement is used.
# ============================================================

MODEL_NAME_34 = "parsimonious_clinical_logistic"
MODEL_VERSION_34 = "clinical8_v1_nested_cv"

BOOTSTRAP_REPLICATES_34 = 2000
BOOTSTRAP_SEED_34 = 20260723

EXPECTED_CLINICAL_PROTOCOL_SHA_34 = (
    "94b0abb218dbef4e349702ea2824ca4e"
    "31dfbe36c53efa05ba1a8bd1f38f835e"
)

EXPECTED_CONVERGENCE_AMENDMENT_SHA_34 = (
    "fcbcf857caa9aaad7ffd6691b4b61ac6"
    "503d09a6eee80db84bc2967ddf1f1a6a"
)

EXPECTED_FULL_REFIT_CORRECTION_SHA_34 = (
    "ea7ab2747aad0c20157da0988b2fbeb1"
    "6ce007b4c92bf785d9ef9d363862ecfa"
)

EXPECTED_TOTAL_ROWS_34 = 58491
EXPECTED_TOTAL_EVENTS_34 = 3032
EXPECTED_TOTAL_NONEVENTS_34 = 55459
EXPECTED_TOTAL_HOSPITALS_34 = 198

EXPECTED_FOLD_STRUCTURE_34 = {
    1: {"rows": 11688, "events": 606, "nonevents": 11082, "hospitals": 40},
    2: {"rows": 11691, "events": 606, "nonevents": 11085, "hospitals": 40},
    3: {"rows": 11736, "events": 608, "nonevents": 11128, "hospitals": 40},
    4: {"rows": 11688, "events": 606, "nonevents": 11082, "hospitals": 39},
    5: {"rows": 11688, "events": 606, "nonevents": 11082, "hospitals": 39},
}

# ------------------------------------------------------------
# 1. Required objects and locked protocol
# ------------------------------------------------------------

required_objects_34 = [
    "core_df_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_objects_34 = [
    name for name in required_objects_34
    if name not in globals()
]

if missing_objects_34:
    raise RuntimeError(
        "Eksik çalışma nesneleri var: "
        + ", ".join(missing_objects_34)
        + ". Önce 07A ve 07B hücrelerini çalıştır."
    )

if len(core_df_07B) != EXPECTED_TOTAL_ROWS_34:
    raise RuntimeError(
        f"core_df_07B rows={len(core_df_07B)}, "
        f"expected={EXPECTED_TOTAL_ROWS_34}."
    )

clinical_guard_files_34 = [
    (
        "33A_locked_parsimonious_clinical_baseline_protocol_v1_SHA256.txt",
        EXPECTED_CLINICAL_PROTOCOL_SHA_34,
        "33A protocol",
    ),
    (
        "33A1_parsimonious_clinical_baseline_convergence_amendment_v1_SHA256.txt",
        EXPECTED_CONVERGENCE_AMENDMENT_SHA_34,
        "33A1 convergence amendment",
    ),
    (
        "33A2_parsimonious_clinical_baseline_full_inner_refit_correction_v1_SHA256.txt",
        EXPECTED_FULL_REFIT_CORRECTION_SHA_34,
        "33A2 full inner-refit correction",
    ),
]

for filename, expected_sha, label in clinical_guard_files_34:
    path = os.path.join(MODEL_OUTPUT_DIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    with open(path, "r", encoding="utf-8") as fh:
        observed_sha = fh.read().strip()
    if observed_sha != expected_sha:
        raise RuntimeError(
            f"{label} SHA mismatch: {observed_sha}"
        )
    print(f"{label} SHA guard: PASS")

# ------------------------------------------------------------
# 2. Source parsimonious clinical baseline prediction tables
# ------------------------------------------------------------

source_clinical_tables_34 = {
    1: f"{TARGET_DATASET}.model_clinical_lr_outer_predictions_outer1_v2",
    2: f"{TARGET_DATASET}.model_clinical_lr_outer_predictions_outer2_v2",
    3: f"{TARGET_DATASET}.model_clinical_lr_outer_predictions_outer3_v2",
    4: f"{TARGET_DATASET}.model_clinical_lr_outer_predictions_outer4_v2",
    5: f"{TARGET_DATASET}.model_clinical_lr_outer_predictions_outer5_v2",
}

for fold_number, table_id in source_clinical_tables_34.items():
    if not table_id.endswith("_v2"):
        raise RuntimeError(
            f"Fold {fold_number}: only corrected v2 clinical predictions are allowed."
        )
    try:
        client.get_table(table_id)
    except Exception as exc:
        raise RuntimeError(
            f"parsimonious clinical baseline outer-fold-{fold_number} table unavailable: "
            f"{table_id}. {type(exc).__name__}: {exc}"
        )

# ------------------------------------------------------------
# 3. Robust prediction-table normalization
# ------------------------------------------------------------

def resolve_column_34(
    columns,
    exact_candidates,
    contains_all=None,
    contains_any=None,
    required=True,
):
    original_columns = list(columns)
    lower_to_original = {
        str(column).lower(): column
        for column in original_columns
    }

    for candidate in exact_candidates:
        if candidate.lower() in lower_to_original:
            return lower_to_original[candidate.lower()]

    matches = []

    for original in original_columns:
        lowered = str(original).lower()

        if contains_all and not all(
            token.lower() in lowered
            for token in contains_all
        ):
            continue

        if contains_any and not any(
            token.lower() in lowered
            for token in contains_any
        ):
            continue

        if contains_all or contains_any:
            matches.append(original)

    if len(matches) == 1:
        return matches[0]

    if required:
        raise RuntimeError(
            "Required column could not be resolved. "
            f"Candidates={exact_candidates}; columns={original_columns}; "
            f"heuristic_matches={matches}"
        )

    return None


def normalize_prediction_table_34(
    raw_dataframe,
    expected_fold,
    source_table,
):
    df = raw_dataframe.copy()

    id_col = resolve_column_34(
        df.columns,
        ["id_row", "patient_id", "patientunitstayid"],
        contains_any=["id_row"],
    )

    label_col = resolve_column_34(
        df.columns,
        ["label_stage23", "label", "outcome", "y_true"],
        contains_any=["label_stage23"],
    )

    fold_col = resolve_column_34(
        df.columns,
        ["outer_fold", "fold"],
        contains_all=["outer", "fold"],
        required=False,
    )

    raw_col = resolve_column_34(
        df.columns,
        [
            "prediction_raw",
            "raw_probability",
            "probability_raw",
            "prediction_raw_probability",
        ],
        contains_all=["raw"],
        contains_any=["prediction", "probability", "prob"],
    )

    platt_col = resolve_column_34(
        df.columns,
        [
            "prediction_platt",
            "prediction_platt_calibrated",
            "platt_probability",
            "probability_platt",
            "calibrated_probability",
            "prediction_calibrated",
        ],
        contains_any=["platt", "calibrated"],
    )

    model_name_col = resolve_column_34(
        df.columns,
        ["model_name", "model"],
        contains_all=["model", "name"],
        required=False,
    )

    model_version_col = resolve_column_34(
        df.columns,
        ["model_version", "version"],
        contains_all=["model", "version"],
        required=False,
    )

    out = pd.DataFrame({
        "id_row": df[id_col].astype(str),
        "label_stage23": pd.to_numeric(
            df[label_col], errors="raise"
        ).astype(np.int64),
        "prediction_raw": pd.to_numeric(
            df[raw_col], errors="raise"
        ).astype(np.float64),
        "prediction_platt": pd.to_numeric(
            df[platt_col], errors="raise"
        ).astype(np.float64),
    })

    if fold_col is None:
        out["outer_fold"] = np.full(
            len(out), expected_fold, dtype=np.int64
        )
    else:
        out["outer_fold"] = pd.to_numeric(
            df[fold_col], errors="raise"
        ).astype(np.int64)

    out["model_name"] = (
        df[model_name_col].astype(str)
        if model_name_col is not None
        else MODEL_NAME_34
    )

    out["model_version"] = (
        df[model_version_col].astype(str)
        if model_version_col is not None
        else MODEL_VERSION_34
    )

    out["source_table"] = source_table

    if set(out["outer_fold"].unique()) != {expected_fold}:
        raise RuntimeError(
            f"{source_table}: expected outer_fold={expected_fold}; "
            f"found={sorted(out['outer_fold'].unique())}"
        )

    if out["id_row"].duplicated().any():
        raise RuntimeError(
            f"{source_table}: duplicate id_row values found."
        )

    for col in ["prediction_raw", "prediction_platt"]:
        if out[col].isna().any():
            raise RuntimeError(
                f"{source_table}: missing values in {col}."
            )
        if not out[col].between(0, 1).all():
            raise RuntimeError(
                f"{source_table}: probabilities outside 0-1 in {col}."
            )

    return out[
        [
            "id_row",
            "outer_fold",
            "label_stage23",
            "prediction_raw",
            "prediction_platt",
            "model_name",
            "model_version",
            "source_table",
        ]
    ].copy()

# ------------------------------------------------------------
# 4. Load all five parsimonious clinical baseline folds
# ------------------------------------------------------------

clinical_frames_34 = []
source_schema_rows_34 = []

for fold_number in range(1, 6):
    table_id = source_clinical_tables_34[fold_number]

    print(
        f"Loading parsimonious clinical baseline outer-fold-{fold_number}: {table_id}"
    )

    query_job = client.query(
        f"SELECT * FROM `{table_id}`",
        location=BQ_LOCATION,
    )

    try:
        raw_df = query_job.to_dataframe(
            create_bqstorage_client=True
        )
        load_method = "BigQuery Storage API"
    except Exception:
        raw_df = query_job.to_dataframe(
            create_bqstorage_client=False
        )
        load_method = "Standard BigQuery API"

    source_schema_rows_34.append({
        "outer_fold": fold_number,
        "source_table": table_id,
        "source_rows": len(raw_df),
        "source_columns": " | ".join(map(str, raw_df.columns)),
        "load_method": load_method,
    })

    normalized = normalize_prediction_table_34(
        raw_dataframe=raw_df,
        expected_fold=fold_number,
        source_table=table_id,
    )

    clinical_frames_34.append(normalized)

source_schema_summary_34 = pd.DataFrame(
    source_schema_rows_34
)

clinical_pooled_34 = pd.concat(
    clinical_frames_34,
    ignore_index=True,
)

# ------------------------------------------------------------
# 5. Locked-core audit and hospital attachment
# ------------------------------------------------------------

required_core_columns_34 = {
    "id_row",
    "outer_fold",
    "label_stage23",
    "group_hospital",
}

missing_core_columns_34 = (
    required_core_columns_34
    - set(core_df_07B.columns)
)

if missing_core_columns_34:
    raise RuntimeError(
        "core_df_07B missing audit columns: "
        + ", ".join(sorted(missing_core_columns_34))
    )

core_audit_34 = core_df_07B[
    [
        "id_row",
        "outer_fold",
        "label_stage23",
        "group_hospital",
    ]
].copy()

core_audit_34["id_row"] = (
    core_audit_34["id_row"].astype(str)
)
core_audit_34["outer_fold"] = pd.to_numeric(
    core_audit_34["outer_fold"],
    errors="raise",
).astype(np.int64)
core_audit_34["label_stage23"] = pd.to_numeric(
    core_audit_34["label_stage23"],
    errors="raise",
).astype(np.int64)
core_audit_34["group_hospital"] = (
    core_audit_34["group_hospital"].astype(str)
)

if core_audit_34["id_row"].duplicated().any():
    raise RuntimeError(
        "core_df_07B contains duplicate id_row values."
    )

if len(clinical_pooled_34) != EXPECTED_TOTAL_ROWS_34:
    raise RuntimeError(
        f"parsimonious clinical baseline pooled rows={len(clinical_pooled_34)}, "
        f"expected={EXPECTED_TOTAL_ROWS_34}."
    )

if clinical_pooled_34["id_row"].duplicated().any():
    raise RuntimeError(
        "parsimonious clinical baseline pooled predictions contain duplicate patients."
    )

clinical_audited_34 = clinical_pooled_34.merge(
    core_audit_34,
    on="id_row",
    how="inner",
    suffixes=("_prediction", "_core"),
    validate="one_to_one",
)

if len(clinical_audited_34) != EXPECTED_TOTAL_ROWS_34:
    raise RuntimeError(
        "parsimonious clinical baseline-core merge did not retain all 58,491 patients."
    )

fold_mismatch_34 = int(
    (
        clinical_audited_34["outer_fold_prediction"]
        != clinical_audited_34["outer_fold_core"]
    ).sum()
)

label_mismatch_34 = int(
    (
        clinical_audited_34["label_stage23_prediction"]
        != clinical_audited_34["label_stage23_core"]
    ).sum()
)

if fold_mismatch_34 != 0:
    raise RuntimeError(
        f"parsimonious clinical baseline/core outer-fold mismatches={fold_mismatch_34}"
    )

if label_mismatch_34 != 0:
    raise RuntimeError(
        f"parsimonious clinical baseline/core outcome mismatches={label_mismatch_34}"
    )

clinical_audited_34 = clinical_audited_34.rename(
    columns={
        "outer_fold_prediction": "outer_fold",
        "label_stage23_prediction": "label_stage23",
    }
).drop(
    columns=[
        "outer_fold_core",
        "label_stage23_core",
    ]
)

if int(clinical_audited_34["label_stage23"].sum()) != EXPECTED_TOTAL_EVENTS_34:
    raise RuntimeError(
        "parsimonious clinical baseline pooled event count is not 3,032."
    )

if (
    clinical_audited_34["group_hospital"].nunique()
    != EXPECTED_TOTAL_HOSPITALS_34
):
    raise RuntimeError(
        "parsimonious clinical baseline pooled hospital count is not 198."
    )

hospital_fold_count_34 = (
    clinical_audited_34
    .groupby("group_hospital")["outer_fold"]
    .nunique()
)

if (hospital_fold_count_34 > 1).any():
    raise RuntimeError(
        "A hospital appears in more than one outer-test fold."
    )

# ------------------------------------------------------------
# 6. Fold integrity
# ------------------------------------------------------------

fold_integrity_rows_34 = []

for fold_number in range(1, 6):
    part = clinical_audited_34.loc[
        clinical_audited_34["outer_fold"] == fold_number
    ].copy()

    expected = EXPECTED_FOLD_STRUCTURE_34[fold_number]

    actual = {
        "rows": len(part),
        "events": int(part["label_stage23"].sum()),
        "nonevents": int(
            len(part) - part["label_stage23"].sum()
        ),
        "hospitals": int(
            part["group_hospital"].nunique()
        ),
    }

    for key, expected_value in expected.items():
        if actual[key] != expected_value:
            raise RuntimeError(
                f"Fold {fold_number} {key}: "
                f"found={actual[key]}, expected={expected_value}"
            )

    fold_integrity_rows_34.append({
        "outer_fold": fold_number,
        "prediction_rows": actual["rows"],
        "distinct_patients": part["id_row"].nunique(),
        "hospitals": actual["hospitals"],
        "events": actual["events"],
        "nonevents": actual["nonevents"],
        "minimum_raw_probability": float(
            part["prediction_raw"].min()
        ),
        "maximum_raw_probability": float(
            part["prediction_raw"].max()
        ),
        "minimum_platt_probability": float(
            part["prediction_platt"].min()
        ),
        "maximum_platt_probability": float(
            part["prediction_platt"].max()
        ),
    })

fold_integrity_summary_34 = pd.DataFrame(
    fold_integrity_rows_34
)

# ------------------------------------------------------------
# 7. Secure pooled parsimonious clinical baseline BigQuery table
# ------------------------------------------------------------

clinical_pooled_table_id_34 = (
    f"{TARGET_DATASET}."
    "model_clinical_lr_outer_predictions_all5_v2"
)

clinical_secure_df_34 = clinical_audited_34[
    [
        "id_row",
        "outer_fold",
        "label_stage23",
        "prediction_raw",
        "prediction_platt",
        "model_name",
        "model_version",
        "group_hospital",
    ]
].copy()

clinical_secure_df_34["id_row"] = (
    clinical_secure_df_34["id_row"].astype(str)
)
clinical_secure_df_34["group_hospital"] = (
    clinical_secure_df_34["group_hospital"].astype(str)
)

pooled_load_config_34 = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField("id_row", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("outer_fold", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("label_stage23", "INTEGER", mode="REQUIRED"),
        bigquery.SchemaField("prediction_raw", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("prediction_platt", "FLOAT", mode="REQUIRED"),
        bigquery.SchemaField("model_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("model_version", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("group_hospital", "STRING", mode="REQUIRED"),
    ],
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)

print("\nUploading secure pooled parsimonious clinical baseline prediction table:")
print(clinical_pooled_table_id_34)

client.load_table_from_dataframe(
    clinical_secure_df_34,
    clinical_pooled_table_id_34,
    job_config=pooled_load_config_34,
    location=BQ_LOCATION,
).result()

verify_sql_34 = f"""
SELECT
  COUNT(*) AS prediction_rows,
  COUNT(DISTINCT id_row) AS distinct_patients,
  COUNT(DISTINCT group_hospital) AS hospitals,
  COUNT(DISTINCT outer_fold) AS outer_folds,
  COUNTIF(label_stage23 = 1) AS events,
  COUNTIF(label_stage23 = 0) AS nonevents,
  COUNTIF(prediction_raw IS NULL) AS missing_raw_predictions,
  COUNTIF(prediction_platt IS NULL) AS missing_platt_predictions,
  COUNTIF(prediction_raw < 0 OR prediction_raw > 1)
    AS invalid_raw_predictions,
  COUNTIF(prediction_platt < 0 OR prediction_platt > 1)
    AS invalid_platt_predictions,
  MIN(prediction_raw) AS minimum_raw_probability,
  MAX(prediction_raw) AS maximum_raw_probability,
  MIN(prediction_platt) AS minimum_platt_probability,
  MAX(prediction_platt) AS maximum_platt_probability
FROM `{clinical_pooled_table_id_34}`
"""

clinical_bigquery_verification_34 = (
    client.query(
        verify_sql_34,
        location=BQ_LOCATION,
    )
    .to_dataframe()
)

vr = clinical_bigquery_verification_34.iloc[0]

expected_verification_34 = {
    "prediction_rows": EXPECTED_TOTAL_ROWS_34,
    "distinct_patients": EXPECTED_TOTAL_ROWS_34,
    "hospitals": EXPECTED_TOTAL_HOSPITALS_34,
    "outer_folds": 5,
    "events": EXPECTED_TOTAL_EVENTS_34,
    "nonevents": EXPECTED_TOTAL_NONEVENTS_34,
    "missing_raw_predictions": 0,
    "missing_platt_predictions": 0,
    "invalid_raw_predictions": 0,
    "invalid_platt_predictions": 0,
}

for field, expected_value in expected_verification_34.items():
    actual_value = int(vr[field])
    if actual_value != expected_value:
        raise RuntimeError(
            f"parsimonious clinical baseline pooled BigQuery verification {field}: "
            f"found={actual_value}, expected={expected_value}"
        )

# ------------------------------------------------------------
# 8. Metric helpers
# ------------------------------------------------------------

def logit_34(probabilities):
    p = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        p / (1 - p)
    ).reshape(-1, 1)


def calibration_intercept_slope_34(
    y_true,
    probabilities,
):
    model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    model.fit(
        logit_34(probabilities),
        y_true,
    )
    return (
        float(model.intercept_[0]),
        float(model.coef_[0][0]),
    )


def probability_metrics_34(
    y_true,
    probabilities,
):
    p = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    intercept, slope = calibration_intercept_slope_34(
        y_true,
        p,
    )

    return {
        "auroc": float(
            roc_auc_score(y_true, p)
        ),
        "auprc": float(
            average_precision_score(y_true, p)
        ),
        "brier": float(
            brier_score_loss(y_true, p)
        ),
        "log_loss": float(
            log_loss(
                y_true,
                p,
                labels=[0, 1],
            )
        ),
        "mean_predicted_risk": float(p.mean()),
        "observed_event_rate": float(np.mean(y_true)),
        "calibration_intercept": intercept,
        "calibration_slope": slope,
    }


def weighted_metric_bundle_34(
    y_true,
    probabilities,
    sample_weight,
):
    p = np.clip(
        np.asarray(probabilities, dtype=float),
        1e-8,
        1 - 1e-8,
    )

    return {
        "auroc": float(
            roc_auc_score(
                y_true,
                p,
                sample_weight=sample_weight,
            )
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                p,
                sample_weight=sample_weight,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                p,
                sample_weight=sample_weight,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                p,
                labels=[0, 1],
                sample_weight=sample_weight,
            )
        ),
    }

# ------------------------------------------------------------
# 9. Fold-level, macro, and pooled point metrics
# ------------------------------------------------------------

y_clinical_34 = clinical_audited_34[
    "label_stage23"
].to_numpy(dtype=np.int8)

raw_clinical_34 = clinical_audited_34[
    "prediction_raw"
].to_numpy(dtype=float)

platt_clinical_34 = clinical_audited_34[
    "prediction_platt"
].to_numpy(dtype=float)

fold_metric_rows_34 = []

for fold_number in range(1, 6):
    mask = (
        clinical_audited_34["outer_fold"]
        .to_numpy(dtype=int)
        == fold_number
    )

    fold_y = y_clinical_34[mask]

    for probability_type, vector in [
        ("raw", raw_clinical_34),
        ("platt_calibrated", platt_clinical_34),
    ]:
        metrics = probability_metrics_34(
            fold_y,
            vector[mask],
        )

        fold_metric_rows_34.append({
            "outer_fold": fold_number,
            "model": MODEL_NAME_34,
            "probability_type": probability_type,
            "patients": int(mask.sum()),
            "events": int(fold_y.sum()),
            **metrics,
        })

clinical_fold_metrics_34 = pd.DataFrame(
    fold_metric_rows_34
)

macro_rows_34 = []

for probability_type in ["raw", "platt_calibrated"]:
    subset = clinical_fold_metrics_34.loc[
        clinical_fold_metrics_34["probability_type"]
        == probability_type
    ]

    for metric_name in [
        "auroc",
        "auprc",
        "brier",
        "log_loss",
        "mean_predicted_risk",
        "calibration_intercept",
        "calibration_slope",
    ]:
        values = subset[
            metric_name
        ].to_numpy(dtype=float)

        macro_rows_34.append({
            "probability_type": probability_type,
            "metric": metric_name,
            "fold_mean": float(np.mean(values)),
            "fold_sd": float(np.std(values, ddof=1)),
            "fold_minimum": float(np.min(values)),
            "fold_maximum": float(np.max(values)),
        })

clinical_macro_summary_34 = pd.DataFrame(
    macro_rows_34
)

pooled_point_rows_34 = []

for probability_type, vector in [
    ("raw", raw_clinical_34),
    ("platt_calibrated", platt_clinical_34),
]:
    metrics = probability_metrics_34(
        y_clinical_34,
        vector,
    )

    pooled_point_rows_34.append({
        "model": MODEL_NAME_34,
        "probability_type": probability_type,
        "patients": EXPECTED_TOTAL_ROWS_34,
        "hospitals": EXPECTED_TOTAL_HOSPITALS_34,
        "events": EXPECTED_TOTAL_EVENTS_34,
        "nonevents": EXPECTED_TOTAL_NONEVENTS_34,
        **metrics,
    })

clinical_pooled_point_metrics_34 = pd.DataFrame(
    pooled_point_rows_34
)

# ------------------------------------------------------------
# 10. Selected-model consistency audit
# ------------------------------------------------------------

selected_model_paths_34 = {
    1: os.path.join(
        MODEL_OUTPUT_DIR,
        "33B_R2_clinical_selected_model_outer1.csv",
    ),
    2: os.path.join(
        MODEL_OUTPUT_DIR,
        "33C_clinical_selected_model_outer2.csv",
    ),
    3: os.path.join(
        MODEL_OUTPUT_DIR,
        "33D_clinical_selected_model_outer3.csv",
    ),
    4: os.path.join(
        MODEL_OUTPUT_DIR,
        "33E_clinical_selected_model_outer4.csv",
    ),
    5: os.path.join(
        MODEL_OUTPUT_DIR,
        "33F_clinical_selected_model_outer5.csv",
    ),
}

selection_rows_34 = []

expected_selected_candidates_34 = {
    1: "CLIN01",
    2: "CLIN01",
    3: "CLIN01",
    4: "CLIN03",
    5: "CLIN01",
}

for fold_number, path in selected_model_paths_34.items():
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    df = pd.read_csv(path)

    if len(df) != 1:
        raise RuntimeError(
            f"Fold {fold_number} selected-model file must have one row."
        )

    row = df.iloc[0]

    file_outer_fold = int(
        pd.to_numeric(
            row["outer_fold"],
            errors="raise",
        )
    )

    if file_outer_fold != fold_number:
        raise RuntimeError(
            f"Fold {fold_number} selection file outer_fold="
            f"{file_outer_fold}."
        )

    observed_protocol_sha = str(row["protocol_sha256"])
    if observed_protocol_sha != EXPECTED_CLINICAL_PROTOCOL_SHA_34:
        raise RuntimeError(
            f"Fold {fold_number}: clinical protocol SHA mismatch "
            "in selected-model file."
        )

    if "full_inner_refit_correction_sha256" in row.index:
        observed_correction_sha = str(
            row["full_inner_refit_correction_sha256"]
        )
        if observed_correction_sha != EXPECTED_FULL_REFIT_CORRECTION_SHA_34:
            raise RuntimeError(
                f"Fold {fold_number}: full-refit correction SHA mismatch."
            )

    selected_candidate = str(row["selected_candidate"])
    expected_candidate = expected_selected_candidates_34[fold_number]

    if selected_candidate != expected_candidate:
        raise RuntimeError(
            f"Fold {fold_number}: selected candidate={selected_candidate}; "
            f"expected={expected_candidate} from completed corrected run."
        )

    selection_rows_34.append({
        "outer_fold": fold_number,
        "selected_candidate": selected_candidate,
        "selected_C": float(row["selected_C"]),
        "selected_l1_ratio": float(row["selected_l1_ratio"]),
        "inner_oof_auprc": float(row["inner_oof_auprc"]),
        "inner_oof_auroc": float(row["inner_oof_auroc"]),
        "inner_oof_brier": float(row["inner_oof_brier"]),
        "inner_oof_log_loss": float(row["inner_oof_log_loss"]),
        "platt_intercept": float(row["platt_intercept"]),
        "platt_slope": float(row["platt_slope"]),
        "protocol_sha256": observed_protocol_sha,
        "selection_file": path,
    })

selected_model_consistency_34 = pd.DataFrame(
    selection_rows_34
)

if not (
    selected_model_consistency_34.loc[
        selected_model_consistency_34["outer_fold"].isin([1, 2, 3, 5]),
        "selected_C",
    ].eq(0.03).all()
    and selected_model_consistency_34.loc[
        selected_model_consistency_34["outer_fold"].isin([1, 2, 3, 5]),
        "selected_l1_ratio",
    ].eq(0.0).all()
    and selected_model_consistency_34.loc[
        selected_model_consistency_34["outer_fold"].eq(4),
        "selected_C",
    ].eq(0.30).all()
    and selected_model_consistency_34.loc[
        selected_model_consistency_34["outer_fold"].eq(4),
        "selected_l1_ratio",
    ].eq(0.0).all()
):
    raise RuntimeError(
        "Selected clinical hyperparameters do not match the five completed folds."
    )

# ------------------------------------------------------------
# 11. Hospital-cluster bootstrap
# ------------------------------------------------------------

print(
    f"\nRunning {BOOTSTRAP_REPLICATES_34:,} "
    "hospital-cluster bootstrap replicates for parsimonious clinical baseline..."
)

hospital_categorical_34 = pd.Categorical(
    clinical_audited_34["group_hospital"]
)

hospital_codes_34 = (
    hospital_categorical_34.codes.astype(int)
)

hospital_labels_34 = list(
    hospital_categorical_34.categories
)

n_hospitals_34 = len(hospital_labels_34)

if n_hospitals_34 != EXPECTED_TOTAL_HOSPITALS_34:
    raise RuntimeError(
        "Bootstrap hospital count is not 198."
    )

rng_34 = np.random.default_rng(
    BOOTSTRAP_SEED_34
)

bootstrap_rows_34 = []

for bootstrap_index in range(
    BOOTSTRAP_REPLICATES_34
):
    sampled_codes = rng_34.integers(
        low=0,
        high=n_hospitals_34,
        size=n_hospitals_34,
    )

    multiplicity = np.bincount(
        sampled_codes,
        minlength=n_hospitals_34,
    )

    weights = multiplicity[
        hospital_codes_34
    ].astype(float)

    positive_weight = float(
        weights[y_clinical_34 == 1].sum()
    )
    negative_weight = float(
        weights[y_clinical_34 == 0].sum()
    )

    if positive_weight <= 0 or negative_weight <= 0:
        continue

    raw_metrics = weighted_metric_bundle_34(
        y_clinical_34,
        raw_clinical_34,
        weights,
    )
    platt_metrics = weighted_metric_bundle_34(
        y_clinical_34,
        platt_clinical_34,
        weights,
    )

    bootstrap_rows_34.append({
        "bootstrap_replicate": bootstrap_index + 1,
        "unique_sampled_hospitals": int(
            np.count_nonzero(multiplicity)
        ),
        "weighted_patients": float(weights.sum()),
        "weighted_events": positive_weight,
        "auroc_raw": raw_metrics["auroc"],
        "auroc_platt": platt_metrics["auroc"],
        "auprc_raw": raw_metrics["auprc"],
        "auprc_platt": platt_metrics["auprc"],
        "brier_raw": raw_metrics["brier"],
        "brier_platt": platt_metrics["brier"],
        "log_loss_raw": raw_metrics["log_loss"],
        "log_loss_platt": platt_metrics["log_loss"],
    })

    if (
        bootstrap_index == 0
        or (bootstrap_index + 1) % 100 == 0
    ):
        print(
            "  Completed parsimonious clinical baseline bootstrap replicate",
            bootstrap_index + 1,
            "/",
            BOOTSTRAP_REPLICATES_34,
        )

clinical_bootstrap_replicates_34 = pd.DataFrame(
    bootstrap_rows_34
)

if len(clinical_bootstrap_replicates_34) < (
    BOOTSTRAP_REPLICATES_34 * 0.99
):
    raise RuntimeError(
        "Fewer than 99% of parsimonious clinical baseline bootstrap replicates completed."
    )

clinical_ci_rows_34 = []

point_lookup_34 = {
    "raw": clinical_pooled_point_metrics_34.loc[
        clinical_pooled_point_metrics_34["probability_type"]
        == "raw"
    ].iloc[0],
    "platt_calibrated": clinical_pooled_point_metrics_34.loc[
        clinical_pooled_point_metrics_34["probability_type"]
        == "platt_calibrated"
    ].iloc[0],
}

ci_specs_34 = [
    ("auroc", "raw", "auroc_raw"),
    ("auroc", "platt_calibrated", "auroc_platt"),
    ("auprc", "raw", "auprc_raw"),
    ("auprc", "platt_calibrated", "auprc_platt"),
    ("brier", "raw", "brier_raw"),
    ("brier", "platt_calibrated", "brier_platt"),
    ("log_loss", "raw", "log_loss_raw"),
    ("log_loss", "platt_calibrated", "log_loss_platt"),
]

for metric, probability_type, column in ci_specs_34:
    values = clinical_bootstrap_replicates_34[
        column
    ].dropna().to_numpy(dtype=float)

    clinical_ci_rows_34.append({
        "metric": metric,
        "probability_type": probability_type,
        "point_estimate": float(
            point_lookup_34[
                probability_type
            ][metric]
        ),
        "bootstrap_replicates": len(values),
        "bootstrap_mean": float(
            np.mean(values)
        ),
        "bootstrap_standard_error": float(
            np.std(values, ddof=1)
        ),
        "ci_95_lower": float(
            np.quantile(values, 0.025)
        ),
        "ci_95_upper": float(
            np.quantile(values, 0.975)
        ),
        "bootstrap_unit": "hospital",
        "bootstrap_seed": BOOTSTRAP_SEED_34,
    })

clinical_bootstrap_ci_34 = pd.DataFrame(
    clinical_ci_rows_34
)

# ------------------------------------------------------------
# 12. Pooled calibration deciles
# ------------------------------------------------------------

def calibration_quantiles_34(
    dataframe,
    probability_column,
    probability_type,
    requested_bins=10,
):
    working = dataframe[
        ["label_stage23", probability_column]
    ].copy()

    working["risk_group"] = pd.qcut(
        working[probability_column],
        q=requested_bins,
        labels=False,
        duplicates="drop",
    )

    summary = (
        working
        .groupby(
            "risk_group",
            observed=True,
        )
        .agg(
            patients=("label_stage23", "size"),
            events=("label_stage23", "sum"),
            mean_predicted_risk=(probability_column, "mean"),
            minimum_predicted_risk=(probability_column, "min"),
            maximum_predicted_risk=(probability_column, "max"),
        )
        .reset_index()
    )

    summary["observed_event_rate"] = (
        summary["events"]
        / summary["patients"]
    )

    summary["probability_type"] = probability_type
    summary["requested_bins"] = requested_bins
    summary["actual_bins"] = len(summary)

    return summary[
        [
            "probability_type",
            "requested_bins",
            "actual_bins",
            "risk_group",
            "patients",
            "events",
            "mean_predicted_risk",
            "observed_event_rate",
            "minimum_predicted_risk",
            "maximum_predicted_risk",
        ]
    ]


clinical_calibration_deciles_34 = pd.concat(
    [
        calibration_quantiles_34(
            clinical_audited_34,
            "prediction_raw",
            "raw",
        ),
        calibration_quantiles_34(
            clinical_audited_34,
            "prediction_platt",
            "platt_calibrated",
        ),
    ],
    ignore_index=True,
)

# ------------------------------------------------------------
# 13. Save aggregate outputs only
# ------------------------------------------------------------

paths_34 = {
    "source_schema": os.path.join(
        MODEL_OUTPUT_DIR,
        "34A_clinical_baseline_source_prediction_schema_audit.csv",
    ),
    "fold_integrity": os.path.join(
        MODEL_OUTPUT_DIR,
        "34A_clinical_baseline_five_fold_prediction_integrity.csv",
    ),
    "selected_models": os.path.join(
        MODEL_OUTPUT_DIR,
        "34A_clinical_baseline_selected_model_consistency.csv",
    ),
    "fold_metrics": os.path.join(
        MODEL_OUTPUT_DIR,
        "34B_clinical_baseline_outer_fold_metrics.csv",
    ),
    "macro_summary": os.path.join(
        MODEL_OUTPUT_DIR,
        "34B_clinical_baseline_macro_fold_summary.csv",
    ),
    "pooled_point": os.path.join(
        MODEL_OUTPUT_DIR,
        "34B_clinical_baseline_pooled_point_metrics.csv",
    ),
    "bootstrap_replicates": os.path.join(
        MODEL_OUTPUT_DIR,
        "34C_clinical_baseline_hospital_bootstrap_replicates.csv",
    ),
    "bootstrap_ci": os.path.join(
        MODEL_OUTPUT_DIR,
        "34C_clinical_baseline_hospital_bootstrap_confidence_intervals.csv",
    ),
    "calibration_deciles": os.path.join(
        MODEL_OUTPUT_DIR,
        "34D_clinical_baseline_pooled_calibration_deciles.csv",
    ),
}

source_schema_summary_34.to_csv(
    paths_34["source_schema"],
    index=False,
)
fold_integrity_summary_34.to_csv(
    paths_34["fold_integrity"],
    index=False,
)
selected_model_consistency_34.to_csv(
    paths_34["selected_models"],
    index=False,
)
clinical_fold_metrics_34.to_csv(
    paths_34["fold_metrics"],
    index=False,
)
clinical_macro_summary_34.to_csv(
    paths_34["macro_summary"],
    index=False,
)
clinical_pooled_point_metrics_34.to_csv(
    paths_34["pooled_point"],
    index=False,
)
clinical_bootstrap_replicates_34.to_csv(
    paths_34["bootstrap_replicates"],
    index=False,
)
clinical_bootstrap_ci_34.to_csv(
    paths_34["bootstrap_ci"],
    index=False,
)
clinical_calibration_deciles_34.to_csv(
    paths_34["calibration_deciles"],
    index=False,
)

# ------------------------------------------------------------
# 14. Final parsimonious clinical baseline pooled manifest
# ------------------------------------------------------------

manifest_34 = {
    "analysis_version": "34",
    "analysis_type": "five_fold_pooled_parsimonious_clinical_baseline_evaluation",
    "analysis_role": "additional_post_hoc_clinical_baseline",
    "analysis_status": "post_hoc_exploratory_corrected",
    "patients": EXPECTED_TOTAL_ROWS_34,
    "hospitals": EXPECTED_TOTAL_HOSPITALS_34,
    "events": EXPECTED_TOTAL_EVENTS_34,
    "nonevents": EXPECTED_TOTAL_NONEVENTS_34,
    "outer_folds": 5,
    "clinical_protocol_sha256": EXPECTED_CLINICAL_PROTOCOL_SHA_34,
    "convergence_amendment_sha256": EXPECTED_CONVERGENCE_AMENDMENT_SHA_34,
    "full_inner_refit_correction_sha256": EXPECTED_FULL_REFIT_CORRECTION_SHA_34,
    "clinical_secure_pooled_prediction_table": clinical_pooled_table_id_34,
    "selected_candidates_by_fold": expected_selected_candidates_34,
    "bootstrap_replicates": BOOTSTRAP_REPLICATES_34,
    "bootstrap_unit": "hospital",
    "bootstrap_seed": BOOTSTRAP_SEED_34,
    "predictive_model_fit_performed": False,
    "retuning_performed": False,
    "recalibration_performed": False,
    "diagnostic_calibration_intercept_slope_evaluated": True,
    "model_comparison_performed": False,
    "patient_level_file_written_to_drive": False,
    "patient_level_predictions_written_to_bigquery": True,
    "bigquery_dml_used": False,
    "outputs": paths_34,
}

manifest_path_34 = os.path.join(
    MODEL_OUTPUT_DIR,
    "34E_final_clinical_baseline_pooled_evaluation_manifest.json",
)
manifest_sha_path_34 = os.path.join(
    MODEL_OUTPUT_DIR,
    "34E_final_clinical_baseline_pooled_evaluation_manifest_SHA256.txt",
)

with open(
    manifest_path_34,
    "w",
    encoding="utf-8",
) as fh:
    json.dump(
        manifest_34,
        fh,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )

with open(
    manifest_path_34,
    "rb",
) as fh:
    manifest_sha_34 = hashlib.sha256(
        fh.read()
    ).hexdigest()

with open(
    manifest_sha_path_34,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(manifest_sha_34 + "\n")

# ------------------------------------------------------------
# 15. Display final audit
# ------------------------------------------------------------

print("\n30 CLINICAL BASELINE FIVE-FOLD PREDICTION INTEGRITY")
display(fold_integrity_summary_34)

print("\n30 CLINICAL BASELINE SELECTED MODEL CONSISTENCY")
display(selected_model_consistency_34)

print("\n30 CLINICAL BASELINE OUTER-FOLD METRICS")
display(clinical_fold_metrics_34)

print("\n30 CLINICAL BASELINE MACRO FOLD SUMMARY")
display(clinical_macro_summary_34)

print("\n30 CLINICAL BASELINE PRIMARY POOLED POINT METRICS")
display(clinical_pooled_point_metrics_34)

print("\n30 CLINICAL BASELINE HOSPITAL-CLUSTER BOOTSTRAP CIs")
display(clinical_bootstrap_ci_34)

print("\n30 CLINICAL BASELINE POOLED CALIBRATION DECILES")
display(clinical_calibration_deciles_34)

print("\n30 CLINICAL BASELINE POOLED BIGQUERY VERIFICATION")
display(clinical_bigquery_verification_34)

print("\nFinal parsimonious clinical baseline pooled manifest SHA-256:")
print(manifest_sha_34)

print("\nSaved aggregate outputs:")
for path in paths_34.values():
    print(path)
print(manifest_path_34)
print(manifest_sha_path_34)

print(
    "\n34 PASS: Five locked parsimonious clinical baseline outer-test folds were pooled "
    "and evaluated with 2,000 hospital-cluster bootstrap replicates."
)
print(
    "No model fitting, retuning, model comparison, class weighting, "
    "or SMOTE was performed."
)
print(
    "Patient-level parsimonious clinical baseline prediction data remained only in "
    "BigQuery/RAM. No patient-level prediction file was written "
    "to Google Drive."
)

In [ ]:
import os
import re
import json
import hashlib
import pandas as pd
from google.cloud import bigquery
from IPython.display import display

print("STARTING KIDNEY/TEMPORAL FEATURE PROVENANCE DISCOVERY AUDIT — CODE VERSION 35A")

# ============================================================
# Purpose
# ============================================================
# This is a READ-ONLY provenance discovery audit.
# It does NOT fit models, access outer-test performance for tuning,
# alter predictors, or modify BigQuery tables.
#
# Critical targets:
#   x_reference_creatinine
#   x_stage1_at_landmark
#   x_lab_creatinine_last
#   label_stage23
#
# We need to establish whether their construction is based only on
# information available by the 12 h landmark, while the outcome
# belongs strictly to the >12–72 h window.
#
# This stage discovers SQL lineage / view definitions / historical
# construction queries. It does NOT declare PASS unless the actual
# temporal logic is visible and auditable.
# ============================================================

if "client" not in globals():
    raise RuntimeError("Önce BigQuery client'ını oluşturan temel hücreyi çalıştır.")

if "MODEL_OUTPUT_DIR" not in globals():
    raise RuntimeError("Önce MODEL_OUTPUT_DIR tanımlı temel hücreyi çalıştır.")

PROJECT_ID_35A = client.project
DATASET_ID_35A = "aki_jcmc_v2"
LOCATION_35A = "US"

if "TARGET_DATASET" in globals():
    parts_35A = str(TARGET_DATASET).split(".")
    if len(parts_35A) == 2:
        PROJECT_ID_35A = parts_35A[0]
        DATASET_ID_35A = parts_35A[1]

TARGET_FEATURES_35A = [
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
    "label_stage23",
]

TARGET_OBJECTS_35A = [
    "feature_matrix_core_view_v1",
    "feature_matrix_extended_view_v1",
    "feature_matrix_core_outerfold_v1",
    "feature_matrix_extended_outerfold_v1",
]

EXPECTED_LANDMARK_MINUTES_35A = 12 * 60
EXPECTED_OUTCOME_END_MINUTES_35A = 72 * 60

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def bq_query_35A(sql, params=None):
    cfg = bigquery.QueryJobConfig()
    if params:
        cfg.query_parameters = params
    return client.query(
        sql,
        job_config=cfg,
        location=LOCATION_35A,
    ).to_dataframe()

def sha256_text_35A(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

def extract_backtick_refs_35A(sql_text):
    if not isinstance(sql_text, str):
        return []
    refs = re.findall(r"`([^`]+)`", sql_text)
    cleaned = []
    for ref in refs:
        if ref.upper().endswith("INFORMATION_SCHEMA.VIEWS"):
            continue
        cleaned.append(ref)
    return sorted(set(cleaned))

def context_snippets_35A(sql_text, token, radius=450):
    if not isinstance(sql_text, str):
        return []
    low = sql_text.lower()
    tok = token.lower()
    out = []
    start = 0
    while True:
        idx = low.find(tok, start)
        if idx < 0:
            break
        a = max(0, idx - radius)
        b = min(len(sql_text), idx + len(token) + radius)
        out.append(sql_text[a:b])
        start = idx + len(tok)
    return out[:10]

# ------------------------------------------------------------
# 1. Dataset table/view inventory
# ------------------------------------------------------------

tables_sql_35A = f"""
SELECT
  table_name,
  table_type,
  creation_time,
  ddl
FROM `{PROJECT_ID_35A}.{DATASET_ID_35A}.INFORMATION_SCHEMA.TABLES`
ORDER BY table_name
"""

tables_35A = bq_query_35A(tables_sql_35A)

views_sql_35A = f"""
SELECT
  table_name,
  view_definition,
  check_option,
  use_standard_sql
FROM `{PROJECT_ID_35A}.{DATASET_ID_35A}.INFORMATION_SCHEMA.VIEWS`
ORDER BY table_name
"""

views_35A = bq_query_35A(views_sql_35A)

columns_sql_35A = f"""
SELECT
  table_name,
  column_name,
  data_type,
  is_nullable,
  ordinal_position
FROM `{PROJECT_ID_35A}.{DATASET_ID_35A}.INFORMATION_SCHEMA.COLUMNS`
WHERE LOWER(column_name) IN (
  'x_reference_creatinine',
  'x_stage1_at_landmark',
  'x_lab_creatinine_last',
  'label_stage23'
)
ORDER BY table_name, ordinal_position
"""

target_column_inventory_35A = bq_query_35A(columns_sql_35A)

# ------------------------------------------------------------
# 2. Target object definitions and references
# ------------------------------------------------------------

target_view_rows_35A = []
lineage_rows_35A = []
snippet_rows_35A = []

view_map_35A = {
    str(row["table_name"]): str(row["view_definition"])
    for _, row in views_35A.iterrows()
}

for object_name in TARGET_OBJECTS_35A:
    definition = view_map_35A.get(object_name)

    target_view_rows_35A.append({
        "object_name": object_name,
        "is_view": definition is not None,
        "definition_sha256": (
            sha256_text_35A(definition)
            if definition is not None
            else None
        ),
        "definition_length": (
            len(definition)
            if definition is not None
            else None
        ),
    })

    if definition is not None:
        for ref in extract_backtick_refs_35A(definition):
            lineage_rows_35A.append({
                "source_object": object_name,
                "referenced_object": ref,
            })

        for token in TARGET_FEATURES_35A:
            snippets = context_snippets_35A(
                definition,
                token,
            )
            for i, snippet in enumerate(snippets, start=1):
                snippet_rows_35A.append({
                    "source_type": "target_view_definition",
                    "source_object": object_name,
                    "target_token": token,
                    "snippet_index": i,
                    "snippet": snippet,
                })

target_view_audit_35A = pd.DataFrame(target_view_rows_35A)
lineage_refs_35A = pd.DataFrame(lineage_rows_35A)
definition_snippets_35A = pd.DataFrame(snippet_rows_35A)

# ------------------------------------------------------------
# 3. Search ALL current dataset view definitions for target tokens
# ------------------------------------------------------------

all_view_hits_35A = []

for _, row in views_35A.iterrows():
    table_name = str(row["table_name"])
    definition = str(row["view_definition"])

    for token in TARGET_FEATURES_35A:
        if token.lower() in definition.lower():
            snippets = context_snippets_35A(
                definition,
                token,
            )
            for i, snippet in enumerate(snippets, start=1):
                all_view_hits_35A.append({
                    "view_name": table_name,
                    "target_token": token,
                    "snippet_index": i,
                    "view_definition_sha256": sha256_text_35A(
                        definition
                    ),
                    "snippet": snippet,
                })

all_dataset_view_hits_35A = pd.DataFrame(all_view_hits_35A)

# ------------------------------------------------------------
# 4. Search BigQuery query-job history for construction SQL
# ------------------------------------------------------------

job_search_terms_35A = (
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
    "label_stage23",
    "feature_matrix_core",
    "feature_matrix_extended",
)

job_history_sql_35A = f"""
SELECT
  creation_time,
  job_id,
  user_email,
  query
FROM `{PROJECT_ID_35A}.region-us.INFORMATION_SCHEMA.JOBS_BY_PROJECT`
WHERE job_type = 'QUERY'
  AND query IS NOT NULL
  AND (
    REGEXP_CONTAINS(LOWER(query), r'x_reference_creatinine')
    OR REGEXP_CONTAINS(LOWER(query), r'x_stage1_at_landmark')
    OR REGEXP_CONTAINS(LOWER(query), r'x_lab_creatinine_last')
    OR REGEXP_CONTAINS(LOWER(query), r'label_stage23')
    OR REGEXP_CONTAINS(LOWER(query), r'feature_matrix_core')
    OR REGEXP_CONTAINS(LOWER(query), r'feature_matrix_extended')
  )
ORDER BY creation_time DESC
LIMIT 500
"""

try:
    job_history_35A = bq_query_35A(job_history_sql_35A)
    job_history_access_35A = "PASS"
except Exception as exc:
    job_history_35A = pd.DataFrame(
        columns=[
            "creation_time",
            "job_id",
            "user_email",
            "query",
        ]
    )
    job_history_access_35A = (
        "UNAVAILABLE: " + str(exc)
    )

job_snippet_rows_35A = []

for _, row in job_history_35A.iterrows():
    query_text = str(row.get("query", ""))
    for token in TARGET_FEATURES_35A:
        if token.lower() in query_text.lower():
            for i, snippet in enumerate(
                context_snippets_35A(
                    query_text,
                    token,
                    radius=700,
                ),
                start=1,
            ):
                job_snippet_rows_35A.append({
                    "creation_time": row.get(
                        "creation_time"
                    ),
                    "job_id": row.get("job_id"),
                    "target_token": token,
                    "snippet_index": i,
                    "query_sha256": sha256_text_35A(
                        query_text
                    ),
                    "snippet": snippet,
                })

job_history_snippets_35A = pd.DataFrame(
    job_snippet_rows_35A
)

# ------------------------------------------------------------
# 5. Temporal-keyword evidence scan
# ------------------------------------------------------------

TEMPORAL_PATTERNS_35A = {
    "landmark_12h_text": [
        "12 hour",
        "12-hour",
        "12h",
        "720",
    ],
    "outcome_72h_text": [
        "72 hour",
        "72-hour",
        "72h",
        "4320",
    ],
    "strict_post_landmark_operators": [
        "> 720",
        ">720",
        "> 12",
        ">12",
    ],
    "pre_or_at_landmark_operators": [
        "<= 720",
        "<=720",
        "< 720",
        "<720",
    ],
}

evidence_rows_35A = []

combined_sources_35A = []

for _, row in all_dataset_view_hits_35A.iterrows():
    combined_sources_35A.append((
        "view",
        str(row["view_name"]),
        str(row["target_token"]),
        str(row["snippet"]),
    ))

for _, row in job_history_snippets_35A.iterrows():
    combined_sources_35A.append((
        "job_history",
        str(row["job_id"]),
        str(row["target_token"]),
        str(row["snippet"]),
    ))

for source_type, source_id, token, snippet in combined_sources_35A:
    low = snippet.lower()
    for evidence_type, patterns in TEMPORAL_PATTERNS_35A.items():
        matched = [
            p for p in patterns
            if p.lower() in low
        ]
        if matched:
            evidence_rows_35A.append({
                "source_type": source_type,
                "source_id": source_id,
                "target_token": token,
                "evidence_type": evidence_type,
                "matched_patterns": " | ".join(matched),
                "snippet": snippet,
            })

temporal_evidence_35A = pd.DataFrame(evidence_rows_35A)

# ------------------------------------------------------------
# 6. Aggregate discovery status
# ------------------------------------------------------------

status_rows_35A = []

for token in TARGET_FEATURES_35A:
    view_hits = (
        int(
            (
                all_dataset_view_hits_35A[
                    "target_token"
                ] == token
            ).sum()
        )
        if not all_dataset_view_hits_35A.empty
        else 0
    )

    job_hits = (
        int(
            (
                job_history_snippets_35A[
                    "target_token"
                ] == token
            ).sum()
        )
        if not job_history_snippets_35A.empty
        else 0
    )

    temporal_hits = (
        int(
            (
                temporal_evidence_35A[
                    "target_token"
                ] == token
            ).sum()
        )
        if not temporal_evidence_35A.empty
        else 0
    )

    if view_hits + job_hits == 0:
        discovery_status = "NOT_FOUND"
    elif temporal_hits == 0:
        discovery_status = "FOUND_BUT_TEMPORAL_LOGIC_NOT_YET_EXPLICIT"
    else:
        discovery_status = "FOUND_WITH_TEMPORAL_EVIDENCE_REQUIRES_MANUAL_AUDIT"

    status_rows_35A.append({
        "target_feature": token,
        "view_definition_hits": view_hits,
        "job_history_hits": job_hits,
        "temporal_evidence_hits": temporal_hits,
        "discovery_status": discovery_status,
    })

discovery_summary_35A = pd.DataFrame(
    status_rows_35A
)

# ------------------------------------------------------------
# 7. Save only metadata / SQL provenance to Drive
#    (no patient-level data)
# ------------------------------------------------------------

paths_35A = {
    "tables_inventory": os.path.join(
        MODEL_OUTPUT_DIR,
        "35A_dataset_table_inventory.csv",
    ),
    "target_column_inventory": os.path.join(
        MODEL_OUTPUT_DIR,
        "35A_target_column_inventory.csv",
    ),
    "target_view_audit": os.path.join(
        MODEL_OUTPUT_DIR,
        "35A_target_view_definition_audit.csv",
    ),
    "lineage_refs": os.path.join(
        MODEL_OUTPUT_DIR,
        "35A_target_view_lineage_references.csv",
    ),
    "view_hits": os.path.join(
        MODEL_OUTPUT_DIR,
        "35A_target_feature_view_definition_hits.csv",
    ),
    "job_hits": os.path.join(
        MODEL_OUTPUT_DIR,
        "35A_target_feature_job_history_hits.csv",
    ),
    "temporal_evidence": os.path.join(
        MODEL_OUTPUT_DIR,
        "35A_temporal_keyword_evidence.csv",
    ),
    "discovery_summary": os.path.join(
        MODEL_OUTPUT_DIR,
        "35A_provenance_discovery_summary.csv",
    ),
    "manifest": os.path.join(
        MODEL_OUTPUT_DIR,
        "35A_provenance_discovery_manifest.json",
    ),
    "manifest_sha": os.path.join(
        MODEL_OUTPUT_DIR,
        "35A_provenance_discovery_manifest_SHA256.txt",
    ),
}

tables_35A.to_csv(
    paths_35A["tables_inventory"],
    index=False,
)
target_column_inventory_35A.to_csv(
    paths_35A["target_column_inventory"],
    index=False,
)
target_view_audit_35A.to_csv(
    paths_35A["target_view_audit"],
    index=False,
)
lineage_refs_35A.to_csv(
    paths_35A["lineage_refs"],
    index=False,
)
all_dataset_view_hits_35A.to_csv(
    paths_35A["view_hits"],
    index=False,
)
job_history_snippets_35A.to_csv(
    paths_35A["job_hits"],
    index=False,
)
temporal_evidence_35A.to_csv(
    paths_35A["temporal_evidence"],
    index=False,
)
discovery_summary_35A.to_csv(
    paths_35A["discovery_summary"],
    index=False,
)

manifest_35A = {
    "analysis_version": "35A",
    "analysis_type": "kidney_temporal_feature_provenance_discovery",
    "project_id": PROJECT_ID_35A,
    "dataset_id": DATASET_ID_35A,
    "location": LOCATION_35A,
    "landmark_minutes": EXPECTED_LANDMARK_MINUTES_35A,
    "outcome_end_minutes": EXPECTED_OUTCOME_END_MINUTES_35A,
    "target_features": TARGET_FEATURES_35A,
    "target_objects": TARGET_OBJECTS_35A,
    "job_history_access": job_history_access_35A,
    "patient_level_data_written_to_drive": False,
    "bigquery_dml_used": False,
    "model_fit_performed": False,
    "retuning_performed": False,
    "clinical_performance_result_used": False,
    "discovery_only_not_final_leakage_pass": True,
    "output_files": paths_35A,
}

manifest_text_35A = json.dumps(
    manifest_35A,
    indent=2,
    sort_keys=True,
)

with open(
    paths_35A["manifest"],
    "w",
    encoding="utf-8",
) as fh:
    fh.write(manifest_text_35A)

manifest_sha_35A = sha256_text_35A(
    manifest_text_35A
)

with open(
    paths_35A["manifest_sha"],
    "w",
    encoding="utf-8",
) as fh:
    fh.write(manifest_sha_35A + "\n")

# ------------------------------------------------------------
# 8. Display compact audit output
# ------------------------------------------------------------

print("\n35A TARGET COLUMN INVENTORY")
display(target_column_inventory_35A)

print("\n35A TARGET VIEW DEFINITION AUDIT")
display(target_view_audit_35A)

print("\n35A LINEAGE REFERENCES")
display(lineage_refs_35A)

print("\n35A PROVENANCE DISCOVERY SUMMARY")
display(discovery_summary_35A)

print("\n35A TEMPORAL EVIDENCE — FIRST 30")
display(temporal_evidence_35A.head(30))

print("\nBigQuery job-history access:")
print(job_history_access_35A)

print("\n35A manifest SHA-256:")
print(manifest_sha_35A)

print("\nSaved:")
for key, value in paths_35A.items():
    print(value)

print(
    "\n35A COMPLETE: provenance discovery finished."
)
print(
    "IMPORTANT: This is discovery only. Do NOT call leakage PASS yet."
)
print(
    "Next step is a targeted 35B audit based on the exact SQL lineage "
    "found here."
)

In [ ]:
import os
import re
import json
import math
import hashlib
from collections import deque

import numpy as np
import pandas as pd
from google.cloud import bigquery
from IPython.display import display

print("STARTING TARGETED KIDNEY/TEMPORAL SQL LINEAGE AUDIT — CODE VERSION 35B")

# ============================================================
# 35B — TARGETED SQL LINEAGE / CONSTRUCTION AUDIT
#
# Purpose:
#   Reconstruct the actual SQL provenance for:
#     x_reference_creatinine
#     x_stage1_at_landmark
#     x_lab_creatinine_last
#     label_stage23
#
# Locked temporal rule:
#   predictors: information available at or before 12 h (<= 720 min)
#   outcome:    strictly after 12 h and through 72 h (>720 to <=4320 min)
#
# Important:
#   - READ ONLY BigQuery audit
#   - no model fitting
#   - no retuning
#   - no recalibration
#   - no BigQuery DML
#   - no patient-level data written to Drive
#
# 35B finds and ranks the historical SQL that actually constructed
# the relevant feature/cohort objects and surfaces the exact temporal
# predicate evidence for manual scientific adjudication.
# ============================================================

EXPECTED_35A_MANIFEST_SHA = (
    "b383f97b97a285c6b88f1c909855d7ff"
    "611e20d1cd225dbddb4720684a0ef804"
)

TARGET_FEATURES = [
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
    "label_stage23",
]

LANDMARK_MIN = 720
OUTCOME_END_MIN = 4320

if "client" not in globals():
    raise RuntimeError(
        "Önce BigQuery client'ını oluşturan temel hücreyi çalıştır."
    )

if "MODEL_OUTPUT_DIR" not in globals():
    raise RuntimeError(
        "Önce MODEL_OUTPUT_DIR tanımlı temel hücreyi çalıştır."
    )

PROJECT_ID = client.project
DATASET_ID = "aki_jcmc_v2"
BQ_LOCATION = "US"

if "TARGET_DATASET" in globals():
    parts = str(TARGET_DATASET).split(".")
    if len(parts) == 2:
        PROJECT_ID = parts[0]
        DATASET_ID = parts[1]

# ------------------------------------------------------------
# 1. Verify 35A provenance-discovery lock
# ------------------------------------------------------------

manifest_35a_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "35A_provenance_discovery_manifest.json",
)
manifest_35a_sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "35A_provenance_discovery_manifest_SHA256.txt",
)

for path in [manifest_35a_path, manifest_35a_sha_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

with open(
    manifest_35a_sha_path,
    "r",
    encoding="utf-8",
) as fh:
    observed_35a_sha = fh.read().strip()

if observed_35a_sha != EXPECTED_35A_MANIFEST_SHA:
    raise RuntimeError(
        "35A manifest SHA mismatch: " + observed_35a_sha
    )

print("35A manifest SHA guard: PASS")

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def q(sql):
    return client.query(
        sql,
        location=BQ_LOCATION,
    ).to_dataframe()

def sha_text(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

def normalize_ref(ref):
    ref = str(ref).strip()
    if not ref:
        return None
    if ref.count(".") == 2:
        return ref
    if ref.count(".") == 1:
        return f"{PROJECT_ID}.{ref}"
    return f"{PROJECT_ID}.{DATASET_ID}.{ref}"

def backtick_refs(sql_text):
    if not isinstance(sql_text, str):
        return []
    refs = re.findall(r"`([^`]+)`", sql_text)
    out = []
    for ref in refs:
        # Keep only actual table/view-like references.
        if "." in ref or ref in set(table_meta["table_name"]):
            out.append(normalize_ref(ref))
    return sorted({
        x for x in out if x is not None
    })

def contexts(text, token, radius=2200, max_hits=20):
    if not isinstance(text, str):
        return []
    low = text.lower()
    tok = token.lower()
    out = []
    start = 0
    while len(out) < max_hits:
        idx = low.find(tok, start)
        if idx < 0:
            break
        a = max(0, idx - radius)
        b = min(
            len(text),
            idx + len(token) + radius,
        )
        out.append(text[a:b])
        start = idx + len(tok)
    return out

def destination_fqn(row):
    p = row.get("destination_project")
    d = row.get("destination_dataset")
    t = row.get("destination_table")
    if (
        isinstance(p, str)
        and isinstance(d, str)
        and isinstance(t, str)
        and p
        and d
        and t
    ):
        return f"{p}.{d}.{t}"
    return None

def construction_score(query_text, dest_fqn, lineage_names):
    low = query_text.lower()
    score = 0

    # Target feature mentions.
    for token in TARGET_FEATURES:
        if token.lower() in low:
            score += 12

    # High-value construction syntax.
    if re.search(
        r"\bcreate\s+(or\s+replace\s+)?(table|view)\b",
        low,
    ):
        score += 18

    if re.search(
        r"\bselect\b",
        low,
    ):
        score += 2

    # Destination or explicit lineage-object construction.
    if dest_fqn and dest_fqn in lineage_names:
        score += 30

    for name in lineage_names:
        short = name.split(".")[-1].lower()
        if short and short in low:
            score += 6

    # Time-window evidence.
    if "720" in low:
        score += 8
    if "4320" in low:
        score += 8
    if "offset" in low:
        score += 6

    # eICU source cues.
    for cue in [
        "eicu_crd.lab",
        "eicu_crd.patient",
        "labresultoffset",
        "observationoffset",
        "intakeoutputoffset",
        "treatmentoffset",
    ]:
        if cue in low:
            score += 5

    # Deprioritize obvious evaluation/model-only queries.
    for cue in [
        "roc_auc_score",
        "average_precision",
        "model_lr_outer_predictions",
        "model_xgb_outer_predictions",
        "model_random_forest_outer_predictions",
        "model_catboost_outer_predictions",
    ]:
        if cue in low:
            score -= 15

    return score

def comparator_hits(text):
    low = text.lower()
    hits = []

    regexes = {
        "predictor_le_720": [
            r"<=\s*720\b",
            r"<\s*721\b",
            r"between\s+0\s+and\s+720\b",
        ],
        "predictor_lt_720": [
            r"<\s*720\b",
        ],
        "post_landmark_gt_720": [
            r">\s*720\b",
            r">=\s*721\b",
        ],
        "outcome_le_4320": [
            r"<=\s*4320\b",
            r"<\s*4321\b",
        ],
        "outcome_between_720_4320": [
            r"between\s+721\s+and\s+4320\b",
            r"between\s+720\s+and\s+4320\b",
        ],
    }

    for label, patterns in regexes.items():
        for pattern in patterns:
            if re.search(pattern, low):
                hits.append(label)
                break

    # Identify named offset fields near temporal predicates.
    offset_fields = sorted(set(
        re.findall(
            r"\b[a-z_]*offset[a-z_]*\b",
            low,
        )
    ))

    return sorted(set(hits)), offset_fields

# ------------------------------------------------------------
# 2. Current metadata + recursive same-dataset lineage
# ------------------------------------------------------------

table_meta = q(f"""
SELECT
  table_name,
  table_type,
  creation_time,
  ddl
FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.TABLES`
ORDER BY table_name
""")

view_meta = q(f"""
SELECT
  table_name,
  view_definition
FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.VIEWS`
ORDER BY table_name
""")

table_type_map = {
    f"{PROJECT_ID}.{DATASET_ID}.{row['table_name']}":
        row["table_type"]
    for _, row in table_meta.iterrows()
}

view_def_map = {
    f"{PROJECT_ID}.{DATASET_ID}.{row['table_name']}":
        str(row["view_definition"])
    for _, row in view_meta.iterrows()
}

roots = [
    f"{PROJECT_ID}.{DATASET_ID}.feature_matrix_core_view_v1",
    f"{PROJECT_ID}.{DATASET_ID}.feature_matrix_extended_view_v1",
    f"{PROJECT_ID}.{DATASET_ID}.feature_matrix_core_outerfold_v1",
    f"{PROJECT_ID}.{DATASET_ID}.feature_matrix_extended_outerfold_v1",
]

lineage_rows = []
seen = set()
queue = deque([
    (root, 0, None)
    for root in roots
])

while queue:
    obj, depth, parent = queue.popleft()

    if (obj, parent) in seen:
        continue
    seen.add((obj, parent))

    obj_type = table_type_map.get(
        obj,
        "EXTERNAL_OR_UNKNOWN",
    )
    definition = view_def_map.get(obj)

    lineage_rows.append({
        "root_or_parent": parent,
        "object_fqn": obj,
        "object_name": obj.split(".")[-1],
        "depth": depth,
        "object_type": obj_type,
        "has_view_definition": definition is not None,
        "view_definition_sha256": (
            sha_text(definition)
            if definition is not None
            else None
        ),
    })

    if definition is not None and depth < 12:
        for ref in backtick_refs(definition):
            # Traverse only current-project references; keep external source
            # objects in the lineage output but do not recurse them.
            if ref.startswith(
                f"{PROJECT_ID}.{DATASET_ID}."
            ):
                queue.append(
                    (ref, depth + 1, obj)
                )
            else:
                lineage_rows.append({
                    "root_or_parent": obj,
                    "object_fqn": ref,
                    "object_name": ref.split(".")[-1],
                    "depth": depth + 1,
                    "object_type": "EXTERNAL_SOURCE",
                    "has_view_definition": False,
                    "view_definition_sha256": None,
                })

recursive_lineage = (
    pd.DataFrame(lineage_rows)
    .drop_duplicates()
    .sort_values(
        ["depth", "object_fqn"],
        kind="stable",
    )
    .reset_index(drop=True)
)

same_dataset_lineage_names = set(
    recursive_lineage.loc[
        recursive_lineage[
            "object_fqn"
        ].str.startswith(
            f"{PROJECT_ID}.{DATASET_ID}.",
            na=False,
        ),
        "object_fqn",
    ].tolist()
)

# ------------------------------------------------------------
# 3. Historical BigQuery query jobs
#    Retrieve full SQL, not snippets only.
# ------------------------------------------------------------

job_sql = f"""
SELECT
  creation_time,
  job_id,
  user_email,
  statement_type,
  destination_table.project_id AS destination_project,
  destination_table.dataset_id AS destination_dataset,
  destination_table.table_id AS destination_table,
  query
FROM `{PROJECT_ID}.region-us.INFORMATION_SCHEMA.JOBS_BY_PROJECT`
WHERE job_type = 'QUERY'
  AND query IS NOT NULL
  AND creation_time < TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 1 MINUTE)
  AND (
    REGEXP_CONTAINS(LOWER(query), r'x_reference_creatinine')
    OR REGEXP_CONTAINS(LOWER(query), r'x_stage1_at_landmark')
    OR REGEXP_CONTAINS(LOWER(query), r'x_lab_creatinine_last')
    OR REGEXP_CONTAINS(LOWER(query), r'label_stage23')
    OR REGEXP_CONTAINS(LOWER(query), r'feature_matrix')
    OR REGEXP_CONTAINS(LOWER(query), r'feature_labs')
    OR REGEXP_CONTAINS(LOWER(query), r'reference_creatinine')
  )
ORDER BY creation_time DESC
LIMIT 1500
"""

job_history = q(job_sql)

candidate_rows = []

for _, row in job_history.iterrows():
    query_text = str(row["query"])
    dest = destination_fqn(row)
    score = construction_score(
        query_text,
        dest,
        same_dataset_lineage_names,
    )

    mentioned_targets = [
        token
        for token in TARGET_FEATURES
        if token.lower() in query_text.lower()
    ]

    lineage_mentions = sorted([
        name
        for name in same_dataset_lineage_names
        if name.split(".")[-1].lower()
        in query_text.lower()
    ])

    candidate_rows.append({
        "creation_time": row["creation_time"],
        "job_id": row["job_id"],
        "statement_type": row["statement_type"],
        "destination_fqn": dest,
        "construction_score": score,
        "mentioned_target_count": len(mentioned_targets),
        "mentioned_targets": " | ".join(mentioned_targets),
        "lineage_object_mentions": " | ".join(
            x.split(".")[-1]
            for x in lineage_mentions
        ),
        "query_sha256": sha_text(query_text),
        "query_length": len(query_text),
        "contains_720": "720" in query_text,
        "contains_4320": "4320" in query_text,
        "contains_offset": "offset" in query_text.lower(),
        "contains_create_or_replace": bool(
            re.search(
                r"\bcreate\s+(or\s+replace\s+)?(table|view)\b",
                query_text,
                flags=re.I,
            )
        ),
    })

candidate_jobs = (
    pd.DataFrame(candidate_rows)
    .sort_values(
        [
            "construction_score",
            "mentioned_target_count",
            "creation_time",
        ],
        ascending=[False, False, False],
        kind="stable",
    )
    .reset_index(drop=True)
)

candidate_jobs["candidate_rank"] = (
    np.arange(len(candidate_jobs)) + 1
)

# ------------------------------------------------------------
# 4. Keep a focused set of top/high-value construction jobs
# ------------------------------------------------------------

high_value_mask = (
    candidate_jobs["construction_score"].ge(20)
    | candidate_jobs[
        "destination_fqn"
    ].isin(same_dataset_lineage_names)
)

high_value_jobs = candidate_jobs.loc[
    high_value_mask
].copy()

if high_value_jobs.empty:
    raise RuntimeError(
        "No plausible construction SQL was found in BigQuery job history."
    )

high_value_jobs = high_value_jobs.head(60).copy()

job_query_map = {
    str(row["job_id"]): str(row["query"])
    for _, row in job_history.iterrows()
}

# ------------------------------------------------------------
# 5. Exact target contexts and temporal predicate evidence
# ------------------------------------------------------------

context_rows = []
evidence_rows = []

for _, cand in high_value_jobs.iterrows():
    job_id = str(cand["job_id"])
    query_text = job_query_map[job_id]

    for token in TARGET_FEATURES:
        if token.lower() not in query_text.lower():
            continue

        token_contexts = contexts(
            query_text,
            token,
            radius=2600,
            max_hits=12,
        )

        for idx, snippet in enumerate(
            token_contexts,
            start=1,
        ):
            temporal_hits, offset_fields = comparator_hits(
                snippet
            )

            context_rows.append({
                "candidate_rank": int(
                    cand["candidate_rank"]
                ),
                "creation_time": cand["creation_time"],
                "job_id": job_id,
                "destination_fqn": cand["destination_fqn"],
                "construction_score": int(
                    cand["construction_score"]
                ),
                "target_feature": token,
                "context_index": idx,
                "temporal_predicate_hits": " | ".join(
                    temporal_hits
                ),
                "offset_fields": " | ".join(
                    offset_fields
                ),
                "snippet": snippet,
            })

            for hit in temporal_hits:
                evidence_rows.append({
                    "candidate_rank": int(
                        cand["candidate_rank"]
                    ),
                    "job_id": job_id,
                    "destination_fqn": cand["destination_fqn"],
                    "target_feature": token,
                    "evidence_type": hit,
                    "offset_fields": " | ".join(
                        offset_fields
                    ),
                    "snippet": snippet,
                })

target_contexts = pd.DataFrame(context_rows)
temporal_predicate_evidence = pd.DataFrame(
    evidence_rows
)

# ------------------------------------------------------------
# 6. Provisional evidence status
#    IMPORTANT: not the final leakage verdict.
# ------------------------------------------------------------

status_rows = []

for token in TARGET_FEATURES:
    if target_contexts.empty:
        sub = pd.DataFrame()
    else:
        sub = target_contexts.loc[
            target_contexts[
                "target_feature"
            ].eq(token)
        ].copy()

    all_hits = set()

    if not sub.empty:
        for cell in sub[
            "temporal_predicate_hits"
        ].fillna(""):
            for item in str(cell).split(" | "):
                if item:
                    all_hits.add(item)

    if token == "label_stage23":
        required = {
            "post_landmark_gt_720",
            "outcome_le_4320",
        }
    else:
        required = {
            "predictor_le_720",
        }

    missing = sorted(
        required - all_hits
    )

    if sub.empty:
        provisional = "NO_CONSTRUCTION_CONTEXT_FOUND"
    elif not missing:
        provisional = (
            "REQUIRED_TEMPORAL_PATTERNS_FOUND_MANUAL_SQL_REVIEW_REQUIRED"
        )
    else:
        provisional = (
            "INCOMPLETE_TEMPORAL_EVIDENCE_MANUAL_SQL_REVIEW_REQUIRED"
        )

    status_rows.append({
        "target_feature": token,
        "construction_contexts": len(sub),
        "evidence_types_found": " | ".join(
            sorted(all_hits)
        ),
        "required_evidence_types": " | ".join(
            sorted(required)
        ),
        "missing_required_evidence": " | ".join(
            missing
        ),
        "provisional_status": provisional,
    })

provisional_status = pd.DataFrame(
    status_rows
)

# ------------------------------------------------------------
# 7. Save top candidate full SQL for transparent audit
# ------------------------------------------------------------

sql_dir = os.path.join(
    MODEL_OUTPUT_DIR,
    "35B_candidate_construction_sql",
)
os.makedirs(
    sql_dir,
    exist_ok=True,
)

saved_sql_rows = []

for _, cand in high_value_jobs.head(20).iterrows():
    job_id = str(cand["job_id"])
    query_text = job_query_map[job_id]
    sha12 = str(
        cand["query_sha256"]
    )[:12]

    filename = (
        f"35B_rank{int(cand['candidate_rank']):02d}_"
        f"{sha12}.sql"
    )
    path = os.path.join(
        sql_dir,
        filename,
    )

    with open(
        path,
        "w",
        encoding="utf-8",
    ) as fh:
        fh.write(query_text)

    saved_sql_rows.append({
        "candidate_rank": int(
            cand["candidate_rank"]
        ),
        "job_id": job_id,
        "query_sha256": cand["query_sha256"],
        "destination_fqn": cand["destination_fqn"],
        "sql_file": path,
    })

saved_sql_files = pd.DataFrame(
    saved_sql_rows
)

# ------------------------------------------------------------
# 8. Drive outputs — metadata / SQL only, no patient records
# ------------------------------------------------------------

paths = {
    "recursive_lineage": os.path.join(
        MODEL_OUTPUT_DIR,
        "35B_recursive_sql_lineage.csv",
    ),
    "candidate_jobs": os.path.join(
        MODEL_OUTPUT_DIR,
        "35B_candidate_construction_jobs.csv",
    ),
    "target_contexts": os.path.join(
        MODEL_OUTPUT_DIR,
        "35B_target_feature_sql_contexts.csv",
    ),
    "temporal_evidence": os.path.join(
        MODEL_OUTPUT_DIR,
        "35B_temporal_predicate_evidence.csv",
    ),
    "provisional_status": os.path.join(
        MODEL_OUTPUT_DIR,
        "35B_provisional_temporal_status.csv",
    ),
    "saved_sql_files": os.path.join(
        MODEL_OUTPUT_DIR,
        "35B_saved_candidate_sql_files.csv",
    ),
    "manifest": os.path.join(
        MODEL_OUTPUT_DIR,
        "35B_targeted_sql_lineage_audit_manifest.json",
    ),
    "manifest_sha": os.path.join(
        MODEL_OUTPUT_DIR,
        "35B_targeted_sql_lineage_audit_manifest_SHA256.txt",
    ),
}

recursive_lineage.to_csv(
    paths["recursive_lineage"],
    index=False,
)
candidate_jobs.to_csv(
    paths["candidate_jobs"],
    index=False,
)
target_contexts.to_csv(
    paths["target_contexts"],
    index=False,
)
temporal_predicate_evidence.to_csv(
    paths["temporal_evidence"],
    index=False,
)
provisional_status.to_csv(
    paths["provisional_status"],
    index=False,
)
saved_sql_files.to_csv(
    paths["saved_sql_files"],
    index=False,
)

manifest = {
    "analysis_version": "35B",
    "analysis_type": "targeted_kidney_temporal_sql_lineage_audit",
    "project_id": PROJECT_ID,
    "dataset_id": DATASET_ID,
    "bigquery_location": BQ_LOCATION,
    "landmark_minutes": LANDMARK_MIN,
    "outcome_end_minutes": OUTCOME_END_MIN,
    "target_features": TARGET_FEATURES,
    "source_35a_manifest_sha256": EXPECTED_35A_MANIFEST_SHA,
    "historical_query_jobs_scanned": int(
        len(job_history)
    ),
    "candidate_construction_jobs": int(
        len(candidate_jobs)
    ),
    "high_value_construction_jobs_reviewed": int(
        len(high_value_jobs)
    ),
    "top_full_sql_files_saved": int(
        len(saved_sql_files)
    ),
    "model_fit_performed": False,
    "retuning_performed": False,
    "recalibration_performed": False,
    "bigquery_dml_used": False,
    "patient_level_data_written_to_drive": False,
    "final_leakage_pass_declared": False,
    "requires_manual_scientific_sql_adjudication": True,
    "output_files": paths,
    "candidate_sql_directory": sql_dir,
}

manifest_text = json.dumps(
    manifest,
    indent=2,
    sort_keys=True,
)

with open(
    paths["manifest"],
    "w",
    encoding="utf-8",
) as fh:
    fh.write(manifest_text)

manifest_sha = sha_text(
    manifest_text
)

with open(
    paths["manifest_sha"],
    "w",
    encoding="utf-8",
) as fh:
    fh.write(
        manifest_sha + "\n"
    )

# ------------------------------------------------------------
# 9. Compact display for the next audit decision
# ------------------------------------------------------------

print("\n35B RECURSIVE LINEAGE — FIRST 60")
display(
    recursive_lineage.head(60)
)

print("\n35B TOP CONSTRUCTION JOB CANDIDATES")
display(
    high_value_jobs[
        [
            "candidate_rank",
            "creation_time",
            "statement_type",
            "destination_fqn",
            "construction_score",
            "mentioned_targets",
            "lineage_object_mentions",
            "contains_720",
            "contains_4320",
            "contains_offset",
            "query_sha256",
        ]
    ].head(30)
)

print("\n35B TARGET TEMPORAL STATUS")
display(
    provisional_status
)

print("\n35B TEMPORAL PREDICATE EVIDENCE — FIRST 40")
display(
    temporal_predicate_evidence.head(40)
)

print("\n35B TARGET SQL CONTEXTS — FIRST 20")
display(
    target_contexts[
        [
            "candidate_rank",
            "creation_time",
            "job_id",
            "destination_fqn",
            "target_feature",
            "temporal_predicate_hits",
            "offset_fields",
            "snippet",
        ]
    ].head(20)
)

print("\n35B manifest SHA-256:")
print(manifest_sha)

print("\nSaved candidate full SQL directory:")
print(sql_dir)

print(
    "\n35B COMPLETE: targeted construction SQL was recovered and ranked."
)
print(
    "IMPORTANT: Do not declare leakage PASS from regex evidence alone."
)
print(
    "Send the complete 35B output next; the exact SQL contexts will be "
    "scientifically adjudicated before any further model analysis."
)

In [ ]:
import os
import re
import json
import hashlib
import pandas as pd
from IPython.display import display

print("STARTING EXACT CONSTRUCTION-SQL TEMPORAL AUDIT — CODE VERSION 35C")

# ============================================================
# 35C — EXACT SQL ADJUDICATION FOR CRITICAL AKI VARIABLES
#
# This audit directly recovers the CTAS / construction SQL for:
#   feature_static_v1
#   feature_labs_v1
#   creatinine_staged_v1
#   cohort_outcome_v1
#
# It prints:
#   1) the exact relevant SQL lines with line numbers,
#   2) temporal predicates around 720 / 4320,
#   3) contexts around the critical feature aliases,
#   4) source tables referenced by each construction query.
#
# READ ONLY:
# - no model fitting
# - no retuning/recalibration
# - no BigQuery DML
# - no patient-level Drive export
# ============================================================

EXPECTED_35B_MANIFEST_SHA = (
    "ebd05e182c0030c1375ddd82821b1c09"
    "ad6fd2c326a9740cc321e1f79a4c4db3"
)

if "client" not in globals():
    raise RuntimeError("Önce BigQuery client'ını oluşturan temel hücreyi çalıştır.")

if "MODEL_OUTPUT_DIR" not in globals():
    raise RuntimeError("Önce MODEL_OUTPUT_DIR tanımlı temel hücreyi çalıştır.")

PROJECT_ID = client.project
DATASET_ID = "aki_jcmc_v2"
BQ_LOCATION = "US"

if "TARGET_DATASET" in globals():
    parts = str(TARGET_DATASET).split(".")
    if len(parts) == 2:
        PROJECT_ID, DATASET_ID = parts

# ------------------------------------------------------------
# 1. Verify 35B lock
# ------------------------------------------------------------

manifest_sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "35B_targeted_sql_lineage_audit_manifest_SHA256.txt",
)

if not os.path.exists(manifest_sha_path):
    raise FileNotFoundError(manifest_sha_path)

with open(manifest_sha_path, "r", encoding="utf-8") as fh:
    observed_sha = fh.read().strip()

if observed_sha != EXPECTED_35B_MANIFEST_SHA:
    raise RuntimeError(
        "35B manifest SHA mismatch: " + observed_sha
    )

print("35B manifest SHA guard: PASS")

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def bq(sql):
    return client.query(
        sql,
        location=BQ_LOCATION,
    ).to_dataframe()

def sha_text(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

def numbered_lines(text):
    return [
        (i + 1, line)
        for i, line in enumerate(text.splitlines())
    ]

def context_by_line(text, patterns, radius=8):
    lines = text.splitlines()
    hit_indices = set()

    compiled = [
        re.compile(p, re.I)
        for p in patterns
    ]

    for idx, line in enumerate(lines):
        if any(rx.search(line) for rx in compiled):
            for j in range(
                max(0, idx - radius),
                min(len(lines), idx + radius + 1),
            ):
                hit_indices.add(j)

    rows = []
    for j in sorted(hit_indices):
        rows.append({
            "line_no": j + 1,
            "sql_line": lines[j],
        })
    return pd.DataFrame(rows)

def source_refs(text):
    return sorted(set(
        re.findall(r"`([^`]+)`", text)
    ))

# ------------------------------------------------------------
# 2. Recover exact construction jobs by destination table
# ------------------------------------------------------------

TARGET_TABLES = [
    "feature_static_v1",
    "feature_labs_v1",
    "creatinine_staged_v1",
    "cohort_outcome_v1",
]

destination_list = ", ".join(
    [f"'{x}'" for x in TARGET_TABLES]
)

jobs_sql = f"""
SELECT
  creation_time,
  job_id,
  statement_type,
  destination_table.project_id AS destination_project,
  destination_table.dataset_id AS destination_dataset,
  destination_table.table_id AS destination_table,
  query
FROM `{PROJECT_ID}.region-us.INFORMATION_SCHEMA.JOBS_BY_PROJECT`
WHERE job_type = 'QUERY'
  AND query IS NOT NULL
  AND destination_table.project_id = '{PROJECT_ID}'
  AND destination_table.dataset_id = '{DATASET_ID}'
  AND destination_table.table_id IN ({destination_list})
  AND statement_type IN (
    'CREATE_TABLE_AS_SELECT',
    'CREATE_TABLE',
    'CREATE_VIEW'
  )
ORDER BY destination_table, creation_time ASC
"""

jobs = bq(jobs_sql)

if jobs.empty:
    raise RuntimeError("Critical construction jobs could not be recovered.")

# Pick the earliest successful construction query per exact destination.
# This reconstructs the original feature/cohort build provenance rather than
# later inspection SELECTs.
selected_rows = []

for table_name in TARGET_TABLES:
    sub = jobs.loc[
        jobs["destination_table"].eq(table_name)
    ].sort_values(
        "creation_time",
        kind="stable",
    )

    if sub.empty:
        raise RuntimeError(
            f"No construction job found for {table_name}."
        )

    row = sub.iloc[0].copy()
    selected_rows.append(row)

selected_jobs = pd.DataFrame(selected_rows)

# ------------------------------------------------------------
# 3. Save exact SQL + produce focused evidence
# ------------------------------------------------------------

sql_dir = os.path.join(
    MODEL_OUTPUT_DIR,
    "35C_exact_construction_sql",
)
os.makedirs(sql_dir, exist_ok=True)

job_summary_rows = []
source_ref_rows = []
evidence_frames = []
feature_context_frames = []

TARGET_PATTERNS = {
    "feature_static_v1": [
        r"x_reference_creatinine",
        r"x_stage1_at_landmark",
        r"reference_offset",
        r"audit_reference_offset",
        r"720",
        r"offset",
        r"creatinine",
        r"stage",
    ],
    "feature_labs_v1": [
        r"x_lab_creatinine_last",
        r"offset_min",
        r"labresultoffset",
        r"\b720\b",
        r"creatinine",
        r"row_number",
        r"order by",
        r"where",
    ],
    "creatinine_staged_v1": [
        r"\b720\b",
        r"\b4320\b",
        r"offset",
        r"creatinine",
        r"stage",
        r"kdigo",
        r"reference",
        r"where",
    ],
    "cohort_outcome_v1": [
        r"\b720\b",
        r"\b4320\b",
        r"label_stage23",
        r"stage23",
        r"offset",
        r"stage",
        r"where",
        r"between",
    ],
}

CRITICAL_ALIAS_PATTERNS = [
    r"x_reference_creatinine",
    r"x_stage1_at_landmark",
    r"x_lab_creatinine_last",
    r"label_stage23",
]

for _, row in selected_jobs.iterrows():
    table_name = str(row["destination_table"])
    query_text = str(row["query"])
    query_sha = sha_text(query_text)

    sql_path = os.path.join(
        sql_dir,
        f"35C_{table_name}_{query_sha[:12]}.sql",
    )
    with open(sql_path, "w", encoding="utf-8") as fh:
        fh.write(query_text)

    refs = source_refs(query_text)
    for ref in refs:
        source_ref_rows.append({
            "destination_table": table_name,
            "source_reference": ref,
        })

    job_summary_rows.append({
        "destination_table": table_name,
        "creation_time": row["creation_time"],
        "job_id": row["job_id"],
        "statement_type": row["statement_type"],
        "query_sha256": query_sha,
        "query_length": len(query_text),
        "contains_720": "720" in query_text,
        "contains_4320": "4320" in query_text,
        "contains_offset": "offset" in query_text.lower(),
        "sql_file": sql_path,
    })

    evidence = context_by_line(
        query_text,
        TARGET_PATTERNS[table_name],
        radius=8,
    )
    evidence.insert(
        0,
        "destination_table",
        table_name,
    )
    evidence_frames.append(evidence)

    alias_context = context_by_line(
        query_text,
        CRITICAL_ALIAS_PATTERNS,
        radius=12,
    )
    alias_context.insert(
        0,
        "destination_table",
        table_name,
    )
    feature_context_frames.append(alias_context)

job_summary = pd.DataFrame(job_summary_rows)
source_refs_df = pd.DataFrame(source_ref_rows).drop_duplicates()

temporal_sql_evidence = pd.concat(
    evidence_frames,
    ignore_index=True,
)

critical_alias_context = pd.concat(
    feature_context_frames,
    ignore_index=True,
)

# ------------------------------------------------------------
# 4. Machine-readable predicate inventory
# ------------------------------------------------------------

predicate_rows = []

PREDICATE_REGEXES = {
    "le_720": r"(?:<=\s*720\b|<\s*721\b|between\s+\S+\s+and\s+720\b)",
    "lt_720": r"<\s*720\b",
    "gt_720": r">\s*720\b",
    "ge_721": r">=\s*721\b",
    "le_4320": r"<=\s*4320\b",
    "lt_4321": r"<\s*4321\b",
    "between_721_4320": r"between\s+721\s+and\s+4320\b",
    "between_720_4320": r"between\s+720\s+and\s+4320\b",
}

for _, row in selected_jobs.iterrows():
    table_name = str(row["destination_table"])
    query_text = str(row["query"])
    low = query_text.lower()

    for label, pattern in PREDICATE_REGEXES.items():
        matches = list(
            re.finditer(pattern, low, flags=re.I)
        )
        predicate_rows.append({
            "destination_table": table_name,
            "predicate_type": label,
            "match_count": len(matches),
        })

predicate_inventory = pd.DataFrame(predicate_rows)

# ------------------------------------------------------------
# 5. Explicit scientific checklist
# ------------------------------------------------------------

def has_any(table_name, predicate_types):
    sub = predicate_inventory.loc[
        predicate_inventory[
            "destination_table"
        ].eq(table_name)
        & predicate_inventory[
            "predicate_type"
        ].isin(predicate_types)
    ]
    return bool(
        (sub["match_count"] > 0).any()
    )

checklist = pd.DataFrame([
    {
        "critical_item": "x_lab_creatinine_last",
        "construction_table": "feature_labs_v1",
        "required_temporal_rule": "lab data restricted to <=720 min",
        "automatic_pattern_support": has_any(
            "feature_labs_v1",
            ["le_720", "lt_720"],
        ),
        "final_status": "MANUAL_SQL_REVIEW_REQUIRED",
    },
    {
        "critical_item": "x_reference_creatinine",
        "construction_table": "feature_static_v1 + upstream creatinine_staged_v1",
        "required_temporal_rule": "reference creatinine must not use >720 min outcome-window information",
        "automatic_pattern_support": (
            has_any(
                "feature_static_v1",
                ["le_720", "lt_720"],
            )
            or has_any(
                "creatinine_staged_v1",
                ["le_720", "lt_720"],
            )
        ),
        "final_status": "MANUAL_SQL_REVIEW_REQUIRED",
    },
    {
        "critical_item": "x_stage1_at_landmark",
        "construction_table": "feature_static_v1 + upstream creatinine_staged_v1",
        "required_temporal_rule": "landmark stage uses only <=720 min information",
        "automatic_pattern_support": (
            has_any(
                "feature_static_v1",
                ["le_720", "lt_720"],
            )
            or has_any(
                "creatinine_staged_v1",
                ["le_720", "lt_720"],
            )
        ),
        "final_status": "MANUAL_SQL_REVIEW_REQUIRED",
    },
    {
        "critical_item": "label_stage23",
        "construction_table": "cohort_outcome_v1",
        "required_temporal_rule": "outcome strictly >720 and <=4320 min",
        "automatic_pattern_support": (
            has_any(
                "cohort_outcome_v1",
                ["gt_720", "ge_721", "between_721_4320"],
            )
            and has_any(
                "cohort_outcome_v1",
                ["le_4320", "lt_4321", "between_721_4320", "between_720_4320"],
            )
        ),
        "final_status": "MANUAL_SQL_REVIEW_REQUIRED",
    },
])

# ------------------------------------------------------------
# 6. Save audit outputs
# ------------------------------------------------------------

paths = {
    "job_summary": os.path.join(
        MODEL_OUTPUT_DIR,
        "35C_exact_construction_job_summary.csv",
    ),
    "source_refs": os.path.join(
        MODEL_OUTPUT_DIR,
        "35C_exact_construction_source_references.csv",
    ),
    "temporal_sql_evidence": os.path.join(
        MODEL_OUTPUT_DIR,
        "35C_temporal_sql_evidence_with_line_numbers.csv",
    ),
    "critical_alias_context": os.path.join(
        MODEL_OUTPUT_DIR,
        "35C_critical_alias_sql_context.csv",
    ),
    "predicate_inventory": os.path.join(
        MODEL_OUTPUT_DIR,
        "35C_temporal_predicate_inventory.csv",
    ),
    "checklist": os.path.join(
        MODEL_OUTPUT_DIR,
        "35C_critical_temporal_checklist.csv",
    ),
    "manifest": os.path.join(
        MODEL_OUTPUT_DIR,
        "35C_exact_construction_sql_audit_manifest.json",
    ),
    "manifest_sha": os.path.join(
        MODEL_OUTPUT_DIR,
        "35C_exact_construction_sql_audit_manifest_SHA256.txt",
    ),
}

job_summary.to_csv(
    paths["job_summary"],
    index=False,
)
source_refs_df.to_csv(
    paths["source_refs"],
    index=False,
)
temporal_sql_evidence.to_csv(
    paths["temporal_sql_evidence"],
    index=False,
)
critical_alias_context.to_csv(
    paths["critical_alias_context"],
    index=False,
)
predicate_inventory.to_csv(
    paths["predicate_inventory"],
    index=False,
)
checklist.to_csv(
    paths["checklist"],
    index=False,
)

manifest = {
    "analysis_version": "35C",
    "analysis_type": "exact_construction_sql_temporal_audit",
    "source_35b_manifest_sha256": EXPECTED_35B_MANIFEST_SHA,
    "critical_tables": TARGET_TABLES,
    "landmark_minutes": 720,
    "outcome_end_minutes": 4320,
    "read_only": True,
    "bigquery_dml_used": False,
    "model_fit_performed": False,
    "retuning_performed": False,
    "recalibration_performed": False,
    "patient_level_data_written_to_drive": False,
    "final_leakage_pass_declared": False,
    "requires_manual_scientific_adjudication": True,
    "outputs": paths,
    "sql_directory": sql_dir,
}

manifest_text = json.dumps(
    manifest,
    indent=2,
    sort_keys=True,
)

with open(
    paths["manifest"],
    "w",
    encoding="utf-8",
) as fh:
    fh.write(manifest_text)

manifest_sha = sha_text(manifest_text)

with open(
    paths["manifest_sha"],
    "w",
    encoding="utf-8",
) as fh:
    fh.write(manifest_sha + "\n")

# ------------------------------------------------------------
# 7. Display exactly what is needed for adjudication
# ------------------------------------------------------------

print("\n35C EXACT CONSTRUCTION JOB SUMMARY")
display(job_summary)

print("\n35C CRITICAL TEMPORAL CHECKLIST")
display(checklist)

print("\n35C TEMPORAL PREDICATE INVENTORY")
display(predicate_inventory)

for table_name in TARGET_TABLES:
    print(
        f"\n================ {table_name} ================\n"
    )

    sub = temporal_sql_evidence.loc[
        temporal_sql_evidence[
            "destination_table"
        ].eq(table_name)
    ]

    display(sub)

print("\n35C CRITICAL FEATURE ALIAS CONTEXT")
display(critical_alias_context)

print("\n35C manifest SHA-256:")
print(manifest_sha)

print("\nExact SQL files saved under:")
print(sql_dir)

print(
    "\n35C COMPLETE: exact construction SQL and temporal predicates recovered."
)
print(
    "Do NOT declare leakage PASS automatically."
)
print(
    "Send the complete 35C output; the four critical temporal rules "
    "will then be adjudicated explicitly."
)

In [ ]:
import os
import re
import json
import hashlib
import pandas as pd

print("STARTING CRITICAL AKI TEMPORAL INVARIANCE AUDIT — CODE VERSION 35D")

# ============================================================
# 35D — CRITICAL AKI TEMPORAL INVARIANCE AUDIT
#
# Goal:
#   Close the leakage audit with two independent evidence layers:
#
#   A) exact, untruncated construction SQL lines
#   B) aggregate empirical invariance checks in BigQuery
#
# Critical rules:
#   1. x_lab_creatinine_last: source labs <= 720 min
#   2. x_reference_creatinine: selected reference offset <= 720 min
#      and its construction must not select a post-landmark measurement
#   3. x_stage1_at_landmark: reconstructed only from creatinine stages
#      available at/before 720 min
#   4. label_stage23: stage 2/3 only when offset >720 and <=4320 min
#
# READ ONLY:
#   - no model fitting
#   - no tuning/recalibration
#   - no BigQuery DML
#   - no patient-level output to Drive
# ============================================================

EXPECTED_35C_SHA = (
    "6757330aaf0f3e7e642ee523fbe7d07c"
    "69f1d56803b7ad5b018ac30f7fba91bc"
)

if "client" not in globals():
    raise RuntimeError(
        "Önce BigQuery client'ını oluşturan temel hücreyi çalıştır."
    )

if "MODEL_OUTPUT_DIR" not in globals():
    raise RuntimeError(
        "Önce MODEL_OUTPUT_DIR tanımlı temel hücreyi çalıştır."
    )

PROJECT_ID = client.project
DATASET_ID = "aki_jcmc_v2"
BQ_LOCATION = "US"

if "TARGET_DATASET" in globals():
    parts = str(TARGET_DATASET).split(".")
    if len(parts) == 2:
        PROJECT_ID, DATASET_ID = parts

# ------------------------------------------------------------
# 1. Verify 35C lock
# ------------------------------------------------------------

sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "35C_exact_construction_sql_audit_manifest_SHA256.txt",
)

if not os.path.exists(sha_path):
    raise FileNotFoundError(sha_path)

with open(sha_path, "r", encoding="utf-8") as fh:
    observed_35c_sha = fh.read().strip()

if observed_35c_sha != EXPECTED_35C_SHA:
    raise RuntimeError(
        "35C manifest SHA mismatch: " + observed_35c_sha
    )

print("35C manifest SHA guard: PASS")

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def bq(sql):
    return client.query(
        sql,
        location=BQ_LOCATION,
    ).to_dataframe()

def sha_text(text):
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

def fetch_job_query(job_id):
    sql = f"""
    SELECT
      creation_time,
      job_id,
      statement_type,
      query
    FROM `{PROJECT_ID}.region-us.INFORMATION_SCHEMA.JOBS_BY_PROJECT`
    WHERE job_id = '{job_id}'
      AND query IS NOT NULL
    LIMIT 1
    """
    df = bq(sql)
    if len(df) != 1:
        raise RuntimeError(
            f"Could not recover exact SQL for job {job_id}."
        )
    return str(df.iloc[0]["query"])

def fetch_original_construction_sql(table_name):
    sql = f"""
    SELECT
      creation_time,
      job_id,
      statement_type,
      query
    FROM `{PROJECT_ID}.region-us.INFORMATION_SCHEMA.JOBS_BY_PROJECT`
    WHERE job_type = 'QUERY'
      AND query IS NOT NULL
      AND destination_table.project_id = '{PROJECT_ID}'
      AND destination_table.dataset_id = '{DATASET_ID}'
      AND destination_table.table_id = '{table_name}'
      AND statement_type IN (
        'CREATE_TABLE_AS_SELECT',
        'CREATE_TABLE',
        'CREATE_VIEW'
      )
    ORDER BY creation_time ASC
    LIMIT 1
    """
    df = bq(sql)
    if df.empty:
        return None
    return {
        "creation_time": df.iloc[0]["creation_time"],
        "job_id": str(df.iloc[0]["job_id"]),
        "statement_type": str(df.iloc[0]["statement_type"]),
        "query": str(df.iloc[0]["query"]),
    }

def print_focus_lines(label, sql_text, patterns, radius=3):
    lines = sql_text.splitlines()
    regexes = [
        re.compile(p, re.I)
        for p in patterns
    ]
    selected = set()

    for i, line in enumerate(lines):
        if any(rx.search(line) for rx in regexes):
            for j in range(
                max(0, i - radius),
                min(len(lines), i + radius + 1),
            ):
                selected.add(j)

    print(
        "\n" + "=" * 78
        + f"\n{label} — EXACT UNTRUNCATED SQL LINES\n"
        + "=" * 78
    )

    previous = None
    for j in sorted(selected):
        if previous is not None and j > previous + 1:
            print("    ...")
        print(f"{j + 1:04d}: {lines[j]}")
        previous = j

def table_columns(table_name):
    sql = f"""
    SELECT column_name, data_type
    FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.COLUMNS`
    WHERE table_name = '{table_name}'
    ORDER BY ordinal_position
    """
    return bq(sql)

# ------------------------------------------------------------
# 2. Recover exact SQL for the four 35C jobs
# ------------------------------------------------------------

KNOWN_JOBS = {
    "feature_static_v1":
        "46fd138b-90d5-4651-b661-cf1f20bd3b58",
    "feature_labs_v1":
        "a3374266-5e8d-4c8f-8f2e-b7b6bde32b87",
    "creatinine_staged_v1":
        "38d5a6d9-ad7e-4f3f-84ad-7bbb8c1d9ac1",
    "cohort_outcome_v1":
        "c511307d-6ef3-4587-afdb-e6d2e3b3832a",
}

exact_sql = {
    table: fetch_job_query(job_id)
    for table, job_id in KNOWN_JOBS.items()
}

# Recover two important upstream builders as well.
for upstream in [
    "base_stays_v1",
    "creatinine_clean_v1",
]:
    recovered = fetch_original_construction_sql(
        upstream
    )
    if recovered is not None:
        exact_sql[upstream] = recovered["query"]

# ------------------------------------------------------------
# 3. Print exact SQL lines WITHOUT pandas truncation
# ------------------------------------------------------------

print_focus_lines(
    "feature_labs_v1",
    exact_sql["feature_labs_v1"],
    [
        r"720",
        r"labResultOffset",
        r"offset_min",
        r"x_lab_creatinine_last",
        r"feature\s*=\s*'creatinine'",
        r"ROW_NUMBER",
        r"ORDER BY",
        r"WHERE",
    ],
    radius=4,
)

print_focus_lines(
    "feature_static_v1",
    exact_sql["feature_static_v1"],
    [
        r"x_reference_creatinine",
        r"x_stage1_at_landmark",
        r"reference_offset",
        r"cohort_outcome_v1",
    ],
    radius=5,
)

print_focus_lines(
    "creatinine_staged_v1",
    exact_sql["creatinine_staged_v1"],
    [
        r"reference",
        r"ref_offset",
        r"ref_creatinine",
        r"labResultOffset",
        r"base_stays_v1",
        r"BETWEEN",
        r"RANGE BETWEEN",
        r"kdigo_creatinine_stage",
    ],
    radius=5,
)

print_focus_lines(
    "cohort_outcome_v1",
    exact_sql["cohort_outcome_v1"],
    [
        r"720",
        r"4320",
        r"stage1_at_prediction",
        r"outcome_creatinine_stage23",
        r"reference_creatinine",
        r"reference_offset",
        r"kdigo_creatinine_stage",
        r"prediction",
        r"outcome",
    ],
    radius=5,
)

if "base_stays_v1" in exact_sql:
    print_focus_lines(
        "base_stays_v1",
        exact_sql["base_stays_v1"],
        [
            r"720",
            r"prediction",
            r"landmark",
            r"horizon",
            r"offset",
        ],
        radius=4,
    )

# ------------------------------------------------------------
# 4. Exact source-SQL text checks
# ------------------------------------------------------------

labs_sql_low = exact_sql[
    "feature_labs_v1"
].lower()

labs_has_720_bound = bool(
    re.search(
        r"(?:labresultoffset|offset_min|offset)"
        r"[\s\S]{0,160}"
        r"(?:<=\s*720\b|between[\s\S]{0,80}\b720\b)",
        labs_sql_low,
        flags=re.I,
    )
    or re.search(
        r"(?:<=\s*720\b|between[\s\S]{0,80}\b720\b)"
        r"[\s\S]{0,160}"
        r"(?:labresultoffset|offset_min|offset)",
        labs_sql_low,
        flags=re.I,
    )
)

# ------------------------------------------------------------
# 5. Aggregate empirical audit of reference offsets
# ------------------------------------------------------------

cohort_cols = set(
    table_columns(
        "cohort_outcome_v1"
    )["column_name"].astype(str)
)

required_cohort_cols = {
    "patientUnitStayID",
    "reference_offset",
    "reference_creatinine",
    "stage1_at_prediction",
    "outcome_creatinine_stage23",
}

missing_cols = sorted(
    required_cohort_cols - cohort_cols
)

if missing_cols:
    raise RuntimeError(
        "cohort_outcome_v1 missing expected audit columns: "
        + ", ".join(missing_cols)
    )

reference_offset_audit = bq(f"""
SELECT
  COUNT(*) AS rows_total,
  COUNTIF(reference_offset IS NULL) AS reference_offset_missing,
  MIN(reference_offset) AS min_reference_offset,
  MAX(reference_offset) AS max_reference_offset,
  COUNTIF(reference_offset > 720) AS reference_offset_gt_720,
  COUNTIF(reference_offset = 720) AS reference_offset_eq_720
FROM `{PROJECT_ID}.{DATASET_ID}.cohort_outcome_v1`
WHERE eligible_main_cohort = 1
  AND deterministic_eligible_patient_stay_rank = 1
""")

# ------------------------------------------------------------
# 6. Stage-1 landmark empirical reconstruction
# ------------------------------------------------------------

stage_cols = set(
    table_columns(
        "creatinine_staged_v1"
    )["column_name"].astype(str)
)

for col in [
    "patientUnitStayID",
    "labResultOffset",
    "kdigo_creatinine_stage",
]:
    if col not in stage_cols:
        raise RuntimeError(
            f"creatinine_staged_v1 missing expected column {col}."
        )

stage1_reconstruction = bq(f"""
WITH pre AS (
  SELECT
    patientUnitStayID,

    MAX(
      IF(
        labResultOffset <= 720,
        kdigo_creatinine_stage,
        NULL
      )
    ) AS max_stage_le_720,

    ARRAY_AGG(
      IF(
        labResultOffset <= 720,
        STRUCT(
          labResultOffset AS offset_min,
          kdigo_creatinine_stage AS stage
        ),
        NULL
      )
      IGNORE NULLS
      ORDER BY labResultOffset DESC
      LIMIT 1
    )[SAFE_OFFSET(0)] AS last_stage_le_720

  FROM `{PROJECT_ID}.{DATASET_ID}.creatinine_staged_v1`
  GROUP BY patientUnitStayID
),
audit AS (
  SELECT
    c.patientUnitStayID,
    c.stage1_at_prediction,

    IF(
      p.max_stage_le_720 = 1,
      1,
      0
    ) AS expected_stage1_from_max_severity,

    IF(
      p.last_stage_le_720.stage = 1,
      1,
      0
    ) AS expected_stage1_from_last_stage,

    p.max_stage_le_720,
    p.last_stage_le_720.offset_min AS last_stage_offset_le_720

  FROM `{PROJECT_ID}.{DATASET_ID}.cohort_outcome_v1` c
  LEFT JOIN pre p
    USING(patientUnitStayID)
  WHERE c.eligible_main_cohort = 1
    AND c.deterministic_eligible_patient_stay_rank = 1
)
SELECT
  COUNT(*) AS patients,
  COUNTIF(stage1_at_prediction = 1) AS stored_stage1_positive,

  COUNTIF(
    stage1_at_prediction
    != expected_stage1_from_max_severity
  ) AS mismatch_vs_max_severity_definition,

  COUNTIF(
    stage1_at_prediction
    != expected_stage1_from_last_stage
  ) AS mismatch_vs_last_stage_definition,

  MAX(last_stage_offset_le_720) AS maximum_last_stage_offset_used,

  COUNTIF(
    max_stage_le_720 >= 2
  ) AS severe_stage_present_by_landmark
FROM audit
""")

# ------------------------------------------------------------
# 7. Outcome empirical reconstruction (>720 and <=4320)
# ------------------------------------------------------------

outcome_reconstruction = bq(f"""
WITH derived AS (
  SELECT
    patientUnitStayID,

    MAX(
      IF(
        labResultOffset > 720
        AND labResultOffset <= 4320
        AND kdigo_creatinine_stage >= 2,
        1,
        0
      )
    ) AS expected_outcome_stage23,

    MIN(
      IF(
        labResultOffset > 720
        AND labResultOffset <= 4320
        AND kdigo_creatinine_stage >= 2,
        labResultOffset,
        NULL
      )
    ) AS first_stage23_offset_in_window,

    MAX(
      IF(
        labResultOffset > 4320
        AND kdigo_creatinine_stage >= 2,
        1,
        0
      )
    ) AS has_stage23_only_after_72h_candidate

  FROM `{PROJECT_ID}.{DATASET_ID}.creatinine_staged_v1`
  GROUP BY patientUnitStayID
),
audit AS (
  SELECT
    c.patientUnitStayID,
    c.outcome_creatinine_stage23,
    COALESCE(d.expected_outcome_stage23, 0)
      AS expected_outcome_stage23,
    d.first_stage23_offset_in_window,
    COALESCE(
      d.has_stage23_only_after_72h_candidate,
      0
    ) AS has_stage23_after_72h
  FROM `{PROJECT_ID}.{DATASET_ID}.cohort_outcome_v1` c
  LEFT JOIN derived d
    USING(patientUnitStayID)
  WHERE c.eligible_main_cohort = 1
    AND c.deterministic_eligible_patient_stay_rank = 1
)
SELECT
  COUNT(*) AS patients,
  COUNTIF(outcome_creatinine_stage23 = 1)
    AS stored_outcome_events,
  COUNTIF(expected_outcome_stage23 = 1)
    AS reconstructed_outcome_events,
  COUNTIF(
    outcome_creatinine_stage23
    != expected_outcome_stage23
  ) AS outcome_window_mismatches,
  MIN(first_stage23_offset_in_window)
    AS minimum_positive_outcome_offset,
  MAX(first_stage23_offset_in_window)
    AS maximum_first_positive_outcome_offset,
  COUNTIF(
    outcome_creatinine_stage23 = 0
    AND has_stage23_after_72h = 1
  ) AS negatives_with_stage23_after_72h
FROM audit
""")

# ------------------------------------------------------------
# 8. Compact adjudication table
# ------------------------------------------------------------

ref_row = reference_offset_audit.iloc[0]
stage_row = stage1_reconstruction.iloc[0]
out_row = outcome_reconstruction.iloc[0]

reference_pass = (
    int(ref_row["reference_offset_gt_720"]) == 0
    and float(ref_row["max_reference_offset"]) <= 720
)

stage1_empirical_pass = (
    int(stage_row[
        "mismatch_vs_max_severity_definition"
    ]) == 0
    or int(stage_row[
        "mismatch_vs_last_stage_definition"
    ]) == 0
)

outcome_pass = (
    int(out_row["outcome_window_mismatches"]) == 0
    and (
        pd.isna(out_row["minimum_positive_outcome_offset"])
        or float(out_row["minimum_positive_outcome_offset"]) > 720
    )
    and (
        pd.isna(out_row["maximum_first_positive_outcome_offset"])
        or float(out_row["maximum_first_positive_outcome_offset"]) <= 4320
    )
)

adjudication = pd.DataFrame([
    {
        "critical_item": "x_lab_creatinine_last",
        "sql_window_evidence": labs_has_720_bound,
        "empirical_invariance_evidence": None,
        "provisional_decision": (
            "PASS_CANDIDATE"
            if labs_has_720_bound
            else "NEEDS_REVIEW"
        ),
    },
    {
        "critical_item": "x_reference_creatinine",
        "sql_window_evidence": None,
        "empirical_invariance_evidence": reference_pass,
        "provisional_decision": (
            "PASS_CANDIDATE"
            if reference_pass
            else "FAIL_CANDIDATE"
        ),
    },
    {
        "critical_item": "x_stage1_at_landmark",
        "sql_window_evidence": None,
        "empirical_invariance_evidence": stage1_empirical_pass,
        "provisional_decision": (
            "PASS_CANDIDATE"
            if stage1_empirical_pass
            else "FAIL_CANDIDATE"
        ),
    },
    {
        "critical_item": "label_stage23",
        "sql_window_evidence": None,
        "empirical_invariance_evidence": outcome_pass,
        "provisional_decision": (
            "PASS_CANDIDATE"
            if outcome_pass
            else "FAIL_CANDIDATE"
        ),
    },
])

# ------------------------------------------------------------
# 9. Save aggregate audit only
# ------------------------------------------------------------

paths = {
    "reference_offset_audit": os.path.join(
        MODEL_OUTPUT_DIR,
        "35D_reference_offset_aggregate_audit.csv",
    ),
    "stage1_reconstruction": os.path.join(
        MODEL_OUTPUT_DIR,
        "35D_stage1_landmark_reconstruction_audit.csv",
    ),
    "outcome_reconstruction": os.path.join(
        MODEL_OUTPUT_DIR,
        "35D_outcome_window_reconstruction_audit.csv",
    ),
    "adjudication": os.path.join(
        MODEL_OUTPUT_DIR,
        "35D_critical_temporal_adjudication.csv",
    ),
    "manifest": os.path.join(
        MODEL_OUTPUT_DIR,
        "35D_critical_temporal_invariance_manifest.json",
    ),
    "manifest_sha": os.path.join(
        MODEL_OUTPUT_DIR,
        "35D_critical_temporal_invariance_manifest_SHA256.txt",
    ),
}

reference_offset_audit.to_csv(
    paths["reference_offset_audit"],
    index=False,
)
stage1_reconstruction.to_csv(
    paths["stage1_reconstruction"],
    index=False,
)
outcome_reconstruction.to_csv(
    paths["outcome_reconstruction"],
    index=False,
)
adjudication.to_csv(
    paths["adjudication"],
    index=False,
)

manifest = {
    "analysis_version": "35D",
    "analysis_type":
        "critical_aki_temporal_invariance_audit",
    "source_35c_manifest_sha256":
        EXPECTED_35C_SHA,
    "landmark_minutes": 720,
    "outcome_end_minutes": 4320,
    "read_only_bigquery": True,
    "bigquery_dml_used": False,
    "model_fit_performed": False,
    "retuning_performed": False,
    "recalibration_performed": False,
    "patient_level_data_written_to_drive": False,
    "final_leakage_pass_declared_by_script": False,
    "outputs": paths,
}

manifest_text = json.dumps(
    manifest,
    indent=2,
    sort_keys=True,
)

with open(
    paths["manifest"],
    "w",
    encoding="utf-8",
) as fh:
    fh.write(manifest_text)

manifest_sha = sha_text(manifest_text)

with open(
    paths["manifest_sha"],
    "w",
    encoding="utf-8",
) as fh:
    fh.write(manifest_sha + "\n")

# ------------------------------------------------------------
# 10. User-facing compact results
# ------------------------------------------------------------

print("\n35D REFERENCE OFFSET AGGREGATE AUDIT")
print(reference_offset_audit.to_string(index=False))

print("\n35D STAGE-1 LANDMARK RECONSTRUCTION AUDIT")
print(stage1_reconstruction.to_string(index=False))

print("\n35D OUTCOME WINDOW RECONSTRUCTION AUDIT")
print(outcome_reconstruction.to_string(index=False))

print("\n35D CRITICAL TEMPORAL ADJUDICATION")
print(adjudication.to_string(index=False))

print("\n35D manifest SHA-256:")
print(manifest_sha)

print(
    "\n35D COMPLETE: exact SQL lines and aggregate temporal "
    "invariance checks completed."
)
print(
    "Send the complete 35D output before any further model analysis."
)

In [ ]:
import os
import re
import json
import hashlib
import pandas as pd

print("STARTING FINAL CRITICAL TEMPORAL LEAKAGE ADJUDICATION LOCK — CODE VERSION 35E")

# ============================================================
# 35E — FINAL ADJUDICATION LOCK
#
# This script does NOT reanalyse patient data or fit any model.
# It verifies the already-completed 35C/35D evidence and writes
# a final immutable audit record for the four critical temporal
# variables:
#
#   x_lab_creatinine_last
#   x_reference_creatinine
#   x_stage1_at_landmark
#   label_stage23
#
# Scope:
#   "critical kidney/outcome temporal leakage audit"
# It does NOT claim that every one of the 159 predictors has been
# independently re-derived in this script.
# ============================================================

EXPECTED_35D_SHA = (
    "baf7d530a45578f7d111ce6da5be09b1"
    "f087f42c050e937056d9ab202e97b874"
)

if "MODEL_OUTPUT_DIR" not in globals():
    raise RuntimeError(
        "Önce MODEL_OUTPUT_DIR tanımlı temel hücreyi çalıştır."
    )

# ------------------------------------------------------------
# 1. Verify 35D lock
# ------------------------------------------------------------

sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "35D_critical_temporal_invariance_manifest_SHA256.txt",
)

if not os.path.exists(sha_path):
    raise FileNotFoundError(sha_path)

with open(sha_path, "r", encoding="utf-8") as fh:
    observed_sha = fh.read().strip()

if observed_sha != EXPECTED_35D_SHA:
    raise RuntimeError(
        "35D manifest SHA mismatch: " + observed_sha
    )

print("35D manifest SHA guard: PASS")

# ------------------------------------------------------------
# 2. Load aggregate audit outputs
# ------------------------------------------------------------

reference_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "35D_reference_offset_aggregate_audit.csv",
)
stage1_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "35D_stage1_landmark_reconstruction_audit.csv",
)
outcome_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "35D_outcome_window_reconstruction_audit.csv",
)
adjudication_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "35D_critical_temporal_adjudication.csv",
)

for path in [
    reference_path,
    stage1_path,
    outcome_path,
    adjudication_path,
]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

reference = pd.read_csv(reference_path)
stage1 = pd.read_csv(stage1_path)
outcome = pd.read_csv(outcome_path)
prior_adjudication = pd.read_csv(adjudication_path)

if len(reference) != 1 or len(stage1) != 1 or len(outcome) != 1:
    raise RuntimeError(
        "35D aggregate audit tables must each contain exactly one row."
    )

r = reference.iloc[0]
s = stage1.iloc[0]
o = outcome.iloc[0]

# ------------------------------------------------------------
# 3. Verify exact expected aggregate invariants
# ------------------------------------------------------------

checks = []

def add_check(name, condition, observed, expected):
    checks.append({
        "check": name,
        "pass": bool(condition),
        "observed": str(observed),
        "expected": str(expected),
    })

add_check(
    "reference_rows_total",
    int(r["rows_total"]) == 58491,
    int(r["rows_total"]),
    58491,
)
add_check(
    "reference_offset_missing",
    int(r["reference_offset_missing"]) == 0,
    int(r["reference_offset_missing"]),
    0,
)
add_check(
    "reference_offset_gt_720",
    int(r["reference_offset_gt_720"]) == 0,
    int(r["reference_offset_gt_720"]),
    0,
)
add_check(
    "reference_offset_maximum",
    float(r["max_reference_offset"]) <= 360,
    float(r["max_reference_offset"]),
    "<= 360 min",
)
add_check(
    "reference_offset_minimum",
    float(r["min_reference_offset"]) >= -1440,
    float(r["min_reference_offset"]),
    ">= -1440 min",
)

add_check(
    "stage1_patients",
    int(s["patients"]) == 58491,
    int(s["patients"]),
    58491,
)
add_check(
    "stage1_stored_positive",
    int(s["stored_stage1_positive"]) == 2949,
    int(s["stored_stage1_positive"]),
    2949,
)
add_check(
    "stage1_mismatch_vs_locked_definition",
    int(s["mismatch_vs_max_severity_definition"]) == 0,
    int(s["mismatch_vs_max_severity_definition"]),
    0,
)
add_check(
    "stage1_maximum_landmark_offset",
    float(s["maximum_last_stage_offset_used"]) <= 720,
    float(s["maximum_last_stage_offset_used"]),
    "<= 720 min",
)
add_check(
    "stage23_present_by_landmark_in_final_cohort",
    int(s["severe_stage_present_by_landmark"]) == 0,
    int(s["severe_stage_present_by_landmark"]),
    0,
)

add_check(
    "outcome_patients",
    int(o["patients"]) == 58491,
    int(o["patients"]),
    58491,
)
add_check(
    "outcome_stored_events",
    int(o["stored_outcome_events"]) == 3032,
    int(o["stored_outcome_events"]),
    3032,
)
add_check(
    "outcome_reconstructed_events",
    int(o["reconstructed_outcome_events"]) == 3032,
    int(o["reconstructed_outcome_events"]),
    3032,
)
add_check(
    "outcome_window_mismatches",
    int(o["outcome_window_mismatches"]) == 0,
    int(o["outcome_window_mismatches"]),
    0,
)
add_check(
    "minimum_positive_outcome_offset",
    float(o["minimum_positive_outcome_offset"]) > 720,
    float(o["minimum_positive_outcome_offset"]),
    "> 720 min",
)
add_check(
    "maximum_first_positive_outcome_offset",
    float(o["maximum_first_positive_outcome_offset"]) <= 4320,
    float(o["maximum_first_positive_outcome_offset"]),
    "<= 4320 min",
)
add_check(
    "negative_cases_with_stage23_after_72h",
    int(o["negatives_with_stage23_after_72h"]) == 0,
    int(o["negatives_with_stage23_after_72h"]),
    0,
)

check_df = pd.DataFrame(checks)

if not check_df["pass"].all():
    failed = check_df.loc[~check_df["pass"]]
    raise RuntimeError(
        "35E final audit cannot PASS. Failed checks:\n"
        + failed.to_string(index=False)
    )

# ------------------------------------------------------------
# 4. Verify the explicit first-12-hour lab evidence captured in 35D
# ------------------------------------------------------------

lab_row = prior_adjudication.loc[
    prior_adjudication["critical_item"].eq(
        "x_lab_creatinine_last"
    )
]

if len(lab_row) != 1:
    raise RuntimeError(
        "x_lab_creatinine_last adjudication row missing."
    )

lab_sql_evidence = str(
    lab_row.iloc[0]["sql_window_evidence"]
).strip().lower()

if lab_sql_evidence not in {"true", "1", "1.0"}:
    raise RuntimeError(
        "35D did not confirm the <=720-minute SQL window "
        "for x_lab_creatinine_last."
    )

# ------------------------------------------------------------
# 5. Final scoped scientific adjudication
# ------------------------------------------------------------

final_rows = [
    {
        "critical_item": "x_lab_creatinine_last",
        "temporal_rule": "eICU labResultOffset BETWEEN 0 AND 720; last creatinine selected within that window",
        "decision": "PASS",
        "basis": "Exact construction SQL",
    },
    {
        "critical_item": "x_reference_creatinine",
        "temporal_rule": "reference creatinine selected from offset max 360 min; no reference offset >720",
        "decision": "PASS",
        "basis": "Exact construction SQL + 58,491-patient aggregate offset audit",
    },
    {
        "critical_item": "x_stage1_at_landmark",
        "temporal_rule": "max KDIGO creatinine stage through 720 min; stage1 flag equals max-stage-by-12h definition",
        "decision": "PASS",
        "basis": "Exact cohort SQL + zero-mismatch 58,491-patient reconstruction",
    },
    {
        "critical_item": "label_stage23",
        "temporal_rule": "incident KDIGO stage 2/3 strictly >720 and <=min(discharge,4320)",
        "decision": "PASS",
        "basis": "Exact cohort SQL + zero-mismatch 58,491-patient outcome reconstruction",
    },
]

final_adjudication = pd.DataFrame(final_rows)

# ------------------------------------------------------------
# 6. Save final immutable audit record
# ------------------------------------------------------------

final_csv = os.path.join(
    MODEL_OUTPUT_DIR,
    "35E_final_critical_temporal_leakage_adjudication.csv",
)
manifest_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "35E_final_critical_temporal_leakage_adjudication_manifest.json",
)
manifest_sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "35E_final_critical_temporal_leakage_adjudication_manifest_SHA256.txt",
)
checks_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "35E_final_critical_temporal_invariant_checks.csv",
)

final_adjudication.to_csv(
    final_csv,
    index=False,
)
check_df.to_csv(
    checks_path,
    index=False,
)

manifest = {
    "analysis_version": "35E",
    "analysis_type": "final_critical_temporal_leakage_adjudication",
    "scope": "critical kidney/outcome temporal variables",
    "source_35d_manifest_sha256": EXPECTED_35D_SHA,
    "landmark_minutes": 720,
    "outcome_window": {
        "start": ">720",
        "end": "<=4320",
    },
    "patients_audited": 58491,
    "critical_items": [
        "x_lab_creatinine_last",
        "x_reference_creatinine",
        "x_stage1_at_landmark",
        "label_stage23",
    ],
    "critical_temporal_leakage_verdict": "PASS",
    "important_scope_note": (
        "This verdict closes the critical kidney/outcome temporal "
        "leakage concern. It does not assert that all 159 predictors "
        "were independently reconstructed in Code 35E."
    ),
    "model_fit_performed": False,
    "retuning_performed": False,
    "recalibration_performed": False,
    "bigquery_dml_used": False,
    "patient_level_data_written_to_drive": False,
    "outputs": {
        "final_adjudication_csv": final_csv,
        "invariant_checks_csv": checks_path,
    },
}

manifest_text = json.dumps(
    manifest,
    indent=2,
    sort_keys=True,
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(manifest_text)

manifest_sha = hashlib.sha256(
    manifest_text.encode("utf-8")
).hexdigest()

with open(
    manifest_sha_path,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(manifest_sha + "\n")

print("\n35E FINAL CRITICAL TEMPORAL ADJUDICATION")
print(final_adjudication.to_string(index=False))

print("\n35E INVARIANT CHECKS")
print(check_df.to_string(index=False))

print("\n35E manifest SHA-256:")
print(manifest_sha)

print("\nSaved:")
print(final_csv)
print(checks_path)
print(manifest_path)
print(manifest_sha_path)

print(
    "\n35E PASS: Critical kidney/outcome temporal leakage audit is closed."
)
print(
    "No model was fitted, retuned, or recalibrated."
)

In [ ]:
import os
import json
import hashlib

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)
from sklearn.linear_model import LogisticRegression
from IPython.display import display

print("STARTING CLINICAL BASELINE VS LOCKED MODELS PAIRED COMPARISON — CODE VERSION 36")

# ============================================================
# 36 — CLINICAL BASELINE VS LOCKED MODELS
#      PAIRED HOSPITAL-CLUSTER COMPARISON
#
# Purpose:
# - compare the corrected parsimonious clinical baseline against
#   the already-frozen LR, RF, XGBoost, and CatBoost predictions;
# - use exactly the same 58,491 patients and 198 hospitals;
# - use paired hospital-cluster bootstrap (2,000 replicates);
# - primary probability type = fold-specific Platt calibrated;
# - raw probabilities retained as secondary audit;
# - no predictive model fitting, retuning, or recalibration.
#
# Provenance:
# - LR/XGB belong to the original model-development set.
# - RF/CatBoost are additional/post-hoc benchmarks.
# - The parsimonious clinical baseline is additional/post-hoc.
# Therefore all model-vs-clinical pairwise CIs in Code 36 are
# explicitly NOMINAL / EXPLORATORY and must not be presented as
# multiplicity-adjusted confirmatory inference.
#
# Patient-level predictions remain in BigQuery/RAM only.
# ============================================================

ANALYSIS_VERSION_36 = "36"
BOOTSTRAP_REPLICATES_36 = 2000
BOOTSTRAP_SEED_36 = 20260723

EXPECTED_ROWS_36 = 58491
EXPECTED_EVENTS_36 = 3032
EXPECTED_NONEVENTS_36 = 55459
EXPECTED_HOSPITALS_36 = 198
EXPECTED_OUTER_FOLDS_36 = {1, 2, 3, 4, 5}

EXPECTED_PROTOCOL_SHA_36 = {
    "logistic_regression": (
        "400c3b4b510c836794543bc685c62fae"
        "f49df0c1caa197d78dffda8c2207952d"
    ),
    "random_forest": (
        "c3b9586b0edf60b6f0abac3894ad66c"
        "886974e1019b01ffa40daf6c27475d026"
    ),
    "xgboost": (
        "3434db5dd0b4b5145950fc07fba3d007"
        "829738d800b88ec40fbdb09879188264"
    ),
    "catboost": (
        "9aa1d57b907c9e31a881506a4309ba55"
        "aad311e8b747d835caa4dc156f3dcb9e"
    ),
    "clinical_baseline": (
        "94b0abb218dbef4e349702ea2824ca4e"
        "31dfbe36c53efa05ba1a8bd1f38f835e"
    ),
}

PROTOCOL_SHA_FILES_36 = {
    "logistic_regression":
        "08B_locked_logistic_model_protocol_v1_SHA256.txt",
    "random_forest":
        "19A_locked_random_forest_model_protocol_v1_SHA256.txt",
    "xgboost":
        "13A_locked_xgboost_model_protocol_v1_SHA256.txt",
    "catboost":
        "25A_locked_catboost_benchmark_protocol_v1_SHA256.txt",
    "clinical_baseline":
        "33A_locked_parsimonious_clinical_baseline_protocol_v1_SHA256.txt",
}

EXPECTED_CLINICAL_POOLED_MANIFEST_SHA_36 = (
    "a1eb23d936154d521048c88e8b34d583"
    "14c38755c8b8e5126741e17c1a9f1924"
)

EXPECTED_TEMPORAL_AUDIT_SHA_36 = (
    "7230605e14a2314ec1724f48c61e42d2"
    "07f96c70ae2d4fe480bf02145ad24544"
)

POOLED_TABLES_36 = {
    "logistic_regression":
        "model_lr_outer_predictions_all5_v1",
    "random_forest":
        "model_random_forest_outer_predictions_all5_v1",
    "xgboost":
        "model_xgb_outer_predictions_all5_v1",
    "catboost":
        "model_catboost_outer_predictions_all5_v1",
    "clinical_baseline":
        "model_clinical_lr_outer_predictions_all5_v2",
}

MODEL_ORDER_36 = [
    "clinical_baseline",
    "logistic_regression",
    "random_forest",
    "catboost",
    "xgboost",
]

# Every pair is oriented model_a vs clinical baseline.
PAIR_ORDER_36 = [
    ("xgboost", "clinical_baseline"),
    ("catboost", "clinical_baseline"),
    ("random_forest", "clinical_baseline"),
    ("logistic_regression", "clinical_baseline"),
]

MODEL_PROVENANCE_36 = {
    "xgboost": "original_model_set",
    "logistic_regression": "original_model_set",
    "random_forest": "additional_post_hoc_benchmark",
    "catboost": "additional_post_hoc_benchmark",
    "clinical_baseline": "additional_post_hoc_clinical_baseline",
}

# ------------------------------------------------------------
# 1. Required runtime objects
# ------------------------------------------------------------

required_objects = [
    "core_df_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing = [
    name for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Eksik çalışma nesneleri var: "
        + ", ".join(missing)
        + ". Önce 07A/07B temel hücrelerini çalıştır."
    )

if len(core_df_07B) != EXPECTED_ROWS_36:
    raise RuntimeError(
        f"core_df_07B rows={len(core_df_07B)}, "
        f"expected={EXPECTED_ROWS_36}."
    )

# ------------------------------------------------------------
# 2. Verify frozen protocols + clinical pooled/audit manifests
# ------------------------------------------------------------

protocol_rows = []

for model_name, filename in PROTOCOL_SHA_FILES_36.items():
    path = os.path.join(MODEL_OUTPUT_DIR, filename)

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    with open(path, "r", encoding="utf-8") as fh:
        observed = fh.read().strip()

    expected = EXPECTED_PROTOCOL_SHA_36[model_name]

    if observed != expected:
        raise RuntimeError(
            f"{model_name}: protocol SHA mismatch. "
            f"observed={observed}, expected={expected}"
        )

    protocol_rows.append({
        "model": model_name,
        "protocol_sha256": observed,
        "status": "PASS",
    })

protocol_audit = pd.DataFrame(protocol_rows)

clinical_manifest_sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "34E_final_clinical_baseline_pooled_evaluation_manifest_SHA256.txt",
)
temporal_sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "35E_final_critical_temporal_leakage_adjudication_manifest_SHA256.txt",
)

for path in [clinical_manifest_sha_path, temporal_sha_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(path)

with open(
    clinical_manifest_sha_path,
    "r",
    encoding="utf-8",
) as fh:
    observed_clinical_manifest_sha = fh.read().strip()

with open(
    temporal_sha_path,
    "r",
    encoding="utf-8",
) as fh:
    observed_temporal_sha = fh.read().strip()

if observed_clinical_manifest_sha != EXPECTED_CLINICAL_POOLED_MANIFEST_SHA_36:
    raise RuntimeError(
        "Clinical pooled Code34 manifest SHA mismatch."
    )

if observed_temporal_sha != EXPECTED_TEMPORAL_AUDIT_SHA_36:
    raise RuntimeError(
        "Final critical temporal leakage audit SHA mismatch."
    )

print("All five model protocol SHA guards: PASS")
print("Clinical pooled Code34 manifest SHA guard: PASS")
print("35E critical temporal leakage audit SHA guard: PASS")

# ------------------------------------------------------------
# 3. Helpers
# ------------------------------------------------------------

def resolve_column(columns, exact_candidates, contains_any=None):
    lower_map = {
        str(c).lower(): c
        for c in columns
    }

    for candidate in exact_candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]

    if contains_any:
        matches = []
        for c in columns:
            low = str(c).lower()
            if any(token.lower() in low for token in contains_any):
                matches.append(c)

        if len(matches) == 1:
            return matches[0]

    raise RuntimeError(
        "Required column could not be resolved. "
        f"Candidates={exact_candidates}; columns={list(columns)}"
    )


def load_predictions(model_name, table_suffix):
    table_id = f"{TARGET_DATASET}.{table_suffix}"
    client.get_table(table_id)

    job = client.query(
        f"SELECT * FROM `{table_id}`",
        location=BQ_LOCATION,
    )

    try:
        df = job.to_dataframe(
            create_bqstorage_client=True
        )
        load_method = "BigQuery Storage API"
    except Exception:
        df = job.to_dataframe(
            create_bqstorage_client=False
        )
        load_method = "Standard BigQuery API"

    id_col = resolve_column(
        df.columns,
        ["id_row"],
        ["id_row"],
    )
    fold_col = resolve_column(
        df.columns,
        ["outer_fold"],
        ["outer_fold"],
    )
    label_col = resolve_column(
        df.columns,
        ["label_stage23"],
        ["label_stage23"],
    )
    raw_col = resolve_column(
        df.columns,
        ["prediction_raw"],
        ["prediction_raw", "raw_probability"],
    )
    platt_col = resolve_column(
        df.columns,
        ["prediction_platt"],
        ["prediction_platt", "platt"],
    )

    out = pd.DataFrame({
        "id_row": df[id_col].astype(str),
        "outer_fold": pd.to_numeric(
            df[fold_col],
            errors="raise",
        ).astype(np.int64),
        "label_stage23": pd.to_numeric(
            df[label_col],
            errors="raise",
        ).astype(np.int64),
        f"{model_name}_raw": pd.to_numeric(
            df[raw_col],
            errors="raise",
        ).astype(float),
        f"{model_name}_platt": pd.to_numeric(
            df[platt_col],
            errors="raise",
        ).astype(float),
    })

    if len(out) != EXPECTED_ROWS_36:
        raise RuntimeError(
            f"{model_name}: rows={len(out)}; expected={EXPECTED_ROWS_36}"
        )

    if out["id_row"].duplicated().any():
        raise RuntimeError(
            f"{model_name}: duplicate id_row."
        )

    if set(out["outer_fold"].unique()) != EXPECTED_OUTER_FOLDS_36:
        raise RuntimeError(
            f"{model_name}: outer folds are not exactly 1..5."
        )

    if int(out["label_stage23"].sum()) != EXPECTED_EVENTS_36:
        raise RuntimeError(
            f"{model_name}: event count mismatch."
        )

    for col in [
        f"{model_name}_raw",
        f"{model_name}_platt",
    ]:
        values = out[col].to_numpy(dtype=float)
        if not np.isfinite(values).all():
            raise RuntimeError(
                f"{model_name}: non-finite predictions."
            )
        if ((values < 0) | (values > 1)).any():
            raise RuntimeError(
                f"{model_name}: invalid probabilities."
            )

    return out, {
        "model": model_name,
        "table_id": table_id,
        "load_method": load_method,
        "rows": len(out),
        "events": int(out["label_stage23"].sum()),
    }


def calibration_intercept_slope(y, p):
    eps = 1e-12
    p = np.clip(
        np.asarray(p, dtype=float),
        eps,
        1 - eps,
    )
    y = np.asarray(y, dtype=np.int8)

    logits = np.log(
        p / (1 - p)
    ).reshape(-1, 1)

    model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=1000,
    )
    model.fit(logits, y)

    return (
        float(model.intercept_[0]),
        float(model.coef_[0, 0]),
    )


def metric_bundle(y, p, sample_weight=None, calibration=False):
    y = np.asarray(y, dtype=np.int8)
    p = np.asarray(p, dtype=float)

    result = {
        "auroc": float(
            roc_auc_score(
                y,
                p,
                sample_weight=sample_weight,
            )
        ),
        "auprc": float(
            average_precision_score(
                y,
                p,
                sample_weight=sample_weight,
            )
        ),
        "brier": float(
            brier_score_loss(
                y,
                p,
                sample_weight=sample_weight,
            )
        ),
        "log_loss": float(
            log_loss(
                y,
                p,
                sample_weight=sample_weight,
                labels=[0, 1],
            )
        ),
    }

    if calibration:
        ci, cs = calibration_intercept_slope(y, p)
        result["calibration_intercept"] = ci
        result["calibration_slope"] = cs

    return result

# ------------------------------------------------------------
# 4. Core audit and patient/hospital alignment
# ------------------------------------------------------------

needed_core = {
    "id_row",
    "outer_fold",
    "label_stage23",
    "group_hospital",
}

missing_core = needed_core - set(core_df_07B.columns)

if missing_core:
    raise RuntimeError(
        "core_df_07B missing: "
        + ", ".join(sorted(missing_core))
    )

core = core_df_07B[
    [
        "id_row",
        "outer_fold",
        "label_stage23",
        "group_hospital",
    ]
].copy()

core["id_row"] = core["id_row"].astype(str)
core["outer_fold"] = pd.to_numeric(
    core["outer_fold"],
    errors="raise",
).astype(np.int64)
core["label_stage23"] = pd.to_numeric(
    core["label_stage23"],
    errors="raise",
).astype(np.int64)
core["group_hospital"] = core[
    "group_hospital"
].astype(str)

if core["id_row"].duplicated().any():
    raise RuntimeError("Duplicate id_row in core.")

if core["group_hospital"].nunique() != EXPECTED_HOSPITALS_36:
    raise RuntimeError("Core hospital count is not 198.")

loaded = {}
source_rows = []

for model_name in MODEL_ORDER_36:
    print(f"Loading pooled predictions: {model_name}...")
    model_df, source_row = load_predictions(
        model_name,
        POOLED_TABLES_36[model_name],
    )
    loaded[model_name] = model_df
    source_rows.append(source_row)

source_audit = pd.DataFrame(source_rows)

comparison = core.copy()

for model_name in MODEL_ORDER_36:
    df = loaded[model_name]

    comparison = comparison.merge(
        df,
        on="id_row",
        how="inner",
        validate="one_to_one",
        suffixes=("", f"_{model_name}"),
    )

    if len(comparison) != EXPECTED_ROWS_36:
        raise RuntimeError(
            f"{model_name}: alignment lost rows."
        )

    fold_col = f"outer_fold_{model_name}"
    label_col = f"label_stage23_{model_name}"

    if (
        comparison["outer_fold"]
        != comparison[fold_col]
    ).any():
        raise RuntimeError(
            f"{model_name}: outer-fold mismatch."
        )

    if (
        comparison["label_stage23"]
        != comparison[label_col]
    ).any():
        raise RuntimeError(
            f"{model_name}: outcome mismatch."
        )

    comparison = comparison.drop(
        columns=[fold_col, label_col]
    )

hospital_fold_n = comparison.groupby(
    "group_hospital"
)["outer_fold"].nunique()

if (hospital_fold_n != 1).any():
    raise RuntimeError(
        "Hospital cross-fold violation detected."
    )

alignment_integrity = pd.DataFrame([{
    "patients": len(comparison),
    "distinct_patients": comparison["id_row"].nunique(),
    "hospitals": comparison["group_hospital"].nunique(),
    "outer_folds": comparison["outer_fold"].nunique(),
    "events": int(comparison["label_stage23"].sum()),
    "nonevents": int(
        len(comparison)
        - comparison["label_stage23"].sum()
    ),
    "duplicate_patients": int(
        comparison["id_row"].duplicated().sum()
    ),
    "hospital_cross_fold_violations": int(
        (hospital_fold_n != 1).sum()
    ),
}])

print("Five-model patient/hospital/outcome alignment: PASS")

# ------------------------------------------------------------
# 5. Pooled point metrics
# ------------------------------------------------------------

y = comparison["label_stage23"].to_numpy(
    dtype=np.int8
)

point_rows = []

for model_name in MODEL_ORDER_36:
    for probability_type, suffix in [
        ("raw", "raw"),
        ("platt_calibrated", "platt"),
    ]:
        p = comparison[
            f"{model_name}_{suffix}"
        ].to_numpy(dtype=float)

        metrics = metric_bundle(
            y,
            p,
            calibration=True,
        )

        point_rows.append({
            "model": model_name,
            "model_provenance":
                MODEL_PROVENANCE_36[model_name],
            "probability_type": probability_type,
            "patients": EXPECTED_ROWS_36,
            "hospitals": EXPECTED_HOSPITALS_36,
            "events": EXPECTED_EVENTS_36,
            "auroc": metrics["auroc"],
            "auprc": metrics["auprc"],
            "brier": metrics["brier"],
            "log_loss": metrics["log_loss"],
            "mean_predicted_risk": float(p.mean()),
            "observed_event_rate": float(y.mean()),
            "calibration_intercept":
                metrics["calibration_intercept"],
            "calibration_slope":
                metrics["calibration_slope"],
        })

pooled_metrics = pd.DataFrame(point_rows)

# ------------------------------------------------------------
# 6. Point differences vs clinical baseline
#    Positive values ALWAYS favor model_a.
# ------------------------------------------------------------

lookup = {
    (row["model"], row["probability_type"]): row
    for _, row in pooled_metrics.iterrows()
}

point_diff_rows = []

for model_a, model_b in PAIR_ORDER_36:
    for probability_type in [
        "raw",
        "platt_calibrated",
    ]:
        a = lookup[(model_a, probability_type)]
        b = lookup[(model_b, probability_type)]

        point_diff_rows.append({
            "model_a": model_a,
            "model_b": model_b,
            "model_a_provenance":
                MODEL_PROVENANCE_36[model_a],
            "model_b_provenance":
                MODEL_PROVENANCE_36[model_b],
            "probability_type": probability_type,
            "auroc_improvement": (
                float(a["auroc"]) - float(b["auroc"])
            ),
            "auprc_improvement": (
                float(a["auprc"]) - float(b["auprc"])
            ),
            "brier_improvement": (
                float(b["brier"]) - float(a["brier"])
            ),
            "log_loss_improvement": (
                float(b["log_loss"])
                - float(a["log_loss"])
            ),
            "positive_value_favors": model_a,
            "inference_status":
                "exploratory_nominal_95_ci",
        })

point_differences = pd.DataFrame(
    point_diff_rows
)

# ------------------------------------------------------------
# 7. Paired hospital-cluster bootstrap
# ------------------------------------------------------------

print(
    f"\nRunning {BOOTSTRAP_REPLICATES_36:,} paired "
    "hospital-cluster bootstrap replicates..."
)

hospital_cat = pd.Categorical(
    comparison["group_hospital"]
)
hospital_codes = hospital_cat.codes.astype(int)

if len(hospital_cat.categories) != EXPECTED_HOSPITALS_36:
    raise RuntimeError(
        "Bootstrap hospital count is not 198."
    )

prob_vectors = {}

for model_name in MODEL_ORDER_36:
    prob_vectors[(model_name, "raw")] = comparison[
        f"{model_name}_raw"
    ].to_numpy(dtype=float)
    prob_vectors[(model_name, "platt_calibrated")] = comparison[
        f"{model_name}_platt"
    ].to_numpy(dtype=float)

rng = np.random.default_rng(
    BOOTSTRAP_SEED_36
)

bootstrap_rows = []

for b in range(BOOTSTRAP_REPLICATES_36):
    sampled_codes = rng.integers(
        0,
        EXPECTED_HOSPITALS_36,
        size=EXPECTED_HOSPITALS_36,
    )

    multiplicity = np.bincount(
        sampled_codes,
        minlength=EXPECTED_HOSPITALS_36,
    )

    weights = multiplicity[
        hospital_codes
    ].astype(float)

    if (
        weights[y == 1].sum() <= 0
        or weights[y == 0].sum() <= 0
    ):
        continue

    metrics_cache = {}

    for probability_type in [
        "raw",
        "platt_calibrated",
    ]:
        for model_name in MODEL_ORDER_36:
            metrics_cache[
                (model_name, probability_type)
            ] = metric_bundle(
                y,
                prob_vectors[
                    (model_name, probability_type)
                ],
                sample_weight=weights,
                calibration=False,
            )

    row = {
        "bootstrap_replicate": b + 1,
        "unique_sampled_hospitals": int(
            np.count_nonzero(multiplicity)
        ),
        "weighted_patients": float(
            weights.sum()
        ),
        "weighted_events": float(
            weights[y == 1].sum()
        ),
        "weighted_nonevents": float(
            weights[y == 0].sum()
        ),
    }

    for model_a, model_b in PAIR_ORDER_36:
        pair = f"{model_a}_vs_{model_b}"

        for probability_type, suffix in [
            ("raw", "raw"),
            ("platt_calibrated", "platt"),
        ]:
            a = metrics_cache[
                (model_a, probability_type)
            ]
            bmet = metrics_cache[
                (model_b, probability_type)
            ]

            row[
                f"{pair}__auroc_improvement__{suffix}"
            ] = a["auroc"] - bmet["auroc"]

            row[
                f"{pair}__auprc_improvement__{suffix}"
            ] = a["auprc"] - bmet["auprc"]

            row[
                f"{pair}__brier_improvement__{suffix}"
            ] = bmet["brier"] - a["brier"]

            row[
                f"{pair}__log_loss_improvement__{suffix}"
            ] = bmet["log_loss"] - a["log_loss"]

    bootstrap_rows.append(row)

    if b == 0 or (b + 1) % 100 == 0:
        print(
            "  Completed paired bootstrap replicate",
            b + 1,
            "/",
            BOOTSTRAP_REPLICATES_36,
        )

bootstrap_df = pd.DataFrame(
    bootstrap_rows
)

if len(bootstrap_df) < BOOTSTRAP_REPLICATES_36 * 0.99:
    raise RuntimeError(
        "Fewer than 99% of bootstrap replicates completed."
    )

# ------------------------------------------------------------
# 8. Nominal 95% bootstrap CIs
# ------------------------------------------------------------

point_lookup = {
    (
        row["model_a"],
        row["model_b"],
        row["probability_type"],
    ): row
    for _, row in point_differences.iterrows()
}

metric_names = [
    "auroc_improvement",
    "auprc_improvement",
    "brier_improvement",
    "log_loss_improvement",
]

ci_rows = []

for model_a, model_b in PAIR_ORDER_36:
    pair = f"{model_a}_vs_{model_b}"

    for probability_type, suffix in [
        ("raw", "raw"),
        ("platt_calibrated", "platt"),
    ]:
        point = point_lookup[
            (model_a, model_b, probability_type)
        ]

        for metric in metric_names:
            col = f"{pair}__{metric}__{suffix}"
            vals = bootstrap_df[
                col
            ].dropna().to_numpy(dtype=float)

            lo = float(
                np.quantile(vals, 0.025)
            )
            hi = float(
                np.quantile(vals, 0.975)
            )
            pe = float(point[metric])

            if lo > 0:
                direction = f"nominally_favors_{model_a}"
            elif hi < 0:
                direction = f"nominally_favors_{model_b}"
            else:
                direction = "nominal_95_ci_includes_zero"

            ci_rows.append({
                "model_a": model_a,
                "model_b": model_b,
                "probability_type": probability_type,
                "metric": metric,
                "point_estimate": pe,
                "bootstrap_replicates": len(vals),
                "bootstrap_mean": float(
                    np.mean(vals)
                ),
                "bootstrap_standard_error": float(
                    np.std(vals, ddof=1)
                ),
                "nominal_ci_95_lower": lo,
                "nominal_ci_95_upper": hi,
                "positive_value_favors": model_a,
                "statistical_direction": direction,
                "bootstrap_unit": "hospital",
                "paired_resampling": True,
                "bootstrap_seed": BOOTSTRAP_SEED_36,
                "multiplicity_adjusted": False,
                "inference_status":
                    "exploratory_nominal_95_ci",
            })

pairwise_ci = pd.DataFrame(ci_rows)

# ------------------------------------------------------------
# 9. Primary Platt comparison table
# ------------------------------------------------------------

primary_platt = pooled_metrics.loc[
    pooled_metrics[
        "probability_type"
    ].eq("platt_calibrated")
].copy()

primary_platt = primary_platt.sort_values(
    ["auprc", "auroc", "brier"],
    ascending=[False, False, True],
    kind="stable",
).reset_index(drop=True)

primary_platt.insert(
    0,
    "rank_by_auprc",
    np.arange(1, len(primary_platt) + 1),
)

primary_pairwise_ci = pairwise_ci.loc[
    pairwise_ci[
        "probability_type"
    ].eq("platt_calibrated")
].copy()

# ------------------------------------------------------------
# 10. Save aggregate outputs only
# ------------------------------------------------------------

os.makedirs(
    MODEL_OUTPUT_DIR,
    exist_ok=True,
)

paths = {
    "protocol_audit": os.path.join(
        MODEL_OUTPUT_DIR,
        "36A_model_protocol_sha_audit.csv",
    ),
    "source_audit": os.path.join(
        MODEL_OUTPUT_DIR,
        "36A_source_prediction_audit.csv",
    ),
    "alignment": os.path.join(
        MODEL_OUTPUT_DIR,
        "36A_five_model_alignment_integrity.csv",
    ),
    "pooled_metrics": os.path.join(
        MODEL_OUTPUT_DIR,
        "36B_five_model_pooled_metrics.csv",
    ),
    "point_differences": os.path.join(
        MODEL_OUTPUT_DIR,
        "36B_model_vs_clinical_point_differences.csv",
    ),
    "primary_platt": os.path.join(
        MODEL_OUTPUT_DIR,
        "36B_primary_platt_model_summary.csv",
    ),
    "bootstrap_replicates": os.path.join(
        MODEL_OUTPUT_DIR,
        "36C_paired_hospital_bootstrap_replicates.csv",
    ),
    "pairwise_ci": os.path.join(
        MODEL_OUTPUT_DIR,
        "36C_model_vs_clinical_nominal_95CI.csv",
    ),
    "primary_pairwise_ci": os.path.join(
        MODEL_OUTPUT_DIR,
        "36C_primary_platt_model_vs_clinical_nominal_95CI.csv",
    ),
    "manifest": os.path.join(
        MODEL_OUTPUT_DIR,
        "36D_model_vs_clinical_paired_comparison_manifest.json",
    ),
    "manifest_sha": os.path.join(
        MODEL_OUTPUT_DIR,
        "36D_model_vs_clinical_paired_comparison_manifest_SHA256.txt",
    ),
}

protocol_audit.to_csv(
    paths["protocol_audit"],
    index=False,
)
source_audit.to_csv(
    paths["source_audit"],
    index=False,
)
alignment_integrity.to_csv(
    paths["alignment"],
    index=False,
)
pooled_metrics.to_csv(
    paths["pooled_metrics"],
    index=False,
)
point_differences.to_csv(
    paths["point_differences"],
    index=False,
)
primary_platt.to_csv(
    paths["primary_platt"],
    index=False,
)
bootstrap_df.to_csv(
    paths["bootstrap_replicates"],
    index=False,
)
pairwise_ci.to_csv(
    paths["pairwise_ci"],
    index=False,
)
primary_pairwise_ci.to_csv(
    paths["primary_pairwise_ci"],
    index=False,
)

manifest = {
    "analysis_version": ANALYSIS_VERSION_36,
    "analysis_type":
        "paired_hospital_cluster_model_vs_clinical_baseline_comparison",
    "patients": EXPECTED_ROWS_36,
    "hospitals": EXPECTED_HOSPITALS_36,
    "events": EXPECTED_EVENTS_36,
    "nonevents": EXPECTED_NONEVENTS_36,
    "outer_folds": 5,
    "models": MODEL_ORDER_36,
    "model_provenance": MODEL_PROVENANCE_36,
    "pair_order": PAIR_ORDER_36,
    "primary_probability_type": "platt_calibrated",
    "bootstrap_replicates": BOOTSTRAP_REPLICATES_36,
    "bootstrap_seed": BOOTSTRAP_SEED_36,
    "bootstrap_unit": "hospital",
    "paired_resampling": True,
    "confidence_intervals":
        "nominal_95_percent_exploratory_not_multiplicity_adjusted",
    "clinical_pooled_manifest_sha256":
        EXPECTED_CLINICAL_POOLED_MANIFEST_SHA_36,
    "critical_temporal_leakage_audit_sha256":
        EXPECTED_TEMPORAL_AUDIT_SHA_36,
    "predictive_model_fit_performed": False,
    "retuning_performed": False,
    "recalibration_performed": False,
    "diagnostic_calibration_intercept_slope_evaluated": True,
    "bigquery_dml_used": False,
    "patient_level_data_written_to_drive": False,
    "outputs": paths,
}

manifest_text = json.dumps(
    manifest,
    indent=2,
    sort_keys=True,
)

with open(
    paths["manifest"],
    "w",
    encoding="utf-8",
) as fh:
    fh.write(manifest_text)

manifest_sha = hashlib.sha256(
    manifest_text.encode("utf-8")
).hexdigest()

with open(
    paths["manifest_sha"],
    "w",
    encoding="utf-8",
) as fh:
    fh.write(manifest_sha + "\n")

# ------------------------------------------------------------
# 11. Display
# ------------------------------------------------------------

print("\n36A FIVE-MODEL ALIGNMENT INTEGRITY")
display(alignment_integrity)

print("\n36B PRIMARY PLATT MODEL SUMMARY")
display(primary_platt)

print("\n36B MODEL VS CLINICAL POINT DIFFERENCES")
display(
    point_differences.loc[
        point_differences[
            "probability_type"
        ].eq("platt_calibrated")
    ]
)

print("\n36C PRIMARY PLATT MODEL VS CLINICAL NOMINAL 95% CIs")
display(primary_pairwise_ci)

print("\n36D manifest SHA-256:")
print(manifest_sha)

print(
    "\n36 PASS: Paired hospital-cluster comparison against the "
    "parsimonious clinical baseline is complete."
)
print(
    "All pairwise confidence intervals are exploratory nominal 95% CIs."
)
print(
    "No predictive model was fitted, retuned, or recalibrated."
)
print(
    "No patient-level prediction file was written to Google Drive."
)

In [ ]:
import os
import json
import hashlib

print("STARTING DECISION CURVE ANALYSIS PROTOCOL LOCK — CODE VERSION 37A")

# ============================================================
# 37A — DECISION CURVE ANALYSIS (DCA) PROTOCOL LOCK
#
# This script ONLY locks the DCA analysis plan.
# It does NOT calculate or display any DCA result.
#
# Primary DCA comparison:
#   1) XGBoost (fold-specific Platt-calibrated probabilities)
#   2) Parsimonious clinical baseline (fold-specific Platt-calibrated probabilities)
#   3) Treat-all
#   4) Treat-none
#
# Rationale:
# - XGBoost is the numerically strongest model in the locked pooled analysis.
# - The parsimonious clinical model is the clinically interpretable benchmark.
# - RF and CatBoost remain additional/post-hoc benchmarks and are not added to
#   the primary DCA figure to avoid an algorithm-zoo clinical-utility analysis.
#
# Thresholds are locked BEFORE any DCA values are viewed:
#   Primary range:     1.0% to 10.0% in 0.5 percentage-point steps
#   Sensitivity range: 0.5% to 15.0% in 0.5 percentage-point steps
#
# These ranges are chosen a priori for an outcome prevalence near 5% and for
# an early-warning setting where plausible intervention thresholds are expected
# to be low. They are NOT chosen after observing DCA results.
#
# Net benefit:
#   NB = TP/N - FP/N * pt/(1-pt)
#
# No threshold will be selected as "optimal" after viewing the curve.
# Results will be presented across the full locked threshold range.
# ============================================================

EXPECTED_CODE36_MANIFEST_SHA = (
    "f31ad0df192bc0872d1626771a773c50"
    "b7fff798534751144d9a060f82d5f718"
)

EXPECTED_35E_TEMPORAL_AUDIT_SHA = (
    "7230605e14a2314ec1724f48c61e42d2"
    "07f96c70ae2d4fe480bf02145ad24544"
)

EXPECTED_XGB_PROTOCOL_SHA = (
    "3434db5dd0b4b5145950fc07fba3d007"
    "829738d800b88ec40fbdb09879188264"
)

EXPECTED_CLINICAL_PROTOCOL_SHA = (
    "94b0abb218dbef4e349702ea2824ca4e"
    "31dfbe36c53efa05ba1a8bd1f38f835e"
)

if "MODEL_OUTPUT_DIR" not in globals():
    raise RuntimeError(
        "Önce MODEL_OUTPUT_DIR tanımlı temel hücreyi çalıştır."
    )

# ------------------------------------------------------------
# 1. Verify upstream locked analyses
# ------------------------------------------------------------

sha_guards = [
    (
        "36D_model_vs_clinical_paired_comparison_manifest_SHA256.txt",
        EXPECTED_CODE36_MANIFEST_SHA,
        "Code36 paired comparison",
    ),
    (
        "35E_final_critical_temporal_leakage_adjudication_manifest_SHA256.txt",
        EXPECTED_35E_TEMPORAL_AUDIT_SHA,
        "35E critical temporal leakage audit",
    ),
    (
        "13A_locked_xgboost_model_protocol_v1_SHA256.txt",
        EXPECTED_XGB_PROTOCOL_SHA,
        "XGBoost protocol",
    ),
    (
        "33A_locked_parsimonious_clinical_baseline_protocol_v1_SHA256.txt",
        EXPECTED_CLINICAL_PROTOCOL_SHA,
        "Clinical baseline protocol",
    ),
]

for filename, expected_sha, label in sha_guards:
    path = os.path.join(MODEL_OUTPUT_DIR, filename)

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    with open(path, "r", encoding="utf-8") as fh:
        observed_sha = fh.read().strip()

    if observed_sha != expected_sha:
        raise RuntimeError(
            f"{label} SHA mismatch: {observed_sha}"
        )

    print(f"{label} SHA guard: PASS")

# ------------------------------------------------------------
# 2. Lock threshold grids without looking at DCA results
# ------------------------------------------------------------

primary_thresholds = [
    round(x / 1000.0, 4)
    for x in range(10, 101, 5)
]  # 0.010 to 0.100 by 0.005

sensitivity_thresholds = [
    round(x / 1000.0, 4)
    for x in range(5, 151, 5)
]  # 0.005 to 0.150 by 0.005

assert primary_thresholds[0] == 0.01
assert primary_thresholds[-1] == 0.10
assert sensitivity_thresholds[0] == 0.005
assert sensitivity_thresholds[-1] == 0.15

# ------------------------------------------------------------
# 3. Immutable DCA protocol
# ------------------------------------------------------------

protocol = {
    "analysis_version": "37A",
    "analysis_type": "decision_curve_analysis_protocol_lock",
    "timing": "locked_before_any_dca_values_or_curves_are_viewed",

    "study_context": {
        "outcome": "incident KDIGO creatinine stage 2-3 after 12 h and through 72 h",
        "landmark_hours": 12,
        "outcome_window_hours": ">12 to 72",
        "observed_event_rate_reference": 0.051837,
        "patients": 58491,
        "hospitals": 198,
    },

    "primary_models": [
        {
            "name": "xgboost",
            "prediction_table":
                "model_xgb_outer_predictions_all5_v1",
            "probability": "prediction_platt",
            "role": "strongest_locked_machine_learning_model",
        },
        {
            "name": "parsimonious_clinical_baseline",
            "prediction_table":
                "model_clinical_lr_outer_predictions_all5_v2",
            "probability": "prediction_platt",
            "role": "clinically_interpretable_post_hoc_benchmark",
        },
    ],

    "reference_strategies": [
        "treat_all",
        "treat_none",
    ],

    "primary_threshold_range": {
        "lower": 0.01,
        "upper": 0.10,
        "step": 0.005,
        "thresholds": primary_thresholds,
        "status": "a_priori_locked",
    },

    "sensitivity_threshold_range": {
        "lower": 0.005,
        "upper": 0.15,
        "step": 0.005,
        "thresholds": sensitivity_thresholds,
        "status": "a_priori_locked",
    },

    "threshold_rationale": (
        "The outcome prevalence is approximately 5.18%. "
        "In an early-warning setting, plausible intervention thresholds "
        "are expected to be low because preventive responses may be initiated "
        "before high absolute predicted risk. The 1-10% primary range and "
        "0.5-15% sensitivity range are therefore fixed before inspecting "
        "decision-curve results and will not be altered based on observed "
        "net benefit."
    ),

    "net_benefit_definition": (
        "TP/N - FP/N * pt/(1-pt)"
    ),

    "analysis_unit": "patient",
    "uncertainty_unit": "hospital_cluster",
    "bootstrap_replicates": 2000,
    "bootstrap_seed": 20260723,

    "primary_reporting": {
        "probability_type": "fold_specific_platt_calibrated",
        "show_full_locked_threshold_range": True,
        "select_optimal_threshold_post_hoc": False,
        "claim_clinical_utility_from_single_threshold": False,
        "compare_xgb_vs_clinical_baseline": True,
        "compare_vs_treat_all": True,
        "compare_vs_treat_none": True,
    },

    "interpretation_rules": [
        (
            "A model may be described as having higher net benefit only over "
            "threshold regions where its curve exceeds the comparator."
        ),
        (
            "No threshold will be chosen or emphasized because it maximizes "
            "observed net benefit."
        ),
        (
            "Clinical utility claims will remain conditional on the locked "
            "threshold range and the retrospective eICU setting."
        ),
        (
            "DCA does not replace prospective impact evaluation."
        ),
    ],

    "excluded_from_primary_dca": {
        "random_forest":
            "additional/post-hoc benchmark; omitted to keep DCA clinically focused",
        "catboost":
            "additional/post-hoc benchmark; omitted to keep DCA clinically focused",
        "qSOFA":
            "standard definition not reconstructable in the locked 0-12 h data window",
        "SIRS":
            "standard definition not adequately reconstructable because of low temperature/full-case coverage",
    },

    "upstream_sha256": {
        "code36_manifest": EXPECTED_CODE36_MANIFEST_SHA,
        "critical_temporal_leakage_audit":
            EXPECTED_35E_TEMPORAL_AUDIT_SHA,
        "xgboost_protocol": EXPECTED_XGB_PROTOCOL_SHA,
        "clinical_baseline_protocol": EXPECTED_CLINICAL_PROTOCOL_SHA,
    },

    "predictive_model_fit_performed": False,
    "retuning_performed": False,
    "recalibration_performed": False,
    "dca_result_calculated": False,
    "dca_result_viewed": False,
    "patient_level_data_written_to_drive": False,
    "bigquery_dml_used": False,
}

protocol_text = json.dumps(
    protocol,
    indent=2,
    sort_keys=True,
)

protocol_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "37A_locked_decision_curve_analysis_protocol_v1.json",
)
sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "37A_locked_decision_curve_analysis_protocol_v1_SHA256.txt",
)

with open(protocol_path, "w", encoding="utf-8") as fh:
    fh.write(protocol_text)

protocol_sha = hashlib.sha256(
    protocol_text.encode("utf-8")
).hexdigest()

with open(sha_path, "w", encoding="utf-8") as fh:
    fh.write(protocol_sha + "\n")

print("\n37A LOCKED DCA PROTOCOL")
print("Primary models:")
print("- XGBoost, Platt calibrated")
print("- Parsimonious clinical baseline, Platt calibrated")
print("- Treat-all")
print("- Treat-none")
print()
print("Primary thresholds:")
print("1.0% to 10.0%, step 0.5 percentage points")
print()
print("Sensitivity thresholds:")
print("0.5% to 15.0%, step 0.5 percentage points")
print()
print("Bootstrap:")
print("2,000 hospital-cluster replicates; seed 20260723")
print()
print("Post-hoc optimal-threshold selection: PROHIBITED")

print("\n37A protocol SHA-256:")
print(protocol_sha)

print("\nSaved:")
print(protocol_path)
print(sha_path)

print(
    "\n37A PASS: DCA protocol locked before any DCA result was calculated or viewed."
)

In [ ]:

import os
import json
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

print("STARTING LOCKED DECISION CURVE ANALYSIS — CODE VERSION 37B")

# ============================================================
# 37B — DECISION CURVE ANALYSIS
#
# Upstream protocol:
#   37A locked BEFORE any DCA values were calculated or viewed.
#
# Primary comparison:
#   - XGBoost, fold-specific Platt-calibrated probabilities
#   - Parsimonious clinical baseline, fold-specific Platt-calibrated probabilities
#   - Treat-all
#   - Treat-none
#
# Locked primary thresholds:
#   0.010 to 0.100 by 0.005
#
# Locked sensitivity thresholds:
#   0.005 to 0.150 by 0.005
#
# Uncertainty:
#   2,000 paired hospital-cluster bootstrap replicates
#   seed = 20260723
#
# IMPORTANT:
# - No post-hoc "optimal" threshold is selected.
# - Full locked threshold ranges are reported.
# - CIs are pointwise percentile 95% intervals and are NOT
#   multiplicity-adjusted.
# - No predictive model is fitted, retuned, or recalibrated.
# - No patient-level file is written to Drive.
# - BigQuery is read only; no DML.
# ============================================================

EXPECTED_37A_PROTOCOL_SHA = (
    "e0f574c228be1d76e75669e7e446df6d"
    "9461dee11883acf5e4b22735dcffcbfb"
)

EXPECTED_35E_TEMPORAL_AUDIT_SHA = (
    "7230605e14a2314ec1724f48c61e42d2"
    "07f96c70ae2d4fe480bf02145ad24544"
)

EXPECTED_CODE36_MANIFEST_SHA = (
    "f31ad0df192bc0872d1626771a773c50"
    "b7fff798534751144d9a060f82d5f718"
)

EXPECTED_ROWS = 58491
EXPECTED_EVENTS = 3032
EXPECTED_HOSPITALS = 198
EXPECTED_OUTER_FOLDS = {1, 2, 3, 4, 5}

BOOTSTRAP_REPLICATES = 2000
BOOTSTRAP_SEED = 20260723

PRIMARY_THRESHOLDS = np.round(
    np.arange(0.010, 0.100 + 0.0001, 0.005),
    3,
)
SENSITIVITY_THRESHOLDS = np.round(
    np.arange(0.005, 0.150 + 0.0001, 0.005),
    3,
)

XGB_TABLE_SUFFIX = "model_xgb_outer_predictions_all5_v1"
CLINICAL_TABLE_SUFFIX = "model_clinical_lr_outer_predictions_all5_v2"

if "client" not in globals():
    raise RuntimeError(
        "Önce BigQuery client'ını oluşturan temel hücreyi çalıştır."
    )

if "MODEL_OUTPUT_DIR" not in globals():
    raise RuntimeError(
        "Önce MODEL_OUTPUT_DIR tanımlı temel hücreyi çalıştır."
    )

if "TARGET_DATASET" not in globals():
    raise RuntimeError(
        "Önce TARGET_DATASET tanımlı temel hücreyi çalıştır."
    )

if "BQ_LOCATION" not in globals():
    raise RuntimeError(
        "Önce BQ_LOCATION tanımlı temel hücreyi çalıştır."
    )

if "core_df_07B" not in globals():
    raise RuntimeError(
        "Önce core_df_07B oluşturan 07B hücresini çalıştır."
    )

# ------------------------------------------------------------
# 1. Guard locked upstream artifacts
# ------------------------------------------------------------

guards = [
    (
        "37A_locked_decision_curve_analysis_protocol_v1_SHA256.txt",
        EXPECTED_37A_PROTOCOL_SHA,
        "37A DCA protocol",
    ),
    (
        "35E_final_critical_temporal_leakage_adjudication_manifest_SHA256.txt",
        EXPECTED_35E_TEMPORAL_AUDIT_SHA,
        "35E temporal leakage audit",
    ),
    (
        "36D_model_vs_clinical_paired_comparison_manifest_SHA256.txt",
        EXPECTED_CODE36_MANIFEST_SHA,
        "Code36 paired model comparison",
    ),
]

for filename, expected_sha, label in guards:
    path = os.path.join(MODEL_OUTPUT_DIR, filename)

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    with open(path, "r", encoding="utf-8") as fh:
        observed_sha = fh.read().strip()

    if observed_sha != expected_sha:
        raise RuntimeError(
            f"{label} SHA mismatch: {observed_sha}"
        )

    print(f"{label} SHA guard: PASS")

# ------------------------------------------------------------
# 2. Verify threshold grids exactly match locked protocol
# ------------------------------------------------------------

protocol_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "37A_locked_decision_curve_analysis_protocol_v1.json",
)

with open(protocol_path, "r", encoding="utf-8") as fh:
    protocol = json.load(fh)

locked_primary = np.asarray(
    protocol["primary_threshold_range"]["thresholds"],
    dtype=float,
)
locked_sensitivity = np.asarray(
    protocol["sensitivity_threshold_range"]["thresholds"],
    dtype=float,
)

if not np.array_equal(
    PRIMARY_THRESHOLDS,
    locked_primary,
):
    raise RuntimeError(
        "Primary threshold grid does not match locked 37A protocol."
    )

if not np.array_equal(
    SENSITIVITY_THRESHOLDS,
    locked_sensitivity,
):
    raise RuntimeError(
        "Sensitivity threshold grid does not match locked 37A protocol."
    )

print("Locked primary threshold grid guard: PASS")
print("Locked sensitivity threshold grid guard: PASS")

# ------------------------------------------------------------
# 3. Load pooled Platt predictions from BigQuery
# ------------------------------------------------------------

def resolve_column(columns, exact_candidates, contains_candidates=None):
    lower_map = {
        str(c).lower(): c
        for c in columns
    }

    for candidate in exact_candidates:
        key = candidate.lower()
        if key in lower_map:
            return lower_map[key]

    if contains_candidates:
        matches = []
        for c in columns:
            low = str(c).lower()
            if any(
                token.lower() in low
                for token in contains_candidates
            ):
                matches.append(c)

        if len(matches) == 1:
            return matches[0]

    raise RuntimeError(
        "Could not resolve required column. "
        f"Candidates={exact_candidates}; columns={list(columns)}"
    )


def load_model_predictions(table_suffix, model_name):
    table_id = f"{TARGET_DATASET}.{table_suffix}"
    client.get_table(table_id)

    query = client.query(
        f"SELECT * FROM `{table_id}`",
        location=BQ_LOCATION,
    )

    try:
        raw = query.to_dataframe(
            create_bqstorage_client=True
        )
        load_method = "BigQuery Storage API"
    except Exception:
        raw = query.to_dataframe(
            create_bqstorage_client=False
        )
        load_method = "Standard BigQuery API"

    id_col = resolve_column(
        raw.columns,
        ["id_row"],
        ["id_row"],
    )
    fold_col = resolve_column(
        raw.columns,
        ["outer_fold"],
        ["outer_fold"],
    )
    label_col = resolve_column(
        raw.columns,
        ["label_stage23"],
        ["label_stage23"],
    )
    platt_col = resolve_column(
        raw.columns,
        ["prediction_platt"],
        ["prediction_platt", "platt"],
    )

    out = pd.DataFrame({
        "id_row": raw[id_col].astype(str),
        "outer_fold": pd.to_numeric(
            raw[fold_col],
            errors="raise",
        ).astype(np.int64),
        "label_stage23": pd.to_numeric(
            raw[label_col],
            errors="raise",
        ).astype(np.int64),
        f"{model_name}_platt": pd.to_numeric(
            raw[platt_col],
            errors="raise",
        ).astype(float),
    })

    if len(out) != EXPECTED_ROWS:
        raise RuntimeError(
            f"{model_name}: rows={len(out)}, expected={EXPECTED_ROWS}"
        )

    if out["id_row"].duplicated().any():
        raise RuntimeError(
            f"{model_name}: duplicate id_row detected."
        )

    if int(out["label_stage23"].sum()) != EXPECTED_EVENTS:
        raise RuntimeError(
            f"{model_name}: event-count mismatch."
        )

    if set(out["outer_fold"].unique()) != EXPECTED_OUTER_FOLDS:
        raise RuntimeError(
            f"{model_name}: outer folds are not exactly 1..5."
        )

    p = out[f"{model_name}_platt"].to_numpy(dtype=float)

    if not np.isfinite(p).all():
        raise RuntimeError(
            f"{model_name}: non-finite predictions."
        )

    if ((p < 0) | (p > 1)).any():
        raise RuntimeError(
            f"{model_name}: invalid probabilities."
        )

    return out, {
        "model": model_name,
        "table_id": table_id,
        "load_method": load_method,
        "rows": len(out),
        "events": int(out["label_stage23"].sum()),
    }


print("Loading XGBoost pooled Platt predictions...")
xgb, xgb_source = load_model_predictions(
    XGB_TABLE_SUFFIX,
    "xgboost",
)

print("Loading clinical-baseline pooled Platt predictions...")
clinical, clinical_source = load_model_predictions(
    CLINICAL_TABLE_SUFFIX,
    "clinical_baseline",
)

# ------------------------------------------------------------
# 4. Align against core patient/hospital map
# ------------------------------------------------------------

required_core = {
    "id_row",
    "outer_fold",
    "label_stage23",
    "group_hospital",
}

missing_core = required_core - set(core_df_07B.columns)

if missing_core:
    raise RuntimeError(
        "core_df_07B missing required columns: "
        + ", ".join(sorted(missing_core))
    )

core = core_df_07B[
    [
        "id_row",
        "outer_fold",
        "label_stage23",
        "group_hospital",
    ]
].copy()

core["id_row"] = core["id_row"].astype(str)
core["outer_fold"] = pd.to_numeric(
    core["outer_fold"],
    errors="raise",
).astype(np.int64)
core["label_stage23"] = pd.to_numeric(
    core["label_stage23"],
    errors="raise",
).astype(np.int64)
core["group_hospital"] = core[
    "group_hospital"
].astype(str)

aligned = (
    core
    .merge(
        xgb,
        on="id_row",
        how="inner",
        validate="one_to_one",
        suffixes=("", "_xgb"),
    )
    .merge(
        clinical,
        on="id_row",
        how="inner",
        validate="one_to_one",
        suffixes=("", "_clinical"),
    )
)

if len(aligned) != EXPECTED_ROWS:
    raise RuntimeError(
        f"Aligned rows={len(aligned)}, expected={EXPECTED_ROWS}"
    )

for suffix in ["_xgb", "_clinical"]:
    if (
        aligned["outer_fold"]
        != aligned[f"outer_fold{suffix}"]
    ).any():
        raise RuntimeError(
            f"Outer-fold mismatch for {suffix}."
        )

    if (
        aligned["label_stage23"]
        != aligned[f"label_stage23{suffix}"]
    ).any():
        raise RuntimeError(
            f"Outcome mismatch for {suffix}."
        )

aligned = aligned.drop(
    columns=[
        "outer_fold_xgb",
        "label_stage23_xgb",
        "outer_fold_clinical",
        "label_stage23_clinical",
    ]
)

if aligned["group_hospital"].nunique() != EXPECTED_HOSPITALS:
    raise RuntimeError(
        "Aligned hospital count is not 198."
    )

hospital_fold_n = aligned.groupby(
    "group_hospital"
)["outer_fold"].nunique()

if (hospital_fold_n != 1).any():
    raise RuntimeError(
        "Hospital cross-fold violation detected."
    )

print("Patient/hospital/outcome alignment: PASS")

# ------------------------------------------------------------
# 5. DCA helpers
# ------------------------------------------------------------

def net_benefit(y, p, threshold, sample_weight=None):
    y = np.asarray(y, dtype=np.int8)
    p = np.asarray(p, dtype=float)

    if sample_weight is None:
        w = np.ones(len(y), dtype=float)
    else:
        w = np.asarray(sample_weight, dtype=float)

    total_weight = float(w.sum())

    if total_weight <= 0:
        return np.nan

    predicted_positive = p >= threshold

    tp = float(
        w[
            predicted_positive & (y == 1)
        ].sum()
    )
    fp = float(
        w[
            predicted_positive & (y == 0)
        ].sum()
    )

    odds = threshold / (1.0 - threshold)

    return (
        tp / total_weight
        - fp / total_weight * odds
    )


def treat_all_net_benefit(y, threshold, sample_weight=None):
    y = np.asarray(y, dtype=np.int8)

    if sample_weight is None:
        w = np.ones(len(y), dtype=float)
    else:
        w = np.asarray(sample_weight, dtype=float)

    total_weight = float(w.sum())

    if total_weight <= 0:
        return np.nan

    prevalence = float(
        w[y == 1].sum()
        / total_weight
    )

    return (
        prevalence
        - (1.0 - prevalence)
        * threshold / (1.0 - threshold)
    )


def calculate_point_curve(thresholds, y, p_xgb, p_clinical):
    rows = []

    for pt in thresholds:
        nb_xgb = net_benefit(
            y,
            p_xgb,
            pt,
        )
        nb_clinical = net_benefit(
            y,
            p_clinical,
            pt,
        )
        nb_all = treat_all_net_benefit(
            y,
            pt,
        )
        nb_none = 0.0

        rows.append({
            "threshold": float(pt),
            "threshold_percent": float(pt * 100.0),
            "xgboost_net_benefit": nb_xgb,
            "clinical_baseline_net_benefit": nb_clinical,
            "treat_all_net_benefit": nb_all,
            "treat_none_net_benefit": nb_none,
            "xgboost_minus_clinical": (
                nb_xgb - nb_clinical
            ),
            "xgboost_minus_treat_all": (
                nb_xgb - nb_all
            ),
            "xgboost_minus_treat_none": (
                nb_xgb - nb_none
            ),
            "clinical_minus_treat_all": (
                nb_clinical - nb_all
            ),
            "clinical_minus_treat_none": (
                nb_clinical - nb_none
            ),
        })

    return pd.DataFrame(rows)


y = aligned["label_stage23"].to_numpy(
    dtype=np.int8
)
p_xgb = aligned["xgboost_platt"].to_numpy(
    dtype=float
)
p_clinical = aligned[
    "clinical_baseline_platt"
].to_numpy(dtype=float)

primary_point = calculate_point_curve(
    PRIMARY_THRESHOLDS,
    y,
    p_xgb,
    p_clinical,
)

sensitivity_point = calculate_point_curve(
    SENSITIVITY_THRESHOLDS,
    y,
    p_xgb,
    p_clinical,
)

# ------------------------------------------------------------
# 6. Paired hospital-cluster bootstrap
# ------------------------------------------------------------

print(
    f"\nRunning {BOOTSTRAP_REPLICATES:,} paired "
    "hospital-cluster DCA bootstrap replicates..."
)

hospital_cat = pd.Categorical(
    aligned["group_hospital"]
)
hospital_codes = hospital_cat.codes.astype(int)

if len(hospital_cat.categories) != EXPECTED_HOSPITALS:
    raise RuntimeError(
        "Bootstrap hospital count is not 198."
    )

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

bootstrap_rows = []

for b in range(BOOTSTRAP_REPLICATES):
    sampled_codes = rng.integers(
        0,
        EXPECTED_HOSPITALS,
        size=EXPECTED_HOSPITALS,
    )

    multiplicity = np.bincount(
        sampled_codes,
        minlength=EXPECTED_HOSPITALS,
    )

    weights = multiplicity[
        hospital_codes
    ].astype(float)

    if (
        weights[y == 1].sum() <= 0
        or weights[y == 0].sum() <= 0
    ):
        continue

    for pt in SENSITIVITY_THRESHOLDS:
        nb_xgb = net_benefit(
            y,
            p_xgb,
            pt,
            sample_weight=weights,
        )
        nb_clinical = net_benefit(
            y,
            p_clinical,
            pt,
            sample_weight=weights,
        )
        nb_all = treat_all_net_benefit(
            y,
            pt,
            sample_weight=weights,
        )

        bootstrap_rows.append({
            "bootstrap_replicate": b + 1,
            "threshold": float(pt),
            "xgboost_net_benefit": nb_xgb,
            "clinical_baseline_net_benefit": nb_clinical,
            "treat_all_net_benefit": nb_all,
            "treat_none_net_benefit": 0.0,
            "xgboost_minus_clinical": (
                nb_xgb - nb_clinical
            ),
            "xgboost_minus_treat_all": (
                nb_xgb - nb_all
            ),
            "xgboost_minus_treat_none": nb_xgb,
            "clinical_minus_treat_all": (
                nb_clinical - nb_all
            ),
            "clinical_minus_treat_none": nb_clinical,
        })

    if b == 0 or (b + 1) % 100 == 0:
        print(
            "  Completed DCA bootstrap replicate",
            b + 1,
            "/",
            BOOTSTRAP_REPLICATES,
        )

bootstrap_df = pd.DataFrame(
    bootstrap_rows
)

completed_replicates = bootstrap_df[
    "bootstrap_replicate"
].nunique()

if completed_replicates < BOOTSTRAP_REPLICATES * 0.99:
    raise RuntimeError(
        "Fewer than 99% of DCA bootstrap replicates completed."
    )

# ------------------------------------------------------------
# 7. Pointwise percentile CIs
# ------------------------------------------------------------

CI_COLUMNS = [
    "xgboost_net_benefit",
    "clinical_baseline_net_benefit",
    "treat_all_net_benefit",
    "xgboost_minus_clinical",
    "xgboost_minus_treat_all",
    "xgboost_minus_treat_none",
    "clinical_minus_treat_all",
    "clinical_minus_treat_none",
]

ci_rows = []

for pt, sub in bootstrap_df.groupby(
    "threshold",
    sort=True,
):
    row = {
        "threshold": float(pt),
        "threshold_percent": float(pt * 100.0),
        "bootstrap_replicates": int(
            sub["bootstrap_replicate"].nunique()
        ),
        "ci_type":
            "pointwise_percentile_95_not_multiplicity_adjusted",
    }

    for col in CI_COLUMNS:
        values = sub[col].to_numpy(dtype=float)

        row[f"{col}_ci95_lower"] = float(
            np.quantile(values, 0.025)
        )
        row[f"{col}_ci95_upper"] = float(
            np.quantile(values, 0.975)
        )

    ci_rows.append(row)

pointwise_ci = pd.DataFrame(ci_rows)

primary_result = primary_point.merge(
    pointwise_ci,
    on=[
        "threshold",
        "threshold_percent",
    ],
    how="left",
    validate="one_to_one",
)

sensitivity_result = sensitivity_point.merge(
    pointwise_ci,
    on=[
        "threshold",
        "threshold_percent",
    ],
    how="left",
    validate="one_to_one",
)

# ------------------------------------------------------------
# 8. Locked-range descriptive summaries
#    No optimal threshold selection.
# ------------------------------------------------------------

def summarize_curve(result, range_name):
    xgb_gt_clin = result[
        "xgboost_minus_clinical"
    ] > 0
    xgb_gt_all = result[
        "xgboost_minus_treat_all"
    ] > 0
    xgb_gt_none = result[
        "xgboost_minus_treat_none"
    ] > 0

    clinical_gt_all = result[
        "clinical_minus_treat_all"
    ] > 0
    clinical_gt_none = result[
        "clinical_minus_treat_none"
    ] > 0

    xgb_ci_gt_clin = (
        result[
            "xgboost_minus_clinical_ci95_lower"
        ] > 0
    )

    return pd.DataFrame([{
        "range_name": range_name,
        "threshold_count": len(result),
        "minimum_threshold": float(
            result["threshold"].min()
        ),
        "maximum_threshold": float(
            result["threshold"].max()
        ),
        "xgboost_nb_positive_thresholds": int(
            xgb_gt_none.sum()
        ),
        "clinical_nb_positive_thresholds": int(
            clinical_gt_none.sum()
        ),
        "xgboost_above_treat_all_thresholds": int(
            xgb_gt_all.sum()
        ),
        "clinical_above_treat_all_thresholds": int(
            clinical_gt_all.sum()
        ),
        "xgboost_above_clinical_thresholds": int(
            xgb_gt_clin.sum()
        ),
        "xgboost_minus_clinical_pointwise_ci_lower_gt_zero_thresholds":
            int(xgb_ci_gt_clin.sum()),
        "optimal_threshold_selected": False,
    }])


primary_summary = summarize_curve(
    primary_result,
    "primary_1_to_10_percent",
)

sensitivity_summary = summarize_curve(
    sensitivity_result,
    "sensitivity_0_5_to_15_percent",
)

range_summary = pd.concat(
    [
        primary_summary,
        sensitivity_summary,
    ],
    ignore_index=True,
)

# ------------------------------------------------------------
# 9. Figures
# ------------------------------------------------------------

primary_figure_png = os.path.join(
    MODEL_OUTPUT_DIR,
    "37B_primary_decision_curve_1_to_10_percent.png",
)
sensitivity_figure_png = os.path.join(
    MODEL_OUTPUT_DIR,
    "37B_sensitivity_decision_curve_0_5_to_15_percent.png",
)

fig = plt.figure(figsize=(8.5, 6.0))
ax = fig.add_subplot(111)

ax.plot(
    primary_result["threshold_percent"],
    primary_result["xgboost_net_benefit"],
    label="XGBoost",
)
ax.plot(
    primary_result["threshold_percent"],
    primary_result["clinical_baseline_net_benefit"],
    label="Parsimonious clinical baseline",
)
ax.plot(
    primary_result["threshold_percent"],
    primary_result["treat_all_net_benefit"],
    label="Treat all",
)
ax.plot(
    primary_result["threshold_percent"],
    primary_result["treat_none_net_benefit"],
    label="Treat none",
)

ax.set_xlabel("Threshold probability (%)")
ax.set_ylabel("Net benefit")
ax.set_title("Decision Curve Analysis: Locked Primary Threshold Range")
ax.legend()
ax.grid(True, alpha=0.2)
fig.tight_layout()
fig.savefig(
    primary_figure_png,
    dpi=300,
    bbox_inches="tight",
)
plt.close(fig)

fig = plt.figure(figsize=(8.5, 6.0))
ax = fig.add_subplot(111)

ax.plot(
    sensitivity_result["threshold_percent"],
    sensitivity_result["xgboost_net_benefit"],
    label="XGBoost",
)
ax.plot(
    sensitivity_result["threshold_percent"],
    sensitivity_result["clinical_baseline_net_benefit"],
    label="Parsimonious clinical baseline",
)
ax.plot(
    sensitivity_result["threshold_percent"],
    sensitivity_result["treat_all_net_benefit"],
    label="Treat all",
)
ax.plot(
    sensitivity_result["threshold_percent"],
    sensitivity_result["treat_none_net_benefit"],
    label="Treat none",
)

ax.set_xlabel("Threshold probability (%)")
ax.set_ylabel("Net benefit")
ax.set_title("Decision Curve Analysis: Locked Sensitivity Threshold Range")
ax.legend()
ax.grid(True, alpha=0.2)
fig.tight_layout()
fig.savefig(
    sensitivity_figure_png,
    dpi=300,
    bbox_inches="tight",
)
plt.close(fig)

# ------------------------------------------------------------
# 10. Save aggregate outputs only
# ------------------------------------------------------------

paths = {
    "source_audit": os.path.join(
        MODEL_OUTPUT_DIR,
        "37B_DCA_source_prediction_audit.csv",
    ),
    "alignment": os.path.join(
        MODEL_OUTPUT_DIR,
        "37B_DCA_alignment_integrity.csv",
    ),
    "primary_curve": os.path.join(
        MODEL_OUTPUT_DIR,
        "37B_DCA_primary_1_to_10_percent.csv",
    ),
    "sensitivity_curve": os.path.join(
        MODEL_OUTPUT_DIR,
        "37B_DCA_sensitivity_0_5_to_15_percent.csv",
    ),
    "pointwise_ci": os.path.join(
        MODEL_OUTPUT_DIR,
        "37B_DCA_pointwise_hospital_bootstrap_95CI.csv",
    ),
    "range_summary": os.path.join(
        MODEL_OUTPUT_DIR,
        "37B_DCA_locked_range_summary.csv",
    ),
    "primary_figure_png": primary_figure_png,
    "sensitivity_figure_png": sensitivity_figure_png,
    "manifest": os.path.join(
        MODEL_OUTPUT_DIR,
        "37B_decision_curve_analysis_manifest.json",
    ),
    "manifest_sha": os.path.join(
        MODEL_OUTPUT_DIR,
        "37B_decision_curve_analysis_manifest_SHA256.txt",
    ),
}

pd.DataFrame(
    [xgb_source, clinical_source]
).to_csv(
    paths["source_audit"],
    index=False,
)

alignment_integrity = pd.DataFrame([{
    "patients": len(aligned),
    "distinct_patients": aligned[
        "id_row"
    ].nunique(),
    "hospitals": aligned[
        "group_hospital"
    ].nunique(),
    "outer_folds": aligned[
        "outer_fold"
    ].nunique(),
    "events": int(
        aligned["label_stage23"].sum()
    ),
    "nonevents": int(
        len(aligned)
        - aligned["label_stage23"].sum()
    ),
    "hospital_cross_fold_violations": int(
        (hospital_fold_n != 1).sum()
    ),
}])

alignment_integrity.to_csv(
    paths["alignment"],
    index=False,
)

primary_result.to_csv(
    paths["primary_curve"],
    index=False,
)
sensitivity_result.to_csv(
    paths["sensitivity_curve"],
    index=False,
)
pointwise_ci.to_csv(
    paths["pointwise_ci"],
    index=False,
)
range_summary.to_csv(
    paths["range_summary"],
    index=False,
)

manifest = {
    "analysis_version": "37B",
    "analysis_type":
        "decision_curve_analysis_locked_thresholds",
    "protocol_sha256": EXPECTED_37A_PROTOCOL_SHA,
    "critical_temporal_audit_sha256":
        EXPECTED_35E_TEMPORAL_AUDIT_SHA,
    "code36_manifest_sha256":
        EXPECTED_CODE36_MANIFEST_SHA,
    "patients": EXPECTED_ROWS,
    "hospitals": EXPECTED_HOSPITALS,
    "events": EXPECTED_EVENTS,
    "primary_models": [
        "xgboost_platt_calibrated",
        "parsimonious_clinical_baseline_platt_calibrated",
        "treat_all",
        "treat_none",
    ],
    "primary_thresholds":
        PRIMARY_THRESHOLDS.tolist(),
    "sensitivity_thresholds":
        SENSITIVITY_THRESHOLDS.tolist(),
    "net_benefit_formula":
        "TP/N - FP/N * pt/(1-pt)",
    "bootstrap_replicates":
        BOOTSTRAP_REPLICATES,
    "completed_bootstrap_replicates":
        int(completed_replicates),
    "bootstrap_unit": "hospital",
    "paired_resampling": True,
    "bootstrap_seed": BOOTSTRAP_SEED,
    "confidence_intervals":
        "pointwise_percentile_95_not_multiplicity_adjusted",
    "optimal_threshold_selected": False,
    "predictive_model_fit_performed": False,
    "retuning_performed": False,
    "recalibration_performed": False,
    "patient_level_data_written_to_drive": False,
    "bigquery_dml_used": False,
    "outputs": paths,
}

manifest_text = json.dumps(
    manifest,
    indent=2,
    sort_keys=True,
)

with open(
    paths["manifest"],
    "w",
    encoding="utf-8",
) as fh:
    fh.write(manifest_text)

manifest_sha = hashlib.sha256(
    manifest_text.encode("utf-8")
).hexdigest()

with open(
    paths["manifest_sha"],
    "w",
    encoding="utf-8",
) as fh:
    fh.write(manifest_sha + "\n")

# ------------------------------------------------------------
# 11. Display compact results
# ------------------------------------------------------------

print("\n37B DCA ALIGNMENT INTEGRITY")
display(alignment_integrity)

print("\n37B PRIMARY DCA — 1% TO 10%")
display(
    primary_result[
        [
            "threshold_percent",
            "xgboost_net_benefit",
            "clinical_baseline_net_benefit",
            "treat_all_net_benefit",
            "treat_none_net_benefit",
            "xgboost_minus_clinical",
            "xgboost_minus_clinical_ci95_lower",
            "xgboost_minus_clinical_ci95_upper",
        ]
    ]
)

print("\n37B LOCKED-RANGE DCA SUMMARY")
display(range_summary)

print("\n37B manifest SHA-256:")
print(manifest_sha)

print("\nSaved figures:")
print(primary_figure_png)
print(sensitivity_figure_png)

print(
    "\n37B PASS: Decision curve analysis completed over the full "
    "pre-locked threshold ranges."
)
print(
    "No post-hoc optimal threshold was selected."
)
print(
    "Pointwise 95% bootstrap CIs are not multiplicity-adjusted."
)
print(
    "No predictive model was fitted, retuned, or recalibrated."
)
print(
    "No patient-level file was written to Google Drive."
)

In [ ]:
import os
import json
import hashlib

print("STARTING XGBOOST EXPLAINABILITY / TREESHAP PROTOCOL LOCK — CODE VERSION 38A")

# ============================================================
# 38A — XGBOOST EXPLAINABILITY / TREESHAP PROTOCOL LOCK
#
# This script ONLY locks the explainability analysis plan.
# It does NOT refit XGBoost and does NOT calculate/view SHAP values.
#
# Scientific purpose:
# - Explain the numerically strongest locked model (XGBoost).
# - Preserve outer-test separation for all primary explanations.
# - Treat SHAP as descriptive model attribution, not causality.
#
# Because model binaries were not persisted, the later execution step
# may reconstruct the exact fold-specific final XGBoost models using:
#   - the same locked outer training data,
#   - the same locked preprocessing,
#   - the same selected hyperparameters,
#   - the same XGBoost version/settings,
# and MUST reproduce the already-stored outer-test raw probabilities
# within a strict tolerance BEFORE any SHAP result is accepted.
#
# No retuning, no re-selection, no recalibration.
# ============================================================

EXPECTED_37B_MANIFEST_SHA = (
    "03a14b54c4fde8f23cfbc0ff4276f84c"
    "5443663cb0337d052fdf1989933238b4"
)

EXPECTED_35E_TEMPORAL_AUDIT_SHA = (
    "7230605e14a2314ec1724f48c61e42d2"
    "07f96c70ae2d4fe480bf02145ad24544"
)

EXPECTED_XGB_PROTOCOL_SHA = (
    "3434db5dd0b4b5145950fc07fba3d007"
    "829738d800b88ec40fbdb09879188264"
)

EXPECTED_CODE36_MANIFEST_SHA = (
    "f31ad0df192bc0872d1626771a773c50"
    "b7fff798534751144d9a060f82d5f718"
)

if "MODEL_OUTPUT_DIR" not in globals():
    raise RuntimeError(
        "Önce MODEL_OUTPUT_DIR tanımlı temel hücreyi çalıştır."
    )

# ------------------------------------------------------------
# 1. Upstream SHA guards
# ------------------------------------------------------------

guards = [
    (
        "37B_decision_curve_analysis_manifest_SHA256.txt",
        EXPECTED_37B_MANIFEST_SHA,
        "37B locked DCA",
    ),
    (
        "35E_final_critical_temporal_leakage_adjudication_manifest_SHA256.txt",
        EXPECTED_35E_TEMPORAL_AUDIT_SHA,
        "35E critical temporal leakage audit",
    ),
    (
        "13A_locked_xgboost_model_protocol_v1_SHA256.txt",
        EXPECTED_XGB_PROTOCOL_SHA,
        "XGBoost locked protocol",
    ),
    (
        "36D_model_vs_clinical_paired_comparison_manifest_SHA256.txt",
        EXPECTED_CODE36_MANIFEST_SHA,
        "Code36 five-model paired comparison",
    ),
]

for filename, expected_sha, label in guards:
    path = os.path.join(MODEL_OUTPUT_DIR, filename)

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    with open(path, "r", encoding="utf-8") as fh:
        observed_sha = fh.read().strip()

    if observed_sha != expected_sha:
        raise RuntimeError(
            f"{label} SHA mismatch: {observed_sha}"
        )

    print(f"{label} SHA guard: PASS")

# ------------------------------------------------------------
# 2. Lock explainability protocol
# ------------------------------------------------------------

protocol = {
    "analysis_version": "38A",
    "analysis_type": "locked_xgboost_treeshap_explainability_protocol",
    "timing": "locked_before_any_shap_values_or_explainability_figures_are_viewed",

    "target_model": {
        "model": "xgboost",
        "role": "numerically strongest locked model",
        "model_set_provenance": "original_model_set",
        "performance_probability_type": "fold_specific_platt_calibrated",
        "explanation_target": "uncalibrated_xgboost_raw_margin_log_odds",
        "important_note": (
            "Performance reporting remains based on fold-specific Platt-calibrated "
            "probabilities, but TreeSHAP explains the underlying XGBoost model score. "
            "The Platt calibration layer is not assigned feature-level SHAP values."
        ),
    },

    "locked_selected_candidates_by_outer_fold": {
        "1": "XGB04",
        "2": "XGB04",
        "3": "XGB04",
        "4": "XGB04",
        "5": "XGB06",
    },

    "analysis_population": {
        "primary_explanation_set": "outer_test_patients_only",
        "patients_total": 58491,
        "hospitals_total": 198,
        "outer_folds": 5,
        "pooling": (
            "Each patient is explained only by the fold-specific final XGBoost model "
            "for which that patient's hospital belonged to the untouched outer test fold."
        ),
    },

    "model_reconstruction_rule": {
        "needed_because": "final trained XGBoost binary model files were not persisted",
        "allowed_action": (
            "Refit each fold-specific final XGBoost model using the already-locked "
            "selected candidate, the same outer-training data, preprocessing, software "
            "version, random seed, and fixed hyperparameters."
        ),
        "prohibited_actions": [
            "hyperparameter retuning",
            "candidate reselection",
            "early stopping",
            "class weighting",
            "SMOTE or resampling",
            "recalibration",
            "test-result-driven changes",
        ],
        "mandatory_reproduction_check": {
            "comparison_target":
                "stored secure outer-test raw XGBoost probabilities",
            "maximum_absolute_probability_difference_tolerance": 1e-8,
            "mean_absolute_probability_difference_tolerance": 1e-10,
            "rule": (
                "If either tolerance is exceeded in any outer fold, stop and do not "
                "calculate or report SHAP values until the reconstruction discrepancy "
                "is resolved."
            ),
        },
    },

    "shap_method": {
        "method": "native_xgboost_tree_shap",
        "implementation": "Booster.predict(pred_contribs=True)",
        "output_scale": "raw_margin_log_odds",
        "additivity_check": True,
        "expected_value_included": True,
        "causal_interpretation": False,
    },

    "feature_attribution_levels": {
        "transformed_feature_level": {
            "purpose": "transparent audit and signed SHAP visualization",
            "includes": [
                "numeric feature values",
                "numeric missingness indicators",
                "one-hot encoded categorical levels",
            ],
        },
        "source_predictor_level": {
            "purpose": "primary global importance reporting",
            "aggregation_rule": (
                "Sum absolute SHAP contributions of all transformed columns originating "
                "from the same locked source predictor for each patient, then average "
                "across outer-test patients."
            ),
            "source_predictor_count": 159,
            "numeric_missing_indicator_rule": (
                "The numeric value column and its missingness indicator are grouped "
                "under the same source predictor."
            ),
            "categorical_rule": (
                "All one-hot levels of a categorical predictor are grouped under the "
                "same source predictor."
            ),
        },
    },

    "primary_global_outputs": {
        "source_predictor_importance": "mean absolute grouped SHAP",
        "top_predictors_to_display": 20,
        "ranking_rule": "descending pooled mean absolute grouped SHAP",
        "no_manual_feature_cherry_picking": True,
    },

    "directionality_outputs": {
        "primary_signed_plot": (
            "TreeSHAP beeswarm at transformed-feature level for the deterministic "
            "top 20 transformed features by pooled mean absolute SHAP."
        ),
        "interpretation": (
            "Positive SHAP values increase the model's raw log-odds score; "
            "negative values decrease it. This is model contribution, not causal effect."
        ),
    },

    "predefined_dependence_outputs": {
        "data_driven_set": (
            "Top 5 source predictors by pooled mean absolute grouped SHAP; "
            "selection rule fixed before SHAP values are viewed."
        ),
        "kidney_variables_of_a_priori_interest": [
            "x_reference_creatinine",
            "x_stage1_at_landmark",
            "x_lab_creatinine_last",
        ],
        "rule": (
            "All three a-priori kidney variables are reported regardless of their rank; "
            "top-5 plots are generated by the deterministic ranking rule."
        ),
    },

    "stability_analysis": {
        "fold_specific_mean_absolute_shap": True,
        "fold_specific_source_predictor_ranks": True,
        "pairwise_fold_rank_correlation": "Spearman",
        "top20_presence_frequency_across_folds": True,
        "purpose": (
            "Assess whether global attribution patterns are stable across unseen hospital groups."
        ),
    },

    "reporting_and_claim_rules": [
        "Use association/contribution language, never causal language.",
        "Do not call a high-SHAP feature an independent risk factor.",
        "A SHAP sign is conditional on the fitted model and correlated predictor set.",
        "Do not interpret missingness-indicator contribution as biological effect.",
        "Do not choose features for discussion solely because they support a preferred narrative.",
        "Report that fold 5 used XGB06 whereas folds 1-4 used XGB04.",
    ],

    "privacy_and_storage": {
        "patient_level_shap_written_to_drive": False,
        "patient_level_shap_written_to_bigquery": False,
        "patient_level_shap_retained": "RAM_only_during_execution",
        "aggregate_tables_and_figures_may_be_written_to_drive": True,
    },

    "upstream_sha256": {
        "37B_dca_manifest": EXPECTED_37B_MANIFEST_SHA,
        "35E_temporal_audit": EXPECTED_35E_TEMPORAL_AUDIT_SHA,
        "xgboost_protocol": EXPECTED_XGB_PROTOCOL_SHA,
        "36_model_comparison_manifest": EXPECTED_CODE36_MANIFEST_SHA,
    },

    "shap_result_calculated": False,
    "shap_result_viewed": False,
    "model_reconstruction_performed": False,
    "retuning_performed": False,
    "candidate_reselection_performed": False,
    "recalibration_performed": False,
    "bigquery_dml_used": False,
    "patient_level_data_written_to_drive": False,
}

protocol_text = json.dumps(
    protocol,
    indent=2,
    sort_keys=True,
)

protocol_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "38A_locked_xgboost_treeshap_explainability_protocol_v1.json",
)

sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "38A_locked_xgboost_treeshap_explainability_protocol_v1_SHA256.txt",
)

with open(protocol_path, "w", encoding="utf-8") as fh:
    fh.write(protocol_text)

protocol_sha = hashlib.sha256(
    protocol_text.encode("utf-8")
).hexdigest()

with open(sha_path, "w", encoding="utf-8") as fh:
    fh.write(protocol_sha + "\n")

print("\n38A LOCKED EXPLAINABILITY PROTOCOL")
print("Target model: XGBoost")
print("Primary explanation population: outer-test patients only")
print("Fold 1-4 selected candidate: XGB04")
print("Fold 5 selected candidate: XGB06")
print("Method: native TreeSHAP on raw XGBoost margin/log-odds")
print("Primary global importance: grouped source-predictor mean(|SHAP|)")
print("Top global predictors displayed: 20")
print("A-priori kidney variables always reported:")
print("- x_reference_creatinine")
print("- x_stage1_at_landmark")
print("- x_lab_creatinine_last")
print("Patient-level SHAP export to Drive/BigQuery: PROHIBITED")
print("Causal interpretation: PROHIBITED")
print("Retuning/reselection/recalibration: PROHIBITED")

print("\n38A protocol SHA-256:")
print(protocol_sha)

print("\nSaved:")
print(protocol_path)
print(sha_path)

print(
    "\n38A PASS: XGBoost explainability protocol locked before any SHAP "
    "value was calculated or viewed."
)

In [ ]:
import os
import sys
import json
import time
import hashlib

import numpy as np
import pandas as pd
import sklearn
import xgboost
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

from IPython.display import display

print("STARTING VERIFIED XGBOOST MODEL RECONSTRUCTION AUDIT — CODE VERSION 38B")

# ============================================================
# 38B — VERIFIED XGBOOST MODEL RECONSTRUCTION AUDIT
#
# IMPORTANT:
# - This step performs NO SHAP calculation.
# - It reconstructs each already-locked final XGBoost model exactly.
# - It compares reconstructed outer-test raw probabilities against
#   the already-stored secure predictions.
# - SHAP is allowed only after all five folds reproduce within the
#   strict tolerances locked in 38A.
#
# Allowed:
#   exact refit of the already-selected final fold model
#
# Prohibited:
#   hyperparameter tuning
#   model reselection
#   early stopping
#   class weighting
#   SMOTE/resampling
#   recalibration
#   test-driven changes
#
# Outputs to Drive:
#   aggregate reconstruction audit
#   verified XGBoost model JSON files
#   verified preprocessing objects
#   hashes / manifest
#
# No patient-level predictions are written to Drive.
# ============================================================

EXPECTED_38A_PROTOCOL_SHA = (
    "06fe49d742e43aedfb518e805f12949f5"
    "2cc926e880a57683144dd6915e30a49"
)

EXPECTED_XGB_PROTOCOL_SHA = (
    "3434db5dd0b4b5145950fc07fba3d007"
    "829738d800b88ec40fbdb09879188264"
)

EXPECTED_ROWS = 58491
EXPECTED_EVENTS = 3032
EXPECTED_HOSPITALS = 198

EXPECTED_SELECTED = {
    1: "XGB04",
    2: "XGB04",
    3: "XGB04",
    4: "XGB04",
    5: "XGB06",
}

EXPECTED_SPLITS = {
    1: {
        "training_rows": 46803,
        "test_rows": 11688,
        "training_hospitals": 158,
        "test_hospitals": 40,
        "training_events": 2426,
        "test_events": 606,
    },
    2: {
        "training_rows": 46800,
        "test_rows": 11691,
        "training_hospitals": 158,
        "test_hospitals": 40,
        "training_events": 2426,
        "test_events": 606,
    },
    3: {
        "training_rows": 46755,
        "test_rows": 11736,
        "training_hospitals": 158,
        "test_hospitals": 40,
        "training_events": 2424,
        "test_events": 608,
    },
    4: {
        "training_rows": 46803,
        "test_rows": 11688,
        "training_hospitals": 159,
        "test_hospitals": 39,
        "training_events": 2426,
        "test_events": 606,
    },
    5: {
        "training_rows": 46803,
        "test_rows": 11688,
        "training_hospitals": 159,
        "test_hospitals": 39,
        "training_events": 2426,
        "test_events": 606,
    },
}

MAX_ABS_PROB_DIFF_TOL = 1e-8
MEAN_ABS_PROB_DIFF_TOL = 1e-10
MODEL_RANDOM_SEED = 20260721

SELECTION_FILES = {
    1: "13C_xgboost_selected_model_outer1.csv",
    2: "14B_xgboost_selected_model_outer2.csv",
    3: "15B_xgboost_selected_model_outer3.csv",
    4: "16B_xgboost_selected_model_outer4.csv",
    5: "17B_xgboost_selected_model_outer5.csv",
}

POOLED_PREDICTION_TABLE = (
    "model_xgb_outer_predictions_all5_v1"
)

CANDIDATES = {
    "XGB04": {
        "candidate_id": "XGB04",
        "n_estimators": 350,
        "max_depth": 4,
        "learning_rate": 0.03,
        "min_child_weight": 10.0,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "gamma": 0.10,
        "reg_alpha": 0.10,
        "reg_lambda": 10.0,
    },
    "XGB06": {
        "candidate_id": "XGB06",
        "n_estimators": 450,
        "max_depth": 4,
        "learning_rate": 0.02,
        "min_child_weight": 15.0,
        "subsample": 0.90,
        "colsample_bytree": 0.80,
        "gamma": 0.20,
        "reg_alpha": 0.50,
        "reg_lambda": 15.0,
    },
}

# ------------------------------------------------------------
# 1. Runtime requirements
# ------------------------------------------------------------

required_runtime = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_runtime = [
    name for name in required_runtime
    if name not in globals()
]

if missing_runtime:
    raise RuntimeError(
        "Eksik çalışma nesneleri: "
        + ", ".join(missing_runtime)
        + ". Önce 07A/07B temel hücrelerini çalıştır."
    )

if len(core_df_07B) != EXPECTED_ROWS:
    raise RuntimeError(
        f"Expected {EXPECTED_ROWS} rows; found {len(core_df_07B)}."
    )

if int(core_df_07B["label_stage23"].sum()) != EXPECTED_EVENTS:
    raise RuntimeError(
        "Cohort event count is not 3,032."
    )

if core_df_07B["group_hospital"].astype(str).nunique() != EXPECTED_HOSPITALS:
    raise RuntimeError(
        "Cohort hospital count is not 198."
    )

if len(predictor_columns_07B) != 159:
    raise RuntimeError(
        "Expected 159 core predictors."
    )

if len(numeric_columns_07B) != 156:
    raise RuntimeError(
        "Expected 156 numeric predictors."
    )

if len(categorical_columns_07B) != 3:
    raise RuntimeError(
        "Expected 3 categorical predictors."
    )

# Keep exact software environment locked to original development.
expected_versions = {
    "python_major_minor": "3.12",
    "sklearn": "1.6.1",
    "xgboost": "3.3.0",
}

observed_python_major_minor = (
    f"{sys.version_info.major}.{sys.version_info.minor}"
)

if observed_python_major_minor != expected_versions["python_major_minor"]:
    raise RuntimeError(
        "Python major/minor differs from locked environment: "
        + observed_python_major_minor
    )

if sklearn.__version__ != expected_versions["sklearn"]:
    raise RuntimeError(
        "scikit-learn version differs from locked environment: "
        + sklearn.__version__
    )

if xgboost.__version__ != expected_versions["xgboost"]:
    raise RuntimeError(
        "XGBoost version differs from locked environment: "
        + xgboost.__version__
    )

print("Software version guards: PASS")

# ------------------------------------------------------------
# 2. Protocol SHA guards
# ------------------------------------------------------------

guard_files = [
    (
        "38A_locked_xgboost_treeshap_explainability_protocol_v1_SHA256.txt",
        EXPECTED_38A_PROTOCOL_SHA,
        "38A explainability protocol",
    ),
    (
        "13A_locked_xgboost_model_protocol_v1_SHA256.txt",
        EXPECTED_XGB_PROTOCOL_SHA,
        "XGBoost model protocol",
    ),
]

for filename, expected_sha, label in guard_files:
    path = os.path.join(
        MODEL_OUTPUT_DIR,
        filename,
    )

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    with open(path, "r", encoding="utf-8") as fh:
        observed_sha = fh.read().strip()

    if observed_sha != expected_sha:
        raise RuntimeError(
            f"{label} SHA mismatch: {observed_sha}"
        )

    print(f"{label} SHA guard: PASS")

# ------------------------------------------------------------
# 3. Verify selected candidates from locked selection files
# ------------------------------------------------------------

selection_audit_rows = []

for outer_fold, filename in SELECTION_FILES.items():
    path = os.path.join(
        MODEL_OUTPUT_DIR,
        filename,
    )

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    df = pd.read_csv(path)

    candidate_column = None
    for col in [
        "selected_candidate",
        "selected_candidate_id",
        "candidate_id",
    ]:
        if col in df.columns:
            candidate_column = col
            break

    if candidate_column is None:
        raise RuntimeError(
            f"Cannot resolve selected candidate column in {filename}."
        )

    if len(df) != 1:
        raise RuntimeError(
            f"{filename} must contain exactly one selected-model row."
        )

    observed_candidate = str(
        df.iloc[0][candidate_column]
    )

    expected_candidate = EXPECTED_SELECTED[
        outer_fold
    ]

    if observed_candidate != expected_candidate:
        raise RuntimeError(
            f"Outer fold {outer_fold}: selected candidate "
            f"{observed_candidate}, expected {expected_candidate}."
        )

    selection_audit_rows.append({
        "outer_fold": outer_fold,
        "selection_file": filename,
        "selected_candidate": observed_candidate,
        "expected_candidate": expected_candidate,
        "status": "PASS",
    })

selection_audit = pd.DataFrame(
    selection_audit_rows
)

print("Selected-candidate guards for all five folds: PASS")

# ------------------------------------------------------------
# 4. Exact original preprocessing / XGBoost constructors
# ------------------------------------------------------------

def make_preprocessor():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                numeric_columns_07B,
            ),
            (
                "categorical",
                categorical_pipeline,
                categorical_columns_07B,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_model(candidate):
    return XGBClassifier(
        n_estimators=int(
            candidate["n_estimators"]
        ),
        max_depth=int(
            candidate["max_depth"]
        ),
        learning_rate=float(
            candidate["learning_rate"]
        ),
        min_child_weight=float(
            candidate["min_child_weight"]
        ),
        subsample=float(
            candidate["subsample"]
        ),
        colsample_bytree=float(
            candidate["colsample_bytree"]
        ),
        gamma=float(
            candidate["gamma"]
        ),
        reg_alpha=float(
            candidate["reg_alpha"]
        ),
        reg_lambda=float(
            candidate["reg_lambda"]
        ),
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        max_bin=256,
        scale_pos_weight=1.0,
        importance_type="gain",
        random_state=MODEL_RANDOM_SEED,
        n_jobs=-1,
        verbosity=0,
    )

# ------------------------------------------------------------
# 5. Prepare the feature matrix exactly as in original XGB scripts
# ------------------------------------------------------------

X_all = core_df_07B[
    predictor_columns_07B
].copy()

for column in numeric_columns_07B:
    X_all[column] = pd.to_numeric(
        X_all[column],
        errors="coerce",
    ).astype("float64")

for column in categorical_columns_07B:
    category_series = X_all[
        column
    ].astype("object")

    X_all[column] = category_series.where(
        pd.notna(category_series),
        np.nan,
    )

meta = core_df_07B[
    [
        "id_row",
        "outer_fold",
        "group_hospital",
        "label_stage23",
    ]
].copy()

meta["id_row"] = meta["id_row"].astype(str)
meta["outer_fold"] = pd.to_numeric(
    meta["outer_fold"],
    errors="raise",
).astype(np.int64)
meta["group_hospital"] = meta[
    "group_hospital"
].astype(str)
meta["label_stage23"] = pd.to_numeric(
    meta["label_stage23"],
    errors="raise",
).astype(np.int64)

# ------------------------------------------------------------
# 6. Load only secure stored raw XGB predictions into RAM
# ------------------------------------------------------------

prediction_table_id = (
    f"{TARGET_DATASET}.{POOLED_PREDICTION_TABLE}"
)

client.get_table(
    prediction_table_id
)

stored = client.query(
    f"""
    SELECT
      id_row,
      outer_fold,
      label_stage23,
      prediction_raw
    FROM `{prediction_table_id}`
    """,
    location=BQ_LOCATION,
).to_dataframe()

stored["id_row"] = stored[
    "id_row"
].astype(str)
stored["outer_fold"] = pd.to_numeric(
    stored["outer_fold"],
    errors="raise",
).astype(np.int64)
stored["label_stage23"] = pd.to_numeric(
    stored["label_stage23"],
    errors="raise",
).astype(np.int64)
stored["prediction_raw"] = pd.to_numeric(
    stored["prediction_raw"],
    errors="raise",
).astype(float)

if len(stored) != EXPECTED_ROWS:
    raise RuntimeError(
        "Stored pooled XGB prediction table does not contain 58,491 rows."
    )

if stored["id_row"].duplicated().any():
    raise RuntimeError(
        "Duplicate patient rows in stored pooled XGB predictions."
    )

if int(stored["label_stage23"].sum()) != EXPECTED_EVENTS:
    raise RuntimeError(
        "Stored pooled XGB event count mismatch."
    )

print("Stored pooled XGBoost raw-prediction integrity: PASS")

# ------------------------------------------------------------
# 7. Reconstruct and verify all five final models
# ------------------------------------------------------------

verified_dir = os.path.join(
    MODEL_OUTPUT_DIR,
    "38B_verified_xgboost_models",
)

os.makedirs(
    verified_dir,
    exist_ok=True,
)

audit_rows = []
artifact_rows = []

for outer_fold in range(1, 6):
    print(
        f"\nReconstructing locked XGBoost outer fold {outer_fold}..."
    )

    training_mask = (
        meta["outer_fold"].to_numpy()
        != outer_fold
    )
    test_mask = (
        meta["outer_fold"].to_numpy()
        == outer_fold
    )

    X_train = (
        X_all.loc[training_mask]
        .reset_index(drop=True)
    )
    X_test = (
        X_all.loc[test_mask]
        .reset_index(drop=True)
    )

    train_meta = (
        meta.loc[
            training_mask,
            [
                "id_row",
                "group_hospital",
                "label_stage23",
            ],
        ]
        .reset_index(drop=True)
    )
    test_meta = (
        meta.loc[
            test_mask,
            [
                "id_row",
                "group_hospital",
                "label_stage23",
            ],
        ]
        .reset_index(drop=True)
    )

    expected_split = EXPECTED_SPLITS[
        outer_fold
    ]

    observed_split = {
        "training_rows": len(train_meta),
        "test_rows": len(test_meta),
        "training_hospitals": train_meta[
            "group_hospital"
        ].nunique(),
        "test_hospitals": test_meta[
            "group_hospital"
        ].nunique(),
        "training_events": int(
            train_meta["label_stage23"].sum()
        ),
        "test_events": int(
            test_meta["label_stage23"].sum()
        ),
    }

    if observed_split != expected_split:
        raise RuntimeError(
            f"Outer fold {outer_fold}: split mismatch. "
            f"Observed={observed_split}; expected={expected_split}"
        )

    if (
        set(train_meta["group_hospital"])
        & set(test_meta["group_hospital"])
    ):
        raise RuntimeError(
            f"Outer fold {outer_fold}: hospital overlap."
        )

    candidate_id = EXPECTED_SELECTED[
        outer_fold
    ]
    candidate = CANDIDATES[
        candidate_id
    ]

    preprocessor = make_preprocessor()

    preprocessing_started = time.time()

    X_train_processed = (
        preprocessor.fit_transform(
            X_train
        )
    )
    X_test_processed = (
        preprocessor.transform(
            X_test
        )
    )

    preprocessing_seconds = (
        time.time()
        - preprocessing_started
    )

    processed_feature_names = (
        preprocessor.get_feature_names_out()
    )

    model = make_model(
        candidate
    )

    fit_started = time.time()

    model.fit(
        X_train_processed,
        train_meta[
            "label_stage23"
        ].to_numpy(dtype=np.int8),
    )

    fit_seconds = (
        time.time()
        - fit_started
    )

    reconstructed_prob = (
        model.predict_proba(
            X_test_processed
        )[:, 1]
    )

    stored_fold = (
        stored.loc[
            stored[
                "outer_fold"
            ].eq(outer_fold),
            [
                "id_row",
                "label_stage23",
                "prediction_raw",
            ],
        ]
        .copy()
    )

    aligned_fold = (
        test_meta[
            [
                "id_row",
                "label_stage23",
            ]
        ]
        .merge(
            stored_fold,
            on="id_row",
            how="inner",
            validate="one_to_one",
            suffixes=(
                "_core",
                "_stored",
            ),
        )
    )

    if len(aligned_fold) != len(test_meta):
        raise RuntimeError(
            f"Outer fold {outer_fold}: stored prediction alignment failed."
        )

    if (
        aligned_fold[
            "label_stage23_core"
        ].to_numpy()
        != aligned_fold[
            "label_stage23_stored"
        ].to_numpy()
    ).any():
        raise RuntimeError(
            f"Outer fold {outer_fold}: label mismatch."
        )

    # Merge reconstruction by exact test-meta id order to avoid any
    # dependence on BigQuery row ordering.
    reconstructed_df = pd.DataFrame({
        "id_row": test_meta[
            "id_row"
        ].to_numpy(),
        "reconstructed_probability":
            reconstructed_prob.astype(float),
    })

    comparison = (
        stored_fold[
            [
                "id_row",
                "prediction_raw",
            ]
        ]
        .merge(
            reconstructed_df,
            on="id_row",
            how="inner",
            validate="one_to_one",
        )
    )

    absolute_difference = np.abs(
        comparison[
            "reconstructed_probability"
        ].to_numpy(dtype=float)
        - comparison[
            "prediction_raw"
        ].to_numpy(dtype=float)
    )

    max_abs_diff = float(
        absolute_difference.max()
    )
    mean_abs_diff = float(
        absolute_difference.mean()
    )

    reproduction_pass = (
        max_abs_diff
        <= MAX_ABS_PROB_DIFF_TOL
        and mean_abs_diff
        <= MEAN_ABS_PROB_DIFF_TOL
    )

    print(
        "Candidate:",
        candidate_id,
        "| processed columns:",
        X_train_processed.shape[1],
    )
    print(
        "Max abs probability difference:",
        f"{max_abs_diff:.12g}",
    )
    print(
        "Mean abs probability difference:",
        f"{mean_abs_diff:.12g}",
    )

    if not reproduction_pass:
        raise RuntimeError(
            f"Outer fold {outer_fold}: exact reconstruction FAILED. "
            f"max_abs_diff={max_abs_diff:.12g}; "
            f"mean_abs_diff={mean_abs_diff:.12g}. "
            "Do NOT proceed to SHAP."
        )

    print(
        f"Outer fold {outer_fold} probability reproduction: PASS"
    )

    # --------------------------------------------------------
    # Persist only verified model/preprocessor artifacts.
    # No patient-level predictions are persisted.
    # --------------------------------------------------------

    model_path = os.path.join(
        verified_dir,
        f"38B_verified_xgboost_outer{outer_fold}_{candidate_id}.json",
    )

    preprocessor_path = os.path.join(
        verified_dir,
        f"38B_verified_preprocessor_outer{outer_fold}.joblib",
    )

    feature_names_path = os.path.join(
        verified_dir,
        f"38B_verified_processed_feature_names_outer{outer_fold}.json",
    )

    model.save_model(
        model_path
    )

    joblib.dump(
        preprocessor,
        preprocessor_path,
        compress=3,
    )

    with open(
        feature_names_path,
        "w",
        encoding="utf-8",
    ) as fh:
        json.dump(
            [
                str(x)
                for x in processed_feature_names
            ],
            fh,
            indent=2,
            ensure_ascii=False,
        )

    def file_sha256(path):
        digest = hashlib.sha256()
        with open(path, "rb") as fh:
            for block in iter(
                lambda: fh.read(1024 * 1024),
                b"",
            ):
                digest.update(block)
        return digest.hexdigest()

    model_sha = file_sha256(
        model_path
    )
    preprocessor_sha = file_sha256(
        preprocessor_path
    )
    feature_names_sha = file_sha256(
        feature_names_path
    )

    audit_rows.append({
        "outer_fold": outer_fold,
        "selected_candidate": candidate_id,
        "training_rows": observed_split[
            "training_rows"
        ],
        "test_rows": observed_split[
            "test_rows"
        ],
        "training_hospitals": observed_split[
            "training_hospitals"
        ],
        "test_hospitals": observed_split[
            "test_hospitals"
        ],
        "training_events": observed_split[
            "training_events"
        ],
        "test_events": observed_split[
            "test_events"
        ],
        "processed_columns": int(
            X_train_processed.shape[1]
        ),
        "preprocessing_seconds": float(
            preprocessing_seconds
        ),
        "fit_seconds": float(
            fit_seconds
        ),
        "maximum_absolute_probability_difference":
            max_abs_diff,
        "mean_absolute_probability_difference":
            mean_abs_diff,
        "maximum_allowed_absolute_difference":
            MAX_ABS_PROB_DIFF_TOL,
        "maximum_allowed_mean_difference":
            MEAN_ABS_PROB_DIFF_TOL,
        "reproduction_status": "PASS",
    })

    artifact_rows.extend([
        {
            "outer_fold": outer_fold,
            "artifact_type": "xgboost_model_json",
            "path": model_path,
            "sha256": model_sha,
        },
        {
            "outer_fold": outer_fold,
            "artifact_type": "preprocessor_joblib",
            "path": preprocessor_path,
            "sha256": preprocessor_sha,
        },
        {
            "outer_fold": outer_fold,
            "artifact_type": "processed_feature_names_json",
            "path": feature_names_path,
            "sha256": feature_names_sha,
        },
    ])

    # Release patient-level matrices/predictions from this fold.
    del (
        X_train,
        X_test,
        train_meta,
        test_meta,
        X_train_processed,
        X_test_processed,
        reconstructed_prob,
        stored_fold,
        aligned_fold,
        reconstructed_df,
        comparison,
        absolute_difference,
        model,
        preprocessor,
    )

# ------------------------------------------------------------
# 8. Final aggregate lock
# ------------------------------------------------------------

reconstruction_audit = pd.DataFrame(
    audit_rows
)

artifact_manifest = pd.DataFrame(
    artifact_rows
)

if len(reconstruction_audit) != 5:
    raise RuntimeError(
        "Expected five reconstruction audit rows."
    )

if not (
    reconstruction_audit[
        "reproduction_status"
    ].eq("PASS")
).all():
    raise RuntimeError(
        "Not all five XGBoost folds reproduced."
    )

# ------------------------------------------------------------
# 9. Save aggregate audit + manifest
# ------------------------------------------------------------

reconstruction_csv = os.path.join(
    MODEL_OUTPUT_DIR,
    "38B_xgboost_verified_reconstruction_audit.csv",
)

selection_csv = os.path.join(
    MODEL_OUTPUT_DIR,
    "38B_xgboost_selected_candidate_audit.csv",
)

artifact_csv = os.path.join(
    MODEL_OUTPUT_DIR,
    "38B_verified_xgboost_artifact_manifest.csv",
)

manifest_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "38B_verified_xgboost_reconstruction_manifest.json",
)

manifest_sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "38B_verified_xgboost_reconstruction_manifest_SHA256.txt",
)

reconstruction_audit.to_csv(
    reconstruction_csv,
    index=False,
)

selection_audit.to_csv(
    selection_csv,
    index=False,
)

artifact_manifest.to_csv(
    artifact_csv,
    index=False,
)

manifest = {
    "analysis_version": "38B",
    "analysis_type":
        "verified_exact_xgboost_fold_model_reconstruction",
    "source_38A_protocol_sha256":
        EXPECTED_38A_PROTOCOL_SHA,
    "source_xgboost_protocol_sha256":
        EXPECTED_XGB_PROTOCOL_SHA,
    "patients_total": EXPECTED_ROWS,
    "hospitals_total": EXPECTED_HOSPITALS,
    "events_total": EXPECTED_EVENTS,
    "selected_candidates_by_outer_fold":
        EXPECTED_SELECTED,
    "software_versions": {
        "python_major_minor":
            observed_python_major_minor,
        "scikit_learn":
            sklearn.__version__,
        "xgboost":
            xgboost.__version__,
    },
    "reproduction_tolerances": {
        "maximum_absolute_probability_difference":
            MAX_ABS_PROB_DIFF_TOL,
        "mean_absolute_probability_difference":
            MEAN_ABS_PROB_DIFF_TOL,
    },
    "all_five_folds_reproduced": True,
    "shap_calculated": False,
    "shap_viewed": False,
    "retuning_performed": False,
    "candidate_reselection_performed": False,
    "recalibration_performed": False,
    "patient_level_predictions_written_to_drive":
        False,
    "patient_level_shap_written_to_drive":
        False,
    "bigquery_dml_used": False,
    "verified_artifact_directory":
        verified_dir,
    "outputs": {
        "reconstruction_audit_csv":
            reconstruction_csv,
        "selection_audit_csv":
            selection_csv,
        "verified_artifact_manifest_csv":
            artifact_csv,
    },
}

manifest_text = json.dumps(
    manifest,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(
        manifest_text
    )

manifest_sha = hashlib.sha256(
    manifest_text.encode("utf-8")
).hexdigest()

with open(
    manifest_sha_path,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(
        manifest_sha + "\n"
    )

# ------------------------------------------------------------
# 10. Display
# ------------------------------------------------------------

print("\n38B VERIFIED XGBOOST RECONSTRUCTION AUDIT")
display(
    reconstruction_audit
)

print("\n38B VERIFIED ARTIFACT MANIFEST")
display(
    artifact_manifest
)

print("\n38B manifest SHA-256:")
print(
    manifest_sha
)

print("\nVerified model directory:")
print(
    verified_dir
)

print(
    "\n38B PASS: All five locked XGBoost final models were "
    "reconstructed and reproduced their stored outer-test "
    "raw probabilities within the pre-locked strict tolerance."
)
print(
    "No SHAP value was calculated or viewed in Code 38B."
)
print(
    "TreeSHAP may proceed only from these verified artifacts."
)

In [ ]:
import os
import json
import math
import hashlib
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import xgboost as xgb

from IPython.display import display

print("STARTING VERIFIED OUTER-TEST TREESHAP ANALYSIS — CODE VERSION 38C")

# ============================================================
# 38C — VERIFIED OUTER-TEST TREESHAP ANALYSIS
#
# Upstream requirements:
#   - 38A explainability protocol locked before SHAP inspection
#   - 38B reconstructed all five locked XGBoost models and reproduced
#     stored outer-test raw probabilities EXACTLY (max/mean diff = 0)
#
# Primary scientific rules:
#   - explain OUTER-TEST patients only
#   - each patient explained by the model for their own outer test fold
#   - folds 1-4 use XGB04; fold 5 uses XGB06
#   - native XGBoost TreeSHAP, raw margin/log-odds scale
#   - primary global importance = source-predictor grouped mean(|SHAP|)
#   - top 20 chosen by deterministic pooled importance ranking
#   - a-priori kidney variables always reported
#   - SHAP is descriptive model attribution, NOT causality
#
# Privacy:
#   - patient-level SHAP stays in RAM only
#   - no patient-level SHAP or prediction table is written to Drive/BQ
# ============================================================

EXPECTED_38A_PROTOCOL_SHA = (
    "06fe49d742e43aedfb518e805f12949f5"
    "2cc926e880a57683144dd6915e30a49"
)

EXPECTED_38B_MANIFEST_SHA = (
    "e6f0d5e03a4ac49bd2ef9d02c9ad4163"
    "606017b41501514052c21e163e6f32b1"
)

EXPECTED_ROWS = 58491
EXPECTED_EVENTS = 3032
EXPECTED_HOSPITALS = 198

EXPECTED_SELECTED = {
    1: "XGB04",
    2: "XGB04",
    3: "XGB04",
    4: "XGB04",
    5: "XGB06",
}

APRIORI_KIDNEY_VARIABLES = [
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
]

TOP_SOURCE_N = 20
TOP_TRANSFORMED_N = 20
TOP_DEPENDENCE_N = 5

# Visualization-only deterministic sampling.
# This does NOT affect any importance/ranking/statistical calculation.
VISUALIZATION_MAX_POINTS = 8000
VISUALIZATION_SEED = 20260724

# Native TreeSHAP additivity check tolerances.
# These are numerical-consistency checks, not inferential thresholds.
ADDITIVITY_MAX_ABS_TOL = 1e-4
ADDITIVITY_MEAN_ABS_TOL = 1e-6

required_runtime = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
]

missing_runtime = [
    x for x in required_runtime
    if x not in globals()
]

if missing_runtime:
    raise RuntimeError(
        "Eksik çalışma nesneleri: "
        + ", ".join(missing_runtime)
        + ". Önce 07A/07B temel hücrelerini çalıştır."
    )

if len(core_df_07B) != EXPECTED_ROWS:
    raise RuntimeError("Cohort row count is not 58,491.")

if int(core_df_07B["label_stage23"].sum()) != EXPECTED_EVENTS:
    raise RuntimeError("Cohort event count is not 3,032.")

if core_df_07B["group_hospital"].astype(str).nunique() != EXPECTED_HOSPITALS:
    raise RuntimeError("Cohort hospital count is not 198.")

# ------------------------------------------------------------
# 1. SHA guards
# ------------------------------------------------------------

guards = [
    (
        "38A_locked_xgboost_treeshap_explainability_protocol_v1_SHA256.txt",
        EXPECTED_38A_PROTOCOL_SHA,
        "38A explainability protocol",
    ),
    (
        "38B_verified_xgboost_reconstruction_manifest_SHA256.txt",
        EXPECTED_38B_MANIFEST_SHA,
        "38B verified reconstruction",
    ),
]

for filename, expected_sha, label in guards:
    path = os.path.join(MODEL_OUTPUT_DIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    with open(path, "r", encoding="utf-8") as fh:
        observed = fh.read().strip()

    if observed != expected_sha:
        raise RuntimeError(
            f"{label} SHA mismatch: {observed}"
        )

    print(f"{label} SHA guard: PASS")

# ------------------------------------------------------------
# 2. Verify 38B artifact manifest and hashes
# ------------------------------------------------------------

artifact_manifest_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "38B_verified_xgboost_artifact_manifest.csv",
)

if not os.path.exists(artifact_manifest_path):
    raise FileNotFoundError(artifact_manifest_path)

artifact_manifest = pd.read_csv(
    artifact_manifest_path
)

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(
            lambda: fh.read(1024 * 1024),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()

for _, row in artifact_manifest.iterrows():
    path = str(row["path"])
    expected = str(row["sha256"])

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    observed = file_sha256(path)

    if observed != expected:
        raise RuntimeError(
            f"Artifact SHA mismatch: {path}"
        )

print("All verified 38B model/preprocessor artifact SHA guards: PASS")

verified_dir = os.path.join(
    MODEL_OUTPUT_DIR,
    "38B_verified_xgboost_models",
)

# ------------------------------------------------------------
# 3. Prepare original outer-test feature data
# ------------------------------------------------------------

X_all = core_df_07B[
    predictor_columns_07B
].copy()

for column in numeric_columns_07B:
    X_all[column] = pd.to_numeric(
        X_all[column],
        errors="coerce",
    ).astype("float64")

for column in categorical_columns_07B:
    s = X_all[column].astype("object")
    X_all[column] = s.where(
        pd.notna(s),
        np.nan,
    )

meta = core_df_07B[
    [
        "id_row",
        "outer_fold",
        "group_hospital",
        "label_stage23",
    ]
].copy()

meta["id_row"] = meta["id_row"].astype(str)
meta["outer_fold"] = pd.to_numeric(
    meta["outer_fold"],
    errors="raise",
).astype(np.int64)
meta["group_hospital"] = meta[
    "group_hospital"
].astype(str)
meta["label_stage23"] = pd.to_numeric(
    meta["label_stage23"],
    errors="raise",
).astype(np.int64)

# ------------------------------------------------------------
# 4. Helpers for transformed-column -> source-predictor mapping
# ------------------------------------------------------------

def build_source_mapping(preprocessor, processed_names):
    """
    Returns:
      source_by_transformed_index: list[str] length = processed columns
      source_to_indices: dict[str, list[int]]
    """
    numeric_pipe = preprocessor.named_transformers_[
        "numeric"
    ]
    numeric_imputer = numeric_pipe.named_steps[
        "imputer"
    ]

    categorical_pipe = preprocessor.named_transformers_[
        "categorical"
    ]
    onehot = categorical_pipe.named_steps[
        "onehot"
    ]

    source_by_index = []

    # Numeric value columns are first and preserve numeric-column order.
    source_by_index.extend(
        list(numeric_columns_07B)
    )

    # Then SimpleImputer missing-indicator columns.
    indicator_features = getattr(
        numeric_imputer.indicator_,
        "features_",
        np.array([], dtype=int),
    )

    for feature_idx in indicator_features:
        source_by_index.append(
            numeric_columns_07B[
                int(feature_idx)
            ]
        )

    # Then OneHotEncoder columns grouped by input categorical predictor.
    for cat_col, categories in zip(
        categorical_columns_07B,
        onehot.categories_,
    ):
        for _ in categories:
            source_by_index.append(
                cat_col
            )

    if len(source_by_index) != len(processed_names):
        raise RuntimeError(
            "Transformed source mapping length mismatch: "
            f"{len(source_by_index)} vs {len(processed_names)}"
        )

    source_to_indices = defaultdict(list)

    for idx, source in enumerate(
        source_by_index
    ):
        source_to_indices[source].append(
            idx
        )

    if set(source_to_indices.keys()) != set(
        predictor_columns_07B
    ):
        missing = (
            set(predictor_columns_07B)
            - set(source_to_indices.keys())
        )
        extra = (
            set(source_to_indices.keys())
            - set(predictor_columns_07B)
        )
        raise RuntimeError(
            f"Source mapping mismatch. missing={missing}, extra={extra}"
        )

    return (
        source_by_index,
        dict(source_to_indices),
    )

# ------------------------------------------------------------
# 5. Pass 1: exact outer-test TreeSHAP
# ------------------------------------------------------------

pooled_source_abs_sum = defaultdict(float)
pooled_source_signed_sum = defaultdict(float)

pooled_transformed_abs_sum = None
pooled_transformed_signed_sum = None

fold_source_rows = []
fold_transformed_rows = []
additivity_rows = []

# Patient-level SHAP stays RAM-only.
shap_by_fold = {}
test_index_by_fold = {}

reference_processed_names = None
reference_source_by_index = None
reference_source_to_indices = None

for outer_fold in range(1, 6):
    candidate_id = EXPECTED_SELECTED[
        outer_fold
    ]

    print(
        f"\nComputing verified native TreeSHAP for outer fold "
        f"{outer_fold} ({candidate_id})..."
    )

    model_path = os.path.join(
        verified_dir,
        f"38B_verified_xgboost_outer{outer_fold}_{candidate_id}.json",
    )
    preprocessor_path = os.path.join(
        verified_dir,
        f"38B_verified_preprocessor_outer{outer_fold}.joblib",
    )
    feature_names_path = os.path.join(
        verified_dir,
        f"38B_verified_processed_feature_names_outer{outer_fold}.json",
    )

    preprocessor = joblib.load(
        preprocessor_path
    )

    with open(
        feature_names_path,
        "r",
        encoding="utf-8",
    ) as fh:
        processed_names = json.load(fh)

    processed_names = [
        str(x)
        for x in processed_names
    ]

    if reference_processed_names is None:
        reference_processed_names = processed_names

        (
            reference_source_by_index,
            reference_source_to_indices,
        ) = build_source_mapping(
            preprocessor,
            processed_names,
        )
    else:
        if processed_names != reference_processed_names:
            raise RuntimeError(
                f"Outer fold {outer_fold}: processed feature-name space differs."
            )

        (
            fold_source_by_index,
            fold_source_to_indices,
        ) = build_source_mapping(
            preprocessor,
            processed_names,
        )

        if fold_source_by_index != reference_source_by_index:
            raise RuntimeError(
                f"Outer fold {outer_fold}: source mapping differs."
            )

    test_mask = (
        meta["outer_fold"].to_numpy()
        == outer_fold
    )

    test_indices = np.flatnonzero(
        test_mask
    )

    X_test = (
        X_all.loc[
            test_mask,
            predictor_columns_07B,
        ]
        .reset_index(drop=True)
    )

    X_test_processed = preprocessor.transform(
        X_test
    )

    if X_test_processed.shape[1] != len(
        processed_names
    ):
        raise RuntimeError(
            f"Outer fold {outer_fold}: transformed-column count mismatch."
        )

    booster = xgb.Booster()
    booster.load_model(
        model_path
    )

    dtest = xgb.DMatrix(
        X_test_processed
    )

    raw_margin = booster.predict(
        dtest,
        output_margin=True,
    ).astype(np.float64)

    contributions = booster.predict(
        dtest,
        pred_contribs=True,
        approx_contribs=False,
    ).astype(np.float64)

    if contributions.shape != (
        len(X_test),
        len(processed_names) + 1,
    ):
        raise RuntimeError(
            f"Outer fold {outer_fold}: unexpected SHAP contribution shape "
            f"{contributions.shape}."
        )

    shap_values = contributions[
        :, :-1
    ]
    expected_value = contributions[
        :, -1
    ]

    reconstructed_margin = (
        shap_values.sum(axis=1)
        + expected_value
    )

    additivity_diff = np.abs(
        reconstructed_margin
        - raw_margin
    )

    max_additivity_diff = float(
        additivity_diff.max()
    )
    mean_additivity_diff = float(
        additivity_diff.mean()
    )

    additivity_pass = (
        max_additivity_diff
        <= ADDITIVITY_MAX_ABS_TOL
        and mean_additivity_diff
        <= ADDITIVITY_MEAN_ABS_TOL
    )

    additivity_rows.append({
        "outer_fold": outer_fold,
        "selected_candidate": candidate_id,
        "test_patients": len(X_test),
        "processed_columns": len(processed_names),
        "expected_value_mean": float(
            expected_value.mean()
        ),
        "maximum_absolute_additivity_difference":
            max_additivity_diff,
        "mean_absolute_additivity_difference":
            mean_additivity_diff,
        "maximum_allowed_absolute_difference":
            ADDITIVITY_MAX_ABS_TOL,
        "maximum_allowed_mean_difference":
            ADDITIVITY_MEAN_ABS_TOL,
        "additivity_status":
            "PASS" if additivity_pass else "FAIL",
    })

    if not additivity_pass:
        raise RuntimeError(
            f"Outer fold {outer_fold}: TreeSHAP additivity check FAILED."
        )

    print(
        "TreeSHAP additivity:",
        f"max={max_additivity_diff:.3g}, "
        f"mean={mean_additivity_diff:.3g} -> PASS"
    )

    abs_shap = np.abs(
        shap_values
    )

    # Transformed-feature level.
    fold_mean_abs = abs_shap.mean(
        axis=0
    )
    fold_mean_signed = shap_values.mean(
        axis=0
    )

    if pooled_transformed_abs_sum is None:
        pooled_transformed_abs_sum = np.zeros(
            len(processed_names),
            dtype=np.float64,
        )
        pooled_transformed_signed_sum = np.zeros(
            len(processed_names),
            dtype=np.float64,
        )

    pooled_transformed_abs_sum += abs_shap.sum(
        axis=0
    )
    pooled_transformed_signed_sum += shap_values.sum(
        axis=0
    )

    for idx, transformed_name in enumerate(
        processed_names
    ):
        fold_transformed_rows.append({
            "outer_fold": outer_fold,
            "selected_candidate": candidate_id,
            "transformed_feature":
                transformed_name,
            "source_predictor":
                reference_source_by_index[idx],
            "mean_absolute_shap":
                float(fold_mean_abs[idx]),
            "mean_signed_shap":
                float(fold_mean_signed[idx]),
        })

    # Source-predictor grouped level.
    for source in predictor_columns_07B:
        indices = reference_source_to_indices[
            source
        ]

        patient_abs_group = abs_shap[
            :, indices
        ].sum(axis=1)

        patient_signed_group = shap_values[
            :, indices
        ].sum(axis=1)

        mean_abs_group = float(
            patient_abs_group.mean()
        )
        mean_signed_group = float(
            patient_signed_group.mean()
        )

        pooled_source_abs_sum[
            source
        ] += float(
            patient_abs_group.sum()
        )

        pooled_source_signed_sum[
            source
        ] += float(
            patient_signed_group.sum()
        )

        fold_source_rows.append({
            "outer_fold": outer_fold,
            "selected_candidate":
                candidate_id,
            "source_predictor":
                source,
            "transformed_column_count":
                len(indices),
            "mean_absolute_grouped_shap":
                mean_abs_group,
            "mean_signed_grouped_shap":
                mean_signed_group,
        })

    # Keep patient-level SHAP in RAM only.
    shap_by_fold[
        outer_fold
    ] = shap_values.astype(
        np.float32,
        copy=False,
    )

    test_index_by_fold[
        outer_fold
    ] = test_indices

    del (
        X_test,
        X_test_processed,
        dtest,
        contributions,
        raw_margin,
        reconstructed_margin,
        additivity_diff,
        abs_shap,
        booster,
        preprocessor,
    )

# ------------------------------------------------------------
# 6. Aggregate global rankings
# ------------------------------------------------------------

additivity_audit = pd.DataFrame(
    additivity_rows
)

fold_source_importance = pd.DataFrame(
    fold_source_rows
)

fold_transformed_importance = pd.DataFrame(
    fold_transformed_rows
)

pooled_source_rows = []

for source in predictor_columns_07B:
    pooled_source_rows.append({
        "source_predictor": source,
        "transformed_column_count":
            len(
                reference_source_to_indices[
                    source
                ]
            ),
        "pooled_mean_absolute_grouped_shap":
            pooled_source_abs_sum[source]
            / EXPECTED_ROWS,
        "pooled_mean_signed_grouped_shap":
            pooled_source_signed_sum[source]
            / EXPECTED_ROWS,
    })

pooled_source_importance = pd.DataFrame(
    pooled_source_rows
).sort_values(
    [
        "pooled_mean_absolute_grouped_shap",
        "source_predictor",
    ],
    ascending=[False, True],
    kind="stable",
).reset_index(drop=True)

pooled_source_importance.insert(
    0,
    "pooled_rank",
    np.arange(
        1,
        len(pooled_source_importance) + 1,
    ),
)

pooled_transformed_importance = pd.DataFrame({
    "transformed_feature":
        reference_processed_names,
    "source_predictor":
        reference_source_by_index,
    "pooled_mean_absolute_shap":
        pooled_transformed_abs_sum
        / EXPECTED_ROWS,
    "pooled_mean_signed_shap":
        pooled_transformed_signed_sum
        / EXPECTED_ROWS,
}).sort_values(
    [
        "pooled_mean_absolute_shap",
        "transformed_feature",
    ],
    ascending=[False, True],
    kind="stable",
).reset_index(drop=True)

pooled_transformed_importance.insert(
    0,
    "pooled_rank",
    np.arange(
        1,
        len(pooled_transformed_importance) + 1,
    ),
)

top20_source = pooled_source_importance.head(
    TOP_SOURCE_N
).copy()

top20_transformed = pooled_transformed_importance.head(
    TOP_TRANSFORMED_N
).copy()

kidney_importance = pooled_source_importance.loc[
    pooled_source_importance[
        "source_predictor"
    ].isin(
        APRIORI_KIDNEY_VARIABLES
    )
].copy()

if set(
    kidney_importance["source_predictor"]
) != set(APRIORI_KIDNEY_VARIABLES):
    raise RuntimeError(
        "One or more a-priori kidney variables are absent from source importance."
    )

# ------------------------------------------------------------
# 7. Fold-specific ranks and stability
# ------------------------------------------------------------

fold_source_importance[
    "fold_rank"
] = (
    fold_source_importance.groupby(
        "outer_fold"
    )[
        "mean_absolute_grouped_shap"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

rank_matrix = (
    fold_source_importance.pivot(
        index="source_predictor",
        columns="outer_fold",
        values="fold_rank",
    )
    .sort_index()
)

rank_correlation = rank_matrix.corr(
    method="spearman"
)

rank_correlation.index.name = "outer_fold"
rank_correlation.columns = [
    f"outer_fold_{int(x)}"
    for x in rank_correlation.columns
]

rank_correlation = rank_correlation.reset_index()

top20_presence_rows = []

for source in predictor_columns_07B:
    sub = fold_source_importance.loc[
        fold_source_importance[
            "source_predictor"
        ].eq(source)
    ]

    top20_presence_rows.append({
        "source_predictor": source,
        "top20_presence_count_across_5_folds":
            int(
                (sub["fold_rank"] <= 20).sum()
            ),
        "minimum_fold_rank":
            int(sub["fold_rank"].min()),
        "maximum_fold_rank":
            int(sub["fold_rank"].max()),
        "median_fold_rank":
            float(sub["fold_rank"].median()),
        "pooled_rank":
            int(
                pooled_source_importance.loc[
                    pooled_source_importance[
                        "source_predictor"
                    ].eq(source),
                    "pooled_rank",
                ].iloc[0]
            ),
    })

top20_presence = pd.DataFrame(
    top20_presence_rows
).sort_values(
    [
        "top20_presence_count_across_5_folds",
        "pooled_rank",
    ],
    ascending=[False, True],
    kind="stable",
).reset_index(drop=True)

# ------------------------------------------------------------
# 8. Visualization 1: top-20 source importance
# ------------------------------------------------------------

figure_dir = os.path.join(
    MODEL_OUTPUT_DIR,
    "38C_treeshap_figures",
)
os.makedirs(
    figure_dir,
    exist_ok=True,
)

top20_source_png = os.path.join(
    figure_dir,
    "38C_top20_source_predictor_mean_absolute_treeshap.png",
)

plot_df = top20_source.sort_values(
    "pooled_mean_absolute_grouped_shap",
    ascending=True,
)

fig = plt.figure(
    figsize=(9.0, 7.5)
)
ax = fig.add_subplot(111)

ax.barh(
    plot_df["source_predictor"],
    plot_df[
        "pooled_mean_absolute_grouped_shap"
    ],
)

ax.set_xlabel(
    "Mean absolute grouped TreeSHAP contribution (raw margin)"
)
ax.set_ylabel(
    "Source predictor"
)
ax.set_title(
    "XGBoost Global Feature Attribution Across Outer-Test Patients"
)

fig.tight_layout()

fig.savefig(
    top20_source_png,
    dpi=300,
    bbox_inches="tight",
)

plt.close(fig)

# ------------------------------------------------------------
# 9. Visualization 2: transformed-feature SHAP beeswarm
# ------------------------------------------------------------

name_to_index = {
    name: idx
    for idx, name in enumerate(
        reference_processed_names
    )
}

top_transformed_names = top20_transformed[
    "transformed_feature"
].tolist()

top_transformed_indices = [
    name_to_index[name]
    for name in top_transformed_names
]

# Second pass only for transformed feature values.
transformed_value_by_feature = {
    name: []
    for name in top_transformed_names
}
shap_value_by_feature = {
    name: []
    for name in top_transformed_names
}

for outer_fold in range(1, 6):
    candidate_id = EXPECTED_SELECTED[
        outer_fold
    ]

    preprocessor_path = os.path.join(
        verified_dir,
        f"38B_verified_preprocessor_outer{outer_fold}.joblib",
    )

    preprocessor = joblib.load(
        preprocessor_path
    )

    test_mask = (
        meta["outer_fold"].to_numpy()
        == outer_fold
    )

    X_test = (
        X_all.loc[
            test_mask,
            predictor_columns_07B,
        ]
        .reset_index(drop=True)
    )

    X_proc = preprocessor.transform(
        X_test
    )

    selected_values = X_proc[
        :, top_transformed_indices
    ]

    if hasattr(
        selected_values,
        "toarray",
    ):
        selected_values = selected_values.toarray()

    selected_values = np.asarray(
        selected_values,
        dtype=np.float32,
    )

    fold_shap = shap_by_fold[
        outer_fold
    ]

    for local_j, name in enumerate(
        top_transformed_names
    ):
        transformed_value_by_feature[
            name
        ].append(
            selected_values[
                :, local_j
            ]
        )
        shap_value_by_feature[
            name
        ].append(
            fold_shap[
                :, top_transformed_indices[
                    local_j
                ]
            ]
        )

    del (
        preprocessor,
        X_test,
        X_proc,
        selected_values,
    )

beeswarm_png = os.path.join(
    figure_dir,
    "38C_top20_transformed_feature_treeshap_beeswarm.png",
)

rng = np.random.default_rng(
    VISUALIZATION_SEED
)

fig = plt.figure(
    figsize=(10.0, 8.5)
)
ax = fig.add_subplot(111)

# Reverse order so rank 1 appears at top.
plot_names = list(
    reversed(
        top_transformed_names
    )
)

for y_pos, name in enumerate(
    plot_names
):
    shap_vals = np.concatenate(
        shap_value_by_feature[
            name
        ]
    ).astype(float)

    feat_vals = np.concatenate(
        transformed_value_by_feature[
            name
        ]
    ).astype(float)

    n = len(shap_vals)

    if n > VISUALIZATION_MAX_POINTS:
        chosen = rng.choice(
            n,
            size=VISUALIZATION_MAX_POINTS,
            replace=False,
        )
    else:
        chosen = np.arange(n)

    # Small deterministic vertical jitter for density visibility.
    jitter = rng.normal(
        loc=0.0,
        scale=0.10,
        size=len(chosen),
    )

    ax.scatter(
        shap_vals[chosen],
        np.full(
            len(chosen),
            y_pos,
            dtype=float,
        ) + jitter,
        c=feat_vals[chosen],
        s=5,
        alpha=0.45,
        linewidths=0,
    )

ax.axvline(
    0.0,
    linewidth=1.0,
)

ax.set_yticks(
    np.arange(
        len(plot_names)
    )
)
ax.set_yticklabels(
    plot_names
)

ax.set_xlabel(
    "TreeSHAP contribution to XGBoost raw margin"
)
ax.set_ylabel(
    "Transformed feature"
)
ax.set_title(
    "Top 20 Transformed-Feature TreeSHAP Contributions"
)

fig.tight_layout()

fig.savefig(
    beeswarm_png,
    dpi=300,
    bbox_inches="tight",
)

plt.close(fig)

# ------------------------------------------------------------
# 10. Predefined dependence plots:
#     top-5 pooled source predictors + 3 a-priori kidney predictors
# ------------------------------------------------------------

top5_sources = top20_source.head(
    TOP_DEPENDENCE_N
)["source_predictor"].tolist()

dependence_sources = []

for source in (
    top5_sources
    + APRIORI_KIDNEY_VARIABLES
):
    if source not in dependence_sources:
        dependence_sources.append(
            source
        )

dependence_manifest_rows = []

for source in dependence_sources:
    source_indices = reference_source_to_indices[
        source
    ]

    pooled_grouped_shap_parts = []
    pooled_source_value_parts = []

    for outer_fold in range(1, 6):
        test_indices = test_index_by_fold[
            outer_fold
        ]

        fold_shap = shap_by_fold[
            outer_fold
        ]

        grouped_signed_shap = fold_shap[
            :, source_indices
        ].sum(axis=1)

        pooled_grouped_shap_parts.append(
            grouped_signed_shap.astype(
                np.float32,
                copy=False,
            )
        )

        raw_values = X_all.iloc[
            test_indices
        ][source]

        pooled_source_value_parts.append(
            raw_values.reset_index(
                drop=True
            )
        )

    grouped_shap_all = np.concatenate(
        pooled_grouped_shap_parts
    ).astype(float)

    raw_source_all = pd.concat(
        pooled_source_value_parts,
        ignore_index=True,
    )

    source_rank = int(
        pooled_source_importance.loc[
            pooled_source_importance[
                "source_predictor"
            ].eq(source),
            "pooled_rank",
        ].iloc[0]
    )

    safe_name = "".join(
        ch if (
            ch.isalnum()
            or ch in {"_", "-"}
        ) else "_"
        for ch in source
    )

    figure_path = os.path.join(
        figure_dir,
        f"38C_dependence_{safe_name}.png",
    )

    fig = plt.figure(
        figsize=(8.0, 5.5)
    )
    ax = fig.add_subplot(111)

    if source in numeric_columns_07B:
        x_numeric = pd.to_numeric(
            raw_source_all,
            errors="coerce",
        ).to_numpy(dtype=float)

        valid = np.isfinite(
            x_numeric
        )

        valid_indices = np.flatnonzero(
            valid
        )

        if len(valid_indices) > VISUALIZATION_MAX_POINTS:
            chosen = rng.choice(
                valid_indices,
                size=VISUALIZATION_MAX_POINTS,
                replace=False,
            )
        else:
            chosen = valid_indices

        ax.scatter(
            x_numeric[chosen],
            grouped_shap_all[chosen],
            s=7,
            alpha=0.35,
            linewidths=0,
        )

        ax.set_xlabel(
            source
        )

        missing_count = int(
            (~valid).sum()
        )

        ax.set_title(
            f"TreeSHAP Dependence: {source} "
            f"(pooled rank {source_rank}; missing n={missing_count})"
        )

    else:
        cat = raw_source_all.astype(
            "object"
        ).where(
            pd.notna(raw_source_all),
            "__MISSING__",
        ).astype(str)

        category_order = (
            cat.value_counts(
                dropna=False
            )
            .index
            .tolist()
        )

        data = [
            grouped_shap_all[
                cat.to_numpy() == level
            ]
            for level in category_order
        ]

        ax.boxplot(
            data,
            labels=category_order,
            showfliers=False,
        )

        ax.set_xlabel(
            source
        )

        ax.tick_params(
            axis="x",
            labelrotation=45,
        )

        ax.set_title(
            f"TreeSHAP by Category: {source} "
            f"(pooled rank {source_rank})"
        )

    ax.axhline(
        0.0,
        linewidth=1.0,
    )

    ax.set_ylabel(
        "Grouped signed TreeSHAP contribution (raw margin)"
    )

    fig.tight_layout()

    fig.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)

    dependence_manifest_rows.append({
        "source_predictor": source,
        "pooled_rank": source_rank,
        "selection_basis": (
            "top5_pooled_importance"
            if source in top5_sources
            else "a_priori_kidney_variable"
        ),
        "figure_path": figure_path,
    })

dependence_manifest = pd.DataFrame(
    dependence_manifest_rows
)

# ------------------------------------------------------------
# 11. Save aggregate tables only
# ------------------------------------------------------------

paths = {
    "additivity_audit": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_treeshap_additivity_audit.csv",
    ),
    "fold_source_importance": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_fold_source_predictor_treeshap_importance.csv",
    ),
    "pooled_source_importance": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_pooled_source_predictor_treeshap_importance.csv",
    ),
    "top20_source": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_top20_source_predictor_treeshap_importance.csv",
    ),
    "kidney_importance": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_apriori_kidney_variable_treeshap_importance.csv",
    ),
    "fold_transformed_importance": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_fold_transformed_feature_treeshap_importance.csv",
    ),
    "pooled_transformed_importance": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_pooled_transformed_feature_treeshap_importance.csv",
    ),
    "top20_transformed": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_top20_transformed_feature_treeshap_importance.csv",
    ),
    "fold_rank_correlation": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_fold_source_rank_spearman_correlation.csv",
    ),
    "top20_presence": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_source_top20_presence_frequency_across_folds.csv",
    ),
    "dependence_manifest": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_dependence_plot_manifest.csv",
    ),
    "manifest": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_verified_outer_test_treeshap_manifest.json",
    ),
    "manifest_sha": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_verified_outer_test_treeshap_manifest_SHA256.txt",
    ),
}

additivity_audit.to_csv(
    paths["additivity_audit"],
    index=False,
)

fold_source_importance.to_csv(
    paths["fold_source_importance"],
    index=False,
)

pooled_source_importance.to_csv(
    paths["pooled_source_importance"],
    index=False,
)

top20_source.to_csv(
    paths["top20_source"],
    index=False,
)

kidney_importance.to_csv(
    paths["kidney_importance"],
    index=False,
)

fold_transformed_importance.to_csv(
    paths["fold_transformed_importance"],
    index=False,
)

pooled_transformed_importance.to_csv(
    paths["pooled_transformed_importance"],
    index=False,
)

top20_transformed.to_csv(
    paths["top20_transformed"],
    index=False,
)

rank_correlation.to_csv(
    paths["fold_rank_correlation"],
    index=False,
)

top20_presence.to_csv(
    paths["top20_presence"],
    index=False,
)

dependence_manifest.to_csv(
    paths["dependence_manifest"],
    index=False,
)

manifest = {
    "analysis_version": "38C",
    "analysis_type":
        "verified_outer_test_native_xgboost_treeshap",
    "source_38A_protocol_sha256":
        EXPECTED_38A_PROTOCOL_SHA,
    "source_38B_verified_reconstruction_sha256":
        EXPECTED_38B_MANIFEST_SHA,
    "patients_explained": EXPECTED_ROWS,
    "hospitals": EXPECTED_HOSPITALS,
    "events": EXPECTED_EVENTS,
    "outer_folds": 5,
    "selected_candidates_by_outer_fold":
        EXPECTED_SELECTED,
    "shap_method":
        "native_xgboost_pred_contribs",
    "shap_scale":
        "raw_margin_log_odds",
    "primary_global_importance":
        "source_predictor_grouped_mean_absolute_shap",
    "top_source_predictors_displayed":
        TOP_SOURCE_N,
    "top_transformed_features_displayed":
        TOP_TRANSFORMED_N,
    "a_priori_kidney_variables":
        APRIORI_KIDNEY_VARIABLES,
    "fold_stability_assessed": True,
    "additivity_check": {
        "maximum_absolute_tolerance":
            ADDITIVITY_MAX_ABS_TOL,
        "mean_absolute_tolerance":
            ADDITIVITY_MEAN_ABS_TOL,
        "all_folds_passed": bool(
            additivity_audit[
                "additivity_status"
            ].eq("PASS").all()
        ),
    },
    "dependence_plot_selection": {
        "top_n_data_driven":
            TOP_DEPENDENCE_N,
        "plus_a_priori_kidney_variables":
            APRIORI_KIDNEY_VARIABLES,
        "visualization_max_points":
            VISUALIZATION_MAX_POINTS,
        "visualization_seed":
            VISUALIZATION_SEED,
    },
    "causal_interpretation_permitted":
        False,
    "patient_level_shap_written_to_drive":
        False,
    "patient_level_shap_written_to_bigquery":
        False,
    "patient_level_shap_retained":
        "RAM_only_during_execution",
    "retuning_performed": False,
    "candidate_reselection_performed":
        False,
    "recalibration_performed": False,
    "bigquery_dml_used": False,
    "figures": {
        "top20_source_importance":
            top20_source_png,
        "top20_transformed_beeswarm":
            beeswarm_png,
        "dependence_figure_directory":
            figure_dir,
    },
    "outputs": paths,
}

manifest_text = json.dumps(
    manifest,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
)

with open(
    paths["manifest"],
    "w",
    encoding="utf-8",
) as fh:
    fh.write(
        manifest_text
    )

manifest_sha = hashlib.sha256(
    manifest_text.encode("utf-8")
).hexdigest()

with open(
    paths["manifest_sha"],
    "w",
    encoding="utf-8",
) as fh:
    fh.write(
        manifest_sha + "\n"
    )

# ------------------------------------------------------------
# 12. Compact display
# ------------------------------------------------------------

print("\n38C TREESHAP ADDITIVITY AUDIT")
display(
    additivity_audit
)

print("\n38C TOP 20 SOURCE-PREDICTOR IMPORTANCE")
display(
    top20_source
)

print("\n38C A-PRIORI KIDNEY VARIABLE IMPORTANCE")
display(
    kidney_importance
)

print("\n38C FOLD SOURCE-RANK SPEARMAN CORRELATION")
display(
    rank_correlation
)

print("\n38C TOP-20 PRESENCE FREQUENCY — FIRST 30")
display(
    top20_presence.head(30)
)

print("\n38C DEPENDENCE PLOT MANIFEST")
display(
    dependence_manifest
)

print("\n38C manifest SHA-256:")
print(
    manifest_sha
)

print("\nFigures saved under:")
print(
    figure_dir
)

print(
    "\n38C PASS: Verified outer-test native TreeSHAP analysis completed."
)
print(
    "All global rankings were generated by the pre-locked deterministic rules."
)
print(
    "SHAP values are descriptive model attributions, not causal effects."
)
print(
    "No patient-level SHAP values were written to Drive or BigQuery."
)

In [ ]:
import os
import json
import hashlib

import numpy as np
import pandas as pd
import joblib
import xgboost as xgb

from IPython.display import display

print("STARTING TREESHAP ADDITIVITY NUMERICAL DIAGNOSTIC — CODE VERSION 38C-D1")

# ============================================================
# 38C-D1 — NUMERICAL DIAGNOSTIC ONLY
#
# Purpose:
#   Diagnose why the first 38C run stopped at the native TreeSHAP
#   additivity check.
#
# IMPORTANT:
#   - 38B already proved exact model reconstruction: stored raw
#     probabilities were reproduced with max/mean difference = 0.
#   - This diagnostic does NOT rank features, produce SHAP figures,
#     or interpret any attribution.
#   - It only measures floating-point additivity discrepancies between:
#         model raw margin
#         sum(TreeSHAP contributions + bias term)
#   - Patient-level values remain RAM-only.
# ============================================================

EXPECTED_38A_PROTOCOL_SHA = (
    "06fe49d742e43aedfb518e805f12949f5"
    "2cc926e880a57683144dd6915e30a49"
)

EXPECTED_38B_MANIFEST_SHA = (
    "e6f0d5e03a4ac49bd2ef9d02c9ad4163"
    "606017b41501514052c21e163e6f32b1"
)

EXPECTED_ROWS = 58491
EXPECTED_EVENTS = 3032
EXPECTED_HOSPITALS = 198

EXPECTED_SELECTED = {
    1: "XGB04",
    2: "XGB04",
    3: "XGB04",
    4: "XGB04",
    5: "XGB06",
}

required_runtime = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
]

missing_runtime = [
    x for x in required_runtime
    if x not in globals()
]

if missing_runtime:
    raise RuntimeError(
        "Eksik çalışma nesneleri: "
        + ", ".join(missing_runtime)
        + ". Önce 07A/07B temel hücrelerini çalıştır."
    )

if len(core_df_07B) != EXPECTED_ROWS:
    raise RuntimeError("Cohort row count is not 58,491.")

if int(core_df_07B["label_stage23"].sum()) != EXPECTED_EVENTS:
    raise RuntimeError("Cohort event count is not 3,032.")

if core_df_07B["group_hospital"].astype(str).nunique() != EXPECTED_HOSPITALS:
    raise RuntimeError("Cohort hospital count is not 198.")

# ------------------------------------------------------------
# 1. SHA guards
# ------------------------------------------------------------

guards = [
    (
        "38A_locked_xgboost_treeshap_explainability_protocol_v1_SHA256.txt",
        EXPECTED_38A_PROTOCOL_SHA,
        "38A explainability protocol",
    ),
    (
        "38B_verified_xgboost_reconstruction_manifest_SHA256.txt",
        EXPECTED_38B_MANIFEST_SHA,
        "38B verified reconstruction",
    ),
]

for filename, expected_sha, label in guards:
    path = os.path.join(MODEL_OUTPUT_DIR, filename)

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    with open(path, "r", encoding="utf-8") as fh:
        observed = fh.read().strip()

    if observed != expected_sha:
        raise RuntimeError(
            f"{label} SHA mismatch: {observed}"
        )

    print(f"{label} SHA guard: PASS")

# ------------------------------------------------------------
# 2. Verify 38B artifact hashes
# ------------------------------------------------------------

artifact_manifest_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "38B_verified_xgboost_artifact_manifest.csv",
)

artifact_manifest = pd.read_csv(
    artifact_manifest_path
)

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(
            lambda: fh.read(1024 * 1024),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()

for _, row in artifact_manifest.iterrows():
    path = str(row["path"])
    expected = str(row["sha256"])

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    if file_sha256(path) != expected:
        raise RuntimeError(
            f"Artifact SHA mismatch: {path}"
        )

print("All verified 38B artifact SHA guards: PASS")

verified_dir = os.path.join(
    MODEL_OUTPUT_DIR,
    "38B_verified_xgboost_models",
)

# ------------------------------------------------------------
# 3. Prepare exact outer-test data
# ------------------------------------------------------------

X_all = core_df_07B[
    predictor_columns_07B
].copy()

for column in numeric_columns_07B:
    X_all[column] = pd.to_numeric(
        X_all[column],
        errors="coerce",
    ).astype("float64")

for column in categorical_columns_07B:
    s = X_all[column].astype("object")
    X_all[column] = s.where(
        pd.notna(s),
        np.nan,
    )

meta = core_df_07B[
    [
        "outer_fold",
        "group_hospital",
        "label_stage23",
    ]
].copy()

meta["outer_fold"] = pd.to_numeric(
    meta["outer_fold"],
    errors="raise",
).astype(np.int64)

# ------------------------------------------------------------
# 4. Numerical diagnostic over all five folds
# ------------------------------------------------------------

rows = []

for outer_fold in range(1, 6):
    candidate_id = EXPECTED_SELECTED[
        outer_fold
    ]

    print(
        f"\nDiagnosing outer fold {outer_fold} ({candidate_id})..."
    )

    model_path = os.path.join(
        verified_dir,
        f"38B_verified_xgboost_outer{outer_fold}_{candidate_id}.json",
    )
    preprocessor_path = os.path.join(
        verified_dir,
        f"38B_verified_preprocessor_outer{outer_fold}.joblib",
    )

    preprocessor = joblib.load(
        preprocessor_path
    )

    test_mask = (
        meta["outer_fold"].to_numpy()
        == outer_fold
    )

    X_test = (
        X_all.loc[
            test_mask,
            predictor_columns_07B,
        ]
        .reset_index(drop=True)
    )

    X_proc = preprocessor.transform(
        X_test
    )

    booster = xgb.Booster()
    booster.load_model(
        model_path
    )

    dtest = xgb.DMatrix(
        X_proc
    )

    raw_margin_native = booster.predict(
        dtest,
        output_margin=True,
    )

    probability_native = booster.predict(
        dtest,
        output_margin=False,
    )

    contrib_native = booster.predict(
        dtest,
        pred_contribs=True,
        approx_contribs=False,
    )

    # Native-precision accumulation.
    contrib_sum_native = contrib_native.sum(
        axis=1
    )

    # Float64 accumulation for diagnosis.
    contrib_sum_float64 = contrib_native.astype(
        np.float64
    ).sum(
        axis=1
    )

    raw_margin_float64 = raw_margin_native.astype(
        np.float64
    )

    diff_native = np.abs(
        contrib_sum_native.astype(np.float64)
        - raw_margin_float64
    )

    diff_float64 = np.abs(
        contrib_sum_float64
        - raw_margin_float64
    )

    # Probability-space effect of the margin discrepancy.
    clipped_margin = np.clip(
        contrib_sum_float64,
        -50.0,
        50.0,
    )

    probability_from_contrib = (
        1.0
        / (
            1.0
            + np.exp(
                -clipped_margin
            )
        )
    )

    probability_diff = np.abs(
        probability_from_contrib
        - probability_native.astype(
            np.float64
        )
    )

    scale = np.maximum(
        1.0,
        np.abs(
            raw_margin_float64
        ),
    )

    relative_diff = (
        diff_float64
        / scale
    )

    def quantile(x, q):
        return float(
            np.quantile(
                x,
                q,
            )
        )

    row = {
        "outer_fold": outer_fold,
        "selected_candidate": candidate_id,
        "test_patients": int(
            len(X_test)
        ),
        "processed_columns": int(
            X_proc.shape[1]
        ),

        "raw_margin_dtype":
            str(raw_margin_native.dtype),
        "contrib_dtype":
            str(contrib_native.dtype),

        "native_sum_max_abs_margin_diff":
            float(diff_native.max()),
        "native_sum_mean_abs_margin_diff":
            float(diff_native.mean()),

        "float64_sum_max_abs_margin_diff":
            float(diff_float64.max()),
        "float64_sum_mean_abs_margin_diff":
            float(diff_float64.mean()),
        "float64_sum_median_abs_margin_diff":
            quantile(diff_float64, 0.50),
        "float64_sum_p95_abs_margin_diff":
            quantile(diff_float64, 0.95),
        "float64_sum_p99_abs_margin_diff":
            quantile(diff_float64, 0.99),
        "float64_sum_p999_abs_margin_diff":
            quantile(diff_float64, 0.999),

        "maximum_relative_margin_diff":
            float(relative_diff.max()),

        "max_abs_probability_diff_from_contrib_sum":
            float(probability_diff.max()),
        "mean_abs_probability_diff_from_contrib_sum":
            float(probability_diff.mean()),

        "allclose_atol1e_5_rtol1e_5":
            bool(
                np.allclose(
                    contrib_sum_float64,
                    raw_margin_float64,
                    atol=1e-5,
                    rtol=1e-5,
                )
            ),
        "allclose_atol1e_4_rtol1e_5":
            bool(
                np.allclose(
                    contrib_sum_float64,
                    raw_margin_float64,
                    atol=1e-4,
                    rtol=1e-5,
                )
            ),
        "allclose_atol5e_4_rtol1e_5":
            bool(
                np.allclose(
                    contrib_sum_float64,
                    raw_margin_float64,
                    atol=5e-4,
                    rtol=1e-5,
                )
            ),
    }

    rows.append(
        row
    )

    print(
        "float64 SHAP-sum vs margin:"
    )
    print(
        "  max abs diff  =",
        f"{row['float64_sum_max_abs_margin_diff']:.12g}",
    )
    print(
        "  mean abs diff =",
        f"{row['float64_sum_mean_abs_margin_diff']:.12g}",
    )
    print(
        "  p99 abs diff  =",
        f"{row['float64_sum_p99_abs_margin_diff']:.12g}",
    )
    print(
        "probability-space impact:"
    )
    print(
        "  max abs diff  =",
        f"{row['max_abs_probability_diff_from_contrib_sum']:.12g}",
    )
    print(
        "  mean abs diff =",
        f"{row['mean_abs_probability_diff_from_contrib_sum']:.12g}",
    )

    del (
        preprocessor,
        X_test,
        X_proc,
        booster,
        dtest,
        raw_margin_native,
        probability_native,
        contrib_native,
        contrib_sum_native,
        contrib_sum_float64,
        raw_margin_float64,
        diff_native,
        diff_float64,
        probability_from_contrib,
        probability_diff,
        relative_diff,
    )

diagnostic = pd.DataFrame(
    rows
)

# ------------------------------------------------------------
# 5. Save aggregate diagnostic only
# ------------------------------------------------------------

diagnostic_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "38C_D1_treeshap_additivity_numerical_diagnostic.csv",
)

manifest_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "38C_D1_treeshap_additivity_numerical_diagnostic_manifest.json",
)

manifest_sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "38C_D1_treeshap_additivity_numerical_diagnostic_manifest_SHA256.txt",
)

diagnostic.to_csv(
    diagnostic_path,
    index=False,
)

manifest = {
    "analysis_version": "38C-D1",
    "analysis_type":
        "treeshap_additivity_numerical_diagnostic_only",
    "source_38A_protocol_sha256":
        EXPECTED_38A_PROTOCOL_SHA,
    "source_38B_verified_reconstruction_sha256":
        EXPECTED_38B_MANIFEST_SHA,
    "folds": 5,
    "feature_ranking_performed": False,
    "shap_figure_generated": False,
    "shap_interpretation_performed": False,
    "patient_level_values_written_to_drive":
        False,
    "bigquery_dml_used": False,
    "retuning_performed": False,
    "reselection_performed": False,
    "recalibration_performed": False,
    "output": diagnostic_path,
}

manifest_text = json.dumps(
    manifest,
    indent=2,
    sort_keys=True,
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(
        manifest_text
    )

manifest_sha = hashlib.sha256(
    manifest_text.encode(
        "utf-8"
    )
).hexdigest()

with open(
    manifest_sha_path,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(
        manifest_sha + "\n"
    )

print("\n38C-D1 ADDITIVITY NUMERICAL DIAGNOSTIC")
display(
    diagnostic
)

print("\n38C-D1 manifest SHA-256:")
print(
    manifest_sha
)

print(
    "\n38C-D1 COMPLETE: numerical additivity discrepancy measured "
    "without feature ranking, figures, or interpretation."
)
print(
    "Send this diagnostic output before rerunning the full TreeSHAP analysis."
)

In [ ]:
import os
import json
import hashlib
import pandas as pd

print("STARTING TREESHAP NUMERICAL ADDITIVITY TOLERANCE AMENDMENT LOCK — CODE VERSION 38C-D2")

# ============================================================
# 38C-D2 — NUMERICAL ADDITIVITY TOLERANCE AMENDMENT
#
# Purpose:
#   Formally document the numerical-equivalence rule used for
#   native XGBoost TreeSHAP additivity after 38C-D1 showed that:
#
#   - raw margins and TreeSHAP contribution sums are float32-derived;
#   - all five folds satisfy np.allclose(atol=1e-5, rtol=1e-5);
#   - maximum absolute margin discrepancy is < 8e-6;
#   - maximum probability-space impact is < 7e-7.
#
# This amendment concerns numerical floating-point equivalence only.
# It does NOT change:
#   - models
#   - hyperparameters
#   - predictions
#   - feature rankings
#   - SHAP values
#   - scientific interpretation
#
# No SHAP ranking/figure is calculated here.
# ============================================================

EXPECTED_D1_MANIFEST_SHA = (
    "819ae644fc5097d8df802b04b64f04b7"
    "4b29434e95e8ae3a10b104c87142c2a7"
)

if "MODEL_OUTPUT_DIR" not in globals():
    raise RuntimeError(
        "Önce MODEL_OUTPUT_DIR tanımlı temel hücreyi çalıştır."
    )

# ------------------------------------------------------------
# 1. Guard 38C-D1 diagnostic
# ------------------------------------------------------------

d1_sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "38C_D1_treeshap_additivity_numerical_diagnostic_manifest_SHA256.txt",
)

if not os.path.exists(d1_sha_path):
    raise FileNotFoundError(d1_sha_path)

with open(d1_sha_path, "r", encoding="utf-8") as fh:
    observed_d1_sha = fh.read().strip()

if observed_d1_sha != EXPECTED_D1_MANIFEST_SHA:
    raise RuntimeError(
        "38C-D1 manifest SHA mismatch: " + observed_d1_sha
    )

print("38C-D1 diagnostic SHA guard: PASS")

# ------------------------------------------------------------
# 2. Read diagnostic table and verify the observed numerical scale
# ------------------------------------------------------------

diagnostic_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "38C_D1_treeshap_additivity_numerical_diagnostic.csv",
)

if not os.path.exists(diagnostic_path):
    raise FileNotFoundError(diagnostic_path)

diag = pd.read_csv(diagnostic_path)

if len(diag) != 5:
    raise RuntimeError(
        "38C-D1 diagnostic must contain exactly five outer folds."
    )

if not diag["allclose_atol1e_5_rtol1e_5"].astype(bool).all():
    raise RuntimeError(
        "Not all folds satisfy np.allclose(atol=1e-5, rtol=1e-5)."
    )

max_margin_diff = float(
    diag["float64_sum_max_abs_margin_diff"].max()
)
max_mean_margin_diff = float(
    diag["float64_sum_mean_abs_margin_diff"].max()
)
max_probability_diff = float(
    diag["max_abs_probability_diff_from_contrib_sum"].max()
)
max_mean_probability_diff = float(
    diag["mean_abs_probability_diff_from_contrib_sum"].max()
)

# Conservative numerical-equivalence limits.
# These are not inferential cutoffs; they only document acceptable
# floating-point agreement for native float32 XGBoost TreeSHAP.
LOCKED_RULES = {
    "margin_allclose_atol": 1e-5,
    "margin_allclose_rtol": 1e-5,
    "maximum_absolute_margin_difference": 1e-5,
    "maximum_absolute_probability_difference": 1e-6,
    "mean_absolute_probability_difference": 1e-7,
}

if max_margin_diff > LOCKED_RULES[
    "maximum_absolute_margin_difference"
]:
    raise RuntimeError(
        "Observed maximum margin discrepancy exceeds the amended limit."
    )

if max_probability_diff > LOCKED_RULES[
    "maximum_absolute_probability_difference"
]:
    raise RuntimeError(
        "Observed maximum probability discrepancy exceeds the amended limit."
    )

if max_mean_probability_diff > LOCKED_RULES[
    "mean_absolute_probability_difference"
]:
    raise RuntimeError(
        "Observed mean probability discrepancy exceeds the amended limit."
    )

# ------------------------------------------------------------
# 3. Lock amendment
# ------------------------------------------------------------

amendment = {
    "analysis_version": "38C-D2",
    "analysis_type":
        "treeshap_numerical_additivity_tolerance_amendment_lock",

    "reason": (
        "The first 38C execution used a mean absolute margin-difference "
        "criterion of 1e-6 and stopped in outer fold 1. 38C-D1 showed "
        "that this discrepancy is the expected numerical scale of native "
        "float32 XGBoost TreeSHAP accumulation: all five folds satisfy "
        "np.allclose(atol=1e-5, rtol=1e-5), maximum absolute margin "
        "difference is below 8e-6, and the corresponding maximum "
        "probability-space discrepancy is below 7e-7."
    ),

    "source_diagnostic_sha256":
        EXPECTED_D1_MANIFEST_SHA,

    "observed_diagnostic_extrema": {
        "maximum_absolute_margin_difference":
            max_margin_diff,
        "maximum_fold_mean_absolute_margin_difference":
            max_mean_margin_diff,
        "maximum_absolute_probability_difference":
            max_probability_diff,
        "maximum_fold_mean_absolute_probability_difference":
            max_mean_probability_diff,
    },

    "locked_numerical_equivalence_rule": {
        "margin_check":
            "numpy.allclose(TreeSHAP_sum, raw_margin, atol=1e-5, rtol=1e-5)",
        "margin_allclose_atol":
            LOCKED_RULES["margin_allclose_atol"],
        "margin_allclose_rtol":
            LOCKED_RULES["margin_allclose_rtol"],
        "maximum_absolute_margin_difference":
            LOCKED_RULES["maximum_absolute_margin_difference"],
        "maximum_absolute_probability_difference":
            LOCKED_RULES["maximum_absolute_probability_difference"],
        "mean_absolute_probability_difference":
            LOCKED_RULES["mean_absolute_probability_difference"],
    },

    "scientific_scope": (
        "Numerical floating-point equivalence only; no change to model, "
        "predictions, SHAP values, ranking rules, plots, or interpretation."
    ),

    "feature_ranking_performed": False,
    "shap_figure_generated": False,
    "shap_interpretation_performed": False,
    "model_refit_performed": False,
    "retuning_performed": False,
    "candidate_reselection_performed": False,
    "recalibration_performed": False,
    "patient_level_data_written_to_drive": False,
    "bigquery_dml_used": False,
}

amendment_text = json.dumps(
    amendment,
    indent=2,
    sort_keys=True,
)

amendment_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "38C_D2_locked_treeshap_numerical_additivity_amendment.json",
)

sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "38C_D2_locked_treeshap_numerical_additivity_amendment_SHA256.txt",
)

with open(
    amendment_path,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(amendment_text)

amendment_sha = hashlib.sha256(
    amendment_text.encode("utf-8")
).hexdigest()

with open(
    sha_path,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(amendment_sha + "\n")

print("\n38C-D2 LOCKED NUMERICAL EQUIVALENCE RULE")
print("Margin: np.allclose(atol=1e-5, rtol=1e-5)")
print("Maximum absolute margin difference <= 1e-5")
print("Maximum absolute probability difference <= 1e-6")
print("Mean absolute probability difference <= 1e-7")

print("\nObserved diagnostic extrema:")
print("max absolute margin difference =", max_margin_diff)
print("max fold mean margin difference =", max_mean_margin_diff)
print("max absolute probability difference =", max_probability_diff)
print("max fold mean probability difference =", max_mean_probability_diff)

print("\n38C-D2 amendment SHA-256:")
print(amendment_sha)

print("\nSaved:")
print(amendment_path)
print(sha_path)

print(
    "\n38C-D2 PASS: TreeSHAP numerical additivity rule is formally "
    "amended and locked before feature ranking or SHAP interpretation."
)

In [ ]:

import os
import json
import math
import hashlib
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import xgboost as xgb

from IPython.display import display

print("STARTING VERIFIED OUTER-TEST TREESHAP ANALYSIS — CODE VERSION 38C-R1")

# ============================================================
# 38C-R1 — VERIFIED OUTER-TEST TREESHAP ANALYSIS
#
# Upstream requirements:
#   - 38A explainability protocol locked before SHAP inspection
#   - 38B reconstructed all five locked XGBoost models and reproduced
#     stored outer-test raw probabilities EXACTLY (max/mean diff = 0)
#
# Primary scientific rules:
#   - explain OUTER-TEST patients only
#   - each patient explained by the model for their own outer test fold
#   - folds 1-4 use XGB04; fold 5 uses XGB06
#   - native XGBoost TreeSHAP, raw margin/log-odds scale\n#   - numerical additivity assessed using the formally locked 38C-D2 rule
#   - primary global importance = source-predictor grouped mean(|SHAP|)
#   - top 20 chosen by deterministic pooled importance ranking
#   - a-priori kidney variables always reported
#   - SHAP is descriptive model attribution, NOT causality
#
# Privacy:
#   - patient-level SHAP stays in RAM only
#   - no patient-level SHAP or prediction table is written to Drive/BQ
# ============================================================

EXPECTED_38A_PROTOCOL_SHA = (
    "06fe49d742e43aedfb518e805f12949f5"
    "2cc926e880a57683144dd6915e30a49"
)

EXPECTED_38B_MANIFEST_SHA = (
    "e6f0d5e03a4ac49bd2ef9d02c9ad4163"
    "606017b41501514052c21e163e6f32b1"
)

EXPECTED_38C_D2_AMENDMENT_SHA = (
    "bcdc5cfb10108c30b98f11b79b472c0d"
    "1768673d1d63ba7ec9ee1108c8f08933"
)

EXPECTED_ROWS = 58491
EXPECTED_EVENTS = 3032
EXPECTED_HOSPITALS = 198

EXPECTED_SELECTED = {
    1: "XGB04",
    2: "XGB04",
    3: "XGB04",
    4: "XGB04",
    5: "XGB06",
}

APRIORI_KIDNEY_VARIABLES = [
    "x_reference_creatinine",
    "x_stage1_at_landmark",
    "x_lab_creatinine_last",
]

TOP_SOURCE_N = 20
TOP_TRANSFORMED_N = 20
TOP_DEPENDENCE_N = 5

# Visualization-only deterministic sampling.
# This does NOT affect any importance/ranking/statistical calculation.
VISUALIZATION_MAX_POINTS = 8000
VISUALIZATION_SEED = 20260724

# Native TreeSHAP numerical-equivalence rule.
# Formally locked in 38C-D2 after the diagnostic-only 38C-D1 run.
# These are floating-point consistency checks, not inferential thresholds.
ADDITIVITY_ATOL = 1e-5
ADDITIVITY_RTOL = 1e-5
ADDITIVITY_MAX_ABS_MARGIN_TOL = 1e-5
ADDITIVITY_MAX_ABS_PROB_TOL = 1e-6
ADDITIVITY_MEAN_ABS_PROB_TOL = 1e-7

required_runtime = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
]

missing_runtime = [
    x for x in required_runtime
    if x not in globals()
]

if missing_runtime:
    raise RuntimeError(
        "Eksik çalışma nesneleri: "
        + ", ".join(missing_runtime)
        + ". Önce 07A/07B temel hücrelerini çalıştır."
    )

if len(core_df_07B) != EXPECTED_ROWS:
    raise RuntimeError("Cohort row count is not 58,491.")

if int(core_df_07B["label_stage23"].sum()) != EXPECTED_EVENTS:
    raise RuntimeError("Cohort event count is not 3,032.")

if core_df_07B["group_hospital"].astype(str).nunique() != EXPECTED_HOSPITALS:
    raise RuntimeError("Cohort hospital count is not 198.")

# ------------------------------------------------------------
# 1. SHA guards
# ------------------------------------------------------------

guards = [
    (
        "38A_locked_xgboost_treeshap_explainability_protocol_v1_SHA256.txt",
        EXPECTED_38A_PROTOCOL_SHA,
        "38A explainability protocol",
    ),
    (
        "38B_verified_xgboost_reconstruction_manifest_SHA256.txt",
        EXPECTED_38B_MANIFEST_SHA,
        "38B verified reconstruction",
    ),
    (
        "38C_D2_locked_treeshap_numerical_additivity_amendment_SHA256.txt",
        EXPECTED_38C_D2_AMENDMENT_SHA,
        "38C-D2 numerical additivity amendment",
    ),
]

for filename, expected_sha, label in guards:
    path = os.path.join(MODEL_OUTPUT_DIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    with open(path, "r", encoding="utf-8") as fh:
        observed = fh.read().strip()

    if observed != expected_sha:
        raise RuntimeError(
            f"{label} SHA mismatch: {observed}"
        )

    print(f"{label} SHA guard: PASS")

# ------------------------------------------------------------
# 2. Verify 38B artifact manifest and hashes
# ------------------------------------------------------------

artifact_manifest_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "38B_verified_xgboost_artifact_manifest.csv",
)

if not os.path.exists(artifact_manifest_path):
    raise FileNotFoundError(artifact_manifest_path)

artifact_manifest = pd.read_csv(
    artifact_manifest_path
)

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(
            lambda: fh.read(1024 * 1024),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()

for _, row in artifact_manifest.iterrows():
    path = str(row["path"])
    expected = str(row["sha256"])

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    observed = file_sha256(path)

    if observed != expected:
        raise RuntimeError(
            f"Artifact SHA mismatch: {path}"
        )

print("All verified 38B model/preprocessor artifact SHA guards: PASS")

verified_dir = os.path.join(
    MODEL_OUTPUT_DIR,
    "38B_verified_xgboost_models",
)

# ------------------------------------------------------------
# 3. Prepare original outer-test feature data
# ------------------------------------------------------------

X_all = core_df_07B[
    predictor_columns_07B
].copy()

for column in numeric_columns_07B:
    X_all[column] = pd.to_numeric(
        X_all[column],
        errors="coerce",
    ).astype("float64")

for column in categorical_columns_07B:
    s = X_all[column].astype("object")
    X_all[column] = s.where(
        pd.notna(s),
        np.nan,
    )

meta = core_df_07B[
    [
        "id_row",
        "outer_fold",
        "group_hospital",
        "label_stage23",
    ]
].copy()

meta["id_row"] = meta["id_row"].astype(str)
meta["outer_fold"] = pd.to_numeric(
    meta["outer_fold"],
    errors="raise",
).astype(np.int64)
meta["group_hospital"] = meta[
    "group_hospital"
].astype(str)
meta["label_stage23"] = pd.to_numeric(
    meta["label_stage23"],
    errors="raise",
).astype(np.int64)

# ------------------------------------------------------------
# 4. Helpers for transformed-column -> source-predictor mapping
# ------------------------------------------------------------

def build_source_mapping(preprocessor, processed_names):
    """
    Returns:
      source_by_transformed_index: list[str] length = processed columns
      source_to_indices: dict[str, list[int]]
    """
    numeric_pipe = preprocessor.named_transformers_[
        "numeric"
    ]
    numeric_imputer = numeric_pipe.named_steps[
        "imputer"
    ]

    categorical_pipe = preprocessor.named_transformers_[
        "categorical"
    ]
    onehot = categorical_pipe.named_steps[
        "onehot"
    ]

    source_by_index = []

    # Numeric value columns are first and preserve numeric-column order.
    source_by_index.extend(
        list(numeric_columns_07B)
    )

    # Then SimpleImputer missing-indicator columns.
    indicator_features = getattr(
        numeric_imputer.indicator_,
        "features_",
        np.array([], dtype=int),
    )

    for feature_idx in indicator_features:
        source_by_index.append(
            numeric_columns_07B[
                int(feature_idx)
            ]
        )

    # Then OneHotEncoder columns grouped by input categorical predictor.
    for cat_col, categories in zip(
        categorical_columns_07B,
        onehot.categories_,
    ):
        for _ in categories:
            source_by_index.append(
                cat_col
            )

    if len(source_by_index) != len(processed_names):
        raise RuntimeError(
            "Transformed source mapping length mismatch: "
            f"{len(source_by_index)} vs {len(processed_names)}"
        )

    source_to_indices = defaultdict(list)

    for idx, source in enumerate(
        source_by_index
    ):
        source_to_indices[source].append(
            idx
        )

    if set(source_to_indices.keys()) != set(
        predictor_columns_07B
    ):
        missing = (
            set(predictor_columns_07B)
            - set(source_to_indices.keys())
        )
        extra = (
            set(source_to_indices.keys())
            - set(predictor_columns_07B)
        )
        raise RuntimeError(
            f"Source mapping mismatch. missing={missing}, extra={extra}"
        )

    return (
        source_by_index,
        dict(source_to_indices),
    )

# ------------------------------------------------------------
# 5. Pass 1: exact outer-test TreeSHAP
# ------------------------------------------------------------

pooled_source_abs_sum = defaultdict(float)
pooled_source_signed_sum = defaultdict(float)

pooled_transformed_abs_sum = None
pooled_transformed_signed_sum = None

fold_source_rows = []
fold_transformed_rows = []
additivity_rows = []

# Patient-level SHAP stays RAM-only.
shap_by_fold = {}
test_index_by_fold = {}

reference_processed_names = None
reference_source_by_index = None
reference_source_to_indices = None

for outer_fold in range(1, 6):
    candidate_id = EXPECTED_SELECTED[
        outer_fold
    ]

    print(
        f"\nComputing verified native TreeSHAP for outer fold "
        f"{outer_fold} ({candidate_id})..."
    )

    model_path = os.path.join(
        verified_dir,
        f"38B_verified_xgboost_outer{outer_fold}_{candidate_id}.json",
    )
    preprocessor_path = os.path.join(
        verified_dir,
        f"38B_verified_preprocessor_outer{outer_fold}.joblib",
    )
    feature_names_path = os.path.join(
        verified_dir,
        f"38B_verified_processed_feature_names_outer{outer_fold}.json",
    )

    preprocessor = joblib.load(
        preprocessor_path
    )

    with open(
        feature_names_path,
        "r",
        encoding="utf-8",
    ) as fh:
        processed_names = json.load(fh)

    processed_names = [
        str(x)
        for x in processed_names
    ]

    if reference_processed_names is None:
        reference_processed_names = processed_names

        (
            reference_source_by_index,
            reference_source_to_indices,
        ) = build_source_mapping(
            preprocessor,
            processed_names,
        )
    else:
        if processed_names != reference_processed_names:
            raise RuntimeError(
                f"Outer fold {outer_fold}: processed feature-name space differs."
            )

        (
            fold_source_by_index,
            fold_source_to_indices,
        ) = build_source_mapping(
            preprocessor,
            processed_names,
        )

        if fold_source_by_index != reference_source_by_index:
            raise RuntimeError(
                f"Outer fold {outer_fold}: source mapping differs."
            )

    test_mask = (
        meta["outer_fold"].to_numpy()
        == outer_fold
    )

    test_indices = np.flatnonzero(
        test_mask
    )

    X_test = (
        X_all.loc[
            test_mask,
            predictor_columns_07B,
        ]
        .reset_index(drop=True)
    )

    X_test_processed = preprocessor.transform(
        X_test
    )

    if X_test_processed.shape[1] != len(
        processed_names
    ):
        raise RuntimeError(
            f"Outer fold {outer_fold}: transformed-column count mismatch."
        )

    booster = xgb.Booster()
    booster.load_model(
        model_path
    )

    dtest = xgb.DMatrix(
        X_test_processed
    )

    raw_margin = booster.predict(
        dtest,
        output_margin=True,
    ).astype(np.float64)

    contributions = booster.predict(
        dtest,
        pred_contribs=True,
        approx_contribs=False,
    ).astype(np.float64)

    if contributions.shape != (
        len(X_test),
        len(processed_names) + 1,
    ):
        raise RuntimeError(
            f"Outer fold {outer_fold}: unexpected SHAP contribution shape "
            f"{contributions.shape}."
        )

    shap_values = contributions[
        :, :-1
    ]
    expected_value = contributions[
        :, -1
    ]

    reconstructed_margin = (
        shap_values.sum(axis=1)
        + expected_value
    )

    additivity_diff = np.abs(
        reconstructed_margin
        - raw_margin
    )

    max_additivity_diff = float(
        additivity_diff.max()
    )
    mean_additivity_diff = float(
        additivity_diff.mean()
    )

    margin_allclose = bool(
        np.allclose(
            reconstructed_margin,
            raw_margin,
            atol=ADDITIVITY_ATOL,
            rtol=ADDITIVITY_RTOL,
        )
    )

    # Translate the tiny raw-margin discrepancy into probability space.
    # This does not recalibrate the model; it is only a numerical diagnostic.
    reconstructed_probability = (
        1.0
        / (
            1.0
            + np.exp(
                -np.clip(
                    reconstructed_margin,
                    -50.0,
                    50.0,
                )
            )
        )
    )

    raw_margin_probability = (
        1.0
        / (
            1.0
            + np.exp(
                -np.clip(
                    raw_margin,
                    -50.0,
                    50.0,
                )
            )
        )
    )

    probability_diff = np.abs(
        reconstructed_probability
        - raw_margin_probability
    )

    max_probability_diff = float(
        probability_diff.max()
    )
    mean_probability_diff = float(
        probability_diff.mean()
    )

    additivity_pass = (
        margin_allclose
        and max_additivity_diff
        <= ADDITIVITY_MAX_ABS_MARGIN_TOL
        and max_probability_diff
        <= ADDITIVITY_MAX_ABS_PROB_TOL
        and mean_probability_diff
        <= ADDITIVITY_MEAN_ABS_PROB_TOL
    )

    additivity_rows.append({
        "outer_fold": outer_fold,
        "selected_candidate": candidate_id,
        "test_patients": len(X_test),
        "processed_columns": len(processed_names),
        "expected_value_mean": float(
            expected_value.mean()
        ),
        "maximum_absolute_margin_difference":
            max_additivity_diff,
        "mean_absolute_margin_difference":
            mean_additivity_diff,
        "margin_allclose_atol_1e_5_rtol_1e_5":
            margin_allclose,
        "maximum_absolute_probability_difference":
            max_probability_diff,
        "mean_absolute_probability_difference":
            mean_probability_diff,
        "maximum_allowed_absolute_margin_difference":
            ADDITIVITY_MAX_ABS_MARGIN_TOL,
        "maximum_allowed_absolute_probability_difference":
            ADDITIVITY_MAX_ABS_PROB_TOL,
        "maximum_allowed_mean_probability_difference":
            ADDITIVITY_MEAN_ABS_PROB_TOL,
        "additivity_status":
            "PASS" if additivity_pass else "FAIL",
    })

    if not additivity_pass:
        raise RuntimeError(
            f"Outer fold {outer_fold}: amended TreeSHAP numerical "
            "additivity check FAILED."
        )

    print(
        "TreeSHAP numerical additivity:",
        f"max margin diff={max_additivity_diff:.3g}, "
        f"max probability diff={max_probability_diff:.3g}, "
        f"mean probability diff={mean_probability_diff:.3g} -> PASS"
    )

    abs_shap = np.abs(
        shap_values
    )

    # Transformed-feature level.
    fold_mean_abs = abs_shap.mean(
        axis=0
    )
    fold_mean_signed = shap_values.mean(
        axis=0
    )

    if pooled_transformed_abs_sum is None:
        pooled_transformed_abs_sum = np.zeros(
            len(processed_names),
            dtype=np.float64,
        )
        pooled_transformed_signed_sum = np.zeros(
            len(processed_names),
            dtype=np.float64,
        )

    pooled_transformed_abs_sum += abs_shap.sum(
        axis=0
    )
    pooled_transformed_signed_sum += shap_values.sum(
        axis=0
    )

    for idx, transformed_name in enumerate(
        processed_names
    ):
        fold_transformed_rows.append({
            "outer_fold": outer_fold,
            "selected_candidate": candidate_id,
            "transformed_feature":
                transformed_name,
            "source_predictor":
                reference_source_by_index[idx],
            "mean_absolute_shap":
                float(fold_mean_abs[idx]),
            "mean_signed_shap":
                float(fold_mean_signed[idx]),
        })

    # Source-predictor grouped level.
    for source in predictor_columns_07B:
        indices = reference_source_to_indices[
            source
        ]

        patient_abs_group = abs_shap[
            :, indices
        ].sum(axis=1)

        patient_signed_group = shap_values[
            :, indices
        ].sum(axis=1)

        mean_abs_group = float(
            patient_abs_group.mean()
        )
        mean_signed_group = float(
            patient_signed_group.mean()
        )

        pooled_source_abs_sum[
            source
        ] += float(
            patient_abs_group.sum()
        )

        pooled_source_signed_sum[
            source
        ] += float(
            patient_signed_group.sum()
        )

        fold_source_rows.append({
            "outer_fold": outer_fold,
            "selected_candidate":
                candidate_id,
            "source_predictor":
                source,
            "transformed_column_count":
                len(indices),
            "mean_absolute_grouped_shap":
                mean_abs_group,
            "mean_signed_grouped_shap":
                mean_signed_group,
        })

    # Keep patient-level SHAP in RAM only.
    shap_by_fold[
        outer_fold
    ] = shap_values.astype(
        np.float32,
        copy=False,
    )

    test_index_by_fold[
        outer_fold
    ] = test_indices

    del (
        X_test,
        X_test_processed,
        dtest,
        contributions,
        raw_margin,
        reconstructed_margin,
        additivity_diff,
        reconstructed_probability,
        raw_margin_probability,
        probability_diff,
        abs_shap,
        booster,
        preprocessor,
    )

# ------------------------------------------------------------
# 6. Aggregate global rankings
# ------------------------------------------------------------

additivity_audit = pd.DataFrame(
    additivity_rows
)

fold_source_importance = pd.DataFrame(
    fold_source_rows
)

fold_transformed_importance = pd.DataFrame(
    fold_transformed_rows
)

pooled_source_rows = []

for source in predictor_columns_07B:
    pooled_source_rows.append({
        "source_predictor": source,
        "transformed_column_count":
            len(
                reference_source_to_indices[
                    source
                ]
            ),
        "pooled_mean_absolute_grouped_shap":
            pooled_source_abs_sum[source]
            / EXPECTED_ROWS,
        "pooled_mean_signed_grouped_shap":
            pooled_source_signed_sum[source]
            / EXPECTED_ROWS,
    })

pooled_source_importance = pd.DataFrame(
    pooled_source_rows
).sort_values(
    [
        "pooled_mean_absolute_grouped_shap",
        "source_predictor",
    ],
    ascending=[False, True],
    kind="stable",
).reset_index(drop=True)

pooled_source_importance.insert(
    0,
    "pooled_rank",
    np.arange(
        1,
        len(pooled_source_importance) + 1,
    ),
)

pooled_transformed_importance = pd.DataFrame({
    "transformed_feature":
        reference_processed_names,
    "source_predictor":
        reference_source_by_index,
    "pooled_mean_absolute_shap":
        pooled_transformed_abs_sum
        / EXPECTED_ROWS,
    "pooled_mean_signed_shap":
        pooled_transformed_signed_sum
        / EXPECTED_ROWS,
}).sort_values(
    [
        "pooled_mean_absolute_shap",
        "transformed_feature",
    ],
    ascending=[False, True],
    kind="stable",
).reset_index(drop=True)

pooled_transformed_importance.insert(
    0,
    "pooled_rank",
    np.arange(
        1,
        len(pooled_transformed_importance) + 1,
    ),
)

top20_source = pooled_source_importance.head(
    TOP_SOURCE_N
).copy()

top20_transformed = pooled_transformed_importance.head(
    TOP_TRANSFORMED_N
).copy()

kidney_importance = pooled_source_importance.loc[
    pooled_source_importance[
        "source_predictor"
    ].isin(
        APRIORI_KIDNEY_VARIABLES
    )
].copy()

if set(
    kidney_importance["source_predictor"]
) != set(APRIORI_KIDNEY_VARIABLES):
    raise RuntimeError(
        "One or more a-priori kidney variables are absent from source importance."
    )

# ------------------------------------------------------------
# 7. Fold-specific ranks and stability
# ------------------------------------------------------------

fold_source_importance[
    "fold_rank"
] = (
    fold_source_importance.groupby(
        "outer_fold"
    )[
        "mean_absolute_grouped_shap"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

rank_matrix = (
    fold_source_importance.pivot(
        index="source_predictor",
        columns="outer_fold",
        values="fold_rank",
    )
    .sort_index()
)

rank_correlation = rank_matrix.corr(
    method="spearman"
)

rank_correlation.index.name = "outer_fold"
rank_correlation.columns = [
    f"outer_fold_{int(x)}"
    for x in rank_correlation.columns
]

rank_correlation = rank_correlation.reset_index()

top20_presence_rows = []

for source in predictor_columns_07B:
    sub = fold_source_importance.loc[
        fold_source_importance[
            "source_predictor"
        ].eq(source)
    ]

    top20_presence_rows.append({
        "source_predictor": source,
        "top20_presence_count_across_5_folds":
            int(
                (sub["fold_rank"] <= 20).sum()
            ),
        "minimum_fold_rank":
            int(sub["fold_rank"].min()),
        "maximum_fold_rank":
            int(sub["fold_rank"].max()),
        "median_fold_rank":
            float(sub["fold_rank"].median()),
        "pooled_rank":
            int(
                pooled_source_importance.loc[
                    pooled_source_importance[
                        "source_predictor"
                    ].eq(source),
                    "pooled_rank",
                ].iloc[0]
            ),
    })

top20_presence = pd.DataFrame(
    top20_presence_rows
).sort_values(
    [
        "top20_presence_count_across_5_folds",
        "pooled_rank",
    ],
    ascending=[False, True],
    kind="stable",
).reset_index(drop=True)

# ------------------------------------------------------------
# 8. Visualization 1: top-20 source importance
# ------------------------------------------------------------

figure_dir = os.path.join(
    MODEL_OUTPUT_DIR,
    "38C_R1_treeshap_figures",
)
os.makedirs(
    figure_dir,
    exist_ok=True,
)

top20_source_png = os.path.join(
    figure_dir,
    "38C_R1_top20_source_predictor_mean_absolute_treeshap.png",
)

plot_df = top20_source.sort_values(
    "pooled_mean_absolute_grouped_shap",
    ascending=True,
)

fig = plt.figure(
    figsize=(9.0, 7.5)
)
ax = fig.add_subplot(111)

ax.barh(
    plot_df["source_predictor"],
    plot_df[
        "pooled_mean_absolute_grouped_shap"
    ],
)

ax.set_xlabel(
    "Mean absolute grouped TreeSHAP contribution (raw margin)"
)
ax.set_ylabel(
    "Source predictor"
)
ax.set_title(
    "XGBoost Global Feature Attribution Across Outer-Test Patients"
)

fig.tight_layout()

fig.savefig(
    top20_source_png,
    dpi=300,
    bbox_inches="tight",
)

plt.close(fig)

# ------------------------------------------------------------
# 9. Visualization 2: transformed-feature SHAP beeswarm
# ------------------------------------------------------------

name_to_index = {
    name: idx
    for idx, name in enumerate(
        reference_processed_names
    )
}

top_transformed_names = top20_transformed[
    "transformed_feature"
].tolist()

top_transformed_indices = [
    name_to_index[name]
    for name in top_transformed_names
]

# Second pass only for transformed feature values.
transformed_value_by_feature = {
    name: []
    for name in top_transformed_names
}
shap_value_by_feature = {
    name: []
    for name in top_transformed_names
}

for outer_fold in range(1, 6):
    candidate_id = EXPECTED_SELECTED[
        outer_fold
    ]

    preprocessor_path = os.path.join(
        verified_dir,
        f"38B_verified_preprocessor_outer{outer_fold}.joblib",
    )

    preprocessor = joblib.load(
        preprocessor_path
    )

    test_mask = (
        meta["outer_fold"].to_numpy()
        == outer_fold
    )

    X_test = (
        X_all.loc[
            test_mask,
            predictor_columns_07B,
        ]
        .reset_index(drop=True)
    )

    X_proc = preprocessor.transform(
        X_test
    )

    selected_values = X_proc[
        :, top_transformed_indices
    ]

    if hasattr(
        selected_values,
        "toarray",
    ):
        selected_values = selected_values.toarray()

    selected_values = np.asarray(
        selected_values,
        dtype=np.float32,
    )

    fold_shap = shap_by_fold[
        outer_fold
    ]

    for local_j, name in enumerate(
        top_transformed_names
    ):
        transformed_value_by_feature[
            name
        ].append(
            selected_values[
                :, local_j
            ]
        )
        shap_value_by_feature[
            name
        ].append(
            fold_shap[
                :, top_transformed_indices[
                    local_j
                ]
            ]
        )

    del (
        preprocessor,
        X_test,
        X_proc,
        selected_values,
    )

beeswarm_png = os.path.join(
    figure_dir,
    "38C_R1_top20_transformed_feature_treeshap_beeswarm.png",
)

rng = np.random.default_rng(
    VISUALIZATION_SEED
)

fig = plt.figure(
    figsize=(10.0, 8.5)
)
ax = fig.add_subplot(111)

# Reverse order so rank 1 appears at top.
plot_names = list(
    reversed(
        top_transformed_names
    )
)

for y_pos, name in enumerate(
    plot_names
):
    shap_vals = np.concatenate(
        shap_value_by_feature[
            name
        ]
    ).astype(float)

    feat_vals = np.concatenate(
        transformed_value_by_feature[
            name
        ]
    ).astype(float)

    n = len(shap_vals)

    if n > VISUALIZATION_MAX_POINTS:
        chosen = rng.choice(
            n,
            size=VISUALIZATION_MAX_POINTS,
            replace=False,
        )
    else:
        chosen = np.arange(n)

    # Small deterministic vertical jitter for density visibility.
    jitter = rng.normal(
        loc=0.0,
        scale=0.10,
        size=len(chosen),
    )

    ax.scatter(
        shap_vals[chosen],
        np.full(
            len(chosen),
            y_pos,
            dtype=float,
        ) + jitter,
        c=feat_vals[chosen],
        s=5,
        alpha=0.45,
        linewidths=0,
    )

ax.axvline(
    0.0,
    linewidth=1.0,
)

ax.set_yticks(
    np.arange(
        len(plot_names)
    )
)
ax.set_yticklabels(
    plot_names
)

ax.set_xlabel(
    "TreeSHAP contribution to XGBoost raw margin"
)
ax.set_ylabel(
    "Transformed feature"
)
ax.set_title(
    "Top 20 Transformed-Feature TreeSHAP Contributions"
)

fig.tight_layout()

fig.savefig(
    beeswarm_png,
    dpi=300,
    bbox_inches="tight",
)

plt.close(fig)

# ------------------------------------------------------------
# 10. Predefined dependence plots:
#     top-5 pooled source predictors + 3 a-priori kidney predictors
# ------------------------------------------------------------

top5_sources = top20_source.head(
    TOP_DEPENDENCE_N
)["source_predictor"].tolist()

dependence_sources = []

for source in (
    top5_sources
    + APRIORI_KIDNEY_VARIABLES
):
    if source not in dependence_sources:
        dependence_sources.append(
            source
        )

dependence_manifest_rows = []

for source in dependence_sources:
    source_indices = reference_source_to_indices[
        source
    ]

    pooled_grouped_shap_parts = []
    pooled_source_value_parts = []

    for outer_fold in range(1, 6):
        test_indices = test_index_by_fold[
            outer_fold
        ]

        fold_shap = shap_by_fold[
            outer_fold
        ]

        grouped_signed_shap = fold_shap[
            :, source_indices
        ].sum(axis=1)

        pooled_grouped_shap_parts.append(
            grouped_signed_shap.astype(
                np.float32,
                copy=False,
            )
        )

        raw_values = X_all.iloc[
            test_indices
        ][source]

        pooled_source_value_parts.append(
            raw_values.reset_index(
                drop=True
            )
        )

    grouped_shap_all = np.concatenate(
        pooled_grouped_shap_parts
    ).astype(float)

    raw_source_all = pd.concat(
        pooled_source_value_parts,
        ignore_index=True,
    )

    source_rank = int(
        pooled_source_importance.loc[
            pooled_source_importance[
                "source_predictor"
            ].eq(source),
            "pooled_rank",
        ].iloc[0]
    )

    safe_name = "".join(
        ch if (
            ch.isalnum()
            or ch in {"_", "-"}
        ) else "_"
        for ch in source
    )

    figure_path = os.path.join(
        figure_dir,
        f"38C_R1_dependence_{safe_name}.png",
    )

    fig = plt.figure(
        figsize=(8.0, 5.5)
    )
    ax = fig.add_subplot(111)

    if source in numeric_columns_07B:
        x_numeric = pd.to_numeric(
            raw_source_all,
            errors="coerce",
        ).to_numpy(dtype=float)

        valid = np.isfinite(
            x_numeric
        )

        valid_indices = np.flatnonzero(
            valid
        )

        if len(valid_indices) > VISUALIZATION_MAX_POINTS:
            chosen = rng.choice(
                valid_indices,
                size=VISUALIZATION_MAX_POINTS,
                replace=False,
            )
        else:
            chosen = valid_indices

        ax.scatter(
            x_numeric[chosen],
            grouped_shap_all[chosen],
            s=7,
            alpha=0.35,
            linewidths=0,
        )

        ax.set_xlabel(
            source
        )

        missing_count = int(
            (~valid).sum()
        )

        ax.set_title(
            f"TreeSHAP Dependence: {source} "
            f"(pooled rank {source_rank}; missing n={missing_count})"
        )

    else:
        cat = raw_source_all.astype(
            "object"
        ).where(
            pd.notna(raw_source_all),
            "__MISSING__",
        ).astype(str)

        category_order = (
            cat.value_counts(
                dropna=False
            )
            .index
            .tolist()
        )

        data = [
            grouped_shap_all[
                cat.to_numpy() == level
            ]
            for level in category_order
        ]

        ax.boxplot(
            data,
            labels=category_order,
            showfliers=False,
        )

        ax.set_xlabel(
            source
        )

        ax.tick_params(
            axis="x",
            labelrotation=45,
        )

        ax.set_title(
            f"TreeSHAP by Category: {source} "
            f"(pooled rank {source_rank})"
        )

    ax.axhline(
        0.0,
        linewidth=1.0,
    )

    ax.set_ylabel(
        "Grouped signed TreeSHAP contribution (raw margin)"
    )

    fig.tight_layout()

    fig.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)

    dependence_manifest_rows.append({
        "source_predictor": source,
        "pooled_rank": source_rank,
        "selection_basis": (
            "top5_pooled_importance"
            if source in top5_sources
            else "a_priori_kidney_variable"
        ),
        "figure_path": figure_path,
    })

dependence_manifest = pd.DataFrame(
    dependence_manifest_rows
)

# ------------------------------------------------------------
# 11. Save aggregate tables only
# ------------------------------------------------------------

paths = {
    "additivity_audit": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_R1_treeshap_additivity_audit.csv",
    ),
    "fold_source_importance": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_R1_fold_source_predictor_treeshap_importance.csv",
    ),
    "pooled_source_importance": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_R1_pooled_source_predictor_treeshap_importance.csv",
    ),
    "top20_source": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_R1_top20_source_predictor_treeshap_importance.csv",
    ),
    "kidney_importance": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_R1_apriori_kidney_variable_treeshap_importance.csv",
    ),
    "fold_transformed_importance": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_R1_fold_transformed_feature_treeshap_importance.csv",
    ),
    "pooled_transformed_importance": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_R1_pooled_transformed_feature_treeshap_importance.csv",
    ),
    "top20_transformed": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_R1_top20_transformed_feature_treeshap_importance.csv",
    ),
    "fold_rank_correlation": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_R1_fold_source_rank_spearman_correlation.csv",
    ),
    "top20_presence": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_R1_source_top20_presence_frequency_across_folds.csv",
    ),
    "dependence_manifest": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_R1_dependence_plot_manifest.csv",
    ),
    "manifest": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_R1_verified_outer_test_treeshap_manifest.json",
    ),
    "manifest_sha": os.path.join(
        MODEL_OUTPUT_DIR,
        "38C_R1_verified_outer_test_treeshap_manifest_SHA256.txt",
    ),
}

additivity_audit.to_csv(
    paths["additivity_audit"],
    index=False,
)

fold_source_importance.to_csv(
    paths["fold_source_importance"],
    index=False,
)

pooled_source_importance.to_csv(
    paths["pooled_source_importance"],
    index=False,
)

top20_source.to_csv(
    paths["top20_source"],
    index=False,
)

kidney_importance.to_csv(
    paths["kidney_importance"],
    index=False,
)

fold_transformed_importance.to_csv(
    paths["fold_transformed_importance"],
    index=False,
)

pooled_transformed_importance.to_csv(
    paths["pooled_transformed_importance"],
    index=False,
)

top20_transformed.to_csv(
    paths["top20_transformed"],
    index=False,
)

rank_correlation.to_csv(
    paths["fold_rank_correlation"],
    index=False,
)

top20_presence.to_csv(
    paths["top20_presence"],
    index=False,
)

dependence_manifest.to_csv(
    paths["dependence_manifest"],
    index=False,
)

manifest = {
    "analysis_version": "38C-R1",
    "analysis_type":
        "verified_outer_test_native_xgboost_treeshap",
    "source_38A_protocol_sha256":
        EXPECTED_38A_PROTOCOL_SHA,
    "source_38B_verified_reconstruction_sha256":
        EXPECTED_38B_MANIFEST_SHA,
    "source_38C_D2_numerical_additivity_amendment_sha256":
        EXPECTED_38C_D2_AMENDMENT_SHA,
    "patients_explained": EXPECTED_ROWS,
    "hospitals": EXPECTED_HOSPITALS,
    "events": EXPECTED_EVENTS,
    "outer_folds": 5,
    "selected_candidates_by_outer_fold":
        EXPECTED_SELECTED,
    "shap_method":
        "native_xgboost_pred_contribs",
    "shap_scale":
        "raw_margin_log_odds",
    "primary_global_importance":
        "source_predictor_grouped_mean_absolute_shap",
    "top_source_predictors_displayed":
        TOP_SOURCE_N,
    "top_transformed_features_displayed":
        TOP_TRANSFORMED_N,
    "a_priori_kidney_variables":
        APRIORI_KIDNEY_VARIABLES,
    "fold_stability_assessed": True,
    "additivity_check": {
        "rule":
            "np.allclose(raw_margin, TreeSHAP_sum, atol=1e-5, rtol=1e-5) "
            "plus locked probability-space limits from 38C-D2",
        "margin_allclose_atol":
            ADDITIVITY_ATOL,
        "margin_allclose_rtol":
            ADDITIVITY_RTOL,
        "maximum_absolute_margin_difference_tolerance":
            ADDITIVITY_MAX_ABS_MARGIN_TOL,
        "maximum_absolute_probability_difference_tolerance":
            ADDITIVITY_MAX_ABS_PROB_TOL,
        "mean_absolute_probability_difference_tolerance":
            ADDITIVITY_MEAN_ABS_PROB_TOL,
        "all_folds_passed": bool(
            additivity_audit[
                "additivity_status"
            ].eq("PASS").all()
        ),
    },
    "dependence_plot_selection": {
        "top_n_data_driven":
            TOP_DEPENDENCE_N,
        "plus_a_priori_kidney_variables":
            APRIORI_KIDNEY_VARIABLES,
        "visualization_max_points":
            VISUALIZATION_MAX_POINTS,
        "visualization_seed":
            VISUALIZATION_SEED,
    },
    "causal_interpretation_permitted":
        False,
    "patient_level_shap_written_to_drive":
        False,
    "patient_level_shap_written_to_bigquery":
        False,
    "patient_level_shap_retained":
        "RAM_only_during_execution",
    "retuning_performed": False,
    "candidate_reselection_performed":
        False,
    "recalibration_performed": False,
    "bigquery_dml_used": False,
    "figures": {
        "top20_source_importance":
            top20_source_png,
        "top20_transformed_beeswarm":
            beeswarm_png,
        "dependence_figure_directory":
            figure_dir,
    },
    "outputs": paths,
}

manifest_text = json.dumps(
    manifest,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
)

with open(
    paths["manifest"],
    "w",
    encoding="utf-8",
) as fh:
    fh.write(
        manifest_text
    )

manifest_sha = hashlib.sha256(
    manifest_text.encode("utf-8")
).hexdigest()

with open(
    paths["manifest_sha"],
    "w",
    encoding="utf-8",
) as fh:
    fh.write(
        manifest_sha + "\n"
    )

# ------------------------------------------------------------
# 12. Compact display
# ------------------------------------------------------------

print("\n38C-R1 TREESHAP ADDITIVITY AUDIT")
display(
    additivity_audit
)

print("\n38C-R1 TOP 20 SOURCE-PREDICTOR IMPORTANCE")
display(
    top20_source
)

print("\n38C-R1 A-PRIORI KIDNEY VARIABLE IMPORTANCE")
display(
    kidney_importance
)

print("\n38C-R1 FOLD SOURCE-RANK SPEARMAN CORRELATION")
display(
    rank_correlation
)

print("\n38C-R1 TOP-20 PRESENCE FREQUENCY — FIRST 30")
display(
    top20_presence.head(30)
)

print("\n38C-R1 DEPENDENCE PLOT MANIFEST")
display(
    dependence_manifest
)

print("\n38C-R1 manifest SHA-256:")
print(
    manifest_sha
)

print("\nFigures saved under:")
print(
    figure_dir
)

print(
    "\n38C-R1 PASS: Verified outer-test native TreeSHAP analysis completed."
)
print(
    "All global rankings were generated by the pre-locked deterministic rules."
)
print(
    "SHAP values are descriptive model attributions, not causal effects."
)
print(
    "No patient-level SHAP values were written to Drive or BigQuery."
)


In [ ]:
import os
import json
import hashlib
import glob

print("STARTING RENAL-MARKER ABLATION SENSITIVITY PROTOCOL LOCK — CODE VERSION 39A-R2")

# ============================================================
# 39A-R2 — RENAL-MARKER ABLATION SENSITIVITY PROTOCOL LOCK
#
# Revision reason:
#   39A-R1 stopped because the assumed feature
#       x_lab_creatinine_delta
#   is NOT present in the locked 159-predictor core registry.
#
# Scientific correction:
#   Instead of assuming a hand-written list of creatinine summary
#   variables, the ablation rule is now defined directly from the
#   already-locked predictor registry:
#
#     REMOVE:
#       1) x_reference_creatinine
#       2) x_stage1_at_landmark
#       3) every existing source predictor whose name starts with
#          "x_lab_creatinine_"
#
#   This is deterministic, registry-based, and locked BEFORE any
#   ablated-model performance result is calculated or viewed.
#
# No model is fitted in 39A-R2.
# No performance result is calculated or viewed.
# ============================================================

EXPECTED_38C_R1_MANIFEST_SHA = (
    "0469f6e3a1f31c5621904d4fbfd3c359"
    "04ccd435eca805ddc60f833f925ec521"
)

EXPECTED_XGB_PROTOCOL_SHA = (
    "3434db5dd0b4b5145950fc07fba3d007"
    "829738d800b88ec40fbdb09879188264"
)

EXPECTED_INNER_FOLD_SHA = (
    "34b3fed6f6216d7a59afb8303e081fb4"
    "8c9dade9bab1bb367e395372098e9b50"
)

EXPECTED_ROWS = 58491
EXPECTED_EVENTS = 3032
EXPECTED_HOSPITALS = 198
EXPECTED_CORE_PREDICTORS = 159

LOCKED_SELECTED_CANDIDATES = {
    "1": "XGB04",
    "2": "XGB04",
    "3": "XGB04",
    "4": "XGB04",
    "5": "XGB06",
}

BOOTSTRAP_REPLICATES = 2000
BOOTSTRAP_SEED = 20260723

required_runtime = [
    "MODEL_OUTPUT_DIR",
    "core_df_07B",
    "predictor_columns_07B",
]

missing_runtime = [
    name for name in required_runtime
    if name not in globals()
]

if missing_runtime:
    raise RuntimeError(
        "Eksik çalışma nesneleri: "
        + ", ".join(missing_runtime)
        + ". Önce 07B temel hücrelerini çalıştır."
    )

# ------------------------------------------------------------
# 1. Direct upstream SHA guards
# ------------------------------------------------------------

direct_guards = [
    (
        "38C_R1_verified_outer_test_treeshap_manifest_SHA256.txt",
        EXPECTED_38C_R1_MANIFEST_SHA,
        "38C-R1 verified TreeSHAP",
    ),
    (
        "13A_locked_xgboost_model_protocol_v1_SHA256.txt",
        EXPECTED_XGB_PROTOCOL_SHA,
        "Locked XGBoost protocol",
    ),
]

for filename, expected_sha, label in direct_guards:
    path = os.path.join(
        MODEL_OUTPUT_DIR,
        filename,
    )

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    with open(path, "r", encoding="utf-8") as fh:
        observed_sha = fh.read().strip()

    if observed_sha != expected_sha:
        raise RuntimeError(
            f"{label} SHA mismatch: {observed_sha}"
        )

    print(f"{label} SHA guard: PASS")

# ------------------------------------------------------------
# 2. Robust 07D inner-fold SHA resolution
# ------------------------------------------------------------

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(
            lambda: fh.read(1024 * 1024),
            b"",
        ):
            digest.update(block)
    return digest.hexdigest()


def find_expected_07d_sha():
    candidates = sorted(
        glob.glob(
            os.path.join(
                MODEL_OUTPUT_DIR,
                "07D*",
            )
        )
    )

    if not candidates:
        raise FileNotFoundError(
            "MODEL_OUTPUT_DIR içinde 07D ile başlayan hiçbir dosya bulunamadı."
        )

    for path in candidates:
        if not os.path.isfile(path):
            continue

        ext = os.path.splitext(path)[1].lower()

        if ext not in {".txt", ".json", ".csv"}:
            continue

        try:
            with open(
                path,
                "r",
                encoding="utf-8",
                errors="ignore",
            ) as fh:
                content = fh.read()
        except Exception:
            continue

        if EXPECTED_INNER_FOLD_SHA in content:
            return {
                "resolution_method":
                    "expected_sha_found_inside_07D_artifact",
                "resolved_path":
                    path,
                "resolved_filename":
                    os.path.basename(path),
                "expected_sha256":
                    EXPECTED_INNER_FOLD_SHA,
                "file_sha256":
                    sha256_file(path),
            }

    for path in candidates:
        if not os.path.isfile(path):
            continue

        observed_file_sha = sha256_file(path)

        if observed_file_sha == EXPECTED_INNER_FOLD_SHA:
            return {
                "resolution_method":
                    "07D_file_sha256_matches_expected_sha",
                "resolved_path":
                    path,
                "resolved_filename":
                    os.path.basename(path),
                "expected_sha256":
                    EXPECTED_INNER_FOLD_SHA,
                "file_sha256":
                    observed_file_sha,
            }

    available = [
        os.path.basename(p)
        for p in candidates
        if os.path.isfile(p)
    ]

    raise RuntimeError(
        "Beklenen kilitli 07D SHA değeri mevcut 07D artefaktlarında "
        "bulunamadı.\n"
        f"Expected SHA: {EXPECTED_INNER_FOLD_SHA}\n"
        "Bulunan 07D dosyaları:\n- "
        + "\n- ".join(available)
    )


inner_fold_guard = find_expected_07d_sha()

print("Locked inner hospital folds SHA guard: PASS")
print(
    "Resolved 07D guard artifact:",
    inner_fold_guard["resolved_filename"],
)
print(
    "Resolution method:",
    inner_fold_guard["resolution_method"],
)

# ------------------------------------------------------------
# 3. Cohort / predictor integrity
# ------------------------------------------------------------

if len(core_df_07B) != EXPECTED_ROWS:
    raise RuntimeError(
        f"Expected {EXPECTED_ROWS} patients; found {len(core_df_07B)}."
    )

if int(core_df_07B["label_stage23"].sum()) != EXPECTED_EVENTS:
    raise RuntimeError(
        "Event count is not 3,032."
    )

if core_df_07B["group_hospital"].astype(str).nunique() != EXPECTED_HOSPITALS:
    raise RuntimeError(
        "Hospital count is not 198."
    )

if len(predictor_columns_07B) != EXPECTED_CORE_PREDICTORS:
    raise RuntimeError(
        "Core predictor count is not 159."
    )

predictor_set = set(
    predictor_columns_07B
)

mandatory_direct_renal_markers = [
    "x_reference_creatinine",
    "x_stage1_at_landmark",
]

missing_mandatory = [
    x for x in mandatory_direct_renal_markers
    if x not in predictor_set
]

if missing_mandatory:
    raise RuntimeError(
        "Mandatory direct renal marker(s) missing: "
        + ", ".join(missing_mandatory)
    )

# Deterministic registry-based rule:
# remove every EXISTING x_lab_creatinine_* source predictor.
creatinine_lab_predictors = sorted(
    [
        x for x in predictor_columns_07B
        if str(x).startswith(
            "x_lab_creatinine_"
        )
    ]
)

if len(creatinine_lab_predictors) == 0:
    raise RuntimeError(
        "No x_lab_creatinine_* predictors found in locked core registry."
    )

LOCKED_RENAL_MARKER_ABLATION_SET = (
    mandatory_direct_renal_markers
    + creatinine_lab_predictors
)

# Preserve original predictor order for actual modeling.
locked_ablation_set_lookup = set(
    LOCKED_RENAL_MARKER_ABLATION_SET
)

remaining_predictors = [
    x for x in predictor_columns_07B
    if x not in locked_ablation_set_lookup
]

if len(remaining_predictors) + len(
    LOCKED_RENAL_MARKER_ABLATION_SET
) != EXPECTED_CORE_PREDICTORS:
    raise RuntimeError(
        "Predictor accounting mismatch after registry-based ablation."
    )

# The failed 39A-R1 assumption must not silently reappear.
if "x_lab_creatinine_delta" in predictor_set:
    print(
        "NOTE: x_lab_creatinine_delta actually exists in this runtime "
        "and is therefore included by the prefix rule."
    )
else:
    print(
        "Confirmed: x_lab_creatinine_delta is not in the locked core registry."
    )

print("Cohort integrity guard: PASS")
print("Registry-based direct renal-marker rule: PASS")
print(
    "Existing x_lab_creatinine_* predictors discovered:",
    len(creatinine_lab_predictors),
)
print(
    "Total predictors removed:",
    len(LOCKED_RENAL_MARKER_ABLATION_SET),
)
print(
    "Predictors retained after ablation:",
    len(remaining_predictors),
)

# ------------------------------------------------------------
# 4. Lock protocol
# ------------------------------------------------------------

protocol = {
    "analysis_version": "39A-R2",
    "analysis_type":
        "post_hoc_renal_marker_ablation_sensitivity_protocol_lock",

    "revision_reason": (
        "39A-R1 stopped before protocol creation because "
        "x_lab_creatinine_delta was assumed to exist but is absent "
        "from the locked 159-predictor core registry. 39A-R2 replaces "
        "the hand-written creatinine-summary list with a deterministic "
        "registry-based rule: remove x_reference_creatinine, "
        "x_stage1_at_landmark, and every existing predictor beginning "
        "with x_lab_creatinine_. No ablated-model result had been "
        "calculated or viewed before this correction."
    ),

    "scientific_question": (
        "How much predictive performance remains after removing "
        "direct, outcome-proximal creatinine/KDIGO-derived predictors "
        "from the locked core XGBoost feature set?"
    ),

    "status":
        "exploratory_post_hoc_sensitivity_analysis",

    "cohort": {
        "patients": EXPECTED_ROWS,
        "events": EXPECTED_EVENTS,
        "hospitals": EXPECTED_HOSPITALS,
        "outer_folds": 5,
        "same_as_primary_xgboost_analysis": True,
    },

    "full_model_reference": {
        "model": "locked_xgboost",
        "prediction_table":
            "model_xgb_outer_predictions_all5_v1",
        "performance_reference_probability":
            "fold_specific_platt_calibrated",
    },

    "ablation_definition": {
        "rule": (
            "Remove x_reference_creatinine, x_stage1_at_landmark, "
            "and every existing source predictor in the locked core "
            "registry whose name starts with x_lab_creatinine_."
        ),
        "source_predictors_removed":
            LOCKED_RENAL_MARKER_ABLATION_SET,
        "creatinine_lab_predictors_removed":
            creatinine_lab_predictors,
        "removed_predictor_count":
            len(LOCKED_RENAL_MARKER_ABLATION_SET),
        "retained_predictor_count":
            len(remaining_predictors),
        "bun_removed": False,
        "other_non_creatinine_physiology_removed":
            False,
        "rationale": (
            "Remove direct creatinine/KDIGO-derived, outcome-proximal "
            "renal-state information while retaining broader physiology "
            "and non-creatinine laboratory information."
        ),
    },

    "modeling_rule": {
        "algorithm": "XGBoost",
        "candidate_selection":
            "fixed_from_original_locked_outer_fold",
        "selected_candidate_by_outer_fold":
            LOCKED_SELECTED_CANDIDATES,
        "hyperparameter_reselection_permitted":
            False,
        "new_grid_search_permitted":
            False,
        "early_stopping_permitted":
            False,
        "class_weighting_permitted":
            False,
        "smote_or_resampling_permitted":
            False,
        "outer_test_retuning_permitted":
            False,
    },

    "preprocessing": {
        "same_as_primary_xgboost":
            True,
        "fit_only_on_training_partition":
            True,
        "numeric":
            "median imputation with missing indicators",
        "categorical":
            "constant missing imputation plus sparse one-hot encoding",
        "unknown_categories":
            "ignore",
    },

    "nested_evaluation": {
        "outer_folds":
            "same five hospital-disjoint outer folds",
        "inner_folds":
            "same locked 07D hospital-disjoint inner folds",
        "inner_fold_guard":
            inner_fold_guard,
        "inner_oof_role": (
            "generate out-of-fold raw predictions for fitting "
            "the ablated-model Platt calibrator"
        ),
        "outer_test_role":
            "untouched until fixed model and calibrator are locked",
    },

    "calibration": {
        "method": "Platt scaling",
        "fit_source":
            "pooled inner-OOF raw model scores within each outer fold",
        "outer_test_used_for_calibration":
            False,
        "new_calibrator_required_for_ablated_model":
            True,
    },

    "primary_metrics": [
        "AUROC",
        "AUPRC",
        "Brier score",
        "log loss",
    ],

    "calibration_diagnostics": [
        "mean predicted risk",
        "calibration intercept",
        "calibration slope",
    ],

    "paired_comparison": {
        "comparison":
            "full locked XGBoost vs renal-marker-ablated XGBoost",
        "same_outer_test_patients":
            True,
        "bootstrap_unit":
            "hospital",
        "paired_resampling":
            True,
        "bootstrap_replicates":
            BOOTSTRAP_REPLICATES,
        "bootstrap_seed":
            BOOTSTRAP_SEED,
        "confidence_intervals":
            "nominal exploratory 95% percentile intervals",
        "multiplicity_adjusted":
            False,
    },

    "interpretation_rules": [
        (
            "The ablated model is a sensitivity analysis, not a "
            "replacement clinical model."
        ),
        (
            "Retained performance does not prove causal importance "
            "of non-renal predictors."
        ),
        (
            "A performance drop does not prove leakage; removed "
            "predictors are legitimate pre-landmark variables whose "
            "temporal validity was separately audited."
        ),
        (
            "The analysis tests dependence of predictive performance "
            "on direct creatinine/KDIGO information."
        ),
    ],

    "dca": {
        "performed_in_39_series":
            False,
        "rule": (
            "Any DCA of the ablated model requires a separate protocol "
            "lock after discrimination/calibration results are available."
        ),
    },

    "privacy_and_storage": {
        "patient_level_predictions_written_to_drive":
            False,
        "patient_level_predictions_may_be_written_to_secure_bigquery":
            True,
        "aggregate_outputs_written_to_drive":
            True,
        "bigquery_dml_used":
            False,
    },

    "upstream_sha256": {
        "38C_R1_treeshap_manifest":
            EXPECTED_38C_R1_MANIFEST_SHA,
        "xgboost_protocol":
            EXPECTED_XGB_PROTOCOL_SHA,
        "inner_fold_configuration":
            EXPECTED_INNER_FOLD_SHA,
    },

    "performance_result_calculated":
        False,
    "performance_result_viewed":
        False,
    "model_fit_performed":
        False,
    "retuning_performed":
        False,
    "candidate_reselection_performed":
        False,
    "outer_test_used":
        False,
}

protocol_text = json.dumps(
    protocol,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
)

protocol_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "39A_R2_locked_renal_marker_ablation_sensitivity_protocol_v1.json",
)

sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "39A_R2_locked_renal_marker_ablation_sensitivity_protocol_v1_SHA256.txt",
)

ablation_list_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "39A_R2_locked_renal_marker_ablation_predictor_list.txt",
)

retained_list_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "39A_R2_locked_renal_marker_ablation_retained_predictor_list.txt",
)

with open(
    protocol_path,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(protocol_text)

protocol_sha = hashlib.sha256(
    protocol_text.encode("utf-8")
).hexdigest()

with open(
    sha_path,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(protocol_sha + "\n")

with open(
    ablation_list_path,
    "w",
    encoding="utf-8",
) as fh:
    for predictor in LOCKED_RENAL_MARKER_ABLATION_SET:
        fh.write(predictor + "\n")

with open(
    retained_list_path,
    "w",
    encoding="utf-8",
) as fh:
    for predictor in remaining_predictors:
        fh.write(predictor + "\n")

print("\n39A-R2 LOCKED RENAL-MARKER ABLATION SET")
for predictor in LOCKED_RENAL_MARKER_ABLATION_SET:
    print("-", predictor)

print("\nRemoved predictor count:")
print(len(LOCKED_RENAL_MARKER_ABLATION_SET))

print("\nRetained predictor count:")
print(len(remaining_predictors))

print("\nFixed XGBoost candidates:")
for fold, candidate in LOCKED_SELECTED_CANDIDATES.items():
    print(f"- Outer fold {fold}: {candidate}")

print("\nPaired comparison:")
print("Full locked XGBoost vs renal-marker-ablated XGBoost")
print("2,000 paired hospital-cluster bootstrap replicates")
print("Seed: 20260723")
print("Inference: exploratory nominal 95% CIs")
print("Hyperparameter reselection: PROHIBITED")
print("Outer-test-driven changes: PROHIBITED")

print("\n39A-R2 protocol SHA-256:")
print(protocol_sha)

print("\nSaved:")
print(protocol_path)
print(sha_path)
print(ablation_list_path)
print(retained_list_path)

print(
    "\n39A-R2 PASS: Registry-based renal-marker ablation sensitivity "
    "protocol locked before any ablated-model performance result "
    "was calculated or viewed."
)

In [ ]:
import os
import gc
import json
import time
import hashlib

import numpy as np
import pandas as pd

from google.cloud import bigquery
from google.api_core.exceptions import NotFound

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)

from xgboost import XGBClassifier
from IPython.display import display

print("STARTING RENAL-MARKER ABLATION XGBOOST SENSITIVITY ANALYSIS — CODE VERSION 39B")

# ============================================================
# 39B — RENAL-MARKER ABLATION XGBOOST SENSITIVITY ANALYSIS
#
# Locked upstream protocol: 39A-R2
#
# Scientific question:
#   How much predictive performance remains after removing direct,
#   outcome-proximal creatinine/KDIGO-derived predictors?
#
# Fixed analysis:
#   - same 58,491-patient cohort
#   - same five hospital-disjoint outer folds
#   - same locked 07D hospital-disjoint inner folds
#   - remove exactly the 39A-R2 locked ablation predictor set
#   - NO hyperparameter reselection
#   - folds 1-4: XGB04
#   - fold 5: XGB06
#   - new Platt calibrator from INNER-OOF predictions only
#   - outer test remains untouched until model/calibrator are fixed
#   - full locked XGBoost vs ablated XGBoost paired comparison
#   - 2,000 paired hospital-cluster bootstrap replicates
#
# Privacy / infrastructure:
#   - patient-level predictions: secure BigQuery only
#   - aggregate outputs: Drive
#   - BigQuery DML: NEVER used
#   - all BigQuery writes use load jobs + WRITE_TRUNCATE
# ============================================================

EXPECTED_39A_R2_PROTOCOL_SHA = (
    "b56eb8ba8ba1f74def121280d17f8acc"
    "861dd4074b875aa6c68c2e655789ec35"
)

EXPECTED_INNER_FOLD_SHA = (
    "34b3fed6f6216d7a59afb8303e081fb4"
    "8c9dade9bab1bb367e395372098e9b50"
)

EXPECTED_ROWS = 58491
EXPECTED_EVENTS = 3032
EXPECTED_HOSPITALS = 198

MODEL_RANDOM_SEED = 20260721
BOOTSTRAP_REPLICATES = 2000
BOOTSTRAP_SEED = 20260723

EXPECTED_SPLITS = {
    1: {
        "training_rows": 46803,
        "test_rows": 11688,
        "training_hospitals": 158,
        "test_hospitals": 40,
        "training_events": 2426,
        "test_events": 606,
    },
    2: {
        "training_rows": 46800,
        "test_rows": 11691,
        "training_hospitals": 158,
        "test_hospitals": 40,
        "training_events": 2426,
        "test_events": 606,
    },
    3: {
        "training_rows": 46755,
        "test_rows": 11736,
        "training_hospitals": 158,
        "test_hospitals": 40,
        "training_events": 2424,
        "test_events": 608,
    },
    4: {
        "training_rows": 46803,
        "test_rows": 11688,
        "training_hospitals": 159,
        "test_hospitals": 39,
        "training_events": 2426,
        "test_events": 606,
    },
    5: {
        "training_rows": 46803,
        "test_rows": 11688,
        "training_hospitals": 159,
        "test_hospitals": 39,
        "training_events": 2426,
        "test_events": 606,
    },
}

SELECTED_CANDIDATE_BY_OUTER_FOLD = {
    1: "XGB04",
    2: "XGB04",
    3: "XGB04",
    4: "XGB04",
    5: "XGB06",
}

CANDIDATES = {
    "XGB04": {
        "candidate_id": "XGB04",
        "n_estimators": 350,
        "max_depth": 4,
        "learning_rate": 0.03,
        "min_child_weight": 10.0,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "gamma": 0.10,
        "reg_alpha": 0.10,
        "reg_lambda": 10.0,
    },
    "XGB06": {
        "candidate_id": "XGB06",
        "n_estimators": 450,
        "max_depth": 4,
        "learning_rate": 0.02,
        "min_child_weight": 15.0,
        "subsample": 0.90,
        "colsample_bytree": 0.80,
        "gamma": 0.20,
        "reg_alpha": 0.50,
        "reg_lambda": 15.0,
    },
}

FULL_XGB_POOLED_TABLE = (
    "model_xgb_outer_predictions_all5_v1"
)

ABLATION_MODEL_NAME = "xgboost_renal_marker_ablation"
ABLATION_MODEL_VERSION = "posthoc_sensitivity_v1"

required_runtime = [
    "core_df_07B",
    "predictor_columns_07B",
    "numeric_columns_07B",
    "categorical_columns_07B",
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
]

missing_runtime = [
    name for name in required_runtime
    if name not in globals()
]

if missing_runtime:
    raise RuntimeError(
        "Eksik çalışma nesneleri: "
        + ", ".join(missing_runtime)
        + ". Önce 07A/07B temel hücrelerini çalıştır."
    )

# ------------------------------------------------------------
# 1. 39A-R2 protocol guard and exact locked ablation set
# ------------------------------------------------------------

protocol_sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "39A_R2_locked_renal_marker_ablation_sensitivity_protocol_v1_SHA256.txt",
)

protocol_json_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "39A_R2_locked_renal_marker_ablation_sensitivity_protocol_v1.json",
)

if not os.path.exists(protocol_sha_path):
    raise FileNotFoundError(protocol_sha_path)

if not os.path.exists(protocol_json_path):
    raise FileNotFoundError(protocol_json_path)

with open(protocol_sha_path, "r", encoding="utf-8") as fh:
    observed_protocol_sha = fh.read().strip()

if observed_protocol_sha != EXPECTED_39A_R2_PROTOCOL_SHA:
    raise RuntimeError(
        "39A-R2 protocol SHA mismatch: "
        + observed_protocol_sha
    )

with open(protocol_json_path, "r", encoding="utf-8") as fh:
    locked_protocol = json.load(fh)

locked_ablation_set = list(
    locked_protocol[
        "ablation_definition"
    ][
        "source_predictors_removed"
    ]
)

locked_retained_count = int(
    locked_protocol[
        "ablation_definition"
    ][
        "retained_predictor_count"
    ]
)

print("39A-R2 renal-marker ablation protocol SHA guard: PASS")

# ------------------------------------------------------------
# 2. Inner-fold SHA and map guard
# ------------------------------------------------------------

inner_sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1_SHA256.txt",
)

inner_map_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "07D_locked_inner_hospital_folds_v1.csv",
)

if not os.path.exists(inner_sha_path):
    raise FileNotFoundError(inner_sha_path)

if not os.path.exists(inner_map_path):
    raise FileNotFoundError(inner_map_path)

with open(inner_sha_path, "r", encoding="utf-8") as fh:
    observed_inner_sha = fh.read().strip()

if observed_inner_sha != EXPECTED_INNER_FOLD_SHA:
    raise RuntimeError(
        "07D inner-fold SHA mismatch: "
        + observed_inner_sha
    )

inner_map = pd.read_csv(
    inner_map_path,
    dtype={"group_hospital": str},
)

required_inner_columns = {
    "outer_fold",
    "group_hospital",
    "inner_fold",
}

if not required_inner_columns.issubset(
    set(inner_map.columns)
):
    raise RuntimeError(
        "07D inner map is missing required columns."
    )

inner_map["outer_fold"] = pd.to_numeric(
    inner_map["outer_fold"],
    errors="raise",
).astype(int)
inner_map["inner_fold"] = pd.to_numeric(
    inner_map["inner_fold"],
    errors="raise",
).astype(int)
inner_map["group_hospital"] = inner_map[
    "group_hospital"
].astype(str)

print("07D locked inner hospital folds SHA/map guard: PASS")

# ------------------------------------------------------------
# 3. Cohort / feature integrity
# ------------------------------------------------------------

if len(core_df_07B) != EXPECTED_ROWS:
    raise RuntimeError(
        f"Cohort rows={len(core_df_07B)}, expected={EXPECTED_ROWS}."
    )

if int(core_df_07B["label_stage23"].sum()) != EXPECTED_EVENTS:
    raise RuntimeError(
        "Cohort event count is not 3,032."
    )

if core_df_07B["group_hospital"].astype(str).nunique() != EXPECTED_HOSPITALS:
    raise RuntimeError(
        "Cohort hospital count is not 198."
    )

if len(predictor_columns_07B) != 159:
    raise RuntimeError(
        "Locked core predictor count is not 159."
    )

missing_locked_ablation = [
    x for x in locked_ablation_set
    if x not in predictor_columns_07B
]

if missing_locked_ablation:
    raise RuntimeError(
        "39A-R2 locked ablation feature(s) missing at runtime: "
        + ", ".join(missing_locked_ablation)
    )

ablation_lookup = set(
    locked_ablation_set
)

retained_predictors = [
    x for x in predictor_columns_07B
    if x not in ablation_lookup
]

if len(retained_predictors) != locked_retained_count:
    raise RuntimeError(
        f"Retained predictor count={len(retained_predictors)}, "
        f"locked={locked_retained_count}."
    )

retained_numeric = [
    x for x in numeric_columns_07B
    if x in retained_predictors
]

retained_categorical = [
    x for x in categorical_columns_07B
    if x in retained_predictors
]

if (
    len(retained_numeric)
    + len(retained_categorical)
    != len(retained_predictors)
):
    raise RuntimeError(
        "Retained numeric/categorical predictor accounting mismatch."
    )

print("Cohort integrity guard: PASS")
print(
    "Locked ablation set:",
    len(locked_ablation_set),
    "predictors removed",
)
print(
    "Retained feature set:",
    len(retained_predictors),
    "source predictors",
)

# ------------------------------------------------------------
# 4. Exact feature matrix preparation
# ------------------------------------------------------------

X_all = core_df_07B[
    retained_predictors
].copy()

for column in retained_numeric:
    X_all[column] = pd.to_numeric(
        X_all[column],
        errors="coerce",
    ).astype("float64")

for column in retained_categorical:
    s = X_all[column].astype("object")
    X_all[column] = s.where(
        pd.notna(s),
        np.nan,
    )

meta = core_df_07B[
    [
        "id_row",
        "outer_fold",
        "group_hospital",
        "label_stage23",
    ]
].copy()

meta["id_row"] = meta["id_row"].astype(str)
meta["outer_fold"] = pd.to_numeric(
    meta["outer_fold"],
    errors="raise",
).astype(int)
meta["group_hospital"] = meta[
    "group_hospital"
].astype(str)
meta["label_stage23"] = pd.to_numeric(
    meta["label_stage23"],
    errors="raise",
).astype(int)

# ------------------------------------------------------------
# 5. Constructors
# ------------------------------------------------------------

def make_preprocessor():
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True,
                    keep_empty_features=True,
                ),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="__MISSING__",
                    keep_empty_features=True,
                ),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                    dtype=np.float32,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                retained_numeric,
            ),
            (
                "categorical",
                categorical_pipeline,
                retained_categorical,
            ),
        ],
        remainder="drop",
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_xgb(candidate):
    return XGBClassifier(
        n_estimators=int(
            candidate["n_estimators"]
        ),
        max_depth=int(
            candidate["max_depth"]
        ),
        learning_rate=float(
            candidate["learning_rate"]
        ),
        min_child_weight=float(
            candidate["min_child_weight"]
        ),
        subsample=float(
            candidate["subsample"]
        ),
        colsample_bytree=float(
            candidate["colsample_bytree"]
        ),
        gamma=float(
            candidate["gamma"]
        ),
        reg_alpha=float(
            candidate["reg_alpha"]
        ),
        reg_lambda=float(
            candidate["reg_lambda"]
        ),
        objective="binary:logistic",
        eval_metric="logloss",
        tree_method="hist",
        max_bin=256,
        scale_pos_weight=1.0,
        importance_type="gain",
        random_state=MODEL_RANDOM_SEED,
        n_jobs=-1,
        verbosity=0,
    )


def clip_probability(p):
    return np.clip(
        np.asarray(p, dtype=float),
        1e-8,
        1 - 1e-8,
    )


def probability_logit(p):
    p = np.clip(
        np.asarray(p, dtype=float),
        1e-6,
        1 - 1e-6,
    )
    return np.log(
        p / (1.0 - p)
    ).reshape(-1, 1)


def fit_platt(y, raw_probability):
    calibrator = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    calibrator.fit(
        probability_logit(
            raw_probability
        ),
        y,
    )

    intercept = float(
        calibrator.intercept_[0]
    )
    slope = float(
        calibrator.coef_[0][0]
    )

    if (
        not np.isfinite(intercept)
        or not np.isfinite(slope)
        or slope <= 0
    ):
        raise RuntimeError(
            "Invalid Platt calibration coefficients."
        )

    return calibrator, intercept, slope


def apply_platt(calibrator, raw_probability):
    return calibrator.predict_proba(
        probability_logit(
            raw_probability
        )
    )[:, 1]


def probability_metrics(
    y_true,
    probability,
    sample_weight=None,
):
    p = clip_probability(
        probability
    )

    return {
        "auroc": float(
            roc_auc_score(
                y_true,
                p,
                sample_weight=sample_weight,
            )
        ),
        "auprc": float(
            average_precision_score(
                y_true,
                p,
                sample_weight=sample_weight,
            )
        ),
        "brier": float(
            brier_score_loss(
                y_true,
                p,
                sample_weight=sample_weight,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                p,
                labels=[0, 1],
                sample_weight=sample_weight,
            )
        ),
        "mean_predicted_risk": float(
            np.average(
                p,
                weights=sample_weight,
            )
            if sample_weight is not None
            else p.mean()
        ),
        "observed_event_rate": float(
            np.average(
                y_true,
                weights=sample_weight,
            )
            if sample_weight is not None
            else np.mean(y_true)
        ),
    }


def calibration_intercept_slope(
    y_true,
    probability,
):
    model = LogisticRegression(
        penalty=None,
        solver="lbfgs",
        max_iter=2000,
    )
    model.fit(
        probability_logit(
            probability
        ),
        y_true,
    )

    return (
        float(model.intercept_[0]),
        float(model.coef_[0][0]),
    )

# ------------------------------------------------------------
# 6. Secure BQ schemas / checkpoint helpers
# ------------------------------------------------------------

inner_oof_load_config = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField(
            "id_row",
            "STRING",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "outer_fold",
            "INTEGER",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "inner_fold",
            "INTEGER",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "candidate_id",
            "STRING",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "label_stage23",
            "INTEGER",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "prediction_raw",
            "FLOAT",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "protocol_sha256",
            "STRING",
            mode="REQUIRED",
        ),
    ],
    write_disposition=(
        bigquery.WriteDisposition.WRITE_TRUNCATE
    ),
)

outer_prediction_load_config = bigquery.LoadJobConfig(
    schema=[
        bigquery.SchemaField(
            "id_row",
            "STRING",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "outer_fold",
            "INTEGER",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "label_stage23",
            "INTEGER",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "prediction_raw",
            "FLOAT",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "prediction_platt",
            "FLOAT",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "candidate_id",
            "STRING",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "model_name",
            "STRING",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "model_version",
            "STRING",
            mode="REQUIRED",
        ),
        bigquery.SchemaField(
            "protocol_sha256",
            "STRING",
            mode="REQUIRED",
        ),
    ],
    write_disposition=(
        bigquery.WriteDisposition.WRITE_TRUNCATE
    ),
)


def inner_table_id(outer_fold, inner_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_xgb_renal_ablation_inner_oof_"
        f"outer{outer_fold}_inner{inner_fold}_v1"
    )


def outer_table_id(outer_fold):
    return (
        f"{TARGET_DATASET}."
        f"model_xgb_renal_ablation_outer_predictions_"
        f"outer{outer_fold}_v1"
    )


def pooled_table_id():
    return (
        f"{TARGET_DATASET}."
        "model_xgb_renal_ablation_outer_predictions_all5_v1"
    )


def verify_inner_checkpoint(
    outer_fold,
    inner_fold,
    expected_rows,
    expected_events,
    candidate_id,
):
    table_id = inner_table_id(
        outer_fold,
        inner_fold,
    )

    try:
        client.get_table(table_id)
    except NotFound:
        return False

    sql = f"""
    SELECT
      COUNT(*) AS n,
      COUNT(DISTINCT id_row) AS n_ids,
      COUNTIF(label_stage23 = 1) AS events,
      COUNTIF(prediction_raw IS NULL) AS missing_pred,
      COUNTIF(prediction_raw < 0 OR prediction_raw > 1) AS invalid_pred,
      COUNT(DISTINCT candidate_id) AS n_candidates,
      ANY_VALUE(candidate_id) AS candidate_id,
      COUNT(DISTINCT protocol_sha256) AS n_protocols,
      ANY_VALUE(protocol_sha256) AS protocol_sha256,
      MIN(outer_fold) AS min_outer,
      MAX(outer_fold) AS max_outer,
      MIN(inner_fold) AS min_inner,
      MAX(inner_fold) AS max_inner
    FROM `{table_id}`
    """

    row = client.query(
        sql,
        location=BQ_LOCATION,
    ).to_dataframe().iloc[0]

    return bool(
        int(row["n"]) == expected_rows
        and int(row["n_ids"]) == expected_rows
        and int(row["events"]) == expected_events
        and int(row["missing_pred"]) == 0
        and int(row["invalid_pred"]) == 0
        and int(row["n_candidates"]) == 1
        and str(row["candidate_id"]) == candidate_id
        and int(row["n_protocols"]) == 1
        and str(row["protocol_sha256"])
            == EXPECTED_39A_R2_PROTOCOL_SHA
        and int(row["min_outer"]) == outer_fold
        and int(row["max_outer"]) == outer_fold
        and int(row["min_inner"]) == inner_fold
        and int(row["max_inner"]) == inner_fold
    )


def read_inner_checkpoint(
    outer_fold,
    inner_fold,
):
    table_id = inner_table_id(
        outer_fold,
        inner_fold,
    )

    return client.query(
        f"""
        SELECT
          id_row,
          outer_fold,
          inner_fold,
          candidate_id,
          label_stage23,
          prediction_raw
        FROM `{table_id}`
        """,
        location=BQ_LOCATION,
    ).to_dataframe()


def verify_outer_checkpoint(
    outer_fold,
    expected_rows,
    expected_events,
    candidate_id,
):
    table_id = outer_table_id(
        outer_fold
    )

    try:
        client.get_table(table_id)
    except NotFound:
        return False

    sql = f"""
    SELECT
      COUNT(*) AS n,
      COUNT(DISTINCT id_row) AS n_ids,
      COUNTIF(label_stage23 = 1) AS events,
      COUNTIF(prediction_raw IS NULL) AS missing_raw,
      COUNTIF(prediction_platt IS NULL) AS missing_platt,
      COUNTIF(prediction_raw < 0 OR prediction_raw > 1) AS invalid_raw,
      COUNTIF(prediction_platt < 0 OR prediction_platt > 1) AS invalid_platt,
      COUNT(DISTINCT candidate_id) AS n_candidates,
      ANY_VALUE(candidate_id) AS candidate_id,
      COUNT(DISTINCT protocol_sha256) AS n_protocols,
      ANY_VALUE(protocol_sha256) AS protocol_sha256,
      MIN(outer_fold) AS min_outer,
      MAX(outer_fold) AS max_outer
    FROM `{table_id}`
    """

    row = client.query(
        sql,
        location=BQ_LOCATION,
    ).to_dataframe().iloc[0]

    return bool(
        int(row["n"]) == expected_rows
        and int(row["n_ids"]) == expected_rows
        and int(row["events"]) == expected_events
        and int(row["missing_raw"]) == 0
        and int(row["missing_platt"]) == 0
        and int(row["invalid_raw"]) == 0
        and int(row["invalid_platt"]) == 0
        and int(row["n_candidates"]) == 1
        and str(row["candidate_id"]) == candidate_id
        and int(row["n_protocols"]) == 1
        and str(row["protocol_sha256"])
            == EXPECTED_39A_R2_PROTOCOL_SHA
        and int(row["min_outer"]) == outer_fold
        and int(row["max_outer"]) == outer_fold
    )

# ------------------------------------------------------------
# 7. Fit all five fixed-candidate ablated outer models
# ------------------------------------------------------------

outer_summary_rows = []
calibration_rows = []

for outer_fold in range(1, 6):
    candidate_id = SELECTED_CANDIDATE_BY_OUTER_FOLD[
        outer_fold
    ]
    candidate = CANDIDATES[
        candidate_id
    ]

    print(
        f"\n===== OUTER FOLD {outer_fold} / "
        f"FIXED {candidate_id} ====="
    )

    outer_train_mask = (
        meta["outer_fold"].to_numpy()
        != outer_fold
    )
    outer_test_mask = (
        meta["outer_fold"].to_numpy()
        == outer_fold
    )

    X_outer_train = (
        X_all.loc[
            outer_train_mask,
            retained_predictors,
        ]
        .reset_index(drop=True)
    )
    X_outer_test = (
        X_all.loc[
            outer_test_mask,
            retained_predictors,
        ]
        .reset_index(drop=True)
    )

    train_meta = (
        meta.loc[
            outer_train_mask,
            [
                "id_row",
                "group_hospital",
                "label_stage23",
            ],
        ]
        .reset_index(drop=True)
    )

    test_meta = (
        meta.loc[
            outer_test_mask,
            [
                "id_row",
                "group_hospital",
                "label_stage23",
            ],
        ]
        .reset_index(drop=True)
    )

    y_outer_train = train_meta[
        "label_stage23"
    ].to_numpy(dtype=np.int8)

    y_outer_test = test_meta[
        "label_stage23"
    ].to_numpy(dtype=np.int8)

    train_hospitals = set(
        train_meta["group_hospital"]
    )
    test_hospitals = set(
        test_meta["group_hospital"]
    )

    if train_hospitals & test_hospitals:
        raise RuntimeError(
            f"Outer fold {outer_fold}: hospital overlap detected."
        )

    observed_split = {
        "training_rows": len(train_meta),
        "test_rows": len(test_meta),
        "training_hospitals": len(
            train_hospitals
        ),
        "test_hospitals": len(
            test_hospitals
        ),
        "training_events": int(
            y_outer_train.sum()
        ),
        "test_events": int(
            y_outer_test.sum()
        ),
    }

    if observed_split != EXPECTED_SPLITS[
        outer_fold
    ]:
        raise RuntimeError(
            f"Outer fold {outer_fold}: split mismatch. "
            f"Observed={observed_split}; "
            f"expected={EXPECTED_SPLITS[outer_fold]}"
        )

    # Locked inner-hospital map for this outer training set.
    inner_part = (
        inner_map.loc[
            inner_map[
                "outer_fold"
            ].eq(outer_fold),
            [
                "group_hospital",
                "inner_fold",
            ],
        ]
        .copy()
    )

    expected_train_hospitals = EXPECTED_SPLITS[
        outer_fold
    ][
        "training_hospitals"
    ]

    if len(inner_part) != expected_train_hospitals:
        raise RuntimeError(
            f"Outer fold {outer_fold}: inner-map hospital count mismatch."
        )

    if inner_part[
        "group_hospital"
    ].duplicated().any():
        raise RuntimeError(
            f"Outer fold {outer_fold}: duplicate hospital in inner map."
        )

    hospital_to_inner = dict(
        zip(
            inner_part[
                "group_hospital"
            ].astype(str),
            inner_part[
                "inner_fold"
            ].astype(int),
        )
    )

    inner_vector = np.asarray(
        [
            hospital_to_inner.get(
                h,
                -1,
            )
            for h in train_meta[
                "group_hospital"
            ].astype(str)
        ],
        dtype=int,
    )

    if (inner_vector == -1).any():
        raise RuntimeError(
            f"Outer fold {outer_fold}: missing inner-fold assignment."
        )

    if set(
        np.unique(inner_vector)
    ) != {1, 2, 3, 4, 5}:
        raise RuntimeError(
            f"Outer fold {outer_fold}: inner folds are not exactly 1..5."
        )

    # --------------------------------------------------------
    # 7A. Build / reuse fixed-candidate inner OOF checkpoints
    # --------------------------------------------------------

    inner_frames = []

    for inner_fold in range(1, 6):
        inner_train_mask = (
            inner_vector != inner_fold
        )
        inner_valid_mask = (
            inner_vector == inner_fold
        )

        validation_rows = int(
            inner_valid_mask.sum()
        )
        validation_events = int(
            y_outer_train[
                inner_valid_mask
            ].sum()
        )

        valid_checkpoint = (
            verify_inner_checkpoint(
                outer_fold=outer_fold,
                inner_fold=inner_fold,
                expected_rows=validation_rows,
                expected_events=validation_events,
                candidate_id=candidate_id,
            )
        )

        if valid_checkpoint:
            print(
                f"Outer {outer_fold} / inner {inner_fold}: "
                "secure OOF checkpoint PASS; reusing."
            )

            inner_frames.append(
                read_inner_checkpoint(
                    outer_fold,
                    inner_fold,
                )
            )
            continue

        inner_train_hospitals = set(
            train_meta.loc[
                inner_train_mask,
                "group_hospital",
            ]
        )
        inner_valid_hospitals = set(
            train_meta.loc[
                inner_valid_mask,
                "group_hospital",
            ]
        )

        if (
            inner_train_hospitals
            & inner_valid_hospitals
        ):
            raise RuntimeError(
                f"Outer {outer_fold} / inner {inner_fold}: hospital overlap."
            )

        print(
            f"Outer {outer_fold} / inner {inner_fold}: "
            f"fitting fixed {candidate_id} "
            f"({validation_rows} validation patients, "
            f"{validation_events} events)"
        )

        preprocessor = make_preprocessor()

        X_inner_train_processed = (
            preprocessor.fit_transform(
                X_outer_train.loc[
                    inner_train_mask
                ]
            )
        )

        X_inner_valid_processed = (
            preprocessor.transform(
                X_outer_train.loc[
                    inner_valid_mask
                ]
            )
        )

        if (
            X_inner_train_processed.shape[1]
            != X_inner_valid_processed.shape[1]
        ):
            raise RuntimeError(
                f"Outer {outer_fold} / inner {inner_fold}: "
                "processed column mismatch."
            )

        model = make_xgb(
            candidate
        )

        fit_started = time.time()

        model.fit(
            X_inner_train_processed,
            y_outer_train[
                inner_train_mask
            ],
        )

        fit_seconds = (
            time.time()
            - fit_started
        )

        valid_probability = (
            model.predict_proba(
                X_inner_valid_processed
            )[:, 1]
        ).astype(float)

        if (
            not np.isfinite(
                valid_probability
            ).all()
            or (
                (valid_probability < 0)
                | (valid_probability > 1)
            ).any()
        ):
            raise RuntimeError(
                f"Outer {outer_fold} / inner {inner_fold}: invalid OOF probabilities."
            )

        oof_df = pd.DataFrame({
            "id_row": train_meta.loc[
                inner_valid_mask,
                "id_row",
            ].astype(str).to_numpy(),
            "outer_fold": np.full(
                validation_rows,
                outer_fold,
                dtype=np.int64,
            ),
            "inner_fold": np.full(
                validation_rows,
                inner_fold,
                dtype=np.int64,
            ),
            "candidate_id": candidate_id,
            "label_stage23": y_outer_train[
                inner_valid_mask
            ].astype(np.int64),
            "prediction_raw":
                valid_probability.astype(
                    np.float64
                ),
            "protocol_sha256":
                EXPECTED_39A_R2_PROTOCOL_SHA,
        })

        if oof_df[
            "id_row"
        ].duplicated().any():
            raise RuntimeError(
                f"Outer {outer_fold} / inner {inner_fold}: duplicate OOF ids."
            )

        target = inner_table_id(
            outer_fold,
            inner_fold,
        )

        client.load_table_from_dataframe(
            oof_df,
            target,
            job_config=inner_oof_load_config,
            location=BQ_LOCATION,
        ).result()

        if not verify_inner_checkpoint(
            outer_fold=outer_fold,
            inner_fold=inner_fold,
            expected_rows=validation_rows,
            expected_events=validation_events,
            candidate_id=candidate_id,
        ):
            raise RuntimeError(
                f"Outer {outer_fold} / inner {inner_fold}: secure checkpoint verification failed."
            )

        print(
            f"Outer {outer_fold} / inner {inner_fold}: "
            f"checkpoint PASS | fit {fit_seconds:.1f}s"
        )

        inner_frames.append(
            oof_df[
                [
                    "id_row",
                    "outer_fold",
                    "inner_fold",
                    "candidate_id",
                    "label_stage23",
                    "prediction_raw",
                ]
            ].copy()
        )

        del (
            preprocessor,
            X_inner_train_processed,
            X_inner_valid_processed,
            model,
            oof_df,
            valid_probability,
        )
        gc.collect()

    pooled_inner_oof = pd.concat(
        inner_frames,
        ignore_index=True,
    )

    if len(
        pooled_inner_oof
    ) != EXPECTED_SPLITS[
        outer_fold
    ][
        "training_rows"
    ]:
        raise RuntimeError(
            f"Outer fold {outer_fold}: pooled inner OOF row count mismatch."
        )

    if pooled_inner_oof[
        "id_row"
    ].duplicated().any():
        raise RuntimeError(
            f"Outer fold {outer_fold}: duplicate patient in pooled inner OOF."
        )

    if int(
        pooled_inner_oof[
            "label_stage23"
        ].sum()
    ) != EXPECTED_SPLITS[
        outer_fold
    ][
        "training_events"
    ]:
        raise RuntimeError(
            f"Outer fold {outer_fold}: pooled inner OOF event mismatch."
        )

    pooled_inner_oof = (
        pooled_inner_oof
        .sort_values(
            "id_row"
        )
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # 7B. Fit Platt ONLY on inner-OOF predictions
    # --------------------------------------------------------

    platt, platt_intercept, platt_slope = (
        fit_platt(
            pooled_inner_oof[
                "label_stage23"
            ].to_numpy(dtype=int),
            pooled_inner_oof[
                "prediction_raw"
            ].to_numpy(dtype=float),
        )
    )

    inner_raw_metrics = probability_metrics(
        pooled_inner_oof[
            "label_stage23"
        ].to_numpy(dtype=int),
        pooled_inner_oof[
            "prediction_raw"
        ].to_numpy(dtype=float),
    )

    calibration_rows.append({
        "outer_fold": outer_fold,
        "fixed_candidate":
            candidate_id,
        "inner_oof_patients":
            len(pooled_inner_oof),
        "inner_oof_events":
            int(
                pooled_inner_oof[
                    "label_stage23"
                ].sum()
            ),
        "inner_oof_raw_auroc":
            inner_raw_metrics[
                "auroc"
            ],
        "inner_oof_raw_auprc":
            inner_raw_metrics[
                "auprc"
            ],
        "inner_oof_raw_brier":
            inner_raw_metrics[
                "brier"
            ],
        "inner_oof_raw_log_loss":
            inner_raw_metrics[
                "log_loss"
            ],
        "platt_intercept":
            platt_intercept,
        "platt_slope":
            platt_slope,
    })

    # --------------------------------------------------------
    # 7C. Final fixed candidate on full outer training set
    #     Outer test touched only AFTER inner OOF + Platt lock.
    # --------------------------------------------------------

    print(
        f"Outer {outer_fold}: inner-OOF model/calibrator locked; "
        "fitting final model on full outer-training data."
    )

    final_preprocessor = make_preprocessor()

    X_outer_train_processed = (
        final_preprocessor.fit_transform(
            X_outer_train
        )
    )

    X_outer_test_processed = (
        final_preprocessor.transform(
            X_outer_test
        )
    )

    final_model = make_xgb(
        candidate
    )

    final_fit_started = time.time()

    final_model.fit(
        X_outer_train_processed,
        y_outer_train,
    )

    final_fit_seconds = (
        time.time()
        - final_fit_started
    )

    outer_raw_probability = (
        final_model.predict_proba(
            X_outer_test_processed
        )[:, 1]
    ).astype(float)

    outer_platt_probability = (
        apply_platt(
            platt,
            outer_raw_probability,
        )
    ).astype(float)

    if (
        not np.isfinite(
            outer_raw_probability
        ).all()
        or not np.isfinite(
            outer_platt_probability
        ).all()
    ):
        raise RuntimeError(
            f"Outer fold {outer_fold}: non-finite test predictions."
        )

    outer_prediction_df = pd.DataFrame({
        "id_row":
            test_meta[
                "id_row"
            ].astype(str).to_numpy(),
        "outer_fold": np.full(
            len(test_meta),
            outer_fold,
            dtype=np.int64,
        ),
        "label_stage23":
            y_outer_test.astype(
                np.int64
            ),
        "prediction_raw":
            outer_raw_probability.astype(
                np.float64
            ),
        "prediction_platt":
            outer_platt_probability.astype(
                np.float64
            ),
        "candidate_id":
            candidate_id,
        "model_name":
            ABLATION_MODEL_NAME,
        "model_version":
            ABLATION_MODEL_VERSION,
        "protocol_sha256":
            EXPECTED_39A_R2_PROTOCOL_SHA,
    })

    if outer_prediction_df[
        "id_row"
    ].duplicated().any():
        raise RuntimeError(
            f"Outer fold {outer_fold}: duplicate outer-test ids."
        )

    target = outer_table_id(
        outer_fold
    )

    client.load_table_from_dataframe(
        outer_prediction_df,
        target,
        job_config=outer_prediction_load_config,
        location=BQ_LOCATION,
    ).result()

    if not verify_outer_checkpoint(
        outer_fold=outer_fold,
        expected_rows=EXPECTED_SPLITS[
            outer_fold
        ][
            "test_rows"
        ],
        expected_events=EXPECTED_SPLITS[
            outer_fold
        ][
            "test_events"
        ],
        candidate_id=candidate_id,
    ):
        raise RuntimeError(
            f"Outer fold {outer_fold}: final secure prediction checkpoint failed."
        )

    outer_summary_rows.append({
        "outer_fold": outer_fold,
        "fixed_candidate":
            candidate_id,
        "source_predictors":
            len(
                retained_predictors
            ),
        "processed_columns":
            int(
                X_outer_train_processed.shape[1]
            ),
        "training_patients":
            len(train_meta),
        "training_hospitals":
            len(
                train_hospitals
            ),
        "training_events":
            int(
                y_outer_train.sum()
            ),
        "test_patients":
            len(test_meta),
        "test_hospitals":
            len(
                test_hospitals
            ),
        "test_events":
            int(
                y_outer_test.sum()
            ),
        "platt_intercept":
            platt_intercept,
        "platt_slope":
            platt_slope,
        "final_fit_seconds":
            float(
                final_fit_seconds
            ),
        "secure_prediction_table":
            target,
    })

    print(
        f"Outer fold {outer_fold}: secure final prediction checkpoint PASS."
    )

    # Do not print test metrics yet.
    del (
        X_outer_train,
        X_outer_test,
        train_meta,
        test_meta,
        y_outer_train,
        y_outer_test,
        inner_part,
        inner_vector,
        inner_frames,
        pooled_inner_oof,
        platt,
        final_preprocessor,
        X_outer_train_processed,
        X_outer_test_processed,
        final_model,
        outer_raw_probability,
        outer_platt_probability,
        outer_prediction_df,
    )
    gc.collect()

# ------------------------------------------------------------
# 8. Pool all five ablated outer-test prediction tables
# ------------------------------------------------------------

print(
    "\nAll five ablated outer folds are locked. "
    "Now pooling outer-test predictions for evaluation..."
)

union_sql = "\nUNION ALL\n".join(
    [
        f"""
        SELECT
          id_row,
          outer_fold,
          label_stage23,
          prediction_raw,
          prediction_platt,
          candidate_id,
          model_name,
          model_version,
          protocol_sha256
        FROM `{outer_table_id(fold)}`
        """
        for fold in range(1, 6)
    ]
)

pooled_ablation = client.query(
    union_sql,
    location=BQ_LOCATION,
).to_dataframe()

if len(pooled_ablation) != EXPECTED_ROWS:
    raise RuntimeError(
        f"Pooled ablation rows={len(pooled_ablation)}, expected={EXPECTED_ROWS}."
    )

if pooled_ablation[
    "id_row"
].duplicated().any():
    raise RuntimeError(
        "Duplicate patients in pooled ablation predictions."
    )

if int(
    pooled_ablation[
        "label_stage23"
    ].sum()
) != EXPECTED_EVENTS:
    raise RuntimeError(
        "Pooled ablation event count mismatch."
    )

# Save secure pooled patient predictions to BigQuery only.
client.load_table_from_dataframe(
    pooled_ablation,
    pooled_table_id(),
    job_config=outer_prediction_load_config,
    location=BQ_LOCATION,
).result()

print(
    "Secure pooled ablation prediction table:",
    pooled_table_id(),
)

# ------------------------------------------------------------
# 9. Load full locked XGBoost predictions and exact alignment
# ------------------------------------------------------------

full_xgb_table_id = (
    f"{TARGET_DATASET}."
    f"{FULL_XGB_POOLED_TABLE}"
)

client.get_table(
    full_xgb_table_id
)

full_xgb = client.query(
    f"""
    SELECT
      id_row,
      outer_fold,
      label_stage23,
      prediction_raw,
      prediction_platt
    FROM `{full_xgb_table_id}`
    """,
    location=BQ_LOCATION,
).to_dataframe()

full_xgb["id_row"] = full_xgb[
    "id_row"
].astype(str)

pooled_ablation["id_row"] = pooled_ablation[
    "id_row"
].astype(str)

comparison = (
    meta[
        [
            "id_row",
            "outer_fold",
            "group_hospital",
            "label_stage23",
        ]
    ]
    .merge(
        full_xgb.rename(
            columns={
                "outer_fold":
                    "outer_fold_full",
                "label_stage23":
                    "label_full",
                "prediction_raw":
                    "full_raw",
                "prediction_platt":
                    "full_platt",
            }
        ),
        on="id_row",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        pooled_ablation[
            [
                "id_row",
                "outer_fold",
                "label_stage23",
                "prediction_raw",
                "prediction_platt",
            ]
        ].rename(
            columns={
                "outer_fold":
                    "outer_fold_ablation",
                "label_stage23":
                    "label_ablation",
                "prediction_raw":
                    "ablation_raw",
                "prediction_platt":
                    "ablation_platt",
            }
        ),
        on="id_row",
        how="inner",
        validate="one_to_one",
    )
)

if len(comparison) != EXPECTED_ROWS:
    raise RuntimeError(
        "Full-vs-ablation alignment is not exactly 58,491 patients."
    )

if comparison[
    "id_row"
].duplicated().any():
    raise RuntimeError(
        "Duplicate patient after full-vs-ablation alignment."
    )

if not (
    comparison[
        "outer_fold"
    ].to_numpy()
    == comparison[
        "outer_fold_full"
    ].to_numpy()
).all():
    raise RuntimeError(
        "Outer-fold mismatch versus full XGBoost."
    )

if not (
    comparison[
        "outer_fold"
    ].to_numpy()
    == comparison[
        "outer_fold_ablation"
    ].to_numpy()
).all():
    raise RuntimeError(
        "Outer-fold mismatch versus ablation."
    )

if not (
    comparison[
        "label_stage23"
    ].to_numpy()
    == comparison[
        "label_full"
    ].to_numpy()
).all():
    raise RuntimeError(
        "Outcome mismatch versus full XGBoost."
    )

if not (
    comparison[
        "label_stage23"
    ].to_numpy()
    == comparison[
        "label_ablation"
    ].to_numpy()
).all():
    raise RuntimeError(
        "Outcome mismatch versus ablation."
    )

hospital_fold_n = comparison.groupby(
    "group_hospital"
)["outer_fold"].nunique()

if (
    comparison[
        "group_hospital"
    ].nunique()
    != EXPECTED_HOSPITALS
    or (
        hospital_fold_n != 1
    ).any()
):
    raise RuntimeError(
        "Hospital alignment / cross-fold integrity failed."
    )

print("Exact full-vs-ablation patient/hospital alignment: PASS")

# ------------------------------------------------------------
# 10. Per-fold and pooled metrics
# ------------------------------------------------------------

metric_rows = []

for model_name, raw_col, platt_col in [
    (
        "full_locked_xgboost",
        "full_raw",
        "full_platt",
    ),
    (
        "renal_marker_ablated_xgboost",
        "ablation_raw",
        "ablation_platt",
    ),
]:
    for fold_scope, subset in [
        ("pooled_all5", comparison),
        *[
            (
                f"outer_fold_{fold}",
                comparison.loc[
                    comparison[
                        "outer_fold"
                    ].eq(fold)
                ],
            )
            for fold in range(1, 6)
        ],
    ]:
        y = subset[
            "label_stage23"
        ].to_numpy(dtype=int)

        for probability_type, col in [
            ("raw", raw_col),
            ("platt_calibrated", platt_col),
        ]:
            metrics = probability_metrics(
                y,
                subset[
                    col
                ].to_numpy(dtype=float),
            )

            intercept, slope = (
                calibration_intercept_slope(
                    y,
                    subset[
                        col
                    ].to_numpy(dtype=float),
                )
            )

            metric_rows.append({
                "model": model_name,
                "scope": fold_scope,
                "probability_type":
                    probability_type,
                "patients":
                    len(subset),
                "events":
                    int(
                        y.sum()
                    ),
                **metrics,
                "calibration_intercept":
                    intercept,
                "calibration_slope":
                    slope,
            })

metrics_table = pd.DataFrame(
    metric_rows
)

# ------------------------------------------------------------
# 11. Point differences: positive ALWAYS favors FULL model
# ------------------------------------------------------------

pooled_lookup = {
    (
        row["model"],
        row["probability_type"],
    ): row
    for _, row in metrics_table.loc[
        metrics_table[
            "scope"
        ].eq("pooled_all5")
    ].iterrows()
}

point_difference_rows = []

for probability_type in [
    "raw",
    "platt_calibrated",
]:
    full = pooled_lookup[
        (
            "full_locked_xgboost",
            probability_type,
        )
    ]
    ablated = pooled_lookup[
        (
            "renal_marker_ablated_xgboost",
            probability_type,
        )
    ]

    point_difference_rows.append({
        "probability_type":
            probability_type,
        "auroc_improvement_full_vs_ablation":
            float(
                full["auroc"]
            )
            - float(
                ablated["auroc"]
            ),
        "auprc_improvement_full_vs_ablation":
            float(
                full["auprc"]
            )
            - float(
                ablated["auprc"]
            ),
        "brier_improvement_full_vs_ablation":
            float(
                ablated["brier"]
            )
            - float(
                full["brier"]
            ),
        "log_loss_improvement_full_vs_ablation":
            float(
                ablated["log_loss"]
            )
            - float(
                full["log_loss"]
            ),
        "positive_value_favors":
            "full_locked_xgboost",
        "inference_status":
            "exploratory_post_hoc_nominal_95_ci",
    })

point_differences = pd.DataFrame(
    point_difference_rows
)

# ------------------------------------------------------------
# 12. Paired hospital-cluster bootstrap
# ------------------------------------------------------------

print(
    f"\nRunning {BOOTSTRAP_REPLICATES:,} paired "
    "hospital-cluster bootstrap replicates..."
)

y_all = comparison[
    "label_stage23"
].to_numpy(dtype=int)

hospital_cat = pd.Categorical(
    comparison[
        "group_hospital"
    ]
)

hospital_codes = hospital_cat.codes.astype(
    int
)

if len(
    hospital_cat.categories
) != EXPECTED_HOSPITALS:
    raise RuntimeError(
        "Bootstrap hospital count is not 198."
    )

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

bootstrap_rows = []

for b in range(
    BOOTSTRAP_REPLICATES
):
    sampled_hospital_codes = rng.integers(
        0,
        EXPECTED_HOSPITALS,
        size=EXPECTED_HOSPITALS,
    )

    multiplicity = np.bincount(
        sampled_hospital_codes,
        minlength=EXPECTED_HOSPITALS,
    )

    weights = multiplicity[
        hospital_codes
    ].astype(float)

    if (
        weights[
            y_all == 1
        ].sum() <= 0
        or weights[
            y_all == 0
        ].sum() <= 0
    ):
        continue

    row = {
        "bootstrap_replicate":
            b + 1,
        "weighted_patients":
            float(
                weights.sum()
            ),
        "weighted_events":
            float(
                weights[
                    y_all == 1
                ].sum()
            ),
    }

    for probability_type, full_col, ablation_col, suffix in [
        (
            "raw",
            "full_raw",
            "ablation_raw",
            "raw",
        ),
        (
            "platt_calibrated",
            "full_platt",
            "ablation_platt",
            "platt",
        ),
    ]:
        full_metrics = probability_metrics(
            y_all,
            comparison[
                full_col
            ].to_numpy(dtype=float),
            sample_weight=weights,
        )

        ablation_metrics = probability_metrics(
            y_all,
            comparison[
                ablation_col
            ].to_numpy(dtype=float),
            sample_weight=weights,
        )

        row[
            f"auroc_improvement__{suffix}"
        ] = (
            full_metrics[
                "auroc"
            ]
            - ablation_metrics[
                "auroc"
            ]
        )

        row[
            f"auprc_improvement__{suffix}"
        ] = (
            full_metrics[
                "auprc"
            ]
            - ablation_metrics[
                "auprc"
            ]
        )

        row[
            f"brier_improvement__{suffix}"
        ] = (
            ablation_metrics[
                "brier"
            ]
            - full_metrics[
                "brier"
            ]
        )

        row[
            f"log_loss_improvement__{suffix}"
        ] = (
            ablation_metrics[
                "log_loss"
            ]
            - full_metrics[
                "log_loss"
            ]
        )

    bootstrap_rows.append(
        row
    )

    if b == 0 or (
        b + 1
    ) % 100 == 0:
        print(
            "  Completed bootstrap replicate",
            b + 1,
            "/",
            BOOTSTRAP_REPLICATES,
        )

bootstrap_df = pd.DataFrame(
    bootstrap_rows
)

if len(
    bootstrap_df
) < BOOTSTRAP_REPLICATES * 0.99:
    raise RuntimeError(
        "Fewer than 99% of paired bootstrap replicates completed."
    )

# ------------------------------------------------------------
# 13. Nominal exploratory 95% percentile CIs
# ------------------------------------------------------------

point_lookup = {
    row["probability_type"]: row
    for _, row in point_differences.iterrows()
}

ci_rows = []

for probability_type, suffix in [
    ("raw", "raw"),
    ("platt_calibrated", "platt"),
]:
    point = point_lookup[
        probability_type
    ]

    for metric in [
        "auroc_improvement",
        "auprc_improvement",
        "brier_improvement",
        "log_loss_improvement",
    ]:
        bootstrap_col = (
            f"{metric}__{suffix}"
        )

        point_col = (
            f"{metric}_full_vs_ablation"
        )

        values = bootstrap_df[
            bootstrap_col
        ].to_numpy(dtype=float)

        ci_rows.append({
            "probability_type":
                probability_type,
            "metric":
                metric,
            "point_difference":
                float(
                    point[
                        point_col
                    ]
                ),
            "ci95_lower":
                float(
                    np.quantile(
                        values,
                        0.025,
                    )
                ),
            "ci95_upper":
                float(
                    np.quantile(
                        values,
                        0.975,
                    )
                ),
            "positive_value_favors":
                "full_locked_xgboost",
            "inference_status":
                "exploratory_post_hoc_nominal_95_ci",
            "multiplicity_adjusted":
                False,
        })

paired_ci = pd.DataFrame(
    ci_rows
)

# ------------------------------------------------------------
# 14. Save aggregate outputs only
# ------------------------------------------------------------

outer_summary = pd.DataFrame(
    outer_summary_rows
)

calibration_table = pd.DataFrame(
    calibration_rows
)

alignment_integrity = pd.DataFrame([{
    "patients":
        len(comparison),
    "distinct_patients":
        comparison[
            "id_row"
        ].nunique(),
    "hospitals":
        comparison[
            "group_hospital"
        ].nunique(),
    "outer_folds":
        comparison[
            "outer_fold"
        ].nunique(),
    "events":
        int(
            comparison[
                "label_stage23"
            ].sum()
        ),
    "nonevents":
        int(
            len(comparison)
            - comparison[
                "label_stage23"
            ].sum()
        ),
    "hospital_cross_fold_violations":
        int(
            (
                hospital_fold_n != 1
            ).sum()
        ),
}])

paths = {
    "outer_model_summary":
        os.path.join(
            MODEL_OUTPUT_DIR,
            "39B_renal_ablation_outer_model_summary.csv",
        ),
    "inner_oof_calibration":
        os.path.join(
            MODEL_OUTPUT_DIR,
            "39B_renal_ablation_inner_oof_platt_calibration.csv",
        ),
    "alignment_integrity":
        os.path.join(
            MODEL_OUTPUT_DIR,
            "39B_full_vs_renal_ablation_alignment_integrity.csv",
        ),
    "performance_metrics":
        os.path.join(
            MODEL_OUTPUT_DIR,
            "39B_full_vs_renal_ablation_performance_metrics.csv",
        ),
    "point_differences":
        os.path.join(
            MODEL_OUTPUT_DIR,
            "39B_full_vs_renal_ablation_point_differences.csv",
        ),
    "paired_bootstrap_ci":
        os.path.join(
            MODEL_OUTPUT_DIR,
            "39B_full_vs_renal_ablation_paired_hospital_bootstrap_95CI.csv",
        ),
    "manifest":
        os.path.join(
            MODEL_OUTPUT_DIR,
            "39B_renal_marker_ablation_sensitivity_manifest.json",
        ),
    "manifest_sha":
        os.path.join(
            MODEL_OUTPUT_DIR,
            "39B_renal_marker_ablation_sensitivity_manifest_SHA256.txt",
        ),
}

outer_summary.to_csv(
    paths[
        "outer_model_summary"
    ],
    index=False,
)

calibration_table.to_csv(
    paths[
        "inner_oof_calibration"
    ],
    index=False,
)

alignment_integrity.to_csv(
    paths[
        "alignment_integrity"
    ],
    index=False,
)

metrics_table.to_csv(
    paths[
        "performance_metrics"
    ],
    index=False,
)

point_differences.to_csv(
    paths[
        "point_differences"
    ],
    index=False,
)

paired_ci.to_csv(
    paths[
        "paired_bootstrap_ci"
    ],
    index=False,
)

manifest = {
    "analysis_version":
        "39B",
    "analysis_type":
        "post_hoc_renal_marker_ablation_xgboost_sensitivity",
    "source_39A_R2_protocol_sha256":
        EXPECTED_39A_R2_PROTOCOL_SHA,
    "source_inner_fold_sha256":
        EXPECTED_INNER_FOLD_SHA,
    "patients":
        EXPECTED_ROWS,
    "events":
        EXPECTED_EVENTS,
    "hospitals":
        EXPECTED_HOSPITALS,
    "removed_source_predictors":
        locked_ablation_set,
    "removed_predictor_count":
        len(
            locked_ablation_set
        ),
    "retained_predictor_count":
        len(
            retained_predictors
        ),
    "selected_candidate_by_outer_fold":
        SELECTED_CANDIDATE_BY_OUTER_FOLD,
    "hyperparameter_reselection_performed":
        False,
    "early_stopping_used":
        False,
    "class_weighting_used":
        False,
    "smote_or_resampling_used":
        False,
    "platt_calibration_source":
        "inner_oof_only_within_each_outer_fold",
    "outer_test_used_for_calibration":
        False,
    "full_model_reference_table":
        full_xgb_table_id,
    "ablated_model_pooled_secure_table":
        pooled_table_id(),
    "paired_hospital_bootstrap": {
        "replicates_requested":
            BOOTSTRAP_REPLICATES,
        "replicates_completed":
            int(
                len(
                    bootstrap_df
                )
            ),
        "seed":
            BOOTSTRAP_SEED,
        "confidence_intervals":
            "nominal_exploratory_percentile_95",
        "multiplicity_adjusted":
            False,
    },
    "patient_level_predictions_written_to_drive":
        False,
    "patient_level_predictions_written_to_secure_bigquery":
        True,
    "bigquery_dml_used":
        False,
    "interpretation":
        (
            "Exploratory post-hoc sensitivity analysis testing "
            "dependence of predictive performance on direct "
            "creatinine/KDIGO-derived information."
        ),
    "outputs":
        paths,
}

manifest_text = json.dumps(
    manifest,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
)

with open(
    paths["manifest"],
    "w",
    encoding="utf-8",
) as fh:
    fh.write(
        manifest_text
    )

manifest_sha = hashlib.sha256(
    manifest_text.encode(
        "utf-8"
    )
).hexdigest()

with open(
    paths["manifest_sha"],
    "w",
    encoding="utf-8",
) as fh:
    fh.write(
        manifest_sha + "\n"
    )

# ------------------------------------------------------------
# 15. Compact display — results only after all five folds locked
# ------------------------------------------------------------

print("\n39B ALIGNMENT INTEGRITY")
display(
    alignment_integrity
)

print("\n39B OUTER-FOLD MODEL / CALIBRATION SUMMARY")
display(
    outer_summary
)

print("\n39B POOLED FULL VS RENAL-ABLATION PERFORMANCE")
display(
    metrics_table.loc[
        metrics_table[
            "scope"
        ].eq(
            "pooled_all5"
        )
    ].reset_index(
        drop=True
    )
)

print("\n39B POOLED POINT DIFFERENCES — POSITIVE FAVORS FULL XGBOOST")
display(
    point_differences
)

print("\n39B PAIRED HOSPITAL-BOOTSTRAP NOMINAL 95% CIs")
display(
    paired_ci
)

print("\n39B manifest SHA-256:")
print(
    manifest_sha
)

print("\nSecure pooled ablation prediction table:")
print(
    pooled_table_id()
)

print(
    "\n39B PASS: Renal-marker ablation sensitivity analysis completed "
    "under the pre-locked 39A-R2 protocol."
)
print(
    "No hyperparameter reselection, test-driven tuning, class weighting, "
    "SMOTE, or early stopping was used."
)
print(
    "The ablated-model Platt calibrators were fitted from inner-OOF "
    "predictions only."
)
print(
    "Patient-level predictions were written only to secure BigQuery; "
    "Drive contains aggregate outputs only."
)
print(
    "All inferential comparisons are exploratory/post-hoc with nominal "
    "95% paired hospital-cluster bootstrap intervals."
)

In [ ]:
import os
import json
import hashlib

print("STARTING RENAL-MARKER ABLATION DCA PROTOCOL LOCK — CODE VERSION 39C")

# ============================================================
# 39C — RENAL-MARKER ABLATION DECISION CURVE PROTOCOL LOCK
#
# Purpose:
#   Evaluate whether the renal-marker-ablated XGBoost model retains
#   clinically meaningful net benefit despite removal of direct
#   creatinine/KDIGO-derived predictors.
#
# This is an EXPLORATORY / POST-HOC supplementary DCA.
#
# IMPORTANT:
#   - Thresholds are NOT newly selected.
#   - We reuse exactly the already-locked primary DCA threshold grids.
#   - No "optimal threshold" may be selected after viewing the curves.
#   - No model fitting, retuning, or recalibration occurs in 39C.
#
# Models/strategies to be compared later:
#   1) Full locked XGBoost, Platt calibrated
#   2) Renal-marker-ablated XGBoost, Platt calibrated
#   3) Treat-all
#   4) Treat-none
#
# Primary thresholds:
#   1.0% to 10.0% in 0.5 percentage-point steps
#
# Sensitivity thresholds:
#   0.5% to 15.0% in 0.5 percentage-point steps
#
# Uncertainty:
#   2,000 paired hospital-cluster bootstrap replicates
#   seed = 20260723
# ============================================================

EXPECTED_39B_MANIFEST_SHA = (
    "fe50ce847b8eed3c886d4b2cb7734513"
    "316edad2babdbfcac207f8649f547215"
)

EXPECTED_37A_DCA_PROTOCOL_SHA = (
    "e0f574c228be1d76e75669e7e446df6"
    "d9461dee11883acf5e4b22735dcffcbfb"
)

EXPECTED_39A_R2_PROTOCOL_SHA = (
    "b56eb8ba8ba1f74def121280d17f8acc"
    "861dd4074b875aa6c68c2e655789ec35"
)

PRIMARY_THRESHOLDS = [
    0.010, 0.015, 0.020, 0.025, 0.030,
    0.035, 0.040, 0.045, 0.050, 0.055,
    0.060, 0.065, 0.070, 0.075, 0.080,
    0.085, 0.090, 0.095, 0.100,
]

SENSITIVITY_THRESHOLDS = [
    0.005, 0.010, 0.015, 0.020, 0.025,
    0.030, 0.035, 0.040, 0.045, 0.050,
    0.055, 0.060, 0.065, 0.070, 0.075,
    0.080, 0.085, 0.090, 0.095, 0.100,
    0.105, 0.110, 0.115, 0.120, 0.125,
    0.130, 0.135, 0.140, 0.145, 0.150,
]

BOOTSTRAP_REPLICATES = 2000
BOOTSTRAP_SEED = 20260723

if "MODEL_OUTPUT_DIR" not in globals():
    raise RuntimeError(
        "Önce MODEL_OUTPUT_DIR tanımlı temel hücreyi çalıştır."
    )

# ------------------------------------------------------------
# 1. SHA guards
# ------------------------------------------------------------

guards = [
    (
        "39B_renal_marker_ablation_sensitivity_manifest_SHA256.txt",
        EXPECTED_39B_MANIFEST_SHA,
        "39B renal-marker ablation analysis",
    ),
    (
        "37A_locked_decision_curve_analysis_protocol_v1_SHA256.txt",
        EXPECTED_37A_DCA_PROTOCOL_SHA,
        "37A primary DCA protocol",
    ),
    (
        "39A_R2_locked_renal_marker_ablation_sensitivity_protocol_v1_SHA256.txt",
        EXPECTED_39A_R2_PROTOCOL_SHA,
        "39A-R2 renal-marker ablation protocol",
    ),
]

for filename, expected_sha, label in guards:
    path = os.path.join(MODEL_OUTPUT_DIR, filename)

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    with open(path, "r", encoding="utf-8") as fh:
        observed_sha = fh.read().strip()

    if observed_sha != expected_sha:
        raise RuntimeError(
            f"{label} SHA mismatch: {observed_sha}"
        )

    print(f"{label} SHA guard: PASS")

# ------------------------------------------------------------
# 2. Confirm exact threshold reuse from 37A
# ------------------------------------------------------------

dca_protocol_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "37A_locked_decision_curve_analysis_protocol_v1.json",
)

with open(dca_protocol_path, "r", encoding="utf-8") as fh:
    dca_protocol = json.load(fh)

locked_primary = [
    float(x)
    for x in dca_protocol[
        "primary_threshold_range"
    ]["thresholds"]
]

locked_sensitivity = [
    float(x)
    for x in dca_protocol[
        "sensitivity_threshold_range"
    ]["thresholds"]
]

if locked_primary != PRIMARY_THRESHOLDS:
    raise RuntimeError(
        "Primary thresholds do not exactly match 37A."
    )

if locked_sensitivity != SENSITIVITY_THRESHOLDS:
    raise RuntimeError(
        "Sensitivity thresholds do not exactly match 37A."
    )

print("Primary threshold reuse from 37A: PASS")
print("Sensitivity threshold reuse from 37A: PASS")

# ------------------------------------------------------------
# 3. Lock supplementary DCA protocol
# ------------------------------------------------------------

protocol = {
    "analysis_version": "39C",
    "analysis_type":
        "post_hoc_renal_marker_ablation_decision_curve_protocol_lock",

    "status":
        "exploratory_post_hoc_supplementary_analysis",

    "scientific_question": (
        "Does the renal-marker-ablated XGBoost model retain clinically "
        "meaningful net benefit across the same prespecified threshold "
        "range used for the primary full-model decision curve analysis?"
    ),

    "comparators": [
        "full_locked_xgboost_platt_calibrated",
        "renal_marker_ablated_xgboost_platt_calibrated",
        "treat_all",
        "treat_none",
    ],

    "primary_thresholds": PRIMARY_THRESHOLDS,
    "sensitivity_thresholds": SENSITIVITY_THRESHOLDS,

    "threshold_provenance": {
        "source_protocol":
            "37A_locked_decision_curve_analysis_protocol_v1",
        "source_protocol_sha256":
            EXPECTED_37A_DCA_PROTOCOL_SHA,
        "new_threshold_selection_performed":
            False,
        "post_hoc_optimal_threshold_selection":
            "PROHIBITED",
    },

    "net_benefit_formula":
        "TP/N - FP/N * pt/(1-pt)",

    "bootstrap": {
        "unit": "hospital",
        "paired_resampling": True,
        "replicates": BOOTSTRAP_REPLICATES,
        "seed": BOOTSTRAP_SEED,
        "confidence_intervals":
            "pointwise percentile 95%, exploratory, not multiplicity-adjusted",
    },

    "primary_contrasts": [
        "full_xgboost_minus_ablated_xgboost",
        "ablated_xgboost_minus_treat_all",
        "ablated_xgboost_minus_treat_none",
        "full_xgboost_minus_treat_all",
        "full_xgboost_minus_treat_none",
    ],

    "interpretation_rules": [
        (
            "This DCA is supplementary and post-hoc; it does not alter "
            "the primary full-model DCA."
        ),
        (
            "Positive net benefit of the ablated model indicates residual "
            "decision value after direct creatinine/KDIGO markers are removed; "
            "it does not establish prospective clinical effectiveness."
        ),
        (
            "Do not select or emphasize a single 'best' threshold after "
            "viewing the curves."
        ),
        (
            "Report the full locked threshold ranges."
        ),
        (
            "Pointwise confidence intervals are not simultaneous bands "
            "and are not multiplicity-adjusted."
        ),
    ],

    "modeling_changes": {
        "model_fit_performed_in_39C": False,
        "retuning_performed": False,
        "candidate_reselection_performed": False,
        "recalibration_performed": False,
        "test_driven_changes_performed": False,
    },

    "privacy_and_storage": {
        "patient_level_predictions_written_to_drive": False,
        "aggregate_dca_outputs_may_be_written_to_drive": True,
        "bigquery_dml_used": False,
    },

    "upstream_sha256": {
        "39B_ablation_manifest":
            EXPECTED_39B_MANIFEST_SHA,
        "39A_R2_ablation_protocol":
            EXPECTED_39A_R2_PROTOCOL_SHA,
        "37A_primary_dca_protocol":
            EXPECTED_37A_DCA_PROTOCOL_SHA,
    },

    "dca_result_calculated": False,
    "dca_result_viewed": False,
}

protocol_text = json.dumps(
    protocol,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
)

protocol_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "39C_locked_renal_ablation_dca_protocol_v1.json",
)

sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "39C_locked_renal_ablation_dca_protocol_v1_SHA256.txt",
)

with open(protocol_path, "w", encoding="utf-8") as fh:
    fh.write(protocol_text)

protocol_sha = hashlib.sha256(
    protocol_text.encode("utf-8")
).hexdigest()

with open(sha_path, "w", encoding="utf-8") as fh:
    fh.write(protocol_sha + "\n")

print("\n39C LOCKED SUPPLEMENTARY DCA PROTOCOL")
print("Comparators:")
print("- Full locked XGBoost, Platt calibrated")
print("- Renal-marker-ablated XGBoost, Platt calibrated")
print("- Treat-all")
print("- Treat-none")

print("\nPrimary thresholds:")
print("1.0% to 10.0%, step 0.5 percentage points")

print("\nSensitivity thresholds:")
print("0.5% to 15.0%, step 0.5 percentage points")

print("\nBootstrap:")
print("2,000 paired hospital-cluster replicates")
print("Seed: 20260723")

print("\nPost-hoc optimal-threshold selection: PROHIBITED")
print("Model refitting/retuning/recalibration in 39C: PROHIBITED")

print("\n39C protocol SHA-256:")
print(protocol_sha)

print("\nSaved:")
print(protocol_path)
print(sha_path)

print(
    "\n39C PASS: Supplementary renal-ablation DCA protocol locked "
    "before any ablation DCA result was calculated or viewed."
)

In [ ]:
import os
import json
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

print("STARTING RENAL-MARKER ABLATION DECISION CURVE ANALYSIS — CODE VERSION 39D")

# ============================================================
# 39D — RENAL-MARKER ABLATION DECISION CURVE ANALYSIS
#
# Locked upstream protocol: 39C
#
# Supplementary / post-hoc exploratory DCA comparing:
#   1) Full locked XGBoost, fold-specific Platt calibrated
#   2) Renal-marker-ablated XGBoost, fold-specific Platt calibrated
#   3) Treat-all
#   4) Treat-none
#
# Thresholds are reused EXACTLY from the original 37A DCA lock:
#   Primary:     1.0% to 10.0%, step 0.5 percentage points
#   Sensitivity: 0.5% to 15.0%, step 0.5 percentage points
#
# Uncertainty:
#   2,000 paired hospital-cluster bootstrap replicates
#   seed = 20260723
#
# IMPORTANT:
# - No model is fitted.
# - No retuning, reselection, or recalibration is performed.
# - No post-hoc optimal threshold is selected.
# - Pointwise 95% CIs are exploratory and not multiplicity-adjusted.
# - Patient-level predictions remain in BigQuery / RAM only.
# - Drive receives aggregate DCA outputs and figures only.
# - BigQuery DML is never used.
# ============================================================

EXPECTED_39C_PROTOCOL_SHA = (
    "20987af143234997c8b6dc7ecc1e6847"
    "1d57d7315bcf32cb1bc8183926334ba9"
)

EXPECTED_39B_MANIFEST_SHA = (
    "fe50ce847b8eed3c886d4b2cb7734513"
    "316edad2babdbfcac207f8649f547215"
)

EXPECTED_37A_DCA_PROTOCOL_SHA = (
    "e0f574c228be1d76e75669e7e446df6"
    "d9461dee11883acf5e4b22735dcffcbfb"
)

EXPECTED_ROWS = 58491
EXPECTED_EVENTS = 3032
EXPECTED_HOSPITALS = 198
EXPECTED_OUTER_FOLDS = {1, 2, 3, 4, 5}

BOOTSTRAP_REPLICATES = 2000
BOOTSTRAP_SEED = 20260723

FULL_XGB_TABLE_SUFFIX = (
    "model_xgb_outer_predictions_all5_v1"
)

ABLATION_XGB_TABLE_SUFFIX = (
    "model_xgb_renal_ablation_outer_predictions_all5_v1"
)

PRIMARY_THRESHOLDS = np.round(
    np.arange(
        0.010,
        0.100 + 0.0001,
        0.005,
    ),
    3,
)

SENSITIVITY_THRESHOLDS = np.round(
    np.arange(
        0.005,
        0.150 + 0.0001,
        0.005,
    ),
    3,
)

required_runtime = [
    "MODEL_OUTPUT_DIR",
    "client",
    "TARGET_DATASET",
    "BQ_LOCATION",
    "core_df_07B",
]

missing_runtime = [
    name
    for name in required_runtime
    if name not in globals()
]

if missing_runtime:
    raise RuntimeError(
        "Eksik çalışma nesneleri: "
        + ", ".join(missing_runtime)
        + ". Önce temel 07A/07B hücrelerini çalıştır."
    )

# ------------------------------------------------------------
# 1. SHA guards
# ------------------------------------------------------------

guards = [
    (
        "39C_locked_renal_ablation_dca_protocol_v1_SHA256.txt",
        EXPECTED_39C_PROTOCOL_SHA,
        "39C renal-ablation DCA protocol",
    ),
    (
        "39B_renal_marker_ablation_sensitivity_manifest_SHA256.txt",
        EXPECTED_39B_MANIFEST_SHA,
        "39B renal-marker ablation analysis",
    ),
    (
        "37A_locked_decision_curve_analysis_protocol_v1_SHA256.txt",
        EXPECTED_37A_DCA_PROTOCOL_SHA,
        "37A primary DCA protocol",
    ),
]

for filename, expected_sha, label in guards:
    path = os.path.join(
        MODEL_OUTPUT_DIR,
        filename,
    )

    if not os.path.exists(path):
        raise FileNotFoundError(path)

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as fh:
        observed_sha = fh.read().strip()

    if observed_sha != expected_sha:
        raise RuntimeError(
            f"{label} SHA mismatch: {observed_sha}"
        )

    print(f"{label} SHA guard: PASS")

# ------------------------------------------------------------
# 2. Confirm exact threshold reuse from 39C + 37A
# ------------------------------------------------------------

protocol_39c_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "39C_locked_renal_ablation_dca_protocol_v1.json",
)

protocol_37a_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "37A_locked_decision_curve_analysis_protocol_v1.json",
)

with open(
    protocol_39c_path,
    "r",
    encoding="utf-8",
) as fh:
    protocol_39c = json.load(fh)

with open(
    protocol_37a_path,
    "r",
    encoding="utf-8",
) as fh:
    protocol_37a = json.load(fh)

locked_39c_primary = np.asarray(
    protocol_39c[
        "primary_thresholds"
    ],
    dtype=float,
)

locked_39c_sensitivity = np.asarray(
    protocol_39c[
        "sensitivity_thresholds"
    ],
    dtype=float,
)

locked_37a_primary = np.asarray(
    protocol_37a[
        "primary_threshold_range"
    ][
        "thresholds"
    ],
    dtype=float,
)

locked_37a_sensitivity = np.asarray(
    protocol_37a[
        "sensitivity_threshold_range"
    ][
        "thresholds"
    ],
    dtype=float,
)

if not (
    np.array_equal(
        PRIMARY_THRESHOLDS,
        locked_39c_primary,
    )
    and np.array_equal(
        PRIMARY_THRESHOLDS,
        locked_37a_primary,
    )
):
    raise RuntimeError(
        "Primary threshold grid does not exactly match 39C/37A."
    )

if not (
    np.array_equal(
        SENSITIVITY_THRESHOLDS,
        locked_39c_sensitivity,
    )
    and np.array_equal(
        SENSITIVITY_THRESHOLDS,
        locked_37a_sensitivity,
    )
):
    raise RuntimeError(
        "Sensitivity threshold grid does not exactly match 39C/37A."
    )

print("Locked primary threshold grid guard: PASS")
print("Locked sensitivity threshold grid guard: PASS")

# ------------------------------------------------------------
# 3. Load secure pooled predictions
# ------------------------------------------------------------

def resolve_column(
    columns,
    exact_candidates,
    contains_candidates=None,
):
    lower_map = {
        str(c).lower(): c
        for c in columns
    }

    for candidate in exact_candidates:
        key = candidate.lower()

        if key in lower_map:
            return lower_map[key]

    if contains_candidates:
        matches = []

        for c in columns:
            low = str(c).lower()

            if any(
                token.lower() in low
                for token in contains_candidates
            ):
                matches.append(c)

        if len(matches) == 1:
            return matches[0]

    raise RuntimeError(
        "Could not resolve required column. "
        f"Candidates={exact_candidates}; columns={list(columns)}"
    )


def load_platt_predictions(
    table_suffix,
    output_name,
):
    table_id = (
        f"{TARGET_DATASET}."
        f"{table_suffix}"
    )

    client.get_table(
        table_id
    )

    query = client.query(
        f"SELECT * FROM `{table_id}`",
        location=BQ_LOCATION,
    )

    try:
        raw = query.to_dataframe(
            create_bqstorage_client=True
        )
        load_method = (
            "BigQuery Storage API"
        )
    except Exception:
        raw = query.to_dataframe(
            create_bqstorage_client=False
        )
        load_method = (
            "Standard BigQuery API"
        )

    id_col = resolve_column(
        raw.columns,
        ["id_row"],
        ["id_row"],
    )

    fold_col = resolve_column(
        raw.columns,
        ["outer_fold"],
        ["outer_fold"],
    )

    label_col = resolve_column(
        raw.columns,
        ["label_stage23"],
        ["label_stage23"],
    )

    platt_col = resolve_column(
        raw.columns,
        ["prediction_platt"],
        [
            "prediction_platt",
            "platt",
        ],
    )

    out = pd.DataFrame({
        "id_row":
            raw[
                id_col
            ].astype(str),
        "outer_fold":
            pd.to_numeric(
                raw[
                    fold_col
                ],
                errors="raise",
            ).astype(np.int64),
        "label_stage23":
            pd.to_numeric(
                raw[
                    label_col
                ],
                errors="raise",
            ).astype(np.int64),
        f"{output_name}_platt":
            pd.to_numeric(
                raw[
                    platt_col
                ],
                errors="raise",
            ).astype(float),
    })

    if len(out) != EXPECTED_ROWS:
        raise RuntimeError(
            f"{output_name}: rows={len(out)}, "
            f"expected={EXPECTED_ROWS}."
        )

    if out[
        "id_row"
    ].duplicated().any():
        raise RuntimeError(
            f"{output_name}: duplicate id_row."
        )

    if int(
        out[
            "label_stage23"
        ].sum()
    ) != EXPECTED_EVENTS:
        raise RuntimeError(
            f"{output_name}: event count mismatch."
        )

    if set(
        out[
            "outer_fold"
        ].unique()
    ) != EXPECTED_OUTER_FOLDS:
        raise RuntimeError(
            f"{output_name}: outer folds are not exactly 1..5."
        )

    p = out[
        f"{output_name}_platt"
    ].to_numpy(dtype=float)

    if not np.isfinite(p).all():
        raise RuntimeError(
            f"{output_name}: non-finite probabilities."
        )

    if (
        (p < 0)
        | (p > 1)
    ).any():
        raise RuntimeError(
            f"{output_name}: probabilities outside [0,1]."
        )

    return (
        out,
        {
            "model":
                output_name,
            "table_id":
                table_id,
            "load_method":
                load_method,
            "rows":
                len(out),
            "events":
                int(
                    out[
                        "label_stage23"
                    ].sum()
                ),
        },
    )


print(
    "Loading full locked XGBoost pooled Platt predictions..."
)

full_xgb, full_source = (
    load_platt_predictions(
        FULL_XGB_TABLE_SUFFIX,
        "full_xgboost",
    )
)

print(
    "Loading renal-marker-ablated XGBoost pooled Platt predictions..."
)

ablated_xgb, ablated_source = (
    load_platt_predictions(
        ABLATION_XGB_TABLE_SUFFIX,
        "ablated_xgboost",
    )
)

# ------------------------------------------------------------
# 4. Exact patient / hospital / outcome alignment
# ------------------------------------------------------------

required_core = {
    "id_row",
    "outer_fold",
    "label_stage23",
    "group_hospital",
}

missing_core = (
    required_core
    - set(
        core_df_07B.columns
    )
)

if missing_core:
    raise RuntimeError(
        "core_df_07B missing required columns: "
        + ", ".join(
            sorted(
                missing_core
            )
        )
    )

core = core_df_07B[
    [
        "id_row",
        "outer_fold",
        "label_stage23",
        "group_hospital",
    ]
].copy()

core[
    "id_row"
] = core[
    "id_row"
].astype(str)

core[
    "outer_fold"
] = pd.to_numeric(
    core[
        "outer_fold"
    ],
    errors="raise",
).astype(np.int64)

core[
    "label_stage23"
] = pd.to_numeric(
    core[
        "label_stage23"
    ],
    errors="raise",
).astype(np.int64)

core[
    "group_hospital"
] = core[
    "group_hospital"
].astype(str)

aligned = (
    core
    .merge(
        full_xgb,
        on="id_row",
        how="inner",
        validate="one_to_one",
        suffixes=(
            "",
            "_full",
        ),
    )
    .merge(
        ablated_xgb,
        on="id_row",
        how="inner",
        validate="one_to_one",
        suffixes=(
            "",
            "_ablated",
        ),
    )
)

if len(aligned) != EXPECTED_ROWS:
    raise RuntimeError(
        f"Aligned rows={len(aligned)}, expected={EXPECTED_ROWS}."
    )

for suffix in [
    "_full",
    "_ablated",
]:
    if (
        aligned[
            "outer_fold"
        ]
        != aligned[
            f"outer_fold{suffix}"
        ]
    ).any():
        raise RuntimeError(
            f"Outer-fold mismatch for {suffix}."
        )

    if (
        aligned[
            "label_stage23"
        ]
        != aligned[
            f"label_stage23{suffix}"
        ]
    ).any():
        raise RuntimeError(
            f"Outcome mismatch for {suffix}."
        )

aligned = aligned.drop(
    columns=[
        "outer_fold_full",
        "label_stage23_full",
        "outer_fold_ablated",
        "label_stage23_ablated",
    ]
)

if aligned[
    "group_hospital"
].nunique() != EXPECTED_HOSPITALS:
    raise RuntimeError(
        "Aligned hospital count is not 198."
    )

hospital_fold_n = aligned.groupby(
    "group_hospital"
)[
    "outer_fold"
].nunique()

if (
    hospital_fold_n != 1
).any():
    raise RuntimeError(
        "Hospital cross-fold violation detected."
    )

print(
    "Exact full-vs-ablated patient/hospital/outcome alignment: PASS"
)

# ------------------------------------------------------------
# 5. DCA helpers
# ------------------------------------------------------------

def net_benefit(
    y,
    p,
    threshold,
    sample_weight=None,
):
    y = np.asarray(
        y,
        dtype=np.int8,
    )

    p = np.asarray(
        p,
        dtype=float,
    )

    if sample_weight is None:
        w = np.ones(
            len(y),
            dtype=float,
        )
    else:
        w = np.asarray(
            sample_weight,
            dtype=float,
        )

    total_weight = float(
        w.sum()
    )

    if total_weight <= 0:
        return np.nan

    predicted_positive = (
        p >= threshold
    )

    tp = float(
        w[
            predicted_positive
            & (y == 1)
        ].sum()
    )

    fp = float(
        w[
            predicted_positive
            & (y == 0)
        ].sum()
    )

    odds = (
        threshold
        / (
            1.0
            - threshold
        )
    )

    return (
        tp / total_weight
        - fp / total_weight * odds
    )


def treat_all_net_benefit(
    y,
    threshold,
    sample_weight=None,
):
    y = np.asarray(
        y,
        dtype=np.int8,
    )

    if sample_weight is None:
        w = np.ones(
            len(y),
            dtype=float,
        )
    else:
        w = np.asarray(
            sample_weight,
            dtype=float,
        )

    total_weight = float(
        w.sum()
    )

    if total_weight <= 0:
        return np.nan

    prevalence = float(
        w[
            y == 1
        ].sum()
        / total_weight
    )

    return (
        prevalence
        - (
            1.0
            - prevalence
        )
        * threshold
        / (
            1.0
            - threshold
        )
    )


def calculate_point_curve(
    thresholds,
    y,
    p_full,
    p_ablated,
):
    rows = []

    for pt in thresholds:
        nb_full = net_benefit(
            y,
            p_full,
            pt,
        )

        nb_ablated = net_benefit(
            y,
            p_ablated,
            pt,
        )

        nb_all = treat_all_net_benefit(
            y,
            pt,
        )

        nb_none = 0.0

        rows.append({
            "threshold":
                float(pt),
            "threshold_percent":
                float(
                    pt * 100.0
                ),
            "full_xgboost_net_benefit":
                nb_full,
            "ablated_xgboost_net_benefit":
                nb_ablated,
            "treat_all_net_benefit":
                nb_all,
            "treat_none_net_benefit":
                nb_none,
            "full_minus_ablated":
                nb_full
                - nb_ablated,
            "ablated_minus_treat_all":
                nb_ablated
                - nb_all,
            "ablated_minus_treat_none":
                nb_ablated
                - nb_none,
            "full_minus_treat_all":
                nb_full
                - nb_all,
            "full_minus_treat_none":
                nb_full
                - nb_none,
        })

    return pd.DataFrame(
        rows
    )


y = aligned[
    "label_stage23"
].to_numpy(
    dtype=np.int8
)

p_full = aligned[
    "full_xgboost_platt"
].to_numpy(
    dtype=float
)

p_ablated = aligned[
    "ablated_xgboost_platt"
].to_numpy(
    dtype=float
)

primary_point = calculate_point_curve(
    PRIMARY_THRESHOLDS,
    y,
    p_full,
    p_ablated,
)

sensitivity_point = calculate_point_curve(
    SENSITIVITY_THRESHOLDS,
    y,
    p_full,
    p_ablated,
)

# ------------------------------------------------------------
# 6. Paired hospital-cluster bootstrap
# ------------------------------------------------------------

print(
    f"\nRunning {BOOTSTRAP_REPLICATES:,} paired "
    "hospital-cluster DCA bootstrap replicates..."
)

hospital_cat = pd.Categorical(
    aligned[
        "group_hospital"
    ]
)

hospital_codes = (
    hospital_cat.codes.astype(
        int
    )
)

if len(
    hospital_cat.categories
) != EXPECTED_HOSPITALS:
    raise RuntimeError(
        "Bootstrap hospital count is not 198."
    )

rng = np.random.default_rng(
    BOOTSTRAP_SEED
)

bootstrap_rows = []

for b in range(
    BOOTSTRAP_REPLICATES
):
    sampled_codes = rng.integers(
        0,
        EXPECTED_HOSPITALS,
        size=EXPECTED_HOSPITALS,
    )

    multiplicity = np.bincount(
        sampled_codes,
        minlength=EXPECTED_HOSPITALS,
    )

    weights = (
        multiplicity[
            hospital_codes
        ].astype(float)
    )

    if (
        weights[
            y == 1
        ].sum() <= 0
        or weights[
            y == 0
        ].sum() <= 0
    ):
        continue

    for pt in (
        SENSITIVITY_THRESHOLDS
    ):
        nb_full = net_benefit(
            y,
            p_full,
            pt,
            sample_weight=weights,
        )

        nb_ablated = net_benefit(
            y,
            p_ablated,
            pt,
            sample_weight=weights,
        )

        nb_all = treat_all_net_benefit(
            y,
            pt,
            sample_weight=weights,
        )

        bootstrap_rows.append({
            "bootstrap_replicate":
                b + 1,
            "threshold":
                float(pt),
            "full_xgboost_net_benefit":
                nb_full,
            "ablated_xgboost_net_benefit":
                nb_ablated,
            "treat_all_net_benefit":
                nb_all,
            "treat_none_net_benefit":
                0.0,
            "full_minus_ablated":
                nb_full
                - nb_ablated,
            "ablated_minus_treat_all":
                nb_ablated
                - nb_all,
            "ablated_minus_treat_none":
                nb_ablated,
            "full_minus_treat_all":
                nb_full
                - nb_all,
            "full_minus_treat_none":
                nb_full,
        })

    if (
        b == 0
        or (
            b + 1
        ) % 100 == 0
    ):
        print(
            "  Completed DCA bootstrap replicate",
            b + 1,
            "/",
            BOOTSTRAP_REPLICATES,
        )

bootstrap_df = pd.DataFrame(
    bootstrap_rows
)

completed_replicates = bootstrap_df[
    "bootstrap_replicate"
].nunique()

if (
    completed_replicates
    < BOOTSTRAP_REPLICATES * 0.99
):
    raise RuntimeError(
        "Fewer than 99% of DCA bootstrap replicates completed."
    )

# ------------------------------------------------------------
# 7. Pointwise percentile 95% CIs
# ------------------------------------------------------------

CI_COLUMNS = [
    "full_xgboost_net_benefit",
    "ablated_xgboost_net_benefit",
    "treat_all_net_benefit",
    "full_minus_ablated",
    "ablated_minus_treat_all",
    "ablated_minus_treat_none",
    "full_minus_treat_all",
    "full_minus_treat_none",
]

ci_rows = []

for pt, sub in bootstrap_df.groupby(
    "threshold",
    sort=True,
):
    row = {
        "threshold":
            float(pt),
        "threshold_percent":
            float(
                pt * 100.0
            ),
        "bootstrap_replicates":
            int(
                sub[
                    "bootstrap_replicate"
                ].nunique()
            ),
        "ci_type":
            (
                "pointwise_percentile_95_"
                "exploratory_not_multiplicity_adjusted"
            ),
    }

    for col in CI_COLUMNS:
        values = sub[
            col
        ].to_numpy(
            dtype=float
        )

        row[
            f"{col}_ci95_lower"
        ] = float(
            np.quantile(
                values,
                0.025,
            )
        )

        row[
            f"{col}_ci95_upper"
        ] = float(
            np.quantile(
                values,
                0.975,
            )
        )

    ci_rows.append(
        row
    )

pointwise_ci = pd.DataFrame(
    ci_rows
)

primary_result = (
    primary_point
    .merge(
        pointwise_ci,
        on=[
            "threshold",
            "threshold_percent",
        ],
        how="left",
        validate="one_to_one",
    )
)

sensitivity_result = (
    sensitivity_point
    .merge(
        pointwise_ci,
        on=[
            "threshold",
            "threshold_percent",
        ],
        how="left",
        validate="one_to_one",
    )
)

# ------------------------------------------------------------
# 8. Locked-range summaries
#    No threshold optimization.
# ------------------------------------------------------------

def summarize_range(
    result,
    range_name,
):
    return pd.DataFrame([{
        "range_name":
            range_name,
        "threshold_count":
            len(result),
        "minimum_threshold":
            float(
                result[
                    "threshold"
                ].min()
            ),
        "maximum_threshold":
            float(
                result[
                    "threshold"
                ].max()
            ),

        "full_nb_positive_thresholds":
            int(
                (
                    result[
                        "full_xgboost_net_benefit"
                    ] > 0
                ).sum()
            ),

        "ablated_nb_positive_thresholds":
            int(
                (
                    result[
                        "ablated_xgboost_net_benefit"
                    ] > 0
                ).sum()
            ),

        "full_above_ablated_thresholds":
            int(
                (
                    result[
                        "full_minus_ablated"
                    ] > 0
                ).sum()
            ),

        "full_minus_ablated_ci_lower_gt_zero_thresholds":
            int(
                (
                    result[
                        "full_minus_ablated_ci95_lower"
                    ] > 0
                ).sum()
            ),

        "ablated_above_treat_all_thresholds":
            int(
                (
                    result[
                        "ablated_minus_treat_all"
                    ] > 0
                ).sum()
            ),

        "ablated_minus_treat_all_ci_lower_gt_zero_thresholds":
            int(
                (
                    result[
                        "ablated_minus_treat_all_ci95_lower"
                    ] > 0
                ).sum()
            ),

        "ablated_above_treat_none_thresholds":
            int(
                (
                    result[
                        "ablated_minus_treat_none"
                    ] > 0
                ).sum()
            ),

        "ablated_minus_treat_none_ci_lower_gt_zero_thresholds":
            int(
                (
                    result[
                        "ablated_minus_treat_none_ci95_lower"
                    ] > 0
                ).sum()
            ),

        "optimal_threshold_selected":
            False,
    }])


range_summary = pd.concat(
    [
        summarize_range(
            primary_result,
            "primary_1_to_10_percent",
        ),
        summarize_range(
            sensitivity_result,
            "sensitivity_0_5_to_15_percent",
        ),
    ],
    ignore_index=True,
)

# ------------------------------------------------------------
# 9. Figures
# ------------------------------------------------------------

primary_figure_png = os.path.join(
    MODEL_OUTPUT_DIR,
    "39D_primary_full_vs_renal_ablation_DCA_1_to_10_percent.png",
)

sensitivity_figure_png = os.path.join(
    MODEL_OUTPUT_DIR,
    "39D_sensitivity_full_vs_renal_ablation_DCA_0_5_to_15_percent.png",
)

fig = plt.figure(
    figsize=(
        8.5,
        6.0,
    )
)

ax = fig.add_subplot(
    111
)

ax.plot(
    primary_result[
        "threshold_percent"
    ],
    primary_result[
        "full_xgboost_net_benefit"
    ],
    label="Full XGBoost",
)

ax.plot(
    primary_result[
        "threshold_percent"
    ],
    primary_result[
        "ablated_xgboost_net_benefit"
    ],
    label="Renal-marker-ablated XGBoost",
)

ax.plot(
    primary_result[
        "threshold_percent"
    ],
    primary_result[
        "treat_all_net_benefit"
    ],
    label="Treat all",
)

ax.plot(
    primary_result[
        "threshold_percent"
    ],
    primary_result[
        "treat_none_net_benefit"
    ],
    label="Treat none",
)

ax.set_xlabel(
    "Threshold probability (%)"
)

ax.set_ylabel(
    "Net benefit"
)

ax.set_title(
    "Supplementary Decision Curve Analysis: Locked Primary Range"
)

ax.legend()

ax.grid(
    True,
    alpha=0.2,
)

fig.tight_layout()

fig.savefig(
    primary_figure_png,
    dpi=300,
    bbox_inches="tight",
)

plt.close(
    fig
)

fig = plt.figure(
    figsize=(
        8.5,
        6.0,
    )
)

ax = fig.add_subplot(
    111
)

ax.plot(
    sensitivity_result[
        "threshold_percent"
    ],
    sensitivity_result[
        "full_xgboost_net_benefit"
    ],
    label="Full XGBoost",
)

ax.plot(
    sensitivity_result[
        "threshold_percent"
    ],
    sensitivity_result[
        "ablated_xgboost_net_benefit"
    ],
    label="Renal-marker-ablated XGBoost",
)

ax.plot(
    sensitivity_result[
        "threshold_percent"
    ],
    sensitivity_result[
        "treat_all_net_benefit"
    ],
    label="Treat all",
)

ax.plot(
    sensitivity_result[
        "threshold_percent"
    ],
    sensitivity_result[
        "treat_none_net_benefit"
    ],
    label="Treat none",
)

ax.set_xlabel(
    "Threshold probability (%)"
)

ax.set_ylabel(
    "Net benefit"
)

ax.set_title(
    "Supplementary Decision Curve Analysis: Locked Sensitivity Range"
)

ax.legend()

ax.grid(
    True,
    alpha=0.2,
)

fig.tight_layout()

fig.savefig(
    sensitivity_figure_png,
    dpi=300,
    bbox_inches="tight",
)

plt.close(
    fig
)

# ------------------------------------------------------------
# 10. Save aggregate outputs only
# ------------------------------------------------------------

paths = {
    "source_audit":
        os.path.join(
            MODEL_OUTPUT_DIR,
            "39D_DCA_source_prediction_audit.csv",
        ),

    "alignment_integrity":
        os.path.join(
            MODEL_OUTPUT_DIR,
            "39D_DCA_alignment_integrity.csv",
        ),

    "primary_curve":
        os.path.join(
            MODEL_OUTPUT_DIR,
            "39D_DCA_primary_1_to_10_percent.csv",
        ),

    "sensitivity_curve":
        os.path.join(
            MODEL_OUTPUT_DIR,
            "39D_DCA_sensitivity_0_5_to_15_percent.csv",
        ),

    "pointwise_ci":
        os.path.join(
            MODEL_OUTPUT_DIR,
            "39D_DCA_pointwise_hospital_bootstrap_95CI.csv",
        ),

    "range_summary":
        os.path.join(
            MODEL_OUTPUT_DIR,
            "39D_DCA_locked_range_summary.csv",
        ),

    "primary_figure_png":
        primary_figure_png,

    "sensitivity_figure_png":
        sensitivity_figure_png,

    "manifest":
        os.path.join(
            MODEL_OUTPUT_DIR,
            "39D_renal_ablation_decision_curve_analysis_manifest.json",
        ),

    "manifest_sha":
        os.path.join(
            MODEL_OUTPUT_DIR,
            "39D_renal_ablation_decision_curve_analysis_manifest_SHA256.txt",
        ),
}

pd.DataFrame(
    [
        full_source,
        ablated_source,
    ]
).to_csv(
    paths[
        "source_audit"
    ],
    index=False,
)

alignment_integrity = pd.DataFrame([{
    "patients":
        len(aligned),
    "distinct_patients":
        aligned[
            "id_row"
        ].nunique(),
    "hospitals":
        aligned[
            "group_hospital"
        ].nunique(),
    "outer_folds":
        aligned[
            "outer_fold"
        ].nunique(),
    "events":
        int(
            aligned[
                "label_stage23"
            ].sum()
        ),
    "nonevents":
        int(
            len(aligned)
            - aligned[
                "label_stage23"
            ].sum()
        ),
    "hospital_cross_fold_violations":
        int(
            (
                hospital_fold_n != 1
            ).sum()
        ),
}])

alignment_integrity.to_csv(
    paths[
        "alignment_integrity"
    ],
    index=False,
)

primary_result.to_csv(
    paths[
        "primary_curve"
    ],
    index=False,
)

sensitivity_result.to_csv(
    paths[
        "sensitivity_curve"
    ],
    index=False,
)

pointwise_ci.to_csv(
    paths[
        "pointwise_ci"
    ],
    index=False,
)

range_summary.to_csv(
    paths[
        "range_summary"
    ],
    index=False,
)

manifest = {
    "analysis_version":
        "39D",

    "analysis_type":
        "post_hoc_renal_marker_ablation_decision_curve_analysis",

    "source_39C_protocol_sha256":
        EXPECTED_39C_PROTOCOL_SHA,

    "source_39B_manifest_sha256":
        EXPECTED_39B_MANIFEST_SHA,

    "source_37A_dca_protocol_sha256":
        EXPECTED_37A_DCA_PROTOCOL_SHA,

    "patients":
        EXPECTED_ROWS,

    "hospitals":
        EXPECTED_HOSPITALS,

    "events":
        EXPECTED_EVENTS,

    "comparators": [
        "full_locked_xgboost_platt_calibrated",
        "renal_marker_ablated_xgboost_platt_calibrated",
        "treat_all",
        "treat_none",
    ],

    "primary_thresholds":
        PRIMARY_THRESHOLDS.tolist(),

    "sensitivity_thresholds":
        SENSITIVITY_THRESHOLDS.tolist(),

    "net_benefit_formula":
        "TP/N - FP/N * pt/(1-pt)",

    "bootstrap_replicates_requested":
        BOOTSTRAP_REPLICATES,

    "bootstrap_replicates_completed":
        int(
            completed_replicates
        ),

    "bootstrap_unit":
        "hospital",

    "paired_resampling":
        True,

    "bootstrap_seed":
        BOOTSTRAP_SEED,

    "confidence_intervals":
        (
            "pointwise_percentile_95_"
            "exploratory_not_multiplicity_adjusted"
        ),

    "post_hoc_optimal_threshold_selected":
        False,

    "model_fit_performed":
        False,

    "retuning_performed":
        False,

    "candidate_reselection_performed":
        False,

    "recalibration_performed":
        False,

    "patient_level_data_written_to_drive":
        False,

    "bigquery_dml_used":
        False,

    "interpretation_scope":
        (
            "Exploratory supplementary clinical-utility analysis. "
            "Does not establish prospective clinical effectiveness."
        ),

    "outputs":
        paths,
}

manifest_text = json.dumps(
    manifest,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
)

with open(
    paths[
        "manifest"
    ],
    "w",
    encoding="utf-8",
) as fh:
    fh.write(
        manifest_text
    )

manifest_sha = hashlib.sha256(
    manifest_text.encode(
        "utf-8"
    )
).hexdigest()

with open(
    paths[
        "manifest_sha"
    ],
    "w",
    encoding="utf-8",
) as fh:
    fh.write(
        manifest_sha
        + "\n"
    )

# ------------------------------------------------------------
# 11. Compact display
# ------------------------------------------------------------

print(
    "\n39D DCA ALIGNMENT INTEGRITY"
)

display(
    alignment_integrity
)

print(
    "\n39D PRIMARY DCA — 1% TO 10%"
)

display(
    primary_result[
        [
            "threshold_percent",
            "full_xgboost_net_benefit",
            "ablated_xgboost_net_benefit",
            "treat_all_net_benefit",
            "treat_none_net_benefit",
            "full_minus_ablated",
            "full_minus_ablated_ci95_lower",
            "full_minus_ablated_ci95_upper",
            "ablated_minus_treat_all",
            "ablated_minus_treat_all_ci95_lower",
            "ablated_minus_treat_all_ci95_upper",
        ]
    ]
)

print(
    "\n39D LOCKED-RANGE DCA SUMMARY"
)

display(
    range_summary
)

print(
    "\n39D manifest SHA-256:"
)

print(
    manifest_sha
)

print(
    "\nSaved figures:"
)

print(
    primary_figure_png
)

print(
    sensitivity_figure_png
)

print(
    "\n39D PASS: Supplementary renal-marker-ablation DCA completed "
    "over the full pre-locked threshold ranges."
)

print(
    "No post-hoc optimal threshold was selected."
)

print(
    "Pointwise 95% bootstrap CIs are exploratory and not "
    "multiplicity-adjusted."
)

print(
    "No model was fitted, retuned, reselected, or recalibrated."
)

print(
    "No patient-level file was written to Google Drive."
)

In [ ]:
import os
import glob
import json
import hashlib
import pandas as pd

print("STARTING FINAL ANALYSIS FREEZE AND RESULTS REGISTRY — CODE VERSION 40A")

# ============================================================
# 40A — FINAL ANALYSIS FREEZE AND RESULTS REGISTRY
#
# Purpose:
#   Freeze the completed eICU AKI modeling/validation analysis before
#   manuscript assembly. This step performs NO model fitting, tuning,
#   recalibration, bootstrap, SHAP calculation, or DCA calculation.
#
# It only:
#   1) verifies the critical upstream aggregate/protocol SHA locks,
#   2) records the canonical results already obtained,
#   3) freezes claim language / provenance,
#   4) writes a final aggregate results registry and freeze manifest.
#
# After 40A PASS:
#   - no further eICU model retuning is permitted;
#   - no new post-hoc eICU comparison should be added merely to improve
#     the manuscript;
#   - any genuinely new analysis must be separately labeled and locked;
#   - an independent external validation dataset (e.g. MIMIC-IV) would be
#     a new validation phase rather than an alteration of this frozen work.
#
# No patient-level data are read or written.
# No BigQuery action is performed.
# ============================================================

if "MODEL_OUTPUT_DIR" not in globals():
    raise RuntimeError(
        "Önce MODEL_OUTPUT_DIR tanımlı temel hücreyi çalıştır."
    )

# ------------------------------------------------------------------
# 1. Critical provenance locks
# ------------------------------------------------------------------

EXPECTED_SHA = {
    "31_four_model_paired_comparison":
        "90ba7b07afd94c733dd231f0eaca300a8c47e70565221373ddba701cc8fc8928",

    "34_clinical_baseline":
        "a1eb23d936154d521048c88e8b34d58314c38755c8b8e5126741e17c1a9f1924",

    "35E_temporal_leakage_audit":
        "7230605e14a2314ec1724f48c61e42d207f96c70ae2d4fe480bf02145ad24544",

    "36_model_vs_clinical_comparison":
        "f31ad0df192bc0872d1626771a773c50b7fff798534751144d9a060f82d5f718",

    "37B_primary_dca":
        "03a14b54c4fde8f23cfbc0ff4276f84c5443663cb0337d052fdf1989933238b4",

    "38C_R1_treeshap":
        "0469f6e3a1f31c5621904d4fbfd3c35904ccd435eca805ddc60f833f925ec521",

    "39B_renal_marker_ablation":
        "fe50ce847b8eed3c886d4b2cb7734513316edad2babdbfcac207f8649f547215",

    "39D_renal_ablation_dca":
        "294d18b1194af2b582a056f0a245d9a9918243436b73f59b4aeb044b6d93ea05",
}


def resolve_sha_artifact(expected_sha):
    candidates = sorted(
        glob.glob(
            os.path.join(
                MODEL_OUTPUT_DIR,
                "**",
                "*.txt",
            ),
            recursive=True,
        )
    )

    matches = []

    for path in candidates:
        if not os.path.isfile(path):
            continue

        try:
            with open(
                path,
                "r",
                encoding="utf-8",
                errors="ignore",
            ) as fh:
                content = fh.read()
        except Exception:
            continue

        if expected_sha in content:
            matches.append(path)

    if not matches:
        raise RuntimeError(
            "Expected SHA not found in MODEL_OUTPUT_DIR text artifacts:\n"
            + expected_sha
        )

    # Prefer an explicitly named SHA file when available.
    matches = sorted(
        matches,
        key=lambda p: (
            "sha256" not in os.path.basename(p).lower(),
            len(p),
            p,
        )
    )

    return matches[0]


resolved_sha_artifacts = {}

for label, expected_sha in EXPECTED_SHA.items():
    path = resolve_sha_artifact(
        expected_sha
    )

    resolved_sha_artifacts[label] = {
        "sha256": expected_sha,
        "artifact":
            os.path.relpath(
                path,
                MODEL_OUTPUT_DIR,
            ),
    }

    print(
        f"{label} SHA guard: PASS -> "
        f"{os.path.basename(path)}"
    )

# ------------------------------------------------------------------
# 2. Canonical frozen results registry
#    Values below are already-established results, not new calculations.
# ------------------------------------------------------------------

registry_rows = [
    {
        "section": "cohort",
        "item": "patients",
        "value": 58491,
        "value_2": None,
        "status": "frozen",
        "scope": "primary cohort",
    },
    {
        "section": "cohort",
        "item": "hospitals",
        "value": 198,
        "value_2": None,
        "status": "frozen",
        "scope": "primary cohort",
    },
    {
        "section": "cohort",
        "item": "events_stage23_12_to_72h",
        "value": 3032,
        "value_2": 0.051837,
        "status": "frozen",
        "scope": "primary cohort / event rate",
    },

    # Pooled Platt-calibrated model performance.
    {
        "section": "model_performance",
        "item": "XGBoost_AUROC",
        "value": 0.875804,
        "value_2": None,
        "status": "frozen",
        "scope": "main nonlinear model / pooled Platt",
    },
    {
        "section": "model_performance",
        "item": "XGBoost_AUPRC",
        "value": 0.398832,
        "value_2": None,
        "status": "frozen",
        "scope": "main nonlinear model / pooled Platt",
    },
    {
        "section": "model_performance",
        "item": "XGBoost_Brier",
        "value": 0.037800,
        "value_2": None,
        "status": "frozen",
        "scope": "main nonlinear model / pooled Platt",
    },
    {
        "section": "model_performance",
        "item": "XGBoost_log_loss",
        "value": 0.142019,
        "value_2": None,
        "status": "frozen",
        "scope": "main nonlinear model / pooled Platt",
    },

    {
        "section": "model_performance",
        "item": "CatBoost_AUROC",
        "value": 0.871126,
        "value_2": None,
        "status": "frozen",
        "scope": "additional/post-hoc benchmark / pooled Platt",
    },
    {
        "section": "model_performance",
        "item": "CatBoost_AUPRC",
        "value": 0.393057,
        "value_2": None,
        "status": "frozen",
        "scope": "additional/post-hoc benchmark / pooled Platt",
    },

    {
        "section": "model_performance",
        "item": "RandomForest_AUROC",
        "value": 0.867524,
        "value_2": None,
        "status": "frozen",
        "scope": "additional/post-hoc benchmark / pooled Platt",
    },
    {
        "section": "model_performance",
        "item": "RandomForest_AUPRC",
        "value": 0.389310,
        "value_2": None,
        "status": "frozen",
        "scope": "additional/post-hoc benchmark / pooled Platt",
    },

    {
        "section": "model_performance",
        "item": "FullLogistic_AUROC",
        "value": 0.847925,
        "value_2": None,
        "status": "frozen",
        "scope": "main linear model / pooled Platt",
    },
    {
        "section": "model_performance",
        "item": "FullLogistic_AUPRC",
        "value": 0.314647,
        "value_2": None,
        "status": "frozen",
        "scope": "main linear model / pooled Platt",
    },

    {
        "section": "clinical_baseline",
        "item": "ParsimoniousClinical_AUROC",
        "value": 0.814786,
        "value_2": None,
        "status": "frozen",
        "scope": "additional/post-hoc exploratory benchmark / pooled Platt",
    },
    {
        "section": "clinical_baseline",
        "item": "ParsimoniousClinical_AUPRC",
        "value": 0.262471,
        "value_2": None,
        "status": "frozen",
        "scope": "additional/post-hoc exploratory benchmark / pooled Platt",
    },

    # Critical provenance.
    {
        "section": "temporal_audit",
        "item": "critical_kidney_outcome_leakage_audit",
        "value": 1,
        "value_2": 0,
        "status": "PASS",
        "scope":
            "reference creatinine, last creatinine, stage1 landmark, outcome",
    },

    # Primary DCA.
    {
        "section": "primary_dca",
        "item": "XGB_positive_net_benefit_primary_thresholds",
        "value": 19,
        "value_2": 19,
        "status": "frozen",
        "scope": "1-10% locked threshold range",
    },
    {
        "section": "primary_dca",
        "item": "XGB_above_clinical_primary_thresholds",
        "value": 19,
        "value_2": 19,
        "status": "frozen",
        "scope": "point estimates / 1-10% locked threshold range",
    },

    # SHAP.
    {
        "section": "explainability",
        "item": "SHAP_rank1_reference_creatinine",
        "value": 0.494474,
        "value_2": 1,
        "status": "frozen",
        "scope": "pooled grouped mean absolute SHAP / rank",
    },
    {
        "section": "explainability",
        "item": "SHAP_rank2_last_creatinine",
        "value": 0.340731,
        "value_2": 2,
        "status": "frozen",
        "scope": "pooled grouped mean absolute SHAP / rank",
    },
    {
        "section": "explainability",
        "item": "SHAP_rank3_stage1_landmark",
        "value": 0.291037,
        "value_2": 3,
        "status": "frozen",
        "scope": "pooled grouped mean absolute SHAP / rank",
    },
    {
        "section": "explainability",
        "item": "SHAP_fold_rank_spearman_range",
        "value": 0.818737,
        "value_2": 0.890065,
        "status": "frozen",
        "scope": "pairwise fold source-rank correlations",
    },

    # Renal-marker ablation.
    {
        "section": "renal_marker_ablation",
        "item": "AblatedXGB_AUROC",
        "value": 0.786269,
        "value_2": None,
        "status": "frozen",
        "scope": "post-hoc sensitivity / pooled Platt",
    },
    {
        "section": "renal_marker_ablation",
        "item": "AblatedXGB_AUPRC",
        "value": 0.209285,
        "value_2": None,
        "status": "frozen",
        "scope": "post-hoc sensitivity / pooled Platt",
    },
    {
        "section": "renal_marker_ablation",
        "item": "Full_minus_Ablated_AUROC",
        "value": 0.089535,
        "value_2": None,
        "status": "frozen",
        "scope": "post-hoc exploratory paired comparison / Platt",
    },
    {
        "section": "renal_marker_ablation",
        "item": "Full_minus_Ablated_AUROC_CI95",
        "value": 0.081057,
        "value_2": 0.097795,
        "status": "frozen",
        "scope": "nominal paired hospital bootstrap 95% CI",
    },
    {
        "section": "renal_marker_ablation",
        "item": "Full_minus_Ablated_AUPRC",
        "value": 0.189547,
        "value_2": None,
        "status": "frozen",
        "scope": "post-hoc exploratory paired comparison / Platt",
    },
    {
        "section": "renal_marker_ablation",
        "item": "Full_minus_Ablated_AUPRC_CI95",
        "value": 0.172183,
        "value_2": 0.206503,
        "status": "frozen",
        "scope": "nominal paired hospital bootstrap 95% CI",
    },

    # Supplementary ablation DCA.
    {
        "section": "renal_marker_ablation_dca",
        "item": "Full_above_Ablated_primary",
        "value": 19,
        "value_2": 19,
        "status": "frozen",
        "scope": "point estimate / lower pointwise CI >0",
    },
    {
        "section": "renal_marker_ablation_dca",
        "item": "Ablated_above_TreatAll_primary",
        "value": 19,
        "value_2": 19,
        "status": "frozen",
        "scope": "point estimate / lower pointwise CI >0",
    },
    {
        "section": "renal_marker_ablation_dca",
        "item": "Ablated_above_TreatNone_primary",
        "value": 19,
        "value_2": 19,
        "status": "frozen",
        "scope": "point estimate / lower pointwise CI >0",
    },
    {
        "section": "renal_marker_ablation_dca",
        "item": "Ablated_above_TreatAll_sensitivity",
        "value": 30,
        "value_2": 29,
        "status": "frozen",
        "scope":
            "point estimate / thresholds with lower pointwise CI >0; 0.5-15%",
    },
]

registry = pd.DataFrame(
    registry_rows
)

# ------------------------------------------------------------------
# 3. Manuscript reporting limits
# ------------------------------------------------------------------

reporting_notes = [
    (
        "Use 'internal-external cross-validation' for the eICU "
        "hospital-disjoint outer-fold design. Do not call it independent "
        "external validation."
    ),
    (
        "The primary outcome is incident/progressive KDIGO stage 2-3 "
        "after the 12-hour landmark through 72 hours."
    ),
    (
        "Main scientific framing: early prediction of AKI progression "
        "across unseen hospitals."
    ),
    (
        "XGBoost had the best overall numerical performance, but the three "
        "nonlinear ensemble learners formed a broadly comparable upper "
        "performance cluster. Do not claim universal or dramatic superiority."
    ),
    (
        "CatBoost and Random Forest are additional/post-hoc benchmarks "
        "unless separate prior provenance demonstrates otherwise."
    ),
    (
        "The parsimonious clinical logistic baseline is an additional/post-hoc "
        "exploratory benchmark. qSOFA and SIRS were not adequately reconstructable "
        "within 0-12 hours; do not state that they underperformed."
    ),
    (
        "Temporal leakage audit supports the validity of the critical kidney/outcome "
        "timing. It does not claim that every one of the 159 predictors was independently "
        "reconstructed from source SQL."
    ),
    (
        "SHAP values are descriptive model attributions on the raw-margin scale, "
        "not causal effects."
    ),
    (
        "Do not interpret pooled mean signed SHAP as a monotonic direction-of-risk coefficient. "
        "Use dependence/beeswarm plots for directionality."
    ),
    (
        "Primary DCA supports potential clinical utility across the prespecified threshold "
        "range; it does not establish prospective clinical effectiveness."
    ),
    (
        "DCA confidence intervals are pointwise, not simultaneous, and are not "
        "multiplicity-adjusted."
    ),
    (
        "Renal-marker ablation is explicitly post-hoc/exploratory. The performance drop "
        "shows substantial reliance on legitimate pre-landmark renal-state information; "
        "it does not demonstrate leakage."
    ),
    (
        "The ablated model retained residual discrimination (AUROC about 0.786) and positive "
        "net benefit across the locked threshold ranges, supporting additional predictive "
        "information beyond direct creatinine/KDIGO markers."
    ),
]

# ------------------------------------------------------------------
# 4. Freeze manifest
# ------------------------------------------------------------------

freeze_manifest = {
    "analysis_version": "40A",
    "analysis_type":
        "final_eicu_aki_analysis_freeze_and_results_registry",

    "status": "FROZEN",

    "cohort": {
        "patients": 58491,
        "hospitals": 198,
        "events": 3032,
        "event_rate": 0.051837,
        "landmark_hours": 12,
        "outcome_window_hours": [
            12,
            72,
        ],
    },

    "critical_upstream_sha_locks":
        resolved_sha_artifacts,

    "canonical_primary_result": {
        "model":
            "XGBoost",
        "probability":
            "fold-specific Platt calibrated",
        "AUROC":
            0.875804,
        "AUPRC":
            0.398832,
        "Brier":
            0.037800,
        "log_loss":
            0.142019,
    },

    "canonical_clinical_baseline": {
        "status":
            "additional_post_hoc_exploratory",
        "AUROC":
            0.814786,
        "AUPRC":
            0.262471,
        "Brier":
            0.043773,
        "log_loss":
            0.167491,
    },

    "renal_marker_ablation": {
        "status":
            "post_hoc_exploratory_sensitivity",
        "removed_source_predictors":
            8,
        "retained_source_predictors":
            151,
        "AUROC":
            0.786269,
        "AUPRC":
            0.209285,
        "Brier":
            0.044940,
        "log_loss":
            0.174346,
        "full_minus_ablated_AUROC":
            0.089535,
        "full_minus_ablated_AUROC_CI95": [
            0.081057,
            0.097795,
        ],
        "full_minus_ablated_AUPRC":
            0.189547,
        "full_minus_ablated_AUPRC_CI95": [
            0.172183,
            0.206503,
        ],
    },

    "explainability": {
        "method":
            "verified native TreeSHAP on locked outer-test XGBoost models",
        "top3_source_predictors": [
            {
                "rank": 1,
                "predictor":
                    "x_reference_creatinine",
                "pooled_mean_abs_grouped_shap":
                    0.494474,
            },
            {
                "rank": 2,
                "predictor":
                    "x_lab_creatinine_last",
                "pooled_mean_abs_grouped_shap":
                    0.340731,
            },
            {
                "rank": 3,
                "predictor":
                    "x_stage1_at_landmark",
                "pooled_mean_abs_grouped_shap":
                    0.291037,
            },
        ],
        "fold_rank_spearman_range": [
            0.818737,
            0.890065,
        ],
        "causal_interpretation_permitted":
            False,
    },

    "decision_curve_analysis": {
        "primary_threshold_range":
            "1%-10% in 0.5 percentage-point increments",
        "sensitivity_threshold_range":
            "0.5%-15% in 0.5 percentage-point increments",
        "bootstrap":
            "2000 paired hospital-cluster replicates",
        "optimal_threshold_selected":
            False,
        "prospective_clinical_effectiveness_claim_permitted":
            False,
    },

    "analysis_freeze_rules": {
        "eicu_model_retuning_after_40A":
            "PROHIBITED",
        "test_driven_model_changes_after_40A":
            "PROHIBITED",
        "new_eicu_post_hoc_analysis_for_result_improvement":
            "PROHIBITED",
        "new_analysis_only_if_scientifically_necessary_and_separately_locked":
            True,
        "independent_external_validation":
            "allowed_as_new_validation_phase_without altering frozen eICU results",
    },

    "patient_level_data_written_by_40A":
        False,
    "bigquery_action_performed_by_40A":
        False,
}

# ------------------------------------------------------------------
# 5. Write aggregate freeze artifacts
# ------------------------------------------------------------------

registry_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "40A_final_results_registry.csv",
)

reporting_notes_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "40A_manuscript_reporting_notes.txt",
)

manifest_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "40A_FINAL_ANALYSIS_FREEZE_manifest.json",
)

manifest_sha_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "40A_FINAL_ANALYSIS_FREEZE_manifest_SHA256.txt",
)

registry.to_csv(
    registry_path,
    index=False,
)

with open(
    reporting_notes_path,
    "w",
    encoding="utf-8",
) as fh:
    for i, rule in enumerate(
        reporting_notes,
        start=1,
    ):
        fh.write(
            f"{i}. {rule}\n"
        )

manifest_text = json.dumps(
    freeze_manifest,
    indent=2,
    ensure_ascii=False,
    sort_keys=True,
)

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(
        manifest_text
    )

manifest_sha = hashlib.sha256(
    manifest_text.encode(
        "utf-8"
    )
).hexdigest()

with open(
    manifest_sha_path,
    "w",
    encoding="utf-8",
) as fh:
    fh.write(
        manifest_sha
        + "\n"
    )

# ------------------------------------------------------------------
# 6. Display
# ------------------------------------------------------------------

print("\n40A FINAL RESULTS REGISTRY — SELECTED HEADLINES")
display(
    registry.loc[
        registry[
            "section"
        ].isin(
            [
                "model_performance",
                "clinical_baseline",
                "renal_marker_ablation",
                "renal_marker_ablation_dca",
                "explainability",
            ]
        )
    ].reset_index(
        drop=True
    )
)

print("\n40A FREEZE MANIFEST SHA-256:")
print(
    manifest_sha
)

print("\nSaved:")
print(
    registry_path
)
print(
    reporting_notes_path
)
print(
    manifest_path
)
print(
    manifest_sha_path
)

print(
    "\n40A PASS: eICU AKI statistical/modeling analysis is now FROZEN "
    "for manuscript assembly."
)
print(
    "No further eICU model retuning or test-driven modification is permitted."
)
print(
    "Any future independent external validation is a new validation phase "
    "and must not alter these frozen eICU results."
)